# Autotuned tiled matmul

The tile sizes that make a matmul fast depend on the GPU and on the problem shape, so Triton lets you list candidate configurations and benchmark them on first use with `@triton.autotune`.

Write `matmul(a, b)` returning `a @ b` from an autotuned Triton kernel.

## Inputs
- `a`: `float32` CUDA tensor `(M, K)`; `b`: `float32` CUDA tensor `(K, N)`. Any sizes $\ge 1$, not necessarily multiples of the tile sizes. Use the tensors' strides; do not assume contiguity.

## Output
- `float32` tensor `(M, N)`.

## Requirements
- A 2-D grid over output tiles: program `(pid_m, pid_n)` computes `C[pid_m*BM : +BM, pid_n*BN : +BN]`, looping over `K` in steps of `BK` and accumulating in a `float32` tile with `tl.dot`.
- Decorate the kernel with `@triton.autotune(configs=[...], key=["M", "N", "K"])` and at least three `triton.Config`s that vary the block sizes, `num_warps` and `num_stages`. The grid must be a function of the chosen config (`lambda meta: ...`).
- All block sizes at least 16 (a `tl.dot` requirement). Mask loads with `other=0.0` on both the M/N and K edges, and mask the store.
- Pass `input_precision="ieee"` to `tl.dot` so float32 inputs are not rounded to TF32 (the tests compare against exact float32 maths).

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def matmul(a, b):
    """a: (M, K), b: (K, N) float32 CUDA tensors. Return a @ b, (M, N), from an autotuned Triton kernel."""
    raise NotImplementedError


# @triton.autotune(configs=[triton.Config({...}, num_warps=..., num_stages=...), ...], key=["M", "N", "K"])
@triton.jit
def _matmul_kernel(a_ptr, b_ptr, c_ptr, M, N, K,
                   stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
                   BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0], [3.0, 4.0]], "dtype": "float32"}, {"__nd__": [[5.0, 6.0], [7.0, 8.0]], "dtype": "float32"}], "e": {"__nd__": [[19.0, 22.0], [43.0, 50.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 0.0, -1.0]], "dtype": "float32"}, {"__nd__": [[2.0], [3.0], [4.0]], "dtype": "float32"}], "e": {"__nd__": [[-2.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.39530128240585327]], "dtype": "float32"}, {"__nd__": [[0.26391488313674927]], "dtype": "float32"}], "e": {"__nd__": [[-0.10432589054107666]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.6071282625198364, -0.9721598029136658, 0.7676642537117004, 0.2550581097602844, 0.7829798460006714, 0.27241823077201843, 1.1620081663131714, -0.937756359577179, 1.776099681854248, 1.2023898363113403, -0.5999578237533569, 0.660166323184967, 0.44476738572120667, -1.7457635402679443, 0.5952162742614746, -0.5853990912437439, -0.250013530254364, -0.6023864150047302, -0.42808401584625244, 0.0717347040772438, 0.09671912342309952, -1.5591518878936768, -0.2687060236930847, -1.344811201095581, -1.2707778215408325, -0.3469536006450653, 0.85530024766922, 0.6308565139770508, -0.6046580672264099, -0.7104208469390869, -0.8274099230766296, 0.14274021983146667, 0.9366500377655029, 0.018049856647849083, 0.6927666664123535, 0.3154090642929077, 1.504185676574707, -2.006690740585327, -2.1310341358184814, -0.1984395831823349, 0.6365395188331604, -0.4105492830276489, 0.42578068375587463, -1.1735447645187378, -1.259383201599121, -0.7165758609771729, 0.8413227200508118, 0.6963583827018738, -0.7132527232170105, 0.18111936748027802, -0.6475708484649658, 0.02005981095135212, -1.1142964363098145, -0.04655434936285019, -0.7357539534568787, -0.04435950890183449, -0.6520426273345947, -0.8217414617538452, 0.11343128979206085, -0.33779534697532654, 0.021671175956726074, -0.41908517479896545, 0.6137428879737854, -0.8156847357749939], [0.7687510251998901, 1.3319742679595947, -0.2688876986503601, 1.7452430725097656, 2.029698133468628, 0.4430750012397766, -1.3576050996780396, -1.6156939268112183, -0.17577868700027466, 1.7759298086166382, 0.5745498538017273, 0.758030891418457, -0.2509133219718933, -0.7441468834877014, 0.3719210922718048, -1.108778715133667, -0.19694438576698303, 0.5111522674560547, 0.03593415394425392, 0.5245642066001892, -0.445976585149765, 0.598179817199707, 0.40465039014816284, -1.649706244468689, -1.0422629117965698, -0.7234762907028198, 0.655586302280426, 0.3020801544189453, 0.2829882800579071, 0.172932431101799, 0.636476457118988, 0.9453478455543518, -0.554716169834137, -1.5909122228622437, 0.6673244833946228, -0.20942920446395874, 0.04880537837743759, -0.17380526661872864, -1.1125682592391968, -1.5106513500213623, 0.9642801880836487, 1.0930925607681274, 0.931239664554596, 0.20823465287685394, 0.2685207724571228, -0.41892966628074646, 0.06801223754882812, -0.4028360843658447, -0.5436861515045166, 0.20520316064357758, 1.6981004476547241, -1.883541464805603, 0.6015112996101379, 0.6756512522697449, -1.214406132698059, 0.3134499490261078, 0.10157496482133865, -0.4639827311038971, -0.4963243007659912, -1.4061732292175293, 1.707177758216858, 2.347567558288574, 1.5863759517669678, -0.28779324889183044], [0.28568199276924133, 0.3772081434726715, -0.8129329681396484, -0.25851336121559143, 0.5332453846931458, -0.4216229319572449, -1.759756088256836, -1.2680186033248901, 0.2549883723258972, -0.33706140518188477, -0.2982936203479767, 1.3559778928756714, 0.08478662371635437, -0.7174362540245056, -0.06496331840753555, 2.1579108238220215, -1.1264636516571045, 1.786948323249817, 1.4415949583053589, -0.16009530425071716, -0.33588939905166626, 1.4188083410263062, 0.24397017061710358, -0.5308330059051514, 1.8227617740631104, 0.17449237406253815, 0.3321458697319031, 0.03851839900016785, -0.3013107180595398, 0.12420020252466202, -0.3700959384441376, 0.5075311064720154, 1.055453896522522, 1.085198163986206, 1.1913931369781494, 1.4954686164855957, 0.7610350251197815, -0.6044836044311523, 0.9621555209159851, -1.6492067575454712, -0.7133713364601135, 0.28996866941452026, -1.6823936700820923, 1.4718807935714722, -1.0173481702804565, -1.4978022575378418, -0.7614302039146423, -0.7640095949172974, -0.026283925399184227, -0.15793462097644806, 1.031081199645996, -0.1664392650127411, 0.17555119097232819, -0.9899791479110718, 0.4972487688064575, -0.7749791145324707, -1.948292851448059, 1.3038350343704224, -1.056431770324707, -1.0897020101547241, -0.8766761422157288, 0.651117742061615, -1.5234259366989136, -0.39813995361328125], [0.5768841505050659, 1.8969395160675049, -0.28968507051467896, 0.05582650750875473, -0.8672504425048828, 1.981290578842163, -0.1405663788318634, -0.06256305426359177, 0.1467183381319046, -0.9161278009414673, -0.010194802656769753, -1.2612591981887817, 1.4888421297073364, -0.14947304129600525, -0.16486458480358124, 1.1681957244873047, -1.2158558368682861, 1.4012563228607178, 1.5222817659378052, -0.7692632079124451, 0.6907475590705872, -0.09757599979639053, 0.8827047944068909, 0.041959505528211594, -0.9001685976982117, 0.6281111240386963, -0.21847528219223022, 0.17924445867538452, 0.976440966129303, 1.215274691581726, 0.6291961669921875, -1.4211156368255615, 0.6944321990013123, -0.30050158500671387, 1.0722490549087524, -0.19584167003631592, 1.425998568534851, 0.5140710473060608, 6.793543434469029e-05, 0.29327264428138733, -1.7639750242233276, 0.5738072395324707, 0.11494091153144836, -1.0967977046966553, 0.4070016145706177, 0.3192577362060547, 0.35762062668800354, -0.11539345979690552, 0.7885177135467529, 0.8014461398124695, 0.09359147399663925, -0.06111520156264305, -0.4537959694862366, -0.595069944858551, 0.611762523651123, -0.5458992719650269, -1.412223219871521, 0.24595977365970612, 1.1074165105819702, -0.2736063599586487, 0.8682156205177307, 0.48149847984313965, -1.0081942081451416, 0.3224697709083557], [-0.10889124125242233, 0.6822238564491272, -2.7856249809265137, 0.8312948346138, 0.7414124608039856, -0.9247534871101379, 1.0697388648986816, -0.15130722522735596, 0.18450219929218292, 0.8662132024765015, 1.3465386629104614, 0.173604816198349, 0.6882915496826172, 0.44626668095588684, -0.7248194217681885, 1.6449662446975708, 0.7813403010368347, 0.15670013427734375, 1.0839478969573975, 0.553984522819519, -0.5421837568283081, 1.016693353652954, 0.07845883071422577, 0.6362290978431702, 0.9726846814155579, 0.9654582738876343, 1.899163007736206, 0.2182973176240921, -0.14668899774551392, -0.7031407356262207, -0.6027897000312805, 0.6930890679359436, 0.4513877332210541, -0.8346272706985474, -0.5825162529945374, -0.40847155451774597, 1.5894498825073242, -0.8972784876823425, 0.39727333188056946, -1.5898147821426392, -0.19121955335140228, 1.4235073328018188, -1.2102335691452026, 1.1903412342071533, -1.8462027311325073, 0.09528207778930664, -1.0633293390274048, 0.01901480183005333, -0.20140987634658813, 0.8510773777961731, 0.3089255690574646, 0.2063516080379486, -0.5036148428916931, 0.48655936121940613, -0.7586924433708191, -0.6019229292869568, -0.5874203443527222, 0.6054588556289673, 2.2983410358428955, -0.02576146461069584, 1.957404613494873, -0.9767245650291443, -0.5579162836074829, 0.5591204762458801], [0.6481375098228455, 0.801069438457489, 1.616340160369873, -0.16196657717227936, 1.1249645948410034, 0.29025769233703613, -2.780587911605835, -0.9478462338447571, -0.48086994886398315, 0.1566597819328308, 0.5753792524337769, -0.22836971282958984, 1.3561968803405762, -0.9229505658149719, 0.021755414083600044, 0.9650008082389832, -1.075555443763733, 1.3008931875228882, 0.9983919262886047, -0.11492114514112473, 0.05099400505423546, -0.04040123522281647, 0.0683920830488205, 1.291157841682434, 0.5613875985145569, -1.3276869058609009, -1.4929348230361938, 0.6474104523658752, -2.5776150226593018, -1.5289541482925415, -0.570802628993988, 0.016263792291283607, 1.3869352340698242, 0.21359480917453766, 3.5430376529693604, 0.2497892826795578, 0.8905138969421387, 1.0490736961364746, -0.07181970775127411, 0.9846828579902649, 0.6284646391868591, 0.26901647448539734, -0.5988590717315674, -0.39397433400154114, 0.82786625623703, -0.0032894716132432222, 0.4351484179496765, 0.43553125858306885, 0.3101162612438202, 0.047154705971479416, 0.6190009713172913, 0.8614473342895508, 1.4039926528930664, 0.5322123169898987, -0.528978168964386, -0.642220139503479, 0.6691192388534546, -1.4880560636520386, -0.485151469707489, 1.0709654092788696, -0.2962842881679535, -0.618796706199646, 1.2623355388641357, -2.6131718158721924], [-0.0430445522069931, -0.42336416244506836, -0.4543963670730591, 0.9381310343742371, 1.5284554958343506, 1.1630781888961792, 0.6534404754638672, 0.35796645283699036, -0.20916731655597687, 0.036517105996608734, -1.5834388732910156, -0.48761776089668274, 0.6277427077293396, -0.08652953058481216, 0.8559591770172119, -1.5165858268737793, -0.5270207524299622, -0.9652652740478516, 1.082616925239563, -0.2285294234752655, -0.2593539357185364, -1.1851441860198975, -0.46822699904441833, -0.9727951288223267, -0.5356854796409607, 2.1979448795318604, -1.1169816255569458, -0.2275369018316269, -0.8591797351837158, 1.3312807083129883, -0.9384475946426392, -0.8350302577018738, 0.39290156960487366, 1.6792041063308716, 0.25836071372032166, -0.541017472743988, -2.5206310749053955, -3.5844645500183105, -0.05720273032784462, 0.048199690878391266, 1.1993061304092407, 0.48744362592697144, -0.2391422688961029, 1.2977532148361206, 1.7428299188613892, -0.6324559450149536, 0.9449775815010071, 1.2103668451309204, -0.7433685064315796, 0.5822667479515076, -0.013637988828122616, 1.1899149417877197, 1.1226704120635986, -2.099175453186035, -0.2080022692680359, 1.387617588043213, 0.5265130996704102, -1.0083043575286865, -0.46258363127708435, -1.0528122186660767, 0.3435070812702179, -1.283640742301941, 1.2366055250167847, 0.8631665706634521], [1.6956835985183716, 1.0935087203979492, 1.0380775928497314, 0.9365491271018982, 0.5971190333366394, -1.9021104574203491, 0.5834382772445679, -0.7524120211601257, -2.3580312728881836, 1.6068954467773438, 0.026157980784773827, -1.5893728733062744, 0.3385200798511505, 0.4669429361820221, 0.22740769386291504, 1.5045219659805298, -1.0760059356689453, 0.9320504665374756, -0.9419798851013184, 0.5046001076698303, 0.6818916201591492, -0.4501282274723053, 2.5535380840301514, -1.7746614217758179, -1.4509872198104858, -0.4346076548099518, -0.8626020550727844, 1.1667430400848389, -0.5529673099517822, 1.3640046119689941, -0.4121207892894745, 0.836799144744873, 0.8300654888153076, 0.14660567045211792, 0.7459577322006226, 0.380903959274292, 2.1447653770446777, 2.410651445388794, -0.30921292304992676, 0.5258227586746216, 1.1899629831314087, 0.6051384806632996, 2.0072734355926514, 2.677976131439209, -1.9439350366592407, -0.511328935623169, 1.3721483945846558, 0.5701846480369568, -0.6720284223556519, -0.19949555397033691, 0.1727953553199768, 1.6986083984375, -0.9121341109275818, 0.2549067735671997, 1.5241605043411255, 1.0937745571136475, -0.0931420624256134, -0.23793412744998932, -1.7494406700134277, -1.6729521751403809, -0.8457815647125244, 0.8755425214767456, 0.8541547060012817, 1.3367605209350586], [0.9500565528869629, 0.7979133129119873, 0.20529170334339142, -0.7635186910629272, 1.2554240226745605, -0.34386709332466125, -1.7838717699050903, 0.4823305308818817, 0.12101549655199051, 2.466702938079834, -0.23714642226696014, 0.38016489148139954, -1.1281780004501343, 0.7105526328086853, 0.28890636563301086, -0.7916283011436462, 0.2642103135585785, -1.2588025331497192, 0.5971057415008545, -0.44453105330467224, 0.374627947807312, -0.8981724977493286, -0.7606667876243591, -0.15165750682353973, 0.8465197682380676, 0.14029012620449066, -2.3733479976654053, 0.6793525218963623, -0.10356657952070236, 0.644068717956543, 1.160842776298523, 0.8357692956924438, -0.7258838415145874, -0.3577231764793396, 0.5077568888664246, 1.0190932750701904, -0.7686432600021362, -0.6319220662117004, -0.9879582524299622, 0.4113899767398834, -0.7186739444732666, 2.3346049785614014, 0.8425583839416504, 1.195311427116394, 0.6160955429077148, -0.5910398364067078, -2.5523478984832764, 1.470738172531128, 0.3168754577636719, -1.8925117254257202, 2.857417345046997, 0.7595958709716797, 1.6143240928649902, -0.23962128162384033, -1.7735244035720825, -0.7770264148712158, 0.8052572011947632, -2.1145687103271484, -0.5269166827201843, 1.1634153127670288, 1.0872116088867188, 0.08793649077415466, -2.569814920425415, -0.25493672490119934], [1.3037420511245728, 0.8854002356529236, -0.3881882131099701, 0.39085161685943604, 0.5884590744972229, -0.38637521862983704, -0.7310280203819275, -0.6946631669998169, -0.8359198570251465, -0.06688866764307022, -0.030821338295936584, 0.5965975522994995, 0.6009596586227417, -0.12646372616291046, -2.136857509613037, 0.5723321437835693, -0.8873356580734253, 1.7576425075531006, -0.2896389663219452, 2.2650320529937744, 0.6562109589576721, 1.6972239017486572, 0.2148139327764511, 0.5325136780738831, -0.407450795173645, -0.4558207094669342, -0.4552024006843567, 0.7678586840629578, -1.1112866401672363, 1.0416642427444458, -0.24414706230163574, 0.6470335125923157, -0.9680948853492737, 0.07737495005130768, -1.6943919658660889, -0.6313762068748474, -0.18627436459064484, -1.0059627294540405, -0.223226860165596, 0.12271004170179367, 0.14007771015167236, -2.0414700508117676, -0.30083709955215454, -0.21195778250694275, -0.8498441576957703, -0.9611077308654785, 1.9240530729293823, 0.8394057750701904, 0.9947872161865234, 1.195285439491272, -0.9367231726646423, -0.25608575344085693, -1.7697302103042603, 0.18869368731975555, -1.9147093296051025, -0.7230842113494873, 0.7446659207344055, 0.5401005744934082, -0.9814414978027344, -0.8986513018608093, 1.3031266927719116, 0.44594690203666687, -0.5042356848716736, 0.1442878097295761], [0.1238861083984375, -1.076439380645752, -0.554142415523529, -2.1052255630493164, 0.23808343708515167, 0.08277343213558197, 1.0373444557189941, 1.0296210050582886, -0.45418041944503784, 1.6009646654129028, -0.7315324544906616, 0.939730167388916, -0.5193395614624023, -0.620844304561615, -1.2433080673217773, -0.06228756159543991, 2.256570339202881, 0.15589439868927002, 0.4730592966079712, -0.3394375145435333, 0.5049340724945068, 0.6251394748687744, 0.4784151613712311, 1.3767495155334473, -0.02247784100472927, 0.42109084129333496, 1.5518392324447632, 0.6295466423034668, -0.744873046875, 1.281235933303833, 0.6465003490447998, -0.7998771071434021, -0.7727622985839844, -0.3857782781124115, 0.8346342444419861, 0.34398818016052246, 0.9958153367042542, 2.4794325828552246, 1.747847318649292, 0.12441655248403549, -0.6500998735427856, 1.407261610031128, -0.694019079208374, -0.7654364705085754, -0.10100942105054855, 0.2969019412994385, -0.24168118834495544, 0.27938079833984375, -2.6746253967285156, -0.3541891574859619, 0.04927453771233559, 0.7242475748062134, 0.3641577363014221, -0.21119356155395508, 2.510408639907837, 0.999980628490448, 0.6375816464424133, 0.33139145374298096, 0.4794115126132965, -0.8001515865325928, -1.0052052736282349, 0.24999484419822693, -1.5738492012023926, 0.6134696006774902], [-0.3702804744243622, 0.72633957862854, 0.13713262975215912, -0.8094255924224854, -0.6870294213294983, -0.39463916420936584, 1.3211604356765747, 1.3954188823699951, -0.17094288766384125, 0.15117396414279938, -0.7797881364822388, 1.4296504259109497, 0.2601883113384247, -0.6279731392860413, -1.2920438051223755, 0.5351314544677734, 1.8897227048873901, 0.10749457776546478, -0.8788115382194519, -0.8536117076873779, -0.5556792616844177, -2.2601118087768555, 0.9099363088607788, -0.560704231262207, -1.8212615251541138, -2.9268746376037598, 0.345007985830307, 0.21463781595230103, 1.4253185987472534, 0.9984395503997803, -0.40859562158584595, -1.7188010215759277, 0.20790106058120728, -0.5753607749938965, 0.08453218638896942, -1.5889387130737305, -0.2998954653739929, -0.22755073010921478, -1.2832175493240356, -1.75264310836792, -2.219750165939331, 0.07247970998287201, 1.2008987665176392, -0.040300674736499786, -2.580258369445801, -0.05143554508686066, -1.6800457239151, 0.23618084192276, 1.7267365455627441, 0.34548091888427734, -2.5918996334075928, 1.0434794425964355, 0.32806137204170227, 1.8798422813415527, 0.49978941679000854, 0.6902827620506287, -1.799019694328308, 1.4237289428710938, -1.2426913976669312, -2.018174648284912, 0.5033662915229797, 0.7265804409980774, 1.4438488483428955, 1.1507611274719238], [0.36703798174858093, 0.32908591628074646, 0.8603413105010986, 0.35934245586395264, 0.4574005901813507, -2.0337002277374268, -0.1102357879281044, -0.36842232942581177, 0.5084225535392761, -0.7955746054649353, 0.9359046220779419, 0.7447285652160645, -1.1264729499816895, 0.025520415976643562, 0.5436481237411499, -1.084868311882019, -2.4155595302581787, -0.2500218152999878, 0.2444121539592743, 0.017199071124196053, 1.0841052532196045, 0.29803767800331116, 1.6115491390228271, -0.570375382900238, 1.6535919904708862, -0.2910756766796112, 0.5009312033653259, -1.8803141117095947, -0.7762830257415771, 1.1902472972869873, 0.06678731739521027, 0.010361981578171253, -1.2082505226135254, 0.7952855229377747, -0.9990519881248474, 1.4648138284683228, 0.6153947710990906, -0.5622377991676331, -0.8193256855010986, -0.5262561440467834, 0.37818050384521484, 0.2504798173904419, -1.619558572769165, 0.34786733984947205, -1.0747138261795044, 1.8406562805175781, -1.8914341926574707, 0.9392959475517273, -1.197153925895691, 0.38289394974708557, 1.1850558519363403, 0.4724362790584564, -1.1781024932861328, -1.5447272062301636, 1.4815179109573364, -0.8105404376983643, 0.7845354080200195, 1.2006276845932007, -1.6609454154968262, 1.9807394742965698, -0.1273321956396103, -0.19877490401268005, -0.9882063865661621, 0.23311442136764526], [0.48385265469551086, 1.056944489479065, 1.0433348417282104, 1.0324852466583252, -0.11037301272153854, -0.09188999235630035, 0.12951816618442535, 0.19163034856319427, 1.2744083404541016, 0.9260583519935608, -0.16222140192985535, -0.9224324822425842, -1.916060209274292, -0.21795952320098877, 0.019501537084579468, -1.8780906200408936, -0.3607490062713623, 0.2546367645263672, 0.21745382249355316, -1.4168403148651123, -0.6711909174919128, -1.4620904922485352, 0.8534194231033325, 0.2778284251689911, 2.608417510986328, -0.6026420593261719, 1.5243656635284424, 0.03268630430102348, 0.08564428240060806, 1.0573983192443848, -1.9373326301574707, -1.1975990533828735, 1.5261446237564087, 1.0069515705108643, 1.8757092952728271, 0.5658403635025024, -0.47745105624198914, 0.8596386313438416, -0.4365011155605316, -0.06746035814285278, -0.12113279849290848, -0.8670384883880615, 2.1249022483825684, 0.7252494692802429, 0.13607437908649445, 0.3960166573524475, -0.5698494911193848, -0.3717188537120819, 0.12299338728189468, 0.32667025923728943, -2.1719141006469727, 1.3202437162399292, -2.0562822818756104, -0.9680041074752808, -1.3976435661315918, 0.10635692626237869, -1.7719594240188599, 0.9893237352371216, 0.6910830140113831, 0.9293277263641357, 0.7979755401611328, -0.3330443799495697, 0.24175015091896057, -0.96907639503479], [-0.4400922954082489, 0.43155866861343384, -1.2734566926956177, -0.3043397068977356, -0.5072780251502991, 0.888058066368103, 0.5288904905319214, -0.4425778090953827, 0.7964950799942017, 1.9388543367385864, 1.096276879310608, 0.0933796763420105, 0.12725476920604706, -0.973216712474823, -0.5435022711753845, 0.27907440066337585, 0.3771715462207794, -1.4315332174301147, -1.252223014831543, -1.2332903146743774, -1.160212516784668, -0.6441863179206848, -0.16845296323299408, -1.1936999559402466, -0.7124638557434082, -1.994870901107788, -0.2539384961128235, 0.2153652161359787, -0.5748698115348816, -1.5984984636306763, 2.36599063873291, -0.8623560070991516, 0.387052059173584, -2.224165201187134, -0.7924401760101318, 0.09402283281087875, -0.1259334832429886, 0.10445970296859741, 0.0061616250313818455, -0.7984717488288879, -0.9546368718147278, -0.2424338012933731, -0.7110884189605713, -0.658466100692749, 0.3487228751182556, 0.16605930030345917, -0.5205350518226624, 1.0163006782531738, -0.13195392489433289, -0.3093743920326233, 0.760957658290863, 0.181550532579422, -1.6489514112472534, 0.3982948064804077, -1.6703999042510986, -0.3820629119873047, 0.2241460233926773, 0.5866102576255798, -0.9699174761772156, -0.4309283196926117, 0.7094202041625977, 1.9471172094345093, 1.902819275856018, 0.7833452224731445], [1.6065279245376587, -0.5948908925056458, -0.2795731723308563, 0.7550939321517944, 0.7547466158866882, -1.5758415460586548, 1.1248247623443604, -0.9019489884376526, -0.15830595791339874, -1.197464942932129, -0.39931556582450867, 1.016038417816162, -1.3385413885116577, -0.6790691018104553, -0.038533806800842285, 0.5473030209541321, 1.6678781509399414, 0.08769802749156952, 1.0137735605239868, -1.5906102657318115, -0.5659031867980957, -2.4218907356262207, 1.1161198616027832, -0.6145498156547546, 0.5712592601776123, -1.093191146850586, -1.1368608474731445, -1.0485683679580688, -0.749842643737793, -0.34458667039871216, -1.7575623989105225, 0.0414111390709877, 0.5199729800224304, -1.861875295639038, -1.5957642793655396, -1.197749376296997, -0.9671710729598999, 1.6798584461212158, 0.6902992725372314, -1.3196817636489868, 2.663024663925171, 0.7475103735923767, -1.3248032331466675, -1.743302822113037, 0.3325316905975342, -1.7233796119689941, 0.10282749682664871, 0.8172793388366699, -0.9056351780891418, -0.8009445071220398, -0.8977340459823608, 1.248950481414795, 0.5956138968467712, 0.4380834996700287, 0.7920514345169067, -0.07773006707429886, -0.6983513236045837, 0.2639898359775543, 0.8653379082679749, -0.20127259194850922, -0.26872262358665466, -0.9930107593536377, 1.352907419204712, 0.7305850982666016], [0.17557495832443237, 0.16556747257709503, 0.3526182472705841, 0.06217703968286514, 0.9667825102806091, 0.17878633737564087, 0.7529708743095398, 0.05850669741630554, -0.5802467465400696, 1.1388977766036987, 1.1667718887329102, 1.5389353036880493, -0.4277893900871277, -0.6329684257507324, -1.8614089488983154, 0.05647444725036621, -0.8268758654594421, 1.2008172273635864, -1.1632015705108643, 0.9568781852722168, -1.2364312410354614, -0.1177176982164383, 1.2709680795669556, -0.47149476408958435, 0.39011892676353455, 0.7021298408508301, -1.3497400283813477, 2.2599878311157227, -1.3521645069122314, -0.29846620559692383, 0.19918473064899445, 0.41100385785102844, 1.0057315826416016, 1.1937477588653564, -1.3408817052841187, -0.1683889776468277, -2.0791146755218506, 0.2757605016231537, -0.6715865731239319, 0.3488397002220154, -1.7127935886383057, -1.1289076805114746, 0.49591368436813354, 0.7209324836730957, -0.32580047845840454, -0.457854688167572, -0.5625005960464478, -0.5895718932151794, -0.6080636382102966, 2.5658037662506104, 0.7130829691886902, -1.6024072170257568, -0.43358489871025085, -1.024545431137085, 0.15823616087436676, 2.1363377571105957, 1.242508888244629, 0.14431312680244446, 0.4005371332168579, -0.12533962726593018, -1.0109376907348633, -0.597538411617279, 1.1114342212677002, -1.5791317224502563], [1.2760738134384155, 2.3408937454223633, 0.07145849615335464, 1.137880802154541, 0.1933186650276184, -0.27564772963523865, -2.037242889404297, -0.7284643054008484, 1.559209942817688, 0.08894914388656616, -0.310205340385437, 1.5832279920578003, -0.0630025863647461, -1.9606120586395264, -0.5291662812232971, 1.5230520963668823, 1.1610426902770996, -0.316072016954422, 0.8403627872467041, -0.09827636927366257, 2.4841859340667725, 0.7708772420883179, 1.2341911792755127, 0.5275670886039734, 1.1437045335769653, 0.74323970079422, 0.16990184783935547, -0.3748229742050171, -0.26700758934020996, -0.0444413498044014, 0.8545214533805847, 0.7438057065010071, 1.4893542528152466, -0.08204462379217148, 0.28158095479011536, -2.0109128952026367, -0.6208733320236206, -0.5788090229034424, 0.06847189366817474, 0.1984790861606598, -0.4194193482398987, 0.4534952640533447, -0.4088594317436218, 0.0363771952688694, -0.5812825560569763, 0.23909351229667664, -0.551124095916748, -0.18548189103603363, -0.7710667848587036, -1.3114919662475586, 1.3195737600326538, -0.7346956133842468, 1.0865870714187622, -0.32012486457824707, -0.015232699923217297, 0.0892879068851471, -0.8618122339248657, -1.6927355527877808, 0.7885525822639465, 0.22052112221717834, -1.636721134185791, -0.7226472496986389, -0.6365121603012085, 1.0886918306350708], [0.5133900046348572, -0.018810195848345757, 1.1033525466918945, 0.4072924852371216, -0.7357279658317566, -1.3951853513717651, 0.9713966846466064, -1.7120462656021118, 1.1864359378814697, -1.7992291450500488, -0.6686965227127075, 1.211043119430542, 0.8755678534507751, 1.159079909324646, -1.4996263980865479, -0.31722262501716614, -0.07410409301519394, -0.42930421233177185, -0.6066575050354004, 0.27643337845802307, -0.01000590343028307, -0.9770870208740234, -0.5885673761367798, -0.2868419885635376, -0.05276638641953468, -0.28219765424728394, -1.5538517236709595, 0.06785271316766739, 0.4814397394657135, 1.5541962385177612, -0.5245168805122375, 0.8973386883735657, -0.5631275177001953, 1.2553337812423706, -0.13288670778274536, 0.10351976752281189, -0.2072608917951584, 0.1613154113292694, 0.19005785882472992, -1.3765336275100708, 0.6732777953147888, -1.0067921876907349, -0.16962914168834686, 0.9558424949645996, -0.20405058562755585, -0.6415495276451111, 1.6983895301818848, 1.4190880060195923, 1.4384448528289795, 1.0718811750411987, 0.5219932794570923, 0.24017846584320068, -1.4156137704849243, -0.9019532203674316, -0.8231095671653748, 1.1121766567230225, 2.4439971446990967, 0.8278530240058899, -0.170423686504364, 1.4081063270568848, -0.03601275756955147, -0.20406493544578552, -0.04072649031877518, 1.1453052759170532], [1.2806695699691772, 0.32934287190437317, -1.0143349170684814, 0.8835310339927673, -0.7706426978111267, -0.902008056640625, -0.5607842206954956, -1.0693374872207642, -0.7160670757293701, 1.7539781332015991, -0.9805360436439514, -0.07138761878013611, -1.5117987394332886, -0.821042537689209, -0.3673943877220154, 0.10949774831533432, 0.10488089919090271, -0.07412540912628174, -0.7119231224060059, 1.4907256364822388, 0.7163267135620117, -0.5494666695594788, -0.241074338555336, -0.455698698759079, -0.022792333737015724, -0.6422622203826904, 0.7384718060493469, 0.9447696805000305, -0.12790685892105103, 0.36719992756843567, -0.5758433938026428, 1.2699543237686157, -0.3965204954147339, 0.5071042776107788, 0.37386178970336914, -0.9333222508430481, -1.8182196617126465, -0.762016773223877, -1.0573756694793701, -0.6522806882858276, -0.08684450387954712, -0.09038156270980835, -0.5495466589927673, 1.7365121841430664, -1.2332539558410645, 0.9479604959487915, 0.7367309927940369, -0.8318459987640381, 2.0073864459991455, 1.3258764743804932, 0.33327996730804443, 0.4792458713054657, 0.07478315383195877, 0.6380620002746582, -0.4972573518753052, 1.1710660457611084, 1.0253074169158936, -0.3666056990623474, -1.485581874847412, 1.1606673002243042, 0.20228974521160126, -0.8818290829658508, 1.4737221002578735, 0.22606192529201508], [-1.5169410705566406, 0.9616043567657471, -1.1634855270385742, -1.9017298221588135, -0.7434409260749817, -0.21291783452033997, -1.8864433765411377, 2.3671247959136963, 0.28997206687927246, 0.20578743517398834, -0.20330671966075897, 0.29932501912117004, -0.8330298066139221, -0.28392890095710754, 1.1698709726333618, -0.2862740159034729, 0.22469429671764374, -0.2165294587612152, 1.9224507808685303, 1.23215913772583, 0.4450613856315613, 0.7926515936851501, 0.37791576981544495, 0.33248457312583923, 2.796947479248047, -2.2408392429351807, -0.14594390988349915, 0.06401021778583527, 0.9883536696434021, -1.517061710357666, -0.6423249244689941, 0.4277935326099396, -1.9352651834487915, 0.5749775767326355, 2.235380172729492, -0.9469984173774719, -2.065579652786255, 0.7013248801231384, -0.27445995807647705, 2.143908977508545, -0.39855459332466125, -0.6417160034179688, 1.2213025093078613, 1.331790566444397, 1.1363247632980347, 2.3492887020111084, 0.625083863735199, 0.07349542528390884, -0.5230858325958252, -0.10912200808525085, -0.45288896560668945, -0.5412031412124634, -1.0778834819793701, -2.388991594314575, 0.4621421694755554, 0.4371509850025177, -1.7099487781524658, 0.9079504609107971, -2.8377530574798584, -0.5012834668159485, 0.1580505520105362, -0.35745254158973694, 0.7663711905479431, 1.2538657188415527], [1.242722988128662, 0.13456971943378448, 1.6645251512527466, -0.3650932312011719, -0.5505802035331726, -0.6419580578804016, -0.7297185063362122, -0.41791266202926636, -1.3465346097946167, -1.6593711376190186, 1.5958982706069946, 0.6155574917793274, 1.0175031423568726, 0.3708692193031311, -1.0681431293487549, 1.3424733877182007, 0.7730174660682678, -1.227430820465088, 0.7871042490005493, -2.1277594566345215, 1.047837495803833, -0.8636338710784912, 1.0577000379562378, -0.8564956188201904, 1.1116591691970825, 0.38216203451156616, 0.6298742890357971, 0.30002835392951965, 0.5550928115844727, 0.7000190019607544, 0.8188809156417847, -2.3078105449676514, -0.3860924243927002, -0.5445693135261536, -0.3070026934146881, -0.14081896841526031, 2.068286418914795, 0.5758486390113831, -1.4435968399047852, -0.4586981534957886, 1.001440167427063, -1.1170812845230103, 0.23886314034461975, -0.5986393094062805, 0.37026718258857727, -0.2082880139350891, -0.03453323245048523, -0.2704053521156311, -0.8501878976821899, 1.2763862609863281, 0.39090201258659363, 0.127121239900589, 1.5881447792053223, 0.28787338733673096, -0.030354520305991173, 0.22298510372638702, -0.39855870604515076, -0.8161457777023315, 1.89461088180542, -0.8175868988037109, 1.278489351272583, 0.4841803312301636, -1.173993706703186, -0.2429685890674591], [-0.7764348983764648, -0.06656701862812042, 0.8569794297218323, -0.9269225597381592, -0.9074987769126892, -1.2219282388687134, 1.498293161392212, 0.12948356568813324, 0.34813180565834045, 0.1666726917028427, -1.169100046157837, 1.6975351572036743, -1.141953945159912, -2.238913059234619, 0.7431966066360474, -0.929270327091217, 1.0217982530593872, 0.714447021484375, 0.11888036131858826, 0.6983495354652405, 0.770033061504364, -0.5231799483299255, -0.7880510091781616, -1.329290509223938, -0.43187710642814636, -0.7718830704689026, -1.1236203908920288, -0.7599384188652039, 2.5893547534942627, 0.651188313961029, -1.4156911373138428, 1.6827160120010376, -1.8601897954940796, -0.136321023106575, -1.1217654943466187, -1.108107089996338, -0.5309687852859497, 0.4733832776546478, 1.1485931873321533, -1.897813320159912, -0.4746876358985901, -0.895353376865387, 0.5254103541374207, 0.8141991496086121, -1.3237956762313843, -0.8701514005661011, -0.8324870467185974, 0.7285921573638916, -0.49559661746025085, -0.8622303605079651, -0.7319139242172241, -0.7962369322776794, 0.07172638922929764, 1.7861484289169312, -1.816834568977356, -0.7733543515205383, 0.3606375455856323, -2.3214900493621826, 0.3697643280029297, 0.3699345886707306, -0.0044198487885296345, -1.419907569885254, 1.797398328781128, 0.16916128993034363], [-0.2941310703754425, 1.1454647779464722, -1.6133061647415161, -0.8350261449813843, -1.974084734916687, -0.23795346915721893, 0.07282033562660217, -0.37092065811157227, -0.2899620532989502, 0.6528443694114685, -0.73223477602005, -0.35075896978378296, -1.3670276403427124, 0.5532200932502747, -0.3557664155960083, 1.6331441402435303, -0.6340839266777039, 0.5070893168449402, 0.3408569395542145, -1.3321740627288818, 0.7329670786857605, 0.4675173759460449, 0.9785704612731934, 0.8971565365791321, -0.31600746512413025, 1.5839923620224, 0.40332767367362976, 0.7881970405578613, -2.0838770866394043, -2.0058677196502686, -1.8160291910171509, -0.11146848648786545, -0.02098330296576023, 0.5927072763442993, 0.06871888786554337, -1.7125850915908813, -0.4834531247615814, 1.4651403427124023, 1.0944557189941406, -1.6030099391937256, -0.32791799306869507, -1.4845526218414307, 0.39017534255981445, 1.028442144393921, 0.8197654485702515, -1.1034584045410156, -0.5302280187606812, -0.16193526983261108, 1.7558610439300537, 0.3010441064834595, 0.11188463866710663, 0.13028736412525177, 0.21534182131290436, -0.8566216826438904, -1.1431984901428223, -0.5524165034294128, -0.30635273456573486, -0.6523127555847168, 0.7024654746055603, 0.01721538044512272, 0.7682072520256042, -0.7387204170227051, -2.502399206161499, -0.5335042476654053], [0.8856715559959412, 0.18127025663852692, -2.5932250022888184, -0.5830081701278687, -0.7456207871437073, -0.3103739023208618, -0.723774790763855, -0.09977691620588303, -0.6369823813438416, -0.5175726413726807, 0.9211836457252502, 0.23797492682933807, -0.1975410431623459, -1.4349889755249023, -0.8966085314750671, -0.6369910836219788, 0.4017646610736847, -1.6777031421661377, 0.22319309413433075, 1.3716219663619995, 0.2082240730524063, -1.4136356115341187, 0.9913966059684753, 0.01726238988339901, 0.6399971842765808, 0.9502632021903992, 0.6627886891365051, -0.1418604552745819, -0.8931775689125061, 2.2415056228637695, -0.8269386887550354, 0.9045872092247009, -0.9062013626098633, 1.308894395828247, 1.9591957330703735, 0.9125440716743469, -0.7150051593780518, -0.31525719165802, 1.524300456047058, 1.5398415327072144, 0.6837772727012634, 0.27478545904159546, 1.631017804145813, 0.5903589725494385, 0.38863691687583923, 0.9356130361557007, 0.42686840891838074, 1.2409826517105103, 1.5597609281539917, 0.5542755126953125, -2.318284273147583, 0.061330683529376984, -0.42365649342536926, -1.1071279048919678, -0.4453732371330261, 1.3659889698028564, -1.397536039352417, -0.3309434950351715, -0.6956726312637329, -1.4850434064865112, -1.481382131576538, 0.5644018054008484, -1.3105069398880005, 1.8737797737121582], [-0.17642158269882202, 0.3696555197238922, 0.5792036652565002, -1.5607069730758667, 1.085168719291687, 0.18349777162075043, -0.05573485419154167, 0.5815010666847229, 0.19042985141277313, 0.06299126893281937, -1.226818561553955, -0.09675292670726776, 0.6513888239860535, -0.7975999712944031, 0.8178561925888062, -0.32976946234703064, 0.5654786229133606, 0.0629562959074974, -1.0139226913452148, -0.19818051159381866, -0.5546740293502808, -0.8010703325271606, -0.8306614756584167, -2.13759183883667, -0.3161543011665344, -0.15853561460971832, -1.0643061399459839, -1.611366868019104, 0.6388095021247864, 0.3449711799621582, 1.1053842306137085, 1.0607335567474365, -0.9872229695320129, 0.006011100485920906, -0.14337396621704102, 0.8788109421730042, -0.9638579487800598, -0.9212531447410583, 1.2514957189559937, -1.098459005355835, 0.10447307676076889, 0.17372959852218628, -1.4837582111358643, -0.7096300721168518, -0.8622817993164062, -0.5376877188682556, -1.0777266025543213, 0.9172459244728088, -0.3221224844455719, 0.9326865077018738, 0.8435156345367432, -1.741216778755188, 0.8417612314224243, -0.5435598492622375, 0.7108392119407654, 0.9810962677001953, 1.300045132637024, 0.9913492202758789, -0.12238753587007523, -0.590514063835144, 0.11537322402000427, 0.24891464412212372, 0.05770569294691086, -0.20709142088890076], [0.5895617604255676, -0.38739392161369324, 0.27945199608802795, 0.1662137359380722, 1.1289621591567993, -0.580706775188446, -0.590343177318573, -0.8522908687591553, 0.7905972003936768, 0.39944154024124146, -0.2521337866783142, 1.731917142868042, -0.8924501538276672, 0.8430669903755188, 0.40805092453956604, -0.3831961452960968, -1.0461496114730835, -0.748110294342041, 0.8560652136802673, -1.433154821395874, -0.27392515540122986, -0.8372604846954346, 0.021118052303791046, -0.7579536437988281, -0.4881409704685211, -1.3631590604782104, 0.15051445364952087, -0.863938570022583, 0.013623547740280628, -1.7913415431976318, -1.2790045738220215, -0.4130018651485443, -0.7944735288619995, -0.25389647483825684, -0.662933886051178, 0.6742764115333557, 0.4666413962841034, 1.043978214263916, 0.6179019808769226, 1.1284162998199463, 1.1188491582870483, 0.3137950301170349, -0.4609452486038208, -0.07417035102844238, 0.13502639532089233, -0.7611386179924011, -0.42137280106544495, 1.014068603515625, 0.7174562215805054, -0.7676675915718079, -0.9818805456161499, -0.5521361827850342, -1.6505910158157349, 0.9940154552459717, 0.4067111015319824, -1.182672142982483, -0.2834731936454773, -0.7048225402832031, 1.2588847875595093, -1.2719426155090332, -0.7016236186027527, 1.593687653541565, 1.6516796350479126, 1.5819830894470215], [0.33990252017974854, -0.45507946610450745, -0.8417765498161316, 0.7698848247528076, -0.7895463705062866, -0.11766523122787476, -0.15935048460960388, 0.9994983077049255, -0.6867736577987671, 0.9862896800041199, -0.5718997716903687, -0.4978431761264801, -1.1066347360610962, 0.24022512137889862, -0.16395355761051178, 0.09595482051372528, -0.3228030800819397, 0.09821973741054535, 1.649096131324768, 0.4677417278289795, 1.0111498832702637, -0.039082810282707214, -0.01975339837372303, 1.1805462837219238, 2.706997871398926, -0.5614765286445618, 0.4950632154941559, -0.4581010937690735, 0.5397771596908569, 0.21497447788715363, -0.535953164100647, -1.5146708488464355, -1.5947984457015991, 1.3328665494918823, 0.08029218018054962, 0.034501221030950546, 0.6044061779975891, -0.0438220240175724, 2.1327152252197266, 0.1625083088874817, 1.040649175643921, 0.22493454813957214, -1.426974892616272, 0.5814372897148132, 0.3071701228618622, -0.4289567768573761, 1.4003562927246094, 0.6868408918380737, 0.8429736495018005, -1.185624599456787, -0.4367891550064087, -1.648008942604065, 0.6263909935951233, 1.0961966514587402, -0.562407374382019, 2.3475914001464844, -1.0561238527297974, 0.27564582228660583, -2.088779926300049, -1.9814151525497437, 0.38466957211494446, 0.14990583062171936, 2.0490009784698486, 0.26843613386154175], [1.357608675956726, -0.7339189052581787, 0.6531333923339844, 1.4412437677383423, 0.12190188467502594, 1.4399827718734741, 1.1341078281402588, 0.3171612322330475, -0.9942716360092163, -0.5715850591659546, -1.6239469051361084, 1.023038387298584, 1.5597801208496094, 0.3450878858566284, 0.4606638252735138, -1.9386208057403564, 1.1480703353881836, -1.32541823387146, -0.5225912928581238, 2.0383994579315186, 0.5231010913848877, -0.7101239562034607, -0.8089675903320312, 0.3188669681549072, 1.1807032823562622, 0.26481893658638, 0.15744300186634064, 0.5361405611038208, 0.7515402436256409, -1.5019824504852295, -1.7881343364715576, 1.336923599243164, -0.6989886164665222, -2.116579055786133, -0.47635364532470703, -0.15916402637958527, -0.32158127427101135, 1.2122578620910645, -1.0972261428833008, -0.042619772255420685, -1.431931495666504, 2.087813377380371, -1.1723508834838867, -1.6459803581237793, 0.7118059396743774, 0.2203470915555954, -1.2629014253616333, 0.5768149495124817, 0.7711194157600403, -0.4352603554725647, 0.5531688332557678, 0.9342617988586426, 0.6839355230331421, -0.47568652033805847, -0.06457053869962692, -0.45926329493522644, -0.417304664850235, -0.06472206115722656, -0.4638548195362091, -0.1706823706626892, -0.1456434577703476, -0.2297898530960083, 0.5316944718360901, -1.180626630783081], [-0.8719335198402405, -0.6028330326080322, -0.8558589220046997, 0.47813329100608826, 0.23166213929653168, -0.06634406745433807, -0.27579572796821594, 0.5284015536308289, 0.8276770114898682, 0.8860006332397461, -0.22702281177043915, 0.0584217794239521, -1.682310938835144, -0.24150313436985016, 2.1277525424957275, -0.7219653725624084, 0.9315422177314758, 0.6167035698890686, -0.9663041234016418, -1.8166522979736328, -0.8082622289657593, 1.2729274034500122, 1.0335557460784912, 0.6294488310813904, -0.4164833426475525, 0.6719072461128235, -1.1388332843780518, 0.8662670850753784, 0.425368994474411, 0.7013697624206543, 3.041808843612671, 2.796520471572876, -0.8405587673187256, 0.24161167442798615, -0.014524023979902267, 0.33684056997299194, 2.3723554611206055, -1.117590069770813, -0.07313593477010727, 1.3581374883651733, -0.5669440627098083, 1.6469626426696777, 0.12543562054634094, -0.06365928798913956, 0.6444167494773865, -0.21281465888023376, 1.023133635520935, 1.7723079919815063, -2.4254024028778076, 1.073793888092041, -1.915401577949524, -1.8193901777267456, 0.8421201109886169, 0.39246222376823425, 0.39966636896133423, -0.8667198419570923, -0.5541118383407593, -0.2867090404033661, -1.631782054901123, 0.5047762393951416, -0.242048978805542, 0.5690425038337708, -2.4422295093536377, -2.5752313137054443], [-0.6026793718338013, -0.24536707997322083, 0.5068963170051575, -0.5248728394508362, -0.822106122970581, -0.9800125360488892, 0.9104640483856201, 0.10770312696695328, 1.7145326137542725, -0.0882062166929245, 0.38826677203178406, -0.11422665417194366, -0.31359055638313293, 0.3381616473197937, -1.193047285079956, -0.5050275325775146, 0.38806551694869995, -0.7648290991783142, -2.157587766647339, -0.5743316411972046, -1.1641981601715088, 0.39249855279922485, -1.2926898002624512, -0.9422195553779602, 0.05112798884510994, -0.472433477640152, 0.5312844514846802, 0.10164063423871994, -0.23883602023124695, 0.5898531079292297, 0.27663856744766235, -0.5996166467666626, 0.9347057938575745, -0.057475704699754715, 0.4167570173740387, -0.09198576211929321, 0.37612175941467285, -0.6218780279159546, 0.40677499771118164, 0.16356946527957916, -0.16659262776374817, 0.03212245553731918, -0.31984782218933105, 0.5216467380523682, 1.5401562452316284, -0.7902487516403198, -0.23368416726589203, -1.454316258430481, -0.3306570053100586, -0.018986856564879417, 0.3833114504814148, -1.2580193281173706, -0.7116373181343079, 0.00971771590411663, 2.8945326805114746, 0.14780673384666443, 0.11597201228141785, -0.7200832962989807, -0.3916977643966675, -2.4376916885375977, -0.7258762717247009, -0.7916591167449951, -0.3798825144767761, -0.4574677050113678], [2.0578134059906006, 0.9482309222221375, -0.12670288980007172, -1.9253596067428589, 1.7139101028442383, -0.5928947329521179, -0.8967542052268982, -0.16522587835788727, -0.17783090472221375, 1.3780368566513062, 0.06516106426715851, 0.44490957260131836, 0.5867453217506409, 1.4881126880645752, 0.6615608930587769, -0.15770094096660614, -0.4221228063106537, -1.4141532182693481, -1.001049518585205, -3.045227527618408, 1.033078670501709, -0.7734594941139221, -0.583071231842041, 0.699135422706604, -0.3691132664680481, 0.2385948896408081, -0.6749811172485352, -0.3245817720890045, 0.004309454001486301, 1.5899133682250977, -1.6316137313842773, -0.8884301781654358, 0.33174246549606323, 0.5779879093170166, -0.35144466161727905, -0.05376312509179115, -0.11737789213657379, 0.6800209879875183, -1.4887759685516357, -0.15335382521152496, 1.3254250288009644, 0.9988937377929688, -0.6531203389167786, 0.41468992829322815, -1.0936578512191772, 0.4876522123813629, 0.39430370926856995, 0.004936356097459793, 0.33719873428344727, -0.23300108313560486, 1.097511649131775, 0.8791930079460144, 0.664338231086731, 1.119281530380249, 1.157707691192627, 0.13879485428333282, 0.6288110613822937, -0.047983959317207336, -1.6968454122543335, 1.198263168334961, -0.14099064469337463, -0.08276232331991196, 2.0166094303131104, -0.14843785762786865], [-0.32232949137687683, -1.7911189794540405, 0.3308659493923187, 0.13306526839733124, -0.6317476034164429, 1.088612675666809, -0.43680039048194885, -1.2928142547607422, -0.6556351780891418, -0.2569425702095032, -0.21911968290805817, -0.007533632684499025, -1.145237922668457, -0.9618180990219116, -1.8902125358581543, -0.1815442144870758, 1.4658130407333374, 0.9782484173774719, -1.4745771884918213, -0.6175504922866821, 1.2035138607025146, -0.4486704170703888, -0.47798028588294983, -1.4043794870376587, 1.7543905973434448, 1.018310546875, -1.0867611169815063, 0.9233622550964355, -0.9715720415115356, 0.08269614726305008, 1.1927883625030518, -0.9237930178642273, -0.9263288974761963, -1.0717871189117432, -0.09740140289068222, 0.7975313067436218, 0.803774356842041, 1.610862374305725, 1.8154600858688354, 0.6281017065048218, -1.5510892868041992, -1.0333486795425415, 0.49007222056388855, 0.18129578232765198, -0.20465540885925293, -0.608925461769104, 0.3759596049785614, -1.5140380859375, -0.27821290493011475, 0.3415115475654602, 1.258568286895752, 1.2598960399627686, -0.7283245921134949, -1.2907634973526, -0.04927917942404747, 0.31073886156082153, -1.5506380796432495, -1.5970622301101685, -0.7613483667373657, -0.20387405157089233, 0.2251632809638977, -0.23103412985801697, -0.11593269556760788, 0.868544340133667], [-0.37449970841407776, -0.41391521692276, -1.6036187410354614, 0.2540867030620575, 0.9545295238494873, 0.585625946521759, -0.896536648273468, 1.0611566305160522, -0.33628013730049133, 1.725168228149414, 0.2218218743801117, 1.16360604763031, 0.1932741105556488, 0.1070946455001831, -0.8070469498634338, -0.763579785823822, 1.0815950632095337, -0.9894365668296814, 1.5696872472763062, -1.7992751598358154, -0.08373437076807022, -0.5624606609344482, 1.6910053491592407, -0.5243498086929321, -0.4891621172428131, -0.8612989783287048, 0.6380282640457153, -0.23994135856628418, -0.08621274679899216, -1.586259365081787, -0.37554654479026794, -0.3536261022090912, -0.7131007313728333, 1.6880460977554321, 2.3780171871185303, -0.12507225573062897, -0.5359636545181274, 0.6569600105285645, 0.754786491394043, -0.952599823474884, 0.5562176704406738, -0.574507474899292, -0.4487312436103821, -0.19477738440036774, 0.865637481212616, 0.11707557737827301, 0.35041582584381104, 0.1451188176870346, -1.8156999349594116, 1.321658730506897, -0.09933432191610336, -0.5501397848129272, -0.9905555844306946, -1.4028539657592773, 0.20934441685676575, 1.0511484146118164, -0.08692548424005508, -0.3612525165081024, 0.348230242729187, -3.224491596221924, 0.1247795820236206, -0.4385205805301666, 0.6617234349250793, 1.1108675003051758], [-0.29015251994132996, -1.2061835527420044, 0.6953930258750916, 0.02646462246775627, 0.6057180762290955, 0.7859053015708923, -0.015358809381723404, -0.6968434453010559, 0.3012450635433197, 0.09845366328954697, 1.3940821886062622, 0.15971991419792175, 1.2277708053588867, 0.9298015832901001, 0.21374496817588806, -2.2982242107391357, -1.1127679347991943, 0.7342526316642761, 0.9073333740234375, -0.7276571393013, -1.6337405443191528, 2.3843703269958496, -0.16077569127082825, -1.923143744468689, 0.5100880861282349, 1.5365526676177979, -2.176708936691284, 0.07462306320667267, -0.49384811520576477, -0.018738247454166412, 0.6513833403587341, -0.9898629784584045, -1.1298154592514038, -1.2502506971359253, 0.9931125044822693, 1.127929449081421, -1.1267832517623901, 1.169392704963684, -0.6516363024711609, 0.051742181181907654, -0.46720778942108154, -0.2750492990016937, 1.4502936601638794, 0.19528569281101227, -0.8219290375709534, 1.668402910232544, 1.9558252096176147, 0.12453431636095047, 1.6790506839752197, 1.8746813535690308, -0.6237233877182007, -1.660935401916504, -1.2799203395843506, -1.4385846853256226, -0.4877850115299225, 1.9013103246688843, -0.43376001715660095, -0.5533792972564697, 1.5642826557159424, -0.224837988615036, 0.49692994356155396, 2.1449472904205322, -1.4690020084381104, -1.1578396558761597], [-0.2794715166091919, 0.204557403922081, 0.10837044566869736, 0.5260710120201111, -1.1909754276275635, -0.4085238575935364, -1.247475266456604, -0.1335364580154419, 0.5189558863639832, 0.5897164344787598, 0.23320581018924713, 1.1037718057632446, -1.4286059141159058, -0.06824305653572083, 0.3761138319969177, -1.5544397830963135, 2.824267625808716, -0.6412263512611389, 1.000571846961975, -0.3022804856300354, 0.024814268574118614, -0.035564735531806946, 1.7192403078079224, -0.025694016367197037, 0.7371478080749512, 0.7662299275398254, 0.5262432098388672, 0.786394476890564, -0.7985987663269043, -1.77780020236969, -0.34172579646110535, -0.8353809118270874, -2.1892096996307373, 1.2746626138687134, -1.8489303588867188, 1.7520184516906738, -1.4449436664581299, -0.8226516842842102, 0.7535606026649475, 0.6186255216598511, 1.1930792331695557, 0.12593384087085724, -0.2567519545555115, -2.6261353492736816, 0.4020499885082245, -0.21299532055854797, -0.08540154248476028, 0.5178050398826599, 1.4697988033294678, -0.8404070734977722, -0.21909770369529724, -0.3316660225391388, -1.8387988805770874, 1.0151218175888062, -0.3595835566520691, -1.8765302896499634, 0.23853051662445068, -0.0528184175491333, 0.17381885647773743, -1.439370036125183, -1.3185142278671265, -0.021586738526821136, 0.7504955530166626, -1.060465931892395], [0.0030895674135535955, -0.36227846145629883, 0.22790959477424622, 0.7929930090904236, 0.6399904489517212, 1.2300634384155273, 1.8659862279891968, -0.6061990261077881, 1.2326486110687256, -0.3593926727771759, -0.17261764407157898, -0.5603771805763245, 0.5344216823577881, -1.5987300872802734, -1.3088809251785278, -0.8061280250549316, -1.2061843872070312, -1.8928346633911133, 0.051666516810655594, -0.39738476276397705, -1.0032391548156738, 0.4449435770511627, 1.5066033601760864, -1.1608433723449707, 1.0253621339797974, -0.9048135876655579, 1.0774368047714233, 0.8606973886489868, -1.0883678197860718, 1.9755864143371582, -0.9630612134933472, -1.6113311052322388, -0.7434550523757935, 0.25388234853744507, -0.7123774290084839, -1.1274149417877197, 0.9529921412467957, 0.5498971343040466, -0.49390003085136414, 1.9670010805130005, -1.6690130233764648, -0.7000252604484558, -1.1276410818099976, -0.4355676770210266, -0.0008397595374844968, 0.7066558003425598, -0.5911007523536682, 0.4340110719203949, -0.4554445445537567, 1.730695366859436, -1.4155488014221191, -0.3395735025405884, 0.6114683747291565, 1.0599368810653687, 0.4211975932121277, 0.2866182029247284, 1.7552164793014526, 0.046052753925323486, 0.152700275182724, -0.7063516974449158, 1.015310287475586, 0.0774463564157486, -1.699202537536621, -1.6448347568511963], [-2.317603588104248, 0.23083838820457458, 0.30430594086647034, 1.138100504875183, -0.6772985458374023, -2.6808066368103027, 0.148153617978096, -0.27729275822639465, -0.3707493543624878, 0.5346999764442444, -1.2848137617111206, -0.12828215956687927, -0.2582663595676422, 0.3547110855579376, 1.493564248085022, -1.1525547504425049, 0.36935263872146606, 0.04079541191458702, 0.15093190968036652, 1.4542782306671143, 1.562679648399353, -0.6404628753662109, 0.5460312366485596, -0.021851837635040283, -0.880208432674408, 0.6993321776390076, 0.3731306493282318, 1.1035387516021729, 0.2556496560573578, -0.4081430733203888, -0.42048361897468567, -0.06222017854452133, 0.7759094834327698, -0.40318241715431213, 0.21780799329280853, 0.582694411277771, 2.2423739433288574, 0.4636194407939911, 1.0752795934677124, -0.1547420471906662, -0.43175750970840454, 0.8798948526382446, -0.8145606517791748, -0.733024537563324, 1.036995530128479, -0.26421982049942017, 0.21685750782489777, -0.9972425103187561, -0.31328731775283813, -0.5867933630943298, -0.7565609216690063, 0.5301102995872498, -0.5645835995674133, -0.8680527806282043, -0.5903776288032532, -0.33917859196662903, 1.5640389919281006, -0.8352389335632324, -0.195774108171463, 0.7797223925590515, -1.5019391775131226, -0.11768701672554016, 0.23592321574687958, 0.6503839492797852], [-0.07185132801532745, -0.24243684113025665, 0.0075653051026165485, -0.7560111880302429, -0.9526999592781067, 1.3689967393875122, -0.3148098886013031, -1.11441969871521, 0.7561405301094055, -1.2572342157363892, -1.0652713775634766, -0.7700673341751099, 1.4602688550949097, 0.1048336997628212, 1.2596626281738281, 0.09520581364631653, -0.04037799686193466, -0.6206956505775452, 0.02708575502038002, 0.9303048253059387, 0.6208170056343079, -0.5176187753677368, -0.43882524967193604, 0.6494749784469604, -0.41322821378707886, -0.23011192679405212, -1.4405193328857422, -0.7710946202278137, 0.609382152557373, 0.5866888165473938, 0.23318172991275787, 0.8293365836143494, -1.2887177467346191, -0.20597201585769653, -0.5664637684822083, -0.09592663496732712, 2.306199073791504, 0.6444377899169922, -1.6930495500564575, 0.08719131350517273, 1.2433866262435913, 1.9563161134719849, 0.7608733773231506, 0.6158071756362915, -0.6847764849662781, -0.94638991355896, -0.004995333030819893, 0.19822219014167786, 1.6334720849990845, 1.9010591506958008, 1.0735666751861572, -0.5501655340194702, -0.48032841086387634, -0.6536504030227661, 0.29375115036964417, -0.7032967805862427, -1.5103538036346436, -0.9232296347618103, -0.33647826313972473, 0.9320165514945984, 0.878496527671814, -0.1979023516178131, -1.2674282789230347, -1.412337064743042], [1.2675851583480835, -0.12051791697740555, 0.568472146987915, -0.10699751228094101, -0.33485499024391174, 0.66900634765625, -0.17155961692333221, 0.5367477536201477, -1.940983533859253, 0.3932126462459564, 2.8199679851531982, -0.03929729759693146, -0.11485201865434647, -1.135359287261963, 0.4142841696739197, -1.1134716272354126, -0.7722303867340088, -0.6198148131370544, -1.4829809665679932, 0.3570759892463684, -1.8858693838119507, 0.703945517539978, -1.503037929534912, -0.8265016078948975, 0.1258811503648758, -0.8725082278251648, 1.9202243089675903, 0.08086372166872025, 0.7053170204162598, 0.2166479527950287, 0.99095219373703, -0.5063328742980957, 0.4255485534667969, 0.5441550612449646, 1.523224949836731, 0.8254519104957581, -1.3580515384674072, -1.0864801406860352, 1.2322602272033691, 1.5270333290100098, 1.296142816543579, 1.7953957319259644, 1.0505014657974243, 1.275642991065979, 0.9603399634361267, -0.18616938591003418, -1.152594804763794, 0.3116554021835327, -0.08653487265110016, -0.6650367975234985, -0.521784782409668, -0.32471010088920593, 0.06366623193025589, -0.054504938423633575, 0.5966482758522034, -0.340360164642334, 2.2953851222991943, 1.3747888803482056, -0.20455044507980347, 0.4272025227546692, -0.4868201017379761, 0.5061949491500854, 2.93243145942688, 0.047218453139066696], [-0.5521990656852722, 0.3108726441860199, 0.7038331031799316, 1.3768737316131592, -0.7587260603904724, -0.6622610688209534, 0.42158588767051697, -0.6519075632095337, 3.405093193054199, -2.471132278442383, -0.32332170009613037, -0.4271334409713745, -0.8674018979072571, 0.6776543855667114, -0.15263226628303528, -0.07003601640462875, 1.303123116493225, 0.47329285740852356, 0.05415739864110947, -0.7080597281455994, 1.2211116552352905, -0.9952791929244995, 1.295983910560608, -0.5677501559257507, 2.443694591522217, 0.3186003863811493, 0.2115015983581543, -0.08053342998027802, 0.06522689014673233, 0.9624903798103333, -0.5019647479057312, -0.5260521769523621, -1.1439729928970337, 0.634745180606842, -0.9797247648239136, -0.9307444095611572, 1.5339455604553223, -2.1033337116241455, -2.095374345779419, -0.3090737462043762, 0.09490223228931427, 1.1145062446594238, 0.4033743143081665, 0.024968111887574196, 0.9116709232330322, -1.4780460596084595, -1.23866868019104, 0.03276406601071358, 2.399785280227661, -0.4090713858604431, -0.10317166894674301, -0.5080703496932983, -1.06298828125, -1.8243926763534546, 0.5912156105041504, 0.27421361207962036, -0.5250180959701538, 0.028891824185848236, 0.5128256678581238, -0.6438460946083069, -0.8233340978622437, -1.6243027448654175, -0.6145217418670654, 1.149399757385254], [-0.9380699992179871, -0.37301039695739746, 0.5596039891242981, 0.26263344287872314, 0.48348137736320496, 0.6144848465919495, 0.754574179649353, 0.044080451130867004, 0.13646672666072845, 1.7981910705566406, -0.04836972802877426, 0.6551452279090881, -0.7704393267631531, 0.8804508447647095, 1.1279473304748535, 0.20963092148303986, 0.3732384741306305, -0.5864093899726868, -0.5637984275817871, 0.23724418878555298, 1.6184184551239014, -0.559736430644989, 1.848016381263733, 0.8346541523933411, 0.11916391551494598, -0.7995566725730896, 1.7249829769134521, 0.4766424000263214, 0.5644365549087524, 1.4355108737945557, -0.07599560916423798, -1.344127893447876, 1.1918145418167114, -1.628157615661621, -1.1218360662460327, -0.39114055037498474, 0.7916488647460938, -0.398160457611084, 1.6009358167648315, -1.039705753326416, 1.3555961847305298, 0.9679147005081177, -0.4108569622039795, -0.9031884074211121, -0.5688915252685547, -0.920672595500946, 1.5431288480758667, -0.39017173647880554, -0.8450312614440918, 0.5188255906105042, -0.8142350912094116, -1.8512636423110962, 0.1370290368795395, -0.6024309992790222, 1.1891900300979614, 1.7377464771270752, 0.35043951869010925, -0.24286137521266937, 0.950039803981781, 0.20662634074687958, -1.7338558435440063, -0.29308539628982544, -0.9747093915939331, -0.08813578635454178], [-1.755212664604187, 2.02445912361145, -0.25080886483192444, 0.1108301505446434, -0.5767167806625366, 1.0589672327041626, -0.27546510100364685, -0.9094944596290588, -1.844867467880249, 1.3201634883880615, -1.1941624879837036, 1.3667373657226562, 1.8920491933822632, -0.4041399359703064, -0.9160735011100769, 1.5992487668991089, -0.7019301056861877, 0.3986811637878418, -2.8988425731658936, -1.3148564100265503, 0.4566376507282257, 1.2964603900909424, -0.6144024729728699, 1.2692779302597046, 1.3159910440444946, 0.2562914490699768, 1.2005786895751953, -1.0699063539505005, 0.32843610644340515, 1.8785420656204224, -1.5805760622024536, 0.6645386219024658, 0.37337008118629456, -0.7661038637161255, -1.530394196510315, -1.4440457820892334, -1.161252737045288, -0.7063549160957336, -0.29055964946746826, 0.22663167119026184, 1.1393316984176636, 0.1188315898180008, -0.3164459764957428, -1.8259459733963013, -0.22503240406513214, -1.6334309577941895, -0.031294308602809906, -0.24193894863128662, 0.6534927487373352, -0.9180907011032104, -0.716122567653656, -0.7532541155815125, -0.284773051738739, -0.8380352854728699, 1.3007832765579224, -0.7555516362190247, 1.4433553218841553, -1.363839864730835, 0.20391497015953064, -1.6262463331222534, 0.7918094992637634, 0.27194732427597046, 0.8710519075393677, 1.815883994102478], [1.3783801794052124, 0.4908565878868103, 0.6560582518577576, -0.12995612621307373, 0.9112417697906494, -1.6461379528045654, 0.7747710347175598, -1.6666624546051025, -0.53074711561203, 1.1076666116714478, -0.3572756350040436, -0.2293250858783722, -0.45160752534866333, 0.4163306951522827, -1.4123289585113525, 0.1723415106534958, 1.5504701137542725, -0.33714473247528076, 0.23556062579154968, 0.35607588291168213, 0.2952713668346405, -0.15200406312942505, -1.3982300758361816, -0.396374374628067, -1.1510423421859741, 0.8291864991188049, 0.06930073350667953, -1.0624974966049194, 1.6773852109909058, -1.507110834121704, -0.6169936656951904, 2.1353631019592285, -1.427730917930603, -1.7038501501083374, -0.29270198941230774, -0.4512298107147217, -1.6939817667007446, 1.9033063650131226, 1.5991148948669434, 0.18938422203063965, -1.6291064023971558, -0.8988388180732727, 2.7623817920684814, 0.7210248112678528, 0.5076454281806946, -1.1057273149490356, 0.7742037773132324, 1.2021660804748535, -0.33413177728652954, -0.911850094795227, 0.4765254855155945, -1.6185272932052612, -0.7408373355865479, 1.7061264514923096, 0.9521316289901733, 0.5348427891731262, 0.83037269115448, -1.1811598539352417, 1.0564639568328857, 1.9361603260040283, 2.253722906112671, 0.10663115978240967, 0.27886784076690674, -0.5601019263267517], [1.0193589925765991, 1.2225356101989746, 0.9769282937049866, -0.3086978793144226, -0.5386741161346436, 2.239581346511841, 0.5984726548194885, 1.2622519731521606, -0.28401148319244385, -1.3553608655929565, 0.577350914478302, 0.8780901432037354, -1.48331880569458, -1.1788012981414795, -0.15793803334236145, 0.7247247695922852, -0.18487268686294556, -0.3227672278881073, -1.5116770267486572, -1.185002088546753, -0.021176014095544815, -0.05999140068888664, -0.4069406986236572, -0.5001567006111145, -0.08301471173763275, -1.4900728464126587, 1.3320550918579102, 1.3667771816253662, -0.6699172854423523, -0.9087045788764954, -0.6393089294433594, 0.18899153172969818, -1.2325162887573242, 0.08752259612083435, 0.09211654216051102, 0.3927516043186188, 0.9563741683959961, -0.13867643475532532, -0.9537501335144043, 1.958818793296814, 0.8680161833763123, -0.15328404307365417, 0.24692030251026154, -1.8460471630096436, -1.0362040996551514, -0.3706152141094208, -0.06620851904153824, 0.022223200649023056, 0.33459681272506714, -0.5368839502334595, -2.8115005493164062, -0.23335658013820648, 0.2678682208061218, -1.0526587963104248, -0.24564282596111298, 1.0763542652130127, -0.04225011542439461, 0.22511468827724457, -2.340111494064331, 0.5865025520324707, -1.2794163227081299, 1.2659327983856201, -1.728687047958374, -1.0849229097366333], [-1.149674415588379, 0.34497418999671936, 1.3283100128173828, 0.7227637767791748, 0.37356287240982056, 0.31604257225990295, -0.28773701190948486, -0.6851685643196106, -0.5070162415504456, 0.9196624755859375, 1.736995816230774, -0.047211576253175735, 0.5701106190681458, -0.6628034710884094, -0.29157477617263794, -0.03313067927956581, 0.32169046998023987, 0.7906284332275391, 0.6590471267700195, 0.7120749950408936, -1.0607458353042603, -1.254838228225708, -0.9097328782081604, 0.5189409852027893, 0.7030797004699707, 0.7762919068336487, 1.1854649782180786, 1.3827205896377563, 0.09716233611106873, -0.2100396752357483, 1.5797642469406128, 2.1336967945098877, 1.0676276683807373, -1.1474138498306274, -0.4870895445346832, -0.9518053531646729, -0.12628453969955444, -1.634821891784668, 1.0323491096496582, 0.4768337309360504, -0.35306909680366516, 1.2866007089614868, 0.13763412833213806, 0.2716231048107147, 0.33095207810401917, -0.3462531268596649, -1.2422268390655518, 0.41626155376434326, -0.10950889438390732, -1.9342538118362427, -0.6053978800773621, -0.42389944195747375, 0.6006156206130981, 0.1343783587217331, 0.8332249522209167, 0.4994369447231293, -0.22128698229789734, -1.4775853157043457, 1.0209639072418213, -0.8010880351066589, 0.5515941977500916, 0.7875939011573792, -0.7429482340812683, -0.952029287815094], [0.1426464319229126, -0.28481900691986084, 0.4202203154563904, 0.9710729718208313, -1.1283817291259766, -0.7138400673866272, 2.511859893798828, -0.30739080905914307, 1.4182765483856201, 0.21206960082054138, -1.3434807062149048, -0.5715283155441284, 0.6048880815505981, -0.8796365857124329, 0.7562265992164612, 0.485970675945282, -0.7164905667304993, -0.38018712401390076, -0.6645033955574036, 0.1975683867931366, 2.2758374214172363, 0.61850905418396, 0.45977646112442017, 0.6593265533447266, -0.9101352095603943, -1.1817280054092407, 1.6974260807037354, -0.5439929366111755, 0.026914535090327263, 1.0905756950378418, -0.843208372592926, 0.4110807776451111, 0.2061416357755661, -0.6258403658866882, 0.2105216085910797, -2.034177303314209, -0.8153554797172546, 1.8914846181869507, -1.1347997188568115, 0.036744970828294754, 0.1860194206237793, 1.4019101858139038, -0.3446711003780365, 0.532168447971344, 0.7459409236907959, 0.04730027914047241, -0.4460695683956146, 2.5215606689453125, 0.21843628585338593, 0.3677274286746979, 1.1977728605270386, -1.6726003885269165, 0.16666126251220703, 1.4320708513259888, 0.9172114133834839, -1.729885220527649, 0.7942548990249634, 1.2718780040740967, -1.4713962078094482, -0.19567719101905823, -1.1123547554016113, -0.07324334233999252, 1.4670617580413818, -0.05238177254796028], [-0.1768849790096283, 0.12544842064380646, -0.6758129596710205, 0.03284185007214546, -1.3253772258758545, -1.2792067527770996, 1.1849538087844849, 1.5690433979034424, 2.796027421951294, -0.5035671591758728, 0.4033116102218628, -0.40450775623321533, 1.3773677349090576, -0.5675932765007019, -2.2241110801696777, 0.08324781805276871, -1.537400722503662, 0.7496113777160645, 0.8426847457885742, -0.43110737204551697, 0.42361822724342346, 0.5513917207717896, -0.8587797284126282, 0.14618001878261566, 1.4880932569503784, 0.5721510648727417, -1.8665629625320435, -1.4915807247161865, -0.6163665652275085, -0.729128897190094, 1.9562854766845703, -0.7583827972412109, 0.026237500831484795, -0.43627291917800903, -1.153766393661499, -1.146627426147461, -0.563393235206604, -1.4726829528808594, -1.3199623823165894, -1.6415611505508423, -0.6266104578971863, 0.32938647270202637, 0.5289802551269531, 0.0013490236597135663, 0.08466144651174545, 0.2429686337709427, 0.7955305576324463, 1.964160680770874, -1.435804843902588, -0.05304797366261482, -0.6753721833229065, -1.0824530124664307, 1.430241584777832, 0.06265860050916672, 0.5415238738059998, 1.2086297273635864, 0.6913131475448608, -1.4865288734436035, 0.5694543123245239, 0.7620674967765808, 1.451922059059143, 0.16807310283184052, -0.1413593292236328, 0.707771360874176], [1.562440276145935, 0.5266812443733215, -0.5614021420478821, -0.9577469825744629, 1.472348690032959, 0.21449244022369385, -0.11162565648555756, -0.4497433304786682, -0.6436250805854797, 1.5913994312286377, 2.0092124938964844, 0.4643824100494385, -0.7282096743583679, -0.3300969898700714, -0.38651448488235474, -0.7289742231369019, -0.23437900841236115, -2.0626933574676514, 0.39986950159072876, -1.8102869987487793, -0.45619869232177734, 1.1461377143859863, 1.0466591119766235, 0.7170097231864929, -0.09358064085245132, 1.2623672485351562, 0.6803700923919678, 0.25706765055656433, -1.7463343143463135, 0.7094098329544067, -2.655705690383911, 0.6492984890937805, 1.509900450706482, -1.4889055490493774, 0.8351528644561768, 0.23282644152641296, 0.11003581434488297, 0.6199315786361694, -1.4393582344055176, -0.5470804572105408, 0.22371429204940796, 0.2225644588470459, -1.6233919858932495, 0.6417849063873291, 1.1722571849822998, 0.5644933581352234, -0.5918545722961426, -0.2996283769607544, -0.38550275564193726, -0.8641086220741272, -1.4851771593093872, 0.5430212616920471, 0.26277029514312744, -0.39615094661712646, 1.1653437614440918, -1.056809425354004, -0.5721661448478699, -0.024475477635860443, 2.1046030521392822, -0.8900365233421326, 0.8075022101402283, 1.0474836826324463, -0.1028946116566658, -0.7444872260093689], [-0.4195629358291626, 0.4676630198955536, -2.681089162826538, 1.6191837787628174, 0.10404638946056366, 1.0033982992172241, -0.8069639801979065, 1.2697142362594604, -0.9810017347335815, -1.2035772800445557, 0.004557126667350531, 1.9066270589828491, 1.091678261756897, 0.3379460275173187, -1.2939226627349854, 0.888338029384613, -0.00529170036315918, -1.7518776655197144, -1.7125874757766724, 0.16303838789463043, -0.09062404930591583, 0.08642075210809708, -0.19582106173038483, 0.04536421597003937, -1.406673550605774, 0.006051309406757355, 0.39110103249549866, 1.0825068950653076, 0.4695269763469696, 0.20056533813476562, -0.10794202983379364, 0.3746899366378784, -0.7952514290809631, -0.1039913073182106, -0.37158459424972534, -0.4433084726333618, 0.609311580657959, -0.7852832078933716, 0.8742951154708862, -0.5564832091331482, -1.2180575132369995, 0.23210696876049042, 1.299512505531311, 1.3665841817855835, -1.027733564376831, -0.5127779245376587, -0.7949264049530029, 0.12154065072536469, 2.102626323699951, -1.3076671361923218, -0.6617797613143921, 1.767103672027588, -2.1737117767333984, 2.255693197250366, -1.0673540830612183, -0.84840327501297, 0.9708932638168335, 0.47646552324295044, -0.5833768844604492, -2.859271764755249, -0.2807609736919403, -0.6014509797096252, -0.9322144985198975, -0.5580859184265137], [-1.881761908531189, -1.0511889457702637, -0.9999222159385681, -0.9951390027999878, -0.2354602962732315, -0.09736482053995132, -1.3208047151565552, 0.037510115653276443, -0.7467065453529358, -1.6869856119155884, -0.07823362946510315, 0.9460312724113464, -0.6704940795898438, 0.14921098947525024, -0.9488611221313477, -1.3654297590255737, -0.3027817904949188, -2.5767147541046143, -0.09832094609737396, -0.25772422552108765, 0.1312631070613861, 0.68690425157547, -0.08542712032794952, 2.4430036544799805, -1.3815258741378784, 1.0109143257141113, -1.0990720987319946, -0.5203834176063538, 0.6210251450538635, 1.0150772333145142, -0.2606612741947174, 0.7429084777832031, -2.0413601398468018, 0.43862590193748474, -1.7162392139434814, -0.6247623562812805, -0.30070313811302185, -0.5282707810401917, 0.654954731464386, 0.7703357934951782, 1.3034675121307373, 0.9593673348426819, 0.16090130805969238, -1.8464586734771729, 0.27826276421546936, -0.3741249144077301, -1.59794020652771, 2.2974541187286377, 0.14894551038742065, -1.462508201599121, -1.7518389225006104, -1.097150206565857, -0.6481035947799683, -0.11783275008201599, -1.2878620624542236, -0.11678853631019592, -0.03605472296476364, 1.9071545600891113, -0.23111699521541595, -0.8507407307624817, 0.3304744362831116, 0.3241695761680603, -0.25696825981140137, 0.11890967935323715], [0.16798239946365356, 0.3195394277572632, 2.9341015815734863, -0.6519294381141663, 0.12678229808807373, -1.5989432334899902, 0.24518471956253052, 1.6575334072113037, 2.112159490585327, 0.5273448824882507, -0.3903883695602417, 0.007541370578110218, 0.657318115234375, 1.2182915210723877, 0.00673384265974164, -0.48043107986450195, -0.21016739308834076, -2.4707298278808594, -1.5641509294509888, 0.804144024848938, 1.3773736953735352, 1.3114174604415894, 0.8064336776733398, -0.7781050205230713, -0.8713681101799011, -0.5957586169242859, -0.5899972915649414, -0.5352120995521545, -0.0833452120423317, -0.7226793766021729, 0.30089542269706726, -0.7849071621894836, 0.5049628615379333, -1.6345458030700684, 1.9129081964492798, -0.07873252034187317, 1.1000008583068848, 0.3805326521396637, 0.9094441533088684, 0.3345299959182739, -0.04257833957672119, 0.6873087286949158, 1.6002308130264282, 0.16647544503211975, 0.3093777894973755, -1.2426563501358032, 0.6286816000938416, 1.326579213142395, 0.30015692114830017, 1.387783408164978, 1.123692274093628, 0.23989669978618622, -0.22182877361774445, -0.3918221890926361, -0.6660348176956177, -0.13027912378311157, 0.5451162457466125, -0.2987990975379944, 2.053400993347168, 0.06732756644487381, 0.5064035654067993, -0.13987547159194946, -0.9815133213996887, 0.512509286403656], [0.19526158273220062, 0.4512850046157837, 1.3981469869613647, -0.9363887906074524, 0.5583919286727905, 1.9095869064331055, 1.2990734577178955, 0.8542481660842896, 1.2480323314666748, -0.43055543303489685, -0.9318676590919495, 1.2640409469604492, 0.04835736006498337, 0.3296387791633606, -0.5570622682571411, 0.4785968065261841, -1.9443937540054321, -1.3085529804229736, -0.5339511632919312, -2.085007905960083, -0.05104440450668335, -0.9263933897018433, -0.6684150695800781, -0.026313643902540207, 0.5765026807785034, -0.6023757457733154, -0.2815917134284973, 1.220013976097107, -1.5128629207611084, -0.7882461547851562, 1.7360095977783203, -0.7686837315559387, -0.16891250014305115, -1.142741322517395, 0.11204568296670914, -0.3685208261013031, 0.7559606432914734, -1.2795604467391968, 1.7885440587997437, 0.019217224791646004, 0.19936755299568176, 1.5736172199249268, 0.34261664748191833, 2.697705030441284, -0.9305880665779114, -0.1148933544754982, 0.3787984251976013, 1.3897024393081665, -0.7541962265968323, 0.5790390372276306, 0.9857234358787537, 1.3484764099121094, 0.4128541052341461, 0.20552465319633484, 0.5678221583366394, -0.296762079000473, 2.1846976280212402, -0.9269956350326538, 0.47387006878852844, 0.5032997727394104, -0.6917968392372131, -2.478299617767334, -0.12864790856838226, 0.3504706919193268], [-0.5816283226013184, 0.8090667724609375, -0.23766225576400757, -0.8901152014732361, 0.22451962530612946, 1.463551640510559, 0.5499188899993896, -0.8591734766960144, 0.7822176218032837, 0.19689339399337769, 0.6716262698173523, -0.3133483827114105, -0.22235313057899475, -0.15553785860538483, 0.33414480090141296, -0.005074082408100367, 0.025681594386696815, -0.7962396740913391, 1.0251636505126953, 0.6282404065132141, 1.3398852348327637, -0.39673635363578796, 1.4786961078643799, -0.7563491463661194, 0.6324169039726257, -1.5272918939590454, -0.22075137495994568, -0.6450886726379395, -0.846025824546814, -0.14404191076755524, 0.2580612897872925, -0.5657461881637573, 0.19574515521526337, 0.24315698444843292, -0.7152749300003052, -0.6421187520027161, 0.8137378692626953, 0.5949718356132507, 0.008683660998940468, 0.21332411468029022, -0.06681747734546661, -0.9025719165802002, -1.9146919250488281, 0.20844170451164246, 1.4208205938339233, 0.8729856610298157, 1.6736900806427002, 0.11661763489246368, -0.1498711258172989, -1.773857831954956, -0.983301043510437, 1.226231575012207, 0.5280555486679077, 0.22620312869548798, -0.6493909955024719, -0.19264395534992218, 0.2086186558008194, -1.1964789628982544, 0.6232577562332153, 1.4276434183120728, 0.612588107585907, -0.2509115934371948, 1.972968339920044, 1.859768033027649], [-3.811659336090088, -0.9490697979927063, -0.08553364127874374, 1.8392634391784668, 0.6782851815223694, 0.8415499329566956, 0.0709318220615387, -1.6193667650222778, 1.3748465776443481, 0.7403830885887146, -0.10715357214212418, 0.9548525214195251, 0.5060359239578247, -1.7078076601028442, 0.08343501389026642, 0.15565823018550873, -1.1355899572372437, -0.49485889077186584, -0.5274193286895752, 0.19032157957553864, 0.01503626350313425, 0.14035506546497345, -0.08692008256912231, -0.37212249636650085, -1.266440510749817, 1.003228783607483, -1.167004942893982, 0.9004623293876648, 0.3308318853378296, -2.8857102394104004, 0.18210995197296143, 0.07105883955955505, 0.7809556126594543, -0.5032644867897034, 0.22836920619010925, -0.2308822125196457, 1.0711610317230225, 1.3669092655181885, 0.9788625240325928, 0.560565710067749, 0.2920278012752533, -0.8539565801620483, 1.1718968152999878, 0.5846782922744751, 0.10866809636354446, 0.7000033259391785, -0.5971130728721619, 1.5486345291137695, 0.6147864460945129, 1.4769219160079956, -1.5658609867095947, -0.11606381088495255, -0.3700828552246094, 1.0433598756790161, -0.46002477407455444, 0.6767868399620056, 1.8590689897537231, -0.6573333740234375, 0.15594929456710815, 2.2300615310668945, -0.2533438801765442, 0.16843757033348083, -0.995868980884552, -0.7797017693519592], [2.143880605697632, 0.769781768321991, -0.801004946231842, -0.39125046133995056, -0.6678170561790466, -0.47487425804138184, -1.3534862995147705, -1.2280665636062622, 1.023777961730957, 1.0806373357772827, -1.126198649406433, 1.3906198740005493, 1.3329792022705078, -0.18239165842533112, 0.19770528376102448, -0.957210898399353, 0.41337525844573975, -0.022219209000468254, 0.06206672266125679, 1.4587382078170776, 0.39631325006484985, -0.4337722659111023, 0.07028113305568695, 1.1892967224121094, -1.2795350551605225, -0.37272530794143677, -0.8071539402008057, 0.31169500946998596, 1.4799250364303589, 0.11461524665355682, 0.11550251394510269, 0.1506142020225525, 1.0008107423782349, 0.16085107624530792, -0.14270927011966705, -0.268838495016098, -1.1107007265090942, 1.3990370035171509, -0.42538800835609436, -0.3130349814891815, 0.6272138357162476, 0.7939443588256836, 0.5642881393432617, 0.36533209681510925, -0.5536523461341858, 1.5805734395980835, 0.3651195466518402, -0.18778425455093384, 0.25705471634864807, 0.4248192012310028, 1.4422590732574463, 1.6618067026138306, -0.3134397566318512, -2.400331974029541, 0.05965911224484444, -1.0701453685760498, -0.06832560896873474, 1.4842616319656372, 0.9728794097900391, -1.7194792032241821, -1.1941337585449219, -0.40972182154655457, -0.8145588636398315, -0.9519559741020203], [-0.9214923977851868, -2.1267330646514893, 1.727352261543274, -0.5332608819007874, 0.03032439760863781, -0.28399500250816345, 1.1948986053466797, 2.055563449859619, 0.2778700590133667, -0.0199847761541605, 0.23897981643676758, -2.3524632453918457, -1.4767086505889893, -1.7907267808914185, 1.9945482015609741, 0.685340404510498, 1.3165589570999146, 1.1261285543441772, 0.9587540030479431, -1.6173030138015747, 0.26385825872421265, 0.5860938429832458, 0.06385701894760132, 1.0221803188323975, 1.4936256408691406, 0.04681522399187088, -0.5880642533302307, 1.1752989292144775, -0.9253667593002319, -0.7274905443191528, 1.1582653522491455, -0.16164109110832214, -0.043839193880558014, 0.6882750391960144, 0.620474100112915, 3.849456548690796, -0.009047340601682663, 0.7825290560722351, -1.9578876495361328, 0.33276984095573425, -0.24014712870121002, 1.220069408416748, -1.878706455230713, 2.2305026054382324, -1.2703908681869507, 1.3290760517120361, -0.5336093306541443, -0.3474494218826294, 0.8047051429748535, -1.1284141540527344, -0.23110517859458923, -1.2442777156829834, -1.2876700162887573, 0.18405188620090485, 1.0275934934616089, -0.6940233111381531, -0.8171844482421875, -0.11358094215393066, -2.535811424255371, -0.8070148825645447, 1.3789559602737427, 0.2180359661579132, -1.119856595993042, 0.6328763961791992], [0.969650149345398, -0.42620187997817993, 0.33623695373535156, 2.4258224964141846, 0.41896840929985046, -0.7117501497268677, -1.2509849071502686, 0.7650424838066101, 2.0045533180236816, 0.431636244058609, 1.886602520942688, -1.6230711936950684, 0.552923321723938, 0.38353899121284485, 0.9348800182342529, 1.424277424812317, -0.7661314010620117, -1.008033275604248, 0.23547369241714478, -0.010815880261361599, -1.9333486557006836, -1.0930715799331665, 1.2521727085113525, 0.4229010045528412, -0.44977813959121704, -0.07734110206365585, 0.7483704090118408, 0.8558588624000549, -1.1254583597183228, -0.14020761847496033, 2.765707492828369, -0.038572683930397034, -0.048151955008506775, -2.5175223350524902, -0.24176247417926788, -2.5082647800445557, 0.07754871249198914, -0.6695734858512878, -0.18705035746097565, -0.7219650149345398, 0.4683317542076111, 0.5985889434814453, 1.6977910995483398, -1.612883448600769, 0.1139996200799942, 1.2220335006713867, 0.7884876728057861, 1.5136078596115112, -2.6100733280181885, 0.5767808556556702, -0.786256730556488, -0.2341437041759491, -0.20728057622909546, 1.9178647994995117, 0.4899426996707916, -0.854049026966095, 0.49969539046287537, -0.2810579538345337, 1.6714093685150146, -0.33198174834251404, 1.0525506734848022, 0.6040558218955994, 0.10545288771390915, -0.6648502945899963], [-2.0310966968536377, 0.5781850814819336, -0.2120012789964676, 0.9349936842918396, 0.9539846777915955, 0.09087476134300232, 0.5131747722625732, 0.40474575757980347, -0.6522701978683472, -1.3878474235534668, 0.14378073811531067, -0.22015371918678284, -0.10421591997146606, -1.9485440254211426, 0.5121590495109558, 0.015611995942890644, -0.3097951114177704, 0.7583926320075989, -0.7691839337348938, 0.5807602405548096, 2.530667304992676, 0.8326645493507385, 1.8098140954971313, -1.0025007724761963, 0.17990893125534058, 2.1511740684509277, -1.0721577405929565, -0.743423581123352, 2.0630855560302734, 1.6486402750015259, -1.8630539178848267, 0.8291099667549133, -2.3324458599090576, 0.6866012215614319, 1.4747612476348877, 0.6119412183761597, 1.1866075992584229, -0.2611806094646454, 1.9808893203735352, 0.4848375618457794, 0.20461063086986542, -0.7528674602508545, -0.32378074526786804, -0.3737555146217346, 0.5064473152160645, -0.872840166091919, 0.8937932252883911, 1.311989665031433, -2.4166057109832764, 0.3278309404850006, 0.4110788404941559, -0.6904850602149963, -0.1812448799610138, 0.3535141944885254, 1.1104589700698853, 0.04214699938893318, 1.1192642450332642, 0.4911262094974518, -0.5052476525306702, -0.7677038311958313, -1.5003005266189575, 0.5858179926872253, 0.8193961381912231, 0.32749271392822266], [-1.2622038125991821, 1.2701936960220337, 0.013988642022013664, 0.4290356934070587, 0.02878216840326786, 0.6458601355552673, 1.2728322744369507, -0.9175511002540588, -0.2009044736623764, 0.5779882669448853, -0.5184026956558228, 0.3249603509902954, 0.3093316853046417, 0.06072552874684334, 0.275468111038208, -0.4909061789512634, 1.2853763103485107, 2.0213990211486816, 0.15344026684761047, -0.12713263928890228, -1.016937255859375, 1.0111361742019653, -2.311368942260742, -0.6958268880844116, 0.8134344816207886, -0.621779203414917, 1.6147781610488892, -0.02810145914554596, -1.4803404808044434, -0.689685046672821, 1.1613725423812866, 2.2899317741394043, 0.7103922367095947, -0.3401677906513214, 0.6128280758857727, -2.380296468734741, 0.9255342483520508, -0.8010022640228271, -0.9019746780395508, -0.23475320637226105, 0.317131906747818, 0.309499591588974, -0.8702654838562012, 1.6856801509857178, -0.45701947808265686, -0.14414386451244354, -0.3399183452129364, -1.515211820602417, -0.7644208669662476, -0.16121678054332733, 1.6989216804504395, -0.2756926417350769, -3.4567434787750244, 1.289565920829773, 1.8118807077407837, 0.03778679296374321, -1.2589164972305298, 1.1934796571731567, -1.0761923789978027, 0.26301702857017517, -0.0855495035648346, 0.7710897326469421, 1.0679211616516113, 2.0915563106536865], [0.9276145696640015, -0.7612578868865967, -0.13859134912490845, -0.808019757270813, 0.729282557964325, 0.47182098031044006, -0.07308086007833481, -1.4973677396774292, 1.500344157218933, 0.8899437189102173, 0.6930904984474182, 0.09973669052124023, 1.5474419593811035, -0.18474093079566956, 1.0799403190612793, 0.692713737487793, -0.24588577449321747, 0.8793264031410217, -0.8163728713989258, 0.7582213878631592, 0.06741669774055481, 1.0287137031555176, 0.965793788433075, 0.4190085530281067, 1.3686341047286987, -0.7009303569793701, -0.3045986294746399, 1.2985033988952637, -1.948455810546875, 0.36065220832824707, -0.3296741247177124, 0.3334673345088959, -0.8905437588691711, 1.302941918373108, -0.4054121673107147, 1.7459583282470703, 0.21083717048168182, 0.2958589196205139, -2.075031280517578, 1.3310166597366333, -0.3562961220741272, -0.7336258292198181, -2.027531862258911, -1.6977269649505615, 1.0768091678619385, 0.9859091639518738, -0.04354551061987877, 1.2553473711013794, -1.0371251106262207, -0.43038204312324524, -1.4382375478744507, 0.1291547566652298, 0.7365620732307434, 0.6194618344306946, -0.3077872395515442, 0.7444428205490112, 0.5891668200492859, -0.7659062147140503, -0.17417781054973602, 0.05521946772933006, -0.677551805973053, -1.8775683641433716, 0.18326738476753235, 0.0019489987753331661], [-0.7791467905044556, 0.3194921314716339, 0.6553874015808105, 0.1260683685541153, 0.2721262276172638, -0.2747580409049988, -1.1780405044555664, 0.8551486730575562, -1.1833512783050537, 0.7372494339942932, -0.7616084218025208, -1.2905343770980835, 1.9046621322631836, -0.1750231385231018, -1.0759233236312866, -1.904725432395935, -0.44523969292640686, 0.9727225303649902, 0.6636234521865845, -0.2988799214363098, -0.43173664808273315, -0.6071680784225464, 0.2581838369369507, -0.473726361989975, -0.7766861915588379, -1.4253329038619995, 1.8757556676864624, -1.6868585348129272, 0.39308351278305054, 0.7442672252655029, 0.5344154834747314, 0.9182289838790894, 0.7356182932853699, -0.32015281915664673, 0.7894550561904907, 0.0800081416964531, -0.21548472344875336, -2.0076935291290283, -1.6241304874420166, -1.0705955028533936, 0.5915070176124573, 1.2797088623046875, 0.9712615609169006, 0.04594510793685913, -0.37744972109794617, -0.043502967804670334, -0.0821346715092659, -0.27599331736564636, -1.0089932680130005, -0.5544371008872986, 0.18659690022468567, 0.5720610618591309, 1.0334327220916748, -0.27343103289604187, -0.6913633942604065, -0.6384674310684204, 0.5164036154747009, -0.5635775327682495, -0.7863935232162476, 0.35683727264404297, 1.321489930152893, 1.4021610021591187, -0.04510820284485817, 0.3125307261943817], [-0.9226137399673462, -1.5504157543182373, 1.8086724281311035, 0.3954853415489197, -0.8943411111831665, -0.11282844841480255, -1.0210777521133423, -0.37399157881736755, 0.17387384176254272, -0.4957004487514496, 0.3510899245738983, 0.27764540910720825, 1.5314139127731323, -1.8011621236801147, -0.5979447960853577, -1.3537228107452393, -0.36379364132881165, 0.5375390648841858, 1.5263324975967407, -0.13230344653129578, 1.1069564819335938, -0.1561630219221115, -1.3030219078063965, 0.709595263004303, 0.10345921665430069, -1.3840792179107666, 1.0062083005905151, 1.9257853031158447, 0.3007815182209015, 1.8628056049346924, -1.3755686283111572, -0.41405102610588074, -0.20001378655433655, -0.3424002528190613, 0.5190984606742859, -0.10266757756471634, 0.42919832468032837, -0.3123913109302521, -0.038548704236745834, 0.26486310362815857, -1.7155667543411255, 1.0485622882843018, -1.4687292575836182, 0.28024327754974365, 0.7844158411026001, 0.07742249220609665, -1.6776267290115356, 0.5355592370033264, 1.9489766359329224, -0.2912593483924866, 0.7386458516120911, 0.45985856652259827, -0.7644302845001221, -1.3588366508483887, 0.6363857388496399, -0.03240315988659859, -1.0168851613998413, 1.3779056072235107, -0.7157629132270813, 0.10562018305063248, -0.8781885504722595, 0.8572721481323242, -0.3425840139389038, -0.47191622853279114], [0.04421635717153549, -0.6380417943000793, 1.2056173086166382, 0.5063096284866333, 1.3535187244415283, 1.6906096935272217, -1.804856538772583, -0.5256558060646057, -1.5246447324752808, 0.2254450023174286, 1.1265608072280884, 0.7886822819709778, 0.9002076983451843, 0.8396652340888977, -1.103677749633789, 0.8591316938400269, -1.4018640518188477, -1.018190860748291, 1.9411396980285645, -0.2980954349040985, -0.9543898701667786, -0.9705638885498047, -0.49414747953414917, -1.0748374462127686, -1.808276653289795, -0.5676971077919006, -0.38630467653274536, -0.5674358010292053, 1.1411925554275513, -1.5564486980438232, 1.088760256767273, -0.7514284253120422, -0.7902284860610962, -0.739877462387085, 0.28146567940711975, 1.4820654392242432, 0.17336831986904144, 0.027095289900898933, 0.6212239265441895, 0.4525015354156494, -0.4953819811344147, 0.3064015209674835, -0.5251451134681702, -0.19272570312023163, -1.0734598636627197, 0.26101332902908325, -0.0069779809564352036, -0.8197966814041138, -0.15221209824085236, -1.0859222412109375, -0.6722441911697388, 1.2633416652679443, -0.5940476059913635, 1.2483264207839966, -0.671086311340332, -0.19134797155857086, -1.7521101236343384, 0.4817129373550415, 1.761755108833313, -0.7950394153594971, 0.7612340450286865, 0.39202257990837097, -0.7725585103034973, 1.133861780166626]], "dtype": "float32"}, {"__nd__": [[-0.7755532264709473, -0.7416783571243286, -0.17867441475391388, 0.7736780643463135, 1.8053739070892334, -0.007643287070095539, -0.9427714347839355, -0.7370923161506653, 1.2976109981536865, 0.1676948517560959, -2.345228910446167, 1.9126415252685547, -0.36397621035575867, -1.410677194595337, -0.5672017931938171, -0.8661342263221741, -0.21766163408756256, -0.6203020811080933, -1.4115294218063354, -0.17350833117961884, 1.2972474098205566, -1.2865445613861084, 0.4471193552017212, -0.10883140563964844, 0.9983662962913513, -0.6915088891983032, -0.1476772278547287, 0.34215426445007324, -0.024394849315285683, 1.0535646677017212, 0.48605620861053467, 1.0024665594100952, -1.2168142795562744, 0.5540397763252258, 0.4206130802631378, 0.9091464877128601, -0.7175635099411011, -0.9927796721458435, -0.4137157201766968, 1.4950546026229858, -0.38749784231185913, -0.5573775768280029, 0.332729309797287, -0.740331768989563, 0.03574349358677864, 1.963663101196289, -0.6781560778617859, -1.4307565689086914, -1.6382147073745728, 0.9985597729682922, 0.8605501651763916, -1.0459386110305786, -0.21562151610851288, 0.832273542881012, -1.140263557434082, -0.925728976726532, -1.469959020614624, 0.01378985121846199, 1.0738348960876465, 0.23515041172504425, -0.3344825804233551, 1.3644299507141113, -0.27167636156082153, -0.01280574407428503], [-0.4387211799621582, 0.3149746060371399, -1.62285315990448, 2.1647861003875732, -1.3861788511276245, 0.9228933453559875, 0.4489322006702423, 1.936388373374939, -0.5115767121315002, -0.8946376442909241, -0.28058403730392456, -0.4313584268093109, 0.11104656755924225, -0.5732607245445251, 0.8961217999458313, -1.5070923566818237, 2.031628131866455, 0.1342199593782425, 1.0520323514938354, 0.2501178979873657, 1.4818646907806396, -0.9274280667304993, -0.8544062972068787, -0.259896457195282, -0.41952207684516907, -0.8601365685462952, 0.0013852586271241307, 0.6866524815559387, -0.6862213015556335, 1.550815224647522, -1.535637378692627, 0.4851592779159546, -0.8872090578079224, 0.2965516746044159, 0.15110039710998535, -0.7430460453033447, 0.2664313316345215, -1.2748395204544067, 1.5376582145690918, -0.05895756930112839, 0.33608800172805786, -1.1087932586669922, -1.2233126163482666, -1.6457546949386597, -0.42979303002357483, 0.9666886925697327, -0.031746938824653625, -0.8840404152870178, -0.5631085634231567, 0.6860194802284241, -0.9376417994499207, 0.49083948135375977, 0.21730013191699982, 0.45635196566581726, 1.182111382484436, -0.9119201302528381, -2.2148420810699463, 1.3251723051071167, -0.31926625967025757, -1.3480080366134644, 1.3482991456985474, 1.6126112937927246, 1.9642882347106934, 1.106439232826233], [-0.8716168403625488, -0.427624374628067, 0.5977921485900879, 1.7180054187774658, -1.076514720916748, 0.14559733867645264, -1.0074150562286377, -0.799258291721344, 0.33465704321861267, 0.7253454327583313, 0.6503633856773376, 1.145829200744629, 1.2061879634857178, -0.050139352679252625, -0.0801139622926712, 1.3455569744110107, -0.03305361047387123, 0.9740418195724487, 0.4930768609046936, 0.7752686142921448, -0.03576136752963066, 2.032893419265747, 2.8468668460845947, 0.11201788485050201, -0.4829789996147156, 0.001226064981892705, -0.186508908867836, 0.6232095956802368, -1.0708558559417725, 1.3265595436096191, -1.86713707447052, -1.764225959777832, -0.21897530555725098, -0.9247896671295166, 0.4577324390411377, -0.06443700939416885, 0.4197002947330475, 0.4200262427330017, -0.011837633326649666, 0.47688260674476624, -0.522736668586731, -0.8851114511489868, 0.8923041820526123, -0.5243527889251709, 1.5059118270874023, -0.49944835901260376, -0.9190884828567505, -0.4833231568336487, -0.3716028928756714, 0.2370448261499405, -0.5715782046318054, 1.3701598644256592, -1.654995083808899, -0.423025518655777, -1.0565435886383057, -0.6753241419792175, -1.7113568782806396, 0.5801916122436523, -0.23719047009944916, 0.08150365203619003, -0.0361468680202961, 0.6124975085258484, 1.2719777822494507, 2.669438600540161], [-1.1526237726211548, 0.19681580364704132, 1.0723427534103394, -1.3529925346374512, -0.43332669138908386, -0.5609911680221558, 0.27841249108314514, -1.4031718969345093, -0.4480656087398529, 0.17746524512767792, -0.6548914313316345, 0.26948145031929016, 2.3320887088775635, -0.28139910101890564, 0.8631836175918579, -0.07188892364501953, 0.5205933451652527, -0.4734826683998108, 0.08109414577484131, -1.3722718954086304, -1.1445344686508179, 1.352914810180664, 0.46560364961624146, -0.3752160966396332, -1.779340147972107, 0.7451924085617065, 1.371262550354004, 1.2721527814865112, -0.427564412355423, -1.1553089618682861, 0.7554200887680054, -1.2366169691085815, 1.7683069705963135, 0.8350586891174316, -1.3659149408340454, 1.0515458583831787, -0.918475329875946, -0.05552949383854866, -0.16100747883319855, -0.7418755888938904, -0.1837652325630188, -0.24283777177333832, 0.024510260671377182, 0.9755367636680603, -0.4066746234893799, -1.587185025215149, -0.013162430375814438, -1.1897047758102417, -1.3970168828964233, 0.44411057233810425, -0.01231252308934927, -0.43879932165145874, 1.5718135833740234, -0.4383891224861145, -0.27606916427612305, 0.3581135869026184, 1.9011874198913574, 0.6626419425010681, -0.25232672691345215, -0.4471132159233093, -1.3863227367401123, 0.3110411763191223, 1.2323769330978394, -2.133284568786621], [0.6299040913581848, -0.7820435166358948, -1.256056785583496, -1.0284909009933472, -0.8710450530052185, 0.8690187335014343, -0.23645006120204926, 0.6755936741828918, -1.5440144538879395, 0.06561215221881866, -0.36042964458465576, -1.2703248262405396, -1.0109833478927612, 0.07135817408561707, -0.8853330612182617, 0.0668613389134407, 0.24309639632701874, -0.20860882103443146, 0.2109490931034088, 0.4649919271469116, 0.9853065609931946, 0.11255385726690292, 0.5286532640457153, 0.9096788763999939, 1.7581381797790527, 0.05555516481399536, -1.2188657522201538, 0.3187655210494995, 0.49667343497276306, 0.21666721999645233, 0.8280251622200012, -0.8936837315559387, -0.9664633870124817, -0.6058192849159241, -1.5893203020095825, -1.0561676025390625, -1.157974123954773, 0.9250725507736206, -0.78779536485672, -1.9570529460906982, 0.4577088952064514, 0.4837741255760193, -0.4816843867301941, -0.95433509349823, 0.3902744948863983, -1.5329205989837646, 0.7484543919563293, 0.005627204664051533, 1.1310912370681763, 0.0718177780508995, 1.3333145380020142, 0.37693503499031067, -0.356254518032074, 1.1427078247070312, -0.7645042538642883, -0.6926603317260742, -1.3513277769088745, -0.5710223317146301, 0.10879496484994888, 0.5287162065505981, -1.1099847555160522, 0.45670071244239807, -1.327942967414856, -0.9614124894142151], [0.6284714341163635, -0.4159768521785736, -0.982801616191864, 0.009626505896449089, -0.46884891390800476, -0.45640069246292114, 1.3273414373397827, -0.24229943752288818, -0.6039170026779175, -0.39531978964805603, 1.0784127712249756, -0.6264981031417847, -0.4945847690105438, -0.5897804498672485, 1.1421226263046265, 1.0940109491348267, 0.8392840623855591, 1.2561767101287842, 0.2155630737543106, 2.69340181350708, -0.7652854919433594, -0.2781030833721161, 0.601898193359375, -0.9495545625686646, 0.30603793263435364, 0.21906869113445282, 0.039168111979961395, 1.3993000984191895, 0.44885921478271484, 0.45748159289360046, -1.2963238954544067, 2.6204993724823, 1.6370093822479248, -1.6941779851913452, 1.359703779220581, -0.9834599494934082, -1.3764158487319946, -0.594241201877594, -0.168533593416214, 0.23869481682777405, 0.023736199364066124, -0.7707979679107666, 1.2359685897827148, -1.5011872053146362, 0.0277425404638052, 1.0955021381378174, 2.10915470123291, -0.08038093894720078, -0.046531736850738525, 0.07842597365379333, -0.4393942058086395, 0.7266579866409302, -0.08669018745422363, 2.20025372505188, -0.1235198900103569, -1.114632248878479, -1.4387879371643066, 1.5070449113845825, 0.012861196883022785, -1.527058482170105, -0.3764778971672058, -1.0921108722686768, -0.5086376070976257, -1.4636881351470947], [-0.4238145351409912, -1.563726544380188, 1.742309331893921, 0.32834696769714355, 0.0570262111723423, 0.11989287286996841, 0.5242345333099365, -0.6103991270065308, -1.289716124534607, -1.3136059045791626, -0.39792400598526, -0.5182112455368042, 2.423037528991699, 2.2097442150115967, 1.4725505113601685, 0.22049318253993988, 1.1946576833724976, -0.001386890304274857, -0.004446882754564285, 0.992567777633667, -0.8393170833587646, 0.484647661447525, 0.0668674036860466, 0.26684829592704773, 0.5101777911186218, -0.43527352809906006, -1.8717613220214844, 0.5790966749191284, 0.4745319187641144, -0.6905228495597839, -0.13136570155620575, -1.0234047174453735, -0.17010891437530518, 0.9111788272857666, -0.04852103441953659, 1.238338828086853, -0.49810755252838135, -0.06721348315477371, -2.888472080230713, 0.5067550539970398, 0.4039998948574066, -1.4100072383880615, -0.05707014724612236, -0.8474336862564087, 0.08941572159528732, 1.0390925407409668, -0.9560987949371338, 0.7725398540496826, -1.0729438066482544, 0.5714625716209412, -0.5702464580535889, 0.6368204951286316, -0.8838762044906616, -1.1148334741592407, 0.3625791072845459, 1.4660853147506714, 1.1733609437942505, 2.2430224418640137, 1.4833608865737915, 1.1487985849380493, 1.0374447107315063, -1.0950367450714111, 1.3817342519760132, -0.7061311602592468], [0.011081484146416187, -1.1909273862838745, -0.25544416904449463, 1.1686041355133057, -0.18401974439620972, -1.095463752746582, -1.883569359779358, 0.6825790405273438, -0.1318415254354477, 0.01641181856393814, -0.4230215549468994, 0.387527734041214, -0.7763632535934448, 0.45263636112213135, 0.1272968351840973, -0.11071228981018066, 1.3351309299468994, 0.2594771981239319, -1.307754397392273, 1.5388128757476807, 1.2128623723983765, -0.03409167379140854, 1.2944955825805664, -0.0811663344502449, 0.1425265669822693, -1.9484585523605347, 1.2347779273986816, -0.683244526386261, -0.10089895874261856, 0.27740049362182617, -0.0012057448038831353, 0.19769778847694397, 0.5228643417358398, 0.7973373532295227, -0.3404255509376526, -2.410418748855591, 0.10783375799655914, 0.8760798573493958, -0.0024711659643799067, -0.573686420917511, 1.6953437328338623, -0.6484339833259583, -0.46676596999168396, -0.6055164933204651, 0.8330762982368469, -1.2423888444900513, -0.5042945146560669, 0.014320452697575092, 1.0012012720108032, 2.363164186477661, 1.1855047941207886, 0.09356506168842316, 0.8051729798316956, -0.06587433069944382, 0.2605563700199127, -0.3303265869617462, -0.1096700057387352, -0.5585409998893738, 1.586503028869629, 0.19757068157196045, 1.2841109037399292, -0.6030019521713257, 0.9256312847137451, -0.5015477538108826], [0.5907519459724426, -0.5526022911071777, -0.20078115165233612, -0.5860559344291687, -0.42529383301734924, 0.36391860246658325, -2.480529308319092, 1.566240906715393, 0.7323427796363831, -0.34772947430610657, 2.4696149826049805, -0.3048211932182312, 0.21970129013061523, 2.112528085708618, 2.017976760864258, -0.6349530816078186, 0.583600640296936, -0.2216637134552002, 1.3340526819229126, -0.9000834226608276, 1.1423434019088745, 0.20266467332839966, -0.7559719085693359, -0.005917834118008614, -0.785387396812439, -0.7796306610107422, -0.5477727651596069, 1.0821845531463623, 0.36019569635391235, 0.3888739049434662, 0.269076406955719, 0.8371524214744568, 0.8161718249320984, 0.8455578684806824, 0.7562157511711121, 0.7993483543395996, 0.10913205146789551, 1.5880181789398193, 0.7998939156532288, -1.5281649827957153, 1.9129588603973389, 0.5824617743492126, 0.8304614424705505, -0.26071280241012573, -0.20857274532318115, -0.179561048746109, -0.08627957850694656, -0.10683280974626541, -0.9912658333778381, -0.15833494067192078, -0.008935072459280491, -1.4407976865768433, 0.42353367805480957, 0.07065670937299728, 0.3604074716567993, -0.9607217907905579, 0.9356127977371216, 1.3340896368026733, 0.6326368451118469, 1.7214419841766357, -0.9931938052177429, -0.0917411670088768, 0.22872939705848694, -0.10865289717912674], [0.10523626208305359, -2.0433084964752197, -0.10423095524311066, 1.6144933700561523, -0.6887224316596985, 0.23701408505439758, -0.2604100704193115, 0.7484580874443054, 1.7639333009719849, 0.41337138414382935, 0.41764187812805176, -1.0065550804138184, -0.3533196449279785, -0.7641633749008179, 0.9247996807098389, 0.9979618787765503, 0.5633165240287781, -0.965327799320221, 0.5034492611885071, 0.576546311378479, -1.6492313146591187, 0.3306023180484772, -0.10996954888105392, -0.805007815361023, 1.8663015365600586, -0.40464070439338684, -0.16037724912166595, -2.1903293132781982, -0.7835491895675659, 0.36703068017959595, -0.6576635837554932, 0.7634619474411011, 1.6095917224884033, 0.9888337850570679, -0.18190382421016693, -2.0114099979400635, 0.49293458461761475, -0.7362993359565735, -1.3173882961273193, 0.7671893239021301, 0.7495613098144531, 0.576356053352356, 1.1803815364837646, 1.3825801610946655, -0.1319369077682495, -1.5949558019638062, 1.4349632263183594, -0.3030001223087311, -0.6819115877151489, -1.5070161819458008, -2.1774728298187256, 0.11659888178110123, 0.6742792129516602, -0.5717948079109192, -1.3246077299118042, -1.0367612838745117, 2.210341691970825, -1.1107001304626465, 1.3105058670043945, 0.08177805691957474, -0.6106868386268616, -1.3513976335525513, -0.32741814851760864, -0.5250536203384399], [-0.201793372631073, -0.14041362702846527, 1.3549402952194214, 1.913834571838379, 1.4202089309692383, 1.2482343912124634, -1.2005211114883423, -0.10286775976419449, -0.63921719789505, 0.2479972392320633, 0.30451536178588867, -1.0951128005981445, -0.1477862149477005, 0.13681067526340485, 0.5733964443206787, -0.4493615925312042, -1.3592959642410278, 0.14802685379981995, 0.13906703889369965, -0.43027833104133606, 0.3571733236312866, -1.1933618783950806, -0.28134897351264954, -2.3671841621398926, 0.27165892720222473, 1.0289252996444702, -1.1405750513076782, 1.980780839920044, 0.915934145450592, 0.24698150157928467, -1.0937039852142334, 1.1905227899551392, -0.05061018094420433, 0.559790849685669, 1.0741976499557495, -0.051450084894895554, -0.5511945486068726, -0.22712595760822296, -1.2272149324417114, -0.5303667187690735, 0.728435754776001, -0.7588919997215271, -0.7374091148376465, 0.058992937207221985, 0.22770337760448456, -0.3806458115577698, 0.7807407379150391, 0.6621145009994507, 0.4698553681373596, 0.04672931879758835, -0.12148535251617432, -0.05847499519586563, -0.93962562084198, -0.305442214012146, 0.21269091963768005, 0.029520627111196518, 2.0581438541412354, 1.1031135320663452, -0.02207537181675434, -0.44676119089126587, -0.13772545754909515, -1.0574514865875244, -0.03224340081214905, 0.9397276043891907], [0.028108615428209305, 1.9797475337982178, -0.08718137443065643, 0.37036946415901184, 0.6820257902145386, 0.4998352527618408, 0.873878538608551, 1.220319390296936, -1.0515832901000977, -0.2887845039367676, -0.47818270325660706, 1.019622802734375, 0.09224211424589157, 0.8864065408706665, 0.6322674751281738, -0.18028204143047333, -0.0868779718875885, -0.9748047590255737, -0.0034698122180998325, 1.059370994567871, 1.6642982959747314, -0.5597704648971558, -1.1917608976364136, -1.1443153619766235, -0.42714744806289673, 0.6840871572494507, -1.2046042680740356, -1.083517074584961, 0.08900649845600128, -0.3894493877887726, 1.1153405904769897, -1.2464510202407837, 0.343410462141037, -0.3980851173400879, -0.7138679027557373, -1.476669192314148, 2.4713082313537598, 0.383809357881546, -0.0704508051276207, 0.14995183050632477, 0.23521120846271515, 1.3782449960708618, -0.02833080291748047, 1.396020770072937, 0.6889883279800415, -0.6178118586540222, 0.834562361240387, -0.4271372854709625, 1.3473272323608398, -0.7120340466499329, -0.8709720373153687, 1.5169535875320435, 0.04954595863819122, -0.12277640402317047, 0.2880222201347351, 0.17544271051883698, 1.438085675239563, 0.5253822207450867, -0.44034647941589355, -0.5816487669944763, 0.03942412510514259, 2.1768388748168945, 0.4381687641143799, -0.7748941779136658], [-1.3134472370147705, -0.2806819677352905, -0.34874624013900757, 0.6092431545257568, 0.5364946722984314, 0.4204261004924774, -1.0139408111572266, 0.3095717132091522, -0.26748529076576233, 0.43539878726005554, -0.5915374755859375, -0.1553461253643036, 0.041680172085762024, 1.1671931743621826, -0.8015417456626892, 0.011206370778381824, 1.0706959962844849, 0.8833918571472168, 1.25853431224823, -1.7825742959976196, -1.0478326082229614, 0.9564780592918396, -0.9535076022148132, -1.1204991340637207, -0.6178498864173889, 1.2105456590652466, -1.8988171815872192, -0.068275585770607, 1.8585959672927856, -0.9456964731216431, 0.8129147887229919, -0.9632331132888794, -0.4335383176803589, -1.9050291776657104, -0.3546396791934967, -1.3667024374008179, -0.25016871094703674, -0.5185268521308899, 0.5237380862236023, 0.5059852600097656, 1.1798657178878784, -2.2397046089172363, 0.6077013611793518, -1.319874882698059, -0.4742615818977356, 0.3332670331001282, -0.7776390314102173, -0.32314878702163696, 0.7784199714660645, 1.1122108697891235, -0.023283811286091805, 2.4630889892578125, 0.1296483427286148, 0.39242690801620483, -0.10997378081083298, 1.7327210903167725, 2.1211068630218506, -0.19142025709152222, -0.3928903341293335, -2.418063163757324, -1.3001837730407715, 1.2639470100402832, 1.7635489702224731, -0.1045893207192421], [-0.3753542900085449, 2.2463910579681396, 0.3131828010082245, 0.2753616273403168, 2.3120970726013184, 0.3985275626182556, 1.4564924240112305, -1.3875452280044556, 1.7753623723983765, -1.103057861328125, -0.6134088635444641, -0.4993542730808258, -1.5895880460739136, -1.3101109266281128, 0.4201863706111908, -0.37538596987724304, 1.1541825532913208, 0.20857061445713043, -0.33169785141944885, 1.650579810142517, 0.21832053363323212, -0.07299362868070602, -0.7447560429573059, 0.5658249855041504, 0.14842022955417633, 1.269582748413086, -1.2666908502578735, -3.6932129859924316, 0.3178190290927887, 1.6882423162460327, -0.0915420725941658, 1.4275047779083252, -0.7546213865280151, -1.2870451211929321, 1.2920528650283813, 0.6668890714645386, -0.0031759897246956825, -0.05730775371193886, -1.2644798755645752, 0.1245666965842247, 0.45392438769340515, -0.9098275899887085, 0.8395626544952393, 0.683019757270813, 0.23630180954933167, -2.0562453269958496, -0.21721793711185455, -0.276260644197464, -0.5410972833633423, 0.15315388143062592, 0.6830772161483765, 0.23403500020503998, 0.49401193857192993, 0.7444097399711609, -0.1173771396279335, -0.9970346093177795, -1.8429991006851196, 0.7908396124839783, -0.3398674726486206, -1.430219054222107, -0.33988305926322937, 2.066434144973755, 0.3315664231777191, -0.20951104164123535], [1.7587745189666748, 0.11337055265903473, 0.8054394721984863, -0.9746285080909729, 0.29374977946281433, 0.8297019600868225, 0.412415474653244, 0.8278915882110596, 0.3453773558139801, -1.731571078300476, 0.07179651409387589, -0.24377602338790894, -1.080731749534607, -0.10310431569814682, -0.40575820207595825, 0.10403604805469513, -0.6441218852996826, 0.75567626953125, 1.5229465961456299, -0.076271191239357, -1.0501658916473389, 1.029167890548706, -0.6679984927177429, -1.3988559246063232, -0.4045047461986542, -0.2592160999774933, 1.0912238359451294, -0.10593786835670471, -0.6435304284095764, -0.16066798567771912, -1.7841178178787231, 1.240182876586914, -0.2875232398509979, 0.6300643682479858, 0.1316218227148056, 0.7237352728843689, -3.458622932434082, -0.9318503737449646, -1.0713504552841187, 0.377636194229126, -0.5602123737335205, 0.19716592133045197, -0.6403546333312988, 1.4382017850875854, 0.33019882440567017, 0.9305024743080139, -0.11576209217309952, -1.304505705833435, 0.026447843760252, 0.19976510107517242, 0.5531846880912781, -0.4442425072193146, 1.5052276849746704, 0.211170956492424, 0.5219599008560181, -0.304492712020874, -2.723966121673584, -0.16762541234493256, 1.9338983297348022, -0.21744710206985474, -0.726993978023529, -1.6249440908432007, 0.6489527225494385, -0.27048447728157043], [1.0596147775650024, -0.6011446118354797, -0.6923906207084656, -0.7899906039237976, 1.1913173198699951, -1.2705354690551758, 0.01814272068440914, -1.164298415184021, -1.2205967903137207, -1.107323169708252, 0.14171895384788513, 0.2761429250240326, 0.6207518577575684, 1.2738311290740967, 0.23370857536792755, -0.8920660018920898, 1.4348251819610596, 1.3585197925567627, -0.6622467637062073, 0.19753195345401764, -0.4127466380596161, -0.37633970379829407, 0.8476428985595703, 0.27270862460136414, 0.63957279920578, -0.23597955703735352, 1.8183550834655762, 0.05801357328891754, -0.1630048304796219, 0.9091517329216003, 0.28116145730018616, 1.5856517553329468, 1.6048389673233032, 2.4412128925323486, 2.543595790863037, 0.06565167009830475, -1.7277904748916626, -0.08047527819871902, 0.47406473755836487, 0.40379634499549866, -0.4687976837158203, -0.9685847759246826, 0.5986932516098022, -0.03459692746400833, -0.2674108147621155, 0.603264331817627, 1.0760433673858643, 0.4363565146923065, -1.0138030052185059, 1.866654634475708, 0.9307582974433899, 1.1139745712280273, -0.9327989220619202, -1.391555905342102, 0.7480497360229492, 0.46820998191833496, 1.1203042268753052, 0.3448396921157837, 2.7385501861572266, 1.6816320419311523, -1.3898032903671265, 0.25632622838020325, -1.2234995365142822, 0.7513064742088318], [-0.2576815187931061, -0.5670255422592163, 1.4931724071502686, 0.8770014643669128, -0.1706518828868866, -1.5070750713348389, 0.38563692569732666, -0.5251778960227966, 0.3825940489768982, -0.8069777488708496, -0.1863538920879364, -0.5718247294425964, -1.1228981018066406, 0.10344958305358887, 2.197413682937622, 1.5243555307388306, -0.21164898574352264, 0.4720046818256378, -0.7542760968208313, 0.09732324630022049, 1.9401875734329224, -0.763588011264801, 0.6246511340141296, 0.8764883279800415, -0.7715653777122498, -0.5682355761528015, -0.10493293404579163, 0.8960861563682556, -1.5750412940979004, 0.38981521129608154, 0.8354350924491882, -0.7206818461418152, 0.551430344581604, -0.37674480676651, -0.10907622426748276, -0.6077247262001038, -0.030726108700037003, 0.2676219046115875, 0.5023733377456665, -0.2585115134716034, -0.7622863054275513, 0.5087353587150574, -1.0559563636779785, -1.0485774278640747, -1.1607277393341064, -0.752728283405304, 0.3558153510093689, 0.44086992740631104, 1.1800768375396729, -0.838158369064331, 0.3129374384880066, -1.6308035850524902, 0.9888847470283508, 0.06951200217008591, 0.5816857218742371, -0.7292375564575195, 1.1688894033432007, 1.0171287059783936, -0.2523129880428314, -1.7705967426300049, -1.0518287420272827, 1.579539179801941, -1.9693630933761597, -1.2606536149978638], [0.06897429376840591, 0.36358872056007385, 0.10593187063932419, -1.399782419204712, -0.554224967956543, -0.6028959155082703, 0.0595850944519043, 0.6670078039169312, 2.2873706817626953, -0.9289864301681519, -1.0762805938720703, -0.20860561728477478, -0.3857201039791107, -1.5976157188415527, 0.03588173910975456, -0.9924113154411316, 0.8525913953781128, -0.20192518830299377, -0.3141610026359558, 1.1911345720291138, -1.9226744174957275, 0.7780667543411255, -1.092705488204956, 0.24232415854930878, 0.12400983273983002, -0.07292073965072632, -0.001321956398896873, -0.2991292476654053, 1.1011364459991455, 0.16521677374839783, 0.7742576003074646, 0.6787566542625427, 1.7186429500579834, 1.0077683925628662, 0.6293628215789795, 0.6322586536407471, -0.7099118828773499, -1.7904380559921265, -2.3997325897216797, -0.3795487880706787, -0.27523696422576904, 1.2331898212432861, -0.28400948643684387, -0.6353617310523987, -0.7553156018257141, 1.2830530405044556, 0.6002838015556335, -2.1813812255859375, 1.9300286769866943, -0.8618643283843994, -0.08000113070011139, 1.6598726511001587, -1.630212426185608, 0.5629082322120667, -0.017765790224075317, 0.48969051241874695, -0.9629902243614197, -0.0034275318030267954, -0.6147291660308838, -0.22983905673027039, -0.6312973499298096, 1.1403042078018188, 1.0971667766571045, 0.40987539291381836], [0.5708464980125427, -0.6478478312492371, 1.012460708618164, 1.654097318649292, 1.4736632108688354, 1.7273164987564087, -0.17746491730213165, 0.49752429127693176, 0.05633620545268059, 0.46918660402297974, 0.3596298098564148, 1.0222933292388916, -0.22449876368045807, -1.5128182172775269, -0.08255326747894287, 2.4877209663391113, 0.5216817259788513, -0.29389989376068115, 0.6626209020614624, 1.2423170804977417, -1.2497960329055786, -0.42046624422073364, -2.4940831661224365, 0.06328163295984268, 1.0163360834121704, 0.3498663604259491, -0.0924314484000206, -1.534327745437622, -1.3448666334152222, 0.6175875663757324, -0.0473862960934639, 0.07011724263429642, 0.12974075973033905, 0.2170836329460144, 0.4862942099571228, 1.1095185279846191, -0.5129671692848206, 1.8163630962371826, 0.8373480439186096, 0.3893544375896454, -1.0000689029693604, -0.36915746331214905, -1.2159984111785889, -1.4742552042007446, -0.7396743893623352, 0.06677351891994476, 1.0168163776397705, -1.1685607433319092, -0.37251338362693787, -0.08738841116428375, 0.4534853398799896, 0.22781866788864136, -0.21020184457302094, -0.9467834830284119, 0.2178173065185547, -0.7358320355415344, 0.07489795237779617, -0.05977948382496834, -1.4219080209732056, 0.4930550158023834, -0.9081273674964905, 1.265750765800476, -0.5433085560798645, 0.6501234769821167], [1.1952942609786987, -0.180653914809227, 0.4062875807285309, 0.4632647931575775, 0.29834574460983276, 0.18676120042800903, -0.5952776670455933, -0.8833515048027039, 1.1376307010650635, 0.19115355610847473, -0.561457097530365, -0.08298151195049286, 0.15982477366924286, -0.15268486738204956, -0.7409915924072266, 0.19732093811035156, -0.2023632824420929, -0.20440004765987396, 1.7496546506881714, -0.5002069473266602, 0.5578641295433044, 0.3942517042160034, 0.35858941078186035, -2.3214221000671387, 1.847912073135376, -1.1629494428634644, 0.20373116433620453, -1.779963731765747, -1.1888477802276611, -0.3162192106246948, 1.488946557044983, 2.1247305870056152, -0.17679400742053986, -1.340035319328308, 0.2036435604095459, -1.896896481513977, -0.9844412207603455, -1.402454137802124, -0.5179427862167358, -1.2991666793823242, 0.7607173323631287, -0.555353045463562, -0.22400955855846405, 1.797683835029602, -1.4472935199737549, -0.7205079197883606, 0.2748028039932251, -0.06645367294549942, 1.725827693939209, -1.427085280418396, -0.36518245935440063, 0.13533857464790344, -3.104639768600464, 1.384968876838684, 0.10299889743328094, -0.031941577792167664, 0.25087136030197144, -1.4640443325042725, 0.5520589351654053, -0.5516522526741028, -0.6553683876991272, -0.98021399974823, -1.601590633392334, 0.7843841314315796], [0.5993131399154663, -0.20672813057899475, 0.5754839181900024, -0.15109144151210785, 0.24517589807510376, -0.5198770761489868, -0.9384567737579346, -0.4412620961666107, -2.917567491531372, 1.3534457683563232, -0.046090591698884964, 0.9730262160301208, -1.4066987037658691, 1.360999584197998, 1.1525115966796875, 0.06132616475224495, -0.2599357068538666, 0.6923761963844299, -0.01740599051117897, 0.6384494304656982, 0.9290999174118042, -1.2368634939193726, 0.48200592398643494, 0.30354538559913635, 0.906683623790741, 1.0770630836486816, 0.7124836444854736, -1.9846051931381226, 1.5573397874832153, -0.5365244150161743, -1.541818618774414, 0.5594825744628906, -0.24041247367858887, 0.5622928142547607, -0.4781978130340576, -0.16744081676006317, 0.20645318925380707, 1.237623691558838, 0.11682688444852829, 0.6132766604423523, -0.07741905748844147, 0.4849851131439209, 1.1574114561080933, -1.7113608121871948, -1.0860611200332642, -0.3175083100795746, -2.341341257095337, -1.599898099899292, -1.662895917892456, -1.6116068363189697, -0.8884825110435486, 1.5583574771881104, -0.7797078490257263, 0.5392534732818604, 1.2018425464630127, 0.49712657928466797, -0.6213067770004272, 0.325430303812027, 0.8841657042503357, -0.34585413336753845, -1.4806493520736694, 0.019817408174276352, 1.4116846323013306, -2.290457010269165], [1.2834116220474243, 0.03706251084804535, 0.5473029017448425, -0.9832512140274048, 0.258088082075119, -1.1661841869354248, 0.44766879081726074, -1.1306216716766357, 1.4521814584732056, 0.9830554723739624, 1.9511419534683228, -0.7739084362983704, -1.0834205150604248, 1.0279473066329956, -1.2245004177093506, -0.6420740485191345, -0.5470308065414429, -0.720643937587738, -0.6433575749397278, -0.45980212092399597, 1.968976378440857, 0.043974634259939194, 1.043225884437561, -0.2590999901294708, 0.22984017431735992, 0.15531811118125916, 0.5319746136665344, -2.6547985076904297, 0.549470067024231, 1.2809350490570068, 0.25902923941612244, 0.00932400207966566, -1.109068751335144, 0.17472602427005768, 0.6627540588378906, 0.7912644743919373, -0.4194275438785553, 1.0864540338516235, 1.830744743347168, 0.15658560395240784, -0.0078212795779109, -0.5159002542495728, 2.0199859142303467, -0.8309118151664734, -0.6905723810195923, -0.45453259348869324, 1.1943447589874268, -0.05352950096130371, -0.36686888337135315, -0.49825140833854675, 0.9078758358955383, 0.021973518654704094, -0.5347628593444824, 1.340431571006775, -0.11542757600545883, 0.9453255534172058, -0.11242686957120895, -0.5613308548927307, -1.2013767957687378, 0.16981348395347595, -0.8335723876953125, -0.2947230935096741, 0.9516035914421082, -0.46688005328178406], [-0.4714404344558716, -0.2717275023460388, -1.1305031776428223, -0.05289773270487785, 1.738265872001648, 0.6342845559120178, -2.1491453647613525, -0.901508629322052, 1.3514842987060547, -1.237602710723877, -1.5898979902267456, -1.5220154523849487, -0.0891488641500473, -1.4962942600250244, -0.28093570470809937, -0.3486911654472351, 0.46568506956100464, 0.11562976986169815, -1.1256887912750244, 0.16579210758209229, 0.7734971642494202, -1.0895673036575317, -1.5480947494506836, -1.1396328210830688, -0.17850850522518158, 0.6007871627807617, -0.40500205755233765, -0.3513987958431244, 0.6306049227714539, 1.431681752204895, -1.2859002351760864, 0.8281065821647644, 0.2478848099708557, 0.9958755970001221, -1.0270731449127197, -0.2021091878414154, -0.9365894794464111, -0.8907572031021118, 0.04387307167053223, -0.9357489347457886, -0.4681461453437805, 1.39173424243927, -0.7725585699081421, -0.3427838385105133, 0.12804582715034485, -1.965567946434021, -0.45814990997314453, 0.5774984359741211, -0.9531984925270081, 0.12256629019975662, 0.707287609577179, -0.8688473701477051, -0.24539832770824432, -1.732528567314148, -0.07827773690223694, -2.157057523727417, 2.3766582012176514, -0.6443353891372681, 2.726418972015381, -0.8053898811340332, -0.9630233645439148, 1.2598233222961426, -0.43783673644065857, -0.18048198521137238], [0.9243674278259277, 0.4543251395225525, -1.1531257629394531, 0.31904375553131104, 1.711802363395691, -0.8858056664466858, 0.11193565279245377, -0.9480963349342346, -0.5929170846939087, 0.27659907937049866, 2.4829742908477783, -0.23449186980724335, -0.5912302732467651, 0.3591400980949402, 0.2572096884250641, 1.7857670783996582, -0.44258859753608704, -0.8431145548820496, 0.43571746349334717, -1.9743636846542358, 0.3412089943885803, 0.09308905154466629, 0.15413491427898407, -0.3706623613834381, -1.5398285388946533, -1.1809357404708862, -0.4201042950153351, 0.9522272944450378, 0.48547104001045227, -1.6379772424697876, 1.688228964805603, -0.7682881951332092, -0.4685133695602417, -0.11206801235675812, 1.1061862707138062, -1.2560789585113525, -1.4040474891662598, 1.9567118883132935, -0.22027547657489777, -1.0500835180282593, -1.1920336484909058, 0.5395601987838745, 1.4284684658050537, 0.6876431703567505, -0.8730467557907104, -1.491620421409607, 2.3229329586029053, -1.265545129776001, -0.5337572693824768, -0.2939336895942688, -1.704677939414978, 0.6272746920585632, 0.30946484208106995, -0.5441139340400696, 0.06797096133232117, -0.9310495257377625, 0.5034313797950745, 0.8946105241775513, 0.15376263856887817, -0.32598593831062317, 0.3111508786678314, 0.17285005748271942, -0.17115958034992218, -0.9710906147956848], [-0.7174873352050781, 0.3957963287830353, 1.9320180416107178, 0.6223089098930359, 0.2827800512313843, -1.0931025743484497, -1.1645559072494507, -2.0431277751922607, 0.1968006044626236, 0.6695841550827026, 0.3142538368701935, 1.9588732719421387, 0.04369625076651573, 0.12457043677568436, 0.2768954634666443, 0.542226254940033, 0.23791271448135376, 0.567485511302948, 0.21178488433361053, 0.8293756246566772, -1.5981709957122803, -0.01054970733821392, -1.1979498863220215, -0.9530875086784363, 0.9377337694168091, -0.49983832240104675, 1.7687928676605225, -1.1498839855194092, 0.369935005903244, 1.6489025354385376, -1.0649064779281616, -0.03402261435985565, -0.8416520357131958, -0.3481677770614624, 0.6565401554107666, 1.2225316762924194, -0.2993066608905792, -0.4772063195705414, 1.0888174772262573, -2.229768991470337, 1.6987254619598389, -0.21545632183551788, 0.025207528844475746, 0.49195876717567444, -0.3938846290111542, -0.5267042517662048, 0.7111414074897766, -0.9003169536590576, 1.3432966470718384, 2.5986809730529785, -0.15613840520381927, -1.546502709388733, 1.6295366287231445, -0.7270790934562683, -1.753885269165039, 0.6253148317337036, 0.19494712352752686, -0.5285641551017761, 0.7325975894927979, 1.683518409729004, 0.354568749666214, -1.0996962785720825, -1.1907811164855957, -1.179613709449768], [1.9593567848205566, 0.8108476400375366, 0.5484966039657593, -0.4929182827472687, 1.0377477407455444, 0.7274337410926819, 1.347495675086975, 0.7710073590278625, 0.47149229049682617, -0.5716707110404968, 0.44550344347953796, -1.5113306045532227, 0.2204390913248062, 0.8712990283966064, -0.8215474486351013, -1.5829334259033203, 1.6821974515914917, -0.5385991930961609, 0.7865878939628601, 1.260382890701294, -0.5727942585945129, 0.3319362699985504, -0.9737482666969299, 0.7400122880935669, 0.3986157774925232, 0.051583532243967056, 0.21294385194778442, 1.191928505897522, 1.777359962463379, 0.7761373519897461, -0.06608368456363678, 2.025211811065674, 0.053939007222652435, -0.9090463519096375, -0.3016932010650635, -0.27733156085014343, -1.1464508771896362, -2.0247445106506348, 0.14980091154575348, 0.08476653695106506, -0.7845685482025146, -1.0853819847106934, -0.648400068283081, -1.0596028566360474, -0.9817651510238647, -0.447970449924469, -1.0412474870681763, 0.7555753588676453, -1.4709513187408447, -1.362223744392395, -2.0716121196746826, -0.00927666388452053, 1.4141101837158203, 2.0537126064300537, -0.5679174065589905, 0.5706602931022644, 0.11139766126871109, 0.08814835548400879, -1.042869210243225, -1.2907668352127075, 0.6543149352073669, -1.1957041025161743, 0.5886408090591431, -1.2959965467453003], [-1.3300648927688599, -1.132699966430664, 0.5719430446624756, -0.5575735569000244, 0.17241758108139038, 0.0420512892305851, -0.8736108541488647, 1.0448287725448608, -1.674337387084961, 0.5745865106582642, 0.9677722454071045, 0.3735902011394501, 0.07616866379976273, 0.38841190934181213, -0.05500122904777527, -0.06665854156017303, -1.4108271598815918, -0.4052668511867523, -0.8833252787590027, 0.05963214114308357, 1.0251189470291138, 0.05921727046370506, 0.08552110195159912, 1.0292316675186157, 1.434078574180603, 0.06653156131505966, -1.992896556854248, 0.7895679473876953, 0.29553017020225525, -0.6581209897994995, 0.7395796775817871, -0.10431931167840958, 0.14547576010227203, -0.4879383146762848, -1.598671793937683, -0.46494486927986145, -1.7481756210327148, 0.09113648533821106, -0.618587076663971, -0.47786206007003784, -1.1653612852096558, 0.2156333476305008, 0.029328079894185066, -1.034488558769226, 0.4664992690086365, 0.31727689504623413, -0.018351148813962936, 1.56270170211792, -0.7136725783348083, -2.2316064834594727, 0.17759455740451813, 1.1749858856201172, -0.29375290870666504, 0.8300009965896606, -1.1182880401611328, 1.7417237758636475, -0.4159994423389435, 0.35734260082244873, 1.7467116117477417, 1.366808533668518, 0.7266677021980286, 0.6874732375144958, -0.8523837924003601, 0.13205504417419434], [0.020097343251109123, 0.9411090016365051, 0.5163654685020447, -0.04101534187793732, -0.6720257997512817, 0.30396538972854614, -2.261852979660034, -0.6261449456214905, -0.23011861741542816, -0.08439873903989792, -0.4121570289134979, -0.9161949157714844, 1.701185941696167, -0.5014172196388245, 0.1233297735452652, 1.4144301414489746, 0.3042229413986206, -0.8878554105758667, -1.1153620481491089, -0.47020167112350464, -0.5198407769203186, 0.9928944706916809, -0.6444304585456848, -0.4192308485507965, 1.4984811544418335, -1.310487151145935, -0.6266355514526367, -0.21849282085895538, -1.5835167169570923, 0.4475093185901642, 0.6603357195854187, -0.5250667333602905, -1.1093645095825195, 0.2784710228443146, -0.5102279782295227, 0.2601398825645447, 0.4155575931072235, -0.510723888874054, -0.5624845623970032, -0.9393887519836426, 1.283012866973877, -0.3875783085823059, -2.4257946014404297, -0.6672754883766174, -0.5613924860954285, 0.49920153617858887, -1.2821582555770874, -0.08671879768371582, 0.0508762001991272, -3.0619771480560303, 0.15511128306388855, 0.26985597610473633, 1.037064552307129, 0.07561026513576508, -1.3173925876617432, -1.70106840133667, -1.6521316766738892, -0.26462695002555847, -0.7540456652641296, -1.584266185760498, -0.22288452088832855, 1.6822092533111572, 0.585032045841217, 2.2892961502075195], [1.867545485496521, -0.023696469143033028, -0.7864766120910645, 0.02971603162586689, -0.8953982591629028, -0.5990004539489746, 1.1730170249938965, 1.2160316705703735, 0.4575027525424957, -0.42648717761039734, 1.375264048576355, 1.1161067485809326, -1.0190882682800293, 1.4147183895111084, -0.44146957993507385, 0.4377937614917755, 0.6391639709472656, -1.2252874374389648, 0.6925689578056335, 0.16664089262485504, -1.0127023458480835, -0.38039910793304443, 0.44561803340911865, 0.5460838079452515, 0.22626663744449615, -0.7143549919128418, 1.2046056985855103, -1.6434741020202637, 0.6245993971824646, -1.1211023330688477, -3.248417854309082, 0.0905115082859993, -0.16191142797470093, 0.38803908228874207, -0.9846689701080322, 0.08791700005531311, -0.5015994906425476, 0.9704058766365051, 0.6672098636627197, 1.3272204399108887, -1.700020432472229, -1.0353219509124756, -0.16810302436351776, 0.15542088449001312, 1.5775878429412842, -0.06474079191684723, -1.1526575088500977, 0.9780644178390503, -0.26734310388565063, -1.9095960855484009, 1.2499644756317139, -1.7703074216842651, -0.146631121635437, -0.33314818143844604, -0.024083757773041725, -0.35699552297592163, -0.21643106639385223, -0.1786402016878128, 1.0183253288269043, 0.1463809609413147, -2.2869505882263184, -1.3619310855865479, -0.9199525117874146, 0.5153880715370178], [-0.1330556869506836, -0.033838726580142975, -0.7663230299949646, 0.010583980940282345, 0.5450007915496826, -1.7845839262008667, 0.1843710094690323, 0.5818100571632385, -0.3605591654777527, 0.4791982173919678, -1.385237693786621, -0.44094231724739075, -1.031878113746643, -0.15533997118473053, 0.22982728481292725, -1.6399965286254883, -1.2711033821105957, 0.7418385744094849, 0.5899134874343872, -0.1387484222650528, -0.7050046920776367, -1.1703134775161743, 2.5813839435577393, 0.4429464638233185, 0.39251723885536194, 1.3751463890075684, 0.3611027002334595, -0.5142675638198853, -0.334700345993042, -0.13805674016475677, -0.2975083291530609, -1.207339882850647, -0.23164135217666626, 0.06880400329828262, 0.8139942288398743, -0.23710602521896362, -0.5850708484649658, -1.2749336957931519, 0.5684313774108887, 0.741436779499054, -0.5889307856559753, -0.5674102306365967, 2.2830052375793457, 0.056934598833322525, 0.18049462139606476, 0.47672805190086365, -0.007092916872352362, -2.7654895782470703, -0.3744792640209198, 0.8095903992652893, 2.0416829586029053, 0.34401020407676697, -0.6395689845085144, 2.9177286624908447, 0.6636360287666321, 0.6832203269004822, -1.6960936784744263, 0.037062931805849075, -0.9191224575042725, 0.16078561544418335, 0.47994017601013184, 0.5360414385795593, -1.4312782287597656, -0.18684494495391846], [0.6035536527633667, 1.2963048219680786, 0.08878212422132492, -0.7270740866661072, 1.0702897310256958, -1.6182321310043335, -1.0041263103485107, -0.7493161559104919, -0.13604292273521423, -0.5839630365371704, -0.7424390316009521, -0.6215301156044006, -1.5875896215438843, -1.1338708400726318, -1.3615427017211914, -0.2620600759983063, 0.13789725303649902, 0.3886391520500183, -1.702233076095581, -2.1829237937927246, -0.6035423278808594, 0.5255755186080933, 0.3155026435852051, -0.15933340787887573, -0.9777219295501709, 0.8481524586677551, -0.7476168870925903, 2.1425533294677734, 0.29317814111709595, 1.8416813611984253, -0.5239748954772949, -0.04236234724521637, 0.40531855821609497, -0.5936011672019958, -0.03869200125336647, 1.7526339292526245, 0.548585057258606, 0.011388765648007393, -0.5312739014625549, 0.2015279084444046, -1.1067540645599365, -1.187684416770935, -0.7595692873001099, -0.012633448466658592, -1.1040958166122437, -1.3364006280899048, 0.5890966057777405, -1.889214277267456, -0.7378137111663818, -0.07836224883794785, -1.6088844537734985, 1.3101539611816406, -0.21394863724708557, 2.1078579425811768, 1.1287959814071655, -0.5100487470626831, -1.4522712230682373, -1.1045961380004883, -0.7369336485862732, -0.22704775631427765, 0.47506409883499146, 1.1352171897888184, -2.22988224029541, 1.6154009103775024], [0.9927647709846497, -0.724449098110199, 0.3818366229534149, 0.15187492966651917, -1.2257280349731445, -0.9755680561065674, 0.766091525554657, -0.42979195713996887, 0.004433429799973965, -1.852596640586853, -1.369301676750183, -0.8547898530960083, -0.4141022861003876, -0.15550857782363892, -0.3173035979270935, 1.3828486204147339, -1.6387282609939575, 0.05396469309926033, 0.9542078375816345, -0.17191965878009796, 0.23154506087303162, 0.05452924966812134, 0.4548397660255432, -0.013676492497324944, -0.39385589957237244, 1.3191126585006714, 2.2339930534362793, 1.3090201616287231, 1.3751294612884521, -1.3314881324768066, 0.3478766977787018, 0.3024497926235199, 0.5774208307266235, 0.9606632590293884, -0.9056143760681152, -1.1353124380111694, -1.7486413717269897, 0.7897266745567322, -3.4333508014678955, -0.4710718095302582, -0.5874510407447815, -0.81491619348526, 0.7332466244697571, -0.3515181541442871, -1.0913907289505005, 0.011597855016589165, 1.0990874767303467, -0.11242938786745071, -1.0028672218322754, 0.20323918759822845, 0.0766400694847107, 0.49147284030914307, 2.9579858779907227, -0.30777880549430847, 0.14084702730178833, 0.9658681750297546, -0.9753976464271545, -1.3714920282363892, -1.0711716413497925, 0.5515204668045044, 0.9912708401679993, -0.16489745676517487, -0.5418311953544617, 0.07401116192340851], [0.3425549566745758, -0.9706212282180786, 0.7131249308586121, -0.639597475528717, 1.7449443340301514, -0.416644424200058, -0.3242293894290924, 0.7686778903007507, -0.07880514860153198, -0.5146073698997498, 0.25510501861572266, 0.4991757273674011, 1.3117481470108032, -0.11862394213676453, -1.0634675025939941, 0.5869463682174683, 0.9327155947685242, -0.935961902141571, 0.7359321117401123, -1.0611753463745117, 2.2764945030212402, 1.4150151014328003, -0.05492760241031647, 1.7339645624160767, 0.00420512817800045, -2.223073959350586, -0.6843287348747253, -1.2300801277160645, -0.542984127998352, -1.7340863943099976, 0.7893562912940979, -0.34669482707977295, 1.1026122570037842, 0.7632985711097717, -0.7980895042419434, -0.38442322611808777, 1.5884568691253662, -1.3802493810653687, 0.36325570940971375, 1.683279275894165, -0.3000403940677643, -0.05991169437766075, 1.8148059844970703, 0.8078741431236267, 0.7398629188537598, 1.1454179286956787, -0.402919739484787, -1.275704264640808, -1.4796274900436401, -0.3028848171234131, 1.6869689226150513, -1.6699743270874023, -0.7161673307418823, -1.6168125867843628, 0.058489441871643066, -1.5392816066741943, -1.3792527914047241, -1.0115796327590942, 1.2129324674606323, -0.964970588684082, 0.25172629952430725, -2.5705769062042236, 0.1466352343559265, -0.3468286991119385], [-0.6844878792762756, -0.15340638160705566, -0.6047052145004272, 0.0044029890559613705, -0.8713692426681519, -0.6703438758850098, -0.4205864369869232, -0.2903117835521698, 0.34227925539016724, 0.25498807430267334, 0.7502628564834595, 0.21044839918613434, -1.74663507938385, -0.10625284910202026, -0.6652277112007141, 0.8128170371055603, 1.0608365535736084, 0.7796863317489624, 0.5047032833099365, 0.3842880427837372, -0.6101850867271423, -0.08883360773324966, 0.16507196426391602, -0.8792680501937866, -0.3568267822265625, -0.697333574295044, 1.2092293500900269, 1.1974029541015625, -1.3228152990341187, -0.7284706234931946, -0.690857470035553, 0.45634394884109497, 0.3033718168735504, 0.14318324625492096, -0.000450680818175897, -0.5852354764938354, -1.854587435722351, 0.6651721596717834, -1.4310418367385864, -0.46875035762786865, 1.1342343091964722, 1.6893349885940552, -0.18120074272155762, 1.385158658027649, 1.33623206615448, -0.2432880848646164, -0.652801513671875, 2.4206671714782715, -0.2264651209115982, 0.6333624124526978, 0.4006105661392212, 0.9371936917304993, 0.4070674777030945, -1.2097402811050415, -0.16245873272418976, -0.24156637489795685, 1.632203221321106, 0.7079259753227234, -1.2942180633544922, -0.6156396269798279, -0.870908260345459, 0.8733847141265869, -0.9261369705200195, 0.9369191527366638], [0.33698487281799316, -0.14309683442115784, 0.6456031203269958, 0.367753803730011, -0.5681109428405762, 0.2420516163110733, 0.5712316632270813, 0.7868455052375793, -0.39747512340545654, -0.6448667049407959, -0.7950474619865417, -1.6911704540252686, 0.7848426103591919, 0.6738439798355103, -0.5875180959701538, -0.3117386996746063, -0.5091241598129272, -0.033956270664930344, 0.009902216494083405, -1.327481746673584, 0.6651787161827087, -0.2841043770313263, 1.584464192390442, -1.349386215209961, 0.19428588449954987, 1.3810584545135498, -0.8713402152061462, -0.8035206198692322, -0.6867114305496216, 0.0993293970823288, 1.0372083187103271, -0.3227422535419464, -0.5291146636009216, -0.35405686497688293, 0.9432036280632019, 1.0764615535736084, -0.5784702897071838, -1.248698353767395, 0.5747318863868713, -0.25543123483657837, 0.5451489686965942, 0.35181811451911926, -0.5438130497932434, -0.024137159809470177, -0.32664763927459717, 0.3360885679721832, -0.9321009516716003, -1.3824959993362427, -0.9399609565734863, 1.0790419578552246, 0.060999929904937744, 0.010627669282257557, 0.15971530973911285, 0.0013978553470224142, -0.36402031779289246, 0.1836673468351364, 0.5015982985496521, -0.1624826341867447, -0.6909419298171997, -0.3377710282802582, -2.2531535625457764, 0.9203122854232788, -0.37973058223724365, -0.5728649497032166], [-0.4670018255710602, -1.309809923171997, -1.1926249265670776, -0.6345624923706055, 0.30065709352493286, -0.3520987629890442, -0.08467532694339752, 0.12599541246891022, 2.51149320602417, -1.4260073900222778, -0.9512670040130615, -0.6042259931564331, 0.7850762605667114, 0.22241206467151642, -0.6833987236022949, 1.0583266019821167, 0.2740604877471924, -0.7744988799095154, -0.23957376182079315, 0.0647975280880928, 0.979324996471405, 0.4135408401489258, 1.5350618362426758, 0.3562510013580322, -0.9871811866760254, -0.5175118446350098, 1.0536994934082031, -0.6704939007759094, 0.8384782075881958, -2.477137327194214, -2.114502191543579, -1.1961511373519897, -1.008524775505066, 0.5869141817092896, 1.7905421257019043, -1.7723931074142456, 1.8354229927062988, -1.6341689825057983, -1.4502474069595337, 2.172628164291382, -0.18597863614559174, 0.8926548957824707, 2.143444776535034, 1.6008050441741943, -2.7488973140716553, 0.7689651846885681, 0.5680238008499146, -1.1473283767700195, -0.2638399004936218, -2.2061495780944824, -0.3516906201839447, 0.7628124952316284, -1.2975555658340454, 1.2137303352355957, 0.6830630898475647, -0.6891375780105591, -0.38704952597618103, 0.06192617863416672, 1.7446320056915283, 0.5014417767524719, 0.8719435334205627, -0.7676185369491577, 2.188795804977417, -1.6263911724090576], [-1.0181108713150024, -0.9177629351615906, 0.34895360469818115, -0.15149062871932983, 0.8380622863769531, 0.07601583749055862, -1.1801352500915527, -3.3448853492736816, -1.08934485912323, -1.335262417793274, -1.3222029209136963, -0.8010282516479492, 0.5962110161781311, -1.1054890155792236, -0.8625468015670776, -1.77251398563385, 1.038780689239502, -1.6814824342727661, 0.2506735026836395, -0.5636376738548279, -0.696893036365509, -0.006267718505114317, -0.6700879335403442, -0.37190061807632446, 0.4279729127883911, 0.9472533464431763, 1.3774867057800293, -0.03431584686040878, 0.21454575657844543, 0.7722441554069519, 1.2421320676803589, -1.2106281518936157, 0.3262898027896881, -0.10109397768974304, -1.0260480642318726, 1.5009849071502686, -1.8904871940612793, 0.12292015552520752, -2.285440444946289, -1.027480959892273, -1.2647244930267334, -0.529647171497345, 0.4897172152996063, 0.4399479329586029, 0.17389394342899323, 1.3667516708374023, 0.0617915503680706, 0.2347727119922638, -0.5579524040222168, 0.26307106018066406, -0.36618003249168396, -0.9287655353546143, 1.5842732191085815, 0.5405188202857971, -0.6378239393234253, -2.488083600997925, -0.9107456803321838, 0.9775080680847168, -1.816817283630371, 1.4296643733978271, -0.048459749668836594, -0.7866856455802917, 1.3119890689849854, -1.7979345321655273], [0.6422282457351685, 0.8443087935447693, 2.1401872634887695, -1.3894699811935425, -0.8420606255531311, -1.6738741397857666, 0.8530993461608887, 1.1793612241744995, 0.6169725060462952, -1.148682713508606, 0.9478691816329956, 0.5919989943504333, -0.07650106400251389, 0.1376628577709198, -0.060702886432409286, 0.5700681805610657, 0.8529723286628723, -0.3438224494457245, -0.6384153366088867, -0.2978113889694214, -0.262331485748291, -0.38227447867393494, 1.981677532196045, 0.1434103548526764, 1.5685768127441406, -1.4010587930679321, -0.7090238928794861, 0.6423386335372925, 1.492180585861206, 1.707746148109436, 1.4070872068405151, 1.382028579711914, 0.5475022196769714, -0.209687739610672, 0.7473593354225159, -1.3696554899215698, 0.44917285442352295, 1.3237509727478027, -0.7277783155441284, 0.5366883277893066, -0.7031500339508057, -1.3239781856536865, -2.3756096363067627, 0.6192632913589478, -0.01693987101316452, 1.6802905797958374, 0.7798415422439575, 1.6441423892974854, 0.6201130151748657, -0.07918454706668854, -0.5731468796730042, -0.3066801130771637, -0.10830499976873398, 0.6955310106277466, -0.6861361265182495, 0.501571774482727, -0.8266080021858215, 2.4201598167419434, -0.43846720457077026, 0.12372822314500809, -1.709852695465088, 1.6223782300949097, 0.29311099648475647, -0.509331226348877], [-0.3824390470981598, 0.5396720767021179, -2.2520127296447754, 0.8888798952102661, -0.9723812937736511, -1.2872545719146729, 0.22643594443798065, 0.03665011003613472, -1.042985200881958, 0.2421693503856659, -0.08908766508102417, 0.5340263843536377, 1.412835955619812, -1.1266961097717285, -1.19771409034729, 0.5819401144981384, 0.040550895035266876, -0.9064649939537048, -0.35613149404525757, 1.46884286403656, 0.9234662055969238, -1.1792596578598022, 1.2948853969573975, 0.6249229311943054, -1.6157768964767456, -0.2108682245016098, 1.53428053855896, 1.0756179094314575, 0.39783233404159546, 0.8094498515129089, 0.945106029510498, -0.9791913628578186, -1.724333643913269, -0.7376576662063599, -0.15721966326236725, 0.5451104044914246, -0.08634865283966064, 0.3464648425579071, -0.8569558262825012, 0.42938968539237976, -2.7012414932250977, -0.8254826068878174, -1.553247332572937, 1.3137977123260498, 0.10708329826593399, -1.0658059120178223, 0.7189614176750183, 0.08746509999036789, 0.6280458569526672, 1.0233495235443115, 0.46129152178764343, -0.5769460797309875, 0.17880050837993622, -0.9698954224586487, 1.5057423114776611, 1.6727759838104248, -0.22152218222618103, 1.0094391107559204, 0.5740653872489929, -0.3826877772808075, -1.3212018013000488, -0.5954049229621887, -1.4795405864715576, -0.6630476117134094], [-0.4952324628829956, -0.3817182779312134, -0.05060768872499466, -1.1343059539794922, -2.4257118701934814, -0.34518688917160034, 0.0010883054928854108, 0.5775970220565796, 0.15834477543830872, -1.619119644165039, 0.9057018160820007, 0.09399397671222687, -0.41863173246383667, -0.7172384262084961, -0.4273691177368164, 1.4885624647140503, 0.07378607988357544, 0.5535597801208496, -0.9106501340866089, 1.6257704496383667, -0.33570364117622375, 0.3689712882041931, 1.3049955368041992, 0.0275675430893898, -0.8927763104438782, 0.05964554101228714, 0.3652906119823456, 2.1212029457092285, -0.14706985652446747, 1.0888055562973022, -0.7833417057991028, 1.157930612564087, -1.2755202054977417, -0.15590080618858337, 0.7329790592193604, 0.202840656042099, 0.4866175949573517, -0.43437573313713074, 0.04774926230311394, 0.37867704033851624, 1.324194312095642, -1.8305996656417847, -0.7650626301765442, 0.058435190469026566, 0.2674313485622406, 0.9209557771682739, 0.9500649571418762, 1.7406489849090576, -1.1553726196289062, -0.27188438177108765, 0.04807528480887413, 0.527854859828949, 0.0988810583949089, -0.3629531264305115, 1.235453724861145, 0.37970998883247375, -0.6672321557998657, 1.361324667930603, 0.07544906437397003, 1.6842187643051147, 0.4220940172672272, 1.1648108959197998, -0.8525307774543762, -1.2334198951721191], [0.46384504437446594, 1.1646044254302979, 0.025141602382063866, -0.8413407206535339, -1.0193047523498535, -0.2750184237957001, -0.5546680092811584, -0.5451608896255493, 1.0303397178649902, 1.6224558353424072, -0.10767919570207596, -0.5797777771949768, -2.522777557373047, 1.3565254211425781, -1.0444824695587158, -1.2424439191818237, 0.02663884498178959, 1.7068238258361816, -0.19211548566818237, 5.06883430480957, -1.4826759099960327, 0.6277399063110352, 0.4720754623413086, 1.1034283638000488, -0.9452861547470093, 1.3053276538848877, 0.8187456727027893, 0.5258957147598267, -0.5436804890632629, -1.1329131126403809, -2.8642046451568604, -2.4069271087646484, 1.1928648948669434, 0.116496741771698, -0.48300793766975403, -1.3252629041671753, -0.7771583795547485, 0.44981929659843445, -0.6634334325790405, 0.7319304943084717, 0.23106621205806732, 0.4518151581287384, 0.2420535832643509, 0.26162752509117126, -0.6084694266319275, 1.1112827062606812, 0.7656811475753784, -1.2054669857025146, 1.326745629310608, 0.8328198790550232, -0.28425052762031555, 0.9465662240982056, 1.5908257961273193, -0.0665644034743309, 0.13399428129196167, -0.4412923753261566, 0.1328570395708084, -0.0718112587928772, 1.6290650367736816, -0.12907083332538605, -1.2487831115722656, -0.30733299255371094, -0.7565962076187134, -1.3673583269119263], [-0.6670296788215637, 0.79872065782547, 0.07357994467020035, -2.3947694301605225, 0.6441331505775452, -0.6608012914657593, -1.6673336029052734, -2.9668667316436768, 0.8403513431549072, -0.8749079704284668, 1.6130871772766113, 1.3047282695770264, 0.7778075933456421, -0.5783951282501221, 0.8563864827156067, -0.30061861872673035, 0.7384811639785767, -1.274705171585083, 0.025341464206576347, -0.37904495000839233, -2.038189172744751, 1.3465564250946045, -0.2489362359046936, 0.899526834487915, 0.3947937786579132, -0.03809351101517677, -1.3837388753890991, 1.017706274986267, -0.15154971182346344, -0.7281270623207092, 0.32405468821525574, -1.0750946998596191, -0.6358539462089539, -0.5950692892074585, 0.4914902448654175, -1.2925997972488403, 0.6617017984390259, 0.6202102303504944, -0.2896408140659332, 0.8986055254936218, 0.6852725148200989, -0.5669347047805786, 0.8922169208526611, -0.8686186075210571, 0.40128380060195923, 1.684199571609497, 0.5267022252082825, -0.7492054104804993, 1.5255411863327026, 0.5542409420013428, 0.3013596832752228, 0.524630606174469, 0.9335340261459351, -0.42448508739471436, -1.049307107925415, 1.406947374343872, -0.4622034430503845, -1.0446218252182007, 0.036001287400722504, 1.4324642419815063, -0.0662621408700943, 0.05926651880145073, -0.20596840977668762, -0.1302965134382248], [1.7015559673309326, -0.5928544998168945, 1.4172853231430054, 0.8990195393562317, -0.3150273859500885, -0.2692766785621643, 0.9593015313148499, 2.0336761474609375, 0.03406171873211861, -1.401128888130188, -0.764920711517334, -0.26915043592453003, -0.8650668859481812, 1.0726293325424194, -0.9322682619094849, -0.3007545471191406, 1.6939527988433838, 0.877275288105011, 1.813961148262024, 0.4435407519340515, 0.2044808566570282, 1.4455673694610596, 0.06980717182159424, 0.0957578793168068, 1.1870622634887695, 0.8767192959785461, -0.3089992105960846, -1.3116703033447266, -1.0577967166900635, 0.14891037344932556, -0.6365554928779602, -0.1972499042749405, -0.8096851110458374, -0.6426846385002136, 0.8900681138038635, 0.42684611678123474, -1.6937342882156372, -0.9247022867202759, -1.4019584655761719, -0.19393713772296906, 0.9368463754653931, 0.6755982041358948, 0.8198315501213074, 0.0879305973649025, -0.04802216216921806, 2.5873985290527344, 0.34841492772102356, 1.5154850482940674, -1.1616415977478027, -0.6104562282562256, -0.09035272896289825, 0.8245442509651184, -0.9706960320472717, -1.0936661958694458, -1.1309335231781006, 0.7356988191604614, -0.5916687846183777, 0.5532344579696655, -0.341000497341156, 0.6740144491195679, 1.3706203699111938, -1.2013646364212036, -0.3439933657646179, -0.6545083522796631], [-0.3093799650669098, 1.4872474670410156, 2.4824037551879883, 0.9604960083961487, 0.04303868114948273, -0.7259787321090698, -0.34020674228668213, -0.1671968400478363, 0.03600798919796944, -0.23555880784988403, -0.8600793480873108, -1.1761220693588257, 0.784076452255249, 0.43034613132476807, -0.19765371084213257, -0.46736666560173035, 0.3890174627304077, -0.9994913339614868, 0.26506751775741577, 0.09318523854017258, 0.902930736541748, -0.05441810563206673, 1.0822651386260986, 0.8123624920845032, 1.3985122442245483, 0.3382989764213562, 0.30300647020339966, 0.03708072751760483, -0.23737676441669464, -1.432077407836914, 0.4299969971179962, 0.26060351729393005, -0.27343231439590454, -0.5642479062080383, -0.5148661136627197, -2.047344207763672, 0.41882580518722534, 0.6209110617637634, 1.0628979206085205, -2.642625331878662, 2.122020721435547, -0.8206411004066467, 0.20728884637355804, -0.051727332174777985, 0.044656042009592056, -1.370427131652832, -0.06949570775032043, -1.4853445291519165, 0.9724719524383545, -2.7347826957702637, 0.7770406007766724, -0.5936412811279297, 1.4963960647583008, -1.1316536664962769, -0.8137686848640442, -0.06385160982608795, 0.6604962944984436, 0.44203758239746094, 0.918048083782196, 0.8535606861114502, 0.012541444040834904, -0.41730666160583496, 0.855811357498169, 0.23967984318733215], [0.11237189173698425, 0.6828855276107788, -2.1280078887939453, 0.32969045639038086, -0.09352359920740128, 1.4552663564682007, 0.1444741189479828, 0.2416992336511612, 1.6983013153076172, -0.9437962174415588, 1.4497915506362915, 0.31611236929893494, -2.0117461681365967, -0.8622532486915588, -0.22736965119838715, -1.2153640985488892, -1.5210140943527222, 0.6165108680725098, -0.37570157647132874, 0.05810048431158066, 0.6013650894165039, -2.374248743057251, 0.0008899170206859708, -3.020364999771118, -0.33624279499053955, -1.2864961624145508, -1.4968643188476562, -1.8363721370697021, -0.18586722016334534, 0.5437564849853516, -0.9121758341789246, -1.4039145708084106, 0.9673156142234802, 0.8894649147987366, 1.1079906225204468, 0.9281330704689026, -0.17318934202194214, 0.8389669060707092, 1.1699501276016235, -0.9229337573051453, -0.8860832452774048, 0.2845005691051483, 0.20593318343162537, -0.7847999930381775, 0.6143047213554382, -1.9787968397140503, -0.1807747632265091, 0.17650151252746582, -0.7544282674789429, -0.8717642426490784, -0.10361117869615555, -0.5124151110649109, 0.7599613666534424, -1.916529655456543, 0.5739699602127075, -0.9522503614425659, -0.172170028090477, -0.7834829092025757, 0.9427743554115295, 1.0479696989059448, 0.1507188230752945, 0.01856939122080803, 0.19376999139785767, -0.19341889023780823], [0.7449236512184143, 0.20688915252685547, 1.7883268594741821, 0.5219178199768066, 1.5165883302688599, -0.004025123082101345, 0.24336400628089905, 1.8907614946365356, 0.1999288648366928, 2.2280313968658447, -0.21731455624103546, 1.3565073013305664, 1.2844310998916626, 1.1586990356445312, -1.545710802078247, 0.11920492351055145, 0.3453027904033661, -1.2711166143417358, -0.15793153643608093, 0.4620403051376343, 1.1066445112228394, 0.9761176109313965, -0.06078660860657692, 0.01706632226705551, -0.4298156797885895, -0.6765215396881104, 3.276094675064087, 0.773433268070221, -1.9457725286483765, -0.9311792850494385, -0.9700567722320557, -1.1763187646865845, 0.7298513054847717, -0.2377757877111435, -0.24103102087974548, 0.14176471531391144, 0.31603556871414185, 0.4798760712146759, -0.8897068500518799, 1.2180439233779907, 1.4618113040924072, -0.27271464467048645, 0.38485679030418396, 0.5014246106147766, 0.6532522439956665, 0.9113112688064575, -0.4869440197944641, 1.1955310106277466, 2.236774444580078, -0.24770969152450562, 0.06290062516927719, 0.5393196940422058, 0.5601412057876587, 0.9913639426231384, -0.5665833353996277, -1.0772197246551514, 0.17379845678806305, 0.12919870018959045, -1.7501755952835083, -0.27360010147094727, -1.050710678100586, 0.725929319858551, 0.8180553913116455, 0.9250288009643555], [-1.7122886180877686, 0.810105562210083, -0.06432879716157913, 0.6468183398246765, 0.7729119062423706, -1.5523955821990967, 1.441161036491394, -0.1403934210538864, 0.4727388024330139, 0.9129729270935059, 2.459635019302368, -0.6374484300613403, 1.5234814882278442, -1.6650341749191284, 0.18472634255886078, 0.14307895302772522, -0.04027874395251274, 0.40482625365257263, -2.101806640625, 1.6452975273132324, -0.9429180026054382, 0.5571908950805664, -2.3620846271514893, -0.8594812154769897, 0.09029039740562439, 1.460246205329895, -0.7075944542884827, 1.2890304327011108, 0.8340284824371338, 2.6856331825256348, 1.265432357788086, 1.771849274635315, 0.5575563311576843, -0.5645411610603333, -0.38018545508384705, 0.5403163433074951, -0.48899656534194946, -1.3363065719604492, 0.42674604058265686, 0.5074805617332458, -0.05328943952918053, -1.2035363912582397, 0.22770512104034424, -1.0784510374069214, 0.21046032011508942, 0.3352526128292084, 1.39034104347229, 1.3958040475845337, 0.6588808298110962, 0.6899276971817017, 0.8090121746063232, -0.3902324140071869, -0.9696508646011353, 1.8282865285873413, 2.0166139602661133, 0.13211539387702942, -0.661886990070343, -0.0788789689540863, 0.7269502878189087, 0.29472988843917847, 1.5005302429199219, 0.32144859433174133, -1.3654271364212036, 0.2794002592563629], [0.01930742710828781, -0.5535388588905334, 1.5601435899734497, -0.29966744780540466, -0.6082196235656738, -0.6134516000747681, -1.0850447416305542, 1.1738051176071167, -0.6427195072174072, 2.2572672367095947, 1.6619811058044434, -0.24925409257411957, 1.0475406646728516, -0.03460776433348656, 1.7383183240890503, -1.484319806098938, -0.09165728837251663, -0.36703941226005554, -1.0575886964797974, -0.2911551892757416, -1.6500473022460938, 0.8862249255180359, -0.3098657429218292, -0.49508416652679443, -0.4675620198249817, -0.37984171509742737, 0.39987459778785706, 0.7443748712539673, -0.5454174280166626, -0.6128979325294495, 0.9170138835906982, -0.7386934757232666, 1.3575559854507446, 1.6463955640792847, -0.5923562049865723, -0.7107316255569458, -1.0981574058532715, -1.0393593311309814, -0.03400062397122383, 0.24732835590839386, -0.852641224861145, -0.8506393432617188, 0.8914968967437744, 1.342476487159729, 1.8234810829162598, 0.1897849589586258, -0.8797179460525513, -0.6940491795539856, -0.24511176347732544, -0.5834852457046509, -0.736481249332428, -1.6629350185394287, -1.2248311042785645, -0.9292028546333313, 0.1673009693622589, 0.657263457775116, 0.6082335114479065, -0.4900001585483551, 0.6965548992156982, -1.8148995637893677, -0.2420472502708435, 0.113380566239357, -1.4860684871673584, 0.2801953852176666], [1.8728407621383667, 1.1830849647521973, -0.6319872140884399, -0.7164256572723389, 0.7153283357620239, 1.2089307308197021, 1.5647563934326172, -0.33050742745399475, -1.4463484287261963, 0.348615437746048, -1.1861662864685059, 1.4193191528320312, -0.9361609816551208, -1.9081717729568481, 1.3748196363449097, 0.8997678160667419, 2.2090256214141846, -0.14886589348316193, -0.6966836452484131, -0.3327266573905945, -0.728266179561615, -0.34386110305786133, -0.0058040739968419075, 0.4710727035999298, -0.9889413118362427, 0.7393303513526917, 0.2342587113380432, -2.0105502605438232, 1.0383280515670776, -0.7366012334823608, -1.6539148092269897, 0.7812394499778748, 0.551228940486908, -0.008972130715847015, -0.4169212281703949, 0.9464700222015381, 0.41330355405807495, -0.3051084876060486, -0.7449194192886353, -0.20937004685401917, -0.1963600218296051, -0.35494932532310486, -0.32995155453681946, -1.622754454612732, 0.5420859456062317, 0.16597287356853485, -0.634907066822052, 0.18322108685970306, -2.025115966796875, -0.3305107057094574, -1.4447969198226929, -0.21690049767494202, 0.7702213525772095, 0.5832262635231018, 0.15425804257392883, -1.0209182500839233, -0.2074144184589386, 0.21878685057163239, -0.3894718885421753, -0.4963749945163727, -1.3263416290283203, 0.19802674651145935, 0.7134859561920166, 0.4572793245315552], [0.639759361743927, 1.5867185592651367, 0.954960286617279, -0.39195406436920166, -0.7298973798751831, -0.07122313976287842, -0.10170365869998932, -0.8075085878372192, 0.6143489480018616, 1.4015027284622192, 0.24835291504859924, -0.5651304125785828, -1.1799582242965698, 0.6642553210258484, 0.029903382062911987, 2.2153820991516113, -0.3368031978607178, 0.27439701557159424, 0.363738477230072, 0.3636873662471771, 0.617526113986969, -0.40960702300071716, 1.8073543310165405, -0.4512457847595215, -0.5857100486755371, 0.48104697465896606, 0.08430541306734085, -1.123518466949463, -2.4256598949432373, -0.6589314341545105, -2.3198089599609375, -1.0676754713058472, -0.9193591475486755, -0.23848621547222137, -1.3284928798675537, -0.1345420628786087, -1.619106411933899, -0.4886118769645691, -0.5591263771057129, 0.7682615518569946, -0.5796895027160645, 0.46559417247772217, -0.7102900147438049, -1.1756088733673096, -0.6995104551315308, 0.6666811108589172, 0.0017959719989448786, -0.3005259037017822, -0.12000144273042679, -0.15918777883052826, -1.3339630365371704, -0.3051297068595886, 0.41958463191986084, -0.19682899117469788, -0.5447000861167908, -1.231840968132019, 0.2682884931564331, 0.05209009721875191, -2.626389980316162, 0.8134528994560242, 0.026454608887434006, 1.045960545539856, 0.37352660298347473, -1.0277905464172363], [0.8850323557853699, 0.22840867936611176, -0.5664215087890625, 0.048248715698719025, 0.4910993278026581, 0.6164938807487488, -0.9341509938240051, -0.7275220155715942, 0.34728479385375977, 0.34180083870887756, -1.050408124923706, 0.3037174642086029, 1.1310012340545654, -0.8222836852073669, 0.2053234577178955, 0.4128502309322357, 0.7153761982917786, 0.9357770085334778, 0.1738237589597702, -0.4656168222427368, 0.09030049294233322, -0.040038883686065674, 0.4873420298099518, -0.3239385485649109, -0.480131596326828, 0.6547868251800537, 0.6730603575706482, 1.6871296167373657, -1.1580439805984497, 0.19683654606342316, -0.7430464029312134, 1.2584922313690186, 0.288094162940979, 1.0314761400222778, -0.056854479014873505, -1.0929105281829834, -2.3330953121185303, 0.4120309054851532, 0.47853386402130127, -0.31096151471138, 1.7028142213821411, -0.7608949542045593, -0.3287956118583679, 0.34273242950439453, 0.4792075753211975, -0.11380881071090698, -0.579542875289917, -0.5838378071784973, -0.6939703226089478, -0.46231013536453247, 1.107247233390808, -0.04912344366312027, 1.9757490158081055, 1.5973546504974365, 0.6547378301620483, 0.19745294749736786, 0.7196342349052429, -0.24958102405071259, -0.7283739447593689, -0.3581944406032562, -2.737530469894409, 0.13519461452960968, -1.1792066097259521, -0.10697763413190842], [0.6177727580070496, -0.3343704342842102, -1.5016233921051025, -0.164260134100914, -3.124478816986084, -0.12201343476772308, -0.09297190606594086, -0.6528428792953491, -0.00033099279971793294, -0.09961258620023727, 1.3368823528289795, 0.28920289874076843, -1.08221435546875, 0.7956089973449707, -0.16206303238868713, -0.36195456981658936, -0.03758297488093376, -0.2693462073802948, 1.948363184928894, -1.6455142498016357, 1.7702240943908691, 0.18248644471168518, 0.943278431892395, 0.046557482331991196, -0.6932432055473328, -0.6228625774383545, 1.1504526138305664, 0.19417093694210052, 0.3219964802265167, -0.05622858554124832, -0.8854328989982605, -0.6189191341400146, 0.4078078866004944, -0.9178134799003601, -1.9713178873062134, -0.466592401266098, -1.51957106590271, -0.8218203186988831, -0.6610032320022583, 0.31405726075172424, -0.05850550904870033, -0.2929607927799225, -1.9485280513763428, 1.2102453708648682, 0.7458688616752625, -0.10942026972770691, -0.34649425745010376, -0.8691230416297913, -0.006529327481985092, 1.2436809539794922, -1.8901501893997192, 0.6085847020149231, -0.14750106632709503, 2.053481101989746, 0.7483261823654175, 1.0823248624801636, -1.5801877975463867, -0.9246031641960144, -0.04442787542939186, 0.5902601480484009, 1.0650407075881958, 1.1461467742919922, 0.6711815595626831, 0.9100896120071411], [-1.0250232219696045, -1.9128174781799316, -0.9376235008239746, -0.8161434531211853, -0.9437628388404846, 0.24448463320732117, -0.7934668064117432, 0.5643008947372437, 1.2194091081619263, -0.8421519994735718, -1.1482136249542236, 1.8978166580200195, 1.0773353576660156, -1.2455289363861084, 1.797678828239441, 0.8272342681884766, 1.252320647239685, -0.8723564743995667, 1.4617514610290527, -2.348435401916504, -0.23987901210784912, -0.3238527178764343, 1.446291446685791, 1.566215991973877, 0.5425898432731628, 2.2048068046569824, 0.8091225028038025, 0.7247763276100159, -0.5274603962898254, -0.9600517749786377, 0.9279468059539795, 0.8933505415916443, -1.2861249446868896, -1.0592340230941772, 0.23184342682361603, -0.7415593862533569, -1.6922200918197632, -1.7201073169708252, 0.25380027294158936, 0.41577640175819397, 0.03305989131331444, -1.862642765045166, -1.4236797094345093, 0.8546645641326904, -0.7854040861129761, 0.6201546788215637, 0.9258955121040344, -1.6397662162780762, 1.341405987739563, 0.7442923784255981, -0.7363094091415405, 0.29911085963249207, -0.3421321213245392, 0.9239521026611328, -0.7983503937721252, 0.24018852412700653, -1.157365083694458, 1.759853720664978, -0.26555922627449036, 0.4180862605571747, 1.1491621732711792, 0.8299702405929565, -0.9145736694335938, -0.6945289373397827], [0.7965319156646729, 0.6550034880638123, -0.5365551710128784, -0.2272079885005951, 1.0104784965515137, -0.6748504638671875, 1.4129350185394287, -0.124233178794384, -0.11928163468837738, 1.1759490966796875, 2.0015182495117188, -0.46978941559791565, 1.5607842206954956, -0.4877513647079468, -1.9203177690505981, -0.7981486916542053, 0.33660656213760376, -1.3470572233200073, 0.3524448871612549, 0.42316654324531555, 1.6699647903442383, 0.79301917552948, 0.11572115123271942, -2.1635689735412598, -1.3022900819778442, 0.2783873379230499, -0.1977831870317459, -0.15038755536079407, 0.03281187266111374, -0.07810328155755997, 0.25943851470947266, -0.28617966175079346, -0.5974246263504028, 0.9385437369346619, 1.4477779865264893, -0.5317581295967102, 0.74983149766922, -1.3329309225082397, 0.6027015447616577, -1.9493850469589233, -0.1187768504023552, -1.778045892715454, -0.4458293914794922, -0.03542179986834526, 0.17191673815250397, 0.6561073660850525, 0.769909143447876, -0.397598534822464, -1.4079642295837402, 2.52121639251709, -0.1556268185377121, -1.1501133441925049, 0.0012007555924355984, 0.4705025553703308, -0.5084975361824036, -0.9798495769500732, -0.4242241382598877, 0.4727070927619934, 0.26437675952911377, -0.46821025013923645, -0.9957625865936279, -0.3635876178741455, 0.395303338766098, 0.19643224775791168], [0.2729779779911041, 0.4033966660499573, -1.2675979137420654, 0.6231351494789124, 0.12091539055109024, -1.1083959341049194, -1.2740764617919922, 0.2285667061805725, 1.214370608329773, -0.051011763513088226, -1.4229167699813843, 0.17242439091205597, -0.05313441902399063, -0.04772942140698433, 0.5820422768592834, -1.585627794265747, -0.08113601803779602, 1.10956609249115, -0.5600779056549072, -0.6059455275535583, 0.4459489583969116, 0.38334953784942627, 0.7503240704536438, 0.72562575340271, -1.1480143070220947, -1.2831292152404785, 0.27269062399864197, -0.5463642477989197, -0.2932019531726837, -0.6277908682823181, 1.0581271648406982, 1.254415512084961, -1.6513700485229492, -1.951343059539795, 0.9893104434013367, 0.18533746898174286, 1.918003797531128, -0.8823439478874207, -0.16731515526771545, 0.7214416861534119, -1.3214694261550903, 0.7635071277618408, -0.6455822587013245, -1.4950436353683472, -2.3082923889160156, -0.07343751192092896, 0.2675607204437256, 0.14493262767791748, -0.1334807574748993, 0.4039474129676819, 1.6684967279434204, 1.3572272062301636, -0.06535395979881287, 1.5391219854354858, 0.5271250009536743, -0.8241980671882629, 0.44007188081741333, 0.7113232016563416, 1.1564092636108398, -1.1828432083129883, -1.382915735244751, 0.45369860529899597, 0.3659968674182892, -0.37645551562309265], [-0.02265535667538643, -0.20417404174804688, -0.7315387725830078, 0.25570812821388245, -1.0861049890518188, 0.731508195400238, 0.6725322604179382, -1.3542945384979248, -0.32185080647468567, 0.656119167804718, 0.19633899629116058, -1.6235237121582031, -0.7543030381202698, 0.04123213514685631, 0.06390175968408585, -0.33891528844833374, 0.9615891575813293, 0.8517924547195435, 0.1725766956806183, 0.8631408214569092, -0.28538352251052856, -0.8175328969955444, 0.6105749607086182, 0.29215288162231445, 1.0338375568389893, 0.04065241292119026, -0.7746512293815613, -2.8353030681610107, -1.296488642692566, -0.5604188442230225, -0.883771538734436, -0.5667105913162231, 0.15957629680633545, 2.030789375305176, -0.5110751986503601, 0.6483270525932312, -1.041879415512085, 0.1457998901605606, 0.5720508694648743, -0.056214991956949234, -0.34120434522628784, 1.620023488998413, -0.5072674751281738, 1.7279733419418335, -1.1973161697387695, -0.03320468217134476, -1.4007046222686768, 0.044667214155197144, -1.086773157119751, -0.24010491371154785, -1.9336072206497192, -0.7673454284667969, -0.17143484950065613, -1.7043209075927734, 0.1748497188091278, 1.1118868589401245, 1.1785945892333984, -1.5807050466537476, -0.29971304535865784, 1.2958228588104248, 0.796019434928894, 2.0272228717803955, -1.8607109785079956, -2.282827138900757], [0.12431728839874268, 0.7498170733451843, 0.4512280821800232, -0.10896774381399155, 3.256012439727783, 0.20892009139060974, 0.2876458764076233, -0.36589667201042175, 0.37460923194885254, 1.0163713693618774, 0.4042219817638397, 0.547016978263855, -1.0522211790084839, 1.6781725883483887, 1.0047866106033325, -0.35014447569847107, -1.4487454891204834, 1.9741134643554688, -0.4819934666156769, 0.7886350154876709, -1.3286257982254028, -2.252821922302246, 1.153090000152588, 0.9344983696937561, 0.17918550968170166, 0.3874139189720154, -1.154538869857788, 1.1244105100631714, 0.5752537250518799, -0.7135704755783081, -0.9669417142868042, 0.8567420244216919, 0.8571500182151794, 0.5974982976913452, -0.48952266573905945, 2.8628804683685303, 0.9114320278167725, 1.3212746381759644, 0.4789738357067108, 0.6707254648208618, 0.45929276943206787, -0.7584382891654968, 0.7946746945381165, 1.0546492338180542, 1.2535197734832764, 2.052297830581665, -1.3583183288574219, -0.060020990669727325, -0.5260578393936157, -0.4748167395591736, 0.6352329254150391, 1.863581895828247, -1.3291893005371094, 0.8779756426811218, 0.852553129196167, 0.14080817997455597, 1.575719952583313, -1.0361080169677734, -0.28984156250953674, 0.06652693450450897, -0.06055206432938576, -0.655728816986084, 1.2846332788467407, -0.3574754595756531], [-1.0583491325378418, -0.4291229248046875, 1.396044373512268, -0.1039709746837616, -0.5388273596763611, -0.6657439470291138, -1.1676671504974365, -0.30041512846946716, 1.143047571182251, 0.8662771582603455, 0.09503738582134247, 0.1072937548160553, -0.7590823173522949, -0.7006293535232544, -0.5406942963600159, 0.8273695707321167, 0.9844505190849304, 0.45063191652297974, 0.5953219532966614, 1.516252875328064, -0.20931300520896912, -1.0946823358535767, 0.6435750722885132, 0.7960368394851685, 1.173382043838501, -0.36837178468704224, 0.04085517302155495, -0.021310674026608467, 1.6363074779510498, -0.5719156861305237, 0.2588622570037842, 0.9178465008735657, -0.1440734565258026, -0.8828989267349243, -2.6368818283081055, 0.48664695024490356, -0.9106544256210327, -0.9948564171791077, -0.7434132099151611, -1.7285593748092651, -1.9486134052276611, 2.703824043273926, 0.11952003091573715, 0.6602752804756165, 1.4282323122024536, -0.3422108292579651, -1.7951085567474365, -0.3739234507083893, 0.2495957612991333, -0.008069325238466263, -1.4539433717727661, -0.40836191177368164, 0.01003135833889246, 0.3271753787994385, -0.8146757483482361, 2.458862543106079, -0.011082914657890797, 0.3326471745967865, 0.45651406049728394, -1.1701014041900635, -0.5468390583992004, 1.1206804513931274, -0.22019067406654358, 0.9535393714904785], [-1.8931418657302856, 2.3658883571624756, 0.255565345287323, -0.9291702508926392, 0.5107485055923462, -0.8856282234191895, -0.6305657625198364, -1.3452590703964233, 1.2155160903930664, -0.6943910717964172, 1.8493602275848389, 0.24525316059589386, -1.4952715635299683, 0.27910327911376953, 0.1890597641468048, 0.4754360318183899, -0.44714388251304626, 1.0848196744918823, -2.177394151687622, 1.0793908834457397, -1.1069202423095703, 0.5245326161384583, 0.9971394538879395, 0.36916956305503845, 0.8396116495132446, 1.1198019981384277, 0.08531274646520615, 1.3423291444778442, -0.26734575629234314, -1.6726411581039429, -0.22050194442272186, -0.48302534222602844, -0.2646729350090027, 0.8153480887413025, 0.7920293807983398, 0.2904214859008789, 0.8138971924781799, -1.726391077041626, -0.34443777799606323, -0.8078091144561768, -0.38092878460884094, -0.6340775489807129, -0.3528190851211548, -1.2825757265090942, 1.661662220954895, -0.6781996488571167, 1.606400489807129, -2.310999631881714, 0.7012427449226379, -0.6058392524719238, -1.6101548671722412, -2.0549588203430176, 0.09758919477462769, 1.0018895864486694, -0.8639637231826782, -0.16685537993907928, 1.182823896408081, 0.5497864484786987, 1.899444818496704, 1.1325942277908325, 0.8254464268684387, 1.6261141300201416, -0.8812686204910278, 0.4775264263153076], [0.1755540370941162, 0.7750722765922546, -1.521523356437683, -0.1585526019334793, 0.8057013750076294, 0.9400935173034668, 1.4848030805587769, 0.724764347076416, -1.2150344848632812, -0.8533080220222473, 1.0630972385406494, -0.9716184139251709, -0.20399126410484314, -0.2789210081100464, 2.0906479358673096, -0.08834782987833023, -0.1752690225839615, 0.8556954860687256, 0.475493460893631, -0.16317196190357208, -1.7009944915771484, 0.7054464221000671, -0.04917769879102707, -1.40568208694458, -0.4383206367492676, 0.40546539425849915, -0.5047818422317505, -0.3612949848175049, 0.7209970355033875, 0.5209999680519104, 1.2119510173797607, 0.5774129033088684, -0.6520782113075256, 0.738959550857544, 0.47338414192199707, 0.6810041069984436, 1.3227688074111938, 0.440082311630249, -1.7560439109802246, -3.103882074356079, 0.8286175727844238, -1.217042088508606, -0.28666388988494873, -1.1690459251403809, 0.26544272899627686, -0.5819820165634155, -1.2113851308822632, -0.9659404754638672, -0.8735065460205078, -0.5165386199951172, -0.7873759269714355, -1.4706146717071533, 1.5713484287261963, 0.7728807926177979, -0.3497752845287323, -0.11655816435813904, 0.045903854072093964, 0.4047507345676422, -0.3533801734447479, -0.3579944372177124, 1.7390201091766357, 0.13173289597034454, -0.9266892075538635, -2.5310797691345215], [-1.4029786586761475, 0.3727450966835022, -1.947189211845398, -1.4238018989562988, 0.02950216457247734, 1.0266566276550293, 0.5192927718162537, -1.4746801853179932, 0.2445410043001175, -0.8040867447853088, -0.6319654583930969, -0.07309556007385254, -0.6798950433731079, 0.043070655316114426, -0.4583589732646942, -0.4552910625934601, 0.5444380640983582, 1.5568444728851318, -0.10287366062402725, 1.0781049728393555, -0.2776900827884674, -1.0105818510055542, 0.9659439325332642, -1.229972004890442, -1.0871691703796387, 0.1338014006614685, -0.5895017981529236, 1.5000932216644287, 0.05622699111700058, -0.6929482817649841, 1.0547419786453247, -0.49216488003730774, -1.316144347190857, -1.4970756769180298, -1.081715703010559, 0.7424526810646057, -1.6033594608306885, -1.2115648984909058, 2.139995574951172, 0.4782327115535736, 0.6424487829208374, 1.6015865802764893, -0.7327314615249634, -0.46995577216148376, 0.06202259287238121, -0.16275259852409363, 0.012357239611446857, 0.16005752980709076, -1.1088467836380005, 0.4678569436073303, 0.7082147598266602, 1.273078203201294, -0.992851734161377, -1.4451971054077148, -0.808112621307373, 0.5548660755157471, 0.7362563014030457, 0.3656643033027649, -1.5702708959579468, 0.4266173541545868, 1.3704149723052979, 0.838690996170044, -0.35853514075279236, -1.3781334161758423], [0.33998674154281616, -0.8302087783813477, 0.06448850780725479, 1.0814390182495117, -2.1062498092651367, -1.2286171913146973, 0.1779032051563263, 0.9884976744651794, -0.22656464576721191, 0.9156922101974487, -1.1504621505737305, -0.7695266008377075, -0.4909123480319977, -1.3221454620361328, 0.958307147026062, -0.9710040092468262, -0.8251721262931824, 0.08104655146598816, 1.7282686233520508, 0.8507773876190186, -1.2502472400665283, -1.0072178840637207, -0.419393926858902, 0.8757919669151306, 0.5678427219390869, -0.4221939146518707, -1.4424571990966797, -0.8167896866798401, -0.6219717860221863, -1.9985580444335938, -0.05172289162874222, -2.5539028644561768, 1.4653362035751343, 0.5892771482467651, -0.26752522587776184, -0.15548525750637054, 0.899720311164856, -1.1608468294143677, 1.4598323106765747, 0.6251131296157837, -0.26578813791275024, 0.022857822477817535, 1.7915621995925903, 0.30851686000823975, -1.3535510301589966, 1.1749929189682007, -0.13971255719661713, 0.4116430878639221, -0.3778281509876251, 1.3840534687042236, -0.6470963954925537, 0.8671424388885498, 1.194102168083191, -0.21510730683803558, 1.5798918008804321, -1.120182991027832, 0.7822919487953186, 0.11227227002382278, -0.6094042658805847, -0.13428373634815216, -1.7462849617004395, -0.3423738181591034, 0.4284205138683319, 0.592488706111908], [0.7799844741821289, -0.18105565011501312, 0.7875969409942627, 0.875484049320221, 0.2833060920238495, 0.4099891483783722, 0.49059954285621643, 0.40728721022605896, -1.2158868312835693, 1.870728611946106, 0.894720733165741, 0.22051024436950684, 1.5610370635986328, 1.3612191677093506, 0.6101936101913452, -0.1681818813085556, 0.06471172720193863, 0.23479680716991425, -1.0888501405715942, 1.4974697828292847, -1.9026060104370117, -1.603530764579773, -0.9552572965621948, -0.7019413709640503, -1.5515071153640747, -0.8947374820709229, 0.034818634390830994, -0.5384913682937622, 0.7865421772003174, 0.6701551675796509, -0.7939352989196777, -1.0490225553512573, 0.8317994475364685, -1.0011539459228516, -0.4892081320285797, -0.3128170669078827, 1.7507051229476929, -0.7082421183586121, -0.5004776120185852, -1.7935255765914917, 0.16713359951972961, -0.559884250164032, -1.2782665491104126, 1.0281399488449097, -0.019235074520111084, -0.5997828841209412, -0.9357885718345642, 0.7687830328941345, -0.19098302721977234, -1.659934639930725, 0.98537278175354, 1.157120704650879, 1.0243746042251587, 0.23323549330234528, 1.7390148639678955, 0.4126490354537964, 0.1488185077905655, 1.3298499584197998, 2.3966586589813232, 0.3865134119987488, 1.340736746788025, -1.4786155223846436, 0.15048031508922577, -0.2691308557987213], [0.1001582145690918, -2.1313154697418213, -1.049538254737854, -0.1588364839553833, -1.404685139656067, -0.2568175792694092, 0.6924452185630798, -0.9836001396179199, 2.1342806816101074, -0.24716219305992126, 0.5260322093963623, -0.6329491138458252, 0.5987270474433899, -1.376695156097412, 0.45506012439727783, -0.7909343838691711, 0.7027227282524109, -0.8143100142478943, -1.5597305297851562, 1.637303113937378, -1.6400097608566284, -1.7725504636764526, -0.4114856719970703, 0.6094897985458374, -2.0465331077575684, -0.8674558401107788, -0.4854968190193176, -2.5130269527435303, -1.3920669555664062, 1.4003571271896362, 0.8428020477294922, 0.6310214400291443, 0.213986337184906, 0.12367217242717743, 0.7736032009124756, 0.46763697266578674, -1.1380741596221924, -0.046076469123363495, -2.2930262088775635, -0.6070111393928528, -0.36959803104400635, 0.0705946758389473, -0.8537777066230774, 1.3786941766738892, -0.3177413046360016, -0.7522397637367249, -0.7206579446792603, 0.8730534911155701, -0.15689094364643097, -1.8555959463119507, -0.6690759658813477, -0.3928087651729584, -1.389906883239746, 0.3068062961101532, 0.5420651435852051, -0.4081636071205139, -0.45335617661476135, 0.440478652715683, 2.5053391456604004, -0.828583836555481, 0.6170767545700073, -1.8270046710968018, -0.6886968016624451, 0.8961442708969116]], "dtype": "float32"}], "e": {"__nd__": [[-6.6081929206848145, -15.774353981018066, 6.672307968139648, -1.2414082288742065, -5.419824600219727, 12.493639945983887, -8.468232154846191, 4.216192245483398, -12.556058883666992, 2.4963247776031494, 0.36233532428741455, -8.599201202392578, 13.204922676086426, 13.217891693115234, 2.413471221923828, -0.018676500767469406, -4.4687652587890625, -0.7449946999549866, 5.110889434814453, -1.0624538660049438, -2.243572473526001, 17.930706024169922, -12.645214080810547, -3.7620291709899902, 4.494787693023682, 3.5699338912963867, -9.589933395385742, 8.84285831451416, -5.674189567565918, -9.400330543518066, 0.19413302838802338, -11.258313179016113, 10.293904304504395, 5.088912487030029, -12.918182373046875, -0.44906291365623474, -0.4338698089122772, -8.033652305603027, -9.38233470916748, 4.320718765258789, 8.362000465393066, 10.061753273010254, 9.996707916259766, -1.0679353475570679, 4.2294087409973145, 12.738863945007324, -6.2989630699157715, 4.021118640899658, -8.900641441345215, -6.373748779296875, -1.6147922277450562, -0.6273393034934998, -3.574958562850952, -4.739395618438721, -5.323029041290283, -7.559876441955566, 5.097222805023193, -5.438361167907715, 5.045586585998535, 6.253875255584717, 7.767590045928955, -10.368391990661621, 5.510924339294434, -2.2589619159698486], [1.9532967805862427, -2.978972911834717, 1.7077405452728271, -0.9181700944900513, -4.50564432144165, 9.68259048461914, 1.3769229650497437, 3.767472982406616, 4.351657867431641, 5.278212070465088, -16.535009384155273, -9.874402046203613, 3.1306538581848145, -9.84304428100586, -2.273242950439453, -9.339848518371582, -5.7961530685424805, -3.147839069366455, 6.921745300292969, 3.5631821155548096, -7.1537275314331055, -5.781474590301514, -5.657675266265869, -6.619568824768066, 9.518932342529297, 14.571680068969727, -14.695327758789062, 0.00687933387234807, -8.00753402709961, -4.318478584289551, -4.183664798736572, -12.475624084472656, 8.606588363647461, 1.5145819187164307, -10.905614852905273, -6.556562423706055, -6.876113414764404, -9.924914360046387, 10.710515022277832, -0.2834916412830353, 4.5001373291015625, 5.305915355682373, 4.612786769866943, 0.17881503701210022, -8.288909912109375, 5.8640875816345215, 6.576364517211914, -8.69803237915039, -0.4683901369571686, -10.411812782287598, 5.6614813804626465, 8.18833065032959, 7.016441345214844, 2.715207576751709, -4.3360419273376465, -6.570775985717773, 0.9459922313690186, -8.528076171875, -7.8001389503479, 0.19953344762325287, -11.904638290405273, -1.140419602394104, -2.686765193939209, -1.19355309009552], [-0.6026161313056946, -3.2802071571350098, -3.0940141677856445, -3.716444969177246, -1.252648949623108, -4.870270252227783, -13.137399673461914, -6.560913562774658, 4.9178242683410645, -6.544637680053711, -18.253555297851562, 2.0811569690704346, 6.088024616241455, -12.454204559326172, -10.382851600646973, -1.4878876209259033, 4.86414098739624, -13.122261047363281, 10.444414138793945, -13.80495548248291, 9.829865455627441, -0.472640722990036, -8.148126602172852, 7.693962097167969, 10.794452667236328, 0.5572222471237183, 12.64150619506836, -10.440316200256348, 5.660435676574707, -3.3633198738098145, 11.499465942382812, 5.2369513511657715, -2.5019428730010986, 2.9332995414733887, 1.4021644592285156, -6.8028788566589355, -6.359638690948486, -4.036210536956787, 7.499011993408203, -1.561753273010254, -6.729552268981934, 15.78071403503418, 7.908583641052246, 2.451843023300171, -12.980972290039062, -2.6707046031951904, 8.549882888793945, -17.490436553955078, 7.079408168792725, 4.7317705154418945, 9.579471588134766, 0.4489321708679199, 4.956711292266846, -2.1183221340179443, -5.45449686050415, 3.4172332286834717, -1.362134337425232, -8.261701583862305, -3.9877400398254395, 1.6178503036499023, -21.079557418823242, 3.035086154937744, 1.3483104705810547, 7.578528881072998], [-1.807435154914856, 0.24433046579360962, -12.067410469055176, -3.558431386947632, 3.9878408908843994, -0.29698461294174194, -2.4690582752227783, 0.17503595352172852, -0.3815138339996338, -11.327159881591797, -2.4647865295410156, 0.7114114165306091, -0.3455672860145569, -13.797733306884766, -2.0310492515563965, -5.827763557434082, 16.811304092407227, 0.594913899898529, -1.9465148448944092, -6.954135417938232, -6.215327739715576, -1.7731717824935913, -3.5441460609436035, -3.4443728923797607, 0.33221369981765747, 0.6822748780250549, -2.6799676418304443, 0.43503856658935547, 3.369976282119751, 8.622414588928223, -1.9127726554870605, 10.413768768310547, -0.7208850979804993, -2.0597951412200928, 9.38139533996582, 11.238801002502441, -10.81479549407959, -14.215693473815918, 6.9358062744140625, 13.197566032409668, -11.702274322509766, -7.594632148742676, 0.1812705397605896, -24.241832733154297, -0.385979026556015, 13.02129077911377, -0.09806516766548157, -8.253211975097656, -13.392245292663574, 6.487545967102051, -0.1154194176197052, 2.0290677547454834, -9.023789405822754, 9.181928634643555, 2.7394824028015137, -8.93671703338623, -12.034784317016602, 9.704548835754395, -8.138766288757324, -6.105807304382324, -9.20930290222168, 10.887626647949219, 7.907701015472412, 4.640571117401123], [-8.415034294128418, 2.7608561515808105, 8.947531700134277, -8.459712028503418, 17.150278091430664, 1.2702652215957642, -8.814769744873047, -23.612428665161133, -4.254297733306885, -3.998586416244507, 2.199965715408325, -6.321186065673828, 6.971047878265381, 5.922677516937256, 0.23106861114501953, -1.9588991403579712, 7.824774265289307, -13.501816749572754, -2.5801198482513428, -2.9353978633880615, 1.985898733139038, 0.053889527916908264, -11.619696617126465, 3.2123422622680664, 8.832544326782227, 3.543395519256592, 1.4084371328353882, 4.780265808105469, 3.700258731842041, -10.95269775390625, 18.81826400756836, 0.7399692535400391, -3.225731611251831, 4.47166109085083, -10.40770435333252, -5.213481426239014, -15.472173690795898, -2.9217681884765625, -6.069408893585205, -14.397195816040039, 0.8863353729248047, -1.554910659790039, -4.19633150100708, -7.594565391540527, -1.7834044694900513, -8.742757797241211, 11.289306640625, -9.885936737060547, 4.434395790100098, -3.641695261001587, -3.933128833770752, -10.834372520446777, 9.384071350097656, -3.328665256500244, -15.47326374053955, 7.325584888458252, 20.083911895751953, 0.09078548103570938, 2.5880930423736572, 5.683058738708496, 0.45743823051452637, 0.9793941974639893, -2.2590019702911377, -7.449522495269775], [-1.2656559944152832, 5.001166343688965, -5.430626392364502, 0.1454422026872635, 4.058949947357178, 10.697999000549316, -7.344357967376709, 0.2607669532299042, -5.7541728019714355, 2.2185606956481934, 2.562128782272339, 3.676344871520996, 8.227785110473633, -8.685253143310547, -5.4781928062438965, 8.286355018615723, -2.138479471206665, 3.278474807739258, 5.3614349365234375, -18.125614166259766, 2.298003673553467, 8.670527458190918, 3.2516531944274902, -18.800151824951172, -2.0018205642700195, 7.067320346832275, 1.6863254308700562, 12.276676177978516, -0.17959076166152954, 7.048607349395752, 8.033697128295898, -2.9386117458343506, 0.4362160861492157, -0.6844676733016968, 12.243528366088867, -3.287964344024658, 2.418194055557251, -3.833817958831787, 8.268158912658691, -5.294896125793457, 11.937212944030762, -10.504600524902344, -5.708859920501709, -1.6125472784042358, -1.7718816995620728, 10.937273025512695, 11.67125415802002, -19.29629135131836, 1.7796117067337036, 17.191862106323242, 0.38730067014694214, 10.63921070098877, 3.439223051071167, -3.0858633518218994, -2.408923387527466, -13.12605094909668, -7.654749870300293, -1.9729207754135132, -13.205878257751465, -0.3820111155509949, -10.938194274902344, 13.74994945526123, 7.19846248626709, -2.777401924133301], [-2.8104770183563232, -3.8617758750915527, -11.735574722290039, 1.42349374294281, -10.859550476074219, 15.7931489944458, 2.3709733486175537, 4.354249954223633, 4.0659499168396, 10.622621536254883, 1.7500765323638916, -4.91003942489624, -3.9398508071899414, 5.128578186035156, 5.332500457763672, -2.504265785217285, 1.4925206899642944, 12.178772926330566, 5.734025478363037, 15.425252914428711, -12.65677261352539, -3.6199288368225098, -4.821086406707764, 4.718594551086426, -5.630707740783691, 3.6740026473999023, -3.343986749649048, -1.5903165340423584, -15.046016693115234, -8.41458797454834, -11.776741981506348, -5.139252662658691, 2.3330295085906982, -8.724644660949707, -12.11989688873291, -3.6137845516204834, -14.55274772644043, -7.501770496368408, 11.072864532470703, 5.402134418487549, 11.53770637512207, 5.44770622253418, -0.44144755601882935, 3.085916757583618, 4.509232521057129, -12.837786674499512, -11.944267272949219, -8.296501159667969, 1.738142728805542, -5.668228626251221, 5.645080089569092, 5.634740352630615, 2.1528937816619873, -3.3866608142852783, 6.8884196281433105, 8.420391082763672, 1.3039379119873047, -13.789596557617188, 5.2656636238098145, -4.15444803237915, 16.342296600341797, -7.818825721740723, -8.537035942077637, -8.888816833496094], [-0.4662783741950989, -12.6308012008667, 14.281943321228027, 4.981357574462891, -12.385856628417969, -14.082104682922363, -7.234999179840088, -9.6083984375, 8.239943504333496, -6.096618175506592, -30.994396209716797, -17.897138595581055, 15.887595176696777, -10.326482772827148, -14.693256378173828, -20.634733200073242, 14.404821395874023, -5.18074369430542, -0.5022103190422058, 4.316791534423828, -5.304664134979248, 5.552653789520264, 9.540414810180664, 9.608927726745605, 13.986916542053223, 2.2787532806396484, 4.507645606994629, -12.689599990844727, -5.074233055114746, 10.552772521972656, -0.8995128870010376, 0.5641454458236694, 2.8016297817230225, 6.0568366050720215, -7.707996368408203, -11.186962127685547, -12.27924633026123, -20.584495544433594, -22.0379695892334, 10.383651733398438, 0.15543681383132935, -6.435232162475586, -3.8914499282836914, 13.334538459777832, -9.121428489685059, 19.508562088012695, -11.31711196899414, 2.0342655181884766, -10.304006576538086, -2.677931547164917, 10.91598129272461, 11.894204139709473, -6.824036121368408, -1.454728126525879, 1.82228684425354, -4.315934181213379, -10.821294784545898, -2.5663442611694336, 15.798727035522461, -1.646091103553772, -10.698742866516113, 1.5821963548660278, 10.09670639038086, 7.101259708404541], [8.784605979919434, -2.4128410816192627, -11.606216430664062, -0.2734885513782501, -1.6701334714889526, 11.837494850158691, -7.051860809326172, 0.548603892326355, 7.619633674621582, -16.304889678955078, -12.595662117004395, 5.471130847930908, -10.321431159973145, -12.443248748779297, 11.785365104675293, 1.7189077138900757, 2.545412302017212, -4.221770286560059, 14.155855178833008, -19.61084747314453, -1.8281797170639038, 3.265256404876709, 9.386734962463379, -1.4444444179534912, 8.68474292755127, 6.651739120483398, 3.90220308303833, 2.8518104553222656, -11.811610221862793, -1.902435541152954, -6.20044469833374, 1.9192208051681519, -7.159997940063477, 8.911667823791504, 11.35649585723877, -13.860604286193848, -0.6840602159500122, 6.985074996948242, 9.60389518737793, 1.818426489830017, 25.807893753051758, -9.649165153503418, 6.401787281036377, 6.943297386169434, 2.8652467727661133, -4.779004096984863, 2.3747401237487793, -26.08402442932129, -4.430834770202637, -2.4010071754455566, -4.452221393585205, -4.764742374420166, 14.790995597839355, 0.8494918942451477, -12.054557800292969, -10.571160316467285, -11.073670387268066, -16.94278335571289, -14.89862060546875, 1.6598635911941528, -0.7783511281013489, 1.321828007698059, -7.819159984588623, -1.7752110958099365], [-3.2353971004486084, 5.814084053039551, -3.4179139137268066, 2.6563875675201416, 2.722947120666504, -3.162522077560425, 6.012109279632568, -7.556003093719482, -2.8564462661743164, 17.878711700439453, -6.188093662261963, -6.529974937438965, -6.42836856842041, -8.873993873596191, -6.885115623474121, -3.553257465362549, -4.936265468597412, 2.5680882930755615, -6.329224586486816, 9.67241096496582, 2.222799777984619, -8.940398216247559, -8.09265422821045, -9.71651840209961, 3.563960075378418, 6.320464134216309, -2.576509714126587, -6.146032333374023, 7.31405782699585, 9.036008834838867, 9.415063858032227, 3.9622480869293213, 4.02183723449707, 1.8377699851989746, -10.05309772491455, 1.582851529121399, -6.054876804351807, -5.900925159454346, 4.320417404174805, -3.000373125076294, -0.047838371247053146, 4.741830825805664, 10.726572036743164, -4.323935031890869, -3.2061610221862793, -1.4913618564605713, 3.8161637783050537, 0.3227253556251526, -2.2800333499908447, -1.735917329788208, -1.8580279350280762, 12.366875648498535, -22.32124137878418, 6.0926899909973145, 6.048011302947998, 2.1028811931610107, 2.620605945587158, -10.178829193115234, -15.047599792480469, -8.716571807861328, 4.491319179534912, 9.705060958862305, 2.156233072280884, 9.832551956176758], [-7.13476037979126, -9.723427772521973, -0.6348047256469727, 1.3405590057373047, -2.0814309120178223, -22.694019317626953, -15.291427612304688, -4.24910831451416, 8.029223442077637, -3.1471312046051025, 4.588437557220459, -3.7400014400482178, -2.471879482269287, 0.6304335594177246, 2.5417399406433105, -0.028850408270955086, -6.626981258392334, -7.840651512145996, -12.858280181884766, -1.0920008420944214, 9.771430969238281, -8.388263702392578, 15.60111141204834, 12.623068809509277, 14.001340866088867, -11.230960845947266, -9.32950496673584, 2.1113662719726562, -3.3962459564208984, 6.795465469360352, 23.140424728393555, -2.8684628009796143, -7.072766304016113, -3.5089149475097656, 5.741394996643066, -9.411009788513184, 4.593887805938721, 6.807394981384277, -5.661071300506592, 11.131393432617188, -16.90009117126465, 4.047863960266113, -5.227511882781982, 1.54393470287323, -10.995862007141113, 1.4913743734359741, 10.205540657043457, 0.519740879535675, 13.178674697875977, -1.6528106927871704, -4.578831195831299, 4.56425142288208, -7.7397379875183105, 12.72615909576416, -1.6536859273910522, 6.064999580383301, 4.839587211608887, 4.041406631469727, 3.8395111560821533, 2.0375726222991943, -7.472988128662109, 16.505023956298828, -13.577534675598145, -8.728964805603027], [-2.20931339263916, -17.209640502929688, 2.569054126739502, 11.131390571594238, -14.647360801696777, -4.3945112228393555, 0.2685719430446625, 12.832151412963867, -16.941953659057617, 5.333969593048096, -14.551435470581055, 6.10893440246582, 12.552820205688477, 3.5012314319610596, 11.68687915802002, -1.8837854862213135, 17.580320358276367, -12.840165138244629, 3.2423696517944336, -12.686844825744629, 13.075833320617676, -4.194123268127441, 0.8532748222351074, 12.22028923034668, -2.0481228828430176, -12.513854026794434, -12.243673324584961, -20.991817474365234, -10.561622619628906, -9.706894874572754, 6.316181182861328, -15.564873695373535, 3.8857805728912354, -4.512081146240234, -13.651695251464844, -9.221656799316406, 15.411983489990234, -9.565046310424805, 2.6077215671539307, 2.855647563934326, -7.309700965881348, 16.1732120513916, -11.064468383789062, 1.727245569229126, 8.95916748046875, 6.5304975509643555, -15.285245895385742, 10.625168800354004, 1.400614857673645, 8.008257865905762, -0.1387149691581726, 1.34403657913208, -19.171648025512695, -15.300108909606934, -3.0240299701690674, -1.2448943853378296, 4.170416355133057, 10.263372421264648, 8.040148735046387, -9.951008796691895, 3.5149378776550293, 9.094416618347168, 5.5973100662231445, 21.6339111328125], [-4.037399768829346, -0.7514616847038269, 3.8077704906463623, -0.10720406472682953, 9.705699920654297, 7.532862186431885, -20.08101463317871, -2.8605971336364746, 1.8450689315795898, 11.45977783203125, -15.077568054199219, 5.678996562957764, -3.399932384490967, 5.645664691925049, -0.45388707518577576, -13.660346031188965, -12.096643447875977, 1.5931237936019897, 7.362821102142334, -3.3455259799957275, 0.5848317742347717, -2.308323621749878, 1.299501895904541, -0.20270970463752747, -3.825986623764038, 0.5683621764183044, 14.882206916809082, 1.7579580545425415, 3.1792187690734863, -3.9152257442474365, -12.087129592895508, -4.675382614135742, -7.239402770996094, 3.634355306625366, -10.359135627746582, 4.154768466949463, 1.3334264755249023, 11.875147819519043, -8.882010459899902, -3.8532819747924805, 5.646994113922119, 15.809800148010254, 11.611851692199707, 10.060454368591309, 1.5595964193344116, -6.606394290924072, -17.515125274658203, -9.204330444335938, 4.126291275024414, 1.8620542287826538, 4.573122978210449, 6.007152080535889, 1.5909438133239746, 7.219949245452881, -3.1786510944366455, 2.8344919681549072, 5.053194046020508, -7.195126533508301, -4.740174770355225, 4.55580997467041, -2.47225284576416, 0.7226515412330627, 11.834969520568848, -2.340158462524414], [-9.555974960327148, -3.3873538970947266, 13.406224250793457, 8.084541320800781, -14.427021980285645, 3.9081108570098877, -1.0583447217941284, 14.797149658203125, -1.5226719379425049, -3.2927727699279785, 6.869830131530762, -0.2471502125263214, -2.4829862117767334, 5.000020980834961, -0.45586052536964417, 6.910083770751953, 3.3392670154571533, -1.6533488035202026, 8.558485984802246, 7.7120747566223145, 5.590271949768066, 6.750055313110352, 2.717700958251953, -2.590613603591919, 10.038163185119629, -13.761860847473145, 0.47000306844711304, -8.283899307250977, -6.1140360832214355, -0.9762611389160156, -12.028196334838867, -14.163922309875488, -0.19508759677410126, 2.022926092147827, -7.829347610473633, 4.162950038909912, 0.31046727299690247, -7.431166172027588, -0.8904982805252075, -12.1734619140625, 7.592151641845703, 23.265243530273438, 0.9194842576980591, 1.3343145847320557, 13.960530281066895, 0.39743533730506897, -6.4926886558532715, -1.1282706260681152, -8.680801391601562, -0.2608001232147217, -4.6824727058410645, -19.750539779663086, 4.855792999267578, -19.718017578125, -19.90389633178711, -3.744893789291382, -5.534457683563232, 7.021049976348877, 4.945991516113281, 13.082935333251953, 13.724968910217285, -0.22342462837696075, 5.3021559715271, -5.089233875274658], [-0.6426872611045837, -9.678948402404785, 0.7223962545394897, 6.545293807983398, -3.234008550643921, 4.118420600891113, -1.6653000116348267, 5.437471389770508, -9.125972747802734, 8.51506519317627, 4.250164031982422, -7.339264392852783, 12.725696563720703, -0.7658562064170837, 7.846679210662842, -4.2914838790893555, -4.433906555175781, -3.652113199234009, -8.355742454528809, -5.919896602630615, -2.777062177658081, -10.635210037231445, -11.615287780761719, -7.901611804962158, -8.534998893737793, -11.150787353515625, -16.127147674560547, 8.51722526550293, 0.8768198490142822, 5.569783687591553, -1.5432748794555664, -10.016961097717285, 18.821544647216797, 8.536412239074707, -6.604971885681152, 3.524613380432129, 19.006988525390625, 0.663734495639801, 6.580749988555908, 1.3537348508834839, 1.953620195388794, 0.1777455359697342, -0.5650911927223206, 10.064247131347656, 4.541390419006348, -1.8519089221954346, -1.8432670831680298, 10.131775856018066, -5.232432842254639, -2.5487728118896484, -8.065383911132812, -3.4881656169891357, 1.5196824073791504, -7.706686496734619, 11.490684509277344, -4.353500843048096, 12.261029243469238, -2.738809823989868, 9.177191734313965, -3.607645273208618, 2.1548426151275635, -13.837138175964355, 0.20700037479400635, 11.76533031463623], [-11.526205062866211, -0.2408631294965744, -2.716283082962036, -7.180474758148193, -4.334596157073975, 0.09962312132120132, -2.368541955947876, -7.1125569343566895, 1.389915943145752, 0.4792190194129944, 1.2740800380706787, 15.144688606262207, 6.868436336517334, -2.0362319946289062, 13.082198143005371, -5.839161396026611, -5.406206130981445, 2.755509853363037, -11.877458572387695, 14.799747467041016, -5.9016218185424805, -7.689638614654541, -10.305399894714355, 12.311561584472656, -10.16592788696289, -5.146241188049316, -3.672781467437744, 8.606935501098633, 3.989760637283325, 4.728104591369629, 9.529083251953125, -14.975733757019043, 6.357579231262207, 8.77855110168457, -5.81768274307251, 2.1610820293426514, 9.93659496307373, 2.3905904293060303, -3.895364999771118, 1.693568229675293, -20.107513427734375, 7.343329906463623, -21.661514282226562, 2.984543561935425, 5.262791156768799, 0.9330036044120789, 3.853623151779175, -7.594402313232422, 10.828906059265137, 20.870595932006836, 5.4991068840026855, -17.16741371154785, 1.9221891164779663, -15.76587963104248, 3.5022926330566406, 6.013524055480957, 1.8552699089050293, 7.150700092315674, 28.544198989868164, 4.449553489685059, 3.72856068611145, 6.7914204597473145, -11.483704566955566, -5.417975902557373], [0.4111352562904358, 9.76936149597168, 10.771327018737793, 8.421883583068848, 5.110875129699707, 5.119283199310303, -6.894649982452393, 3.2997989654541016, 4.3166680335998535, 3.192512035369873, -10.874383926391602, -10.258604049682617, 5.779881000518799, 4.101603031158447, 2.01265811920166, 13.91080379486084, 2.852328062057495, 6.716851711273193, -4.6095356941223145, 1.1389931440353394, 3.0469553470611572, -1.226274013519287, 2.904684543609619, -5.850183010101318, 16.358205795288086, -8.4824857711792, -10.564130783081055, -0.16113430261611938, -7.5517096519470215, 0.8037446141242981, -3.2560436725616455, 9.359859466552734, 0.932070791721344, 8.471976280212402, -4.326653003692627, -7.225427627563477, 14.505044937133789, -6.674981117248535, -5.068491458892822, -8.373907089233398, 16.135927200317383, 9.541010856628418, -6.644867420196533, 3.654158592224121, -2.724710702896118, -5.758242607116699, -2.535503625869751, 6.354176998138428, 2.4006106853485107, -15.971632957458496, -4.392501354217529, 5.58458948135376, -6.405300617218018, -10.534455299377441, -7.264875888824463, -3.2560031414031982, 20.35087776184082, -6.760705471038818, -8.842572212219238, -2.386277437210083, 10.008075714111328, 9.614306449890137, -0.6655686497688293, 4.629745006561279], [4.07609748840332, -4.195616722106934, -3.244520902633667, 2.895479679107666, 4.741512775421143, -3.756989002227783, -13.474099159240723, 3.8052754402160645, -10.862977981567383, -1.3227773904800415, 1.5039618015289307, 9.801486015319824, 5.261697292327881, -1.0052871704101562, 8.533038139343262, -3.586350440979004, 2.8952717781066895, -5.367095470428467, -0.8759130239486694, -16.627593994140625, 14.965147018432617, -1.45625638961792, -9.353346824645996, 3.910705327987671, -0.5985145568847656, 2.184070587158203, 10.749503135681152, 7.252143383026123, -6.540933609008789, 11.767141342163086, 8.214588165283203, 10.203621864318848, 3.787282705307007, 12.927628517150879, 6.067760467529297, -4.904445171356201, -0.26542094349861145, 8.178170204162598, 16.96658706665039, -5.476524829864502, 3.443437099456787, -6.463526248931885, -0.7494091987609863, -10.056488990783691, -6.3540425300598145, -8.020127296447754, 5.729795932769775, -16.62631607055664, -5.847558498382568, -0.006830225698649883, 0.6264607906341553, -8.324092864990234, 7.493523120880127, -0.3899449408054352, 2.2440130710601807, -5.978804111480713, 3.1785144805908203, -2.104395627975464, 7.926857948303223, -2.271251678466797, -11.739362716674805, 7.281488418579102, -12.953458786010742, -2.2101199626922607], [-4.593984603881836, 17.092344284057617, 3.287043809890747, 4.50916862487793, 5.367391586303711, -0.6359068751335144, 11.235910415649414, -7.057192802429199, -1.9648716449737549, 11.945219039916992, -0.6457326412200928, -1.6043779850006104, -0.7085597515106201, 7.600607872009277, 12.10350227355957, -4.107693672180176, -0.7230908870697021, 11.46961784362793, -1.0644625425338745, 7.433822154998779, -11.72496509552002, -10.00767993927002, 3.108288049697876, 2.0608901977539062, -9.590011596679688, 10.155989646911621, -6.203550338745117, -5.88046932220459, 2.64111328125, -2.768077850341797, -2.2313992977142334, -7.471611022949219, 3.906754732131958, 4.127871990203857, -9.173017501831055, 14.325132369995117, 3.4385881423950195, 6.846352577209473, -6.528960227966309, -4.4208760261535645, 1.1694282293319702, 0.4982297122478485, 6.535027980804443, 7.655694484710693, 9.974417686462402, 1.369045376777649, -23.04868507385254, -2.077711582183838, -7.58981466293335, -5.541725158691406, -3.5272696018218994, -2.4934628009796143, -4.0750041007995605, 0.8229827284812927, 8.751823425292969, 7.486586093902588, 10.561820030212402, -0.3462940752506256, -10.709348678588867, -6.596594333648682, 1.1910333633422852, 11.731973648071289, -0.11461645364761353, 2.7543702125549316], [8.396708488464355, 3.7386457920074463, 5.114927768707275, 2.3560962677001953, 1.0669950246810913, 4.114501953125, 11.892526626586914, -0.5557567477226257, -6.142220973968506, 15.825777053833008, -8.278099060058594, 1.987538456916809, 6.413546562194824, -0.35568469762802124, 7.7371296882629395, 6.936328411102295, -3.1183536052703857, -2.7695810794830322, 0.0597226619720459, -3.3263163566589355, -1.0943695306777954, -5.940156936645508, 1.49880051612854, -2.1933226585388184, 3.882859945297241, 1.8614287376403809, 8.770798683166504, -12.019543647766113, -11.347821235656738, -5.828564643859863, -3.5696804523468018, 9.856893539428711, 4.006321430206299, 5.9537739753723145, -13.29000473022461, -2.951638698577881, 0.44702011346817017, -2.5691797733306885, -2.8359532356262207, -8.390047073364258, 12.818135261535645, 4.773952960968018, -6.871788501739502, 5.139157295227051, -4.352972030639648, 0.3492133319377899, -11.900964736938477, -1.211778163909912, -7.0960516929626465, -15.810014724731445, -3.1231796741485596, 0.2648119032382965, 6.539652347564697, 2.101317882537842, -2.8990800380706787, 1.4383729696273804, 4.9878249168396, -12.475071907043457, -2.6827175617218018, 2.1395864486694336, 3.3721694946289062, 1.5707038640975952, -5.838155746459961, 0.6042960286140442], [13.653059005737305, -6.1045379638671875, 12.587485313415527, 20.268890380859375, -21.41114616394043, 0.981609582901001, -2.797013521194458, 22.65873908996582, 5.8642659187316895, 4.303353786468506, 5.526543617248535, 4.217639923095703, -12.628951072692871, 1.6528956890106201, -3.4833881855010986, 14.349029541015625, -1.0699400901794434, 2.642423152923584, 0.5272730588912964, 13.515480041503906, 5.019139766693115, -11.5383939743042, -11.936929702758789, -17.270753860473633, 5.615803241729736, -18.219877243041992, 17.313838958740234, -26.860456466674805, -15.01330280303955, 19.720348358154297, -7.597692012786865, 14.030927658081055, -5.04058313369751, -6.880550861358643, 3.6775577068328857, -8.212488174438477, -6.280456066131592, 19.75684928894043, 13.007643699645996, -12.412969589233398, 19.4799747467041, 16.97700309753418, -14.84019660949707, 6.352086544036865, -4.859408855438232, -12.634024620056152, 0.09280359745025635, 22.984302520751953, 20.44256591796875, -5.522950172424316, 9.336089134216309, 1.8476496934890747, 4.638479232788086, -21.04620933532715, 4.987258434295654, 4.6718292236328125, 0.2081557810306549, -5.632584571838379, 3.2734248638153076, 5.210733413696289, -5.463713645935059, 2.0919089317321777, -12.289179801940918, 6.916560649871826], [-17.84812355041504, 7.6106791496276855, 2.138615846633911, 3.709296703338623, 10.93991470336914, 2.214219808578491, -12.09231948852539, -14.746828079223633, -3.5693018436431885, 1.750949740409851, -10.092938423156738, 8.917220115661621, -9.386528968811035, -0.9917873740196228, -3.044671058654785, -7.56097412109375, 1.8975491523742676, 10.015477180480957, -9.609273910522461, 0.22767899930477142, 2.964423894882202, -7.311337947845459, 5.795130252838135, 1.5911263227462769, 4.982165813446045, 17.879228591918945, -5.227860450744629, 5.567941665649414, -6.700458526611328, 8.132904052734375, -13.175432205200195, -13.295413970947266, -7.725584030151367, -1.543305516242981, 2.9048867225646973, 10.622501373291016, -6.069124698638916, -7.618325710296631, 18.727088928222656, 11.484445571899414, -7.247641563415527, -6.660394668579102, -7.808046340942383, -14.357699394226074, 3.3392131328582764, 6.7640910148620605, -2.692828416824341, -6.386751174926758, -3.399454116821289, 12.92229175567627, -2.418128252029419, -0.7136074304580688, -4.464746475219727, 0.7071142792701721, -8.810641288757324, -12.577999114990234, 2.1254751682281494, 10.838456153869629, -6.074443340301514, 4.2843122482299805, -1.0983271598815918, 22.82134246826172, -0.685605525970459, 0.43956831097602844], [11.275325775146484, -2.1542322635650635, 7.4590163230896, 8.308518409729004, -9.525616645812988, -1.118545413017273, 16.698572158813477, 15.876798629760742, -14.064345359802246, 1.9532921314239502, 4.81032133102417, 8.840288162231445, 11.610824584960938, 19.6754093170166, 8.218634605407715, 1.2282378673553467, -7.6697845458984375, -7.955951690673828, 6.109888076782227, 5.950364112854004, -5.35136079788208, -1.5791302919387817, -0.7613746523857117, -3.7661688327789307, -2.6664841175079346, 0.9828267097473145, 12.865874290466309, -1.6917695999145508, 0.9650793671607971, -3.446169376373291, -2.6100101470947266, -8.584308624267578, 2.447354316711426, 10.36573314666748, -1.7478550672531128, -6.980567932128906, 9.458205223083496, 12.527369499206543, -5.583071708679199, -7.063657283782959, -1.4287657737731934, -7.26548433303833, -9.607796669006348, 7.093846797943115, 10.881284713745117, 2.143108367919922, -4.491677284240723, 2.144138813018799, 10.027013778686523, -9.181474685668945, 5.613907337188721, -9.734046936035156, -4.608531475067139, -10.884090423583984, 2.6500115394592285, 10.204337120056152, 3.3255066871643066, 4.812951564788818, 4.80088472366333, 5.919025897979736, 6.976146221160889, -18.056997299194336, 0.24899615347385406, 4.893643379211426], [3.606670618057251, 11.972710609436035, -2.625249147415161, -1.6255375146865845, 4.9325785636901855, 9.301251411437988, 8.778709411621094, 0.6056333780288696, -3.9111554622650146, -11.33707046508789, 0.09050893783569336, -1.8188128471374512, -5.637976169586182, -6.0186052322387695, 6.078475475311279, 0.9115362763404846, 10.443655967712402, 0.07174424082040787, -2.3979029655456543, -1.4881815910339355, 9.957639694213867, -7.5871429443359375, -9.476177215576172, 2.018493890762329, 10.248372077941895, -0.2725917398929596, 0.8923166394233704, -5.804939270019531, 3.6156113147735596, 6.936065673828125, 10.425378799438477, 19.701168060302734, 2.8591442108154297, 11.899493217468262, 4.063982963562012, 4.030911922454834, -12.62180233001709, 4.921291828155518, 5.85505485534668, -3.4082818031311035, -2.0237197875976562, 8.222922325134277, -11.586259841918945, -11.40979290008545, 2.2082509994506836, -3.7320451736450195, 3.607044219970703, 5.339035511016846, -12.816424369812012, -1.808806300163269, -15.065666198730469, -7.480466842651367, 7.330585479736328, -20.252910614013672, -8.838752746582031, -1.3837499618530273, 7.74515438079834, 5.827048301696777, -5.511776447296143, -2.042479991912842, 1.1420013904571533, 7.815650939941406, 8.502227783203125, -0.07987424731254578], [2.398893356323242, -7.813535213470459, 1.9626673460006714, 9.147909164428711, -14.9386625289917, -5.885817527770996, 2.0147109031677246, 10.396077156066895, -2.946992874145508, 2.8721442222595215, -11.909693717956543, -7.0773210525512695, -7.022037506103516, -6.697834014892578, -5.412917137145996, -2.2530369758605957, -4.320664882659912, -7.328085899353027, -0.5925260782241821, 5.863982677459717, 6.045875549316406, -14.148207664489746, 0.4622613489627838, -1.4392427206039429, 3.023087501525879, 3.4200868606567383, 7.656142234802246, -8.65333366394043, -10.587831497192383, -6.273073196411133, -1.7387547492980957, -3.3289973735809326, -5.333030700683594, 0.027244530618190765, 1.5479819774627686, -9.550355911254883, -9.56905746459961, -10.193949699401855, -6.439181804656982, 6.3946146965026855, -2.6802244186401367, 8.836788177490234, -1.2150911092758179, 8.551004409790039, -11.362467765808105, 5.342587947845459, 0.38553106784820557, 12.027812957763672, 0.8002801537513733, -2.0700433254241943, -8.770881652832031, -6.740852355957031, -2.5084846019744873, -7.002101421356201, 4.8185296058654785, 6.473811626434326, 7.604371547698975, -5.120145797729492, 1.9881101846694946, 0.2153434455394745, 3.6466681957244873, 1.6397905349731445, -18.710309982299805, -4.584655284881592], [-0.3658243417739868, -4.871863842010498, -9.372858047485352, -0.42004284262657166, -8.568632125854492, -2.377124309539795, -2.981733560562134, 5.707842826843262, 5.103493690490723, 4.936893939971924, -14.826495170593262, 1.1165621280670166, -6.535233497619629, 2.620593309402466, 3.1351680755615234, -0.7812734842300415, -2.299556255340576, 11.359040260314941, 1.884987711906433, 6.27126932144165, -6.888904094696045, -7.23051118850708, 13.936373710632324, 12.088560104370117, -4.543651103973389, 3.377387046813965, 4.7532877922058105, 7.3496317863464355, -0.48827382922172546, -8.798709869384766, -11.513099670410156, -2.8166697025299072, -3.0029361248016357, -0.6930356621742249, -6.193048477172852, -2.9558768272399902, 2.855193614959717, -2.674532175064087, 1.7695688009262085, 11.575974464416504, -7.945788383483887, 4.65747594833374, 2.3914127349853516, 3.987837553024292, -1.3186876773834229, 2.3620665073394775, -8.723663330078125, -7.394862651824951, 10.369926452636719, 7.942479610443115, 3.2087361812591553, 3.9921112060546875, -1.159943699836731, 5.753171443939209, 6.623322010040283, 9.869217872619629, 0.42387956380844116, -9.106245040893555, -1.5521564483642578, -5.068513870239258, -4.697307109832764, -1.9416577816009521, -8.116321563720703, 1.180970549583435], [-2.929399251937866, 3.161642074584961, -0.10078156739473343, -1.4465399980545044, -8.089845657348633, 4.189840316772461, 2.980868339538574, 4.591671943664551, 0.9504761099815369, 4.252647876739502, 8.577625274658203, 6.485228061676025, 1.7922323942184448, -5.145671844482422, 1.0451945066452026, -4.149612903594971, -4.6536688804626465, -7.82065486907959, -12.953429222106934, 22.7545108795166, -7.688253402709961, -4.711045265197754, -11.841903686523438, 3.562920570373535, -9.792474746704102, -4.352313041687012, -7.127070426940918, -6.5621418952941895, 2.2068612575531006, 0.253446102142334, -3.9520130157470703, -13.820225715637207, 4.98630952835083, 4.043521404266357, 3.5054235458374023, -1.2454811334609985, 21.753087997436523, 8.416936874389648, -5.181651592254639, 0.8525288701057434, -6.115675926208496, 3.806013345718384, -1.4230338335037231, 7.774296760559082, 8.710457801818848, -0.6085195541381836, 7.448907375335693, 11.427440643310547, -0.9265428781509399, -2.674333333969116, 1.8110179901123047, -3.8871748447418213, 3.964937448501587, -12.105345726013184, 5.530043125152588, -10.524347305297852, 2.908071994781494, 10.485230445861816, 21.622262954711914, 7.311674118041992, -10.346948623657227, -3.342956781387329, 7.628263473510742, 4.072750091552734], [-5.217493057250977, -6.691235065460205, -0.23928730189800262, 8.037751197814941, -1.266296148300171, -6.521772861480713, -0.08918585628271103, -14.257697105407715, -4.473484992980957, 20.82434844970703, 5.141477108001709, 9.399032592773438, 7.460900783538818, -14.092151641845703, -1.2687398195266724, 1.1391059160232544, 8.889975547790527, -8.01710033416748, -10.26853084564209, 22.59386444091797, -16.386947631835938, -15.74982738494873, -10.701777458190918, -5.608917713165283, -0.3570280075073242, -3.9883439540863037, 12.034225463867188, -19.025785446166992, -1.3663253784179688, 9.908625602722168, -2.1194231510162354, -2.3486788272857666, 2.071284770965576, 3.2811830043792725, 2.106832265853882, -0.4197840690612793, -8.62207317352295, 5.812429428100586, 11.25943374633789, -6.962513446807861, -6.44580602645874, 5.836126327514648, -3.875465154647827, 16.730873107910156, -6.750795841217041, -12.324681282043457, 2.881788492202759, 4.7035064697265625, 2.5089035034179688, 8.75930404663086, 2.445533275604248, -3.8375604152679443, 3.425330877304077, -16.508214950561523, 5.748615264892578, 5.346070289611816, 7.657819747924805, 3.3668160438537598, 14.080292701721191, 8.76678466796875, -9.441457748413086, -5.068662166595459, -14.650766372680664, -12.55184555053711], [0.9615015387535095, -2.7469375133514404, 7.546591281890869, -5.229750633239746, -4.324978351593018, 7.125514984130859, 0.822529137134552, -9.540544509887695, -5.089851379394531, -3.068082809448242, -2.0729753971099854, 21.634370803833008, 11.315838813781738, 3.2088465690612793, 13.671707153320312, 15.011235237121582, -1.152351975440979, -7.899019718170166, 9.115219116210938, -13.684118270874023, -0.36516624689102173, 7.384042263031006, 0.5344334840774536, -5.704392433166504, 4.234506130218506, -8.965994834899902, -2.7613682746887207, -2.1151795387268066, 8.014561653137207, -9.222611427307129, 6.828123569488525, -1.8864328861236572, -3.369598865509033, -15.11357307434082, -10.384317398071289, -12.970709800720215, 2.157665729522705, 3.5198862552642822, -1.952285885810852, 3.529144763946533, 1.2993853092193604, -12.525476455688477, -4.2679009437561035, -4.208276271820068, 0.06822036951780319, 1.5761226415634155, -6.484623908996582, -0.04710341989994049, 6.930922031402588, -0.9287686347961426, -6.572002410888672, -1.043529987335205, 10.77170181274414, 7.528842449188232, -11.372079849243164, 8.451085090637207, -7.390283107757568, -1.448441982269287, -3.633875608444214, -7.458880424499512, -3.82110595703125, 3.3503358364105225, 10.464750289916992, -10.721261978149414], [1.4144701957702637, 1.2750810384750366, 3.754030227661133, -12.84299373626709, -1.5562783479690552, -17.245105743408203, -9.043664932250977, -2.086656093597412, 7.51049280166626, -21.06519317626953, -0.05624601989984512, -19.995098114013672, -6.582204341888428, -9.924639701843262, -2.9832541942596436, -2.892265796661377, -12.35059928894043, -11.231362342834473, 1.4477806091308594, -15.694443702697754, -2.005915403366089, 15.92081356048584, -2.248915672302246, -6.1980719566345215, -2.5849504470825195, 10.281905174255371, 10.286615371704102, 23.599227905273438, 0.7204634547233582, 3.892200231552124, 6.023092746734619, -5.862695217132568, -2.2041099071502686, 9.328471183776855, -0.7844451665878296, 5.4619526863098145, -11.212491989135742, 1.7357193231582642, -9.055644035339355, 2.215792417526245, -10.089424133300781, -0.40106281638145447, 13.829988479614258, -2.4632084369659424, -9.316960334777832, -0.637569785118103, 17.745420455932617, -6.734490871429443, -0.9207959175109863, 6.787800312042236, 2.2527718544006348, -8.383204460144043, 21.93890380859375, -2.736320972442627, -1.4763718843460083, -12.866059303283691, -12.094155311584473, -8.030233383178711, -22.151107788085938, 5.586777687072754, 6.275908946990967, 0.5532206892967224, -7.047418117523193, -11.590303421020508], [-6.724844932556152, 2.9604969024658203, -0.6954941749572754, 4.539878845214844, 0.37260523438453674, -5.396495342254639, -10.421323776245117, -2.1773290634155273, 7.051595211029053, -2.985311269760132, -3.4843950271606445, -0.8280516862869263, 5.605271339416504, 9.494657516479492, -3.5445780754089355, -11.3384428024292, -7.215989589691162, 0.434568852186203, -6.602079391479492, -8.984212875366211, 18.374286651611328, -0.2025660127401352, 10.043354034423828, 6.348133087158203, -5.226561069488525, -8.419425964355469, -10.148415565490723, -0.2636171877384186, -4.949252605438232, 0.7372167110443115, 2.1295690536499023, -5.658961772918701, -6.807162284851074, -5.492945194244385, 8.947304725646973, 5.9740519523620605, 19.978378295898438, 4.5252509117126465, 10.380329132080078, 6.601254463195801, 2.6550304889678955, 3.837684392929077, 6.715552806854248, -2.494582176208496, -4.0085673332214355, -2.9957871437072754, -0.2683781087398529, 9.81978988647461, -1.2148874998092651, 0.9814201593399048, 15.862503051757812, -0.3682350814342499, -0.5111846327781677, -3.8851850032806396, 2.3082098960876465, -2.1759841442108154, 4.806569576263428, 1.967435598373413, 2.514474868774414, 7.870619773864746, -2.2658331394195557, -7.440354347229004, 5.885016441345215, 6.943356990814209], [3.910419464111328, 0.5177639722824097, -9.445917129516602, 4.165746212005615, 8.465015411376953, 5.298316955566406, 4.243008136749268, 9.143706321716309, -6.233875274658203, 13.112227439880371, -0.01799975335597992, 3.7653748989105225, -5.781294345855713, 4.817635536193848, 7.232847690582275, -11.482707977294922, 5.825804233551025, 1.2803962230682373, 3.411663055419922, 6.686052322387695, 0.06068652123212814, -7.003634452819824, 1.769584059715271, 9.356241226196289, -2.0716605186462402, 1.323111891746521, -4.381960868835449, -5.77002477645874, 5.659420013427734, 1.483898401260376, -9.837162017822266, -0.15263375639915466, -7.606825828552246, -3.5657334327697754, -0.6102444529533386, -14.082917213439941, 8.826200485229492, 1.8091602325439453, 2.466460704803467, 5.7130537033081055, 11.008223533630371, -3.9789366722106934, 5.2103447914123535, -0.20302173495292664, 6.600184917449951, 10.131118774414062, -10.660064697265625, -3.7562971115112305, -6.049921035766602, 7.824519157409668, 9.470698356628418, 7.105472087860107, 15.525440216064453, 13.762022972106934, 0.26821088790893555, -7.21419620513916, -12.505508422851562, 6.8200483322143555, 4.936418533325195, -6.243238925933838, -1.6604554653167725, 3.3115181922912598, 7.613030433654785, -9.745536804199219], [-0.9979638457298279, 2.627591133117676, -0.017202284187078476, -2.5984883308410645, -17.37798500061035, -11.08836555480957, 5.827417373657227, -12.310025215148926, -3.7053656578063965, -8.197468757629395, -11.243560791015625, -1.9398568868637085, 9.639212608337402, -7.455310344696045, -0.08511544018983841, 3.9736974239349365, 0.4646090567111969, -1.2015740871429443, -10.094086647033691, -3.019439458847046, 2.6321210861206055, -3.41672682762146, 8.428888320922852, 2.165189027786255, -1.9857466220855713, 0.8379682898521423, 7.884470462799072, 4.843177795410156, 2.6048409938812256, 22.4871768951416, 5.172330379486084, 3.430662155151367, -2.9679770469665527, -9.559659957885742, 5.453367233276367, 7.126284599304199, 7.105159282684326, -3.540375232696533, -2.1115550994873047, 1.9542306661605835, -1.2937878370285034, 1.315369963645935, -14.248747825622559, 0.29901280999183655, -15.845743179321289, -3.3206875324249268, 2.2170233726501465, 7.079655647277832, 2.3163955211639404, -9.20715045928955, -7.241992950439453, -1.6778478622436523, -0.7120556235313416, 10.651826858520508, 3.1511118412017822, 2.8893790245056152, -5.538856506347656, 6.224701404571533, -13.072279930114746, 5.203444004058838, 1.3825808763504028, -0.24418742954730988, -6.683729648590088, -10.16026496887207], [-9.453152656555176, -4.511212348937988, 1.6410473585128784, 11.00423812866211, -7.271665096282959, 2.992920160293579, -7.001436233520508, 3.8470892906188965, 6.700430393218994, 8.1842679977417, -0.18126654624938965, -9.478537559509277, -9.211146354675293, 1.8856556415557861, -4.870518207550049, 8.80166244506836, -4.463924884796143, 0.3505585789680481, -16.659690856933594, 15.1265869140625, 7.102630138397217, -18.797748565673828, -7.757048606872559, -3.205364227294922, 1.3311371803283691, -1.873043417930603, -7.1217427253723145, -7.250951290130615, -13.732357025146484, 3.6237614154815674, -3.0907535552978516, -8.122666358947754, 8.462499618530273, -0.07338457554578781, -9.86652660369873, -7.73455810546875, -4.15159273147583, 12.133936882019043, 5.7701239585876465, 8.65627384185791, -3.056903123855591, 24.60626220703125, -13.116490364074707, 5.815112113952637, 0.5114738345146179, -16.795780181884766, 4.53203821182251, 16.1280517578125, 8.47919750213623, -3.9408042430877686, 3.709484815597534, 1.6459500789642334, 4.392049789428711, -19.841781616210938, 1.1023508310317993, -5.666405200958252, 27.544063568115234, -3.441298484802246, 5.596742630004883, -3.9289684295654297, -14.25154972076416, 12.17119312286377, -8.569110870361328, -4.295148849487305], [2.912436008453369, 20.251352310180664, 20.918569564819336, 0.49943825602531433, 2.7703680992126465, 4.857089519500732, 13.856386184692383, 7.498822212219238, 16.57486915588379, 6.369377613067627, -3.1094534397125244, -14.387253761291504, -7.221711158752441, -4.679763317108154, -6.590911865234375, 4.547577381134033, 4.106240749359131, 4.841726303100586, -3.232555389404297, 12.023700714111328, -13.114611625671387, 12.307286262512207, -3.8991806507110596, -4.620925426483154, 11.704625129699707, 17.442367553710938, -13.021306037902832, -12.183788299560547, -0.38820451498031616, 5.053839683532715, -12.548637390136719, -2.9476699829101562, -3.8692641258239746, -6.413322448730469, 5.775294780731201, 13.763494491577148, 9.390583038330078, -11.509940147399902, 8.105655670166016, 13.2973051071167, 11.532821655273438, -0.484464555978775, 8.687005043029785, -12.148290634155273, -6.856673240661621, 7.924992084503174, 5.789758682250977, 1.859407901763916, 6.5788702964782715, -1.7025824785232544, -11.048237800598145, -2.2856645584106445, -3.3175101280212402, 10.067858695983887, -7.297739505767822, -2.461873769760132, 8.14306354522705, -4.192025184631348, -31.896780014038086, -1.7384847402572632, -5.005444049835205, 11.447418212890625, 7.447718620300293, -2.2381069660186768], [1.2378668785095215, 8.157194137573242, 0.8898894786834717, 5.796318531036377, 3.7889649868011475, 4.372270107269287, -5.08039665222168, 3.1578116416931152, 7.950618267059326, 10.181330680847168, 11.586041450500488, 8.14975357055664, -8.150330543518066, -6.221489906311035, 5.281700611114502, 13.064265251159668, -6.543067455291748, -0.24701619148254395, -13.416377067565918, 18.42432975769043, 3.9720051288604736, -1.5378189086914062, -21.252155303955078, -7.915287017822266, -16.424367904663086, -7.413359642028809, 5.08188009262085, 9.258316993713379, -0.763237476348877, 7.799495697021484, -20.122007369995117, -4.315933704376221, 5.128143787384033, 2.2754104137420654, 1.6667790412902832, -1.789418339729309, 16.784452438354492, 5.193118095397949, 2.907968521118164, 1.8674752712249756, -5.44713020324707, 14.66070556640625, -7.372844696044922, 0.0657213032245636, -1.1434195041656494, -11.244232177734375, 6.48399543762207, 16.419363021850586, 2.720460891723633, -6.164670467376709, -7.951355934143066, -3.1800284385681152, 8.213737487792969, -2.7897815704345703, 7.197872638702393, -10.385650634765625, 13.874256134033203, 1.430882215499878, 8.671391487121582, -12.464715957641602, -8.130736351013184, 4.634101867675781, 2.6562464237213135, 6.608636856079102], [-19.708452224731445, -4.59319543838501, 3.45011043548584, -5.325568675994873, -0.5542214512825012, 2.818664312362671, -14.228948593139648, -3.7353317737579346, -10.130753517150879, 9.130928039550781, 3.4693973064422607, 2.4631998538970947, 9.616844177246094, 6.149484157562256, 2.9453556537628174, 2.184810161590576, -7.405653953552246, 3.2822117805480957, -0.3505619168281555, -8.39531135559082, 4.117773056030273, -3.3166680335998535, 12.489179611206055, -5.388067722320557, 10.119770050048828, 1.9784764051437378, -9.552019119262695, 18.152698516845703, -5.4376068115234375, 0.619040310382843, 2.6282286643981934, -5.471440315246582, -9.297554016113281, -2.5355589389801025, -7.571310520172119, 16.128650665283203, 0.6704991459846497, 0.21275019645690918, 18.49015235900879, -0.910801112651825, 6.838733673095703, -4.986391067504883, -0.024613577872514725, -9.618989944458008, 10.451167106628418, 10.546974182128906, -0.9655252695083618, 11.654727935791016, -6.092818737030029, 16.433712005615234, 3.9420435428619385, -3.3968429565429688, -10.31348991394043, -1.6759434938430786, -12.394219398498535, -2.323078155517578, 13.357096672058105, 9.025259971618652, -17.49053192138672, 13.234426498413086, 2.7694413661956787, 12.381668090820312, 4.924771308898926, -3.6806395053863525], [0.8969971537590027, 0.6143090724945068, 0.9402132034301758, -3.798058032989502, 2.520927667617798, 5.282811164855957, -0.6971186995506287, -5.7798614501953125, -1.6052747964859009, -9.626480102539062, 2.861873149871826, -2.782291889190674, 7.420699119567871, -0.4163817763328552, 0.9912029504776001, 0.18372288346290588, -5.470195770263672, -11.307430267333984, 4.310351848602295, -9.213380813598633, -5.6568427085876465, 2.3516740798950195, -8.332260131835938, 1.717932939529419, -1.7704899311065674, -5.476016521453857, -0.9038204550743103, -14.973895072937012, 0.7783978581428528, -0.2651057541370392, 3.6507489681243896, -9.601212501525879, 3.1696012020111084, 3.9100406169891357, -10.337757110595703, 7.5915632247924805, 8.278763771057129, 7.495635509490967, -13.413118362426758, -0.4944274425506592, -9.567548751831055, -1.5671545267105103, -5.8521246910095215, 11.648248672485352, -2.9020822048187256, -6.441674709320068, -14.665205001831055, 0.618865430355072, -0.5971798300743103, -20.38507080078125, -4.8696088790893555, -1.1847014427185059, 4.031599521636963, -2.3692290782928467, 8.73840045928955, -5.127208232879639, -4.785190105438232, -7.520779132843018, 3.7444515228271484, 0.03887561708688736, 0.06424086540937424, -9.426352500915527, 15.605645179748535, -4.724086284637451], [9.59885025024414, 13.32886028289795, 0.6588274836540222, -13.07658863067627, 1.0517523288726807, 0.1863056719303131, 0.6897696256637573, -16.60849952697754, -3.9181764125823975, -11.602391242980957, -5.328044891357422, -0.2783244848251343, -13.826526641845703, -0.11067212373018265, 4.167050361633301, 2.126511812210083, 7.501075267791748, 7.839544296264648, 14.398640632629395, -7.249252796173096, -13.846227645874023, 13.797107696533203, 1.9711885452270508, -4.939612865447998, -2.0825562477111816, 15.0443115234375, 5.49810266494751, 5.610677242279053, 1.5370632410049438, -6.4039506912231445, -8.656037330627441, 3.726430892944336, -4.5, -9.662707328796387, 5.159337520599365, -6.658225059509277, -17.550573348999023, 5.129413604736328, -2.1370012760162354, 3.920168161392212, 3.706207036972046, -7.703328609466553, 10.450799942016602, -16.7879695892334, -7.495009899139404, 16.864076614379883, 5.566793441772461, -7.36170768737793, -3.1014106273651123, 4.435749053955078, -2.8702244758605957, 0.6278585195541382, 13.960871696472168, 7.881494522094727, -4.23852014541626, -7.6925482749938965, -9.955602645874023, 3.164562225341797, -20.518129348754883, 3.5609242916107178, -8.012609481811523, 1.6896467208862305, 1.732927680015564, -11.880852699279785], [-2.722510576248169, -2.133772373199463, 6.423287391662598, 7.11358118057251, -2.1389615535736084, -2.242166757583618, -4.03375244140625, 6.7575178146362305, 2.895474910736084, 10.000471115112305, 2.040116310119629, 3.507643222808838, -0.6601587533950806, 5.636637210845947, -6.870696544647217, -4.971860408782959, -14.741437911987305, 4.585029602050781, -5.813878536224365, 10.892721176147461, -9.588091850280762, -7.680880069732666, 20.088991165161133, -4.028391361236572, -1.7883530855178833, -4.620838642120361, -10.327957153320312, 11.176634788513184, -0.7028833031654358, -11.228476524353027, -14.326037406921387, -9.60486888885498, -5.972538948059082, -9.804876327514648, 0.2360740751028061, -1.0225551128387451, 12.809261322021484, -9.512255668640137, -4.878235816955566, -1.1459087133407593, 3.0782599449157715, -6.956589221954346, 2.3263251781463623, 12.942835807800293, 10.863408088684082, 10.507527351379395, -4.273561477661133, 1.5238317251205444, 4.904709815979004, -9.460273742675781, 8.241710662841797, 11.814627647399902, -1.965203046798706, 0.9654383659362793, 0.09986772388219833, 16.235950469970703, 0.5639852285385132, -1.8012632131576538, 17.582273483276367, 1.8718160390853882, 6.693284034729004, -18.535198211669922, -4.344338417053223, 7.082456588745117], [-4.584630489349365, 5.492674350738525, 5.634949207305908, -7.710816383361816, 4.077027320861816, 8.398820877075195, -18.550334930419922, -21.92555046081543, -2.3572428226470947, -9.246684074401855, -3.2130634784698486, 7.690027713775635, -10.356057167053223, 2.4646615982055664, 22.597103118896484, -11.002010345458984, 8.245321273803711, 8.712411880493164, -2.572650909423828, -1.8423588275909424, -10.047286033630371, -2.0705509185791016, -15.269192695617676, 6.551369667053223, -6.931179523468018, 4.33329439163208, 1.8149610757827759, -7.8001708984375, -0.25154972076416016, 6.351146697998047, -13.795707702636719, 0.3237234652042389, 4.0096869468688965, 5.87757682800293, 0.8412392139434814, 23.176969528198242, -8.27825927734375, 11.186793327331543, 0.7465357780456543, -18.440561294555664, 5.186650276184082, 15.725937843322754, 4.316324710845947, -15.87779426574707, 3.211221218109131, -9.17800521850586, -16.107845306396484, -3.917837381362915, -10.13088321685791, -4.547650337219238, 0.7008861303329468, -13.396774291992188, 9.191832542419434, -9.093459129333496, -4.799806118011475, -5.91674280166626, 4.1317458152771, 3.8379476070404053, 5.762552738189697, 10.916011810302734, 0.12001005560159683, 3.51106595993042, 3.1567466259002686, -1.7268612384796143], [-7.835227012634277, 0.9647791385650635, -3.2923500537872314, 0.6467257738113403, 13.795218467712402, -8.527395248413086, -10.292959213256836, -9.867047309875488, -4.9925384521484375, 2.614945411682129, 5.758563995361328, -5.156546592712402, -0.9345988035202026, -2.1619415283203125, 10.765450477600098, 0.49872493743896484, -3.8984084129333496, 1.3061494827270508, -8.7042875289917, 14.594215393066406, -5.415947914123535, 2.4276185035705566, -1.4998234510421753, 10.429418563842773, 11.117226600646973, 4.589079856872559, -11.458572387695312, -15.227652549743652, -3.2815096378326416, 2.0436182022094727, -4.3418498039245605, -2.3725719451904297, 1.5328209400177002, 6.421904563903809, -1.9806416034698486, -1.414483904838562, 2.3577964305877686, 2.0339159965515137, -6.3307952880859375, 6.663398742675781, -17.16507339477539, 4.936783313751221, 11.825215339660645, -0.3331688642501831, -2.743152141571045, -7.426095962524414, 5.867542266845703, -9.790114402770996, 2.566840648651123, -6.710130214691162, 1.125781774520874, -7.112164497375488, -5.737371444702148, -0.46676576137542725, 1.3523582220077515, -8.01173210144043, 0.8143051862716675, -1.342038869857788, 26.73743438720703, 1.477820873260498, -3.661036491394043, -4.57562780380249, -11.829094886779785, -15.795023918151855], [-2.408294439315796, -0.9870747923851013, -19.916379928588867, -4.8268866539001465, -5.567687511444092, -10.768898963928223, 15.405292510986328, -1.1024770736694336, -7.191330909729004, 5.747395992279053, 5.75732421875, 5.742086410522461, -7.124450206756592, 10.88088607788086, 9.913290977478027, -15.578435897827148, -2.925889253616333, 13.973834991455078, -4.660367488861084, 13.313000679016113, -16.054222106933594, -9.174802780151367, -0.5580940842628479, 10.97085189819336, -12.654306411743164, 2.404952049255371, -5.455602169036865, -6.8457207679748535, 14.020231246948242, -2.787738084793091, -2.6075222492218018, -3.2451674938201904, 7.672432899475098, -5.522359371185303, 4.457730770111084, -7.043064117431641, 7.342977523803711, -5.431339740753174, 12.242822647094727, 14.4649658203125, -6.325840473175049, -2.4863836765289307, 20.139915466308594, -4.893725872039795, -0.8001843690872192, 3.350828170776367, 12.087164878845215, 4.4810051918029785, -1.9634040594100952, 5.772913455963135, 3.17626953125, 25.314008712768555, -5.162604808807373, 3.4146640300750732, 8.546365737915039, 15.317740440368652, 5.9590654373168945, 0.40171438455581665, 15.390034675598145, -7.504652976989746, 8.111042022705078, -12.984694480895996, 6.090404510498047, -5.675378799438477], [5.910868167877197, 5.486734867095947, -0.12133888155221939, 9.470379829406738, -0.04854194447398186, -0.4577200412750244, 28.438961029052734, 15.016118049621582, -3.313844680786133, -8.795527458190918, 4.978654384613037, -3.186393976211548, 5.135229110717773, 3.7270798683166504, 5.439493179321289, -0.5967308878898621, -3.3822691440582275, 4.320987701416016, -3.833793878555298, 4.520535469055176, 1.2435193061828613, -3.755667209625244, 7.750634670257568, -2.65366530418396, 5.410671234130859, 3.0797276496887207, -12.681884765625, 2.3144047260284424, 7.702986240386963, 5.284246921539307, 16.557872772216797, 7.386526584625244, -12.59780502319336, 3.3954715728759766, 7.566848278045654, 3.6610805988311768, 9.260921478271484, 2.233832359313965, 1.3052699565887451, -4.492681503295898, -8.950072288513184, -14.392459869384766, -7.624440670013428, -10.545350074768066, -2.2160837650299072, -0.3454437553882599, 2.004657745361328, 13.458548545837402, -17.81191635131836, -13.587132453918457, 0.407907634973526, -7.790180206298828, -6.223849773406982, -9.640890121459961, -1.2424631118774414, 13.514420509338379, 8.394524574279785, 7.5011420249938965, -0.5381834506988525, 2.68101167678833, 2.3845996856689453, 3.494950771331787, -12.61848258972168, -9.479183197021484], [-6.643155097961426, -14.32604694366455, -1.5809053182601929, 2.033141851425171, -20.73204803466797, -1.863774299621582, -2.420419931411743, 5.389279842376709, -18.52779197692871, -5.820271968841553, -6.426510334014893, 1.1989853382110596, -1.486339807510376, 0.11761200428009033, 10.604015350341797, 0.21528136730194092, -2.1469154357910156, 4.241789817810059, -1.663589358329773, 9.739501953125, 5.016467094421387, 2.829374313354492, 5.154780864715576, 4.7667927742004395, 6.4178571701049805, -2.7313742637634277, -0.4399266242980957, 10.58554458618164, 0.5955497026443481, 5.246975421905518, -7.132988452911377, -1.7942588329315186, 1.0928748846054077, 5.548924922943115, 1.5378907918930054, 4.432216167449951, 2.856961727142334, 4.216171741485596, -6.3568925857543945, 9.861727714538574, 0.12124044448137283, 4.8009114265441895, 3.86859393119812, -6.297818183898926, -2.2197721004486084, 26.025327682495117, -0.326839804649353, 17.12637710571289, -2.3068058490753174, 14.328465461730957, -6.535594940185547, 10.32961654663086, -3.5241968631744385, -4.326814651489258, 0.04142768308520317, -2.1344010829925537, -15.237524032592773, 9.034213066101074, -4.036792755126953, 5.9160895347595215, 12.657210350036621, 7.716578006744385, 8.466851234436035, -2.5225446224212646], [-3.0237889289855957, -1.2627261877059937, -0.6793463826179504, 5.097151756286621, -3.153444528579712, 1.1425414085388184, -9.346466064453125, -2.9454874992370605, -7.001168251037598, -9.259486198425293, -3.7120442390441895, -0.7420010566711426, 8.112550735473633, -7.972021102905273, 0.1258779764175415, 2.517169952392578, -5.710682392120361, -3.4338743686676025, 7.385895252227783, -10.864514350891113, -8.359530448913574, 14.058889389038086, -3.298227071762085, 1.4326462745666504, 3.368089437484741, 2.1610379219055176, -5.3587117195129395, 19.1779842376709, -2.348315477371216, -11.00720500946045, 8.253104209899902, -8.512203216552734, 2.1813719272613525, -2.3981480598449707, 2.1018106937408447, -3.305960178375244, 2.5168750286102295, -6.333841800689697, -1.6088546514511108, -2.036060333251953, -3.7615368366241455, -12.022798538208008, -0.999058187007904, -3.947268486022949, -5.765472412109375, -5.2857666015625, 14.793344497680664, -10.691943168640137, 2.732041835784912, -3.6993067264556885, -4.763945579528809, 6.7627739906311035, 4.395779132843018, -8.159505844116211, -6.207431793212891, 2.067730188369751, -0.5583032965660095, -8.594656944274902, -3.082087516784668, -0.3546411991119385, 4.475727558135986, -3.730685234069824, -9.807940483093262, 11.221948623657227], [6.657591342926025, -7.600581645965576, 21.642797470092773, -8.566213607788086, 6.202284812927246, -8.948652267456055, -10.726934432983398, 8.791260719299316, -11.634875297546387, 15.171894073486328, 11.276500701904297, 10.314318656921387, 16.773889541625977, 15.510807991027832, 15.44308090209961, -7.020930767059326, -4.769201755523682, -0.8103426694869995, -0.5401918888092041, -8.377988815307617, -7.381315231323242, 0.8527399897575378, 7.052789688110352, -0.18543694913387299, -0.34172385931015015, -4.9406657218933105, -8.90342903137207, 1.6155251264572144, 6.115028381347656, -6.794396877288818, 5.998879432678223, 0.33566269278526306, 10.183173179626465, 9.702939987182617, -2.974990129470825, -3.8891780376434326, -2.897615432739258, 11.625597953796387, 0.25211793184280396, -3.5102593898773193, -1.5970585346221924, -10.938074111938477, 12.414902687072754, -5.927282810211182, 9.349088668823242, 10.338509559631348, -15.288413047790527, -0.4813554584980011, -13.362491607666016, -3.491894006729126, 9.39150333404541, 4.40386438369751, 0.40591686964035034, -4.615551471710205, 1.8114354610443115, 12.997541427612305, 5.610180377960205, 10.788556098937988, 17.80280113220215, -0.5121932625770569, -19.396137237548828, -8.038801193237305, 9.088801383972168, 0.49511581659317017], [-7.542884349822998, -1.7291725873947144, 1.8121920824050903, 3.290900468826294, 14.291723251342773, 0.2147863209247589, -12.085746765136719, -1.0118417739868164, 3.718975782394409, 11.722042083740234, 11.556694030761719, 3.907545328140259, 6.573462009429932, 12.499624252319336, 13.242770195007324, -19.03656768798828, 8.11668872833252, 4.9745378494262695, -2.02114200592041, -11.025514602661133, -14.263716697692871, 2.907268524169922, -17.373291015625, -2.8754477500915527, -6.0976881980896, 2.258143901824951, -7.662816524505615, 0.7571631669998169, 1.6801565885543823, 1.6772105693817139, 12.822136878967285, -2.049586057662964, 4.046945095062256, 8.542640686035156, 9.732714653015137, 14.169130325317383, 10.20401382446289, 3.371367931365967, 17.15860939025879, -4.0825371742248535, 16.52098274230957, -8.021947860717773, 9.33106803894043, -6.618507385253906, -5.345265865325928, -2.944218873977661, -3.0109376907348633, -10.364334106445312, 0.8854724764823914, 15.956159591674805, -5.945083141326904, -1.003293752670288, -12.153559684753418, -9.470627784729004, 2.0577585697174072, 5.983010292053223, 22.473304748535156, -4.913074970245361, -4.761203289031982, 8.1715087890625, 20.775978088378906, -5.486319541931152, 0.47948983311653137, -2.702664375305176], [-9.288837432861328, -4.046945571899414, 1.458364725112915, 6.043600559234619, 8.680378913879395, 8.929758071899414, -6.218249320983887, 3.0972466468811035, 3.7863121032714844, -0.9623444080352783, 0.12060122191905975, -5.755948543548584, -2.3149893283843994, 0.7980168461799622, 0.7665889263153076, -6.856931209564209, -15.203450202941895, -8.308835983276367, -3.3974432945251465, -0.6066823601722717, 18.279586791992188, -9.925209999084473, 0.5309406518936157, 2.0497281551361084, 14.785592079162598, -3.5704591274261475, -11.414185523986816, 1.6087356805801392, 6.406079292297363, -10.27599811553955, 7.981751441955566, -7.308908939361572, -4.680016994476318, -1.2156116962432861, -0.34378528594970703, -11.847766876220703, 12.501029968261719, -7.762184143066406, 11.450911521911621, 4.481589317321777, -1.242771863937378, 4.684404373168945, 8.0676851272583, -13.392801284790039, 0.7092338800430298, -3.620577335357666, 12.886419296264648, -6.6607818603515625, -2.3163537979125977, 0.2904551029205322, 1.728464126586914, -7.826254844665527, 10.010026931762695, 0.18680231273174286, -13.199576377868652, -12.210458755493164, 4.059482574462891, 2.0917460918426514, 8.293244361877441, 1.8420103788375854, 6.296690464019775, 0.6438394784927368, 12.091835021972656, -3.4072089195251465], [6.679678916931152, 10.980721473693848, -2.469092845916748, -1.890799641609192, 4.772603511810303, -7.1889142990112305, 8.796402931213379, -4.41145658493042, -23.90277862548828, 1.0301042795181274, -6.403478145599365, -2.172922134399414, 2.7806224822998047, 7.720309734344482, -12.09824275970459, -10.139591217041016, 7.1523237228393555, -12.234192848205566, -0.37235167622566223, -10.313132286071777, 24.027729034423828, -3.64418625831604, -0.9571517109870911, -0.6074681878089905, -2.7983927726745605, 2.3376662731170654, 3.419524908065796, 3.084102153778076, 12.429547309875488, -10.144993782043457, 8.117295265197754, 0.1856933981180191, 2.932426929473877, -5.02652645111084, -12.166576385498047, -9.649356842041016, 2.045707941055298, 3.528322458267212, 6.737603664398193, -4.194143295288086, 1.2870749235153198, -2.9609317779541016, 1.2171329259872437, 3.6154675483703613, 10.05054759979248, -1.3422104120254517, -0.11511685699224472, 11.264147758483887, -5.859353542327881, 4.735753059387207, 3.108114242553711, 5.795823574066162, 2.4515366554260254, 3.946850061416626, 1.907423734664917, 8.233302116394043, 6.19589900970459, -4.980789661407471, -4.639763832092285, -12.947531700134277, -6.450789928436279, 0.44676724076271057, 12.454389572143555, 7.3695597648620605], [6.697247505187988, 7.042505264282227, -8.349241256713867, -7.155910015106201, -3.2358319759368896, -1.7850245237350464, 6.363549709320068, -0.6225312948226929, -8.905590057373047, 11.661296844482422, 16.26926612854004, -0.36616647243499756, -20.442136764526367, 4.70306921005249, -4.247853755950928, -1.3919960260391235, -11.083649635314941, -1.2343077659606934, 0.18745186924934387, 10.746184349060059, -6.284172058105469, -9.752737045288086, -1.849502682685852, 4.329222202301025, -11.33588981628418, 2.9686601161956787, 1.86880362033844, 5.078794956207275, 10.966778755187988, -11.411717414855957, 0.738102912902832, -14.475420951843262, -3.5283987522125244, -5.214345932006836, -5.451600074768066, -5.659261703491211, 3.097975969314575, 19.56272315979004, 7.194424629211426, 0.5524600744247437, -17.81081771850586, 11.652754783630371, 11.030163764953613, 10.716492652893066, 10.005966186523438, -10.490377426147461, 1.5544511079788208, 13.789993286132812, 3.3170647621154785, -0.42345118522644043, -9.311232566833496, -0.3475866913795471, 1.1647319793701172, -7.609962463378906, 5.0504255294799805, 12.947558403015137, 5.710152626037598, -3.4771299362182617, -11.458806991577148, -10.197704315185547, 6.426953315734863, -2.0104637145996094, -11.33193588256836, -0.5330108404159546], [-4.385868072509766, 3.3593716621398926, 1.9215821027755737, 2.5449459552764893, -2.3506991863250732, -0.2080988734960556, -14.76598072052002, -8.226500511169434, 3.6635758876800537, -2.8677492141723633, 8.822090148925781, -3.731536388397217, -1.9744021892547607, 8.967330932617188, 2.6573312282562256, -3.8409881591796875, 3.7930209636688232, 4.285177707672119, -4.043415546417236, -4.075432300567627, 14.962407112121582, 5.158011436462402, 21.407928466796875, -2.404008150100708, 1.053224802017212, 7.111475467681885, -12.072223663330078, -11.22458553314209, -5.7569756507873535, 9.744843482971191, -4.39569091796875, -6.950676441192627, -11.780563354492188, 0.6432331204414368, -1.3310202360153198, 0.6521540284156799, 0.3813106119632721, 1.1604598760604858, 7.166498184204102, 2.0265750885009766, 6.946451187133789, -10.690370559692383, 7.340635299682617, -3.0286643505096436, 12.960854530334473, -5.815399646759033, -11.160484313964844, -6.2878522872924805, -12.94987964630127, 1.2599124908447266, -5.616281986236572, -11.145864486694336, -8.808282852172852, -2.7316627502441406, -1.5434443950653076, -4.1517109870910645, -0.09203611314296722, -1.5055553913116455, 8.604820251464844, 6.622791767120361, -2.995056390762329, 1.9652910232543945, 4.566275596618652, -1.9063502550125122], [-8.467130661010742, 11.931269645690918, -2.001896619796753, 1.3051128387451172, 7.325716018676758, -1.5935838222503662, -14.502660751342773, -10.91628646850586, -13.959065437316895, 2.1569416522979736, 8.882265090942383, 3.4209697246551514, 17.869070053100586, 10.326519966125488, 6.665337085723877, -3.139242649078369, 7.825042724609375, 1.525875210762024, -11.126086235046387, 3.3968334197998047, 0.48953238129615784, 9.184680938720703, 8.090137481689453, 8.173014640808105, -5.292184829711914, 6.807937145233154, -2.9797160625457764, 22.013704299926758, -2.7459564208984375, 9.805776596069336, 0.2036268711090088, -1.3168132305145264, -9.083354949951172, -6.722090721130371, 1.6091272830963135, 3.520404815673828, 7.843064308166504, 9.143913269042969, -3.9564108848571777, -6.067835807800293, 14.33142375946045, -19.442548751831055, -3.130728006362915, -3.88520884513855, 14.041849136352539, 3.7810726165771484, 0.39370548725128174, -15.8829984664917, 1.9616273641586304, 4.049677848815918, -1.9236072301864624, 6.7406206130981445, 3.2845330238342285, 6.102594375610352, -2.6130354404449463, 0.8157232999801636, -11.092883110046387, 4.62992525100708, 7.236058712005615, 8.35860538482666, 14.951252937316895, 0.7009634375572205, 4.867612361907959, 1.4597667455673218], [-7.860851764678955, -6.3593573570251465, -9.839143753051758, 6.280020236968994, 2.4682083129882812, 7.291708946228027, 2.0480353832244873, -2.745166540145874, -6.628861904144287, 1.9338761568069458, 16.35667610168457, 0.16738516092300415, 2.8686139583587646, -6.613118648529053, 11.313879013061523, -4.678439617156982, -3.6168107986450195, 3.7966976165771484, -8.94562816619873, 9.665435791015625, -6.651189804077148, -15.971566200256348, -15.060249328613281, -16.07843589782715, -4.985266208648682, -1.5083333253860474, -2.1962294578552246, -0.0004156246723141521, 4.038253307342529, 19.028718948364258, 0.30045968294143677, 8.254572868347168, 9.374088287353516, 0.15188655257225037, 4.216607570648193, 6.4863200187683105, 4.58850622177124, 2.6581547260284424, -0.04612767696380615, -13.250531196594238, 4.565618991851807, -10.476186752319336, -10.137428283691406, 0.3295406401157379, 5.197787284851074, -10.377739906311035, 2.5904808044433594, 3.4378087520599365, 1.9513779878616333, -1.1666780710220337, -0.4213241636753082, -5.108654499053955, -6.982944011688232, 2.7592339515686035, 14.58327579498291, -11.14183235168457, -2.0737860202789307, 8.913254737854004, 19.982629776000977, 3.2557501792907715, 9.953224182128906, -4.752333641052246, -6.0025634765625, -0.3500889241695404], [15.485291481018066, 10.803836822509766, 7.917501926422119, -13.274227142333984, -8.105627059936523, 7.054512023925781, 11.994793891906738, 11.580032348632812, -15.151771545410156, 0.3701306879520416, 20.863340377807617, -15.885419845581055, 10.005568504333496, 17.795501708984375, 1.392910361289978, 5.524605751037598, 4.874616622924805, -4.0545973777771, 12.46615982055664, -4.3983612060546875, -7.777063369750977, 16.499801635742188, 1.218868613243103, -7.888391017913818, -4.418996810913086, 3.247502565383911, -0.6253822445869446, 5.415064811706543, 2.037788152694702, -13.502262115478516, 1.276808738708496, -6.242984294891357, 12.316869735717773, 7.06818151473999, -9.031027793884277, 6.348540782928467, 8.259197235107422, 8.456098556518555, -6.93093204498291, -10.709329605102539, 1.1844394207000732, -5.225500106811523, -4.768494129180908, 7.500972270965576, 3.130709409713745, 3.5133678913116455, -0.3905382752418518, 11.53050422668457, -6.675152778625488, -15.02929401397705, -22.897640228271484, -4.905309677124023, 3.5513315200805664, -13.967291831970215, -3.2129597663879395, -5.56789493560791, 20.270727157592773, 1.9141520261764526, -14.646652221679688, -1.9608439207077026, -8.20075798034668, -11.061609268188477, 10.205785751342773, -16.241573333740234], [6.619579792022705, 9.208864212036133, 7.390895366668701, 0.8057141900062561, 0.038624923676252365, -5.017948627471924, -10.229633331298828, 12.282758712768555, 16.08991241455078, 2.8557162284851074, -3.2643227577209473, 14.308877944946289, -12.258733749389648, 3.910252809524536, -3.9416511058807373, 10.527254104614258, 3.323848247528076, -6.567215442657471, 9.252004623413086, -13.304159164428711, 11.35927963256836, 3.9203429222106934, -0.30378273129463196, 9.010209083557129, 8.917162895202637, -6.308879852294922, 5.027294158935547, 2.529104232788086, -3.338186502456665, -11.237574577331543, -4.08328104019165, 6.205381393432617, 3.3535494804382324, -8.479072570800781, -12.567717552185059, -16.564252853393555, -0.45131149888038635, 1.6710760593414307, 1.1001062393188477, 19.97205352783203, 0.4022842049598694, 6.995715141296387, 2.48580002784729, 5.0368828773498535, 4.3647260665893555, 8.323850631713867, 0.20288503170013428, -9.717215538024902, 11.734674453735352, -15.330378532409668, -5.217169284820557, 1.3013131618499756, 0.4038291573524475, 12.717676162719727, -1.6962639093399048, -0.9133945107460022, -1.9843978881835938, -6.615867614746094, -3.8295209407806396, -11.586318016052246, -18.880929946899414, 16.21940803527832, 0.6536552309989929, 13.94310188293457], [5.4630608558654785, -16.587596893310547, 19.083572387695312, -9.260873794555664, 3.65439772605896, -11.138598442077637, -16.447267532348633, -11.137992858886719, 9.671077728271484, -7.542794227600098, -6.462718963623047, -4.177599906921387, 11.982694625854492, 0.14707645773887634, -3.9234211444854736, 13.269087791442871, 4.727699279785156, -5.891355037689209, -10.047746658325195, -2.7835566997528076, -3.2705185413360596, 16.031156539916992, 9.293506622314453, 6.64960241317749, -4.532564163208008, -19.836917877197266, 14.906245231628418, 9.652108192443848, -1.629089117050171, -6.937138080596924, -6.154467582702637, -2.2648561000823975, 0.9177452325820923, 7.664574146270752, 13.74008560180664, -5.243768692016602, 2.641873836517334, 10.630638122558594, -4.565951347351074, 4.127885818481445, 11.981566429138184, 10.680074691772461, 7.896206378936768, -2.856264352798462, -18.111555099487305, 2.2800498008728027, 0.3023606538772583, -1.401747226715088, 2.93660306930542, -4.388953685760498, 9.951311111450195, 6.492170333862305, 3.9759676456451416, -7.6038312911987305, -6.237283706665039, -11.954429626464844, -3.5224196910858154, -0.37610819935798645, 6.331305980682373, 7.745585918426514, -9.00232219696045, -15.116701126098633, 16.243492126464844, 3.4074416160583496], [-6.043552875518799, 4.744643688201904, 7.403411388397217, -0.8134176135063171, 18.880300521850586, -2.9719552993774414, -21.670495986938477, -1.8139680624008179, 2.590373992919922, 6.57853889465332, 10.01766586303711, -7.0166521072387695, 7.756265640258789, -0.00899389386177063, -7.446834087371826, -15.065726280212402, -9.647464752197266, 5.284040927886963, -6.55517053604126, -11.95738697052002, -6.185115814208984, 14.518989562988281, -8.534674644470215, -13.667097091674805, -3.903862953186035, 5.5200276374816895, -7.955591201782227, 29.85099983215332, -9.949049949645996, 11.541803359985352, 12.412400245666504, -5.3000640869140625, 7.193310260772705, 4.975360870361328, 1.5392779111862183, 14.56433391571045, -12.354240417480469, -9.319502830505371, 5.887856483459473, -7.197447299957275, -1.8609205484390259, -12.904947280883789, -4.036363124847412, -2.990004301071167, 5.814091682434082, -2.286804437637329, 12.318017959594727, -7.423288822174072, -4.369786262512207, 22.711889266967773, 3.3513622283935547, -6.333445072174072, 0.669719934463501, -0.9129005074501038, -7.107436180114746, -9.375601768493652, 5.211085796356201, -6.239744186401367, 7.0588178634643555, 2.591198444366455, 2.7599287033081055, 3.233766794204712, -13.331027030944824, 13.715710639953613], [6.5542731285095215, -6.942802429199219, -3.8825604915618896, 2.501354932785034, -16.383066177368164, -11.503305435180664, 0.16434797644615173, 4.809873104095459, -5.843159198760986, 5.26582670211792, 0.46162986755371094, -15.068853378295898, 4.8456034660339355, 13.086834907531738, -3.0000109672546387, -16.709062576293945, -7.970273971557617, 4.701937198638916, 3.968451976776123, 17.06519317626953, -7.7589874267578125, -15.233368873596191, 7.031190872192383, -6.7014946937561035, -3.2920212745666504, 7.943997859954834, 18.068023681640625, 2.880887269973755, 11.819595336914062, 1.6681326627731323, -11.37613296508789, -0.2250007838010788, -1.314411997795105, 6.1944756507873535, -4.937833309173584, -0.8330559134483337, -15.320780754089355, -4.473556995391846, -0.16801270842552185, -4.718325614929199, -14.698607444763184, -1.1889292001724243, -0.5049948692321777, 2.572204351425171, -5.30965518951416, -2.368915319442749, -12.067689895629883, 5.643829345703125, 2.300532817840576, 0.3279372453689575, 9.600174903869629, 2.634188652038574, -2.1432278156280518, 6.869032382965088, 18.429828643798828, 12.585942268371582, 9.534322738647461, 3.3549611568450928, 7.5707106590271, -4.070783615112305, -16.674036026000977, -13.197115898132324, 2.8808157444000244, -16.8533935546875], [2.692680597305298, 1.3137567043304443, 15.810409545898438, 6.44962215423584, -11.689762115478516, -6.711823463439941, 7.154803276062012, -9.019371032714844, -4.268260478973389, 0.49433937668800354, -2.6927080154418945, -16.671428680419922, 7.897019863128662, -3.6457693576812744, -4.116489887237549, -7.115764617919922, -2.3201205730438232, 0.3193885087966919, 1.4272162914276123, 11.523931503295898, -3.2035300731658936, 6.953823089599609, -12.831440925598145, -1.8698089122772217, -6.507778644561768, -6.968165874481201, -8.430042266845703, -2.174107313156128, -5.164310932159424, 7.301148891448975, 16.012540817260742, -3.156222343444824, 10.536541938781738, -5.670112133026123, -6.6277689933776855, 2.256422519683838, -0.03354376554489136, 3.3762893676757812, -13.133021354675293, -22.968687057495117, 4.068056106567383, 3.7959775924682617, -1.0695555210113525, -10.411413192749023, -9.65931510925293, -2.9204189777374268, 2.590630531311035, 0.03807976469397545, 0.953021764755249, -3.295841693878174, 3.1255900859832764, 8.783644676208496, 21.900379180908203, 4.720583438873291, 1.4864087104797363, 6.6885247230529785, -5.24652099609375, -1.3148598670959473, -7.531775951385498, -4.911290168762207, -5.3062663078308105, -5.587124347686768, 2.2657594680786133, 7.549104690551758], [-6.451876640319824, -9.020220756530762, -2.4068713188171387, -9.285955429077148, 6.815042018890381, 5.5106425285339355, -13.4487943649292, -5.2407097816467285, 13.48771858215332, 0.41500961780548096, 11.221647262573242, -3.922781229019165, -1.9974957704544067, 4.149485111236572, 1.663910984992981, 0.6351553797721863, -9.916479110717773, 1.877929449081421, -5.312805652618408, -10.14729118347168, -2.7675628662109375, 1.421047568321228, -5.374152660369873, -22.209077835083008, 1.6820875406265259, -2.589134454727173, 0.9985312223434448, 1.838220238685608, 4.208711624145508, 8.118626594543457, 1.5537583827972412, 8.615100860595703, 1.2664238214492798, 9.23281478881836, 15.240336418151855, 0.7690511345863342, -4.4288201332092285, -4.107491493225098, -9.362072944641113, -3.617445945739746, 12.29183578491211, 0.4533393085002899, 5.495147228240967, 9.280719757080078, -6.446643829345703, 0.946077287197113, 6.604201316833496, -1.7871321439743042, 4.889462947845459, -1.5533307790756226, -4.85944938659668, 4.486053466796875, -11.304262161254883, 6.336915493011475, -4.61377477645874, -6.570488452911377, 3.351241111755371, -3.026431083679199, 6.349937438964844, 2.080049753189087, 3.42452335357666, 10.415326118469238, 2.3039004802703857, -2.3264617919921875], [-11.330365180969238, -15.863381385803223, -8.227649688720703, 3.0056731700897217, -5.151686191558838, 3.7326667308807373, -3.6006970405578613, 5.928136825561523, 4.754544734954834, 3.953511953353882, -11.295186042785645, -0.7187550067901611, -5.178426742553711, -5.793691635131836, -3.466219663619995, -4.062521457672119, -6.084967613220215, 2.5658278465270996, 12.829882621765137, -7.634407997131348, -8.007833480834961, 6.625144958496094, -1.346182107925415, 7.735416889190674, -0.5201027393341064, 12.361294746398926, -11.149212837219238, -1.4629712104797363, -2.918799877166748, -16.11355972290039, -0.28109264373779297, -18.296396255493164, 5.360374927520752, -9.35515022277832, -12.712684631347656, -9.047430038452148, -2.827608346939087, -4.864195346832275, 4.643825054168701, 6.551826477050781, 6.406119346618652, 2.011934518814087, 15.680378913879395, -4.247647762298584, 0.5985199213027954, 4.026208400726318, -2.4557723999023438, -10.465383529663086, 5.027810096740723, 4.265319347381592, 7.788515567779541, 17.15374183654785, -3.3650290966033936, 1.555402398109436, -4.069221496582031, 4.856472492218018, -2.101616144180298, -17.207868576049805, -7.684712886810303, -1.3557401895523071, 10.717795372009277, -4.301762580871582, 2.182894229888916, 3.956434726715088], [-3.254516839981079, -3.7993319034576416, 4.134160041809082, 3.274850845336914, -11.204648971557617, 4.2555131912231445, -17.551286697387695, -0.6077693104743958, -13.698833465576172, 13.69373893737793, -1.3275038003921509, 13.547712326049805, 6.560557842254639, -5.489333152770996, 9.092982292175293, 13.301316261291504, -10.532764434814453, -7.10123872756958, 8.72000789642334, -20.808116912841797, -6.568726539611816, 2.258016586303711, 6.687197685241699, 0.20997104048728943, 2.7066164016723633, -5.190957069396973, -2.8996756076812744, -8.836387634277344, -0.49347370862960815, -17.44113540649414, 4.036903381347656, -17.57114601135254, 0.1095481887459755, -4.583226680755615, -7.70170783996582, -1.7814526557922363, 9.059855461120605, 12.142074584960938, 6.382768630981445, -2.3620362281799316, -3.9466195106506348, 6.70932149887085, 6.60435676574707, -7.092576026916504, 4.164920330047607, -1.7750142812728882, -14.2754487991333, -10.310787200927734, 2.4813334941864014, -14.0700044631958, -0.852357804775238, 5.533198356628418, 3.1899783611297607, -0.48877567052841187, -4.889301300048828, 7.501999378204346, 0.8261723518371582, -3.7865688800811768, -16.443771362304688, -3.484682559967041, -16.57242202758789, 7.75098991394043, 10.156423568725586, 11.978544235229492], [-8.100090026855469, 0.8941554427146912, -13.76686954498291, 0.7258313298225403, -7.0976409912109375, 4.989511489868164, 8.00943374633789, -5.458638668060303, 2.030184268951416, -3.5267550945281982, 2.5786044597625732, 0.753555178642273, 3.082496404647827, -14.735304832458496, -17.929840087890625, 8.069236755371094, 6.223727226257324, -7.979729652404785, -4.60943603515625, 6.333883285522461, -0.07152195274829865, 5.913464069366455, 1.145243763923645, -1.4025733470916748, -1.4343749284744263, 9.15283489227295, -1.9257848262786865, 3.818937063217163, 2.5290064811706543, 1.2767912149429321, -1.5048556327819824, -1.7231662273406982, -2.2474758625030518, -11.495587348937988, 4.096116065979004, -7.162829399108887, 8.41891860961914, -4.2329816818237305, -0.045276906341314316, 9.397900581359863, -9.740973472595215, -5.749624252319336, -6.790138244628906, 1.6624131202697754, 5.715377330780029, -7.613857746124268, 14.140573501586914, 1.5422683954238892, 4.182103633880615, 3.7050046920776367, -7.604425430297852, 4.512235164642334, -12.587706565856934, 6.101521015167236, -3.0364911556243896, -3.3214468955993652, -3.633751392364502, 0.6422430872917175, 0.1527993381023407, -3.0550994873046875, -6.546776294708252, 9.594060897827148, -2.2943227291107178, 15.441727638244629]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.6274216771125793, 0.08466003835201263, 0.7877344489097595, 0.27408039569854736, -0.0847887173295021, -1.8539637327194214, 1.2328466176986694, -0.7831361293792725, 0.718513011932373, -0.198934406042099, 1.610634446144104, 0.09951186180114746, 0.5836406946182251, -0.28434720635414124, -2.1903791427612305, 0.4204488694667816, 0.8060762286186218, -0.419689416885376, -0.7063706517219543, -0.20510198175907135, 0.504331648349762, -0.5120492577552795, 0.3452692925930023, 1.7551920413970947, -1.423267126083374, 0.0906490832567215, 0.06880832463502884, 0.943221926689148, 0.1307533085346222, 0.5273001194000244, 0.32064494490623474, -0.7722295522689819, 0.31727468967437744, -0.7536248564720154, -0.6881061792373657, -1.1265995502471924, 0.2760523855686188, 0.13997797667980194, 0.43208444118499756, -0.5705560445785522, 0.18242943286895752, 0.19329674541950226, 1.0715802907943726, 1.178445816040039, -0.5155948996543884, -0.47574883699417114, -0.4721270501613617, -1.3391733169555664, 0.2318645864725113, 0.3829006552696228], [-1.368687629699707, 0.9714089035987854, 0.6931559443473816, 0.14163820445537567, -0.8199588060379028, -0.15171270072460175, 0.999488353729248, 1.247326135635376, 0.6879739761352539, -1.656544804573059, -1.4228500127792358, -1.961264967918396, -1.4920092821121216, -0.3584320843219757, 1.5691202878952026, 0.12535831332206726, -0.22182737290859222, -0.766668975353241, 1.3096460103988647, -0.6694276332855225, 0.37670499086380005, -0.8605184555053711, 0.4404805600643158, 1.291680932044983, -2.0257554054260254, -0.1392512172460556, -0.3070557415485382, -1.4269810914993286, -2.882603883743286, 0.3764532506465912, 1.1919366121292114, 0.3149791359901428, 0.2643436789512634, 1.8051196336746216, -0.0900738388299942, 1.3201607465744019, -0.8783330917358398, -0.2350306510925293, -0.8851614594459534, 1.354245662689209, -0.3224785327911377, 0.22593247890472412, 1.0436338186264038, 0.2911122441291809, 0.04333319142460823, -0.018846873193979263, 1.496996521949768, -1.2778760194778442, -0.3990733027458191, 0.7934364676475525], [-0.7830122709274292, 0.8948994278907776, -1.2445485591888428, -0.6578305959701538, 1.1048699617385864, -1.7876793146133423, -1.1413166522979736, -0.1877613514661789, 0.256925106048584, -0.876937210559845, -2.1183598041534424, -0.8444182872772217, 0.39140915870666504, -1.0197423696517944, 0.13754361867904663, 1.3625527620315552, -0.7517143487930298, 0.5705992579460144, 0.524709165096283, -0.07421667128801346, 0.2682388722896576, 0.10309093445539474, 0.47108662128448486, -0.5571721792221069, -0.1459493637084961, -1.4875435829162598, 1.1899675130844116, -0.24379117786884308, -0.8414660692214966, -0.7522206902503967, -0.6285680532455444, -1.579418659210205, 1.9638028144836426, -0.8391250371932983, 1.8757725954055786, -0.2832161784172058, 0.10991424322128296, -0.05854390934109688, 0.8335139155387878, -0.32641705870628357, -0.14745676517486572, 0.04416980966925621, 0.3993046283721924, 0.8753138184547424, -0.984125018119812, -0.7528343200683594, 2.4675681591033936, -2.011098623275757, -1.14149010181427, 0.3181551992893219], [0.7331120371818542, -0.18234676122665405, 0.40257689356803894, 0.9837394952774048, -0.4083012342453003, -0.48897284269332886, 0.25955185294151306, -0.9741649627685547, -1.7278746366500854, -1.0627819299697876, 2.4599645137786865, 0.3230093717575073, 0.8506758213043213, 1.8583221435546875, 0.7717157006263733, -0.269192099571228, 0.8420374393463135, -0.7712479829788208, 1.4935493469238281, 0.11027023941278458, -0.5752870440483093, 0.5452777147293091, -0.8197746276855469, 1.802902102470398, 0.8709561228752136, -0.056198690086603165, -0.39086657762527466, -1.01860773563385, 0.11314605921506882, 1.2932757139205933, -0.3861905336380005, -0.17556330561637878, -1.9915541410446167, 1.1737197637557983, -0.3457319438457489, 0.5835152864456177, -3.504775047302246, -0.21079570055007935, -1.4568002223968506, 0.0416429378092289, -0.25463294982910156, -0.13486701250076294, 1.0314351320266724, -1.6331274509429932, 0.1887504607439041, 0.14344336092472076, 1.087834119796753, 0.6115335822105408, -0.710648238658905, 0.08214811980724335], [0.4428028166294098, 0.11445438861846924, 0.14922842383384705, -0.9915013909339905, -1.0009069442749023, -0.3968521058559418, -0.25617581605911255, 0.06802535802125931, 0.16200993955135345, 0.9837853312492371, 0.03514692187309265, 0.8494049310684204, 2.399710178375244, -0.9941948056221008, 0.469310998916626, -0.45928990840911865, 0.22624601423740387, -1.4148039817810059, -0.7691219449043274, 0.6946938037872314, -0.3218387961387634, -0.7782226800918579, 1.125823736190796, -0.5883173942565918, 1.2066677808761597, 1.108421802520752, 0.33879730105400085, 0.5421857237815857, 0.8545241951942444, -0.9472605586051941, -0.07859557121992111, 1.4431898593902588, -0.9809945821762085, -1.5941616296768188, 0.3876218795776367, 0.5273995399475098, 0.1244516372680664, -0.841217577457428, 0.8780772686004639, 0.47582516074180603, -0.005297964904457331, 1.6185657978057861, -0.8588000535964966, -1.540982723236084, 0.04572809860110283, 0.7080836892127991, -0.523844301700592, 0.37209731340408325, -0.40882056951522827, -1.6394331455230713], [-0.4013959467411041, 0.3875523507595062, -0.06235542148351669, 0.8515306115150452, -0.3281479477882385, 1.1066349744796753, -0.7991427183151245, 1.2126057147979736, 0.6813001036643982, -0.7751761078834534, 0.09884197264909744, 1.0233713388442993, -1.294045329093933, -0.6745168566703796, 0.7555966377258301, -0.05342482402920723, 1.973121166229248, -1.4720412492752075, -1.993195652961731, 1.6624634265899658, 0.8931323289871216, -1.1699563264846802, -1.005316972732544, 0.5275196433067322, 0.7709140777587891, 1.6584348678588867, -2.367661714553833, 0.151169091463089, 0.05525745451450348, -0.8494938611984253, 1.5373455286026, -0.7353658080101013, 0.46282780170440674, 0.99110347032547, -1.2521278858184814, 0.4341388940811157, 0.4626563787460327, 1.1297028064727783, -0.25725167989730835, 0.10989871621131897, -0.3685271739959717, 0.37108731269836426, 0.623542070388794, 0.3394700586795807, 0.1500227004289627, 0.9684004783630371, -0.9880155324935913, 0.07751750946044922, 1.0791009664535522, 1.5436136722564697], [-0.025693729519844055, -0.3480672538280487, -1.8785706758499146, -0.12339853495359421, 0.4324471950531006, -0.011142379604279995, -0.5196244120597839, 0.18989916145801544, -0.6231726408004761, -0.6472933888435364, -1.7184771299362183, -1.834174633026123, -1.6512439250946045, 1.5727665424346924, 1.1778713464736938, 0.5494465231895447, 0.9697062373161316, 0.039032772183418274, 0.9753648638725281, -2.765475034713745, 1.6246111392974854, 0.6078463792800903, 0.24316184222698212, 1.2785955667495728, 0.3076114058494568, -1.6787933111190796, 0.4515857696533203, 0.2884392738342285, 0.1583518534898758, 0.07519131898880005, 0.7596268057823181, -0.8804618120193481, 0.6106253266334534, 1.2967755794525146, -0.127366304397583, -0.9163560271263123, 0.958019495010376, -1.8776558637619019, 0.8503013253211975, -0.40627676248550415, -0.16109922528266907, 1.5345430374145508, -0.669140100479126, 0.02008708566427231, 0.7487393617630005, -0.10943739116191864, -0.1588602066040039, 0.23666483163833618, -0.8973785042762756, 1.1179561614990234], [-0.35374021530151367, 1.165406584739685, -0.2603283226490021, -1.255073070526123, -0.39751431345939636, -1.518183708190918, 0.6500459909439087, -0.11483955383300781, 1.6057051420211792, 1.109950304031372, 0.4797677993774414, 0.030357711017131805, -0.46939411759376526, 2.0652921199798584, -1.9461613893508911, 1.5767353773117065, 1.1824926137924194, 1.2020962238311768, 0.3457651138305664, 0.5219199657440186, -0.06329630315303802, 0.17626015841960907, -2.408572196960449, 0.020677288994193077, 0.5753668546676636, 0.10786470025777817, 0.9539623856544495, 1.8025566339492798, 0.2828853130340576, -0.076693095266819, -1.4457571506500244, 0.2710036337375641, -3.1868748664855957, -0.9416815638542175, 0.14300554990768433, -0.7334814071655273, 3.5024216175079346, -0.27101194858551025, 0.8078019618988037, 1.0946744680404663, 1.015445590019226, -1.0758310556411743, 0.3974570333957672, -0.4001525044441223, 0.936862587928772, 0.300266832113266, -0.14738936722278595, 0.09530230611562729, -0.35971081256866455, -0.36890652775764465], [-0.29731568694114685, 0.2614525556564331, -0.6658653020858765, -0.14675620198249817, 0.11162634938955307, -0.42485880851745605, 1.7321044206619263, -0.7288488149642944, -0.5976665019989014, -0.6212602257728577, 1.2345458269119263, 0.6541309952735901, -0.2840021848678589, -0.22895874083042145, 0.33149078488349915, -0.5864449739456177, 0.6250020861625671, -0.9664908647537231, 0.8919695019721985, 0.2411651462316513, -0.6631447076797485, -0.06998952478170395, 0.07919186353683472, -0.24243344366550446, -0.6401780247688293, 2.897150754928589, -0.7305206656455994, -1.1749401092529297, -0.13356934487819672, -0.3548020124435425, -0.42648500204086304, -1.8777410984039307, -0.6914088129997253, 0.47203725576400757, -0.6147512197494507, 0.8915473222732544, -0.5670289397239685, -0.26428142189979553, -0.38904932141304016, 0.6326841115951538, -1.598591685295105, 0.05372008681297302, -0.3818061351776123, 2.389986991882324, 0.790560245513916, 0.02087339758872986, 0.22063961625099182, 0.011563897132873535, 0.8431342840194702, -0.1690247356891632], [-0.7906821966171265, -1.7298915386199951, 0.3703993260860443, -0.3873215615749359, 0.48005011677742004, 0.46794408559799194, -0.7588824033737183, 1.2961832284927368, -0.2371681183576584, 0.46605420112609863, -2.025217056274414, 0.48531481623649597, -1.2495367527008057, -1.4933809041976929, -0.2227613478899002, -1.6915541887283325, -0.5288894176483154, 0.3582611680030823, 1.6955199241638184, 1.5296255350112915, -0.5123346447944641, -1.2072162628173828, -1.602449893951416, -1.1871247291564941, -0.4308823347091675, 1.6303961277008057, -0.6728568077087402, -0.34699973464012146, 1.5076723098754883, -1.1475423574447632, 1.3054331541061401, -0.184299036860466, -0.33321595191955566, 0.25534942746162415, 0.323814332485199, -0.20014025270938873, 0.7483678460121155, 0.6057931780815125, 0.5567761063575745, 0.6125465035438538, -0.019374342635273933, 0.12317516654729843, -1.2343685626983643, -1.0342930555343628, 2.239614248275757, -0.5443294048309326, -0.464213103055954, -1.3587526082992554, 0.11846430599689484, 0.7628320455551147], [0.6897096633911133, 0.4541633129119873, -0.5219942331314087, 0.5749796032905579, 0.08092738687992096, -1.8905524015426636, 0.12946096062660217, -0.795691192150116, 0.876117467880249, -0.965563178062439, 0.22242802381515503, 2.1121909618377686, -0.2681804895401001, -0.45168253779411316, 0.3909599483013153, -0.6796768307685852, -0.45510947704315186, 0.7358618974685669, 0.3684535324573517, 1.963731050491333, -2.7468631267547607, -0.5919743180274963, 1.8255022764205933, -1.33836829662323, -0.29110202193260193, -0.6534792184829712, 0.3946925401687622, -0.6542317271232605, 0.4116052985191345, 0.24375411868095398, 0.6444031596183777, 0.0960843414068222, -0.021678855642676353, 0.1376919448375702, 2.580695629119873, 1.2006464004516602, -0.6692975759506226, 0.33376017212867737, -1.797295331954956, 0.6805500388145447, -0.8642133474349976, -1.2639524936676025, -0.8579505681991577, 0.8367854356765747, -0.29631391167640686, 0.39664584398269653, -0.471000075340271, -1.1254937648773193, 1.3735277652740479, -0.5699760913848877], [2.0307912826538086, -0.5181184411048889, -0.7347143292427063, 1.4476226568222046, -0.801591694355011, 0.6234304904937744, -0.7080742716789246, 1.096811294555664, 0.668565571308136, 0.5007023215293884, -0.833146870136261, -0.6031514406204224, 0.7954819202423096, 0.6557971239089966, 0.17487530410289764, -0.3780059516429901, 0.6004705429077148, -1.320898413658142, 0.7263570427894592, 0.1884603649377823, 1.0206917524337769, -0.02340555563569069, -1.3509520292282104, 0.487903356552124, 1.1186827421188354, 1.1428918838500977, 1.0966683626174927, -0.9134544134140015, -0.5301273465156555, -1.3573004007339478, 0.3749430477619171, -0.08911184966564178, 1.095109462738037, 1.1126002073287964, 0.5044218301773071, 1.212997555732727, -1.26912522315979, 0.4327874481678009, -1.7029420137405396, -1.1820414066314697, 0.5227194428443909, -0.3699382543563843, 0.5323494672775269, -0.20534023642539978, 1.142952799797058, 0.7070254683494568, 0.533462643623352, -0.3697056472301483, 0.39126044511795044, 1.393442153930664], [-1.655791163444519, -0.34195631742477417, 0.2569615840911865, -1.349711298942566, -0.3539242744445801, -0.14636021852493286, -0.1834048330783844, 0.509168267250061, -0.008046554401516914, 1.092179536819458, 0.7516171932220459, -1.2730823755264282, -1.4759609699249268, -0.3299793004989624, -1.0614757537841797, 0.6830105185508728, -0.4919959008693695, 0.3756619095802307, -0.7464702129364014, -0.32435861229896545, -1.7971014976501465, 0.8097833395004272, -0.2644459307193756, -0.9357908368110657, -0.34964194893836975, 0.010222095996141434, 0.2158871591091156, 0.6927841305732727, -1.5010523796081543, 0.7096003293991089, -0.3804777264595032, -0.4307766258716583, -0.593876838684082, 0.5103958249092102, 0.8083558678627014, -0.21477197110652924, 1.1386992931365967, 1.084486961364746, 1.5124977827072144, -1.1040583848953247, -0.21281521022319794, -0.12657271325588226, 0.6624929308891296, 0.7831548452377319, 1.6469188928604126, -0.5766653418540955, 0.4067332446575165, 0.3568185269832611, -0.6183897852897644, -0.7346031069755554], [0.0618523508310318, 0.04617234319448471, -0.023418275639414787, 0.9478952288627625, -0.5268567800521851, -0.35347980260849, -1.6948401927947998, 1.9416284561157227, -0.20250436663627625, -0.7622822523117065, -1.7296257019042969, 1.1653269529342651, -0.8368384838104248, 0.24921099841594696, -0.5936942100524902, 0.42942216992378235, -0.49277564883232117, 0.7495878338813782, -1.9236944913864136, 0.2519759237766266, 0.31524038314819336, -1.317962884902954, -1.148329734802246, -0.15762309730052948, -0.35989636182785034, -0.3551148772239685, -0.36466169357299805, 0.09277858585119247, -0.04381462559103966, 0.06059868261218071, -0.9331406354904175, 0.5849217176437378, 0.33233362436294556, -0.4745466113090515, 0.33159375190734863, -0.41086506843566895, -0.5098278522491455, -0.7070890665054321, -0.40774837136268616, -0.25241878628730774, -0.30287960171699524, 0.14516517519950867, 1.435920238494873, 0.25663256645202637, 0.9571853876113892, 0.058863915503025055, 0.4589698612689972, -1.1837937831878662, 0.97478848695755, 0.7045741677284241], [-2.0851237773895264, 2.247959613800049, -0.31601059436798096, -0.00483382074162364, -0.4929744601249695, 0.2878357470035553, 1.8666658401489258, 1.212487816810608, 2.276683807373047, -0.9004456400871277, 0.5082778334617615, -1.190260887145996, -1.1465421915054321, -0.6600198745727539, 0.3060220777988434, -1.6713829040527344, 0.3343709409236908, 0.49994027614593506, 0.8361988067626953, 0.8543522953987122, 0.1832198053598404, 0.4654181897640228, -0.07948450744152069, -0.6705971956253052, 1.0727055072784424, -0.35035109519958496, 2.059586763381958, -1.5250961780548096, -0.6563708186149597, 0.3346044421195984, -0.20921850204467773, -0.5094965696334839, 0.22580952942371368, -0.24092601239681244, 0.4076859652996063, 0.48106369376182556, -0.42685291171073914, 0.9614667892456055, -1.0386956930160522, 1.4144952297210693, 0.1775701344013214, -1.642338752746582, -0.5636554956436157, 1.2857235670089722, -0.23231083154678345, -0.2838238775730133, -0.7106814384460449, 1.555879831314087, 0.6290816068649292, 0.8635035753250122], [0.6320793628692627, 0.9925686717033386, -0.07626958191394806, 1.5268008708953857, -0.1836962252855301, -0.34944862127304077, 1.5217477083206177, 0.7576138377189636, 0.18497706949710846, 0.18296615779399872, 0.430911123752594, -0.346454918384552, 0.20701929926872253, 1.839714765548706, -0.8797217607498169, 0.15448184311389923, -2.1062228679656982, -1.0232036113739014, -2.2723388671875, -0.050943415611982346, 0.9835307002067566, 1.9639606475830078, 0.6108376979827881, 1.0338249206542969, 0.30229389667510986, 0.6522290706634521, -0.5625874400138855, 0.48726320266723633, 0.6667070388793945, 1.512251377105713, -0.44452041387557983, -0.10563468933105469, 0.20168229937553406, 1.5431969165802002, -1.653435230255127, 0.631168782711029, -1.601271629333496, 0.9328005909919739, 0.011858616024255753, 0.3255534768104553, -0.029811806976795197, 1.1532114744186401, 0.9573513269424438, 0.38719868659973145, 0.36416521668434143, 0.3037361204624176, -0.15841282904148102, 0.1803804188966751, -1.3998301029205322, -1.085870623588562], [-1.046012282371521, -0.380049467086792, -0.31885823607444763, -0.1930316984653473, 0.6369385719299316, -1.2147197723388672, -1.9432404041290283, -0.6381719708442688, 0.8904784917831421, -1.1936715841293335, -0.5795546174049377, -0.15438753366470337, -2.3017261028289795, 0.46516692638397217, -0.9372819662094116, 0.6125259399414062, -2.1428489685058594, 0.5147833824157715, 0.5817719101905823, -1.549482822418213, -0.9369655251502991, 0.8659062385559082, -0.09466435760259628, -0.6191813945770264, -1.1959352493286133, -0.19818541407585144, -0.3258768618106842, -0.3385123610496521, 0.6982588171958923, 0.15565600991249084, 0.27395835518836975, 0.03564343973994255, -1.5092726945877075, 0.47298407554626465, -0.19175176322460175, 0.21600410342216492, 2.0066096782684326, -1.1805716753005981, 0.7713542580604553, -0.6478456258773804, -0.3402782678604126, 0.6485404372215271, -0.6908904910087585, 0.3889111280441284, 0.43613535165786743, 0.3139052093029022, -0.9489508867263794, 1.7678462266921997, 0.36837348341941833, 0.32445114850997925], [-1.3103759288787842, -0.19733034074306488, 1.1828004121780396, -0.7414742708206177, 2.429218053817749, 1.6072396039962769, 0.16989947855472565, -0.6591107845306396, -0.5674992203712463, -0.6696877479553223, -2.0946156978607178, -1.2725129127502441, 0.07706911861896515, -0.23921658098697662, 1.2693694829940796, 2.1179606914520264, -0.34441885352134705, 1.1109241247177124, -0.9963061213493347, -0.30084553360939026, 1.5165934562683105, 1.2198725938796997, -1.5669554471969604, 0.11534205079078674, -0.14847233891487122, -1.2987786531448364, -1.2459903955459595, 0.8816453814506531, 1.4991542100906372, 0.7226612567901611, 1.5997103452682495, 0.1239648386836052, -1.238966703414917, -2.1020853519439697, -0.13248983025550842, -0.33452415466308594, 0.36393317580223083, -0.6625156402587891, 1.8128275871276855, -1.2456130981445312, 0.008934574201703072, -0.539583683013916, 1.0628297328948975, 1.4026765823364258, -1.565930724143982, 0.3507760763168335, 0.34096065163612366, 1.007912516593933, 0.1983497589826584, 0.4166507422924042], [0.39995044469833374, -1.6476720571517944, -0.15703269839286804, 1.1427503824234009, 1.3319694995880127, -1.5017744302749634, 0.04571346938610077, 0.31842145323753357, -0.6190908551216125, -0.022151941433548927, 0.03931998088955879, -0.9193094372749329, -1.0574891567230225, -0.6602388024330139, -0.8455770015716553, 0.383355051279068, 0.6390817761421204, -0.7467887997627258, 0.5564128756523132, -0.7617822885513306, 0.5195606350898743, 1.4637099504470825, -0.8288904428482056, -1.33572256565094, -0.05119921267032623, 0.16280649602413177, -0.610489010810852, -0.47634223103523254, 0.27105993032455444, 0.422794908285141, -0.29202479124069214, 0.31401559710502625, -0.23956601321697235, 0.594208836555481, 0.24281421303749084, -0.8281471729278564, 1.2230651378631592, 0.4804796278476715, -1.0673390626907349, -2.2812180519104004, 1.0948694944381714, 1.0977234840393066, 0.8312716484069824, -0.4965282082557678, -0.22355429828166962, -0.9065014719963074, -0.9994988441467285, -0.42915844917297363, 1.047566533088684, -1.68873929977417], [0.24537405371665955, -0.47956204414367676, -0.4725286662578583, 0.01658451370894909, -0.32075533270835876, -0.5889206528663635, -0.024810487404465675, 2.004941463470459, -1.2127068042755127, 0.9816377758979797, 2.047232151031494, 0.5887895822525024, 1.7993335723876953, 1.4726470708847046, -0.04559678956866264, 1.2830588817596436, 0.15955233573913574, 0.38603177666664124, 0.03940875828266144, 0.8889411091804504, 0.4531940519809723, -0.43605950474739075, 1.0766710042953491, -0.1663040816783905, -0.010906241834163666, -0.7029040455818176, -0.0795462504029274, -1.073525309562683, 0.6311545372009277, 1.1357470750808716, 0.30986133217811584, -0.741438627243042, -1.043400526046753, 0.6526889801025391, -1.7729263305664062, -1.5826436281204224, -1.639797329902649, -0.4701351523399353, -0.13663412630558014, 0.08783397823572159, 0.6230825781822205, 0.1890316754579544, -0.5333746075630188, 0.891739010810852, -0.3436185121536255, -0.2348867803812027, 0.3059749901294708, 0.3113652467727661, -0.03774319589138031, -0.9212104082107544], [0.22551082074642181, -0.6142206788063049, 1.724981427192688, 0.992977499961853, 0.795843243598938, -1.057796835899353, -0.2867259383201599, -0.6577091813087463, 1.9811939001083374, 0.008307975716888905, 1.455371379852295, 0.7955187559127808, 1.6784228086471558, 0.6064637899398804, 0.6670189499855042, 0.14989019930362701, 0.114576056599617, -0.4645141661167145, 0.614584743976593, -0.2342204749584198, -0.6281680464744568, 1.0112498998641968, -1.733048439025879, 0.5460717678070068, -0.3747817277908325, 0.7317501306533813, 2.885166883468628, -0.22755779325962067, 0.4159937798976898, -0.1530277132987976, 1.9885122776031494, -1.0402536392211914, 1.1822633743286133, -0.558220386505127, -0.8568907380104065, 1.621021032333374, 1.459665298461914, -1.1955020427703857, -0.6233923435211182, 1.6146092414855957, 1.1520335674285889, 2.9631166458129883, -0.33371251821517944, 1.595618724822998, -2.0284550189971924, 0.0706343948841095, -0.2500004768371582, -0.23961152136325836, 0.5963101983070374, 0.5696661472320557], [-0.27813082933425903, -0.5017542243003845, 0.4942858815193176, -0.6473421454429626, -0.31296998262405396, -0.2873844504356384, -0.8899339437484741, 0.3554638922214508, -2.1193110942840576, 0.9468868970870972, 0.3704964220523834, -0.5024011135101318, 0.03281736373901367, 0.027562838047742844, 0.711062490940094, -0.6961373686790466, 0.23219583928585052, 0.10049886256456375, -0.5824981331825256, -0.06295491009950638, -0.011032095178961754, -1.32416570186615, 0.6971244215965271, -0.502356767654419, -0.9117842316627502, 0.6240474581718445, -0.8571675419807434, 1.1825950145721436, 0.7077697515487671, -0.9739086627960205, 0.037996791303157806, 0.7390015125274658, 0.18862442672252655, -1.1274080276489258, -0.408864289522171, 0.5300203561782837, -0.5944549441337585, 0.6292545795440674, 0.039952944964170456, 1.8021734952926636, 0.23834089934825897, -1.5217788219451904, -0.5250245928764343, -1.8766517639160156, -2.067502737045288, -0.4114660322666168, 0.06543748080730438, 1.2862404584884644, -0.4959145486354828, -0.47230538725852966], [0.7498611807823181, -0.38707104325294495, -0.42925968766212463, 1.3758251667022705, 0.20018370449543, 0.46565738320350647, -0.591581404209137, 1.3548136949539185, -0.23642587661743164, -0.5239212512969971, -2.3787779808044434, 0.841700553894043, -0.9737821221351624, -0.004269395489245653, 1.1836862564086914, 0.425944983959198, -0.4111568331718445, 0.2588263750076294, -0.17363569140434265, -0.23035627603530884, -0.5608566999435425, 2.288299560546875, 0.6097890138626099, 0.5136327743530273, -2.0444416999816895, -1.0736479759216309, 1.4703432321548462, 1.6565147638320923, 0.2569428086280823, 1.6891191005706787, 0.245836079120636, 1.064157485961914, 0.19393561780452728, -0.5258048176765442, 1.9138299226760864, 0.7603946328163147, 1.2873088121414185, -1.1147997379302979, 0.743167519569397, -2.0173325538635254, 0.6365676522254944, 0.28782665729522705, 0.3563290536403656, -0.13835713267326355, -1.000239610671997, 1.056189775466919, -0.6122055649757385, 0.546992301940918, -0.9095592498779297, -1.183849811553955], [-0.5486230254173279, -0.46883517503738403, 1.0126723051071167, -1.3504260778427124, 0.5611757040023804, -2.388819694519043, -0.6669646501541138, 0.37408795952796936, -0.0035603721626102924, 0.7675009369850159, -0.45495089888572693, -0.19065777957439423, 0.13837258517742157, 0.18082746863365173, -0.9429802894592285, 1.0615415573120117, -0.49722176790237427, 0.8265931010246277, 0.9726518988609314, 1.7286027669906616, 0.23107203841209412, -0.34627389907836914, -1.023209571838379, -0.03880538046360016, -0.8322644233703613, 0.37757980823516846, 1.2340370416641235, -1.8434898853302002, 0.4057851731777191, -0.9233013391494751, -1.995843768119812, -0.8945567011833191, 1.3417165279388428, -0.5182885527610779, -0.6638827323913574, 0.8526543378829956, -1.1083029508590698, 0.438711017370224, 1.2232367992401123, -0.22073809802532196, -1.1348888874053955, -0.38604822754859924, 0.480535089969635, 0.6547544598579407, -1.0953991413116455, 0.4306015074253082, -1.8433958292007446, 0.2731586694717407, 0.5824980735778809, -1.301811695098877], [1.4590098857879639, -0.47412627935409546, 0.2297949194908142, -0.9377191066741943, 0.8888692855834961, 2.351252555847168, -0.032371170818805695, 0.84134441614151, 2.3118679523468018, 1.8590168952941895, 0.3589613437652588, -0.015261951833963394, -0.19613538682460785, 1.0285043716430664, -0.593228816986084, 0.2908920645713806, 0.02233685739338398, -1.2503268718719482, 0.34999099373817444, -1.744486927986145, 1.6570168733596802, -0.4155905544757843, -0.6745190024375916, 0.0989280417561531, -1.2235997915267944, 0.8240934610366821, -2.1527137756347656, -1.494651198387146, -1.7350306510925293, 1.3598837852478027, 1.3129359483718872, -0.36534255743026733, 1.4136089086532593, 0.040353044867515564, -0.5983055830001831, -1.5920838117599487, -1.3029334545135498, 0.5987251996994019, 1.1210252046585083, 0.6076670289039612, -0.08192464709281921, -1.0457857847213745, 0.6094587445259094, -0.6243178248405457, 0.15860426425933838, -0.16054855287075043, -0.4678371846675873, -1.3134084939956665, 0.37447506189346313, -1.0203135013580322], [-2.293487548828125, -1.4592865705490112, 0.6943454146385193, -1.2886598110198975, 1.774776816368103, -0.9496752023696899, -0.4778221547603607, -1.0147989988327026, 0.7118127942085266, 0.5202846527099609, 1.697677493095398, -0.07889724522829056, -0.032107479870319366, 0.7798603773117065, 0.5067064762115479, -0.03405475243926048, 0.9281623959541321, -0.9530957937240601, 0.44269055128097534, -0.48708370327949524, -0.15298141539096832, 0.6289390921592712, -1.6282991170883179, -2.255650758743286, 1.864202618598938, -1.7877222299575806, 0.2159481793642044, 1.1834877729415894, -0.7564406991004944, 0.8930180072784424, -0.9939424991607666, 0.0158368106931448, -0.7366138696670532, 1.9028563499450684, 1.1732242107391357, -0.8264380097389221, -0.22005553543567657, -1.90887451171875, 0.2719675302505493, 0.7719283103942871, 0.28740599751472473, -1.6926237344741821, -0.0352625846862793, -0.14450190961360931, -0.47659948468208313, 0.3990698456764221, -0.16987749934196472, 0.5267492532730103, -1.3654776811599731, -0.0027886496391147375], [-0.7167490124702454, 1.0498895645141602, 1.0799566507339478, 1.4949480295181274, 1.1384968757629395, -0.020352160558104515, 0.8184946775436401, -0.18751879036426544, -1.0457475185394287, -0.06326628476381302, -1.6830649375915527, 0.4560244679450989, -0.787516176700592, 0.5142949819564819, 0.482933908700943, -1.1486035585403442, 0.3657870292663574, -0.1751696914434433, 0.6133679747581482, 1.2280975580215454, 1.339880108833313, 0.590613067150116, 0.7399211525917053, 0.6339571475982666, 0.21029166877269745, -0.28103700280189514, 1.0412908792495728, 1.3513271808624268, 0.8156242966651917, 0.5447571277618408, 2.4478600025177, -1.1281225681304932, -1.3564373254776, 2.18517804145813, 0.12829774618148804, 2.1655378341674805, -1.3209643363952637, -1.025992751121521, 0.38596588373184204, 1.0911152362823486, 0.14789144694805145, 0.3341987729072571, 1.8821842670440674, -0.7849523425102234, 1.619385838508606, 0.6494847536087036, 0.4603360593318939, 0.05170300230383873, -2.9274725914001465, -1.3213926553726196], [1.0671437978744507, -0.7827229499816895, -1.5932724475860596, -0.8070520758628845, -0.21402499079704285, 0.9845767617225647, -0.18095402419567108, 0.9876746535301208, 0.1742098331451416, -0.9830872416496277, -1.2005994319915771, 0.416786789894104, -1.2439771890640259, -1.0895946025848389, -0.6373493075370789, 1.964850902557373, -1.6032249927520752, 0.4819287955760956, -0.8626992106437683, 0.713921308517456, 1.495184063911438, -0.8800328373908997, -0.15157482028007507, 0.31777796149253845, -0.1330752819776535, -0.6568654179573059, 0.13129840791225433, -0.22730126976966858, 0.23384790122509003, 1.1829818487167358, 0.6663560271263123, -0.3393383026123047, -1.5820897817611694, -0.23469054698944092, 1.289858102798462, 0.9872974753379822, 0.9458084106445312, -1.1540807485580444, 0.42903488874435425, 0.5264729261398315, -0.851003885269165, -0.21590164303779602, -1.5926618576049805, 0.8758953809738159, -0.6763713955879211, -1.0320826768875122, 0.880172073841095, -0.23720794916152954, 1.5205624103546143, 1.438598394393921], [-0.7290539741516113, -0.629641056060791, 0.2290888875722885, -0.040568407624959946, -1.3812999725341797, 1.1153041124343872, 1.523991584777832, 0.5231077075004578, 0.4822549819946289, -0.6142449378967285, 0.9366973638534546, -0.03981854021549225, -1.2142823934555054, -1.015007734298706, 0.022920995950698853, -0.17328397929668427, -0.3020193874835968, -1.0452364683151245, -0.0009310705936513841, 1.635385513305664, 0.23280471563339233, -2.0293173789978027, 0.2151399850845337, 0.3586718738079071, 2.259753465652466, -0.5638220310211182, -1.2175114154815674, -0.31055039167404175, 1.044028639793396, 1.4563978910446167, -0.850849986076355, -0.5802187323570251, -0.87058025598526, 1.395035982131958, -0.7424542307853699, 2.5519866943359375, 2.3626046180725098, -2.18424654006958, 1.301727294921875, 0.6697015762329102, 0.8857160806655884, 0.37592458724975586, 0.09066569805145264, 0.7104489207267761, -0.9417110681533813, -0.7800290584564209, -0.16083765029907227, -0.12935562431812286, 0.36098429560661316, 0.14998751878738403], [0.5781201720237732, -0.11210702359676361, 1.1899465322494507, -0.5193714499473572, -0.28608566522598267, -1.1201881170272827, -0.24216075241565704, -0.1088467687368393, 0.8225898742675781, 1.7807565927505493, -0.10986609756946564, -0.4564794600009918, -2.433884620666504, -0.29030346870422363, 0.011152645573019981, -0.47612428665161133, -1.0133696794509888, -2.8158090114593506, -0.8584591746330261, -0.5028923749923706, -0.23903845250606537, -0.7662497162818909, 0.18980440497398376, -0.7481143474578857, 0.8439516425132751, -0.6946173906326294, -1.0148955583572388, 1.3993395566940308, -0.6044689416885376, -0.03763238713145256, 0.4180136024951935, 1.1199429035186768, -1.316441535949707, -1.2099289894104004, -0.23620226979255676, 1.867087960243225, -0.4139779210090637, 1.7071934938430786, 0.050858475267887115, -0.4118799865245819, 1.5819721221923828, -0.26330214738845825, 0.04848046228289604, -0.1062268391251564, -1.2444231510162354, -0.7558042407035828, -1.3847763538360596, 0.9081651568412781, 0.4506453275680542, -0.9880053997039795], [1.4760414361953735, 0.4945303797721863, 0.15345796942710876, 0.8138575553894043, 0.9904208183288574, -0.4904859960079193, -0.45201244950294495, -0.05590934306383133, -0.5207340121269226, -0.35885974764823914, -0.19906100630760193, -0.5086296796798706, 0.3509117066860199, -0.37855982780456543, -0.9754544496536255, -0.8601492047309875, -1.403206706047058, 0.031749654561281204, 0.009146506898105145, 2.486551284790039, 0.3648291528224945, 1.0127689838409424, -1.157333254814148, -0.2489713579416275, -1.141109824180603, -0.12720124423503876, -0.4888162314891815, -0.12409133464097977, -0.6772556304931641, 0.4911109507083893, -0.46938082575798035, -1.5954704284667969, -1.0916969776153564, 1.049166202545166, -0.4309261441230774, 2.496722936630249, 0.2186988741159439, 0.007981534115970135, 0.8114772439002991, 0.8892413973808289, -0.31849223375320435, 2.2295584678649902, 0.9087187647819519, -1.1460635662078857, -2.0925583839416504, -0.6374803781509399, 0.4999361038208008, 0.9952759742736816, -0.31030797958374023, -0.0843808650970459], [-0.5057461857795715, 0.07425273954868317, 0.9176135659217834, -1.0293443202972412, -0.5989026427268982, 0.5474258661270142, -1.9600111246109009, 0.4254760444164276, -0.19175629317760468, 1.0131045579910278, -2.2749276161193848, -2.917025089263916, -0.8210232853889465, 1.37603759765625, 0.34337925910949707, -1.7004448175430298, -2.0723793506622314, 0.45838692784309387, -0.4000222682952881, 0.26036208868026733, -0.7057755589485168, -0.9359472990036011, 1.4378870725631714, -1.454997181892395, 0.5169116854667664, 0.7082180976867676, 0.5441598296165466, 2.1101009845733643, -0.2303074151277542, 0.5432922840118408, 1.6747808456420898, 1.7676349878311157, 0.5204547643661499, 0.2561796009540558, 0.8937281370162964, -1.938092589378357, 0.8747254014015198, -0.7246880531311035, 1.4892653226852417, 0.4304664134979248, 0.18193981051445007, 0.49992311000823975, -0.03331895172595978, 0.6717065572738647, -0.8453133702278137, 0.11327265202999115, -2.528353452682495, 0.327860951423645, 0.43496617674827576, 0.7671352028846741], [-1.2511180639266968, 0.7978135943412781, -1.2221322059631348, -0.12821662425994873, 0.25836390256881714, 0.1772363930940628, 0.13047447800636292, -0.5655919909477234, 1.5663247108459473, 1.6184983253479004, 0.9287732839584351, -0.9372379183769226, 1.1973631381988525, 0.73077791929245, -0.7916011810302734, 2.3818140029907227, -0.2574232518672943, 2.0181491374969482, -0.2835875451564789, -0.58594810962677, -0.8018880486488342, -0.21033374965190887, 1.0300699472427368, 0.7040505409240723, 1.8629831075668335, 1.3050166368484497, 1.1507741212844849, -0.5617406368255615, -0.6212443709373474, 0.7833542823791504, -1.6771180629730225, -0.04686838015913963, 0.5592342615127563, 1.5193835496902466, -0.6919646263122559, 0.849307119846344, -0.05948216840624809, 0.9557867050170898, 0.8924364447593689, 0.7844741344451904, 1.0767223834991455, 0.553315281867981, -0.23314566910266876, -0.9669280052185059, 0.6020478010177612, 0.6749427914619446, -0.0870242565870285, -0.1468338966369629, 0.4668603539466858, -0.7704951763153076], [-2.190058946609497, 0.4422679543495178, -0.32343336939811707, 1.1931042671203613, 0.7644243240356445, 0.13322992622852325, -0.3025968074798584, -0.9355876445770264, 0.8832448720932007, -0.6921591758728027, -2.0154807567596436, 0.2400314211845398, 0.7691492438316345, -1.8724137544631958, 0.8164538145065308, 0.7884538173675537, 0.5607001185417175, 0.42620009183883667, -0.8823024034500122, 0.3935759961605072, 0.7343646287918091, -0.34900471568107605, 2.181178331375122, -1.054702877998352, 0.23572613298892975, 0.08083920925855637, 2.231952428817749, 0.1352771818637848, 0.15125544369220734, 0.5384592413902283, -0.12882837653160095, 1.9336321353912354, -0.7770941257476807, 0.8527229428291321, -0.06077426299452782, 0.47470328211784363, 1.1354396343231201, 0.7727139592170715, 0.020697975531220436, -0.698701024055481, 1.3065861463546753, 0.8491582274436951, 0.10972024500370026, -0.27805525064468384, 0.7810330390930176, -0.722565770149231, 0.8348882794380188, -0.48873379826545715, 0.31414130330085754, -1.403994083404541], [1.2423518896102905, -0.7878615856170654, 1.2644106149673462, 2.1792001724243164, -1.2703393697738647, -0.20454752445220947, 1.5169167518615723, -0.8696569204330444, 0.3555465042591095, -0.5720736384391785, -0.6776219010353088, -0.3144794702529907, -0.61115562915802, -0.6545351147651672, -0.8998702764511108, 0.6630635857582092, -0.6618275046348572, 0.34476011991500854, 0.8020198941230774, -0.08116172254085541, 0.6307632327079773, 0.42705389857292175, 0.7319315671920776, -2.2914109230041504, 1.4049094915390015, -0.5965367555618286, 0.6321814656257629, 0.8641690611839294, 0.05665013939142227, 0.8592253923416138, -0.7000479102134705, -0.06840759515762329, -0.8181267976760864, 0.906319260597229, -0.951178252696991, -0.9273601174354553, 1.2272168397903442, 0.42732512950897217, 2.336442708969116, 0.7126463055610657, -0.5009744763374329, -0.7623293399810791, 0.5832525491714478, -0.46329280734062195, -1.046691656112671, 0.06855952739715576, 0.050287581980228424, 0.009722636081278324, -0.30931127071380615, 0.840823769569397], [0.6186749935150146, 2.9563381671905518, -0.4779447615146637, -0.36266523599624634, 0.9538794159889221, -0.7984486222267151, 0.5484716296195984, -1.1466643810272217, 0.36299383640289307, -0.6900646686553955, 0.7440642714500427, 0.09887780249118805, -0.9117841720581055, 1.2854957580566406, 0.32594186067581177, -0.5143758654594421, -0.10970050096511841, 1.9338587522506714, 0.13062772154808044, -0.9527354836463928, -1.0042357444763184, 0.779555082321167, 0.2562841475009918, 1.7207355499267578, 1.5289522409439087, 0.01931508257985115, 0.6135551333427429, -0.9839514493942261, 0.2949317991733551, 0.48228979110717773, 1.2539350986480713, 1.6266896724700928, 0.060142338275909424, -0.805250883102417, 0.3739851415157318, 0.49823814630508423, -1.0536495447158813, 1.7895424365997314, 0.08833375573158264, 0.9579775929450989, 0.008158874697983265, -0.6285070776939392, -0.3025085926055908, 0.44748565554618835, 1.5455667972564697, 0.2730717062950134, -0.7107659578323364, -0.21899980306625366, 0.2523462176322937, 0.2283688634634018], [-0.7244232892990112, -0.2998126149177551, 0.14243227243423462, -1.675187110900879, -0.06909922510385513, -0.670311450958252, 1.3685249090194702, 1.2345280647277832, 0.08162564784288406, -0.6995822787284851, -0.8510111570358276, -1.7373226881027222, 0.18421444296836853, 0.8041709065437317, 0.5939607620239258, 1.897727131843567, 0.4184357523918152, 0.4841715395450592, 0.4878360331058502, -0.7673704624176025, -0.08441504836082458, 1.648119330406189, -0.9302037358283997, 0.2596513032913208, -0.20464752614498138, -0.011036919429898262, 1.249566912651062, -2.9285430908203125, -1.4099799394607544, 0.9703565239906311, 1.2498377561569214, 0.0960502028465271, 1.2005589008331299, 1.2827171087265015, -0.923395574092865, 0.9023100137710571, 0.5421391129493713, 0.9036728143692017, -0.012314614839851856, 0.6383993625640869, -1.4864287376403809, -0.21636316180229187, -2.166172504425049, -0.299807608127594, -0.6240342259407043, 0.9135182499885559, 0.04969849810004234, 0.043113965541124344, 0.319758802652359, 0.42557621002197266]], "dtype": "float32"}, {"__nd__": [[-1.2741751670837402, 0.5601800084114075, 0.6685380935668945, -0.34736931324005127, -0.5881518721580505, -0.7391358613967896, 1.3067716360092163, 0.49061319231987, 0.6066668629646301, 0.9427269101142883, 0.7841540575027466, -0.7326277494430542, -2.221729040145874, -1.3933721780776978, -0.19684293866157532, -1.38802969455719, -0.4536024034023285, -1.4429752826690674, -0.25842535495758057, 0.5684420466423035, -0.4794972538948059, 0.28888586163520813, 0.8444748520851135], [-0.10693977773189545, 0.7902222275733948, -1.362669587135315, 0.3495943546295166, 1.1391942501068115, 0.2039412409067154, -0.5137237906455994, 1.0365371704101562, -0.802960991859436, -0.37197163701057434, -0.7003905773162842, 0.782568097114563, 0.82816082239151, 0.2526412010192871, -0.7622161507606506, -0.8097622394561768, 1.1868655681610107, -1.2122241258621216, -0.24934031069278717, 0.6182247400283813, -0.15432019531726837, 0.4680716395378113, 1.4069738388061523], [-1.9710230827331543, 0.43370893597602844, -0.38506805896759033, -1.9124606847763062, 0.8763430714607239, -0.8437200784683228, 0.55704665184021, -0.9004959464073181, 2.2755110263824463, 0.3303315043449402, 0.007754487916827202, -0.38463443517684937, -0.542960524559021, 0.31955134868621826, 0.8220008611679077, -0.6058735251426697, 0.9675489664077759, 0.04090017452836037, -0.6012097597122192, 0.054531604051589966, 0.8022439479827881, 1.808021903038025, 0.0043279631063342094], [0.0932079628109932, -0.9543783068656921, -0.2743132412433624, 0.11659218370914459, -1.551066279411316, 1.4290943145751953, -0.628141462802887, 2.538691520690918, -0.6619682312011719, -1.7777000665664673, -1.3456604480743408, -0.15915927290916443, 0.1812218278646469, -0.1956404745578766, 0.919143795967102, -0.13380366563796997, 1.9257389307022095, -0.2753399610519409, 1.2516300678253174, 0.54258793592453, 0.08171642571687698, 0.06056073680520058, 0.6979735493659973], [-0.12746721506118774, -0.1550859808921814, -0.7780192494392395, 1.1662932634353638, -0.5500727891921997, 1.0066360235214233, -0.7681631445884705, 1.0987285375595093, -0.09889061003923416, -2.0149006843566895, -1.5452839136123657, 0.9526420831680298, 0.7385250926017761, 0.6519224643707275, 0.5327005386352539, 1.7647358179092407, 0.13703958690166473, 0.8923711180686951, 1.0325103998184204, -0.3323957920074463, 1.8853918313980103, 0.9937902688980103, -0.05146826431155205], [-1.2629133462905884, 0.39538413286209106, -1.0789943933486938, 1.0775141716003418, 0.703597366809845, 0.2418380081653595, -1.5629578828811646, 0.053305841982364655, -0.9091019034385681, -0.40730535984039307, 1.863402247428894, 0.2954445779323578, 0.4357004463672638, -0.7912507057189941, -1.1869391202926636, 2.2002015113830566, 0.1828555315732956, -1.5433483123779297, 1.4324711561203003, -1.1263045072555542, 0.05158854275941849, 0.20802997052669525, -0.4820028841495514], [0.9055671095848083, -1.6670939922332764, -0.8299764394760132, -0.4614707827568054, 0.2915198802947998, 2.239311456680298, -1.3936710357666016, -0.08782497048377991, 1.4867862462997437, -1.946968674659729, 0.7233482599258423, 0.4528874158859253, -0.5174944996833801, -1.38825261592865, 0.38918787240982056, 0.3272121548652649, 0.08937237411737442, -0.23213127255439758, -1.3439462184906006, -0.520767867565155, 0.9857766032218933, 0.4721125364303589, 0.21063366532325745], [-0.6754747033119202, -0.11589054763317108, -0.7055695652961731, 0.0005014571943320334, -0.7336400747299194, 0.8981615900993347, -0.40014466643333435, -0.6148183345794678, 1.6163338422775269, -0.38835379481315613, 0.8953262567520142, 0.6194396615028381, -0.3883795738220215, 1.4635411500930786, 0.9046441912651062, 0.17127865552902222, -0.7473787665367126, 0.8726068139076233, 0.38554656505584717, -1.2490850687026978, 0.3316635191440582, -0.10649245232343674, -0.21350562572479248], [-1.0265069007873535, 0.38082507252693176, 0.749873161315918, 0.7353631258010864, 1.2827671766281128, -0.5310168862342834, 0.6329901814460754, 1.0217642784118652, -0.40077921748161316, 0.3170253038406372, -0.2584526836872101, 0.6184489130973816, 0.9538518786430359, -0.9320114254951477, -1.5147809982299805, 0.678813099861145, 0.37634041905403137, 0.18586190044879913, 0.12039066851139069, 0.5343894958496094, 0.18686816096305847, -2.406647205352783, -2.2696406841278076], [0.15597030520439148, 0.0007716689142398536, 2.035543918609619, -0.27949264645576477, 1.8235573768615723, 1.4232828617095947, -1.1885690689086914, 0.27031058073043823, -0.19191104173660278, -0.9875018000602722, -0.23075424134731293, 0.7936573028564453, 0.2069557011127472, 0.019749386236071587, 1.1564393043518066, -0.32701778411865234, -0.9390391111373901, 0.5863343477249146, -0.7356469631195068, 0.9759660363197327, -0.08439412713050842, -0.5361930727958679, -1.3573174476623535], [-0.009531029500067234, -2.1960561275482178, 1.2261940240859985, 0.4511590898036957, 0.7207968831062317, -0.30406010150909424, 0.6630923748016357, -1.1062431335449219, -0.2599681317806244, 0.4096640944480896, -0.3611794114112854, -0.05116867274045944, -0.2663208246231079, 0.39064571261405945, -0.6593473553657532, -1.1302695274353027, -1.356812596321106, -0.8802230954170227, 0.7267560958862305, -0.9489368796348572, 0.16322746872901917, 1.7172802686691284, 1.1558057069778442], [0.9179646968841553, 0.10965041071176529, 0.5750311613082886, -0.3578280508518219, 0.17824073135852814, -1.2825868129730225, -0.5825093984603882, -0.2589856684207916, -2.481250762939453, -1.0976043939590454, 2.090880870819092, -1.3592901229858398, -0.7423044443130493, 0.3024674952030182, -0.874580442905426, -0.3975381553173065, -1.4519771337509155, 0.23132027685642242, -0.6909350156784058, 0.13116616010665894, -1.2207286357879639, 0.8517287373542786, 0.1591920405626297], [-0.549812912940979, -1.0472605228424072, 0.14849767088890076, -1.0396552085876465, 0.6662871241569519, -0.16671088337898254, -1.3685028553009033, 0.842732310295105, 0.6642673015594482, 0.07988249510526657, -2.43438458442688, 0.6416032314300537, 0.8521116971969604, -0.21808014810085297, 0.29228606820106506, 0.27469566464424133, -0.8175674080848694, -0.8992387652397156, 0.08205481618642807, -1.2511619329452515, -0.549323558807373, 2.1029956340789795, 0.5997421145439148], [1.9282658100128174, 1.4043618440628052, 1.1995006799697876, 0.6414415240287781, -1.1905580759048462, 0.12093564122915268, 0.4356297552585602, -0.1709604263305664, 1.518457055091858, 0.7568372488021851, -0.7128764390945435, -2.686429977416992, 0.63889080286026, 0.7451480627059937, 0.41405948996543884, 0.7353426814079285, 1.1209423542022705, 2.1077630519866943, -1.285846471786499, -0.08222240209579468, -2.2405552864074707, 0.45831361413002014, -1.4412901401519775], [-2.729790449142456, -1.5261367559432983, 0.10736435651779175, 0.7332632541656494, 0.017418310046195984, 0.8586380481719971, -0.27438414096832275, 0.40394675731658936, -0.8750337362289429, 0.8715863227844238, -1.0020356178283691, -1.216322422027588, 1.0832115411758423, -0.5518847703933716, -1.1002000570297241, 1.5637632608413696, 1.3411697149276733, -0.5044832229614258, -0.13283395767211914, -1.1151562929153442, -0.8252584338188171, 1.1021833419799805, 1.0036683082580566], [0.18669968843460083, 1.0052012205123901, 0.935530424118042, -0.797775149345398, -0.8632593750953674, 1.0326323509216309, -0.6626972556114197, -0.2963287830352783, 1.3153607845306396, 0.3480786085128784, 0.7695905566215515, 1.6866947412490845, 0.4856637716293335, 0.9274539947509766, 0.27272146940231323, 0.5452592372894287, 0.35295578837394714, 0.570819616317749, -0.7419285774230957, -1.087796688079834, 0.4133794903755188, 0.04575569927692413, -0.03235403820872307], [1.0240452289581299, -2.3529984951019287, 0.5773994326591492, -0.016254737973213196, 2.0817949771881104, 0.6273368000984192, -0.9622372388839722, 1.3231533765792847, 1.5373374223709106, -1.003167986869812, 0.24966169893741608, 1.54657781124115, -0.24412737786769867, -0.05124596878886223, -0.8319297432899475, 1.2741904258728027, 0.27851584553718567, 0.8356161117553711, -0.5645173192024231, -0.5798274874687195, 1.851614236831665, -1.0115163326263428, 0.17489442229270935], [-0.11279210448265076, -1.0266772508621216, 0.6205900311470032, 0.6412513256072998, 0.39597055315971375, -0.31995975971221924, 0.6213882565498352, 1.2248231172561646, -0.7084752917289734, 1.2595781087875366, -0.4171701967716217, 1.752699613571167, -1.0831758975982666, 1.3327282667160034, 1.5372884273529053, 1.0156883001327515, 0.3574490547180176, 0.20794324576854706, 2.004185914993286, -0.9110491275787354, -0.27900296449661255, 0.07325822860002518, 0.11473692208528519], [-1.12165367603302, -1.4041645526885986, -1.5735256671905518, 0.8008232116699219, 0.6650000214576721, 1.1168124675750732, -1.1455891132354736, -0.35523009300231934, -0.5248220562934875, -0.9373598694801331, -0.6488726735115051, -0.340842604637146, 0.4992488920688629, 0.18403281271457672, -0.13122160732746124, 1.2013721466064453, 0.32316115498542786, 1.5027226209640503, -1.5663188695907593, 0.3510628342628479, -0.6183998584747314, -1.3079890012741089, 0.1743393987417221], [0.053461942821741104, -0.6601404547691345, 0.006681067869067192, -1.549943447113037, -1.1088870763778687, -0.4151577353477478, 0.6256623864173889, -0.735668957233429, -0.08845051378011703, 0.04974045231938362, -0.9160916805267334, 2.009429693222046, 1.7177464962005615, -1.294735074043274, 1.1077224016189575, -0.3014754354953766, 0.34556788206100464, -0.22487609088420868, 1.1377501487731934, 0.16298966109752655, -0.688439130783081, -0.6410821080207825, 1.464461326599121], [-0.005451866891235113, 0.6095724701881409, 0.37223783135414124, 1.2292728424072266, 1.682503581047058, 1.7437111139297485, -1.1789339780807495, -0.3962055444717407, -0.2623918652534485, -0.6187548637390137, 1.1949167251586914, 1.1754076480865479, 0.33133846521377563, 2.1326558589935303, 0.06739231199026108, -0.48625174164772034, 1.0769906044006348, -0.9184339046478271, -0.1076369658112526, 0.6044164896011353, -0.3274357318878174, 1.2530806064605713, -0.348360151052475], [1.4172794818878174, -0.8457555174827576, -0.48024386167526245, -1.2568726539611816, 1.1218949556350708, -0.6474248766899109, 0.8173754811286926, 0.08312293887138367, -0.1635432094335556, 1.0501424074172974, -0.9945455193519592, -1.3872058391571045, 0.19922618567943573, -1.1199883222579956, 1.4225728511810303, 0.34332865476608276, -0.8746291995048523, 2.052631378173828, -1.5914369821548462, -2.094792127609253, 1.201421856880188, -0.12565608322620392, -0.013989409431815147], [-1.0055736303329468, 1.2806129455566406, -2.1312832832336426, 0.49917072057724, -0.2874181866645813, 0.06253023445606232, 0.9544639587402344, -0.670494794845581, 1.2303876876831055, -2.0626578330993652, -1.0781632661819458, 1.846299409866333, -0.8109658360481262, -0.5664526224136353, 2.1903722286224365, -1.0070005655288696, -1.0402013063430786, 0.2781202495098114, -0.8299869894981384, -1.2585755586624146, 1.54329514503479, 0.4451715648174286, 0.728263258934021], [-0.18148238956928253, 0.452301561832428, 0.423002690076828, -0.8007433414459229, 1.2303147315979004, 1.0700056552886963, 0.1303548812866211, 0.10178772360086441, 1.9955955743789673, 1.0426727533340454, -0.1765783280134201, -1.1994904279708862, -0.4068111777305603, 1.2064576148986816, -0.12989197671413422, 0.8324490785598755, 2.08539080619812, 0.02745034173130989, -0.1792576014995575, -1.2023171186447144, -3.268317937850952, -0.42572471499443054, -0.013946822844445705], [0.843046247959137, 0.5879229307174683, 0.1665450781583786, 0.26309579610824585, -0.6571648716926575, -1.5150649547576904, 0.08911237865686417, -1.7241917848587036, -0.5409042835235596, -0.9915269017219543, -0.6491968631744385, 0.9175847768783569, -0.3533931374549866, -0.02245599590241909, -0.7093132734298706, -1.0606043338775635, -1.2536036968231201, 1.6154959201812744, 0.5427369475364685, 1.0709781646728516, -0.5883631706237793, -0.18649351596832275, 1.1323918104171753], [1.1201868057250977, 0.20504730939865112, -0.34848999977111816, -0.909679651260376, -0.3762684762477875, 0.6614270210266113, -0.6505891680717468, -0.9950419068336487, -0.5379589796066284, -0.13822440803050995, 0.4780794680118561, 1.0449610948562622, 0.44153764843940735, 0.057780586183071136, -1.0173475742340088, 1.263533353805542, 1.0600217580795288, -1.3019227981567383, 0.22770321369171143, 1.2331643104553223, 0.18568776547908783, -0.18185549974441528, -1.7977561950683594], [-0.17380467057228088, -0.028031833469867706, 1.309597134590149, 1.0986337661743164, 2.368215322494507, -0.2668910026550293, -0.66802579164505, 0.9262645840644836, -0.7580581307411194, 0.4441833794116974, 0.473881334066391, 0.8495962619781494, -0.22971300780773163, 0.32068273425102234, -1.6947492361068726, 1.5148463249206543, 1.4406765699386597, 1.4658663272857666, 1.5770595073699951, 0.565506637096405, 0.32617250084877014, 0.38651368021965027, 0.9369569420814514], [-1.0489598512649536, -0.2826172411441803, -0.07046623528003693, 0.29209083318710327, -0.06166684255003929, -0.08236439526081085, -0.9451426863670349, -0.6162880659103394, -0.6833082437515259, -1.4902441501617432, -1.9833778142929077, -0.2069420963525772, 1.4377939701080322, 2.1862761974334717, -0.03564948961138725, 1.29279363155365, 0.016915760934352875, 0.6256231069564819, 1.297546148300171, 1.6295526027679443, 0.3188134729862213, 0.1728174239397049, -0.05078992620110512], [0.5710586905479431, -0.6866061687469482, 1.9608280658721924, -2.4539947509765625, -2.198556661605835, -0.8087570667266846, 2.761293888092041, -0.6277496814727783, -2.4959588050842285, 0.46779903769493103, 0.5595028400421143, 0.5922237634658813, 1.0064308643341064, 0.6922832727432251, 1.1141948699951172, -0.1898377239704132, 0.6640177369117737, 0.8835868239402771, 0.05402802303433418, -0.10560459643602371, -0.15142033994197845, -0.676294207572937, 0.26903998851776123], [0.8056325316429138, -0.8512647747993469, -0.21054627001285553, -0.402596652507782, 1.4818378686904907, 0.3855206370353699, 0.7753780484199524, -0.5428545475006104, -0.04030197113752365, -0.7018119692802429, -0.9979034662246704, 1.0709667205810547, -0.3307582437992096, -1.288835883140564, 0.8017266988754272, 0.41541048884391785, 0.31781160831451416, 1.1825659275054932, 0.8973096013069153, -0.6869728565216064, -0.05860477685928345, 0.9008389115333557, -0.6759684085845947], [0.5774690508842468, 0.6296478509902954, 1.287194848060608, 0.07021874189376831, 0.773772656917572, -0.7208479642868042, 0.9956420063972473, 0.47262343764305115, -1.0707474946975708, -0.3931896388530731, -0.31730595231056213, -1.085203766822815, 0.537992537021637, 0.2361118197441101, 1.2334190607070923, -1.4559650421142578, -1.021335482597351, -0.9619981050491333, -0.2744617164134979, -0.04468151181936264, 0.7291125655174255, -0.32312023639678955, 0.6275198459625244], [0.781283974647522, -0.4496005177497864, 1.4753097295761108, -0.5261463522911072, -0.1731591820716858, 1.395498275756836, -0.01134122908115387, 0.27368560433387756, 0.7554941177368164, -0.015315761789679527, -1.080551028251648, -0.19250626862049103, 1.0360978841781616, -0.9104768633842468, 0.23723025619983673, -1.293088674545288, -0.8111468553543091, -0.33354660868644714, 0.6640622615814209, 0.9505181908607483, -0.7568985819816589, -0.527362585067749, -0.008909894153475761], [0.2387809157371521, -0.9698646664619446, -0.675565242767334, -0.08599637448787689, 2.246628761291504, 0.8920031785964966, 1.1200999021530151, -0.8285396695137024, 0.8249329328536987, 0.3166402578353882, -2.3229856491088867, 1.8091927766799927, -2.7842445373535156, -0.3996486961841583, 1.2674096822738647, -0.5948665142059326, -0.13544346392154694, 0.5933967232704163, 0.03783339262008667, 0.3193429410457611, 0.8032119274139404, 0.2694682478904724, -1.206868052482605], [0.16709809005260468, -1.1593612432479858, -0.47188082337379456, 0.14887678623199463, 0.30720457434654236, -0.17305676639080048, -0.6405214071273804, 0.5989769697189331, 1.151976466178894, 0.3798608183860779, 0.07579828798770905, -0.9698542356491089, -0.8718351721763611, 0.9756965041160583, 1.1798030138015747, -0.37582868337631226, -1.0403791666030884, 1.307176113128662, 0.6297589540481567, 0.2328128218650818, 0.4858114719390869, -0.9055427312850952, 0.7538682222366333], [0.20705826580524445, -0.0460287369787693, 0.6169124245643616, 0.4252415895462036, -1.1699793338775635, -0.5406611561775208, 0.6510765552520752, 0.2462087869644165, 0.8629385232925415, -0.1415688395500183, 1.2275691032409668, 0.08619781583547592, -0.07004639506340027, -0.6174803972244263, 0.877924382686615, -0.8919560313224792, 0.48216351866722107, -0.3604881763458252, -0.42140719294548035, -0.19799299538135529, -0.6032393574714661, -0.4777064919471741, -0.7662858963012695], [0.9192937016487122, 0.5381969213485718, -1.268790602684021, -0.6434862017631531, -0.22828567028045654, -0.8582198023796082, 1.2840490341186523, -0.5389916300773621, 1.1199767589569092, 0.12739397585391998, -1.5257277488708496, 0.011649121530354023, -1.087848424911499, -1.001997709274292, -1.4962642192840576, 0.2326355129480362, -0.7183434367179871, -1.3457722663879395, 0.1969735324382782, -0.3145764470100403, -0.2989741861820221, 0.1497776061296463, -0.5508447289466858], [-0.07737796008586884, -0.26155808568000793, -0.9667113423347473, -0.13336586952209473, -0.3330951929092407, 0.10205566138029099, 0.21867790818214417, -0.5219585299491882, -1.098933219909668, 0.3794306516647339, 0.7636518478393555, -0.21991336345672607, -0.161437526345253, 1.7836142778396606, 0.43040457367897034, -0.7749455571174622, 0.32223978638648987, 0.17208808660507202, -0.9009674191474915, -0.4140181839466095, 0.86953204870224, -0.9687318205833435, -0.863032877445221], [-0.28393563628196716, -0.40175512433052063, -0.022178970277309418, -0.2552780508995056, 0.7469360828399658, 0.21709145605564117, -0.9530187249183655, 0.11738214641809464, -0.3950445055961609, 0.34338921308517456, 0.5642474293708801, 1.2419575452804565, 0.8882955312728882, 1.4893425703048706, 0.7507636547088623, 0.7307122945785522, -0.6093393564224243, -0.4451720416545868, -1.1953390836715698, -0.23304976522922516, -0.6030237078666687, 1.0642030239105225, 2.000880479812622], [-0.5977365374565125, 1.173374891281128, -0.2898857891559601, -2.5512592792510986, -0.8565748333930969, 0.9539690017700195, 0.3731900155544281, -0.31729593873023987, -0.9793456196784973, -0.9769212007522583, -0.3344142436981201, 0.5707880854606628, -2.1733510494232178, 0.05451654642820358, -0.7777581810951233, -0.6930718421936035, 0.8669103980064392, 0.29290562868118286, 0.292714923620224, 0.3634087145328522, -0.2044059932231903, -0.32780593633651733, 0.3965226411819458], [-0.15267498791217804, -0.6555135846138, -0.11745364218950272, -0.7888516187667847, 0.8910126090049744, -0.5144728422164917, -1.0189934968948364, 1.0581247806549072, 1.3510395288467407, 1.724802851676941, -0.19892844557762146, -0.741980791091919, 1.824775218963623, -0.5864651799201965, -1.4384492635726929, -0.7012190222740173, 0.2530204951763153, 1.9144197702407837, 0.7531193494796753, 1.7895441055297852, 0.9218630790710449, -0.2037365436553955, 0.2016010284423828], [-0.2700548768043518, -0.2787875831127167, 0.45712026953697205, 0.22723017632961273, -0.1025836169719696, -0.4643124043941498, -0.28756183385849, 0.39291203022003174, -0.09810546785593033, -0.28687578439712524, 1.6699310541152954, -0.801885187625885, 0.8848478198051453, -0.48259666562080383, -0.6076353192329407, 0.23010879755020142, -0.9557624459266663, -1.3968274593353271, -1.234166145324707, -1.0822473764419556, 0.12614524364471436, -1.7863514423370361, -0.6097423434257507], [-0.6263531446456909, 0.9892746210098267, -0.8729888200759888, -0.3222801387310028, -0.8003085851669312, 0.36722248792648315, 0.8034132719039917, -0.8844369053840637, 1.509183645248413, -1.1070767641067505, 0.4276600778102875, -0.1494010090827942, 1.510249376296997, -1.6417529582977295, 0.9943262934684753, -1.3123738765716553, 0.6379928588867188, -0.17265501618385315, -0.5327695608139038, -2.5286598205566406, -0.8959521055221558, -0.9443570971488953, -0.8885782957077026], [-0.9594286680221558, 0.46176645159721375, 0.23038208484649658, 1.0583341121673584, -0.6154110431671143, 0.2648431062698364, 0.06286899000406265, -0.14769892394542694, -1.1226718425750732, -1.2151989936828613, 1.1627851724624634, 0.06299647688865662, -1.1631474494934082, -1.2632746696472168, 0.051497310400009155, -2.2293922901153564, 0.46095120906829834, -0.46098849177360535, 1.4247585535049438, -1.947429895401001, -0.5578222274780273, -0.018827712163329124, 0.1330256462097168], [-0.8786660432815552, 0.6635536551475525, 1.3554168939590454, 0.8132862448692322, -0.4406571388244629, 1.4280651807785034, 2.7209324836730957, -0.08435802161693573, -2.1528842449188232, -0.11438937485218048, 0.5490686893463135, -0.7336528897285461, 0.028594348579645157, -0.05553971230983734, 1.1596523523330688, 1.1680866479873657, -0.27530789375305176, 0.014423553831875324, 0.6907674670219421, 0.6447545886039734, -0.1072346493601799, -0.1212623342871666, 1.3460851907730103], [0.4409453868865967, -0.6820592284202576, -0.37949076294898987, 0.40704819560050964, 0.032207977026700974, -0.4555888772010803, -0.8326557874679565, 2.5492429733276367, 1.4816640615463257, 0.10185051709413528, -0.21820758283138275, 0.29705846309661865, -0.39520207047462463, -0.14112381637096405, 0.7306396961212158, -0.4548914432525635, -0.7439265251159668, 0.3404523432254791, 1.4334720373153687, -1.6463395357131958, 0.4356625974178314, -2.013622283935547, -0.7571472525596619], [-0.3434291183948517, 0.3393472135066986, 1.3003851175308228, 1.3490140438079834, 0.7572458982467651, -0.7604317665100098, 0.26212331652641296, 0.3651699721813202, -0.6995369791984558, -0.06077021732926369, 0.4019142687320709, 0.0850251242518425, -0.4410569965839386, 0.7713515758514404, 0.4945788085460663, -1.9233075380325317, 0.2327968180179596, 0.8531253337860107, -2.247603178024292, 0.14966674149036407, -2.1282994747161865, -0.6327253580093384, 0.08678513020277023], [0.7737473249435425, 0.04598235338926315, 0.838731586933136, -0.3206802010536194, -0.17831802368164062, 0.6866726875305176, 1.5123937129974365, 0.6331034302711487, -0.1834968477487564, -0.3333774209022522, 1.2279986143112183, 0.11682911217212677, -0.4530888497829437, -0.33404192328453064, 0.4371207058429718, 1.0202109813690186, 0.7121424674987793, 0.5655393600463867, 1.4079620838165283, 0.6967121958732605, -0.4874992370605469, 1.2269700765609741, 1.78220534324646], [0.5838505625724792, -0.36226752400398254, -0.6620391011238098, 0.2617891728878021, 1.1062994003295898, 0.539539635181427, 0.2085549235343933, 0.5729383230209351, 1.2285512685775757, -0.5142675042152405, 0.4541875720024109, 0.2744453549385071, 0.06947971135377884, 0.7569038271903992, 0.8340368270874023, 0.18892070651054382, -0.8761867880821228, 0.4689984917640686, -0.7331588864326477, 1.4111653566360474, 1.2901993989944458, -0.47420066595077515, -2.4049952030181885], [-1.6994620561599731, 0.574147641658783, 0.38166165351867676, 0.20290569961071014, 0.9360243678092957, 1.3260103464126587, -1.174540400505066, -0.4874890446662903, 0.3972409665584564, 0.6160262227058411, -0.0793701708316803, -0.2868598401546478, 1.97352933883667, 0.21158629655838013, 1.1625211238861084, 0.3536085784435272, 0.46316295862197876, 0.9991716146469116, -2.4004580974578857, -0.392458975315094, -1.7124758958816528, 0.3386407494544983, -1.10490083694458], [-0.4282940626144409, 0.07956970483064651, -0.5507736802101135, -0.7777042984962463, 1.5336363315582275, -1.193627119064331, -0.7940545082092285, -0.8362432718276978, -0.968837559223175, 0.5741819739341736, -0.540526270866394, -1.4237322807312012, 0.47219401597976685, 0.7089207172393799, -0.5230019688606262, -0.09585712850093842, 0.9503681659698486, -0.361318439245224, -0.9197629690170288, -0.5930061340332031, -0.6534758806228638, 0.009789524599909782, -0.6697127223014832]], "dtype": "float32"}], "e": {"__nd__": [[-1.0641146898269653, 0.8563287854194641, 7.042234897613525, -6.156109809875488, 5.573812961578369, 5.7696709632873535, 4.591124534606934, -3.4450337886810303, 0.4997546970844269, -4.6139349937438965, -0.28799083828926086, 2.3704230785369873, -5.22019100189209, 3.658176898956299, 3.6196677684783936, -6.113574028015137, 5.761282444000244, -5.39083194732666, -2.455108642578125, -3.855550765991211, -0.26979175209999084, 4.818124771118164, 3.0460946559906006], [-9.021530151367188, 1.9429302215576172, -16.425275802612305, 9.734626770019531, 9.610485076904297, 11.873556137084961, -4.77646541595459, 6.554194927215576, 20.717382431030273, 3.9340732097625732, 0.16813428699970245, -9.1511869430542, -1.4459993839263916, -7.568634510040283, -4.4756693840026855, 2.2222936153411865, 13.303096771240234, -4.446998119354248, 1.3306416273117065, -5.879607200622559, -1.3146164417266846, -2.9083404541015625, 3.7926151752471924], [-3.79303240776062, 8.971068382263184, -4.507193565368652, 4.702024459838867, -4.0070953369140625, 1.146683692932129, 9.49621868133545, 1.8632868528366089, -7.942479610443115, 0.04947337880730629, -4.238317012786865, 11.759697914123535, -8.84140682220459, -0.04388672485947609, 1.5036007165908813, 4.701455116271973, 9.54698657989502, 4.28302526473999, 7.381132125854492, -0.8543363213539124, 3.633949041366577, 2.8899927139282227, 12.284429550170898], [10.277799606323242, -12.809167861938477, 4.149845123291016, -1.2452423572540283, -1.7338910102844238, -5.691704750061035, -2.697031021118164, 2.839918851852417, 14.19200325012207, 2.453122854232788, -3.808793544769287, -16.428659439086914, 0.6714968085289001, -9.532594680786133, -6.482278347015381, 0.29339277744293213, 1.6689079999923706, 3.588597297668457, 1.237276554107666, -7.662041664123535, -15.835624694824219, 12.699483871459961, 12.730979919433594], [-0.9984726905822754, 4.190743446350098, 3.1980862617492676, -11.484880447387695, -7.023157596588135, -7.198563575744629, -1.203378438949585, -5.676251411437988, 3.9139223098754883, -6.054553985595703, -3.048868179321289, 8.35634708404541, 7.659129619598389, -9.259177207946777, -8.39687728881836, -13.081851959228516, -8.261780738830566, -7.659168720245361, -3.2023069858551025, 4.501615047454834, -1.7491636276245117, -0.7124788761138916, -1.795375943183899], [-2.899139881134033, 1.7554562091827393, -1.7298616170883179, -2.1172163486480713, 0.8293401002883911, -1.9063793420791626, -4.976482391357422, -5.339336395263672, -5.6466593742370605, -1.6632136106491089, 7.007049560546875, 0.34308889508247375, 7.300539493560791, 7.67892599105835, -4.596503257751465, -5.675596237182617, 1.670943260192871, -11.532654762268066, -2.8850367069244385, 0.333795428276062, -3.911196708679199, -6.017031669616699, -2.0271425247192383], [1.485257863998413, 5.14310884475708, -5.169021129608154, 11.820202827453613, 2.9218366146087646, 7.796930313110352, 1.041197657585144, 4.772804260253906, 6.043738842010498, -0.2385016232728958, -0.7614707350730896, -12.459330558776855, -6.686314105987549, 11.273977279663086, 2.9570062160491943, 4.963366508483887, 11.314753532409668, 15.258545875549316, -4.067622184753418, -8.134732246398926, 1.4800589084625244, -13.211132049560547, -8.719066619873047], [12.145566940307617, -0.7777993679046631, 14.804346084594727, -2.8024277687072754, -2.1863293647766113, -3.225749969482422, -9.203400611877441, 4.911359786987305, -3.8742499351501465, 5.637720584869385, 11.332855224609375, -1.0915831327438354, 12.477896690368652, 16.580263137817383, -12.281756401062012, 2.6691040992736816, 5.376278877258301, 14.284567832946777, -2.2844021320343018, 8.744384765625, -2.509337902069092, -16.21681785583496, -8.836735725402832], [4.342014789581299, -5.920200347900391, -7.065189838409424, -0.38886502385139465, -2.493328809738159, 6.076741695404053, 1.1584450006484985, -1.9402693510055542, -3.1785948276519775, -1.7429786920547485, 3.233015298843384, -3.008848190307617, 0.801448404788971, -6.808666229248047, -4.263018608093262, 8.3023042678833, 0.6085858345031738, -2.221482038497925, -3.9868645668029785, 2.354282855987549, 2.6052803993225098, 2.6709401607513428, 3.315843343734741], [-1.9959908723831177, -3.6668176651000977, -4.353485107421875, -9.3715238571167, -10.82492733001709, -7.3738861083984375, -6.349648475646973, 0.6222175359725952, -10.030441284179688, 2.658086061477661, 7.3324408531188965, 0.8055142760276794, 7.85660457611084, 4.422561168670654, 0.6568197011947632, 3.0549726486206055, -1.2518680095672607, -0.9861276149749756, 4.828863620758057, 5.693263530731201, 3.1827468872070312, -17.811437606811523, -5.912370204925537], [-0.2812800407409668, -5.178979873657227, 2.7816545963287354, 2.1623144149780273, -9.714457511901855, -14.42910385131836, 13.903183937072754, 4.375617504119873, -7.235568046569824, 3.4620749950408936, -7.921900749206543, 0.9461418390274048, 2.6376960277557373, -13.54737377166748, 6.305858135223389, -6.109405994415283, -8.498583793640137, 2.335078239440918, -2.294246196746826, 5.750500202178955, -1.9703452587127686, -0.07826896756887436, 12.214184761047363], [0.8382341265678406, -0.47731447219848633, 5.507455825805664, 8.354278564453125, 6.627781391143799, -1.1125067472457886, -6.695432186126709, 6.683123588562012, 5.42453145980835, 3.9725379943847656, 1.9574192762374878, -4.642415523529053, -3.9385645389556885, -2.763317108154297, -5.22879695892334, 1.2008132934570312, 2.1602742671966553, -5.822506427764893, -1.6198188066482544, 0.5260505080223083, -12.27648639678955, -6.7035980224609375, -2.893658399581909], [3.822500467300415, 2.022409677505493, -2.017293930053711, 0.10883871465921402, -2.0303823947906494, 0.46014997363090515, 2.9388465881347656, -1.5250120162963867, 2.435976505279541, -1.227073311805725, 4.829906940460205, 5.587684154510498, -7.505204677581787, 2.7783124446868896, 6.68255615234375, 1.2064634561538696, -8.283857345581055, 5.342334747314453, 6.792635917663574, -5.042654514312744, 10.897080421447754, -7.331007480621338, -3.4584293365478516], [-1.654162049293518, 11.218642234802246, 6.058149814605713, 1.844406008720398, -11.844574928283691, -2.188490867614746, 2.141730308532715, 2.840238094329834, -2.6664257049560547, 3.8933558464050293, 8.677528381347656, -1.2853864431381226, -2.5907089710235596, 3.393566846847534, 2.283540964126587, -6.2618865966796875, 7.190149784088135, -1.3218437433242798, 11.858869552612305, -1.5453431606292725, -12.031638145446777, -1.7344237565994263, -3.2219223976135254], [-0.3598833382129669, -10.675738334655762, -8.581161499023438, 11.628419876098633, 20.815574645996094, 1.9089759588241577, -4.675050735473633, 5.042696475982666, -2.2788872718811035, 3.4313464164733887, -0.8883476853370667, 9.076921463012695, 8.441657066345215, -2.690873146057129, -8.672210693359375, 10.202290534973145, 2.4612066745758057, 5.91585636138916, 0.1399361491203308, 7.848422050476074, 9.799476623535156, -5.441033363342285, 1.2222917079925537], [11.661592483520508, 6.511746406555176, -1.2802420854568481, -6.639016628265381, -5.9194016456604, 4.66887903213501, 7.372519016265869, -0.6991439461708069, 9.889520645141602, -4.5978498458862305, -6.535562992095947, -3.202861785888672, -3.446817398071289, -4.986964702606201, 10.076704025268555, -8.762162208557129, -0.9744511246681213, 2.5581085681915283, 4.688647747039795, -5.161262035369873, -3.6509292125701904, 7.106401443481445, 2.4009041786193848], [3.3704330921173096, 12.87591552734375, -4.2326436042785645, 1.5656285285949707, -12.563813209533691, -7.1115875244140625, 13.572031021118164, -0.46497872471809387, -8.894246101379395, 6.751687526702881, 3.2212719917297363, -14.188831329345703, 2.3275766372680664, 9.28193473815918, 5.2180562019348145, -2.9892592430114746, 0.29118281602859497, 9.460841178894043, -6.720894813537598, 1.1769349575042725, 3.1530070304870605, -15.810556411743164, -17.162616729736328], [-6.73038387298584, 10.795531272888184, 6.372662544250488, -4.658268451690674, -6.840675354003906, 9.331786155700684, 6.920400619506836, -2.9784908294677734, -16.15070152282715, -4.539158344268799, 2.403193950653076, 1.1575020551681519, 8.660351753234863, 8.510293960571289, 6.726341724395752, 12.056406021118164, 12.420845985412598, -1.9220508337020874, 0.7058886885643005, -6.838611125946045, 2.4112627506256104, 8.46092414855957, -1.383528470993042], [4.523739337921143, -5.962263107299805, -0.4653293192386627, 3.6531453132629395, -11.100058555603027, 8.09666919708252, -0.10680687427520752, -2.227259397506714, 2.0355100631713867, -8.390377044677734, 5.4209818840026855, 1.6538234949111938, 0.6393876671791077, -0.9198982119560242, 13.631312370300293, -3.447331666946411, -5.929834365844727, 2.6864430904388428, -8.668113708496094, -9.857976913452148, 10.89100456237793, -6.322192668914795, -5.728383541107178], [2.495790719985962, -4.162913799285889, 10.969560623168945, -1.91769540309906, -3.541360855102539, 5.69787073135376, 1.36042320728302, -1.8608756065368652, 4.7557692527771, -2.0638742446899414, 1.6388132572174072, 2.792268991470337, 2.925459384918213, 3.3747520446777344, 11.32633113861084, 0.30418598651885986, -9.398655891418457, 7.233675956726074, 0.7543061375617981, -7.890285491943359, -2.244307518005371, 10.931974411010742, 10.704455375671387], [-5.637258052825928, -2.8013851642608643, 9.808633804321289, -5.38779878616333, 13.670339584350586, -4.1367340087890625, 10.900118827819824, -2.1254007816314697, -4.468296051025391, 7.519859313964844, -8.423787117004395, -13.82896900177002, 10.510148048400879, -5.746673107147217, -9.030855178833008, 3.901181936264038, 9.27854061126709, 4.5092291831970215, -5.972833156585693, -3.231755256652832, -1.0367578268051147, 1.8700454235076904, -6.470127582550049], [-0.05413525551557541, -2.8712456226348877, 0.8099325895309448, -7.476736545562744, -1.823823094367981, -0.8600897789001465, -3.8330211639404297, -6.394969463348389, 4.396873950958252, 3.6723544597625732, -4.737246513366699, 2.43648624420166, 3.9245572090148926, 8.295132637023926, -4.177903652191162, -0.08687038719654083, -3.805920124053955, -5.348038673400879, -0.31910836696624756, 15.664224624633789, 5.634488105773926, 11.94998550415039, 0.45702460408210754], [-2.182267427444458, 3.8487982749938965, -1.6739436388015747, 1.5810837745666504, -7.089995384216309, 3.495943546295166, 11.071242332458496, 4.7578444480896, -5.2970428466796875, -6.775712013244629, 0.1202978640794754, -4.7611284255981445, -7.977668762207031, -2.536292552947998, 9.41023063659668, 3.660050868988037, 6.688574314117432, 3.709794044494629, -0.9978750944137573, -9.965344429016113, -0.47267088294029236, -5.939765453338623, -6.713656425476074], [-0.9407216310501099, 4.341451644897461, 3.9893975257873535, -7.8591389656066895, 0.5507168769836426, 4.274110317230225, 4.796120643615723, -8.043510437011719, 3.9697721004486084, 8.956384658813477, -4.548123359680176, 14.738492012023926, -7.623647689819336, 3.4913320541381836, 1.779842734336853, 0.7529621720314026, 4.358531475067139, 8.843775749206543, -5.344522476196289, 5.204164505004883, -4.165170192718506, 1.9364526271820068, -3.464169979095459], [-4.8508477210998535, 6.566761493682861, 3.194793224334717, 6.670865058898926, 15.303327560424805, 12.389156341552734, -12.224438667297363, 1.9702244997024536, 6.725424289703369, -1.1721093654632568, 7.930692672729492, -0.07924097776412964, -8.065081596374512, -4.409646511077881, -4.24874210357666, 0.3928350806236267, -3.273711919784546, -5.4013543128967285, -5.235185146331787, -0.1498778611421585, 0.823143720626831, 1.7202540636062622, -14.851415634155273], [6.897495269775391, -13.250558853149414, 8.039264678955078, 8.104838371276855, 3.9213685989379883, -9.262874603271484, -5.490628719329834, -1.5818095207214355, -1.7858606576919556, -2.8545889854431152, -6.981753349304199, -7.514636993408203, 2.8736653327941895, 3.6168041229248047, -9.016983032226562, 2.7216057777404785, -10.805633544921875, 22.041994094848633, 5.333383083343506, 11.172886848449707, 13.743770599365234, -0.2138628214597702, -3.0869245529174805], [7.501734733581543, -0.7689427137374878, -7.345057487487793, 0.8385362029075623, -4.116774082183838, -4.781364440917969, 0.7817153930664062, 13.286029815673828, 2.8439204692840576, -17.044408798217773, -4.17537260055542, -2.8644678592681885, -3.284512996673584, -2.541933059692383, 2.3268144130706787, -4.801926136016846, 5.842085838317871, 5.241090774536133, 16.730382919311523, -0.6473934054374695, 6.80573034286499, -3.8460733890533447, 10.93643856048584], [-1.5863515138626099, 14.759069442749023, 1.3462988138198853, -2.803576946258545, -8.4816312789917, -0.12280653417110443, 6.120170593261719, -6.807373523712158, -6.152958393096924, 6.690013408660889, 14.765273094177246, 0.45053163170814514, 3.47265887260437, 3.945732831954956, -1.4665557146072388, 4.474766731262207, 3.963940382003784, -3.488692283630371, 4.673165798187256, 3.1597976684570312, -5.419426441192627, -1.0671775341033936, -1.9011749029159546], [-2.7479732036590576, 0.4068473279476166, -10.104501724243164, -9.549616813659668, -12.004343032836914, -1.1015626192092896, 8.426047325134277, -18.667451858520508, -1.691847801208496, -6.339867115020752, 7.009936332702637, 1.254610538482666, -3.141714572906494, -6.641687870025635, -9.664762496948242, -6.6456122398376465, -1.2479636669158936, -3.345878839492798, 5.539487361907959, 1.335994839668274, -0.6725097298622131, -8.398284912109375, -0.8990217447280884], [-4.337534427642822, 13.906999588012695, 1.7685879468917847, -1.429498314857483, -5.684140205383301, -2.1939756870269775, 2.989961862564087, -11.828341484069824, 2.6933395862579346, -6.976994514465332, 2.3467535972595215, -6.956805229187012, 7.054526329040527, -3.5863420963287354, -6.774005889892578, -9.508810043334961, -12.176581382751465, -9.415200233459473, -12.550665855407715, 14.502240180969238, 2.1933488845825195, -3.376728057861328, -5.78397798538208], [-0.3863343596458435, 4.191066741943359, -14.234100341796875, -10.736612319946289, -8.595357894897461, -4.2746357917785645, 5.282581806182861, -3.4390649795532227, 4.56218957901001, 0.18389226496219635, -4.189988136291504, -2.364985942840576, 1.4519144296646118, -9.285698890686035, 0.15571928024291992, -5.982199192047119, 4.429670810699463, -6.083197593688965, 4.868459224700928, -0.8765857815742493, -0.7662091851234436, 5.358197212219238, 2.4955368041992188], [-13.449186325073242, 17.185653686523438, -1.4060512781143188, 2.234006643295288, -4.249006271362305, -2.404949188232422, 4.59959077835083, -10.04549503326416, -4.017364501953125, 1.3892433643341064, -12.96190071105957, -3.0747413635253906, 9.075807571411133, 4.399913311004639, 11.39059829711914, -8.68025016784668, 3.0465915203094482, 4.886481285095215, 2.250887393951416, 13.252325057983398, 1.3833653926849365, -10.08807373046875, -13.108878135681152], [6.270112991333008, 0.011286046355962753, 0.9966040849685669, 1.4703001976013184, 9.18237018585205, 3.8885581493377686, -9.948468208312988, 3.7516872882843018, 10.946797370910645, 1.068025827407837, -9.093965530395508, 19.74897003173828, -0.4488179385662079, 2.3291373252868652, -2.5313520431518555, 5.433235168457031, -1.7540158033370972, 10.511723518371582, 3.0386502742767334, -4.714752197265625, -2.7212765216827393, -5.745812892913818, -6.2614593505859375], [-2.1109063625335693, -0.1403169333934784, -4.733827590942383, 3.7805514335632324, -0.9711086750030518, 5.909337997436523, -5.908514499664307, 11.478625297546387, -4.668854713439941, -13.248917579650879, -1.7027771472930908, 19.605098724365234, 10.513090133666992, -2.7183165550231934, 2.346846580505371, 6.48126220703125, 2.037081241607666, -1.544721245765686, 11.68076229095459, -5.051126956939697, 11.219265937805176, -4.422492980957031, 6.431312561035156], [-4.690639019012451, -0.06260167807340622, -6.029971122741699, -8.308476448059082, -4.520267009735107, -0.44994404911994934, -3.1509664058685303, -2.8055577278137207, -9.029553413391113, -8.281035423278809, 1.1705981492996216, 4.569764137268066, -8.962498664855957, 0.11258501559495926, 0.9584167003631592, -8.616814613342285, 3.4674394130706787, 8.435914993286133, 1.1849960088729858, 11.703557968139648, 10.250284194946289, 1.6727135181427002, 8.272843360900879], [7.8214030265808105, -0.06130458414554596, 5.06460428237915, 3.021270275115967, 8.321391105651855, -3.78943133354187, 5.384477615356445, 10.378934860229492, -0.5292669534683228, 7.623669624328613, -10.144452095031738, -2.5084877014160156, 0.43679293990135193, -1.3942657709121704, -1.7470648288726807, -2.478034496307373, 1.4866892099380493, 3.561086654663086, -0.9200555682182312, -0.8802427649497986, -9.100567817687988, -0.9194750785827637, 8.509320259094238], [9.013489723205566, -3.758193016052246, -3.6592681407928467, -3.1565494537353516, 19.102100372314453, 6.477422714233398, -2.362576961517334, -0.5949868559837341, 20.152467727661133, 17.11414909362793, -6.044991970062256, 1.5373965501785278, -4.961540222167969, 2.626408338546753, 2.7284724712371826, 8.347131729125977, 1.484452486038208, 15.095041275024414, -15.016870498657227, -5.774376392364502, 4.218139171600342, 2.683727264404297, -2.0915627479553223]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.5316926836967468, 0.108302541077137, 0.9395898580551147, 0.27584949135780334, 1.4003417491912842, 0.10063718259334564, 0.8270308375358582], [0.4823078215122223, 1.7655218839645386, 0.2824983298778534, -0.8890823125839233, 0.11794790625572205, -0.9304470419883728, 0.36792081594467163], [-0.5309395790100098, -0.0524309016764164, 1.1089551448822021, 1.3712873458862305, 1.71701979637146, 0.17821970582008362, 0.642472505569458], [0.9184796810150146, 0.3464799225330353, 0.9279161095619202, 0.15786443650722504, -0.3156261742115021, 1.271267056465149, -0.6818531155586243], [0.9796932339668274, -0.7551509737968445, -0.16893412172794342, -1.8611726760864258, 0.29394063353538513, -1.1617183685302734, -1.9506741762161255], [-1.224092960357666, 1.8253858089447021, 1.9166992902755737, -1.0200382471084595, 0.23109237849712372, -1.4485172033309937, -0.8392366170883179], [-0.47705140709877014, -0.5879063606262207, 1.765042781829834, 0.4361303746700287, -0.8219111561775208, 0.04302185773849487, 0.7778013944625854], [-0.08617416769266129, -0.2812102138996124, 0.14292998611927032, 1.4276007413864136, -0.968253493309021, 1.0397148132324219, -1.0828977823257446], [1.8570533990859985, -0.6846092939376831, -0.3992469608783722, -1.1055611371994019, -0.8465620279312134, -1.69256591796875, 1.1466361284255981], [1.1543914079666138, 1.2396674156188965, -0.1637628674507141, -1.3252837657928467, -1.2065168619155884, -0.8094038963317871, -0.6167886853218079], [0.5068864226341248, -0.12148407846689224, 0.7821733355522156, -0.5704326033592224, 0.22598035633563995, -0.9385896921157837, -0.8778327107429504], [-1.1541619300842285, 1.477279543876648, -0.5507529377937317, -1.800665259361267, 0.6162254214286804, 0.03068145364522934, 0.7790794372558594], [-0.4725643992424011, -0.8775875568389893, 0.6798542737960815, -0.17813047766685486, -0.168282613158226, -1.7429147958755493, -0.7610775232315063], [0.3340264856815338, 1.1538057327270508, 0.037855397909879684, 0.7748825550079346, -0.9003248810768127, 1.2562954425811768, 1.0279730558395386], [-0.7146114110946655, 0.16392655670642853, -1.0847605466842651, -0.8037735223770142, -2.579561233520508, -0.11219995468854904, 0.7433081865310669], [0.8971532583236694, -0.11147814989089966, 0.14611710608005524, 0.7324818968772888, 0.22172434628009796, -0.4090610146522522, 0.13018858432769775], [0.7847183346748352, -0.2524278163909912, -0.09023293107748032, -1.5667885541915894, -0.44442182779312134, -1.3586902618408203, 0.4053231179714203], [1.5176539421081543, -1.1661608219146729, -0.12280897051095963, 0.46382057666778564, 0.12195459008216858, 2.168617010116577, 0.22332271933555603], [-0.7792380452156067, -1.5008474588394165, -0.7918564677238464, -0.7100681066513062, 0.14934970438480377, -0.813927948474884, -0.09786989539861679], [1.4165846109390259, -0.30491527915000916, 0.8936108946800232, 0.2131310999393463, -0.01357701700180769, 0.04179186373949051, -1.113508939743042], [0.8012439608573914, 0.10117973387241364, -0.1362532377243042, -0.9770100712776184, -1.2001087665557861, 0.3437110483646393, -0.839292585849762], [0.6420489549636841, -0.050714604556560516, -0.6486693024635315, -0.24609695374965668, 0.2864365577697754, -1.2703173160552979, -0.11887967586517334], [-0.33688265085220337, 1.4461809396743774, 1.1507463455200195, 2.2623026371002197, 0.14045540988445282, 1.4131742715835571, -0.7202066779136658], [-0.919010579586029, 0.8156039714813232, 2.1155600547790527, 0.7404741048812866, 1.8030449151992798, -0.16420863568782806, 0.3694465458393097], [0.24231542646884918, -2.891434907913208, 0.8496648669242859, 1.0811896324157715, -0.27410662174224854, -0.6241243481636047, -1.1219922304153442], [-0.9175057411193848, -1.0741844177246094, 0.16423046588897705, -0.25301873683929443, -0.9483239650726318, 0.42996087670326233, 0.461845338344574], [1.2766656875610352, -0.9212231636047363, 0.3665711283683777, 0.2207028865814209, 0.4930598735809326, 0.5917983055114746, 1.228583812713623], [0.1505279839038849, -0.3659626245498657, 0.31193429231643677, 0.732713520526886, 0.11457858979701996, -0.183292955160141, -0.4202406108379364], [-0.6918617486953735, -0.6044450402259827, 1.135697841644287, -0.22713394463062286, -1.28618586063385, 0.09247423708438873, -0.36160314083099365], [0.9948983788490295, 1.5208064317703247, 0.5357342958450317, -1.5109513998031616, 0.5418857336044312, 0.028328368440270424, 0.33203479647636414], [0.6100534796714783, 0.7218762040138245, -0.4189639389514923, -1.247040867805481, 0.06903894245624542, -0.20546115934848785, 0.3321802020072937], [1.0991907119750977, -1.297555685043335, 0.5380215644836426, -1.73883056640625, -0.5984659790992737, -1.45750892162323, -1.1413358449935913], [-0.0444389209151268, 1.1476744413375854, 2.2065958976745605, 1.0232778787612915, 1.777414083480835, 2.427929401397705, -0.44380131363868713], [0.15876339375972748, 2.001554489135742, -2.048051357269287, -1.1472760438919067, -0.21865548193454742, -1.9711930751800537, 0.5961285829544067], [0.14750729501247406, 0.4685463309288025, -0.32976698875427246, 0.8518956303596497, -1.582174301147461, 0.8235959410667419, 0.09963195770978928], [-0.8810932040214539, 0.2202865034341812, 1.2195526361465454, 0.7540549039840698, 0.7264626026153564, -0.8407629132270813, 0.12489498406648636], [-0.8578858971595764, 0.6664788722991943, 0.6518722176551819, 0.7962898015975952, -0.6883708238601685, 0.1800159364938736, -0.2036861926317215], [0.15531331300735474, -0.2383410930633545, -0.49117380380630493, 0.511508584022522, 1.8324084281921387, 1.334304690361023, 0.8543260097503662], [1.4463918209075928, 1.284488320350647, -0.34153181314468384, -1.0274924039840698, -0.07213236391544342, -0.8052240610122681, 0.23295846581459045], [0.041492633521556854, 0.14279961585998535, 1.3225016593933105, -1.0089877843856812, -0.4601529836654663, 0.8959817886352539, -1.3881864547729492], [0.03368782624602318, 1.7095152139663696, 0.14660803973674774, 0.2934970557689667, 0.17065459489822388, -0.29106369614601135, -0.36408162117004395], [-0.44055041670799255, 0.3819747269153595, -0.003991064615547657, 0.65572589635849, -0.45115864276885986, -0.7303915619850159, 0.2164490520954132], [1.2737712860107422, 1.835079312324524, 0.08374044299125671, 1.0459034442901611, -0.011264109052717686, 1.9494664669036865, -1.2546918392181396], [-0.4143032729625702, 1.9683914184570312, -1.4228425025939941, 0.7904121279716492, -0.7943589091300964, -0.38480475544929504, 0.06241346895694733], [1.7934057712554932, 0.4223323166370392, 0.46059924364089966, 0.23978731036186218, -0.9005505442619324, 2.151644468307495, 0.24393774569034576], [-0.8553788661956787, 0.6020293831825256, -0.5385346412658691, 0.17865924537181854, 0.5292209982872009, 0.8374118804931641, 1.0990146398544312], [-0.5269327759742737, 0.4513266384601593, -2.575859308242798, 0.048635102808475494, -0.9861567616462708, 0.5010846853256226, 1.976167917251587], [-0.06953611969947815, 0.4293624758720398, -0.6377875804901123, 0.8233001232147217, 1.2472245693206787, -0.8607107996940613, -1.0229871273040771], [-0.5616817474365234, 0.850155234336853, 0.6916135549545288, 0.48377761244773865, 0.19466324150562286, -0.9450578093528748, -1.0603923797607422], [-0.11736897379159927, 1.6824363470077515, 1.2372549772262573, -0.21698600053787231, 0.14709697663784027, -0.8797685503959656, -0.09178469330072403], [-0.07788824290037155, -2.102139711380005, 0.5445811152458191, 0.6751107573509216, -0.9832181334495544, -0.039114076644182205, 0.8210110664367676], [0.10711442679166794, -1.164136528968811, 1.4831304550170898, -2.680656909942627, 0.04537738114595413, 0.4923931360244751, -1.1101188659667969], [0.3192766010761261, -1.6104050874710083, -0.650735080242157, 1.727233648300171, -1.5610761642456055, -2.390315055847168, 0.13246896862983704], [0.846499502658844, 0.7292286157608032, 0.9455763697624207, 0.07685326784849167, 1.144106388092041, 0.19811083376407623, 0.41755440831184387], [0.3270966112613678, 0.534783661365509, -1.200166940689087, -0.7176992297172546, -0.22348198294639587, -0.4088555574417114, -0.9630873203277588], [-0.5710461735725403, -0.6099624037742615, 1.4911669492721558, -0.9732748866081238, 0.3041277527809143, -0.18137063086032867, 0.30403533577919006], [-1.7522672414779663, 0.3917539715766907, -0.47376739978790283, -0.20691989362239838, 1.2576721906661987, 0.42119258642196655, -0.5057084560394287], [0.3308888375759125, -0.6631413102149963, 2.2466487884521484, 0.06253889203071594, -0.45045337080955505, 0.5768535733222961, 0.18648776412010193], [0.5807619690895081, -0.07060977816581726, 0.4041145443916321, -0.742438793182373, -0.8348308205604553, -0.8994172811508179, -0.15041202306747437], [-1.342690348625183, 0.4429031312465668, 0.6363136172294617, 0.4962632358074188, 0.029529619961977005, -0.16269756853580475, 0.6198003888130188], [-0.5812458992004395, -0.5725831985473633, -0.520784854888916, 0.7114992141723633, 1.0740711688995361, -0.8382266759872437, 2.1732594966888428], [-0.17623171210289001, -0.2779223620891571, -0.2505461275577545, 0.13201962411403656, 0.5523408651351929, -1.3687950372695923, 2.120851755142212], [1.1153477430343628, 0.2929222583770752, -0.8974577784538269, 0.2723999619483948, 0.8169280886650085, -1.1757947206497192, 0.010135816410183907], [-0.1550416499376297, 1.6265076398849487, -0.14233700931072235, 0.9178040027618408, -0.0920678973197937, -0.6018420457839966, -1.54146409034729], [0.15320225059986115, -1.628234624862671, 0.898867666721344, 0.2802349030971527, -0.6178785562515259, -0.9583951830863953, -0.5114608407020569], [0.7582541704177856, -1.1115999221801758, -0.39505791664123535, 1.9076300859451294, 0.025488484650850296, -0.5549867153167725, -0.21941065788269043], [-0.30985718965530396, -1.4353840351104736, -1.4377470016479492, -1.1177133321762085, 0.41173800826072693, 0.8749695420265198, 0.370299369096756], [-1.2224884033203125, -0.5735321044921875, -1.0806645154953003, 0.5485576391220093, 0.9955588579177856, -0.8672714233398438, 0.6168010830879211], [2.095898151397705, -0.07134538888931274, 0.23125821352005005, 0.6270251870155334, 0.5583595037460327, 0.012158463709056377, -0.5209984183311462], [0.6062623262405396, -1.1784650087356567, -0.6428824067115784, -1.5321069955825806, 0.7926790714263916, -1.07211434841156, 0.8220996856689453], [0.08660682290792465, 1.2852089405059814, -0.175240159034729, -1.1622625589370728, 0.2365308701992035, -1.2603718042373657, 0.3640406131744385], [-0.24918998777866364, 0.2512950301170349, -1.523246169090271, -0.6749951839447021, -1.14457368850708, 1.2982935905456543, 0.9555535912513733], [-0.2308485209941864, 0.6160274744033813, 0.21075297892093658, 0.14694048464298248, -0.44393736124038696, -0.6631889343261719, 0.4716795086860657], [0.3960307538509369, -0.6140096187591553, 0.07846225798130035, 0.0009706323617137969, -0.46108338236808777, -0.37421905994415283, 0.18309466540813446], [-0.280684232711792, -0.047778911888599396, -0.6171413064002991, -0.013815357349812984, 0.3032014071941376, -1.1987557411193848, -0.8751996755599976], [0.22490781545639038, 1.9501596689224243, 1.443535327911377, 0.6710930466651917, 0.9100059270858765, -1.8339084386825562, -0.017808349803090096], [1.0247427225112915, 0.98496013879776, 0.2698616683483124, -0.43932098150253296, 0.3932355046272278, 1.6966458559036255, 1.664595603942871], [-1.3348482847213745, 1.981419563293457, -0.6374776363372803, -1.0012887716293335, 2.190518379211426, -0.36476442217826843, 1.0063791275024414], [-0.9527817368507385, 0.1929541975259781, 1.0098869800567627, 0.21027790009975433, -1.1335135698318481, -0.05999795347452164, -1.5778535604476929], [1.2063251733779907, 0.23069319128990173, -0.9788964986801147, -0.10662626475095749, 0.7010174989700317, -0.45503634214401245, 1.544832468032837], [-0.4865529239177704, -0.0098944790661335, 0.9858795404434204, -0.7396360039710999, -0.33825981616973877, -0.42409777641296387, 1.143005132675171], [-0.5611292719841003, -0.6527647972106934, -0.5436903834342957, -0.6546037197113037, -1.6954931020736694, 0.7995638847351074, -0.261772483587265], [-0.683760941028595, 0.6178090572357178, 0.4704613983631134, -0.705323338508606, 1.2522629499435425, 1.2600712776184082, 1.0810778141021729], [0.7875881791114807, 0.9272593855857849, 0.7129992246627808, -0.17462748289108276, -0.9923995137214661, 0.1227412074804306, 0.48211610317230225], [-0.5700246095657349, 0.9454272389411926, 0.21737447381019592, 0.3137872517108917, -0.8284628391265869, 0.49652060866355896, -0.484941303730011], [-0.6440224051475525, -0.862410306930542, -1.3907358646392822, -0.2165890783071518, -0.6773430705070496, -1.0350251197814941, -0.39927586913108826], [-0.1932431012392044, -0.7746002078056335, 0.9935919046401978, -0.5895090699195862, 0.3210619390010834, -2.637277841567993, 1.4475717544555664], [-0.18453550338745117, 2.549365758895874, -2.0438010692596436, 0.7643698453903198, 0.828295886516571, -0.6295698285102844, -0.6326881647109985], [0.25013330578804016, -0.3552362620830536, 0.7086896300315857, 0.2443261742591858, -0.5943905115127563, 2.487837076187134, -0.022234147414565086], [-0.010665705427527428, -0.9561730027198792, 0.44977638125419617, 0.46457040309906006, 0.33682942390441895, -0.5080127120018005, -0.7940552830696106], [0.37898844480514526, -0.22619828581809998, -0.159792959690094, -0.12604649364948273, -0.766007661819458, 1.7404857873916626, -0.5325913429260254], [1.5441464185714722, 1.7677241563796997, -0.3463570475578308, -1.9856866598129272, -0.15211723744869232, -0.11830243468284607, 0.340631365776062], [-0.8935301303863525, 1.2999138832092285, 0.2675841450691223, 0.19971641898155212, -1.2768062353134155, -0.5570028424263, -1.0252248048782349], [-0.5670810341835022, 0.12590238451957703, 0.03481779992580414, -0.38156673312187195, -0.3799764811992645, -0.5104485154151917, -1.1158697605133057], [0.9263070225715637, 0.9373038411140442, 1.2080869674682617, 1.2859524488449097, 1.2207075357437134, -0.9393122792243958, 0.5471421480178833], [0.7904478907585144, -1.3189642429351807, 0.7980301976203918, -1.3288066387176514, -0.7702293395996094, -1.7337754964828491, 0.8306564688682556], [-1.5818549394607544, 0.22140109539031982, 0.6402811408042908, -0.4679920971393585, 0.7670249342918396, -0.2106490433216095, 0.10669530928134918], [0.5464368462562561, -1.1108477115631104, 0.18633638322353363, 0.05204760655760765, 0.519763708114624, 0.49968209862709045, -0.05413537472486496], [-0.8824049234390259, -0.038138214498758316, -0.0128603745251894, 2.10400390625, -0.6330848336219788, 3.4847629070281982, 1.706426978111267], [-1.4136983156204224, -1.511898159980774, 1.235079050064087, 0.6912373304367065, 2.547626495361328, 0.8914250135421753, -0.5253822207450867], [0.8523408770561218, 1.056718111038208, 1.336557149887085, -2.0513687133789062, -1.3793665170669556, -0.9533104300498962, -1.2823231220245361], [1.772242546081543, -0.5889188051223755, 0.46517619490623474, 0.44397586584091187, 1.7950265407562256, 1.827480435371399, 0.37464720010757446], [-0.2035045474767685, -0.3085111379623413, -0.8441242575645447, -0.1899692267179489, 1.224191665649414, 0.9736207723617554, -0.9867090582847595], [1.8325506448745728, 0.5796326994895935, -0.6333127617835999, 2.2997097969055176, 1.3311116695404053, -1.5232993364334106, 0.6559661030769348], [-0.39570459723472595, 0.24126766622066498, 0.9647198915481567, 2.601106882095337, 1.1237114667892456, -0.7933887839317322, -0.5896162390708923], [-1.4129955768585205, -1.4693782329559326, 0.1177973523736, -1.556105613708496, 0.35709962248802185, 1.2657470703125, -0.8222950100898743], [-1.254135012626648, 0.8778378963470459, -0.14391492307186127, 1.4640765190124512, -0.6046241521835327, -0.5894787907600403, -1.3851219415664673], [-1.1701693534851074, 0.7904773354530334, 0.961285412311554, 0.7055973410606384, -0.5945588946342468, 0.5929792523384094, 0.051588479429483414], [-1.8830657005310059, -1.653660535812378, 2.060089588165283, -0.40148434042930603, 2.868952989578247, 0.47769695520401, -0.41510888934135437], [0.4984797239303589, -0.7669531106948853, 1.042712688446045, -0.7930936217308044, 2.578699827194214, -1.8740534782409668, -1.7279828786849976], [-0.057230960577726364, -0.3995237946510315, -0.02274826169013977, -1.1770015954971313, -0.15131758153438568, 1.290574073791504, -0.7393150329589844], [-2.1038661003112793, -0.6041582822799683, 1.7116669416427612, 0.5901307463645935, 0.15623542666435242, 1.0904335975646973, 1.678009033203125], [1.7656031847000122, -0.37055283784866333, -1.1546664237976074, -1.6860090494155884, 0.8619664907455444, 1.6956300735473633, 0.2886674404144287], [-0.3661014139652252, -1.056320071220398, 0.09205593913793564, 0.5312361121177673, -0.362432599067688, -0.08728952705860138, 0.39583060145378113], [1.5921248197555542, -0.17274321615695953, -0.3916223347187042, -0.09147526323795319, -0.05679691210389137, -1.4979667663574219, -0.8389946818351746], [1.0559730529785156, -0.3376036584377289, 0.3097309470176697, -1.0436418056488037, 1.0167790651321411, 0.16991880536079407, 1.245645523071289], [-1.0643864870071411, -0.06255733966827393, 2.09226393699646, -1.8210065364837646, -0.6360787153244019, 1.0878580808639526, 1.0267103910446167], [2.2608211040496826, 0.8797078132629395, -0.7909476161003113, 0.6946881413459778, 1.014251708984375, -0.463943213224411, 0.26284411549568176], [-2.2682130336761475, -1.3335506916046143, 1.01259446144104, -1.480857253074646, -0.2911725640296936, 0.6985810399055481, -0.48147037625312805], [-0.9778085947036743, 0.2972475588321686, -1.3125817775726318, 0.3109906017780304, 0.15770503878593445, -0.5024617314338684, -0.629132866859436], [1.0036386251449585, -0.49167126417160034, 0.205500990152359, -0.8193748593330383, -0.32115647196769714, 0.41464754939079285, 0.2752436101436615], [1.539752721786499, -1.6190265417099, -1.4377521276474, 0.5272345542907715, -0.7876570820808411, 0.06250043213367462, 0.05718052014708519], [-1.1185967922210693, -0.13152088224887848, 1.823944330215454, 0.045825012028217316, 0.03149150684475899, 0.3102845549583435, -2.0150411128997803], [2.184985399246216, -1.1317721605300903, -1.124853491783142, -0.61226487159729, -0.30381134152412415, -0.25050777196884155, -0.7788349390029907], [0.5286316275596619, 1.0082882642745972, -0.9524636268615723, -0.9920937418937683, -1.5215528011322021, 0.5845856070518494, 0.3431571125984192], [-1.3607500791549683, -0.2582867741584778, 1.5847829580307007, 0.2567571699619293, -0.8907705545425415, -0.9040433168411255, -0.5298779010772705], [-0.9983117580413818, 1.3332555294036865, 0.07815851271152496, 0.4718017876148224, 2.7973530292510986, 0.856904923915863, 0.8493647575378418], [-0.9135231971740723, 0.9888080358505249, -2.1343889236450195, -2.051016092300415, -0.047574732452631, 1.198361873626709, 1.4020164012908936], [-0.9312834739685059, -0.9365540146827698, 0.6560602784156799, -0.7950050830841064, 0.6820273399353027, -0.33436521887779236, 1.0986539125442505], [-0.20817361772060394, 0.7500197887420654, 1.0032603740692139, -0.6139070391654968, 1.3320437669754028, 0.13873323798179626, -1.011117696762085]], "dtype": "float32"}, {"__nd__": [[-1.1000548601150513, -0.8091102242469788, -0.08711143583059311, -2.1262426376342773, -0.9053568840026855, 0.21180318295955658, 0.2753010094165802, 0.05395427346229553, -0.7037528157234192, 0.07378912717103958, 0.5335745215415955, 0.1531415581703186, -0.6297889947891235, -1.2132214307785034, 0.18072465062141418, 1.2006571292877197, -0.0720941349864006, -1.1843661069869995, -2.331022024154663, -0.7581645846366882, 0.34725627303123474, 0.9545786380767822, 0.6375526189804077, -0.8110338449478149, 0.8311370015144348, -1.6404671669006348, 0.10462714731693268, 1.307442307472229, 0.011430719867348671, 1.7616643905639648, 0.02829436957836151, -0.15332138538360596, 0.814155101776123, 0.21546754240989685, -0.8425366878509521, 0.01817253604531288, 1.6842900514602661, -0.5028913021087646, -0.27892887592315674, 0.5900399684906006, 0.25524958968162537, 0.2608755826950073, 1.2383824586868286, -0.4833531975746155, 0.08398762345314026, 1.4414275884628296, -0.8303266167640686, 0.486439049243927, 1.35860013961792, -0.4674912691116333, -1.086936593055725, 0.3605014681816101, 0.8233720660209656, 0.39704567193984985, 0.6069139838218689, 1.0035114288330078, -1.149973750114441, -0.2917144298553467, -0.46137478947639465, 0.9245463013648987, -1.8155779838562012, 0.3007011413574219, -0.3698539137840271, -1.0301166772842407, 1.3060940504074097, -0.2683025300502777, -1.5480260848999023, 2.288576126098633, -0.6178976893424988, 1.3224481344223022], [-0.002071493538096547, -2.408261775970459, 0.36683347821235657, -1.070972204208374, -1.6436028480529785, 0.6745505332946777, -0.4661179780960083, -0.2315911054611206, 0.02265596203505993, 1.1155669689178467, 0.2781811058521271, -0.6681670546531677, -0.32536330819129944, 0.22703036665916443, -0.6740387678146362, -0.3834874927997589, 0.9198267459869385, 0.019879449158906937, 0.4164646565914154, 0.42120108008384705, 0.440571665763855, 0.6883955597877502, -0.2640017867088318, -0.5417231321334839, 1.1917206048965454, -0.4363400638103485, -1.6280306577682495, 0.3124534487724304, 0.3426929712295532, -0.9425266981124878, 1.4005234241485596, 0.6875373125076294, -0.06419875472784042, -1.7997101545333862, 0.2926488220691681, -0.47965720295906067, 0.20527006685733795, 0.18180319666862488, -2.011812686920166, 0.20438575744628906, -0.10712101310491562, -0.20906126499176025, 1.244338035583496, 0.6161960363388062, -0.034408751875162125, -0.8183632493019104, 1.1057567596435547, -0.2426852434873581, -1.3425498008728027, -0.20141291618347168, -1.6696140766143799, 2.087634325027466, 0.3912948966026306, 0.23108084499835968, -1.2119812965393066, -0.3390480875968933, -1.0989257097244263, 0.0287897028028965, 0.8453167676925659, -2.38084077835083, 0.7606760859489441, 0.06915698945522308, -0.7847150564193726, -0.783486008644104, -0.7185065746307373, 0.1527704894542694, 0.1480731964111328, 0.30378058552742004, 0.7229630351066589, 0.07176150381565094], [-1.0225669145584106, 1.0800375938415527, 0.47020402550697327, 1.3401859998703003, -0.8582104444503784, -0.23013556003570557, 1.373876690864563, -0.4944196045398712, -0.4921938180923462, -0.4612347483634949, 0.647083580493927, -0.8186252117156982, -0.38895827531814575, -0.268001526594162, -1.1533702611923218, 1.1744786500930786, -2.3624155521392822, -0.7396417260169983, 0.6722909808158875, 0.5929768085479736, -0.5060340166091919, 0.1932045966386795, 0.8527702689170837, 0.5346518158912659, -1.0936481952667236, -0.909462571144104, -0.27957379817962646, -0.011105785146355629, 1.3648227453231812, 1.2778130769729614, 0.16891659796237946, -0.972452700138092, 0.20288418233394623, -0.43419814109802246, -1.8434971570968628, 0.025571808218955994, 0.05398876219987869, -0.14222382009029388, -0.8775766491889954, 1.2867109775543213, 2.1235101222991943, 1.1948878765106201, 0.4758320152759552, 0.838725745677948, -2.682204008102417, 1.1580615043640137, 0.5049727559089661, -0.15468452870845795, 0.987841010093689, -2.0319807529449463, 0.996905505657196, -1.9761700630187988, -0.6776686906814575, -0.1772036999464035, 0.995602011680603, 0.8309719562530518, 0.8318270444869995, -0.02889031171798706, 0.02458038739860058, 1.2509119510650635, -0.9852043390274048, -1.3167659044265747, -0.8721354007720947, -0.11592540889978409, 1.8306372165679932, 3.325392723083496, 0.01475941576063633, -1.0238187313079834, -0.8271717429161072, -1.6120365858078003], [0.7331114411354065, 0.30359312891960144, 0.6720337867736816, -1.6001336574554443, 0.4467397630214691, 0.11745165288448334, -0.9803134799003601, 0.7589876651763916, 1.0259665250778198, 1.0223468542099, -0.775664210319519, -0.08006616681814194, 0.08100278675556183, 0.7088561654090881, 0.1216738298535347, 0.945457398891449, -0.9758513569831848, -0.6309910416603088, 0.8769654035568237, -1.0434118509292603, 0.4172409176826477, 1.375626564025879, 0.2896488606929779, 0.19841909408569336, -0.421210914850235, -1.170424222946167, 0.48284032940864563, -0.3251582682132721, 1.206682562828064, 0.6048488616943359, 1.057811975479126, -0.6812560558319092, -1.2958009243011475, -0.24975740909576416, 1.3781063556671143, -0.414262592792511, 1.9712860584259033, -0.1955341398715973, 1.1418267488479614, 0.16398681700229645, 0.8694852590560913, 0.9380462765693665, -0.5506090521812439, -0.6776599884033203, 0.2695576548576355, -0.7352404594421387, 0.85373455286026, 0.08138038963079453, 2.1511995792388916, -0.20405429601669312, -0.4153985381126404, -0.6633259654045105, 0.7944678068161011, -0.09953255206346512, 0.3393006920814514, 1.4647021293640137, 1.2076185941696167, 0.9313800930976868, -1.7110850811004639, 1.053689956665039, 0.9972596764564514, 0.7055522799491882, -0.69917231798172, -1.5197252035140991, 1.615975260734558, -0.12403695285320282, -1.0667084455490112, -0.21708600223064423, 1.3779182434082031, -1.0649809837341309], [0.18690799176692963, -0.24223072826862335, 1.1400030851364136, -0.5728675723075867, 1.4179679155349731, 1.3901708126068115, -0.17650087177753448, -0.3889889717102051, -0.0534343346953392, 1.1294256448745728, 2.467782497406006, -0.2944994866847992, -0.011400577612221241, 0.947589635848999, -0.6478811502456665, -0.9505161643028259, -0.5636218190193176, -0.9435321688652039, -0.7886868119239807, 0.9451203346252441, -2.2776002883911133, -1.2630975246429443, 1.2227301597595215, -0.8152902722358704, 0.9149892926216125, -0.8050913214683533, 0.22382883727550507, -0.9546950459480286, -0.21465903520584106, 0.9071950912475586, -0.520025372505188, -1.4629034996032715, 0.16552232205867767, 0.2773789167404175, -1.6584354639053345, -0.8444481492042542, 0.28099390864372253, -0.1972375512123108, -0.47830289602279663, 1.282217025756836, 0.13554668426513672, -1.2310092449188232, 0.9246163964271545, -0.14719158411026, -1.033940076828003, -1.0799834728240967, -1.2474100589752197, 1.1040818691253662, -0.9274593591690063, 0.9744282960891724, -0.19550447165966034, 1.580307126045227, 1.662923812866211, 1.3204785585403442, -1.0646222829818726, -0.9571042656898499, -0.1711643934249878, 0.8830277323722839, -0.43133771419525146, 1.5476220846176147, -0.130707249045372, 0.5194091796875, -0.9008641242980957, -0.47467249631881714, 1.3689073324203491, -0.18300184607505798, 1.282460331916809, -1.3282495737075806, -1.558637022972107, -0.1267494261264801], [0.044961344450712204, 0.49856245517730713, -0.06413029879331589, -0.8414664268493652, -1.9358327388763428, 1.8187124729156494, -0.3702629804611206, -0.1000765934586525, -0.8181808590888977, 0.13862286508083344, 0.029248852282762527, -2.0743417739868164, 0.2636848986148834, 1.1837050914764404, 0.0070642433129251, -0.289988249540329, 2.6651437282562256, -0.8443664312362671, -0.3805667459964752, -0.34523069858551025, 0.0024601442273706198, 1.6971304416656494, 0.7452377676963806, 0.764542281627655, -0.07907047122716904, 0.44416189193725586, -1.0778086185455322, -0.3167816698551178, 0.24595975875854492, 0.7577558755874634, 0.058897390961647034, -0.2053418755531311, 0.9367595314979553, 1.6210936307907104, -0.40018385648727417, 1.0263416767120361, 0.49990081787109375, 0.05993996188044548, -0.4776250123977661, 0.13943418860435486, -1.6276085376739502, 0.23605866730213165, 0.34811341762542725, -0.41265690326690674, -0.3311503827571869, -0.10465358942747116, -0.4896373450756073, -1.532902479171753, -0.20026463270187378, 0.621773898601532, 1.8785542249679565, -0.1738615483045578, 0.2908017933368683, -0.327179491519928, 1.2023711204528809, 2.894320249557495, -0.6166844964027405, 0.8289279341697693, 1.2327182292938232, 0.6263208985328674, -0.46978867053985596, -0.9875054359436035, -1.013787031173706, 0.5279309153556824, -1.2049760818481445, 0.6529529690742493, -0.2884352207183838, 0.6710038185119629, -0.3698498606681824, -0.18307392299175262], [0.5943455100059509, -0.11122681945562363, 1.578979253768921, 0.3614061772823334, 2.9946324825286865, 0.20848232507705688, -1.4884016513824463, 0.46173208951950073, -0.5748655200004578, -0.42447054386138916, -1.4681357145309448, 0.798811137676239, 1.3183789253234863, -1.6490145921707153, 0.8953957557678223, -1.167306900024414, -1.4129949808120728, -0.0601278655230999, -1.2605091333389282, 1.4539875984191895, -1.0054243803024292, -0.011931688524782658, 0.4954260587692261, -0.09599862992763519, -0.020086195319890976, 2.124483108520508, -0.5352742075920105, -0.4632002115249634, -1.4657398462295532, -0.036173541098833084, -0.9506707191467285, -0.9607535600662231, -1.638240098953247, -0.07303182035684586, -0.08062849193811417, -1.6554150581359863, 0.48592624068260193, -0.03243517503142357, 0.3689327836036682, 1.223238468170166, -0.4090445041656494, -0.8480253219604492, 0.00805855356156826, -0.4006132483482361, -1.7265396118164062, -0.026438426226377487, -0.3219771385192871, 1.017977237701416, 0.26367390155792236, 0.3601371645927429, 0.3399035334587097, 2.0111746788024902, -0.08509460091590881, -0.22861310839653015, -0.11140794306993484, 1.0122783184051514, 2.0807788372039795, -0.5772402882575989, 1.3843052387237549, -0.9758347272872925, 0.029834676533937454, 0.11313286423683167, 2.599134922027588, -0.4432402551174164, 0.32328304648399353, 0.42723777890205383, -0.12973502278327942, -0.4456266462802887, -0.18842005729675293, 2.2881715297698975]], "dtype": "float32"}], "e": {"__nd__": [[0.5839036107063293, 0.8868955969810486, 3.6090292930603027, 1.2443486452102661, 3.8877086639404297, 2.078773021697998, -0.6917734742164612, -0.48187652230262756, -0.4354113042354584, 1.1747102737426758, 2.3849456310272217, -0.9055609703063965, 1.0574116706848145, 0.6956631541252136, -1.3852455615997314, -1.6412110328674316, -4.040579319000244, -1.6931164264678955, -0.027130374684929848, 3.209299087524414, -4.517975330352783, -1.4798301458358765, 2.7105510234832764, -0.21449945867061615, -0.199886292219162, 0.32189416885375977, -0.5991618037223816, -2.513307809829712, 0.1582179069519043, 1.6454503536224365, -0.9213744401931763, -3.8094515800476074, -1.6354657411575317, -0.295171320438385, -3.3016481399536133, -2.6001646518707275, 0.5668842196464539, -0.19748935103416443, -0.9919053316116333, 3.7838621139526367, 1.775481104850769, -1.181303858757019, 1.1080045700073242, 0.3458903431892395, -5.403293609619141, -1.5144835710525513, -0.7911567687988281, 1.8259127140045166, -0.4470342993736267, -0.013817884959280491, 1.4155776500701904, 1.8534291982650757, 1.4745714664459229, 1.2470903396606445, -0.8868684768676758, 0.40271925926208496, 3.026231527328491, 1.2305622100830078, 0.5528532862663269, 2.1397640705108643, 0.19147942960262299, -0.4734494090080261, -0.11463150382041931, -1.043433427810669, 3.4565951824188232, 3.4122745990753174, 2.2182881832122803, -4.366794109344482, -2.365947961807251, -0.8073198795318604], [-1.2760088443756104, -5.140268802642822, 0.916046679019928, -0.2667371928691864, -0.9078851342201233, -0.3278919458389282, 0.3456138074398041, -0.9802154302597046, -0.8071720004081726, 0.813959002494812, 1.3446089029312134, 0.9233517646789551, -0.821716845035553, -2.4865760803222656, -1.2904229164123535, -0.8785396218299866, -1.2766869068145752, 0.4681522846221924, -1.1814603805541992, 2.4408111572265625, -0.20943573117256165, -1.225146770477295, -0.5421280860900879, -2.2158124446868896, 2.744511842727661, -0.504482626914978, -2.499819755554199, 1.47991144657135, -0.87018221616745, -1.602526307106018, 1.1276297569274902, 1.1359226703643799, 0.0338924340903759, -4.476606845855713, -1.4886430501937866, -2.126162052154541, -0.8158349990844727, 0.12112877517938614, -4.425792694091797, 1.3346779346466064, 1.140723466873169, -1.4165676832199097, 3.206266403198242, 1.9134172201156616, -1.4666815996170044, 0.19147889316082, 1.1253684759140015, 1.6211390495300293, -3.1746127605438232, -1.3047776222229004, -4.466927528381348, 4.979239463806152, 0.08442558348178864, 1.0139687061309814, -3.1527724266052246, -3.6155498027801514, -2.0143356323242188, -1.805603265762329, 2.109600305557251, -5.100196838378906, -0.26496806740760803, 0.28955915570259094, 0.6047258377075195, -1.2719587087631226, -0.1566140204668045, 0.7180790305137634, 0.8392674922943115, 0.5989565849304199, -0.3894031345844269, 2.2532384395599365], [1.1662852764129639, 1.7713611125946045, 4.43043327331543, -0.4243758022785187, 4.241401672363281, 2.603053331375122, -1.2677538394927979, 0.08690987527370453, 0.62663733959198, 2.484015703201294, 2.655254364013672, -1.4260245561599731, 0.9056208729743958, 2.0856308937072754, -1.7086981534957886, -0.4521311819553375, -5.368514537811279, -2.866891622543335, 0.9320505261421204, 2.102633237838745, -4.752687454223633, -0.31624940037727356, 3.5687813758850098, -0.001282041659578681, -0.7501165270805359, -1.657951831817627, 0.230219304561615, -3.162036657333374, 1.8777625560760498, 3.030017614364624, 0.05625481903553009, -5.132936000823975, -2.5821971893310547, -0.12578026950359344, -2.693272352218628, -2.8547933101654053, 2.7418105602264404, -0.5171975493431091, 0.17680640518665314, 4.34012508392334, 3.0971505641937256, -0.13256974518299103, 0.7046878337860107, -0.3584911525249481, -5.591158390045166, -2.336381435394287, -0.32236728072166443, 1.971071481704712, 1.935686707496643, -0.25911945104599, 1.4180188179016113, 0.5726019740104675, 2.732691526412964, 1.506178617477417, -0.37460049986839294, 1.9378297328948975, 4.179680347442627, 2.6915712356567383, -1.7500320672988892, 4.608041763305664, 0.9100768566131592, 0.132528156042099, -1.7460174560546875, -2.6302313804626465, 5.933666706085205, 3.728703498840332, 1.4350022077560425, -5.111426830291748, -1.6007936000823975, -2.734145164489746], [-2.2513113021850586, 0.2586459219455719, -0.9284830093383789, -2.4683475494384766, -7.077258110046387, 1.964390754699707, 1.8113200664520264, -0.6889309883117676, -1.5645710229873657, 0.296877384185791, 1.3237903118133545, -3.951866626739502, -1.5994420051574707, 1.1576694250106812, -1.5156314373016357, 2.936267375946045, 2.43576717376709, -2.601478338241577, -0.6098181009292603, -1.8934969902038574, 1.475456953048706, 4.076025009155273, 1.5547951459884644, 0.8895424008369446, -0.2806366980075836, -3.3164212703704834, -1.7270301580429077, 1.4619290828704834, 2.966020107269287, 3.2743144035339355, 1.7221981287002563, -0.05672625079751015, 2.96490740776062, 1.1551036834716797, -2.0958306789398193, 2.5088648796081543, 2.1948893070220947, -0.4011749029159546, -2.295088768005371, 0.7710906863212585, 0.47202742099761963, 2.6908743381500244, 2.0683951377868652, 0.23585796356201172, -1.2984833717346191, 2.2247467041015625, 0.21462762355804443, -2.759305000305176, 1.8972653150558472, -2.1795623302459717, 1.500737190246582, -2.975135326385498, 0.2912713289260864, -0.41222912073135376, 3.0554308891296387, 5.097844123840332, -2.6232097148895264, 1.0309405326843262, 0.38117367029190063, 2.324471950531006, -2.7370827198028564, -2.306779623031616, -4.307920455932617, -0.44189801812171936, 0.7201145887374878, 3.469130277633667, -2.208221912384033, 2.799083709716797, -0.7168124914169312, -2.1773788928985596], [-3.4245195388793945, -0.1549963504076004, -4.363040924072266, 1.5815743207931519, -3.508143901824951, -2.5924904346466064, 5.495785236358643, -2.000105619430542, -0.47675853967666626, -1.5960294008255005, 5.202258110046387, 1.7069247961044312, -3.3377599716186523, -0.5139628052711487, -1.2908272743225098, 1.8423223495483398, 0.9445573091506958, 0.9448699951171875, -1.6748102903366089, -1.3764294385910034, 0.6053536534309387, -4.497154235839844, -1.3319424390792847, -1.7856624126434326, 1.2830228805541992, -3.84246563911438, 2.8425397872924805, 2.642934799194336, -0.21364820003509521, 0.552971601486206, -1.3940232992172241, 2.4454729557037354, 5.379612922668457, 0.4490774869918823, -3.165186643600464, 2.935349225997925, -3.628978729248047, -0.30635783076286316, -1.0363210439682007, -2.2700884342193604, 1.0825412273406982, -0.5161287784576416, 1.069619059562683, 1.3982853889465332, 3.5083911418914795, 3.0586178302764893, -2.492504596710205, 0.6540871858596802, -2.380103588104248, -0.7212620377540588, -2.1021952629089355, -2.911543369293213, -0.5360321402549744, 1.6438474655151367, -0.7823017239570618, -7.245619297027588, -6.077697277069092, -1.613525152206421, -2.169062852859497, 2.162081480026245, -3.593628406524658, 0.2308562844991684, -2.4782822132110596, 2.5422914028167725, -0.32311320304870605, -2.3548779487609863, 1.3195453882217407, 2.2890231609344482, -3.23704195022583, -0.7922093868255615], [-1.885697603225708, -2.329958438873291, 0.023187335580587387, 5.631900310516357, -3.374027967453003, -2.0770020484924316, 4.190092086791992, -2.5430705547332764, 0.56815105676651, 0.4355694353580475, 3.646146297454834, -0.6282383799552917, -2.142153739929199, 0.5510520935058594, -4.697782516479492, 0.29703372716903687, -4.570263862609863, 1.7675237655639648, 5.4345011711120605, 3.3960397243499756, -0.702488124370575, -3.684983491897583, -1.1359833478927612, -0.3889956474304199, -0.16574348509311676, -1.9500640630722046, -2.066060781478882, -0.09271988272666931, 2.820866584777832, -2.9023196697235107, 2.358964204788208, 1.0393832921981812, 0.6530551910400391, -6.349173545837402, -3.109522819519043, -0.7187613248825073, -4.6613240242004395, 0.7691153287887573, -5.905962944030762, 1.0175355672836304, 5.407463550567627, 0.7177262306213379, 1.9318313598632812, 4.915224075317383, -3.891839027404785, -0.3644362986087799, 3.8230648040771484, 0.20332109928131104, -4.560162544250488, -4.4596638679504395, -2.4342451095581055, -0.8124681711196899, -2.368422269821167, 0.6686098575592041, -3.287266254425049, -7.011806964874268, -1.1282860040664673, -1.107993721961975, 0.8532282114028931, -3.885502338409424, 1.330633282661438, -2.0298776626586914, -2.8590950965881348, 0.6563512682914734, 0.7405539155006409, 5.760935306549072, 4.104640007019043, -4.892752647399902, -0.5812443494796753, -5.175697326660156], [-0.6485462784767151, 3.974567174911499, 1.237309455871582, 4.027324199676514, 1.1587475538253784, -1.7547831535339355, 1.111565351486206, 0.243305504322052, -0.5372939109802246, -2.3117501735687256, -2.783205032348633, -0.38593775033950806, 0.8866761326789856, -1.729099154472351, -0.443380743265152, 1.9988583326339722, -5.622865200042725, -0.33497151732444763, 2.087700843811035, 1.0448787212371826, -0.04580378904938698, 1.1827588081359863, 0.8949926495552063, 2.3639276027679443, -3.9822187423706055, 1.256664514541626, -0.022335102781653404, -0.5580559372901917, 1.7752748727798462, 1.4917311668395996, -0.38687002658843994, -1.898335337638855, -1.9276587963104248, -0.13507574796676636, -1.1397770643234253, -0.41158053278923035, 0.19936642050743103, -0.06382540613412857, 0.9243764877319336, 1.8445472717285156, 3.5689196586608887, 2.878936767578125, -1.4613078832626343, 0.8447850942611694, -5.143824577331543, 2.3794374465942383, 1.7634259462356567, -0.5085340142250061, 3.7817158699035645, -3.8281264305114746, 3.584398031234741, -4.918690204620361, -2.8929097652435303, -1.9586539268493652, 3.1683616638183594, 3.5246293544769287, 4.922135353088379, -0.6616389751434326, 0.5045313239097595, 1.6220465898513794, -0.7746508121490479, -2.5819406509399414, 1.5119177103042603, 0.15272168815135956, 2.8097589015960693, 6.364352226257324, -0.9551236629486084, -2.398162603378296, 0.12928903102874756, -2.1068053245544434], [0.21797102689743042, 2.208082437515259, -1.9494106769561768, -2.3199737071990967, -5.57322883605957, 0.24597783386707306, 0.30194175243377686, 0.8459204435348511, 1.272179365158081, 0.5837274193763733, -1.9082465171813965, -2.7932145595550537, -0.9366610050201416, 3.113286256790161, -0.15214085578918457, 3.404888391494751, 2.8636090755462646, -0.8092584609985352, 3.1647772789001465, -4.306510925292969, 3.6661324501037598, 4.716067314147949, -0.39088496565818787, 2.270179510116577, -2.110783815383911, -2.5960826873779297, 0.3404570519924164, 0.4303063154220581, 3.8712000846862793, 1.1079860925674438, 2.7322287559509277, 0.9516778588294983, 0.7147478461265564, 1.5649032592773438, 2.9712250232696533, 3.2229597568511963, 2.3405323028564453, -0.018841015174984932, 1.661427617073059, -2.111485719680786, 0.17237980663776398, 3.90193772315979, -1.716727375984192, -0.8318804502487183, 2.530371904373169, 0.18732817471027374, 2.0989646911621094, -3.6447811126708984, 3.876974582672119, -1.1718318462371826, 1.8870205879211426, -5.736349582672119, -0.3592979609966278, -1.6377846002578735, 3.3167991638183594, 5.058440685272217, -0.4776988923549652, 1.9744981527328491, -2.436927556991577, 2.4823200702667236, 0.8312230706214905, -0.8784738183021545, -3.866633415222168, -0.3885515332221985, -0.2702430188655853, 0.6718054413795471, -2.830111265182495, 1.7274141311645508, 3.0274887084960938, -4.430391788482666], [-1.996516466140747, -1.3870081901550293, -0.38963407278060913, 0.34223851561546326, 4.802549839019775, -4.122546195983887, 0.4350960850715637, 0.6451618671417236, -1.4892799854278564, -3.2502880096435547, -2.4224250316619873, 5.833395004272461, 0.19397996366024017, -7.781641006469727, 2.6862382888793945, 0.9350743889808655, -4.395535945892334, 0.9388176798820496, -5.985435485839844, 0.6719185709953308, 0.8551112413406372, -2.113450050354004, -1.024390697479248, -2.2820069789886475, 0.9661181569099426, 1.2751586437225342, 1.9076850414276123, 3.3912582397460938, -4.007596969604492, 0.645880401134491, -2.892709732055664, 0.8703570365905762, -0.6966506242752075, -0.9806597828865051, -0.5637013912200928, -2.1105237007141113, 0.259541392326355, -0.7570726275444031, 1.5836869478225708, 0.3419300615787506, 0.9093349575996399, -1.2163333892822266, 0.5039046406745911, -0.5414322018623352, 0.4084509611129761, 4.648658752441406, -1.928873896598816, 3.8683884143829346, 2.095890998840332, -1.1577714681625366, -3.43855357170105, 2.025113582611084, -1.3441556692123413, -0.0663079172372818, -0.07737135142087936, -2.7832558155059814, 0.5241481065750122, -4.392039775848389, 0.31237784028053284, -1.8066387176513672, -3.6615731716156006, 1.6181843280792236, 6.430360794067383, -0.6501213312149048, 1.6512713432312012, -2.2537200450897217, -2.5489661693573, 4.168551445007324, -1.106114387512207, 7.268596649169922], [-2.705069065093994, -4.541369915008545, -2.9108688831329346, -0.7316577434539795, -5.525157928466797, -2.315174102783203, 2.2448503971099854, -0.8841850757598877, -0.9821352958679199, -1.0243173837661743, -0.21275301277637482, 1.1302523612976074, -2.1868538856506348, -3.09892201423645, -0.37563830614089966, 1.566809058189392, 2.131571054458618, 1.4736970663070679, -1.409894585609436, -0.8250358700752258, 3.843045234680176, 0.258247435092926, -2.49882435798645, -1.5342758893966675, 2.1465563774108887, -1.4330846071243286, -1.5590722560882568, 4.023340702056885, -0.4207225441932678, -1.8311893939971924, 1.5053932666778564, 4.261238098144531, 2.5968732833862305, -3.1819398403167725, 0.24025596678256989, 1.1803478002548218, -1.4659175872802734, 0.13673174381256104, -3.44938325881958, -1.9078917503356934, 0.06794404983520508, 0.4203493595123291, 2.221632957458496, 1.725327968597412, 2.716714859008789, 2.8382604122161865, 1.2980399131774902, -0.5410614609718323, -1.9902212619781494, -2.087218999862671, -4.431529998779297, 1.2004376649856567, -1.6955854892730713, -0.28161555528640747, -1.0345537662506104, -3.1513636112213135, -5.004223346710205, -2.9109647274017334, 1.4477779865264893, -5.257748603820801, -1.7936668395996094, -0.18372540175914764, -0.0259628314524889, 0.29140809178352356, -2.7000763416290283, -1.0717564821243286, -1.4260225296020508, 4.808163642883301, 0.7883604168891907, 2.180764675140381], [-2.297065019607544, 0.12898923456668854, -1.172558307647705, 1.3564494848251343, -1.6767466068267822, -1.7974772453308105, 3.4441916942596436, -1.163486361503601, -0.0692039504647255, -0.5443342924118042, 3.004257917404175, 0.7433447241783142, -2.0375287532806396, -0.705842137336731, -1.737096905708313, 2.1165926456451416, -2.8279194831848145, -0.18926820158958435, 0.07892563194036484, -0.11521504074335098, -0.14572466909885406, -2.1012191772460938, -0.001032897736877203, -0.8578459620475769, -0.040016382932662964, -3.2859928607940674, 1.2887930870056152, 1.2897588014602661, 1.3506696224212646, 1.1874523162841797, 0.034646205604076385, 0.17226794362068176, 1.9146137237548828, -1.2640446424484253, -2.619060754776001, 0.6228276491165161, -1.0857151746749878, -0.34905779361724854, -1.2183886766433716, 0.27222684025764465, 3.224726438522339, 0.8018273711204529, 1.0379558801651, 1.4284496307373047, -0.6121835112571716, 2.0326483249664307, -0.18691447377204895, 0.9032931923866272, 0.14422005414962769, -2.36498761177063, -1.4371623992919922, -2.483377695083618, -0.4358813166618347, 0.8975306153297424, -0.2312612682580948, -3.4571714401245117, -1.7740780115127563, -0.7770062685012817, -1.8109512329101562, 1.753738522529602, -1.9669601917266846, -0.34346237778663635, -1.909127116203308, 0.13556957244873047, 2.4159350395202637, 1.487977147102356, 0.49178656935691833, -0.09259705245494843, -1.673716425895691, -1.8572224378585815], [1.0892740488052368, -3.985961675643921, 1.1040637493133545, 2.917827844619751, 1.4325510263442993, 1.7421848773956299, -1.2774863243103027, -1.3818258047103882, -1.236527681350708, 0.34549927711486816, 1.213261604309082, -0.19157163798809052, 1.3427737951278687, -0.057635482400655746, -0.48964065313339233, -5.805624961853027, 3.133955478668213, 2.2857019901275635, -0.12349772453308105, 4.754115104675293, -2.40929913520813, -3.4038147926330566, -0.9547511339187622, -1.0696954727172852, 2.7077925205230713, 5.029850959777832, -3.553431510925293, -1.4146977663040161, -3.698110818862915, -4.664407253265381, -1.0207821130752563, 1.2986559867858887, 0.04147934913635254, -2.0547261238098145, -1.1585131883621216, -1.7762778997421265, -4.652994632720947, 1.134440541267395, -4.244769096374512, 0.364400714635849, -3.473116636276245, -4.369145393371582, 1.7250689268112183, 1.810996174812317, -1.1483298540115356, -2.875797748565674, -0.2581165134906769, 0.4451275169849396, -8.34125804901123, 2.628690242767334, -0.8110669851303101, 7.48610782623291, -0.4622326195240021, 0.7855026125907898, -4.3561625480651855, -4.466526985168457, -1.432117223739624, -1.162115454673767, 5.699328422546387, -6.957881927490234, 1.8943601846694946, -0.41222986578941345, 2.445626974105835, 2.2102293968200684, -5.428490161895752, -0.8326561450958252, 4.598435401916504, -2.3829405307769775, -1.363043189048767, 3.0841448307037354], [-0.8662814497947693, 2.4324698448181152, -1.362596869468689, 4.42877197265625, 2.0634336471557617, -4.43192195892334, 3.1954493522644043, -0.40511637926101685, 1.6678388118743896, -1.6181769371032715, 0.732917308807373, 3.0287251472473145, -1.1567636728286743, -0.9019166827201843, -0.8844243288040161, 1.9530067443847656, -5.680300712585449, 1.8279942274093628, 2.7922823429107666, -0.08629487454891205, 0.17510472238063812, -3.9052326679229736, -1.4231435060501099, 0.06454555690288544, -2.1079723834991455, -1.507210612297058, 3.3514657020568848, 0.22362764179706573, 1.1297669410705566, -0.6902037262916565, -0.6076505184173584, 0.26458895206451416, -0.37336623668670654, -1.5896443128585815, -0.3195275366306305, 0.11671043187379837, -2.578917980194092, -0.030353492125868797, 1.7294986248016357, -1.0024094581604004, 4.387469291687012, 1.146584391593933, -2.0241165161132812, 1.4274660348892212, 0.18417437374591827, 1.3395662307739258, 0.9215793013572693, 1.5746055841445923, 1.1290158033370972, -2.46919584274292, -0.7693272233009338, -4.721363544464111, -2.0566470623016357, 0.028858456760644913, -0.43844467401504517, -5.526559352874756, 1.3782461881637573, -1.2269843816757202, -3.3318026065826416, 1.7058560848236084, 0.1610708087682724, 0.32394614815711975, 0.33545589447021484, 0.8633517622947693, 2.5938148498535156, 0.8431915044784546, 1.1872830390930176, -2.6122963428497314, -0.09996181726455688, -2.995227336883545], [0.65870600938797, -2.042706251144409, 1.448912501335144, -3.304945230484009, -2.515336751937866, 2.178896427154541, -2.9897592067718506, 1.0193597078323364, -1.0032750368118286, 0.8074905872344971, -3.771622896194458, -2.3324973583221436, 1.1590596437454224, -0.6653619408607483, 0.8458996415138245, 0.02716241031885147, 2.5947494506835938, -1.1627140045166016, -0.6569080948829651, -0.3433041274547577, 1.9486051797866821, 5.443421840667725, 0.5097475647926331, 0.8738740086555481, 0.34107479453086853, 1.4739731550216675, -3.5857338905334473, 0.530258059501648, 0.3814423680305481, 0.11601832509040833, 2.0163779258728027, 0.2488546520471573, -1.4547805786132812, -0.5027467608451843, 1.9618122577667236, -0.5194540023803711, 3.203554630279541, 0.10442431271076202, -1.353007197380066, 0.8869047164916992, -1.8714814186096191, 1.1511523723602295, 1.0538945198059082, -0.741557240486145, -1.1642842292785645, -0.1749611645936966, 1.8560986518859863, -1.9336811304092407, 1.463561773300171, -0.3495444059371948, 0.31182560324668884, 2.3665552139282227, 0.0971585288643837, -1.5194898843765259, 1.459456443786621, 6.648853778839111, 0.8335334658622742, 0.30937299132347107, 2.8562912940979004, -3.1840155124664307, 0.5648399591445923, -0.9148287177085876, 0.6055538058280945, -1.7951161861419678, -1.4851975440979004, 1.5406692028045654, -2.822603940963745, 2.4887185096740723, 2.4091248512268066, 1.874571442604065], [1.260353446006775, -0.74594646692276, -2.687681198120117, 3.017036199569702, -0.2651764452457428, -3.5206663608551025, -1.5850224494934082, 1.207611083984375, 0.01822615973651409, -3.4357540607452393, -7.8745245933532715, 2.3195836544036865, 1.733322262763977, -3.1777472496032715, 3.2497031688690186, -1.3380463123321533, 3.6538991928100586, 4.64307165145874, 1.440106987953186, -0.5122319459915161, 5.165219783782959, 1.1743711233139038, -4.526211738586426, 1.6972709894180298, -1.240010142326355, 6.634167194366455, -1.2807927131652832, 1.5442436933517456, -2.965764284133911, -5.737766742706299, -0.19592413306236267, 4.9072771072387695, -1.5206753015518188, -0.728934109210968, 5.805131435394287, 1.04628586769104, -3.2327330112457275, 1.1785699129104614, 1.4653584957122803, -4.329685211181641, -3.673413038253784, 0.24779444932937622, -3.1727540493011475, 0.20949985086917877, 4.048142433166504, 0.9485090374946594, 2.57401967048645, -2.204397678375244, -1.3807001113891602, 0.3536161184310913, 0.30172091722488403, 0.1993662714958191, -4.813204765319824, -3.513103723526001, 0.5434476733207703, 0.045207131654024124, 0.8260371685028076, -3.30399489402771, 3.820251226425171, -8.042647361755371, 2.1013216972351074, -0.4872334599494934, 6.013239860534668, 2.7907140254974365, -7.4914960861206055, -2.5744094848632812, -1.400363802909851, 2.7192177772521973, 4.2718610763549805, 3.719719648361206], [-0.4986834228038788, -0.34937024116516113, 0.9264759421348572, -2.5001769065856934, 1.0689501762390137, -0.24136090278625488, -0.2998131215572357, 0.5727255344390869, 0.29368147253990173, 0.761749804019928, 0.31814536452293396, 0.9208460450172424, -0.46499985456466675, -1.122476577758789, 0.12790463864803314, 1.7399660348892212, -2.626333713531494, -1.5066725015640259, -1.580416202545166, -0.864711582660675, -0.14279139041900635, 0.8396715521812439, 0.9689398407936096, -0.9497832655906677, 0.37708142399787903, -2.496924877166748, 1.0090088844299316, 0.7559496164321899, 0.7163191437721252, 2.201774835586548, 0.4056049585342407, -1.2207410335540771, -0.7416988611221313, -0.4635832607746124, -0.2629498243331909, -1.0525157451629639, 2.8610785007476807, -0.7079183459281921, 0.8195269107818604, 1.2012128829956055, 1.8306952714920044, 0.6391362547874451, 0.702177882194519, -0.7921451926231384, -0.43384891748428345, 0.8149824142456055, -0.2872750759124756, 1.50485360622406, 2.9992001056671143, -0.8347346186637878, -1.7151700258255005, -0.0005825274274684489, 1.416661024093628, 0.6285078525543213, 0.33121052384376526, 0.8680071830749512, 0.5821103453636169, 0.19463226199150085, -2.177579879760742, 2.009359359741211, -0.9600556492805481, 1.120311975479126, -0.3305670917034149, -2.3458480834960938, 3.5415408611297607, -0.11474920809268951, -1.7990612983703613, 1.0837273597717285, 0.03466596454381943, 0.507492184638977], [-1.822324275970459, -1.2149547338485718, -1.0358319282531738, 2.5323572158813477, 2.295747995376587, -3.1717121601104736, 1.7238998413085938, -0.5477623343467712, -1.2186317443847656, -2.646214008331299, -0.22614726424217224, 3.761197328567505, -0.3227235972881317, -4.793592929840088, 0.8666579127311707, -0.20503003895282745, -2.4899816513061523, 1.6631451845169067, -3.0123367309570312, 1.5183954238891602, 0.15456941723823547, -3.3468070030212402, -1.3189735412597656, -1.5741649866104126, 0.802666187286377, 1.3541343212127686, 0.9097557663917542, 2.1245081424713135, -2.9241859912872314, -0.8900355696678162, -2.2381789684295654, 1.4009904861450195, 0.6566828489303589, -1.3015645742416382, -1.4797935485839844, -0.9080736041069031, -2.430715560913086, -0.1282561868429184, -0.4098081886768341, -0.22510011494159698, 0.658810019493103, -1.4374191761016846, 0.596796452999115, 0.9149383306503296, 0.1039055585861206, 2.9966073036193848, -1.2247459888458252, 2.3341012001037598, -1.2634414434432983, -0.9448286890983582, -2.198310136795044, 1.322598934173584, -1.8049136400222778, 0.19019778072834015, -1.0449190139770508, -4.593627452850342, -0.8348070979118347, -3.445511817932129, 1.1811643838882446, -2.3715853691101074, -2.3818511962890625, 0.3886055648326874, 3.9132790565490723, 1.094975233078003, -0.3309372365474701, -0.9874871373176575, 0.18718871474266052, 1.6497089862823486, -1.6328072547912598, 4.066213130950928], [-0.9484452605247498, 2.6154561042785645, 0.04654247313737869, -4.698708534240723, -2.5010929107666016, 3.777736186981201, -0.8189113736152649, 0.6033581495285034, -2.467381477355957, -0.3145502209663391, 0.0826597809791565, -3.2809715270996094, 0.3738276958465576, 0.569994330406189, 1.39466392993927, 1.558196783065796, 4.050811767578125, -3.9820759296417236, -4.902146339416504, -2.5073013305664062, -0.2280549257993698, 4.783977508544922, 3.1809628009796143, 0.9643707275390625, -0.25378814339637756, -1.0725176334381104, -0.11397721618413925, 0.563572108745575, 0.18966533243656158, 5.6421895027160645, -1.2684040069580078, -2.069296360015869, 2.3703391551971436, 5.896286487579346, -1.8424681425094604, 2.144718885421753, 4.451364517211914, -0.949763834476471, 1.5484308004379272, 1.3070992231369019, -2.9496779441833496, 1.1004704236984253, 0.9840030670166016, -2.8717737197875977, -0.6077921986579895, 2.2941246032714844, -3.501542329788208, -1.8842943906784058, 4.015448093414307, 1.2279430627822876, 4.108275890350342, -1.6875461339950562, 2.0594308376312256, -0.2908456325531006, 4.822323322296143, 8.881689071655273, -0.8993171453475952, 1.7356518507003784, 0.4471991956233978, 5.843748092651367, -4.08701753616333, -1.1882392168045044, -1.5913251638412476, -0.35232722759246826, 0.9708062410354614, 0.43783459067344666, -3.516695261001587, 4.337700366973877, -2.0743792057037354, 1.7258714437484741], [1.0826315879821777, 2.7430312633514404, -1.2642823457717896, 3.903148651123047, 5.028966426849365, -2.371690273284912, 0.5138999223709106, 0.136286199092865, 0.8898504972457886, -1.9951061010360718, -0.30646947026252747, 3.154770851135254, 0.8842051029205322, -0.347006618976593, 1.5075658559799194, -1.753084659576416, -0.8758482336997986, 2.4790256023406982, 0.35163766145706177, 0.5098178386688232, -1.0711498260498047, -4.475616931915283, -1.4539636373519897, 0.14612333476543427, -1.068166732788086, 2.794760227203369, 3.2035036087036133, -1.0874876976013184, -2.5496087074279785, -1.8772245645523071, -3.0414581298828125, 0.3840445280075073, -0.35601601004600525, 1.7834779024124146, 0.7844762802124023, 0.18016605079174042, -3.475515127182007, 0.2954068183898926, 3.4021265506744385, -1.9435745477676392, -0.9520037770271301, -1.794761061668396, -2.96441388130188, -0.378036767244339, 2.2028024196624756, -0.3634517192840576, -1.7748847007751465, 1.3628212213516235, -1.35475754737854, 2.0247113704681396, 1.266900897026062, -1.197591781616211, -1.2363929748535156, 0.04067255184054375, -0.809972882270813, -4.568946361541748, 1.3019719123840332, -0.9406735301017761, -0.916893720626831, 0.930964469909668, 0.7050599455833435, 1.0738471746444702, 3.0892364978790283, 2.692293643951416, -1.3828738927841187, -3.1659743785858154, 2.1687915325164795, -1.9753196239471436, -0.8402925133705139, 0.8006415367126465], [-2.97768497467041, 0.7659578919410706, -1.4482085704803467, -2.258697986602783, -4.88775110244751, -0.26127660274505615, 3.195160388946533, -0.6460523009300232, -0.6183478832244873, 0.033215463161468506, 2.6864540576934814, -1.3001002073287964, -2.580106258392334, -0.003469334449619055, -1.5311349630355835, 4.369389533996582, -1.0092450380325317, -2.434776544570923, -1.2430095672607422, -2.5412158966064453, 1.1448851823806763, 1.7095390558242798, 1.270305871963501, -0.3137422502040863, -0.24642819166183472, -5.589111328125, 1.0456527471542358, 2.193105936050415, 3.0338103771209717, 4.113348007202148, 1.0575408935546875, -0.35993659496307373, 2.939115524291992, 0.5580559968948364, -2.540836811065674, 2.0042340755462646, 2.267730236053467, -0.8952900767326355, -0.7468240261077881, 0.5846223831176758, 2.862765312194824, 2.6718525886535645, 1.6757341623306274, 0.1633049100637436, -0.2872125506401062, 3.209318161010742, -0.5251858234405518, -0.570382297039032, 3.3857884407043457, -2.8483781814575195, -0.5256584882736206, -4.301356315612793, 0.6951472759246826, 0.5353851914405823, 2.3800504207611084, 1.5864516496658325, -2.6336593627929688, 0.4160885512828827, -2.7381088733673096, 4.469819068908691, -3.5227837562561035, -0.7957158088684082, -4.137319564819336, -1.1257836818695068, 3.620645761489868, 2.0725553035736084, -2.3372244834899902, 2.7304563522338867, -1.3257269859313965, -2.3698651790618896], [-2.1662371158599854, -0.7803176641464233, -3.4687275886535645, -0.33630117774009705, -6.091701984405518, -1.0636601448059082, 2.277763843536377, -0.609470784664154, -1.231512427330017, -1.715535044670105, -0.594024658203125, -0.7851144671440125, -1.5658708810806274, -0.9515113830566406, 0.1433362364768982, 1.8602328300476074, 4.0889692306518555, 0.6628963947296143, -0.9003444314002991, -2.099456787109375, 3.5621728897094727, 1.5733673572540283, -1.5421291589736938, 0.3504367172718048, 0.23865601420402527, -0.7553321123123169, -0.7043609023094177, 2.824007272720337, 0.2512677013874054, -0.2468147724866867, 0.5500884652137756, 3.236229181289673, 3.382500648498535, 0.5793222784996033, 0.17971758544445038, 3.1228556632995605, -1.1362621784210205, 0.1104046031832695, -1.3228403329849243, -2.359619379043579, -1.3239420652389526, 1.3788065910339355, 0.594511866569519, 0.5939071774482727, 2.742003917694092, 2.9149975776672363, 0.1426415592432022, -2.3995063304901123, -0.4606894850730896, -1.1766961812973022, -0.17478758096694946, -2.226850986480713, -1.8088738918304443, -1.0423997640609741, 1.680938720703125, 0.5193427205085754, -4.078721046447754, -1.427196979522705, 1.1637673377990723, -1.5230427980422974, -2.247499942779541, -1.3197014331817627, -1.0225603580474854, 1.7190569639205933, -3.1827831268310547, -0.44595691561698914, -1.7145352363586426, 4.414724826812744, 0.24607926607131958, 0.49574992060661316], [-0.2975262403488159, -1.8621485233306885, -0.32463017106056213, -0.9245121479034424, 2.4581034183502197, -1.71477472782135, 0.147192120552063, 0.14113441109657288, 0.7061746120452881, 0.23626776039600372, 0.9438577890396118, 3.138685703277588, -0.6504414081573486, -1.827286720275879, 0.5674349665641785, 0.03069022111594677, -1.6993998289108276, 0.683137834072113, -1.762283444404602, -0.09958522766828537, -0.1098095029592514, -2.4021618366241455, -0.8570687770843506, -2.0822205543518066, 1.6511865854263306, -1.200545072555542, 1.7091710567474365, 1.0948392152786255, -1.3920063972473145, -0.4972912073135376, -0.5335131883621216, 0.6211816072463989, -0.23454558849334717, -1.398442029953003, 0.3437899053096771, -1.2275108098983765, -0.06147436052560806, -0.3205072581768036, 0.6370725035667419, -0.46180739998817444, 0.7329137325286865, -1.379502534866333, 0.380513072013855, -0.1892032027244568, 2.0589535236358643, 0.22345104813575745, -0.8238826394081116, 2.5474421977996826, -0.2724152207374573, 0.5248117446899414, -3.6403934955596924, 2.005138635635376, 0.8698952198028564, 1.2036774158477783, -2.0972790718078613, -4.309180736541748, -1.0323848724365234, -1.1306724548339844, -1.7880067825317383, -0.5927144885063171, -0.2548234760761261, 2.259842872619629, 1.2609302997589111, -0.9263652563095093, 1.1742256879806519, -3.239231824874878, 0.0006923512555658817, 0.9916447401046753, -0.19015054404735565, 2.0774435997009277], [0.5111373066902161, -0.5298989415168762, 1.5535842180252075, -4.440186023712158, -6.742132186889648, 3.5203170776367188, -0.8796970844268799, 0.26640552282333374, 1.2747880220413208, 4.030786037445068, 0.657698392868042, -5.689124584197998, -1.1011825799942017, 5.025792598724365, -2.813524007797241, 2.8287580013275146, 1.1331082582473755, -3.133347749710083, 4.4044060707092285, -2.215897560119629, 1.2894588708877563, 6.2379021644592285, 1.9081016778945923, 1.5889846086502075, -0.7367292046546936, -4.788264274597168, -2.7252469062805176, -0.9851366877555847, 6.1652607917785645, 2.106565475463867, 5.298220634460449, -1.4180145263671875, -0.5382189154624939, -1.3575273752212524, 0.962958574295044, 0.9164795279502869, 4.64716911315918, -0.09332270920276642, -2.250044822692871, 1.444624900817871, 2.1832950115203857, 3.8783762454986572, 1.3002744913101196, 0.17074723541736603, -1.9245010614395142, -2.280336856842041, 3.7560901641845703, -3.253073215484619, 3.0009849071502686, -2.177551507949829, 0.5414982438087463, -2.349259614944458, 2.0118112564086914, -0.34090694785118103, 1.585947871208191, 6.668145656585693, 0.09326700866222382, 3.9249048233032227, -1.7803202867507935, 1.873932957649231, 2.1303558349609375, -1.324420690536499, -7.026691913604736, -3.358900547027588, 2.5399317741394043, 4.446727752685547, -1.7946339845657349, -0.9183010458946228, 2.8132176399230957, -6.5305585861206055], [-0.061979785561561584, 0.7293667197227478, 4.520971775054932, 1.9697270393371582, 1.9876039028167725, 2.2405362129211426, 0.7401261925697327, -1.2367806434631348, 0.20929642021656036, 2.4801223278045654, 4.433432102203369, -2.372089147567749, -0.026253532618284225, 2.1629958152770996, -3.904282331466675, -0.328885942697525, -6.879856109619141, -2.512122869491577, 2.7283101081848145, 3.82010817527771, -5.199859619140625, -1.448971152305603, 3.4826221466064453, -0.04948641359806061, -0.7620958089828491, -2.378633499145508, -1.2751023769378662, -3.0514488220214844, 3.080937385559082, 2.2613677978515625, 0.9583826661109924, -4.818981170654297, -1.79149329662323, -2.5624310970306396, -4.82089900970459, -2.9632625579833984, 0.7975218296051025, -0.2126830518245697, -3.0432472229003906, 5.208907127380371, 5.1748433113098145, 0.2405695617198944, 2.0886788368225098, 1.8737272024154663, -8.027739524841309, -2.0264666080474854, 1.077720046043396, 1.706551194190979, -0.2027645707130432, -2.3966293334960938, 0.9031819701194763, 0.3204318583011627, 1.6362242698669434, 1.725142478942871, -1.3469187021255493, -0.18321314454078674, 3.375924587249756, 2.1628754138946533, -0.5702783465385437, 2.962184429168701, 0.7956264615058899, -1.342725396156311, -3.160475254058838, -2.1691906452178955, 6.068584442138672, 7.035055160522461, 3.0965428352355957, -6.851871013641357, -2.3912837505340576, -4.708881378173828], [-1.0829225778579712, 7.893211841583252, -1.9997313022613525, 2.2667975425720215, 1.7463793754577637, -3.717719793319702, 3.47133469581604, 0.7342480421066284, 1.6253141164779663, -2.4141013622283936, -0.011338720098137856, 1.6660579442977905, -1.0954036712646484, 0.4399324655532837, 0.31285834312438965, 5.171139717102051, -5.662949562072754, -0.8020578026771545, 1.6183515787124634, -3.7008445262908936, 0.5822666883468628, -0.8072792291641235, 0.5994234681129456, 1.8926514387130737, -4.807946681976318, -3.6142354011535645, 6.229102611541748, 0.0314871147274971, 3.026068687438965, 4.210786819458008, -1.5830200910568237, -1.9808390140533447, 0.3623533546924591, 3.6111435890197754, -0.3318779468536377, 2.4133989810943604, 1.0575828552246094, -0.9267363548278809, 6.253585338592529, -0.9883708953857422, 4.553555488586426, 3.8387370109558105, -3.968609571456909, -1.1714774370193481, 0.5595634579658508, 3.295565128326416, -1.0375442504882812, 0.2880609631538391, 7.4596662521362305, -2.537264585494995, 3.461878538131714, -10.926350593566895, -1.190548300743103, -0.7313713431358337, 4.530595779418945, 0.8333346843719482, 3.0084421634674072, 0.7167822122573853, -6.589415550231934, 9.589936256408691, -2.1026947498321533, -0.13605383038520813, -1.354176640510559, 0.5721052289009094, 5.710712909698486, 1.347902774810791, -1.969977855682373, -0.9831365942955017, -0.5836836695671082, -4.826488494873047], [0.7746822237968445, 3.82254695892334, -0.7863584160804749, 4.0746049880981445, 1.5482566356658936, -1.4265055656433105, 0.04254654794931412, 0.465138703584671, -0.285673052072525, -2.8079471588134766, -3.4915802478790283, 0.219363734126091, 1.5760339498519897, -0.5053675174713135, 1.3689947128295898, -0.49840837717056274, -0.035170238465070724, 1.6074491739273071, 1.5820297002792358, 0.2313617616891861, 0.7160693407058716, -0.009609628468751907, -0.8449183106422424, 2.421187162399292, -3.026716947555542, 4.056267738342285, 0.5618352890014648, -0.8995416164398193, -0.8273967504501343, -1.0982880592346191, -1.6908724308013916, 0.27009350061416626, -0.827661395072937, 2.1276450157165527, 1.1706610918045044, 1.0851342678070068, -2.082862138748169, 0.4900687038898468, 2.402564287185669, -1.1821465492248535, -1.0076416730880737, 0.821344256401062, -2.9788498878479004, -0.13208623230457306, -0.5080663561820984, 0.8997403383255005, 0.26468414068222046, -1.4675838947296143, 0.7287649512290955, -0.12721063196659088, 4.209659099578857, -3.3745250701904297, -2.9793381690979004, -2.114933490753174, 2.2978291511535645, 1.8289519548416138, 2.9247846603393555, -0.7512626051902771, 1.5306648015975952, -0.0010182724799960852, 0.3703119456768036, -1.6098604202270508, 2.8347649574279785, 2.6246514320373535, -2.201719284057617, 1.3111869096755981, 0.13346482813358307, -1.1970077753067017, 0.537894070148468, -0.18745797872543335], [-0.7665695548057556, 1.6874604225158691, 2.3355813026428223, -1.9261958599090576, 3.3749661445617676, 1.6084375381469727, -1.066641092300415, 0.5847560167312622, -2.0901334285736084, -0.7595089077949524, -0.07871359586715698, 0.10189494490623474, 1.141165018081665, -2.5580384731292725, 1.2405318021774292, 0.4509001672267914, -2.4574267864227295, -2.9795303344726562, -5.08233118057251, 0.6791808605194092, -2.412731409072876, 1.3258569240570068, 3.0862555503845215, -0.36406606435775757, -0.15094569325447083, 0.1919412910938263, 0.45232048630714417, 0.07821999490261078, -1.2955459356307983, 4.570533275604248, -2.348217010498047, -3.3591299057006836, -0.4898003935813904, 2.7251272201538086, -2.870448350906372, -1.4597758054733276, 3.4474234580993652, -1.0064224004745483, 1.3623149394989014, 3.290440082550049, -0.004062640480697155, -0.3384484350681305, 1.1593974828720093, -1.8358142375946045, -3.61177134513855, 2.2294464111328125, -3.005558490753174, 1.6937249898910522, 3.5562922954559326, 0.08969025313854218, 1.8571194410324097, 0.8134496212005615, 1.5050996541976929, 0.3836730420589447, 2.3809397220611572, 4.705974578857422, 2.2227365970611572, 0.012779991142451763, 0.48120254278182983, 3.99955415725708, -3.4655001163482666, -0.19609613716602325, 1.9258265495300293, -1.4374229907989502, 3.716090202331543, 1.5294322967529297, -2.0404915809631348, 1.4133856296539307, -2.6728334426879883, 3.436617851257324], [-0.18323753774166107, 1.2464922666549683, -0.029455678537487984, -0.745793879032135, -0.21633026003837585, -0.4623928368091583, 0.5954227447509766, 0.2745043635368347, 0.8694078922271729, 0.4904431104660034, 0.5063861012458801, -0.03567056357860565, -0.6413818597793579, 0.7546728253364563, -0.4485563635826111, 1.8149778842926025, -1.7586954832077026, -0.806684672832489, 0.8580861687660217, -1.2872767448425293, 0.20001272857189178, 0.509189248085022, 0.4661259353160858, 0.1951202154159546, -0.8330168724060059, -2.294982433319092, 1.3262673616409302, -0.015919987112283707, 1.7324808835983276, 1.4321393966674805, 0.6486150622367859, -0.8034310340881348, -0.20439794659614563, 0.13795620203018188, 0.11799634993076324, 0.29350876808166504, 1.3760021924972534, -0.3498225808143616, 1.1348488330841064, 0.1428474485874176, 1.8628607988357544, 1.347882866859436, -0.485232412815094, -0.3060401678085327, 0.05386413261294365, 0.2455338090658188, 0.3355365991592407, 0.15309251844882965, 2.3998184204101562, -0.9336811304092407, -0.05556478723883629, -2.4444339275360107, 0.5244641900062561, 0.15433555841445923, 0.7985190749168396, 0.5419785380363464, 0.5923707485198975, 0.8107950091362, -2.4819839000701904, 2.6453335285186768, -0.06969272345304489, 0.31915125250816345, -1.5625007152557373, -0.982904851436615, 2.4564900398254395, 0.5299326777458191, -0.8098676800727844, -0.3330124318599701, 0.36239394545555115, -2.0529212951660156], [-1.0166614055633545, 3.570972204208374, -1.8232437372207642, 4.532215595245361, -2.5419416427612305, -2.5375308990478516, 2.605227470397949, -0.30715411901474, -0.11787089705467224, -2.767726182937622, -2.2666735649108887, -0.7155023813247681, -0.2654290497303009, -0.27624523639678955, -0.5449625253677368, 2.1380417346954346, -1.485123872756958, 1.2679295539855957, 3.3603551387786865, -0.5929029583930969, 2.1171834468841553, 0.6162703633308411, -1.0617140531539917, 2.604728937149048, -3.6186392307281494, 0.9400392770767212, 0.2904885709285736, 0.333927184343338, 1.8897526264190674, -0.41896289587020874, 0.1035192683339119, 0.9508110880851746, 0.46638375520706177, 0.321919709444046, 0.12456336617469788, 2.180119276046753, -2.166696310043335, 0.3918860852718353, 0.5906120538711548, -1.1863003969192505, 1.925389289855957, 3.0016303062438965, -2.1034109592437744, 1.3644341230392456, -1.22117018699646, 2.368535041809082, 1.9612367153167725, -2.313930034637451, 1.58383309841156, -3.1422107219696045, 3.289997100830078, -6.380838871002197, -3.8374202251434326, -2.2389883995056152, 2.887087821960449, 1.2543152570724487, 1.540981650352478, -0.9102858304977417, 0.3930274546146393, 0.40101224184036255, -0.43518173694610596, -2.7058329582214355, 0.02360253967344761, 2.219411611557007, -0.7463295459747314, 4.039366245269775, -0.3886767327785492, -0.9488716721534729, 0.7767477631568909, -3.228536605834961], [-2.4532132148742676, -4.501652717590332, 0.8479204773902893, -0.8226956725120544, -2.8272550106048584, 1.8098864555358887, 1.1819268465042114, -1.7705086469650269, -2.722586154937744, 0.45316603779792786, 3.3231804370880127, -1.1344966888427734, -1.0131206512451172, -2.076900005340576, -1.7005980014801025, -1.0988826751708984, 0.8369060754776001, -1.1461164951324463, -3.507342576980591, 2.7656331062316895, -1.4539868831634521, -0.6187127232551575, 1.1002074480056763, -2.0961334705352783, 3.1767079830169678, -0.732745885848999, -3.3381242752075195, 1.5811939239501953, -1.1555439233779907, 0.590998649597168, 0.05448536202311516, 0.283893346786499, 2.3512303829193115, -2.205904483795166, -4.399843215942383, -1.0499308109283447, -0.6339479088783264, -0.12054168432950974, -5.6826934814453125, 2.444342851638794, -0.19354937970638275, -1.7775501012802124, 4.724898815155029, 1.7050045728683472, -2.9559366703033447, 1.3238569498062134, -0.960600733757019, 0.8019139766693115, -3.831928253173828, -0.8864813446998596, -2.3986926078796387, 4.996302127838135, 0.7319009900093079, 1.4322766065597534, -1.7984936237335205, -1.3856862783432007, -3.6137051582336426, -1.3588718175888062, 3.6858997344970703, -3.0905203819274902, -2.7583236694335938, -1.0761003494262695, -0.6260653138160706, -0.3717002868652344, -0.43920594453811646, 1.995530128479004, 0.9484133124351501, 1.6696891784667969, -2.958012819290161, 2.8562307357788086], [-0.9572929739952087, -3.2192630767822266, -0.20699986815452576, -0.38289812207221985, -0.4459463357925415, 0.35765957832336426, 0.04782671108841896, -0.726525068283081, -1.5127264261245728, -0.322858989238739, 0.8991795778274536, 0.7251248359680176, -0.17415601015090942, -2.073486566543579, 0.20641975104808807, -1.6092512607574463, 1.7708488702774048, 0.47694942355155945, -2.8916640281677246, 1.513442873954773, -0.27015402913093567, -1.1569887399673462, -0.424250066280365, -1.6025270223617554, 2.423522710800171, 1.083713173866272, -1.537305474281311, 1.2786102294921875, -2.374483585357666, -1.0003807544708252, -0.7254362106323242, 1.2818084955215454, 1.2560189962387085, -1.0125317573547363, -1.3079899549484253, -0.6483474373817444, -1.2271026372909546, 0.09116780012845993, -2.491008758544922, 0.2301235944032669, -1.6876782178878784, -2.0773508548736572, 2.136000633239746, 0.5851670503616333, 0.2371295690536499, 0.6584388613700867, -1.0770041942596436, 0.814216136932373, -3.1721365451812744, 0.7343543767929077, -2.0545506477355957, 4.194974422454834, 0.10474367439746857, 0.6898403763771057, -1.7024421691894531, -2.131734848022461, -2.543207883834839, -1.6076408624649048, 2.6290295124053955, -3.3387224674224854, -1.2919342517852783, 0.18152543902397156, 1.4546722173690796, 0.4612460136413574, -2.0545718669891357, -1.2968088388442993, 0.5912637114524841, 1.937509536743164, -1.3210322856903076, 3.6509761810302734], [-3.887139320373535, 1.8339285850524902, -3.8782401084899902, 3.7127108573913574, -1.546027660369873, -4.691209316253662, 5.695254325866699, -1.3742831945419312, -0.9711496233940125, -3.785747528076172, 2.0785598754882812, 3.0220069885253906, -2.5024125576019287, -3.415199041366577, -0.4033631980419159, 3.1290533542633057, -2.7814223766326904, 1.2355740070343018, -1.8004651069641113, -0.9684823155403137, 1.319273591041565, -3.8360447883605957, -1.3848931789398193, -0.7627649307250977, -0.8981566429138184, -2.2814507484436035, 3.285360813140869, 3.1528542041778564, -0.35313212871551514, 1.1890721321105957, -2.2242226600646973, 1.872069001197815, 3.7459347248077393, 0.3273354768753052, -3.026158332824707, 2.275303602218628, -3.265058994293213, -0.4574955999851227, 0.4075590670108795, -1.5762182474136353, 2.8081612586975098, 0.9303337931632996, -0.10988304764032364, 1.4455169439315796, 1.2971611022949219, 5.376838207244873, -1.7326143980026245, 1.0364662408828735, 0.5723018646240234, -2.8913917541503906, -0.7786198854446411, -5.210153579711914, -2.6706573963165283, 0.16184934973716736, 1.1972237825393677, -5.357856750488281, -2.864034652709961, -3.070868730545044, -1.733986735343933, 2.221043109893799, -4.517919063568115, -0.6951647996902466, 0.40844035148620605, 2.4849936962127686, 1.1110082864761353, 0.18188218772411346, -0.22998371720314026, 2.273576498031616, -2.7713735103607178, 0.0761341080069542], [-1.2821028232574463, 0.7952148914337158, 3.3199048042297363, -3.0364084243774414, -6.791443347930908, 7.1712117195129395, 0.9291388988494873, -1.7218085527420044, -1.8052825927734375, 3.8378188610076904, 6.038527965545654, -8.576265335083008, -1.0859628915786743, 5.738503456115723, -4.733903408050537, 1.1901112794876099, 0.9434823393821716, -5.90274715423584, 1.1960111856460571, 0.954246461391449, -3.7955150604248047, 4.462367057800293, 5.609601974487305, 1.24686861038208, -0.0702313706278801, -4.927796840667725, -3.9773764610290527, -2.317182779312134, 5.505308151245117, 5.746838092803955, 2.7018611431121826, -4.7193989753723145, 2.3075003623962402, 1.1726021766662598, -6.167922019958496, 0.806849479675293, 3.7945516109466553, -0.4735654592514038, -5.238076686859131, 5.29009485244751, 1.9119091033935547, 2.1064648628234863, 4.3446502685546875, 0.8002369403839111, -7.561439037322998, -1.3621875047683716, 0.030750848352909088, -2.7693400382995605, 0.528129518032074, -1.8211898803710938, 3.9694833755493164, -1.1652966737747192, 3.4296252727508545, 1.4088221788406372, 2.2026002407073975, 7.77548885345459, -0.8638186454772949, 4.773580551147461, 0.905905544757843, 5.769464015960693, -1.5859586000442505, -3.642184019088745, -8.7401704788208, -2.0295112133026123, 4.174459934234619, 8.468572616577148, 0.8165006041526794, -2.7682840824127197, -3.142747402191162, -6.30856466293335], [1.2992041110992432, -8.505105018615723, -0.19518323242664337, -1.390755534172058, 3.1026666164398193, -2.044363498687744, -2.6971588134765625, 0.24442867934703827, 1.0463707447052002, 1.2430548667907715, -1.2662997245788574, 5.084901332855225, 0.2210964560508728, -3.5261058807373047, 1.5636439323425293, -3.9834506511688232, 1.8149561882019043, 3.925374746322632, -1.7483164072036743, 2.0459468364715576, 1.388451099395752, -3.5208044052124023, -3.947045087814331, -3.9217069149017334, 5.184162616729736, 2.6385951042175293, -1.46681547164917, 1.785818338394165, -4.803558826446533, -6.631396770477295, 0.6790501475334167, 4.276931285858154, -1.7874431610107422, -5.6918840408325195, 3.7498977184295654, -3.359588384628296, -2.4510855674743652, 0.7052999138832092, -2.3176987171173096, -2.1466305255889893, -2.585632085800171, -4.602102279663086, 1.4608250856399536, 0.8231145143508911, 4.978096008300781, -2.5107223987579346, 1.113710641860962, 3.201995611190796, -6.207921504974365, 2.694335699081421, -8.537144660949707, 10.240154266357422, 0.4027845859527588, 1.2225921154022217, -6.961540222167969, -8.794133186340332, -2.977787733078003, -3.169235944747925, 2.0210556983947754, -10.544121742248535, 3.0803136825561523, 3.9739367961883545, 4.7037200927734375, -0.9518614411354065, -4.565339088439941, -7.397481918334961, 1.454993486404419, 2.019366502761841, 2.419715404510498, 6.629584789276123], [0.5990313291549683, -0.5624794363975525, -1.1227104663848877, -2.3711767196655273, -3.7795209884643555, -0.15758846700191498, -1.6399571895599365, 1.2880979776382446, 0.29654937982559204, -0.15845777094364166, -4.691771030426025, -1.2516130208969116, 0.31848543882369995, -0.06898705661296844, 1.3149256706237793, 1.5642987489700317, 3.314040184020996, 0.3324076533317566, 1.1854935884475708, -2.6337168216705322, 4.285384654998779, 4.966531753540039, -1.3355554342269897, 1.5293067693710327, -0.8319994807243347, 0.7076761722564697, -1.5389962196350098, 1.269361972808838, 1.13631272315979, -0.9027316570281982, 2.2823872566223145, 2.089578628540039, -0.7343727946281433, 0.007940449751913548, 4.081077575683594, 1.4330204725265503, 2.0216987133026123, 0.24952803552150726, 0.6784875392913818, -1.8937954902648926, -1.5677969455718994, 2.4032068252563477, -1.0356712341308594, -0.7833550572395325, 2.3015248775482178, 0.4408371150493622, 2.49466609954834, -2.8295376300811768, 2.4069323539733887, -0.6608262062072754, 0.26511168479919434, -1.3252145051956177, -1.1949406862258911, -2.2409842014312744, 2.1459732055664062, 4.961822986602783, 0.040153902024030685, 0.001511322450824082, 0.6978764533996582, -2.52402925491333, 1.0859081745147705, -0.5117879509925842, 0.11907819658517838, -0.6338126063346863, -2.4970908164978027, -0.3003910481929779, -3.3521130084991455, 3.242363691329956, 3.836869716644287, 0.1307739019393921], [0.44673725962638855, 1.1194465160369873, 2.317045211791992, 2.401780605316162, 2.757565498352051, -0.7232819199562073, 0.5882475972175598, -0.26998472213745117, 1.3757216930389404, 1.0400571823120117, 1.3802073001861572, 0.28900328278541565, 0.004635454621165991, 0.8138667941093445, -1.9873342514038086, 0.4104035198688507, -6.177468776702881, -0.3129563331604004, 3.2163472175598145, 1.85562264919281, -2.2936580181121826, -1.7624750137329102, 0.9620857238769531, 0.14985060691833496, -1.3924905061721802, -1.3353832960128784, 0.5742454528808594, -1.8269438743591309, 2.0939979553222656, 0.27206549048423767, 0.7412056922912598, -2.4232101440429688, -2.333120584487915, -2.4747283458709717, -1.2806557416915894, -2.0859873294830322, -0.041974987834692, -0.035482004284858704, -0.3064856231212616, 2.1850380897521973, 4.412691116333008, 0.6899923086166382, -0.27188190817832947, 1.2634791135787964, -3.8377444744110107, -1.2922818660736084, 1.7000408172607422, 1.6086828708648682, 0.8615898489952087, -2.0343401432037354, -0.18654873967170715, -1.222589373588562, 0.08627787232398987, 0.6157135367393494, -1.1299221515655518, -1.8433040380477905, 3.350242853164673, 0.8029084205627441, -1.844430685043335, 1.4568397998809814, 1.6214932203292847, -0.10183253139257431, -0.9152863621711731, -1.3963509798049927, 4.18996000289917, 3.6034507751464844, 1.7681788206100464, -4.946560859680176, -0.11093720048666, -3.5707852840423584], [0.6178989410400391, 0.3140123784542084, 0.042959414422512054, 0.8790076375007629, -2.4569766521453857, -0.460643470287323, -0.0738445371389389, 0.2371433675289154, 1.1215476989746094, 0.5275668501853943, -1.8626271486282349, -1.5074886083602905, -0.07883034646511078, 1.4785429239273071, -0.9943616986274719, 1.0727243423461914, -0.4866032004356384, 0.554444432258606, 4.145031452178955, -0.5220702886581421, 1.7711681127548218, 2.0386440753936768, -0.7448058724403381, 1.5596566200256348, -1.6070835590362549, -0.20689831674098969, -1.2116438150405884, -0.48504650592803955, 2.5597431659698486, -1.3055857419967651, 2.4237940311431885, 0.579116702079773, -1.2524399757385254, -1.75047767162323, 1.8994940519332886, 0.4547616243362427, 0.09437388181686401, 0.4573470652103424, -0.596261203289032, -0.5073131918907166, 1.4832653999328613, 2.2253506183624268, -0.936782956123352, 0.941108763217926, -0.6249986290931702, -0.8825836181640625, 3.294407367706299, -1.8583984375, 0.8452978134155273, -1.8524552583694458, 0.5422980785369873, -2.26309871673584, -1.3297230005264282, -1.3026914596557617, 0.5627572536468506, 1.5948275327682495, 1.3409804105758667, 0.6512066721916199, -0.15043827891349792, -1.4792840480804443, 2.215742349624634, -1.0667710304260254, -1.4227453470230103, -0.41209593415260315, -0.3443020284175873, 2.557445764541626, -0.32137638330459595, -1.4752485752105713, 2.614657163619995, -3.397331953048706], [1.6171401739120483, 0.1994711011648178, 3.364182949066162, -3.4154646396636963, 3.4748640060424805, 5.197425842285156, -3.111443519592285, 0.24280256032943726, -1.0288972854614258, 2.386958599090576, 2.6087255477905273, -2.0808329582214355, 1.6694834232330322, 2.158679723739624, 0.40466541051864624, -2.941311836242676, 1.7469406127929688, -3.055097818374634, -3.372809886932373, 1.4702715873718262, -4.618271350860596, 0.5327185988426208, 3.549412965774536, -0.7137898802757263, 1.7207387685775757, 0.6296258568763733, -0.6967050433158875, -2.600072145462036, -1.4504165649414062, 2.8225350379943848, -1.5577898025512695, -3.833928346633911, -0.4670783579349518, 3.156834840774536, -2.2319979667663574, -1.6995002031326294, 2.7915353775024414, -0.46074944734573364, 0.25271841883659363, 3.075446844100952, -2.805894613265991, -2.6819610595703125, 1.5460517406463623, -2.1431045532226562, -2.3349289894104004, -2.6671664714813232, -3.418008327484131, 1.0984504222869873, -0.5952860116958618, 3.791938543319702, 1.9656962156295776, 4.571750164031982, 4.136324405670166, 1.8305052518844604, -0.3740037679672241, 3.5506386756896973, 0.9336239099502563, 2.669388771057129, 0.8766434788703918, 3.4735004901885986, -0.31013211607933044, 0.7686658501625061, -0.5826191902160645, -1.237711787223816, 1.4783087968826294, -0.8739697933197021, 1.025693655014038, -1.244408130645752, -2.6677017211914062, 1.9136446714401245], [-1.9090267419815063, -5.354374408721924, -0.16865846514701843, -2.461538314819336, -1.4324822425842285, -0.3854590058326721, 0.3016524016857147, -0.6142238974571228, -1.3461204767227173, 0.3547588288784027, 1.1614941358566284, 1.6027464866638184, -1.1836106777191162, -3.5056405067443848, -0.08586764335632324, -0.09840220212936401, 0.45219865441322327, -0.052616771310567856, -3.8976261615753174, 0.8625351190567017, 0.7403782606124878, -0.4927402138710022, -0.5786846280097961, -2.8345673084259033, 3.5321974754333496, -1.224679946899414, -1.613450527191162, 2.8463454246520996, -1.773294448852539, -0.40452730655670166, 0.4639148414134979, 1.7405329942703247, 1.2093762159347534, -2.937504768371582, -1.2060335874557495, -1.3240773677825928, 0.346289724111557, -0.28596392273902893, -3.356045961380005, 0.5882108211517334, -0.18151865899562836, -1.5619752407073975, 3.447632074356079, 0.7517897486686707, 0.6553897261619568, 1.5496424436569214, -0.4210768938064575, 1.7529033422470093, -2.017559051513672, -0.5182958841323853, -5.0497636795043945, 5.053981781005859, 0.7347334027290344, 1.148840069770813, -2.2849369049072266, -2.79852557182312, -3.606133460998535, -2.1977128982543945, 1.5291874408721924, -4.074081897735596, -1.942492127418518, 1.0325829982757568, 0.9601429104804993, -1.3893526792526245, -0.3725679814815521, -1.6131634712219238, -0.8483347296714783, 3.724790096282959, -0.7320334911346436, 4.339383602142334], [-3.0087709426879883, 1.4571284055709839, -2.781412124633789, 2.1537246704101562, -8.40207290649414, 0.38268154859542847, 4.566587448120117, -2.002155065536499, -1.6225467920303345, -1.285415768623352, 2.6289634704589844, -3.922863721847534, -2.2573750019073486, 1.8261021375656128, -2.675377607345581, 2.3923499584198, 2.597456216812134, -0.6267179846763611, 1.7387666702270508, -0.8969345092773438, 1.4330655336380005, 1.1238130331039429, 0.241625115275383, 1.5893038511276245, -1.280689001083374, -2.332951068878174, -1.4106897115707397, 1.2107441425323486, 2.990734815597534, 1.3298310041427612, 0.9690117835998535, 1.3160090446472168, 4.637712478637695, 0.8559322357177734, -3.305185556411743, 3.9902472496032715, -2.1743595600128174, 0.20378609001636505, -3.3315436840057373, -0.573283314704895, 0.9734899997711182, 2.569908618927002, 1.289172649383545, 2.1150271892547607, -1.244803547859192, 2.6562249660491943, 0.5121263861656189, -3.5957956314086914, -1.118145227432251, -2.9207942485809326, 2.755284070968628, -5.305974006652832, -1.9943046569824219, -0.6678637266159058, 2.5482938289642334, 1.2427592277526855, -3.6853065490722656, 0.15174217522144318, 1.2418394088745117, 1.4932212829589844, -2.6780588626861572, -3.711810350418091, -4.6772308349609375, 2.5321874618530273, -1.416208267211914, 4.609829425811768, 0.3842640221118927, 0.8343920111656189, -1.759243130683899, -4.274364471435547], [-0.17293037474155426, -4.042726993560791, 0.5286839008331299, -2.160047769546509, -3.1198246479034424, 0.7929948568344116, -0.25431013107299805, -0.44917842745780945, 0.6823033094406128, 2.448935031890869, 1.3078886270523071, -1.0179234743118286, -1.169372320175171, 0.9335499405860901, -1.7181909084320068, 0.18173688650131226, 0.5798047780990601, -0.1929081678390503, 1.4244799613952637, 0.2076086848974228, 0.789790153503418, 0.9358657002449036, -0.408425897359848, -1.143498420715332, 1.9677766561508179, -2.3182015419006348, -2.1321022510528564, 0.5790508389472961, 1.5659021139144897, -1.239626169204712, 2.970632553100586, 0.9875834584236145, -0.0808473750948906, -3.6042492389678955, 0.46891775727272034, -0.777338445186615, 0.7196673154830933, 0.17631737887859344, -3.319084882736206, 0.3389228880405426, 1.0377838611602783, 0.131853848695755, 2.130624771118164, 1.2020325660705566, 0.17842935025691986, -1.5406731367111206, 2.2338030338287354, -0.13331803679466248, -1.6691291332244873, -0.8636656999588013, -3.5705063343048096, 2.6846401691436768, 1.0606095790863037, 0.7570294737815857, -2.2969956398010254, -1.3684065341949463, -2.048274040222168, 0.4280954599380493, -0.005464972462505102, -3.1091959476470947, 1.4910486936569214, 0.47726088762283325, -2.4919662475585938, -1.9104050397872925, 0.025005366653203964, 0.3264193534851074, 0.24011671543121338, 0.12286825478076935, 1.408507227897644, -1.1831039190292358], [0.9801206588745117, -0.6476141810417175, 0.49157798290252686, 0.42431506514549255, 1.4897774457931519, -1.668148159980774, -0.9197257161140442, 0.7359619140625, 1.490680456161499, 0.363155335187912, -2.092514991760254, 1.4489258527755737, 0.30575335025787354, -0.561919093132019, 0.22824883460998535, 0.32781702280044556, -2.2455079555511475, 1.1479474306106567, 2.1193270683288574, -0.0511922687292099, 1.099057674407959, 0.07137756794691086, -1.1839154958724976, 0.06698267161846161, -0.5421852469444275, 0.29084402322769165, 0.2201470136642456, -0.10798097401857376, 0.5116105079650879, -1.715185284614563, 1.2012840509414673, 0.48935890197753906, -2.347172260284424, -2.269390344619751, 2.4170384407043457, -1.189931869506836, 0.24208323657512665, 0.20152924954891205, 0.7511473894119263, -0.4950331449508667, 1.4474022388458252, 0.6149583458900452, -1.1028754711151123, 0.2817016839981079, 0.4719493091106415, -0.8763943314552307, 2.1966874599456787, 0.5888231992721558, 0.9170776605606079, -0.8124839067459106, -1.6455711126327515, 0.060866132378578186, -0.6706746816635132, -0.5574697256088257, -0.9338099956512451, -1.0775530338287354, 1.853432536125183, -0.37841346859931946, -1.0020860433578491, -1.9976919889450073, 2.1568307876586914, 0.8732635974884033, 1.117687702178955, -1.1088966131210327, 0.5349633097648621, -0.21992410719394684, -0.356986939907074, -1.0177592039108276, 2.3877511024475098, -0.5609235763549805], [-1.3840612173080444, -3.927786111831665, -0.8145273327827454, -8.322434425354004, -11.321133613586426, 4.87949275970459, -0.2672988772392273, -0.37388357520103455, -0.6961344480514526, 3.9618947505950928, 2.3043336868286133, -6.226170063018799, -2.4871110916137695, 3.956129312515259, -2.078413724899292, 2.822835922241211, 7.35247802734375, -3.754018783569336, -0.3828918933868408, -3.742410898208618, 2.9367761611938477, 7.271823883056641, 1.5194242000579834, -0.1547955870628357, 2.574202060699463, -5.981232166290283, -3.8047704696655273, 1.872117280960083, 4.340753078460693, 2.7663519382476807, 5.040101528167725, 1.094048261642456, 3.4607479572296143, -0.07696090638637543, 0.09052537381649017, 2.7991678714752197, 4.950069904327393, -0.3635961413383484, -4.314998626708984, 0.12849608063697815, -1.4454948902130127, 2.567883253097534, 3.9829468727111816, -0.4236011207103729, 1.6335155963897705, -0.4964078664779663, 1.370217204093933, -4.031598091125488, 0.8887380957603455, -0.5993755459785461, -1.5614618062973022, 0.550768256187439, 3.1959776878356934, 0.44499602913856506, 1.4829925298690796, 6.640650749206543, -5.959723472595215, 2.983247756958008, -0.152909055352211, 0.44336962699890137, -0.9079925417900085, -0.9353002309799194, -7.942733287811279, -2.758425235748291, -0.5814995765686035, 0.8262485861778259, -3.228506326675415, 5.041983127593994, 1.4444917440414429, -2.6590943336486816], [2.357412576675415, -5.708317756652832, -0.162024587392807, -3.597402572631836, -1.4805070161819458, -0.1308210790157318, -3.57143497467041, 1.2015039920806885, 2.1688201427459717, 2.6526362895965576, -3.2704763412475586, 0.804839551448822, 0.22780679166316986, 0.5799963474273682, 0.9034063220024109, -1.3823026418685913, 3.7644364833831787, 2.154134511947632, 2.2163870334625244, -1.052409291267395, 3.5186803340911865, 2.1215033531188965, -2.9953510761260986, -0.9867596626281738, 2.5269320011138916, 0.7908796072006226, -2.264979362487793, 0.6835054755210876, -0.33394524455070496, -4.939655780792236, 3.6719095706939697, 3.4431610107421875, -2.3707664012908936, -4.06011962890625, 6.1037445068359375, -1.142971396446228, 0.8023099899291992, 0.7456067204475403, -1.1065436601638794, -2.539163112640381, -2.1576666831970215, -0.6441916227340698, -0.04389369115233421, -0.06512388586997986, 4.767877578735352, -3.5404138565063477, 3.636085033416748, -0.6184507012367249, -2.080488681793213, 1.5362954139709473, -5.129269599914551, 5.184498310089111, 0.5830972790718079, -0.473480761051178, -3.4094367027282715, -1.3980454206466675, -1.4125828742980957, -0.10163579136133194, 0.4223073422908783, -7.547754287719727, 4.726012229919434, 2.4172346591949463, 0.5648228526115417, -2.005446195602417, -3.8863892555236816, -4.496904373168945, -0.8471611142158508, 1.7040282487869263, 5.313807010650635, 1.3592053651809692], [-2.1955182552337646, -0.6341496706008911, -0.4030230939388275, -5.238404750823975, -7.3176589012146, 3.2990570068359375, -0.30619412660598755, 0.2008286714553833, -3.0857818126678467, -0.1862068921327591, -1.331111192703247, -4.406974792480469, -0.5273837447166443, -0.7420793771743774, 0.35444894433021545, 2.7062649726867676, 4.834383964538574, -3.5893807411193848, -3.9007177352905273, -2.398142099380493, 2.48693585395813, 7.2077202796936035, 2.117342948913574, 0.9663541913032532, 0.39010801911354065, -1.6269088983535767, -3.1641273498535156, 2.4588072299957275, 1.4481936693191528, 4.29953670501709, 1.33681321144104, 0.04536600783467293, 2.682612657546997, 2.5868732929229736, -1.2932928800582886, 2.307431936264038, 4.54595947265625, -0.6388218998908997, -1.987248420715332, 1.220185399055481, -2.1247787475585938, 2.564493417739868, 2.751903533935547, -1.2358489036560059, -1.2372634410858154, 3.3374972343444824, -0.5935230255126953, -3.326064348220825, 3.3089940547943115, -1.3601576089859009, 2.0060739517211914, -0.847711980342865, 0.6276636719703674, -1.2447285652160645, 4.635162353515625, 9.726876258850098, -2.5189340114593506, 0.5465591549873352, 2.509094715118408, 1.1972763538360596, -4.0352959632873535, -2.4337499141693115, -2.300074577331543, -1.1408514976501465, -0.477000892162323, 2.7592198848724365, -4.7698655128479, 6.240229606628418, -0.2915147542953491, 1.6825271844863892], [2.411142587661743, -1.1180778741836548, 2.4471330642700195, -0.44426584243774414, 2.7473902702331543, 2.8576912879943848, -3.4703714847564697, 0.4340701103210449, -0.28124189376831055, 1.286829948425293, -1.058998703956604, -1.1217254400253296, 2.230466604232788, 1.125856637954712, 0.7295852303504944, -3.750230312347412, 2.093968629837036, 0.038139138370752335, -0.08214179426431656, 2.2053706645965576, -1.9930076599121094, 0.479255348443985, 0.703862190246582, 0.218393012881279, 0.9161700010299683, 3.7019145488739014, -2.2051830291748047, -2.261949300765991, -1.8413876295089722, -1.5794987678527832, -0.353713721036911, -1.0549849271774292, -2.004225969314575, 0.345479279756546, 0.8344603180885315, -1.7988497018814087, 0.1073569729924202, 0.4914373755455017, -0.5436114072799683, 1.0943948030471802, -3.011857748031616, -2.210696220397949, 0.12491564452648163, -0.6520753502845764, -1.321921944618225, -3.168903350830078, -0.16751717031002045, -0.044946543872356415, -2.4867806434631348, 2.7686262130737305, 1.1567203998565674, 4.79522705078125, 1.0682178735733032, 0.05073273554444313, -1.4033162593841553, 1.7814031839370728, 1.7696659564971924, 0.9758920669555664, 2.9100089073181152, -2.4385154247283936, 2.2899017333984375, 0.19186677038669586, 1.7194790840148926, -0.0958632305264473, -2.176234006881714, -0.5720441341400146, 1.5093525648117065, -1.892918586730957, 0.3137635886669159, 1.8842244148254395], [4.261098861694336, -3.1589291095733643, 0.9969403147697449, -2.0354340076446533, 5.5171589851379395, 0.7437443733215332, -6.894834518432617, 2.4234325885772705, 0.20546162128448486, -0.18075306713581085, -7.180375099182129, 2.5520992279052734, 3.9395620822906494, -2.133504629135132, 4.98930025100708, -5.299798011779785, 5.5898871421813965, 2.8961269855499268, -2.176727294921875, 0.7797268033027649, 1.6000422239303589, 1.4493675231933594, -2.4909589290618896, -0.1872764527797699, 1.9148626327514648, 8.168052673339844, -1.8648747205734253, -0.6677396893501282, -5.869874954223633, -5.202137470245361, -1.1028201580047607, 2.3040103912353516, -3.97487473487854, 0.574935257434845, 6.667270183563232, -2.236410617828369, 0.09560290724039078, 0.8643260598182678, 2.516463041305542, -2.302351713180542, -7.368002414703369, -3.607646942138672, -2.0648539066314697, -2.5138983726501465, 4.304065227508545, -3.187300682067871, 0.025877682492136955, 0.19133473932743073, -2.4264004230499268, 5.441919803619385, -0.963070809841156, 8.139188766479492, -0.13539014756679535, -1.5712337493896484, -1.982617735862732, 1.6435508728027344, 1.9977997541427612, -1.3097418546676636, 4.256775379180908, -7.873408794403076, 3.83870267868042, 2.5154082775115967, 7.5699462890625, 0.27061209082603455, -6.964266300201416, -7.00950288772583, -0.8729782104492188, 2.3232924938201904, 3.82896089553833, 7.991200923919678], [0.9177650809288025, -2.0340917110443115, 0.2787191867828369, -2.8440780639648438, 0.6436565518379211, 0.47356927394866943, -0.2814417779445648, -0.03434456139802933, 2.4429099559783936, 3.3332808017730713, 3.5856196880340576, 0.7595836520195007, -1.37100350856781, 2.786320447921753, -1.1963064670562744, 0.03941195458173752, -0.4481523036956787, -0.3454001545906067, 1.267504334449768, -1.0151561498641968, -0.9829854965209961, -1.7853752374649048, -0.08633331954479218, -1.9305297136306763, 2.0344202518463135, -4.0165863037109375, 1.6239655017852783, -0.6615922451019287, 1.289344310760498, -0.32791367173194885, 1.6357704401016235, -0.2997865378856659, -0.20434479415416718, -1.6910382509231567, 0.8530768752098083, -0.8077138066291809, 0.9826391339302063, -0.22165584564208984, 0.09250473976135254, -0.41106855869293213, 1.2861608266830444, -0.9686986207962036, 0.5366989970207214, -0.2132449597120285, 2.6736862659454346, -3.025385856628418, 0.10833670943975449, 1.6826775074005127, -0.7839773297309875, 1.3857502937316895, -3.8275468349456787, 1.6487809419631958, 3.1078412532806396, 2.265089988708496, -3.166965961456299, -4.259893417358398, -1.739472508430481, 1.7962532043457031, -4.044499397277832, 1.372571349143982, 2.1130595207214355, 2.8115251064300537, -3.2404844760894775, -2.035013198852539, 2.1773109436035156, -3.366070032119751, 1.2640788555145264, -1.3327502012252808, 0.5824941992759705, -2.251089572906494], [-0.3727850019931793, -1.0994733572006226, -0.3807053565979004, 0.7370468378067017, -2.336207866668701, -1.3170840740203857, 1.818885326385498, -0.6727204322814941, 1.9428874254226685, 1.6215077638626099, 2.018625497817993, -0.20297762751579285, -1.8021055459976196, 1.8464242219924927, -2.495636463165283, 1.596093773841858, -2.413586378097534, 0.5433950424194336, 4.095337390899658, -0.3422892093658447, 0.6518329381942749, -0.9889188408851624, -0.8442492485046387, -0.3186901807785034, -0.13970109820365906, -3.4740240573883057, 0.22714775800704956, -0.029010813683271408, 3.092644691467285, -1.1155909299850464, 2.6545333862304688, 0.5965555310249329, -0.1143304780125618, -3.472770929336548, 0.2546020746231079, 0.020338859409093857, -0.7135286331176758, 0.18341922760009766, -1.6411757469177246, -0.3677007555961609, 3.653165102005005, 1.3924610614776611, 0.2674795091152191, 1.8337293863296509, 0.14142578840255737, -1.1434154510498047, 2.7300407886505127, 0.036994338035583496, -0.45144858956336975, -2.1925268173217773, -2.494236946105957, -1.7760194540023804, -0.07503146678209305, 0.6114049553871155, -1.7439637184143066, -3.5636212825775146, -0.7857754230499268, 0.6195357441902161, -2.5498504638671875, -0.4243623614311218, 1.854438066482544, 0.2349274605512619, -3.374194622039795, -1.0241895914077759, 1.7658560276031494, 1.4147179126739502, 1.149347186088562, -2.2604587078094482, 1.3021348714828491, -4.589924335479736], [-1.3652375936508179, -3.150418281555176, 1.142523169517517, 1.0759278535842896, -2.180955648422241, -0.6148841381072998, 1.5324199199676514, -1.183937430381775, 0.05383995175361633, 1.1588488817214966, 1.8463361263275146, -0.42929545044898987, -1.3269680738449097, -0.7116840481758118, -2.792353391647339, 0.6843107342720032, -3.453078031539917, 0.0038196775130927563, 1.9502637386322021, 2.0669920444488525, -0.26106178760528564, -0.6910892128944397, -0.04799995571374893, -0.9815137982368469, 0.8517190217971802, -2.1170270442962646, -2.1717350482940674, 0.6098204851150513, 1.8885871171951294, -0.8726530075073242, 2.291379928588867, 0.17303113639354706, -0.32079511880874634, -4.914889335632324, -1.8731364011764526, -1.562814712524414, -0.6563374400138855, 0.152587890625, -4.369575500488281, 1.7846862077713013, 3.7178759574890137, 0.5815681219100952, 2.4853811264038086, 2.656363010406494, -3.147092342376709, -0.018033429980278015, 2.6741814613342285, 0.7431338429450989, -1.6472222805023193, -3.1905336380004883, -3.0705337524414062, 1.369727611541748, -0.4525637626647949, 0.6475927829742432, -2.1563096046447754, -2.757934331893921, -0.6203773021697998, -0.7015582323074341, 0.6030232310295105, -3.028865337371826, 0.4488815367221832, -0.766414999961853, -1.6833442449569702, -1.5045289993286133, 1.783974289894104, 3.7892096042633057, 1.1348458528518677, -1.7219443321228027, 0.0798555389046669, -1.8654954433441162], [0.3305310010910034, 6.045996189117432, 0.12340734153985977, 2.9594004154205322, 4.500006198883057, -2.747343063354492, 0.010796650312840939, 1.491246223449707, 0.044358693063259125, -3.3761961460113525, -4.4304728507995605, 1.9193172454833984, 1.6591780185699463, -2.3819921016693115, 2.2287490367889404, 1.9780499935150146, -4.583489418029785, 0.13303396105766296, 0.019702810794115067, -0.9298821687698364, 0.46673402190208435, 0.678184449672699, 0.34064894914627075, 2.3199517726898193, -4.362875461578369, 2.278010368347168, 2.9705376625061035, -0.4134494364261627, -0.1653302162885666, 1.9970142841339111, -2.411687135696411, -1.7652645111083984, -2.2371788024902344, 2.9652981758117676, 0.956935465335846, 0.17216205596923828, 0.9006642699241638, -0.38751259446144104, 5.3356404304504395, 0.07396262139081955, 1.5432851314544678, 2.2080345153808594, -3.740920305252075, -1.4264698028564453, -1.6008728742599487, 2.7865798473358154, -0.4271374046802521, 0.25314822793006897, 5.8428730964660645, -1.4712464809417725, 4.254687786102295, -5.836370944976807, -2.435634136199951, -2.15360164642334, 4.179993152618408, 3.7348577976226807, 5.5686936378479, -0.7992995381355286, -1.3704118728637695, 3.978109836578369, -1.1495144367218018, -0.7887424826622009, 3.790738582611084, 0.7202693223953247, 2.4631805419921875, 1.9321181774139404, -2.258976459503174, -0.6071028709411621, 0.4003945589065552, 0.15967658162117004], [-4.226414680480957, 3.8628575801849365, -3.2731947898864746, 6.454555511474609, -4.8672380447387695, -0.6915866732597351, 6.699615001678467, -3.0719971656799316, -3.349130868911743, -4.124680519104004, 4.528500556945801, -2.1267871856689453, -1.8169467449188232, -0.23547965288162231, -2.2526495456695557, 0.8924269080162048, 0.8889411687850952, 0.05265406146645546, -0.9121179580688477, 1.3637440204620361, -1.3306832313537598, -3.3085875511169434, 0.7363979816436768, 1.2508606910705566, -1.7663155794143677, -0.055363595485687256, 0.2711525559425354, 0.9463785290718079, 0.13030490279197693, 2.0141375064849854, -3.151714563369751, 0.46618831157684326, 6.223851680755615, 3.0356056690216064, -7.042104244232178, 4.01350736618042, -5.5433573722839355, 0.10428360849618912, -2.7167017459869385, 0.0629367008805275, 0.629510760307312, 0.5306963324546814, 1.0702035427093506, 2.5262629985809326, -2.944906711578369, 4.724380016326904, -2.856086015701294, -1.9477128982543945, -3.0264980792999268, -2.331716775894165, 4.958096504211426, -5.790986061096191, -3.188979387283325, -0.0698772668838501, 2.710385322570801, -1.9337719678878784, -3.468712329864502, -1.5152872800827026, 2.640469551086426, 3.3632559776306152, -5.484874725341797, -4.48084020614624, -1.9707599878311157, 5.434133052825928, -1.5305472612380981, 4.896827697753906, 2.6033685207366943, -0.2802024781703949, -5.872012615203857, -2.1139371395111084], [1.2632720470428467, 2.613190174102783, -1.180956244468689, 0.36346369981765747, 6.498288631439209, -7.155892848968506, -0.7853147983551025, 2.930488109588623, 3.7941677570343018, -1.8576830625534058, -6.155252456665039, 7.043213367462158, 0.2780608534812927, -3.8813369274139404, 3.2169923782348633, 3.8920228481292725, -7.330382823944092, 2.4645497798919678, 1.6362195014953613, -3.566041946411133, 3.8677780628204346, -0.6399831771850586, -3.0504302978515625, 0.04076544567942619, -2.911670207977295, -0.7743053436279297, 5.827081680297852, 1.5460617542266846, 0.2008724808692932, -0.9387622475624084, 0.015873687341809273, 0.9472207427024841, -4.721407413482666, -1.4993939399719238, 6.374483108520508, -1.3082475662231445, 2.0077223777770996, -0.5381905436515808, 7.631263256072998, -2.867711305618286, 3.998677968978882, 2.507741928100586, -5.1436028480529785, -1.6998255252838135, 4.470120429992676, 1.6871763467788696, 2.175213098526001, 2.8627660274505615, 7.6300787925720215, -1.8147562742233276, -3.164602518081665, -4.891570568084717, -1.8563765287399292, -1.6115593910217285, 0.8568852543830872, -2.434586524963379, 4.964005947113037, -1.9483286142349243, -6.56990385055542, 1.0929324626922607, 1.8898783922195435, 3.624753952026367, 4.6794114112854, -2.1962828636169434, 3.960127830505371, -3.92836332321167, -3.9145185947418213, 0.9433120489120483, 4.84896183013916, 0.4547809958457947], [-1.3723580837249756, -1.621304988861084, 2.6409170627593994, -2.107792615890503, -0.2528977692127228, 2.500471353530884, 0.2201230823993683, -0.8044641613960266, -1.429025650024414, 1.660810112953186, 3.4229564666748047, -1.5521794557571411, -0.5422544479370117, -0.43027743697166443, -1.7857626676559448, 0.2875707745552063, -2.4059572219848633, -3.007836103439331, -2.470478057861328, 1.765924334526062, -2.856346845626831, 0.48457902669906616, 2.9292306900024414, -1.3821758031845093, 1.5288877487182617, -2.6027932167053223, -1.5068378448486328, -0.04933345690369606, 0.8339676260948181, 3.231627941131592, 0.3060199022293091, -2.7158641815185547, 0.4255252480506897, -0.9517567157745361, -4.147428512573242, -1.7960890531539917, 2.4014132022857666, -0.6699615120887756, -2.9330496788024902, 3.883183717727661, 1.8745512962341309, -0.44540920853614807, 3.4935059547424316, 0.3637559413909912, -4.438974857330322, 0.3945353031158447, -1.012034296989441, 1.479349136352539, 0.279745876789093, -1.0912758111953735, -0.9364838600158691, 2.521298885345459, 2.3272385597229004, 1.6798924207687378, -0.42892223596572876, 1.4015913009643555, -0.34461861848831177, 0.7517877817153931, 0.4463571608066559, 1.7975242137908936, -2.0672779083251953, -0.4400411546230316, -1.9099684953689575, -2.2933096885681152, 3.8992977142333984, 3.1175453662872314, 0.08550950884819031, -0.398777574300766, -2.607294797897339, 0.33978360891342163], [-0.2923944294452667, -3.109254837036133, -2.628203868865967, -1.6042593717575073, -2.875247001647949, -0.633130669593811, 0.5197561979293823, -0.37438109517097473, 0.5364053249359131, 0.5402634739875793, 0.9538618326187134, 0.8773142099380493, -1.346296787261963, 0.4298839867115021, 0.275118350982666, -0.44527143239974976, 4.401125431060791, 1.5777791738510132, -0.430175244808197, -1.4559417963027954, 2.1333768367767334, -0.9388921856880188, -2.219081401824951, -1.3769973516464233, 2.371227502822876, -0.8861567378044128, 0.0587422214448452, 1.6304268836975098, -0.9579988718032837, -2.373218297958374, 0.804027795791626, 3.3097527027130127, 2.0762529373168945, -0.8460622429847717, 1.7162514925003052, 1.3794595003128052, -1.5540590286254883, 0.2945685386657715, -0.9865137338638306, -2.88130521774292, -2.1172807216644287, -1.1384540796279907, 0.5378918647766113, 0.23861078917980194, 5.06397008895874, -0.5187392830848694, -0.1099758893251419, -0.44383859634399414, -2.9678597450256348, 1.5057121515274048, -3.1984598636627197, 1.8631312847137451, 0.31313103437423706, 0.5963993668556213, -2.034409284591675, -3.845968723297119, -4.5424628257751465, -0.6941198706626892, -0.24112297594547272, -2.8904807567596436, 0.47215792536735535, 1.3880186080932617, -0.8795017600059509, 0.7910045981407166, -3.4384958744049072, -4.545599937438965, 0.2769628167152405, 2.747271776199341, 0.8693373203277588, 1.0694622993469238], [-1.3795000314712524, 3.048114776611328, 0.7114725112915039, 5.511518955230713, 1.497821569442749, -0.8335686326026917, 2.6908481121063232, -1.3252828121185303, -1.387070894241333, -2.216099977493286, 1.5443195104599, -0.29314857721328735, 0.248800128698349, -0.9630811810493469, -1.4564414024353027, -0.2119494378566742, -4.177267074584961, 0.023308413103222847, 0.6719858646392822, 2.8678996562957764, -2.6265149116516113, -2.7113466262817383, 1.174003005027771, 0.9819024205207825, -2.1358726024627686, 1.3064268827438354, 0.1472761034965515, -1.0110105276107788, 0.08965708315372467, 1.0131330490112305, -2.105956792831421, -1.8186278343200684, 0.5202997326850891, 0.338469922542572, -4.243924140930176, -0.2227543145418167, -2.7825818061828613, 0.07379184663295746, -0.9801826477050781, 2.0340652465820312, 2.4519002437591553, 0.17232008278369904, -0.00022000462922733277, 1.7186683416366577, -5.068258285522461, 1.8009932041168213, -0.6666935682296753, 0.4836902618408203, -0.7431690096855164, -2.1485345363616943, 3.233112096786499, -2.656825542449951, -2.0654847621917725, -0.14362134039402008, 0.9713351726531982, -1.0609428882598877, 2.0844626426696777, -0.8578387498855591, 1.5159815549850464, 1.8244436979293823, -1.812391996383667, -2.4926438331604004, 0.7699599862098694, 1.9975144863128662, 1.5825790166854858, 5.095279693603516, 2.2567861080169678, -3.468721866607666, -3.1269078254699707, -1.4759060144424438], [2.2129828929901123, -0.1385825276374817, 0.542762279510498, 1.7446855306625366, 0.7102615237236023, 2.3868257999420166, -0.7382910847663879, -0.8729527592658997, 1.1418308019638062, 2.0081961154937744, 2.886376142501831, -1.7737435102462769, 0.5736194849014282, 4.719367504119873, -1.3241386413574219, -3.7334301471710205, 2.9360833168029785, 1.052208662033081, 3.2330029010772705, 1.7364329099655151, -2.63746976852417, -2.646883726119995, -0.08339399099349976, 0.2597673535346985, 0.743389368057251, 1.476824402809143, -0.6903474926948547, -3.195915460586548, -0.2072107344865799, -2.7082784175872803, 0.05171971395611763, -0.30079299211502075, 0.15144051611423492, 0.2433655560016632, -0.03432111442089081, 0.06125927343964577, -2.9861714839935303, 0.8538517951965332, -1.1091699600219727, -0.5446373224258423, -1.9834003448486328, -2.319149971008301, -0.4885925352573395, 0.6748883724212646, 0.48760536313056946, -4.6318535804748535, -0.1399935632944107, -0.6628726720809937, -5.2038445472717285, 3.050447940826416, 1.2376388311386108, 2.156855583190918, 1.1241192817687988, 1.1378804445266724, -2.85634183883667, -3.0845818519592285, -0.5866991281509399, 2.0950193405151367, 0.7586957812309265, -0.659735918045044, 3.3624346256256104, 0.15813599526882172, -1.9758661985397339, 1.7170175313949585, -2.7211544513702393, -1.1910059452056885, 4.541330337524414, -4.523741722106934, -0.5480329990386963, -2.698739767074585], [-2.56154203414917, 4.150712490081787, 0.570273220539093, 2.7575576305389404, -2.306748628616333, -0.4251244366168976, 3.0138559341430664, -0.6882912516593933, -1.8446182012557983, -2.195605754852295, 0.028802845627069473, -2.2653732299804688, -0.4583105742931366, -1.1613103151321411, -1.6139302253723145, 3.3925533294677734, -4.474600315093994, -2.1795246601104736, 0.41842788457870483, 0.38304346799850464, -0.4481724500656128, 1.9251861572265625, 2.291520118713379, 2.0948374271392822, -3.4601686000823975, -1.3548345565795898, -0.30605778098106384, 0.34105923771858215, 2.8835043907165527, 4.138286590576172, -0.38279810547828674, -2.372682571411133, 0.8470404744148254, 1.0702154636383057, -4.027194976806641, 1.0193556547164917, 0.9181797504425049, -0.5013436675071716, -0.6496413946151733, 2.6917173862457275, 3.9044225215911865, 3.500653028488159, 0.4050046503543854, 1.0269317626953125, -6.005765914916992, 3.996601104736328, 0.39927545189857483, -1.2122979164123535, 4.04513692855835, -4.612131595611572, 4.196357250213623, -6.183442115783691, -2.0570263862609863, -1.2523826360702515, 4.414902210235596, 4.804897785186768, 2.4019806385040283, -0.14951662719249725, 0.3985373377799988, 4.243195056915283, -3.4627888202667236, -3.643065929412842, -1.2994134426116943, 0.25891998410224915, 3.871068239212036, 7.811912536621094, -1.4122370481491089, -0.8556410670280457, -2.002466917037964, -2.920083522796631], [-1.8821206092834473, -0.3182581961154938, -1.5169479846954346, 1.7510873079299927, -0.9813053011894226, -2.9325215816497803, 2.180065870285034, -0.3703162968158722, -0.5039713382720947, -1.9850512742996216, -0.7380459904670715, 1.8561490774154663, -0.9860522150993347, -2.9628992080688477, -0.004038174636662006, 1.7269712686538696, -2.051010847091675, 1.0365052223205566, -0.5722818970680237, -0.15296043455600739, 1.706719160079956, -0.907627284526825, -1.247092843055725, -0.35659387707710266, -0.42041289806365967, -0.46738940477371216, 0.5673112869262695, 2.1257712841033936, -0.1834537535905838, -0.27647334337234497, -0.2754051685333252, 1.5256924629211426, 0.7870948910713196, -1.4164429903030396, -0.5215479731559753, 0.39317718148231506, -1.235355019569397, -0.10157222300767899, -0.4489213228225708, -0.6533640027046204, 1.7806706428527832, 0.8956204056739807, 0.14621756970882416, 1.0721275806427002, 0.1878584325313568, 2.9084787368774414, 0.5401157140731812, 0.48059242963790894, 0.6006207466125488, -2.353820562362671, -1.3796049356460571, -1.7095818519592285, -2.0501585006713867, -0.5571604371070862, 0.4125807583332062, -2.1013426780700684, -0.7661197781562805, -2.270524263381958, -0.004177537746727467, -1.2802858352661133, -1.7195024490356445, -0.4486568868160248, 1.2802070379257202, 0.526634693145752, 0.2416323721408844, 0.7705591320991516, -0.9032599329948425, 1.6274800300598145, -0.10500360280275345, 0.8285019397735596], [1.5558383464813232, 0.7004562616348267, 1.9348878860473633, 2.7832272052764893, 2.3761682510375977, -0.19940845668315887, -1.0558491945266724, 0.1780126839876175, 0.9261526465415955, 0.3565862476825714, -1.408240795135498, -0.23829388618469238, 1.268100619316101, 0.7241155505180359, -0.6800284385681152, -1.2698112726211548, -2.8093528747558594, 0.8875067830085754, 3.4346513748168945, 2.0493030548095703, -1.0768814086914062, -0.4920124113559723, -0.06466790288686752, 1.0797463655471802, -1.4656431674957275, 2.070558547973633, -0.9496191740036011, -2.0492782592773438, 0.6488974094390869, -1.7884799242019653, 0.6005780696868896, -1.0517574548721313, -2.798454999923706, -1.7874573469161987, 0.7379045486450195, -1.6440998315811157, -0.9297924041748047, 0.5325333476066589, -0.21604135632514954, 0.9717559218406677, 1.4078311920166016, 0.1826084554195404, -1.1064505577087402, 0.9337952136993408, -2.747725009918213, -1.9570802450180054, 2.192878484725952, 0.09428046643733978, -0.5540406107902527, -0.704925537109375, 1.0474025011062622, 0.17540928721427917, -1.0201215744018555, -0.64238440990448, -0.8459017276763916, -0.11368528008460999, 3.5708868503570557, 0.38169676065444946, 0.8050645589828491, -1.6380043029785156, 2.733734607696533, -0.6147326231002808, 0.9964047074317932, -0.1664598137140274, 0.33174076676368713, 2.635511636734009, 1.6285136938095093, -4.122098445892334, 1.2046822309494019, -1.8538743257522583], [3.149465799331665, 0.5829551219940186, 4.783598899841309, 0.8881165385246277, 11.885910987854004, 0.11580710858106613, -4.420005798339844, 1.568298578262329, 0.7614792585372925, 0.46036002039909363, -1.922878623008728, 3.8214080333709717, 3.4444634914398193, -2.3390631675720215, 2.212261915206909, -3.731963872909546, -5.858949661254883, 0.17694619297981262, -1.8772404193878174, 3.6127078533172607, -4.527118682861328, -2.8760366439819336, 1.3078736066818237, -1.080841302871704, 0.10980009287595749, 4.2242536544799805, 1.3410735130310059, -2.9309470653533936, -3.6772518157958984, -0.45879504084587097, -2.827666759490967, -3.7699356079101562, -5.631840229034424, -0.26597052812576294, 0.6416826248168945, -5.408935070037842, 1.2167521715164185, -0.2094288170337677, 3.271907329559326, 2.905320167541504, 0.04665090888738632, -3.3498294353485107, -1.353036642074585, -1.6736574172973633, -3.025639057159424, -2.6251800060272217, -1.4351844787597656, 4.6777873039245605, 0.7399064898490906, 2.608182668685913, -0.2729000151157379, 5.3662214279174805, 1.572960376739502, 0.8540763854980469, -2.3293464183807373, -1.0339279174804688, 6.5788116455078125, -0.1700926572084427, 0.0657910406589508, -0.05940045416355133, 2.1606149673461914, 2.604875087738037, 6.151810646057129, -1.8891829252243042, 3.031564950942993, -1.5669714212417603, 1.38563072681427, -4.082982540130615, -0.4171944856643677, 4.262144088745117], [1.8496383428573608, -0.47073763608932495, 3.950546979904175, 1.7271969318389893, 10.674477577209473, -1.4310699701309204, -3.1399660110473633, 1.1803271770477295, 0.24769532680511475, -0.5386720299720764, -2.2265493869781494, 4.724087715148926, 2.7384190559387207, -4.282731533050537, 1.9919977188110352, -2.8782155513763428, -6.7359724044799805, 0.8123000264167786, -2.3456671237945557, 3.8085033893585205, -3.3955085277557373, -3.272333860397339, 0.4916073679924011, -1.5146915912628174, 0.31174010038375854, 3.936776638031006, 1.0315096378326416, -1.43348228931427, -3.843752145767212, -0.9016613364219666, -2.680974006652832, -2.574923038482666, -5.01281213760376, -1.6826056241989136, 0.17171311378479004, -5.313157558441162, 0.3943674862384796, -0.21186058223247528, 2.150937080383301, 2.650153398513794, 1.002747893333435, -2.964994430541992, -0.7046821713447571, -0.7517749071121216, -3.0771806240081787, -0.9231394529342651, -0.8764420747756958, 4.898258686065674, 0.49125778675079346, 1.0714627504348755, -1.6074916124343872, 5.140069961547852, 0.3607984185218811, 0.589404284954071, -2.4448843002319336, -2.4409356117248535, 5.621693134307861, -1.697534441947937, 0.6246432662010193, -1.7476396560668945, 1.121174931526184, 2.229365587234497, 6.8119378089904785, -1.6971584558486938, 2.8152995109558105, -0.9334446787834167, 0.9151492714881897, -2.85711407661438, -0.4571387767791748, 5.043744087219238], [-0.004291078075766563, -3.27968168258667, 0.7940767407417297, -3.7987849712371826, 2.865536689758301, -0.32829394936561584, -1.0534350872039795, 0.4473820626735687, 0.8554377555847168, 1.8568660020828247, 1.851314902305603, 2.8944685459136963, -0.7325910329818726, -1.4874447584152222, 0.5438721179962158, -0.017052553594112396, -1.5650482177734375, -0.6018474102020264, -3.0519840717315674, -0.345881849527359, -0.7895537614822388, -1.5597965717315674, 0.07499343156814575, -3.054996967315674, 2.9831011295318604, -2.6185357570648193, 1.4669488668441772, 1.0590293407440186, -1.2624545097351074, 0.5565400123596191, 0.07464457303285599, -0.24583564698696136, -0.6286172866821289, -1.6454282999038696, 0.29075324535369873, -2.1694278717041016, 2.0739192962646484, -0.6652028560638428, 0.3728034794330597, 0.5038017630577087, 0.604720413684845, -1.878907322883606, 1.5148131847381592, -0.9350305199623108, 2.0913989543914795, -0.6311002373695374, -1.269431471824646, 3.3471038341522217, 0.30196499824523926, 1.2562285661697388, -5.074276447296143, 4.122245788574219, 2.873260736465454, 2.103567600250244, -2.7637720108032227, -3.501579523086548, -1.4157390594482422, -0.29642388224601746, -2.54290509223938, 0.016154080629348755, -0.20045003294944763, 3.316148519515991, 0.43228253722190857, -2.701378107070923, 2.581934690475464, -4.185595512390137, -0.6015272736549377, 1.622678279876709, -0.200038880109787, 2.787548780441284], [0.025156637653708458, -3.773002862930298, -1.340269923210144, -3.06957745552063, -5.58239221572876, -0.3390580713748932, 0.637290894985199, -0.23377346992492676, 2.5411243438720703, 3.273895740509033, 1.5839974880218506, -1.023285150527954, -2.4917354583740234, 2.9883546829223633, -2.1733319759368896, 1.9520764350891113, 1.5738786458969116, 0.42984163761138916, 3.992662191390991, -2.359926700592041, 2.875765562057495, 1.3200151920318604, -1.7085591554641724, -0.8864569067955017, 1.5728683471679688, -4.8681464195251465, -0.7281103134155273, 1.0012048482894897, 3.599973440170288, -1.9167182445526123, 4.698251724243164, 2.394449234008789, 0.4974507987499237, -4.016678810119629, 2.651675224304199, 0.8449785709381104, 0.7985381484031677, 0.24653759598731995, -2.2933037281036377, -1.8792312145233154, 1.8795666694641113, 1.5888465642929077, 0.9517860412597656, 1.2152835130691528, 3.5160820484161377, -2.191026449203491, 3.5447885990142822, -1.1216998100280762, -0.7610608339309692, -1.1722365617752075, -4.706812858581543, -0.1288725584745407, 1.1374619007110596, 0.6759030222892761, -2.3495819568634033, -2.6952147483825684, -3.4397003650665283, 1.2606040239334106, -2.9635391235351562, -2.241987466812134, 2.8230390548706055, 1.2729575634002686, -5.049961090087891, -2.0837342739105225, -0.04773062840104103, -1.3317853212356567, -0.24477849900722504, 0.4911300241947174, 3.3106350898742676, -4.241588592529297], [-1.341428518295288, 4.581883430480957, -1.4501655101776123, 3.149860382080078, 1.33876633644104, -3.948457717895508, 2.9865150451660156, 0.2537305951118469, 0.8115665316581726, -2.546795129776001, -0.8088529109954834, 2.1145646572113037, -0.8136088252067566, -1.4743248224258423, 0.058132369071245193, 3.5912606716156006, -5.38902473449707, 0.3674963712692261, 1.3115839958190918, -1.558119297027588, 0.917072057723999, -1.25544011592865, -0.3478933870792389, 1.1141016483306885, -3.3934497833251953, -1.7011632919311523, 3.7192931175231934, 0.7208486199378967, 1.6552913188934326, 1.854365587234497, -1.076680064201355, -0.6158960461616516, -0.11366444081068039, 0.8154011964797974, -0.42696163058280945, 1.0754787921905518, -0.37649378180503845, -0.4746862053871155, 3.328716993331909, -0.5913808345794678, 4.051288604736328, 2.6853954792022705, -2.4719936847686768, 0.17796985805034637, -0.4272346496582031, 3.169341564178467, 0.17020325362682343, 0.6197191476821899, 4.5150370597839355, -3.0095176696777344, 1.4782326221466064, -7.144578456878662, -2.160139799118042, -0.8880022168159485, 2.6188101768493652, -0.8370845913887024, 2.331799030303955, -0.9013397097587585, -3.5274038314819336, 4.380490779876709, -1.607066035270691, -0.4847880005836487, 0.440048485994339, 0.6018247604370117, 3.612025260925293, 1.9332482814788818, -1.2135366201400757, -0.7195929884910583, -0.21531371772289276, -2.5782320499420166], [0.8200653791427612, 1.9575130939483643, 0.34061291813850403, -3.630549192428589, 2.7852442264556885, -1.2939296960830688, -1.1583880186080933, 1.885857343673706, 2.1716508865356445, 0.993347704410553, -1.271164894104004, 1.9979804754257202, -0.24358049035072327, 0.014837725088000298, 1.357159972190857, 3.0691158771514893, -3.1888766288757324, -1.3738890886306763, -0.3554413318634033, -3.371124744415283, 0.9306060671806335, 1.5349910259246826, 0.501407265663147, -0.26952680945396423, -0.9943518042564392, -3.3654603958129883, 3.6419007778167725, 0.2812630236148834, 1.5700730085372925, 2.6430351734161377, 0.5784600377082825, -1.5084580183029175, -2.019582748413086, 0.9824339747428894, 2.5905563831329346, -0.48130980134010315, 4.411203861236572, -0.9314085245132446, 4.721644878387451, -0.2883880138397217, 2.128871202468872, 1.771277666091919, -1.8539364337921143, -2.3623716831207275, 2.212026596069336, 0.17894789576530457, -0.11904866993427277, 1.5104931592941284, 6.2656331062316895, -0.11635413765907288, -1.2766276597976685, -2.836444139480591, 1.8722959756851196, 0.189723938703537, 1.3913816213607788, 1.7508212327957153, 2.2060070037841797, 1.2240614891052246, -5.562189102172852, 4.769406795501709, 0.32022717595100403, 2.5537333488464355, -0.4279734492301941, -2.971285343170166, 4.781223773956299, -2.384385824203491, -3.1578803062438965, 1.0795097351074219, 1.8900457620620728, -0.8754364848136902], [1.3310000896453857, 2.110642433166504, -0.928764820098877, 1.2194299697875977, 3.373242139816284, 1.4066367149353027, -1.2436184883117676, 0.10147731006145477, -1.204300045967102, -1.6745458841323853, -0.12997889518737793, 0.5376480221748352, 1.8450686931610107, 0.4583055377006531, 2.5047519207000732, -3.6442813873291016, 4.765935897827148, 0.9575862288475037, -2.946765422821045, 0.56950843334198, -1.7867263555526733, -2.1387765407562256, -0.0294578168541193, 0.336132675409317, 0.37518566846847534, 4.59423828125, 1.117606520652771, -1.3159914016723633, -4.222371578216553, -0.6830511689186096, -3.95885968208313, 0.08245028555393219, 1.2776645421981812, 4.925506591796875, -0.1116890162229538, 1.0464495420455933, -2.3644535541534424, 0.2771247923374176, 2.4814229011535645, -1.4065526723861694, -5.470003604888916, -3.161496639251709, -1.5502551794052124, -1.7531667947769165, 2.2236127853393555, -0.6612116098403931, -4.071408271789551, -0.18063701689243317, -2.7780447006225586, 4.662109851837158, 3.453378200531006, 1.7176655530929565, 0.17716383934020996, 0.08406957238912582, 0.3133755922317505, -0.14291366934776306, -0.45156413316726685, -0.07530239224433899, 2.2203683853149414, 0.9786008596420288, -0.6812884211540222, 0.303846150636673, 2.98085880279541, 3.4114320278167725, -4.182534694671631, -4.12441349029541, 1.6658039093017578, 0.4446604549884796, -2.232252836227417, 3.6302056312561035], [3.366875171661377, 0.627574622631073, 1.9210931062698364, 1.2698590755462646, 8.159603118896484, -0.397403359413147, -2.864323139190674, 1.0018457174301147, 2.2438440322875977, 1.071608543395996, -0.4107029139995575, 3.0352730751037598, 1.9944254159927368, 0.9310817718505859, 1.3799511194229126, -3.4132189750671387, -2.165806531906128, 1.645508050918579, 1.1327236890792847, 1.6092473268508911, -2.7912306785583496, -3.7526772022247314, -0.5141009092330933, -0.7007070183753967, 0.21837502717971802, 2.720144748687744, 2.2002432346343994, -2.9053232669830322, -2.3545894622802734, -2.4384634494781494, -1.5952765941619873, -1.4006174802780151, -3.54665207862854, -0.07382608205080032, 2.256586790084839, -2.7538747787475586, -1.0078176259994507, 0.2885914444923401, 3.2351601123809814, -0.2289954423904419, -0.7742176055908203, -2.929046630859375, -2.420259952545166, -1.0763822793960571, 1.1564087867736816, -3.9482998847961426, -0.7123233675956726, 2.812836170196533, -1.3653675317764282, 3.4239604473114014, -0.6330455541610718, 3.098238706588745, 1.2880089282989502, 0.9763411283493042, -3.10801362991333, -3.9654951095581055, 3.447476387023926, 0.6864002346992493, -1.5306652784347534, -0.14290225505828857, 3.690690279006958, 2.846069097518921, 3.0466654300689697, -0.20353519916534424, 0.3308231234550476, -3.9062530994415283, 2.6533138751983643, -4.163821220397949, 0.643320620059967, 0.9439671635627747], [-2.2869973182678223, -1.1551072597503662, 0.13447821140289307, -5.591763019561768, -2.4906232357025146, 1.1059250831604004, 0.9856994152069092, 0.0321972519159317, -0.6874070167541504, 1.4628963470458984, 2.9049150943756104, -0.47671428322792053, -2.0259523391723633, -0.7738713026046753, -0.5917282700538635, 3.4821579456329346, -0.9210701584815979, -3.556194305419922, -3.9982199668884277, -2.3702070713043213, 0.09311172366142273, 2.1804068088531494, 2.1675755977630615, -1.80905282497406, 1.6603262424468994, -5.902309894561768, 0.964289128780365, 2.215934991836548, 1.718536615371704, 4.968874931335449, 0.8673722743988037, -1.3412226438522339, 1.9027162790298462, 0.5356196165084839, -2.237828254699707, 0.2219114452600479, 4.673793315887451, -1.3149770498275757, -0.393151193857193, 1.7027961015701294, 1.8478941917419434, 1.1835347414016724, 2.7878425121307373, -1.1664537191390991, 0.045409999787807465, 2.2957518100738525, -1.7017854452133179, 1.119568943977356, 3.862910509109497, -1.1992933750152588, -2.452322483062744, -0.433849573135376, 3.0156009197235107, 1.5647222995758057, 1.2797107696533203, 2.2114083766937256, -2.5694022178649902, 0.7677266001701355, -3.0415873527526855, 4.437743663787842, -3.5563042163848877, 0.9822595119476318, -3.228754758834839, -3.110527753829956, 4.806571006774902, -0.19881628453731537, -3.140350341796875, 3.900752544403076, -1.450541615486145, 0.460898756980896], [-0.5417299270629883, 0.3700781464576721, 0.4534635841846466, 2.308196544647217, 6.9166340827941895, -1.3750394582748413, 0.3683535158634186, -0.36082425713539124, -1.3465954065322876, -2.1420507431030273, 1.48590886592865, 4.176395416259766, 0.9196629524230957, -3.790415048599243, 1.6739308834075928, -2.425950050354004, -2.5795602798461914, 0.8086926937103271, -4.933237552642822, 2.5760135650634766, -3.2572100162506104, -5.29489803314209, 0.28317907452583313, -2.0458757877349854, 1.2414660453796387, 2.5297162532806396, 2.3148884773254395, 0.13181717693805695, -4.761946201324463, 0.3075641393661499, -4.619492530822754, -0.9635599851608276, 0.20421861112117767, 1.3351540565490723, -2.7337779998779297, -1.9361166954040527, -2.1894445419311523, -0.3753938674926758, 1.4527521133422852, 0.9109295010566711, -0.9001730680465698, -3.72686767578125, 0.18839535117149353, -0.5237630605697632, -0.48112061619758606, 1.454650640487671, -4.167687892913818, 3.9111742973327637, -1.8288222551345825, 1.974758267402649, -0.5854076147079468, 3.137547731399536, 0.19293959438800812, 1.4443556070327759, -1.588244915008545, -4.799875259399414, 0.44897735118865967, -2.2824809551239014, 0.8043823838233948, 0.7007467150688171, -2.46709942817688, 1.4297987222671509, 4.841965675354004, 1.3950450420379639, 0.6285862326622009, -2.784364938735962, 1.7309738397598267, -0.11834125965833664, -3.799816370010376, 5.3621063232421875], [-0.566900372505188, -4.433475017547607, 0.5257214307785034, 1.1209770441055298, 1.3058109283447266, -1.0984382629394531, 0.206489697098732, -0.8862558603286743, -0.3287236988544464, 0.2706218361854553, 1.204243779182434, 2.22662615776062, -0.35378336906433105, -2.458286762237549, -0.6261158585548401, -1.9778425693511963, -1.282662272453308, 1.6051194667816162, -0.9694844484329224, 2.772454023361206, -0.7078033685684204, -3.1074118614196777, -1.2398767471313477, -2.282172203063965, 2.5935699939727783, 0.8400108814239502, -1.3789722919464111, 0.899491012096405, -2.094594717025757, -2.7393319606781006, 4.2070252675330266e-05, 1.395593523979187, -0.2793900668621063, -3.932128667831421, -0.8927640318870544, -2.2338335514068604, -2.2776272296905518, 0.30827948451042175, -3.159914493560791, 0.4705541729927063, 0.436283141374588, -2.4431512355804443, 2.045935869216919, 1.7301656007766724, -0.3359321355819702, -0.4084983766078949, 0.47332265973091125, 2.226510763168335, -4.152137756347656, -0.12817567586898804, -4.222015857696533, 5.156602382659912, -0.2345835268497467, 1.1195875406265259, -3.8817200660705566, -5.702618598937988, -1.5670299530029297, -2.1117420196533203, 1.8791084289550781, -5.202260494232178, 0.40600916743278503, 0.9343025088310242, 1.9357573986053467, -0.24854525923728943, -1.0491082668304443, -0.9761908650398254, 1.9130924940109253, -0.30175960063934326, -0.5520146489143372, 2.760791063308716], [1.748748540878296, -1.4354009628295898, -0.935243546962738, -0.7920806407928467, -0.4564209580421448, 1.357291579246521, -3.3177201747894287, 0.9256746768951416, -1.3121228218078613, -1.2438982725143433, -4.714621543884277, -0.4977778196334839, 2.2281484603881836, -0.8343791961669922, 3.0666379928588867, -3.2267379760742188, 7.261401653289795, 1.7789512872695923, -1.7263504266738892, -0.044786516577005386, 2.1626980304718018, 2.3499598503112793, -1.678265929222107, 0.9516592025756836, 0.8734503984451294, 6.002700328826904, -2.5022270679473877, 0.22794416546821594, -3.6457691192626953, -3.1196632385253906, -0.8631690144538879, 2.6418654918670654, -0.19208510220050812, 2.041414260864258, 3.4629783630371094, 0.8327973484992981, -0.9892432689666748, 0.7922072410583496, 0.40987592935562134, -2.284034252166748, -6.571172714233398, -1.6657081842422485, -0.947685182094574, -1.294960379600525, 2.97782039642334, -0.7575826644897461, -0.3762909770011902, -2.2826387882232666, -2.579202175140381, 3.334892988204956, 1.6006277799606323, 3.7799980640411377, -1.21795654296875, -1.8583736419677734, 0.4717380702495575, 3.2307322025299072, -0.6882383227348328, -0.9908255338668823, 4.861832141876221, -5.336045265197754, 1.0393303632736206, -0.29650822281837463, 3.893908739089966, 2.067365884780884, -7.20762300491333, -3.4109861850738525, -0.8458092212677002, 3.177727699279785, 1.7893047332763672, 4.9567389488220215], [0.3124319612979889, -1.3001115322113037, 0.7251476049423218, 0.8612518906593323, 1.148110270500183, -1.3895580768585205, -0.583332359790802, 0.3090498447418213, 0.5186194181442261, -0.07033811509609222, -1.7368372678756714, 1.2519482374191284, 0.32692214846611023, -1.5158921480178833, 0.023129725828766823, -0.06326602399349213, -2.2417585849761963, 0.98753821849823, 1.0731745958328247, 0.9013398885726929, 0.6811442971229553, -0.12384869158267975, -0.8908921480178833, -0.19503386318683624, -0.11335018277168274, 0.8111693263053894, -0.6520899534225464, 0.2559681534767151, -0.08576157689094543, -1.5514557361602783, 0.7906509637832642, 0.4863347113132477, -1.8425962924957275, -2.5192971229553223, 1.1523633003234863, -1.4417614936828613, -0.18839478492736816, 0.20189209282398224, -0.48900270462036133, 0.20025312900543213, 1.2766849994659424, 0.19059626758098602, -0.13749176263809204, 0.7184156179428101, -0.7020121812820435, -0.16447769105434418, 1.8313528299331665, 0.7241830825805664, 0.05252707377076149, -1.1494554281234741, -1.627267599105835, 1.0512412786483765, -0.946334958076477, -0.478336900472641, -1.0043582916259766, -1.067284107208252, 1.4076846837997437, -0.9981714487075806, 0.40790900588035583, -2.8243279457092285, 1.2102807760238647, 0.27702805399894714, 1.6136505603790283, -0.8410491943359375, 0.223038449883461, 0.688387930393219, -0.14429374039173126, -0.654384434223175, 1.4644925594329834, 0.4996565580368042], [-0.508089005947113, 1.1480522155761719, -0.4347275495529175, 0.5643346905708313, 1.2026605606079102, -1.6316524744033813, 0.4494972229003906, 0.42685773968696594, -0.10467979311943054, -1.3412963151931763, -1.3270820379257202, 1.4649059772491455, 0.06788931787014008, -1.822020411491394, 0.8550867438316345, 1.137090802192688, -1.7758244276046753, 0.20011793076992035, -0.8499957919120789, -0.5537344813346863, 0.6928645372390747, -0.08303441107273102, -0.2701713740825653, 0.1258051097393036, -0.8847653865814209, 0.13972891867160797, 1.2217216491699219, 0.7986811995506287, -0.35906779766082764, 0.6687584519386292, -0.7907772660255432, 0.015617894940078259, -0.35031577944755554, 0.4081498384475708, 0.24300386011600494, 0.0055020698346197605, 0.3194788098335266, -0.25956475734710693, 1.5238831043243408, -0.21012288331985474, 0.8060092926025391, 0.6503358483314514, -0.7919180393218994, -0.3556809425354004, 0.1287340223789215, 1.695768117904663, -0.2678908109664917, 0.5805517435073853, 1.9928383827209473, -0.837136447429657, 0.12190262973308563, -1.5901122093200684, -0.857728898525238, -0.5269157290458679, 1.0835025310516357, 0.21575962007045746, 0.9764408469200134, -0.9576074481010437, -0.7104485630989075, 0.800545871257782, -1.0208871364593506, 0.12475869059562683, 1.536880373954773, 0.0026860604993999004, 0.9825637936592102, -0.021004553884267807, -1.2109986543655396, 0.9190199971199036, 0.0703907310962677, 0.8980539441108704], [0.4124092161655426, -0.902315080165863, -1.2519378662109375, 0.3617108464241028, 0.9857333302497864, -1.8924294710159302, 0.8036534786224365, -0.11152025312185287, 1.9537485837936401, 0.7442758679389954, 1.4464017152786255, 2.1934707164764404, -1.0421547889709473, 0.7968408465385437, -0.29696711897850037, 0.024489831179380417, -0.6813812255859375, 1.5753966569900513, 1.5276415348052979, -0.7309754490852356, 0.3744371235370636, -2.8460404872894287, -1.6528398990631104, -1.1588482856750488, 0.7803201675415039, -1.5771554708480835, 2.0426504611968994, 0.12511052191257477, 0.04434328153729439, -1.848028302192688, 0.4100339710712433, 1.2631891965866089, 0.02826852723956108, -1.498354196548462, 1.3886122703552246, -0.029793601483106613, -1.4824576377868652, 0.11967204511165619, 0.8048709630966187, -1.8206837177276611, 0.9611541628837585, -0.7276380062103271, -0.8371070623397827, 0.3986409902572632, 3.2241880893707275, -1.2488750219345093, 0.34732693433761597, 1.2507988214492798, -1.2284550666809082, 0.6325801014900208, -2.833322763442993, -0.04480226710438728, 0.38751357793807983, 0.9809108972549438, -2.3981988430023193, -5.444256782531738, -1.2884937524795532, -0.13528020679950714, -2.7224717140197754, -0.359812468290329, 1.5649125576019287, 1.9574264287948608, -0.6434271931648254, 0.03025166690349579, 0.09223174303770065, -3.194650411605835, 1.281211256980896, -0.8391261100769043, 0.7660210728645325, -1.1866282224655151], [-1.1585280895233154, -4.248426914215088, 2.9524447917938232, -0.690584659576416, 0.4392067790031433, -0.964271605014801, 1.0231883525848389, -0.8225393891334534, 1.3260036706924438, 2.993532180786133, 3.2942497730255127, 1.0178951025009155, -1.8006961345672607, -1.0204070806503296, -3.4755895137786865, 1.5396909713745117, -7.662888526916504, -1.0278161764144897, 1.8495711088180542, 2.273942232131958, -1.5724211931228638, -1.5023510456085205, 0.7911013960838318, -2.4762179851531982, 1.6275945901870728, -4.903215408325195, -1.041111946105957, 0.38956430554389954, 2.8305368423461914, 0.24515493214130402, 3.1270265579223633, -1.4921948909759521, -2.0569510459899902, -6.974873065948486, -2.128946542739868, -3.793605089187622, 1.5102506875991821, -0.38391974568367004, -4.0525383949279785, 3.3880844116210938, 6.612889766693115, 0.46731555461883545, 3.225404977798462, 2.4788920879364014, -4.042016506195068, -0.8838582038879395, 3.040070056915283, 3.2652454376220703, 0.07558967918157578, -3.828059434890747, -5.969237327575684, 2.575594425201416, 1.4846755266189575, 2.0230796337127686, -3.7340354919433594, -4.44992733001709, 0.5476045608520508, -0.13246358931064606, -2.245943307876587, -1.6451143026351929, 1.0642526149749756, 1.056835651397705, -2.3485560417175293, -4.339061737060547, 6.069379806518555, 3.5830678939819336, 0.9443689584732056, -2.947802782058716, 0.26484036445617676, -2.424708366394043], [-0.5882091522216797, -2.4776031970977783, 3.0715463161468506, -3.2204267978668213, -0.7164766192436218, 4.747159481048584, -2.550764322280884, -0.19384446740150452, -3.6484973430633545, 0.5735482573509216, -0.08764509111642838, -2.9924614429473877, 1.5310717821121216, -1.7673509120941162, 0.4042781591415405, -2.0546457767486572, 2.5714049339294434, -3.020193099975586, -5.236404895782471, 2.4625790119171143, -2.0951523780822754, 3.466909646987915, 3.066086769104004, -0.49080684781074524, 2.107628345489502, 2.131328582763672, -4.415554523468018, 0.10347067564725876, -1.9195241928100586, 2.538181781768799, -0.697727382183075, -1.9659799337387085, 0.3225206732749939, 1.1786348819732666, -3.1433959007263184, -1.6112512350082397, 2.8442111015319824, -0.3185991644859314, -3.39016056060791, 3.858123302459717, -3.0419411659240723, -1.523423433303833, 3.832581043243408, -0.7892065644264221, -4.632489204406738, 0.660302996635437, -1.8577643632888794, -0.29016396403312683, -0.8742067813873291, 0.901442289352417, 1.3693500757217407, 5.858022212982178, 1.7029114961624146, 0.21398428082466125, 0.9836918711662292, 6.4944682121276855, -0.21682700514793396, 0.1052166223526001, 5.344338893890381, -1.4760916233062744, -2.6140518188476562, -1.5719268321990967, 1.1721062660217285, -1.2197058200836182, -0.5531821846961975, 2.574465036392212, -1.1688916683197021, 2.3378472328186035, -2.303725481033325, 4.907119274139404], [2.3732712268829346, -5.508626937866211, 3.9801290035247803, 0.8797950148582458, 4.877564907073975, 3.674551248550415, -2.934765100479126, -1.326589822769165, -0.12637029588222504, 3.3785560131073, 4.120667934417725, -0.010869004763662815, 1.568469524383545, 1.514789342880249, -1.4840412139892578, -7.209023475646973, 0.7731112837791443, 0.9043067693710327, -0.227279931306839, 6.172852039337158, -5.687631130218506, -4.808679103851318, 0.6973846554756165, -2.69167423248291, 4.3837151527404785, 3.2893505096435547, -3.3259551525115967, -3.2353601455688477, -3.4495432376861572, -3.9648802280426025, -0.5469439625740051, -1.2274863719940186, -1.6736489534378052, -3.3839378356933594, -2.06817889213562, -4.466291904449463, -2.927586555480957, 0.8314040899276733, -4.700004577636719, 2.6218135356903076, -2.298309803009033, -6.0995259284973145, 2.9670093059539795, 1.4349384307861328, -2.6219751834869385, -5.901815414428711, -0.7553189992904663, 2.8890700340270996, -8.950589179992676, 3.9947426319122314, -2.8482956886291504, 11.128324508666992, 2.7637031078338623, 2.9223010540008545, -7.0687689781188965, -6.14121150970459, -0.4377802908420563, 0.5832779407501221, 3.987044334411621, -5.62447452545166, 3.475311040878296, 1.4804226160049438, 1.207038402557373, -0.2601983845233917, -2.1886541843414307, -1.5439008474349976, 6.202350616455078, -5.1857404708862305, -2.0640347003936768, 2.5628128051757812], [-0.9831579327583313, 1.8809348344802856, -3.009812355041504, 2.9657535552978516, -6.4435200691223145, -2.2932093143463135, 3.399841785430908, -0.7174196839332581, 1.4102836847305298, -0.7246536612510681, -0.4468442499637604, -1.9205228090286255, -1.9216063022613525, 2.5349252223968506, -2.1202831268310547, 3.1036007404327393, 0.3636707067489624, 1.4676799774169922, 6.070377349853516, -2.1617021560668945, 3.4988057613372803, 1.0564427375793457, -1.9486840963363647, 2.2796149253845215, -2.755692720413208, -2.1519453525543213, 0.06090812757611275, 0.5670210123062134, 4.228573322296143, -1.4594181776046753, 2.722240924835205, 2.3398940563201904, 1.4853898286819458, -1.3400062322616577, 1.318372130393982, 3.33646559715271, -2.211336612701416, 0.6006319522857666, -0.7798842191696167, -2.5806941986083984, 2.6528937816619873, 3.834317684173584, -1.6567188501358032, 2.1076602935791016, 1.1773780584335327, 0.7558047771453857, 3.6453335285186768, -3.4151384830474854, 0.5437219142913818, -3.3985018730163574, 1.2054466009140015, -7.030059814453125, -2.994429111480713, -1.6500272750854492, 1.5750889778137207, -0.5603554248809814, -1.0745288133621216, 0.3103107213973999, -1.5015474557876587, -0.10754095762968063, 1.2206518650054932, -2.162597179412842, -3.8458900451660156, 1.5994044542312622, -1.1839931011199951, 3.111440658569336, 0.062424659729003906, -1.033023715019226, 2.2688419818878174, -6.5538177490234375], [0.6240474581718445, -3.18973445892334, 2.7152059078216553, -3.4136734008789062, 5.822242259979248, 1.0928972959518433, -3.27036190032959, 0.9008666276931763, -1.0245436429977417, 0.7617922425270081, -0.3942570984363556, 2.811964511871338, 1.4460197687149048, -3.646205425262451, 2.104418992996216, -2.2282555103302, -1.2488460540771484, -1.0029343366622925, -5.794492721557617, 1.7791730165481567, -2.1795623302459717, -0.7016067504882812, 1.125929594039917, -2.715596914291382, 3.0393974781036377, 1.4509477615356445, -0.20672982931137085, 0.45414578914642334, -3.8985610008239746, 0.8276371955871582, -1.780893325805664, -1.4180611371994019, -1.9341427087783813, -0.35962218046188354, -0.39626213908195496, -3.685927152633667, 2.5362987518310547, -0.6202869415283203, 0.3886949419975281, 2.206995725631714, -1.6844682693481445, -3.28365159034729, 1.8760885000228882, -1.7239885330200195, -0.5511023998260498, -0.25549590587615967, -2.480954170227051, 3.717672109603882, -0.018877580761909485, 2.3569583892822266, -3.0947110652923584, 7.215543270111084, 2.564136266708374, 1.4377399682998657, -2.023777961730957, -0.26143357157707214, 0.7912859320640564, -1.0662038326263428, 1.0720385313034058, -1.4783943891525269, -0.9883831739425659, 2.580681562423706, 4.146104335784912, -2.405592441558838, 1.4528638124465942, -3.2958006858825684, -0.9041153788566589, 1.931330919265747, -1.1312237977981567, 6.8327226638793945], [-0.41805949807167053, 1.001106858253479, 1.4516204595565796, 4.513636112213135, 3.04447078704834, -1.4267396926879883, 0.46569567918777466, -0.3709891736507416, -1.193907618522644, -2.1838276386260986, -1.5759516954421997, 1.076640009880066, 1.2652084827423096, -2.9078330993652344, -0.06874316185712814, -1.0115216970443726, -4.135997295379639, 0.922092616558075, 0.13161776959896088, 3.2096989154815674, -1.3606406450271606, -1.604382872581482, 0.15552347898483276, 0.6221276521682739, -1.4817752838134766, 3.283815860748291, -0.8979875445365906, -0.48184049129486084, -1.262956142425537, -0.7049921751022339, -1.579184651374817, -0.9032633304595947, -1.5628379583358765, -1.1951707601547241, -1.7911797761917114, -1.714255928993225, -1.9779746532440186, 0.2515164911746979, -0.7680591344833374, 1.7634551525115967, 1.5041652917861938, -0.293666809797287, -0.1896710842847824, 1.3240771293640137, -4.367491722106934, 1.3717650175094604, 0.5210277438163757, 0.993217945098877, -0.5649248361587524, -1.8045744895935059, 1.4933967590332031, 0.18425790965557098, -2.4432947635650635, -0.8657684922218323, 0.170136958360672, -0.4957004487514496, 3.1950528621673584, -1.8857429027557373, 2.7113118171691895, -1.8768857717514038, -0.5555031299591064, -1.5946006774902344, 3.5505259037017822, 0.9487558603286743, 0.39867517352104187, 3.7725319862365723, 1.095490574836731, -2.3099477291107178, -1.0724513530731201, 1.2901806831359863], [0.25815245509147644, 2.0785539150238037, -3.283613443374634, 2.4148778915405273, -2.9808175563812256, -1.468356728553772, 0.4373689591884613, 0.3515159785747528, -0.43699949979782104, -2.881052017211914, -4.1014556884765625, -0.5206397175788879, 0.6092740893363953, -0.01425060722976923, 1.755743384361267, 0.004448891617357731, 4.819694995880127, 2.407158374786377, 1.4594604969024658, -1.7479844093322754, 3.6463680267333984, 1.511124610900879, -2.4456207752227783, 2.406890869140625, -1.9832782745361328, 3.630002975463867, -0.2612127661705017, 0.7679299116134644, -0.8177481293678284, -2.3867437839508057, -0.5367240905761719, 3.1795566082000732, 1.22019362449646, 2.2984354496002197, 2.894918918609619, 3.2459046840667725, -2.6027846336364746, 0.7596668004989624, 1.5319327116012573, -3.654137134552002, -3.2211225032806396, 1.224286675453186, -2.696878671646118, -0.11892954260110855, 3.1973910331726074, 1.3313839435577393, 0.7184783816337585, -3.447805404663086, -0.48788994550704956, 0.3828788995742798, 3.1742138862609863, -3.4012651443481445, -3.4337470531463623, -2.6527557373046875, 2.4827706813812256, 1.9195990562438965, -0.6277052760124207, -1.132361888885498, 2.168410539627075, -2.2022714614868164, 0.24324490129947662, -1.6596630811691284, 1.688062071800232, 3.4902524948120117, -5.686049938201904, -0.9554429054260254, -0.9088314175605774, 2.12146258354187, 1.818793535232544, 0.2542246878147125], [0.6859821081161499, -0.4359804689884186, 3.0871825218200684, 1.1643223762512207, 1.4585763216018677, 4.338761806488037, -1.4350793361663818, -1.0619597434997559, -2.1793549060821533, 0.8308019042015076, 2.1985466480255127, -2.965203285217285, 1.7327451705932617, 1.2392287254333496, -1.0028588771820068, -3.989842414855957, 1.319348692893982, -1.391330361366272, -1.2809927463531494, 4.113938331604004, -4.4336066246032715, -0.5628967881202698, 2.603689670562744, 0.17009320855140686, 0.9749854207038879, 3.097994089126587, -3.205935478210449, -2.5722768306732178, -1.5485635995864868, 0.43945568799972534, -1.4254639148712158, -2.576728343963623, 0.0296540055423975, 1.0237724781036377, -3.7506332397460938, -1.558382511138916, -0.8827154636383057, 0.32065147161483765, -3.072376012802124, 3.3162951469421387, -2.1783204078674316, -2.5678915977478027, 2.139446258544922, 0.4463532567024231, -5.109245777130127, -1.9406548738479614, -1.6408429145812988, -0.061166711151599884, -3.9396660327911377, 1.7762295007705688, 2.963449239730835, 4.515528678894043, 1.1564371585845947, 0.8522830009460449, -0.8732358813285828, 2.005078077316284, 0.9050363898277283, 1.0729831457138062, 4.565853118896484, -0.5854744911193848, -0.17891070246696472, -1.7515968084335327, 0.25522851943969727, 0.8293066024780273, -1.0701265335083008, 2.9852709770202637, 3.0115537643432617, -3.0052740573883057, -3.1134402751922607, 1.2171331644058228], [-1.618847131729126, -1.905319094657898, 0.11148276925086975, -0.7932230830192566, -3.128047466278076, -0.44816094636917114, 0.34750470519065857, -0.060954757034778595, -1.3879034519195557, -0.7233253121376038, -1.8782426118850708, -0.6458754539489746, -0.40989217162132263, -2.6499881744384766, -0.2507631480693817, 1.6072512865066528, -0.512619137763977, -0.5278012156486511, -0.9952445030212402, 0.11912142485380173, 2.0242156982421875, 2.7437214851379395, -0.06833788752555847, 0.0621304027736187, 0.12598846852779388, -0.2629341185092926, -2.3233401775360107, 2.0535550117492676, 0.6257257461547852, 0.4942238926887512, 1.3216155767440796, 0.9057653546333313, 0.11352154612541199, -1.8765723705291748, -0.389438658952713, -0.17397776246070862, 1.2278918027877808, -0.10729432851076126, -2.316347360610962, 0.8774001598358154, 0.9324296116828918, 1.541534423828125, 1.6936030387878418, 0.8093199133872986, -1.7721964120864868, 2.376689910888672, 1.604927659034729, -0.7594771385192871, 1.1767477989196777, -2.685196876525879, -1.0324233770370483, 0.3065161406993866, -1.2662200927734375, -1.042803406715393, 1.155189037322998, 2.6057910919189453, -0.44513842463493347, -1.4391682147979736, 1.9835433959960938, -2.7010488510131836, -1.5147473812103271, -1.3432326316833496, 0.5040043592453003, -1.0329010486602783, 0.03492826595902443, 2.7907421588897705, -2.255768060684204, 2.5777363777160645, 0.7638846039772034, 1.3511598110198975], [0.21211619675159454, -0.983423113822937, -1.032448410987854, -0.1297616958618164, -4.672338485717773, 0.15405632555484772, 0.07759551703929901, -0.07036149501800537, 0.5543215870857239, 0.5721497535705566, -1.4618685245513916, -2.0954177379608154, -0.5067105293273926, 1.6727465391159058, -0.8467670679092407, 0.714568018913269, 2.5664443969726562, 0.7267307043075562, 3.219507932662964, -1.027633786201477, 2.6152102947235107, 2.475222110748291, -1.2199697494506836, 1.230232834815979, -0.5045393705368042, -0.185105100274086, -1.9690991640090942, 0.3039487302303314, 2.003553628921509, -1.7855186462402344, 2.5976955890655518, 1.8881714344024658, 0.2351539433002472, -1.3865466117858887, 2.003000497817993, 1.4237014055252075, -0.35594630241394043, 0.5751647353172302, -1.5953035354614258, -1.398186445236206, -0.2344173789024353, 1.7560232877731323, -0.19589053094387054, 0.9390922784805298, 0.950563371181488, -0.7187435626983643, 2.8428361415863037, -2.684279441833496, -0.6129044890403748, -1.1028696298599243, 0.05731283873319626, -1.2403537034988403, -1.189431071281433, -1.2231621742248535, 0.3641190230846405, 1.486784815788269, -0.9971364736557007, 0.43942996859550476, 0.8287174105644226, -2.67331862449646, 1.7134153842926025, -1.1463528871536255, -1.9574980735778809, 0.214715838432312, -2.4079527854919434, 1.2499363422393799, -0.4518766701221466, 0.3416622579097748, 2.487300395965576, -2.466097354888916], [1.5631382465362549, 0.7226517200469971, -2.395989179611206, 1.8903580904006958, 2.9448156356811523, -3.330803871154785, -0.3766234815120697, 0.8708997368812561, 2.0085532665252686, -1.3285776376724243, -2.431079626083374, 3.6609675884246826, 0.4179935157299042, -0.4038587510585785, 2.1166036128997803, -0.8706437349319458, 0.9374663829803467, 3.4479691982269287, 1.448545217514038, -1.3370461463928223, 1.9514076709747314, -2.6713500022888184, -3.228992462158203, 0.0022174022160470486, -0.48071932792663574, 2.188474655151367, 2.798555612564087, 0.1338634490966797, -1.986307978630066, -3.6141462326049805, -1.0192217826843262, 2.5928077697753906, -0.8980457186698914, 0.23467884957790375, 4.125287055969238, 0.6267778873443604, -2.66554856300354, 0.49173933267593384, 3.558837652206421, -3.8824844360351562, -1.4574412107467651, -0.9245721697807312, -3.4029829502105713, -0.5530310869216919, 5.379885196685791, -0.8234647512435913, 0.1742088347673416, 0.5258124470710754, -0.8266839385032654, 1.8975518941879272, -1.104200005531311, -0.834041178226471, -1.4907225370407104, -0.6514899134635925, -1.2826365232467651, -4.578370571136475, 0.1933411955833435, -1.2241016626358032, -1.6319071054458618, -1.816974401473999, 2.230285882949829, 2.050262212753296, 2.9010009765625, 1.7815515995025635, -2.926560163497925, -5.279289245605469, 0.5614467263221741, 0.11810488253831863, 2.1401519775390625, 0.9207335114479065], [-0.43221619725227356, 1.362313151359558, 2.6245319843292236, 6.073764324188232, 10.227550506591797, -4.909662246704102, 1.0160819292068481, 0.03771335631608963, 0.3330451250076294, -2.556762456893921, -0.6284487247467041, 6.254188537597656, 1.1488887071609497, -5.830163478851318, 0.3389888405799866, -0.555527925491333, -11.725664138793945, 1.6874021291732788, -0.79537034034729, 4.343188285827637, -3.3502931594848633, -6.235283851623535, -0.09781396389007568, -1.4264326095581055, -1.4488279819488525, 2.0868184566497803, 2.817929267883301, -0.4556220769882202, -2.4622790813446045, -0.4567980468273163, -3.2445223331451416, -2.386451482772827, -3.93095326423645, -3.223700523376465, -2.301741600036621, -4.7365498542785645, -2.11767578125, -0.33804383873939514, 1.7073030471801758, 2.724125385284424, 5.374837398529053, -1.499589443206787, -1.0153487920761108, 1.310053825378418, -4.771413326263428, 1.8304173946380615, -0.2728831171989441, 5.763078689575195, 1.1028268337249756, -2.45792818069458, -1.7862788438796997, 0.6180010437965393, -1.960084319114685, 0.5827703475952148, -2.063347101211548, -6.444173812866211, 5.771537780761719, -3.2818942070007324, -0.9181274175643921, -0.28021669387817383, -0.5649706721305847, 1.098921775817871, 6.3717732429504395, -0.5996612310409546, 5.255772113800049, 2.148399829864502, 1.8125792741775513, -4.407994270324707, -1.8725429773330688, 2.469383955001831], [2.5984833240509033, -8.409701347351074, 0.48957961797714233, -6.473499298095703, -1.4290181398391724, 2.1152806282043457, -3.7677552700042725, 0.4389505088329315, 2.812342882156372, 5.67127799987793, 1.6498295068740845, 0.43685659766197205, -0.8659560680389404, 2.975207567214966, -0.40904930233955383, -2.7431349754333496, 5.189920425415039, 1.0867136716842651, 1.1720104217529297, -0.7155099511146545, 1.1603007316589355, 0.12830255925655365, -2.082029104232788, -3.2683393955230713, 5.618366241455078, -2.136162519454956, -2.026677370071411, 0.03117293305695057, -0.4008278548717499, -4.579950332641602, 4.1622161865234375, 2.7732632160186768, -1.1351916790008545, -4.67600679397583, 4.651952266693115, -1.8933337926864624, 1.2195303440093994, 0.5169148445129395, -2.739901542663574, -1.8919209241867065, -2.599853515625, -2.9379329681396484, 2.091968059539795, -0.1807251274585724, 6.029149055480957, -6.093076705932617, 2.071455717086792, 0.9054038524627686, -4.856952667236328, 3.7575953006744385, -7.970542907714844, 8.933464050292969, 4.086050987243652, 2.246300220489502, -6.545554161071777, -4.883707046508789, -4.436387538909912, 1.4729453325271606, -1.1271686553955078, -6.486476421356201, 5.218741416931152, 4.331676959991455, -2.4366037845611572, -3.1771087646484375, -2.8910727500915527, -7.2852396965026855, 1.1435619592666626, 1.0379873514175415, 3.761979818344116, 0.9821231961250305], [-0.832443356513977, 2.8794984817504883, -0.5269379019737244, -1.353534460067749, -5.867093563079834, 3.3726766109466553, 0.18542969226837158, -0.09721329808235168, -2.2731804847717285, -0.7719299793243408, -1.0577062368392944, -5.327387809753418, 0.3356584906578064, 2.017437219619751, -0.12024444341659546, 1.3693821430206299, 4.739437103271484, -2.5201380252838135, -0.4902721047401428, -1.6269408464431763, 1.0559343099594116, 5.440474987030029, 2.044610023498535, 2.805746078491211, -1.8335461616516113, 0.3504813611507416, -2.2782068252563477, -0.08161580562591553, 1.9152706861495972, 3.1755731105804443, 0.36448386311531067, -0.7581644058227539, 2.3221819400787354, 4.194252967834473, -1.2925070524215698, 3.1839540004730225, 1.9341232776641846, -0.07186145335435867, -0.6102092862129211, 0.5844826698303223, -2.301450490951538, 2.553344488143921, 0.3867028057575226, -0.8411983251571655, -1.9726547002792358, 1.6744877099990845, -0.5035689473152161, -4.374355792999268, 2.0896053314208984, -0.5756115913391113, 5.708417892456055, -3.6305646896362305, -0.4822635054588318, -1.7264409065246582, 4.997404098510742, 9.065203666687012, -0.491444855928421, 1.674100399017334, 2.4760708808898926, 2.880962371826172, -2.2706336975097656, -3.478145122528076, -2.6471142768859863, 1.172594428062439, -1.544521689414978, 3.9287006855010986, -2.1669564247131348, 2.1546809673309326, -0.6504238247871399, -1.5283353328704834], [-0.5374590158462524, 2.6916141510009766, -0.6633647084236145, 0.853662371635437, 0.48590144515037537, -1.3174160718917847, 1.915786623954773, -0.0957329049706459, 1.0952225923538208, -0.15290774405002594, 1.6411559581756592, 0.5521509647369385, -1.0041531324386597, 1.0318800210952759, -0.7524704337120056, 2.07542085647583, -2.81643009185791, -0.4733048975467682, 1.2650362253189087, -1.2735066413879395, -0.4287808835506439, -1.2205700874328613, 0.40361520648002625, 0.27250194549560547, -1.4716212749481201, -2.7018613815307617, 2.7021005153656006, -0.2615920603275299, 1.8132826089859009, 1.6874983310699463, -0.2222369909286499, -1.0351881980895996, 0.42268356680870056, 0.7350987195968628, -0.7510585784912109, 0.7861503958702087, 0.18068493902683258, -0.39441025257110596, 1.8509416580200195, -0.157070592045784, 2.6560513973236084, 1.3091552257537842, -1.1165963411331177, -0.043450891971588135, 0.14177648723125458, 0.6568094491958618, -0.34044888615608215, 0.5373853445053101, 2.2928802967071533, -1.0847845077514648, 0.5733522772789001, -4.173346996307373, 0.16131988167762756, 0.441388875246048, 0.8768661022186279, -1.2288429737091064, 0.6015653014183044, 0.729965329170227, -3.4579505920410156, 4.296751022338867, -0.5168556571006775, 0.2529791295528412, -1.8150901794433594, -0.07414678484201431, 3.0637247562408447, 0.5622466206550598, 0.067520372569561, -1.31063711643219, -0.6040801405906677, -3.069159746170044], [-0.726908266544342, 1.1397819519042969, -2.101653814315796, -1.794251561164856, -5.940892696380615, 1.939184308052063, 0.39727362990379333, -0.06595727056264877, -1.3994419574737549, -0.677343487739563, -0.9238518476486206, -3.4601337909698486, -0.3476279079914093, 1.6549370288848877, 0.4216199815273285, 1.0800142288208008, 6.088046073913574, -0.970465362071991, -0.5824943780899048, -2.445065498352051, 2.344640016555786, 3.929523229598999, 0.2251613587141037, 1.7110424041748047, -0.5545365214347839, 0.02811712771654129, -1.3705589771270752, 0.8942349553108215, 0.9297886490821838, 1.2436344623565674, 0.5407745838165283, 1.3025239706039429, 2.8301265239715576, 3.2375261783599854, 0.3521660566329956, 3.4783616065979004, 0.7308206558227539, 0.0883440151810646, -0.3157412111759186, -1.4398818016052246, -3.0467560291290283, 1.6424561738967896, 0.07456453889608383, -0.7632840275764465, 1.5694175958633423, 1.2982295751571655, -0.47830474376678467, -3.8021864891052246, 0.6110277771949768, 0.3627651631832123, 3.097113609313965, -2.52047061920166, -0.49067026376724243, -1.320123314857483, 3.269857168197632, 5.371149063110352, -2.5228099822998047, 0.8439151644706726, 1.5843511819839478, 0.9805669784545898, -1.5618464946746826, -1.957067608833313, -2.1938695907592773, 1.5154259204864502, -3.1567063331604004, 0.3971131443977356, -1.9033775329589844, 3.41225004196167, 0.21134574711322784, -0.5634174346923828], [-2.635162115097046, -6.5434675216674805, -0.611336886882782, -2.153435468673706, -3.859895944595337, 1.0103498697280884, 0.6355361342430115, -1.433650255203247, -3.004317283630371, -0.11714521795511246, 1.7528232336044312, 0.06015913560986519, -1.1541458368301392, -3.632692575454712, -0.3518594205379486, -1.3268059492111206, 3.559783458709717, -0.06162600591778755, -5.1018500328063965, 1.8327038288116455, 0.6654745936393738, -0.1202785074710846, -0.4581267237663269, -2.788278341293335, 4.468542575836182, 0.12823258340358734, -3.667158842086792, 3.2456438541412354, -2.741093397140503, -0.8294455409049988, 0.10875000804662704, 2.5877649784088135, 2.95243763923645, -2.461200475692749, -2.609502077102661, -0.5629517436027527, -0.9057663083076477, -0.005766395013779402, -5.695459842681885, 0.7062485814094543, -2.224637985229492, -2.372788906097412, 4.861331939697266, 1.3327655792236328, 0.07094968855381012, 2.005648374557495, -1.05962073802948, 0.574263870716095, -4.634544372558594, -0.06804939359426498, -4.226956367492676, 6.713891983032227, 0.30391183495521545, 1.040565848350525, -2.2421083450317383, -2.098026752471924, -5.596615791320801, -2.667994260787964, 4.562339782714844, -5.94851016998291, -3.0122430324554443, -0.28200772404670715, 0.8744563460350037, -0.059019654989242554, -3.051758050918579, -0.9535945653915405, -0.22074322402477264, 4.827442169189453, -1.9090744256973267, 5.662328720092773], [-0.019995547831058502, -1.9123215675354004, -2.2239270210266113, 1.376347303390503, -5.270378112792969, -2.352266550064087, 1.277481198310852, -0.250947505235672, 1.8447949886322021, 0.38087522983551025, -1.758919596672058, -0.5279658436775208, -1.432062029838562, 1.270422339439392, -1.4166966676712036, 1.503672480583191, 1.1168560981750488, 2.4968395233154297, 5.490527629852295, -1.3298524618148804, 4.1478142738342285, 1.0480008125305176, -3.1110286712646484, 0.9167172908782959, -0.673906147480011, -0.976034939289093, -1.3248326778411865, 1.040305495262146, 2.6812474727630615, -3.8798813819885254, 3.6575565338134766, 3.6016743183135986, -0.06898236274719238, -3.8802990913391113, 3.338256597518921, 1.4880508184432983, -1.9653903245925903, 0.8602695465087891, -1.8742363452911377, -2.8533825874328613, 1.5274251699447632, 2.311906099319458, -0.8543583154678345, 2.150487184524536, 2.491029977798462, -0.7243941426277161, 4.680467128753662, -2.374778985977173, -1.2397799491882324, -2.3882882595062256, -2.1605658531188965, -2.2524538040161133, -2.447690963745117, -1.3910577297210693, -0.9797855615615845, -2.250441789627075, -1.5084348917007446, -0.5210083723068237, -0.3791801333427429, -4.700260639190674, 2.9446053504943848, -0.6193508505821228, -2.012378692626953, 0.33386877179145813, -2.6965367794036865, 0.7353215217590332, 0.02281400002539158, -0.18830262124538422, 3.9350125789642334, -3.8144960403442383], [-0.44895830750465393, 0.03905481472611427, -2.3068442344665527, 1.9720532894134521, -2.7861361503601074, -1.7772408723831177, 2.1240158081054688, -0.6829156279563904, 1.072743535041809, -0.33380576968193054, 0.7365602850914001, 0.11045872420072556, -1.3296780586242676, 1.3125746250152588, -1.0305044651031494, 0.7627460956573486, 0.8772526383399963, 1.74576735496521, 2.9636054039001465, -0.9036134481430054, 1.6678237915039062, -1.3458553552627563, -1.873456358909607, 0.3612765669822693, -0.483539342880249, -1.0011814832687378, 0.6041374802589417, 0.46292930841445923, 1.2153443098068237, -1.9951156377792358, 0.990897536277771, 2.1323554515838623, 1.318724274635315, -1.119982361793518, 0.8490163087844849, 1.7324672937393188, -2.5837574005126953, 0.45826977491378784, -0.5474892258644104, -2.250000476837158, 0.8196815252304077, 0.8029617667198181, -0.8569552898406982, 1.3530577421188354, 2.2402989864349365, -0.10628525167703629, 1.3851109743118286, -1.1158301830291748, -1.5654865503311157, -0.8426512479782104, -0.6645211577415466, -2.5132393836975098, -1.4297479391098022, -0.243895024061203, -0.6764622926712036, -3.3850033283233643, -1.8601077795028687, -0.3018689751625061, -0.9882320761680603, -1.0014053583145142, 0.9667040109634399, -0.2964111864566803, -1.693148136138916, 1.4668383598327637, -1.649802565574646, -0.4060095250606537, 1.1087273359298706, -0.5529237389564514, 0.8781397342681885, -2.802356481552124], [-0.8024147152900696, -2.136427402496338, 4.011168003082275, -3.1231777667999268, 2.3462648391723633, 0.8041895627975464, -0.4647855758666992, 0.08342164754867554, 0.4828420579433441, 2.8876967430114746, 2.7209479808807373, 0.4496690332889557, -0.7943304777145386, -1.1806172132492065, -2.0088775157928467, 1.8608229160308838, -7.278054714202881, -3.174981117248535, -1.1239289045333862, 1.3406225442886353, -2.672870397567749, 0.38933053612709045, 2.8094682693481445, -2.1238620281219482, 1.2042292356491089, -4.769972801208496, -0.15312494337558746, -0.04888090118765831, 2.2373197078704834, 3.4457828998565674, 1.6930131912231445, -3.667027473449707, -2.3014583587646484, -3.557084798812866, -2.653754711151123, -3.835207223892212, 4.492092609405518, -1.0335116386413574, -1.6692777872085571, 4.606992721557617, 5.2900261878967285, 0.5070828795433044, 2.986342668533325, 0.26038798689842224, -4.743897914886475, -0.212802916765213, 0.7362380027770996, 3.4855144023895264, 3.1600635051727295, -2.5365407466888428, -3.718834161758423, 2.243093252182007, 3.042653799057007, 2.0364654064178467, -1.4246580600738525, 0.16603748500347137, 1.971402883529663, 0.9030399918556213, -2.7327675819396973, 2.2580108642578125, -0.578535258769989, 1.2834241390228271, -1.7561652660369873, -5.100754737854004, 7.805787563323975, 3.1495611667633057, -0.8836073279380798, -1.6068730354309082, -0.7804129719734192, -0.7555452585220337], [-2.3852243423461914, 2.2251243591308594, -0.5257501602172852, 5.128025054931641, 4.925356864929199, -5.112826824188232, 2.7729947566986084, -0.19833938777446747, -1.3600761890411377, -4.602493762969971, -1.5690339803695679, 4.942256450653076, 0.16002032160758972, -6.566143989562988, 1.180316686630249, 1.4010615348815918, -7.219122886657715, 1.4265410900115967, -2.8004255294799805, 1.7832245826721191, -0.3500262498855591, -3.8066625595092773, -0.6745186448097229, -0.5408849120140076, -1.8122687339782715, 1.7230455875396729, 2.6169564723968506, 1.944361925125122, -2.435871124267578, 0.8090945482254028, -3.586958885192871, -0.21408699452877045, -0.500436007976532, -0.5554768443107605, -2.450147867202759, -1.2862114906311035, -2.195281505584717, -0.469919890165329, 1.7183845043182373, 0.7924879789352417, 3.260028123855591, 0.023496361449360847, -0.8600025773048401, 0.8710552453994751, -2.450547933578491, 5.111245155334473, -1.3039875030517578, 3.12591814994812, 2.0550692081451416, -2.98370361328125, -0.13352537155151367, -2.4093356132507324, -3.317467451095581, -0.6398123502731323, 1.0647693872451782, -3.4828057289123535, 2.5290215015411377, -4.126034259796143, 0.15853619575500488, 0.3806576430797577, -3.6098475456237793, -0.4358654320240021, 5.58627462387085, 1.2281579971313477, 2.596991539001465, 1.7687742710113525, -0.5851813554763794, 0.36927565932273865, -2.2478411197662354, 3.395095109939575], [0.939153790473938, 0.9934656620025635, 1.261960744857788, 4.509649276733398, 2.1245925426483154, 0.31741613149642944, 0.5835704803466797, -1.0364086627960205, 0.39299237728118896, 0.14830173552036285, 1.7249184846878052, -0.5805615186691284, 0.713623046875, 1.7676061391830444, -1.633431315422058, -2.467170000076294, -1.882708191871643, 1.1473537683486938, 3.140320062637329, 3.113328218460083, -2.8258039951324463, -3.2052698135375977, 0.17723383009433746, 0.6158280372619629, -0.8376762270927429, 1.9793974161148071, -0.5893136262893677, -2.5689122676849365, -0.0059043667279183865, -1.927917718887329, -0.6342849135398865, -1.0904031991958618, -0.8109217286109924, -1.0369380712509155, -1.6241014003753662, -0.9652338624000549, -3.344762086868286, 0.6688246130943298, -1.3273497819900513, 0.943640410900116, 0.9284254312515259, -1.2173084020614624, -0.48436278104782104, 1.686463475227356, -2.8915047645568848, -2.18489408493042, 0.5940490961074829, 0.3180423974990845, -3.461664915084839, 0.14422684907913208, 1.6730183362960815, 0.40041202306747437, -0.8163574934005737, 0.4135812222957611, -1.8314656019210815, -3.051691770553589, 1.7638615369796753, 0.4545501172542572, 1.2906758785247803, -0.7307882905006409, 1.9447661638259888, -1.0151598453521729, -0.020002735778689384, 1.5704395771026611, -0.4709647297859192, 2.413147211074829, 4.0207929611206055, -5.314568519592285, -1.174696445465088, -2.4243054389953613], [-0.6637548804283142, 2.5793774127960205, 0.1425056755542755, -0.5435143709182739, 0.8019997477531433, 0.9496967196464539, 0.6770235896110535, -0.04306468740105629, -0.8535212874412537, -0.5523573160171509, 1.4395086765289307, -0.5636184215545654, 0.003490176983177662, 0.24507974088191986, 0.2572391927242279, 0.7743832468986511, -0.436909556388855, -1.7490016222000122, -2.0973238945007324, -0.5859717726707458, -1.5003862380981445, 0.05667279660701752, 1.7967183589935303, 0.23201216757297516, -0.6582102179527283, -0.9536153078079224, 1.4454559087753296, -0.281077116727829, 0.03336562588810921, 3.1313483715057373, -1.643170714378357, -1.8751498460769653, 1.1293619871139526, 2.981193780899048, -2.1148593425750732, 0.6895074248313904, 1.1745312213897705, -0.5842426419258118, 1.4710679054260254, 1.0135765075683594, -0.021276090294122696, 0.17028993368148804, 0.008526516146957874, -1.0886222124099731, -1.0110530853271484, 1.2620490789413452, -2.41910982131958, 0.2635960578918457, 1.9333906173706055, 0.37669771909713745, 2.243546962738037, -1.8991743326187134, 0.9445701241493225, 0.45729199051856995, 1.9346258640289307, 2.0500328540802, 0.300449401140213, 0.7561236619949341, -0.9587743878364563, 4.608076572418213, -2.2730653285980225, -0.35074102878570557, -0.644816517829895, 0.2478158175945282, 2.0289735794067383, 0.5048931837081909, -0.5336809754371643, 0.3800716996192932, -2.2078888416290283, 0.00587797025218606], [3.578950881958008, 3.1316051483154297, 3.2200071811676025, -3.419794797897339, -0.7209358811378479, 5.8509016036987305, -6.023789405822754, 2.2499263286590576, -1.013205647468567, 1.0930134057998657, -6.087417125701904, -5.946615219116211, 3.919384241104126, 3.26788067817688, 2.0997776985168457, -1.471405267715454, 5.238763809204102, -2.7214479446411133, 0.89966881275177, -0.8702890276908875, 0.2959649860858917, 8.716644287109375, 2.714237689971924, 4.163501262664795, -2.54009747505188, 4.696081638336182, -3.821756362915039, -3.1395246982574463, 0.9899872541427612, 1.742156744003296, 1.0573010444641113, -2.740670680999756, -3.0808873176574707, 4.707500457763672, 3.173344373703003, 0.4166385531425476, 5.046185493469238, 0.30564096570014954, 2.004502296447754, 1.561542272567749, -4.874703884124756, 1.8909101486206055, -1.663331151008606, -3.0620169639587402, -2.9167768955230713, -2.5286407470703125, 1.014295220375061, -4.550442218780518, 3.7050211429595947, 2.1813857555389404, 7.386081218719482, 0.057634931057691574, 0.7542033195495605, -2.932535409927368, 4.885630130767822, 14.617806434631348, 5.09684944152832, 3.5608785152435303, 3.7054545879364014, 1.0134704113006592, 2.180529832839966, -2.3435583114624023, 0.3691931664943695, -0.8732912540435791, -2.2626571655273438, 3.0474770069122314, -2.9226279258728027, -0.05586504936218262, 2.8038318157196045, -0.042792804539203644], [1.0060734748840332, 6.214425086975098, 2.631375551223755, 2.774796962738037, 3.327183246612549, 3.531022787094116, 1.3370071649551392, -1.1349347829818726, 0.49847525358200073, 1.5700196027755737, 6.172530174255371, -3.291783094406128, 0.4712038040161133, 5.866518974304199, -2.6914970874786377, -1.0802441835403442, -3.1988394260406494, -2.8302667140960693, 2.415956497192383, 1.782297968864441, -6.765644550323486, -2.8995304107666016, 4.270386695861816, 1.417992353439331, -2.347517251968384, -1.724772334098816, 2.1926372051239014, -5.0304412841796875, 2.4279489517211914, 3.936518430709839, -1.9904863834381104, -5.8999176025390625, 0.4184064567089081, 3.8975725173950195, -5.115076065063477, 0.07802560180425644, -0.35592392086982727, -0.30676424503326416, 1.3032392263412476, 3.3076319694519043, 2.134169816970825, -0.4087121784687042, -0.7632625699043274, -0.2132081240415573, -5.2153120040893555, -2.7091851234436035, -2.7293946743011475, 0.4559363126754761, 0.13632851839065552, 1.162235140800476, 6.002943515777588, -3.750760316848755, 2.497037649154663, 1.9942047595977783, 0.8566790223121643, 0.7426088452339172, 3.070307731628418, 4.268815517425537, -2.5054962635040283, 9.579651832580566, 0.12170340865850449, -1.2847293615341187, -4.415498733520508, 0.9413505792617798, 4.861361503601074, 4.061084270477295, 4.323722839355469, -7.660806655883789, -4.49020528793335, -6.393457412719727], [-4.873231410980225, -2.4122722148895264, -3.972851276397705, 3.258633613586426, -8.522507667541504, -3.5738821029663086, 6.094390869140625, -2.376650094985962, -1.747527837753296, -2.617677688598633, 1.6555484533309937, -0.14604918658733368, -3.492879629135132, -2.9274380207061768, -2.610628604888916, 3.3328306674957275, -0.19647575914859772, 1.500868558883667, 0.6199027299880981, -0.10719127207994461, 3.6578617095947266, -0.8829392790794373, -2.2222959995269775, -0.43732473254203796, 0.2090904712677002, -2.7110846042633057, -1.5902026891708374, 4.30957555770874, 1.6618551015853882, -0.9546892046928406, 1.440104365348816, 4.139250755310059, 4.534843921661377, -3.620482921600342, -2.9273431301116943, 2.701768636703491, -3.8064422607421875, 0.2310161292552948, -5.236911296844482, -1.3678996562957764, 3.048095703125, 2.2346115112304688, 2.518338680267334, 3.860436201095581, -0.14680224657058716, 5.043236255645752, 1.9846432209014893, -1.5825012922286987, -2.221198081970215, -5.307215690612793, -2.4632344245910645, -3.360281467437744, -3.882094383239746, -0.6664237380027771, 0.3363555669784546, -4.134003162384033, -5.351151466369629, -3.4354710578918457, 1.6876155138015747, -3.6979212760925293, -3.516310453414917, -2.7980306148529053, -1.9997180700302124, 1.976483941078186, -1.6682933568954468, 3.7138760089874268, -0.2826874554157257, 3.1124939918518066, -0.9507431387901306, -1.3517171144485474], [-1.4581934213638306, 1.0561586618423462, 2.6673758029937744, -5.655176162719727, -0.7079164981842041, 5.8203654289245605, -0.5848326086997986, -0.3691585063934326, -2.840512990951538, 1.8347886800765991, 4.6715826988220215, -3.7716333866119385, -0.11415550112724304, 1.1525697708129883, -0.5798547267913818, 0.6463205814361572, 1.127741813659668, -5.994154930114746, -6.257753372192383, -0.16876962780952454, -4.154725074768066, 2.816663980484009, 5.553009510040283, -0.883764386177063, 1.5658081769943237, -3.4305686950683594, -0.539913535118103, -0.4825853109359741, 0.5040932893753052, 7.539785861968994, -1.4084227085113525, -4.792602062225342, 2.3950252532958984, 4.56193733215332, -5.649702072143555, -0.11772602796554565, 5.3644022941589355, -1.4079434871673584, -0.804016649723053, 4.611390590667725, -0.995073676109314, -0.5382537841796875, 3.737689256668091, -2.2986419200897217, -4.066874027252197, 1.1090259552001953, -4.76336669921875, 0.5310745239257812, 2.681013345718384, 1.274612307548523, 2.5456910133361816, 1.4681265354156494, 4.750809669494629, 2.127683162689209, 2.647674798965454, 6.965511322021484, -1.122420310974121, 2.7497739791870117, -0.06660535931587219, 7.64736795425415, -4.7631306648254395, -0.6370011568069458, -3.405441999435425, -2.146178960800171, 4.683152198791504, 1.9511865377426147, -1.5710721015930176, 1.97942316532135, -4.838128566741943, 1.3739047050476074], [0.6345505714416504, 0.2368963360786438, -0.8448704481124878, -1.9413721561431885, -1.7728500366210938, 3.1876018047332764, 0.006337099242955446, -0.6955931782722473, 0.062011223286390305, 1.7723722457885742, 3.9048655033111572, -2.2871367931365967, -0.5165995955467224, 4.208029270172119, -0.5481042265892029, -1.591202974319458, 5.209521770477295, -0.9387189149856567, -0.4804796874523163, -0.8917633295059204, -1.6524595022201538, -0.7131931185722351, 0.9104278683662415, -0.31580397486686707, 1.529349684715271, -1.190885305404663, 0.37803617119789124, -1.3114322423934937, -0.06641682982444763, 0.6227778792381287, -0.4226050078868866, -0.27343785762786865, 2.660170555114746, 2.915302038192749, -0.9647958278656006, 1.800307035446167, -0.47490930557250977, 0.052358709275722504, -0.21328766644001007, -0.8019725680351257, -2.991708755493164, -1.6158307790756226, 0.5299155116081238, -0.8576663136482239, 2.321866035461426, -2.2766501903533936, -2.4466984272003174, -1.0543135404586792, -2.6953513622283936, 3.354191303253174, 1.2279704809188843, 0.8576018214225769, 2.5356650352478027, 1.5399432182312012, -0.6771895289421082, -0.4318438768386841, -3.2215981483459473, 2.3555614948272705, -0.5563396215438843, 2.7575228214263916, 0.1301426887512207, 0.45772242546081543, -3.468092203140259, 1.2081676721572876, -1.712770700454712, -2.785872220993042, 1.876692533493042, -0.18701040744781494, -1.7430716753005981, -1.319360375404358], [0.8866176009178162, -4.019325256347656, 3.9516069889068604, -8.2894926071167, 5.759792804718018, 0.4117705225944519, -3.537466287612915, 1.9607492685317993, 2.1927223205566406, 4.43883752822876, 1.2227283716201782, 3.5194897651672363, -0.4621308743953705, -1.9152904748916626, 0.664936900138855, 1.8192304372787476, -6.083913803100586, -3.1507256031036377, -3.73628830909729, -1.1826003789901733, -1.5232669115066528, 0.9151248931884766, 1.9587143659591675, -3.9953975677490234, 3.2630443572998047, -5.72948694229126, 2.1241695880889893, 0.7442302107810974, 0.508362352848053, 3.293301820755005, 1.7837855815887451, -3.097987174987793, -3.934952974319458, -3.095797061920166, 1.311540126800537, -4.986978054046631, 7.636009216308594, -1.5509223937988281, 1.8372856378555298, 3.058755397796631, 3.4518280029296875, -0.7970892190933228, 2.1288390159606934, -2.4483160972595215, 0.4481455385684967, -1.552618384361267, -0.3629307150840759, 5.508355140686035, 5.276512145996094, 0.4303296208381653, -7.4451518058776855, 5.284445762634277, 5.706639766693115, 2.8510119915008545, -2.762319803237915, -0.534321665763855, 1.5824837684631348, 1.1763596534729004, -5.8499860763549805, 2.4110965728759766, 0.5923378467559814, 5.317495346069336, -0.1380867213010788, -7.490174293518066, 8.403692245483398, -3.7523751258850098, -3.1521060466766357, 1.4366713762283325, 1.344469666481018, 2.647946357727051], [1.1791306734085083, 0.9685747027397156, 2.7255454063415527, -2.4754602909088135, 1.659361720085144, 0.15870393812656403, -0.4728788137435913, 0.7900497317314148, 3.4058046340942383, 3.863664150238037, 2.078153610229492, -0.3759755790233612, -0.9931813478469849, 3.218081474304199, -2.2919058799743652, 2.87488055229187, -6.481629371643066, -2.2362616062164307, 4.11142635345459, -1.2616877555847168, -1.4025108814239502, 0.7940901517868042, 1.750735878944397, -0.2439998835325241, -1.0892791748046875, -5.887622833251953, 1.9742629528045654, -1.8468217849731445, 4.96140718460083, 2.3210673332214355, 3.1705880165100098, -3.3981034755706787, -3.103726387023926, -2.5194015502929688, 0.711584210395813, -1.962932825088501, 4.195283889770508, -0.6530212163925171, 1.3723249435424805, 2.092668294906616, 5.868189334869385, 2.36844539642334, -0.4049021601676941, -0.2153928279876709, -1.809086799621582, -2.6780285835266113, 2.4797463417053223, 1.6680546998977661, 4.648196220397949, -1.9653401374816895, -2.0020065307617188, -2.5428779125213623, 2.8694307804107666, 1.3470042943954468, -0.7741242051124573, 0.1639205366373062, 3.2036099433898926, 3.1920833587646484, -6.319418907165527, 4.82480525970459, 2.753748655319214, 1.7630399465560913, -4.4434380531311035, -4.537118911743164, 7.582855224609375, 2.052873134613037, -0.3656471371650696, -4.146857261657715, 1.8581280708312988, -6.177602291107178], [-0.06891092658042908, 4.972733974456787, -2.378760576248169, 5.659045219421387, -1.5083198547363281, 1.126702070236206, 2.6754324436187744, -1.6205079555511475, -1.2753642797470093, -2.460846424102783, 2.2460548877716064, -2.594054698944092, 0.4416945278644562, 3.438689708709717, -0.5488425493240356, -2.212526321411133, 4.324564456939697, 1.1828080415725708, 1.6694923639297485, 0.8508102893829346, -1.8303792476654053, -2.7712924480438232, 0.10932193696498871, 2.4517195224761963, -2.1556854248046875, 3.2010397911071777, 0.615920901298523, -2.1628496646881104, -0.7967113852500916, -0.582149863243103, -3.0534675121307373, 0.15965379774570465, 3.576162338256836, 4.8885016441345215, -2.6336071491241455, 3.6855452060699463, -5.409167289733887, 0.7630703449249268, 0.39133399724960327, -1.6091487407684326, -2.9815139770507812, -0.8238458037376404, -1.9011974334716797, 0.6854020357131958, -0.172178715467453, -0.05010375380516052, -2.5209903717041016, -2.858675003051758, -3.9796087741851807, 1.8735203742980957, 6.781426906585693, -4.087010383605957, -2.0226008892059326, -0.5211565494537354, 1.745915412902832, -0.6118078231811523, -1.0942530632019043, 0.756363034248352, 2.3433268070220947, 2.847520112991333, -0.8860551118850708, -2.9370124340057373, -1.0812560319900513, 5.821180820465088, -4.390923976898193, 1.1491776704788208, 3.830988645553589, -2.721470594406128, -3.3006327152252197, -2.6652684211730957], [1.635538935661316, -0.8036506772041321, -1.4910393953323364, -0.4673309624195099, -3.3939430713653564, -1.6698023080825806, -0.0008193165995180607, 0.5660349130630493, 3.7862837314605713, 2.273268222808838, -1.1295088529586792, -0.4836220145225525, -1.2958637475967407, 3.810610055923462, -1.3268948793411255, 1.7352802753448486, 0.5360443592071533, 1.8369351625442505, 6.9233503341674805, -2.6742725372314453, 3.4032199382781982, 1.1731504201889038, -2.5948057174682617, 0.9303897619247437, -0.9343056678771973, -2.6260883808135986, 0.4282194972038269, -0.43432745337486267, 3.871778964996338, -3.2801902294158936, 4.314851760864258, 2.274683713912964, -1.3868826627731323, -3.175407886505127, 4.9468231201171875, 1.1444743871688843, -0.19143317639827728, 0.6533264517784119, 0.4415135681629181, -3.0574471950531006, 1.997298240661621, 2.4704790115356445, -2.110797882080078, 0.9214054346084595, 3.8569443225860596, -3.0179500579833984, 4.678099155426025, -1.8556536436080933, 0.4385422170162201, -1.0513488054275513, -2.314099073410034, -2.9449989795684814, -0.4874369502067566, -0.7041891813278198, -1.3823573589324951, -2.0609023571014404, -0.2892424464225769, 1.5359042882919312, -3.571315288543701, -1.8401228189468384, 4.861214637756348, 1.0174369812011719, -3.580962896347046, -1.0144426822662354, -0.7314481735229492, -1.055608868598938, 0.08186987042427063, -1.7492045164108276, 4.967384338378906, -5.907593727111816], [0.7661105394363403, 0.729478120803833, 0.6837244629859924, 1.6610116958618164, -2.58606219291687, 0.4096945524215698, -0.25302594900131226, 0.009811733849346638, 0.6091486811637878, 0.4622642397880554, -1.8553882837295532, -2.5645344257354736, 0.3941740393638611, 1.8951225280761719, -1.3315715789794922, 0.4209723174571991, -0.30546215176582336, 0.3025825321674347, 4.500169277191162, 0.36228686571121216, 1.0536322593688965, 2.3402390480041504, -0.19011716544628143, 2.1079273223876953, -1.9709947109222412, 0.7262579798698425, -2.137223720550537, -1.1671664714813232, 2.6187925338745117, -1.243282437324524, 2.277806282043457, 0.005852593574672937, -1.3501791954040527, -1.4757909774780273, 1.1620814800262451, 0.3571292757987976, -0.21137887239456177, 0.6086324453353882, -1.281639814376831, 0.2071538269519806, 1.204613208770752, 2.168121576309204, -0.7394932508468628, 1.2029439210891724, -2.184340000152588, -1.1604756116867065, 3.368217945098877, -2.3652327060699463, 0.2627094089984894, -1.9015390872955322, 1.8650320768356323, -2.0782530307769775, -1.565690279006958, -1.513426423072815, 0.868445873260498, 2.7275516986846924, 1.9721342325210571, 0.9302664995193481, 1.0832282304763794, -1.6170111894607544, 2.283109426498413, -1.9537094831466675, -1.4506653547286987, -0.025261690840125084, -0.7080658674240112, 4.061885833740234, 0.24979542195796967, -2.4106225967407227, 2.1693060398101807, -3.7069931030273438], [-0.015021059662103653, 7.198525905609131, 2.840801954269409, 6.982691764831543, 4.375701427459717, 2.735013484954834, 3.4108831882476807, -2.3973631858825684, -0.4436366558074951, 0.13833029568195343, 7.883054256439209, -3.005155563354492, 0.43614721298217773, 5.041005611419678, -3.877634048461914, -1.9677664041519165, -5.617648601531982, -2.1583709716796875, 2.812415838241577, 4.314675807952881, -8.708246231079102, -5.898281097412109, 4.534806728363037, 1.5108646154403687, -3.0240631103515625, -0.5724877715110779, 2.074876070022583, -5.569046974182129, 1.8490595817565918, 3.6105551719665527, -3.515143394470215, -6.474340438842773, 1.113669753074646, 3.3766515254974365, -8.164117813110352, -0.26725396513938904, -3.3480474948883057, -0.09192132204771042, -0.1677512526512146, 4.373283386230469, 3.5031979084014893, -1.1274845600128174, -0.37272223830223083, 1.438014268875122, -8.142870903015137, -1.8175495862960815, -3.246462106704712, 1.1467101573944092, -2.0328001976013184, 0.05235990881919861, 7.223628044128418, -4.319941520690918, 1.03253173828125, 2.2721104621887207, 0.34241631627082825, -1.9886707067489624, 3.5621237754821777, 3.2372169494628906, -1.014715552330017, 9.494396209716797, -0.8808349370956421, -2.7050564289093018, -3.669602632522583, 2.6810972690582275, 5.0687127113342285, 6.762546539306641, 6.724232196807861, -10.139056205749512, -6.859371662139893, -6.903224945068359], [-2.823734998703003, 0.9623149633407593, -0.03600511699914932, 1.903177261352539, 1.6697849035263062, -0.9286695718765259, 5.515444278717041, -2.5263733863830566, 0.6937945485115051, 1.2755836248397827, 10.1882905960083, 1.5463695526123047, -3.3359076976776123, 1.4541656970977783, -3.9232470989227295, 1.4768717288970947, -6.437182903289795, -1.6232264041900635, -0.6183313131332397, 1.3165533542633057, -5.163858413696289, -7.358717441558838, 2.080110549926758, -2.957982301712036, 1.2363835573196411, -7.082681655883789, 4.148327827453613, -0.40940698981285095, 1.727264642715454, 3.4355340003967285, -1.5314675569534302, -2.8048269748687744, 3.1964643001556396, -0.9635048508644104, -7.046957492828369, -0.5083222985267639, -1.8768818378448486, -0.9482370615005493, -1.3925244808197021, 2.2803921699523926, 5.840602874755859, -1.3590691089630127, 2.313800096511841, 1.7844935655593872, -2.004493236541748, 0.5936669707298279, -3.155233144760132, 4.163564205169678, -1.4411176443099976, -1.310288906097412, -2.5043702125549316, -2.0302248001098633, 2.6638736724853516, 4.328166484832764, -2.8050413131713867, -9.176311492919922, -2.7020442485809326, 0.7847795486450195, -5.310149669647217, 7.258847236633301, -2.8148365020751953, 1.1588224172592163, -4.85183048248291, -0.2756848633289337, 7.058894157409668, 0.8810557723045349, 4.047967910766602, -3.900174617767334, -5.818360805511475, -4.169764995574951], [-1.1854920387268066, 1.3888826370239258, -2.365885019302368, 1.1359559297561646, -4.724693775177002, 1.5680655241012573, 1.9423003196716309, -1.2043050527572632, -1.7879735231399536, -1.3209116458892822, 1.5063047409057617, -2.852057456970215, -0.5531258583068848, 1.7539149522781372, -0.41284844279289246, -0.4224385619163513, 5.40845251083374, -0.08314786106348038, -0.5203580856323242, -0.5737951397895813, 0.4156668186187744, 0.43705108761787415, 0.11915907263755798, 1.1981849670410156, -0.22869323194026947, 0.7908796072006226, -0.9466197490692139, 0.26138848066329956, -0.15532836318016052, 0.40217316150665283, -0.9525051116943359, 1.224702000617981, 3.8946852684020996, 3.1147003173828125, -1.8547046184539795, 3.35382342338562, -2.2564516067504883, 0.32070785760879517, -1.3210304975509644, -1.2561391592025757, -2.862149715423584, 0.05521361157298088, 0.37262454628944397, 0.34589290618896484, 0.7582220435142517, 1.131400227546692, -1.6157056093215942, -2.9211440086364746, -2.4088640213012695, 0.7823628187179565, 3.398204803466797, -1.9794024229049683, -0.9365447163581848, -0.44690844416618347, 1.8226860761642456, 1.4669348001480103, -3.2437584400177, 0.2725519835948944, 2.3348169326782227, 0.9252194166183472, -1.9599415063858032, -2.2820067405700684, -1.9161843061447144, 3.2441864013671875, -3.7325963973999023, 0.5791767835617065, 0.814227283000946, 1.422881841659546, -1.9586409330368042, -0.722970724105835], [2.0735034942626953, 5.504220008850098, 3.9207863807678223, 6.069407939910889, 4.828070640563965, 1.3724699020385742, -1.4533555507659912, 0.2329152673482895, -0.635253369808197, -1.4000334739685059, -2.686882495880127, -2.3345048427581787, 3.401599407196045, 1.046619176864624, -0.46641337871551514, -2.1495437622070312, -4.576506614685059, -0.3276839256286621, 3.6674609184265137, 3.9508485794067383, -3.65696120262146, 0.3515419363975525, 2.2837584018707275, 3.611051082611084, -4.566092014312744, 5.390988349914551, -1.4686353206634521, -4.4221906661987305, 0.5922737121582031, 0.3145785927772522, -1.6045458316802979, -4.2239813804626465, -3.793156147003174, 1.431886076927185, -1.5771876573562622, -1.739719033241272, -1.0074106454849243, 0.5694660544395447, 0.9975167512893677, 3.339324474334717, 1.235556960105896, 0.8183637261390686, -2.3300585746765137, 0.5351374745368958, -8.007660865783691, -1.3170355558395386, 1.1779168844223022, -0.8843823075294495, 0.9923135042190552, -1.0587118864059448, 7.344965934753418, -2.3616175651550293, -2.225653648376465, -1.8710681200027466, 2.3175642490386963, 5.085440635681152, 8.012149810791016, 1.1697566509246826, 3.091958999633789, 1.5435377359390259, 1.7797045707702637, -3.3177342414855957, 2.4619643688201904, 1.303066611289978, 1.2157199382781982, 7.491261005401611, 2.231295347213745, -7.102535247802734, -0.702482283115387, -2.593217134475708], [-1.587887167930603, -1.690648078918457, -0.6360161900520325, -4.023159027099609, -1.9474822282791138, 4.5340352058410645, -0.4843781888484955, -0.8993930816650391, -4.011748313903809, -0.3881586194038391, 3.1525492668151855, -1.9423644542694092, 0.13901066780090332, -0.7639802694320679, 1.4074678421020508, -2.336183786392212, 7.530351161956787, -2.442974090576172, -8.213815689086914, 0.22881373763084412, -1.9185867309570312, 0.6734278202056885, 2.2110891342163086, -1.6171914339065552, 3.647644519805908, 0.9611863493919373, -1.4924052953720093, 1.2599165439605713, -3.9082798957824707, 3.0208308696746826, -3.070338010787964, -0.14050719141960144, 4.66989803314209, 4.936549663543701, -3.9222571849823, 1.4132927656173706, 0.741904616355896, -0.5391198992729187, -1.7744742631912231, 0.8986015319824219, -6.188605308532715, -3.328798294067383, 3.493889808654785, -1.8498823642730713, 0.852487325668335, 1.6346880197525024, -5.896661758422852, -0.3635101020336151, -2.9342339038848877, 3.9377102851867676, 1.3637877702713013, 4.911042213439941, 2.6536672115325928, 1.5052610635757446, 0.8879681825637817, 2.843358278274536, -5.212279796600342, -0.0626116544008255, 3.8467228412628174, 1.4080076217651367, -4.931908130645752, -0.35792240500450134, 0.07836262136697769, 1.5257459878921509, -3.0359559059143066, -3.088162422180176, -0.4277227222919464, 5.3405585289001465, -4.751945495605469, 6.2061004638671875], [0.8638356328010559, 3.1010611057281494, 0.2621236741542816, 1.6071659326553345, 3.066371440887451, -1.3289475440979004, -0.49558788537979126, 0.9150546193122864, 0.5966696739196777, -1.29422128200531, -2.319774627685547, 1.1358380317687988, 1.0844498872756958, -0.5432486534118652, 1.192921757698059, 0.4836573004722595, -2.2687904834747314, 0.4011761248111725, 0.7613517642021179, -0.40394365787506104, 0.009838410653173923, -0.023147419095039368, -0.03426795452833176, 1.2145309448242188, -2.220228672027588, 1.4499551057815552, 1.713263750076294, -0.7921499609947205, -0.12335983663797379, 0.3803500533103943, -1.1052334308624268, -0.953723669052124, -1.6901739835739136, 1.3785923719406128, 1.1658058166503906, -0.15649595856666565, 0.2656034827232361, -0.07148613780736923, 3.1141068935394287, -0.21903930604457855, 0.608124852180481, 0.8235273361206055, -2.378801107406616, -0.8259390592575073, -0.37789303064346313, 0.4428562521934509, 0.003364932956174016, 0.2438574582338333, 2.6125011444091797, -0.17643681168556213, 2.0740954875946045, -2.6329851150512695, -1.0168685913085938, -0.9991594552993774, 1.5667487382888794, 1.3402875661849976, 3.239431142807007, -0.05237499624490738, -1.0340646505355835, 1.8495190143585205, 0.400444358587265, 0.013189212419092655, 1.956416130065918, 0.3372405171394348, 1.0448060035705566, 0.35552430152893066, -0.6459746360778809, -1.1218763589859009, 0.6409891843795776, -0.30646026134490967], [-1.9942874908447266, -1.962684988975525, -1.741115927696228, -2.5889065265655518, -0.555495023727417, -2.678168535232544, 1.883893370628357, 0.03471890836954117, 0.6854792833328247, 0.0962134525179863, 1.6667869091033936, 3.1409811973571777, -2.3020401000976562, -2.374161958694458, 0.11971060186624527, 2.899146318435669, -2.034025192260742, -0.17284290492534637, -2.4542925357818604, -2.1730525493621826, 1.6059976816177368, -1.2611006498336792, -0.9012402296066284, -2.4436306953430176, 1.6675682067871094, -4.475265979766846, 2.5640311241149902, 2.979104995727539, 0.18762493133544922, 1.2555862665176392, 0.37911927700042725, 1.2770289182662964, 1.3082451820373535, -1.5359981060028076, -0.034781645983457565, 0.03909151628613472, 1.272189974784851, -0.8098590970039368, 0.5757669806480408, -0.9227787852287292, 2.2873342037200928, 0.3255058228969574, 1.0399904251098633, -0.17985817790031433, 3.168750286102295, 2.2903218269348145, -0.7144031524658203, 2.248978614807129, 1.94278085231781, -1.1839710474014282, -4.882620811462402, -0.4687536954879761, 0.9773671627044678, 1.2776403427124023, -0.8924672603607178, -3.9336585998535156, -2.8895668983459473, -1.3508614301681519, -3.7171902656555176, 1.0896077156066895, -2.041304588317871, 2.272773265838623, -0.6586695313453674, -1.7123030424118042, 2.7948734760284424, -3.0706710815429688, -1.9303585290908813, 3.4562084674835205, -0.11013171821832657, 1.1835089921951294], [-1.304727554321289, -0.32381120324134827, 2.3435254096984863, -0.07390407472848892, 3.7098793983459473, 1.7842953205108643, -0.19969604909420013, -0.6474542617797852, -2.883415937423706, -0.8653315901756287, 2.1648457050323486, 0.5604207515716553, 0.9152392745018005, -3.070040225982666, 0.3919740319252014, -1.6954081058502197, -1.9802520275115967, -2.005666971206665, -5.745838165283203, 3.0432815551757812, -3.942030429840088, -1.6109949350357056, 2.711207628250122, -1.5336626768112183, 1.4680718183517456, 1.258060097694397, -0.5527098774909973, 0.009523005224764347, -2.942500352859497, 2.94911789894104, -3.1975436210632324, -2.70332932472229, 0.5833973288536072, 1.427804946899414, -4.852428436279297, -2.124897003173828, 0.6446135640144348, -0.6631654500961304, -1.1867432594299316, 3.632601022720337, -0.5922779440879822, -2.5307257175445557, 2.618943691253662, -0.4702259600162506, -4.269991874694824, 1.7755861282348633, -3.7373058795928955, 2.5929324626922607, -0.6998277902603149, 0.7029644250869751, 0.7020080089569092, 3.8385720252990723, 1.3325624465942383, 1.3925167322158813, -0.012640492059290409, 0.6824740767478943, 0.4670715630054474, -0.9790797233581543, 2.5160322189331055, 1.532317042350769, -3.6955015659332275, -0.3487502634525299, 2.4832844734191895, -0.21817654371261597, 2.0921006202697754, 1.2815943956375122, 0.5265334844589233, 0.4319559335708618, -4.473150730133057, 4.674670696258545], [-1.76322340965271, 3.3009843826293945, 0.6560431718826294, 7.868070125579834, -1.4758799076080322, 0.34527087211608887, 2.577119827270508, -1.84689462184906, -3.5967400074005127, -3.9784622192382812, -0.8642080426216125, -2.9372990131378174, 1.3770794868469238, -1.582526445388794, -1.4458181858062744, -1.427687168121338, -1.3395017385482788, 0.48076876997947693, 1.0581923723220825, 4.437446117401123, -1.7966012954711914, -0.5224786400794983, 1.1363002061843872, 2.9061920642852783, -3.169020175933838, 5.178417205810547, -3.338163137435913, -1.055215835571289, -0.47619307041168213, -0.033897705376148224, -2.271807909011841, -0.9531499743461609, 1.1533676385879517, 0.9416995644569397, -4.951375961303711, 0.7725591063499451, -4.4183502197265625, 0.7397599816322327, -3.329216957092285, 2.3447067737579346, 0.31782659888267517, 0.6963790059089661, 0.40111181139945984, 2.698185443878174, -7.665221214294434, 2.824768304824829, 0.2467130869626999, -2.2991080284118652, -2.569819211959839, -2.943319320678711, 6.62053918838501, -2.570500612258911, -4.594223976135254, -2.057145833969116, 2.7658298015594482, 2.8211865425109863, 2.4084489345550537, -1.700377345085144, 6.642193794250488, -1.4416347742080688, -2.3897247314453125, -5.652721881866455, 2.029940128326416, 4.091501712799072, -2.3074240684509277, 8.724864959716797, 2.34907603263855, -3.084418535232544, -3.231785535812378, -0.6148001551628113], [-0.8458364009857178, -5.09739351272583, 1.821738362312317, -8.016453742980957, 0.6198184490203857, 1.9568760395050049, -1.9537708759307861, 0.6098288893699646, -0.2948087155818939, 3.1928608417510986, 2.5038726329803467, 1.2239454984664917, -1.1335173845291138, -1.8602584600448608, 0.38736727833747864, 0.9686166644096375, -0.34270572662353516, -3.0945212841033936, -5.780869007110596, -1.036470651626587, -0.712713897228241, 1.4949281215667725, 1.7604949474334717, -3.802049160003662, 4.4592604637146, -4.650620460510254, -0.052729249000549316, 2.0705795288085938, -0.6310134530067444, 3.122218608856201, 1.0926252603530884, -1.0869154930114746, 0.02621089480817318, -1.4161204099655151, -0.7495166063308716, -2.456645965576172, 5.495978832244873, -1.2367398738861084, -1.0796418190002441, 2.167295455932617, 0.19235575199127197, -1.4685333967208862, 3.9139623641967773, -1.747991919517517, 1.1195014715194702, 0.058389607816934586, -1.833458423614502, 3.1637067794799805, 1.8251219987869263, 1.0258452892303467, -5.983705997467041, 5.9658966064453125, 4.822971343994141, 2.6029484272003174, -1.912736177444458, 0.2832790017127991, -2.726208448410034, 0.39499449729919434, -2.153106212615967, 0.7609782218933105, -1.8702677488327026, 3.2869954109191895, -1.0825828313827515, -4.8250627517700195, 4.027860164642334, -3.5648152828216553, -2.721797466278076, 4.324661731719971, -0.6082729697227478, 4.14596700668335], [0.06766913086175919, 6.163205146789551, -1.9476333856582642, 9.38256549835205, -0.4922432005405426, -1.0215648412704468, 3.3493897914886475, -1.6170992851257324, -0.7308805584907532, -3.6636765003204346, 0.23139792680740356, -1.9146413803100586, 0.9013379216194153, 2.472949981689453, -1.0966614484786987, -1.7865554094314575, 0.6960622072219849, 2.559157133102417, 4.6846604347229, 2.087158203125, -1.356481909751892, -3.3655595779418945, -0.73341304063797, 3.6272733211517334, -4.270070552825928, 4.636952877044678, 0.3752414584159851, -2.6322624683380127, 0.05219753086566925, -2.058091402053833, -2.6769895553588867, 0.2001262605190277, 1.7582416534423828, 2.9283297061920166, -2.1445529460906982, 2.9976818561553955, -6.925146579742432, 1.1586835384368896, 0.36399132013320923, -1.4157131910324097, -0.5529792308807373, 0.439537912607193, -3.201022148132324, 2.0748934745788574, -2.3587801456451416, 0.33737874031066895, -0.16796875, -2.9393205642700195, -3.473375082015991, -0.4491957426071167, 7.522119998931885, -6.170347690582275, -4.492170333862305, -1.7437622547149658, 1.9488955736160278, -1.3383992910385132, 1.7450522184371948, -0.18533320724964142, 2.798231601715088, 1.2409523725509644, 0.32481423020362854, -4.048011302947998, 0.34030774235725403, 6.234874248504639, -3.939683198928833, 4.259518623352051, 4.395928859710693, -5.241268157958984, -2.1544930934906006, -4.343218803405762], [2.2781834602355957, -1.466651201248169, -0.9953468441963196, -0.3909277319908142, 0.9743928909301758, -0.49375593662261963, -1.4213275909423828, 0.4618556499481201, 2.4243249893188477, 1.5583083629608154, -0.23148176074028015, 1.1945334672927856, 0.09110528230667114, 2.4181265830993652, 0.5056151747703552, -1.8053765296936035, 2.602236270904541, 2.2518908977508545, 2.653245687484741, -0.8285169005393982, 0.857498049736023, -1.598994493484497, -2.224442481994629, -0.46039050817489624, 1.0427184104919434, 0.6173914670944214, 0.8445039391517639, -0.9720712900161743, -0.5607799887657166, -3.706782341003418, 0.9823822975158691, 1.895758032798767, -0.8983662128448486, -0.9782438278198242, 3.7494070529937744, 0.06986182183027267, -1.556289553642273, 0.6308262348175049, 1.1141709089279175, -2.7915358543395996, -1.7017749547958374, -1.3731200695037842, -1.6709941625595093, -0.21968193352222443, 4.6016459465026855, -3.5025012493133545, 0.9951369762420654, -0.015536079183220863, -2.5666654109954834, 2.615609884262085, -2.05975604057312, 1.7269245386123657, 0.6174492835998535, 0.39856353402137756, -2.856945276260376, -3.9593162536621094, -0.9447012543678284, 0.7072840929031372, -1.4203203916549683, -2.382669687271118, 3.8013641834259033, 2.1812331676483154, -0.21217811107635498, 0.39264264702796936, -2.7730422019958496, -4.72139835357666, 1.635377049446106, -1.1374242305755615, 2.391906499862671, -0.8546047210693359], [-1.791664481163025, 0.5991182327270508, -0.6799169778823853, -0.08635126054286957, -1.0767736434936523, 0.10243246704339981, 1.084539532661438, -0.3449632227420807, -2.1395742893218994, -1.8289796113967896, -0.01723291724920273, -0.16607941687107086, -0.14253868162631989, -2.2325384616851807, 0.6335248947143555, 0.7239758968353271, 0.6866887807846069, -0.8970714807510376, -3.3761305809020996, -0.03769274801015854, 0.14178569614887238, 0.6382163763046265, 0.7602748274803162, -0.0479150153696537, 0.03644056245684624, 0.3677017092704773, -0.21374176442623138, 1.4704803228378296, -1.0977827310562134, 2.0113751888275146, -1.5624666213989258, -0.013322369195520878, 1.8364731073379517, 1.7795330286026, -2.1530261039733887, 0.8398913145065308, 0.23615339398384094, -0.3838494122028351, -0.3496098816394806, 0.6044623255729675, -0.7982067465782166, 0.1013675183057785, 1.0296382904052734, -0.2945631742477417, -0.9513264894485474, 2.985630750656128, -1.8638081550598145, -0.20094405114650726, 0.7513951063156128, -0.5765419006347656, 1.2105236053466797, -0.5532630681991577, -0.5931492447853088, -0.29265519976615906, 1.9414076805114746, 1.930614709854126, -1.0604139566421509, -1.1747761964797974, 1.559091567993164, 0.9862792491912842, -3.360384225845337, -1.1260550022125244, 0.9926310181617737, 0.8221048712730408, -0.13406850397586823, 0.8877246379852295, -1.3165730237960815, 2.6971724033355713, -1.9792747497558594, 2.427920341491699], [0.05583646520972252, 1.4760435819625854, -1.8614134788513184, -3.891160011291504, 1.6698132753372192, -1.3425747156143188, -1.2828234434127808, 1.895582914352417, 0.08637195825576782, -1.3955652713775635, -2.993999481201172, 2.6003406047821045, 0.25983133912086487, -2.2432610988616943, 3.6539180278778076, 1.9432693719863892, 1.8115519285202026, -0.4381067156791687, -4.24233341217041, -3.934866189956665, 2.5055580139160156, 1.9030587673187256, -0.5524457693099976, -0.3513467013835907, -0.026152033358812332, -0.3455987572669983, 3.1791815757751465, 2.0574874877929688, -1.7626620531082153, 2.050955057144165, -1.5501405000686646, 0.7742419838905334, 0.21714645624160767, 3.61679744720459, 2.88262939453125, 1.184004306793213, 3.0604629516601562, -0.8100350499153137, 5.059427738189697, -2.117189407348633, -2.260085344314575, 0.45265382528305054, -1.7883048057556152, -3.237809658050537, 4.878462314605713, 2.334348201751709, -2.4111363887786865, 0.4999777674674988, 4.712528705596924, 1.7121106386184692, -0.33194276690483093, -1.4739444255828857, 0.7309588193893433, -0.634081244468689, 2.551525115966797, 2.664246082305908, -0.335492342710495, -0.6399080753326416, -2.5205302238464355, 2.7996087074279785, -2.0095300674438477, 2.151846170425415, 2.3811073303222656, -0.5706896185874939, 0.2592829465866089, -5.2975640296936035, -4.242522239685059, 5.452216625213623, 0.9876298308372498, 3.895505428314209], [-1.778513789176941, 3.5768401622772217, -2.228088140487671, 3.882979154586792, -6.906271457672119, -0.5520155429840088, 5.093050479888916, -1.870618462562561, 0.8363427519798279, -0.08977263420820236, 3.5564017295837402, -3.8427693843841553, -2.533583641052246, 4.5908684730529785, -4.034085273742676, 3.1251540184020996, -0.7375183701515198, -0.22631393373012543, 6.216165065765381, -1.1807751655578613, 0.6047677993774414, -0.2320351004600525, 0.16167360544204712, 2.367727518081665, -3.055744171142578, -3.988492488861084, 0.3605044186115265, -0.7337414026260376, 5.509882926940918, 0.8483269810676575, 2.0582597255706787, 0.1023356169462204, 3.005397319793701, -0.14882269501686096, -2.409242868423462, 3.698009490966797, -2.537432909011841, 0.34799835085868835, -1.8783992528915405, -0.7137265205383301, 3.96506404876709, 3.7013611793518066, -0.5853541493415833, 2.6329457759857178, -1.6255240440368652, 0.5605947375297546, 2.201122522354126, -3.282750368118286, -0.06546653807163239, -3.668224573135376, 3.126507520675659, -8.369424819946289, -1.8580398559570312, -0.4015668034553528, 1.8760221004486084, -0.6670264005661011, -1.38615882396698, 1.760684847831726, -2.0491836071014404, 3.8182365894317627, -0.03040594980120659, -3.2328526973724365, -6.686138153076172, 2.0162551403045654, 1.0643333196640015, 5.6756134033203125, 1.9024916887283325, -3.412954092025757, -0.6336468458175659, -9.149348258972168], [-2.2308247089385986, -0.4077278673648834, -3.1059274673461914, -3.8581581115722656, -1.7043476104736328, -1.1540179252624512, 1.4894654750823975, 0.2550819516181946, -0.9689283967018127, -1.2557309865951538, 0.984420657157898, 2.0476491451263428, -1.7093077898025513, -2.340468645095825, 1.8783135414123535, 2.428013324737549, 2.660377264022827, -0.846993625164032, -5.541080474853516, -3.494521379470825, 2.0482795238494873, 0.2149505764245987, -0.3887726664543152, -1.7509422302246094, 1.7128201723098755, -2.8722293376922607, 2.7089099884033203, 3.4448516368865967, -1.4917389154434204, 2.670989513397217, -1.4772672653198242, 1.6419901847839355, 3.4076993465423584, 2.7154946327209473, -0.2753470838069916, 2.096184730529785, 1.591097116470337, -0.9547010660171509, 1.9331276416778564, -1.8670363426208496, -1.5569137334823608, -0.1364557296037674, 0.7250273823738098, -1.8219455480575562, 4.816268444061279, 3.598130226135254, -3.404046058654785, 0.7174328565597534, 2.186283588409424, 0.8848189115524292, -2.0282812118530273, -0.9489376544952393, 1.120263934135437, 0.7251131534576416, 1.4791460037231445, -0.4777926802635193, -4.358105659484863, -1.2340809106826782, -2.20072078704834, 2.7953901290893555, -4.196140289306641, 1.6294050216674805, -0.0075463661924004555, 0.05398229882121086, 0.2525607943534851, -4.864494323730469, -3.129822254180908, 6.523770332336426, -1.3686007261276245, 3.5759289264678955], [-0.3911226689815521, -3.563981771469116, -2.020975351333618, -1.3890854120254517, -4.023162841796875, -0.08570301532745361, -1.1191037893295288, 0.20475168526172638, -1.49250328540802, -1.1942381858825684, -3.5258049964904785, -0.22402174770832062, 0.2530239522457123, -2.1761269569396973, 1.6909191608428955, -0.9324232339859009, 6.038295745849609, 1.6458312273025513, -1.777698040008545, -0.6466453671455383, 3.817739486694336, 2.5598440170288086, -2.2835352420806885, -0.026532968506217003, 1.6551833152770996, 2.9339122772216797, -2.9532759189605713, 2.4478464126586914, -2.1780898571014404, -2.78598952293396, 0.7162002921104431, 3.990436553955078, 1.191564917564392, -0.5388230085372925, 2.5001232624053955, 1.229394555091858, -0.8783457279205322, 0.570933997631073, -1.8977247476577759, -2.3199288845062256, -4.156337261199951, -0.4215678870677948, 0.8017574548721313, 0.08448711037635803, 3.0841140747070312, 1.1362545490264893, 0.8493083715438843, -2.147660732269287, -2.3259644508361816, 0.6920353770256042, -1.2831568717956543, 3.01981258392334, -1.702361822128296, -1.5684797763824463, 0.09845205396413803, 1.439670443534851, -3.0923373699188232, -2.078758716583252, 4.134537220001221, -6.472151756286621, -0.3093107044696808, -0.5458796620368958, 2.2075648307800293, 1.1623516082763672, -6.057172775268555, -2.225292205810547, -1.7892837524414062, 4.966972827911377, 1.9138237237930298, 4.234447479248047], [-0.456949919462204, 3.336580514907837, -0.8526613712310791, 5.962475299835205, -0.6886852979660034, -3.789985179901123, 2.9519968032836914, -0.4099617302417755, 1.527061104774475, -1.7634694576263428, -1.4183199405670166, 0.3606444001197815, -0.5814010500907898, 0.3091026246547699, -1.7721500396728516, 2.2966861724853516, -5.2925825119018555, 1.9079835414886475, 6.069477081298828, 0.2945001721382141, 1.2782000303268433, -1.220184087753296, -1.3989509344100952, 2.2277116775512695, -4.013038635253906, -0.20695549249649048, 1.017671823501587, -0.5786536335945129, 3.1142215728759766, -1.4473663568496704, 1.0527756214141846, 0.3128378391265869, -1.2287050485610962, -2.2545149326324463, 0.3849826157093048, 0.7348417639732361, -2.7129287719726562, 0.5004451274871826, 0.4639433026313782, -0.6908074021339417, 4.836317539215088, 3.1659886837005615, -2.5364081859588623, 2.3702216148376465, -2.1516664028167725, 0.9670855402946472, 3.5881574153900146, -0.9605630040168762, 1.4833835363388062, -4.205410957336426, 1.6792584657669067, -6.648064613342285, -3.7905335426330566, -1.6656720638275146, 1.072482705116272, -1.8853881359100342, 3.084413766860962, -0.6472269296646118, -1.4546176195144653, 0.18210548162460327, 1.4941245317459106, -1.9625473022460938, -0.5139560699462891, 1.210593581199646, 1.423056721687317, 4.9101433753967285, 1.0048604011535645, -4.058234691619873, 1.51956307888031, -5.580265998840332], [2.4275882244110107, -2.5202906131744385, 5.405031204223633, -1.972029447555542, 3.707449436187744, 6.349668979644775, -3.3266353607177734, -0.824901282787323, -0.16049382090568542, 4.777625560760498, 5.20416784286499, -3.068329095840454, 1.5165961980819702, 4.091806411743164, -2.157606601715088, -5.070943832397461, 0.16026265919208527, -2.560645818710327, -0.2543374001979828, 4.455487251281738, -6.825087070465088, -1.4602208137512207, 3.694653272628784, -1.4842411279678345, 2.949660062789917, 0.3655953109264374, -2.821171760559082, -4.378429889678955, -0.5131873488426208, 0.5262666940689087, 0.139597550034523, -4.4119439125061035, -1.619603157043457, -0.6633303761482239, -3.3132340908050537, -3.7398922443389893, 1.1536420583724976, 0.11313354969024658, -3.367579460144043, 4.606662273406982, -1.1844050884246826, -3.9647750854492188, 3.091763973236084, -0.05570501461625099, -4.854702472686768, -5.9196858406066895, -1.4370524883270264, 1.8567153215408325, -4.5962042808532715, 3.5075862407684326, 0.09255968034267426, 7.935959339141846, 4.850277900695801, 3.070211887359619, -4.026305198669434, -0.035266097635030746, 1.0777474641799927, 3.456939935684204, 1.8077458143234253, 0.5347508192062378, 2.477337121963501, 0.7248384356498718, -2.2561721801757812, -1.9941916465759277, 1.715010404586792, 1.0833971500396729, 4.470850944519043, -5.581233024597168, -2.67081880569458, -0.42094099521636963], [2.560079574584961, -4.1170220375061035, 0.1430239975452423, 0.8303778767585754, 1.9285775423049927, 3.1294517517089844, -4.156219005584717, -0.23376578092575073, -2.172349452972412, -0.5594484210014343, -2.143296957015991, -0.24096077680587769, 3.0825765132904053, -0.4875803291797638, 2.6752424240112305, -7.86083984375, 9.258770942687988, 2.9231903553009033, -2.878143787384033, 3.5633385181427, -0.9555974006652832, -1.3480051755905151, -1.7281951904296875, -0.522483766078949, 3.4508543014526367, 8.957983016967773, -4.151699542999268, -1.1784274578094482, -6.809602737426758, -5.695013999938965, -2.4086573123931885, 2.7692956924438477, 0.2353077232837677, 1.289670467376709, 1.6535639762878418, -0.7466267943382263, -4.227067470550537, 1.3795145750045776, -2.235666275024414, -1.5985063314437866, -9.185222625732422, -5.7668375968933105, 0.597293496131897, -0.39859747886657715, 2.2929656505584717, -3.20084547996521, -1.955762505531311, -0.9833755493164062, -8.915409088134766, 6.187131404876709, 0.8032581806182861, 9.849518775939941, -0.3982430398464203, -0.3272715210914612, -3.238430976867676, -1.0965216159820557, -2.10196852684021, -1.411611557006836, 8.152913093566895, -8.721061706542969, 1.9532190561294556, 0.10759151726961136, 5.329432010650635, 3.5645334720611572, -10.181170463562012, -5.056943893432617, 3.128354549407959, 1.0827082395553589, -0.4145210087299347, 7.482572555541992], [0.5381329655647278, 3.022080183029175, 2.0454859733581543, 5.4222211837768555, 6.368701457977295, -0.5042890310287476, 0.22904784977436066, -0.4856724739074707, -0.8988332748413086, -1.9712737798690796, 0.34408435225486755, 1.3800898790359497, 1.924154281616211, -1.383346676826477, 0.1490514576435089, -2.57389235496521, -4.396337032318115, 0.6735062599182129, -0.2708488702774048, 3.8875949382781982, -4.058532238006592, -3.9426164627075195, 1.1117616891860962, 0.5385191440582275, -1.6443508863449097, 3.9066920280456543, 0.7849813103675842, -2.3131165504455566, -2.2344930171966553, -0.07479484379291534, -3.486985206604004, -2.582132339477539, -1.5349990129470825, 0.9654686450958252, -2.880361318588257, -1.9594192504882812, -2.7341997623443604, 0.17001080513000488, 0.8992468118667603, 2.1446776390075684, 0.7517842054367065, -1.8591831922531128, -1.0456931591033936, 0.5594898462295532, -4.511297702789307, 0.03770933300256729, -1.6505537033081055, 1.9920629262924194, -1.3459075689315796, 0.305476576089859, 3.17217755317688, 0.2854803204536438, -1.26602303981781, 0.13552404940128326, -0.2972252070903778, -1.7446824312210083, 4.061335563659668, -0.8238028883934021, 1.8289436101913452, 1.1257520914077759, -0.3600633442401886, -0.9608701467514038, 3.6431498527526855, 1.838013768196106, 1.0645867586135864, 2.513305425643921, 2.989276170730591, -4.5347747802734375, -2.8861591815948486, 0.9789651036262512], [-1.5942604541778564, -0.8816589713096619, 0.26554059982299805, 0.7210178971290588, -3.5872232913970947, 2.052128314971924, 2.791743755340576, -2.1458089351654053, -0.5635805130004883, 1.6838375329971313, 5.998654365539551, -2.7929179668426514, -1.8645185232162476, 2.812588691711426, -3.542368173599243, -0.06575888395309448, -0.018460216000676155, -1.4063911437988281, 1.1048821210861206, 1.4501011371612549, -2.5226027965545654, -1.768068552017212, 1.5781902074813843, -0.7057520747184753, 1.1103092432022095, -3.338557004928589, -1.1298930644989014, -0.696647584438324, 2.113351583480835, 1.1871377229690552, 0.8413137197494507, -1.0155155658721924, 2.7883071899414062, -1.0087347030639648, -4.483753204345703, 0.6078037619590759, -1.4003692865371704, -0.0032206475734710693, -4.108664512634277, 1.7111713886260986, 1.8315141201019287, -0.33775264024734497, 2.762661933898926, 1.9720134735107422, -2.577183723449707, -0.7270211577415466, -0.41927993297576904, -0.2597053050994873, -3.149143695831299, -0.9469789266586304, -0.11428527534008026, -0.03728890419006348, 1.2959388494491577, 1.918682336807251, -1.3834725618362427, -2.4256060123443604, -2.909113645553589, 1.3564441204071045, 0.0019208203302696347, 1.76505446434021, -0.9216130971908569, -1.324446201324463, -4.92597770690918, 0.3326048254966736, 1.363155722618103, 2.9976484775543213, 2.9024384021759033, -2.3680808544158936, -2.9418840408325195, -3.692814350128174]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.6024714708328247, 0.15217216312885284, -0.052321065217256546, -1.249622106552124, -0.43425506353378296, -0.9883988499641418, -0.7053906917572021, -0.08811341971158981, -1.4559544324874878, -0.867866575717926, -0.2043866664171219, 0.32575523853302, 2.798933982849121, -1.359233021736145, -0.05762934684753418, 0.4889642894268036, 0.7545438408851624, 0.8628664016723633, -0.21840538084506989, -0.2236139327287674, 0.5436845421791077, -1.2198169231414795, 0.15989980101585388, 1.2248139381408691, -0.5953030586242676, 0.9698208570480347, -0.34026890993118286, -0.6449416875839233, 0.15052424371242523, -0.3922019898891449, 0.08946572244167328, 0.6178564429283142, 0.22721165418624878, -1.532105803489685, 0.7578801512718201, 0.30465710163116455, 0.3157505393028259, 0.7836583852767944, -0.9051414728164673, -1.5995264053344727, 1.1173591613769531, 1.5754129886627197, -1.0168849229812622, -0.6378488540649414, -3.263096570968628, 1.3572301864624023, -0.575509786605835, -0.5951182246208191, 2.575059413909912, 0.10995521396398544, 1.0243929624557495, 0.32355761528015137, -0.38073641061782837, -1.0856928825378418, 1.4325488805770874, -0.2274131029844284, -0.1929309219121933, -0.06039012223482132, 1.049695611000061, -0.5097801685333252, -0.4655202031135559, 0.48990851640701294, 2.4114208221435547, 0.6561275720596313, -0.7812783122062683, -1.5944968461990356, -1.2193366289138794, -1.2819409370422363, -0.605411946773529, 1.4908647537231445, -0.42295655608177185, 0.12795962393283844, 0.8296756148338318, -0.43300050497055054, -0.2885030508041382, -0.8679003119468689, -1.30172598361969, -0.5897526144981384, 1.684854507446289, -0.7530861496925354, -1.0317939519882202, -0.435315877199173, 0.2624228298664093, 0.1014375239610672, 0.3899049460887909, 1.1122887134552002, 0.3860378861427307, -1.6068555116653442, -0.13005995750427246, 0.5357650518417358, -0.10961733758449554, 1.6050350666046143, -0.7291628122329712, -0.38741615414619446, 0.36838895082473755, 0.12537163496017456, -1.5051701068878174, 0.33811718225479126, -0.470210462808609, 1.5455570220947266, 0.5670846700668335, -0.18761497735977173, -0.7389857172966003, 2.227444648742676, 0.6856937408447266, 0.6837888956069946, 0.7619785666465759, 0.04302678629755974, -1.0762861967086792, 0.051019273698329926, 0.6536162495613098, 0.7784174084663391, -0.024325337260961533, -1.2304497957229614, 0.21426904201507568, -1.3443328142166138, 0.5804272890090942, -1.854526400566101, -0.3164030611515045, 0.6357357501983643, -0.48577186465263367, -0.8363283276557922, 0.32731935381889343, -0.7216921448707581, 0.21081341803073883, 0.6548902988433838, -1.5693795680999756, 1.3963897228240967, -0.8186811804771423, -0.9249936938285828, 0.000992140849120915, 0.14599739015102386, 2.0509371757507324, -1.6451094150543213, 0.22860755026340485, 0.6026695966720581, -1.1991863250732422, -2.1174583435058594, -2.080031156539917, 0.4514335095882416, 0.5236981511116028, 1.2854275703430176, 0.3445824384689331, 0.6550014615058899, -2.052258014678955, -1.5828914642333984, -0.3512890040874481, 1.962202787399292, 2.4166414737701416, -0.6815577745437622, -0.41612666845321655, -0.7083982825279236, 0.4312048554420471, -1.0594905614852905, -1.2272136211395264, 0.34428519010543823, 0.003450743155553937, -0.4661143124103546, -1.229994297027588, 0.812347412109375, -0.7622594237327576, -0.35762709379196167, -1.3737874031066895, 0.4922129213809967, 0.7309850454330444, -0.48334309458732605, 0.6438211798667908, 0.8624505400657654, 1.504902958869934, 0.2898227572441101, 0.22829878330230713, -1.0350662469863892, -1.74277925491333, -0.8553168177604675, -1.0268241167068481, 0.10179100185632706, -1.7704548835754395, -0.4544965624809265, -0.7373494505882263, 2.4550817012786865, -0.2271346002817154, -1.8192135095596313, -0.6156113147735596, -0.7393118739128113, -0.19467075169086456, -0.00013697256508748978, 0.9432396292686462, -0.12529616057872772, -1.5275338888168335, 0.30957409739494324, -1.4958778619766235, 0.6421400904655457, -0.4490562677383423, -1.4094425439834595, -1.67376708984375, -0.13847753405570984, 0.6010148525238037, 0.20169301331043243, 0.5843239426612854, 1.2076479196548462, -0.3020520806312561, 1.1744067668914795, -0.7480173707008362, -2.284501552581787, -0.1742849498987198, 1.0749725103378296, 0.38052448630332947, -1.7298269271850586, -1.4712984561920166, 0.6892832517623901, 1.4981013536453247, 0.11320488154888153, 0.688029408454895, -1.1742076873779297, -2.0118377208709717, 2.0334038734436035, -0.2528223693370819, -1.090401291847229, -0.516960620880127, 0.2416638433933258, 0.19554650783538818, -0.5689058899879456, -1.440504789352417, 0.029240848496556282, -1.037786602973938, -0.19785749912261963, 1.1244603395462036, 0.404450923204422, 0.9159460663795471, 0.5603212118148804, 0.5356990694999695, -0.35344550013542175, 1.1943522691726685, 0.21279539167881012, 0.5394499897956848, 0.43818992376327515, -2.4727072715759277, 1.6198663711547852, 0.0020921179093420506, -0.23188483715057373, 0.4370138943195343, -0.7309179306030273, 0.1984357088804245, 0.5587461590766907, 0.3251519203186035, -1.2736668586730957, 1.3642005920410156, -0.8292697072029114, 0.47676002979278564, -1.6916126012802124, 0.11749221384525299, 1.0700197219848633, -0.8690604567527771, -1.3218721151351929, -0.39533326029777527, 1.0060869455337524, -0.3112088441848755, -0.7550779581069946, 0.10771092772483826, -0.025041379034519196, 0.34362471103668213, -0.41278553009033203, -0.6803703904151917, 1.1084964275360107, 0.5213014483451843, -0.1890823096036911, 0.05782749876379967, 0.9619918465614319, 0.4735327959060669, 0.6992740035057068, 0.6040494441986084, 0.31893303990364075, 0.8102219104766846, 0.6491551995277405, -0.0591583177447319, 0.27421021461486816, 0.36288824677467346, -1.1790409088134766, -0.8018478155136108, 0.6127197742462158, -0.09531621634960175, 0.9633651375770569, -1.4051291942596436, 0.008278285153210163, -0.9789342284202576, 1.3338322639465332, 1.9423142671585083, 1.8383196592330933, -0.7806556820869446, 0.47908464074134827, -0.1067286804318428, 0.2371467649936676, 0.6666209697723389, 1.3282055854797363, 1.7020535469055176, 0.9114386439323425, 0.4600178301334381, 1.3757058382034302, 0.9927112460136414, -1.296156644821167], [0.8601850271224976, 0.20568369328975677, -0.2798599898815155, 0.27898934483528137, -0.44407176971435547, -0.9358401894569397, -1.0395393371582031, -1.0798155069351196, -0.5607089400291443, -0.5389447212219238, 1.2207701206207275, 0.40227147936820984, 1.0071669816970825, 0.5200112462043762, -0.23784276843070984, 0.3256429433822632, -0.5220944881439209, 1.2551730871200562, -0.4626840353012085, 1.5278151035308838, 0.8401552438735962, -0.7708509564399719, 0.30026447772979736, 0.9537788033485413, -0.29120540618896484, 1.315026879310608, 2.040876626968384, -0.9531570076942444, 1.2336208820343018, -1.519497275352478, -0.578839898109436, -0.36192557215690613, 0.9168997406959534, 1.5740619897842407, 0.8176670074462891, -0.8150253891944885, 1.354984164237976, -1.2938518524169922, 1.2340140342712402, 2.5479421615600586, -1.5489282608032227, -0.06464378535747528, -0.39161351323127747, -1.416486382484436, -1.0620752573013306, 1.438003420829773, -1.1812827587127686, -0.48480838537216187, -1.3669826984405518, -0.20555156469345093, -0.9500733017921448, -0.9022502303123474, 0.1289355754852295, 0.5790589451789856, 1.4819328784942627, -0.9438382983207703, 0.6523358225822449, 0.3778342008590698, 1.0168869495391846, 1.0189238786697388, 1.7545833587646484, 0.48621809482574463, -2.397423267364502, -2.2446982860565186, 0.43294164538383484, 0.5373196601867676, 0.9521604180335999, -0.16246594488620758, 0.983559250831604, -1.9880481958389282, -0.7529412508010864, -1.2682756185531616, 0.9934738874435425, -1.66791570186615, -0.15230968594551086, -1.1267672777175903, 1.5285810232162476, 1.4682594537734985, -0.1653345674276352, -0.4773867428302765, 0.22440414130687714, -1.2682644128799438, 0.9118569493293762, 0.27609777450561523, 1.3332799673080444, 0.12108524888753891, 0.9531975388526917, -0.05596700683236122, 1.4755014181137085, -1.3362445831298828, 0.25657472014427185, -0.8135324120521545, -0.7695290446281433, -0.14012020826339722, -0.8111698627471924, 0.2963086664676666, -0.6586673259735107, 1.3245437145233154, -0.545498788356781, 1.3522950410842896, -0.5731021165847778, 0.9965424537658691, -0.34421899914741516, -0.48745185136795044, -0.4105118215084076, 0.35055631399154663, -0.3189551532268524, 1.6624475717544556, 0.9110022783279419, -0.015767866745591164, 0.045074813067913055, -1.0668047666549683, 0.2987639009952545, -1.2611753940582275, 0.7280218601226807, -1.700927495956421, 0.24469108879566193, 0.34197643399238586, -0.20780235528945923, 0.17205142974853516, 0.8653978705406189, 0.33895382285118103, 0.7009208798408508, -0.17396420240402222, -0.8991086483001709, 0.505040168762207, -0.5556254982948303, 0.6313607692718506, -0.6256076097488403, -1.181164264678955, 0.08017949014902115, -0.2249114066362381, -0.7886171340942383, 0.5649703145027161, -0.10829884558916092, -1.1817045211791992, -0.5244978666305542, 0.6230685114860535, -0.7757425904273987, 1.2069231271743774, 0.1145327091217041, -0.046023063361644745, -0.4025757610797882, -2.8532919883728027, 0.46980783343315125, 1.1058443784713745, -0.14367914199829102, 0.20108413696289062, 1.0248266458511353, -0.40746375918388367, -0.26281601190567017, -0.007384529337286949, 1.0613106489181519, -1.3088324069976807, 2.0088376998901367, -0.6037983894348145, -0.2837621569633484, 0.8729838728904724, 2.049116611480713, 0.2005302608013153, 0.028302595019340515, 0.6686697006225586, -0.42253726720809937, -0.09521318972110748, 1.5352708101272583, 0.1438785344362259, 1.2017966508865356, -0.49569401144981384, -1.4982482194900513, -0.4852995276451111, -0.23298566043376923, 1.3949180841445923, 0.3331192135810852, -0.4733515977859497, 1.1113379001617432, 0.1985965520143509, 0.29259154200553894, 0.00608966825529933, 1.3825623989105225, 0.08845826983451843, 1.879105806350708, -1.137877345085144, -0.20969513058662415, 1.202872633934021, 0.874740481376648, 0.19239027798175812, 0.5231375098228455, -1.0315018892288208, -1.0668349266052246, -2.173171281814575, -0.2519656717777252, 0.932836651802063, 1.052135944366455, -1.9202477931976318, -2.258328914642334, 0.4922764301300049, -0.45853614807128906, -1.1050379276275635, -0.8457801342010498, -0.3050486147403717, -1.1329151391983032, 0.25957098603248596, 0.17173057794570923, -0.46353745460510254, 0.08873239904642105, -0.2099587619304657, 0.08195080608129501, 0.544948160648346, -0.23863017559051514, 1.1896531581878662, -0.44231247901916504, 1.3102866411209106, -0.8345475792884827, 0.31768956780433655, -1.0815088748931885, -0.7469362616539001, 0.21215353906154633, -0.5125657320022583, -1.1726717948913574, -0.18631526827812195, -0.2418973296880722, 1.1248984336853027, 1.2650024890899658, -0.42403003573417664, -1.273621678352356, -0.22352340817451477, 1.6900466680526733, 2.9992780685424805, -1.7228100299835205, 0.6458953619003296, -0.9505934119224548, -0.12109029293060303, -0.19818314909934998, -0.29992103576660156, -2.488332748413086, 0.2892170548439026, -0.7571038007736206, -2.0536630153656006, 0.7281721830368042, -0.38594886660575867, 2.4980807304382324, -0.6577230095863342, -0.7924382090568542, -0.15596792101860046, 0.9139934182167053, 1.7581311464309692, 1.674290418624878, -0.9847438335418701, 1.8682763576507568, -0.5661308765411377, -1.6828664541244507, 0.10605378448963165, -0.7701717615127563, -0.5051189064979553, 0.8004432320594788, -0.8601149916648865, 0.0904807597398758, -0.6479615569114685, -0.07481326162815094, 1.199081540107727, -0.13110558688640594, 0.13096760213375092, 0.47352585196495056, -0.4401632249355316, -0.7039638757705688, -0.2121923565864563, -1.2259857654571533, -0.2956256866455078, -1.7670165300369263, -0.22507168352603912, 0.39434269070625305, 0.36884182691574097, -0.2001124769449234, -1.1084074974060059, 1.60946524143219, -1.1717230081558228, 0.6429644823074341, 0.12860319018363953, -1.2720909118652344, -0.4880066514015198, 2.2966954708099365, -0.3720174729824066, -1.0578423738479614, 1.2073872089385986, -1.2312812805175781, 0.1855393499135971, 0.9280512928962708, 1.1011375188827515, 1.7996129989624023, -0.8282144665718079, 0.08489801734685898, 0.22708900272846222, -0.6601057648658752, -0.38540953397750854, -1.2032392024993896, 0.9964439272880554, 1.5918720960617065, -0.2517883777618408, 1.6352996826171875, -0.08540935814380646], [0.7250745892524719, -2.623965263366699, 1.5186190605163574, -0.7485287189483643, -0.24028362333774567, -0.9944754242897034, -1.1830826997756958, -0.21248646080493927, 1.174651861190796, -0.13313722610473633, 1.087959885597229, -0.6345618963241577, -0.8598902225494385, 0.20480485260486603, -1.2335697412490845, -1.6900579929351807, 0.16046744585037231, 0.00602705916389823, 0.2010146677494049, -0.5830100178718567, 0.6427831649780273, -0.8800278306007385, -1.2651004791259766, 0.02141411043703556, -1.046638011932373, -0.08684735745191574, 0.3009924590587616, -1.2085498571395874, 0.2564713954925537, 1.1032451391220093, 0.453754723072052, 0.3479144275188446, 1.9670413732528687, 0.1807015836238861, 0.2670103907585144, 0.5219805240631104, -0.23657172918319702, -0.6040443778038025, 0.6042901873588562, -0.556797981262207, 0.644090473651886, 1.1563079357147217, 0.9460110068321228, 0.7394858598709106, -0.2391355037689209, 0.5147121548652649, 0.904811680316925, -0.39434143900871277, -0.08524978160858154, 0.5307640433311462, 0.6824862957000732, -0.7498296499252319, 1.5553205013275146, 0.008251161314547062, 1.5355010032653809, -0.003242245176807046, -0.38897189497947693, 0.09835803508758545, -1.1751580238342285, 0.165300652384758, -1.7465466260910034, 0.0703928992152214, 0.12803563475608826, -0.7383334040641785, 2.0521891117095947, -0.4183070957660675, 0.0059916917234659195, 0.5810446739196777, 1.1263349056243896, 2.351630210876465, -0.41637805104255676, 0.25110140442848206, -1.5400607585906982, 0.03083239123225212, 2.4091975688934326, -1.6848934888839722, 0.014778745360672474, -1.623547911643982, 0.6543001532554626, 0.17446857690811157, -1.7348434925079346, 1.913216471672058, -0.5508900284767151, 0.21785955131053925, 0.8153290748596191, -1.9289368391036987, 0.9035123586654663, 0.086892269551754, 0.2381609082221985, 0.19020415842533112, 0.2144441455602646, -1.1382042169570923, -1.24863862991333, -1.6565358638763428, 0.49297839403152466, -0.8083553314208984, -1.591083288192749, -0.31073683500289917, -1.0977792739868164, -1.916167140007019, -0.9966250658035278, -0.8176683187484741, 0.027698246762156487, -0.7658863663673401, -1.863648533821106, 0.3542396128177643, -0.634026288986206, -1.1795108318328857, 1.407378911972046, -0.3643217384815216, 1.1323060989379883, 0.4472475051879883, -0.8413589596748352, 0.41316041350364685, -2.5598762035369873, 2.310014247894287, -0.8453322649002075, 0.04876342788338661, 3.3404085636138916, -1.3552910089492798, -0.6106038689613342, -1.5092462301254272, -1.2688785791397095, -0.7290022969245911, 1.5405397415161133, 0.768420398235321, -0.8919448852539062, -0.3966977894306183, 0.6351262331008911, 0.25355035066604614, 0.8145956993103027, -0.11085709184408188, 0.0018442345317453146, 0.32580721378326416, -1.0879255533218384, 0.8280760645866394, 1.5533684492111206, -0.4070872962474823, -0.9952086210250854, -0.16055123507976532, -2.156940221786499, 0.45357537269592285, 0.03151857852935791, 0.6934848427772522, -0.8400900959968567, 0.7476111054420471, 1.918186068534851, -1.3958853483200073, -1.192444920539856, 0.10820899158716202, -0.5142761468887329, 0.5304306149482727, -0.09267067164182663, -0.25971657037734985, 0.534810483455658, -1.6809697151184082, -1.0780489444732666, -1.0353167057037354, 0.7701342701911926, 0.9667240977287292, -2.537905216217041, -1.705061912536621, -0.3326030671596527, -1.8386067152023315, 0.5235762000083923, 0.34325268864631653, -0.5164355635643005, 0.4583090841770172, 0.21653510630130768, 0.12698164582252502, 0.14991863071918488, 1.4546736478805542, 0.2642771601676941, 0.06403829157352448, 1.13701331615448, 0.7120686769485474, 2.23358416557312, -0.5708491802215576, 1.4158765077590942, -1.334966778755188, -1.181817650794983, 0.9528136253356934, 0.46960023045539856, 2.2606358528137207, -1.1925276517868042, -1.8163412809371948, -0.06822464615106583, 0.5458320379257202, -0.5790723562240601, -0.4291220009326935, -0.6113547682762146, 1.0612834692001343, -1.7541569471359253, 1.2440543174743652, 0.44928082823753357, 0.0770229622721672, 0.634809672832489, -0.34343767166137695, 0.38221558928489685, -0.21849806606769562, -0.5306516289710999, -0.11250623315572739, 0.4193914532661438, 0.21568036079406738, -0.5391634106636047, 1.0108000040054321, -0.013491571880877018, -0.1276046484708786, -0.2120266556739807, 0.19516074657440186, 0.24887101352214813, 1.4862492084503174, 0.644068717956543, 1.0162534713745117, -0.08605160564184189, 0.2928882837295532, 0.9720962643623352, 0.8113565444946289, 0.11734548211097717, -0.3958984613418579, 0.18249069154262543, -1.1211514472961426, -0.8936141133308411, -0.829232931137085, -1.3197335004806519, 0.6621562838554382, 0.8881540894508362, -1.248019814491272, -0.7008965015411377, -0.7921403646469116, 1.2255548238754272, 0.5031999945640564, 1.658934473991394, -0.5117306113243103, 1.6098953485488892, 0.25044944882392883, -0.8059335947036743, 0.3241589367389679, 0.5165333151817322, -0.39793694019317627, 0.3387981951236725, 0.40948033332824707, -0.2999972701072693, -0.016370709985494614, -0.5869988203048706, 0.7777630090713501, 0.4944257438182831, 2.365023612976074, 1.211214303970337, 0.30600401759147644, 1.0992302894592285, -0.9799636602401733, -1.5272071361541748, -0.6149263978004456, 0.4046075642108917, 1.195052981376648, -1.4804171323776245, 0.3556048572063446, -0.000850471667945385, 0.010638321749866009, -0.28669649362564087, -0.8180872201919556, -0.6090834140777588, 0.289763867855072, -0.5721715092658997, -0.7069982886314392, -0.3392486870288849, -1.4113529920578003, -0.620611846446991, -0.45892444252967834, 0.8819087147712708, -0.21734853088855743, 2.2703919410705566, 0.15184259414672852, -1.7635116577148438, 1.9800459146499634, -3.147221565246582, -0.44813311100006104, -0.6017425060272217, 0.41939279437065125, 1.6347538232803345, 0.7679498195648193, 1.7738947868347168, -0.5160808563232422, 0.4534030556678772, 0.6544883251190186, -0.22103114426136017, -0.14327910542488098, -0.45063284039497375, 0.6452739238739014, 1.0788681507110596, -1.278855800628662, -0.3296576142311096, 1.676206350326538, 1.3139578104019165, 1.383297324180603, 0.10013668984174728, -0.5779746174812317, 0.10434374958276749, 0.2800854444503784], [1.2361729145050049, 0.9506175518035889, 1.1649296283721924, 1.4131009578704834, 0.6017175316810608, 0.8226119875907898, -0.36506038904190063, -1.1676596403121948, 0.5244255065917969, 2.280109167098999, -1.534455418586731, 0.2850828468799591, 1.837213397026062, -0.28841501474380493, -0.3318774700164795, 1.6912182569503784, -0.7937514781951904, -0.4218849241733551, 0.19433258473873138, -0.3271741569042206, -1.0335288047790527, -0.9857786297798157, -1.9009289741516113, 1.379917860031128, 0.9215644001960754, 0.8000501394271851, 0.05211851745843887, -0.2757546007633209, -1.2131913900375366, -0.0794748067855835, -0.28802961111068726, -0.28707465529441833, 1.2915153503417969, 0.8720484375953674, 0.09662160277366638, -0.4455811083316803, -1.6440173387527466, -0.7925112843513489, 1.0137230157852173, -0.4975248873233795, 0.07365170866250992, -0.1168835312128067, -0.0857560783624649, 0.487905889749527, -0.8174000382423401, 1.4704389572143555, -1.795269250869751, -0.2136017233133316, -0.12704290449619293, 1.0195503234863281, 0.3410627841949463, -0.06609352678060532, -1.436646819114685, 0.443632036447525, -0.5644488334655762, 0.7225137948989868, 1.2259550094604492, 0.12224750965833664, 0.7507190704345703, 0.4638530910015106, -0.5169642567634583, 0.4095778167247772, -1.1931613683700562, -1.2540005445480347, 1.0379505157470703, 1.4312304258346558, -0.5034210085868835, 0.6424033045768738, 0.4208410680294037, 1.2884395122528076, -1.7957843542099, -1.0225214958190918, -0.3177493214607239, -0.48773500323295593, 0.39062759280204773, 0.21091948449611664, 1.972833514213562, 0.9341841340065002, -2.078920364379883, 1.260654330253601, -0.1866954267024994, 0.08427746593952179, 0.5821331143379211, 0.31656304001808167, -0.7798696160316467, 0.05898674204945564, 0.23663198947906494, -0.948657214641571, -0.2825736701488495, -0.1643795371055603, -1.1534911394119263, -0.03916352614760399, -0.8787639141082764, 0.610523521900177, 0.4094031751155853, 0.44725605845451355, 1.543177843093872, 0.35421907901763916, 1.0536233186721802, 0.44195547699928284, 0.1868944764137268, 0.3561496138572693, -0.09188412874937057, -0.1651761531829834, 0.4859149754047394, 0.4211515486240387, 1.203701376914978, -0.619472086429596, 0.568122923374176, 1.5306396484375, 0.6303374171257019, -0.0041176266968250275, 0.2558523714542389, -0.03265692666172981, -0.07872562110424042, -0.6812139749526978, 1.044662594795227, 0.40567678213119507, -0.3301774561405182, -0.5078241229057312, 0.3979399502277374, 1.7064329385757446, 0.9201910495758057, 0.16508261859416962, 1.3741081953048706, -0.5400207042694092, 1.978173851966858, -0.581144392490387, -0.43713876605033875, -2.2338614463806152, 0.4351796507835388, 0.6447848081588745, -0.2830684781074524, -1.3069835901260376, -0.36370769143104553, -0.6677844524383545, -0.771386981010437, -0.9927104115486145, 1.0066156387329102, -1.584933876991272, -0.6915537714958191, 0.8074256777763367, 1.3928582668304443, -1.891160011291504, 0.804789125919342, -0.4790734648704529, -1.570937991142273, -0.09648429602384567, -1.92410409450531, -0.28582286834716797, 0.5682278871536255, -0.06760899722576141, 0.4653719365596771, -0.4371402859687805, -1.8518035411834717, 0.5194598436355591, -1.0664597749710083, -0.4846187233924866, -1.0629374980926514, -0.060208506882190704, -0.6454170346260071, 0.46320420503616333, 1.7585909366607666, 0.9493042826652527, -0.5416390895843506, 1.2950019836425781, -0.08248911052942276, 0.20624187588691711, -1.6066619157791138, 1.6246999502182007, 1.0510518550872803, -0.8917253017425537, 1.554638147354126, -1.0842455625534058, 0.29521501064300537, 0.7871761918067932, 0.2741227149963379, -0.4713166356086731, 0.3331637978553772, -0.6176999807357788, -0.4385360777378082, 0.12347740679979324, -0.34340915083885193, 0.16505514085292816, -0.684253454208374, 0.2639085650444031, 1.0571153163909912, -0.7200581431388855, 1.1802892684936523, 0.004298998974263668, 1.257103681564331, 1.4717211723327637, -1.3216140270233154, -1.0174040794372559, -0.1313997358083725, -0.39056292176246643, 0.829463005065918, 0.597088098526001, 0.019541170448064804, -0.7606574892997742, 0.6882472634315491, -1.663893222808838, -1.3905484676361084, -0.49293792247772217, 0.48572108149528503, -0.12628412246704102, 1.056076169013977, 0.16259463131427765, -0.240874245762825, 2.267915725708008, 0.6137881875038147, -2.6938986778259277, 1.0691876411437988, -0.5816088914871216, -0.04888121783733368, -0.7876706123352051, 1.4226716756820679, -1.9355804920196533, 0.9867111444473267, 1.2471264600753784, 1.718121886253357, 1.6445186138153076, 0.7243739366531372, -1.5242080688476562, 1.4093012809753418, -0.6641944050788879, -2.0213050842285156, 0.09714443981647491, 1.528108835220337, 0.042196180671453476, 0.5102795362472534, 0.4576570391654968, -0.13552677631378174, 1.5468288660049438, -1.099759817123413, 0.1036895141005516, 1.002620816230774, -0.0270240381360054, 0.8294357657432556, -0.49363189935684204, 0.6567716598510742, 0.04096101596951485, -0.9666383266448975, -1.1363312005996704, -0.22031338512897491, -2.9421772956848145, 0.8697664737701416, -0.5576610565185547, 0.36118263006210327, -0.8036584258079529, 0.7551615238189697, 0.7160612344741821, 0.7528737187385559, -1.6747773885726929, -0.2572607398033142, -0.06205393746495247, 1.428135871887207, 0.4452989399433136, -1.2909547090530396, 0.626429557800293, -1.3694781064987183, 0.34040597081184387, -1.2989076375961304, 1.2348315715789795, -2.060945510864258, 0.1763841211795807, 1.0779588222503662, 0.5776826739311218, 0.6498889923095703, -0.5630055069923401, 1.0102338790893555, 1.1613413095474243, 0.3904896080493927, -0.8215264081954956, 0.608273446559906, 0.6322399377822876, -1.3140078783035278, -1.2625815868377686, -0.06923521310091019, -0.06881357729434967, 1.9078041315078735, 0.9597765207290649, 2.903000593185425, 0.7523767948150635, -0.9012709856033325, -1.7571709156036377, -0.7685000896453857, -0.4522280991077423, -1.1105592250823975, 0.5088851451873779, 0.05557693541049957, -0.3095077574253082, 1.6906827688217163, 0.3135794997215271, -0.20152020454406738, -1.6989421844482422, -0.014277821406722069, -0.4404636323451996, 1.0500657558441162, 0.8829517960548401], [-2.5620267391204834, -0.591342031955719, -0.6634653806686401, -1.0301650762557983, 0.567349374294281, 0.07384001463651657, -1.0900683403015137, -0.7091494798660278, 1.3928979635238647, -0.10620256513357162, 1.0010098218917847, 0.7841792702674866, -0.43377286195755005, 1.5644184350967407, -1.0226699113845825, -0.18158213794231415, -0.834058403968811, -0.335591197013855, -0.5049126744270325, 0.031181225553154945, -0.15959258377552032, -0.8210738301277161, 0.8257238268852234, -1.3764923810958862, -1.0307940244674683, -0.25254201889038086, -0.7501486539840698, -0.3587651252746582, -0.19296780228614807, -1.2040328979492188, 2.1387767791748047, 0.7277427315711975, 0.17189958691596985, 1.4670294523239136, -0.8136330842971802, -1.5692944526672363, 1.4229990243911743, -2.785452365875244, 0.5110273361206055, 0.4473665952682495, -0.6837566494941711, 1.0509648323059082, 0.981736421585083, 0.9646196365356445, -0.531730592250824, 1.467566967010498, -0.24039512872695923, -0.27921807765960693, 0.2677730321884155, -0.9405320286750793, 0.8178317546844482, 0.24772049486637115, 0.6575416922569275, -0.4354155361652374, 0.28323954343795776, -2.5114173889160156, 1.310775876045227, 1.4089086055755615, -1.2980483770370483, -0.391812264919281, 0.8731295466423035, -0.4838438332080841, -0.16306021809577942, 0.7732642889022827, -0.29141172766685486, -0.27795645594596863, 2.3836231231689453, 0.08396056294441223, -1.0474108457565308, 2.2489397525787354, 1.5322139263153076, -0.06571809202432632, 0.9101940989494324, 0.7473583221435547, -1.05217444896698, 0.984410285949707, -1.7803541421890259, -0.14351972937583923, -0.23122765123844147, -0.5428572297096252, 3.1678857803344727, 0.5291264057159424, -2.0947911739349365, -0.9073731899261475, 0.4397205412387848, -0.3425459861755371, 0.11727084964513779, -0.6148151159286499, 0.6214728355407715, -1.5954684019088745, 1.6584653854370117, 1.924465537071228, -0.9655214548110962, -0.8668066263198853, 1.1995066404342651, -0.4137565791606903, -1.8656049966812134, 0.007142374757677317, 1.269171118736267, -1.0781255960464478, -1.3824841976165771, -1.0042728185653687, -0.9330394268035889, -1.9415905475616455, -1.8396670818328857, -1.6095105409622192, 1.5149894952774048, 0.9396584033966064, -0.4470632076263428, -0.5190662145614624, 0.4092601537704468, -0.7451457977294922, -0.4928610324859619, -0.26020851731300354, -0.8049366474151611, -0.25624242424964905, -0.4102073907852173, -0.6190662384033203, -0.6734108924865723, 1.4322409629821777, 0.7191251516342163, -1.8066755533218384, 0.19724559783935547, 1.0398188829421997, 0.2960491180419922, -3.3471274375915527, -0.25685158371925354, 0.24450169503688812, -0.8288957476615906, -0.40589240193367004, -0.9713430404663086, 0.9196028113365173, -1.8289719820022583, -0.6021894812583923, -0.26242679357528687, 0.4261319041252136, 0.7217798233032227, -0.8425613641738892, -0.8118959069252014, 1.9367666244506836, 0.643546998500824, -0.6557328104972839, -0.42760947346687317, 0.01951761171221733, -0.22219498455524445, 0.5372984409332275, -2.0163495540618896, 0.5607418417930603, -1.1098153591156006, 0.39936327934265137, 0.955656886100769, 1.598257064819336, -0.430016428232193, 0.5103185176849365, 1.1201823949813843, -0.22449688613414764, 1.8309197425842285, 0.4862474203109741, -2.514237880706787, 1.075379490852356, -0.08881998807191849, 0.13669051229953766, -1.4702837467193604, -0.5948917865753174, 0.31083735823631287, -2.8785362243652344, 0.6038373112678528, 0.8622578382492065, 1.4002188444137573, 0.15010380744934082, -0.8855607509613037, -1.3726054430007935, 0.1703394055366516, -0.0052534774877130985, 1.3195120096206665, -2.328446865081787, 0.42477649450302124, 1.581889033317566, 0.9222176671028137, 0.34436994791030884, 0.5934765338897705, 0.5401414632797241, 1.367404580116272, -1.5197961330413818, 0.454215943813324, -1.747331142425537, -1.370087742805481, 0.8318485617637634, 2.014742374420166, 0.4296005070209503, 2.147348642349243, 0.6939417719841003, 0.09780661761760712, -0.4765307903289795, -2.0533487796783447, 0.1532740443944931, -0.08058348298072815, -0.40293362736701965, -1.3304460048675537, -0.9672868847846985, -0.39482393860816956, -0.49735790491104126, -0.6851382255554199, -0.17806564271450043, -0.5448307394981384, -0.2601340115070343, -0.067670539021492, 1.5373839139938354, 0.7558659315109253, -1.303208589553833, 2.369924306869507, 0.3280002176761627, 0.05912122502923012, -1.124550223350525, -0.28621309995651245, 0.9930647015571594, 0.47444644570350647, -0.343564510345459, 0.23110507428646088, -0.05190132185816765, 1.2801281213760376, 0.44067975878715515, -0.15153996646404266, -0.0007458726176992059, -0.38026320934295654, 0.6982361674308777, 0.547737181186676, -1.177150845527649, -1.6860922574996948, -0.504017174243927, 0.33984318375587463, -0.40614384412765503, 0.6129041314125061, -0.1469743698835373, 0.22239242494106293, -0.023395324125885963, 0.49636948108673096, -1.015972375869751, 0.17369228601455688, -1.1343594789505005, 1.9388552904129028, -0.4500756859779358, -0.10145265609025955, -0.7015888094902039, -0.19839400053024292, -0.37791702151298523, -0.753856897354126, 0.975773274898529, -0.40552443265914917, 1.6525566577911377, 0.5404434204101562, 0.708346962928772, 1.517701268196106, -1.1070079803466797, 0.8152648210525513, 2.4383230209350586, -0.5681846141815186, -0.029777931049466133, 0.0313563272356987, 0.683404803276062, 1.5825316905975342, 2.3729913234710693, 1.128737449645996, 0.05876564979553223, -1.191085696220398, -0.04079817980527878, 0.3767492175102234, 0.9311193227767944, -0.3006453514099121, -0.3027002811431885, 0.590893566608429, 1.0767499208450317, 1.4701955318450928, 0.4605330526828766, -0.022882234305143356, 0.9637783765792847, 0.0354427844285965, 1.576089859008789, 1.4930641651153564, 1.1518018245697021, 1.4008338451385498, 0.016340015456080437, -0.8823919296264648, -1.1220707893371582, 1.540250301361084, 0.6870324015617371, 0.7242886424064636, -0.18126225471496582, 0.7823852896690369, -0.7440152764320374, -0.30476823449134827, 0.7733994126319885, -1.0303224325180054, 0.29973822832107544, -0.8046281933784485, 1.7655260562896729, -0.5679773688316345, 1.280514121055603, 1.7417393922805786, 0.6919807195663452]], "dtype": "float32"}, {"__nd__": [[1.3052546977996826, 0.09697571396827698, -0.8764106035232544, -0.665199339389801, -1.2906806468963623, 0.4416272044181824, -0.7129784226417542, -0.3246438503265381, 0.6876093745231628, 1.497972011566162, -0.5588985085487366, -2.416317939758301, 1.1122208833694458, -0.07976175099611282, -0.16394715011119843, 0.37629029154777527, 0.6084325313568115, -0.5717180967330933, -1.201910376548767, -0.8108160495758057, 1.0987282991409302, 2.6983306407928467, -0.31701451539993286, -1.0563535690307617, -0.3318440914154053, 1.1599149703979492, 1.2032983303070068, 0.722989559173584, -0.8732137084007263, 0.8550718426704407, -0.2988390624523163, -0.08398409187793732, 0.18820030987262726, -0.640116810798645, -1.086762547492981, 0.8091577887535095, 0.3400361239910126, -0.9055938124656677, -0.5751994252204895, -1.8668321371078491, -0.7587279081344604, -1.3401870727539062, 1.258202314376831, 0.31845834851264954, -1.2357664108276367, -1.2015761137008667, -1.2224082946777344, 0.7034634351730347, 1.48851478099823, 0.611819863319397, -0.4226993918418884, 0.4955327808856964, 1.5284417867660522, 1.8254755735397339, -0.4250548481941223, 0.27266934514045715, 0.6949211359024048, 0.7424011826515198, 0.5033906698226929, -0.5092213153839111, -1.2655614614486694, 0.4380541741847992, 0.9730731248855591, -1.4272613525390625, 0.2629803717136383, 0.6741212606430054, -0.47372931241989136, 0.34095299243927, 0.9838210940361023, 1.0450057983398438, -1.8034356832504272, -1.7725297212600708, -1.0745656490325928, 0.5189752578735352, 0.06989718973636627, 1.1842288970947266, 0.012911795638501644, 1.6246364116668701, 0.34543755650520325, 0.2422664910554886, 0.13024181127548218, 0.5649123787879944, 0.9989446401596069, 1.4174631834030151, -0.5025471448898315, 2.7922229766845703, -1.9179567098617554, -0.12737321853637695, -0.762639045715332, -0.3036368489265442, -2.218458890914917, 1.454392433166504, -0.11938924342393875, -2.1905949115753174, -0.2970203757286072, 0.1503576934337616, 0.6986941695213318, 0.08140187710523605, 0.6018126010894775, -0.5237413048744202, 0.6099967956542969, 0.5143560171127319, 1.6833113431930542, -0.7694379687309265, -2.251565933227539, -0.23965239524841309, -1.0474449396133423, 0.9743165373802185, -0.6739389896392822, 0.5638508200645447, 0.4421986937522888, 0.06507766246795654, 1.515069842338562, -0.2568173408508301, 0.3420352339744568, -0.919138491153717, -1.3334850072860718, 0.023749936372041702, 0.5182270407676697, 0.728804349899292, 0.3473917543888092, 1.4212937355041504, 0.29341697692871094, 1.7049362659454346, -0.3683348000049591, -1.1300153732299805, -0.4706915616989136, 0.9765520095825195, 0.1412399858236313], [-0.5129562020301819, -0.7225220203399658, 0.8370854258537292, 0.2586771547794342, 0.5059887766838074, 2.455487012863159, -1.6350256204605103, -1.4227961301803589, 0.7574756145477295, -0.8957967758178711, -0.5862575769424438, 1.9202699661254883, 0.5381529927253723, -0.2686719596385956, 0.22515860199928284, 0.5566520094871521, 0.2058812528848648, -0.051398150622844696, 0.7336888909339905, 1.8061829805374146, 0.08839414268732071, -0.5636056065559387, 0.10072004050016403, -0.46931353211402893, -2.8777108192443848, 0.5372365713119507, -1.4172775745391846, 0.376814067363739, 0.36953720450401306, -0.5472118258476257, -0.8117736577987671, -0.5614443421363831, 0.7975039482116699, -0.8593828678131104, 0.7797993421554565, 0.398952454328537, 0.007374617271125317, 0.35512104630470276, -0.8813610672950745, 0.31324177980422974, -1.3481248617172241, 0.65077805519104, -0.637203574180603, -0.8496027588844299, 1.3532497882843018, 0.15603473782539368, -0.39342203736305237, -0.8622039556503296, 1.7137700319290161, -1.6720517873764038, 0.3868713676929474, 0.4888206422328949, -0.1646614372730255, 0.6934785842895508, 0.042316701263189316, -1.5031464099884033, 1.800413727760315, -0.19197693467140198, 0.22274230420589447, 0.11387214809656143, -0.8519794344902039, -0.03979932889342308, -0.48345789313316345, 1.2179286479949951, 1.0350781679153442, 0.3790905177593231, 0.24854175746440887, 0.5210092067718506, -1.5827593803405762, 1.1287999153137207, 0.0225446168333292, -0.7227088212966919, 0.5758272409439087, -0.3311660587787628, 1.6850318908691406, 0.4657149910926819, 1.3302664756774902, 0.7356795072555542, 1.4184778928756714, 0.8896039724349976, -0.28013908863067627, -1.14738130569458, 1.296823263168335, -0.5979111790657043, 1.1108088493347168, -0.8263368010520935, 0.5817758440971375, -0.8023285269737244, 2.104982852935791, 2.0461766719818115, -0.5972363352775574, -0.5299072265625, 0.9164566397666931, -0.988642692565918, 0.6896657943725586, 0.7500230669975281, 0.29566261172294617, -0.31017786264419556, -0.15741918981075287, 0.4217565655708313, -0.808872401714325, 0.30039411783218384, -2.575453758239746, -0.01592087931931019, 0.026093991473317146, -1.3938320875167847, -2.332817316055298, 1.1194822788238525, 1.4951348304748535, -1.1961320638656616, 0.6628590822219849, 0.1598329097032547, 0.24600645899772644, 1.4224144220352173, -0.3035065233707428, -0.8486570715904236, -0.28024300932884216, -1.724660873413086, 0.13241922855377197, 0.08158432692289352, -0.8798655271530151, 1.818274736404419, -0.824367105960846, -0.08830364048480988, 0.7296609282493591, 0.9759581685066223, -1.2355257272720337, -0.7041693329811096, -0.653275728225708], [-0.9016299843788147, -0.38247671723365784, 0.22187037765979767, -2.3249621391296387, 1.2683168649673462, 0.3028298616409302, 3.0551650524139404, 0.9484046697616577, 1.0024173259735107, 1.242668867111206, -1.2032233476638794, 0.7577493786811829, 0.23314988613128662, 1.0913985967636108, 1.9012665748596191, 1.1183637380599976, 1.0624542236328125, -0.8711403608322144, -0.42355775833129883, -0.7774549126625061, 0.0439012236893177, -1.5806083679199219, -0.7906338572502136, -0.6200449466705322, -0.013915936462581158, -0.47906428575515747, -0.515454888343811, -0.652481198310852, -1.9620623588562012, -1.1001715660095215, -0.4401812255382538, -0.19790732860565186, -1.0189226865768433, -1.5544623136520386, 0.6257271766662598, 0.2907848656177521, -1.4723066091537476, 0.005287733394652605, -0.3484339118003845, 0.4730018675327301, 2.5264053344726562, 0.37122443318367004, 2.0840890407562256, -0.35222238302230835, -0.9200071692466736, 0.5618602633476257, -0.28474161028862, -0.9115882515907288, 0.9916843175888062, 0.15331947803497314, 0.7307738661766052, -0.5055683255195618, -0.07812024652957916, 0.2577626705169678, -0.03097549080848694, 0.5898497700691223, -2.2354393005371094, 1.87106454372406, -0.7487155199050903, 1.4222012758255005, 0.7663393616676331, -0.694509744644165, 0.4096048176288605, -1.9486355781555176, -1.6201573610305786, -0.35079053044319153, 0.929615318775177, -0.006348784081637859, -0.29174789786338806, 0.8298455476760864, -1.6013725996017456, 0.12217795848846436, 1.2108782529830933, 0.7428793907165527, 0.03641233593225479, 0.11383092403411865, -1.176580548286438, 1.0590496063232422, -0.09889139980077744, -0.6403706073760986, 2.424621105194092, -0.20538903772830963, 0.14101411402225494, 0.8378739356994629, -0.18524745106697083, -0.6096993088722229, 0.8416022062301636, -0.45134735107421875, -0.7721748352050781, 0.34816357493400574, 1.00221586227417, -0.8709354996681213, -0.42362987995147705, 0.47950154542922974, 0.8866509199142456, 0.5563132762908936, 2.1425702571868896, -0.06838858127593994, 1.3523043394088745, -1.155802845954895, -1.3948664665222168, 0.3321652114391327, 0.1452292650938034, -0.3897441327571869, -0.8599065542221069, 0.846595287322998, 0.198964923620224, -2.284848690032959, -0.8176571726799011, 0.7446868419647217, 0.849552571773529, 0.7009602189064026, -0.7224656939506531, -0.6802375316619873, -0.8417349457740784, -0.2160080373287201, -0.49072811007499695, 2.306382179260254, 1.000056266784668, -0.0777035728096962, -1.0036269426345825, 0.6363652348518372, 0.26413026452064514, -0.869787871837616, 0.3751900792121887, 1.4873437881469727, -0.9279117584228516, 0.5861974954605103, -0.3466799259185791], [-0.46095848083496094, -1.0913814306259155, 1.3030279874801636, -0.4332105815410614, 1.7437711954116821, 0.5782419443130493, -1.805221676826477, 0.18695449829101562, 0.2909190356731415, -0.36157146096229553, 0.2760714292526245, -0.3951357901096344, 0.7532647252082825, 0.27814528346061707, 2.318934440612793, 0.3019290864467621, -1.2690551280975342, -0.594380795955658, -1.5630170106887817, -1.8218483924865723, 0.7575019001960754, 0.45577964186668396, 0.3709827959537506, -0.772100567817688, -0.3211352527141571, -1.145340085029602, -1.314167857170105, -0.018210019916296005, -1.0869094133377075, 1.8310086727142334, -1.5979647636413574, -1.2027169466018677, 0.14807263016700745, -1.6467243432998657, 0.07555602490901947, -0.8719619512557983, 0.8090298175811768, -0.12864620983600616, 0.620887815952301, -0.3065769076347351, -0.8858338594436646, -0.05339909717440605, -0.34305793046951294, 0.5491018295288086, 0.527596652507782, -2.539803981781006, 0.9806868433952332, -0.5461393594741821, -1.4294811487197876, -0.13483040034770966, -0.313662052154541, 0.9987747669219971, -1.9818590879440308, -0.8045269846916199, 1.4277760982513428, -0.10153927654027939, -1.2935612201690674, -0.9074579477310181, -0.5846895575523376, -2.4647252559661865, 1.8560532331466675, -0.5707427859306335, -1.6535898447036743, -1.8815813064575195, -0.10926166921854019, -0.7510412931442261, -1.1008344888687134, -1.9016480445861816, -0.20835460722446442, -1.1709856986999512, 1.884193778038025, 0.3595520555973053, 0.5505588054656982, -0.015780556946992874, 1.2505443096160889, -2.8733088970184326, 0.39226096868515015, 0.29624617099761963, 0.4111776649951935, 0.31758731603622437, 0.7152738571166992, 1.4383975267410278, 0.5580158829689026, -1.2581350803375244, -0.2627745270729065, 2.205509662628174, -1.889057993888855, 1.0460045337677002, 0.25217968225479126, 1.8995165824890137, -0.12855350971221924, -0.11278177052736282, -0.7553000450134277, 1.712799310684204, 0.6816138625144958, 2.414062261581421, 0.3725038468837738, -0.9052993059158325, 0.9696051478385925, -0.1380421370267868, -2.0607187747955322, 1.1321274042129517, 0.9705649614334106, 0.8285631537437439, 1.3218777179718018, 0.08871878683567047, 0.4072291851043701, -0.6241302490234375, 2.1984105110168457, -0.13143225014209747, 0.527266263961792, 0.3863864541053772, -1.721390724182129, 1.0635085105895996, -0.3116323947906494, -1.460009217262268, 0.7701133489608765, 0.7779147624969482, 0.1356276571750641, 0.6059826016426086, 1.1569783687591553, 0.9263682961463928, -0.14655965566635132, 2.455913543701172, -0.27074649930000305, 0.017821284011006355, 0.18864159286022186, -0.6553393006324768, -0.8932369351387024], [0.17334692180156708, 0.2221260517835617, -0.773717999458313, -0.08456706255674362, -2.0828142166137695, 0.12670943140983582, 0.776664674282074, 0.08028703927993774, -1.0780833959579468, -0.05564447492361069, -0.1862170398235321, 0.5680862069129944, 1.1161787509918213, 1.6870335340499878, -0.5003395676612854, -0.7114630341529846, -1.508765697479248, -0.6153193116188049, 0.6291972994804382, 0.3496181070804596, 0.12704820930957794, 0.7321299910545349, -0.09130019694566727, -0.6405022740364075, -0.3988703787326813, -0.6283466219902039, 0.7836290597915649, -0.7516381144523621, -0.9946921467781067, 0.4128071963787079, -0.24575552344322205, 1.3548142910003662, -0.7009066343307495, 0.45451420545578003, 1.4166561365127563, -0.5067180395126343, 0.2804291844367981, 1.250808835029602, -1.881043553352356, 0.446573406457901, 2.800421714782715, -0.5455501079559326, 1.852317214012146, 0.5221719145774841, 0.4891519844532013, 0.36660346388816833, 0.2591843903064728, -0.5712624192237854, -0.513486385345459, -0.4776119887828827, 1.2725613117218018, 0.23389293253421783, 0.47304412722587585, 1.12213134765625, -0.008382624946534634, 0.4036492109298706, -0.7464398741722107, 0.13587245345115662, -1.2727863788604736, -0.9336721897125244, 0.5891947150230408, -0.14994876086711884, 0.5148153901100159, 2.4208788871765137, -1.9762108325958252, -0.6785159707069397, -0.7185403108596802, 0.24149048328399658, -0.5259588360786438, -0.3172031044960022, 0.14708998799324036, 0.010723401792347431, 0.19230462610721588, 1.0007795095443726, -1.7951511144638062, 0.5560591220855713, 0.5990646481513977, -0.12510313093662262, -0.29557424783706665, -1.629870057106018, -0.3037092983722687, -1.1983410120010376, -0.5820552706718445, -1.0219730138778687, 0.2059001624584198, 1.0834627151489258, 1.4857474565505981, -0.5052409768104553, 1.3600382804870605, -2.2300944328308105, -0.2780175805091858, 0.16245697438716888, -0.9122536778450012, -0.4893183708190918, -1.4687515497207642, -1.6859627962112427, 1.0362266302108765, 1.132472276687622, -1.4385757446289062, -0.6370856165885925, -0.43671026825904846, 1.6297504901885986, 1.3424415588378906, 0.25420045852661133, 0.3355155885219574, 1.5813156366348267, -0.5052664875984192, -0.36498740315437317, 1.3560346364974976, -1.0884389877319336, -1.3311747312545776, 1.1529663801193237, 0.0595487505197525, 1.1781197786331177, 0.32344961166381836, 0.8957799673080444, -0.45827269554138184, 0.017088452354073524, -1.1460564136505127, 1.2090402841567993, 0.12679815292358398, 0.20202700793743134, 0.8176764249801636, 2.0151755809783936, -0.5023691058158875, 0.3322969675064087, 0.4791092276573181, -0.5811744332313538, -0.1006554514169693], [-1.6765398979187012, 1.0689867734909058, 0.39714333415031433, 0.4503515064716339, -0.2598132789134979, 0.0881597250699997, -1.8709418773651123, 0.20977166295051575, 0.41316667199134827, -0.3574138581752777, -1.6873369216918945, -0.186135396361351, -0.05266759917140007, 0.003899502335116267, 0.6872859597206116, 0.312007337808609, 0.9611374735832214, 0.7198418974876404, 0.7622979879379272, 0.965395450592041, 0.8736119866371155, 1.3425229787826538, -0.4729735255241394, -0.4394229054450989, 0.7476863861083984, 0.21797530353069305, -1.2589738368988037, 0.23023417592048645, 0.7682068943977356, -2.8368659019470215, 1.0208021402359009, -0.215139701962471, 0.3628833591938019, -0.1977638304233551, 0.3057314157485962, 0.23678070306777954, 2.222860813140869, 0.5235174298286438, -0.2964227497577667, 0.41395044326782227, 1.5148614645004272, 1.3464632034301758, -1.20981764793396, -0.7082381844520569, 0.9154897928237915, -0.24552121758460999, -0.7297177910804749, -1.048490285873413, -1.988233208656311, -0.7654743194580078, -0.2797906994819641, -0.487729549407959, -1.0467581748962402, 1.0647284984588623, -0.932928204536438, 0.37193018198013306, 0.15498939156532288, -1.5260584354400635, 1.0007944107055664, -0.48396554589271545, -0.4511260390281677, 1.1451414823532104, 1.7097309827804565, -0.4669567048549652, -0.26571106910705566, -1.4727109670639038, 0.9198142886161804, 0.09157571941614151, 0.3254390358924866, -0.28519830107688904, -1.2681745290756226, -1.1296013593673706, -0.380167692899704, 0.5866168737411499, 0.21308307349681854, -2.1847944259643555, 0.14007464051246643, -1.348326325416565, -1.0004392862319946, -0.5959066152572632, -0.7173939943313599, -0.2741013765335083, -0.03582697734236717, -0.46044623851776123, -1.0382691621780396, -0.25219428539276123, 0.6532940864562988, 1.0217682123184204, 1.0839552879333496, 1.598458170890808, 0.9033578038215637, -1.6648750305175781, -0.1996673345565796, -0.35458990931510925, -0.9420321583747864, -0.8423120379447937, -0.06162266805768013, 1.522359848022461, 0.10232971608638763, 1.0785571336746216, -0.3457123041152954, -0.052722565829753876, -1.5420112609863281, -1.3968013525009155, 0.08884299546480179, -1.1214711666107178, 0.46443915367126465, -0.6539356708526611, 0.24813327193260193, 0.504633903503418, -0.6328632235527039, 0.656136691570282, -1.3971257209777832, 0.30166545510292053, 1.7716313600540161, -0.21205143630504608, -0.47126978635787964, 1.8687556982040405, -1.0563396215438843, -0.48117727041244507, -0.8939406275749207, -0.9411517381668091, 1.2662487030029297, 0.22442612051963806, -1.2327420711517334, -0.8516501784324646, -0.1314486414194107, 1.3262304067611694, -0.43909260630607605], [0.429246187210083, -0.26023587584495544, -1.328755497932434, 1.5836373567581177, -0.7165399193763733, -0.24202096462249756, 0.32582855224609375, 0.39324986934661865, 0.47974884510040283, -0.36250558495521545, 1.0887714624404907, -0.5816617608070374, 0.45148247480392456, -2.695314884185791, 2.4105656147003174, -0.5945241451263428, 0.29189345240592957, -1.891880750656128, 0.9122565984725952, -0.13978374004364014, -0.8517225384712219, 1.3610354661941528, 1.1547815799713135, -0.06421195715665817, -0.23402221500873566, 1.5990180969238281, -1.3913602828979492, -0.11990632116794586, -1.0203015804290771, -0.28938597440719604, 0.21518757939338684, -2.393787145614624, -0.24270470440387726, 1.3877570629119873, 0.06221752613782883, 1.191182255744934, -0.03833536431193352, 1.2111958265304565, 1.3642841577529907, 2.43015718460083, 0.8533586263656616, -0.3935028910636902, 0.4177034795284271, -1.2897863388061523, -1.8109867572784424, 0.6270321011543274, 0.46429574489593506, -0.2588998079299927, 0.8475173711776733, -0.5319809913635254, 0.7877927422523499, 1.362223744392395, 0.9246717691421509, 0.48790305852890015, -1.5193623304367065, 0.5740255117416382, -0.5530180931091309, -0.40258288383483887, -0.21623747050762177, -1.1732641458511353, 0.8503462672233582, 0.6276408433914185, -0.06926852464675903, 0.17876513302326202, -2.290135145187378, 0.15614768862724304, 0.6515660285949707, -0.06872652471065521, 0.48092910647392273, 0.6480278372764587, -1.1838799715042114, -0.4554491639137268, 0.4020081162452698, -0.14598318934440613, 0.6299214959144592, 0.16114483773708344, -1.5807058811187744, 0.46151718497276306, -0.3245566785335541, 0.5409231781959534, -0.6355332136154175, -0.441297322511673, 0.2740074098110199, -0.04796598479151726, -0.3105195462703705, 0.04045606404542923, 0.5774030685424805, -0.029857557266950607, 0.016609881073236465, -0.8806291222572327, 0.6541747450828552, 1.198788046836853, -0.2215268760919571, -1.758931279182434, -0.9952728748321533, -0.9829593896865845, -0.6530715227127075, -0.26279476284980774, -0.28321272134780884, -0.2719636559486389, 0.4815235733985901, -1.0151792764663696, 0.9124128222465515, 0.6541718244552612, -0.25831401348114014, -0.6341985464096069, -1.3482342958450317, 0.00582205830141902, 1.630239725112915, 0.921513557434082, -1.4277467727661133, -0.01208607666194439, -0.6983541250228882, -0.6950318813323975, 0.9969182014465332, -0.6156066060066223, 0.8584043979644775, 0.018818946555256844, 1.3012715578079224, -0.6130814552307129, -1.3852684497833252, -0.3294629752635956, -0.7703458070755005, 0.6155805587768555, -1.9302668571472168, -1.1213418245315552, -0.0011912892805412412, 1.2855830192565918, 1.2219913005828857], [-0.37724143266677856, 0.7634612917900085, -0.5238062143325806, -1.2493571043014526, 1.3814793825149536, 0.3619697391986847, 0.38537949323654175, 0.16129480302333832, 0.8546242713928223, 0.6919781565666199, 0.5611733794212341, -0.7974445819854736, -0.3173108696937561, -0.5211756229400635, -1.1168155670166016, -0.30888622999191284, 1.7452867031097412, 0.5177791118621826, 0.20592249929904938, 1.1847540140151978, -0.9314131140708923, 0.9428238272666931, -1.8631747961044312, -0.4075866639614105, 1.0667006969451904, 0.23897333443164825, 1.7746387720108032, 0.6232932209968567, -0.5455648899078369, 0.11374546587467194, 0.5323307514190674, -1.8007287979125977, 0.6560932993888855, 1.9351383447647095, -0.4167925715446472, -0.693554162979126, -0.5710339546203613, -1.0187581777572632, -1.3696595430374146, -0.8286300897598267, 1.6071122884750366, -1.0351622104644775, 0.043354764580726624, 0.945084273815155, -0.042935650795698166, 0.052294157445430756, -1.356603980064392, -0.05412202700972557, 0.47859111428260803, 0.04556725174188614, 0.8509684801101685, -0.7961559891700745, 0.39161065220832825, 0.8664831519126892, -0.4124950170516968, -1.3420146703720093, 1.5927504301071167, -0.552433967590332, 1.0305428504943848, -0.9857625365257263, -0.048593562096357346, -0.7745262384414673, 0.37000834941864014, 0.1768275946378708, 0.13017788529396057, -0.7877116799354553, -0.6806154251098633, 0.9155105948448181, 0.4796927273273468, 0.8617581725120544, 0.30016252398490906, 1.1910793781280518, -0.3054964244365692, 1.561704158782959, -0.7737024426460266, -0.37031805515289307, -1.68860924243927, -0.4325830042362213, 0.4466676115989685, 0.7103499174118042, -0.794379472732544, 0.8223862648010254, 0.20291900634765625, -1.3254543542861938, -0.2468162178993225, 1.6168708801269531, -1.3705052137374878, -0.05296211689710617, -1.3202319145202637, -1.53758704662323, -0.5797479748725891, 2.957127332687378, -0.2202722430229187, 0.40484702587127686, -0.6254091262817383, -1.3038451671600342, 0.4022185802459717, -1.380202054977417, -0.028597867116332054, -0.8016475439071655, -0.6238064169883728, -1.3502681255340576, -0.26905640959739685, 0.13146774470806122, 0.49791714549064636, -1.14830482006073, 0.6628214716911316, 0.6939529776573181, 1.069825530052185, -2.2612826824188232, 2.257730722427368, 1.3945497274398804, -1.6229472160339355, -0.1325148642063141, 0.9956833124160767, 0.2242213785648346, -0.11577867716550827, 0.5087608098983765, 1.2175304889678955, 0.9060553908348083, 0.45116549730300903, -1.189491868019104, -0.7136309742927551, -1.1028635501861572, 0.1292264759540558, 1.3469929695129395, 0.9071258902549744, -2.415426254272461, -0.18108704686164856], [0.7695972323417664, -0.9546186923980713, 0.26353132724761963, 1.0359234809875488, 0.828641414642334, 1.4476629495620728, 0.7006910443305969, -0.353534460067749, -0.9391935467720032, 0.9320589303970337, 0.3141467869281769, 1.1805686950683594, 0.04106250777840614, -0.4810657501220703, 0.5815837979316711, 0.9010834097862244, 0.02217887155711651, 0.5063108801841736, -0.6648004055023193, -0.5477370023727417, -0.2678828835487366, -2.1152503490448, -0.021695192903280258, 1.1226638555526733, 0.654750406742096, -0.7987619638442993, -0.27685704827308655, 0.6496670842170715, 0.5654298663139343, -1.706060767173767, 0.3949272930622101, -0.9922387003898621, 0.6623690724372864, -0.2701278328895569, -0.9249849319458008, -1.554421067237854, -1.9086633920669556, -1.909867286682129, 0.5258265733718872, -0.09138405323028564, 0.8653582334518433, -0.42700469493865967, 0.6701040863990784, -0.008588063530623913, 0.8051632046699524, -1.2463737726211548, 0.043400317430496216, 0.7805792093276978, -0.10468316823244095, -0.6441748142242432, -0.2822728753089905, -0.8376553654670715, 0.696189820766449, 0.290596604347229, 0.2955018877983093, -0.8013680577278137, -0.26412850618362427, 0.4638449549674988, -3.012514591217041, -0.8917284607887268, -1.466681718826294, 1.334299087524414, -0.31112751364707947, -1.0871202945709229, 1.5309301614761353, -2.1383955478668213, 1.266158103942871, 0.9800606966018677, 0.6785228848457336, 0.2068537026643753, -0.29282233119010925, -0.07975476235151291, 1.0156837701797485, -0.46659451723098755, 1.053443431854248, 2.3241281509399414, -1.3209764957427979, -1.1988260746002197, 0.613805890083313, -1.0807278156280518, 0.7497194409370422, -0.168088898062706, 0.03832119330763817, 1.3220508098602295, 0.4925837516784668, -1.491933822631836, -0.8682911992073059, -1.2175767421722412, -0.9433004856109619, 0.5014745593070984, -1.337349772453308, -0.5711701512336731, -0.04357753321528435, -0.4742056429386139, 0.5322157740592957, -0.21524803340435028, 0.9460477232933044, -0.006149597465991974, -1.8644009828567505, -0.1604967564344406, 0.5363042950630188, -0.15470771491527557, 0.5437222719192505, -0.6123287081718445, -0.9489752054214478, 0.7112806439399719, -0.9416785836219788, -1.1306815147399902, -0.4018820524215698, -1.426844596862793, -1.3642008304595947, 1.8228329420089722, 1.4642212390899658, 0.0980539321899414, -0.48273444175720215, 1.0629583597183228, -0.49663040041923523, 0.556103527545929, -1.3056210279464722, -0.20763318240642548, 0.8299105167388916, 0.27567756175994873, 1.452643871307373, -1.3523818254470825, -2.0410315990448, -0.2236192524433136, 0.9308968186378479, 0.18089096248149872, 1.5623447895050049], [-1.3874444961547852, 0.34469959139823914, 0.37104567885398865, 0.051839280873537064, 0.2282608598470688, -1.115195870399475, 0.5191082954406738, -1.072765588760376, 0.7278289794921875, -0.9347856044769287, -0.18307727575302124, 0.5985687971115112, -0.526504635810852, 0.08297647535800934, -0.35998210310935974, 0.5772096514701843, 0.20091670751571655, -0.9283820986747742, 2.028503656387329, -0.1611492931842804, 0.5850428938865662, 1.1675671339035034, 0.655412495136261, 1.3615124225616455, -0.16774018108844757, 1.7739051580429077, 0.04635041579604149, 1.477537989616394, 0.1706053912639618, -1.5467054843902588, 0.17591899633407593, -0.1942405104637146, -0.9585238695144653, 1.8962575197219849, 0.9803416132926941, -1.9123162031173706, -0.8807364702224731, -0.4124777317047119, -2.25049090385437, -0.4400162696838379, 0.8117578625679016, 2.268568754196167, 1.39250648021698, 0.7173092365264893, -0.7441129684448242, -0.4159105718135834, 0.9520552754402161, -0.7274385094642639, -1.3292021751403809, -0.550957441329956, -0.9168108105659485, -0.15550605952739716, 0.6929538249969482, -1.7237943410873413, 1.6418993473052979, -1.462105631828308, -2.0160603523254395, -0.5860363841056824, 0.0876862034201622, -1.0325859785079956, -0.598721981048584, 1.4598188400268555, -2.1981964111328125, -0.119631826877594, 0.401632696390152, -0.016233671456575394, -0.5583934187889099, 1.5410003662109375, 0.7025048136711121, 0.8465116024017334, -0.4262145757675171, 1.6594523191452026, 0.3213355839252472, -1.1058709621429443, 0.29582101106643677, -0.8241012692451477, 1.6225186586380005, -0.49286437034606934, 0.8652981519699097, -0.371812641620636, 1.9173901081085205, -0.09754588454961777, -0.3774224519729614, 1.0147058963775635, -0.22197434306144714, -2.1298110485076904, -0.8552668690681458, -1.9730199575424194, -0.41895461082458496, -0.20499595999717712, -0.8667641282081604, -0.2830173969268799, 0.99344402551651, 1.0690504312515259, -1.128264307975769, 1.1833847761154175, -0.04392601177096367, -0.3057316839694977, -0.409812331199646, -1.5034995079040527, -1.661088228225708, -1.4330614805221558, 0.05633547902107239, 0.5144869685173035, 0.24827304482460022, 2.0239791870117188, -0.7603484988212585, 2.2049567699432373, 0.4422479271888733, -0.8320215940475464, -0.939516544342041, 1.089443564414978, -0.34179016947746277, -0.17242562770843506, -1.3582285642623901, 0.017215967178344727, 0.3395538330078125, -1.039299726486206, 1.224683403968811, -1.0708903074264526, 0.2376968115568161, -0.001963583752512932, -0.8435937762260437, 0.6952230930328369, -0.12179652601480484, -0.7023745179176331, 0.8599611520767212, 1.8623136281967163, -1.545052409172058], [-0.9494693279266357, -1.5268350839614868, -1.7965658903121948, 0.09626349806785583, 0.004438744857907295, 1.1457668542861938, 1.1506279706954956, 0.7086566090583801, -1.0288591384887695, -0.9188584089279175, 0.3682650923728943, 0.12180379778146744, 0.07792027294635773, -0.5250368118286133, -0.9212570786476135, -0.46734434366226196, -0.08257831633090973, -0.10017819702625275, -1.0497002601623535, -0.2729007601737976, 1.9757417440414429, -1.8259165287017822, -1.5219476222991943, -0.1731652319431305, -0.23655197024345398, 0.4957009553909302, -1.2561184167861938, -0.01940999925136566, -0.4594143331050873, 0.05159977823495865, 1.9687920808792114, -0.12166222929954529, -0.1776413917541504, 0.47472697496414185, -2.2145252227783203, -0.6723958253860474, -1.5642393827438354, 1.027279019355774, 0.17249521613121033, -1.4629160165786743, -0.06340006738901138, 0.27399900555610657, -1.3713546991348267, 0.29253366589546204, 0.2029397338628769, 0.776309609413147, -1.3414250612258911, 0.6790306568145752, 1.7086297273635864, -1.9151976108551025, 3.3370375633239746, -0.7299097776412964, 0.3255453109741211, 0.7099735736846924, -0.27246174216270447, -0.08088086545467377, -0.4445531368255615, -1.4969868659973145, 0.3520047068595886, 1.8764687776565552, -2.9184789657592773, 0.6983161568641663, 0.6772281527519226, -1.2102407217025757, -0.012898351065814495, 1.8886560201644897, -0.6100001931190491, -0.29892995953559875, 0.037077464163303375, -0.047469817101955414, -0.3997809886932373, -1.4436451196670532, -0.30688443779945374, -0.6788085103034973, -0.8955913186073303, -0.2214880734682083, -0.8526495695114136, -0.48061075806617737, 0.9109363555908203, 0.5709143280982971, 0.7333378195762634, 1.8329602479934692, -0.949600875377655, 0.8056343197822571, -0.0540463961660862, -0.44611820578575134, -0.9569258093833923, 0.6076923608779907, 0.6449247002601624, -0.3348456919193268, -0.9700016379356384, 0.13420985639095306, 1.1972846984863281, -0.7441791892051697, 0.44061920046806335, -0.6369999647140503, -0.9555704593658447, 0.8725625872612, -0.8773057460784912, -0.3923793137073517, -0.9929635524749756, 0.5315732359886169, 0.7080272436141968, 0.6635101437568665, -0.993313729763031, 0.4856744706630707, -0.3945602774620056, -2.9051084518432617, 0.1429944634437561, -1.2186938524246216, 1.1765220165252686, -0.09649922698736191, -2.122277021408081, -0.5479053854942322, -0.6255528926849365, -0.05365603789687157, 0.010863943956792355, -0.4734572172164917, -1.1219781637191772, 0.5780410170555115, 1.0415030717849731, -0.13506299257278442, 1.437756896018982, -0.7660733461380005, -1.0074857473373413, -2.162761926651001, -0.8535007834434509, -0.045812319964170456, -0.23366838693618774], [-1.2374794483184814, -0.6638401746749878, -0.8656349778175354, 0.30807632207870483, 0.13859772682189941, 0.18305514752864838, 1.3529778718948364, 0.37821024656295776, 2.5664448738098145, 0.3583764135837555, 0.5055168271064758, 2.0222017765045166, 1.450245976448059, -0.5483577847480774, -0.23809154331684113, -0.3850628733634949, -1.4184969663619995, 0.8268629908561707, 0.1578267216682434, -2.3905720710754395, 0.48787572979927063, 0.6230184435844421, -1.2934991121292114, 0.5840632319450378, -1.7382266521453857, -0.49410349130630493, -0.8998570442199707, -0.5476253032684326, -0.22120361030101776, -0.8706920742988586, 0.45840802788734436, -1.2012741565704346, -0.22590042650699615, -0.26618489623069763, -0.08503760397434235, 0.250066876411438, 0.6322796940803528, 1.7349082231521606, 0.03392236679792404, 1.0891246795654297, -1.772955298423767, -0.11410940438508987, 1.2797800302505493, -0.37018883228302, 1.215083122253418, -0.8409888744354248, -0.22623024880886078, 0.37453407049179077, -1.5314933061599731, -1.2861865758895874, -0.4818291962146759, -0.43771958351135254, -0.21344906091690063, 0.6856626868247986, 0.30861249566078186, 2.0575525760650635, 0.16836921870708466, -0.8432275056838989, -0.6556547284126282, -2.050516128540039, 0.4842597246170044, 0.3023280203342438, 0.10874982178211212, -0.5819878578186035, -2.2504968643188477, -0.3533302843570709, -1.6300601959228516, 1.5366120338439941, -2.4495110511779785, 0.23974129557609558, -1.0978676080703735, -0.2819153964519501, 0.13170354068279266, -0.6555712819099426, -1.6776149272918701, -0.1158088743686676, -0.525274395942688, 0.3532300889492035, -1.5108022689819336, -0.4439356029033661, 0.9653874635696411, 1.5891135931015015, 0.21658451855182648, -0.2140147089958191, -0.9321781396865845, 0.4624456465244293, 0.8047897219657898, -0.2851516008377075, -0.6793206930160522, -1.313132882118225, -0.06242850795388222, -2.4021265506744385, 1.045225739479065, -0.9811868667602539, -0.16996827721595764, -0.5768144726753235, -1.6176502704620361, 0.5618425011634827, -0.4397110342979431, -1.2760077714920044, 0.29351571202278137, -0.2520706057548523, -0.8363614678382874, -0.12866464257240295, 0.3789858818054199, -0.9416449069976807, -0.12081751227378845, -1.4800511598587036, 0.4234435558319092, 0.6561716198921204, -0.29830026626586914, -0.4623628556728363, 1.4243862628936768, -0.9911802411079407, 0.1845787912607193, -0.576483964920044, -0.4967823326587677, -0.0530855692923069, 1.5604643821716309, -0.31604066491127014, 0.5110499858856201, 0.42754065990448, -1.8923571109771729, -0.05637281388044357, 0.4598531424999237, 0.836654007434845, -0.8009741902351379, -1.3840149641036987, -1.8080593347549438], [-0.8681480288505554, 1.458388328552246, 0.15165023505687714, -0.7952345609664917, 0.2998400330543518, 0.6698450446128845, -0.9815080165863037, 0.31108152866363525, 0.549288809299469, -0.5600747466087341, -1.0592291355133057, 0.7546976208686829, -0.23953019082546234, -0.13769763708114624, -0.6253036260604858, -0.2801573574542999, 0.1542641818523407, -0.13352592289447784, -0.3920990526676178, -0.03753294423222542, 0.45672404766082764, 2.690416097640991, 0.5752017498016357, 0.2185392826795578, 0.8564994931221008, 1.2509173154830933, 1.3375300168991089, -2.0632376670837402, -1.5941234827041626, -0.757785975933075, 0.45021170377731323, -0.3771991729736328, -0.9608026146888733, -0.6700757145881653, 0.9797195792198181, -1.276352882385254, -0.23140287399291992, 1.3081563711166382, 0.5854794979095459, -0.1253679096698761, -0.19638319313526154, 0.5812408328056335, -2.1210694313049316, 0.5291344523429871, 0.633569598197937, -0.319049596786499, -0.22437839210033417, -0.21801595389842987, 0.19879809021949768, -1.3403831720352173, -0.8170302510261536, -0.7195445895195007, -1.7865597009658813, 0.8390074968338013, -0.36782532930374146, -0.4226694405078888, -1.938438892364502, 0.4898250699043274, 0.5180255770683289, -1.9534618854522705, -0.7819063067436218, -0.2013433575630188, 1.0446910858154297, -0.5746314525604248, -0.026006484404206276, -1.431434988975525, -1.029250979423523, 1.2052106857299805, -1.5299654006958008, 0.9013144373893738, -1.840282917022705, -0.09478916972875595, -0.3020629286766052, 0.515196681022644, -0.4806016981601715, 1.71291184425354, -0.5048964023590088, -1.8036772012710571, 0.19699259102344513, -0.445986807346344, -1.5008533000946045, 0.018647363409399986, -2.2814722061157227, 1.4936745166778564, -0.5513138175010681, -2.2183096408843994, -2.2485146522521973, -0.18360713124275208, -1.6139857769012451, -0.570751428604126, 0.8999055027961731, -0.5073176026344299, -0.9121577143669128, -0.4774211347103119, -0.2418932169675827, -1.2122724056243896, -0.7219399213790894, -0.14136247336864471, 0.42460158467292786, -0.692524254322052, 0.7589114904403687, 1.898634910583496, -1.6403032541275024, 0.3628297746181488, -0.06111621856689453, 0.06492455303668976, 0.9098193645477295, -2.5880892276763916, -1.9277480840682983, -0.5593371391296387, 0.7840982675552368, -0.04172278568148613, 0.5914629697799683, 0.30784130096435547, 0.01208383496850729, -0.12065626680850983, -0.7087571024894714, -0.08433663845062256, -0.05705517530441284, -0.3155525326728821, 1.6144593954086304, -0.7695355415344238, -1.7623858451843262, 1.8681745529174805, -2.056020975112915, -1.2450664043426514, -1.0675177574157715, 0.7336583733558655, -0.3406221568584442], [-0.2618372440338135, 1.632072925567627, 0.8488761186599731, -0.4035053551197052, -2.650160789489746, -1.7412430047988892, 0.43398812413215637, -1.0169813632965088, -0.6755569577217102, 1.244066596031189, 1.1917307376861572, 1.777020812034607, -1.0341564416885376, -0.12010151892900467, 1.0909228324890137, -0.6900550723075867, 0.10972266644239426, 0.16490048170089722, 0.4620601534843445, -0.9975413084030151, -2.214073419570923, -0.9792447686195374, -0.11486253142356873, 0.09571617841720581, -1.247802495956421, -0.23233453929424286, -0.595227837562561, 0.09177863597869873, -1.1848185062408447, -0.7059837579727173, 1.277822732925415, -0.08580607175827026, 0.2339298576116562, -0.6464843153953552, -1.1225720643997192, -0.3718537390232086, 0.2442542314529419, -0.32351934909820557, 1.6141092777252197, -0.0502927228808403, 2.2680578231811523, -1.1730029582977295, 1.7759811878204346, -0.4776264429092407, -1.1199532747268677, 0.48952487111091614, -1.5836236476898193, 2.328216791152954, -1.2223602533340454, 0.5456968545913696, 2.040107488632202, -0.21913759410381317, 0.677461564540863, -0.06969063729047775, 0.5663011074066162, 0.8066994547843933, -0.36113402247428894, 0.2341192215681076, -0.7945765256881714, 0.3747199475765228, -0.435220330953598, 0.796531617641449, -0.5828443765640259, -1.1642476320266724, -0.46755388379096985, 0.9426811933517456, -0.7654129266738892, 0.8333050012588501, 0.03617014363408089, 0.5733775496482849, -1.4961237907409668, -0.3158186078071594, 0.8779171109199524, 1.0949310064315796, -0.2863827645778656, 0.27683359384536743, -0.4914133846759796, -2.2037808895111084, -0.9300486445426941, -0.3413226902484894, -1.0382198095321655, -0.2576300799846649, 0.6116149425506592, -0.7699970006942749, -2.1601293087005615, -0.6795796155929565, -0.3642873466014862, 1.275065302848816, -1.342037320137024, 0.27065470814704895, 0.10737843811511993, -0.2423725724220276, -1.763731837272644, -1.068367600440979, -0.8094484806060791, -0.36209771037101746, -0.3650946617126465, -0.7382575273513794, 0.8650830388069153, -0.46146151423454285, -0.3484659194946289, -0.8495764136314392, -1.6660387516021729, 1.962698221206665, 1.3610647916793823, -0.9032233357429504, 0.21903541684150696, 0.9387642741203308, 0.23931680619716644, -0.19738991558551788, 1.1510940790176392, 0.11646782606840134, -0.4006524085998535, 0.721233606338501, -0.07553502917289734, -0.9174954891204834, -0.3705235719680786, -0.5255448222160339, -0.8431521654129028, 0.8826168775558472, 0.9115107655525208, -1.2616727352142334, 0.2941545248031616, -0.17268556356430054, 1.6595003604888916, -0.061442725360393524, 1.1811821460723877, 1.6172627210617065, 1.0738976001739502], [1.5125290155410767, 0.8151815533638, 0.5878584980964661, -0.986718475818634, 0.24682265520095825, 0.8277019262313843, 0.38930198550224304, 0.8579431176185608, -0.048806022852659225, -0.8939329385757446, -0.5362053513526917, -0.021903524175286293, 0.8534327745437622, -0.8978006839752197, -0.686595618724823, -0.847213089466095, 0.9381204843521118, 1.6031180620193481, -0.5302940607070923, 0.2442595511674881, 1.4786778688430786, 1.0849360227584839, -1.2006583213806152, -0.6575515866279602, 1.1998118162155151, -1.0832141637802124, 0.0775061547756195, 1.321405291557312, 0.5904326438903809, -0.976256251335144, 1.2995247840881348, -0.6665079593658447, 1.2993786334991455, -0.0037239925004541874, 0.09602432698011398, -0.10045590996742249, -1.419664740562439, -0.4498952329158783, -0.10875096172094345, 0.18486922979354858, -1.777468204498291, 0.9574097394943237, 2.079453945159912, -0.22793516516685486, 0.650971531867981, 0.22205382585525513, 0.21353478729724884, 0.5200448036193848, -1.9554359912872314, -1.371392846107483, 0.1208135113120079, -0.6069481372833252, 0.34568992257118225, 0.797734260559082, -0.7844995260238647, 1.3376327753067017, -0.05346573516726494, 0.22143177688121796, -0.5574284195899963, -1.8527679443359375, 0.985736072063446, 1.4356756210327148, 0.04363875836133957, 0.2038978487253189, 1.2681808471679688, -0.15086247026920319, -0.3886304199695587, -2.0000481605529785, 0.9779491424560547, -0.8951624631881714, -0.7438153028488159, 1.2126185894012451, -1.027734637260437, 0.560137152671814, -1.3850988149642944, -0.2273816466331482, 0.4221465587615967, -0.3543022572994232, -0.9401610493659973, 0.6307520866394043, 0.3346371352672577, -1.1587393283843994, 0.28726860880851746, 0.7592515349388123, -0.48763349652290344, 0.43197011947631836, -0.07852113246917725, 2.2142493724823, 0.3883728086948395, -0.9060779809951782, -2.407301425933838, -0.5700646042823792, -0.961324155330658, 1.4643359184265137, -0.4573543667793274, 1.932604193687439, -0.702936053276062, -1.4265217781066895, 0.38393327593803406, 0.8788588643074036, -0.6087501049041748, 0.7228348851203918, 0.8098825216293335, -0.8728656768798828, -0.328749418258667, 1.1896159648895264, -0.4493088722229004, -0.6727569699287415, 1.6048115491867065, -1.0214972496032715, -0.3435342311859131, 0.9806432127952576, 0.28948912024497986, 2.5762789249420166, -0.1324625313282013, -0.06683658808469772, -0.925434410572052, 1.047118067741394, 0.1767347753047943, 0.117113396525383, 0.17838634550571442, 0.31582704186439514, -0.27975621819496155, 0.33832666277885437, 0.6493450403213501, 0.5013781785964966, -0.410191148519516, -1.777436375617981, -0.3006080985069275], [-0.8847779035568237, -1.3803712129592896, 0.315172404050827, -0.47596412897109985, -0.26009732484817505, 0.21054573357105255, -1.585961937904358, 1.9647752046585083, -0.16384632885456085, -1.2065542936325073, 1.6206274032592773, -0.08065758645534515, 0.19207461178302765, -0.7717487215995789, 0.953054666519165, 0.32359638810157776, -1.0872491598129272, -1.2403295040130615, -0.21499274671077728, -0.04506201297044754, 1.4188963174819946, -1.4098149538040161, 0.49271360039711, -0.17368848621845245, -1.0078911781311035, -0.11028555780649185, 0.29999807476997375, 0.24377170205116272, -0.10105956345796585, -2.1098201274871826, 0.7355823516845703, 0.4088596999645233, -0.9318972229957581, 1.736056923866272, 0.10512686520814896, -0.20026224851608276, -0.9977766275405884, 1.0046504735946655, -0.661601185798645, -0.36366215348243713, -0.3063221871852875, 1.1505568027496338, 0.413219153881073, 1.986840844154358, -0.5719968676567078, -0.22310194373130798, -1.0768399238586426, -0.9329614043235779, -1.8094655275344849, -0.24465540051460266, 0.9969501495361328, -1.3827208280563354, -0.15988828241825104, -0.5005776286125183, 0.3543797433376312, -0.11623860895633698, 0.12275638431310654, 0.44073596596717834, -0.816506028175354, -0.19351017475128174, 0.17760813236236572, -0.20514512062072754, 1.4582605361938477, -1.183774471282959, -0.9898490309715271, 1.0509519577026367, -1.2467005252838135, -1.9997727870941162, 0.46960967779159546, 1.3001327514648438, 0.3215520679950714, 1.4236345291137695, 1.1917542219161987, -1.4246282577514648, 1.1061705350875854, 1.0120673179626465, 0.4724559485912323, -1.330816626548767, 0.029112132266163826, -0.1823563277721405, 0.3327450454235077, 0.5664183497428894, -1.499755620956421, -0.7818575501441956, -2.4856526851654053, -0.40596359968185425, -0.33862683176994324, 1.8412514925003052, -0.7651994228363037, -0.8186543583869934, 1.071363925933838, -1.0766836404800415, -0.022234758362174034, 0.27135804295539856, -0.28465092182159424, 0.16156069934368134, 0.9665120244026184, -0.44651058316230774, 0.5782827138900757, 0.6597731113433838, -1.5286074876785278, 0.7740576267242432, -2.0606977939605713, 0.14168770611286163, -1.1805481910705566, 0.8189470767974854, -0.18495520949363708, 1.1863197088241577, 0.20179300010204315, 0.4782281219959259, 0.8047401905059814, 1.055755615234375, 0.3199310600757599, -0.4871123433113098, -0.5763792991638184, 0.09608462452888489, -0.6248999834060669, 0.1425115466117859, -1.1969711780548096, 1.007009744644165, 0.29698339104652405, 0.028846070170402527, -0.024152632802724838, 0.15214481949806213, 0.7798001170158386, 1.7330217361450195, -0.08337138593196869, -1.5237071514129639, -1.014954686164856], [-0.4089847505092621, -0.9968331456184387, -0.6592944264411926, -0.9948623776435852, -0.08516974002122879, 1.0217760801315308, 0.9859644770622253, -0.0251863282173872, 0.4801558256149292, -0.7078877687454224, 0.484561949968338, 0.673214852809906, -0.5582997798919678, 1.3164455890655518, -1.2328126430511475, 0.4442216455936432, 2.3805150985717773, -0.15930019319057465, 1.8395766019821167, -0.4009753167629242, 1.258548617362976, 0.5372745394706726, -0.05251447111368179, 0.7979691624641418, 1.2553868293762207, 0.3927558660507202, -1.058717966079712, -0.3782613277435303, -0.8371590971946716, -0.2169896513223648, 2.2161221504211426, -0.4104984700679779, -1.0148593187332153, -1.1954734325408936, 1.205704927444458, 0.45744839310646057, 0.33284619450569153, 0.14640256762504578, -0.10360105335712433, 0.9447033405303955, -0.6027219295501709, 0.8324581980705261, -0.30402523279190063, -1.2812939882278442, -0.410146027803421, 0.32991909980773926, 0.1495165079832077, 1.5397239923477173, 0.6064897179603577, 0.012104673311114311, 1.022870659828186, -0.046732064336538315, 0.9474762082099915, 0.1020079180598259, -0.3960144519805908, -0.2296891063451767, -0.0871768370270729, 0.930084764957428, 0.16077075898647308, -0.35692450404167175, 0.6022354364395142, 1.6500194072723389, 0.4878964126110077, 1.0210446119308472, 1.3155468702316284, 0.6624082922935486, 0.28090348839759827, -0.5044419169425964, 0.5865217447280884, -0.7581791877746582, -0.3079778850078583, 1.1200634241104126, -1.7988985776901245, -0.4302766025066376, -0.7875027060508728, -0.002603425644338131, -0.8094355463981628, -0.009688863530755043, 0.6126683354377747, 0.5112422704696655, 1.9382706880569458, -0.42845869064331055, 1.48646080493927, -0.13395431637763977, -0.07029475271701813, -1.1851866245269775, -0.7601690888404846, -2.426176071166992, 0.888214111328125, -1.095950961112976, 1.7499867677688599, -1.119666576385498, 0.13211321830749512, 1.2259953022003174, -0.12198396772146225, -0.02572599984705448, -0.8706241250038147, -0.0655769631266594, -1.1421141624450684, -1.2031610012054443, 0.7480001449584961, -2.0010464191436768, 0.18384644389152527, 0.7709935307502747, 1.1171172857284546, -0.8952778577804565, -1.1404165029525757, 0.020224109292030334, -0.16743861138820648, -0.5946806073188782, -1.2921804189682007, 0.017466997727751732, -0.7533605098724365, 0.0015984808560460806, -0.00036395739880390465, -1.322964072227478, -1.9086376428604126, -0.5790805816650391, 0.015719328075647354, -0.669274628162384, 0.7780042886734009, -0.5440341830253601, 0.09767478704452515, -0.04338167980313301, -0.7839183807373047, 1.1468805074691772, -0.6140841245651245, 1.1792984008789062, 0.13225790858268738], [-1.2328280210494995, 0.20231053233146667, 0.29488545656204224, 2.7345330715179443, -0.981733500957489, -0.5919595956802368, 1.2645666599273682, 1.2981442213058472, -0.9451639652252197, 0.11502605676651001, -2.100555181503296, -0.2781442403793335, -0.47002312541007996, 1.1010419130325317, -0.6184300184249878, 0.9895139932632446, 0.30570077896118164, 0.18186864256858826, -1.6703864336013794, 0.12123148143291473, 0.1928451955318451, 0.3535204529762268, 0.7008676528930664, -0.7706788182258606, 1.4065853357315063, -0.19349391758441925, -0.8136246204376221, 0.7646204233169556, 0.007831609807908535, 0.3502500355243683, -0.10374956578016281, -0.4463139772415161, -0.47244349122047424, -0.030205683782696724, 0.4727250039577484, 0.05588744580745697, 0.8061082363128662, 0.03698026388883591, 0.0760434940457344, 0.5468323230743408, -0.058265797793865204, -1.787714958190918, 0.4409167468547821, -1.4142481088638306, -0.9383708834648132, 1.4578471183776855, -0.6491478681564331, 0.32150161266326904, -0.12868736684322357, 0.9369297027587891, 1.322219967842102, 0.7103819251060486, 0.050671450793743134, 0.2791185677051544, -0.01074147131294012, -0.20894093811511993, 2.3913097381591797, 0.7222374677658081, -0.18091657757759094, 0.30349504947662354, -0.763274610042572, -0.04513760283589363, -0.2973482012748718, 0.38718903064727783, 1.5408306121826172, 1.0679045915603638, -1.2288118600845337, -0.23770730197429657, 0.07892221957445145, -0.27422580122947693, -0.8648697733879089, -0.8846844434738159, -0.8172982335090637, -1.1868155002593994, 0.3184548616409302, 0.5604128241539001, 0.6876909732818604, 1.7982722520828247, -1.2762234210968018, -1.739557147026062, -0.4561827480792999, 0.6948206424713135, -0.830049991607666, 0.22550983726978302, -0.3707727789878845, -0.7919231653213501, 1.4757589101791382, -0.680164635181427, -0.8988108038902283, 0.9024344086647034, 1.3587895631790161, 1.4013038873672485, 0.07555889338254929, -1.1564487218856812, 0.17966027557849884, 1.4242967367172241, -0.903421938419342, -0.7330157160758972, -0.33127495646476746, 0.4868834912776947, 1.2361959218978882, 0.2913420796394348, -0.23805435001850128, -0.3005983531475067, 0.577509880065918, 1.3862320184707642, -0.10308220982551575, -1.3194636106491089, 0.3762176036834717, -0.13457274436950684, -2.3217732906341553, 0.6037744283676147, -0.28072643280029297, 0.8228981494903564, -1.2748380899429321, 0.24544261395931244, -0.26981985569000244, 0.43826237320899963, 1.121285080909729, 0.012811604887247086, 0.6019165515899658, 0.8430293202400208, -0.1014300286769867, -0.7742825150489807, 0.6039206385612488, 1.4804747104644775, -0.22941136360168457, -0.022802207618951797, 0.4584987163543701], [1.7536615133285522, -0.3942997455596924, -0.3159889578819275, -1.7031227350234985, -1.0929718017578125, 0.15880970656871796, -2.0020363330841064, -1.795375943183899, 0.8059285879135132, -1.042971134185791, -0.5119016170501709, 1.3901314735412598, 0.1761382818222046, -0.20198875665664673, 0.38144198060035706, -0.4398142397403717, -0.9479273557662964, 0.9118773937225342, -0.6861045360565186, -1.3068203926086426, -0.16741636395454407, 0.8659208416938782, -0.09290847927331924, 0.9152730107307434, 1.5067594051361084, 0.13661441206932068, 0.581899881362915, 0.13322964310646057, 1.0187058448791504, 0.006883140187710524, 0.5218297243118286, -1.6761530637741089, 0.7566165924072266, 1.6382954120635986, 0.16976985335350037, -0.4030432403087616, -1.4042041301727295, -0.782931387424469, -0.5919373035430908, -1.7559572458267212, 0.37366431951522827, -0.5431403517723083, -0.23840923607349396, -0.7271428108215332, -0.7697902917861938, 0.39773204922676086, 0.9391324520111084, -1.307910680770874, 1.2019774913787842, 1.568120002746582, -0.6113309264183044, 0.25403204560279846, -0.19387799501419067, 0.3246479332447052, 0.07499080896377563, 1.9354610443115234, -0.25582006573677063, -0.056538160890340805, 1.515178918838501, 1.39644193649292, -0.2522636353969574, 0.8513137698173523, 0.6033897995948792, -1.083053708076477, -0.7383138537406921, 1.523688793182373, -0.1877007633447647, 0.45569029450416565, 0.8099573254585266, -0.6052442193031311, -1.8251036405563354, -0.9794929623603821, -0.3525174558162689, 1.0012179613113403, 1.3057782649993896, 0.77451092004776, 0.07741604745388031, 1.4338159561157227, 0.9946125149726868, -1.291223406791687, 1.5682059526443481, -0.8531302213668823, 1.1230745315551758, 1.3677388429641724, 0.6477591395378113, 0.3457302451133728, 0.033167868852615356, -1.2500661611557007, 0.49480727314949036, 1.2168481349945068, 0.9474106431007385, 0.8393383026123047, -0.11177054047584534, -0.9440575242042542, -1.3088089227676392, -0.5519751310348511, 0.3102079927921295, -0.6949228644371033, -0.2870903015136719, -0.9344101548194885, 1.9100524187088013, -0.1299489438533783, -0.18528932332992554, -0.9285903573036194, 0.8398998379707336, -1.020017385482788, 0.4242144525051117, -0.20043328404426575, -1.518257737159729, -0.17074204981327057, 0.0039422838017344475, -0.13482187688350677, -1.5645841360092163, 0.7295844554901123, -0.5320609211921692, -0.4463964104652405, 0.4670613706111908, 0.30517029762268066, 1.5520408153533936, 0.2096748948097229, -0.3989112377166748, -0.4153450131416321, -1.2293522357940674, -1.0868303775787354, -1.3128042221069336, 0.46302860975265503, 0.22743764519691467, -2.170117139816284, 1.2803510427474976], [2.129594564437866, -0.1670927256345749, 0.1152956411242485, -1.8778727054595947, -0.8476690053939819, 0.27080395817756653, -0.3951910138130188, 0.22920319437980652, -0.693444013595581, 0.06092073395848274, -1.0500783920288086, -0.5130996108055115, 0.2773987054824829, 0.11175230890512466, -1.4419713020324707, -1.1901980638504028, -1.1036821603775024, -0.8318265676498413, -0.43319767713546753, 0.213373601436615, -0.9159802198410034, 0.0012496240669861436, -1.1868622303009033, -0.14384590089321136, 0.425508975982666, -1.0165444612503052, -0.11990769952535629, -0.7479849457740784, -0.2527537941932678, 0.2535250186920166, -0.5673108696937561, 1.8866183757781982, -0.16851788759231567, -0.14935070276260376, 0.6448985934257507, 0.2360735386610031, -0.2100873440504074, -1.5094224214553833, 0.4666683077812195, 0.9789540767669678, -0.14465564489364624, 0.17750979959964752, -0.9400202035903931, 1.648061752319336, 0.24287831783294678, -0.7294510006904602, -2.2502646446228027, 1.5270918607711792, 1.3423899412155151, -0.7722551226615906, 0.8669283390045166, 1.8506395816802979, -2.045848846435547, -0.4168551564216614, -1.1210525035858154, -1.9973533153533936, -0.9867731332778931, 0.6838785409927368, 0.6027347445487976, -0.1780543029308319, -1.2519947290420532, 1.4472225904464722, -0.6904333829879761, -0.9381285309791565, 0.11630526185035706, -0.9878177642822266, -0.17687183618545532, 2.1862001419067383, 0.9780945181846619, -1.0090813636779785, -0.09873715043067932, 0.3287866413593292, 0.23453272879123688, 0.3343462646007538, 4.169489860534668, 1.6704777479171753, 0.4682534337043762, 0.09882374852895737, -0.697189211845398, 0.30552008748054504, -1.5421520471572876, -0.9208494424819946, 0.520776629447937, -0.3087122440338135, 1.2413897514343262, -2.147966146469116, 1.286731481552124, 1.4330933094024658, -1.1140763759613037, 1.353920340538025, -0.2862747311592102, -0.8506174087524414, -0.3456747531890869, -1.0787814855575562, -0.7930487394332886, -0.02634875662624836, 1.5202953815460205, -0.44270262122154236, -1.137420892715454, 0.13403446972370148, -0.5456852316856384, 1.2183446884155273, -0.9025622010231018, 1.5044738054275513, -1.2615996599197388, -0.016872815787792206, -1.6312224864959717, -0.7450335025787354, -0.4026128947734833, 0.8680811524391174, -2.5194015502929688, 0.3745495080947876, 0.6735071539878845, -2.1144375801086426, -0.5268791913986206, -0.42960017919540405, 0.09491527080535889, 0.6078527569770813, 0.8189107179641724, 0.8324666619300842, 0.6210314631462097, -1.7340247631072998, 0.9713425636291504, 0.011167019605636597, -0.07451902329921722, 0.37225815653800964, 0.6060327887535095, 0.5328134298324585, -1.4872938394546509], [0.5887776017189026, -1.3309803009033203, 0.4644910991191864, -0.4943963289260864, -0.2515365481376648, -0.6793237924575806, -0.1620251089334488, -1.8713891506195068, 1.1862400770187378, 0.7880027890205383, 0.25759217143058777, -0.543317437171936, -0.4167744219303131, -2.0666627883911133, -2.3069045543670654, 0.02646494284272194, 0.5558791756629944, -1.8177157640457153, 0.39437636733055115, 0.46320921182632446, -0.4422200322151184, -0.41693931818008423, -0.30770501494407654, -0.12794651091098785, 0.8041340112686157, -1.6623945236206055, -1.9057632684707642, -1.8940703868865967, 1.216407060623169, -0.42942357063293457, -0.31441205739974976, 1.9645549058914185, -0.8786423206329346, 0.6695986986160278, 0.4858034551143646, 1.0531400442123413, -0.11738916486501694, -0.1689286231994629, 0.7411261796951294, -0.6394475698471069, 0.6627837419509888, -0.3615064322948456, 0.5479783415794373, 0.3640206456184387, 1.6331607103347778, 0.6915522813796997, -0.10644165426492691, 1.2177342176437378, -2.325183629989624, -1.2671730518341064, -1.6838518381118774, 0.6819231510162354, -0.36913180351257324, -0.13253583014011383, -0.5206851959228516, -0.6461988687515259, 0.7004769444465637, 1.39834725856781, 0.16903461515903473, 0.274580180644989, 0.9899938702583313, 0.9808362722396851, -1.187725305557251, -0.5313603281974792, -0.5159241557121277, 0.8664969801902771, 0.5631725192070007, 0.7713561058044434, 0.12548361718654633, -0.7212104797363281, 2.248018503189087, -1.160936951637268, 1.3531607389450073, -0.025564802810549736, 1.1200274229049683, -0.8015799522399902, -0.7884875535964966, 1.006973385810852, 1.5025194883346558, 1.6051989793777466, 1.2501609325408936, 0.07282751798629761, 2.442272186279297, -0.49901431798934937, 0.15088264644145966, -0.5639398694038391, 1.1756863594055176, -0.6200929880142212, 0.5356390476226807, 1.552372932434082, -1.6856282949447632, 0.5527762174606323, 0.027571171522140503, 0.4634127914905548, -0.6969861388206482, -0.15419204533100128, 0.9523064494132996, 0.11727708578109741, 0.012073756195604801, 0.11434407532215118, 0.7947618961334229, 0.7607219219207764, 1.3407068252563477, -1.7905720472335815, 0.8558802604675293, 0.6021541357040405, 0.661449670791626, 0.19641968607902527, -0.5236148834228516, -0.223959282040596, -0.5059393048286438, -0.13754090666770935, -1.5249780416488647, 0.9921809434890747, -0.518764078617096, 0.7207422852516174, 0.4751896560192108, 1.0334534645080566, 1.0185179710388184, 0.39337408542633057, -0.6273007988929749, 0.018619762733578682, 0.06481414288282394, -0.6065183281898499, -1.2354176044464111, -1.0245544910430908, -2.07757830619812, 0.5010901689529419, -0.15825018286705017], [-0.5938064455986023, 0.06375004351139069, 1.1895333528518677, -0.6548145413398743, -0.015388463623821735, -1.1920500993728638, 1.08878493309021, -0.2747955024242401, -0.6692409515380859, -0.40214791893959045, 0.6440131068229675, 0.6547489762306213, 0.6643983125686646, 0.45676225423812866, -1.4553587436676025, -0.9816555976867676, 0.30201342701911926, 3.061391592025757, 1.0419634580612183, 0.34329521656036377, 0.2802658677101135, -1.2825640439987183, 1.651963472366333, 1.0928832292556763, 0.4477524757385254, 0.016688361763954163, -0.47072699666023254, -0.34213414788246155, -0.229363352060318, 1.941285490989685, -0.7826374173164368, 0.9860022664070129, 0.09034042805433273, 1.969262957572937, -0.1347958892583847, -0.9523227214813232, -0.17272815108299255, 1.251691460609436, 0.8307031989097595, 0.3487309217453003, -0.05278075858950615, 0.5371997356414795, -0.3689563274383545, -1.090254783630371, 0.6470710039138794, 0.179458886384964, 0.16767150163650513, -0.5423449277877808, -0.10954605042934418, 0.015118720009922981, -1.4422687292099, -1.788339376449585, -1.4285920858383179, 0.6349011659622192, -0.39163443446159363, -0.49364158511161804, 0.3340715169906616, 0.7302714586257935, -0.38843366503715515, -1.983559489250183, -1.223836898803711, 1.638263463973999, -0.6333625912666321, 0.08077710121870041, -1.2764455080032349, 0.511665403842926, 0.8248648047447205, -1.719979166984558, 0.8439149260520935, -0.2566917836666107, -0.5625073909759521, -0.02546468935906887, -0.3267395794391632, 2.1727287769317627, -0.4099554121494293, 0.6038903594017029, 0.16401636600494385, 1.1095408201217651, -1.3444534540176392, 0.9873523116111755, -0.8268920183181763, -0.14843469858169556, 1.3487679958343506, -0.6196773052215576, 0.43002521991729736, -0.7076472640037537, -0.01473519578576088, 0.7817381024360657, -0.5211833715438843, -0.6646935343742371, -1.2474387884140015, -1.517535924911499, -0.7031643986701965, 0.9837333559989929, 0.1400400996208191, 0.7084944248199463, -0.9375806450843811, -0.47366973757743835, 0.03943396732211113, 0.02899743989109993, -0.6748231649398804, 1.1156812906265259, -0.10370749980211258, -1.6762912273406982, -0.9983558058738708, -0.25687095522880554, -0.4183811843395233, 0.03978291153907776, -0.23407435417175293, 1.1347665786743164, -0.0737612247467041, -0.15277805924415588, 1.7846300601959229, 0.8134029507637024, 0.17371121048927307, 0.045647166669368744, -0.39797982573509216, -0.2309868335723877, 0.10603132843971252, 0.6467716693878174, -0.33505767583847046, -1.3608531951904297, -0.008290793746709824, -0.7851359844207764, 1.6742035150527954, 1.9233636856079102, -2.309469699859619, 0.14222241938114166, -0.8806430101394653], [0.5980390310287476, -0.9366763830184937, -0.15197418630123138, -0.3146268129348755, 0.8014872074127197, -0.692755401134491, 2.1163454055786133, 1.811638355255127, 0.019607244059443474, 0.4277961254119873, -0.5808988213539124, 0.7740796208381653, -1.1122474670410156, -1.8567473888397217, 1.2620232105255127, 0.06323479861021042, -0.31578198075294495, 0.15054741501808167, 2.4527747631073, 0.33146989345550537, -0.7414396405220032, 0.29501086473464966, -0.10114660859107971, 0.10363948345184326, -0.11638165265321732, 2.0711705684661865, -0.7706514000892639, -0.8404441475868225, 1.3163747787475586, 1.13288414478302, 0.12463944405317307, -0.2890952527523041, 0.8055064082145691, -0.3951393663883209, 0.054346714168787, 0.6209154725074768, -0.3092799782752991, -0.3147374987602234, 2.4187204837799072, 0.5458448529243469, -1.7342119216918945, -0.0021387210581451654, -1.2644788026809692, 0.698609471321106, -0.3449791967868805, 0.5631956458091736, -0.5502186417579651, -1.2054567337036133, -1.3284828662872314, 2.3479161262512207, 0.6504652500152588, -0.007828126661479473, -1.9051278829574585, -1.007429838180542, -2.1977150440216064, -2.8312149047851562, -0.9179394841194153, 1.3412261009216309, -0.6417028307914734, -1.6201077699661255, 0.7479501962661743, 0.661381185054779, -0.06569909304380417, 0.9334931969642639, 1.5062021017074585, 0.8700358271598816, 1.9344040155410767, 0.08455084264278412, 0.8549585342407227, 2.7069082260131836, 1.6625730991363525, 0.24314863979816437, -1.544398307800293, -0.12210139632225037, -1.7520496845245361, -1.7663918733596802, -0.18720372021198273, 0.9851579070091248, 2.0866928100585938, 0.7070962190628052, 0.8517547249794006, -1.7105623483657837, 0.4908112585544586, -0.6116092205047607, 0.8856306672096252, 1.4491264820098877, 1.1387052536010742, 0.1800002157688141, -1.251220703125, -1.1051146984100342, -1.0553295612335205, 0.04604685306549072, -0.6715294718742371, 1.711637258529663, -1.1766932010650635, 0.9664706587791443, 0.039708446711301804, 0.04458450898528099, -0.12410096824169159, -1.7322559356689453, -0.17065823078155518, -0.30702584981918335, 0.8808484077453613, -0.5489663481712341, 0.7489160895347595, -1.880569577217102, -0.3125418424606323, -2.398399829864502, 0.5286271572113037, 1.6128416061401367, -1.6458412408828735, -2.7950527667999268, -1.1700128316879272, -2.2755959033966064, 1.279484510421753, 0.6245422959327698, -0.04848717898130417, 1.0049470663070679, 0.9155240058898926, 0.9521332383155823, -1.2611260414123535, 0.8690281510353088, 0.684329628944397, 0.26080992817878723, 2.672738790512085, -0.014660101383924484, -0.31912103295326233, -1.2869328260421753, -0.5855193138122559], [-0.013752943836152554, -0.31636151671409607, 0.5365031957626343, 1.9215110540390015, -1.2086341381072998, 0.9716747999191284, -0.6770706176757812, -0.5829454064369202, -1.2788201570510864, 0.44885095953941345, 1.10115385055542, -0.031049327924847603, -1.478129506111145, -1.8833777904510498, 0.5942071080207825, -0.35413408279418945, -0.5074714422225952, 1.1365633010864258, -1.6458587646484375, 0.5984022617340088, -0.1371922641992569, -0.5845732092857361, -0.6204291582107544, -1.2548075914382935, -1.2523561716079712, 1.5340499877929688, 0.759101390838623, -1.3794161081314087, 1.1157026290893555, 0.9831988215446472, -2.00058650970459, 0.7970405220985413, 0.564373254776001, 0.5351877212524414, 1.807997465133667, 0.1753145456314087, 0.24717830121517181, -0.35810402035713196, 0.7796693444252014, 1.0750566720962524, 0.7924578189849854, -0.14930346608161926, 0.7375712990760803, -0.08482593297958374, -0.9183124303817749, 1.0853850841522217, 0.4543263614177704, 0.9333458542823792, 0.6475875377655029, -0.09576834738254547, 2.556488037109375, -0.04852241650223732, 1.6725070476531982, 0.7974081635475159, 1.3042658567428589, 1.0266785621643066, 1.2775081396102905, -0.50239098072052, -0.286850243806839, -1.7361928224563599, 0.1642908751964569, 0.4368579387664795, -0.1843571960926056, -0.3279793858528137, -1.3440001010894775, 0.16653293371200562, -0.29631921648979187, -0.6247058510780334, -1.055673599243164, -0.09885109215974808, 1.0186187028884888, 0.40233689546585083, 0.42521315813064575, 1.1077854633331299, -1.1695817708969116, -0.5153084993362427, -0.6050246953964233, -1.545073390007019, -0.2929990887641907, 0.5071977972984314, -0.5968331098556519, 0.266804963350296, -0.3515406847000122, 1.552632212638855, 0.04899932071566582, 0.08523519337177277, 0.18459711968898773, 0.1889803409576416, -0.6677469611167908, -0.022393101826310158, 0.6233908534049988, 0.05661061778664589, 0.1633213609457016, -0.7447630167007446, -0.768295168876648, -0.6404476165771484, 0.04280822351574898, 0.25808843970298767, 0.35035577416419983, -0.7874724268913269, -0.5774948596954346, 0.7891122698783875, 0.20315873622894287, 0.4796190559864044, -1.0521795749664307, 0.44747111201286316, -0.5508259534835815, 1.8967939615249634, -0.8200365900993347, -1.2071651220321655, -0.34247133135795593, 0.11116080731153488, -1.2426183223724365, -0.9253512620925903, 0.7619010806083679, 0.043657369911670685, -1.5561882257461548, 0.006132629234343767, 1.6107853651046753, 0.5948216319084167, -0.31411558389663696, -0.855512261390686, -0.9559689164161682, 0.49463197588920593, -3.227649450302124, -0.06261616945266724, -0.43677300214767456, 0.272098183631897, 0.49455100297927856], [0.42902764678001404, 0.0666067823767662, 1.321640133857727, -0.8345532417297363, -0.9274377822875977, -1.0064424276351929, -0.19150657951831818, 0.7314916253089905, 0.3523792326450348, -1.5769729614257812, -0.8173723816871643, -2.0563454627990723, 0.6120815277099609, -0.20410121977329254, 1.6631224155426025, -0.540526270866394, 0.2283022552728653, -2.2670106887817383, 0.4199174642562866, 1.819041132926941, 1.1470897197723389, -1.7775908708572388, -0.14404217898845673, 0.30533647537231445, -1.0985808372497559, -0.24345272779464722, 0.13029558956623077, -0.9330301880836487, -0.015333207324147224, -0.40424254536628723, -1.1504621505737305, 1.9745053052902222, -1.4934395551681519, -0.7253015041351318, 1.580275297164917, -0.08314966410398483, 1.9533658027648926, 0.7939225435256958, 1.0577876567840576, 1.3167786598205566, -0.8783447742462158, -0.4460078775882721, -1.655454158782959, -0.4211673140525818, -1.5933634042739868, 0.3556203246116638, -0.4576835334300995, 0.08701310306787491, -0.12041585892438889, 0.7182859182357788, -0.829901397228241, 1.6264780759811401, -0.33810484409332275, 0.8371514678001404, -0.29101377725601196, 0.3584127128124237, -0.37207546830177307, -0.6141443848609924, 0.6921908855438232, -1.2639020681381226, 0.2765093743801117, 1.5631530284881592, 0.5015438795089722, -0.6392320990562439, -1.4675068855285645, 1.776186466217041, 1.1222786903381348, -1.1313420534133911, -0.6728023886680603, -0.5801059007644653, 0.7188825011253357, -0.2697875499725342, 0.5584064722061157, -1.5398211479187012, 1.4379920959472656, 0.5717355012893677, -0.9652780294418335, -1.319563627243042, -0.49353888630867004, 1.4237955808639526, 0.4640643298625946, 1.7792372703552246, 0.9884020686149597, 0.49987292289733887, -0.2782467007637024, 0.2394060641527176, -0.3454461097717285, 0.2686898112297058, 0.17602987587451935, -0.041892584413290024, -0.5704346895217896, -0.3234378397464752, -0.13949954509735107, 0.11544910818338394, -0.8205291628837585, -0.04716166853904724, 0.8120737671852112, 2.1508805751800537, 0.092627614736557, -1.5331249237060547, 0.5990039110183716, -0.09492874145507812, -0.4375101327896118, -0.7834271788597107, -1.8029215335845947, -0.5729186534881592, -0.8037787675857544, 0.15091902017593384, 0.9648725390434265, 0.08982183039188385, 0.498807817697525, 0.5050608515739441, -1.7572269439697266, -0.2812454104423523, -0.39454367756843567, -0.36772647500038147, 0.24838703870773315, 0.30257850885391235, -0.1997804492712021, -0.16921654343605042, 0.22200694680213928, 0.8423899412155151, -0.18331032991409302, -0.6074070930480957, -0.06247347593307495, -0.1264224350452423, -1.2656220197677612, 1.4876375198364258, -0.9143903255462646], [-0.9996699690818787, -0.6042810082435608, -0.06708002090454102, -0.1442939043045044, 1.8919858932495117, -1.269133448600769, -0.6867087483406067, 0.4699936509132385, 0.34458446502685547, 0.11017999798059464, 1.0170730352401733, -0.4707024097442627, -1.0816129446029663, -0.5219457745552063, -1.568806529045105, 1.0950456857681274, -1.5026934146881104, 0.776606559753418, 0.13032811880111694, -1.0278282165527344, 0.09575852751731873, -0.13306103646755219, -2.258145809173584, 1.2872895002365112, -1.3599544763565063, 1.8513656854629517, -0.008012419566512108, -0.12980154156684875, 0.11371307820081711, 0.14145587384700775, -1.1977109909057617, -0.5895703434944153, 0.25679174065589905, 2.668060302734375, 1.2822449207305908, 0.5194385051727295, -2.8362057209014893, 1.9865281581878662, 0.3196348249912262, -0.07109808176755905, 1.4208428859710693, 0.025589898228645325, -0.47727110981941223, 0.7821575403213501, -1.3395024538040161, -0.4269489347934723, 2.376810312271118, 2.1298606395721436, 0.13825444877147675, 0.2765830457210541, 1.1612548828125, -0.6887232661247253, -0.6042652726173401, -0.933509349822998, 1.7309284210205078, 0.6391205787658691, -0.2610012888908386, -0.7248069047927856, 1.271885871887207, -0.2390449047088623, -0.2810109257698059, -0.016235074028372765, -0.5513750314712524, 0.38563111424446106, 0.31352052092552185, 0.7607717514038086, -0.19939973950386047, -0.2863817811012268, -0.4976305365562439, -0.33985814452171326, -0.07514222711324692, 0.19797688722610474, -0.23216910660266876, 0.3311890661716461, -0.07756689935922623, 1.759533166885376, 0.43334734439849854, -1.803866982460022, -0.20617008209228516, 0.6250858902931213, 0.051647499203681946, -0.15417598187923431, -1.2401301860809326, 1.408522129058838, -0.2963796854019165, 0.3814232647418976, 1.6301813125610352, 0.15347707271575928, 1.4265186786651611, -1.7939152717590332, 0.509965181350708, 0.5050508975982666, 0.37555304169654846, -0.4959191679954529, 1.3163070678710938, 0.4164040684700012, 0.2877795696258545, 0.7122474908828735, -0.689850926399231, 2.08233380317688, -0.6250314116477966, -0.9883490204811096, -0.3131609559059143, -0.5593416094779968, 1.330949068069458, 0.0040701329708099365, -0.4800621271133423, -0.4266764223575592, 0.6875944137573242, 1.4488612413406372, 0.796699583530426, 1.214774250984192, -1.3704559803009033, 1.4012243747711182, 1.5815080404281616, 1.5806668996810913, -0.9364981651306152, -0.0006926881615072489, 0.24597972631454468, 0.06569009274244308, -0.3000395894050598, 0.6421961188316345, -0.220119446516037, 0.4906056523323059, -1.4859399795532227, -0.332893431186676, -0.6136547327041626, -0.5230585336685181, -0.11489175260066986], [0.5229508280754089, 0.20607870817184448, 1.632295846939087, -0.01061298418790102, -0.1698482632637024, -1.1237883567810059, 0.6412460207939148, -0.18217948079109192, -2.3016791343688965, -1.5229939222335815, -0.5906599760055542, -0.24209780991077423, 2.4344286918640137, -0.17132720351219177, 0.4143643379211426, 0.8572655320167542, 2.167415142059326, 1.348364233970642, 1.879035234451294, 0.5063832402229309, -0.6135500073432922, 0.6711168885231018, -0.6898134350776672, 0.11959416419267654, 0.9930793046951294, -0.2340187281370163, 0.32628005743026733, 2.121211290359497, -0.7896909713745117, 0.2400938719511032, 0.610925555229187, 0.05556511506438255, 0.4656694829463959, 0.3914220333099365, -0.4017794728279114, 0.28460395336151123, 1.1334244012832642, -0.6018195748329163, 1.9506944417953491, -1.1723219156265259, -1.9061723947525024, 0.6402704119682312, 1.1346158981323242, -0.1815558820962906, 1.2377556562423706, 0.5302443504333496, -0.9398218393325806, -0.5692235231399536, -0.26578518748283386, 0.5057514309883118, -0.9853824973106384, -0.3803572356700897, -0.37486395239830017, -0.5098810791969299, 1.410383701324463, 0.0935446098446846, -1.6703722476959229, -1.8441684246063232, 0.034401990473270416, -0.7107949256896973, 0.33765366673469543, -0.1810912936925888, 1.3837677240371704, -0.5185084342956543, -0.0010130074806511402, 0.2867995500564575, -1.9958969354629517, -0.8753642439842224, -1.874056100845337, 0.4742783010005951, -0.2516702115535736, 0.33869054913520813, -1.4478663206100464, -0.6842451691627502, 1.083578109741211, -0.42233049869537354, 0.03210330754518509, -0.5843626260757446, 0.8031317591667175, -0.6499139070510864, -0.8992387056350708, 1.255037784576416, 0.08758298307657242, -1.2827035188674927, 1.405112862586975, 1.4628187417984009, -0.3611128330230713, -1.6100972890853882, -0.018285652622580528, 1.9727216958999634, -0.1316678822040558, -1.9723403453826904, -0.8575630187988281, 0.9605194330215454, -0.6405436396598816, -1.4659125804901123, -0.418640673160553, -1.622153878211975, -0.046654701232910156, 0.7587475180625916, -0.51775723695755, 1.1625373363494873, -0.1035265251994133, -0.21498563885688782, 0.28879979252815247, 0.9951319098472595, -0.5207895636558533, -0.2617364823818207, 0.137094184756279, -0.8292124271392822, 0.43545982241630554, 0.5281667709350586, -0.05138781666755676, -0.5868920683860779, -0.2995539605617523, 0.09528957307338715, -1.271189570426941, 0.11306054890155792, 1.0239362716674805, 0.05499247834086418, -0.3945751488208771, 0.06725210696458817, 0.2242487668991089, 0.4904008209705353, -1.1584328413009644, -0.3156836926937103, -0.6869442462921143, -0.843619167804718, -0.45878398418426514], [1.210785150527954, 0.9427365660667419, -0.06987445801496506, -1.3326427936553955, 0.5679208040237427, -1.8665727376937866, -0.8518700003623962, -0.6281789541244507, -0.1139979138970375, 0.4989427924156189, -0.21562841534614563, -0.4882383644580841, 0.3467951714992523, -0.7289330363273621, -0.3134273886680603, 0.2559368312358856, -0.49671995639801025, 0.531348466873169, -0.014652199111878872, 0.24210329353809357, -0.08900251239538193, 0.31364354491233826, -2.7868564128875732, -0.19429099559783936, 0.32058125734329224, 0.5303525924682617, 1.927720546722412, 0.4035588502883911, 0.00543942442163825, 1.0939494371414185, -1.0315256118774414, 0.8039366602897644, 1.3808859586715698, 1.5115594863891602, -1.3258320093154907, -0.9255833029747009, -0.14402782917022705, -1.6404695510864258, 0.3145774304866791, 0.25275012850761414, -0.1875486969947815, -1.245612621307373, -0.5152125954627991, 0.22056280076503754, -0.8295974731445312, -0.34154772758483887, 1.1714954376220703, -1.945548415184021, 0.6353188157081604, -0.5967801809310913, 0.33659616112709045, 0.4435134828090668, 1.7331563234329224, 0.44486162066459656, -1.4341399669647217, 0.7676765322685242, 1.040282964706421, 0.749801516532898, -0.7054285407066345, 0.3468315303325653, 0.6206989288330078, 0.2708916962146759, 1.0532066822052002, 0.9044101238250732, 0.6008604168891907, 0.12895680963993073, 0.5272008180618286, 1.2626867294311523, 0.1654556691646576, 0.28607627749443054, 0.07823372632265091, -0.9013131260871887, -1.0518708229064941, -0.3904719054698944, -2.382005453109741, 0.6566939353942871, -0.10502414405345917, 1.279590368270874, 0.48819398880004883, 1.1618127822875977, 0.6406358480453491, -1.5025067329406738, -0.9340123534202576, 0.5014848113059998, -0.7066628336906433, -1.1838773488998413, -1.0981566905975342, -0.9668703675270081, 1.8754712343215942, 0.3868565857410431, 0.3141686022281647, -0.6109137535095215, -1.2190781831741333, 0.9908760190010071, -0.01792694814503193, -0.38631418347358704, -0.9396082162857056, 0.05552418902516365, -0.036999527364969254, -1.110336422920227, 0.6263706684112549, 1.5702859163284302, 1.8884797096252441, -0.5767942667007446, -1.4628604650497437, 0.5640820860862732, -1.0141472816467285, 0.9372534155845642, 0.7953996658325195, 0.028566796332597733, 1.289433479309082, -0.0016672683414071798, 0.4307943880558014, -0.23002943396568298, 0.8525662422180176, -0.46631157398223877, 3.4256553649902344, 0.4610535204410553, -0.43354251980781555, -0.16009825468063354, -0.7443041801452637, -0.35305550694465637, 0.23645955324172974, 1.3891555070877075, -0.9669497609138489, 0.8524203300476074, 1.7013260126113892, -0.30092018842697144, -0.0951346680521965], [-1.2353068590164185, -1.2507630586624146, -0.8260079026222229, -0.5755658745765686, 1.9520912170410156, -0.9626687169075012, 0.5674930214881897, -1.7079399824142456, 0.539825439453125, -0.8314323425292969, 0.7355805039405823, -0.9246474504470825, 0.468717098236084, 0.47548362612724304, -1.3084640502929688, -1.211673617362976, 0.6237810254096985, -0.3466578722000122, 1.3641237020492554, 0.158148393034935, -0.6205736994743347, -0.7835602760314941, -0.24202926456928253, 1.1861060857772827, 0.6893812417984009, 0.18069817125797272, -0.18054869771003723, -0.8884294033050537, -0.8744691610336304, 0.748258650302887, 1.309093713760376, 0.3372734487056732, 0.12230522930622101, 1.1504710912704468, -0.7625522017478943, -0.6613267660140991, -0.19953949749469757, -0.6701716184616089, 0.6567801833152771, -0.9459654688835144, 0.3213701546192169, -1.6864957809448242, -0.9126364588737488, -0.6142930388450623, -0.07952313870191574, 0.5225090384483337, 0.6893690228462219, -0.9433609843254089, -1.3581602573394775, 0.7450159788131714, -0.27107664942741394, -1.3003515005111694, -0.8008242845535278, -1.5293762683868408, 0.25794878602027893, 0.010869698598980904, -0.9654651284217834, 0.2004767805337906, -0.45132043957710266, 0.16039061546325684, -1.4197168350219727, 0.5059018135070801, 1.1057549715042114, 0.7644505500793457, -1.6597764492034912, -0.09209752827882767, 0.5676084160804749, -0.11129351705312729, -1.1223281621932983, -1.0425084829330444, 0.09562425315380096, -0.42418381571769714, 0.19088515639305115, -1.7233238220214844, 1.22958242893219, -0.994392454624176, 2.3865807056427, 0.30073100328445435, 0.07891232520341873, -1.1053400039672852, -1.1728906631469727, -0.2591005563735962, -0.20778648555278778, -0.7038795351982117, 0.005453429650515318, -0.9615040421485901, 0.5971584916114807, 0.38967257738113403, 1.2885888814926147, -0.9471928477287292, -1.3977165222167969, -0.052648890763521194, 1.2319366931915283, 1.9409537315368652, -1.0943844318389893, 1.015037178993225, -1.088655710220337, 0.21311067044734955, -1.3995072841644287, 0.4845981299877167, -0.28464260697364807, -2.176459550857544, -0.4725567698478699, 0.40599367022514343, 1.039100170135498, -1.196826457977295, 0.19816192984580994, -1.9816633462905884, 1.256257176399231, -1.6152877807617188, -0.2669995427131653, 1.4849536418914795, -0.3362154960632324, -1.045591115951538, -0.26534947752952576, 0.09240632504224777, -1.0156587362289429, -0.49553629755973816, -0.06203813850879669, -1.3957642316818237, -0.8363130688667297, 1.9170560836791992, 0.7147743105888367, -1.189645528793335, -0.9517441391944885, 0.06620863080024719, 1.6816089153289795, -1.027580976486206, 0.22487734258174896], [0.9374020099639893, -1.2571380138397217, -2.4045615196228027, -0.7184502482414246, 0.2597636580467224, 0.27345746755599976, -0.6889516711235046, 0.4729284644126892, -0.9808524250984192, 0.4654615819454193, 0.7867318391799927, 0.20859849452972412, 0.2474585622549057, -0.5770053267478943, -0.1588490754365921, 0.74339359998703, 0.8511666059494019, 0.05452873557806015, 0.2805497944355011, 1.4186856746673584, -0.4805145263671875, -1.1060266494750977, 0.6827977299690247, -0.6016177535057068, -1.8718078136444092, -0.8094282746315002, 0.6815518736839294, 1.7570843696594238, 0.4098406136035919, -1.5189940929412842, 0.9240618348121643, -0.6711035370826721, -1.151557207107544, 0.010541330091655254, 1.476746678352356, -1.1598587036132812, 0.9200847148895264, -1.6012194156646729, -1.6670711040496826, -1.0672303438186646, -1.0787760019302368, 1.2143688201904297, 0.5090949535369873, 0.42849501967430115, -1.5342686176300049, 0.046173129230737686, -0.5665662884712219, 0.252719521522522, 0.1593651920557022, -0.941482424736023, -0.7033116817474365, -2.9040658473968506, 1.145268201828003, 0.41532155871391296, -1.0054218769073486, -1.0730332136154175, 0.2552298307418823, -1.3471016883850098, 0.5752613544464111, 0.6369569897651672, 0.30585911870002747, -0.03825092315673828, 0.4043472111225128, 0.8996626138687134, 0.6632406115531921, 0.9809948801994324, -0.6776054501533508, -0.9597499966621399, -0.7263709902763367, -1.467383623123169, 0.7665615677833557, -1.120123028755188, 0.5136722326278687, 0.586360514163971, 0.3562229871749878, -0.10339362174272537, 0.23401235044002533, 1.24732506275177, -0.00970325618982315, -0.7046090960502625, -0.018013333901762962, -0.30425459146499634, -0.4860774278640747, -0.05838267505168915, 0.08340086787939072, 0.46125760674476624, -0.552124559879303, 1.5742486715316772, 1.4149739742279053, 0.8933374285697937, -0.35342109203338623, 1.2281302213668823, -0.08376387506723404, -1.0543441772460938, -0.846369743347168, -0.6754976511001587, 0.21905595064163208, 0.022200554609298706, 0.20596438646316528, -0.1802448332309723, 0.028926944360136986, -0.6030919551849365, 0.47527530789375305, -0.42239969968795776, 1.0245873928070068, -1.2144207954406738, -2.3126094341278076, -1.6652746200561523, 0.49022799730300903, 0.9932294487953186, -1.5430455207824707, 0.8687033653259277, -1.2144031524658203, 0.44041433930397034, -0.3877898156642914, 1.2361160516738892, 0.5370751619338989, -0.20254313945770264, -1.2435322999954224, 0.1160317063331604, 0.987090528011322, -1.5368572473526, 2.1269490718841553, -1.137600064277649, 0.6319947838783264, 2.079369306564331, -1.3371332883834839, -0.49006181955337524, -1.036429524421692], [-0.07790114730596542, -0.001853173365816474, 0.04154578968882561, 0.0934644564986229, -0.34369567036628723, -0.387939989566803, 1.043872594833374, -0.5540007948875427, 0.20056326687335968, 0.3245913088321686, 1.5524775981903076, -0.07061807811260223, -0.7097037434577942, -1.1084593534469604, 0.37320250272750854, 0.09974747151136398, -0.8401981592178345, -0.2573716640472412, 1.1661173105239868, 0.6193522214889526, 0.4047425091266632, -0.18980401754379272, 1.0492693185806274, 0.6123698949813843, -0.7038788199424744, -0.24825984239578247, 1.3168543577194214, -0.04732467606663704, -0.5245702266693115, -1.0211844444274902, -0.06907988339662552, -0.533757209777832, 0.9335826635360718, 2.2367615699768066, -0.3716575503349304, 0.48019272089004517, 0.055838000029325485, -0.9375458359718323, 1.1362314224243164, 0.7166457772254944, -0.05155016854405403, -1.0654597282409668, -0.8209059834480286, 0.01642896607518196, -0.19302387535572052, -1.0673465728759766, -1.1414653062820435, -0.08958074450492859, 1.7538690567016602, -0.6477794051170349, 1.3027229309082031, -1.5291403532028198, -0.0910956859588623, -0.7834892272949219, 1.9543511867523193, 1.818874478340149, 1.3695489168167114, -0.22533756494522095, -1.950423002243042, -0.2622402608394623, -1.3641833066940308, -0.6629064083099365, 0.15046024322509766, 0.10259398072957993, -0.014421791769564152, 1.5804250240325928, -0.6430177092552185, -0.14796549081802368, 0.9514941573143005, 0.39155930280685425, -1.0569725036621094, -2.0021448135375977, 1.0328234434127808, 0.0929235965013504, 0.4671505391597748, -0.2618979215621948, -0.08586371690034866, 1.1388252973556519, 1.3190096616744995, 1.3353043794631958, 1.7826064825057983, -1.1633683443069458, 1.3183809518814087, -1.0401197671890259, -0.7301552891731262, 0.7107799649238586, -0.5150801539421082, 0.111316978931427, 0.17822867631912231, 0.7813687920570374, 1.0692062377929688, 0.2857343852519989, 1.4347058534622192, 2.0072999000549316, -0.3323751389980316, -0.023890862241387367, 1.281378149986267, -0.6473158001899719, 0.4719718098640442, -0.2554241716861725, 0.895220935344696, 0.45129263401031494, 1.8579018115997314, -1.872548222541809, 0.9939553737640381, -0.10838349908590317, -0.3452865481376648, 0.5735890865325928, -0.45797571539878845, -0.018841806799173355, 1.0674864053726196, -0.3703606426715851, 0.06500749289989471, 2.15351939201355, 0.08620196580886841, -0.5879613161087036, -0.10383794456720352, 0.6495105028152466, -1.0724494457244873, -1.8017020225524902, -1.292171835899353, 0.1556914746761322, 0.4003748893737793, -0.7966673970222473, -0.8039678931236267, 0.4739362895488739, 0.7307096123695374, -1.5184904336929321, 1.6809591054916382], [-0.9619867205619812, -2.5070269107818604, -0.5174071788787842, -0.002413496607914567, -0.18881213665008545, -0.39009255170822144, 0.8627969026565552, -0.3965372145175934, -0.8083612322807312, -0.329550564289093, -1.2562189102172852, -1.400755524635315, -0.552784264087677, 0.5631142258644104, -0.4433399438858032, 2.7042064666748047, -0.058309733867645264, -0.17984315752983093, 0.6448807120323181, -0.47690102458000183, 0.48572075366973877, -0.8366009593009949, 0.34677448868751526, -1.5993002653121948, 0.12099935114383698, -0.7821052670478821, 0.23644275963306427, 2.128995895385742, 1.230247974395752, -0.17342646420001984, 0.7088562846183777, -0.4811556935310364, -1.86568284034729, 0.18495731055736542, 1.1001640558242798, -0.8870105147361755, 0.18779896199703217, -0.2836851477622986, 0.7092099189758301, -0.8401650190353394, -0.1555635929107666, 1.7607927322387695, -1.0984325408935547, -1.411843180656433, 0.8983533382415771, 1.2209383249282837, -0.8656550645828247, -1.549923300743103, 0.7533485889434814, 0.5940880179405212, -1.5217499732971191, 1.0559799671173096, -0.802707314491272, 0.1733054369688034, 0.7008717656135559, -0.17877623438835144, 1.1000690460205078, 0.36740076541900635, 0.9087122082710266, -2.076833963394165, 0.4432656764984131, 0.8300180435180664, -0.11157092452049255, -1.0460420846939087, -1.7106702327728271, -1.4420902729034424, 0.545814573764801, 0.5149878263473511, -1.019866704940796, -0.3683956563472748, -0.301287978887558, 2.5413105487823486, -0.15505430102348328, -0.46306970715522766, -0.12174277752637863, 0.041852209717035294, 1.1250780820846558, -0.13188768923282623, 0.052904050797224045, -1.1966979503631592, -0.1775728464126587, -0.8267368674278259, 0.8354431390762329, 0.5138439536094666, 0.956192135810852, 2.1890878677368164, 1.5537813901901245, -0.6332012414932251, -1.331247091293335, 1.2709814310073853, -1.0137879848480225, -1.8004062175750732, -1.6871771812438965, -0.062135327607393265, -0.2159613072872162, 0.05075438320636749, -0.38944631814956665, 0.7686326503753662, 0.6533157825469971, 0.8481783270835876, -0.5028775930404663, 0.002593336161226034, -1.6925078630447388, -0.39489060640335083, -0.9684648513793945, -0.12935997545719147, 0.9256083965301514, -0.7203729152679443, -0.10629771649837494, -2.033250331878662, 0.7741039991378784, 0.3596395254135132, -0.7079896926879883, -0.6505336165428162, -0.8299192190170288, 1.2596936225891113, 1.3797038793563843, -0.2160019427537918, -0.471895694732666, 0.08540889620780945, 0.12928560376167297, 0.5772479772567749, 0.3146391212940216, -2.369568109512329, -0.2845822870731354, -0.10320696979761124, -0.09492204338312149, -0.29845520853996277, -0.38762715458869934], [-0.18939393758773804, -0.3536074161529541, 0.09387114644050598, 0.9878507852554321, 1.5197134017944336, 1.0293192863464355, 1.2614212036132812, 0.5727115273475647, 0.9678208827972412, 2.64969801902771, 0.6837365627288818, -0.0048274691216647625, -0.7530555725097656, 1.0164285898208618, -0.7695785760879517, -1.150877833366394, 1.5796037912368774, -0.02212824672460556, 0.670775830745697, 0.4623735249042511, 0.85052889585495, 0.8008918166160583, -1.4529430866241455, 0.5085510015487671, 0.5740489959716797, 0.15715031325817108, -0.37876442074775696, -1.2880195379257202, 0.13342922925949097, -0.004839784000068903, -1.2279791831970215, 0.7599382400512695, -0.9750282168388367, 0.45838433504104614, 1.4657634496688843, -0.806732714176178, -0.06505780667066574, 0.40548303723335266, 0.6810534000396729, 0.6301769614219666, 2.1958730220794678, -0.6304426193237305, 0.6455972790718079, -0.6207109093666077, -1.922802448272705, 0.4268864095211029, -0.4459688365459442, 1.3222604990005493, 2.6584689617156982, -0.16581827402114868, 0.8553225994110107, 0.08210321515798569, 1.241019368171692, -0.6814925074577332, -1.2288309335708618, -1.0731631517410278, 0.7363417744636536, -1.0048884153366089, 2.2221598625183105, -0.7037208676338196, 0.6671295762062073, -0.36966949701309204, 0.1477125883102417, -1.0911093950271606, 2.8052799701690674, -0.5713509321212769, 1.109215497970581, 0.9273764491081238, -0.007687458302825689, -0.7355384826660156, 1.591388463973999, -1.569697618484497, -1.6774706840515137, 0.3846682608127594, 0.8680453896522522, -0.2899521291255951, 1.291527271270752, 2.170552968978882, 1.0851891040802002, -1.1286545991897583, -0.2553236484527588, -0.6704569458961487, 0.6954143643379211, 0.7663739323616028, -0.12257549911737442, 0.921538770198822, -0.7463024854660034, -1.2493441104888916, 0.7454391121864319, 0.7595796585083008, -1.056209921836853, -0.1731233447790146, 0.9053452610969543, 1.575688123703003, -0.4077974259853363, 0.6799308061599731, -1.1380292177200317, -1.1952508687973022, 0.7821849584579468, -0.96617192029953, 0.6816039085388184, 0.3993496894836426, -1.2999094724655151, -0.2175358682870865, 0.3312870264053345, 1.3535501956939697, 0.0912676528096199, -0.5758153796195984, 1.4162111282348633, 1.70486581325531, 1.257986307144165, 0.023402629420161247, -0.638784646987915, 1.350048542022705, -0.4714653789997101, -0.23609402775764465, -0.7773521542549133, -1.9255762100219727, -1.3747749328613281, -1.7705572843551636, 0.688942015171051, -1.584463357925415, -0.4135132431983948, -0.8270396590232849, 0.2966187298297882, -1.4581592082977295, -1.3589452505111694, 1.4979228973388672, -0.48063039779663086], [0.24722233414649963, 0.07114250212907791, -0.37150901556015015, 1.2166606187820435, 0.23484796285629272, -1.9410741329193115, -0.4881818890571594, -0.7054101824760437, -1.3749700784683228, -0.5065709352493286, 0.13962922990322113, 0.24620942771434784, 0.747958779335022, 1.5448685884475708, 1.3902772665023804, 1.099958896636963, -0.5578480958938599, 1.9238462448120117, 1.5428271293640137, -1.1897650957107544, 1.2329062223434448, 1.9057024717330933, -0.8367776870727539, 0.5893117785453796, -1.901587724685669, -0.14795097708702087, -1.2906866073608398, -2.1823151111602783, 0.2787259519100189, -1.4898991584777832, 1.0765742063522339, -1.0286107063293457, -1.0475627183914185, 0.135334312915802, -0.7902804017066956, 2.2790474891662598, -0.9238271117210388, -0.153861403465271, -0.43464088439941406, -0.5587510466575623, 0.9070978760719299, -0.8506631851196289, 1.160681128501892, -0.7374548316001892, -1.4672735929489136, 1.3566352128982544, -0.44534915685653687, -0.6351451277732849, 1.3725900650024414, 0.1679697334766388, 0.08102265745401382, -0.2593860328197479, -0.48374414443969727, 0.5159783959388733, 0.2875072658061981, -1.3730586767196655, -1.442851185798645, -0.18753130733966827, 1.3267202377319336, -0.12790733575820923, 0.2008051574230194, -0.8811501264572144, -1.173570990562439, 0.5257507562637329, -0.27854177355766296, -0.3264528214931488, -0.24381589889526367, 1.8598709106445312, -1.0109083652496338, -0.27143433690071106, 0.9289980530738831, 0.49925050139427185, 1.0156545639038086, 1.6091547012329102, 0.42728284001350403, 0.20763443410396576, 0.321315199136734, 1.224815845489502, -2.3517417907714844, 1.6975246667861938, 0.3454432785511017, 0.7135321497917175, -1.2234985828399658, -0.5605228543281555, -1.2931259870529175, 1.5175857543945312, -1.5176849365234375, -0.8751357793807983, 1.5752308368682861, -0.7283802032470703, 0.7227553725242615, -1.4814057350158691, 0.6547244191169739, 3.284231185913086, -1.6213322877883911, -0.36136165261268616, -2.083336591720581, 0.5803452134132385, 0.6819153428077698, 0.8688750863075256, -1.7502602338790894, 0.4935397505760193, -0.7885798811912537, -1.3243733644485474, -2.355015516281128, -0.4430685341358185, 0.46975719928741455, -0.5947123765945435, 0.06622104346752167, -1.4396783113479614, -0.7009403705596924, -1.2805639505386353, 0.7443922758102417, 0.4459499418735504, 0.40776291489601135, -0.7632517218589783, 1.0948734283447266, 0.10238007456064224, -1.9605871438980103, -0.7539101243019104, -0.14548040926456451, 0.20967929065227509, 0.6694313883781433, 0.15610283613204956, 0.2355344146490097, -0.3250819742679596, -0.09713640064001083, -0.5075461268424988, -0.8980786204338074], [-0.17780998349189758, -0.019519755616784096, -2.026384115219116, 0.05291542410850525, -0.8424733877182007, 0.28304967284202576, 0.25230634212493896, 0.4985480308532715, -0.8818768262863159, -0.7527809739112854, 1.4011709690093994, -1.1720956563949585, 1.3634474277496338, -2.573274612426758, 1.6551005840301514, 0.5414120554924011, 0.1801566630601883, 2.2562255859375, 0.5827239155769348, 1.0646412372589111, -0.6437790989875793, -0.34472954273223877, -0.20373491942882538, 0.3731251657009125, -0.3912380039691925, -0.7485756874084473, -0.2341889590024948, -1.0571339130401611, -0.5109614729881287, -1.478827953338623, -0.687673807144165, 1.647153377532959, -1.5749322175979614, -0.242604061961174, 0.23458103835582733, 1.5212944746017456, -0.9340437650680542, -0.06586433947086334, -0.2254239022731781, -1.7338160276412964, -0.8668671250343323, -0.07289133220911026, 0.5910199284553528, 0.7486681938171387, -0.4174138903617859, -0.9226844310760498, -0.5712845921516418, -0.5413371920585632, -1.4321208000183105, 0.5401809811592102, -0.2979640066623688, -0.8582535982131958, 0.41852131485939026, -0.3999998867511749, -0.0864291712641716, -0.7044287919998169, -2.4855899810791016, -0.9537772536277771, 1.2938966751098633, 0.5024994611740112, -1.4033831357955933, -0.4684905409812927, 0.5567066669464111, -0.9588291049003601, 0.30678918957710266, -1.1011502742767334, 1.445277214050293, -1.754366397857666, 3.107104778289795, 0.5690979361534119, -0.6022518873214722, 0.27708956599235535, -0.4497235119342804, 0.6510465145111084, 0.6194397807121277, 0.5492419600486755, -0.9393351674079895, 1.119828462600708, -0.9504343867301941, 0.5546926856040955, 1.8704687356948853, 0.8960209488868713, 0.1368127167224884, -1.0336291790008545, -0.045023296028375626, -0.9013118147850037, 0.015432117506861687, 0.030119605362415314, -1.5644584894180298, -0.20385895669460297, -0.8978511095046997, -0.8061754107475281, 0.18004879355430603, 1.067949652671814, 0.18980251252651215, 0.6218132376670837, 0.9296388626098633, 2.2629127502441406, -0.1226770281791687, 1.038596749305725, -0.6452895998954773, -0.3831096589565277, 0.701447606086731, -0.13706545531749725, -0.13557803630828857, 0.21911576390266418, -0.8887053728103638, -0.3454933166503906, 0.7322567105293274, -1.00240159034729, -0.35361841320991516, -0.4442211091518402, -0.015839751809835434, -0.1767883151769638, 1.2988715171813965, -1.191462755203247, -0.7363300919532776, 0.7210323810577393, 1.6600466966629028, 0.3371545076370239, -1.1076078414916992, -0.04672006890177727, -0.6900540590286255, -1.019890308380127, -1.363951563835144, 0.32872048020362854, 0.8144717812538147, 1.9598419666290283, -0.6424432992935181], [0.5019794702529907, -1.3208311796188354, 0.7637919783592224, -0.4664550721645355, -0.5805558562278748, -0.8658624291419983, -0.13197681307792664, 0.6860018968582153, -0.6274389624595642, 0.6850787401199341, -0.6015840768814087, -0.8990887403488159, 1.4184104204177856, -3.3902223110198975, -0.8341805338859558, 0.48964691162109375, -0.43932655453681946, -0.2387813925743103, -0.5625753998756409, 1.2568461894989014, 0.24439847469329834, 1.9102709293365479, 1.5268710851669312, -0.06252659112215042, 0.24228981137275696, 0.9033223986625671, 1.3432062864303589, -1.2380714416503906, 0.25380346179008484, -0.42025232315063477, 0.5051988363265991, 1.25217604637146, -0.3805822432041168, 0.7150113582611084, -1.4291536808013916, 0.6344989538192749, 1.8707308769226074, 1.5201451778411865, -0.15139466524124146, -0.2340322881937027, -0.04299772530794144, -0.6613077521324158, 0.36343833804130554, 0.8574523329734802, -0.9223009943962097, -0.09967518597841263, 0.47372332215309143, -1.5521262884140015, 1.7881642580032349, -0.6805141568183899, -0.7232570052146912, -0.543057918548584, 1.9599586725234985, 0.06046503037214279, -1.970448613166809, 1.020067811012268, -0.8513191342353821, 0.5191528797149658, 0.3711967170238495, -0.753319501876831, -0.5030518174171448, 0.8811968564987183, -0.31743311882019043, -0.4585413932800293, 0.5300779938697815, -2.144341230392456, 1.2267794609069824, -1.5117287635803223, -1.4832061529159546, 0.3498042821884155, -0.4543083906173706, -1.9225683212280273, -0.21375758945941925, 2.957731246948242, -2.867112159729004, -1.1666749715805054, 0.14094121754169464, -0.9871187806129456, 1.1219556331634521, 0.7835149168968201, 0.06361179798841476, -0.12756602466106415, 1.5628005266189575, -2.0578718185424805, -0.6298190355300903, -0.9092019200325012, -1.055197834968567, -0.6259889602661133, -0.19075225293636322, 1.8554052114486694, -0.1669885665178299, -1.7758241891860962, -0.19390515983104706, 0.4405408799648285, 0.8149614334106445, -0.14631128311157227, -0.945107638835907, -0.9094683527946472, 1.1501171588897705, 0.09352857619524002, -1.0075465440750122, -0.9968165159225464, -0.018676258623600006, -1.1618999242782593, 0.8275412917137146, -0.5016250610351562, -0.6186271905899048, 1.2663524150848389, 0.6217076182365417, -0.6218953728675842, -2.2020609378814697, -2.264777898788452, 1.6172508001327515, -0.8779821395874023, 0.9884712100028992, 1.4710168838500977, 0.24539732933044434, 1.1946951150894165, -1.7123795747756958, -0.8918875455856323, -2.529378652572632, 0.9311304688453674, -0.6491254568099976, 0.3341120481491089, -0.26393014192581177, -0.2733122706413269, 0.24137280881404877, 0.4719190001487732, 0.8122530579566956], [-1.4744426012039185, 0.4272521734237671, 0.9554717540740967, 1.2070552110671997, 0.3662204146385193, 1.2567392587661743, -0.4921313226222992, -2.1061136722564697, -0.3765643835067749, -0.8154825568199158, 1.468016505241394, 0.46123984456062317, -0.8049378991127014, -1.0979688167572021, -0.2040739506483078, 0.2144116759300232, 0.3060785233974457, -2.331235885620117, 0.12339308857917786, 0.6710197329521179, 0.07843782007694244, 1.2641044855117798, -2.0102245807647705, 0.10625088214874268, 0.2338273823261261, -0.3114246129989624, 0.44343462586402893, 0.08308220654726028, -0.20556128025054932, 0.12756358087062836, -1.0490756034851074, 1.6950178146362305, 1.4344292879104614, -0.7763218283653259, 0.45408886671066284, -0.9218178987503052, -0.833340585231781, 1.541940450668335, -1.0503795146942139, 0.05411902442574501, 1.2603363990783691, 1.1223413944244385, -1.0591251850128174, -0.043901875615119934, -0.3055737614631653, -0.8880974650382996, -1.000728726387024, -0.14264273643493652, -0.8372148275375366, 0.1545965075492859, 2.011460304260254, 1.6613523960113525, -1.5015124082565308, 0.18302609026432037, -0.5988118648529053, 0.557692289352417, 0.568996787071228, -0.27919089794158936, 0.09173019230365753, -0.318078875541687, 0.3376064598560333, -1.103134036064148, 0.12798698246479034, -1.1110800504684448, 0.0972852036356926, 1.1595433950424194, 1.1479545831680298, -0.5864971876144409, 0.4503971040248871, -0.9851874709129333, 0.8776305317878723, 0.08845937252044678, -0.6636572480201721, 1.5106550455093384, -0.4416598975658417, 0.3269464075565338, -1.092989206314087, -0.4646637737751007, 1.270382046699524, 1.8938020467758179, -0.47997409105300903, 0.577255129814148, 1.6485936641693115, 1.4776488542556763, 0.2803654670715332, 1.3296645879745483, -0.8026956915855408, 0.354876846075058, -0.08764228224754333, 0.8062193989753723, -1.1613118648529053, -1.4644070863723755, -0.7831487059593201, 1.091896653175354, 1.0859463214874268, -0.9203108549118042, 0.9489402174949646, -2.463590383529663, -1.5359724760055542, 1.4697054624557495, -1.495430588722229, 0.9202589988708496, -0.62824547290802, 1.5921913385391235, -0.3346649706363678, -0.7978203892707825, 3.0147600173950195, 1.7892506122589111, -2.093553066253662, -0.003740837099030614, 0.3253583014011383, 1.2511097192764282, -0.6540620923042297, 0.6210580468177795, -0.3904242217540741, -0.29299241304397583, -0.9286474585533142, -1.3919613361358643, 2.142791271209717, -1.4987726211547852, -0.03276282176375389, -0.6150566339492798, 0.5392810106277466, 1.0646435022354126, 0.7172225713729858, -1.4089288711547852, 0.5871065855026245, 1.0322871208190918, 0.7649106979370117], [0.24235029518604279, -0.03061051107943058, -2.2754693031311035, -1.6658575534820557, 1.1813561916351318, 0.3328408896923065, -1.0677666664123535, -0.2631600797176361, 0.3438470661640167, -0.4432455003261566, -1.8130528926849365, -0.9388864040374756, -0.66246098279953, -0.10111507028341293, 0.3129391372203827, 2.2966270446777344, 0.6944561004638672, 1.2961899042129517, 0.637536346912384, -0.3230401575565338, 1.789023518562317, -1.6170620918273926, 1.1237927675247192, 0.10959764569997787, 1.8399711847305298, -0.17423075437545776, 0.7816893458366394, 0.14974528551101685, 0.3633824288845062, 1.0019959211349487, -0.7339659929275513, 0.7051982879638672, -1.6415694952011108, 0.33120593428611755, -0.3279702663421631, 0.37706613540649414, -1.000815987586975, 0.6694265604019165, 0.3141040802001953, 2.17883563041687, 0.39183181524276733, -0.3858986496925354, 2.4110987186431885, -1.860639214515686, 0.1933705061674118, -0.43877923488616943, 1.5801953077316284, -1.0565837621688843, -0.6824286580085754, 2.761448383331299, 0.2362055480480194, -0.23537613451480865, 0.6702038049697876, 1.3340158462524414, -0.23124374449253082, -0.2009022831916809, -0.03597421571612358, 0.8830528259277344, -0.27140721678733826, 0.9492154121398926, -0.007932315580546856, -0.2519604563713074, 0.21221353113651276, 0.23601144552230835, -0.8211278319358826, 0.5087463855743408, 1.1501672267913818, -1.1849790811538696, 0.09103672951459885, 0.10967900604009628, -0.2276753932237625, 0.6387460231781006, 0.3752756416797638, -0.17676642537117004, -0.049778275191783905, -0.8824654221534729, -0.23360170423984528, -1.0124455690383911, 1.04855477809906, 0.1869601607322693, 0.23125958442687988, -0.01837046816945076, 1.2099305391311646, -1.1762834787368774, 1.1046326160430908, -0.17514771223068237, -0.5425289869308472, -0.16998623311519623, 0.18068425357341766, -0.8485285043716431, -0.6429175734519958, 1.0145376920700073, 0.8220093846321106, 0.3523622155189514, -0.8586620092391968, -1.236976981163025, -1.0700076818466187, 0.5631483793258667, 2.3968706130981445, -0.3405888080596924, -0.7449288368225098, 1.3214757442474365, 1.9720293283462524, 0.41590237617492676, -0.294719398021698, 1.2493716478347778, 0.5806237459182739, -0.12911461293697357, 0.028364164754748344, -0.9884920716285706, -2.144541025161743, -0.011113384738564491, 0.8506792187690735, 0.582029402256012, -1.7627909183502197, -0.6208133697509766, 0.7867677211761475, 1.026324987411499, 1.7764250040054321, -0.871937096118927, -0.2592496871948242, 0.5610637068748474, 0.15169072151184082, -1.055612564086914, -0.8666530847549438, 0.5295831561088562, 0.3265920579433441, 0.6022687554359436, 0.048900067806243896], [-0.2179698348045349, 0.4357655942440033, 1.2143919467926025, 1.1588894128799438, 0.4119476079940796, -0.34922903776168823, 1.099601149559021, -0.47198933362960815, -0.40724530816078186, -1.2838096618652344, -0.7184399962425232, 0.22778315842151642, 0.0796743705868721, 0.016865091398358345, -0.5798397660255432, 1.550412654876709, -0.5562142133712769, 0.3408414423465729, 1.1874535083770752, 1.176743984222412, 0.05551452562212944, -0.2369951754808426, -0.3397354483604431, -0.8765078783035278, 0.1238020732998848, -1.816607117652893, -0.07661213725805283, -0.357486754655838, 0.6371264457702637, -0.9672162532806396, -1.2061994075775146, 2.4227871894836426, 1.8814475536346436, -0.27602577209472656, 0.7274309396743774, 1.0677355527877808, 0.012556007131934166, 1.0009758472442627, 0.12503066658973694, -0.24921077489852905, 1.9896352291107178, -0.20969142019748688, -0.16027216613292694, 0.4604112207889557, -0.10619013011455536, -0.0940508097410202, 0.3888467848300934, 0.4054371416568756, 1.74503755569458, -1.4548711776733398, 1.0264023542404175, -0.6727230548858643, -1.4093173742294312, -0.3734036087989807, 0.31536978483200073, -1.1037436723709106, -1.6732109785079956, -1.0970741510391235, -0.9557624459266663, -0.13273920118808746, -0.07321103662252426, -0.5041496157646179, 0.41147541999816895, -0.8756277561187744, -1.540593147277832, -1.0509766340255737, 1.3294106721878052, 0.40500593185424805, -0.5175832509994507, -1.5275522470474243, -0.7182596921920776, 0.06820282340049744, 0.9125757813453674, -0.8304322361946106, -1.1101484298706055, 1.4427391290664673, -0.12179748713970184, -1.1733272075653076, 0.9486716389656067, -0.5966676473617554, -0.1057056337594986, 0.7282063961029053, 0.16812452673912048, 0.5399062037467957, 1.1759084463119507, -1.8456910848617554, -0.0035095945931971073, -1.850633144378662, 1.0208044052124023, 0.6795060038566589, 0.6318866610527039, 0.9866700172424316, 0.02025783620774746, -0.06600261479616165, -1.2686131000518799, -0.019648293033242226, -1.3244848251342773, -0.6183357834815979, 0.0012944169575348496, 1.8507888317108154, -0.046991851180791855, 0.2514668107032776, 0.3470194637775421, 1.203602910041809, 0.33602020144462585, 0.49086955189704895, -0.14718687534332275, 0.1976693868637085, -0.022838778793811798, 0.3872459828853607, 0.8182932734489441, 0.14213544130325317, 1.0679453611373901, 0.11385823041200638, 1.1895278692245483, -1.6046593189239502, 1.0668877363204956, 2.0643136501312256, -1.2456735372543335, 0.5021173357963562, -0.9001480340957642, 1.7195755243301392, 0.05098671466112137, 1.8002943992614746, -0.6912224292755127, 0.5190886855125427, -1.15848708152771, -0.27469390630722046, -1.0935395956039429], [0.4968584179878235, 1.4747283458709717, -0.23435792326927185, 0.34752941131591797, -2.034050464630127, -1.6030527353286743, 1.526045799255371, 0.7694685459136963, 0.713024914264679, -0.7069276571273804, 1.1023021936416626, -0.16589295864105225, -0.39393171668052673, 0.28718122839927673, -0.6427524089813232, -0.585774302482605, 0.18290792405605316, -0.6377859115600586, 0.1753896325826645, 1.1598807573318481, 0.22827178239822388, 0.49686068296432495, 0.34054502844810486, -1.1071205139160156, -0.17129217088222504, 1.2209374904632568, 0.4603351950645447, 0.6774174571037292, 0.7931769490242004, 1.251564621925354, -1.4617283344268799, 1.6619856357574463, -0.376687228679657, -0.5489753484725952, 1.211038589477539, 0.9021729826927185, -2.619396448135376, -0.9259868264198303, -0.5528649687767029, -0.651153028011322, 1.0740656852722168, -1.17891263961792, -0.2103753387928009, 0.23868443071842194, 0.7475259304046631, -0.9334689378738403, -0.34421348571777344, -0.8985892534255981, -0.7299192547798157, -0.4302699565887451, 1.4186787605285645, -0.4202309846878052, 1.3857357501983643, 0.8021367192268372, 0.22127839922904968, -0.24859619140625, 1.5976438522338867, -0.7398999929428101, -2.5728859901428223, -1.5842108726501465, 0.38537466526031494, -0.7479701638221741, 0.2360064685344696, -1.7528575658798218, -2.1077311038970947, -0.4244263768196106, 0.6881340742111206, -1.5136632919311523, -0.8406720757484436, 1.0096086263656616, -0.3057240843772888, 0.22794705629348755, -0.4270494282245636, -1.0956305265426636, 0.2884698510169983, 0.09643122553825378, -0.03415949270129204, -0.28309881687164307, -0.6229844093322754, 0.8197844624519348, 0.5490056276321411, -0.9748612642288208, 0.6802200675010681, -0.04710203781723976, 0.4167059063911438, 1.293366551399231, 0.6582813858985901, 0.17806914448738098, -0.811539888381958, -0.5774955749511719, -0.7698520421981812, -2.9046688079833984, 0.20040297508239746, 0.5900342464447021, 0.37295910716056824, -0.745490312576294, 0.9760805368423462, -0.15719321370124817, 0.14758260548114777, 0.9890386462211609, 1.06766939163208, 1.3166624307632446, 0.2245839238166809, -1.1242231130599976, 0.007273454684764147, -0.8032784461975098, 0.3568938970565796, 0.4551764726638794, -0.3165935277938843, 2.2906482219696045, -0.4479157030582428, 0.7951434254646301, -0.6939374804496765, -0.14995774626731873, 0.5179988145828247, -0.8955652117729187, -0.8233652710914612, -0.14808358252048492, -0.35841089487075806, -1.369545817375183, 1.456032633781433, -0.6764981150627136, -1.2672770023345947, -0.43377286195755005, -1.6146786212921143, 0.9586641192436218, 1.1272457838058472, 1.51180899143219, 0.6865171194076538], [-0.1753680557012558, 0.8093136548995972, 0.9477869272232056, -1.7978392839431763, -0.13769082725048065, 0.9500341415405273, -0.014112168923020363, 0.9058640003204346, 0.17895594239234924, 0.43740957975387573, -0.27188751101493835, 1.3279504776000977, 0.47889426350593567, -0.030802184715867043, -0.7567276954650879, -1.4205269813537598, 0.04896757751703262, -0.6994472742080688, -3.207428455352783, -0.7902549505233765, -1.4069952964782715, 0.7339638471603394, -0.41948962211608887, -2.0848758220672607, 0.8578065633773804, -0.18566392362117767, 0.6246017813682556, -0.3813864290714264, 2.292938709259033, -0.4461737275123596, 1.19740629196167, 0.8657313585281372, -0.15601883828639984, -0.04899478331208229, -0.480807900428772, 0.6058208346366882, -1.573230266571045, -0.07407891005277634, 0.9697121381759644, -0.7148881554603577, -0.1555536389350891, -1.3002935647964478, -1.1142592430114746, 0.2273697406053543, 0.576863706111908, 0.4532610774040222, 1.2192782163619995, 1.5307265520095825, 0.34627875685691833, 1.3552806377410889, 0.5690906047821045, -0.05875099450349808, 1.746614694595337, -0.5461190342903137, 0.9913433790206909, -0.13319021463394165, 2.802189350128174, 1.3830286264419556, 0.3304760158061981, 0.4242812693119049, -0.9177132844924927, -0.46652746200561523, -0.25409018993377686, 0.07888689637184143, -1.192100167274475, -0.5429854393005371, -0.04563302546739578, -1.0267868041992188, -0.6039667725563049, -1.413688063621521, 1.118314504623413, 0.051675014197826385, -0.8615109324455261, -2.177913188934326, 0.6835790872573853, 0.23679512739181519, -0.44345635175704956, -0.9371742606163025, 0.12129570543766022, -0.02186153642833233, -0.13955330848693848, -0.9349886178970337, -1.2039586305618286, 0.2524198293685913, -0.21385109424591064, -0.8805566430091858, 0.36628878116607666, -0.0318618044257164, -0.06662313640117645, 0.26786983013153076, 0.9726200103759766, -0.25846239924430847, 1.1328399181365967, -2.132345676422119, 2.53651762008667, -0.9628936648368835, 0.32106471061706543, 0.28601735830307007, 0.44566255807876587, -1.370119571685791, -0.846889078617096, -0.7709997892379761, -0.7640258073806763, 0.45254334807395935, 0.1076548621058464, 1.5481956005096436, 0.4733361005783081, 1.9179437160491943, 0.6310847401618958, -1.42702317237854, 0.49814552068710327, -0.6974787712097168, 0.9652991890907288, -1.5866258144378662, -0.7446074485778809, 0.277087539434433, 2.014129161834717, 0.07288181781768799, 1.3299503326416016, -0.3193091154098511, -0.21290628612041473, -0.8374535441398621, -1.5817724466323853, 0.021931305527687073, 0.25964510440826416, 0.8166126608848572, 0.2869078516960144, -0.887604832649231, -0.8918898105621338], [-1.7294538021087646, 2.073148727416992, -0.7810067534446716, 0.17045485973358154, 0.29350757598876953, 0.517521321773529, -1.2727067470550537, -0.50068598985672, -0.9925072193145752, 0.718011736869812, 0.07215475291013718, 0.5090500712394714, -0.9904518127441406, -1.303288459777832, -0.2525234818458557, -0.2832106351852417, 0.7660945653915405, 0.06884080171585083, -0.21877747774124146, -0.6405643820762634, 1.6891897916793823, -1.8484852313995361, 1.0876061916351318, 0.3288508355617523, 0.6649850010871887, -1.0342323780059814, 0.9352566599845886, 0.16606974601745605, -1.2602651119232178, -0.9656537175178528, -3.49835205078125, 0.5390281677246094, -0.5568854212760925, -0.060220565646886826, 0.08092736452817917, -0.44960668683052063, 0.7779509425163269, -0.3406004309654236, 1.480780005455017, -0.42191383242607117, 1.6676714420318604, 1.2237303256988525, 0.0956176370382309, 1.6941691637039185, 0.2919117510318756, 1.588796854019165, 0.8195356726646423, -2.401036262512207, -0.9148948192596436, -0.12434666603803635, -0.6786870360374451, -0.5541435480117798, -1.068617820739746, -0.7165274620056152, -0.26326173543930054, 0.8601449728012085, -0.6355849504470825, 0.16651524603366852, 1.4064481258392334, 1.3886502981185913, -1.0367385149002075, 1.1481547355651855, -0.3526545464992523, -0.2769455015659332, 1.84800124168396, 0.8593013882637024, 0.5720822215080261, 0.038050826638936996, -1.3084620237350464, -2.3653013706207275, 0.1438358873128891, 0.5067019462585449, -0.6180970668792725, -1.3299565315246582, 1.0311130285263062, -0.22770923376083374, -0.30487045645713806, 0.01481487974524498, -0.47542497515678406, -0.9709899425506592, 1.0243674516677856, -0.22698859870433807, 0.31709542870521545, 0.7311272621154785, -0.6772695183753967, 0.021346647292375565, -1.3460396528244019, -0.5480499863624573, -0.8438894748687744, 1.133021354675293, -0.40976083278656006, 0.1835295557975769, -0.5563340187072754, -0.5817825198173523, -0.06684131175279617, 0.49175161123275757, 0.31578534841537476, -0.7559000849723816, 0.1282985657453537, 1.25386643409729, -0.624922513961792, -2.219615936279297, 0.5377419590950012, 0.3183663487434387, -0.9623785614967346, 2.2429723739624023, -0.42843982577323914, -2.239382743835449, 0.003731019329279661, -0.8542394638061523, -0.17487657070159912, 1.8513858318328857, -0.3905332684516907, -0.06337126344442368, 0.2994489371776581, -1.1650428771972656, -0.49504193663597107, 1.4577986001968384, 1.5536891222000122, -1.397847294807434, 1.0175836086273193, 2.443838119506836, 0.47868624329566956, -0.8150719404220581, 1.668753743171692, -1.6172364950180054, -0.1893426924943924, -0.022891905158758163, -1.0223629474639893], [-1.3971668481826782, -1.48911714553833, 0.2298990935087204, -1.3007829189300537, -0.47488638758659363, 0.3367721438407898, 0.012663708068430424, -0.190878763794899, -0.36042407155036926, 0.4066632091999054, -1.2907719612121582, -0.1963774412870407, 0.35857149958610535, 2.4357237815856934, -0.9460956454277039, 0.45685648918151855, -0.48731550574302673, -0.39222151041030884, 0.01862681657075882, -1.9524420499801636, 0.2279578000307083, 1.341881275177002, 2.1064345836639404, -0.08283921331167221, 0.945331871509552, -0.09900588542222977, 0.3843619227409363, -0.3026878237724304, 1.7168760299682617, -0.13744506239891052, -0.6355054378509521, 0.7914534211158752, 0.9319722652435303, -1.1705642938613892, 0.7105661034584045, 0.5132514238357544, -0.026368973776698112, -0.45330098271369934, 0.14144966006278992, 0.06126173585653305, 0.034610945731401443, 1.3034757375717163, 0.003805789863690734, 0.9017328023910522, -0.7483624815940857, 1.0189017057418823, -0.3535935878753662, -0.23874524235725403, 0.5451151728630066, -1.060880422592163, 1.8035191297531128, -0.5070917010307312, -1.0157185792922974, 0.805371880531311, -0.5518914461135864, 1.807366967201233, -0.2704523801803589, 0.17944592237472534, -0.284151554107666, 0.46215546131134033, 0.49421533942222595, -0.5771958827972412, 0.07261530309915543, 1.3648552894592285, -0.7443283796310425, 0.39240291714668274, 0.27218085527420044, 0.5019131302833557, -0.47255659103393555, 0.5889361500740051, -0.8908637166023254, -0.32944896817207336, -0.05267651379108429, -0.18744201958179474, -0.9929524064064026, -1.7749719619750977, 0.6088467836380005, 0.6865149140357971, -3.3462586402893066, -0.3004521131515503, 0.2735346555709839, -0.16320864856243134, 0.5857290625572205, -0.041025560349226, -1.1701165437698364, -0.024761417880654335, -1.309551477432251, 0.0050316001288592815, 0.07400897890329361, -0.06603894382715225, 1.0620522499084473, -0.4321728050708771, -0.0015269106952473521, -0.03335215896368027, 0.6816416382789612, 0.922652006149292, 0.4009665250778198, -2.3537213802337646, -0.7631955146789551, 0.018974991515278816, 0.08186311274766922, -1.339759349822998, 0.5965583920478821, -1.606398105621338, 0.7985274195671082, 1.4542794227600098, 0.9656096696853638, -0.7884800434112549, 0.8159729242324829, 1.424276351928711, 0.06561356782913208, -0.053852442651987076, 0.5310854315757751, 0.4289760887622833, 0.2626591622829437, 0.7311722636222839, -0.14756083488464355, -0.4861036241054535, 0.1892988532781601, -1.492345929145813, -1.478390097618103, -0.8201224207878113, -0.26692989468574524, -0.591735303401947, -0.07644324004650116, -0.49542132019996643, -0.058214638382196426, 0.9505934715270996, 0.8963534832000732], [-0.10067915171384811, 0.6465562582015991, 0.13560250401496887, 0.312944233417511, 0.19369535148143768, 0.00720570283010602, 0.6670861840248108, -0.29705333709716797, -0.15686741471290588, 0.27942100167274475, -0.5228263139724731, 0.14158956706523895, 0.3906134366989136, -0.3417888879776001, -1.4795246124267578, 0.43786993622779846, 0.1455114632844925, -0.7246960401535034, 1.6169179677963257, -0.28713780641555786, 2.7227983474731445, -2.2796247005462646, 0.42329782247543335, -0.12340652197599411, -0.7627980709075928, 0.1560954749584198, -0.9707087278366089, -0.34943902492523193, -0.06059114262461662, 0.5405752062797546, 1.0651248693466187, 0.0382092222571373, -1.846429705619812, 0.28178203105926514, 1.413548231124878, -0.9246751666069031, 0.7673308253288269, 0.4363860487937927, -0.3261379599571228, -0.258630633354187, -0.39881977438926697, -0.5566400289535522, 0.718619167804718, -1.4423335790634155, 0.16517174243927002, -0.4174453020095825, -1.9486953020095825, -0.09835416078567505, -2.4896950721740723, -0.8024377822875977, -0.16515807807445526, -0.9220728278160095, 1.3806015253067017, 1.6955344676971436, 0.14363247156143188, -1.5679775476455688, 0.12876395881175995, -0.3032061457633972, -0.4202149510383606, -1.582471251487732, -0.9988027215003967, -0.08852490037679672, -1.215975284576416, 0.055049117654561996, 0.0664912611246109, 0.49771279096603394, 0.4134760797023773, -0.16335450112819672, -0.32507050037384033, -0.2898198366165161, 0.48043957352638245, -1.2868719100952148, 0.9570006728172302, -0.31189894676208496, 1.3027125597000122, -0.5127254724502563, 0.519956111907959, 2.404635190963745, 0.27869871258735657, -1.6669371128082275, 1.6192381381988525, -0.6109628081321716, -0.979157030582428, -1.5808645486831665, 1.604535698890686, 0.29514309763908386, -1.0848674774169922, -0.7455984354019165, 1.8344804048538208, 1.431272268295288, 0.6941980123519897, -0.36980000138282776, -0.4642567038536072, -0.9031045436859131, 0.13796208798885345, 1.4466519355773926, 0.237954780459404, -1.1165751218795776, 0.4327450394630432, 0.7060927748680115, -0.5623821020126343, -0.3245595693588257, -0.11856216192245483, 0.4352543354034424, -0.7850908041000366, -0.9825477600097656, 1.3352473974227905, -1.014880657196045, 0.3159947097301483, -1.811151146888733, -1.4034745693206787, 1.3810548782348633, 0.1553199142217636, 1.2769427299499512, -0.701563835144043, -2.374033212661743, -0.6447160243988037, -2.456249475479126, -1.0160778760910034, 1.6751011610031128, 0.9802735447883606, 0.25157222151756287, 1.5841636657714844, -0.9575180411338806, -0.7273454070091248, 0.26748496294021606, -0.4571787118911743, -0.9979812502861023, -0.6980029344558716], [-0.4556686580181122, 0.4499877691268921, -0.04217348247766495, -0.3066379427909851, -0.4944835603237152, -0.8845775723457336, -0.30646011233329773, -0.04419250413775444, -0.9969350695610046, 0.23655952513217926, 1.4782929420471191, 0.43011805415153503, 1.4085562229156494, -2.2022836208343506, 0.22050954401493073, 0.42989417910575867, -0.38189589977264404, -0.15450716018676758, -0.8636227250099182, 0.1576472669839859, 1.2474966049194336, -0.813398003578186, -0.5607316493988037, -1.1835070848464966, -1.5005183219909668, -1.8263952732086182, 0.526466965675354, -0.6560733318328857, 0.12292471528053284, 1.102481484413147, -0.07137835770845413, -0.450231671333313, -0.5816586017608643, -2.6715991497039795, -2.321777820587158, -0.31726107001304626, -0.8460239171981812, 1.039656162261963, -0.9216517806053162, -0.5445051789283752, 0.23629088699817657, -0.6252946853637695, -0.6911939382553101, -0.6353562474250793, 0.09514191001653671, 0.20894621312618256, 1.009480357170105, -0.9421724677085876, -0.0022813170216977596, -1.2531641721725464, 0.21379980444908142, -0.44449108839035034, 0.7504163384437561, -1.8067508935928345, 0.8887208104133606, -1.240813970565796, -0.1161435917019844, -0.45210838317871094, 0.2857443392276764, 0.12214834988117218, -1.3214102983474731, -0.4544696509838104, 0.6652594208717346, -1.131864070892334, 0.8222338557243347, 0.3708832263946533, 1.0084753036499023, -0.17487463355064392, 0.3332776129245758, 0.5495341420173645, -1.0868492126464844, -1.4582240581512451, 0.16530273854732513, 1.909872055053711, 1.0370025634765625, 1.5081998109817505, -0.4633871018886566, -0.7319574952125549, 0.0657588467001915, 0.779948890209198, 0.9471323490142822, -1.5229289531707764, 0.53427654504776, -0.7778011560440063, -0.5494925379753113, -1.901013731956482, 0.8361526131629944, -0.30274447798728943, 0.5502086281776428, -0.3701087534427643, -0.554232120513916, 1.0420470237731934, -0.03727913275361061, -0.8560734987258911, 2.300562858581543, -0.10791432112455368, -0.5608242750167847, -0.4492502212524414, -0.2878817319869995, -0.03207291662693024, -0.528249979019165, -2.3672430515289307, -1.351346731185913, 2.728848457336426, 1.4468623399734497, -0.7121008634567261, 0.5268566012382507, 0.9839547276496887, -0.2508588135242462, -0.43674996495246887, 0.8377898335456848, -0.25825977325439453, -0.3100118935108185, 1.3778284788131714, 1.1032618284225464, -0.0671943798661232, -1.6008050441741943, -1.1777453422546387, 0.3597152829170227, 1.3156437873840332, 0.03167392313480377, -0.8285582661628723, 0.5872544646263123, 0.4273144006729126, -0.6985999941825867, -0.4914786219596863, -1.4655176401138306, -0.07199268043041229, -0.21907112002372742], [0.9262262582778931, 1.4339087009429932, -0.9021939635276794, 1.1547330617904663, -0.8108046650886536, -1.4703618288040161, -0.3623330295085907, 1.5972462892532349, 1.0576180219650269, -1.1594070196151733, -1.0025148391723633, 0.4651238024234772, -0.810889482498169, 0.8408864736557007, -0.6754856109619141, 0.4474218487739563, 0.5107250213623047, -1.1934070587158203, 0.07224538177251816, -0.7761988043785095, -0.20306256413459778, -0.2098628133535385, 1.0512902736663818, -0.6394765377044678, 0.06005781888961792, -0.21029216051101685, 0.1563272774219513, 1.5806260108947754, -0.7970625758171082, -0.27147459983825684, 0.5831190347671509, 1.9009357690811157, -0.3394874930381775, 1.2746326923370361, -0.26042622327804565, 0.6178152561187744, 0.7477152347564697, -1.9588176012039185, -1.2813732624053955, 1.9895273447036743, -1.7984702587127686, -0.21699881553649902, 1.4338897466659546, 0.24478517472743988, -0.6977275013923645, -0.3233954906463623, 0.6399575471878052, -0.6177006959915161, 0.36263489723205566, -0.5212404131889343, -0.13909770548343658, 0.26897940039634705, -0.4488055109977722, -1.2759960889816284, -1.4377450942993164, -0.6983283758163452, -0.4855100214481354, -0.2422030121088028, 0.6986916065216064, -2.102630615234375, -1.1948833465576172, 1.3807857036590576, 1.8597432374954224, -0.12387941032648087, -0.22142642736434937, -0.3698916435241699, -2.2650508880615234, 0.5564358830451965, 0.018345754593610764, -1.1194615364074707, 0.35615602135658264, 1.2936487197875977, -1.2138842344284058, 1.2700976133346558, -2.692909002304077, -0.7580996751785278, -0.330547571182251, -2.056190252304077, -1.139728307723999, -0.5323672294616699, 0.6377335786819458, -0.3460281789302826, 0.23379671573638916, -0.21125507354736328, -1.1158384084701538, -1.225236415863037, -1.2965550422668457, -0.5191329717636108, 1.1634843349456787, -0.07904645800590515, -0.2290581315755844, -1.5647295713424683, 0.3136964738368988, 0.5394500494003296, -0.9257591962814331, -0.39016464352607727, -0.6974233388900757, -0.08765733242034912, 0.8427819609642029, 0.3399134576320648, 0.7917631268501282, 1.502218246459961, -2.6406192779541016, 0.7191510796546936, 1.492862343788147, -0.5570396780967712, 0.8303317427635193, -0.8348075151443481, -0.18084487318992615, -0.6890348792076111, 1.8984957933425903, -1.1852866411209106, -0.3628155291080475, -0.33698126673698425, -0.6347571015357971, 0.6988307237625122, 0.5945504307746887, -0.1354576200246811, 0.2717382311820984, 0.7992731928825378, 0.746142566204071, 2.0559020042419434, 2.7551004886627197, 0.23464538156986237, 0.3152581453323364, -0.10282948613166809, 0.15089726448059082, 1.426259160041809, -0.948253333568573], [0.2265840321779251, 1.5903356075286865, 0.011423937045037746, 0.0696408674120903, 0.03436315804719925, 0.038126688450574875, 0.028829308226704597, 1.075310230255127, 1.9514964818954468, 0.5605169534683228, -0.023212498053908348, -0.26758795976638794, 0.6226462125778198, 0.0306062214076519, 0.8888486623764038, -0.4710156321525574, -0.4820285141468048, -0.028479650616645813, -0.12785308063030243, 1.9185965061187744, 0.1078682541847229, -0.5914851427078247, -0.45472222566604614, 0.8655257225036621, -0.7542040944099426, 0.8588941097259521, 2.3873801231384277, -0.16485945880413055, 0.004911845549941063, 0.7113736867904663, 0.27022305130958557, -0.33073192834854126, 1.7448124885559082, 0.2979282736778259, -0.3827495872974396, 0.2824428379535675, 0.3298446536064148, 1.3234492540359497, -1.7720261812210083, -0.7174473404884338, -0.0018525727791711688, 0.9925536513328552, 0.23561662435531616, -0.7624301910400391, 0.314229279756546, -0.14295901358127594, -0.5347375273704529, 0.31909510493278503, -0.996600329875946, -0.4650675058364868, 0.44241002202033997, -0.9745826721191406, -0.4001539647579193, 0.5016045570373535, -0.7359535098075867, 1.1872888803482056, 0.7213452458381653, -1.8306430578231812, -1.7110649347305298, -0.7294983267784119, 1.2001889944076538, 0.9883069396018982, 0.8788681030273438, -0.34872785210609436, -1.349952220916748, 0.16742101311683655, 0.04675563797354698, -1.6919864416122437, 0.5967614650726318, 1.0586609840393066, 1.1109577417373657, 0.6037890315055847, -0.9918959736824036, -1.1561328172683716, 0.6408962607383728, -0.31065866351127625, -0.54825758934021, -0.7061024308204651, 0.4340195953845978, 0.0967961773276329, 0.43886035680770874, 0.49631229043006897, 0.5297636389732361, 0.11511632800102234, 1.0058256387710571, -0.7416606545448303, 0.8752107620239258, 1.700871467590332, -1.2850910425186157, -0.9873684644699097, 0.43605613708496094, -0.5002350807189941, -1.1974639892578125, 1.0178184509277344, -0.4013623595237732, 1.248481273651123, 1.1386516094207764, -0.5354238152503967, -0.3582850694656372, 1.583953857421875, -1.0589731931686401, -0.4678597152233124, 0.7699010372161865, 0.604689359664917, 0.21905075013637543, 1.1209248304367065, -0.30025190114974976, 1.2632908821105957, -0.7495266795158386, -0.07015534490346909, -1.8556357622146606, 0.019117986783385277, -0.8879073858261108, 1.2408651113510132, -0.38788488507270813, 2.2460591793060303, 0.5770396590232849, 0.7368671298027039, -0.07824600487947464, 0.7858284711837769, 1.678043007850647, 0.9286178946495056, 0.5781143307685852, -1.1763395071029663, -0.7651751041412354, -1.3216619491577148, -0.206752210855484, 0.8572270274162292, -0.8816669583320618], [-1.087752103805542, 0.1210077702999115, -1.1779009103775024, 0.807658314704895, 1.294760823249817, -0.3228805661201477, 0.4832446277141571, -1.216742992401123, -1.428900122642517, -1.8267185688018799, -0.5846224427223206, 1.0123697519302368, 1.476732611656189, 0.368838369846344, 1.243884563446045, 0.2400854527950287, -0.27710551023483276, -0.4019426703453064, 0.6608864068984985, 0.6284862160682678, 0.3239459693431854, -0.022017119452357292, -1.724495768547058, 0.4602448046207428, -1.5156302452087402, 1.2076635360717773, -0.14146597683429718, 0.9587494730949402, 0.6637285947799683, 0.0819365531206131, -0.7785589694976807, -1.5196611881256104, -3.9958574771881104, 0.5855610966682434, 0.2538608908653259, 0.28896114230155945, 1.3491356372833252, 0.21480970084667206, 0.04057493433356285, 0.695155680179596, -0.7840923070907593, -0.4781021773815155, -0.20463885366916656, -0.058334920555353165, 1.1240791082382202, -3.7030856609344482, 1.0652533769607544, -1.0086017847061157, -0.18852116167545319, 1.0060769319534302, 0.8361237645149231, -0.23070327937602997, 0.3267008662223816, -0.2544056475162506, 0.712260901927948, 0.26894187927246094, -1.377916693687439, 1.4663426876068115, 0.9385696053504944, 1.8761467933654785, 0.6993346810340881, -0.6292044520378113, -0.5368969440460205, -0.3317667543888092, -0.5841402411460876, -0.11762888729572296, -0.7551400065422058, -0.14959408342838287, 0.3861151933670044, 2.017840623855591, 0.866055428981781, 0.24438396096229553, -0.5699374675750732, 0.20346765220165253, -1.2224714756011963, -1.6325852870941162, 1.3078172206878662, 0.22668643295764923, 0.1596440076828003, 1.363579511642456, 0.6799381375312805, 0.60697340965271, -0.35937273502349854, 0.15855367481708527, -1.2703347206115723, -1.601343035697937, -0.12147840857505798, -0.6975237727165222, 0.14174218475818634, -1.5709840059280396, -1.0937371253967285, -0.5283321142196655, -0.25244054198265076, 2.179597854614258, -1.22087562084198, -0.8319138884544373, -0.029567234218120575, 0.4331015646457672, 1.7516865730285645, -0.614224374294281, -0.6728429794311523, -0.4173685610294342, -0.5623027682304382, -0.03574472665786743, 0.5603302717208862, -1.7433083057403564, -0.05895869433879852, -0.3844834864139557, -1.057032585144043, -0.755501389503479, 0.4657852053642273, 0.33331385254859924, 0.7458789348602295, 0.8197880387306213, -2.2292253971099854, 1.2529433965682983, -1.6186587810516357, -0.8872960209846497, 1.330649971961975, 1.2761636972427368, 0.5006352663040161, 0.9362102746963501, -0.6604653596878052, -0.4892619848251343, 1.6644859313964844, -1.1251119375228882, 1.0466279983520508, -1.4041476249694824, 1.7202739715576172], [-1.470414161682129, 2.2154805660247803, 0.04531778395175934, 1.7429790496826172, 0.14870893955230713, -1.5641852617263794, 0.6284257769584656, -0.3379873037338257, -1.3917478322982788, -0.07613428682088852, 0.4478524923324585, 0.41962721943855286, 1.6328866481781006, 0.290242463350296, -0.6228474378585815, -0.46537894010543823, 0.688870370388031, -0.18664705753326416, 1.4985902309417725, -0.5006904006004333, -0.18474026024341583, -0.826177179813385, 1.1787973642349243, 1.8445985317230225, -0.16769272089004517, 0.5857941508293152, 0.023490412160754204, 0.7526764869689941, 0.7667685151100159, -2.8094749450683594, 0.5620677471160889, -1.4318761825561523, -1.4285213947296143, -1.5606517791748047, -0.5975271463394165, -1.7599108219146729, -1.2770888805389404, -0.49811437726020813, 0.13176678121089935, 1.004111886024475, 0.637055516242981, 1.475699543952942, 0.8165208697319031, 0.02160027250647545, -1.1874750852584839, 0.17277792096138, -0.4021015465259552, 0.8519859910011292, 0.9571532011032104, 0.7617993354797363, 0.3950365483760834, 0.38710817694664, 0.8897877931594849, 0.13712617754936218, 1.4852625131607056, 1.2688519954681396, -0.4987998604774475, 0.6060127019882202, -1.611249327659607, -0.9631136059761047, 1.1351404190063477, 1.6747506856918335, -0.7950581908226013, -0.5760340690612793, -0.9574973583221436, 0.03949573263525963, -0.3987363278865814, 0.7111208438873291, 0.021914968267083168, 0.9031031131744385, 0.48966094851493835, 0.8626111149787903, -0.8452181220054626, 0.794061005115509, -0.6495321989059448, -1.1673192977905273, -0.7112459540367126, 0.32338079810142517, 0.009552416391670704, 1.173749566078186, 1.279714822769165, 0.9204591512680054, -2.0679688453674316, 2.0816586017608643, 0.2263939529657364, -0.8171757459640503, 1.3365789651870728, 0.17102135717868805, -0.7535431385040283, 0.17608065903186798, -0.7591038942337036, -0.8079488277435303, -0.7609689831733704, 0.7681035399436951, 0.14061278104782104, -0.510013222694397, -0.4182732403278351, -0.03487945720553398, 0.9498330354690552, 1.2757370471954346, -0.4799747169017792, 2.261762857437134, 1.2451430559158325, -1.0569888353347778, -0.30656859278678894, -0.49205881357192993, -0.7535597085952759, 0.3034912645816803, 0.1636151373386383, -0.39000236988067627, -0.18900339305400848, 0.4166505038738251, -1.2239168882369995, -0.4661034643650055, 0.7181527018547058, 0.7394734025001526, 1.563522458076477, -0.11841510981321335, 1.9916834831237793, 0.461807519197464, -0.5312358140945435, 0.41936585307121277, -0.15026907622814178, 2.417956590652466, -0.9139329791069031, 1.1243014335632324, 0.2728113830089569, 0.9664872884750366, 0.14437498152256012], [-0.5159562826156616, 0.6308929324150085, -1.710810661315918, -1.045326828956604, 0.5019419193267822, 0.8145151734352112, -1.6353634595870972, 0.9465926885604858, 0.7238711714744568, -1.3923112154006958, 0.8316745758056641, 0.8663724064826965, 1.6628592014312744, 0.8000648021697998, 0.2136022001504898, 0.5161838531494141, 0.589340329170227, 0.021189866587519646, 1.1676517724990845, 1.419670820236206, 1.4041863679885864, 0.8120568990707397, -0.404194176197052, 0.6048542857170105, 0.648734450340271, -1.7310326099395752, -1.7385920286178589, -0.5055832266807556, -0.9219039082527161, -0.1105668768286705, -0.444286972284317, -0.0658821314573288, 1.2140069007873535, -0.35666579008102417, -0.3096155822277069, 2.282153606414795, 0.3180690109729767, -0.36600059270858765, 1.3336952924728394, 0.16595406830310822, -0.6878345608711243, -0.12079896777868271, -1.3290859460830688, 0.17954620718955994, -1.0164437294006348, -0.41450947523117065, -1.4138760566711426, -0.1285644769668579, -1.8619089126586914, -0.9469477534294128, 0.8180392980575562, 0.5627294182777405, 0.5524033904075623, -0.13612100481987, 1.0773212909698486, 0.40994009375572205, -0.623443067073822, 1.646209716796875, -0.2605370283126831, -0.3210139572620392, -0.29592910408973694, -0.9202490448951721, 1.8492872714996338, -0.36922791600227356, -0.1586836278438568, -0.40667927265167236, -0.12957021594047546, 0.599829375743866, -0.8175666928291321, -2.759775161743164, -2.1900992393493652, 1.424452543258667, -0.04040538892149925, 1.5541809797286987, 0.45759138464927673, 1.4435571432113647, -0.7519930601119995, -0.4333030581474304, 0.07506159693002701, 0.4946518540382385, -0.9103080630302429, -0.577007532119751, 1.0049468278884888, -0.3788091540336609, -0.1643163561820984, 2.2428817749023438, -1.2611550092697144, 0.6254281401634216, 0.6680178046226501, -0.2523616552352905, 0.9080637097358704, -2.3979251384735107, -1.709561824798584, -1.599412441253662, 0.34410399198532104, 1.8742785453796387, 2.316189765930176, 0.1072256937623024, -0.43203479051589966, -1.8956685066223145, -0.5941433906555176, -0.4085860848426819, -1.223297357559204, -0.7057774066925049, -1.0581800937652588, -1.5204246044158936, 0.3287530541419983, 0.6521737575531006, -0.15082122385501862, 1.5851353406906128, -1.2639689445495605, -0.42223235964775085, -0.3936162292957306, 2.283280849456787, 0.2404373586177826, -0.10179973393678665, 0.41127634048461914, -0.47724980115890503, -0.48224562406539917, 0.517960250377655, 0.6329753398895264, -0.23156383633613586, 0.881837010383606, -0.23230214416980743, -0.6670099496841431, -1.659639835357666, -0.07446276396512985, 1.411569595336914, 0.9336410164833069], [-0.4424424469470978, -1.492790699005127, 0.09187011420726776, -0.06493765115737915, 0.025423606857657433, 1.2277477979660034, 1.7644916772842407, 0.21309266984462738, 2.611107349395752, 0.0734008327126503, -0.8724744915962219, -0.9057843685150146, 0.5788464546203613, -0.7888213396072388, -0.8693632483482361, 0.8307135701179504, -0.5240738987922668, -0.1657477170228958, -0.25536274909973145, -1.6437846422195435, 0.8021522164344788, -0.6497378945350647, -0.08643075078725815, -1.439963459968567, 0.0559946671128273, 0.0015565702924504876, -0.5835208892822266, 0.6995103359222412, 0.09060514718294144, -0.37975040078163147, -0.24879571795463562, -1.8481547832489014, 1.0749952793121338, 1.2306894063949585, -1.213599443435669, 1.0230138301849365, -1.9616608619689941, 0.823567271232605, -1.0675451755523682, 1.3602488040924072, -1.1224822998046875, -0.6871034502983093, 1.1792995929718018, -0.44119521975517273, 0.6115877032279968, -2.063523769378662, 0.46780598163604736, -0.28098517656326294, 0.43127724528312683, -1.026678204536438, -0.13991446793079376, -0.36533457040786743, -1.8354705572128296, 0.12279251962900162, -1.3130996227264404, -0.9112763404846191, -0.5082747340202332, 2.5214760303497314, -0.20267018675804138, -2.148172616958618, 1.3497040271759033, -0.4614129960536957, -1.3417375087738037, 0.988466203212738, 2.2700657844543457, -1.1919002532958984, 0.3995908200740814, 0.35073620080947876, 0.9203018546104431, 0.20828191936016083, 0.14227396249771118, 1.5511722564697266, -0.5337713956832886, 1.3444161415100098, 0.7435154318809509, 0.5262674689292908, 0.881160318851471, 1.1229954957962036, 0.016296813264489174, -2.466809034347534, 1.318079948425293, -1.6879637241363525, 0.1276070922613144, 1.0497792959213257, -0.9252203106880188, -1.1165170669555664, -1.3554375171661377, -1.5351738929748535, -0.042112141847610474, 0.5198254585266113, 1.7355310916900635, -2.0224196910858154, 0.5251171588897705, 0.645247757434845, -0.7995682954788208, -0.4328662157058716, 1.4871838092803955, -0.2713133990764618, -0.3462292551994324, 0.7109583020210266, 0.5692555904388428, -1.4964523315429688, -3.116894483566284, 0.8671886324882507, -1.4014549255371094, -0.2055843025445938, 0.24067406356334686, -1.2431927919387817, 1.1450763940811157, 0.1758159101009369, 1.6979997158050537, -0.7671384215354919, -1.2626575231552124, -2.0186023712158203, -0.685680091381073, -0.6637633442878723, -1.2339744567871094, 1.724103331565857, -0.4798758029937744, -0.253393292427063, -0.17642876505851746, -1.4576518535614014, 0.3269648253917694, 0.24328282475471497, -0.17782141268253326, -1.3343861103057861, -2.1521596908569336, -0.5957682132720947, 0.4951211214065552], [-0.6839128732681274, 1.5616569519042969, -0.6327240467071533, -0.6541017889976501, -0.43864017724990845, 0.2523191273212433, -0.06564532965421677, -0.3440038561820984, -0.8067683577537537, -0.7305118441581726, 0.715943455696106, -0.4203864336013794, -1.6943821907043457, 1.228366494178772, 0.426422655582428, -0.48107242584228516, 0.35727521777153015, 0.486661434173584, 1.1850619316101074, 0.5564273595809937, -1.142682671546936, -0.16639219224452972, 2.4558725357055664, 0.15056809782981873, 0.18164710700511932, 0.1078042984008789, -1.0881834030151367, 1.8131004571914673, 0.5664157867431641, 0.6756008267402649, -1.0801141262054443, 1.1626261472702026, 0.21729741990566254, -0.9332181811332703, 0.21343183517456055, 0.5450267791748047, 0.32170817255973816, -0.8718365430831909, 0.9486256241798401, 1.032829761505127, -0.7228475213050842, -2.1480040550231934, -1.5442193746566772, 0.49139419198036194, -0.43576425313949585, 0.6365578770637512, 1.5721536874771118, 1.4111007452011108, 0.04132958874106407, 0.4537833631038666, -1.9462617635726929, 0.2970273792743683, -0.9477919936180115, 1.7082899808883667, 1.5207792520523071, -2.0258820056915283, 0.2630527913570404, 0.9664539694786072, 1.0022305250167847, 0.7166682481765747, 0.7772620320320129, -0.3756740391254425, -0.7732664346694946, -0.38835909962654114, -1.5864390134811401, -0.6761956810951233, 1.0307602882385254, -0.058501020073890686, -0.22435623407363892, 0.45329830050468445, 0.2421458661556244, 0.04965505003929138, -1.0961329936981201, 0.7680942416191101, 0.5176073908805847, 0.5412290096282959, 1.2776541709899902, 1.519662857055664, -0.4390125572681427, 0.45024242997169495, -1.4484821557998657, 1.0034613609313965, 2.536236047744751, 0.2462233603000641, 0.2160739004611969, 0.2592187523841858, 1.1698945760726929, 0.001883885357528925, -2.2359399795532227, 0.2222454994916916, -0.9712175726890564, 0.6524903774261475, 0.29759758710861206, 1.5689562559127808, 1.3309814929962158, -0.30337873101234436, -0.20229126513004303, 0.9504897594451904, 0.31205424666404724, -0.5293371081352234, 0.33456215262413025, -0.5335721373558044, 0.5733925700187683, 0.7498736381530762, -0.9534133672714233, 0.44708189368247986, 0.27340835332870483, 0.6723991632461548, 0.07387705147266388, 0.43658098578453064, 1.6293025016784668, 1.0292688608169556, -0.28807833790779114, 1.433415412902832, -1.2708793878555298, 0.8030487895011902, 1.7226879596710205, 0.11584111303091049, 2.644967555999756, 0.7001879811286926, -0.5495427250862122, -1.959844708442688, -0.38854071497917175, 0.12324710935354233, -0.1396746039390564, -0.13530392944812775, -0.2291993349790573, -1.0527185201644897, 0.0484519861638546], [0.30523768067359924, 0.16409502923488617, 0.17543329298496246, 1.4179540872573853, 0.5869486331939697, -0.9550046920776367, 2.1844284534454346, -0.6743069291114807, -0.4606419503688812, 1.2994890213012695, -2.0308098793029785, 0.7021802663803101, 0.4389514625072479, 0.22398455440998077, -0.08410262316465378, 1.5820223093032837, -1.3196942806243896, -1.5661391019821167, 0.1243923082947731, 0.8681818842887878, -0.8734061121940613, 1.036889910697937, 1.062758207321167, -0.809441864490509, 1.6694600582122803, 0.2630680501461029, 0.03014775738120079, 0.08844246715307236, -1.4927749633789062, 2.0895836353302, 0.7338172793388367, -0.7412212491035461, 0.7246914505958557, -1.3419419527053833, -0.21372869610786438, -0.7109686732292175, 0.16926327347755432, 0.8687452673912048, -0.7107607126235962, -1.2291831970214844, -0.47937875986099243, -0.0847732350230217, 1.786242127418518, -0.634533703327179, -0.28831034898757935, -1.713761568069458, 0.49991491436958313, -0.3554547429084778, -0.6238887906074524, 0.7568246126174927, 1.6400656700134277, -0.048027485609054565, 1.0916874408721924, -1.4873600006103516, -1.8682208061218262, 0.8649414777755737, 2.778231143951416, 0.9269989728927612, -0.9481923580169678, -0.2811693847179413, 1.095517635345459, 1.3250428438186646, 0.7014946341514587, -1.2097464799880981, 0.41305217146873474, -1.0198091268539429, 0.07830266654491425, -1.3110451698303223, -0.6869496703147888, 0.9660900235176086, -0.7194518446922302, 0.15504586696624756, -1.152746319770813, -1.2463864088058472, -1.3770402669906616, 0.7968844771385193, 0.013684024102985859, 0.1876126229763031, -1.6245999336242676, -0.890596866607666, -0.395648330450058, -0.8617584705352783, 0.6075843572616577, 0.16237957775592804, 0.1627558022737503, 1.6825748682022095, 0.37652066349983215, -0.5926470160484314, 0.2605261504650116, -1.0737892389297485, -0.5744485855102539, -0.3158617913722992, 0.5926972031593323, 0.4637463390827179, 0.5663618445396423, -0.16552433371543884, -0.31234315037727356, -0.6127638816833496, 0.1615474969148636, -0.43931907415390015, -0.5766419768333435, 0.6031794548034668, -0.6119616031646729, 0.9079257249832153, 2.3733479976654053, 0.30327680706977844, 0.19509491324424744, 0.4399552047252655, 0.6764968633651733, -2.347966432571411, -0.4877985119819641, -1.1844900846481323, -0.8114584684371948, 0.518549382686615, -0.28119969367980957, -0.28456297516822815, -0.9931765794754028, 0.6613656282424927, 0.7714983224868774, 0.5418853163719177, 0.4056118428707123, 0.1401159167289734, 0.1680092066526413, 0.6454890370368958, 0.36936745047569275, -1.3821001052856445, -0.2571304142475128, -0.8277248740196228, 0.36424627900123596], [1.1272377967834473, 0.39077192544937134, -0.8455878496170044, 0.6629963517189026, 0.5307710766792297, -1.4181671142578125, 1.237155556678772, -1.177582859992981, 1.462530493736267, 0.9527474641799927, -1.5682263374328613, -2.9431636333465576, -1.1203898191452026, 1.5507999658584595, 0.5450505614280701, -2.5209314823150635, 0.43430474400520325, 0.35146740078926086, -1.2053849697113037, -1.0092772245407104, 0.36394548416137695, 0.11425358802080154, -1.776108980178833, -0.9786970019340515, 0.38662558794021606, 2.584929943084717, -0.6019312143325806, -0.654173731803894, 1.2081416845321655, -1.0808948278427124, 0.09262409806251526, -1.3529249429702759, 1.3415783643722534, -1.1772983074188232, -1.437833309173584, 0.13360515236854553, -0.1898173838853836, 0.7233164310455322, 0.6565392017364502, -1.6008127927780151, -0.9388996958732605, -0.12057354301214218, -0.4534511864185333, -0.6347350478172302, -0.5266680121421814, 1.7862775325775146, -0.3267284333705902, 0.48800474405288696, -0.028527969494462013, -0.1467224806547165, 1.5548653602600098, -0.8088404536247253, 0.2320527583360672, 0.5583425164222717, -0.663345456123352, -1.2662194967269897, 0.08523783087730408, -1.9858136177062988, 0.35340461134910583, 0.5165443420410156, -0.356192946434021, -0.08487528562545776, 1.5128690004348755, -0.08259638398885727, -0.20336580276489258, -1.3031954765319824, -0.5322887897491455, -0.08637381345033646, -0.003745398484170437, -0.6395564079284668, -0.25143110752105713, 0.7627279162406921, 0.0446401983499527, 1.0730232000350952, -0.813093900680542, 1.1798415184020996, -1.236307978630066, 0.601335346698761, -0.7794163823127747, 0.6322251558303833, -1.3654394149780273, 1.0361460447311401, -2.0496513843536377, -0.8256679773330688, -1.1928497552871704, -0.6394283175468445, 1.2554818391799927, -0.0014116597594693303, 0.8498424291610718, 0.40366604924201965, 0.45219263434410095, 0.7689512968063354, 0.26416662335395813, -0.44938409328460693, -1.1140705347061157, -1.5111974477767944, -1.123747706413269, -0.15922178328037262, -0.9752500653266907, 0.16387714445590973, -0.8801155090332031, 0.587421715259552, 0.1360194832086563, 2.3291070461273193, -1.1417185068130493, -0.05434507876634598, -1.8243529796600342, -0.420717716217041, -1.3071240186691284, -1.1116305589675903, 0.5744498372077942, -0.24024498462677002, -1.3621869087219238, 0.1698034554719925, 0.30134153366088867, 1.3772151470184326, -1.9709795713424683, 0.1869240552186966, -1.9381870031356812, 1.1335430145263672, -0.18439331650733948, -0.7921221256256104, -0.688808798789978, 0.5393908023834229, -0.5353244543075562, 0.25837695598602295, 0.2486640065908432, -0.14278382062911987, -0.06507658958435059], [0.8144171237945557, 0.33399948477745056, -0.323359876871109, 0.1683209091424942, -0.003771262476220727, -1.825271487236023, 0.49682679772377014, -0.23168163001537323, -0.006272976752370596, -2.394583225250244, 0.6386803388595581, -0.821539044380188, -0.7252387404441833, 1.2514034509658813, 0.1731593757867813, 0.22816406190395355, -0.8469628691673279, -0.03297659009695053, 0.05255948379635811, -0.2487509399652481, -0.6760173439979553, 1.620389699935913, 2.1513113975524902, -1.419607162475586, 0.6851500868797302, -1.0608537197113037, -0.5061925053596497, -0.0013268403708934784, -2.1066548824310303, 0.34218668937683105, 1.633660078048706, 1.5822970867156982, 0.08630847185850143, -1.041716456413269, 1.2300536632537842, -2.736492156982422, -0.7412735223770142, -1.1513240337371826, 0.43904048204421997, -1.526742696762085, -0.4717366397380829, 0.8885946273803711, 0.02264496311545372, -0.2833142578601837, -0.6490253806114197, -1.145084023475647, 1.855147361755371, -0.04216710478067398, 0.6408266425132751, -0.836829423904419, 2.189495801925659, 0.507067084312439, 2.299781322479248, 1.2139233350753784, 0.6470023393630981, -1.172057867050171, 1.5533020496368408, -0.4418565630912781, -0.28979969024658203, -1.2608915567398071, -1.9594306945800781, 1.0496330261230469, -0.4746082127094269, -0.5638015270233154, -0.8187609314918518, -0.7578572034835815, -1.3199268579483032, -1.5924338102340698, 0.5641428828239441, -0.42522555589675903, 1.099058747291565, -0.3347155451774597, -1.6495050191879272, 1.158813238143921, -1.3037605285644531, 1.3763145208358765, 0.1172930970788002, -2.0866875648498535, -1.3271825313568115, 1.1765202283859253, 1.3236205577850342, -0.6301635503768921, -1.4079488515853882, -0.9942446351051331, 0.5793226361274719, -0.717826247215271, -0.27188611030578613, 0.036778055131435394, -0.7183948159217834, -0.36728304624557495, 1.8116453886032104, 1.8369256258010864, -0.32092803716659546, 0.1468828022480011, 0.649635374546051, -0.20295515656471252, -0.9851571321487427, 0.16577662527561188, -1.4588655233383179, 1.121097207069397, -1.2444654703140259, 0.8057871460914612, 0.10917913913726807, 0.42093661427497864, 0.642296314239502, 0.10554099828004837, -0.5509689450263977, 0.6469801068305969, -0.5053367018699646, -0.06913972645998001, 0.7634716629981995, -0.8733514547348022, 0.5669369697570801, -0.36789470911026, -0.2537377178668976, -0.96658855676651, -0.6032887697219849, -0.0238965917378664, 0.4663035571575165, 0.09351155906915665, -0.44156691431999207, -1.6134936809539795, 2.3171775341033936, 0.11153778433799744, -0.19611401855945587, -0.005712234415113926, -0.5109821557998657, -0.5599254965782166, 0.23165079951286316], [-0.013127932325005531, -0.4337291717529297, 0.8259779810905457, 0.30044275522232056, -0.6188668012619019, 0.9483215808868408, 0.4405975043773651, 2.228705644607544, -0.7950741648674011, -0.09956695139408112, 0.7731630206108093, 0.012201061472296715, 2.3099582195281982, -2.6559619903564453, -1.7480993270874023, -0.9415683746337891, -1.3581711053848267, 0.6477339267730713, -0.20754557847976685, -1.3395980596542358, -1.0153905153274536, -0.5740510821342468, -0.40249285101890564, 1.275625467300415, -1.7495051622390747, -1.1092556715011597, 1.5277581214904785, 1.7620108127593994, -0.133809432387352, 0.30612263083457947, 2.0515520572662354, -0.5085970759391785, -0.15212133526802063, 1.4192322492599487, 0.5332434773445129, 0.1936044543981552, -0.06849679350852966, 1.6290709972381592, -2.057264804840088, 0.4477139711380005, -0.018182337284088135, -2.1729767322540283, -2.2523281574249268, 1.481648325920105, -0.8997902274131775, 1.5512974262237549, 1.4813898801803589, -0.09038303047418594, 0.08899066597223282, -0.5800789594650269, 0.5392615795135498, -0.007085090037435293, -1.9853663444519043, 0.9465290904045105, -0.5610827803611755, -1.2650493383407593, 2.246807336807251, -1.048707365989685, 0.47001972794532776, -0.8608145713806152, 1.4230420589447021, 0.9153986573219299, -1.4723316431045532, -1.0520540475845337, -1.0159022808074951, -0.4060569703578949, -1.1348416805267334, 1.369897484779358, 0.2075541913509369, -0.8061133027076721, 0.43660426139831543, 0.5331993103027344, 1.410400629043579, 1.0459048748016357, -0.08749190717935562, -1.0118577480316162, 0.2949371635913849, -0.61463463306427, 1.3013858795166016, 1.6827735900878906, -0.15281930565834045, -0.019893961027264595, 0.3489076793193817, -0.255118727684021, -1.277671217918396, 0.9765846729278564, -0.6808706521987915, -1.889746069908142, -0.9728296399116516, 0.9888368248939514, 1.5153712034225464, -0.6798254251480103, -1.2101192474365234, 1.6604160070419312, 0.5058869123458862, 1.5291403532028198, 0.698293149471283, -0.116219662129879, 1.215364694595337, -0.48186373710632324, 0.5372816920280457, 1.5412726402282715, -0.9688635468482971, -0.05792118236422539, 0.4182014763355255, -0.6788821220397949, 0.15637922286987305, -0.017534876242280006, -2.5729217529296875, -0.2568133473396301, 1.1197737455368042, -1.4107438325881958, -0.526703417301178, 0.21191059052944183, 0.4153052866458893, -0.36119115352630615, 0.05339309200644493, -0.6456504464149475, -0.5658442378044128, -1.9278156757354736, -1.6367506980895996, -1.7147289514541626, 0.217569500207901, 0.7472773194313049, -0.09784797579050064, -0.07708744704723358, -0.7345110177993774, 1.0889047384262085, -0.4861226975917816], [-0.048358138650655746, 1.3831546306610107, -1.8314828872680664, -0.5102205872535706, -0.7853803634643555, -0.3456813395023346, -0.5247249603271484, -0.9460670948028564, 0.9498456716537476, -0.6851121783256531, 0.6067283153533936, 1.0083684921264648, 0.19724883139133453, -1.114621877670288, 0.009948925115168095, -0.7308409810066223, 1.446635127067566, 1.8785343170166016, -0.6239716410636902, 0.5713210701942444, 0.6103682518005371, -0.6399599313735962, 1.1746224164962769, 1.3728049993515015, -0.322250634431839, 0.7760717272758484, -0.4723188877105713, 1.1847975254058838, 2.518594980239868, -1.1064382791519165, -0.7153862118721008, -1.2116687297821045, -1.4409056901931763, 0.4203360080718994, -1.2582511901855469, 1.297049880027771, 1.954232096672058, -1.4539135694503784, 0.3257581293582916, -0.4571775496006012, -0.5274479985237122, -0.7152888774871826, -0.7949279546737671, -0.9064967632293701, 0.6840713024139404, 0.2197873890399933, 1.2810782194137573, -2.7210795879364014, 0.08570532500743866, 0.11771559715270996, 0.1463875025510788, 0.2611537575721741, -1.1391957998275757, -1.3546533584594727, -1.2066701650619507, -0.45492827892303467, -1.0027692317962646, 1.4590195417404175, 0.5095212459564209, -0.24156196415424347, 2.5590755939483643, -0.10711145401000977, -0.07445527613162994, -0.16224318742752075, 0.3834749162197113, -0.07819807529449463, 1.021134853363037, 1.3165276050567627, 0.28715741634368896, 0.8301218748092651, 0.515094518661499, -0.2162022590637207, 0.3637450635433197, 1.363074541091919, -1.0505577325820923, 0.32864901423454285, -0.8980333209037781, 1.197674036026001, 1.1930876970291138, -1.2911416292190552, -0.11319679021835327, 0.27205902338027954, 0.7218673229217529, -1.398400902748108, 0.3344351649284363, 0.23927699029445648, -1.863397240638733, 0.6755693554878235, -0.5233076214790344, -0.7948485016822815, -0.29229143261909485, 2.5583465099334717, -0.32108965516090393, 1.0090969800949097, 1.7759943008422852, 0.03684290125966072, -0.14473780989646912, -2.036078453063965, -0.7549512982368469, 1.2805055379867554, -1.177956223487854, -0.34910696744918823, -1.3089487552642822, 0.15369006991386414, 1.0230305194854736, -0.5810707211494446, -0.4594923257827759, 0.6410141587257385, 0.3815159201622009, -0.6156188249588013, -0.5204249620437622, -0.13781936466693878, -0.6608442664146423, -0.3088705241680145, 0.834073007106781, 1.1762480735778809, 0.966087818145752, 1.4647448062896729, -0.9936314225196838, -1.4648526906967163, 1.0661429166793823, -0.46527284383773804, -0.3844962418079376, 1.0368587970733643, 0.4398319125175476, -2.0045745372772217, 1.3597031831741333, -0.43540158867836, -0.01945977285504341], [-1.4274390935897827, -1.9782203435897827, -0.43635761737823486, 0.4256666600704193, 0.8736466765403748, -0.8894006013870239, -0.7965401411056519, 0.08775603771209717, 1.8979551792144775, 0.822804868221283, 0.2753349542617798, -0.9306284785270691, 0.2736477553844452, 1.596512794494629, 0.871618926525116, 0.04534931853413582, -0.05017168074846268, -0.8197749853134155, 1.970963478088379, 0.7654749751091003, 1.4751650094985962, -1.5991122722625732, 1.851227045059204, -0.3905058801174164, 0.3012382686138153, -1.8721543550491333, 0.5950695276260376, 1.6568669080734253, -1.6395217180252075, -2.365194320678711, -0.23977217078208923, 0.5291514992713928, 1.6424797773361206, -1.1149331331253052, 0.017238864675164223, -2.138772964477539, 0.12209048122167587, -1.0721327066421509, -1.5943924188613892, -0.9417226910591125, -1.2274932861328125, 0.3361346125602722, 0.657481849193573, 0.6661604046821594, -1.6752325296401978, 0.5484971404075623, -0.7350730299949646, -1.9366956949234009, -0.9928588271141052, 0.8200867772102356, 0.2887715995311737, 0.3161969780921936, -0.6766055226325989, -0.30170804262161255, -1.7914717197418213, 1.285650372505188, 0.6241146922111511, -0.22971975803375244, 0.7835026979446411, -0.6759510636329651, 0.19442644715309143, 1.7075414657592773, 1.527775526046753, 0.47489133477211, -0.7002826929092407, 0.7265158295631409, -1.0145056247711182, -0.007381310220807791, 2.6078386306762695, 0.06818100064992905, -0.19391365349292755, -1.0642116069793701, -0.32191649079322815, 1.4301798343658447, -1.473413109779358, -0.004160744603723288, 0.7427325248718262, -1.029884696006775, 0.15020442008972168, -2.178846597671509, 0.15405763685703278, -1.2296876907348633, 1.3187390565872192, -2.7071943283081055, -1.3538843393325806, -1.0683388710021973, 1.078039288520813, -1.35050368309021, -1.3380098342895508, -0.47825688123703003, -0.6964945197105408, -0.3017350435256958, -1.4628123044967651, -0.9087656736373901, 0.5964107513427734, 0.4400923550128937, 0.29300758242607117, -1.2850145101547241, -0.12162157893180847, 1.7182992696762085, 2.672860860824585, -0.8776992559432983, -1.637697696685791, -1.0266467332839966, -0.5927555561065674, 0.8749442100524902, -0.7401379346847534, 0.004318032413721085, -2.1330549716949463, -0.19076992571353912, 0.6225900650024414, -1.337719440460205, 0.5058315992355347, -2.323659658432007, 0.0041162120178341866, -0.671198844909668, -0.7373209595680237, -0.36373981833457947, 2.590338945388794, -0.08431505411863327, -1.0242846012115479, -2.128962516784668, -1.2206578254699707, 0.13241729140281677, -2.273000478744507, -0.26812058687210083, 0.37090957164764404, -0.4210326373577118, -1.2430880069732666], [0.9849981069564819, -1.314595103263855, 0.6606897115707397, 0.7699838876724243, -0.5378530025482178, 0.5519219040870667, -1.0120673179626465, 1.3843783140182495, 1.026917815208435, -1.5246455669403076, -1.1916837692260742, 1.5646783113479614, -0.19158893823623657, 0.9096128344535828, -2.26789927482605, 1.3236356973648071, 2.2002391815185547, 1.6509782075881958, 0.6927219033241272, -0.9237914681434631, 1.646883487701416, -0.10458679497241974, 1.4447286128997803, 1.9131933450698853, -1.2254970073699951, -0.2057114988565445, 0.7503715753555298, 1.758082389831543, 0.6492125988006592, -0.1673726886510849, 0.8489569425582886, 1.3952337503433228, -1.0725820064544678, -0.34055832028388977, 1.3786896467208862, 0.20916476845741272, -2.140498161315918, -0.34190434217453003, 0.50525963306427, 1.0195623636245728, 0.4611056447029114, 0.7179837822914124, -0.6034020781517029, 0.21443580090999603, 0.30879631638526917, -0.7562282085418701, 0.1051180437207222, -0.3005102574825287, -0.8260223269462585, 0.8518706560134888, 0.35828936100006104, -0.8058592677116394, 0.7565541863441467, -0.6786895394325256, 0.22722333669662476, 0.9613220691680908, -0.2841228246688843, -0.9811137318611145, 0.7947953343391418, -0.12229974567890167, -2.088303327560425, 0.8431922793388367, -1.3671056032180786, 0.18208998441696167, 1.0106875896453857, 0.15044499933719635, -1.7069191932678223, -0.33095288276672363, -1.2791565656661987, 0.6145366430282593, 0.17712579667568207, 0.16164851188659668, 1.571634292602539, 0.42526140809059143, -0.557859480381012, 1.1749956607818604, -0.39938679337501526, -0.3699941337108612, -0.018965253606438637, -1.169590950012207, -1.0629867315292358, 0.4205378293991089, -0.17097778618335724, 1.1017042398452759, 0.830030083656311, -0.7116724252700806, 1.4018356800079346, 1.6359864473342896, -0.03993959352374077, 0.694153368473053, 1.783153772354126, 0.09427589178085327, 0.03760109841823578, 0.11889554560184479, 0.7386507987976074, 0.9555947780609131, -0.07822098582983017, -1.614737629890442, 0.8685804605484009, 0.04478813707828522, -0.2073996514081955, -0.9960402250289917, 0.8580535054206848, 1.2234892845153809, 0.1988733410835266, -1.369183897972107, -1.928676724433899, -0.30089500546455383, -1.3751449584960938, 1.0597862005233765, 0.6516531705856323, -1.14291512966156, 0.36074045300483704, 1.228767991065979, 0.5303268432617188, -0.017792513594031334, 0.837579071521759, -1.2207207679748535, -1.3797122240066528, 0.052431292831897736, 0.2067536860704422, 0.4542962312698364, 0.7777252793312073, -1.3980001211166382, 0.48947328329086304, -1.8012406826019287, -0.2975054383277893, -0.7814396619796753, 1.5957095623016357], [0.355398952960968, -1.461436152458191, -0.27309027314186096, -0.26946213841438293, -0.14284943044185638, -1.259577989578247, 0.12660449743270874, -0.5594449043273926, -1.8854128122329712, -0.50328129529953, 1.570059895515442, -0.5352699160575867, 0.6328574419021606, 0.6097198128700256, 1.318326711654663, -0.10579914599657059, 0.6189385056495667, -0.1766906976699829, -0.4183211028575897, -0.6498657464981079, -2.0716350078582764, -0.1563565582036972, 0.3707309067249298, 0.3734339475631714, 0.20058602094650269, 0.07934260368347168, 1.021836757659912, 1.7493313550949097, -0.6754499673843384, -1.7924705743789673, 1.6369274854660034, -0.8715565800666809, 0.09831950813531876, -1.1509053707122803, 1.4577887058258057, -1.3464254140853882, 0.646462619304657, -0.4701582193374634, 0.7069961428642273, 1.4728686809539795, -0.025313938036561012, 0.3972940742969513, -0.10787607729434967, -0.7514139413833618, 0.02566520683467388, -0.83577960729599, -0.6951444149017334, -1.5284255743026733, -0.6307327151298523, -1.2627849578857422, -0.780762255191803, 1.4910856485366821, 1.5815576314926147, 0.7705625295639038, -1.7944775819778442, 0.8359977602958679, 1.4346954822540283, 1.5576688051223755, 0.6875826716423035, -0.2608558237552643, -1.037491798400879, -0.11082115769386292, 0.40362703800201416, 0.7013476490974426, -1.63639497756958, 1.079115629196167, 0.47867223620414734, 0.29461947083473206, 0.840357780456543, -1.4171299934387207, 1.6043100357055664, -1.017769694328308, 1.0955027341842651, -0.6937416195869446, -0.9525238871574402, 1.2761801481246948, 1.4023975133895874, -0.347436785697937, -0.900948166847229, 0.8402045369148254, -0.6908984780311584, -1.7641205787658691, -0.7920539975166321, -0.665235698223114, -0.6090167760848999, -2.669058322906494, 0.4446392059326172, 0.6346763372421265, 0.5744909644126892, 2.9949398040771484, 0.17217031121253967, -1.2035853862762451, 0.38553598523139954, 0.9909260869026184, 0.5804134607315063, -0.7981827855110168, -1.2708747386932373, 0.7969185709953308, -2.5989136695861816, 1.66972017288208, -0.10786266624927521, -0.369376540184021, 1.998591661453247, -0.24661175906658173, 0.22123253345489502, -1.5314249992370605, -0.12739799916744232, 0.1682305932044983, 2.1547365188598633, 0.6161372065544128, -0.536581814289093, 0.8278518319129944, 0.429931640625, 0.7852711081504822, 1.833816647529602, 0.48742929100990295, 1.8797736167907715, -0.7848559021949768, 0.08122388273477554, -0.4919300377368927, -0.7004457712173462, 0.9708160161972046, 2.2170233726501465, 0.2294400930404663, 0.7438425421714783, -0.8995276093482971, 0.1493224948644638, 1.236511468887329, -0.0690712034702301], [-1.1812665462493896, 1.5665979385375977, 1.8076682090759277, 0.24799413979053497, -1.3892199993133545, -1.4380966424942017, -1.377310872077942, -1.8033764362335205, -1.064144492149353, -0.8928623199462891, -0.7327409982681274, 1.2571253776550293, -0.14528563618659973, 1.362496256828308, -0.2761952579021454, -1.0699735879898071, -0.6239517331123352, 0.08083916455507278, 0.09302274882793427, 1.0122344493865967, -1.2767735719680786, -1.2192339897155762, -0.3958267569541931, -1.3888471126556396, 1.5728026628494263, -0.314557284116745, 0.9790816307067871, -0.8726637363433838, 1.0482450723648071, -0.1482795625925064, 1.3152574300765991, -2.092711925506592, -1.0853921175003052, 1.3354196548461914, -0.38327932357788086, -0.21071937680244446, -0.11649727076292038, 2.652279853820801, 0.45731186866760254, 0.32446780800819397, -0.6569455862045288, -0.3861048221588135, 0.9283831119537354, 0.8754011988639832, 0.1915959119796753, -0.23928532004356384, 0.2802410125732422, 0.9169520735740662, 0.5220487713813782, -1.1509356498718262, 0.649337649345398, -1.4407310485839844, 0.3087197542190552, 0.12137757241725922, -1.0499752759933472, 1.4970057010650635, 0.6547525525093079, -0.5499259233474731, -0.29917943477630615, -0.05853988602757454, -0.16924339532852173, 0.0954912081360817, 0.36389052867889404, 0.013721675612032413, -0.4526398479938507, -0.1412508338689804, 0.5726785063743591, -0.12715068459510803, 2.074099063873291, 0.10292651504278183, 1.160575270652771, 0.1335611790418625, -0.10399992018938065, -0.38750559091567993, -0.5453193187713623, 0.4976896643638611, 0.2589530646800995, 1.0979313850402832, -0.4002614915370941, 0.838798999786377, -1.1026419401168823, 0.34028223156929016, 1.3644064664840698, 0.8129088878631592, 0.6031701564788818, 1.4272832870483398, 1.3554123640060425, -0.5807017087936401, -0.3408730924129486, -0.49142369627952576, -1.2328099012374878, 1.0424721240997314, -1.9451919794082642, 0.9010335803031921, -0.29730114340782166, 0.2384157031774521, 0.8299934267997742, 0.09420283138751984, 0.8313047289848328, -1.395586609840393, -1.1309247016906738, -1.092175006866455, -0.015124023891985416, 1.1930910348892212, 1.1117229461669922, 1.3602992296218872, -2.532813310623169, -1.1132289171218872, 0.6176098585128784, 0.8339995741844177, -1.5470329523086548, 0.22909656167030334, -1.4481693506240845, 0.557756781578064, -0.4659493863582611, 1.3444263935089111, -1.1200042963027954, 0.21705012023448944, 0.07950890809297562, -0.9693726897239685, 0.2842373847961426, 0.9336127042770386, -2.0157110691070557, 1.5140537023544312, 0.7787078022956848, -0.25703340768814087, -0.20248085260391235, 0.08329324424266815, 1.0685772895812988], [-0.38408732414245605, -0.3677651286125183, -0.9542824625968933, -0.2780635952949524, -1.8732552528381348, 0.6422410607337952, -0.5745543241500854, 0.44216030836105347, 0.4414805471897125, -0.5161973237991333, -0.4967528283596039, 1.7284581661224365, -0.7014418840408325, 0.43047910928726196, -0.09400595724582672, 1.1026219129562378, 2.1636297702789307, 1.4238526821136475, -1.1516469717025757, 1.0081493854522705, 0.863614022731781, 0.07935671508312225, -1.197623372077942, -1.1837972402572632, 0.9093624353408813, 0.6519771814346313, -0.8446505069732666, -0.3069716989994049, -1.6136672496795654, 0.6838080286979675, 0.32494568824768066, 0.9633576273918152, 0.186776265501976, 0.10756570845842361, -0.5449378490447998, 0.22335697710514069, -0.09035535901784897, -1.0143752098083496, -0.4882476329803467, 0.6843318939208984, -0.06360892206430435, 0.9626159071922302, -1.2937697172164917, 0.35097572207450867, 0.08095639199018478, 1.3691407442092896, 0.12864452600479126, -0.274567574262619, -0.40774354338645935, -0.6924275755882263, -1.0235499143600464, 1.5949687957763672, -1.297250509262085, -0.8802807927131653, 0.09797642379999161, 0.5582307577133179, -0.6898078322410583, 2.4300270080566406, 0.23436924815177917, -1.2459802627563477, -0.6853581666946411, -1.601563572883606, -1.0866745710372925, -0.09235649555921555, -0.12007421255111694, 0.5196061134338379, 0.794999361038208, 1.7835808992385864, 0.4956049621105194, 0.16456139087677002, -0.7871176600456238, 0.5891866683959961, 0.1433081030845642, -1.1911969184875488, -0.39019736647605896, 0.2988358736038208, -0.13975907862186432, 0.4063071310520172, 0.34626853466033936, 0.16246114671230316, 0.9921492338180542, -1.1480305194854736, -0.47243475914001465, -0.9755983948707581, -0.7321381568908691, -0.2620871961116791, -2.239962100982666, 0.25646182894706726, 0.9916585683822632, -0.3533153831958771, 0.08255358040332794, 0.5812716484069824, 1.7729183435440063, -0.23931267857551575, 0.5738571882247925, 0.7379311919212341, 1.020532250404358, -0.17150495946407318, 0.0069053596816957, 0.0731801688671112, -0.9301049709320068, 0.47188663482666016, -0.13150960206985474, -0.8119098544120789, 1.4119837284088135, -1.1773409843444824, 0.3723911643028259, 0.3602182865142822, 0.3614892363548279, 0.7986975908279419, 2.4865646362304688, -0.20681630074977875, -0.09872347116470337, -1.137616753578186, -1.1935256719589233, -0.6954434514045715, -1.0325440168380737, 0.7923352122306824, -0.5809893608093262, 0.7547197937965393, 0.5717251896858215, -0.2182929962873459, -0.11979597806930542, 0.4223620593547821, -0.2683567702770233, -0.7322778105735779, 0.3847532272338867, 0.8644704818725586, 0.31384941935539246], [-0.6726146340370178, -0.40043357014656067, 0.15022528171539307, -0.6521303653717041, -0.059282559901475906, -0.266061931848526, -0.6095418930053711, -1.6215251684188843, 0.4101454019546509, 0.16164256632328033, 1.1351678371429443, 1.8795133829116821, 1.6285643577575684, -1.4049019813537598, -0.9026786684989929, 1.443234920501709, -1.3882867097854614, 0.2647593021392822, 0.07972361892461777, 0.17481669783592224, 1.3684227466583252, 0.11431223899126053, 0.6264832019805908, -0.08059132099151611, 1.6709259748458862, -1.4619826078414917, 0.5876061320304871, -0.938551127910614, -1.569108486175537, 0.16195131838321686, 0.05193760246038437, -0.12441091984510422, 0.35306796431541443, -0.25203293561935425, -0.9695895910263062, -0.5427804589271545, -2.456021547317505, -0.3445700705051422, 0.41808274388313293, -1.1501613855361938, 0.019711287692189217, -0.2682487964630127, 2.0996012687683105, -0.4305013120174408, 0.6236444115638733, 0.03846016526222229, -0.4910115897655487, -1.6272255182266235, -0.7110493779182434, 0.2500758171081543, 1.340736746788025, 0.28974851965904236, 1.4252375364303589, 0.9395076036453247, 0.958447277545929, -2.2472808361053467, 0.06082757189869881, 0.38485559821128845, 1.0197843313217163, -1.386155366897583, -2.465872287750244, -0.5059506893157959, 0.9476794004440308, -0.05493202432990074, -0.3237515091896057, -1.3213846683502197, -1.055702567100525, -0.7359914183616638, 0.2133972942829132, 1.5779839754104614, 0.17107637226581573, 0.8389151096343994, 1.3211429119110107, 0.1569812297821045, 0.3178861737251282, -1.1276130676269531, -1.2957350015640259, -1.2755870819091797, 1.7668997049331665, 1.8319169282913208, 0.9010920524597168, 0.16330565512180328, -0.34785225987434387, -1.116255760192871, -1.56040358543396, 1.1517318487167358, 0.568610429763794, 1.6121817827224731, -1.3005062341690063, 1.3338004350662231, 1.3026865720748901, -2.2667038440704346, 1.8975967168807983, -0.35309353470802307, -0.058228686451911926, -0.21529406309127808, 0.8294563889503479, 0.33758270740509033, 0.010835351422429085, -0.31588900089263916, 0.5345249772071838, -2.2121918201446533, -0.38743817806243896, -0.8639534711837769, 2.0191023349761963, 1.4268896579742432, -0.4662811756134033, -2.146857976913452, -1.1168314218521118, -0.2758404612541199, 0.4034334123134613, 1.3112910985946655, -0.27113857865333557, -1.4397196769714355, -0.696160614490509, 0.0005152903031557798, -0.5034937858581543, -1.3120207786560059, -0.12133491039276123, -1.7384333610534668, 1.3408135175704956, -2.0911288261413574, -0.12565718591213226, -0.0884750485420227, 0.21334262192249298, 1.356683373451233, -0.9748236536979675, 0.326063871383667, 1.170170545578003], [1.5198808908462524, -0.8635649085044861, 0.7770704627037048, 2.157069206237793, -0.13345763087272644, 0.8927378058433533, -1.2196078300476074, 0.470887690782547, 0.6938292384147644, 1.6361042261123657, -0.5886139869689941, 1.399514079093933, -0.22842666506767273, -0.42974817752838135, -1.4587969779968262, 0.2628924250602722, 1.4568684101104736, 0.5114383697509766, 0.6626471281051636, 0.7124015092849731, 0.7350497841835022, 0.016772253438830376, 0.22753186523914337, 2.7385501861572266, 0.6860719323158264, 0.5236474871635437, 0.7768320441246033, -1.4059804677963257, -1.2859872579574585, -0.10984773933887482, 0.11144615709781647, -0.6792181134223938, -1.0793946981430054, 0.44442152976989746, 0.3043701648712158, 0.04822733253240585, 1.1302454471588135, -0.16502544283866882, -0.10742738097906113, 0.6567085981369019, -0.9404662251472473, -0.7144818305969238, 0.5804181098937988, -0.4066310524940491, 0.6749303340911865, -0.43651217222213745, -0.4122375547885895, 1.5214850902557373, 0.3781895935535431, 0.48168981075286865, -2.1255600452423096, -0.04355819150805473, 0.5549217462539673, -0.494789719581604, 1.375283122062683, 0.10158658027648926, 0.47066766023635864, 0.28786787390708923, 0.7488371729850769, -0.6669362187385559, -1.7910377979278564, -0.16592912375926971, 0.6905907988548279, 0.37996283173561096, -0.8984349966049194, -0.6175729036331177, 0.4228745102882385, 0.5930436849594116, -0.2197328507900238, 0.17530103027820587, -0.9769876003265381, 0.2946828305721283, -0.023754363879561424, 0.47241708636283875, -0.6377760171890259, 0.39407339692115784, 0.5648408532142639, -0.11855079978704453, -0.7951602339744568, -0.1250811368227005, 0.17177002131938934, -0.7584478259086609, -1.5151546001434326, -1.3109699487686157, -0.8409002423286438, 0.3641127943992615, -1.2368814945220947, 0.25397658348083496, 1.2963721752166748, 1.21769118309021, -0.4586755633354187, -0.5785685777664185, -0.9856885075569153, 0.4125531315803528, 0.5150973200798035, -1.7257959842681885, -0.9392181634902954, 1.000175952911377, -0.4284723103046417, 0.17146477103233337, -0.025985678657889366, -1.2626756429672241, -2.587379217147827, -0.64862060546875, -0.5774713754653931, -1.594437837600708, 0.1930234283208847, -1.2802749872207642, -1.3553471565246582, 1.2993409633636475, -1.4893947839736938, 0.041680533438920975, 0.48126471042633057, -0.6811994314193726, -0.47027114033699036, 0.8401994705200195, -1.7653533220291138, 1.0462485551834106, -0.09349028021097183, 0.6283281445503235, -1.1868921518325806, -0.9818354845046997, -0.6205512285232544, -3.1886062622070312, -0.24212057888507843, 1.0468063354492188, -0.5341066718101501, 1.1963026523590088, 0.04240989312529564], [0.30657845735549927, 0.8721997141838074, 0.1812056303024292, -0.8748454451560974, -0.6947781443595886, -1.82668936252594, 0.6325170397758484, 0.24167801439762115, -0.7365321516990662, -1.9846513271331787, 2.9774069786071777, 2.173408269882202, -0.4162082374095917, 0.24859566986560822, 0.23338867723941803, 0.9630616903305054, 0.05834696441888809, -0.26315125823020935, -1.6896004676818848, -0.6614530086517334, 0.4628866910934448, -0.677419900894165, 1.1491429805755615, -0.7804396748542786, 1.278956413269043, 1.804307460784912, -0.2181839644908905, 0.05395900085568428, 1.4647917747497559, 1.1135483980178833, -1.1596590280532837, -1.8602920770645142, -0.32857948541641235, -0.5231980681419373, 0.9232063293457031, -0.25045621395111084, -0.22972239553928375, 0.775219738483429, 1.6421259641647339, 3.7078065872192383, 0.6566729545593262, -2.1046385765075684, 0.14499197900295258, 2.1275954246520996, 1.4790992736816406, 0.29317906498908997, -0.15412072837352753, -1.1707792282104492, 0.016834501177072525, -0.1606570929288864, 0.9128586053848267, 0.25619396567344666, 0.12836866080760956, -0.5133756399154663, -0.7689997553825378, 0.7070425152778625, 1.8254567384719849, -0.45916134119033813, 1.1391633749008179, -1.4286730289459229, 0.9446495175361633, 0.018428131937980652, -0.462533563375473, -0.06617194414138794, -0.563988983631134, -1.0880036354064941, 1.2326992750167847, 0.4346734583377838, -0.4099215269088745, 1.3333796262741089, 1.0291787385940552, -1.8384515047073364, 0.6217413544654846, -1.2563508749008179, 0.09363692998886108, -0.8176367282867432, 0.12120233476161957, -0.5568124651908875, 0.5587670207023621, 0.18635593354701996, -0.4617575705051422, -1.1253670454025269, 0.4029627740383148, -0.25346168875694275, 0.5729840993881226, -1.0188298225402832, -1.451444149017334, 0.6857569217681885, -0.03298274800181389, -0.433280885219574, -0.04112271964550018, 0.4922215938568115, 0.6458832025527954, -0.41286614537239075, 0.5525598526000977, 0.3620002865791321, -0.825389564037323, 0.607902467250824, -1.643172025680542, -0.006298658438026905, 0.9911544322967529, -0.267814040184021, -1.2703076601028442, -1.0511285066604614, 0.9719165563583374, -0.8667280673980713, 0.49549466371536255, -0.4258909225463867, 0.2348606288433075, 0.15849462151527405, 0.275014728307724, -0.15583942830562592, 0.020754488185048103, -0.045664623379707336, 0.9735062122344971, 0.6634024381637573, 0.1119847223162651, -0.3984786570072174, 1.5883067846298218, -0.13140758872032166, 0.7337805032730103, 0.5378669500350952, -0.2732944190502167, 0.6519176959991455, -0.18945792317390442, -0.566704511642456, 0.25926393270492554, 0.3933907747268677, -0.45204296708106995], [-1.6294351816177368, 0.2944078743457794, 0.9274228811264038, 0.2201065570116043, -0.5301889181137085, -1.0661132335662842, 1.983015537261963, 0.6039794683456421, -0.43797099590301514, -0.5168836712837219, 1.3851253986358643, 0.33400478959083557, -1.279646396636963, -1.276002287864685, 0.49691271781921387, 0.41643598675727844, 0.8631747961044312, 1.1332623958587646, -1.4096750020980835, -0.9226112365722656, -0.6953147053718567, -0.4902614951133728, 0.42601487040519714, -0.21669068932533264, -0.9172258377075195, -0.646260678768158, -1.3221179246902466, -0.6908631920814514, -0.5293529629707336, 0.5774242281913757, 0.7452042698860168, -0.4743381142616272, -0.5615980625152588, 0.38589197397232056, -0.02883009798824787, -0.09947936981916428, -1.8823591470718384, 0.0012320500100031495, 1.2134515047073364, 0.4868811368942261, -0.2943151593208313, 0.37281692028045654, 0.44014641642570496, 0.7442749738693237, 1.2892398834228516, -0.7547491192817688, -1.6260297298431396, -0.18803240358829498, -0.04110928624868393, -0.14853322505950928, 0.8343576788902283, 1.318428635597229, 1.793358325958252, 1.5748592615127563, -0.390045702457428, 0.17864084243774414, 0.035713471472263336, 0.804203987121582, 0.5707432627677917, -0.5325078368186951, -1.1761376857757568, -1.485223412513733, 0.475591778755188, -0.8554688096046448, -0.24131956696510315, 0.8601034879684448, 0.7084576487541199, -2.426034688949585, -2.0312066078186035, 0.3177688419818878, -0.8715061545372009, 0.7434425354003906, -0.8788154125213623, -0.2165423184633255, -1.942156434059143, -1.395912766456604, -0.8909605145454407, 0.265298068523407, 2.0795507431030273, -1.0288602113723755, 1.0171010494232178, -0.25075724720954895, -0.04368673637509346, -0.0016694845398887992, 0.29703420400619507, 0.5119091868400574, 0.021416427567601204, -0.5013437271118164, -0.4481159448623657, 0.6221229434013367, -2.1228041648864746, -0.15235070884227753, -0.1870066523551941, 0.37659305334091187, 0.1315458118915558, 0.09630930423736572, -0.27810996770858765, -1.0442084074020386, 0.9576703310012817, 0.8869167566299438, 0.5356888771057129, 0.09964540600776672, 1.6228946447372437, 0.10899943858385086, -0.531948983669281, -1.78315007686615, -1.2631179094314575, -0.24362459778785706, 1.5572712421417236, -1.9646413326263428, 0.6609787940979004, 0.2896040976047516, 1.9070079326629639, -0.07881549000740051, -0.061219483613967896, 1.2390917539596558, -0.7665213942527771, -0.2923976182937622, -0.21117755770683289, -2.1766934394836426, -0.24649640917778015, 0.08115068823099136, -1.6889716386795044, 0.19769592583179474, 0.09698229283094406, 1.0315377712249756, 0.8639991879463196, 2.7513418197631836, -0.5861396193504333], [-0.8698906302452087, -1.9192707538604736, 0.28524982929229736, 0.9822749495506287, 0.9501014947891235, -1.2353549003601074, -2.3421993255615234, -0.3130599856376648, -0.3378314971923828, 0.8746864199638367, 0.2995091676712036, 0.5109136700630188, -0.19954489171504974, -0.8553774356842041, 1.9676792621612549, 0.701259970664978, 0.3838560879230499, 0.7395517230033875, 1.6964679956436157, 1.2471303939819336, -0.7407980561256409, -0.6273585557937622, -0.1937643140554428, 0.20784254372119904, 1.9245489835739136, 0.29764309525489807, -1.0337127447128296, 0.024747882038354874, 0.07982252538204193, -0.13580237329006195, 1.138079285621643, 0.151836097240448, -0.5824424028396606, -1.4393278360366821, 0.7994074821472168, -1.0322535037994385, 1.1585729122161865, -1.6321977376937866, -1.4063411951065063, 1.5921974182128906, -1.480865478515625, 0.320531964302063, -0.9538640975952148, -0.6279395818710327, 0.28275495767593384, 1.6359292268753052, -0.12688563764095306, -1.2325445413589478, -0.43234026432037354, 0.33093273639678955, 1.7799729108810425, 0.4536397457122803, 0.8585644364356995, 1.2841979265213013, 1.3312872648239136, 0.6537232995033264, 1.55619478225708, -1.1421256065368652, -0.9451724886894226, -0.29653504490852356, 0.6473369002342224, -0.4743930697441101, 0.06636293977499008, -0.2929133474826813, -2.4066951274871826, 0.7272568345069885, -0.6685620546340942, -0.610756516456604, 0.9623783230781555, 0.44341379404067993, 0.39717942476272583, -0.2878071069717407, 0.8179948925971985, 0.010805470868945122, 0.5758389234542847, 0.3695504069328308, -3.3377394676208496, -0.2262585312128067, 0.938385546207428, -0.6544358134269714, -0.9828531742095947, 2.3670923709869385, 0.5076873302459717, -0.6920545101165771, 0.7178069949150085, 0.4339403212070465, -0.4044111967086792, 0.8352301120758057, 0.35716867446899414, 0.859093427658081, -0.7433323860168457, -2.79548716545105, 0.35396233201026917, 0.7101027965545654, 0.8174059987068176, -1.9527703523635864, -0.39304250478744507, 1.616791844367981, 0.08636414259672165, 0.45347872376441956, 0.6563472747802734, -1.5495673418045044, 0.6777675747871399, -0.04471565783023834, -0.6110347509384155, -0.6387467980384827, -1.2923327684402466, -1.195600152015686, 0.07002118974924088, -0.43019208312034607, 0.5235644578933716, -1.1733654737472534, -0.7706372141838074, -1.021628975868225, -0.05123095214366913, -1.664360761642456, 0.3722277581691742, -0.3221445083618164, 0.5676655173301697, 0.8802527189254761, -0.39087578654289246, -1.2632683515548706, -0.6139767169952393, 0.5234150886535645, -0.010025805793702602, 0.6741911768913269, 0.13343507051467896, 1.2055599689483643, -0.6919069290161133], [-0.6351604461669922, -0.9183481335639954, 0.5024797916412354, 0.8435330986976624, 0.8275423645973206, -0.9325790405273438, -1.7283598184585571, 1.865090250968933, -0.840857744216919, -1.159656047821045, -0.6559701561927795, -0.5257599949836731, 1.3806977272033691, -0.47716039419174194, -1.1228911876678467, 1.2470242977142334, 0.7044658064842224, 0.9753985404968262, -0.38736963272094727, 0.7163076400756836, 2.037217855453491, -1.5746104717254639, -1.3160710334777832, -2.409553289413452, 0.8158080577850342, 0.6384696960449219, -1.3925156593322754, -0.058523308485746384, 0.36832156777381897, -0.10173575580120087, 1.426081657409668, -0.7957664728164673, 0.26788344979286194, 0.7991176247596741, -1.4578052759170532, -0.2977760434150696, -0.2356952726840973, 2.152275562286377, -0.12115312367677689, 1.2865464687347412, -0.3658263683319092, 1.3394960165023804, -0.00618134206160903, -0.35138222575187683, -0.49915194511413574, 0.6283597350120544, -1.1745281219482422, 1.4346954822540283, -0.9155567288398743, -0.9186917543411255, 0.552684485912323, 0.5072497725486755, -0.29554304480552673, -0.2797331213951111, 1.068016529083252, 0.005371422506868839, 1.3409637212753296, -1.3071708679199219, 0.8788721561431885, -0.60124272108078, -0.7995041608810425, -0.4770919978618622, -1.0513527393341064, 0.5345146059989929, -0.9600028395652771, 0.0359155535697937, 1.721278190612793, 0.14736424386501312, 0.19936957955360413, 0.02085758186876774, -0.04329182580113411, 0.49996721744537354, 0.1968204379081726, 0.6608548760414124, 0.17606419324874878, -0.10827228426933289, -0.08155275136232376, -1.0017677545547485, -0.5852237939834595, -1.653527855873108, -0.5425311326980591, 0.044680606573820114, 0.3900004029273987, 1.5000450611114502, -2.8394577503204346, 0.18366068601608276, 0.29961511492729187, 0.9538655281066895, 0.15435105562210083, 1.0144668817520142, -0.7735353708267212, -0.9916678667068481, 1.0656861066818237, 2.0328004360198975, -0.22278709709644318, 0.7896108627319336, -0.9521585702896118, 1.0280872583389282, 1.1996062994003296, 0.2562006115913391, 1.2539541721343994, 0.7822394967079163, 1.4710160493850708, -0.29380396008491516, 0.05141182988882065, -0.24890217185020447, 0.8399290442466736, 0.6354725956916809, -0.46707019209861755, 1.2420011758804321, -0.1132909283041954, -0.9358819723129272, -0.5024423003196716, 0.2767122983932495, -0.2338171899318695, -0.008636796846985817, -3.3451123237609863, 0.6279051899909973, 0.38832202553749084, -0.7612083554267883, 0.1382301300764084, 1.3095563650131226, 0.44491949677467346, -0.7208231687545776, -1.5907734632492065, 0.8556547164916992, 1.4401150941848755, -0.6906861662864685, 0.40633001923561096], [-0.7757591009140015, 0.18647834658622742, 1.5846604108810425, 1.189931869506836, -1.0303702354431152, 0.6326830387115479, 0.653675377368927, 0.5979925394058228, 0.1669759601354599, 0.6256853938102722, 0.6014965772628784, 0.37448933720588684, -0.362613320350647, -2.0132250785827637, 0.7365207076072693, 1.8205610513687134, 0.5393764972686768, -0.018526075407862663, -0.5257177352905273, 1.1930118799209595, 2.40704345703125, 0.3202284872531891, -0.25540974736213684, -0.6874437928199768, 1.2859103679656982, -0.6484475135803223, -0.5316851735115051, 0.4862363338470459, -1.1313179731369019, 0.22467269003391266, 0.5157526731491089, 0.40579965710639954, 0.6787109375, -0.1701117306947708, -1.7340885400772095, 0.6375912427902222, -2.1437795162200928, 0.049392491579055786, 0.5363139510154724, 0.102306067943573, 0.568039059638977, 3.1117424964904785, -1.4481658935546875, 0.8184508681297302, -1.2050219774246216, -0.6177104115486145, -0.12122351676225662, 0.1243300810456276, 0.18004637956619263, 0.23769554495811462, 0.1966344565153122, -0.9378231167793274, -0.830172061920166, 0.914171576499939, -2.0581881999969482, -1.8374944925308228, -0.5016923546791077, 0.034621067345142365, 0.5214595794677734, -2.3303024768829346, 0.874035656452179, 0.026187235489487648, 0.4626041054725647, -0.43722593784332275, -0.039229393005371094, 1.0711085796356201, 0.6998648047447205, -1.7526646852493286, 1.16989004611969, 0.06121576577425003, -0.5646048188209534, 0.9965248703956604, -0.8037058115005493, 0.09843441843986511, -0.6444227695465088, 0.8264197111129761, 0.298570841550827, 0.2863038182258606, 1.6360117197036743, -0.672462522983551, 1.5254930257797241, 0.653770923614502, -1.1636778116226196, 0.0019591152667999268, -1.9851380586624146, -0.5956504344940186, -0.288322776556015, 0.7936100363731384, 1.4007171392440796, -0.4193606376647949, -0.006872403901070356, -0.09136761724948883, 0.9649654626846313, -0.41198354959487915, 0.08198615908622742, 1.0293290615081787, 0.5960993766784668, -1.7508206367492676, 1.2742923498153687, 0.9969687461853027, 0.5009997487068176, -0.7381331920623779, 0.8524746298789978, -1.510991096496582, 0.47599491477012634, 1.0532505512237549, -1.6646393537521362, -2.616381883621216, -0.31996625661849976, -0.668005645275116, 1.496031641960144, -0.24054796993732452, 0.6671698689460754, 0.030380161479115486, 0.8508610725402832, -0.23852257430553436, -0.8524668216705322, 0.10391238331794739, 0.923407256603241, 0.5758650898933411, -1.2762293815612793, -2.2738354206085205, -0.4258316159248352, -1.103129506111145, -1.2970075607299805, -2.5294010639190674, 0.2368949055671692, 0.8280432820320129, 0.6169952154159546], [1.5342273712158203, 0.20849761366844177, 1.5592327117919922, 0.9204522967338562, 1.4235533475875854, -1.0879542827606201, 0.3766012191772461, -0.5467935800552368, -0.8180171847343445, 0.4635842740535736, -1.6586806774139404, -1.7669662237167358, 2.6729302406311035, -0.08275794237852097, 1.9781250953674316, 0.6667430996894836, -0.7004226446151733, -0.43269145488739014, -0.798884391784668, -0.8121601939201355, 0.012011926621198654, 0.3864765465259552, -1.1767938137054443, 0.11933661997318268, 0.7263367772102356, 0.7662385106086731, 0.06347859650850296, 0.2884584665298462, -0.9512019157409668, 1.8508933782577515, -0.5330588221549988, 0.4537706971168518, 0.3613196611404419, -1.1732381582260132, -1.1519802808761597, -0.7685075402259827, 0.5028370022773743, -1.3341680765151978, 0.9948972463607788, -1.6219348907470703, -0.13384540379047394, 0.9664448499679565, 0.609626293182373, -0.5953936576843262, -0.4813394248485565, -1.0179647207260132, -2.335690498352051, -0.4790053069591522, 0.8214943408966064, -0.940568208694458, 0.31127357482910156, -0.9035440683364868, -0.9650035500526428, 1.53614342212677, -0.5108774900436401, 0.18356673419475555, -0.10208027064800262, -0.1688654124736786, 0.6419801115989685, 0.702528178691864, -0.02508346177637577, -0.992440938949585, -0.6933665871620178, 0.21250484883785248, -0.6145933866500854, 0.7550796270370483, 0.8980944752693176, -1.7739996910095215, -0.744997501373291, 0.7389788031578064, -2.8377645015716553, 1.564860224723816, -0.9336365461349487, 2.26642107963562, -1.3637644052505493, -0.5525898337364197, 0.3391387164592743, 0.7560114860534668, 1.0323092937469482, 0.510035514831543, 0.20801472663879395, 0.5416538119316101, 0.5357642769813538, -0.19414445757865906, -0.7762395739555359, 0.14674629271030426, 0.04424746707081795, 1.1531929969787598, -1.683900237083435, -1.8553814888000488, 0.23220132291316986, 0.3883868455886841, 1.3675365447998047, -0.179204061627388, -0.6674016714096069, -1.1294485330581665, -1.60568368434906, 0.6367502212524414, 0.24665628373622894, 0.17238463461399078, 0.3979525864124298, -0.16969545185565948, 0.08535387367010117, -0.8237780332565308, -1.8559393882751465, -0.5906313061714172, -1.7205501794815063, 0.7792540788650513, -0.5800057053565979, -0.29066604375839233, -0.8682597279548645, -0.7516908645629883, -0.7083362936973572, -1.4738320112228394, 0.15990932285785675, -0.30935704708099365, -0.5738831758499146, -0.7329605221748352, 1.997543454170227, 0.7290987372398376, 0.9112357497215271, 3.743492364883423, -0.4233650267124176, -1.0687199831008911, -1.7443674802780151, -0.2641221284866333, 0.49076148867607117, 1.5893514156341553, -0.48145514726638794], [-1.4923783540725708, 1.9426090717315674, -0.2288157343864441, 1.2546628713607788, 0.9435222148895264, 0.12380363047122955, 0.9929641485214233, 0.6064591407775879, -0.6148713231086731, 1.0200035572052002, 2.2645115852355957, -0.11577892303466797, 0.3731171786785126, 0.9618049263954163, -0.2282210886478424, -1.2708219289779663, 0.0008078335085883737, -0.9653107523918152, 0.11003313958644867, 1.507494330406189, 0.4800182282924652, -0.09197170287370682, -1.248551368713379, -0.43591445684432983, 0.5704718232154846, 2.1532444953918457, -0.6407002806663513, -0.9699509739875793, 0.8641269207000732, 0.12375913560390472, -0.3716728389263153, 0.6058034300804138, -0.8313084840774536, 0.07046660035848618, -0.435570627450943, -0.5721602439880371, -0.01854652911424637, 2.0562617778778076, 0.759601891040802, -0.9391303658485413, 0.35069119930267334, 0.5091346502304077, -0.7817441821098328, -1.877387523651123, 0.802454948425293, -1.5972917079925537, 1.6443337202072144, 1.620394229888916, 0.942276120185852, -0.8408858776092529, 0.8047828078269958, -0.17568932473659515, 0.8491696119308472, 0.8905189633369446, -1.2004082202911377, -0.5806229710578918, -0.7201698422431946, -1.6088511943817139, -1.3643786907196045, 0.48233526945114136, -0.9292349815368652, -0.10877835005521774, 0.7168664932250977, -0.2190236896276474, 0.7201206088066101, 0.28542912006378174, -0.20948268473148346, 1.302690029144287, 0.6898396015167236, 0.24990254640579224, -0.6531592011451721, 0.6677806973457336, -0.29081979393959045, 0.46403294801712036, -1.2938069105148315, -0.4118972718715668, -0.3750888705253601, -0.8659116625785828, -0.052235864102840424, 0.3441806435585022, 0.4365183115005493, -1.0545475482940674, 0.7949779033660889, -0.5833750367164612, -2.0465245246887207, -1.712420105934143, -1.2246416807174683, -0.00784174632281065, 0.8519020676612854, 1.5593708753585815, -0.059423889964818954, 0.9599716663360596, -0.2153138369321823, -0.438198059797287, -0.13469286262989044, -0.4836505651473999, -0.2295019030570984, 2.29660964012146, -0.302045613527298, -1.541764736175537, 1.565636396408081, 1.4729269742965698, -0.03160835802555084, -1.5444002151489258, -0.17336894571781158, 2.041309118270874, 1.8181965351104736, 0.04686586558818817, -0.11996309459209442, -0.40586057305336, 0.20289698243141174, -0.5429273247718811, 0.48860111832618713, 1.6523524522781372, -0.19258110225200653, 1.6390793323516846, -1.8259367942810059, 2.3816452026367188, -0.9370533227920532, 1.9903514385223389, -0.9929962754249573, 1.2693878412246704, 0.1784948855638504, 0.5987337827682495, 0.4225659668445587, -0.38181209564208984, 0.7380731701850891, 1.1217339038848877, 2.0322539806365967], [-0.4745651185512543, 0.44514432549476624, 1.019222378730774, 0.7940056920051575, 0.09576304256916046, 1.0553752183914185, -0.7975534200668335, 0.15055769681930542, 0.36024966835975647, 0.8807153701782227, -1.1760023832321167, 0.28724515438079834, -0.017666814848780632, -0.5513892769813538, -0.7539302706718445, 1.8522467613220215, -1.0639766454696655, -0.0018619124311953783, 0.8853760361671448, 2.233917236328125, 0.15806248784065247, -0.021063614636659622, -1.2641242742538452, -0.3990863263607025, 2.948009729385376, -2.183328866958618, -0.9203068017959595, -1.2991976737976074, 0.40412965416908264, 0.8856797218322754, -0.5816828608512878, 0.5543065071105957, -0.2126733809709549, 0.5352164506912231, -0.8265685439109802, 0.23155277967453003, 0.423902302980423, -0.3303993046283722, 2.0259339809417725, -2.119781255722046, -1.4887568950653076, 0.10618774592876434, -1.9648973941802979, -0.28057900071144104, -1.1659016609191895, -0.601976215839386, 0.41185516119003296, -0.394082248210907, -0.6454547643661499, -0.2990637719631195, -0.017929432913661003, 1.113994836807251, -0.40888068079948425, -1.2673391103744507, 0.6495148539543152, 1.0396846532821655, -0.6672425866127014, -1.5946955680847168, 1.1150134801864624, 1.6844768524169922, -0.17812712490558624, 0.4918062388896942, 0.09688901901245117, -0.342894583940506, -0.8168529868125916, 1.3467307090759277, -1.1522387266159058, 1.0760738849639893, 1.241133213043213, 0.4403408467769623, -1.4070758819580078, 1.024442195892334, 0.9374955296516418, -2.397690773010254, 0.7042123079299927, 0.09281829744577408, 1.8902068138122559, 0.6101009845733643, 0.9521355032920837, -1.2861089706420898, -0.5129925608634949, 1.2883398532867432, -0.20463518798351288, 1.0836118459701538, -0.5154939293861389, 1.3439339399337769, 1.413501262664795, 0.12680590152740479, 1.9698231220245361, 0.01863807439804077, -0.6240001320838928, -0.07554897665977478, 0.6966408491134644, -2.660437822341919, 0.9376184344291687, -0.29500067234039307, 0.8830609321594238, -1.2329471111297607, 0.2947554886341095, 0.45386263728141785, 1.955580234527588, -0.3234020471572876, 0.4750535786151886, 1.0337847471237183, 1.6957824230194092, -0.9624590277671814, -0.6002316474914551, 0.9522058963775635, 0.31343862414360046, -0.36547812819480896, -0.33348968625068665, -0.06665876507759094, 0.3931773900985718, 0.5936934947967529, -0.5043596625328064, -0.4171934127807617, 1.1038639545440674, -1.2601696252822876, -0.059584181755781174, 0.0046317907981574535, -0.6922516226768494, 0.0017508668825030327, -1.525827407836914, -0.1181686669588089, 0.820662796497345, -1.438098669052124, -0.9241743683815002, -0.8722511529922485, -0.28132718801498413], [0.5699012875556946, -2.159494400024414, -1.5319759845733643, 0.8766936659812927, -0.08453942835330963, -0.42083078622817993, 1.2820926904678345, -1.7615758180618286, -0.8991682529449463, 1.235163927078247, 0.4156830906867981, 1.4594041109085083, -0.5639947056770325, 1.499361515045166, 0.3750479221343994, 0.024080196395516396, -0.5027928948402405, -1.1919509172439575, 0.9767625331878662, -0.2137623280286789, 0.17096082866191864, -0.7256197929382324, -0.7426357269287109, -0.11044207960367203, -1.1866241693496704, 0.2495783120393753, 0.28821030259132385, 0.8581681251525879, 0.04022505506873131, -0.6224303841590881, -0.31958651542663574, -1.3159995079040527, -0.5845821499824524, 0.8385592699050903, -0.22443345189094543, 0.9114691615104675, -0.11272435635328293, 1.4120162725448608, 0.4793233275413513, 0.3165387213230133, 0.6105362176895142, 0.6882767081260681, 0.0686379075050354, -0.7750064134597778, 1.2409141063690186, -0.1526213139295578, 0.7080744504928589, 1.377036213874817, -0.8734834790229797, 2.375218629837036, -0.19991719722747803, 0.6544142961502075, -0.8274604678153992, -2.0972626209259033, 0.769352376461029, 0.3342149555683136, -0.3970351219177246, -1.3426241874694824, -2.106757164001465, -1.3420162200927734, -0.02176073007285595, 0.24568061530590057, 0.38537657260894775, 0.032320547848939896, 0.4247179329395294, 0.3337148129940033, -0.421924889087677, 1.5171771049499512, -1.6555427312850952, 1.221813678741455, -0.7292206287384033, 0.8560116291046143, 0.7483007311820984, 0.07446923106908798, 0.3416355848312378, -0.6582506895065308, 1.9292833805084229, -0.25732919573783875, -0.36086270213127136, 0.006629716604948044, 0.7921275496482849, 1.0401039123535156, 0.4871026873588562, -0.024310827255249023, 0.024384941905736923, -0.12845823168754578, 1.1010550260543823, 1.2021512985229492, 0.4610365629196167, -0.7683800458908081, -0.3187664747238159, -0.4920809268951416, -1.1598745584487915, -0.20582106709480286, 0.19761362671852112, -0.6355510950088501, 1.3398447036743164, 0.9888447523117065, 0.42023563385009766, -0.3670368492603302, -0.22989030182361603, -1.0987238883972168, -2.2055463790893555, 0.29203057289123535, -2.3935399055480957, 0.8896524310112, 1.0095821619033813, 1.730532169342041, -1.1434303522109985, 0.09437872469425201, -1.1892874240875244, 0.9031760692596436, -0.565220057964325, -0.0698859691619873, -0.06412030756473541, 0.3022899627685547, -0.13446004688739777, 0.8453527688980103, -0.9117584228515625, 2.2795016765594482, -1.53151535987854, 0.29326537251472473, 0.5556941032409668, -0.22849895060062408, -0.0786733478307724, 0.821150541305542, -1.140079379081726, -0.5066051483154297, 1.5772788524627686], [0.2380405217409134, -0.06400313228368759, 0.30211517214775085, 1.6975127458572388, -0.19101913273334503, -1.6531263589859009, -0.19775764644145966, 1.003852367401123, 0.2998257577419281, -0.6312417387962341, -1.7310664653778076, 0.08018673211336136, -1.6154065132141113, 1.8468339443206787, -1.2794712781906128, -1.1820353269577026, -0.73077791929245, -1.2471063137054443, -1.3811393976211548, -0.23814935982227325, -0.5859065651893616, -1.4861243963241577, -1.0870587825775146, -1.2480661869049072, -0.14649231731891632, 0.4184715747833252, 0.976118266582489, 0.7610527276992798, 1.6577001810073853, -1.6578677892684937, -0.008652307093143463, 2.0549867153167725, 1.109477162361145, 0.17498329281806946, -0.7660090327262878, -0.4768945872783661, 0.4573577344417572, 0.5960509777069092, -1.63620126247406, -1.4994333982467651, -0.09846825897693634, 0.8596622943878174, -0.7131457924842834, 0.18721133470535278, 0.6210607290267944, -0.9512138962745667, 1.4098247289657593, -0.7275106310844421, -0.3145432472229004, 0.20452560484409332, 0.9933186173439026, -1.0650235414505005, -0.45725923776626587, -0.8468329310417175, 1.4574670791625977, 0.014813520014286041, -2.6795883178710938, 1.599134922027588, -0.1208001971244812, -0.17558149993419647, -0.48056793212890625, -1.2389264106750488, 0.4767170548439026, 0.6398504376411438, -0.3346576690673828, -1.4620680809020996, -1.8375260829925537, 2.3676559925079346, -0.0769585371017456, -0.477013498544693, -0.11403798311948776, 1.4356775283813477, 0.4334433078765869, -0.16232430934906006, -0.7595139741897583, -1.18734610080719, 0.7609363198280334, 1.9282695055007935, 0.14273057878017426, 0.35385772585868835, 0.8726121783256531, 1.775648593902588, -0.7978395819664001, 1.1031028032302856, -0.3522385060787201, 0.21299129724502563, 0.9300112724304199, 1.0168850421905518, 1.1880018711090088, -0.7246910929679871, 0.03846350312232971, 0.39786195755004883, 0.9573716521263123, 0.13033078610897064, -0.14997871220111847, -0.08859646320343018, 0.7607930302619934, 0.2629811763763428, -0.8474623560905457, 0.5139198303222656, 0.19621765613555908, 0.717301070690155, -0.5694760680198669, 0.8365012407302856, -0.17989559471607208, 1.6487390995025635, 1.3658570051193237, -1.462412714958191, -1.6858869791030884, 0.2462884485721588, -0.40539953112602234, 1.4992961883544922, -0.48734548687934875, 0.7807894349098206, -0.42836955189704895, -0.9321321845054626, 0.05223244056105614, -0.3677738308906555, 0.8488726019859314, -0.9246417284011841, -1.536686897277832, 0.6148887276649475, 0.3928031921386719, -1.1155256032943726, -0.02155986987054348, 0.5919800996780396, 0.8616287112236023, 1.9282492399215698, 1.8386389017105103], [-0.44048550724983215, -0.5742462277412415, 0.22312411665916443, -0.3203023672103882, -1.6696215867996216, 0.3043082058429718, -0.01318609993904829, -1.932912826538086, 0.30798202753067017, 1.0860183238983154, -0.6691244840621948, 0.03895619511604309, 1.3063976764678955, 0.7374520897865295, -1.2674260139465332, 1.4617424011230469, 1.0622382164001465, -0.1004343256354332, -1.3894084692001343, 0.43458691239356995, -0.5258408188819885, -1.9418766498565674, -1.9176735877990723, 1.040412187576294, -0.7316020727157593, 0.769962728023529, -0.434648722410202, 0.1149003803730011, 0.3098873794078827, -0.24080093204975128, 2.5805137157440186, -1.0376629829406738, -1.1074334383010864, -1.542292594909668, -0.8258810639381409, 0.20128628611564636, -1.6599671840667725, 1.5892009735107422, 0.837038516998291, -0.9236435294151306, -0.7658352255821228, 0.14053970575332642, 2.2803454399108887, -0.7413902282714844, 1.3762215375900269, -0.653204619884491, 0.2552168369293213, -1.2530443668365479, 1.8439719676971436, -0.1182013675570488, 0.27316099405288696, 1.1882656812667847, 0.17149244248867035, 0.7232157588005066, -1.954494833946228, -0.16440114378929138, -0.8220639228820801, 1.272323489189148, 0.20873619616031647, -0.03686845675110817, 0.15353502333164215, 1.5110702514648438, 0.21976789832115173, -0.0837671235203743, 0.09744135290384293, -0.4967802166938782, 3.302609920501709, 0.5015630722045898, 0.5489572286605835, -0.4455328583717346, 1.4198355674743652, -1.0602176189422607, -0.17496870458126068, -0.5617250204086304, -1.5902053117752075, -0.19973032176494598, -1.3370518684387207, 0.3912027180194855, 1.1897392272949219, 1.038018822669983, -0.6749864220619202, 0.36724111437797546, 0.975179135799408, 0.4299094080924988, -0.1273670643568039, 0.48770225048065186, -0.08471761643886566, -1.6264712810516357, 0.657850444316864, -0.19245897233486176, -1.3995000123977661, -0.13109242916107178, -1.0878117084503174, 0.512699544429779, 0.34155359864234924, 0.6356639266014099, 0.09501978754997253, -1.405991554260254, 0.8721609711647034, -2.6064882278442383, -0.04849245771765709, 0.04003957659006119, 1.8713231086730957, 0.956327497959137, 1.4946625232696533, 0.6354604959487915, -0.0390835665166378, 0.6983449459075928, 0.5275534987449646, -1.0833044052124023, 0.801200807094574, 1.1256436109542847, -1.3201195001602173, 0.34958064556121826, -0.17050664126873016, 1.4050465822219849, 1.2783008813858032, 0.4744916558265686, -0.7795701026916504, -1.3729592561721802, 0.6469502449035645, -0.22795221209526062, 0.09436424821615219, -0.6228074431419373, 0.4087763726711273, 0.4897512197494507, -2.241720676422119, 1.2605117559432983, -1.8556184768676758], [1.2648324966430664, 0.03436881676316261, 0.25131890177726746, 1.266635537147522, 0.509901762008667, -0.6538225412368774, -0.2350420355796814, 1.9914958477020264, 0.6688767671585083, 0.606720507144928, 0.19415779411792755, -0.11742967367172241, -1.2266756296157837, -0.06450972706079483, -0.7785398364067078, 0.8764588832855225, 0.5195731520652771, 0.6876504421234131, 0.7078968286514282, -0.198246031999588, -0.11761566996574402, -1.6287307739257812, -0.51496422290802, 2.114103317260742, -0.4649450480937958, -1.2943925857543945, 0.9269019365310669, 1.1317001581192017, 1.5198172330856323, 0.7394260168075562, -0.012106033973395824, -1.0767912864685059, -1.1719528436660767, 0.5536221265792847, -0.015461698174476624, -0.45508962869644165, 0.4248751997947693, -0.03184579685330391, 0.8356154561042786, -0.10208997130393982, -0.38363155722618103, 0.25924041867256165, 1.8374155759811401, 0.025574713945388794, -1.000624656677246, 0.7022785544395447, 1.0083482265472412, 0.11376658827066422, 1.772557020187378, 1.3308813571929932, -1.5857974290847778, -1.6624667644500732, 2.0894975662231445, 0.5687013268470764, -0.7766964435577393, 0.7654113173484802, -0.39324191212654114, 1.272202968597412, 1.061259388923645, 0.9282304048538208, 1.7488466501235962, -0.6085988879203796, -0.7646222114562988, 1.0134057998657227, 0.09308106452226639, 0.8890339732170105, 0.8717772364616394, 0.4407654404640198, 0.5909824967384338, 1.3372691869735718, 1.1010302305221558, 1.4399447441101074, -0.3448798656463623, -0.2836243808269501, -0.04027301073074341, -0.260009229183197, 0.06955035775899887, 0.6657975316047668, 1.0389785766601562, 0.44550493359565735, 0.05906260386109352, -0.8924570679664612, 0.15171745419502258, 0.2866879105567932, 1.4830753803253174, 1.0834661722183228, -0.17789830267429352, -0.24447430670261383, -2.3940680027008057, 0.6482532024383545, -0.2797730267047882, 0.13162437081336975, -0.9394675493240356, -0.4846263527870178, 0.9890695214271545, 1.884442687034607, 0.6955934762954712, 0.6498807668685913, 0.08916265517473221, -0.0795569121837616, 0.0647534653544426, 0.5548201203346252, -1.0560359954833984, -0.10979852825403214, 1.3594250679016113, 0.3823617696762085, 0.8445103764533997, 0.022018762305378914, 1.5052908658981323, 0.5962815880775452, -1.7753857374191284, 1.0929241180419922, 0.8029250502586365, -0.4137488603591919, 0.9957731366157532, 0.44240835309028625, 0.4112301468849182, -1.2061787843704224, 0.7333357930183411, 1.4115612506866455, -0.8501513600349426, -0.8175293207168579, -0.9600480198860168, -0.7490949034690857, -0.6438126564025879, 0.16480416059494019, 1.6117782592773438, 0.7700487971305847, -0.5766183733940125], [-0.2987093925476074, 0.5579540133476257, 0.7388575673103333, 0.24279190599918365, 0.08965945243835449, -0.9775128960609436, -0.7230801582336426, 0.9127636551856995, -1.207176685333252, -0.1654110699892044, 0.873420774936676, -0.5224573612213135, -0.3747071325778961, 0.7265236377716064, -0.10734935849905014, 2.6813743114471436, 0.46924781799316406, -0.32733970880508423, 0.4260059893131256, 2.4993207454681396, -0.1435520350933075, 0.15287373960018158, 0.27188733220100403, 1.4098926782608032, 1.2835248708724976, -0.625845730304718, 0.0035112048499286175, 0.5167675018310547, -0.45361843705177307, 0.618039071559906, 1.6590396165847778, 0.004719841759651899, -1.520897388458252, -1.3189269304275513, -1.0313520431518555, 0.13082963228225708, 0.6985922455787659, -0.4304371476173401, -1.4740911722183228, -0.35384824872016907, -0.6635320782661438, -0.8472809791564941, 0.4648052155971527, 1.236202359199524, 0.33858951926231384, -0.4446090757846832, -0.7619563937187195, 0.7632977962493896, 2.148257255554199, 1.6887654066085815, 1.2156637907028198, -0.6857181191444397, -1.3390884399414062, -1.0638493299484253, -1.3417272567749023, -0.047889500856399536, -2.0801680088043213, 0.24919493496418, 2.1056182384490967, -1.1096982955932617, 0.5750324130058289, 1.0160526037216187, -0.6510019898414612, 0.1425832211971283, 0.5443538427352905, 0.8354216814041138, -1.2097499370574951, -1.5549836158752441, -0.7818524241447449, -0.976438045501709, -0.6065590977668762, 0.5381172299385071, -2.1180384159088135, -0.48170486092567444, -0.9162397384643555, 1.2276374101638794, -1.2681496143341064, -0.21031123399734497, -1.5569099187850952, -1.130632996559143, 0.0017979369731619954, 0.3605509400367737, 1.5117212533950806, 0.1741115152835846, 0.35273972153663635, 1.2499264478683472, -0.43379026651382446, -1.4333516359329224, 0.6305017471313477, 0.6087356209754944, -1.3030369281768799, -0.9861708879470825, 2.2668166160583496, 0.27172407507896423, -2.0899972915649414, -1.3183702230453491, 0.019751792773604393, -0.5253288149833679, 0.7570337653160095, 1.4218103885650635, 0.5597169399261475, -0.32506415247917175, -1.9953389167785645, -1.0123212337493896, -0.32400092482566833, -1.2629847526550293, -0.22839070856571198, -0.18620477616786957, -0.3093355894088745, -0.06657654792070389, 0.7166999578475952, -1.05422842502594, -2.0161185264587402, 1.9170321226119995, 0.9805030822753906, -0.4151301681995392, -0.5041480660438538, -0.2159940004348755, -1.295608639717102, 0.5240545272827148, -0.7837226390838623, -0.5194434523582458, -1.340193510055542, 0.034818485379219055, -1.4160934686660767, -0.23258531093597412, 1.412987232208252, -0.0850682333111763, 0.5321968197822571], [-0.9864227771759033, -0.21885909140110016, 0.3356515169143677, 0.3762730360031128, 0.10752616822719574, 0.4849049746990204, 0.18158094584941864, 0.8571570515632629, 0.4926728904247284, 1.6105684041976929, 0.6493294835090637, -1.3823144435882568, 0.22023802995681763, -0.4983274042606354, -0.5779410004615784, 0.20925700664520264, -0.22353969514369965, -0.521717369556427, 1.4186384677886963, -0.7489327192306519, 0.4147169291973114, -0.513755738735199, 2.105386257171631, -0.2776932418346405, 0.5273477435112, -0.9581493139266968, 1.5578778982162476, 1.4024412631988525, -1.2277028560638428, -0.4583533704280853, -0.20093604922294617, -0.4661189317703247, 0.6270782351493835, 0.3240724802017212, -0.019180839881300926, 0.7692130208015442, -0.006791365798562765, -0.9098353981971741, -0.22734779119491577, -2.080031633377075, -0.8393022418022156, -0.43137669563293457, -0.7800912261009216, -0.1050354614853859, 1.1853880882263184, -0.6031724214553833, -0.12732385098934174, 0.671349048614502, -0.7750565409660339, 0.2110215723514557, 0.7882303595542908, 1.6544301509857178, 0.12974949181079865, 0.6812853813171387, -0.4408451020717621, 0.7615907788276672, -0.7799433469772339, -0.05153851956129074, 0.9258202910423279, 0.33603963255882263, 0.506773829460144, 1.0607200860977173, -0.36430683732032776, -0.4994823932647705, 1.1311765909194946, 0.9494649171829224, -0.843590497970581, -2.170600414276123, 0.39107367396354675, 1.5233668088912964, 1.7777678966522217, -0.13340578973293304, -1.2771316766738892, -0.44145748019218445, 0.8536267280578613, 1.4836574792861938, -0.38890740275382996, -1.2425895929336548, 1.435840368270874, -1.6305468082427979, 1.2621591091156006, -0.012396535836160183, -0.623559832572937, -0.6874534487724304, 0.04039812088012695, 0.8385040163993835, 0.532817542552948, 1.0101033449172974, -0.8566281199455261, -0.5979150533676147, 0.14843447506427765, -1.2774351835250854, -0.37277963757514954, 1.3682703971862793, 0.7530966401100159, 0.6726989150047302, -0.11300636827945709, -0.31028658151626587, 0.25610432028770447, -0.010281491093337536, -0.9868163466453552, 1.781891942024231, 2.0026416778564453, -2.1323766708374023, -0.09959083050489426, 0.24300368130207062, -0.1749284416437149, 0.05600661039352417, -0.7981585264205933, 1.0789365768432617, 0.29421085119247437, -0.8500184416770935, -0.38965561985969543, 1.9162677526474, -0.3645560145378113, -0.6196881532669067, 0.8293454647064209, -0.8877466320991516, -0.8436931371688843, -0.2786431908607483, -0.8957471251487732, 1.2280443906784058, 0.2542215883731842, 0.8770270347595215, -0.6274702548980713, 0.34901008009910583, 1.2655776739120483, -0.3060106933116913, 1.6029990911483765], [0.36777782440185547, 0.547685980796814, 2.5860540866851807, -0.48203667998313904, -0.8286957144737244, 1.7707152366638184, -2.091824531555176, -0.8586148023605347, -0.05415930598974228, -0.5877966284751892, 0.7202163934707642, 0.7338282465934753, -2.1390469074249268, -0.3317830562591553, 0.2944912910461426, -0.3963152766227722, -0.6719964742660522, 0.5464288592338562, 0.8900243043899536, 0.539562463760376, 1.212467908859253, -0.6316667795181274, 0.7165734767913818, 0.03179251775145531, 0.9067409038543701, 0.006698638200759888, -0.6012924909591675, -0.959754228591919, 0.27346423268318176, -0.4938373267650604, 1.457858681678772, -1.1162679195404053, 1.050836443901062, -0.7010088562965393, 0.0023492490872740746, -1.1551170349121094, -1.3942335844039917, -0.18474476039409637, 2.0331530570983887, -0.410281240940094, -0.25346317887306213, -0.7580581307411194, -0.8726430535316467, -1.0576740503311157, 0.6204698085784912, 1.5831313133239746, -0.5888115167617798, -0.20939448475837708, -0.11992708593606949, 0.1635437160730362, 1.4876086711883545, -0.34586790204048157, -0.41408345103263855, 0.5094862580299377, -0.0374632366001606, 0.873273491859436, -1.4725377559661865, -0.2655263841152191, -0.5713062286376953, -0.072238028049469, 0.06389780342578888, 1.9759600162506104, 0.8204624056816101, 0.44239741563796997, -0.17486801743507385, 0.775316596031189, 0.23362770676612854, 1.2072608470916748, -0.46811553835868835, -1.753737211227417, -0.7202418446540833, -1.2618603706359863, -0.8571178913116455, 1.9578614234924316, 0.652686595916748, -0.6054753065109253, -0.6132578253746033, -0.6768231987953186, 0.7847889065742493, 1.1234887838363647, -0.3130957782268524, 0.8314169645309448, -0.4154903292655945, -0.8149054646492004, -0.2853943109512329, -1.1394304037094116, -0.6989565491676331, 0.0036643585190176964, 1.4408106803894043, 0.535399317741394, -0.5165605545043945, -0.7842127084732056, 1.0469439029693604, 0.525009274482727, 1.4042123556137085, -1.1526097059249878, -0.21217375993728638, 0.06232102960348129, 1.1529743671417236, 1.7997632026672363, 0.3686644732952118, 0.7852755784988403, -1.495018720626831, -0.7605651021003723, 0.8175598382949829, -0.47629231214523315, 1.221672773361206, -0.5997117757797241, -0.7978261709213257, -0.389067679643631, -0.7135453820228577, -1.371462106704712, 2.002669095993042, -0.6585858464241028, -1.8908027410507202, 0.4868794083595276, 1.3032008409500122, 0.37250372767448425, -0.00721403444185853, -1.2371004819869995, 0.843865156173706, 1.118350625038147, 0.10312032699584961, 0.2505772113800049, -0.4376694858074188, -0.6949678659439087, -1.5772851705551147, -0.8737128973007202, -0.8708445429801941], [1.7304589748382568, -0.33714523911476135, -1.51248300075531, 1.1098508834838867, -0.6584631204605103, -0.16957080364227295, -0.4348510801792145, 1.0129655599594116, 0.7291236519813538, -1.537813425064087, -0.9851848483085632, 0.8702582120895386, 1.0194734334945679, -0.7242008447647095, -0.6445363759994507, -0.6862918734550476, -0.10336118191480637, -0.612535297870636, -1.4689222574234009, 0.9168661236763, -0.07070991396903992, 1.205779790878296, -0.691815197467804, 1.372959017753601, -1.386627197265625, 0.9604193568229675, -0.8249817490577698, 0.8059060573577881, 0.1955043375492096, 1.9695987701416016, -0.253817081451416, -0.9671163558959961, -1.6402629613876343, -0.37132102251052856, -0.33380869030952454, -0.19014786183834076, 1.6227043867111206, -1.840523362159729, -0.12755271792411804, 0.13481028378009796, 1.1246782541275024, 1.2726565599441528, 0.955290675163269, 0.9011085629463196, -0.28101593255996704, -1.6240359544754028, -0.2152959704399109, -1.307651162147522, -1.7824567556381226, -0.47808095812797546, -0.028878722339868546, -0.31228193640708923, 0.07745134085416794, -1.056945562362671, -1.4508285522460938, 2.0166473388671875, -0.3055605888366699, -0.21648362278938293, 0.1638180911540985, -1.4992977380752563, -1.3571174144744873, 2.102220296859741, -0.5034274458885193, 0.3330669701099396, -0.5691958665847778, -0.2816842198371887, -1.2587634325027466, 1.554909586906433, -1.9984527826309204, 0.5075069069862366, 0.8248885869979858, -0.6319694519042969, -0.36750662326812744, -1.5341315269470215, 0.2142166793346405, -2.292714834213257, 1.7865997552871704, -1.3293697834014893, 0.8013883233070374, 1.0824462175369263, -0.6311678886413574, 0.7626665234565735, -0.03455261141061783, -0.3403906524181366, -0.1263575255870819, -0.3514005243778229, 2.279266834259033, -0.9442917704582214, 0.6849743127822876, -0.3615914583206177, 0.0588451586663723, -0.1450483500957489, 0.9936720728874207, 0.5172659754753113, -0.18110182881355286, 1.3194091320037842, 0.7955124378204346, 1.6844367980957031, -0.5747261643409729, -0.9108526706695557, -0.010840912349522114, 0.5244491100311279, 0.528121292591095, -0.4857524037361145, 0.6940227150917053, -0.678876519203186, -0.7525145411491394, 0.4099273681640625, 0.0010170042514801025, 0.9925121068954468, 0.8853158950805664, -0.807796835899353, -0.7832956910133362, -0.851094126701355, 0.2604331076145172, 0.36168819665908813, 0.04568992927670479, -0.011421524919569492, -1.5626840591430664, -1.4214030504226685, -1.958740234375, -0.043641697615385056, 1.430112600326538, 0.6602901220321655, 0.730185866355896, 0.7583154439926147, 1.0136796236038208, -1.55563223361969, -1.8380078077316284], [1.0726515054702759, 0.6379851698875427, 0.23556475341320038, 0.4254680871963501, -1.208706259727478, 2.098984956741333, -0.24713483452796936, 0.06619177758693695, -0.8140029907226562, 0.2208312600851059, -1.7890628576278687, 0.44396939873695374, 1.0365192890167236, 0.6720846891403198, 1.4694291353225708, -0.30534374713897705, -0.6504955291748047, 0.7335636615753174, 0.652974009513855, 2.018670082092285, -0.3164178133010864, -1.7070289850234985, 0.28480780124664307, 0.16012802720069885, -1.370721459388733, 0.6882957816123962, -0.06232210993766785, 0.994236171245575, 2.0163815021514893, -0.8790727257728577, 0.5443178415298462, 1.5668134689331055, 0.5735267400741577, 0.6273573637008667, 1.0962203741073608, 0.014889183454215527, -0.09870240837335587, -1.1731863021850586, 0.2183501273393631, -0.8666190505027771, 0.49058470129966736, 0.4826664626598358, -0.7235973477363586, 1.1045325994491577, 0.10993705689907074, 0.0224965400993824, 1.3638266324996948, -0.06551285088062286, 0.23600518703460693, 0.1118301972746849, 0.9965525269508362, 0.39527615904808044, -0.9168609380722046, -0.33903831243515015, 1.5743829011917114, -1.2383232116699219, 0.48933640122413635, -1.4343445301055908, -1.5241941213607788, 0.7428380846977234, -0.1281345933675766, 0.40579408407211304, 0.29201677441596985, 0.2454114705324173, 0.1509230136871338, 2.3362314701080322, 0.778287947177887, 1.006603479385376, -0.631663978099823, -1.7730118036270142, -0.0870618000626564, 0.1558777093887329, -0.45742523670196533, 1.826332449913025, -0.7002752423286438, -0.5092549324035645, -0.6976025104522705, 0.4650386869907379, -1.1535093784332275, -0.09132307022809982, 0.43527719378471375, -0.7321683168411255, -0.3328033685684204, -0.0004642113926820457, 1.6471750736236572, 0.27249130606651306, -0.29792317748069763, 0.1536066085100174, 0.48735564947128296, -0.7376841902732849, -0.03956824541091919, 1.4316794872283936, 0.4403570592403412, -0.05642198026180267, 0.3527771234512329, 1.7347744703292847, -0.6662315726280212, 0.09800674021244049, 1.9226675033569336, 2.1051762104034424, -1.342471957206726, -0.08924072980880737, 0.11828526854515076, -1.575865626335144, -0.1016971692442894, -0.34431377053260803, -0.4397388696670532, 2.076158285140991, 1.1440778970718384, -2.33683443069458, 0.3217064142227173, -1.5382229089736938, -2.5354104042053223, -0.8123995661735535, -0.25042852759361267, -0.2495965212583542, -0.720782458782196, 0.5419505834579468, -0.5869728922843933, 0.8102120161056519, 1.056150197982788, 0.010732188820838928, -0.9447590708732605, 0.13603508472442627, 0.6558903455734253, -0.3447422683238983, 0.2602834105491638, -0.33313047885894775, -1.0718623399734497], [-0.19963183999061584, -0.7733997702598572, 1.39632248878479, -0.6829458475112915, 0.6969200968742371, -0.8498423099517822, 0.23202195763587952, -0.34514087438583374, -1.508628487586975, 0.34546515345573425, -0.6430373191833496, -0.05246451124548912, 0.1717182695865631, 0.8749257922172546, -1.437459111213684, 0.017388639971613884, -0.9349192380905151, -0.5705441236495972, -1.2367017269134521, 0.27628982067108154, 1.1109824180603027, 1.1857420206069946, 1.5112524032592773, 1.0268489122390747, 1.0273797512054443, 1.7672145366668701, 0.37000033259391785, -1.1793240308761597, 0.6372413635253906, -1.3601956367492676, -1.115325689315796, 0.05224090442061424, 1.9793026447296143, 0.6864425539970398, -0.760638415813446, -0.5052524209022522, 1.9456974267959595, -0.7919000387191772, -0.2650696337223053, -0.27054497599601746, 0.6293001770973206, -1.0567177534103394, -0.15030018985271454, -0.7309480309486389, -0.36670684814453125, -0.0699191614985466, -2.3856358528137207, -0.2966063618659973, -1.1375010013580322, -2.7488954067230225, -1.3445477485656738, 1.7490564584732056, -0.024766266345977783, 0.1592613309621811, -0.234308123588562, -1.1274265050888062, -0.719336748123169, -1.14501953125, 0.5215928554534912, -1.224581241607666, -0.04675459489226341, -0.1488228291273117, -1.5524667501449585, -1.0018489360809326, -2.4509618282318115, -0.19179682433605194, -0.7684921622276306, 1.1590238809585571, 0.31786012649536133, -0.6818470358848572, -1.1235827207565308, -0.2546564042568207, -0.03722802922129631, 1.2361936569213867, 2.3437681198120117, -1.2876688241958618, 0.5562676191329956, -1.140524983406067, -1.1570316553115845, -1.5840895175933838, 0.7436756491661072, 0.9396015405654907, -1.5593507289886475, 0.15729933977127075, -0.20334023237228394, 0.7264784574508667, -0.4473944902420044, 0.7432875633239746, 0.8146135807037354, -0.4186237156391144, -1.086082100868225, 2.01076602935791, -1.0303244590759277, -1.1116042137145996, -1.182949185371399, 0.22478057444095612, 1.9027676582336426, 0.40780746936798096, -0.851950466632843, -1.2020562887191772, 1.158360481262207, -0.4018658697605133, -1.6973878145217896, 0.7275732755661011, 0.07255670428276062, 1.655836582183838, 2.189300775527954, 0.48059913516044617, 1.2370407581329346, -2.5338311195373535, -1.8722153902053833, -0.8422793745994568, -0.8206257820129395, 1.839399814605713, -0.35306233167648315, 0.6008809804916382, -1.4839062690734863, -0.6407802104949951, 0.6959335207939148, -0.13605239987373352, 0.06707601994276047, -0.4101671278476715, 1.2791543006896973, -1.13905930519104, 0.7042555212974548, -0.2541944980621338, -1.080308198928833, 0.6232794523239136, -2.2356905937194824], [-0.12222056835889816, 0.25200673937797546, -0.6718020439147949, 0.8867943286895752, -1.317644715309143, -0.7698839902877808, 0.7519907355308533, -0.5918158888816833, -0.20270508527755737, 0.572686493396759, 1.2927672863006592, 0.8263779878616333, 0.47544506192207336, 0.1369819939136505, 0.12165910005569458, 0.6346895098686218, 2.566251039505005, 0.06174497678875923, -1.2438006401062012, -0.17969441413879395, 0.35519179701805115, 1.0960123538970947, 0.06609128415584564, -1.1884829998016357, -0.04342735931277275, -0.20609743893146515, -2.6310601234436035, 0.09113496541976929, 0.4659355580806732, -1.654867172241211, -0.6968413591384888, 0.8901838064193726, 0.1423468440771103, 0.7615336179733276, -1.0245509147644043, -0.7571787238121033, -0.005534257739782333, 0.7532327175140381, -0.3747647702693939, 1.9018616676330566, -0.24710483849048615, 1.4212713241577148, 0.2151053100824356, 0.6038811802864075, 1.4424575567245483, 0.8723833560943604, -1.7498245239257812, 2.0784358978271484, -0.8326859474182129, 1.0660653114318848, 0.8841043710708618, 0.7325025200843811, 0.7918628454208374, 0.6181379556655884, -0.40638500452041626, -0.07966481894254684, -1.5633147954940796, -1.1130609512329102, -1.2203660011291504, -1.2074017524719238, 2.0845189094543457, -0.9473106265068054, 0.542799174785614, 1.0159095525741577, 0.9606658816337585, 0.6689018607139587, -0.5877869725227356, 0.5935547947883606, -0.5567693114280701, 1.3494817018508911, 1.6848487854003906, 2.6132304668426514, -0.54513019323349, 1.8318039178848267, 0.7786431312561035, 0.7221559882164001, -0.1976974457502365, -2.880544424057007, -0.5593696236610413, -0.1759473830461502, -1.0233163833618164, 0.44886037707328796, -0.6184449195861816, -0.3883250951766968, -1.6117191314697266, -1.3410042524337769, 0.2667692303657532, -1.5521026849746704, 0.8941529989242554, 0.0976693332195282, 0.5549039840698242, 0.42525652050971985, 1.5919764041900635, -0.2585645020008087, 0.08138889074325562, 1.043050765991211, -0.5926644802093506, 0.9161058664321899, 1.1160647869110107, -0.7150554060935974, 0.3363973796367645, 0.45042091608047485, -0.22884434461593628, -1.2391570806503296, 0.34830963611602783, 0.5057642459869385, -0.7852234244346619, 0.7466674447059631, -1.4105545282363892, -1.2807844877243042, 0.8516069650650024, -0.8425152897834778, -0.33141154050827026, -0.6581714749336243, 1.1514129638671875, -0.6956493258476257, -1.260387659072876, 0.5892065167427063, 0.42440447211265564, -1.0628950595855713, -1.0705771446228027, -0.8277848362922668, -0.3759346008300781, -0.9991542100906372, 0.8953024744987488, 1.3438441753387451, -0.6305029988288879, 0.15690487623214722, -0.4980916976928711], [0.5341274738311768, 0.425719290971756, 4.061577320098877, 2.1907498836517334, -1.7920132875442505, -0.2903958261013031, -0.0982910692691803, 0.09608761966228485, -0.4604611098766327, -0.3116622865200043, -0.6280471682548523, 1.4167675971984863, -0.528302788734436, -0.6531466841697693, 1.0979403257369995, 0.9135054349899292, 0.014998053200542927, 1.521958589553833, -0.8921447396278381, -0.1659601926803589, -1.2063684463500977, 1.6394195556640625, 0.6465184092521667, -1.2919279336929321, -1.0822372436523438, 0.2545585334300995, -0.540542483329773, -0.7080379724502563, -0.21660581231117249, -1.0926517248153687, -0.6707412600517273, -0.5360982418060303, -0.11496490985155106, -0.5369715094566345, 0.16296085715293884, -0.3147808313369751, 0.626488208770752, 0.9532517194747925, -0.28847330808639526, -0.6619012951850891, 0.39402252435684204, -1.2663037776947021, 0.9969546794891357, -0.5936542749404907, -2.3468310832977295, 1.3434123992919922, 0.6486490964889526, 0.6820357441902161, -0.4433346688747406, -0.3485351502895355, -1.2929999828338623, 0.636561393737793, -1.9391229152679443, 0.2217724770307541, -1.569041132926941, -0.6498790979385376, 1.8551868200302124, 0.8267202973365784, 0.6962173581123352, 1.7182015180587769, 0.9261191487312317, -0.42068225145339966, -1.7549775838851929, 0.5667003393173218, -0.7150752544403076, 1.316549301147461, 1.4365564584732056, -1.4454110860824585, 1.6221239566802979, 0.40710318088531494, 0.3282657265663147, -1.1813808679580688, -0.11253562569618225, -0.07530833035707474, 1.6355887651443481, -1.0106689929962158, 0.25377118587493896, -0.08745772391557693, 1.794145107269287, -0.8236090540885925, 0.7672014236450195, 1.6680985689163208, -0.9851769208908081, 0.1879655122756958, -0.1524471789598465, -0.40507715940475464, -1.283798098564148, 0.2915996313095093, 0.6507797241210938, 1.3285410404205322, 0.2204650342464447, 0.5272612571716309, 1.3048886060714722, 0.6021367311477661, -0.8513250946998596, -1.5065333843231201, -0.7089742422103882, 0.01981068216264248, -0.07416141778230667, -1.042634129524231, -0.7016130089759827, 1.4620901346206665, 1.1981532573699951, -1.4677189588546753, -1.0507291555404663, -0.7776179313659668, 1.7832739353179932, 0.06446719169616699, -2.46909236907959, -0.14633752405643463, 2.15214467048645, -0.09826498478651047, -0.6039954423904419, -0.8493412733078003, -0.062271878123283386, -1.1894482374191284, 0.48957085609436035, -0.24461303651332855, -0.5168684720993042, 0.4336015582084656, 0.16637465357780457, -1.0932185649871826, 0.02228546515107155, -0.2548847198486328, 0.3197336792945862, -1.6518751382827759, -0.3463726043701172, 1.0070574283599854, -0.625270664691925], [-2.03578782081604, -1.3312852382659912, -0.6864209175109863, -0.42107295989990234, -0.5568286180496216, 0.6888526082038879, 0.8287513256072998, -1.5173380374908447, -0.09565042704343796, -0.576531171798706, 0.6850528717041016, -1.1856571435928345, 0.020699147135019302, -0.07927148044109344, 1.1264995336532593, -3.4668023586273193, -0.4751042425632477, 1.657835602760315, 0.12726396322250366, 1.2098673582077026, 0.2787275016307831, -0.6818844079971313, -0.5622383952140808, 0.6302973031997681, 1.1725612878799438, -0.12249961495399475, -2.069547176361084, -1.0783686637878418, 0.2815677523612976, -0.5178462266921997, -1.3847153186798096, -1.1375192403793335, 1.2112691402435303, -0.06352221220731735, -0.7364770770072937, 1.054864525794983, 0.3019997179508209, 2.1331446170806885, 0.2881922125816345, -0.15114928781986237, -0.31233707070350647, -0.8023337125778198, -0.07325873523950577, -0.3438669443130493, -1.2784534692764282, 0.982516884803772, 0.5642678737640381, 0.4074568748474121, 0.9446653127670288, -0.0010208712192252278, -0.6279938817024231, -0.01776071824133396, -1.7950681447982788, 1.8800007104873657, 0.07038399577140808, 0.9715016484260559, 1.2005488872528076, 0.3711772859096527, -0.7277501821517944, 1.0298391580581665, 0.8583570122718811, 0.29606279730796814, -0.5940594673156738, 0.88639235496521, -1.6185684204101562, 0.7145177721977234, 1.1207714080810547, 1.5479084253311157, 0.4109261929988861, -2.1039505004882812, -0.28964805603027344, -0.4581747353076935, 1.3394824266433716, -0.12977832555770874, -0.5529142618179321, 1.0775542259216309, -0.8214331269264221, -1.3641501665115356, -0.10661161690950394, 0.7856990098953247, 0.15971937775611877, -0.3474689722061157, -0.09262382984161377, 1.0273133516311646, 0.22715575993061066, -0.24855610728263855, -0.565081000328064, 0.5272159576416016, 0.8812804818153381, -1.1553096771240234, -0.5167505741119385, 0.392301470041275, 0.49898427724838257, -0.4152657687664032, -0.670798659324646, -0.6742554306983948, 1.7942049503326416, -0.193672776222229, -0.5631586909294128, -0.7599561810493469, -0.12768889963626862, -1.0118733644485474, 0.3710947632789612, 2.1643221378326416, -0.027823030948638916, -0.369663268327713, 0.7467538118362427, -2.4329187870025635, 1.8317967653274536, 0.8368067145347595, -1.0037637948989868, 1.3392927646636963, -1.4030483961105347, -1.0219680070877075, -0.2853190302848816, 0.324182391166687, 1.5454717874526978, -0.9209203720092773, 0.9690401554107666, -0.8198775053024292, -0.4683634042739868, 0.2720058560371399, -0.17177094519138336, -0.18213649094104767, -2.141434907913208, -0.3307911455631256, 2.253225564956665, -0.9696075320243835, -0.4790531098842621], [1.0186240673065186, 1.5913846492767334, -1.955370545387268, -1.4556266069412231, -0.37756431102752686, 0.5276293158531189, -0.42442697286605835, 0.317156583070755, -1.5230810642242432, 0.5396425127983093, 0.5895100831985474, -0.6136102676391602, -1.3833048343658447, 0.746967613697052, 0.5624772310256958, -0.15595270693302155, 1.0425206422805786, 0.6828448176383972, 0.11960233002901077, 0.4885179400444031, 0.20651401579380035, 0.28058281540870667, -0.8497777581214905, 0.5497393608093262, -1.3013286590576172, -0.1446601301431656, -0.43239617347717285, -0.7414961457252502, 0.29700419306755066, -0.1408720463514328, 0.6081687808036804, 0.3976668119430542, 1.4560481309890747, 0.6004220247268677, -1.5983192920684814, 0.6795175671577454, -0.0839092880487442, -0.8911104202270508, 1.2094088792800903, -0.6231801509857178, 3.7052247524261475, 1.421101689338684, 0.5106922388076782, 0.8996044397354126, 2.5286386013031006, 1.4366321563720703, 1.342947006225586, -0.30800753831863403, 1.4421409368515015, 1.108437180519104, 0.8585789203643799, -0.1616121083498001, 0.38504883646965027, 0.7383768558502197, 0.3039380609989166, -0.4908859431743622, -1.2924031019210815, 1.131285548210144, 0.8741108179092407, -1.6963082551956177, -1.2085589170455933, 1.5138925313949585, 0.12407255917787552, 0.7538793087005615, 1.448608636856079, -1.531198263168335, 0.7800264358520508, 0.41422611474990845, 1.5039525032043457, 1.2535892724990845, -0.3588182330131531, -0.49279266595840454, -0.4999425411224365, 0.9156548976898193, 1.8389190435409546, 0.5758578181266785, -0.7049576044082642, -0.13484233617782593, 0.12300018966197968, 0.2324734777212143, 1.0380582809448242, 0.07290910929441452, -0.41189438104629517, 0.9170578718185425, 0.6509856581687927, -1.6088248491287231, 0.114143006503582, -1.1166867017745972, 0.3034922480583191, 1.8797229528427124, -1.0636495351791382, -1.6139150857925415, 0.3791772127151489, 0.5449485182762146, -0.3669985830783844, 1.2752776145935059, -2.430340051651001, 1.5809506177902222, 0.044050298631191254, -0.4672509729862213, -0.04316115006804466, -0.3636122941970825, -1.927827000617981, -0.4657326340675354, 2.058983564376831, -1.814836859703064, 0.5368665456771851, -0.31047889590263367, -0.5773164629936218, -0.1706443876028061, 0.23179234564304352, 2.383861780166626, 1.2464616298675537, 1.4367226362228394, -0.270602285861969, 0.7794597744941711, 0.8187292814254761, 0.4882097840309143, 0.8491883277893066, 0.9351027011871338, 0.04389365017414093, 0.3669779896736145, 0.9888756275177002, 1.7501658201217651, -1.7601553201675415, 2.150860071182251, -0.5458948612213135, -0.05936179310083389, 1.1978267431259155], [-0.1915193349123001, 0.2924601435661316, 0.010075557045638561, 1.0694050788879395, 0.2937225103378296, 1.2027842998504639, -1.8916159868240356, 0.18668828904628754, -0.6451731324195862, 1.1667747497558594, -0.5242278575897217, -1.2541073560714722, 0.6359725594520569, -0.42992734909057617, 2.080352306365967, 0.5160567164421082, 0.8153179883956909, 0.7248055338859558, -0.17837263643741608, -0.7789028882980347, 2.0641696453094482, -0.1222510114312172, 0.03392144292593002, 0.2522393465042114, -0.6520370841026306, 1.1219321489334106, -1.728351354598999, -0.17533434927463531, 1.0641810894012451, -0.05398799106478691, 0.9831597208976746, 0.004712807945907116, -0.2975928783416748, -0.09527241438627243, 0.8096364736557007, 0.8618571758270264, -1.1852208375930786, 0.6489623188972473, -0.60146564245224, 0.10226582735776901, 0.9198955297470093, -1.8708648681640625, 1.1745667457580566, 0.4809950590133667, -0.04785597696900368, -2.1612000465393066, -1.2222832441329956, 0.7982990145683289, -1.5770776271820068, 0.4731729030609131, -0.4060365557670593, -1.076657772064209, -0.7480130195617676, 0.5891815423965454, 1.035531997680664, 0.2878018021583557, -0.4328813850879669, -1.9787031412124634, -0.32464200258255005, -0.7342516779899597, -1.6601171493530273, 0.7106925845146179, 0.23281307518482208, 1.1486046314239502, -0.3347085118293762, -0.6899705529212952, 1.0589059591293335, 1.122430443763733, 0.4885087311267853, -1.0431417226791382, 0.21846303343772888, -0.6246122121810913, -0.48693543672561646, 0.7489360570907593, 1.1218382120132446, 1.4367480278015137, 0.10633385926485062, 0.35270848870277405, -1.771923542022705, -0.3978773355484009, 1.0024313926696777, -0.49819889664649963, -0.45124518871307373, 1.039260983467102, 0.364378422498703, 0.13375362753868103, 0.19089820981025696, 0.5732578635215759, -0.4892563819885254, -0.9484813809394836, -0.11306042224168777, -1.6235761642456055, 0.005486015696078539, -0.3996194303035736, -0.9722880721092224, 0.012910532765090466, 0.053254738450050354, -0.1087830439209938, -0.77289217710495, -0.6833407878875732, -1.785951018333435, 0.1206449493765831, -0.31107285618782043, 0.5794922113418579, -0.5582384467124939, 0.5935003161430359, 0.6010815501213074, 0.4121897518634796, -1.0084240436553955, 0.9985496997833252, 0.06937026977539062, 0.12251792848110199, -0.5262805819511414, 0.29570481181144714, -0.42491382360458374, -0.048888664692640305, -0.9838577508926392, -0.05802791565656662, -1.0993725061416626, -3.1432273387908936, 0.3845795691013336, -0.5999808311462402, -1.1154937744140625, 2.248843193054199, 1.7760668992996216, -0.07451362907886505, 1.1449627876281738, -0.4909181594848633, -0.8895147442817688], [1.2776509523391724, 0.5560183525085449, -0.7726796269416809, -0.5325760841369629, -1.2237818241119385, -0.348497599363327, -0.36308974027633667, 0.5408851504325867, 1.5833021402359009, 0.08873879164457321, -0.0463244691491127, -2.035978078842163, -1.29848051071167, -0.737841010093689, -1.094603180885315, 0.043281786143779755, 0.3241746127605438, -0.7711554169654846, -0.5000765323638916, -0.4029916822910309, 0.6164716482162476, -1.212548851966858, 0.6721767783164978, 0.5071574449539185, 0.016454242169857025, 0.5682148337364197, 0.7708301544189453, -0.18467268347740173, 0.026599198579788208, 0.9766743183135986, 0.31075724959373474, 1.3358395099639893, -0.1872844398021698, 0.15654443204402924, -0.2703198790550232, 0.07183793932199478, 1.7012345790863037, 0.6462537050247192, 0.8861231207847595, -1.3484318256378174, -1.065215826034546, -0.7788639664649963, 1.8625507354736328, 0.732831597328186, -0.03949994593858719, 0.3459148705005646, 0.748690128326416, 2.10909104347229, -1.2254315614700317, 0.051966384053230286, -1.3382947444915771, 0.5624590516090393, -0.051878057420253754, -0.31711986660957336, -0.29567232728004456, -0.24348610639572144, 1.2011398077011108, -1.1389232873916626, -0.8622135519981384, 1.5968540906906128, -0.3554687798023224, 0.9850829839706421, -0.7185472249984741, -0.7759121656417847, 1.9303746223449707, -0.5391973853111267, -0.820334792137146, 1.4428672790527344, -1.5116244554519653, 0.5619356036186218, -0.6497710347175598, 0.6215946674346924, -0.32989275455474854, -0.6386659145355225, 0.5159645676612854, -1.9147146940231323, -0.38666650652885437, 0.019103577360510826, 0.41599196195602417, 0.19431637227535248, -2.310609817504883, 0.3086092472076416, -0.07197295874357224, -0.9441856741905212, 1.2870548963546753, 0.015995724126696587, 0.18591450154781342, 1.0015056133270264, -1.4941060543060303, -0.028368188068270683, -0.9492201209068298, 0.34317371249198914, -0.11417441070079803, -1.376420497894287, 1.4015297889709473, -0.17579621076583862, -0.3835478127002716, 0.6678750514984131, 0.3460368812084198, 1.4466121196746826, -0.07299894839525223, 0.3441556394100189, -0.14254774153232574, 1.4075413942337036, 0.2590102255344391, -0.4985690414905548, -0.6861193776130676, -0.13205887377262115, -0.10382454097270966, -0.8956162929534912, 0.06319708377122879, -0.11005076766014099, 0.757156252861023, -0.8771438598632812, 0.8903173208236694, 1.2196534872055054, -0.007883697748184204, 0.4748518168926239, 1.2797363996505737, 0.3621305227279663, -0.7382799983024597, -1.1544907093048096, 1.4116357564926147, 0.5565044283866882, -0.2640947997570038, 0.6522721648216248, 1.0581464767456055, -1.3055492639541626, 0.4982033967971802], [0.5353716611862183, -0.21563801169395447, -0.9002373814582825, 2.0900261402130127, 0.68630450963974, -0.7896310091018677, 1.1966643333435059, -1.0862476825714111, 0.2545487880706787, -1.1854150295257568, 0.2366839051246643, 1.5580617189407349, -0.07612653821706772, 0.9408491253852844, 2.2054028511047363, 0.057369478046894073, -0.8837796449661255, -0.07856126129627228, -1.138007640838623, -0.17001275718212128, -0.0373074933886528, 0.21510405838489532, 0.06850023567676544, 2.344141960144043, 1.3769928216934204, -1.802895188331604, 0.8918585777282715, -2.332885503768921, -1.4243446588516235, -0.9425370097160339, -0.8585119247436523, 0.25286802649497986, 0.14809302985668182, 0.11285828053951263, -1.5070643424987793, -1.8975346088409424, 0.27751731872558594, 1.694290280342102, 1.3094295263290405, 1.5395711660385132, 0.32739558815956116, 0.5049982666969299, -1.670661449432373, 2.206712245941162, 0.2468242049217224, -0.91825932264328, -1.1802864074707031, -0.7038296461105347, -0.08623380213975906, 0.1897251158952713, 0.24377724528312683, -2.4459967613220215, 0.29947274923324585, -1.1991626024246216, 0.9506626725196838, 1.5321892499923706, -0.4094965159893036, -0.11728259176015854, -1.2969872951507568, -0.0766986832022667, -2.612813949584961, -0.9357474446296692, 0.3460422158241272, -0.7814096212387085, 1.6781524419784546, -0.48117300868034363, -0.4629664719104767, -0.352242648601532, -0.22360557317733765, 0.52317875623703, 0.0853574350476265, -0.12522201240062714, -0.7481428384780884, 1.5647871494293213, 0.05375773832201958, -0.6315106749534607, -0.1288188397884369, -0.0143646951764822, -0.41789552569389343, 0.709860622882843, -0.4798997640609741, 0.3561621606349945, 0.6099604964256287, 0.6273279786109924, -0.17115770280361176, 0.2344943881034851, 0.11466023325920105, 0.7218095660209656, 0.8503726720809937, 0.996946394443512, -1.2838590145111084, -0.7783081531524658, -0.22635459899902344, 0.41008785367012024, -2.5933735370635986, -0.09266844391822815, -0.9682649374008179, 0.3810824155807495, 0.5798971056938171, 0.4292438328266144, 0.8385889530181885, -0.5263891220092773, 0.21336153149604797, 0.8268769383430481, -1.4713200330734253, -0.8741526007652283, 0.6253865957260132, -0.0036441355478018522, 0.9370309114456177, 0.7955349683761597, 0.11520969122648239, -1.4777584075927734, 0.9110509753227234, -0.08629217743873596, -0.9424395561218262, 0.8769738078117371, 0.6449947357177734, 0.5815477967262268, -0.556495189666748, -0.6378493309020996, 1.779563546180725, -0.238236665725708, 0.04672618955373764, 0.030613843351602554, -1.783664345741272, 0.6980082392692566, 1.1000136137008667, 0.5504534244537354, -0.27112212777137756], [-0.1389836221933365, 0.9882553815841675, 0.39271974563598633, 0.46588072180747986, 2.0480148792266846, -0.7654651403427124, -0.6296196579933167, 0.030613861978054047, 0.5450307130813599, -1.7174525260925293, 0.08503227680921555, -0.05675823614001274, 0.3534575402736664, 0.3537524938583374, -0.99339359998703, 0.1110721156001091, -0.7731576561927795, -0.32884275913238525, 0.7193437218666077, -0.3708665072917938, 0.8494895100593567, 1.6851468086242676, -0.4320385456085205, -0.4725469648838043, -0.08630432933568954, 1.8407539129257202, 0.6094450354576111, -0.13723154366016388, 0.07184912264347076, -0.6415830254554749, 1.2783547639846802, -0.013740845024585724, -0.773089587688446, -0.5412143468856812, 0.7971221804618835, 0.8191670179367065, -1.2308523654937744, 0.10328293591737747, 1.3266353607177734, 0.8560878038406372, 2.0136826038360596, -0.3993697166442871, 1.1190667152404785, 0.6873320937156677, 0.2740806043148041, -0.46636804938316345, 2.787799596786499, -1.6390950679779053, 1.4983271360397339, 2.138779401779175, -1.1250886917114258, 0.3566628098487854, 0.07350794970989227, -0.9005395770072937, 1.5744316577911377, 1.0793708562850952, 1.5003442764282227, -0.7322573065757751, -1.4845019578933716, -1.7786606550216675, -0.5271109342575073, 0.20700159668922424, -0.31040653586387634, 0.487150639295578, -1.6391218900680542, 0.10508764535188675, 1.0028427839279175, -0.2566550374031067, -0.24762938916683197, 0.2539743483066559, 0.29486823081970215, 0.8189921379089355, -0.9807217121124268, 0.6727650761604309, -1.4177409410476685, 0.25817176699638367, 2.4916248321533203, 0.3223191797733307, -1.4754890203475952, 0.09229214489459991, 0.49558907747268677, -1.1340574026107788, -0.30325430631637573, -0.9370670318603516, 0.45668360590934753, 1.8427131175994873, 0.82371985912323, 0.6147457361221313, -0.4961791932582855, 1.2550045251846313, -0.7562352418899536, -0.8353599905967712, -0.8548857569694519, -0.5008609294891357, 1.0475581884384155, -1.7883753776550293, -1.2378387451171875, -0.011788909323513508, -0.5269898176193237, 0.7343300580978394, -2.056713104248047, -0.5164160132408142, 0.33147841691970825, -1.0861735343933105, -1.4297630786895752, -0.5172010660171509, 0.26069343090057373, -0.7448529601097107, 1.599092721939087, -0.053571511059999466, -0.6006836295127869, -0.4346511960029602, 0.8491847515106201, 0.26809853315353394, 0.9725481867790222, 0.5600420832633972, 0.3173529803752899, -0.602997899055481, -1.3945813179016113, 0.159849613904953, -0.6814166903495789, 0.1354324370622635, 0.38032233715057373, 0.20930251479148865, 0.4338558614253998, 0.5025188326835632, -0.5091960430145264, 0.646060049533844, 0.28042641282081604], [-0.12364070117473602, -2.9248695373535156, -0.6865883469581604, -0.22373682260513306, -0.27237582206726074, 1.6650630235671997, -0.48012682795524597, -0.3542175889015198, 0.11953352391719818, -1.6382982730865479, -0.1739085167646408, 0.7360593676567078, 1.2652244567871094, 0.36000800132751465, -0.9288411736488342, -1.406726598739624, 0.6316772103309631, 0.4087420701980591, -0.5427453517913818, -0.2581179738044739, -0.2963316738605499, -1.8184618949890137, 2.2784972190856934, -0.7830109000205994, -0.21822257339954376, -0.1211867704987526, 1.974496841430664, 0.23715262115001678, 0.817764937877655, 0.003570257220417261, -0.5898684859275818, -0.7561367750167847, 0.06621114909648895, -0.15846629440784454, 0.6322317719459534, 0.8917486667633057, 1.0871857404708862, 1.8488367795944214, 0.7826924920082092, -0.2838214039802551, 1.2008781433105469, -0.19826874136924744, -0.1400933861732483, 1.7502728700637817, 0.3064287602901459, -1.1736319065093994, 0.5786328911781311, -0.7324666976928711, 0.4865298867225647, 0.9674471020698547, -1.0044695138931274, 0.3021451532840729, -0.5450649857521057, 1.7447260618209839, -0.5513522624969482, -2.035259962081909, -0.13369184732437134, -0.4874986410140991, -1.5896414518356323, -0.6179331541061401, -0.22363214194774628, 0.6960927248001099, 0.3766018748283386, 0.37476426362991333, -0.07863122969865799, 0.5168720483779907, 0.9789868593215942, 0.7618529796600342, -1.0437705516815186, -2.1331050395965576, -1.0566108226776123, 0.8190506100654602, -0.017541514709591866, 0.4569249153137207, -0.5787651538848877, 0.7276169657707214, 1.0378037691116333, -0.46190688014030457, -1.4639580249786377, 2.01692533493042, -0.3230949640274048, -0.26640602946281433, -0.753241777420044, 1.5772491693496704, 0.1508689969778061, 1.2145283222198486, 0.13481825590133667, -0.36325526237487793, 1.5459181070327759, 1.1039371490478516, 0.523134171962738, 1.258012056350708, -0.7430614233016968, -0.4315067231655121, -0.10142627358436584, -2.2829034328460693, 0.2689342498779297, -0.5728970766067505, 0.6199434399604797, 0.48958298563957214, 1.4358357191085815, -1.6208069324493408, 0.14666658639907837, 0.8339659571647644, 0.134629026055336, -0.22062690556049347, -0.7909303903579712, -1.0656849145889282, 0.8080175518989563, 1.9135979413986206, -1.3874784708023071, 2.145854949951172, -1.1689705848693848, -0.20435380935668945, 0.5184623599052429, -0.42929014563560486, 0.2611507177352905, -0.29393360018730164, -0.2410896122455597, 0.6081592440605164, -1.3294250965118408, 0.9114495515823364, -1.49210786819458, 0.4388106167316437, 1.0494600534439087, 0.8942983746528625, 0.13630911707878113, -0.06201636418700218, -0.12376636266708374], [-0.32014796137809753, -0.057786572724580765, 0.08338840305805206, 1.6550112962722778, 1.061373233795166, -0.905289888381958, 0.7713621258735657, 0.10212942957878113, 0.08320507407188416, -1.13909113407135, -0.5242494940757751, -0.0448627769947052, 1.2847247123718262, 0.353137731552124, -1.6359440088272095, 1.3649075031280518, 1.7882630825042725, -0.2098250389099121, -1.3831305503845215, -0.509302020072937, -0.12954218685626984, -0.31744444370269775, 1.1323522329330444, 0.11887487024068832, 0.715094268321991, 0.5484521389007568, 0.509081244468689, 0.08151345700025558, -0.2040213793516159, 0.12095589935779572, 0.4985333979129791, -0.4240446090698242, -0.4214886724948883, -1.0915474891662598, -0.8302395939826965, -2.092283010482788, 0.3434927463531494, -1.8220367431640625, 1.4433646202087402, 0.886141836643219, 2.1533799171447754, 0.8098309636116028, 0.08266743272542953, 1.3695528507232666, -0.8697419762611389, 0.8807317614555359, -0.49896878004074097, -0.26883774995803833, -1.044104814529419, -0.1499648541212082, 1.1479437351226807, -0.41375455260276794, 0.6213164925575256, -1.5107753276824951, 1.459701657295227, 1.190708875656128, -2.7631285190582275, -0.8944523930549622, 0.5910912156105042, -1.2795064449310303, 0.5003902316093445, -0.5637205243110657, 0.9740303158760071, 0.681468665599823, 1.8299840688705444, 1.0537134408950806, 1.041785478591919, -2.3219010829925537, 0.37745699286460876, -1.5214499235153198, -1.477307677268982, -1.7456825971603394, -0.9324540495872498, 1.0241371393203735, 1.7992981672286987, -1.445243239402771, -2.179368734359741, -1.5851773023605347, -0.2430148869752884, -1.3480950593948364, 0.15471559762954712, 0.9201596975326538, 1.0995055437088013, 0.04778163507580757, 0.9280065298080444, 1.7141832113265991, -0.8779094219207764, 1.6746810674667358, -1.4601500034332275, 1.8235065937042236, -0.13200843334197998, 0.9160202145576477, 0.3864695727825165, -0.43143317103385925, -0.3065280020236969, -1.411152958869934, 0.8661997318267822, -0.14383070170879364, -0.29439815878868103, -0.5491994023323059, 0.36472439765930176, -0.4963986575603485, -1.084489107131958, -0.9351581931114197, -0.5792341232299805, -0.13938510417938232, -0.13111095130443573, 0.5867482423782349, 0.376146525144577, 0.28660404682159424, 0.16122213006019592, 0.4107172191143036, -1.7347534894943237, -0.9509531259536743, -0.7245000600814819, -2.0206410884857178, -0.7200543880462646, 0.059160396456718445, -0.4318492114543915, -0.1119043380022049, 0.621218740940094, 1.1051114797592163, -2.798435926437378, -0.664532482624054, -1.6477235555648804, 0.21086610853672028, -2.263887643814087, 1.9052989482879639, -0.15408679842948914], [-0.5190401673316956, -0.6182454228401184, 0.06331748515367508, -2.3150391578674316, 0.5708549618721008, -1.6256300210952759, -0.4351256787776947, 0.420504093170166, 0.017852773889899254, 0.29107651114463806, -0.08422471582889557, 1.1759123802185059, -1.976521611213684, 0.34058377146720886, -0.2853775918483734, 0.2871435880661011, 0.8285110592842102, 0.39447012543678284, -0.45795875787734985, 0.82732754945755, 0.8612779378890991, 0.682201087474823, 0.7199908494949341, 1.6859418153762817, 1.2208656072616577, 1.1702330112457275, 1.662992238998413, -0.4435443580150604, -0.4174010157585144, -0.4697563052177429, 0.6380606889724731, -0.31829556822776794, 0.7480482459068298, -0.10327232629060745, -0.5357909202575684, 1.4038461446762085, -0.5118517875671387, 0.20980817079544067, -1.2993431091308594, -0.25247833132743835, 1.933897852897644, -1.3743171691894531, -0.7953830361366272, -0.7652759552001953, -0.621623694896698, 1.294285535812378, 0.7780737280845642, -2.704456090927124, -0.7223612070083618, 1.3799620866775513, 0.314274400472641, -0.9787347316741943, 0.7775714993476868, -0.5160718560218811, -0.37009793519973755, 1.4781460762023926, 1.691672444343567, -0.8073936700820923, -2.127660036087036, 0.5562245845794678, -0.9004008769989014, -0.4627077579498291, -1.0624022483825684, -1.2684211730957031, -0.5492218136787415, -2.167762279510498, 0.31689757108688354, 0.18982431292533875, -0.3149866461753845, -0.14990465342998505, 0.6098911762237549, 1.1588735580444336, -1.1555055379867554, 0.6671020984649658, -0.4325220286846161, -0.12277296185493469, 1.3521754741668701, -0.1883346289396286, 0.5966445207595825, -0.30245503783226013, 0.28848010301589966, 0.6948423385620117, -0.6572306752204895, -0.9109352827072144, 1.0670994520187378, 0.002854739548638463, 0.3425832986831665, -0.8779460191726685, 0.2827151417732239, -0.6448633670806885, -0.40029001235961914, -1.6088457107543945, -0.3963201642036438, -0.5667341351509094, -0.5706655383110046, -0.8342716693878174, 0.09959982335567474, 1.2413758039474487, 0.5651803016662598, -0.09079425781965256, 1.6613713502883911, 0.7432557940483093, 1.2963922023773193, -1.589197039604187, -0.4392494261264801, -0.3032190799713135, 1.090187907218933, 1.7934523820877075, -0.03787602484226227, -0.649658739566803, 0.3954277038574219, -0.03155193477869034, -0.4485197067260742, -0.5387507677078247, -0.25632771849632263, 0.02369031496345997, -0.4304964542388916, 0.07291222363710403, -1.607230305671692, -0.46005135774612427, -0.3556555211544037, 0.09382685273885727, -1.1391392946243286, -0.18836186826229095, 1.1222822666168213, -0.3402469754219055, 0.027054443955421448, 0.8492098450660706, -0.26743030548095703], [0.18465396761894226, -0.5549615025520325, 1.5740303993225098, 0.30987706780433655, 0.15899118781089783, -0.33298614621162415, 0.49723249673843384, 1.0978769063949585, 0.44979649782180786, 0.06815476715564728, -1.082181453704834, -0.15781299769878387, 0.8934547901153564, 2.1774821281433105, 1.5170108079910278, 0.7036807537078857, -0.6709651350975037, -0.9291089177131653, 0.20557132363319397, -0.08747946470975876, -0.012857140973210335, 0.41676065325737, 0.9725382924079895, 1.4378414154052734, 0.6808249950408936, 0.23900821805000305, -0.9711608290672302, 0.443137526512146, 1.0201514959335327, 0.4573749601840973, -0.509174644947052, -0.691586434841156, 2.0712599754333496, -0.45052048563957214, 1.5869369506835938, 1.3369619846343994, 0.8732951283454895, -1.6514569520950317, -0.39593756198883057, 1.758737325668335, 0.7462304830551147, 0.886009931564331, 0.5710964798927307, -0.22615626454353333, -1.2840986251831055, -0.2390158325433731, -0.06756429374217987, -0.23617833852767944, -1.9974504709243774, -0.5242219567298889, 2.024860143661499, -0.8330204486846924, 1.2868294715881348, 0.7290582656860352, -0.44404205679893494, 0.08090657740831375, 0.9016545414924622, -1.4778810739517212, 0.4748331606388092, 0.34112411737442017, 1.4081462621688843, 0.13209855556488037, 0.882931649684906, 0.44702082872390747, 0.3783242702484131, -0.8533765077590942, -0.15676452219486237, -0.81712406873703, -0.4658972918987274, -0.6230521202087402, -0.8213815689086914, -0.7519457340240479, 0.2410757839679718, -1.6547443866729736, -0.9378670454025269, -0.6806671619415283, -0.5070149302482605, -0.10684133321046829, -1.3355326652526855, 0.4495506286621094, -1.2625943422317505, -0.7977285981178284, -1.3715146780014038, 1.220790982246399, -0.6485982537269592, -0.04748678207397461, -0.4146687984466553, -0.2556960880756378, -0.38988766074180603, 0.9688941240310669, 0.23430056869983673, 0.7914807200431824, 1.4381744861602783, -1.389164686203003, 0.7434836626052856, -0.7507544755935669, 1.7356098890304565, 1.6768842935562134, 1.6193206310272217, -1.2895084619522095, 0.5667563080787659, 0.8321309685707092, 0.6759104132652283, -0.050621505826711655, -0.2771904766559601, -0.759109377861023, 0.8241183161735535, -1.3007475137710571, -0.9350056648254395, -1.7333775758743286, -1.4581196308135986, 1.5128469467163086, -0.7017163634300232, -0.9415250420570374, 0.7648159861564636, 2.0801775455474854, -2.068216562271118, -0.24081693589687347, -0.33142930269241333, -0.9264214038848877, -1.8884689807891846, 1.2713932991027832, 1.5384416580200195, 0.7297185063362122, -0.20471033453941345, -0.06100514158606529, 0.5870177149772644, -1.1745398044586182, -1.2804279327392578], [0.1806853860616684, -2.1857428550720215, -0.001552832080051303, 1.5319803953170776, -1.6043444871902466, 1.2323819398880005, -1.0195902585983276, 0.7870386242866516, -1.2137818336486816, -0.5729129910469055, 0.606307327747345, -0.5137166976928711, 0.16993947327136993, -1.5530308485031128, -1.8541574478149414, 0.8202139735221863, 0.13325774669647217, -0.42230260372161865, 0.11800120770931244, -0.888508677482605, -0.2282218337059021, 0.21744674444198608, 1.9191325902938843, -2.6328279972076416, 0.4816228747367859, 1.9355394840240479, -0.1188337653875351, 0.07915911823511124, -0.19760894775390625, -0.09612567722797394, -0.5202136635780334, -0.6273595094680786, -0.32104215025901794, 0.3401295840740204, -2.640894889831543, -0.5258220434188843, -0.4273737370967865, -1.299758791923523, 0.47586825489997864, 0.3176809549331665, 0.3211793303489685, -1.095569133758545, -0.4143015146255493, 1.946797251701355, 0.6647225618362427, 0.9109165668487549, -0.002864277921617031, -0.7931148409843445, -0.3074555993080139, 0.058472760021686554, 0.9868089556694031, -2.189884662628174, 0.44683128595352173, -0.27813005447387695, -1.59423828125, -1.1386417150497437, 0.6619083285331726, 1.082497477531433, -0.06034905090928078, 0.5301252603530884, 0.9886676669120789, -0.47128811478614807, -0.27460840344429016, 0.5496344566345215, 0.09577412903308868, 0.4062064290046692, 0.23935560882091522, -1.1133222579956055, -0.8906047344207764, -0.4521920382976532, 0.9916908144950867, 0.24984757602214813, -0.5953717827796936, -0.9293503165245056, -1.659989833831787, -0.5459052920341492, 0.7164134979248047, -1.016951560974121, -0.9604802131652832, -3.7893662452697754, 0.2909959554672241, -0.4496334195137024, 1.4017177820205688, -0.003941617906093597, -0.9778521060943604, -0.10395749658346176, 0.34767401218414307, 0.20116478204727173, 0.8394590616226196, -0.650722086429596, -0.03196541219949722, -0.3411625325679779, 0.14422905445098877, 0.8815708160400391, -0.18433044850826263, 1.0985959768295288, 0.8917423486709595, 0.053904157131910324, 1.015135645866394, -1.7549715042114258, -0.21819965541362762, -0.24434374272823334, -1.2414357662200928, -0.3533727824687958, 1.265541911125183, -0.36273351311683655, 1.6056132316589355, -1.8607287406921387, 0.08185958117246628, -1.0399545431137085, 0.734097421169281, 0.9786493182182312, -3.274477243423462, 1.703166127204895, -2.0995912551879883, -1.237620234489441, 1.3557425737380981, 0.7157978415489197, 0.3068816065788269, -0.13550259172916412, -1.0061336755752563, -1.3852834701538086, 0.3031716048717499, -2.059309244155884, -0.6326002478599548, -1.60045325756073, 0.4318799078464508, -0.5188279747962952, 0.6194416284561157], [-1.4219434261322021, -1.3573265075683594, -1.335376262664795, -1.36484694480896, 0.26276788115501404, -0.6785125136375427, 0.14136573672294617, 1.7162461280822754, -0.47127509117126465, 0.028632964938879013, -0.48014265298843384, -0.4634909927845001, -0.04573056101799011, -0.256682425737381, 0.06541413068771362, 2.2050669193267822, 0.09639488160610199, 0.11044672131538391, -0.5171129703521729, 0.6741517186164856, 0.17802682518959045, -0.09868607670068741, 1.0017505884170532, 0.6854534149169922, -0.4338621497154236, 0.6979464888572693, 0.6735790967941284, 0.16996589303016663, 0.5538203716278076, -2.2497267723083496, 0.38457727432250977, -0.21205763518810272, 1.17233407497406, 0.8529582023620605, -0.4637574851512909, 1.091719388961792, -0.7831997871398926, 0.3318081498146057, -0.4961976408958435, 0.9021728038787842, -0.2186545729637146, -0.8789637088775635, 0.27529922127723694, 0.7855569124221802, -0.7679994106292725, 0.10554400086402893, 0.5449874997138977, -0.7586711645126343, 0.9446578621864319, 0.8791205883026123, -1.6413129568099976, -0.28203272819519043, -0.11252368241548538, -0.09599167108535767, 0.1788976639509201, 1.2282391786575317, -0.5536730885505676, 0.6789135932922363, -0.3280755579471588, 0.43885868787765503, -0.4559783935546875, 1.4793362617492676, -1.7767704725265503, -0.03357589617371559, -0.007252484560012817, -0.578216016292572, -0.8371137976646423, -0.022090990096330643, 2.2692112922668457, -1.561038613319397, -0.9138640761375427, 1.0481293201446533, -0.8709050416946411, 1.3621805906295776, 0.7098182439804077, -0.8708393573760986, 0.03459681570529938, -0.8314560651779175, 1.0488073825836182, 0.5004603266716003, 0.5823747515678406, -1.1928414106369019, -1.5836050510406494, -1.458351969718933, -1.7518336772918701, -0.920016884803772, -0.49412521719932556, -0.5998182892799377, 0.05659128725528717, -0.7881142497062683, 0.838814914226532, 0.8240516781806946, 0.9693591594696045, 0.5131431221961975, 2.8273932933807373, -0.007168528623878956, -1.3360536098480225, 0.43652939796447754, 1.3351786136627197, 1.1787524223327637, 0.7674475312232971, 0.7089101076126099, -0.6216357946395874, 1.1007224321365356, 1.0641181468963623, -0.5288442969322205, -0.6418676376342773, -1.0968515872955322, 1.1124449968338013, -0.013864601030945778, 0.6024346947669983, -1.2275893688201904, -1.4153199195861816, 0.7628893852233887, -1.278586745262146, -0.4600590169429779, -0.12216981500387192, -1.117357611656189, 3.025169610977173, 1.6810017824172974, -0.6387593150138855, 1.075952172279358, -0.8009644150733948, -0.5246642827987671, 0.12070776522159576, -0.015272945165634155, -0.4983712136745453, 0.3463759422302246, -0.9580378532409668], [1.0504039525985718, 0.96543288230896, -0.4877152740955353, -0.44502949714660645, 0.3988226652145386, 0.7612918019294739, -0.21890702843666077, 0.5035336017608643, 0.9824586510658264, -1.2026406526565552, -0.640040934085846, -0.3052673637866974, 0.3134804964065552, -0.9125837683677673, -0.9485703110694885, -2.014608144760132, 0.8846074938774109, -0.4872375726699829, -0.20463883876800537, 0.6130767464637756, 0.3764738440513611, -1.3054232597351074, 0.6409915685653687, 0.7386592030525208, -0.8600014448165894, -1.1326618194580078, -0.20547737181186676, 1.324578046798706, -0.30282050371170044, 1.977721929550171, -0.6118263602256775, 0.5629698038101196, 0.08550867438316345, 0.08229880034923553, -0.869817316532135, -2.3420236110687256, 0.09653669595718384, 0.20797337591648102, 0.8686204552650452, 0.9852088689804077, 0.47171488404273987, -0.26196208596229553, -0.16903211176395416, 0.04539269581437111, 0.22543279826641083, 0.9873253107070923, 0.43621987104415894, 1.1741182804107666, 2.354458808898926, 0.3011184334754944, 0.537119448184967, -1.5929843187332153, 2.4037985801696777, 0.39500823616981506, -0.45527854561805725, 1.620650291442871, -0.678027868270874, -0.5362146496772766, 0.044796038419008255, 0.23366950452327728, 0.02968987450003624, -0.6863139867782593, -0.4014664888381958, -0.4702564477920532, 0.024278899654746056, 0.4467824101448059, -0.3348754346370697, 0.8372983932495117, 0.8324019908905029, 0.8406370282173157, -1.8174610137939453, 0.25277137756347656, -0.100050188601017, 0.12529630959033966, -0.654951274394989, -1.0927178859710693, 0.7789117693901062, 0.4996604919433594, 1.5166423320770264, -0.2186127007007599, 2.4198203086853027, -1.8142839670181274, -1.368449091911316, -0.22008547186851501, 0.5214122533798218, 0.9950311779975891, -0.5738837718963623, 0.06533677875995636, -1.02571702003479, -0.20557986199855804, 0.029659463092684746, 0.31948381662368774, -0.7258918881416321, 1.43106210231781, 1.0926529169082642, 0.11418384313583374, 0.6731200814247131, 1.624405026435852, 0.9392563700675964, 0.33493971824645996, 0.06499605625867844, 1.5987516641616821, -0.293237566947937, 0.9640470743179321, -1.8044028282165527, -0.6627889275550842, -1.4877363443374634, -0.04931323602795601, -0.34746068716049194, -0.28894996643066406, 0.30250921845436096, -1.719537377357483, 1.7508796453475952, -0.4176701605319977, 0.9795166850090027, -1.4211678504943848, 0.5297529101371765, 1.9426093101501465, -1.8777716159820557, -0.8887647986412048, -0.3645380139350891, -1.4562636613845825, 0.6848496198654175, 0.6381583213806152, -2.0424857139587402, 0.7154930233955383, 1.9541893005371094, 2.067383289337158, -1.2135306596755981], [-1.0834674835205078, -0.34858185052871704, 0.09163554757833481, -1.4158803224563599, 0.3891357481479645, -1.1002269983291626, 0.3386736810207367, 0.0853574201464653, -0.07919926941394806, 0.045836687088012695, -1.3927569389343262, -1.2475003004074097, -1.9828746318817139, 1.952384114265442, -0.34857243299484253, -0.8601560592651367, -1.6552826166152954, -1.5469130277633667, 2.0499563217163086, 0.11297041922807693, 0.9628608822822571, 0.6880711317062378, -0.43385130167007446, -0.9132848978042603, 0.5715788006782532, 1.6252362728118896, -0.09881313890218735, -0.1340777426958084, -1.478412389755249, -0.07457342743873596, -0.7495582699775696, 0.33491435647010803, 0.888003408908844, -0.9668020009994507, 0.4942437410354614, 0.7546077370643616, -0.7659778594970703, -1.0763317346572876, -1.0182135105133057, 0.21421825885772705, -1.436316728591919, -1.3800630569458008, -1.7879103422164917, -1.036229133605957, -0.5777970552444458, -1.938955307006836, 0.010608510114252567, -0.0642777606844902, 0.6347978115081787, 0.18901991844177246, -0.5039456486701965, -0.6805251836776733, 1.5100910663604736, -0.45760735869407654, 1.5482304096221924, -0.4104302227497101, 0.1510515660047531, -0.7723244428634644, 0.025374380871653557, 0.05356014892458916, -0.04508701339364052, -2.1063520908355713, 0.6355475783348083, 0.2611834406852722, 0.17826808989048004, 0.022044051438570023, 1.8518693447113037, -1.3403958082199097, 0.9933168292045593, -0.8156585097312927, -0.1647532880306244, 0.7057428359985352, 1.1928685903549194, 0.7855494022369385, -0.3360261619091034, 0.1624891459941864, 1.6812673807144165, -0.14191050827503204, -0.6649807691574097, -0.83906489610672, -1.509986162185669, 2.222496509552002, 0.06757193058729172, -1.1272834539413452, -1.1167815923690796, 0.23539115488529205, 0.24531832337379456, 1.929322361946106, -0.6022655367851257, 0.7812958359718323, -0.6898083686828613, -0.20096714794635773, 0.22967404127120972, -0.3068820834159851, -1.1717510223388672, 0.7144865393638611, 1.2575910091400146, -0.17294535040855408, -0.4788684844970703, -0.1993466317653656, -0.7170110940933228, -0.7444018125534058, 1.5477129220962524, -0.2906630039215088, -0.28229913115501404, -1.0554896593093872, -1.4736653566360474, -0.5964888334274292, 0.1697370558977127, -1.272323489189148, -0.17796343564987183, -0.35370713472366333, -1.73951256275177, -0.2665208578109741, -0.6363736987113953, 0.44979503750801086, 0.05392534285783768, 0.346919447183609, -0.8882356882095337, -0.34774285554885864, 1.0730422735214233, -0.4957236349582672, 1.6283928155899048, -1.7097092866897583, -1.7044340372085571, -0.45643550157546997, -0.09776738286018372, -0.2127932906150818, -1.4155162572860718], [-0.03755395859479904, -0.7752719521522522, 0.29225581884384155, -0.1792316436767578, 1.3294509649276733, 0.7955471873283386, 0.21829305589199066, 0.6503761410713196, -0.3641493618488312, -0.169894739985466, -0.015116937458515167, -2.4232776165008545, 1.6860419511795044, -0.5152538418769836, -0.5681416392326355, -1.0742768049240112, -0.6398434638977051, -0.7290208339691162, 0.380058616399765, 0.6337660551071167, 0.4208674430847168, 0.5899121165275574, -1.1085795164108276, -0.72682124376297, 0.017904486507177353, 0.6408279538154602, -0.13354894518852234, -1.5929442644119263, -0.4022250473499298, 0.988045334815979, 0.2521836757659912, -0.9460912942886353, 0.875154972076416, 0.17480109632015228, 1.4172098636627197, 0.8347729444503784, -0.8735950589179993, 1.0146812200546265, -1.5047231912612915, -0.29475852847099304, 0.633605420589447, 0.17353542149066925, -0.9065411686897278, 1.4787750244140625, -1.443729281425476, -0.2635802626609802, -0.004357403144240379, -1.2360423803329468, 1.9208369255065918, -1.3540722131729126, -0.6723902821540833, 0.2672719657421112, 1.8152588605880737, 0.3907429873943329, -0.36353975534439087, -0.8468652963638306, -0.8859329223632812, 0.05354367569088936, -0.5032175183296204, -1.3420716524124146, -0.20985600352287292, 0.7481065392494202, 1.3188480138778687, 1.9490453004837036, -0.7098999619483948, -1.2098371982574463, 0.2733921408653259, 0.470159113407135, -2.3745977878570557, 1.5710731744766235, -0.9760952591896057, -1.358616828918457, 1.2940421104431152, 0.6486396193504333, 0.109672412276268, 0.3475501239299774, 0.750175416469574, 0.10001686215400696, -0.4804854094982147, -0.10896780341863632, -1.088049054145813, -2.0473692417144775, 0.6735346913337708, -1.082061529159546, 0.7215014100074768, -1.1602411270141602, -1.0313202142715454, -0.32831549644470215, -0.5559952855110168, -0.5889939665794373, 0.08995406329631805, -0.5049978494644165, 0.9329602718353271, 0.8209205269813538, 0.7767636775970459, -0.7051619291305542, -0.5021161437034607, 1.5277713537216187, 0.4156343340873718, -1.261284351348877, 2.3240318298339844, 0.7742576599121094, -0.20888908207416534, 0.6326162815093994, -0.19895200431346893, -0.346835732460022, 0.5332924127578735, -1.0514353513717651, 1.0559016466140747, 0.02511828951537609, -0.2612917125225067, -2.4269351959228516, -0.22523197531700134, 0.5812865495681763, 1.256152629852295, 1.72160804271698, -0.9998135566711426, -0.15585733950138092, 0.3907422125339508, -1.2681896686553955, -1.0045816898345947, -0.4992371201515198, 3.24727725982666, 0.23515257239341736, 1.6948624849319458, 1.6446410417556763, -1.8354926109313965, 0.8598170280456543, -0.7611609697341919], [-0.514480471611023, 1.9890024662017822, -0.9593050479888916, 0.663312554359436, 0.253740519285202, -1.3897724151611328, -2.4423000812530518, -0.6937224268913269, -0.5808720588684082, 0.6476295590400696, 0.6901981830596924, -0.4496515393257141, -0.01596885547041893, 0.37548941373825073, -0.6073561310768127, -2.3032138347625732, -1.3510817289352417, 1.6087630987167358, -0.24874384701251984, 0.514680802822113, -1.6295801401138306, -1.204109787940979, -0.09408301115036011, 0.7348778247833252, 2.1071979999542236, 0.48212841153144836, -0.474232941865921, 0.8072124123573303, -0.3523516058921814, 0.7573339939117432, -0.44557487964630127, 0.25048041343688965, -1.6126350164413452, -1.1150184869766235, 1.0635071992874146, -0.11408120393753052, -0.6298399567604065, 1.9450534582138062, 0.9088848829269409, 0.33608657121658325, -0.8171741366386414, -1.1646486520767212, 1.4022929668426514, -0.13137467205524445, 0.2706701457500458, 1.8367583751678467, 1.0603567361831665, -0.24491389095783234, -1.7134674787521362, 0.8252755403518677, -0.9766952395439148, -0.4738794267177582, -0.9603872895240784, -2.211446523666382, 2.632263660430908, 0.7735596895217896, -0.36364516615867615, 1.8568285703659058, 0.7741702198982239, 1.8430461883544922, 1.4907561540603638, -0.9746297597885132, 0.714596152305603, 0.9833546280860901, 0.10548365861177444, -0.503373920917511, 0.21462465822696686, 1.5164203643798828, 1.94487464427948, -0.5268340706825256, 0.11172304302453995, 1.3784171342849731, 1.4693374633789062, 0.9407143592834473, 0.0028452842961996794, -0.5022845268249512, 0.7198834419250488, -1.0682555437088013, -1.0710232257843018, -0.08997168391942978, 0.050351448357105255, 1.4901018142700195, -0.6777980923652649, 1.952731728553772, 0.657170295715332, -0.9306150674819946, -0.18759284913539886, -1.014056921005249, -0.8982225656509399, -0.21087078750133514, -0.5369695425033569, -1.1584620475769043, -0.5683689117431641, 0.5851991772651672, 1.9648258686065674, -1.098532795906067, -0.20825989544391632, -0.4908522367477417, -0.07942602783441544, -1.3055838346481323, -1.5209068059921265, 1.1117140054702759, -0.38059377670288086, 0.4665381908416748, -1.1945397853851318, -1.570257306098938, -1.1133002042770386, 0.857720673084259, -0.2498760223388672, -0.2804470658302307, 0.7368831038475037, 2.026702880859375, 0.5265388488769531, -1.074373483657837, -1.3387949466705322, 0.3169337213039398, 0.7438398003578186, -0.7981204986572266, -1.0943920612335205, -0.4706967771053314, -0.30555567145347595, -0.16721750795841217, 0.3320445418357849, 0.646996021270752, -0.5814621448516846, -1.4349496364593506, -1.6383823156356812, 2.267981767654419, 0.3964539170265198], [-0.8291997313499451, -0.5563209652900696, 0.35960930585861206, 0.25659871101379395, 0.4590652883052826, -0.47088301181793213, -0.14864887297153473, 0.3561720550060272, -0.08236096799373627, -0.06059915944933891, 1.427040457725525, 0.8021798729896545, 0.6623266339302063, -0.30185914039611816, -0.27710190415382385, 1.5371118783950806, -0.7061861753463745, 0.09317824989557266, 0.9815305471420288, 0.3582335412502289, -0.5033393502235413, 0.08851098269224167, 0.8328089118003845, 1.6005375385284424, 0.5313060283660889, -1.179553508758545, -0.8114432096481323, 0.11196281760931015, 0.45375770330429077, 0.8428683876991272, 0.29500284790992737, -1.9951072931289673, -0.18184229731559753, 0.09816469252109528, -1.1879650354385376, -0.09969977289438248, -0.636982798576355, -0.5946561098098755, 0.9532305002212524, 0.5918489098548889, -2.027700424194336, 1.5976775884628296, -1.5562350749969482, -0.48588740825653076, 0.5562726259231567, 1.2107123136520386, 0.8841192126274109, -1.4115263223648071, 0.17325818538665771, 2.34088397026062, 0.8523287773132324, 0.7663663029670715, 1.2551660537719727, 1.3197797536849976, -0.7855297923088074, 1.362721562385559, 0.49705320596694946, 1.3604490756988525, -1.1312307119369507, 0.8953393697738647, 2.4310429096221924, -2.230499505996704, 0.9228359460830688, 1.797318458557129, 0.40599963068962097, 0.9017361402511597, 0.14222274720668793, 1.4346474409103394, 0.07397986203432083, 0.4456554055213928, 0.8122087717056274, 1.1410294771194458, -0.8137673735618591, 0.20527426898479462, -1.3767672777175903, -0.8361991047859192, 0.0839046984910965, 0.1345447450876236, -0.267263799905777, 0.3125021755695343, -0.9531081318855286, -0.7884398698806763, -0.46469932794570923, 0.30176180601119995, -0.6200613975524902, -1.4348732233047485, 0.6854134202003479, 0.34871065616607666, 1.358505129814148, -1.772401213645935, -0.6953307390213013, -0.6603333950042725, 1.1723271608352661, 0.0995560735464096, 0.927303671836853, 0.6963914632797241, 1.394834041595459, -0.7220419049263, 0.041891422122716904, 1.5465643405914307, 0.6251856684684753, 1.5026021003723145, 0.6665617227554321, 1.4018031358718872, 0.5618895888328552, 0.10387811809778214, -1.7080376148223877, -1.1602106094360352, -0.2987738847732544, 0.1509791612625122, -0.12383239716291428, 0.6978978514671326, -1.6440578699111938, -0.7265108227729797, -0.8417933583259583, 0.21686142683029175, 0.3887428045272827, -0.5420900583267212, 1.4665933847427368, -1.6840949058532715, 0.9780492782592773, 2.412841558456421, -0.4319276213645935, -0.6125049591064453, -0.4082728326320648, 1.0459367036819458, -0.7824409008026123, 1.5489782094955444, -0.41199713945388794], [0.769231379032135, 0.8118817210197449, -2.199451446533203, 0.6575714349746704, 0.2497626394033432, 0.2581203877925873, 0.4150189757347107, -0.4370352327823639, -2.14650297164917, 0.039548080414533615, -0.21288183331489563, -0.3612377643585205, -0.8920517563819885, 0.11758273094892502, -0.7685936093330383, 0.6491925716400146, 1.4620680809020996, -1.5649312734603882, -1.0335534811019897, 1.221117377281189, -1.722042202949524, 0.0473690964281559, 1.2528042793273926, -0.042134471237659454, -0.07714154571294785, 1.597353219985962, -0.3707893192768097, -0.9907224178314209, -1.4213639497756958, 1.7981873750686646, -0.2875959277153015, 0.5983390212059021, -0.6835191249847412, 0.3323676884174347, -0.2878795266151428, -1.0894370079040527, 2.3053483963012695, -0.9990746974945068, -0.9864662289619446, 0.6885501146316528, -0.8171255588531494, -1.7770594358444214, -0.5903032422065735, -1.4536000490188599, 0.23611398041248322, -0.689143717288971, -0.20683099329471588, 1.264297604560852, 1.616269588470459, -0.06331872940063477, -1.0284583568572998, 2.2207915782928467, -0.4738462567329407, -2.503638982772827, 0.8944307565689087, -0.8562584519386292, 0.29742953181266785, -0.261771023273468, 1.2886892557144165, -0.20648998022079468, -1.4415552616119385, 0.8164252638816833, -0.08683910965919495, 0.35653597116470337, -0.9311914443969727, -1.4022599458694458, -1.3489861488342285, -1.4284555912017822, 1.3005808591842651, -1.5310138463974, 1.8408623933792114, -0.30697178840637207, 0.542255699634552, 0.28261637687683105, -0.4543481767177582, 0.10291183739900589, 0.5035188794136047, -0.577491283416748, -0.3564559817314148, 0.4648817181587219, -1.7647768259048462, 2.2008047103881836, -0.3300703763961792, -0.27686864137649536, 0.46753406524658203, 0.4056676924228668, -0.7250504493713379, 0.36972248554229736, -1.0281928777694702, -0.2262180745601654, -0.7516684532165527, 1.8083246946334839, -1.687961459159851, 0.7710690498352051, 1.8967528343200684, 1.030910849571228, -0.43546873331069946, 0.8282397985458374, 0.1299310177564621, 0.505735456943512, 0.1359395533800125, 2.2508771419525146, -0.15619978308677673, 0.529591977596283, -0.6999428868293762, -0.9638234972953796, 0.7741548418998718, 0.084557443857193, -0.9134385585784912, -1.0136876106262207, -1.255467176437378, 0.37725403904914856, 1.212214708328247, 2.04848051071167, -0.6869150996208191, 1.0334258079528809, -0.2318243384361267, -0.489797443151474, 1.7093154191970825, -0.8141666650772095, -1.2813986539840698, 0.21841968595981598, -1.4056134223937988, 0.5115939974784851, -1.0827387571334839, -0.4488532245159149, -0.5824017524719238, 0.4746391177177429, 0.7202556133270264], [-0.3225714862346649, 1.3559634685516357, -0.341958224773407, 1.0929579734802246, -0.36350688338279724, 0.6802628040313721, 0.2900922894477844, -0.14984959363937378, -0.9848636388778687, -0.367281049489975, 0.8652297258377075, 0.9570389986038208, -0.7200806140899658, -0.6210281848907471, 0.7795736789703369, -0.014755696058273315, -2.2027428150177, -0.6000242233276367, -0.5079113841056824, 0.5872671008110046, -1.6038851737976074, -0.9967731237411499, 1.2646090984344482, -0.6969698667526245, -2.106403350830078, 1.2326313257217407, 0.44697725772857666, -0.4949621856212616, -0.059572484344244, -0.32698532938957214, -0.8510504961013794, -0.3394263982772827, 0.4840039312839508, 0.14223262667655945, 1.2259424924850464, -0.04366050660610199, 0.7434567809104919, -0.6066393256187439, 0.27101853489875793, 0.8293598294258118, -1.9137952327728271, -0.06897425651550293, -0.35639089345932007, 0.5187574028968811, 2.19622802734375, -0.06635672599077225, 0.5476902723312378, 0.340038925409317, -0.5830981731414795, 1.0764901638031006, -1.7400031089782715, 0.8711163401603699, -1.4306490421295166, 1.0535297393798828, -2.6047041416168213, -0.35327714681625366, -1.8737859725952148, 0.7248633503913879, -0.14167693257331848, -0.8543916344642639, 0.23302412033081055, -0.21640722453594208, -0.1264437735080719, -1.3399156332015991, -2.0874507427215576, -0.9992192983627319, -0.3292447626590729, 0.9223368763923645, -1.6206693649291992, -1.0989723205566406, -0.5938538312911987, -0.05314469709992409, -0.6381140947341919, 0.11583895236253738, 0.41095462441444397, -1.2486705780029297, 2.150120258331299, 0.9891096353530884, -0.1735510528087616, 1.2205877304077148, 1.1210650205612183, -1.1157487630844116, -0.47552186250686646, -0.17551371455192566, 0.42321786284446716, 2.412975549697876, -0.7153686881065369, 0.15398584306240082, -0.8550451397895813, 1.4723807573318481, 0.37558305263519287, -0.7027856111526489, 0.24572554230690002, -1.2802969217300415, 0.6717327833175659, -1.491260290145874, 0.4199337959289551, 0.11373291164636612, -0.8985556960105896, -1.0725525617599487, -1.6061434745788574, -0.3336443305015564, 1.0537593364715576, 0.7894800305366516, 0.36773955821990967, 0.48293694853782654, 1.1201860904693604, -1.2224981784820557, -0.44262418150901794, 0.7347147464752197, 0.04662879556417465, -0.6046207547187805, -1.372715950012207, 1.1033737659454346, 0.8206769824028015, 0.9756514430046082, -0.7486364841461182, 0.02275330014526844, 0.693188488483429, 0.2485319823026657, -0.0418497696518898, -1.112065076828003, -0.7117596864700317, 0.8377955555915833, -0.5100241303443909, 0.09852256625890732, -1.1994481086730957, 0.7236155271530151, -0.8245497941970825], [0.3641033172607422, 0.010597245767712593, -1.9756497144699097, 0.23807717859745026, 0.36424097418785095, 1.0245087146759033, -0.2020890712738037, 0.8820926547050476, 1.7317869663238525, -0.8117756843566895, 0.6554638743400574, 0.6641367077827454, -0.455070436000824, 0.04832882434129715, -2.86242413520813, 2.1964941024780273, -1.525532841682434, -0.3957401514053345, 0.4386897385120392, -0.29193028807640076, 0.30939003825187683, -1.0373783111572266, 0.643392026424408, -1.319624662399292, 0.6554442048072815, 0.394737184047699, -1.5733258724212646, -0.12418869882822037, -0.7673956751823425, -0.4356054365634918, 1.3126622438430786, 0.2768844962120056, -0.8091278076171875, -0.9275089502334595, -0.39954283833503723, -0.27766153216362, 0.0011304051149636507, -1.4567103385925293, -0.6807993054389954, 1.2517958879470825, -0.7395105957984924, 1.6177293062210083, 0.7597644329071045, -0.2806382477283478, -1.167060375213623, -0.8738104701042175, -0.4369729459285736, -0.26982343196868896, 0.3811528980731964, 0.07936671376228333, -1.8591629266738892, 0.5431745052337646, 1.0800350904464722, 0.1799582839012146, -1.4784172773361206, 1.3513721227645874, -0.13941194117069244, -0.8979313373565674, -1.010000228881836, 1.9028255939483643, -0.4110361933708191, -0.693152666091919, 1.023504376411438, -0.5166498422622681, 0.03559460490942001, 0.24705097079277039, 0.8490186929702759, -1.5181586742401123, -1.7301400899887085, 1.2012370824813843, 0.6034927368164062, 1.0809059143066406, 0.9051631689071655, 0.6613221168518066, -0.7612059712409973, -0.32391756772994995, 0.8424809575080872, 0.10844424366950989, -0.6677517294883728, 0.9731329083442688, -0.1664103865623474, 0.6322799324989319, 0.16479314863681793, 0.6129941344261169, -0.6351487040519714, -0.07524191588163376, 0.06655588001012802, -0.04535561054944992, -0.33700886368751526, 1.4556498527526855, 0.605566143989563, -0.3904114067554474, -1.0653033256530762, -0.5093501806259155, -1.113752841949463, -0.3714632987976074, -1.0358302593231201, -0.3986653983592987, -0.26644468307495117, -1.5047990083694458, 1.0028713941574097, 1.8706475496292114, 0.9836809635162354, 0.7581520080566406, -0.15621230006217957, -0.8673334717750549, 1.9047561883926392, -0.7359107136726379, 0.033756863325834274, 1.2690938711166382, -0.5339707136154175, -0.10506486892700195, -0.9082266688346863, -0.3928842544555664, 0.10322105139493942, 0.08679665625095367, -0.27540168166160583, -0.6344494819641113, 0.8996807932853699, 2.4271023273468018, 0.6273661255836487, 0.15049126744270325, -1.3341901302337646, 0.19284935295581818, -0.3108310103416443, -0.1848604381084442, -0.24059125781059265, 1.493866205215454, 0.10824219882488251], [-0.7756990790367126, -0.8448035717010498, 0.6031842231750488, -0.16702094674110413, -2.4176645278930664, -1.01821768283844, -0.40245726704597473, 0.9244337677955627, -1.2164056301116943, 0.024264372885227203, 0.3032274544239044, -1.2220481634140015, -0.1541105955839157, 0.895133376121521, -0.2944834232330322, -0.6468339562416077, 0.7670952081680298, -1.1987683773040771, 0.28766870498657227, -0.35243862867355347, -0.5488569140434265, 0.08756474405527115, -2.29860258102417, -0.37739303708076477, -1.3861254453659058, 0.5812000632286072, 0.07334612309932709, -1.1131737232208252, 0.7894102931022644, -1.3112578392028809, 0.3375118672847748, -1.777712106704712, -1.2884231805801392, 0.18640878796577454, 0.24287784099578857, 1.067291021347046, -0.37980353832244873, 1.0590672492980957, 1.2044264078140259, 1.3747352361679077, 0.7126687169075012, -0.34705886244773865, 1.5469545125961304, -0.20607352256774902, -0.8296166658401489, -1.263025164604187, 0.7039697766304016, -0.10860840231180191, -1.079628825187683, -0.018817009404301643, 0.7959873080253601, -0.6881794929504395, -1.0733184814453125, 0.3944265842437744, 0.21778103709220886, 0.7043566107749939, -0.3452870547771454, -0.8527590036392212, -2.352750778198242, -0.4896426200866699, -0.28408172726631165, -0.08425935357809067, -1.4887837171554565, 0.8417739868164062, 0.17811179161071777, 0.3647274374961853, 1.1493273973464966, 0.922884464263916, 1.0516142845153809, 0.47189420461654663, 2.249979257583618, -0.7097615003585815, -2.1254944801330566, -1.1994457244873047, -0.6534547209739685, -0.29302603006362915, 0.5624122619628906, 0.5796418786048889, -0.9317935109138489, -2.226036548614502, -0.8228689432144165, 0.9063061475753784, -0.22966769337654114, 2.4678919315338135, 1.0029233694076538, -0.9168636202812195, 0.9277325868606567, 1.7248915433883667, -2.0912601947784424, 0.0058846729807555676, 0.5564672946929932, 0.4996477961540222, 2.437263250350952, -0.2615969777107239, -0.5041703581809998, -0.7813504934310913, -1.4100171327590942, 1.5584380626678467, -0.18526993691921234, -0.18549491465091705, -0.26365160942077637, -0.725963830947876, -0.38384854793548584, -1.3764780759811401, -0.8547384142875671, 1.1518020629882812, -0.39645278453826904, -0.24077551066875458, 1.0568163394927979, 0.05119423195719719, 0.24102060496807098, -0.21372747421264648, 0.735503077507019, -1.844865083694458, -0.5042736530303955, -0.0853646919131279, 0.270345002412796, -1.0451253652572632, -1.5337204933166504, -0.4192543029785156, 0.9625849723815918, 1.1017111539840698, 0.5130822062492371, 0.40931031107902527, 0.613263726234436, -0.7737702131271362, 1.060794711112976, 0.010547099635004997, -0.9743335843086243], [0.5113104581832886, -1.0758541822433472, 0.20270827412605286, -0.28394895792007446, 0.5990244746208191, 0.22947879135608673, 2.105405330657959, 1.0053720474243164, 0.09271226823329926, 1.1644628047943115, 2.7772886753082275, -0.6938964128494263, 0.846844494342804, -0.059791821986436844, 1.1930557489395142, -0.15717445313930511, -0.308298796415329, -0.29054567217826843, 0.7726612687110901, -1.0098322629928589, 1.5787194967269897, 2.1698834896087646, -1.250766396522522, 0.7347231507301331, -2.179205894470215, -0.1829873025417328, -0.9365143179893494, -0.5250042080879211, 0.16727952659130096, 0.6288283467292786, -0.18671785295009613, 2.194279670715332, 0.7763646245002747, -0.7292286157608032, -0.43640026450157166, 1.8601239919662476, -0.4598907232284546, 2.001193046569824, -0.3034053444862366, -0.48256921768188477, 0.826769232749939, 0.5022803544998169, -1.0637388229370117, 0.9667902588844299, -0.9101737141609192, -1.0607306957244873, 0.6015110015869141, 0.5565662384033203, 1.3668349981307983, 0.059280090034008026, -0.7030317187309265, 1.2925697565078735, -0.48936715722084045, 0.4152962267398834, -1.1162669658660889, 1.4697614908218384, 0.9600002765655518, 1.5527695417404175, -0.046030815690755844, 2.3133962154388428, 0.047245994210243225, 0.06645778566598892, -1.5607253313064575, -0.19770361483097076, -0.27443641424179077, -0.04676065221428871, -1.007155418395996, 0.4657544195652008, -0.6253960728645325, 0.10302642732858658, 0.44100478291511536, 1.2370434999465942, 0.793663501739502, -0.40273404121398926, -2.272122383117676, -0.3563046455383301, 0.34867316484451294, 0.4761635363101959, -3.088967800140381, -1.4536975622177124, 2.1039109230041504, -0.7440119981765747, -0.3471963703632355, -1.5617800951004028, 0.0042395745404064655, 0.7597466707229614, 0.537620484828949, -0.685964822769165, -0.7007216811180115, -0.5426539778709412, -2.1466426849365234, 0.33645716309547424, -1.1502537727355957, -0.9813870787620544, 1.8976171016693115, -1.2209534645080566, -0.44000083208084106, 1.1288719177246094, -0.9903985261917114, -0.8915756344795227, 0.780422031879425, -0.029680607840418816, 0.6486783027648926, 0.3362521529197693, 0.2894599735736847, -0.48985204100608826, -1.26759934425354, 2.168868064880371, -0.6850863099098206, 0.8831782937049866, -0.00172293814830482, 0.5454118251800537, -0.451127290725708, -0.7781935334205627, -0.3619453012943268, 1.6414110660552979, -1.722844123840332, 0.6674596071243286, 1.1973925828933716, 0.38875922560691833, -0.32596728205680847, -1.1645760536193848, 0.8632925748825073, 1.1161466836929321, 2.0672361850738525, 0.6590275168418884, -0.4164782166481018, 0.6738396286964417, -1.2897603511810303], [0.9378952980041504, -1.718343734741211, -0.7305967807769775, -1.6692687273025513, 0.5251760482788086, 0.011074344627559185, -0.4771127998828888, 0.1598357856273651, -0.8784878849983215, -0.2611062228679657, -0.5820494890213013, 0.04849164932966232, 1.6172996759414673, 0.797126829624176, 0.19563081860542297, 0.6614612340927124, -1.3318418264389038, 0.02732153795659542, 0.6368532180786133, 2.570056676864624, -0.20953285694122314, -0.8558359146118164, 1.7603604793548584, -0.9534746408462524, 1.2360283136367798, 2.2189714908599854, -1.7680364847183228, -0.7362157702445984, 0.3960530161857605, -0.8054672479629517, 1.1949115991592407, -2.2651455402374268, -0.39699816703796387, -0.5265328288078308, 0.8376456499099731, -0.24771225452423096, -0.9732362031936646, 0.5031866431236267, 0.39646726846694946, -1.1419098377227783, 1.2770648002624512, 0.44082704186439514, 1.415165662765503, -1.2718690633773804, 0.038019269704818726, 0.26752418279647827, -0.2650526762008667, -0.8259651064872742, 0.10718990117311478, -0.5735458135604858, 0.45909881591796875, -0.36262449622154236, 0.725945234298706, -0.01960163004696369, -0.14586284756660461, 0.28460758924484253, -0.5563557744026184, -0.3965902626514435, 0.46385717391967773, -0.33235687017440796, 0.39869973063468933, -1.2206921577453613, 0.21491087973117828, 1.0107754468917847, -0.7655577659606934, 0.7800498604774475, -0.14357005059719086, -0.8235102891921997, 0.6360308527946472, -0.06252672523260117, 0.18951532244682312, 0.8295208215713501, -0.3402208983898163, 0.7366517782211304, -1.1343568563461304, -1.3584415912628174, -0.5365899801254272, -0.11745833605527878, 0.3104865849018097, 1.2575559616088867, 0.713982105255127, -0.861181914806366, 0.6146439909934998, 1.0628514289855957, -1.3719656467437744, 0.7128432989120483, -2.033329725265503, 0.5774523615837097, 0.24770401418209076, 0.2710789144039154, 0.5764342546463013, 0.49087652564048767, -0.16520893573760986, 0.8619926571846008, 0.19395604729652405, -1.1042007207870483, -1.0883214473724365, -0.5311020612716675, 0.40455353260040283, -0.7954175472259521, -1.9161851406097412, -0.19298772513866425, 0.165462926030159, -0.15810149908065796, -0.2074894905090332, -1.0675381422042847, 0.018735835328698158, -0.20446889102458954, 0.02615736983716488, -1.3808263540267944, -0.8772203326225281, 0.23576124012470245, 1.2450710535049438, -0.7702669501304626, -0.5325933694839478, -0.8164888024330139, -0.9754927754402161, -0.44701087474823, -0.03182214871048927, -1.0570513010025024, -2.640688180923462, 0.6150787472724915, -0.4243307113647461, 1.132340431213379, 0.8876211643218994, -0.8449833989143372, -1.9991379976272583, -1.1935160160064697, -0.3751388490200043], [-0.21648256480693817, 0.8860287666320801, -0.5178627967834473, -1.4754102230072021, -0.02498256042599678, 1.2971065044403076, 1.5574936866760254, -2.0544302463531494, -0.8780205845832825, -1.7222355604171753, -0.6929333209991455, -0.849071204662323, 0.39027139544487, -0.48122426867485046, 0.23509061336517334, -1.2616403102874756, 0.008927171118557453, 0.9871512055397034, -2.118055582046509, -1.3764458894729614, -1.1791476011276245, 2.0295941829681396, -1.2908536195755005, 1.0644748210906982, 1.1620852947235107, -0.6735954284667969, 0.5451997518539429, -1.3059474229812622, 0.27771493792533875, 1.956383466720581, -0.5668309330940247, -0.7942017912864685, -0.9333701729774475, -1.2442373037338257, 0.5318173766136169, -0.2768194377422333, 0.032050758600234985, -1.8341535329818726, -0.22934682667255402, 1.4975428581237793, 1.6015255451202393, 1.7868667840957642, -2.7321653366088867, 0.5163029432296753, 0.6971758008003235, -0.5177052617073059, -0.4097447097301483, 1.2722399234771729, 0.6251712441444397, 1.0410361289978027, 3.369539499282837, -0.005156679078936577, 1.0828313827514648, -0.18302951753139496, 0.06092362850904465, 0.16642776131629944, -0.8973365426063538, -0.030337797477841377, 1.3135727643966675, -1.9196406602859497, 1.0138555765151978, 1.5452638864517212, -0.919420599937439, -0.9476791620254517, 1.3584330081939697, 0.5338427424430847, 0.6942213177680969, 0.7403649091720581, 0.2687709629535675, -0.23562295734882355, -2.137439250946045, -1.3622426986694336, 1.3596946001052856, -0.9315966367721558, 0.2781916558742523, -0.4188154637813568, -0.1501527726650238, 2.5464937686920166, 0.9723564982414246, -0.43206587433815, -0.39052698016166687, 0.30194491147994995, -0.721964418888092, 0.637566089630127, 0.41743335127830505, 0.4558236002922058, 1.7667940855026245, -0.9010972380638123, 0.6752824783325195, 0.2309984266757965, 0.7633285522460938, 1.594297170639038, 1.6543036699295044, 0.46543294191360474, 0.23482400178909302, -0.7015568017959595, -1.0181812047958374, 0.786859929561615, 0.7730100154876709, 0.2486618459224701, -0.7468100786209106, -0.09960488229990005, -0.17136132717132568, -0.6020925045013428, -0.45583343505859375, 1.5082827806472778, 0.9076700806617737, 0.03368367254734039, -0.16440415382385254, -2.2657833099365234, -1.35892653465271, -0.060467712581157684, -0.14903998374938965, -1.8796027898788452, 0.4066850543022156, -1.1737476587295532, 0.34751805663108826, 0.8319811224937439, -0.8080292344093323, -0.22347408533096313, -0.21087411046028137, 0.2968914806842804, -0.21169131994247437, 1.5665169954299927, 0.41885092854499817, 1.1464945077896118, -0.03978593647480011, 1.8725107908248901, -0.6266175508499146], [-1.3347913026809692, -0.22902758419513702, -1.448400855064392, 0.29668128490448, 0.134523406624794, 1.2386351823806763, 1.4211982488632202, 0.760991096496582, 1.0665937662124634, 1.2826684713363647, -1.2780262231826782, 0.21161313354969025, 0.8008008003234863, 1.3423092365264893, 1.3489021062850952, 0.35240933299064636, 0.4050169289112091, -0.39158764481544495, 0.4189799726009369, -2.5863447189331055, 1.0427123308181763, 0.9274866580963135, 0.21441617608070374, 0.6920269727706909, -0.6849734783172607, -0.9896640181541443, 0.11436797678470612, 0.3390914499759674, 1.2324036359786987, 0.7939062118530273, 1.2924686670303345, -1.2852212190628052, -0.5933924913406372, 0.1467900425195694, 1.4051605463027954, -1.6632143259048462, 1.0188274383544922, -0.302071750164032, 0.8282256126403809, 0.8245918154716492, 0.8371545076370239, -0.018715914338827133, -2.2945427894592285, 0.8223139047622681, 0.23238646984100342, -1.0444765090942383, 0.3267621695995331, -0.5298077464103699, 0.35278892517089844, -0.3801307678222656, 1.7206684350967407, -0.6414030194282532, -0.4329898953437805, -0.5499477982521057, 0.9604586362838745, 0.23540084064006805, 0.05817996710538864, -2.2629692554473877, 1.0590108633041382, 1.1571153402328491, 1.0235623121261597, 0.20787166059017181, 1.0747184753417969, -1.3599797487258911, -0.5015006065368652, -0.524878740310669, -0.1805247962474823, -1.3974814414978027, -0.6288034319877625, -1.6028640270233154, 0.09723646938800812, -1.4876700639724731, -0.3977145850658417, -2.0419418811798096, -1.8720847368240356, 0.12045455724000931, -0.25440219044685364, 0.961821436882019, -0.6899272203445435, 1.1490123271942139, -1.3219093084335327, -1.3220988512039185, -0.38530081510543823, 0.3040871024131775, 0.5857328772544861, -0.2600557208061218, -1.3328911066055298, -0.974478006362915, -0.4708721935749054, -1.6789501905441284, 1.6979165077209473, 0.545220136642456, 1.187355875968933, 0.11269404739141464, -1.0217750072479248, 1.1200511455535889, -1.8588001728057861, 1.303062081336975, -1.4883087873458862, -0.8303417563438416, -0.8828848004341125, 0.6209835410118103, -0.0640481561422348, -0.46836698055267334, 0.6737048625946045, -0.09994887560606003, -1.185011386871338, -0.6248700618743896, 1.8540210723876953, -0.20363858342170715, 1.6603240966796875, -1.1090621948242188, -1.7320501804351807, 0.5123737454414368, -2.0457940101623535, -0.05943237617611885, -1.7468328475952148, 0.9496595859527588, -0.3978486955165863, -0.1675245314836502, -1.164910912513733, -0.3766927421092987, 1.6885879039764404, -0.6632122993469238, -0.7581548690795898, 0.023315375670790672, 1.0544151067733765, 0.8671567440032959, -0.8577415347099304], [-0.44203275442123413, 0.2740977704524994, -0.16155663132667542, 1.5738699436187744, -1.2490975856781006, -0.2355337142944336, 0.837602972984314, -0.4109943211078644, -1.0104621648788452, -0.9885052442550659, -0.12375535070896149, -0.11314506083726883, 0.2600240707397461, -1.0801072120666504, -0.7770992517471313, -1.093972086906433, 0.47216570377349854, -0.3206501007080078, 0.4205666780471802, -1.9200552701950073, -0.8910424113273621, 0.4666270315647125, -0.865298330783844, 2.408971071243286, -0.575612485408783, 1.3167775869369507, 0.4508991837501526, 0.13351285457611084, 0.2744389474391937, 0.23266957700252533, -1.6623286008834839, 1.1561555862426758, 0.5943536758422852, 0.8953395485877991, 0.3516533076763153, 0.7069717645645142, -0.824931800365448, -0.5898522734642029, 1.1428980827331543, 0.0812099352478981, -0.5875304341316223, -0.18300576508045197, 1.5322120189666748, -0.41862043738365173, -0.14067769050598145, 0.5571823120117188, 1.6273878812789917, 0.03372468799352646, 0.8168613910675049, -0.5401397347450256, 0.11693830043077469, -0.6214708685874939, 0.06458602100610733, 0.5518240928649902, 0.31468138098716736, -1.1408448219299316, -0.20019765198230743, 1.5565125942230225, 0.06907427310943604, 0.1268475502729416, 0.5049179792404175, 0.6048887372016907, 1.4742980003356934, -1.3335790634155273, -0.3731978237628937, -1.4312081336975098, -1.5282787084579468, -0.9706324338912964, 0.8579009771347046, -0.6879650354385376, 0.9393885135650635, 0.3328741788864136, -0.33133232593536377, 0.38965967297554016, 1.9833576679229736, -1.5784121751785278, -1.282109022140503, -0.3094446659088135, 0.3206792175769806, -0.17306411266326904, 0.07880187779664993, 0.811384379863739, 0.2502872347831726, -0.1603604406118393, 0.9250462651252747, -1.3060598373413086, -1.3197181224822998, -1.2267587184906006, -0.7407790422439575, -1.1136360168457031, -0.8779355883598328, -0.6015846133232117, 0.9982670545578003, -0.15912359952926636, -0.20054154098033905, 1.8060429096221924, 0.07902669906616211, -0.9848132133483887, 0.3063688576221466, -0.16428707540035248, 0.21887056529521942, -1.1679415702819824, 0.6210179328918457, 0.14565494656562805, 0.4632701277732849, -0.5384703278541565, -1.5460550785064697, 1.7165942192077637, 0.40405550599098206, 2.4139444828033447, 0.5220423936843872, -0.9553210139274597, -0.07209956645965576, -1.5892295837402344, 0.5000311136245728, -0.6552333831787109, -1.3320621252059937, -0.9956970810890198, -1.4133740663528442, -0.39749008417129517, 0.6223528981208801, -1.043326735496521, 0.4903707802295685, -1.0793417692184448, -0.6788761019706726, 0.3768748939037323, -0.8943644762039185, -1.664811372756958, 1.3196189403533936], [-0.8908666372299194, 0.8334935307502747, -0.14319634437561035, 0.0313296765089035, -0.1285371482372284, -1.6592731475830078, -1.5054936408996582, -0.04366081953048706, 0.27027788758277893, 0.6298852562904358, -0.3199462890625, 0.7681596279144287, -0.27380213141441345, 0.813596248626709, 0.6940513253211975, -0.14448630809783936, -0.9612019658088684, 0.9597113132476807, -2.3389947414398193, 2.223510265350342, 0.12436860054731369, 1.2495938539505005, 0.13425932824611664, 0.23148579895496368, 0.15512244403362274, 2.5678796768188477, -0.04966188967227936, -1.096866488456726, -0.34606850147247314, -1.5248738527297974, 1.2313849925994873, -0.87608402967453, 0.7285830974578857, -1.040273666381836, 0.6980975270271301, -1.1348127126693726, 1.9352868795394897, -1.1874961853027344, -0.32357603311538696, 0.3860568106174469, -0.4969497323036194, -0.5298472046852112, -1.0671931505203247, 0.13321469724178314, -0.9732123613357544, -0.27314314246177673, 1.602325439453125, 0.2823811173439026, -0.5513203144073486, 0.9874135255813599, -0.07989104837179184, -0.6938154697418213, -0.9023117423057556, -0.9422314167022705, -1.4142978191375732, -0.012692458927631378, -0.9190574288368225, -0.3734397292137146, -1.1335431337356567, 0.10597329586744308, -0.3487977087497711, 0.7458041310310364, -0.27811136841773987, 0.1721901148557663, -0.8115156292915344, 0.17719070613384247, -0.19119174778461456, 0.26710712909698486, -1.5987504720687866, 1.0014728307724, 0.5618998408317566, 1.712691068649292, -0.40717580914497375, -0.4530966579914093, -0.007224082946777344, -0.01611190289258957, -1.1450726985931396, 0.867068886756897, 0.25869932770729065, -1.635768175125122, 1.3422530889511108, -2.1392509937286377, -0.8190606832504272, 0.031231768429279327, 0.3996577560901642, 0.11803845316171646, 1.3760713338851929, 1.3203705549240112, 0.7734300494194031, -0.10915976017713547, 0.3085520267486572, -1.082617998123169, 0.1672808676958084, -0.2781660854816437, 1.1466881036758423, -0.20672237873077393, -1.484506607055664, -0.7586007714271545, -0.15865974128246307, -0.9964471459388733, -1.0580414533615112, -0.7879766225814819, -0.40574365854263306, -0.03454837575554848, -2.855259656906128, -1.4401133060455322, -1.1725536584854126, 0.9041735529899597, 0.09620890021324158, 0.01446156669408083, 0.4482934772968292, -0.24408197402954102, 1.1886496543884277, -1.7838547229766846, -0.7586386799812317, 0.30695149302482605, 0.6647816896438599, 1.420485258102417, -1.2588878870010376, -0.04082904011011124, 0.09461762756109238, 1.0483112335205078, -0.7866472601890564, 0.9524322748184204, -1.3178069591522217, 0.5102176666259766, 1.5277700424194336, 0.5120013356208801, -0.8295901417732239], [1.1910889148712158, 1.2826800346374512, 0.22370101511478424, -0.851373553276062, -0.8848745226860046, 0.3181673288345337, 1.9236234426498413, -0.8042078614234924, 3.7765097618103027, -2.4502315521240234, 0.7574139833450317, 0.1284530609846115, -0.12613657116889954, -0.410646915435791, 0.007874472066760063, 0.7883909344673157, 0.4873177707195282, -0.9948315620422363, 0.38971713185310364, -0.31294310092926025, -0.7885700464248657, -0.4769326150417328, -1.1643552780151367, 1.071871280670166, 0.7843443155288696, 1.2256301641464233, 0.9613505005836487, -1.631787896156311, -1.5480163097381592, 0.33111509680747986, -1.4758143424987793, 1.835154414176941, 0.790901243686676, -0.19132336974143982, -0.13635151088237762, -1.0716428756713867, -0.6941274404525757, -1.3879033327102661, 0.05785854533314705, -1.7358875274658203, 0.3296876847743988, -0.3587854206562042, -1.3745871782302856, -1.680990219116211, 1.950546383857727, -0.7529940605163574, 0.028343824669718742, -0.33931154012680054, -0.45874398946762085, 0.5382031202316284, -2.7392983436584473, -0.8453114032745361, 1.108115315437317, 0.26503926515579224, 0.1433831751346588, 0.7095028758049011, 1.118424654006958, 1.2122200727462769, -0.9155699610710144, -0.25126996636390686, 0.2888427972793579, -1.4773765802383423, 1.0461578369140625, 1.201690912246704, -0.026029543951153755, 2.558676242828369, 0.368888795375824, -0.6016846299171448, -0.39264848828315735, 0.054468557238578796, 1.3395506143569946, -0.6528700590133667, -0.2725372612476349, -0.7374394536018372, 2.0148932933807373, -1.9967645406723022, -0.14213502407073975, 0.8061572909355164, 1.013770341873169, -0.04550401121377945, -1.4424959421157837, -0.7306077480316162, 1.4150031805038452, 1.8885456323623657, 1.0343979597091675, -0.03657173737883568, -0.34419721364974976, 0.9125238656997681, 1.6835238933563232, -0.7601290345191956, -1.2815241813659668, 0.6737791895866394, -0.1504317969083786, 0.47153300046920776, -1.2018014192581177, -0.3559841513633728, -0.5010002851486206, 0.7753605246543884, 0.20806904137134552, 0.9642354846000671, 0.3436431884765625, 0.933549702167511, -1.0341198444366455, 0.11219216883182526, -0.0314086489379406, 1.3149932622909546, -0.5578475594520569, -0.07527291029691696, -0.4933777153491974, 1.1735163927078247, 0.8130842447280884, 0.9225895404815674, -1.116676688194275, 1.7474857568740845, 0.2284812480211258, -0.5576037764549255, -1.1564396619796753, -0.926199197769165, 0.8763814568519592, 0.029230717569589615, 0.16902264952659607, -2.429009199142456, -2.3050031661987305, -0.21297717094421387, -0.5571937561035156, 1.0775549411773682, -0.1074017584323883, 0.7282716035842896, -1.0828375816345215], [-0.4640076160430908, -0.24282103776931763, -0.24265947937965393, -0.06629214435815811, -1.5979485511779785, -0.9356375932693481, 0.7662397027015686, 1.2103899717330933, -1.5241968631744385, 0.29585590958595276, 0.41130530834198, -0.3723626732826233, 0.7195874452590942, 0.21079415082931519, 1.3191219568252563, 0.7050712704658508, 1.0087223052978516, -0.31249746680259705, 0.8025529980659485, -1.125283122062683, 0.5149679183959961, 0.09159670770168304, -1.448418140411377, 0.23344366252422333, -0.04080396145582199, -0.5930189490318298, 0.6184529662132263, 1.2651302814483643, -1.2535361051559448, -0.2695818543434143, 2.2788589000701904, 0.5460035800933838, -1.742394208908081, 1.0353575944900513, -1.977157711982727, 0.5647792816162109, -1.8879892826080322, -1.1612125635147095, -1.2950668334960938, -1.2469123601913452, -0.4594918489456177, -0.4411410093307495, -0.7617698907852173, -0.1216101422905922, 1.1372003555297852, -0.506933331489563, -0.30596160888671875, -0.1783217191696167, -0.8191019296646118, 0.2988608479499817, 0.5655121803283691, -1.054450511932373, -1.113905668258667, 1.878242015838623, -1.2913461923599243, -0.38447558879852295, 0.028859101235866547, 0.7787418365478516, 0.7927747368812561, -0.43306005001068115, -0.6793796420097351, 0.9576550722122192, 1.2341550588607788, -0.14464104175567627, -0.7646217942237854, 0.09682811051607132, -1.3990674018859863, -0.9976435303688049, -1.5453206300735474, -0.31134045124053955, -0.3533061444759369, 0.2993183434009552, -1.2635657787322998, -0.7247207164764404, 0.9237457513809204, -0.4369121789932251, -0.4398553669452667, -0.09123989939689636, 0.9131602644920349, -0.27438080310821533, 0.9493460059165955, 0.24961666762828827, 1.5064886808395386, -0.6558594703674316, -0.3785956799983978, -0.17168821394443512, -2.634890079498291, -0.5903031826019287, 1.3786784410476685, -0.4800129234790802, 0.9213895201683044, -1.3546241521835327, 0.9086375832557678, -2.3198537826538086, -0.8349888324737549, 0.28911808133125305, 0.3056206703186035, 0.43784523010253906, 1.06021249294281, 2.806053400039673, -0.7142930626869202, -1.3163256645202637, 0.3183175027370453, -0.32899683713912964, -0.688906729221344, 0.8266305327415466, 1.4937509298324585, -3.2071993350982666, 0.8361139297485352, 0.15852782130241394, -1.5950559377670288, -0.5481371283531189, -0.5558170676231384, 0.008058437146246433, 0.42850974202156067, -0.5234707593917847, -0.8264172077178955, 1.4145857095718384, -0.8210227489471436, -0.29507482051849365, 0.5537291169166565, 0.0005715755978599191, 0.40179240703582764, 1.1154780387878418, -0.9082596302032471, 0.5371993780136108, -0.9023808240890503, 0.5122771859169006, 0.309720516204834], [-1.9435656070709229, -1.8133294582366943, -0.07916545867919922, 2.9565482139587402, -1.123748779296875, 1.8273910284042358, -0.6085255742073059, -0.07291268557310104, 3.6583597660064697, 0.5485734939575195, 0.48293107748031616, 0.8494755029678345, 1.514146327972412, 0.29904404282569885, -0.38183075189590454, -0.48456811904907227, 0.008347006514668465, 2.094688653945923, 0.31077301502227783, 0.2782786190509796, -0.9785348176956177, 1.491845726966858, -0.18321718275547028, 0.7132734656333923, 0.495742529630661, 1.4645174741744995, -0.25914713740348816, 0.16433903574943542, -0.5266065001487732, -0.3480532169342041, 0.6152828931808472, -0.4000168442726135, 0.6394259333610535, -0.6007261276245117, -0.8041155338287354, -0.17814776301383972, -0.46096307039260864, -2.587083578109741, -0.7140486240386963, 0.6931647658348083, -1.3405956029891968, -0.3855149447917938, 0.5896543264389038, -0.013150694780051708, 0.010406939312815666, -0.3875463306903839, -0.010388343594968319, -0.2844156324863434, 0.6455031633377075, 0.1541529893875122, 0.19881397485733032, -0.8620439767837524, 0.4229378402233124, -0.3533715307712555, -0.7910227179527283, 0.05373780429363251, -0.15648040175437927, -0.36479493975639343, 0.5557567477226257, -1.6937816143035889, -1.2556012868881226, 0.9813887476921082, 0.16739758849143982, 0.338598370552063, 2.084413766860962, -1.477826714515686, 1.0465459823608398, 0.2116374522447586, -0.3067311644554138, -0.1288435161113739, 0.3263317048549652, 2.055699586868286, -0.27143996953964233, 0.5845887660980225, -1.1283246278762817, 0.4939752221107483, -1.4705848693847656, -0.785065233707428, -1.9095441102981567, 0.11153312027454376, -0.40023916959762573, 1.0959792137145996, -0.17978742718696594, -0.41484180092811584, 2.0585215091705322, 0.9153618216514587, 1.0011264085769653, 0.4896174669265747, 1.5087584257125854, -0.11427757143974304, -0.8103480935096741, 0.6491940021514893, 0.059550344944000244, -1.7626012563705444, -0.41305798292160034, -2.380443811416626, 1.027438998222351, 1.2598631381988525, -0.4605293571949005, 3.254295587539673, -0.09629315137863159, -1.1670993566513062, 0.18888436257839203, 0.45570725202560425, 0.9492228031158447, 1.0388754606246948, -1.5883610248565674, -1.067579984664917, 0.07156500965356827, 1.7075976133346558, -1.336410403251648, 0.38747167587280273, 0.9567731618881226, 0.9270098209381104, 0.18882517516613007, 0.28991231322288513, 0.9563606381416321, 2.101051092147827, 0.6590065956115723, -0.055556293576955795, 0.8538227081298828, 0.828192412853241, 0.8877727389335632, 0.44944724440574646, -0.15243835747241974, -0.3228510022163391, 0.5812546610832214, 2.116316795349121, 1.3724859952926636], [-1.6941893100738525, 1.5819909572601318, -1.330145239830017, -0.18065382540225983, -0.5379631519317627, -0.9661774635314941, -1.0108803510665894, -0.43784424662590027, 2.228454351425171, 2.438225030899048, -1.1339762210845947, 0.4168962836265564, -0.387117862701416, 0.356268048286438, -1.1360788345336914, 0.40612614154815674, -1.6170456409454346, -0.3476305902004242, 0.7029092311859131, -0.9745001196861267, -0.3094698488712311, 0.10852238535881042, 0.39735183119773865, -1.113355040550232, 0.17706002295017242, 1.019692301750183, -1.5796455144882202, 0.21945299208164215, 0.18895253539085388, 1.5176695585250854, 0.7659337520599365, -0.9228442311286926, -0.551749587059021, 0.026230059564113617, 1.4285500049591064, 0.5990523099899292, -1.692463755607605, 0.4012128412723541, -1.2673357725143433, -1.556117057800293, 0.024458566680550575, 1.0081621408462524, 0.4821905493736267, 0.45849356055259705, -1.8544930219650269, 0.15807490050792694, -1.4794011116027832, 1.0527331829071045, -0.6330084204673767, -0.7414613962173462, 0.5241517424583435, -0.09660769253969193, 0.12431533634662628, -0.9691638350486755, -0.6476339101791382, 0.5256085395812988, -1.3805618286132812, -0.6282750964164734, -1.325188398361206, 1.6436678171157837, 0.21251137554645538, -0.9032291769981384, -1.1066526174545288, -0.015540850348770618, 1.343015193939209, -1.3360551595687866, 0.15877391397953033, -0.6182212233543396, -0.44025784730911255, 0.6790499687194824, -0.6969764828681946, -0.018555495887994766, -0.44778814911842346, -1.4233897924423218, 2.1520915031433105, 1.171400785446167, -0.4221230745315552, -1.199404001235962, 0.7667250037193298, -0.01818789355456829, 0.9381715655326843, 0.44862526655197144, -1.5780940055847168, -0.16335898637771606, 0.49659067392349243, 1.8811451196670532, -0.861810564994812, -0.5209451913833618, 0.0051884776912629604, 0.15687444806098938, 1.4255788326263428, 0.3676466941833496, 0.19856570661067963, -0.6851783990859985, 0.9446657299995422, -0.6036548018455505, -0.9146169424057007, 0.04125908389687538, -0.7399322986602783, -1.2321038246154785, 0.0927753821015358, -1.676859736442566, 0.11797674745321274, 0.405575156211853, -1.2211636304855347, 1.3491655588150024, 0.585694432258606, -0.6593946218490601, 0.6515432000160217, -0.7080667018890381, -0.35578593611717224, 0.9981878399848938, -0.9805055260658264, -1.4190672636032104, 0.5614643096923828, 0.35157838463783264, 0.24596422910690308, -0.8508239984512329, -1.176435112953186, -0.05921080335974693, -1.4093701839447021, 2.247270345687866, 0.550711989402771, -1.0682858228683472, 0.6927827000617981, 0.3890322744846344, 1.2895087003707886, 1.2534117698669434, 1.9132964611053467], [0.8987314701080322, -1.0574148893356323, 1.896727442741394, 1.0971359014511108, 1.7232764959335327, -0.2823064625263214, 0.13450393080711365, -1.47767972946167, 0.820510983467102, 1.6690082550048828, 1.3268966674804688, 0.25397682189941406, 0.17853258550167084, 0.07143253087997437, 1.5781241655349731, -0.09610875695943832, -0.01324410829693079, 1.6284295320510864, -0.1344417780637741, -0.38567453622817993, 0.7381542325019836, -0.9615229964256287, -1.8000872135162354, 0.8761816620826721, 0.16647818684577942, 0.6783196926116943, -0.4942697286605835, 1.3637113571166992, -1.0280146598815918, -1.5973143577575684, -0.2764018476009369, -0.3470053970813751, -2.3761796951293945, -0.8631241321563721, -0.2758062779903412, -1.0593411922454834, -0.7985597848892212, -1.621887445449829, 0.8879600167274475, 1.0627280473709106, 0.2385559231042862, -1.6175841093063354, -1.4441828727722168, 0.249095618724823, -0.5697543621063232, 1.1173367500305176, 0.02337617427110672, -0.18081501126289368, -1.1178330183029175, 1.0506277084350586, -1.4355289936065674, 0.6530047655105591, 0.6246466040611267, 0.618055522441864, -1.071408748626709, 0.5592638850212097, 0.792298436164856, -1.5996273756027222, 0.27997976541519165, 0.22524715960025787, -0.6833988428115845, 0.8445708155632019, 1.1574015617370605, 0.4479081928730011, 1.1167486906051636, 2.0272340774536133, -0.5678086280822754, -1.5911260843276978, 0.4098900258541107, 0.45242786407470703, -1.5789196491241455, 0.5829446315765381, -0.016642218455672264, 0.4524872303009033, 0.4873695373535156, -2.4116666316986084, -0.6756642460823059, 0.15650701522827148, 0.798678994178772, 0.32931843400001526, -0.05713596194982529, 1.1042858362197876, -0.8443951606750488, 0.3162171542644501, 1.2247289419174194, -0.567793071269989, -0.27916309237480164, 1.0399917364120483, 1.2946670055389404, 0.18750043213367462, -0.8523589372634888, 0.22120797634124756, 1.0822983980178833, -1.3433499336242676, -0.008858200162649155, 0.0833471417427063, 0.1517563760280609, -0.9495967030525208, 0.24590829014778137, -1.1552557945251465, 0.8351772427558899, -2.0119709968566895, -0.2085823267698288, 1.9199506044387817, -0.7459697127342224, -1.6834776401519775, 1.0295666456222534, 0.8383455276489258, -0.3516523241996765, 0.11956320703029633, -1.2112758159637451, -1.350297212600708, -0.035780783742666245, 0.030586091801524162, -2.1368613243103027, 2.1696839332580566, 0.10180927813053131, -0.4439517557621002, 0.5391687750816345, -2.3244571685791016, 2.029280424118042, -0.5883382558822632, -0.7951321005821228, -0.16757851839065552, -1.039852499961853, -0.23273667693138123, -1.0224156379699707, 1.274885892868042, 0.6834645867347717], [0.002603059634566307, 0.9572532773017883, -0.5985902547836304, -0.792614221572876, 0.3042629659175873, 0.07625151425600052, 0.4555363059043884, -0.6154497265815735, 1.599111557006836, -0.7587499022483826, 0.37074288725852966, 0.847550094127655, -1.175315022468567, 1.122664213180542, 0.37982377409935, -0.2605452239513397, -0.15478628873825073, 0.36004844307899475, 0.3365679681301117, 0.5493384003639221, -0.22549864649772644, -0.9751392602920532, -0.3071778416633606, 0.353732705116272, 0.1870575100183487, 0.9058573842048645, -0.4103977680206299, 1.2884033918380737, -2.7872414588928223, -0.4890061914920807, 1.6803256273269653, 0.17053647339344025, -0.7288960218429565, 0.6940268278121948, -0.7188868522644043, 0.20459094643592834, 0.10998160392045975, -1.829594612121582, 1.565893292427063, -1.4098012447357178, -1.2834391593933105, 1.3644521236419678, -1.2230812311172485, -0.4449162781238556, 1.8167665004730225, 0.7242156267166138, -1.0562772750854492, 0.9669324159622192, -0.4524455964565277, 1.3482372760772705, -0.7786086797714233, 0.37777432799339294, -0.29396215081214905, 0.9039871692657471, -0.47594377398490906, -0.4899243116378784, -2.2098145484924316, 0.21150800585746765, -0.09974543005228043, -0.5224553346633911, -0.923626184463501, -0.3238530457019806, 0.5108469128608704, 1.0059311389923096, -0.05533194914460182, 0.0003854910610243678, 0.987250566482544, -0.48825711011886597, 0.22763550281524658, -1.4241828918457031, -0.42147156596183777, 1.4066925048828125, -1.0797772407531738, 1.8841527700424194, 0.27934783697128296, -0.28232836723327637, -0.56255042552948, -0.08248597383499146, -0.781930148601532, -0.7347679734230042, 1.109757661819458, -0.2705588638782501, -0.2537625730037689, -0.41351863741874695, -0.5696500539779663, -1.4707098007202148, -0.9950399994850159, 0.34119725227355957, 0.09756743907928467, 1.287051796913147, -1.4334218502044678, -1.6393524408340454, 1.2719697952270508, 1.1259925365447998, 0.6227192282676697, -2.102155923843384, -0.8201180696487427, 0.11048880219459534, 0.05407145246863365, -0.3171091675758362, 0.8115367889404297, 0.9941580295562744, -1.7753524780273438, 1.3937469720840454, -1.6391223669052124, 1.9142801761627197, -0.390195369720459, -2.032843828201294, -0.4275071918964386, 2.1703872680664062, -0.899190366268158, -0.41436707973480225, -0.081803098320961, 1.5716886520385742, -0.40434888005256653, 1.2907203435897827, 0.0778961256146431, 0.051290955394506454, -0.08620920032262802, -1.7735083103179932, 2.0199530124664307, 0.21536323428153992, 0.6540283560752869, -0.6275267601013184, 1.2634055614471436, 0.4507673978805542, -2.4937665462493896, 0.8316060900688171, -0.3408137261867523], [-0.10606510937213898, -0.5517513155937195, 0.4059339761734009, -0.2720416486263275, 0.43541184067726135, -0.7722874283790588, 1.203061580657959, 0.3886922299861908, 1.6314117908477783, 0.9285082817077637, 0.7610259652137756, -0.997559666633606, 3.2938649654388428, 0.6051352620124817, -1.1155712604522705, -0.6056342720985413, -0.8511351943016052, -0.7650940418243408, 2.0397896766662598, 0.2473340928554535, 0.569325864315033, -0.015897434204816818, -0.3107641637325287, -0.04819899797439575, -2.0046634674072266, -0.043715864419937134, -0.5222532153129578, 1.8781055212020874, 0.7324745655059814, -0.1514725387096405, -0.411307156085968, 1.2052477598190308, 0.7277528643608093, -2.1944308280944824, 2.1379098892211914, 0.632531464099884, 1.2849873304367065, 0.4808521866798401, -0.5919537544250488, -0.11528932303190231, 1.7226532697677612, -0.3734140396118164, -0.22353294491767883, -2.4590141773223877, 2.0257930755615234, 0.5084754824638367, 0.7025536894798279, -2.0955662727355957, 1.8663395643234253, -0.4859965741634369, 0.5712796449661255, -0.9059070348739624, -1.989505648612976, 0.5744410157203674, -0.5603398680686951, -0.15238071978092194, -0.04349035769701004, 1.1346365213394165, 0.08868800848722458, -0.5701842308044434, 0.7987311482429504, 1.6501963138580322, 0.13401134312152863, 0.548490583896637, 0.6889800429344177, -0.6779090166091919, -1.3279361724853516, 1.7602697610855103, -1.9421263933181763, 0.5596095323562622, -0.5994561314582825, 1.7367089986801147, 0.29244402050971985, 0.6197103261947632, -0.24587216973304749, 1.9682250022888184, 0.7476059198379517, 0.679118275642395, -1.0539525747299194, 0.221656933426857, 0.639334499835968, -2.182274103164673, -1.5717254877090454, 0.9266189932823181, 0.12106174975633621, -0.6484276652336121, 0.875742495059967, -1.734544277191162, 0.3194790780544281, 1.21079421043396, 0.14180703461170197, -1.0193455219268799, -0.7110192179679871, -0.12441261857748032, 0.23897096514701843, -1.3550046682357788, -1.2595607042312622, -1.0379153490066528, -0.04772251844406128, 1.181482195854187, 0.9448748230934143, -0.4546983540058136, 0.4688563644886017, -1.208749532699585, -1.2496696710586548, -0.01402596477419138, 0.3617953956127167, 0.8661954402923584, 0.6711782813072205, 1.3787198066711426, -0.4866945743560791, 1.240896463394165, 1.5034289360046387, 0.6164659261703491, -1.2110284566879272, 0.30009251832962036, 0.42516231536865234, 0.5031590461730957, 0.5127696394920349, -0.9321855902671814, -0.08067483454942703, -0.09881874918937683, -1.458174467086792, 0.456694096326828, -0.2275324910879135, 0.6895490288734436, -0.5549414157867432, 0.5704973936080933, -0.07366502285003662], [0.18809190392494202, -0.2232474833726883, 0.4864124357700348, 0.9227615594863892, 0.40425699949264526, -0.4595648944377899, 3.3587169647216797, 0.10110605508089066, 0.22741739451885223, -1.5474753379821777, 0.6659638285636902, 2.4270925521850586, 0.8707227110862732, -0.8371575474739075, 1.666425108909607, -1.3346848487854004, -1.1671278476715088, -1.0301331281661987, 1.739656686782837, -0.07549187541007996, 2.6563169956207275, -0.5159417390823364, 0.44170689582824707, 0.27821046113967896, -1.2027920484542847, 1.620400309562683, 0.05479475110769272, 0.7460775971412659, -0.32325512170791626, -0.7421910762786865, 0.024189407005906105, -2.422384738922119, 0.6809287667274475, -1.1734267473220825, 0.5704973936080933, -0.6859955191612244, 0.9154081344604492, -0.22817716002464294, 1.334672451019287, -0.4174652099609375, 0.1891353875398636, 1.206112027168274, 1.9762517213821411, 0.2809778153896332, -1.8358335494995117, 0.6574546098709106, -0.3244514763355255, -0.1665603220462799, -0.3205552399158478, -1.0070990324020386, 1.3855942487716675, -0.501126766204834, 1.168116569519043, 0.34358322620391846, 1.640204906463623, -0.33874639868736267, -0.21395084261894226, -1.3787803649902344, 0.0721060112118721, -0.26683157682418823, 0.45238304138183594, 0.25125277042388916, -0.17903797328472137, 0.12127175182104111, 1.1055004596710205, 1.820850133895874, 0.3918227255344391, 1.025432825088501, 0.43752652406692505, -0.9849914908409119, 0.051045458763837814, -0.5230258703231812, 0.33008092641830444, 0.24738182127475739, -1.0154300928115845, -0.3885599672794342, -1.527847170829773, 0.8646048307418823, 1.937653660774231, -1.1117558479309082, -0.2690136134624481, 2.2051451206207275, -0.26905423402786255, -1.7513747215270996, 1.3215868473052979, -2.1147749423980713, 0.45575857162475586, 0.10681787878274918, -0.7826032638549805, 1.2230762243270874, -2.44740629196167, -0.30674293637275696, 0.4188193380832672, 0.268426775932312, 0.06955082714557648, 0.851509153842926, -0.13309241831302643, 0.118506520986557, 0.3562828004360199, 0.3708136975765228, 0.23122303187847137, 0.836148202419281, -1.309016227722168, -0.12345859408378601, 0.19086335599422455, -0.8354864716529846, 0.07968831062316895, 0.19335253536701202, -1.599402904510498, 0.16232085227966309, 0.24746859073638916, 0.3511981666088104, -0.03605269640684128, 1.5493322610855103, -0.370726078748703, -1.8354920148849487, 0.16089913249015808, 0.8651376962661743, 0.3448725938796997, -0.5905777215957642, 0.4387141466140747, -3.054769992828369, -0.14779269695281982, -0.5070210695266724, -0.7065929174423218, 1.361052393913269, 0.1991218775510788, -1.1820335388183594, -0.0703955888748169], [-0.040205445140600204, -0.586847186088562, 1.240145206451416, 0.15380829572677612, -2.1618807315826416, 0.4176987111568451, 0.24793782830238342, 0.9543911814689636, 0.026593226939439774, 1.006587266921997, -1.5936851501464844, -0.3507162928581238, 0.0919327512383461, 0.31014102697372437, -0.7953291535377502, 0.0735500231385231, -0.6871293187141418, 0.7933668494224548, 1.6023671627044678, -0.5766992568969727, -0.010159897617995739, -0.5593989491462708, -0.6742822527885437, 0.5583320260047913, 2.613157033920288, -0.6716111302375793, 0.5693368315696716, 0.15618382394313812, -0.0878220722079277, 2.3320071697235107, 0.6040797233581543, 0.6595239043235779, 0.15080974996089935, -0.6964344382286072, 0.2800140976905823, -1.9987562894821167, -0.9823408722877502, 0.783484935760498, -1.4049458503723145, -1.4601519107818604, -0.7794100046157837, -0.42296865582466125, -2.1335160732269287, -0.11816182732582092, 1.3921778202056885, 0.22708792984485626, -1.299951195716858, -0.16487003862857819, 0.9832010269165039, 0.1819043904542923, 1.2332311868667603, -1.3062529563903809, 0.2077547162771225, 1.2191181182861328, 0.6544144153594971, -0.6804026365280151, 0.2807435393333435, 0.5478422045707703, -2.800870418548584, -0.12873542308807373, -0.36424487829208374, -1.9631164073944092, -0.694521963596344, -0.24333073198795319, -1.2830899953842163, 0.2187710553407669, 1.236576795578003, -0.5453557372093201, 0.6036537289619446, -1.0416216850280762, 1.7970646619796753, -1.0291346311569214, 1.0021110773086548, -1.8255022764205933, 0.16702626645565033, 0.0965154841542244, 0.3267395496368408, 0.06199708580970764, 0.5550891757011414, 0.20889274775981903, -0.7607755661010742, 0.18890158832073212, -0.3114921748638153, -0.10992541164159775, 2.029176950454712, -1.73249089717865, 1.1876566410064697, 0.49816983938217163, 1.7418798208236694, 1.1366798877716064, 1.2615840435028076, -1.0922772884368896, 1.3545781373977661, -1.1572984457015991, -0.07909256964921951, -0.29402875900268555, -2.5616815090179443, 0.08648988604545593, 0.6684066653251648, 2.3997914791107178, -0.9456968903541565, 0.6861884593963623, 1.1821141242980957, 0.8662092685699463, -1.3731547594070435, -0.03239208459854126, 0.7690637707710266, -0.2331121861934662, 0.7393026947975159, -1.68612802028656, 0.5085132718086243, -1.018214225769043, 0.6634314060211182, -0.1506127566099167, -0.06232941895723343, -0.6036186814308167, 0.466253399848938, 0.9973196387290955, -0.7941747903823853, 0.24615620076656342, 0.23898108303546906, 0.44931691884994507, -1.0466365814208984, -0.8100181818008423, -0.2461647093296051, -0.5838305950164795, -0.6927222013473511, 0.1475958675146103, 0.5337920784950256], [-0.8433279991149902, -1.1689859628677368, 0.12271549552679062, -1.003979206085205, 1.4899072647094727, 1.0915453433990479, -0.8151384592056274, 0.32745975255966187, 0.2702731490135193, 0.9768017530441284, 0.06576233357191086, 0.23748719692230225, 0.7390519976615906, 0.97887122631073, 0.5365010499954224, 1.6420855522155762, 0.8851467967033386, 1.1283516883850098, -0.6737545728683472, 1.3508814573287964, 1.967374324798584, 0.6166924238204956, 1.0101300477981567, 0.5321665406227112, 2.3437602519989014, -0.4570028483867645, -0.9675626158714294, -0.6413357257843018, -0.746168851852417, -0.3542179465293884, -0.7751529216766357, -0.005573118571192026, 1.7572954893112183, 1.0633567571640015, 1.3554884195327759, 1.8817293643951416, 0.2368706911802292, -1.1966583728790283, 1.3065062761306763, -2.4418768882751465, -1.8387202024459839, 0.2404332160949707, 0.3853217661380768, -0.543178379535675, 0.6600273847579956, 1.175301194190979, 0.19559355080127716, 0.6806382536888123, -0.02212360128760338, 1.7477858066558838, 0.40619176626205444, 0.3638124167919159, -0.4741370677947998, 1.67879319190979, -0.17864863574504852, 1.3795245885849, 2.116589069366455, -1.319407343864441, -0.8177018761634827, 1.061824917793274, -2.6852922439575195, -0.6671327948570251, -0.43548333644866943, 0.27595049142837524, 1.9117878675460815, 0.20542588829994202, 1.6255431175231934, 1.9845216274261475, -0.02152596041560173, -1.2438066005706787, -0.11981471627950668, 2.1127076148986816, 1.03370201587677, 0.7568950057029724, 1.8499493598937988, 0.6608823537826538, -0.2809505760669708, -0.32282817363739014, -0.4095827639102936, -0.13205230236053467, 0.9434077143669128, -0.6201661229133606, 0.30189967155456543, 0.36608555912971497, -0.11273612082004547, 1.7376617193222046, -0.22685423493385315, -0.8625776171684265, 0.9322910904884338, -1.1610021591186523, 0.07106325030326843, 0.23075971007347107, -1.57248854637146, -1.2487397193908691, -2.0336291790008545, -0.13613077998161316, 1.0100014209747314, -1.5150611400604248, -0.2868911623954773, 0.3341403901576996, -0.028920622542500496, -0.9270404577255249, 0.23151053488254547, -1.2040705680847168, -1.1833250522613525, 0.1786334365606308, -0.7459229826927185, 0.5485966205596924, 0.10639633238315582, -1.2161023616790771, 0.4759170711040497, -0.7323862314224243, -0.7312163710594177, 2.833174467086792, -0.17951561510562897, 0.14097070693969727, 1.1135368347167969, 1.0421206951141357, 0.6421560049057007, 1.1802730560302734, -2.6950595378875732, -0.0854453444480896, -0.4363470673561096, 0.4756559729576111, -1.255110740661621, -0.7528639435768127, 0.31859996914863586, 0.057832781225442886, -0.8510614037513733], [0.8489132523536682, 0.7484171986579895, 0.21634188294410706, -0.5766232013702393, -1.9434627294540405, -1.0473233461380005, -1.030714750289917, 1.3044400215148926, -1.7781411409378052, 0.40790855884552, 1.2282658815383911, 1.3300975561141968, -0.0973287895321846, -0.02135527692735195, -0.33547282218933105, 1.2934353351593018, -0.849160373210907, 1.0602835416793823, 0.4766494333744049, -0.051072604954242706, 0.07501933723688126, -0.5863203406333923, 0.2972162961959839, -0.23772987723350525, 0.49811071157455444, -0.4103008806705475, 0.49416035413742065, 1.0065962076187134, 0.9537729620933533, -1.4282357692718506, 0.13889551162719727, -1.806246280670166, -1.3791149854660034, -0.3145342767238617, -0.6901660561561584, 1.321163296699524, 1.7483123540878296, 0.1520201414823532, 0.7186117768287659, 0.5769606232643127, -0.5016481876373291, -0.282741516828537, 1.0114456415176392, -1.7718199491500854, -1.0602248907089233, 0.2102956622838974, -0.14845338463783264, 0.39527085423469543, 1.1125493049621582, 0.07976837456226349, -1.9801125526428223, -0.822562575340271, -0.1665123999118805, 0.8972342014312744, 1.6140539646148682, -0.9872488975524902, -0.35522207617759705, -0.7901365160942078, -0.22140327095985413, 1.1170170307159424, 0.9064521789550781, -0.010519619099795818, 0.4377046525478363, 0.5622897744178772, -1.9104161262512207, -0.47580209374427795, -1.4982002973556519, 1.522114634513855, -0.3825761377811432, -1.7478469610214233, 1.1266518831253052, 0.2795741558074951, -0.24490082263946533, 0.3175791800022125, -0.059810299426317215, -0.4535752832889557, 1.4572491645812988, 1.7610024213790894, -0.7555254697799683, -0.6336279511451721, 0.5942338109016418, -0.9496555328369141, 0.15790899097919464, -0.263285368680954, 1.7542543411254883, 0.2529224157333374, -0.24520844221115112, -0.1327696144580841, -1.1396472454071045, 0.3948567807674408, -0.44046634435653687, -2.039222240447998, -1.798527479171753, 1.8021701574325562, -0.7825672626495361, 1.5265353918075562, -1.9658490419387817, -0.5859153866767883, -2.5524489879608154, 0.1459510177373886, 1.6817741394042969, -1.4821603298187256, -1.8513875007629395, -0.21778331696987152, 1.914899468421936, -0.8041165471076965, -1.4992811679840088, -1.0145701169967651, 1.8886433839797974, -1.492938756942749, 1.5665922164916992, 0.08845440298318863, 1.192805290222168, -0.12558992207050323, -0.002814200008288026, 0.7592607736587524, -1.103082299232483, -0.14629478752613068, -0.4233795702457428, -0.2797139883041382, -2.2012393474578857, 0.7087283730506897, 0.4249192178249359, -0.04301561787724495, 0.5757911801338196, 0.2135404348373413, -0.8508298993110657, -0.060449350625276566, 1.0339865684509277], [-1.7604038715362549, 1.8572152853012085, -0.2962060272693634, 2.015734910964966, 2.290233612060547, -1.016631007194519, -0.42837151885032654, 0.2505165934562683, -0.9536495804786682, 0.3015551269054413, 0.139872208237648, 0.19246871769428253, 0.28734391927719116, -1.8911832571029663, 0.16537708044052124, -0.11601921170949936, 0.9701016545295715, 0.8997583985328674, -0.33772915601730347, 0.2326236218214035, 0.7132824659347534, 0.6656755805015564, 0.45038509368896484, 0.3893730938434601, 0.4786929488182068, 1.0443483591079712, -0.16370190680027008, 0.516294002532959, 0.2893451452255249, -0.004978752229362726, -0.43159615993499756, 1.5595353841781616, -0.7281134724617004, 1.7997536659240723, -1.775341510772705, 0.4277600646018982, -0.40272095799446106, -0.430099755525589, -0.8997090458869934, 1.7133698463439941, 0.4647260010242462, -0.2462858110666275, 0.12538164854049683, 0.47977402806282043, 0.3183014690876007, -1.3056834936141968, 0.8457910418510437, 2.0990374088287354, 1.0108741521835327, 1.2387070655822754, 0.4639016091823578, -1.0091801881790161, -1.3880785703659058, -0.9451260566711426, -1.2212733030319214, 0.5323933959007263, -2.839867115020752, -0.5936009287834167, 1.9031004905700684, -0.508038341999054, -2.4818456172943115, 0.7206268906593323, -0.792195200920105, -0.6207310557365417, 0.3271328806877136, -0.9017364978790283, -0.1226547509431839, -0.8910985589027405, -1.5455716848373413, 0.15428364276885986, 0.7479821443557739, -0.25266921520233154, -2.5461606979370117, -0.43674102425575256, -0.9805081486701965, 1.0148396492004395, 1.3008490800857544, 1.318325400352478, 0.5456681847572327, -0.022741755470633507, 0.383333295583725, -0.22419096529483795, 0.5050439238548279, -0.40224266052246094, 0.13225306570529938, 0.5230961441993713, -0.5763453245162964, -0.338852196931839, -0.5631424188613892, 2.457008123397827, 0.49557968974113464, 0.15142633020877838, -1.0696451663970947, -0.40880194306373596, 1.3830379247665405, -0.18944807350635529, -0.8019140958786011, 0.44791144132614136, 0.9512177109718323, -0.4777265787124634, 1.1951555013656616, -0.4096551239490509, -1.6260892152786255, -1.021870493888855, -0.2511443495750427, 1.3784798383712769, 1.011142373085022, 1.2309547662734985, -1.0306721925735474, 0.6429674029350281, -0.10326731204986572, -0.5597721338272095, 1.029160976409912, 0.31943127512931824, -1.022240161895752, 1.6981582641601562, 0.051798995584249496, -0.16144917905330658, 0.09126941859722137, -0.032932184636592865, 0.5427933931350708, 0.07660506665706635, 1.6370173692703247, 1.4634997844696045, -0.595487654209137, -0.33078449964523315, 2.62153697013855, 1.084938406944275, -0.13462872803211212], [-0.1318867802619934, -0.7061105370521545, -0.3407118022441864, 0.9777670502662659, -0.8269853591918945, -0.7281366586685181, 0.6479883790016174, 0.09111614525318146, -0.6508216857910156, 2.2975575923919678, 0.5027840733528137, 0.32516518235206604, -0.059055011719465256, -1.6210569143295288, 0.7565732002258301, 1.0308616161346436, -1.0416268110275269, 0.8261458277702332, 1.6484777927398682, 0.6437767148017883, 0.24696004390716553, -0.32314908504486084, -0.7496638298034668, -0.6770809292793274, 0.7695936560630798, -2.4103646278381348, 0.9453260898590088, 0.8156810402870178, 0.4128549098968506, -0.15163788199424744, 1.4895687103271484, 0.44488266110420227, 0.7625964879989624, -0.23778319358825684, 0.8639242053031921, -0.3228049874305725, 0.6068116426467896, -1.6791013479232788, 0.41172975301742554, 0.6649288535118103, 0.20280565321445465, 0.288099467754364, -0.4751755893230438, 0.5319660902023315, 1.6243902444839478, -0.320741206407547, -0.4656299352645874, -0.29718419909477234, -0.4976068437099457, 0.7752602696418762, -1.3746870756149292, 0.25725358724594116, 0.06050059199333191, -1.132528305053711, -0.36794519424438477, -0.635852038860321, -1.8414561748504639, -1.0591821670532227, 0.8448533415794373, -0.12132085114717484, -0.7383567094802856, 0.4788512587547302, -2.095006227493286, -0.030400829389691353, -0.21566548943519592, -0.5138904452323914, -0.008946135640144348, 0.3710787892341614, -1.021222472190857, 0.25991612672805786, -0.3634062111377716, 0.32235193252563477, 0.9870254993438721, -0.2382427304983139, -0.26333877444267273, 1.1080392599105835, 0.5069870352745056, 1.1484953165054321, -0.18006287515163422, 1.268184781074524, -0.2506639063358307, -1.0033446550369263, -0.10395180433988571, 0.8732855319976807, 1.3647563457489014, -0.037368740886449814, -0.5091943740844727, -0.5893436074256897, 1.3125101327896118, -0.04873352497816086, 1.404112696647644, 1.3347545862197876, -0.14248989522457123, 1.4427249431610107, -0.09289334714412689, -1.0591456890106201, 0.10449641942977905, 0.41600215435028076, -0.018497329205274582, -0.4424467980861664, 1.7694686651229858, -0.8433146476745605, 0.6457054018974304, -0.4933187663555145, 0.2510579824447632, 1.302753210067749, 2.4969561100006104, -0.08529961854219437, 0.6155651807785034, 0.08235490322113037, -0.358488529920578, -0.41324082016944885, 2.7708139419555664, 1.6710255146026611, -0.28839558362960815, -0.9717587232589722, -1.9416718482971191, -0.32759353518486023, 1.530821442604065, -0.2207014411687851, 1.2598789930343628, -0.43396416306495667, -1.4166593551635742, 0.04486769437789917, 0.1427575647830963, 0.10829529166221619, 0.3966154158115387, -1.3660238981246948, -0.877359926700592], [0.5697283744812012, 1.6765538454055786, 2.115145444869995, 1.5846658945083618, 1.0069166421890259, -0.3018897473812103, -0.2783873677253723, 0.6232991218566895, 1.098070502281189, -0.3262174427509308, -2.392115831375122, -1.3762614727020264, -1.5985299348831177, -0.4119202792644501, 1.0355005264282227, 0.6478529572486877, 0.13577023148536682, -0.7752124071121216, -0.8970409631729126, -0.1755904108285904, 0.7153844833374023, -0.6548086404800415, 0.33322325348854065, -0.3412395715713501, 0.12561365962028503, 0.2743782103061676, 1.0296489000320435, -0.17794564366340637, -0.6360812187194824, 0.22026868164539337, -1.7842084169387817, -1.0786341428756714, -1.5086153745651245, -0.944135308265686, -0.488593190908432, 0.3797771632671356, -1.7719361782073975, 0.840097963809967, 1.1005162000656128, 0.0973808541893959, -1.0321905612945557, 0.5549924969673157, 0.2541607618331909, 0.22908158600330353, -0.8553933501243591, -0.3652791380882263, 1.0662585496902466, -0.005222450476139784, 0.3670189082622528, -0.06709419190883636, -0.05384784936904907, 0.18830506503582, -0.2748967409133911, -1.653044581413269, -0.054255757480859756, 0.7588059902191162, 1.1234426498413086, -0.8724155426025391, 0.4213246703147888, 0.2283402979373932, -0.7720323204994202, -0.5547791719436646, -0.5871915817260742, 0.6834931969642639, 0.2666179835796356, 1.037099838256836, -0.5955402255058289, -0.4660170376300812, 1.2116426229476929, -0.16985942423343658, -0.983812153339386, 1.0482310056686401, 0.1556064933538437, 0.11654640734195709, 0.9490755200386047, -0.09768360108137131, -0.23152536153793335, 0.5136522650718689, -0.20036624372005463, -0.8763449788093567, 0.7285189032554626, 0.4417574405670166, -0.5724267363548279, -0.22046786546707153, -1.354984164237976, -0.007285913918167353, -0.46590280532836914, 1.5796313285827637, 0.716995358467102, -0.09556996077299118, -0.5677722692489624, -0.20174540579319, 0.29718470573425293, -0.5325685143470764, 0.1291123330593109, -2.4599947929382324, 0.13169297575950623, 1.3054219484329224, -0.5300454497337341, 1.2275887727737427, 0.45697036385536194, -0.17728856205940247, 1.0519109964370728, 0.7976808547973633, -2.625291585922241, 0.21362897753715515, 1.6055335998535156, -0.8894025087356567, 0.21269749104976654, 0.3252747654914856, 1.6566007137298584, -2.0048768520355225, 0.02383653074502945, 0.24244648218154907, 0.842613697052002, 1.2971395254135132, 0.0549689456820488, -1.2994444370269775, 0.05693860352039337, 0.029945382848381996, 0.36279454827308655, -0.9739100933074951, -0.28210899233818054, 1.9752596616744995, -1.8334624767303467, -0.20435631275177002, 0.03529103100299835, -0.41051045060157776, -0.9789739847183228], [-0.271945059299469, 0.6059724688529968, -0.5829898118972778, 0.5421932339668274, 0.2859107553958893, -0.6994290351867676, -0.9586316347122192, 0.3191845715045929, 2.786454200744629, 2.7040414810180664, 0.9179382920265198, -0.8850507140159607, -0.2154570072889328, -0.3441547453403473, -0.07711438089609146, -1.5579524040222168, 0.7555301785469055, -1.6022659540176392, 1.0842370986938477, 0.21099945902824402, 1.7590519189834595, 0.8531107902526855, 1.323256254196167, 0.5915937423706055, -0.8545343279838562, -0.2101009041070938, 0.6269415020942688, 0.8833069205284119, 0.5675137042999268, 1.311452031135559, 0.3267952799797058, -1.2301836013793945, 0.7757068872451782, 0.9221073985099792, 1.1740946769714355, -0.9739894866943359, -0.5389301776885986, -0.8667646050453186, -0.3073784410953522, -1.6032824516296387, -0.10129975527524948, 0.4097384214401245, -0.8153671026229858, -1.0258817672729492, -0.04940054938197136, -0.6335015296936035, -0.9130045771598816, -0.6734508872032166, 0.9984493851661682, 0.16941387951374054, 0.24392810463905334, 0.06930530816316605, 2.280097246170044, 0.2546939253807068, 0.675521969795227, -0.07048565149307251, -0.3538322448730469, -0.8539170026779175, -3.751197338104248, -0.940483808517456, 0.18545299768447876, 0.4055548310279846, 1.2997288703918457, -0.396440714597702, 0.2230517715215683, 0.9868946075439453, 0.17427334189414978, 0.948304295539856, -1.3845163583755493, -1.2993674278259277, 0.6387516856193542, 0.4878425598144531, -1.0429998636245728, 0.9899317026138306, 0.8708134889602661, -0.5910907983779907, -1.5513780117034912, -1.308440923690796, -0.22464703023433685, -1.9241282939910889, -0.7364959716796875, -0.9243799448013306, 0.8939591646194458, 1.8373336791992188, -0.6538020968437195, -1.395135760307312, -0.10371298342943192, 0.32527002692222595, -0.42125290632247925, 1.1657902002334595, 0.7938191294670105, -0.3233467936515808, -1.810503363609314, 1.5082907676696777, 0.22288891673088074, 1.835329532623291, -0.31801754236221313, 1.2291585206985474, 0.2455400824546814, 0.06406291574239731, -0.5422200560569763, 0.33989986777305603, -0.6343479752540588, -0.9628981947898865, 0.5817115306854248, 1.75497305393219, 1.5247339010238647, -0.661754310131073, 0.3996865749359131, -0.970291793346405, 0.8583794832229614, 0.03098214790225029, -0.41934138536453247, -0.2372988909482956, -1.3269157409667969, 0.28439033031463623, -0.006668996997177601, -1.8249858617782593, 1.0445362329483032, -0.03794000297784805, -0.9048974514007568, 0.14060497283935547, -0.4867459237575531, -0.6432998776435852, 0.0700410008430481, -0.6467478275299072, -0.09867866337299347, -0.06592606008052826, -2.754645586013794], [-1.5035889148712158, 0.7383853197097778, 0.06598103791475296, -0.7044364809989929, -0.5811588764190674, -0.6699652075767517, -0.5065575242042542, 1.0259488821029663, 0.29362332820892334, -1.3733375072479248, 0.7473763823509216, -0.2530617415904999, -0.5828347206115723, 0.8741989135742188, -0.39083918929100037, 1.135618805885315, 1.0596940517425537, 0.7764907479286194, -0.2697737514972687, 0.13454647362232208, 1.6998987197875977, -0.6418562531471252, -1.3733346462249756, -1.3779926300048828, 1.3397200107574463, -2.5090537071228027, -0.5403806567192078, 0.6568350791931152, 1.1303428411483765, -1.352979063987732, -0.6612339615821838, 1.490972876548767, -0.15436336398124695, 0.4739832878112793, 0.7596530914306641, -0.5865857005119324, 3.4028477668762207, -0.3063916862010956, -0.44268736243247986, -0.19254650175571442, -1.6309877634048462, -1.0347652435302734, -0.7108945846557617, 1.1479929685592651, -0.4749923646450043, -0.2597218155860901, 0.04008237645030022, 0.8733934164047241, -0.7179998159408569, 0.512552797794342, 0.7377486228942871, -0.16283999383449554, 1.6699060201644897, 0.7091487050056458, -1.2570881843566895, 1.5777435302734375, 1.6570031642913818, -0.5007522106170654, -0.9571085572242737, -0.5397503972053528, 2.218005895614624, 1.3505799770355225, 0.6826081275939941, -0.2981210947036743, 0.8071514964103699, 0.22788789868354797, 0.7445524334907532, 1.2428255081176758, -0.9167042374610901, -0.8744956851005554, 0.8852726221084595, -0.6838873028755188, -0.22103478014469147, 0.7594874501228333, -0.8370177149772644, -1.2603847980499268, 0.2690819203853607, -1.2728506326675415, 1.1754412651062012, -0.19471977651119232, -1.9575037956237793, 0.6126402616500854, -1.381403923034668, 0.33912602066993713, -0.8730245232582092, -0.703961193561554, 0.15759751200675964, -0.3905976712703705, -0.050383392721414566, 0.34701716899871826, 0.631505012512207, -0.2661958336830139, -2.010931968688965, -0.10194770991802216, -0.7690209746360779, 0.9451307058334351, -0.3570238947868347, -1.6174908876419067, 0.46116459369659424, 1.2501546144485474, -1.1459136009216309, -1.002074122428894, 0.06425616890192032, -0.7858328223228455, 1.3267523050308228, -0.13383574783802032, -0.880720317363739, 0.4665985405445099, -0.3489575684070587, 0.6220308542251587, -0.02638096548616886, -0.41478419303894043, 0.16833896934986115, -0.12662996351718903, 0.0633557140827179, -0.979518473148346, 0.5121666789054871, -0.7268563508987427, -1.612952709197998, -1.7306790351867676, -0.35215073823928833, -0.6154926419258118, 0.5439668893814087, 0.6481594443321228, 1.029430627822876, -0.8156747221946716, 0.07433302700519562, 0.8413623571395874, 1.1430704593658447], [0.8729637861251831, -0.11844688653945923, -1.2002148628234863, -0.34815725684165955, 1.4862580299377441, -0.16839124262332916, 0.6751275062561035, 1.5642728805541992, -0.025295820087194443, -0.9164305925369263, -1.2503217458724976, -1.4772213697433472, -0.020898200571537018, 0.1355382353067398, -1.075274109840393, 0.643897294998169, 0.797417163848877, -0.40809744596481323, 1.4624378681182861, -0.08021731674671173, 0.4058082401752472, 0.9581747055053711, -0.589599609375, 2.7758562564849854, 0.09617184847593307, -0.7661770582199097, -0.2240547090768814, -1.9394489526748657, 2.5311594009399414, -0.4353947043418884, 0.7263754606246948, -1.3100895881652832, 0.4405909776687622, 0.05224233865737915, 0.14370161294937134, -0.43245929479599, 1.1319462060928345, 0.9961698651313782, 0.27551019191741943, 0.92447429895401, -1.2112325429916382, 0.03419028967618942, -0.6348536610603333, -0.3502713441848755, -0.04075426235795021, 1.9639678001403809, 0.025618435814976692, 0.06876687705516815, 1.008897066116333, -0.24912555515766144, -0.8627894520759583, 0.23891142010688782, -1.467153787612915, 0.08363600820302963, -1.8989479541778564, -1.4764177799224854, 1.4713572263717651, 0.35824650526046753, 0.4267214238643646, 0.004857508931308985, -0.524969220161438, 1.1961404085159302, -1.3594589233398438, -0.23040151596069336, 0.3172566592693329, -0.2619117498397827, -0.1041165292263031, 0.8877648115158081, -0.2811104953289032, -0.40388816595077515, 2.103825330734253, 0.37099573016166687, 0.5899870991706848, -1.2138782739639282, -0.16114658117294312, 1.1376583576202393, 0.18070365488529205, 1.0738422870635986, 1.1785763502120972, -0.20400360226631165, 0.8281484246253967, -1.2191981077194214, -1.208371639251709, -0.978188693523407, -1.7790372371673584, 0.28482112288475037, 0.19867436587810516, 0.2831520438194275, 1.7492491006851196, 0.33735471963882446, -0.7537041902542114, 0.7874643206596375, 0.3798430263996124, 0.5029274225234985, 0.2450762391090393, -2.8305580615997314, -0.7871543169021606, 0.6709768176078796, 0.13873334228992462, 0.019851582124829292, -0.5462088584899902, 0.13585489988327026, 0.3014998733997345, -1.8254133462905884, -0.28961944580078125, -1.092063069343567, -0.8482390642166138, -1.4251000881195068, 1.3676671981811523, -0.8983988761901855, 0.5973596572875977, 0.8116028904914856, 1.8590006828308105, -0.8874167203903198, 0.08778349310159683, 0.172171488404274, 0.66600102186203, -0.8074352741241455, -0.2964046895503998, 1.4035899639129639, 0.7351112365722656, 0.6445024013519287, -1.0319092273712158, -1.484061598777771, -0.05115906149148941, 1.8478137254714966, -1.763503909111023, -0.30786755681037903, 0.5770657658576965], [1.2939441204071045, 0.18328692018985748, 1.1608450412750244, -0.7471480369567871, -1.0591013431549072, -1.011048436164856, 0.39464282989501953, -0.0008480389951728284, -2.717557668685913, -0.6210006475448608, 1.5025333166122437, -1.1091434955596924, 0.6728799939155579, -0.9923692941665649, -0.21301835775375366, -0.296312540769577, 0.9003797769546509, -0.6671113967895508, 0.9228436946868896, -0.8968942761421204, -0.18242929875850677, 0.899097740650177, -0.29886263608932495, -0.20201437175273895, -0.012255961075425148, -0.1306161731481552, -0.8539068698883057, -0.09599391371011734, -0.5184196829795837, -0.4055933356285095, 0.44119691848754883, -0.3296338617801666, -0.07818403095006943, 1.206431269645691, 0.53614342212677, 0.4148499071598053, 0.24202854931354523, 0.32212135195732117, 0.22819159924983978, 0.23887217044830322, 1.108187198638916, 0.5676167607307434, 0.2969830632209778, -0.4083053171634674, -1.0085723400115967, 0.24032960832118988, 0.1828136146068573, 1.217840552330017, -0.4324404001235962, -0.16118170320987701, 1.8784260749816895, -0.5875518918037415, 0.7864941358566284, 0.08105950057506561, 0.23314030468463898, -0.6413642168045044, -0.9567130208015442, -2.69856595993042, 0.48562541604042053, 0.6298253536224365, -0.08115052431821823, 0.6572823524475098, 0.5600073337554932, -2.381235122680664, 0.12890379130840302, 0.6604697704315186, -0.015095453709363937, 0.038268812000751495, -0.7637234926223755, 0.3734792470932007, 0.7416581511497498, -0.381503701210022, -0.7642291188240051, 0.3951011300086975, 0.6662577390670776, 0.3107925355434418, 0.394662082195282, -0.3310595452785492, 0.7927085757255554, -1.7079116106033325, 1.399009108543396, 0.7238755822181702, 0.869515597820282, -0.13097335398197174, -1.797265887260437, -0.4969600439071655, -0.8394005298614502, -1.1095863580703735, 1.4186906814575195, -0.390129417181015, -1.0409976243972778, -0.7051423788070679, 0.7626942992210388, 0.06375104933977127, 0.7392892837524414, -0.08571681380271912, 0.31118470430374146, 1.8730891942977905, 0.49336400628089905, -0.00020595657406374812, -0.29247885942459106, -0.16675607860088348, -0.23512549698352814, 0.6375740170478821, -1.6015434265136719, -0.19242095947265625, -0.6871767044067383, -0.3125910460948944, -0.1436322033405304, -0.07551714032888412, -1.0894759893417358, -0.41971737146377563, -0.13603274524211884, 2.650372266769409, 0.23291736841201782, 1.4215726852416992, -0.4936794340610504, -1.1854612827301025, -1.6449205875396729, 0.8597688674926758, -0.6015576124191284, 0.6275789141654968, -0.8779957890510559, 0.11231506615877151, 0.05414239689707756, 0.23466192185878754, 0.4193880558013916, 1.364063024520874, -1.2387068271636963], [-0.4479082226753235, -0.6216243505477905, 1.202468991279602, 0.3343060314655304, -0.3565647602081299, 0.3013935983181, -1.0784002542495728, -0.23759792745113373, -1.7956124544143677, 0.1287708431482315, 0.7900578379631042, 2.2307424545288086, 2.82688045501709, -0.26428771018981934, -1.4055886268615723, -0.1138901486992836, -0.9757252335548401, 0.43550905585289, -1.4987646341323853, -1.0533673763275146, -0.11303656548261642, 1.290719985961914, 0.5490637421607971, -1.1571743488311768, -1.1071444749832153, -0.39683178067207336, 1.441904902458191, -0.660760223865509, 0.6360202431678772, 0.7146156430244446, 0.6844190359115601, -1.9676518440246582, 0.28715288639068604, -1.1034555435180664, -1.3950707912445068, -0.6352444887161255, 1.4265152215957642, 1.3494371175765991, 0.6810664534568787, -0.3210494816303253, -0.79911869764328, -2.0424838066101074, -1.1836899518966675, 1.3218848705291748, -0.26247209310531616, 1.1339606046676636, 0.7773663997650146, 0.23492251336574554, -0.2789999842643738, 0.6462919116020203, 0.5113317966461182, 0.3077637553215027, -1.704524278640747, 0.18996448814868927, -2.055744171142578, 0.1262405812740326, 0.7180561423301697, -1.063932180404663, 0.4850851893424988, 0.2804325819015503, -1.1441924571990967, 0.06562492251396179, 1.8215821981430054, 1.9672660827636719, -0.5639939308166504, -0.8890678882598877, 0.4315114915370941, 0.9043198227882385, -0.0027635551523417234, 0.1846459060907364, -0.3654698431491852, -0.5245025753974915, -0.7434781789779663, 0.7416693568229675, -0.5540360808372498, 1.5382575988769531, 0.5069352388381958, -0.4156373143196106, 1.0171130895614624, -0.22095248103141785, -0.7569832801818848, -0.951617956161499, -0.05700981616973877, -2.1472113132476807, -0.7070167660713196, 0.01470426470041275, 0.11307302117347717, 0.40678873658180237, -0.09990663826465607, -0.1861274242401123, 0.7353550791740417, -0.6182945966720581, 0.6645644307136536, -0.7566586136817932, 0.4353926181793213, -0.9878664612770081, 0.42246752977371216, 1.1855456829071045, -0.7551266551017761, 1.1951189041137695, -1.5749253034591675, -0.05999830365180969, 0.11007022857666016, 2.0069987773895264, 0.737787663936615, 0.667173445224762, 0.09267541766166687, -0.5555127263069153, 0.06449386477470398, 0.4618014097213745, 0.9578710198402405, 0.3174155056476593, 0.05648050829768181, -0.209001362323761, 0.15332402288913727, -0.22181591391563416, -1.075334906578064, -1.2837377786636353, -0.4343253970146179, 2.5018234252929688, -1.2355986833572388, 0.09704306721687317, 0.5524471402168274, 1.0236690044403076, -0.29175910353660583, -1.0222777128219604, -0.7599260807037354, 1.6244930028915405, -0.6239985227584839], [-0.7283916473388672, -0.07710845023393631, -0.42171627283096313, 0.8779527544975281, -0.5708993673324585, -0.27900245785713196, -1.2183668613433838, 0.32037457823753357, -0.8360154032707214, 0.5208047032356262, 0.0153310252353549, -0.20896504819393158, 0.6248115301132202, -0.19899171590805054, 0.020056692883372307, 0.1849076747894287, -0.5967991948127747, -0.8199921250343323, 1.0508819818496704, 0.7125186324119568, 0.940485954284668, 1.930366039276123, 1.7994122505187988, 1.3995529413223267, 1.8718868494033813, -1.5692437887191772, -1.097886562347412, -0.7136896252632141, 0.2475745677947998, 0.4890318810939789, -0.6261385083198547, -1.3164949417114258, 1.2935186624526978, 2.6742935180664062, 0.16040357947349548, -1.533685326576233, 0.1609012335538864, 0.15378151834011078, -0.042119815945625305, 2.2172608375549316, 0.1657891720533371, 0.5724239349365234, 0.0783369168639183, 0.3343433141708374, 1.9558929204940796, 1.0656944513320923, 1.8164225816726685, -0.7078076004981995, -0.42099103331565857, -0.1839517056941986, 0.5197126865386963, -0.9035757780075073, 1.585419774055481, -0.8132832646369934, -0.7397329211235046, -0.7699295282363892, 0.5779502391815186, -0.9078956842422485, -0.2526153326034546, 1.1982980966567993, -0.12295124679803848, 1.4056243896484375, 1.2702854871749878, 0.5051965713500977, -0.7681183815002441, 0.5544353723526001, -0.8162487745285034, 0.9558936357498169, 0.5991007685661316, 0.3954012095928192, 1.1833562850952148, -0.909733772277832, 1.9757685661315918, 0.5757378339767456, -1.3056293725967407, -1.3356044292449951, 0.5048128962516785, -0.04804806783795357, -0.9417039752006531, -0.18131695687770844, 0.4640296697616577, 1.2081998586654663, -1.411041259765625, 0.09061260521411896, 0.08291036635637283, 0.7565933465957642, -0.1415935903787613, -0.7251401543617249, 0.4854278266429901, -0.5114965438842773, -1.1092203855514526, -0.18540145456790924, 1.6751041412353516, 0.05205455794930458, -1.7519360780715942, 1.050427794456482, 0.9596653580665588, 0.855867862701416, -0.9079905152320862, -1.4130594730377197, 0.7977505326271057, 0.9034005403518677, 0.9406275749206543, -0.5362498164176941, -0.916077196598053, -0.8192195296287537, 1.4270734786987305, 1.5114699602127075, -0.16853880882263184, -1.1377613544464111, 0.2632392644882202, 0.9087007641792297, 0.8020769953727722, 1.53389310836792, 0.5772029757499695, -1.3425198793411255, 2.967343330383301, 0.8614992499351501, 1.1394520998001099, -1.6693068742752075, -0.2497258484363556, -0.2729966640472412, 1.0068881511688232, -0.2185361534357071, 0.06714005023241043, -0.3629063367843628, 0.14163438975811005, 1.4284604787826538, -0.20431138575077057], [-0.6812283992767334, 0.8928045034408569, 1.6283059120178223, 0.6962798237800598, -0.14594998955726624, 1.294569969177246, 0.11370356380939484, -2.379265546798706, -0.8760768175125122, 1.4331393241882324, -0.8426961898803711, -1.2518606185913086, -1.8372182846069336, -0.02290966548025608, 1.0498707294464111, -0.1913590282201767, 0.684587299823761, 0.544715166091919, -0.4103330671787262, -0.3597292900085449, -0.9943239092826843, 0.10864319652318954, 1.1920242309570312, -0.27499696612358093, -0.5532128214836121, 0.0714557021856308, 0.8647586107254028, -1.3010343313217163, -0.7740676999092102, -0.17049668729305267, 1.0109528303146362, 0.006792610045522451, 0.5527790784835815, 0.007394053507596254, 0.7999034523963928, 0.31472721695899963, 0.9742853045463562, -0.40824732184410095, 2.2168712615966797, 1.249877691268921, -0.34870705008506775, 0.23888297379016876, 0.08646060526371002, -1.0628174543380737, -0.13418589532375336, 0.5464887619018555, -0.01393906306475401, -1.0707910060882568, -0.1293467879295349, 0.20872248709201813, -1.8836677074432373, 0.6993228197097778, 1.1252747774124146, 0.7123520374298096, 0.8119996786117554, -0.7996161580085754, 0.19377930462360382, -0.06058831512928009, -1.031059741973877, 0.9096001982688904, -0.13652640581130981, 1.6206653118133545, -0.7501949071884155, 0.5039317607879639, 1.1241981983184814, -1.0057867765426636, 0.7939544916152954, 0.3295946419239044, 0.060803063213825226, 1.3613131046295166, 0.4233570992946625, 1.7649565935134888, -0.4014275372028351, -1.4152776002883911, 0.17315351963043213, -0.6431335210800171, 0.1652352660894394, -0.7216338515281677, 1.2840074300765991, 1.0339553356170654, -2.3114962577819824, -0.5811911225318909, -0.05418099835515022, -1.1736432313919067, -0.7648724317550659, 0.4511273205280304, -0.7417867183685303, 0.5566231608390808, 0.19319380819797516, 0.13524295389652252, 0.6115687489509583, 0.601626455783844, 2.306732416152954, -2.4677653312683105, -1.001935362815857, 0.4229174256324768, 0.2710406184196472, -1.6039814949035645, 0.8377041220664978, 0.3267892897129059, -1.0070804357528687, -0.5883159637451172, 0.34699761867523193, -1.1270167827606201, 0.2860740125179291, -0.09305649995803833, 0.39985716342926025, -0.08023697137832642, -1.205746054649353, 1.0714665651321411, -0.556033194065094, 0.2263476848602295, 0.6750972867012024, 1.2461079359054565, -0.783187747001648, -0.5338526964187622, 0.5697043538093567, -0.17907364666461945, 0.2860196828842163, 0.8911124467849731, -0.5241309404373169, -1.5613707304000854, -0.40773066878318787, 0.8196567893028259, -0.23936371505260468, 0.13817855715751648, -0.2191726416349411, 1.3153122663497925, 0.5019806027412415], [-1.5738439559936523, 0.9972087740898132, 0.08328593522310257, 0.7381197214126587, 1.0351418256759644, 0.7241991758346558, 0.17691943049430847, 0.280519962310791, 0.21454140543937683, 1.2171354293823242, -0.26916152238845825, -1.1153676509857178, 0.8542280197143555, 1.9042978286743164, 0.5415217280387878, 0.18269266188144684, 0.5304369926452637, 0.44965073466300964, 0.1343553066253662, 0.8511143326759338, 0.792881429195404, -0.06779550760984421, 1.3548064231872559, 1.2994667291641235, -1.5991266965866089, -0.748187780380249, 1.620365023612976, -0.44257885217666626, -2.6342616081237793, 0.2524220049381256, -0.12899982929229736, 0.7799520492553711, 1.8395313024520874, 1.0246652364730835, -0.7111156582832336, 0.09468240290880203, 1.2731094360351562, 0.9391366243362427, 1.0848010778427124, 1.7726799249649048, 0.3982887268066406, 0.05913788825273514, 0.4334065020084381, 0.7545838356018066, -0.16588641703128815, 2.2407736778259277, 0.012834515422582626, -1.73422110080719, -0.4153400659561157, 1.4739576578140259, 1.091610074043274, -0.4884294271469116, -2.1678848266601562, 2.7595977783203125, 0.9451127052307129, -0.8068298101425171, -1.0128557682037354, -0.6002243757247925, 1.38448965549469, -1.4307801723480225, 0.05078783258795738, 0.2060176283121109, -0.07197132706642151, 1.6537690162658691, 0.5128267407417297, -0.24089346826076508, 1.3959096670150757, -0.09321993589401245, -0.6646056175231934, -0.5277394652366638, -0.010894554667174816, 1.3337870836257935, -1.0490220785140991, -0.10676643997430801, 0.4236224293708801, 0.4542800486087799, -0.7370175123214722, -0.5103820562362671, 0.29620829224586487, -0.577849805355072, 1.5390303134918213, 1.1676584482192993, -1.5833479166030884, 0.7819843888282776, 0.483717679977417, -0.20649972558021545, -0.26529356837272644, 1.3301410675048828, 1.194211721420288, 0.16869717836380005, -0.14681531488895416, -1.1933680772781372, -1.5938489437103271, -1.4102281332015991, 1.9634071588516235, -0.5110083818435669, 0.013388099148869514, -1.378026008605957, -1.1648048162460327, 1.111626386642456, -1.0182864665985107, 0.4982745945453644, 0.6542660593986511, -0.6426854133605957, -1.0254722833633423, -0.5705212950706482, 1.4581471681594849, 0.2637443244457245, -1.0189919471740723, -0.16251006722450256, 0.6971535682678223, 0.38165998458862305, 0.47489774227142334, -0.11130071431398392, -0.66786128282547, -1.4532138109207153, 0.04681162163615227, 0.33846768736839294, 1.1678296327590942, 0.1885644942522049, 0.4852331280708313, 0.9591450095176697, -1.0447797775268555, -0.25261518359184265, 1.077109694480896, -1.1367911100387573, -0.6060632467269897, -0.3034614622592926, 1.075780987739563], [-0.9711405038833618, -0.4705623388290405, -1.7032113075256348, 0.006677606143057346, -0.3109760880470276, 0.40992581844329834, 1.836076259613037, -0.4272623062133789, -0.019553737714886665, 1.1202653646469116, -0.32430005073547363, 1.3626534938812256, 1.347886085510254, 0.32061153650283813, -0.2018190175294876, 0.016628609970211983, 1.066674828529358, 1.5400032997131348, 0.6672624945640564, 0.9852274656295776, 1.0026328563690186, 0.171726256608963, -0.22189737856388092, -0.7269080281257629, -0.12234503030776978, -1.4348993301391602, 0.20489634573459625, -0.2702561020851135, -1.950571060180664, -1.5467060804367065, 0.7963927984237671, -1.4281256198883057, -0.057038575410842896, -0.17616523802280426, -0.9173799157142639, -0.10742708295583725, -1.7791227102279663, -1.334409475326538, -0.897157609462738, 0.7126231789588928, -1.4346873760223389, 1.4410494565963745, 0.8998228907585144, -1.2177529335021973, -0.7243695855140686, -0.12407492101192474, -0.8078116774559021, 0.781923770904541, -0.2595612406730652, -0.44322606921195984, -1.3933746814727783, 1.6448330879211426, -0.25756901502609253, -0.14676237106323242, -0.6954399943351746, 0.9943811893463135, -2.3100054264068604, -0.28050002455711365, -1.1147671937942505, -0.16085533797740936, -0.4094311594963074, 0.12028619647026062, -1.444722294807434, -0.31943511962890625, 0.5661651492118835, 0.4918117821216583, 0.6207159757614136, 0.4380212724208832, 0.49852126836776733, -0.3270118832588196, -0.06633343547582626, -1.5726776123046875, 1.2986083030700684, 0.4643978476524353, -0.3712215721607208, -0.01720527745783329, -0.12077946960926056, 1.1634982824325562, 1.1864145994186401, -0.9540085196495056, 0.41142237186431885, -0.3366280794143677, -0.7224094867706299, -0.20864266157150269, 0.7407909035682678, -0.5229445695877075, 0.2953967750072479, 0.7682345509529114, -1.4428497552871704, -3.876490592956543, 2.3761487007141113, 0.2259863317012787, 0.33886319398880005, -1.3482248783111572, 0.7913012504577637, -0.269342839717865, 1.0620821714401245, -2.092230796813965, 0.2114538997411728, 0.1586829572916031, 0.7872651815414429, 1.535617470741272, 0.03413394093513489, 0.7006450295448303, 0.6093283295631409, -1.094648838043213, 1.2876667976379395, 0.19148419797420502, 0.1891506314277649, 0.8644804954528809, -0.9752053022384644, -0.21231384575366974, 0.009861690923571587, 0.7956733107566833, -1.2697571516036987, 0.9288341403007507, 0.6658086180686951, 1.2741451263427734, 0.19977116584777832, 0.33282843232154846, 0.577084481716156, -1.8967407941818237, -1.1086362600326538, -0.4321669638156891, 0.485414057970047, 0.4916757643222809, -1.4971847534179688, -0.46371781826019287, 0.6377040147781372], [0.2671017348766327, 0.7148235440254211, 0.19467875361442566, -0.4400198459625244, 0.8447959423065186, 0.4274245798587799, 0.5456327199935913, -0.47349318861961365, -0.6190013885498047, 0.4910601079463959, -1.502766489982605, 0.16323260962963104, -1.094724416732788, 1.119052529335022, -1.9204334020614624, 0.5904585719108582, 0.08431785553693771, -0.05715414881706238, 0.7770194411277771, 0.9643317461013794, 0.7476142644882202, 1.2788333892822266, -0.36588379740715027, 0.26286259293556213, 1.3275833129882812, -0.24701730906963348, 0.002209487371146679, 0.17750373482704163, -0.39118635654449463, 0.6608700752258301, 0.8240007162094116, 1.1504970788955688, 0.39428961277008057, -0.3422929644584656, -0.4507720172405243, -1.8093825578689575, 1.092309594154358, 1.193856954574585, -0.7602090835571289, -0.49449557065963745, 1.0531877279281616, 1.167507290840149, 0.4771783649921417, -0.6741974949836731, 0.27564725279808044, -0.35612478852272034, -0.09303750842809677, -0.11201678961515427, -0.4803614020347595, 1.5525745153427124, -0.9751417636871338, 0.40689411759376526, -0.25832971930503845, 1.2907899618148804, 0.49420472979545593, -0.9523986577987671, -0.05750914663076401, -0.07438004016876221, -1.4890613555908203, -1.163054347038269, -1.5352122783660889, -1.7473493814468384, 0.43850791454315186, -0.04667694494128227, -1.2976341247558594, 1.3591400384902954, -0.49713027477264404, -0.30202171206474304, 0.12711016833782196, 0.1581902652978897, 0.03668222203850746, -0.16441762447357178, 1.463239312171936, 2.0900797843933105, -0.18844252824783325, -0.11264511942863464, 1.4583004713058472, 0.37343600392341614, -0.31958529353141785, 1.3044887781143188, 0.9318526983261108, 1.1610805988311768, -1.1046839952468872, -0.887421727180481, -0.8170216083526611, -0.041274894028902054, -0.17296555638313293, 0.2377409040927887, -1.355294942855835, 0.4966055154800415, 1.668347716331482, 0.43260782957077026, -0.27245697379112244, 1.1364332437515259, -1.2171438932418823, 1.6509215831756592, -1.018141746520996, 0.0801885575056076, 0.6835741996765137, -1.91847562789917, -0.9421548247337341, 1.8149902820587158, 0.2163214087486267, -0.10130763053894043, -2.0614676475524902, -0.22398041188716888, -0.18795911967754364, -0.038172848522663116, 0.5405179262161255, -0.028280330821871758, -0.02710624225437641, -2.6747899055480957, 1.1079055070877075, 0.40925154089927673, 2.779268503189087, -0.08390825986862183, -0.318002313375473, -0.6336813569068909, -1.4942771196365356, -1.1828299760818481, 1.3944573402404785, -0.2723858654499054, -0.19555717706680298, 0.006205747369676828, -0.2904477119445801, -0.5463883280754089, 0.7150158882141113, -1.03378427028656, 0.725455105304718], [0.5672183632850647, -0.1693086326122284, 0.47095444798469543, -0.8072671890258789, 0.036873575299978256, -1.049311637878418, 0.34452739357948303, 2.1372625827789307, 2.1729750633239746, 0.17239804565906525, 0.5790603756904602, -1.5968141555786133, -0.6537142395973206, -0.5836942195892334, 1.2147612571716309, 0.291700541973114, 0.08694212138652802, -1.2431284189224243, -0.4358483850955963, -0.6811023950576782, -0.26991721987724304, -0.5282871127128601, -0.2905673682689667, 0.24462682008743286, -0.19659064710140228, 0.20942644774913788, 0.29314160346984863, -1.8031704425811768, 0.1926715075969696, -1.3539559841156006, -0.9106100797653198, -1.2339695692062378, 0.753667950630188, -0.5654823184013367, -0.8086634874343872, 0.5790634155273438, 0.8978012800216675, 0.96866375207901, -0.24864937365055084, -1.0611474514007568, -0.7079724073410034, -1.089994192123413, -1.3204782009124756, 0.2909899652004242, -0.6615720987319946, 0.2799474596977234, -1.2673667669296265, 1.7281571626663208, -1.2391347885131836, -0.4269600212574005, 1.1549270153045654, 0.5429874062538147, 0.3911740779876709, 1.608070731163025, -0.5355820059776306, 0.3578035235404968, 0.7171924114227295, 0.2763819396495819, -0.361471563577652, -0.6651841998100281, 2.268132448196411, -0.31105440855026245, 0.13072407245635986, -1.1768661737442017, -0.1283736228942871, 0.9737973213195801, 0.30942878127098083, 0.9183699488639832, -1.4115095138549805, -0.7312498688697815, 0.5534654259681702, -0.6295644044876099, -0.005667422898113728, 0.16231176257133484, 0.4092697203159332, 0.6971219778060913, -1.5602139234542847, 3.0988576412200928, 0.7873209118843079, 1.0381574630737305, 0.7988479137420654, -0.9304872751235962, 0.5700907707214355, 0.257526695728302, -2.697786569595337, -0.36670923233032227, -0.34814122319221497, -1.4226207733154297, 0.6136324405670166, -0.8390112519264221, 1.143630027770996, 2.38480544090271, -0.1465105265378952, -1.1589394807815552, -1.0401650667190552, -0.5575853586196899, 0.2045474499464035, 0.0641864612698555, 0.5076276063919067, 0.09838811308145523, -0.30322155356407166, 1.1409038305282593, -0.19522185623645782, 0.4583880305290222, -1.2703220844268799, -0.02453278936445713, -0.5249154567718506, -0.2852308750152588, 1.6556981801986694, 0.5690933465957642, 0.7195746898651123, -0.6501560807228088, 1.4939194917678833, 1.2567675113677979, 0.5136786699295044, -1.5246150493621826, -0.9337915778160095, 0.04762652888894081, -0.8380486369132996, 1.5038541555404663, 1.6968883275985718, 0.08420854061841965, -1.8672385215759277, -0.9750455021858215, -0.01076104398816824, -0.6973702907562256, -0.06902020424604416, 0.6471515893936157, 1.1166315078735352], [0.41278278827667236, -0.4793968200683594, -0.7285423278808594, 0.629730224609375, 0.8021863698959351, 1.319758415222168, 0.19890718162059784, 0.123438261449337, -1.4754085540771484, 1.8253763914108276, -0.4101365804672241, 0.6515260934829712, 0.5178359150886536, 0.5557714700698853, -2.318169116973877, -1.3727625608444214, -0.08313039690256119, 1.4351168870925903, -1.0392645597457886, 0.06360265612602234, 0.8372659683227539, 0.6525011658668518, 0.5156435966491699, -0.24548883736133575, 0.11216374486684799, 0.9856767654418945, -0.2659057676792145, -0.1848548948764801, 0.0516970232129097, 0.9744802117347717, -1.201021671295166, -0.8675000667572021, 1.8488597869873047, -2.1334457397460938, -0.5159375667572021, 0.8865715861320496, 0.0041600922122597694, 0.7499567270278931, -0.46690604090690613, -0.570310652256012, 0.2892850339412689, -0.37483683228492737, -0.713007390499115, 0.8825850486755371, 0.39513590931892395, -0.3690687119960785, -0.15424101054668427, 0.062469638884067535, 0.402299702167511, -0.6227688193321228, 1.6599360704421997, 0.8911101818084717, -0.2606640160083771, -0.39168891310691833, -0.42889663577079773, -0.05159708485007286, 0.6557604074478149, 0.48321446776390076, 0.1055949479341507, -0.18168626725673676, 0.9805626273155212, 1.3208128213882446, -1.1743981838226318, -1.1006711721420288, 1.0472642183303833, -0.9066633582115173, 0.868061363697052, 0.35623347759246826, -0.2663681209087372, -0.5905965566635132, 1.8221352100372314, 0.2275070697069168, -0.4847482442855835, 0.0007541327504441142, 0.8693273067474365, 0.028288695961236954, -1.2620819807052612, -1.7796921730041504, -1.6284184455871582, 1.541410207748413, -0.4925827085971832, -2.704118490219116, -1.3688039779663086, -0.2650877833366394, -0.2432534545660019, -0.9341841340065002, -0.6437345147132874, -0.47653159499168396, 1.6575889587402344, 0.024417554959654808, -0.1941441297531128, 0.8139611482620239, -1.0968985557556152, -1.7965892553329468, 0.08318683505058289, 1.3229931592941284, 2.0805444717407227, 0.938046395778656, 0.03585042059421539, -0.28574338555336, -0.06869136542081833, -2.149670362472534, 0.5744802355766296, -0.39121532440185547, 0.32297173142433167, -0.6989665031433105, 1.31539785861969, 0.3339073359966278, -1.1573615074157715, 0.002593080047518015, 0.874886691570282, -0.27991577982902527, 0.8305571675300598, -0.8027198314666748, -1.6531836986541748, 0.2853030562400818, -0.09595528244972229, -3.2498738765716553, -1.1887422800064087, -0.795934796333313, -0.8020212054252625, -0.7035654187202454, 0.9397724270820618, 0.471700519323349, 0.4514918029308319, -0.6971069574356079, -1.6544673442840576, 0.7455310225486755, 1.0389716625213623], [0.49750620126724243, 1.0492572784423828, -0.6804851293563843, -1.8875560760498047, 0.17233812808990479, -0.14436359703540802, -0.2450738251209259, 0.33187538385391235, 0.6174991130828857, -0.23427285254001617, 2.1887903213500977, -0.9377352595329285, -0.07587804645299911, -1.228337049484253, 2.153496742248535, -0.6024377346038818, 1.0502619743347168, 0.49194496870040894, -1.118477702140808, 1.349778175354004, 0.09059777855873108, -1.2816698551177979, 1.6026033163070679, 0.22263121604919434, -0.9983770251274109, 2.140348196029663, -1.3123658895492554, -1.0991473197937012, 0.887906551361084, 0.49938642978668213, -0.7816570401191711, -0.402676522731781, 0.3771769404411316, -0.3173183798789978, 0.9156783223152161, -0.014213566668331623, -1.718326210975647, -0.4097413420677185, 0.22694070637226105, 0.34982970356941223, -0.2326391041278839, -0.8262350559234619, -0.7719792127609253, 0.521384596824646, 1.69906747341156, 0.24825513362884521, 0.05840575322508812, 0.40501976013183594, -1.5901862382888794, -1.0693124532699585, -0.734411358833313, 1.8141716718673706, -0.31748682260513306, -0.807084321975708, -0.1475643515586853, 0.24039199948310852, 0.3714982867240906, -0.06407083570957184, -0.4378155767917633, 0.9747911095619202, -1.3582165241241455, 1.0541157722473145, -1.2055714130401611, 0.4798600673675537, 2.0735015869140625, -1.0125703811645508, -3.0017898082733154, -0.6259371638298035, -0.6685153245925903, -0.41715672612190247, 0.4290676712989807, 0.0010834995191544294, 1.3552480936050415, 2.631688117980957, 0.8493243455886841, 0.7719029784202576, -1.310613989830017, 1.9495563507080078, -0.11254555732011795, 0.8035120964050293, 0.2348935902118683, -0.2794995903968811, -1.0474470853805542, -1.636803388595581, -0.8691471815109253, -1.5624836683273315, 2.373227596282959, -0.22645708918571472, -2.077491044998169, -0.16615554690361023, -0.6417640447616577, 0.9327070116996765, 0.8613182902336121, -0.21423588693141937, -1.2051935195922852, -0.274584025144577, 0.14984744787216187, -0.1951381415128708, -0.606620728969574, -0.8316195607185364, -0.24287089705467224, -0.42921608686447144, 0.7678017616271973, -1.2132230997085571, 0.23982647061347961, -1.0553832054138184, -0.17522090673446655, -0.9392320513725281, 2.1501479148864746, 0.016673490405082703, -0.04440488666296005, -1.1449940204620361, -1.4593678712844849, -0.5343160033226013, -2.323857307434082, -0.8668292760848999, 1.4896268844604492, -0.03953099623322487, 0.43501612544059753, 1.3580082654953003, 0.11831953376531601, -0.10599526762962341, -0.03979376330971718, 1.541320562362671, -0.19546309113502502, 0.6011919379234314, 1.0349806547164917, -0.9923416972160339, 0.7803471088409424], [1.5456856489181519, -0.04465007036924362, -1.0736528635025024, -1.6725996732711792, -0.7359436750411987, -1.0934019088745117, 0.6377968788146973, 1.6414180994033813, 0.9928754568099976, 0.6767838001251221, 1.2758827209472656, -0.5201370120048523, -0.030214818194508553, -0.15057064592838287, 0.03922652825713158, 1.4516454935073853, -0.6936159133911133, 1.168340802192688, -1.8954331874847412, -0.8479785323143005, -1.810064673423767, -0.25821322202682495, -0.6961190104484558, 0.12886092066764832, 0.804130494594574, 0.2684609293937683, -1.3208001852035522, -0.18455086648464203, 1.4310411214828491, 0.6103189587593079, -1.7304219007492065, 0.5242062211036682, 0.7505080103874207, -2.3732731342315674, -1.084465742111206, 0.09344733506441116, 0.8909484148025513, 0.7997481226921082, 0.21923546493053436, -0.2725782096385956, 0.44036665558815, -0.40180879831314087, 0.19759002327919006, 1.1760492324829102, 0.3306162357330322, -0.5948493480682373, 1.716357707977295, -0.3427014946937561, 0.13262271881103516, -0.9663760662078857, -0.5819794535636902, 0.24268825352191925, 0.5198408365249634, 0.45449531078338623, 1.1163402795791626, -0.623789370059967, 0.2611457407474518, -0.9252204298973083, -1.4140751361846924, -1.8164827823638916, 0.5169351696968079, -0.287701815366745, 1.544825792312622, -1.4633071422576904, 1.64121675491333, 0.5635643601417542, -0.3604243993759155, 1.7490224838256836, -1.0477150678634644, -0.09823627769947052, 2.105395793914795, -0.49633273482322693, 0.504269003868103, 2.233891010284424, -0.5861914157867432, 1.252941608428955, 0.29774010181427, -0.281948983669281, 0.1682148426771164, 0.336453914642334, 0.37997204065322876, -1.3836296796798706, 0.09168768674135208, 1.1251450777053833, 1.730730652809143, -0.5737122893333435, 0.7030555605888367, -1.0399714708328247, 0.047322653234004974, -0.7662895321846008, -1.1075623035430908, -1.2660688161849976, 1.1738539934158325, -0.01902674324810505, 0.7301302552223206, -0.30667757987976074, 0.4334660768508911, -2.2580316066741943, -0.12353893369436264, 0.9028167724609375, -0.7232824563980103, -1.228327989578247, 1.5311130285263062, -0.43325096368789673, -0.7279320955276489, 0.3214673697948456, -1.0659931898117065, 0.12952633202075958, -0.4529477059841156, 0.9333327412605286, 2.561012029647827, 1.7261335849761963, 0.6957216858863831, -0.40718114376068115, -0.3527829647064209, -0.41493532061576843, 0.29989513754844666, 0.05797351896762848, -1.0777302980422974, 1.372357726097107, 0.2811293303966522, -0.08951164036989212, 2.35552716255188, 1.5270878076553345, 1.2735828161239624, 0.4792744815349579, -1.3908833265304565, -1.6725677251815796, -0.7293538451194763], [0.8506270051002502, -1.1258177757263184, -0.6928182244300842, -0.9467033743858337, 0.8055512309074402, 1.069831371307373, 0.18208110332489014, 0.9948704838752747, -0.16172298789024353, 1.0357729196548462, -0.4745377004146576, 1.096958875656128, -1.3204532861709595, -0.051017723977565765, 0.29521095752716064, 1.012271761894226, 0.6992740631103516, -0.306376576423645, 0.4466330111026764, -1.2891571521759033, -1.7373805046081543, 0.3681555688381195, 0.36429354548454285, -0.2953769862651825, -0.15286967158317566, 0.5869902968406677, -0.666863203048706, 0.1662604808807373, 0.6481630206108093, -1.661426305770874, -0.21801793575286865, 0.9468503594398499, 1.277867317199707, -0.2077946811914444, -1.437686562538147, 0.971181333065033, -1.6350502967834473, 0.19391077756881714, 1.0523816347122192, -0.042724572122097015, -1.4772498607635498, 0.8266658782958984, -0.47389358282089233, 0.12748569250106812, 0.015645822510123253, 0.7017816305160522, -2.0661678314208984, 0.4063057005405426, -1.3622634410858154, 1.9880726337432861, 0.1719939410686493, 0.5919127464294434, 0.06390461325645447, -0.7252810001373291, -1.0645633935928345, 0.4844539165496826, -1.2930030822753906, -0.6273781657218933, 0.7061987519264221, 0.42278075218200684, -0.9327794313430786, -0.6132861375808716, -0.5551049113273621, -0.6756556630134583, -1.138479471206665, 0.6507819890975952, 1.7442675828933716, 0.09651673585176468, -0.06939675658941269, 1.5972098112106323, -0.9429166316986084, 1.7398810386657715, -0.11831805855035782, -2.8722496032714844, 0.1875922530889511, 0.3027217388153076, -0.9452217817306519, 1.0098400115966797, -1.3692251443862915, -0.17412741482257843, 0.7438877820968628, -0.40631917119026184, 0.30082637071609497, -1.4264929294586182, -0.21846888959407806, -0.38667964935302734, 0.9313753843307495, 0.5035303831100464, 0.4766123294830322, -0.03541715443134308, 0.8767227530479431, 2.7877848148345947, -1.3954650163650513, -0.3528842031955719, 1.7030926942825317, 0.22512143850326538, 0.3877038061618805, -1.5363131761550903, 1.8514968156814575, 0.027483653277158737, -0.44896912574768066, -1.938804030418396, 1.8520030975341797, 0.7530308365821838, 1.8443610668182373, 0.5533133149147034, 0.07566683739423752, 0.09987686574459076, -1.9445116519927979, 0.013669534586369991, -0.6945335268974304, -0.42271047830581665, 0.5810818076133728, -0.04244934394955635, -1.3249101638793945, 0.8425227999687195, -0.1155037209391594, -0.1950768381357193, -0.3391278088092804, -0.1947288066148758, 0.30804362893104553, -0.5276774764060974, 0.3433081805706024, 0.7446855306625366, -0.022258149459958076, 2.1439085006713867, -1.858893871307373, 0.40626251697540283, 0.13508927822113037], [0.5924674272537231, 1.1566439867019653, -0.24624739587306976, 1.0494674444198608, 0.18525564670562744, 1.1796724796295166, -1.6794489622116089, 0.7154805660247803, -1.0401642322540283, 0.8573023676872253, 0.052103910595178604, -0.179478257894516, 1.1927098035812378, 0.47476062178611755, 0.5314137935638428, -1.5813078880310059, -0.4436971843242645, -0.9487131237983704, -1.1095789670944214, -1.9917821884155273, -0.8577778339385986, -1.6441127061843872, -1.640560507774353, 0.3027443289756775, 1.2103554010391235, -1.5318456888198853, 0.9354309439659119, 1.0908725261688232, 2.4584414958953857, 0.2845807671546936, -1.95514976978302, -0.1562219113111496, 0.15428867936134338, 0.7291639447212219, 0.746468186378479, 2.3376882076263428, 1.1510210037231445, 1.1784998178482056, -0.010347800329327583, -0.26440373063087463, 0.7673958539962769, -0.10360470414161682, 1.295818567276001, 1.161548137664795, 1.27553129196167, -0.4852450489997864, 0.653002142906189, -0.1768786907196045, 0.38933658599853516, 0.3072211742401123, 0.7072214484214783, 1.0755246877670288, 0.045339398086071014, 0.4223368465900421, 1.0333976745605469, 1.9005900621414185, 0.09816481918096542, 0.009569559246301651, 0.3580189645290375, -0.5166031122207642, -0.8977654576301575, 0.16575314104557037, 1.1897321939468384, -0.1809634268283844, -0.6384420990943909, 0.35585811734199524, 1.185915470123291, -0.21509705483913422, 1.641843318939209, -0.48438504338264465, 0.6553691625595093, 0.03590705618262291, 0.290334552526474, -0.15841561555862427, -1.1193913221359253, 1.3451664447784424, -0.7311465740203857, -0.3046627938747406, 0.22061583399772644, 0.31257686018943787, 1.8881003856658936, 1.5930917263031006, 0.66583251953125, -0.8453728556632996, -1.3688546419143677, -0.34408146142959595, 0.13185544312000275, 0.37805619835853577, 0.30207979679107666, 0.06971659511327744, -0.9582633376121521, -0.4662330448627472, 0.3591558337211609, -0.9194172024726868, -0.5859732031822205, -1.9317104816436768, 0.31035369634628296, -0.967298686504364, -1.0216697454452515, -2.302431583404541, 0.0229836143553257, 0.2829509675502777, -1.2138324975967407, -0.7625812888145447, 0.5564737915992737, -0.17201602458953857, 0.403436541557312, 0.16614824533462524, 0.5037844181060791, 0.9701238870620728, -0.4859636127948761, 0.16377666592597961, -1.8913532495498657, 0.37833768129348755, -0.18295007944107056, -0.11545205861330032, -0.2988107204437256, 0.9532033205032349, 1.302304983139038, -1.131860375404358, 1.495720624923706, -0.4011886417865753, 0.9974513649940491, -0.4134373068809509, 0.692000150680542, -0.07277464866638184, -0.1507025510072708, -1.1116962432861328, -0.6956790685653687], [0.14993412792682648, 1.1621922254562378, -0.8518123626708984, -0.4074479639530182, 1.3802878856658936, -1.2299007177352905, 1.1218626499176025, 3.3864541053771973, -0.5291093587875366, 0.32921943068504333, 0.02584126777946949, -0.9029155969619751, 0.22002428770065308, -1.484354853630066, -0.6062760949134827, 1.0162853002548218, -0.4699384868144989, 0.8819770812988281, 1.0478088855743408, 0.5955735445022583, 1.966158390045166, -0.05692402273416519, 0.35063523054122925, -0.9893016219139099, 1.6370412111282349, -0.24447645246982574, 0.5780978202819824, 2.8763506412506104, -1.6015242338180542, -0.9993717670440674, -0.95815110206604, -0.6036238670349121, 1.4692245721817017, 0.34279632568359375, 1.537503957748413, -1.1295390129089355, 0.5680468678474426, 0.3502938151359558, 0.6116449236869812, -0.4181998670101166, 0.5831038951873779, -0.054754987359046936, -2.113391160964966, 0.3434123694896698, 0.7569024562835693, 0.9372413158416748, 0.4936804473400116, 1.1452624797821045, -0.11727557331323624, 1.5848942995071411, -0.3444884717464447, 0.15522156655788422, -0.0378752127289772, -0.9889525771141052, -0.812153160572052, -0.7596550583839417, -1.8795089721679688, 0.26960304379463196, 0.2331124097108841, -0.07948791235685349, 1.7092015743255615, -0.31118980050086975, 2.147066831588745, 0.6853587031364441, -0.21586766839027405, 1.0901551246643066, 0.33073294162750244, -0.7139211893081665, -0.5932425260543823, 0.3642343580722809, -0.8357516527175903, -1.8221434354782104, -0.4720044434070587, 1.0115329027175903, 0.027296146377921104, 1.149418830871582, -2.459944248199463, -0.49877652525901794, 1.284482479095459, -1.5568759441375732, 1.5304287672042847, -0.4350349009037018, -2.5978660583496094, 0.3151114284992218, 0.7259347438812256, 3.060697555541992, -0.133897066116333, 2.0961520671844482, -0.4912585914134979, -0.12276514619588852, 0.1864568293094635, 1.3852620124816895, -0.05668943002820015, -0.03539648279547691, 0.6424115896224976, -0.11913745850324631, -0.25466227531433105, 1.0412691831588745, 0.10286431759595871, 0.20765887200832367, 1.0642695426940918, -0.9669075012207031, 1.2525429725646973, 0.18235301971435547, 1.1320316791534424, -0.7798738479614258, -1.4653159379959106, -1.4635802507400513, 1.3352612257003784, 1.9047576189041138, -0.1969207227230072, 0.6652435064315796, -0.08053819090127945, 0.18064235150814056, 0.07180768996477127, 1.740237832069397, -0.02056344598531723, -0.8073545694351196, 1.0361242294311523, 0.12253826856613159, 0.25916776061058044, -0.5587271451950073, 0.16346390545368195, 0.9881004691123962, -0.8844491243362427, -0.13471390306949615, -0.06597401946783066, 0.2889387607574463, 0.9136358499526978], [-0.6871779561042786, 1.0856084823608398, 1.4585144519805908, 0.6026313304901123, -1.689637541770935, 0.6890799403190613, 1.2478132247924805, 0.2766043245792389, -0.4678064286708832, 0.5468629002571106, 2.9795236587524414, 1.7676854133605957, -1.4665286540985107, 1.3230750560760498, 1.2470163106918335, 0.038343194872140884, 0.5689233541488647, -0.4062730669975281, -1.1567013263702393, -1.015892505645752, 0.5089384913444519, 1.1484941244125366, 0.9030941128730774, 0.023401888087391853, -1.3524770736694336, -0.8527055978775024, -0.06726685911417007, 0.020516593009233475, 0.20634479820728302, -0.0911364033818245, 0.09082688391208649, 0.6355205178260803, 1.226937174797058, -1.8477705717086792, -2.021759271621704, 1.4620906114578247, 0.34773728251457214, 0.5171796679496765, -0.2664128243923187, 0.01138872280716896, -0.41865384578704834, 0.6396999359130859, -1.4418435096740723, -0.5824248194694519, -0.7849522233009338, 0.13996252417564392, 3.20402193069458, 1.430534839630127, -0.5764573216438293, -0.4828832149505615, 0.866902232170105, -0.47187891602516174, -0.7753669619560242, 0.4457932710647583, -1.5339998006820679, 2.045865058898926, 1.57736337184906, 1.111780047416687, 1.3279063701629639, 0.8371092081069946, -1.1654632091522217, -1.024483323097229, 0.02782699465751648, -0.6444287896156311, 0.10017774254083633, 1.7790186405181885, 0.6351838707923889, -0.014109755866229534, 1.2828818559646606, 1.359731674194336, 1.089534044265747, 0.37929973006248474, -0.09871996939182281, 2.4049723148345947, 0.7358299493789673, 0.7996805906295776, -0.9553453922271729, -0.45471489429473877, 1.0192961692810059, 0.6113187670707703, -0.9583624601364136, 0.7258076667785645, 0.5959287285804749, 1.2704063653945923, 0.71480792760849, -0.9866639971733093, 0.9456213116645813, -0.4937659800052643, -0.8394840359687805, 0.6591337323188782, -1.3457986116409302, -0.9374862909317017, -0.14715325832366943, 1.8031762838363647, 0.6541057229042053, -1.1840059757232666, -0.13980649411678314, 0.9655132293701172, -0.8750278353691101, -0.18988798558712006, -0.8337724208831787, 0.9152103066444397, 1.6540647745132446, -0.20585893094539642, 0.3196209669113159, 1.8600785732269287, 1.1833326816558838, -0.04578395187854767, -0.7994590997695923, 0.20753902196884155, 0.7026621103286743, 0.46308374404907227, -0.4141416549682617, -0.1983378380537033, 1.5975574254989624, 0.16827639937400818, 1.2361921072006226, 1.8830426931381226, -0.8138811588287354, -1.4042305946350098, -0.8519036173820496, 0.7239302396774292, 2.1479737758636475, -0.7171035408973694, 2.5198709964752197, 0.6813839673995972, 0.1663276106119156, -1.1432844400405884, 1.643007516860962], [-0.008313946425914764, -0.6877952218055725, 1.1645737886428833, 0.3289327621459961, -2.1324737071990967, 0.4723464548587799, -0.9067223072052002, -0.8232833743095398, 0.1176464706659317, -0.7493337988853455, -0.9170650243759155, 0.7185746431350708, -1.1834064722061157, 1.3149477243423462, 1.1794989109039307, 2.4143214225769043, 0.46460849046707153, -0.023787301033735275, -1.0683550834655762, -0.4309777021408081, -0.2048720270395279, -0.7782411575317383, -1.668927550315857, -0.8626396059989929, -0.7190589308738708, 1.2774337530136108, -0.8985819220542908, -0.17021788656711578, 0.7873900532722473, -0.17600996792316437, -0.3139072060585022, 0.95357745885849, 0.3546925485134125, 0.7190466523170471, 0.14105461537837982, -0.3466653525829315, -0.2748166024684906, 0.1283232420682907, -1.1950690746307373, 0.5902820229530334, -0.24782711267471313, 0.2428468018770218, 0.4737282991409302, -0.38515615463256836, -0.6135132312774658, -0.6522043943405151, -0.871648907661438, 1.4533421993255615, -0.6738566160202026, 0.32966330647468567, -1.0711501836776733, 0.8189934492111206, -0.10234624147415161, -1.363857388496399, 0.19100700318813324, 1.046012282371521, -0.07087098807096481, -0.3982686996459961, 1.2158441543579102, -0.2889135777950287, 0.3874934911727905, -1.2314989566802979, -0.9818992018699646, 1.7087575197219849, -0.11184364557266235, 1.211226463317871, -0.34520578384399414, -0.2145390808582306, -0.09468171000480652, -0.5147124528884888, -0.5840717554092407, 0.3724954128265381, 0.5260714292526245, -1.6571251153945923, -0.776923418045044, 0.16477437317371368, 0.04102001339197159, -0.6875187754631042, 0.3365316092967987, 1.2217124700546265, 2.051997184753418, 1.0362727642059326, -0.27433913946151733, -2.351954936981201, 0.42958348989486694, 1.1308647394180298, -1.4895371198654175, 1.4095728397369385, 0.15076230466365814, 0.6962916851043701, 0.3774861991405487, 1.4586623907089233, -0.34087032079696655, 0.9387901425361633, -0.6664783358573914, 0.4324702024459839, 1.555842638015747, -1.3742412328720093, -0.26687607169151306, -0.4592028856277466, -0.4618293046951294, 0.44375672936439514, -1.2717255353927612, 1.0150307416915894, 0.9987002015113831, 2.0869922637939453, 0.7826549410820007, -0.4560753107070923, 1.019546627998352, 1.1483609676361084, 0.5413116216659546, 0.5498901605606079, -0.34192872047424316, 0.24115443229675293, -0.6807578206062317, -0.15936145186424255, -0.44144293665885925, 0.7489068508148193, -0.18007081747055054, 0.8902878761291504, 0.7058658003807068, 1.5787718296051025, 0.3588979244232178, 0.40308111906051636, 0.5634855031967163, -1.5813535451889038, -1.9444910287857056, -0.07462067902088165, -1.1780225038528442], [-0.6462362408638, 0.27031177282333374, -1.8317787647247314, 1.411306619644165, -0.7103694081306458, 1.9149768352508545, -0.0502813421189785, 0.2834354341030121, 0.4045688211917877, 1.2846952676773071, 0.20780049264431, -0.9483034014701843, 0.5579468607902527, -1.2749273777008057, -0.6772139072418213, 0.1408226490020752, 0.6226776838302612, -0.26855871081352234, 0.6240168213844299, 0.6207579970359802, -1.0614336729049683, -1.8787721395492554, -1.0283477306365967, -0.3483200967311859, -0.9060221314430237, 0.0945620983839035, -0.39305171370506287, -0.7308679819107056, -0.8192893266677856, -1.671875238418579, -0.2909436523914337, 1.2714436054229736, 0.6383281350135803, -0.8646749258041382, -0.759247362613678, -0.7870234251022339, 1.1950693130493164, -0.1558777093887329, 0.09125378727912903, -0.2745586633682251, -0.9584856033325195, 0.4951176643371582, 1.200869083404541, -0.17410032451152802, 0.20854127407073975, 1.7646783590316772, -1.2555795907974243, -0.2676003873348236, -1.1557390689849854, 0.2747916579246521, 0.522067666053772, 0.46169233322143555, 1.556686282157898, 0.12474896013736725, 0.175204336643219, 0.7257543802261353, 0.49749672412872314, 1.321939468383789, -0.272754430770874, -1.6346642971038818, 1.1948412656784058, -0.08189443498849869, 0.04871724918484688, 1.1357041597366333, -0.10977823287248611, -0.9834978580474854, -0.15293467044830322, -0.4945624768733978, 0.27934953570365906, 1.1871249675750732, -0.32162150740623474, 1.5873850584030151, 0.14862029254436493, 1.1331573724746704, 0.7381772398948669, 0.540547788143158, 0.5749177932739258, -0.08026108145713806, -0.4606963098049164, 1.9820482730865479, -0.353180468082428, 1.0592067241668701, -1.8654061555862427, -0.37249818444252014, 0.6126119494438171, -0.909980833530426, 1.2823644876480103, -1.3644133806228638, -0.2324145883321762, 1.7582626342773438, 0.3655563294887543, -0.5609663724899292, 0.6429193019866943, -1.2270828485488892, 0.32817432284355164, -0.7485705614089966, 0.03459925949573517, -1.5815593004226685, 0.07673575729131699, 3.0209460258483887, 1.0047032833099365, -0.06896942108869553, 0.21140891313552856, -0.07576753944158554, 0.39641276001930237, 0.5975149869918823, 0.5685182213783264, 0.7802550792694092, 0.44568130373954773, -0.3747974634170532, -1.1458741426467896, -0.5043808221817017, -0.207655131816864, -0.7176646590232849, -0.7534958720207214, -0.17900507152080536, 1.3665117025375366, -0.23732641339302063, -0.5913445949554443, 0.20209325850009918, -0.41925525665283203, 0.7321287989616394, 0.8926174640655518, -0.19420598447322845, -2.6532535552978516, 0.636138916015625, 1.2843670845031738, 0.03585679456591606, -0.15128260850906372], [-0.6732577681541443, -1.2021535634994507, -1.1146303415298462, 0.44220370054244995, -1.6133275032043457, 0.557503879070282, 0.21899233758449554, 1.1413718461990356, 0.4907686114311218, -0.6438499093055725, 1.2801122665405273, 0.031598933041095734, 1.4844378232955933, -1.697299838066101, 0.6129463315010071, -0.45710957050323486, -0.17498427629470825, 0.1312086433172226, -0.5214084982872009, -0.1537751853466034, -1.0899147987365723, 0.11741005629301071, 0.945088267326355, -0.6656643748283386, -0.7933871746063232, 0.3123650550842285, -1.6316953897476196, 0.051534511148929596, -1.7166677713394165, -1.5206942558288574, -1.153692603111267, -1.2524975538253784, 1.0416496992111206, -1.0135079622268677, -1.1749004125595093, 0.4523675739765167, -1.1097933053970337, 1.0776289701461792, 1.2759193181991577, -2.6015188694000244, 1.0638443231582642, 1.071784257888794, -0.342500776052475, 1.7232775688171387, -1.594022274017334, -0.011196738108992577, -0.2729879319667816, -1.1549911499023438, -1.0865055322647095, -0.2583106458187103, 0.9692808389663696, 0.3092675805091858, 0.5889771580696106, -0.2288438081741333, 0.49504607915878296, -1.6977516412734985, -0.3962767720222473, 2.2873620986938477, 1.205678105354309, 0.44335612654685974, -0.011337146162986755, 0.9992437958717346, 0.807330310344696, 0.9120827317237854, -1.4360606670379639, -0.06718993932008743, -2.5668840408325195, -1.1199921369552612, 1.8826050758361816, -1.1338080167770386, -0.1564679890871048, 0.9640889167785645, -0.051682423800230026, -0.11069982498884201, 0.2572253346443176, -0.7129637598991394, 0.5458419919013977, -0.06092587113380432, 0.37292298674583435, -0.38624104857444763, 0.6901498436927795, -2.1971514225006104, -1.5212011337280273, 0.3653540015220642, 0.3509299159049988, 0.9492596983909607, 1.446632981300354, -1.5839829444885254, 0.819998025894165, 0.5384148359298706, -0.23859025537967682, 0.3718794882297516, 1.616317629814148, -0.4650789797306061, 2.25095534324646, -1.9860938787460327, 0.7105597853660583, 0.2853851616382599, -0.15735021233558655, 0.4455493688583374, 0.06935866922140121, -0.34518155455589294, 0.2243465930223465, 0.2813784182071686, -0.3847123682498932, -1.5675227642059326, 1.7892394065856934, 0.32145121693611145, 0.4270060658454895, 0.9354820847511292, 1.6783188581466675, 0.8281881809234619, -1.6737711429595947, 0.6465586423873901, 1.0936083793640137, 1.6212875843048096, -0.38914594054222107, 1.6338896751403809, 0.9833038449287415, 1.1169028282165527, 0.1270304173231125, -2.171753406524658, 0.7249599099159241, -2.4502909183502197, -0.77923184633255, -1.0649408102035522, -0.6850120425224304, 0.7252815961837769, -0.0473957434296608], [1.1424773931503296, 0.979478120803833, -0.0628105103969574, -0.37153518199920654, -0.4984053373336792, 0.16530759632587433, 0.5978971123695374, 1.1349936723709106, 1.3975945711135864, -1.157994270324707, 0.433303564786911, 1.40790593624115, -0.7083780765533447, -1.5572736263275146, -0.3485230505466461, 0.7092795372009277, -1.492772102355957, 0.4481353163719177, -1.4118694067001343, -0.5088280439376831, -0.22392868995666504, 1.7509628534317017, -1.1948438882827759, 0.06504078209400177, 1.243499994277954, 0.94834303855896, -0.3267711400985718, -0.781119704246521, 0.9582000374794006, 0.646705150604248, -0.6495815515518188, 0.2748260200023651, -1.3448271751403809, -0.2507880628108978, 0.0953640416264534, 2.0602269172668457, -0.2122882604598999, -0.8031663298606873, -0.8762413263320923, 0.6760058403015137, 0.7198777198791504, 0.44409990310668945, 0.5901797413825989, -1.327209234237671, 0.9030980467796326, -1.3375656604766846, 0.1487681269645691, -1.1404433250427246, 1.484347939491272, 0.06312358379364014, -0.04104916378855705, 0.9276931881904602, 1.232385516166687, -1.6045868396759033, 0.22015966475009918, -0.029519617557525635, -1.5001676082611084, 0.20773451030254364, 0.6666510105133057, 0.9917833805084229, 1.880566954612732, 1.1793867349624634, -1.053458571434021, -0.4459283947944641, 0.9263284802436829, -0.1599433273077011, -1.5627578496932983, 0.42012476921081543, 0.11011755466461182, 0.0146655747666955, 1.0515607595443726, -0.014731242321431637, 0.90786212682724, 2.901029109954834, -0.3077280819416046, 0.8611265420913696, 2.0734949111938477, 0.5813381671905518, -0.5823355913162231, -0.4022059738636017, 0.7606845498085022, -0.12344025820493698, -0.08557453751564026, -1.3631700277328491, 0.07390883564949036, 0.06742342561483383, 3.4605438709259033, -0.9205777645111084, -0.7644158005714417, -0.21800576150417328, -0.30577418208122253, -0.34050804376602173, 2.041658639907837, 0.017496459186077118, 1.0191720724105835, -1.5205804109573364, 0.273791640996933, -1.9353073835372925, 0.7330938577651978, 0.9182902574539185, -0.45354655385017395, -0.37648192048072815, -0.4033682346343994, -0.06511987745761871, -1.9776816368103027, 1.1904436349868774, -0.2254629284143448, 0.42479580640792847, -0.718132734298706, 0.5470710396766663, -0.628261387348175, -0.660263180732727, -2.129305839538574, -0.04643666371703148, 0.8367174863815308, -2.9244349002838135, 0.4866250455379486, 0.12110131978988647, 1.4289454221725464, 0.48070788383483887, 0.12099786847829819, -0.3408966660499573, 1.252695083618164, 0.6158453822135925, -0.9536176323890686, 1.5114375352859497, 1.54096519947052, 2.6451263427734375, -0.5186344981193542], [0.4996471405029297, 1.5558122396469116, -2.0469343662261963, 1.064708948135376, -0.6135667562484741, -1.1591885089874268, 0.7914268374443054, -1.0039645433425903, 0.46738648414611816, -0.6097399592399597, 0.38759320974349976, 0.3993339538574219, -0.8528398871421814, -0.8230611085891724, 1.0151231288909912, 1.0670130252838135, -0.06893307715654373, -0.5568867325782776, 1.0169410705566406, 0.42948371171951294, -0.7872721552848816, -0.7151991724967957, -0.42940399050712585, 1.2014108896255493, -0.544133186340332, -1.219830870628357, -1.1100919246673584, 0.8796831965446472, -0.4484822750091553, -0.6172756552696228, -0.03073102980852127, -1.1096874475479126, 0.7853317260742188, -0.2542620301246643, -0.4691180884838104, 0.5684145092964172, 0.13684779405593872, -0.7442633509635925, 1.1078113317489624, -2.089728593826294, 0.7223243713378906, 1.4336047172546387, -0.2183755785226822, -0.7398272752761841, 0.6094383001327515, 1.201582431793213, -1.2952443361282349, 0.18742971122264862, 0.19440162181854248, 0.5836682319641113, 0.28968727588653564, -0.7983375787734985, -1.877821445465088, -1.7294425964355469, -0.766801655292511, 1.2416096925735474, -0.7501541972160339, 1.363736867904663, 0.07840707898139954, -0.38317400217056274, -1.4128884077072144, 0.17209044098854065, 0.35425621271133423, -0.5934956669807434, -1.5681480169296265, -0.5242165923118591, -0.5016013383865356, 0.43397754430770874, 1.2030116319656372, -1.3505244255065918, -0.04908894747495651, -0.0802147388458252, 0.2567262649536133, 0.1170438826084137, -1.2974785566329956, 0.29910191893577576, 0.31205490231513977, -1.0801292657852173, 0.5986344814300537, -0.3579840660095215, 0.8940369486808777, 1.81339430809021, -0.4702240228652954, 0.49478673934936523, -0.39182350039482117, 0.20926722884178162, 1.7984724044799805, 1.0772364139556885, -0.11134159564971924, 0.5519500374794006, -1.9480944871902466, 0.30350378155708313, 0.16360405087471008, 0.5725319981575012, -0.38997235894203186, -0.7453249096870422, -0.8642113208770752, -0.14716091752052307, 1.3630708456039429, -0.38409048318862915, 0.23886102437973022, 1.1355959177017212, 1.9376040697097778, 0.9304329752922058, -0.1813376396894455, -0.7366592884063721, -0.22308498620986938, 0.5510001182556152, -0.9935436844825745, -0.31935057044029236, 0.39357617497444153, 1.0500195026397705, -1.1404941082000732, -0.7178156971931458, 0.9895893931388855, -1.569521188735962, 1.0645266771316528, -0.2871486246585846, -1.2579532861709595, 0.12316373735666275, -0.12373968213796616, 1.416068434715271, -0.4565720856189728, 0.5443756580352783, 1.5780106782913208, -1.1552366018295288, 0.5029629468917847, 1.1022413969039917, -0.9225955605506897], [-0.8607628345489502, -0.13542328774929047, 0.9639670252799988, -2.5397889614105225, -1.5134515762329102, -0.8375843167304993, -0.09612397849559784, 0.0732019916176796, 1.7223682403564453, 0.3709392547607422, 0.8494479656219482, 0.49211835861206055, -0.45048874616622925, -2.7179019451141357, -1.2325527667999268, 0.019985806196928024, 0.7254167795181274, 2.112396717071533, -3.0006344318389893, 1.7279084920883179, 0.9662420749664307, -0.9967710971832275, 2.866476058959961, 0.6658738851547241, 0.4208996891975403, -1.114524006843567, -0.07086322456598282, -0.4408051669597626, -0.5069869756698608, 1.1428191661834717, 0.0941239595413208, -0.8745673298835754, -0.01991603523492813, 0.5953349471092224, 0.2726450562477112, -0.895751416683197, -1.6937344074249268, -0.0368344821035862, -0.6363354921340942, -0.5183109641075134, 1.1262974739074707, -0.8954917192459106, 1.277298092842102, 1.1834080219268799, -0.3736036717891693, 0.060036491602659225, 1.0422970056533813, -0.44066107273101807, -0.1283857524394989, -0.19274362921714783, 0.2706633508205414, 1.3835126161575317, -0.6105270385742188, 0.30947861075401306, 0.33778274059295654, 0.5447243452072144, 0.38534945249557495, -0.01841791905462742, 0.14025473594665527, 0.5754209160804749, 0.05998395383358002, -2.3927724361419678, -1.4005169868469238, -0.9957225322723389, -0.02624046802520752, -0.23479460179805756, -1.4735438823699951, 1.5473634004592896, -0.9993432760238647, 0.7798939943313599, 0.02635260857641697, 1.0220870971679688, -0.7513349056243896, -1.4432635307312012, -0.3706231713294983, -0.20373240113258362, 0.5100534558296204, 0.7704915404319763, 0.19627560675144196, 0.6717664003372192, 0.23433548212051392, -1.5787718296051025, -0.3665476143360138, -0.12229826301336288, 1.51515793800354, 1.55778968334198, 0.1449168622493744, 1.1389045715332031, -0.46467068791389465, 1.3110859394073486, 0.532341480255127, -1.035919427871704, 1.874759316444397, -0.5615004301071167, -0.909019410610199, -0.25804680585861206, -1.040094017982483, -1.6392024755477905, 1.0710580348968506, 1.6839021444320679, -0.25150516629219055, -0.13313448429107666, 2.128082513809204, 0.9191032648086548, 0.6164085268974304, 1.1537357568740845, 1.2845181226730347, 0.3399297893047333, 0.9682731032371521, -0.5946153402328491, 0.03333001583814621, -0.9649650454521179, -0.18906138837337494, -0.3099982440471649, -0.1511348932981491, -0.8027224540710449, 0.9855194091796875, -0.6046077609062195, 1.3841203451156616, -0.43099409341812134, -2.2410449981689453, -0.1724596917629242, 0.5830640196800232, -0.1405590921640396, -0.5016697645187378, 1.5112444162368774, 0.7524035573005676, 0.8883926868438721, -1.2263147830963135], [1.0245893001556396, -2.1330955028533936, 0.949055552482605, 0.503606915473938, 0.38934263586997986, 0.6297687292098999, 1.4540698528289795, 0.0006127294036559761, 0.2634340524673462, 1.049772024154663, 0.3089325428009033, 1.3126745223999023, -0.6389074921607971, -0.7236104011535645, -0.602307915687561, -0.36421918869018555, -0.06352351605892181, -0.20339368283748627, -0.13073372840881348, 0.5003563165664673, 1.520116925239563, -1.2869794368743896, -0.3759419918060303, 0.4468647241592407, -1.3752355575561523, 0.43884801864624023, -0.330186128616333, 0.11133106797933578, -0.13792532682418823, -0.38418203592300415, -0.6290286779403687, 0.22966837882995605, 1.2232028245925903, 0.04622099921107292, -0.7966063022613525, 0.43903473019599915, 0.19796353578567505, 0.8720354437828064, 0.47641998529434204, -2.3454842567443848, 0.08245860785245895, -0.312305212020874, -0.42186227440834045, -0.6600244641304016, -0.7421003580093384, -1.1731159687042236, -0.26240167021751404, -0.1729251593351364, -0.3716815114021301, -0.5072880387306213, 1.138399362564087, 0.36743196845054626, -0.17875665426254272, 1.3047661781311035, 0.15712592005729675, -0.5630238652229309, 0.2656787037849426, 0.28467896580696106, 0.18534033000469208, 2.572511911392212, 0.017449961975216866, -0.8073288202285767, 1.2296507358551025, 1.0032180547714233, 0.20749779045581818, -1.70928156375885, 0.3731231689453125, 0.6953686475753784, 0.5632874369621277, -0.0238956231623888, 1.204859733581543, -0.6580239534378052, 0.6344183683395386, -0.3242349624633789, -0.45040270686149597, -1.3251906633377075, -0.10122335702180862, 1.0138436555862427, -0.6120669841766357, -1.2062495946884155, 0.14730699360370636, -0.8572468161582947, -0.9125115275382996, 0.7630997896194458, 0.7079731822013855, 0.8437460660934448, -0.12776273488998413, 0.7525679469108582, 0.255863219499588, 0.2240123152732849, 1.022565484046936, 0.4213135242462158, -0.46821343898773193, -0.22539448738098145, -1.386120319366455, 0.3415437340736389, 1.1959608793258667, 0.13882213830947876, 1.5125160217285156, 0.24922296404838562, 0.5968568921089172, -1.0086778402328491, 0.23338699340820312, 0.48149374127388, -0.9581197500228882, 2.096342086791992, -0.19371528923511505, -0.45874688029289246, -1.696300983428955, -0.39974528551101685, 0.35407596826553345, 0.6762961745262146, 0.6598024964332581, 1.3992034196853638, 0.16539807617664337, 1.6913968324661255, -1.5161585807800293, 0.41319379210472107, 0.11507807672023773, -1.5897012948989868, 0.7144965529441833, 1.3062103986740112, -1.1393498182296753, 0.47381648421287537, 0.32396894693374634, -0.28197136521339417, -0.31150293350219727, -0.04335765540599823, 0.08309819549322128], [-0.07137038558721542, 2.364349365234375, -1.2984853982925415, 1.6156766414642334, 0.24119746685028076, 1.108306646347046, -1.0435504913330078, -1.2273300886154175, 0.29648739099502563, 0.5090335011482239, -2.839865207672119, -0.739799439907074, 0.3222028315067291, 0.044013261795043945, -0.5554122924804688, 0.16466891765594482, -1.56940495967865, 0.004017187748104334, 1.0201737880706787, 0.49420541524887085, -0.12952777743339539, 1.5448874235153198, 0.2596830129623413, 0.9185229539871216, -0.5075886249542236, 1.3328933715820312, 0.3875860273838043, 1.922666311264038, 0.4204809367656708, 0.9159752726554871, -1.2170275449752808, 0.9669878482818604, -0.40384364128112793, 1.372826099395752, -0.5145487189292908, -1.6802672147750854, 0.7425894141197205, -0.8227320313453674, -0.5979364514350891, 1.111012578010559, -0.005152770783752203, 0.48281797766685486, -0.04199831932783127, -1.940148949623108, 0.4947187900543213, -1.043804407119751, 0.3954022526741028, -0.43914198875427246, -0.5408729910850525, 0.03757310286164284, 0.254907488822937, 0.08607752621173859, -0.2211073935031891, -1.4133129119873047, 0.9996526837348938, 0.19291886687278748, 0.6735583543777466, -0.8278532028198242, -0.7704518437385559, 0.76622474193573, 0.44558197259902954, -0.11759809404611588, 0.6460180282592773, -0.14945706725120544, 1.2409521341323853, -1.7894517183303833, 0.010900272987782955, 1.0099608898162842, -0.4359499514102936, -0.41944292187690735, 0.811837911605835, -1.5144912004470825, -1.684070110321045, 0.6831876039505005, 0.47886571288108826, 0.4985623359680176, -1.553952693939209, 0.6794442534446716, 0.6841075420379639, 2.5578789710998535, 1.342478632926941, 1.8592770099639893, -2.117697238922119, 0.6104637384414673, 2.146965503692627, -3.002272605895996, 2.366877555847168, 1.7775371074676514, 0.7013336420059204, 0.056999076157808304, -0.15857507288455963, -1.869122862815857, 0.8906823992729187, 0.45790672302246094, -0.4144159257411957, 0.2933560907840729, 0.026784298941493034, 1.3135255575180054, 0.7891541719436646, 0.9937993288040161, 1.0173208713531494, -1.7710503339767456, 0.7546810507774353, 0.210529625415802, 0.9926404356956482, -0.6395645141601562, -1.2774474620819092, 1.3177660703659058, -0.5438196063041687, -0.8401513695716858, -0.4072313904762268, 1.1330206394195557, -0.28824952244758606, 1.4004490375518799, 0.30509430170059204, 1.3198951482772827, 0.40683868527412415, 2.200518846511841, -0.26396599411964417, -0.08251552283763885, 0.2684723138809204, -0.31465473771095276, 0.7488076686859131, -0.3743893504142761, 0.6625903248786926, -0.5640023946762085, -0.5882222652435303, -0.9504488110542297, -2.1025731563568115], [0.23754428327083588, 0.7609419822692871, 0.3186953663825989, 1.312888264656067, -1.535712718963623, 0.21958310902118683, 0.3177650570869446, 0.4928824007511139, 1.122713327407837, -0.058776404708623886, 0.93355393409729, 1.5427008867263794, -0.29032137989997864, -0.9491369128227234, -1.9623624086380005, -0.6547052264213562, -1.4699089527130127, -0.6312863230705261, -1.0611813068389893, 0.545488715171814, 2.1007580757141113, 1.1017203330993652, 0.5865086913108826, -1.1012777090072632, -0.6385448575019836, 0.13859644532203674, 2.043240547180176, 1.5057865381240845, -0.6476422548294067, -1.0276519060134888, -0.18443992733955383, 0.6868558526039124, 0.8196254968643188, 1.3958494663238525, -1.471143364906311, 0.2867518663406372, 0.24577297270298004, 0.02362075448036194, -0.6845565438270569, -1.1764211654663086, -0.5863168239593506, 1.9132623672485352, 0.6741269826889038, 0.0027464379090815783, -0.7835659980773926, 0.864810049533844, 1.4812039136886597, -1.0234112739562988, -1.1576979160308838, 1.0410069227218628, 1.3709346055984497, 1.0984134674072266, -1.0237317085266113, 1.4941262006759644, 0.7222280502319336, 0.7080098390579224, -0.16571290791034698, -0.7136866450309753, -0.7211560010910034, 0.4229978919029236, -1.6417500972747803, 0.7703717947006226, 1.8928642272949219, 1.572715401649475, -0.9304731488227844, 1.450332522392273, -0.7231664657592773, -0.687195897102356, 0.5007750391960144, -0.3734588027000427, -0.719435453414917, 0.6197870969772339, 0.15306192636489868, -0.6525754332542419, 0.28894999623298645, -1.163533091545105, -1.811747431755066, 0.1065370962023735, 0.17352642118930817, 0.650505542755127, 0.7067562341690063, 0.4220546782016754, -0.685191810131073, -0.3596165180206299, 1.0804342031478882, 1.1143476963043213, 0.6516638994216919, 0.4527496099472046, 0.8267316818237305, 0.38362917304039, -0.2580559253692627, 0.46149498224258423, -1.9388325214385986, 0.25625473260879517, -0.008015777915716171, 0.47474703192710876, -0.5917902588844299, -0.43818530440330505, 0.08982527256011963, 0.29265981912612915, 0.8388931155204773, -2.1879231929779053, 1.2743104696273804, -0.9483972191810608, -0.0028957799077033997, -0.4583916962146759, -1.4125704765319824, 0.6623451709747314, -1.6344125270843506, 0.6322001218795776, -0.008812838234007359, -0.22097761929035187, -1.1780766248703003, -0.3771754801273346, 0.5909071564674377, -0.593633234500885, 0.2708703577518463, -0.20580576360225677, 0.8282544612884521, -0.9321527481079102, -0.9587403535842896, 1.1962765455245972, -1.9797253608703613, 0.19369173049926758, 0.06809262186288834, -0.44373032450675964, -1.3520753383636475, 0.37672996520996094, 1.1425092220306396], [-1.15082585811615, -0.7415626645088196, 1.3782689571380615, -0.5909835696220398, 0.2532462477684021, -1.3982596397399902, -0.14667421579360962, -0.10351007431745529, 0.17651589214801788, -0.7897525429725647, 0.42763248085975647, 0.23359434306621552, 0.3951483964920044, -1.1420178413391113, -1.133276104927063, -1.8681892156600952, -0.8339789509773254, -0.2941491901874542, -0.4558126926422119, 0.310771644115448, -0.14460495114326477, -0.40109702944755554, 1.1409599781036377, -0.8614056706428528, -0.3074355125427246, -0.6235392689704895, -0.7212110757827759, -0.9074552655220032, -0.4918259382247925, -0.9735201001167297, 0.5981031060218811, 0.8356654644012451, -1.1054270267486572, -0.06591658294200897, -0.0368213877081871, 1.2108794450759888, 1.2791388034820557, -1.6143321990966797, 0.5564398765563965, 0.7924911975860596, -3.190643787384033, 0.322956919670105, 0.3157162666320801, 1.8304951190948486, -1.2977904081344604, -0.5505796670913696, 0.4881584048271179, -1.2769017219543457, -0.16336116194725037, 0.8571585416793823, 0.7186870574951172, -1.0556809902191162, 0.37392204999923706, -1.9337011575698853, 0.6794049739837646, 1.8741880655288696, -0.9289417862892151, -0.7493170499801636, 0.8767385482788086, -0.3941754102706909, -0.106914222240448, 0.29424330592155457, -0.6230648159980774, 0.7088395357131958, 0.3731163740158081, 0.3638899028301239, 0.7853665947914124, -0.11262243986129761, -0.42251959443092346, 0.5770251750946045, -1.157571792602539, 0.6156965494155884, 1.3917467594146729, 1.203590989112854, 0.6587867140769958, -0.18866164982318878, -2.485154151916504, 0.05568687617778778, -1.1007931232452393, 0.982244610786438, -0.9795520901679993, -0.9467228055000305, 1.6668070554733276, 1.055047631263733, -1.4588123559951782, -0.3582094609737396, 1.3138909339904785, -1.3018769025802612, -0.20978564023971558, 1.3180882930755615, 0.5422813296318054, 0.278286337852478, 1.1475815773010254, 0.7024418115615845, -0.1523284912109375, -1.7863463163375854, -1.9651204347610474, -0.07392584532499313, 0.10836531221866608, 1.1881316900253296, -0.20944489538669586, -1.4687074422836304, 0.038498617708683014, 1.618219017982483, 0.996859073638916, 0.48842260241508484, 1.0256565809249878, 0.48136013746261597, 0.7746871709823608, -0.24541404843330383, -1.1650608777999878, 0.2685396671295166, 0.19025181233882904, 0.18442997336387634, -2.607797622680664, -0.4993699789047241, 1.3745208978652954, -0.7258172631263733, 0.8885810375213623, -0.030780324712395668, -1.4761574268341064, 0.04174915328621864, -1.0917543172836304, -2.328765869140625, 0.7605743408203125, -1.3155083656311035, -1.321117877960205, 1.0817255973815918, -0.0011627612402662635], [0.6426967978477478, 0.5447900295257568, 1.9972991943359375, -0.8973638415336609, -1.2180535793304443, -1.0045044422149658, 0.29148241877555847, 1.8474568128585815, -0.35573622584342957, -0.6136911511421204, -1.6267367601394653, 0.663154125213623, -0.6325253248214722, -1.1400519609451294, -0.8464088439941406, 0.0005290163680911064, -1.1454582214355469, 1.3690502643585205, 0.27618712186813354, 0.12093354761600494, -0.003805995685979724, 0.6617330312728882, -0.592540979385376, 0.3484911024570465, 1.0979260206222534, -0.3307705223560333, -1.5753990411758423, 0.6845978498458862, -0.4090780019760132, -0.20335222780704498, -1.3335946798324585, -2.6988766193389893, -0.008365030400454998, -1.1724791526794434, -1.7883390188217163, -0.33486658334732056, 0.20937301218509674, 0.6565219163894653, 0.18450544774532318, -0.35803645849227905, -0.3699943423271179, -0.6377738118171692, 0.1237495169043541, -0.07294055074453354, 0.9344647526741028, -1.5628390312194824, 0.2414599508047104, -0.07820376753807068, -1.367682933807373, -1.6766526699066162, -0.5008714199066162, 0.8412297368049622, -0.3388530910015106, -1.1407625675201416, -0.3583431839942932, 0.6172365546226501, 0.21205373108386993, -0.8434181213378906, 1.3148444890975952, -0.03289377689361572, 0.41497522592544556, -0.710156261920929, -1.512398362159729, 0.48475590348243713, 1.2343087196350098, 0.1213066428899765, -1.8331549167633057, 1.9702939987182617, 1.874045968055725, 1.1215012073516846, 0.3997739255428314, 0.124544657766819, 0.6886981129646301, -0.4968937039375305, 1.197346568107605, -0.40224385261535645, -0.2880644202232361, -0.31537294387817383, -0.20772947371006012, 2.278480052947998, -0.4856261610984802, -0.4021229147911072, -1.3440300226211548, 1.5351639986038208, 0.1644054353237152, 0.7760514616966248, 0.08593446761369705, 0.5803811550140381, -1.2434321641921997, 1.7937425374984741, 0.14065156877040863, -0.8485021591186523, 0.1418415755033493, 1.3389403820037842, -1.774489164352417, 0.8962652683258057, 1.5647801160812378, 0.007033849600702524, -0.31083202362060547, 0.369476854801178, 2.938603639602661, -0.0681416392326355, 0.9964179992675781, 0.0788184329867363, 0.10283807665109634, -0.17644758522510529, 0.4214004874229431, 1.4187604188919067, 0.472119003534317, 0.1888011395931244, 0.39942777156829834, -2.0079872608184814, 0.33960282802581787, -2.7093048095703125, -0.5306371450424194, 0.2193533480167389, 0.4394652545452118, -1.1824586391448975, -0.534925103187561, 1.615704894065857, 0.6901431679725647, 0.4514818787574768, 1.7461005449295044, 0.4427107870578766, 0.15624025464057922, -0.7459928393363953, -1.6685035228729248, 1.681605577468872, -0.5630142688751221], [0.46939048171043396, -0.13493700325489044, 0.4183376133441925, 0.33301979303359985, -1.3666664361953735, -0.6774281859397888, 0.49119874835014343, -0.30827561020851135, -0.10460866987705231, -1.0328290462493896, -0.3002808392047882, -0.9257081151008606, 0.48476657271385193, -0.3275255560874939, 1.7404096126556396, 0.027004774659872055, -0.7990036606788635, 0.9561495184898376, -0.5552051067352295, -0.6314935088157654, 0.14959943294525146, -0.9155777096748352, -1.696799397468567, -0.6891431212425232, -0.3081977665424347, -0.2293192595243454, -0.6578426361083984, -0.3074600100517273, 0.13579930365085602, -0.8974692821502686, -1.3654913902282715, 0.24693293869495392, 0.2751033902168274, -0.6374369263648987, -0.3631148338317871, -0.056542959064245224, 0.8553400039672852, -1.3606971502304077, 0.702639639377594, 1.3582273721694946, -0.7141046524047852, -0.6562561392784119, 1.6377403736114502, 1.3326056003570557, 1.48108971118927, -0.9645712375640869, -2.2055459022521973, 0.903613805770874, 0.17459452152252197, 0.11784124374389648, -1.7112696170806885, 0.06557680666446686, -0.061677560210227966, 0.5876821279525757, -0.378202885389328, 0.8692004084587097, -0.08004391193389893, 0.6970474720001221, 0.323507159948349, -1.5740339756011963, 1.5923280715942383, 1.0013577938079834, 1.3421030044555664, 2.589527130126953, 1.2732642889022827, 0.796552300453186, 0.010798143222928047, 0.8964117169380188, 1.368560552597046, 1.2012138366699219, -0.5548299551010132, 0.3478397727012634, 0.8636466264724731, 0.4001466929912567, 1.3572962284088135, -0.6469893455505371, -0.6079266667366028, 0.17817838490009308, -0.1298082023859024, 0.6842523813247681, -0.8007018566131592, -0.12610331177711487, -0.9326232075691223, -0.5702501535415649, 0.4412703216075897, 0.6376851201057434, 0.309280663728714, 0.7493012547492981, 0.2837693691253662, -1.211798071861267, -0.2396429181098938, 1.1354801654815674, -1.4344130754470825, 0.24673612415790558, -0.4389404356479645, 0.69986492395401, -0.8313659429550171, 0.08918638527393341, 1.1790989637374878, 0.6171891093254089, -1.1325592994689941, 1.0960946083068848, -0.7937864661216736, 0.23839066922664642, -1.1473034620285034, -2.628325939178467, 0.9653381705284119, -2.204589605331421, -0.5788723230361938, -0.6103760600090027, -0.44383084774017334, -1.2414919137954712, -0.009728321805596352, -0.612335741519928, 1.6228713989257812, 1.2438664436340332, 1.0777602195739746, 0.23501387238502502, 0.4547607898712158, -0.13473638892173767, 0.3218863308429718, 0.6054726839065552, 0.4259030818939209, 0.10824014246463776, -1.084929347038269, -0.2812238335609436, -0.9506188035011292, 0.6210853457450867, -0.7015277743339539], [0.36185604333877563, 1.0013917684555054, 0.45519840717315674, 1.2484965324401855, -0.05062847584486008, 2.093022346496582, 0.973781943321228, -1.683571696281433, -0.6866768598556519, -1.8960275650024414, -2.0282514095306396, 0.12627026438713074, 0.07999680191278458, -0.15180698037147522, -1.5160549879074097, 0.9126555323600769, 0.9628647565841675, 1.5350278615951538, -2.030804395675659, -0.6123432517051697, -1.4362027645111084, 0.17608316242694855, 0.543737530708313, 0.8265707492828369, 1.7278640270233154, 0.48225414752960205, -0.28007635474205017, 0.6387172937393188, -0.25278428196907043, -1.0431245565414429, 0.06498919427394867, 0.03725777566432953, 1.2193514108657837, 0.16681019961833954, 0.5965152382850647, -0.42661023139953613, -0.3285383880138397, 1.9638527631759644, 1.3954641819000244, 0.32613465189933777, 0.06051978841423988, 0.36334913969039917, -1.8295807838439941, 1.8203967809677124, -0.1881241351366043, -0.4204852283000946, 0.7839821577072144, -0.09027177095413208, 0.4530028998851776, -0.19569486379623413, 1.1109278202056885, 0.5707942247390747, 1.13469398021698, -0.18896134197711945, 1.1923402547836304, -1.1818691492080688, 1.5774468183517456, -0.05085282400250435, 1.7465375661849976, 0.32970497012138367, 1.0609620809555054, 0.6442084312438965, -1.9665329456329346, -0.5072928667068481, 0.5661496520042419, 0.8023700714111328, -0.5753823518753052, -2.623070240020752, -0.9408404231071472, 0.7099820971488953, -1.3410801887512207, 1.7142629623413086, -0.8800421357154846, -0.1577809602022171, -0.8604363799095154, -1.1373246908187866, 0.6846668720245361, -0.17615444958209991, 0.6597848534584045, -0.6458618640899658, 0.21949942409992218, 0.9318240880966187, 0.07648278772830963, -0.11617347598075867, -1.8273171186447144, 1.8072893619537354, -1.221597671508789, 1.455735445022583, -0.24489973485469818, -0.19104240834712982, -2.2241904735565186, -1.0282918214797974, -0.00046825301251374185, 0.37326040863990784, 0.3790701627731323, -0.14155863225460052, 0.17417265474796295, 0.4187367260456085, 1.3825652599334717, 0.8913907408714294, 0.5771659016609192, 1.3521934747695923, 1.560591459274292, 0.45580366253852844, -0.9756565690040588, -1.536468267440796, 0.2128085047006607, 0.7124202251434326, 1.3977652788162231, -1.2819089889526367, 0.9415320158004761, -0.18432128429412842, -0.7868037819862366, -1.9961445331573486, 0.8633108139038086, 0.5952785015106201, -1.637595295906067, 1.7643033266067505, 0.4625411331653595, -1.6763328313827515, 0.5713384747505188, 0.5210511088371277, 0.7847704291343689, -1.4087094068527222, -1.4154244661331177, -3.125809669494629, 0.23202377557754517, 1.4759622812271118, -0.670671820640564], [0.1939532458782196, -0.6055187582969666, -1.2535721063613892, 1.0192724466323853, -1.4921875, 0.40861040353775024, 0.018243230879306793, 0.352382630109787, 1.4231460094451904, 0.8050065040588379, 0.5584507584571838, -0.4656752049922943, -0.062027886509895325, 0.4298413395881653, -2.3603739738464355, 0.09621874243021011, 1.2431457042694092, 0.2917768061161041, -0.5944523215293884, 1.0111157894134521, 0.11518390476703644, -0.3932788670063019, 0.7694078683853149, -0.37866783142089844, -0.008239839226007462, 0.9504613876342773, 0.04539704695343971, 0.006170033011585474, 0.8210031390190125, -0.3576880693435669, 0.8622009754180908, -0.1999988853931427, -0.6701935529708862, -0.6045644879341125, -0.7022880911827087, 1.0695583820343018, 0.47004759311676025, -0.04874345660209656, -1.259737491607666, 2.0722012519836426, -0.8148826360702515, 0.6291185617446899, 0.5866398811340332, 0.27574920654296875, -0.06251630932092667, -0.7509951591491699, 0.7834660410881042, 1.4927287101745605, 0.8307005763053894, -0.44184187054634094, -0.0543428473174572, 0.8954474925994873, -0.07530566304922104, -1.310396432876587, -1.1013686656951904, 0.03728873282670975, 1.4357032775878906, 0.009611012414097786, 0.8509725332260132, 0.565854549407959, 0.9787296652793884, -3.2974541187286377, -0.17903545498847961, 0.21082356572151184, 0.961492657661438, -1.348274827003479, 0.2922801077365875, 1.2756552696228027, -1.1686958074569702, -0.6204463243484497, 1.2042354345321655, 0.16872459650039673, -0.6873179078102112, -0.0635877251625061, 0.8514414429664612, 0.15299375355243683, 1.2084459066390991, -1.396512508392334, -0.16703882813453674, 0.09792514145374298, -0.7625285983085632, 0.10601774603128433, -0.09768985211849213, -1.063287377357483, 0.5967947244644165, -0.7164196968078613, 1.4564988613128662, 0.5229350924491882, -0.34927108883857727, 0.3201392590999603, 0.24945589900016785, 0.5766752362251282, 0.31890782713890076, 0.017853043973445892, 0.533272385597229, -1.1496837139129639, -0.9155659675598145, 1.1995511054992676, 2.133612871170044, 0.8798002600669861, 0.7409507632255554, 1.1411532163619995, -0.009675947949290276, 0.9393430948257446, -0.3832806944847107, 1.714654564857483, 0.13157370686531067, 0.04064752906560898, 1.1179108619689941, -0.9287672638893127, 0.6218338012695312, 0.18500274419784546, -0.6615374088287354, -0.9708583950996399, -1.4647643566131592, -0.7157098054885864, 0.06363771855831146, 1.3240877389907837, -0.376129150390625, 0.1719561070203781, -0.2596113383769989, 0.780068039894104, 1.0747311115264893, 1.1356228590011597, -0.139562726020813, -0.2895050346851349, 1.4650039672851562, -0.7002396583557129, -1.3341633081436157], [0.6214403510093689, 0.6717830896377563, 0.23980994522571564, -0.7506510019302368, -0.4549306631088257, -0.21556846797466278, -0.5426860451698303, -1.3901376724243164, 0.8573126196861267, -0.7761176228523254, -1.2695095539093018, -0.0430343821644783, 0.3073074519634247, 0.5177255868911743, 0.782006561756134, 0.11974078416824341, -0.7955330610275269, 0.18597406148910522, 1.4428610801696777, -1.1961169242858887, 0.19816255569458008, 1.0292900800704956, -1.4041974544525146, 1.0240224599838257, -0.5151439905166626, 0.5985503792762756, -0.08533813804388046, 0.1863929182291031, -0.6848922967910767, 2.150787115097046, 1.1824835538864136, -0.09784211218357086, 0.21422086656093597, 0.6812556385993958, 0.0569339357316494, -0.4974968433380127, -1.0304197072982788, -0.9322220683097839, -0.531196117401123, -0.28497523069381714, 0.5419960021972656, 1.301592230796814, 0.78346848487854, -0.19231845438480377, -0.10505655407905579, -2.0432956218719482, 0.4278121292591095, 1.3648074865341187, 0.518940806388855, 0.0713738352060318, -0.6238238215446472, -0.524867832660675, 0.1466871052980423, -0.330637663602829, -0.6856294870376587, -1.0750937461853027, -1.0405901670455933, -0.2634894549846649, 0.6676499247550964, -1.275964379310608, 0.3686428368091583, -1.3499623537063599, -2.0272581577301025, 0.3115633428096771, -0.3013884425163269, 2.0829484462738037, -1.1119365692138672, 0.6499421000480652, 1.5932703018188477, 1.6909198760986328, 0.0537668839097023, -1.6411211490631104, -0.5102522373199463, -0.08958066999912262, -0.30281952023506165, 0.9619770050048828, 0.8582705855369568, 2.103484630584717, -0.6457415223121643, -1.9188464879989624, 0.05669991672039032, -1.8730738162994385, 0.3714422881603241, -0.690218985080719, -0.733959436416626, -0.957611620426178, 1.0858060121536255, 0.9408835172653198, -0.6339224576950073, -0.4890071451663971, 1.3463305234909058, -0.5387159585952759, -1.0391271114349365, -1.8800026178359985, 0.24780263006687164, 0.9946171045303345, -1.9006811380386353, -0.06376419961452484, -1.122406244277954, 0.3844965398311615, 1.494979739189148, 0.3119315505027771, 0.4133089482784271, -1.0679951906204224, 0.7147030830383301, 0.5951359868049622, -0.7253110408782959, -0.5246185660362244, 0.8619046211242676, -0.4033564031124115, 0.9053887128829956, 2.5490145683288574, 0.5784786939620972, 1.2900614738464355, 0.6406950354576111, 0.3667520582675934, -1.4890785217285156, 1.249287486076355, -0.2023031860589981, 0.22703349590301514, 1.2422834634780884, -1.4770845174789429, -0.9335165619850159, 0.3107382655143738, -0.07231300324201584, -1.140950083732605, -0.6247334480285645, 0.22749724984169006, -0.3712107837200165], [0.603081226348877, 0.4594651758670807, -1.5235037803649902, -1.8959057331085205, 0.7453246712684631, -0.6311113834381104, 0.48867523670196533, -0.616532027721405, -0.08166321367025375, 0.3011563718318939, -0.19481252133846283, 0.984275221824646, 1.1723865270614624, -0.5638916492462158, 0.6267274022102356, 0.4440518319606781, -1.1429848670959473, -0.2282775342464447, 0.9769620895385742, 0.6454269886016846, 0.4936676025390625, 0.90724116563797, 0.7230406999588013, 0.06404035538434982, -0.12030874937772751, -0.09823259711265564, -1.281498670578003, -1.0715538263320923, 1.0010229349136353, -0.1423400640487671, 1.4238618612289429, 0.10679712146520615, 0.07562461495399475, -0.2905799150466919, 0.8438090085983276, 0.3016452491283417, -0.8818870186805725, 1.9795575141906738, 0.8237636685371399, 0.01025258470326662, -1.2550832033157349, 1.4546325206756592, 0.5350990891456604, 0.12291907519102097, 0.5515251755714417, 0.2268400639295578, 0.578686535358429, 0.12979315221309662, 1.076131820678711, 1.2533478736877441, -0.05134144052863121, 0.12467122077941895, -0.4786588251590729, 0.5470255613327026, 0.8520046472549438, 0.9650270938873291, -0.10283070802688599, 1.394932508468628, -0.08532660454511642, 0.8564659357070923, 0.038905564695596695, 1.039318323135376, -0.3543209135532379, -0.6631119251251221, -0.6678072214126587, -0.527717649936676, 1.0742942094802856, -2.1573994159698486, -0.6992531418800354, -1.7447686195373535, -0.4570470452308655, -1.7095463275909424, 0.8505438566207886, 0.10632547736167908, 0.6499340534210205, 0.041331470012664795, -0.4261241853237152, 0.09101761132478714, -0.05478772148489952, 0.15147972106933594, 0.42319902777671814, 0.046074096113443375, -0.5932998061180115, -0.9778568148612976, -0.5065262913703918, 0.056727245450019836, -1.2198476791381836, -0.9565724730491638, -0.5146059989929199, -1.98021399974823, -0.23872913420200348, 0.39222028851509094, -0.13260401785373688, 1.2727218866348267, 0.3988213539123535, -1.2825459241867065, -0.4910794794559479, -1.6718854904174805, 0.5582765340805054, -1.708718180656433, -0.42433875799179077, -0.64864581823349, 0.5853353142738342, -0.9045767784118652, -0.9801905155181885, 1.3466484546661377, 1.1461118459701538, 1.0594139099121094, 0.2149972766637802, 0.7463553547859192, -2.1871860027313232, 0.5977638959884644, -0.7592080235481262, 0.58284991979599, -0.0786062479019165, 0.6103801727294922, -0.4884781837463379, -0.15698890388011932, 0.9799625277519226, -1.0334612131118774, 0.6413187980651855, -1.1534568071365356, 0.39999598264694214, 0.5634135603904724, -0.018134448677301407, 0.6122766733169556, -0.12274230271577835, -1.4177459478378296, -0.2574929893016815], [0.11756377667188644, 0.7732327580451965, 0.06272420287132263, 0.5199599862098694, 0.9937430620193481, -0.8748707175254822, 0.32281026244163513, -1.9243632555007935, -0.344544380903244, -0.38957253098487854, 2.3588078022003174, 0.05086006969213486, 1.0981494188308716, 0.2860477864742279, -0.4134107828140259, -1.4431178569793701, -1.1694971323013306, 1.2363264560699463, -0.8488003611564636, 1.5900479555130005, 1.6066694259643555, 1.8342089653015137, -2.417250394821167, -1.22242271900177, -0.11778773367404938, 1.3271938562393188, 0.3590141534805298, -0.7427203059196472, 1.4202613830566406, 0.5564141273498535, -0.7813589572906494, 0.6302211880683899, -0.3733769953250885, 1.681706190109253, -0.23470836877822876, 0.0130483228713274, -1.035150170326233, 0.2688954174518585, 1.5704121589660645, 0.26431405544281006, -0.2281918227672577, 0.0925537571310997, -2.8400661945343018, 0.21173736453056335, -0.295090913772583, 0.6342539191246033, -1.1910338401794434, -0.6570596694946289, -1.054129958152771, -0.2532261312007904, -0.3630080223083496, -0.536647379398346, 0.8410665392875671, 0.3469441831111908, 0.6008391380310059, -1.4104984998703003, -0.887927770614624, 1.9640040397644043, 0.8978624939918518, -0.24435774981975555, 0.6836757659912109, 0.6233574748039246, 1.030081868171692, -1.6978507041931152, -0.1398993879556656, 0.05696775019168854, 0.05918070301413536, 1.1279178857803345, -0.28321006894111633, 1.1873689889907837, -0.7698922157287598, 0.6623507738113403, 0.3169825077056885, 1.5681108236312866, 0.009858702309429646, 1.8809422254562378, 0.870305061340332, 0.2950596511363983, 0.058635320514440536, -0.2479843944311142, -1.0613030195236206, -0.44936609268188477, -0.24580413103103638, -0.05291949585080147, -0.413632869720459, 0.23910483717918396, -2.346968173980713, -0.4974655508995056, -0.5745476484298706, -0.6970394849777222, -2.5650012493133545, -0.8799576759338379, -0.49330905079841614, -1.645783543586731, 0.6590819954872131, 1.7158671617507935, 1.310422420501709, 0.8777603507041931, -0.05626903474330902, 0.7560352087020874, -0.8448346257209778, 0.22333946824073792, 0.7702952027320862, 1.547988772392273, -1.3369680643081665, 2.4821414947509766, -0.9643633961677551, -0.12071409821510315, 0.26295772194862366, 1.3581783771514893, -0.22817420959472656, 0.4538629651069641, 0.1701505482196808, 0.7491385340690613, -0.22938790917396545, -0.22197329998016357, -0.13753502070903778, -0.6489784717559814, 1.106520175933838, -0.3452308773994446, 0.5652908682823181, -1.738067865371704, 0.2667241096496582, 1.0060287714004517, 0.9653447866439819, 0.895605742931366, -1.2162086963653564, 0.744390070438385, 1.1069172620773315], [-0.5209303498268127, -0.030207673087716103, 1.8620429039001465, -0.8958550691604614, -1.7518036365509033, 0.34766289591789246, 0.1420331448316574, -0.026272200047969818, -0.6626594066619873, -0.8244186043739319, 0.10665995627641678, -0.022986382246017456, -1.4121015071868896, 0.5977663993835449, 0.7526816129684448, 0.44276532530784607, 1.125537395477295, 0.19815437495708466, -0.18453767895698547, 0.37180575728416443, -1.7285679578781128, 0.3487984836101532, 0.6305879950523376, 2.6986734867095947, 1.1563341617584229, 0.5784435868263245, -0.3068682849407196, -1.32172691822052, 0.34342703223228455, 0.2663159966468811, -1.3799246549606323, 0.7117529511451721, 0.8523127436637878, -1.131657600402832, -1.0570058822631836, 0.16276609897613525, -1.3202203512191772, -0.295844703912735, 2.236158609390259, 0.9249026775360107, -1.0606380701065063, 0.27411872148513794, 0.7490676045417786, -0.6294441223144531, 1.0666145086288452, 0.8694367408752441, -0.24395765364170074, 0.3115466237068176, -1.0382963418960571, 0.6057623624801636, -0.4955254793167114, -1.6082077026367188, -2.1028523445129395, 2.3694334030151367, -1.6351338624954224, 0.36364129185676575, 0.6306215524673462, 0.13920137286186218, 0.7948870658874512, 1.321164608001709, 0.8884861469268799, 0.7813417315483093, 0.44702160358428955, 0.8670487403869629, -1.0717934370040894, 0.476722776889801, -1.0623615980148315, -2.814342498779297, 1.4753034114837646, -0.8065089583396912, -1.3679916858673096, -0.7164723873138428, 0.04346664622426033, 0.4581928551197052, 0.37991711497306824, -0.9569526314735413, -0.5876396894454956, -1.088987112045288, -0.6851450800895691, 0.5611231327056885, -1.8315808773040771, 1.5587259531021118, -0.6387643814086914, 0.18483038246631622, -0.35624992847442627, 0.6136640310287476, -0.37560421228408813, 1.5466660261154175, -0.5701030492782593, 0.8348351716995239, -1.0620534420013428, -0.9431265592575073, 1.1327069997787476, -1.0035369396209717, -0.07531453669071198, -0.4335353672504425, 0.7946271896362305, -0.4137212336063385, 1.1125779151916504, -0.47043412923812866, 1.3796144723892212, -0.373169481754303, -0.3288230895996094, 0.39074280858039856, 0.4358149766921997, 0.15739905834197998, -0.7986113429069519, 0.660629153251648, -1.136201024055481, 0.7536166906356812, -0.03251718729734421, -1.6242016553878784, -0.5558960437774658, 0.02304787188768387, -1.5785619020462036, -0.3284716308116913, -0.3449796140193939, -1.5543831586837769, 0.04379719868302345, -0.6642754673957825, -1.0078973770141602, -0.43585604429244995, 1.1845883131027222, -0.014879842288792133, -0.1892644762992859, -0.9059218764305115, -1.0402356386184692, -0.12879785895347595, -1.0653481483459473], [-1.3769683837890625, 0.918996274471283, 0.319120317697525, 0.6810352802276611, 1.1746480464935303, -0.4104318916797638, 0.32279226183891296, 0.8856363892555237, -0.7863155007362366, -0.5497336387634277, -0.6904085278511047, -0.7016988396644592, 0.6113445162773132, 0.41182246804237366, 1.1546754837036133, -0.6262204647064209, -1.6806097030639648, -1.7584224939346313, 0.9875820875167847, -0.6837645769119263, 0.7438217401504517, -0.3255016803741455, 0.43287432193756104, -0.33503463864326477, 1.6276980638504028, 0.1524665653705597, -0.8964793682098389, -0.6130474209785461, 0.23285821080207825, -0.9749749898910522, -0.6243535280227661, 0.2882451117038727, -1.9700078964233398, 0.31558310985565186, 1.8287158012390137, -0.7554413676261902, -0.032967351377010345, 0.6915014982223511, 0.642978847026825, 0.7039007544517517, 0.7924417853355408, -0.29780516028404236, -1.1799911260604858, 1.0050312280654907, 1.387315034866333, 0.8700094819068909, 0.9325361847877502, -1.4289714097976685, -1.437252163887024, -0.13308484852313995, -2.552696943283081, -0.641115665435791, -1.72727632522583, -2.062936544418335, 0.00888211652636528, 0.3687010705471039, -0.6057354211807251, 0.7076168656349182, 1.103617787361145, 0.9224303960800171, -1.3097741603851318, -0.2793359160423279, -0.4971660375595093, -0.21629738807678223, -0.7936683297157288, -0.17311044037342072, 0.22206220030784607, -0.6656325459480286, -1.4301235675811768, -3.1225454807281494, -0.20742373168468475, -0.5585169196128845, -0.7422334551811218, -0.522503137588501, 0.37489303946495056, 0.9507657289505005, -0.13617199659347534, 1.469801664352417, 0.32097071409225464, -0.9442238807678223, -0.26950162649154663, -1.0252063274383545, 0.5770036578178406, -0.6966812610626221, 0.00437878118827939, -0.7381495237350464, 0.7971128821372986, -0.8159014582633972, 0.4580710828304291, 0.017424894496798515, -0.1809072047472, -0.13292217254638672, 0.09804430603981018, -0.5934122800827026, -0.2208067774772644, 0.3099871873855591, -0.22578634321689606, 0.37456750869750977, 1.0446223020553589, 0.5823568105697632, 1.1916128396987915, -1.0964404344558716, -0.3305916488170624, 0.9906922578811646, 0.5968748927116394, -1.1123088598251343, 1.8126481771469116, -0.22194430232048035, -0.03654762730002403, 1.424608826637268, 1.166764736175537, -1.5759834051132202, -0.6856565475463867, 0.5301434993743896, -0.44178342819213867, 0.034444794058799744, -0.26191022992134094, 2.615119218826294, -0.12903346121311188, 2.520789384841919, 2.8451433181762695, -1.6609017848968506, -0.9130668044090271, -0.7816276550292969, 0.6167691946029663, 0.7187105417251587, -0.5141309499740601, -0.6697009801864624, -0.39449039101600647], [1.222536325454712, 1.5333771705627441, 0.49663054943084717, 1.0047783851623535, -0.5186890959739685, 0.08252108842134476, 1.2443501949310303, -1.1686946153640747, 0.548947274684906, 0.5286832451820374, 1.5707639455795288, -0.8771560788154602, 0.24696442484855652, 1.2240492105484009, 1.3094971179962158, 0.9259516596794128, -2.29378342628479, 0.9681722521781921, 2.333224296569824, -1.15886652469635, 0.29138290882110596, -1.3946946859359741, -0.8002336621284485, -0.2074284553527832, 0.6897275447845459, -1.8281781673431396, 1.383880615234375, -0.6188716292381287, 0.6598458886146545, -0.5079156756401062, 0.2102355808019638, 0.2861883342266083, -1.6237342357635498, 2.2434892654418945, -0.45141392946243286, 0.3908595144748688, 0.14981268346309662, -0.8833467960357666, -1.8470746278762817, 0.8030042052268982, 1.0620779991149902, 1.0645560026168823, 0.33909329771995544, 0.6405824422836304, 0.37639573216438293, 0.35352978110313416, 1.4953855276107788, -0.09898115694522858, 0.6985318064689636, 1.3170052766799927, -0.3651719391345978, -0.6262595057487488, -0.5296647548675537, -2.9427645206451416, 2.2145416736602783, 1.9881352186203003, -1.0532116889953613, 0.9454832673072815, 0.07522891461849213, 0.8451297283172607, -0.28364455699920654, 0.518416702747345, 0.876854658126831, 0.8823681473731995, 1.2652699947357178, -0.8071956634521484, -0.567588746547699, -0.8333717584609985, -1.0879446268081665, 0.3544701635837555, 0.48896515369415283, -1.5715123414993286, -0.8184590339660645, -0.5034276843070984, 1.4130620956420898, 0.08941256254911423, 1.5929747819900513, 1.706656813621521, -0.3603898882865906, -0.6944833993911743, 1.2887521982192993, -1.2077879905700684, -1.0548368692398071, 0.842113733291626, -0.16192574799060822, -1.300103783607483, 1.6498578786849976, 0.27750489115715027, 1.0740704536437988, -1.4260376691818237, 0.02795279771089554, -0.8662141561508179, -2.3038814067840576, -0.14374475181102753, 0.5122799873352051, 1.7363874912261963, -0.395261287689209, -0.70322585105896, -0.5499417185783386, -0.37198111414909363, -0.9756627082824707, -1.302551031112671, 0.8749135732650757, 1.5506694316864014, -0.5485239624977112, -0.2083158940076828, -0.6403155326843262, 0.011651252396404743, 1.3113914728164673, 0.4716877341270447, -0.23302273452281952, -0.7949926257133484, 0.5026839971542358, -0.7785106301307678, -0.08401749283075333, -1.2254153490066528, 0.4327540397644043, -0.36962977051734924, -1.7944711446762085, 0.1987973302602768, 0.5097524523735046, -0.35876548290252686, 0.6653438806533813, 0.7280247807502747, 0.33548909425735474, 0.4891272783279419, 2.501246690750122, -0.12401342391967773, 0.378441721200943], [-0.737542450428009, 0.7703521847724915, -1.5288513898849487, -0.6175204515457153, 1.2933502197265625, 0.3140406608581543, -0.1263548731803894, -0.21139656007289886, -1.0060853958129883, 1.0891590118408203, -0.30473893880844116, -0.23525705933570862, 0.10644523799419403, -0.03112032637000084, -1.963708519935608, -1.087359070777893, -0.3783109486103058, -0.005376275163143873, 0.09286046028137207, 1.1521574258804321, 1.3363401889801025, 0.3989546597003937, 0.2587217688560486, -1.2513628005981445, 0.10614793002605438, -0.9554066061973572, -1.6431920528411865, -0.3917161822319031, -0.5572349429130554, -1.023831844329834, -0.5213506817817688, -0.03839555010199547, -0.5374690294265747, 2.2949378490448, 1.8876439332962036, -0.17051883041858673, -0.32207903265953064, 0.1459961235523224, 0.7488977909088135, -0.805425226688385, -0.2075209617614746, 1.5519556999206543, -0.46002396941185, -0.378056138753891, -1.0979511737823486, 0.03144096955657005, 0.9304996132850647, 0.6672418713569641, -0.7386713624000549, -0.2990347743034363, -0.12373699247837067, 0.6816425919532776, 0.607247531414032, -0.052457552403211594, -0.9991798400878906, 0.9839015603065491, 0.3676571249961853, -0.8931958079338074, -0.6952368021011353, -0.8173657655715942, -0.4036458134651184, -0.19207580387592316, -1.0943129062652588, 0.4451737403869629, -0.2681773006916046, -0.7087646126747131, -0.4896913766860962, 2.4676616191864014, 3.044428825378418, -1.9367272853851318, 0.020278621464967728, -0.6990362405776978, -1.9193394184112549, -0.8100659847259521, 1.563551902770996, -2.3589563369750977, -0.35500872135162354, -1.5005396604537964, -0.4775964319705963, -1.1579948663711548, -1.0204957723617554, 0.5441898703575134, 0.4390884041786194, 0.16755332052707672, -0.08489131927490234, -1.8473411798477173, 0.9646921157836914, -1.0376298427581787, 0.04774845391511917, -0.3829159140586853, -0.10214018076658249, 2.0620765686035156, -0.4703727662563324, 1.164575219154358, -0.041653670370578766, -0.07866113632917404, 0.012271753512322903, -0.5027778744697571, 1.91411292552948, 0.07171561568975449, -0.3836590349674225, 0.29288163781166077, -0.2564721405506134, 0.15003354847431183, 0.5602877140045166, 1.7138514518737793, 0.2882026433944702, 1.2256869077682495, -1.1919318437576294, -0.21670669317245483, 0.8098087310791016, -0.44834309816360474, -0.7882698178291321, -0.3557995557785034, 0.27314817905426025, 1.236159324645996, 2.3548669815063477, -0.7974780797958374, -1.1645039319992065, -1.339766025543213, 0.8177034258842468, 1.387546420097351, 1.340881586074829, -0.828386664390564, 1.4156441688537598, -0.8739281892776489, -0.43790197372436523, -0.11222901940345764, -0.5501825213432312], [-0.7093361020088196, -0.09738674014806747, -1.0033482313156128, 0.8489775061607361, -1.9287251234054565, -0.9127740859985352, -0.2991575300693512, -0.44394615292549133, -0.33993545174598694, -0.9061029553413391, 0.19443635642528534, -0.5111767649650574, -1.4980199337005615, -1.4695333242416382, -1.5297365188598633, 0.144150048494339, -0.35628873109817505, 0.10733238607645035, -0.1267162561416626, 0.13255088031291962, 0.5167080163955688, -1.2123050689697266, 0.41832661628723145, -0.19840657711029053, 0.6320281028747559, -0.8199357986450195, 0.9316419959068298, -1.0622713565826416, -0.6958160996437073, 0.48233845829963684, 0.6340932250022888, 0.8555338382720947, 0.35214942693710327, -1.4649165868759155, -0.5895764827728271, -0.21032245457172394, 0.10196436941623688, -0.7807988524436951, -1.3733731508255005, 1.0330359935760498, 1.1201201677322388, -1.4550082683563232, 0.03760760277509689, 0.11320523172616959, -0.11698447912931442, 0.4700061082839966, 1.4808825254440308, -2.199500322341919, 0.5847747921943665, -0.6945905685424805, 1.0247095823287964, 0.03765484690666199, -0.9492926001548767, -0.030289093032479286, 1.331249475479126, -0.2166042923927307, 0.10579738765954971, 0.7360370755195618, -0.8016372919082642, 0.657234251499176, 0.6079548597335815, 0.5300265550613403, -0.17675605416297913, -0.37450262904167175, 1.4223827123641968, -1.8635905981063843, -1.7021077871322632, -0.01006905734539032, -0.7039520740509033, -2.2934927940368652, -1.0975425243377686, -0.14244772493839264, 0.7755759358406067, 0.8871222734451294, -2.17411208152771, -0.6705016493797302, 0.16181038320064545, -0.09856612235307693, 0.04260785132646561, 0.13011744618415833, -0.6667867302894592, 1.0466240644454956, -1.661049246788025, -0.13266947865486145, 0.8440675735473633, -0.03768286854028702, 2.80627179145813, 0.31746411323547363, -0.8704925179481506, -0.3250459134578705, -0.8053478002548218, -2.1784629821777344, -1.6030668020248413, 0.09664686024188995, -0.8121252059936523, -0.12869466841220856, 0.3308047354221344, 0.9402276873588562, 0.6049027442932129, 0.5891030430793762, -0.17959639430046082, -0.7386468648910522, 1.058305025100708, 1.6532193422317505, -0.1626337170600891, 0.139870747923851, -0.1133275032043457, 0.8157592415809631, 2.408626079559326, 1.6355006694793701, -0.13091057538986206, -0.2278113067150116, -0.5136097073554993, -1.5251907110214233, -0.11636460572481155, 0.2610115110874176, 1.4975930452346802, 0.6803823113441467, -0.09024575352668762, 2.3201897144317627, -0.3332735598087311, -1.6679710149765015, 0.814085066318512, 0.12145832926034927, 0.3936226963996887, -0.18236027657985687, -0.5458554625511169, 1.1313294172286987, -0.8289855122566223], [0.5949336886405945, -0.39685460925102234, 0.21089573204517365, -1.6164498329162598, -0.8627327084541321, -1.5896127223968506, 0.029544709250330925, 0.46139079332351685, -0.551600456237793, 0.7590013146400452, 0.6605903506278992, 1.7166436910629272, -0.13862508535385132, 0.4996589422225952, 0.49282142519950867, -0.06390151381492615, -1.0065685510635376, 0.7942512035369873, -1.01887047290802, -0.6126475930213928, -0.12962841987609863, -2.3256173133850098, 0.014579949900507927, 0.9382597208023071, -0.8008862137794495, 0.7032593488693237, 0.1119861900806427, -1.393141746520996, -0.34266623854637146, 1.000030279159546, 0.7775006890296936, -0.12094616889953613, -0.7219783067703247, -0.54423987865448, 0.03626933693885803, -0.5146672129631042, 1.7845170497894287, 0.38184505701065063, 1.87275230884552, 0.32743269205093384, 0.5369418263435364, 0.0085489172488451, 0.15673185884952545, 0.8355717658996582, -0.47715505957603455, 1.33510422706604, -0.7465050220489502, -0.4486258625984192, 0.3568308353424072, -0.10947199165821075, 0.4660896360874176, 0.33700233697891235, 0.38206419348716736, 0.47210103273391724, -1.7927473783493042, -1.4790692329406738, 1.2196557521820068, 0.6738883852958679, 0.0008582055452279747, 1.908361554145813, 1.872067928314209, 2.1892242431640625, -0.29323580861091614, 0.7886675000190735, -0.9572892785072327, 0.2589642405509949, 0.629112958908081, 1.0498640537261963, -0.24340073764324188, -1.0988579988479614, 0.9002113938331604, 0.6014793515205383, 0.2500222325325012, -0.9171328544616699, -0.8008238077163696, 0.027615711092948914, -1.425520896911621, -0.8538901209831238, -0.10911647230386734, -0.37438273429870605, 0.5205385088920593, 0.5388955473899841, -0.27747830748558044, -0.2407422512769699, -0.4945378601551056, -0.3214556872844696, -0.5811124444007874, 1.160502314567566, 1.7914618253707886, 1.1875622272491455, 0.08272714912891388, -0.17589014768600464, -0.6032319068908691, -2.3188750743865967, -0.37092164158821106, -0.5936365127563477, -0.10639721155166626, 0.9158368110656738, -0.8860163688659668, 0.8686660528182983, -1.3268728256225586, 0.4126409590244293, -0.2825194299221039, -0.7922468185424805, 1.0454013347625732, -0.4235132336616516, -1.791107177734375, 0.14677882194519043, -2.0001540184020996, 0.45811307430267334, -0.2307257503271103, 0.06942154467105865, 0.04706260561943054, 1.4116535186767578, -0.0618518628180027, -0.11443229764699936, -0.5856080055236816, -0.33224377036094666, 0.48513278365135193, 0.9100220799446106, 1.3723963499069214, -0.494643896818161, 1.0980145931243896, 0.31916120648384094, 0.19551981985569, 1.4468027353286743, 0.6343423128128052, -0.02306395210325718, -1.4526475667953491], [1.4469327926635742, -1.5676027536392212, -0.09355242550373077, 0.5250779390335083, -0.9395994544029236, 0.6747859120368958, 0.18670925498008728, 0.9113808870315552, -0.10027406364679337, 0.2996182143688202, 0.02489224076271057, 0.5809260606765747, -1.9457018375396729, 0.0826127901673317, 0.2681983411312103, -0.19971077144145966, -0.40938782691955566, 0.9972352981567383, -1.058870553970337, 0.6956912279129028, 1.290596604347229, -0.5548042058944702, -2.137526512145996, -2.652982473373413, -1.9537142515182495, -0.5999363660812378, 0.7750393152236938, 1.4785081148147583, -0.7687599658966064, 0.971610426902771, -1.6980570554733276, 0.18802468478679657, 0.29583799839019775, 0.10390178114175797, 0.34511798620224, -1.2841984033584595, 0.5238896608352661, 0.6709606051445007, 0.4986412823200226, -2.4356424808502197, 0.6910125613212585, 1.0852380990982056, -1.7916313409805298, 1.120124340057373, -0.5345142483711243, 1.5809935331344604, 1.2832467555999756, 1.0762414932250977, 0.9397984743118286, 1.2505050897598267, 0.4295697510242462, 0.07155856490135193, 0.6260314583778381, -0.6289272904396057, 1.0989012718200684, -1.2588857412338257, 0.31324708461761475, 1.9611839056015015, -1.5963062047958374, -0.002293642610311508, -0.09914370626211166, 0.6273489594459534, 0.49392932653427124, 0.8218505382537842, 0.40825507044792175, 0.09398696571588516, -0.3371959328651428, 0.6756138801574707, 0.08277267217636108, 1.051424503326416, -0.4552857577800751, 0.7640295028686523, -1.3547719717025757, -0.42372986674308777, -1.4171016216278076, -0.8302933573722839, -1.7563010454177856, -1.2925008535385132, 1.9526506662368774, 0.6670745611190796, -1.530484676361084, 1.9431824684143066, 0.692689836025238, 0.13035371899604797, 2.496729612350464, 1.6071325540542603, -0.09142503142356873, 1.4830119609832764, -0.30412009358406067, -1.0866060256958008, -2.084036111831665, -2.394252300262451, 0.10402336716651917, 1.656230092048645, -0.9088626503944397, -0.32848507165908813, 2.5261621475219727, 0.5236690640449524, -0.26757240295410156, -2.906115770339966, 1.20132315158844, 2.5598833560943604, 1.1176719665527344, -0.65854412317276, -0.08896184712648392, -0.5469105839729309, 0.7080478668212891, 0.4071654975414276, 0.25544771552085876, -0.49106815457344055, -2.1258864402770996, 0.03245074674487114, -0.6016477346420288, -0.23402836918830872, 2.5768861770629883, 0.7565858364105225, 1.4228085279464722, -0.18659056723117828, 1.181701421737671, -0.35786017775535583, -0.42065635323524475, -0.16404365003108978, 1.6549444198608398, -0.6268126964569092, -0.3383519649505615, -0.03842083364725113, 0.7336037755012512, 0.9948509931564331, -0.480357825756073], [0.6712921261787415, -1.5997461080551147, 0.8554091453552246, -0.5962657332420349, -0.6457962393760681, -0.7281070947647095, -0.7570313811302185, -0.003527304157614708, 0.2022378295660019, 0.11460072547197342, 1.3991512060165405, -1.0768343210220337, 0.7766494750976562, 0.013035465963184834, 0.2750262916088104, 0.3963240683078766, 0.29408156871795654, -0.3219048082828522, -0.5114267468452454, 2.664583206176758, -0.4759717583656311, 0.3110373914241791, 1.0783183574676514, -1.1056360006332397, -0.05859184265136719, -0.5565578937530518, 0.4646156430244446, 1.7612559795379639, 0.8964202404022217, -1.1909027099609375, -1.9805223941802979, 0.7824692130088806, -0.07147327810525894, 1.7177438735961914, 0.29737934470176697, -0.42548713088035583, 1.0393476486206055, -0.9387934803962708, 1.5424357652664185, -1.6117560863494873, -0.47111421823501587, -1.3009555339813232, 0.07446987181901932, 0.15529316663742065, -0.26727327704429626, -0.32055211067199707, 0.0063146427273750305, 1.0091595649719238, 0.5169834494590759, -0.2673071324825287, 0.5539647340774536, 1.039482831954956, -0.9478712677955627, -0.8970664143562317, 1.6106617450714111, -0.7531170845031738, -0.1958266943693161, -0.3889445662498474, -1.0068943500518799, -0.3752065598964691, 0.9374274015426636, 0.9580760598182678, 0.4908422827720642, 1.0628751516342163, 0.9478567838668823, 1.1865698099136353, -0.8729209899902344, 0.7085918188095093, 0.2130131721496582, 0.16370439529418945, -0.6473509073257446, -0.46042630076408386, -0.20393525063991547, -0.09807061403989792, -0.1610221564769745, -0.13724154233932495, -0.2976825535297394, 0.3483808934688568, 1.3167592287063599, 0.5538492798805237, -0.11019735038280487, 0.3811616897583008, -0.518680989742279, -0.24429816007614136, -0.6709662675857544, 1.069942831993103, 0.8607327938079834, -1.052588939666748, -1.0214407444000244, 0.1251237690448761, 0.6031842231750488, -2.507371425628662, -0.4811408519744873, -1.2973812818527222, -0.8441774249076843, 1.4846152067184448, -1.1202560663223267, 0.36312147974967957, -0.6590919494628906, -0.36091190576553345, 1.1638554334640503, 0.6632539629936218, -0.33586108684539795, -1.5980817079544067, -1.0927629470825195, 1.1538125276565552, -0.032219648361206055, 0.7559599280357361, 1.345165729522705, 0.0852104127407074, -0.24280542135238647, 0.848179042339325, 0.4299221336841583, 0.18907777965068817, -0.17538955807685852, 0.2024277299642563, -0.4916898310184479, 0.09344430267810822, 0.049812428653240204, -1.4741181135177612, -0.8832705020904541, -0.6557594537734985, -1.7784383296966553, 0.8643503189086914, -1.5547236204147339, -0.23383377492427826, 1.7624223232269287, 0.16835318505764008, 0.4969841241836548], [-1.195343255996704, -0.8928303122520447, -0.19787003099918365, -0.631874680519104, 0.17389754951000214, 0.11874938756227493, -1.242305040359497, -0.5642113089561462, 0.5523450374603271, 0.19739794731140137, -0.15513749420642853, -0.5778460502624512, -0.13360965251922607, 0.32125774025917053, -1.3246625661849976, -0.04400394484400749, -0.47041985392570496, -2.1360979080200195, -0.9605150818824768, -1.1970340013504028, -1.5502897500991821, 0.5498937964439392, 1.430189847946167, -0.04638756811618805, 0.9246535897254944, -0.09134043008089066, 0.8976706266403198, 1.2614463567733765, 1.0203559398651123, 0.60477614402771, -0.08184245973825455, -2.4395859241485596, -0.6816452741622925, 1.0759108066558838, 0.16458027064800262, 0.6629100441932678, -1.0578535795211792, -1.672838568687439, -0.38782790303230286, -0.7419552206993103, -0.226490318775177, -1.2057738304138184, 0.598243236541748, -0.5694519281387329, -1.8443653583526611, 0.5591245293617249, -1.8760998249053955, -0.0759940892457962, 0.3349558115005493, -1.2782448530197144, 0.34596291184425354, 0.7263817191123962, 0.790962278842926, 0.23445142805576324, 1.7707422971725464, 0.3766941428184509, -3.342830181121826, 0.00015051118680275977, -0.280767023563385, -0.2415897250175476, -0.8516072630882263, -0.07773848623037338, -0.09748492389917374, -0.475966215133667, -1.9455538988113403, -0.5861462950706482, -0.07132584601640701, 0.6036890149116516, 1.0807262659072876, 0.5222398638725281, 1.7357209920883179, 0.3324262499809265, -0.9320440292358398, 1.4082143306732178, 0.009906549006700516, 0.09057298302650452, 0.6319896578788757, -1.1781467199325562, 1.5902341604232788, 0.9112378358840942, -1.5923658609390259, -0.9243109822273254, 1.4046231508255005, -0.08077601343393326, 0.006439995486289263, -1.0022718906402588, -0.5803959369659424, -0.9448546171188354, -0.977031946182251, -0.14570991694927216, -1.7385977506637573, 0.49960485100746155, 0.16051559150218964, -0.6560296416282654, 1.660529613494873, -0.4635416567325592, -0.40599238872528076, -0.04568612575531006, -1.8361239433288574, -0.05938758701086044, -0.2121383100748062, 1.3593592643737793, -0.3880903720855713, 0.24584098160266876, -0.4796653091907501, -1.4740992784500122, 1.5127414464950562, -0.5897160768508911, -0.18774966895580292, 0.9677688479423523, 1.1518696546554565, -0.6702497005462646, -0.00954299047589302, -1.2146263122558594, 0.4291849136352539, 0.07839780300855637, -0.8818682432174683, 0.40733814239501953, -0.7172014117240906, 0.17084328830242157, 0.6829966306686401, 0.7516506314277649, -2.6314809322357178, -0.2685958743095398, -0.2857404053211212, 1.9142323732376099, -0.9935181736946106, 1.9625310897827148, 2.304550886154175], [0.9460230469703674, -0.8818212747573853, -0.7284239530563354, 0.15846727788448334, 0.9468255043029785, -0.6914898753166199, -0.5336689352989197, -0.1760312020778656, -2.0968735218048096, -1.1899549961090088, -1.2303311824798584, -1.1602983474731445, 2.734752893447876, 0.0483880452811718, 0.09431104362010956, 1.5393191576004028, -0.045159220695495605, 0.5104946494102478, 0.21455781161785126, -0.29050084948539734, -1.1489793062210083, 1.256437063217163, 0.21711157262325287, 1.1966928243637085, -2.453909158706665, 0.3603557050228119, 0.691608190536499, 0.545106828212738, -0.9959744811058044, -0.009093431755900383, 0.23841750621795654, 0.7524499893188477, 0.03773825615644455, 1.3432894945144653, -0.5626400113105774, -0.04746178910136223, -0.4232044816017151, -0.15841007232666016, 1.295593023300171, 0.8968933820724487, -0.7944695353507996, -1.2431607246398926, 1.0167102813720703, -0.48312729597091675, -1.3661607503890991, 0.12653866410255432, 0.1549045294523239, -0.11974510550498962, 0.203537717461586, -1.174217939376831, 0.004908399190753698, -1.3759385347366333, -1.0659888982772827, 0.21194535493850708, -0.28574615716934204, 0.9889967441558838, 0.5114976763725281, 0.44505664706230164, 0.9080572128295898, -0.795249342918396, -0.3660268187522888, 1.319716453552246, -0.3593706786632538, 1.1928622722625732, -1.0234853029251099, 0.14184223115444183, 0.6038495302200317, -2.338785171508789, -1.5922164916992188, -0.45962828397750854, -1.260797142982483, 1.8264261484146118, -0.2313070297241211, 1.0668001174926758, -0.7242101430892944, -1.8881282806396484, 0.4222795367240906, -0.39092767238616943, -1.2650172710418701, 0.9159064292907715, 0.007649641949683428, 1.1152514219284058, -0.34219053387641907, 1.7585515975952148, -0.4565284252166748, -0.48600059747695923, -0.6985281109809875, 1.3553894758224487, 0.3116527199745178, 0.8603970408439636, 0.8095836043357849, 0.39718756079673767, -1.825577735900879, -0.8449337482452393, -2.49210262298584, -0.7522814869880676, -0.3140714466571808, 0.6596124768257141, -1.2277076244354248, 1.3757010698318481, 0.6163827776908875, -0.823922336101532, 0.11288609355688095, 1.5248996019363403, -0.5415516495704651, -0.8122787475585938, 0.5360881686210632, 1.5365493297576904, 0.5617576241493225, -0.3802998960018158, -1.6351341009140015, 0.897332489490509, -2.539931297302246, -0.016025811433792114, 1.7979955673217773, 1.2449833154678345, -1.208956003189087, -0.4673270285129547, -0.5207876563072205, 0.8200623989105225, -0.7838110327720642, -0.833472490310669, 0.5980176329612732, -0.863958477973938, -0.31249505281448364, 0.7914174199104309, -0.0631888136267662, -1.223071575164795, -0.4215000569820404], [1.39858877658844, -0.6731466054916382, -1.162942886352539, 0.9795962572097778, -0.3243916630744934, 0.37841832637786865, 0.6626406311988831, 1.6900492906570435, -0.4801422953605652, 0.3557871878147125, -2.02990984916687, -0.6792397499084473, -0.25396329164505005, 0.16913066804409027, -1.833967685699463, 0.2533196210861206, -0.6255946159362793, 0.5962750315666199, 0.33453577756881714, 1.87090265750885, -0.20709466934204102, 1.1919026374816895, -0.908326268196106, 1.5784615278244019, 1.468299150466919, -0.36277973651885986, 1.83486008644104, -0.16274909675121307, -1.225333571434021, 1.0244224071502686, -0.772132933139801, -0.0666952058672905, -1.0520236492156982, 0.49064621329307556, 1.0583330392837524, 0.9983833432197571, 0.35570845007896423, 0.2253979742527008, 1.4700497388839722, -0.6165205836296082, 0.5965255498886108, -0.9958057999610901, 0.44954606890678406, 1.0249806642532349, 0.2917526662349701, 0.33015337586402893, 0.020253650844097137, -1.005969524383545, -0.6114073991775513, -0.5944361090660095, 0.11823024600744247, -0.3779887855052948, 0.8789194226264954, -0.07360924780368805, -0.02915452979505062, -0.28219515085220337, -0.5907083749771118, -1.837296485900879, 0.60979163646698, -0.022660009562969208, -0.7311980724334717, 1.0044939517974854, -0.059327684342861176, 0.3817862272262573, -0.9002772569656372, -0.2611320912837982, 2.048985719680786, -0.22706511616706848, 1.401800274848938, 0.3794167637825012, 2.0021603107452393, -1.8168379068374634, 1.74171781539917, 0.6281557083129883, 0.0633452981710434, 0.6949669122695923, -1.551118016242981, -0.16366831958293915, -0.10501839965581894, -0.5601822137832642, 1.1323392391204834, 0.7145776152610779, -0.6747378706932068, 0.3034806251525879, -0.8779348134994507, -0.3227413594722748, -2.0970447063446045, -1.0126885175704956, -1.0556215047836304, 0.7556498646736145, -0.27491524815559387, -2.020918130874634, -1.2222723960876465, 0.1049930676817894, 1.451966404914856, -0.7154579162597656, 0.7187675833702087, -0.5557489395141602, -0.19300608336925507, -2.9501874446868896, 0.3451104164123535, -0.5759912729263306, -0.7994256615638733, -1.6034667491912842, -1.1977105140686035, 0.4787306785583496, 0.30703994631767273, 1.3779395818710327, -0.7028364539146423, -0.8486948609352112, -0.46110162138938904, 0.5795623064041138, -0.4837607741355896, 0.05775684863328934, -0.9655221104621887, 1.8069171905517578, 1.7889071702957153, 0.535725474357605, -0.5043195486068726, 0.03989243507385254, -1.651276707649231, -1.2005614042282104, 0.4005929231643677, 0.6149579882621765, 0.20149612426757812, 0.9311106204986572, 0.9970999956130981, -0.9447197914123535, 1.2427672147750854], [-1.2024173736572266, 1.2188266515731812, 0.7186213731765747, -2.2937185764312744, -2.275845527648926, 0.49162328243255615, 2.4317405223846436, -0.23164339363574982, 1.384156346321106, -1.5008502006530762, 0.3620585799217224, 0.1184602603316307, -0.19800467789173126, 1.4136667251586914, -0.8531500697135925, -0.28134840726852417, 0.10623221099376678, 2.199795722961426, -0.014125310815870762, 1.4946656227111816, 1.1438642740249634, -0.8329865336418152, 0.2583046853542328, -0.655251681804657, 0.2289547324180603, -0.47124457359313965, -0.7807604074478149, -1.9015638828277588, -0.11138127744197845, -1.3611358404159546, -1.427085041999817, -0.9743138551712036, 1.1765813827514648, -0.8169646859169006, -0.1228451207280159, 0.5906163454055786, 0.6496379971504211, 0.06766900420188904, -0.33537691831588745, -1.2809280157089233, -2.24580979347229, 0.5516237020492554, -1.7026740312576294, -0.2778860330581665, -0.4825364649295807, 0.4712120592594147, -1.0696207284927368, -1.729021668434143, 0.5265144109725952, 0.8894551396369934, 0.21101133525371552, 0.6933927536010742, -2.431845188140869, 0.7804175019264221, -1.0851441621780396, -0.4220874309539795, 0.11984048783779144, -1.4717237949371338, -0.07515174150466919, -0.4664965569972992, 0.8537351489067078, 0.31250903010368347, -2.222460985183716, -0.18024799227714539, -0.5964937806129456, 0.732153594493866, 0.401969850063324, 0.5327115058898926, -1.7105222940444946, 0.12283632159233093, -0.4494633078575134, -0.9314805269241333, 0.9344329833984375, -0.4202471077442169, -0.4594786465167999, -2.39593505859375, -1.0290714502334595, -0.514849841594696, -0.5855032205581665, 1.5347157716751099, 0.5497358441352844, 0.2738194763660431, -0.8595263361930847, -1.3895491361618042, -0.31085944175720215, -0.8174352645874023, -0.6701328754425049, -1.097578763961792, -1.2610896825790405, 0.17040926218032837, -1.859896183013916, -1.0631234645843506, -2.3512301445007324, -1.3423092365264893, -2.7533106803894043, -0.42623797059059143, -1.7405788898468018, -0.23623856902122498, 0.8983391523361206, 0.9203687310218811, -0.1327066570520401, 1.97798490524292, 0.5878761410713196, -0.8463746309280396, 0.2229464203119278, 0.7789545059204102, 1.5267099142074585, 0.31560853123664856, 0.18900731205940247, -2.1885664463043213, 0.34447750449180603, 1.8011696338653564, 0.021014027297496796, -1.036212682723999, 1.3057180643081665, 0.7695798873901367, -0.5212243795394897, 1.8708704710006714, 0.8807892203330994, 0.6206308603286743, 1.074143886566162, -0.4984143376350403, 0.3599103093147278, 1.401489019393921, 0.7750756740570068, 1.288573145866394, -0.8752954006195068, 1.0023778676986694, 0.39692285656929016], [0.24551242589950562, 0.37181055545806885, 1.085381269454956, -0.3402414321899414, -1.0066853761672974, 1.65330171585083, 0.28116336464881897, -0.8341196775436401, 0.8312502503395081, -0.04744066298007965, 0.9409825205802917, 0.6875844597816467, 0.9208009243011475, -0.29177582263946533, 0.2354198396205902, -0.3025462031364441, 1.6402093172073364, -0.3365773856639862, -0.48904964327812195, 1.2329668998718262, -0.8978471755981445, 0.8032578825950623, -1.1428693532943726, 0.7546796202659607, -0.10485923290252686, -0.8533532023429871, -0.006775691639631987, -0.059137359261512756, -0.3460828959941864, -0.7625631093978882, -0.3004454970359802, 0.5778383016586304, -0.43992337584495544, 1.5335593223571777, -0.16851171851158142, 0.255012571811676, -1.4213979244232178, -0.7950494289398193, 0.5725629925727844, -0.565669059753418, -0.8143887519836426, -0.6576372385025024, -0.47492337226867676, -1.2019643783569336, -0.7699539661407471, -1.3287633657455444, 2.2493274211883545, -0.8547049164772034, -0.844985842704773, -2.801913261413574, -1.2488192319869995, 0.5102394223213196, -0.4852505624294281, 0.18013177812099457, -0.11512505263090134, -1.999135136604309, -1.4360997676849365, -0.447122722864151, 0.17074719071388245, 0.987876832485199, 0.036246199160814285, -0.5895236730575562, 1.23186194896698, 1.3163905143737793, -1.6794954538345337, -0.558276891708374, -0.8877128958702087, 0.9529514908790588, -1.4717938899993896, -0.17534805834293365, 1.0537126064300537, 0.7404857873916626, 0.4973113238811493, -1.7034200429916382, -0.9983353018760681, -1.3386696577072144, -0.359008252620697, -0.3274093270301819, -0.0696326419711113, 0.4354041814804077, -0.7631564736366272, 0.7137369513511658, -0.08203290402889252, -1.5966696739196777, -0.08717077970504761, 0.11300178617238998, -0.04559226706624031, -0.6466689705848694, 2.13847017288208, -0.9467864632606506, 1.1539223194122314, -0.1954605132341385, 1.0472919940948486, -0.3406812250614166, 0.5267865657806396, 1.8295954465866089, -1.7869518995285034, -0.5209852457046509, -0.06961067020893097, -0.5802533626556396, 0.880648136138916, -2.133816957473755, 0.524796724319458, -0.26057615876197815, -0.5429768562316895, -0.7204415798187256, 0.9064793586730957, 0.5018045902252197, 1.3607851266860962, -1.2977122068405151, 0.5698654651641846, -0.4169895350933075, -1.0965824127197266, -0.4502454698085785, 0.41321536898612976, -1.3816883563995361, 0.12025292217731476, 0.4433560073375702, 0.5750558376312256, -0.012827193364501, 1.0665698051452637, -2.051318883895874, -0.5827623605728149, 1.1955187320709229, 0.6143633723258972, -0.7081503868103027, -0.3630824685096741, 0.5341551899909973, -0.5636846423149109], [2.1618480682373047, -0.10969822853803635, -0.016835592687129974, 0.05312933400273323, -0.5404267907142639, -0.18883737921714783, -1.1283397674560547, -1.205941915512085, -0.026733912527561188, -0.5620193481445312, 1.4883512258529663, 0.3232140839099884, -0.2090626060962677, -0.0013521363725885749, -0.17321379482746124, 0.2725251317024231, -1.1900420188903809, 1.3018310070037842, -0.35789087414741516, -0.6463566422462463, -1.6818610429763794, -0.024345342069864273, -0.06637697666883469, -0.6515734195709229, 0.1401260942220688, -0.866368293762207, 0.38437819480895996, 0.3598507344722748, 1.1353811025619507, -0.7377171516418457, -0.5496662259101868, -1.909214973449707, 0.11828694492578506, 0.36344778537750244, 0.020815232768654823, 0.1069842204451561, -0.9914494752883911, 0.017542904242873192, -1.6207637786865234, 1.2209467887878418, 1.5766716003417969, 1.281335473060608, 0.6652234196662903, -1.5799728631973267, 1.069597840309143, -2.6043553352355957, -0.3542332649230957, 0.12365683913230896, 0.7432636618614197, 0.8104940056800842, -0.508658230304718, -0.12205911427736282, 0.6706739664077759, -0.6864311695098877, 1.5742050409317017, 1.1113111972808838, 0.7498078346252441, 0.1751473844051361, 1.1178135871887207, 0.6641136407852173, 2.0560154914855957, 0.7599033713340759, -1.701320767402649, -0.12039158493280411, -0.47748249769210815, -0.4737820327281952, 1.5143377780914307, 0.15064726769924164, -0.5792858004570007, -0.6635547280311584, -0.7079520225524902, -0.32135024666786194, 0.4925266206264496, -1.118334174156189, -1.2928804159164429, 0.5097765326499939, 0.11850511282682419, 0.531730055809021, -2.537921905517578, -1.423871636390686, 0.18183991312980652, 1.386899709701538, 1.7788219451904297, -0.9584852457046509, 0.5499954223632812, 1.8991589546203613, 0.17100131511688232, 1.5489466190338135, 0.8238294124603271, -1.8469828367233276, 0.6223695874214172, -0.18834449350833893, -1.6530413627624512, 0.7708607316017151, 1.7052617073059082, 1.4831440448760986, -0.9931759834289551, 1.0235835313796997, -1.4767773151397705, -0.2556445896625519, -0.42940807342529297, 0.4865686297416687, 1.3238625526428223, -1.1826130151748657, 1.3179490566253662, -0.4144579768180847, -0.16081982851028442, 1.2893438339233398, 0.8027690649032593, -0.8316896557807922, 0.10046722739934921, 0.642446756362915, 1.0573052167892456, -0.9296343326568604, 0.08546429872512817, 1.7621742486953735, 1.1309670209884644, 0.8382669687271118, 0.5464701056480408, -0.12273719906806946, -0.7560039758682251, 0.09965622425079346, 0.6591066718101501, 0.7132192850112915, -0.29404735565185547, 2.9701197147369385, -0.47704336047172546, 1.3449668884277344, -0.8788648247718811], [0.5320166945457458, 0.2095409780740738, -1.7055786848068237, -0.9379512071609497, 0.6662284731864929, 1.3978748321533203, -1.2950334548950195, -0.5917193293571472, 1.684220790863037, 0.576448380947113, -0.4668795168399811, -1.8440821170806885, 1.924687147140503, -2.659125328063965, 0.21852345764636993, -0.5229158401489258, 0.41231289505958557, 0.8930473923683167, -1.085443377494812, 0.415343314409256, -2.0047659873962402, -0.21543575823307037, -0.18500195443630219, 1.1093080043792725, -1.0458729267120361, 0.793024480342865, -0.0273291002959013, 0.8304129838943481, 1.641098141670227, 0.7741218209266663, -0.010944993235170841, -0.24891938269138336, -0.5621076226234436, -0.585087776184082, -0.9513012170791626, 0.2623462378978729, -0.3369784653186798, -1.0200408697128296, -0.711366593837738, 2.5643744468688965, 1.5123319625854492, -0.7320036292076111, -0.7773633003234863, -0.28033503890037537, 1.8811466693878174, 0.7824163436889648, -0.026515204459428787, -0.2651207745075226, 1.8592113256454468, -1.6591744422912598, 0.35042667388916016, 0.10951618105173111, -0.5047603845596313, 0.8068205714225769, 0.47674626111984253, 0.9371619820594788, 1.076974868774414, 0.8731458187103271, 0.7022429704666138, 0.24569492042064667, 0.9616734981536865, -0.13973070681095123, 0.3510322570800781, 1.1561840772628784, 0.5073292851448059, -0.3791300058364868, -0.3954823613166809, 0.05904325842857361, -1.9732633829116821, 0.6256081461906433, 0.01616927981376648, 0.3012807071208954, -0.4884810149669647, 0.021893443539738655, -1.7841871976852417, -1.3275269269943237, -0.15095014870166779, -0.9829146862030029, -0.6605952382087708, -0.17816412448883057, -0.5645365118980408, 1.3803679943084717, -0.9099471569061279, 0.27345171570777893, -0.19235114753246307, 0.41811037063598633, 0.1619202047586441, 0.027007082477211952, -0.5043701529502869, 1.8187404870986938, -0.02358967624604702, -0.27084073424339294, 0.020017385482788086, -1.5252376794815063, 0.5266451835632324, -0.37255722284317017, 1.4905766248703003, -0.11508919298648834, -0.8819827437400818, 1.1468784809112549, 0.26338884234428406, 0.4150124788284302, -0.5448724627494812, 0.5119860768318176, -0.6195615530014038, -0.082749143242836, -1.2475905418395996, 2.389031410217285, 0.0635378286242485, 1.7059884071350098, -1.61467707157135, -1.957945704460144, 0.5078517198562622, 0.5906797647476196, -0.13781365752220154, 0.2519457936286926, 1.6835488080978394, 0.15703724324703217, -0.4575040638446808, 1.2706067562103271, 0.634593665599823, -0.687299370765686, -0.8582937717437744, 1.9886454343795776, -0.452193021774292, -0.5787859559059143, 0.6906017661094666, -0.9462951421737671, 0.2879905104637146], [0.11742742359638214, -0.3330465257167816, -0.5447776913642883, -0.41067248582839966, -1.1819642782211304, 0.20711649954319, 0.9109806418418884, 2.3230700492858887, -1.6552979946136475, 0.30919307470321655, 1.1670494079589844, -0.5130552649497986, 1.1950627565383911, 1.7904627323150635, -0.16897863149642944, -0.34458306431770325, 1.401286005973816, -2.8061299324035645, 0.20075835287570953, 2.1790835857391357, -2.8065834045410156, 0.9903441071510315, -1.6014938354492188, 0.4476771354675293, 1.0484530925750732, -0.34924912452697754, 0.9525855183601379, -0.015269995667040348, 0.8387166261672974, -0.27942195534706116, -0.25736385583877563, -2.230624198913574, 1.3565008640289307, 1.2203878164291382, 0.36264508962631226, -1.4924049377441406, -1.6485413312911987, -2.266240119934082, 0.30989381670951843, -0.18698632717132568, 1.0461305379867554, 1.012217402458191, -0.2796594798564911, -0.5939102172851562, 0.8033364415168762, -0.47699683904647827, 2.504239559173584, 0.5214194655418396, 0.3546808660030365, 0.6421774625778198, -0.2919272780418396, -0.12121367454528809, 1.668444037437439, 0.642726719379425, -0.3913956582546234, 0.9990360736846924, -1.0163367986679077, -0.7434728741645813, 0.23358014225959778, -0.6077397465705872, 0.0025496832095086575, 0.06425967067480087, -0.20225602388381958, -0.17738252878189087, -1.0410984754562378, -1.3326799869537354, 0.30696016550064087, 0.6949794292449951, 0.6012141704559326, -0.5541556477546692, -0.2287554144859314, 0.3706202208995819, 1.9884798526763916, 1.0114721059799194, 0.053816139698028564, -0.6208714246749878, -1.5890644788742065, -0.763896107673645, -0.8700593113899231, -1.548142671585083, 0.9589572548866272, 0.36726927757263184, 0.07038815319538116, 0.6389732956886292, 0.9426792860031128, 0.19343382120132446, 1.346243143081665, -0.1373491883277893, -1.1321853399276733, -0.3774198591709137, 0.5637014508247375, 0.555293083190918, 1.524254560470581, -0.7748997211456299, 0.46252989768981934, 0.4415534734725952, -0.7171705961227417, -1.2598718404769897, 1.151310920715332, -0.4136810302734375, -0.35437238216400146, -0.8768048882484436, 1.3107309341430664, 0.557467520236969, -0.3043534755706787, 0.147184357047081, 1.0445001125335693, 1.1695891618728638, 0.05028929561376572, 1.3435328006744385, 0.7852482795715332, -1.232698917388916, 1.2452495098114014, -0.06529274582862854, 0.4332045614719391, -0.252958208322525, 0.5564019083976746, 0.08154090493917465, 0.6194403767585754, -0.5106731057167053, -1.510741114616394, -0.21491771936416626, 1.5244344472885132, 0.7254034280776978, 0.22480224072933197, -0.16251440346240997, -1.0380287170410156, 0.2698810398578644, 1.289770245552063], [-1.048529863357544, 0.5332894325256348, 2.097027540206909, -1.331827163696289, 0.17695443332195282, 0.33415764570236206, -0.7336964011192322, 0.7286736965179443, 0.12070845067501068, 1.125902771949768, 0.29864442348480225, -0.5119423270225525, 1.8074021339416504, -1.0336517095565796, -0.8492729067802429, -2.1571738719940186, -0.5101655125617981, 0.032549016177654266, -1.2979015111923218, 0.9913256764411926, 0.5728992223739624, 1.122634768486023, 1.1512125730514526, -0.05492991581559181, 0.17582276463508606, -0.7161975502967834, 2.7659761905670166, 0.4941276013851166, -0.8556742668151855, 0.2973445653915405, -1.7456369400024414, -0.15771497786045074, -2.0265607833862305, 0.8694730401039124, 0.7283577919006348, -0.29027241468429565, -0.7616251707077026, -0.8877838850021362, 0.3808369040489197, -0.654575765132904, 0.9910609126091003, -0.31164082884788513, -0.4123043715953827, -0.9773068428039551, -1.4339852333068848, 0.7637583017349243, 0.4840172529220581, -0.47516298294067383, -0.7013754844665527, 0.759619414806366, 0.4679088294506073, -1.791351556777954, 2.2405881881713867, -0.7809543013572693, -0.5980913639068604, 0.9715161323547363, -2.615978240966797, -0.41671228408813477, 0.3402252495288849, 1.2874394655227661, 1.4004966020584106, 0.3666995167732239, -0.17269597947597504, 0.1335502564907074, -0.8989650011062622, 0.07722580432891846, -0.06295648217201233, -0.4694245457649231, 0.5462559461593628, -0.6478909850120544, -0.9188593029975891, -1.542412519454956, -1.4197601079940796, 1.21977698802948, 1.1519153118133545, -0.6467867493629456, 1.302526593208313, 0.5506490468978882, 0.7448769807815552, 2.114668607711792, 0.4540553092956543, 0.2936539053916931, 0.6213847398757935, -1.412244200706482, 0.3828873634338379, -0.08032716810703278, -0.3671342432498932, 1.0182862281799316, 1.46184241771698, 1.2136982679367065, -1.094099760055542, -0.8488510847091675, 0.10293975472450256, -0.3570026159286499, -0.2702879309654236, -0.9187854528427124, -0.40139034390449524, 0.45314645767211914, 0.45228132605552673, -0.1698252111673355, 0.06318559497594833, -0.17616666853427887, -0.4524744153022766, 0.3580668866634369, -0.2618222236633301, 0.33550912141799927, -2.2455711364746094, 0.6992116570472717, 1.7387442588806152, -0.6143496632575989, -0.05944802239537239, -1.1907825469970703, -2.207683563232422, 0.7466887831687927, 0.7119462490081787, -0.12347346544265747, 2.1768076419830322, 0.23806539177894592, 0.36222752928733826, 0.4440785348415375, 1.852049708366394, 0.388303279876709, 1.5642801523208618, 0.6875631213188171, 0.3391422927379608, 0.8151276111602783, 0.36090174317359924, 0.7658792734146118, 0.7925254106521606], [-1.0369735956192017, 1.3652079105377197, 1.5054717063903809, -0.9897621870040894, 0.7139648199081421, -0.15654385089874268, -1.4773402214050293, -0.04446251317858696, 0.07045920193195343, 0.23083961009979248, 0.7745305895805359, 0.37368044257164, -1.2242882251739502, 1.444879412651062, 0.13602057099342346, 0.5162119269371033, -1.2529820203781128, -0.3129047155380249, 0.2990500330924988, -0.048612967133522034, -1.8207052946090698, -0.5963199734687805, -0.27941784262657166, -0.9087874293327332, 0.8690734505653381, -1.0003886222839355, -0.08749482035636902, 0.46175554394721985, -0.27784428000450134, -0.6986403465270996, 0.0887465700507164, 0.5820592641830444, 0.1650293469429016, -0.2597295939922333, 1.2584964036941528, -0.11174376308917999, 1.918213963508606, -2.082841634750366, -0.7347469329833984, -0.24076688289642334, -1.6477996110916138, 0.5125536322593689, -0.824113667011261, 1.6996248960494995, 2.725548267364502, -0.16943439841270447, -0.6527196764945984, 1.2686833143234253, -0.40830057859420776, 1.0094804763793945, -0.03721495345234871, -1.157556414604187, 0.44152694940567017, 0.5595483183860779, 0.5978583693504333, 0.2781953513622284, 1.2602850198745728, 2.145273447036743, -0.05875458940863609, -0.14302895963191986, -0.04588702693581581, 0.8161094784736633, 0.05172239616513252, -0.8455768823623657, 1.5473511219024658, 1.586180329322815, -0.1089865192770958, 1.5186046361923218, 0.41127729415893555, 0.4241029620170593, 0.505836009979248, 0.20789118111133575, 0.7124468684196472, 0.48626917600631714, -0.9151753187179565, -0.14646874368190765, 0.9348170757293701, -1.1595256328582764, -0.2320915162563324, -0.31200602650642395, -0.08240456879138947, -1.0485115051269531, -0.47281739115715027, -1.197666883468628, 1.1017498970031738, -0.7707645893096924, 0.4968844950199127, 2.2787485122680664, 0.7776051163673401, -0.19649788737297058, -0.8886294960975647, -2.6789841651916504, 1.764291763305664, 1.0708627700805664, 0.5555537343025208, -0.5438282489776611, 1.4018570184707642, 0.2042558491230011, 1.4666669368743896, -0.21601398289203644, 0.28087878227233887, -0.022469110786914825, -0.08484766632318497, 0.09323470294475555, 0.5254292488098145, 1.2619640827178955, 1.3495118618011475, 0.6709514856338501, -0.29411745071411133, 0.45856350660324097, 1.1183717250823975, -0.2657577693462372, -0.9019174575805664, -1.1021617650985718, -1.3796755075454712, 0.49084487557411194, -0.787796676158905, -0.9357795715332031, -0.4053840935230255, -1.3482463359832764, -0.6568588614463806, -0.46665337681770325, -0.058946747332811356, 0.0070326137356460094, -1.6668808460235596, 1.3467687368392944, 0.0343356616795063, 0.08596909046173096, -0.2331884205341339], [0.4835715889930725, -0.2071426957845688, -0.052319757640361786, -1.4754931926727295, 0.043034665286540985, 1.6385126113891602, -1.240507960319519, 0.6967215538024902, -0.19074968993663788, -0.5002865195274353, 0.29322099685668945, -0.04388853535056114, 0.6812769174575806, 2.184431791305542, 0.506171703338623, 1.5468769073486328, -0.13518740236759186, 1.6294536590576172, -0.16143973171710968, 0.08436022698879242, -0.40749460458755493, -0.6517626047134399, 0.3768768012523651, -0.4794708490371704, -0.8539754152297974, -1.565545678138733, 0.14992555975914001, 1.7906898260116577, 0.20793047547340393, 0.47346231341362, 1.7891626358032227, 1.9437612295150757, -0.7593560814857483, -0.6664330959320068, -0.7884215712547302, 1.6245017051696777, 0.9726639986038208, -0.9068454504013062, 1.1326121091842651, -0.19437865912914276, -0.3585439622402191, -1.9869154691696167, -0.7572612762451172, -0.1796276867389679, 0.09003719687461853, 0.1573852002620697, -1.311025619506836, 0.046974875032901764, -1.474090337753296, -0.9853016138076782, 0.4956020414829254, 1.254578709602356, 0.8222083449363708, -1.7839754819869995, 0.009908313862979412, -0.21154248714447021, -1.051338791847229, -0.16907371580600739, -0.3731567859649658, 0.4057512879371643, -1.0697849988937378, -1.4360275268554688, -1.8552002906799316, -3.1284313201904297, -1.555496096611023, 0.014824533835053444, -0.5151007771492004, 0.09546617418527603, -2.26511287689209, -1.2267872095108032, -0.21736979484558105, -0.21500521898269653, 2.523927927017212, 1.1511322259902954, -0.06701081991195679, 0.7155449986457825, -0.6370297074317932, -1.4178400039672852, -1.9029629230499268, 1.4115252494812012, -0.39571481943130493, -0.7616042494773865, 0.7104665637016296, 0.12845422327518463, -0.30088526010513306, 0.027304168790578842, -0.313694030046463, -1.605180263519287, 0.8950369358062744, 0.3972962200641632, -0.2806466221809387, -0.42578208446502686, 0.7757201194763184, 0.22905579209327698, 0.46302443742752075, 0.6095010638237, -0.4523441791534424, -0.38510188460350037, -0.28384438157081604, 1.939592719078064, -0.2095135748386383, -0.2238367646932602, -0.27466005086898804, 0.5088120102882385, -0.6714867949485779, -0.1285775750875473, -1.6348212957382202, -0.47218143939971924, -1.3864375352859497, 0.8439793586730957, 0.3739333748817444, 0.16786852478981018, 1.2157573699951172, -1.5179520845413208, -1.5735951662063599, -0.7619497776031494, 0.30988937616348267, -1.3043502569198608, -1.6987003087997437, -0.405854731798172, 0.09445375204086304, 1.8607134819030762, 1.1415538787841797, 0.2605702877044678, -0.7287881374359131, 1.7876067161560059, 0.41685324907302856, -0.9238477349281311, 0.263376384973526], [1.4448570013046265, -0.25363102555274963, -0.05345563217997551, 0.2866252064704895, 2.1334338188171387, 2.476567268371582, -0.8506428599357605, -0.35298866033554077, -0.2143840491771698, -0.8098079562187195, -1.4389535188674927, 1.5207858085632324, -1.310705304145813, -0.6234861612319946, 0.6256742477416992, -0.8521024584770203, 1.180923581123352, -1.3441792726516724, 1.0404161214828491, 0.4748581349849701, -0.08230916410684586, -1.1631354093551636, -1.0822206735610962, -0.2350427508354187, -0.3208611309528351, 1.2556023597717285, -0.4832214117050171, 1.0663707256317139, -1.1284335851669312, 0.31335771083831787, 0.4662450850009918, 2.1162707805633545, 0.03306209668517113, 1.5233685970306396, -1.1187176704406738, -0.5696651339530945, -0.5842064619064331, -1.0549485683441162, -1.8591543436050415, -1.2397894859313965, 0.21631939709186554, 0.9377097487449646, 0.501319169998169, 1.1252323389053345, 1.0215424299240112, -0.5023760199546814, 0.21417318284511566, -1.0709667205810547, -1.1584820747375488, 1.0963548421859741, -1.2101073265075684, 0.26250120997428894, 0.8774595260620117, -0.7252256274223328, 1.2684757709503174, -1.6357628107070923, -0.11720950901508331, -1.675389289855957, -0.47236090898513794, -1.5271679162979126, -1.5618287324905396, 0.29384317994117737, 0.6878069639205933, -0.036653026938438416, -0.5035566091537476, 0.44608521461486816, -0.7541524767875671, -0.39015865325927734, -0.6729891896247864, -1.7377369403839111, 0.33813196420669556, -0.0671846941113472, 0.15865816175937653, -0.8236679434776306, -1.2791900634765625, -0.15786145627498627, 1.0620055198669434, -1.3493937253952026, 0.40650084614753723, -0.37449565529823303, 0.22593779861927032, 1.0184178352355957, 0.3846193253993988, 0.4143848419189453, -0.5194830298423767, 0.01596910133957863, -0.13961850106716156, 0.09633060544729233, 3.0006656646728516, 1.552962064743042, -0.9719991087913513, 1.7147525548934937, -0.10455338656902313, -0.36350664496421814, -1.419936180114746, 1.7208818197250366, -0.4975901246070862, -0.6262763738632202, 1.369123101234436, -0.2587973177433014, -0.32533323764801025, 1.07810378074646, -0.035621900111436844, -0.49783414602279663, -0.055354200303554535, 1.9326459169387817, -1.4411299228668213, 0.345109760761261, -0.15589667856693268, 0.47549915313720703, -0.3363828957080841, -0.4778144657611847, -0.650545060634613, 0.5169157385826111, 0.5375193357467651, 1.227203369140625, -0.07628801465034485, 1.4732626676559448, -0.9672272801399231, 1.1767220497131348, 0.7704476118087769, -0.8579475283622742, -1.2329378128051758, 0.7804866433143616, 0.9162229299545288, -0.3041144609451294, 0.12001682817935944, -1.0332871675491333, 0.4929236173629761], [0.8568821549415588, -0.3533615171909332, 0.0391622893512249, -1.0219576358795166, -1.6085922718048096, -0.11085940897464752, 1.0071886777877808, -0.7566776871681213, 0.47900933027267456, 0.32745593786239624, -0.00811364408582449, 1.2462881803512573, 0.5522764921188354, -0.8049437999725342, 1.4872997999191284, 1.5870789289474487, -2.399092435836792, 0.6144317984580994, -0.3853222727775574, 0.5855878591537476, -0.39310529828071594, -1.2951641082763672, -0.16797733306884766, 0.47230905294418335, 0.268775075674057, 0.9002166986465454, 0.404257208108902, -1.6458901166915894, 2.4626681804656982, -0.5597100257873535, -0.5738098621368408, -1.0528987646102905, 1.308668851852417, -0.46666598320007324, 0.009095382876694202, -0.0065591223537921906, -1.0237171649932861, 1.0265191793441772, -0.37522926926612854, 0.7428945899009705, 0.8463828563690186, -0.38450515270233154, 0.7491792440414429, 1.4694901704788208, 1.1676584482192993, -0.552764892578125, -1.8107730150222778, -0.6742216944694519, 1.053246259689331, 0.5520760416984558, 1.0722285509109497, 0.6674869656562805, -1.618479609489441, -0.2184620499610901, 1.4202690124511719, 0.5164262652397156, -0.6665312647819519, -0.37375664710998535, 0.1046622097492218, -0.65861576795578, -0.7699539065361023, 0.4276604652404785, -1.4550793170928955, 0.3140887916088104, 1.3587371110916138, -0.3320764899253845, -0.34292978048324585, 0.04542137309908867, 0.8792314529418945, -0.46424782276153564, -0.19351497292518616, 1.1497623920440674, 0.5723910331726074, -0.30307427048683167, 0.6796457171440125, 1.3175976276397705, -1.724534511566162, -0.9431037902832031, -1.0464255809783936, -1.1410775184631348, -0.813862681388855, -1.2100114822387695, -0.35962992906570435, -0.3867771625518799, 0.7539948225021362, -0.5763946771621704, 2.33471417427063, 0.9937182664871216, 0.7686913013458252, -0.17642991244792938, -0.21530556678771973, 1.5342720746994019, 0.4830693304538727, 0.16669389605522156, 1.2316629886627197, -0.12500318884849548, -1.4753689765930176, 0.1314612627029419, -0.12625572085380554, -0.6423448920249939, -0.5160367488861084, -0.27848291397094727, 0.2946256399154663, 1.0945569276809692, -0.9579232335090637, -0.03161054104566574, -1.17384934425354, 0.5964795351028442, -2.5983216762542725, 0.13802731037139893, -0.7201313376426697, 0.07054051011800766, 0.28385451436042786, -0.5339872241020203, -0.4283631145954132, -0.2653074264526367, -0.35821497440338135, -1.0552151203155518, -0.08634873479604721, 0.06790696084499359, -0.7354162335395813, 0.3698410093784332, -2.084239959716797, 0.31567859649658203, 0.31090399622917175, 0.45675137639045715, 0.4936324656009674, 1.523773193359375, -0.4062249958515167], [-1.503644585609436, -0.6320934891700745, -0.42596209049224854, -0.08370769023895264, 1.1348824501037598, 0.5765514373779297, -1.3150862455368042, 0.4131263494491577, 0.09868770092725754, 0.19790519773960114, -0.19458720088005066, -1.0643866062164307, -1.986675500869751, -0.6284123063087463, 0.6215347647666931, 0.881293535232544, -0.3262132704257965, 0.10843299329280853, -0.8846874237060547, -0.8091715574264526, -0.5250695943832397, -0.864198625087738, 0.9340633153915405, -1.5251693725585938, 0.8686620593070984, -0.2967284619808197, 0.37287071347236633, -0.6688694953918457, -1.9660006761550903, -2.8620612621307373, -0.8697008490562439, -1.70545494556427, -0.5291324257850647, -0.42659544944763184, 0.6564669609069824, 1.5054523944854736, -0.5176214575767517, 1.5762720108032227, -0.8312760591506958, 0.5678830146789551, -0.26013246178627014, 1.145537257194519, -1.0448167324066162, 0.04442683234810829, -0.08721344918012619, 0.12790383398532867, -1.613860011100769, 0.28506413102149963, -0.2729882597923279, -0.23110295832157135, 0.8564605712890625, 0.6633124351501465, -0.2032533884048462, 0.16669701039791107, -0.6596709489822388, -0.43581774830818176, 0.25805455446243286, -2.529163360595703, -1.8213733434677124, 2.296363353729248, -0.5360676646232605, -0.4160923957824707, 0.6663016676902771, 1.6078031063079834, -1.8272650241851807, 0.008292612619698048, -0.8106430172920227, -0.04136284440755844, 0.8669396042823792, -0.6364990472793579, 1.1748337745666504, -1.1945863962173462, 0.15440763533115387, -0.6642844080924988, 1.392279863357544, -0.25615379214286804, -0.6464084982872009, 0.7883859872817993, 0.891245424747467, -0.5849428176879883, 0.44134247303009033, 0.3033946752548218, 0.5302035808563232, 0.3825348913669586, 0.3501836657524109, 0.9119957089424133, 1.347142219543457, 0.4680050015449524, -0.3620668351650238, 0.251893550157547, 1.9345508813858032, 0.16528929769992828, -2.544677495956421, 0.353608101606369, 0.25385627150535583, -0.09776375442743301, -1.0814933776855469, 0.2994785010814667, 0.06375238299369812, 1.3021330833435059, 1.1279540061950684, -1.062664270401001, 0.2691088020801544, 1.4395315647125244, 0.29311689734458923, 1.1025476455688477, -0.5143251419067383, -0.633948564529419, -0.18626189231872559, -0.12123365700244904, 0.6995596885681152, -0.8632674813270569, 0.06769188493490219, 0.8144314885139465, -0.9485849142074585, -1.8686487674713135, -1.172022819519043, 2.26253342628479, 0.3804640769958496, -0.19282202422618866, 0.6344485878944397, -0.699284553527832, 1.497336745262146, -0.10183224081993103, 0.04239180311560631, -0.832757830619812, -0.193216472864151, 0.8913817405700684, 0.8105437755584717], [0.4779484272003174, -0.1844004988670349, 0.9022834897041321, 0.03594502806663513, 0.7663119435310364, 1.007411241531372, 1.2447738647460938, -0.4224642515182495, 1.650352120399475, 0.6239705085754395, 1.9604823589324951, 2.0404164791107178, 1.5144641399383545, -0.14028020203113556, 1.317694067955017, -1.5242536067962646, 0.5465578436851501, 0.5525684356689453, 0.4473651945590973, -1.905418872833252, 0.7818358540534973, -0.6198164224624634, -0.3506115674972534, 0.8265630006790161, -0.22209970653057098, 1.4280359745025635, -0.10919879376888275, 1.2610743045806885, -0.6964908242225647, 1.1878772974014282, 0.5271229147911072, -0.9322338104248047, -0.06360691785812378, -0.553588330745697, 1.7363160848617554, -1.2748878002166748, -0.5281698703765869, 2.0340681076049805, 1.7109496593475342, -1.6375586986541748, 1.344096302986145, -0.30131059885025024, 0.6336349844932556, -1.986777663230896, -0.4127116799354553, 1.2185871601104736, -0.599077045917511, -1.9999383687973022, -1.2689441442489624, -0.6488840579986572, 0.5828115940093994, 1.9603440761566162, 1.7654577493667603, 0.006838417612016201, 0.29930415749549866, -0.1765204668045044, -0.6905934810638428, 1.2820875644683838, -2.0627708435058594, 1.4599201679229736, 0.16961055994033813, -1.051015019416809, 1.5057463645935059, 0.59505295753479, 0.5642962455749512, 1.4890397787094116, 0.07332661002874374, -0.0536838136613369, 0.15102322399616241, -1.02863347530365, 0.7246111035346985, 0.3191556930541992, 0.30281373858451843, -0.8218855261802673, -0.20812378823757172, 0.3705717623233795, -1.3815209865570068, 0.39357393980026245, 0.7035650610923767, 0.0030849918257445097, 0.0842624083161354, 0.5847460627555847, -1.4643670320510864, 0.05450296774506569, -1.827890157699585, 1.2223584651947021, 1.1748263835906982, -1.0676629543304443, 1.3018156290054321, -0.4618293046951294, -0.5367721915245056, -0.20922109484672546, 0.6743782758712769, -0.46283724904060364, -0.5920853018760681, -0.7286863923072815, 2.1087403297424316, 0.1359710693359375, -0.3157554566860199, -0.5049035549163818, 0.7188504934310913, 0.016091875731945038, -0.766124427318573, 0.2410106062889099, -0.7309379577636719, 0.27283960580825806, 0.23089586198329926, 0.5475611686706543, -1.076531171798706, -0.24251073598861694, 1.458788275718689, -1.7240155935287476, 0.45886510610580444, -0.7877261638641357, 1.076440453529358, -0.8747330904006958, 0.8777292370796204, -0.29164162278175354, -1.2433395385742188, -1.4030405282974243, -1.8505407571792603, -0.5027795433998108, 1.0667014122009277, 1.78745436668396, -0.09737423062324524, -0.11427480727434158, -0.24510709941387177, -0.7591255903244019, 1.1254233121871948], [1.0034235715866089, 0.6712831258773804, -0.47784313559532166, -0.1544516235589981, 0.8165814280509949, 0.29132869839668274, 0.019914112985134125, 0.35122936964035034, 0.9479211568832397, -1.457865595817566, 2.4777863025665283, 1.0636029243469238, 1.4843785762786865, -0.3842296004295349, -0.5235462784767151, -0.7662177681922913, 0.2265259176492691, 1.2671279907226562, 1.1265097856521606, 0.328727126121521, -0.672031581401825, -1.0100131034851074, -1.0652475357055664, -1.4517923593521118, -0.02677936851978302, 1.919622540473938, -0.26428142189979553, -0.3977648615837097, -0.23359452188014984, -0.45511019229888916, -0.7015707492828369, -1.3999630212783813, 1.0192501544952393, -1.3033336400985718, -0.9371095299720764, -1.4318584203720093, -0.5461036562919617, 0.5513546466827393, -0.9704750776290894, -0.5712066292762756, 1.1416712999343872, -0.7110241055488586, -0.10571695864200592, 1.742045283317566, 0.9319804310798645, -0.3434574604034424, 0.9411895275115967, -0.1855326145887375, -1.1896859407424927, 0.19877997040748596, 0.5247405767440796, 0.30927199125289917, -0.1375247687101364, -1.3474292755126953, -0.47885939478874207, -0.32617348432540894, -0.30597299337387085, 0.21386948227882385, 0.47317901253700256, -1.0681285858154297, -0.7755832672119141, -0.8497966527938843, 1.8067232370376587, 0.04945690929889679, -0.4756220281124115, 0.900482177734375, -0.351752370595932, 0.21404580771923065, -0.029061762616038322, 0.6202338337898254, 0.5524424910545349, -1.4308602809906006, -0.733048677444458, -0.0768791139125824, 0.562247097492218, 1.7531044483184814, -1.1282960176467896, -0.6479959487915039, 0.7213660478591919, 1.4755148887634277, 1.1895419359207153, -0.05089125037193298, 0.14884893596172333, 1.1597319841384888, -0.31270354986190796, 0.04960756003856659, 0.034438006579875946, -0.42698433995246887, -1.6714041233062744, -0.8766605854034424, -2.6252965927124023, -1.3457752466201782, -0.5767315626144409, 0.6848272681236267, 0.17479154467582703, 2.0695998668670654, -1.060379147529602, -0.9347038269042969, 0.15663288533687592, -1.6618810892105103, -0.23554448783397675, 0.34588873386383057, 0.40621721744537354, -0.38263750076293945, 0.5627766251564026, 1.5250567197799683, -0.06362695246934891, 0.5657326579093933, 0.0018847599858418107, 1.7784886360168457, -0.568857729434967, 0.19557984173297882, -1.8290406465530396, 0.9291723370552063, -0.008384262211620808, 0.6091704964637756, 1.6131219863891602, 0.6441717147827148, 1.2234407663345337, -0.9811872839927673, -0.2701665759086609, 1.5013870000839233, 1.220575213432312, 1.4779161214828491, 0.004536330234259367, -1.5337238311767578, 0.2725605070590973, -0.8446994423866272, 0.005829835310578346], [-0.3739268481731415, 0.43773314356803894, 0.6937038898468018, 0.21176673471927643, -0.563947856426239, 1.7196542024612427, -0.3080371022224426, -0.3321831524372101, -0.6059901714324951, -1.2707124948501587, 0.2952784299850464, 1.2910761833190918, 1.8453233242034912, 0.3686123490333557, -0.5697282552719116, -0.9403282999992371, -0.04667725786566734, 0.19367139041423798, -1.0355124473571777, 0.16537898778915405, 1.0613713264465332, 0.3337995409965515, 0.962836742401123, 1.864667296409607, 0.7863331437110901, -0.8650261759757996, 0.3512890636920929, 0.5887245535850525, -0.5739607214927673, 0.31856900453567505, -0.3776147663593292, -0.6642571091651917, -1.029736042022705, -0.3781001567840576, 0.5747791528701782, 0.00029879840440116823, 1.6368932723999023, -0.4364352524280548, -0.24275366961956024, -1.4710617065429688, 0.0034124841913580894, 0.6966121792793274, -0.6170644760131836, -0.1149706020951271, 1.2930147647857666, 1.4057085514068604, 1.8193238973617554, 1.4123449325561523, 1.3612937927246094, 0.5723961591720581, -0.10532635450363159, 1.6175768375396729, 1.9949557781219482, -0.5848853588104248, -0.8159766793251038, -0.20391473174095154, 0.8435017466545105, 0.4727552533149719, 0.5864115357398987, -1.6226435899734497, -0.496442049741745, 0.11271656304597855, -0.4410126805305481, 1.1923421621322632, -0.477724552154541, -1.0973392724990845, 0.39003095030784607, 0.6382901072502136, 0.5489669442176819, 1.610664963722229, -0.44891878962516785, -1.8347387313842773, -0.6602129936218262, 2.1739585399627686, -0.5438023805618286, 0.18487420678138733, 1.3267284631729126, -0.3452819585800171, 2.1261849403381348, 0.5671920776367188, 0.025803426280617714, -3.0697731971740723, -0.06605173647403717, 0.0034013071563094854, 1.113716959953308, 1.3984419107437134, 0.14205998182296753, -0.4151363670825958, -1.4714170694351196, -1.6532694101333618, 1.2553515434265137, 0.6374366879463196, -0.19295193254947662, -0.19203363358974457, 2.753124952316284, -0.5997006893157959, -0.9038550853729248, 0.12574823200702667, 0.9725500345230103, 1.0666418075561523, 1.2181950807571411, -0.7985356450080872, 0.6199744343757629, 0.5363653302192688, 0.055129263550043106, 1.9960705041885376, -0.6637141108512878, -1.026675820350647, 0.14610391855239868, -1.026032567024231, 0.3222867548465729, 0.5455276966094971, -0.8900777697563171, -0.6247538924217224, -0.6936661005020142, -0.4595671594142914, 0.39425361156463623, 1.6600264310836792, -0.16019900143146515, 1.056052803993225, -0.3821547329425812, -0.605337381362915, -0.12441287934780121, 0.7585657835006714, 0.38050997257232666, 0.5020362734794617, 0.7975664734840393, -0.3914814591407776, 1.910237431526184], [1.71636164188385, -0.4625522792339325, 0.11844366043806076, -0.3218289613723755, -0.5317844748497009, 0.7790603041648865, -0.19660955667495728, -0.7970778942108154, 0.5922082662582397, 0.5054460167884827, -1.968600869178772, 0.21821576356887817, -0.7857304811477661, 1.4562582969665527, -0.44347724318504333, -0.0737324133515358, -0.4530068635940552, 1.5099066495895386, 0.98518967628479, -1.1592093706130981, 0.19153982400894165, 0.28163468837738037, -1.013158917427063, -0.23254770040512085, -2.2679240703582764, 0.4543696939945221, 0.7285090684890747, -1.2228708267211914, 0.546757698059082, -0.9772209525108337, -1.1254067420959473, -0.06364928930997849, 0.7011948227882385, 0.44289249181747437, 0.47517529129981995, 0.7429372668266296, 0.12563131749629974, 0.723563015460968, 0.6973391771316528, 0.36439499258995056, -0.37396225333213806, 1.253873586654663, 0.9246268272399902, 1.1190940141677856, 0.1243048906326294, 0.19290101528167725, 0.06317571550607681, 0.23206552863121033, -0.5095699429512024, -0.0003419421555008739, -0.2578924596309662, -0.06899605691432953, 1.688016414642334, 2.1403751373291016, -0.09049782156944275, 0.12284529954195023, -0.8539696335792542, -0.8778289556503296, -0.6962725520133972, 0.7639564871788025, 1.459160327911377, 1.0685696601867676, -0.2416931837797165, 0.8796923756599426, -0.8367400169372559, -0.5716630220413208, -0.07401371747255325, -0.7483864426612854, -0.2867274284362793, -0.8343292474746704, 0.6193433403968811, -0.0348331555724144, 0.7339218854904175, -0.03684013709425926, -0.09830670058727264, -1.744490623474121, 0.23368431627750397, 0.9356387853622437, -2.0579264163970947, 0.15236197412014008, 0.9199363589286804, 0.13883927464485168, 0.285793274641037, -1.1820392608642578, -1.4401201009750366, 0.9146326184272766, -2.1619765758514404, 0.9002729058265686, -1.6198511123657227, -2.3688762187957764, -0.6692214012145996, -1.340569257736206, 1.662086009979248, 0.5113713145256042, -1.6737701892852783, -1.1044470071792603, -0.6134362816810608, -1.8697681427001953, 1.6269477605819702, 0.5559548139572144, -1.6600955724716187, 1.0848320722579956, 0.2609747052192688, 1.5301809310913086, -0.6604947447776794, 0.09158230572938919, 0.10336189717054367, -1.2646130323410034, 0.7964154481887817, 0.5689024925231934, -1.6880104541778564, -0.7485610246658325, 0.6821733713150024, -2.14158296585083, -1.1656365394592285, 0.0791187584400177, 1.680600643157959, -2.8668010234832764, 2.0877416133880615, -1.2372488975524902, -2.0322649478912354, 1.1074624061584473, 1.078431248664856, 1.319711446762085, -0.0025879829190671444, -0.6424121856689453, -0.6185994148254395, -0.40496814250946045, -0.3274599313735962], [0.11246044933795929, 0.6609766483306885, 0.017675675451755524, 0.3831685185432434, -0.3997945487499237, -1.598499059677124, -0.468311071395874, 0.06516603380441666, -0.018553465604782104, 0.36617425084114075, -0.6066630482673645, 0.02080494724214077, -0.5177916884422302, -0.965768039226532, 0.39713534712791443, 1.6083691120147705, -1.0026884078979492, -0.8643973469734192, -0.9116728901863098, -0.9170535802841187, 0.7512519359588623, -0.135542094707489, -0.10925406962633133, -1.402417778968811, 0.8931870460510254, 1.0369904041290283, 0.32875195145606995, 0.42274415493011475, -2.963838815689087, -0.8374035358428955, 0.760633647441864, -0.6611988544464111, 0.03145793452858925, -0.4313751459121704, 0.7108275294303894, 0.5017650723457336, 0.9890541434288025, 0.21629269421100616, -0.7480306625366211, -0.23596687614917755, 0.7843806743621826, 1.2191476821899414, -0.2639940083026886, 0.6176751852035522, -0.2811654806137085, 0.39401257038116455, 0.6543506383895874, -0.8614436984062195, -1.783023476600647, 0.17475740611553192, -0.27383795380592346, -0.07409336417913437, -0.4527546763420105, 0.8854929208755493, -0.0769117996096611, 1.111248254776001, -0.2529391944408417, 0.20749247074127197, 0.5110613703727722, -0.35556772351264954, -1.0262326002120972, -1.236264705657959, -1.3206562995910645, -0.03744184225797653, -0.2293727546930313, -0.7886934876441956, 0.9224985837936401, 1.5926392078399658, -0.16350224614143372, 0.14383089542388916, 2.293100118637085, -0.71019446849823, -0.5763363838195801, 1.0687657594680786, 0.5738129019737244, -0.5006446242332458, -0.8435658812522888, -0.4954371452331543, -0.4332248270511627, 0.3456425368785858, -0.46823787689208984, 0.9855790734291077, 0.028911665081977844, 0.8195446729660034, 1.7339292764663696, -1.321794033050537, 0.12083079665899277, -0.020937858149409294, 1.0779118537902832, -1.4727915525436401, -0.18298180401325226, -2.0956361293792725, -0.5138435363769531, 0.5394574403762817, 0.28388941287994385, -0.8999093174934387, 1.7146294116973877, -0.7745051383972168, -0.04720877856016159, -0.0774335190653801, -0.6104370355606079, -0.46327921748161316, 0.8724978566169739, 1.3737878799438477, 0.4592127501964569, 0.05088237300515175, -1.2394601106643677, -1.1255747079849243, -1.0060778856277466, 0.3867076337337494, 1.6612216234207153, -0.626091480255127, -0.3926326632499695, -0.4883086681365967, -1.0339573621749878, -0.1229744628071785, -0.21208207309246063, 1.4616637229919434, -0.8568187952041626, -0.11899740993976593, -1.8666596412658691, 0.22520510852336884, -0.024175049737095833, 0.0640254095196724, -0.24707558751106262, -0.8777891397476196, -2.112083911895752, -0.1599959135055542, 0.12897145748138428], [1.446502685546875, -1.760456919670105, 0.5383739471435547, -0.9703695774078369, -0.3125048279762268, -0.13532088696956635, -2.3407835960388184, -0.5779888033866882, -0.7091172337532043, -0.6297483444213867, -0.7267341017723083, -0.3343288004398346, -2.10556697845459, -0.877463698387146, 0.1280033141374588, -0.11380349844694138, 0.7237969636917114, -0.4276621341705322, -1.2312190532684326, 0.22929547727108002, -0.7733967900276184, -0.8178393244743347, -0.3287261128425598, -0.048158735036849976, -1.1713087558746338, 0.782650351524353, 2.046795606613159, 0.813739538192749, 0.8048185110092163, 1.0504117012023926, -0.16112470626831055, 1.102955937385559, -0.7330230474472046, -0.017895642668008804, -0.43551355600357056, -2.0176382064819336, -0.4651200771331787, 1.1457295417785645, 0.7520699501037598, -2.3867857456207275, -1.069472074508667, -0.08020071685314178, 0.17405202984809875, 0.41686058044433594, 0.3892233669757843, -1.0305101871490479, 1.2351555824279785, -0.6157843470573425, -0.09369798004627228, -0.3916895389556885, 0.4528942406177521, 0.34425511956214905, -0.29856616258621216, 0.7719694375991821, 0.6356091499328613, 0.34242182970046997, -2.6706602573394775, -1.8468327522277832, 0.5580268502235413, 1.5133113861083984, 1.0788874626159668, 1.0491604804992676, -0.06416907906532288, -0.9176434874534607, 0.2665068805217743, -0.37153393030166626, -1.4585247039794922, -0.6410585641860962, 1.5696192979812622, -0.8987096548080444, 0.7293833494186401, -0.028209395706653595, 0.5108416676521301, -0.5777102112770081, 0.2910306453704834, 0.5171397924423218, 0.6275224089622498, -1.856979489326477, -0.3532235324382782, -0.6729640960693359, 1.9028642177581787, 0.22585877776145935, -0.09056690335273743, -1.5698785781860352, 2.6902430057525635, -0.9747495055198669, -1.0551584959030151, 0.12246622890233994, 1.710473895072937, -0.11181414872407913, 2.734173536300659, 0.46534574031829834, -0.5677672028541565, -0.7028332352638245, -0.9808764457702637, 0.20108743011951447, -1.0892891883850098, 0.25349676609039307, 0.5060809254646301, 0.9823848009109497, 0.15528570115566254, 0.23279958963394165, -1.1889938116073608, 0.010132929310202599, 1.4790083169937134, 1.135636806488037, -0.33580589294433594, 0.8391851782798767, 1.6605901718139648, 0.5353502035140991, -0.3665132224559784, 1.2452961206436157, -0.7692821025848389, -1.2691078186035156, -0.2268913984298706, 0.3479292094707489, -1.4576807022094727, 0.6682838797569275, -0.2980414628982544, 1.4427872896194458, -1.1439540386199951, 0.6884461641311646, 2.634309768676758, 0.7532228827476501, 0.2661081552505493, 0.1739814728498459, 0.026894938200712204, 1.6993459463119507, -0.7899350523948669], [0.0363142192363739, 0.21248288452625275, -0.33316469192504883, 0.2655203938484192, -0.8136199712753296, -0.6211860179901123, 1.8380703926086426, 0.5825270414352417, 0.076290562748909, -0.8283588290214539, 0.7106860876083374, -0.9639380574226379, 0.5645073652267456, 0.8495879173278809, 1.2677282094955444, -0.4134383797645569, -1.3601374626159668, 0.15201318264007568, 0.2830425798892975, 0.3381071984767914, -1.074289083480835, 1.587955355644226, 0.8971055150032043, -0.025088509544730186, 1.0425883531570435, 0.9831163883209229, -1.4931520223617554, 0.21094731986522675, -0.7041482925415039, -0.6697218418121338, 1.0368268489837646, 0.36699700355529785, 1.3831846714019775, -0.2305152714252472, -1.3366966247558594, -1.9637480974197388, 0.7817276120185852, 0.2079150676727295, 0.3593199551105499, -0.6643300652503967, 1.185631513595581, -1.3433172702789307, -0.89352947473526, 1.4642410278320312, -0.7026540637016296, 0.37886282801628113, -0.33371126651763916, 1.2906581163406372, 0.2734607458114624, 0.9825095534324646, -1.1993502378463745, -0.014692981727421284, 0.4869835376739502, -0.8628189563751221, -2.5002596378326416, 0.893613874912262, -0.7570786476135254, 0.45988115668296814, 0.15244527161121368, -1.004185438156128, -0.7895091772079468, 0.00411028228700161, 1.4418373107910156, -0.2968522608280182, -0.11728334426879883, -0.36231014132499695, -2.3312065601348877, 2.5717625617980957, 2.0245227813720703, -0.49116605520248413, 0.7057207822799683, 0.4841395318508148, 0.3319494426250458, 2.0290639400482178, -0.4410702884197235, 2.937910795211792, 0.2457103282213211, -1.7464269399642944, 0.21943937242031097, -0.4911893606185913, -0.5923170447349548, 1.0447055101394653, 0.5351470708847046, 0.6818361282348633, -0.9636108875274658, 0.5744402408599854, -0.43326959013938904, -0.3097670376300812, -0.026105213910341263, 0.16812117397785187, -0.03487332910299301, 0.27136948704719543, 1.245727300643921, 1.1258007287979126, -0.3090549409389496, -0.3636082112789154, 0.7801451086997986, 1.2582662105560303, -0.1758151799440384, 0.9719817638397217, -1.587533712387085, 1.7240430116653442, 0.6048948168754578, -0.2961611747741699, -0.34650105237960815, 0.5406238436698914, -0.4946853220462799, 2.145557165145874, -1.0702083110809326, -2.151179075241089, 2.487736225128174, -0.1785629391670227, -0.011158659122884274, 0.11231282353401184, 0.21240583062171936, -0.6040753722190857, 0.6894407272338867, -0.6813892126083374, -1.06553053855896, -1.661656379699707, 1.2973145246505737, -1.037851333618164, -1.5042636394500732, 0.027001015841960907, -0.5141294598579407, 0.8910101056098938, -0.2521219253540039, -1.3597344160079956, -1.1724128723144531], [-2.2903378009796143, -0.07429961115121841, -0.1472928375005722, 1.1600688695907593, 0.3007999062538147, 1.5488172769546509, -0.7167801856994629, 1.4908859729766846, 2.0412983894348145, 0.3037569224834442, 0.4522683620452881, -0.3082584738731384, 1.0938736200332642, -0.6040273308753967, -0.24214446544647217, 1.4806815385818481, 0.11426827311515808, -0.328721284866333, -0.7093865871429443, 1.437211513519287, -0.125081866979599, 0.2559167444705963, -0.6097844839096069, -0.20321117341518402, -1.2953846454620361, -0.4961859881877899, 0.930394172668457, -0.735817551612854, 0.2810255289077759, -1.001198172569275, 1.7097458839416504, -0.20309942960739136, -0.20383276045322418, -0.08318467438220978, -0.4418436884880066, 0.35754817724227905, -0.8238900899887085, -2.6983642578125, -1.5929561853408813, 1.9446038007736206, -1.4310023784637451, 0.10335390269756317, 0.2379254698753357, 0.05222649499773979, -1.3007253408432007, -0.4613337516784668, 0.44586020708084106, 0.8815767765045166, -0.03317335247993469, 0.6117767691612244, -0.768019437789917, -0.8068983554840088, 0.7772006988525391, -0.5471632480621338, -0.9719996452331543, 0.3904073238372803, -1.0771743059158325, -0.11982814222574234, 1.6554381847381592, -1.3891398906707764, -1.9600273370742798, -0.9634960293769836, -0.25163644552230835, -1.1517913341522217, 0.3387208878993988, -1.3722056150436401, 0.1484258770942688, 0.6874770522117615, -2.319629192352295, 1.665941596031189, 0.0491059347987175, 0.6080125570297241, -2.0129151344299316, 0.27285394072532654, -0.2558591663837433, 0.6547377705574036, -1.0094603300094604, 0.946121096611023, 0.25680139660835266, -1.0253260135650635, 0.45327696204185486, 0.4947931468486786, -1.5091618299484253, -1.976611614227295, -0.21493202447891235, 1.2400431632995605, -1.169421672821045, 0.36858922243118286, -0.30404797196388245, -2.278550863265991, 0.642778754234314, -0.4749939441680908, 2.480461597442627, -0.24517381191253662, 0.994865357875824, 0.7952151894569397, -1.1069915294647217, 0.5964255928993225, 1.6336345672607422, 0.5614175200462341, 0.1651492714881897, -0.5225124955177307, -1.9040204286575317, -1.1921272277832031, 0.20408639311790466, -0.5676530003547668, 2.6494593620300293, 0.11779042333364487, 0.27417218685150146, 0.7423730492591858, 1.680031657218933, -0.8914620280265808, 0.5113048553466797, -1.8531526327133179, -2.5050718784332275, -0.6858553290367126, 1.1108168363571167, -0.03549521788954735, -0.19460588693618774, -1.544334053993225, -0.39534252882003784, -0.30154356360435486, -0.7544267773628235, 0.5521390438079834, -0.22478489577770233, 0.6862750053405762, 1.3085857629776, -0.6806392073631287, -1.0622351169586182], [-1.1484421491622925, 0.4034311771392822, -0.4754142761230469, 2.618488311767578, 0.6423285007476807, -0.41385674476623535, 1.0250838994979858, -0.2200251668691635, -1.6493123769760132, 1.279280185699463, 0.8229315876960754, 0.08272100985050201, 0.47635024785995483, 0.41734689474105835, 1.1248095035552979, -0.19947005808353424, 0.47783979773521423, -1.7722127437591553, -0.37703123688697815, 0.6741490364074707, 1.0166501998901367, 1.8664703369140625, -0.24072840809822083, 1.2807244062423706, -0.5029611587524414, 0.84554523229599, 1.1064335107803345, 0.780718982219696, 0.38473397493362427, 0.43271443247795105, 0.3335019052028656, 0.00779691431671381, 1.058735966682434, -1.11944580078125, 0.40295863151550293, -1.5126396417617798, -0.2035612016916275, 1.1409156322479248, -1.031462550163269, 1.404520869255066, 1.3840314149856567, 0.17951631546020508, -0.641807496547699, 0.17157945036888123, 0.34198200702667236, -0.8690257668495178, 0.7961685657501221, 0.7828927040100098, 0.12056119740009308, 0.05177910998463631, 0.6504481434822083, 0.4565315246582031, -0.7056810259819031, -0.07523234933614731, -0.48569032549858093, -0.5478532314300537, 1.6903239488601685, -0.40791967511177063, -0.5738497972488403, 0.1833280324935913, 0.9798116683959961, -1.6835088729858398, 0.5007651448249817, -1.4868758916854858, 0.5248247385025024, 2.3549067974090576, -1.2978266477584839, -1.8139185905456543, 1.2689145803451538, -2.1106016635894775, -1.272820234298706, 0.6361473202705383, 0.3346002697944641, 1.1492056846618652, -0.3388156592845917, -0.30771002173423767, -2.337155818939209, 0.5259305834770203, -0.2835184633731842, -0.8818907737731934, -0.4942907989025116, -1.4455780982971191, 0.29433855414390564, 0.3164810538291931, -1.4349759817123413, 1.2116589546203613, 0.5795868039131165, -0.4848287105560303, 0.5917555093765259, -0.7998068332672119, 0.9324698448181152, -0.7238173484802246, 0.5086691379547119, -0.2571883499622345, -0.45806828141212463, 2.208577871322632, -0.7400673031806946, -0.883711576461792, 1.4663141965866089, 1.167188286781311, 0.9526234269142151, -0.8712437152862549, -0.8489903807640076, 1.6302076578140259, 0.9624683260917664, 0.032299261540174484, 0.35921797156333923, -0.3537905812263489, 0.3644174039363861, 1.1426767110824585, -0.2364301234483719, 0.8608807325363159, -0.3047706186771393, -1.4556350708007812, -0.8359406590461731, -0.8939201831817627, 0.5354920029640198, -1.4313915967941284, -0.6262415051460266, -0.27143415808677673, -0.7213978171348572, -0.5542783737182617, 0.8123618960380554, -0.7662248611450195, 1.6258927583694458, 0.6253833174705505, -0.17974595725536346, -0.5673706531524658, -0.8941827416419983], [-0.1146213486790657, 0.27252528071403503, 1.4933055639266968, -1.1146525144577026, -0.7071759700775146, -1.2558695077896118, -0.14698539674282074, -0.7847700715065002, 0.07995499670505524, 0.3147967457771301, -0.15969301760196686, -0.3914458751678467, 0.1599857658147812, 1.534212350845337, -1.0202934741973877, -0.5717857480049133, -0.571828305721283, 1.4546101093292236, -0.940649688243866, 0.5040750503540039, 1.7761189937591553, -0.5977404713630676, -0.8684818148612976, 0.21746398508548737, 1.099605917930603, -0.18681339919567108, 0.3104289174079895, 0.501561164855957, 0.3964461088180542, -1.393506407737732, 1.2957051992416382, 0.020647257566452026, 0.49023184180259705, -0.29647096991539, 2.269559621810913, 1.6920207738876343, -1.7471646070480347, 0.6429893374443054, -0.08865854144096375, 0.1406479775905609, -0.09672549366950989, -1.5545490980148315, -1.3875223398208618, -1.041730284690857, 1.366808533668518, -0.08256035298109055, 0.5369207859039307, 1.3916912078857422, 0.7537615895271301, -0.24438871443271637, 0.3590796887874603, -0.6915327310562134, 1.8357850313186646, -0.280540406703949, 0.8115378618240356, 0.9775745868682861, -1.4774004220962524, -1.8491181135177612, -0.11210499703884125, -0.1621059775352478, -2.062460422515869, 1.8058778047561646, -1.8830794095993042, -1.1734966039657593, -1.2831251621246338, 0.8535220623016357, 0.06227068975567818, -1.1669293642044067, 0.20647437870502472, -0.4601474404335022, -0.7688915729522705, -0.8232300281524658, 0.05976663902401924, 1.5013631582260132, -0.7115721106529236, 0.35116851329803467, -0.529723584651947, -0.09752973914146423, 0.4918893873691559, 0.7887510061264038, 0.2987041771411896, -0.1627846211194992, -0.3184794783592224, 0.8986658453941345, 0.6717930436134338, 2.7747199535369873, -0.7810794115066528, -1.7305737733840942, 0.5486387014389038, -0.6287228465080261, -0.5647850036621094, 2.420295000076294, 2.074944496154785, 0.19579242169857025, 0.30944785475730896, 0.7897000312805176, -2.063188314437866, -0.3773607909679413, 1.6243395805358887, 1.4852131605148315, -0.2717945873737335, -0.2218286395072937, -0.10521750152111053, -0.9040545225143433, -0.5221695899963379, -1.868553638458252, -0.6638216376304626, 0.5636306405067444, -0.7078819274902344, 1.0419573783874512, -0.3164272904396057, -1.6907464265823364, 0.8582280278205872, 1.56724214553833, -0.5098294615745544, -0.20522545278072357, 0.1361151933670044, -1.1907089948654175, -0.38672545552253723, 1.635416030883789, -1.0104868412017822, -0.2145058661699295, 0.8520230054855347, -0.885452926158905, -0.011981231160461903, 0.5778907537460327, 0.0785130113363266, -0.014172397553920746, 1.0246001482009888], [0.4763000011444092, 0.7829114198684692, -1.922561764717102, -0.1712348610162735, 1.3278383016586304, -1.5089197158813477, 0.9373255968093872, 0.7380853891372681, -0.8335331082344055, 0.8207362294197083, 0.4573734998703003, -1.3584693670272827, -0.8463795185089111, 0.9182868003845215, 0.5813970565795898, 0.06707923114299774, 0.3751678764820099, 0.4471060037612915, 1.136209487915039, -0.35130706429481506, -1.4997146129608154, 0.5833678841590881, 2.4482977390289307, 1.2887389659881592, 0.820673942565918, 0.6039854884147644, 0.2295408546924591, 1.5339832305908203, 0.5770301818847656, -0.9783228635787964, -0.19213876128196716, 0.47549718618392944, 1.7486281394958496, 1.871180772781372, -1.2445546388626099, 1.2814327478408813, 1.2264374494552612, 0.4790513217449188, -0.9391358494758606, 0.21483725309371948, -1.046383261680603, -1.2712174654006958, 2.5322999954223633, 0.20113295316696167, 0.12376785278320312, -1.172553539276123, 0.775999128818512, 1.2663012742996216, 0.42538633942604065, -1.1197481155395508, -0.01084800623357296, -1.0010823011398315, -0.5658494830131531, -0.9007775187492371, 0.04185212776064873, 1.722836971282959, -1.281510829925537, 0.036681994795799255, 1.0556501150131226, 1.3008606433868408, -0.41989004611968994, 1.8631657361984253, 0.7283472418785095, -0.3162478506565094, 0.7492422461509705, 0.06734318286180496, 1.4643676280975342, -0.06692977249622345, -0.7014352083206177, 0.2294500172138214, 0.5135667324066162, -0.06309141218662262, 1.7435539960861206, -1.3857358694076538, 1.137820839881897, -0.006563339848071337, 0.5876340270042419, -0.3807816505432129, -1.678476333618164, 1.0091887712478638, 0.31281012296676636, 0.48801571130752563, -0.10920713096857071, 0.5691927671432495, 0.6871450543403625, 0.07619726657867432, -0.7137198448181152, -0.6130003333091736, -0.6186121106147766, -0.8331837058067322, 0.2866974174976349, -0.4009378254413605, -0.0048263114877045155, 0.3841111958026886, 1.4920248985290527, -1.0710023641586304, -0.476724237203598, -0.0867154449224472, -0.41600173711776733, -0.37674680352211, 0.5574246644973755, 0.16267700493335724, 2.0351173877716064, 0.7446118593215942, -0.3542046844959259, 0.11765383929014206, 1.399355411529541, 0.42628005146980286, 0.7229625582695007, 1.0245612859725952, 0.05602485314011574, 0.2970578670501709, -1.2647794485092163, -0.13583728671073914, -0.47901615500450134, -0.6390349864959717, 0.39937257766723633, 0.002130059991031885, 1.3146196603775024, -0.3701475262641907, -0.5339238047599792, 1.6910631656646729, 0.23809124529361725, 0.5545771718025208, 0.5323922038078308, 0.47698619961738586, -2.09665846824646, -0.09514473378658295, 0.577100932598114], [-1.5757631063461304, -0.9067382216453552, 0.37854817509651184, 0.9493756890296936, 0.6490510106086731, -0.21623191237449646, 0.8265305161476135, -1.3571689128875732, -0.5328425168991089, -0.12077835947275162, -0.5267709493637085, 1.7146776914596558, 1.1359292268753052, 0.38777628540992737, -1.9391728639602661, -0.21441230177879333, -1.858809232711792, 1.4403754472732544, 0.5709811449050903, -1.1659948825836182, -1.2922568321228027, -1.7282438278198242, -0.8919752836227417, -0.49602994322776794, 0.3135165870189667, 1.4981801509857178, 0.03732425346970558, -1.2590384483337402, 1.9203444719314575, 0.712438702583313, -0.3494050204753876, -0.38462913036346436, -0.6108096837997437, -0.831136167049408, -0.9370766282081604, -0.1684587597846985, 1.4244269132614136, -0.810127854347229, 0.8086804747581482, -0.06636679172515869, 0.4144684970378876, 0.3266717195510864, 0.9253680109977722, 0.15582513809204102, 0.9457692503929138, -1.723357081413269, 0.5866923928260803, 0.49207577109336853, 0.4124683737754822, -0.49771368503570557, -0.6843350529670715, 0.16449368000030518, 0.2836282551288605, 0.9695442318916321, -0.3506760001182556, -0.22044427692890167, -1.2508833408355713, 0.24647007882595062, -0.8090466260910034, -0.6608344316482544, -0.4795777201652527, 1.9281903505325317, -1.476576328277588, 0.7305466532707214, -0.45931658148765564, -0.5388355255126953, 0.2599584460258484, 1.0841631889343262, 0.3656582534313202, -0.2008282095193863, -0.8178817629814148, 0.5265145897865295, -2.2626492977142334, 0.7398582100868225, -0.4819464683532715, -1.278222680091858, -0.5112669467926025, -0.6037667393684387, -1.3010092973709106, 0.11470969021320343, 0.09812570363283157, 1.5494557619094849, 0.5579748153686523, -1.6481260061264038, 0.3273615539073944, -1.1834728717803955, 0.7218566536903381, -0.23720020055770874, 0.32889196276664734, 0.9867391586303711, 1.5872139930725098, -0.5014363527297974, -1.114923119544983, 0.15251900255680084, -0.6533536911010742, 1.7841968536376953, -1.99691903591156, -0.7033990025520325, -0.4040369987487793, 0.28695473074913025, 0.7155835032463074, 0.5538813471794128, -1.133569598197937, 0.9755663871765137, -0.8206457495689392, 1.1947475671768188, -0.3172340989112854, 1.8538694381713867, 0.42720597982406616, 0.4513336420059204, 0.867088794708252, -2.3321609497070312, 1.0975159406661987, -0.46270033717155457, -0.5005361437797546, -0.35191333293914795, 0.15425673127174377, 0.48337915539741516, 2.3947367668151855, -1.1977274417877197, 1.489646315574646, -0.20559020340442657, 0.49203038215637207, -1.72129487991333, 0.4085773229598999, 1.2751719951629639, 1.587278127670288, 0.09525792300701141, -0.6690310835838318], [-0.7227270603179932, -1.5392556190490723, -0.327926367521286, -1.002658486366272, -0.22583021223545074, -1.2307322025299072, -1.7331464290618896, 1.735391616821289, 0.554397702217102, 0.3014255464076996, -0.9114127159118652, 0.16005145013332367, 0.1927517205476761, 0.1848336011171341, 1.1709709167480469, 0.703300952911377, -0.5657135248184204, 0.025896096602082253, 1.0129753351211548, -0.39892578125, 0.6299530267715454, 1.1189316511154175, -0.8962699174880981, -2.016576051712036, 1.0591851472854614, -0.8289322257041931, 0.7016949653625488, -0.976728081703186, 1.2837456464767456, -0.263886034488678, 0.5020283460617065, -1.357987880706787, 0.6173121929168701, 0.13604868948459625, 0.17423072457313538, 1.5302000045776367, 0.09739042073488235, -0.49729540944099426, 0.04599743336439133, -0.7673031687736511, 0.4393472969532013, 0.6539523601531982, -2.268948793411255, 1.426306962966919, -0.6629169583320618, -0.8999284505844116, 0.8764386773109436, -1.0217149257659912, -1.724467158317566, -0.3306233882904053, -1.0004216432571411, 0.7360506057739258, -0.08889415860176086, -1.7004053592681885, 0.07132013887166977, -0.6470441222190857, -0.225425124168396, -0.9829319715499878, 1.1476796865463257, -0.46460920572280884, 0.4361906349658966, -0.5120248198509216, 1.0344486236572266, 0.34080028533935547, 0.11057468503713608, 1.3271937370300293, -1.976269006729126, -1.6902165412902832, 1.0739405155181885, 0.37876051664352417, -0.7514442205429077, 0.765631914138794, -1.367920160293579, 0.19621725380420685, 0.3632335364818573, 1.2109400033950806, -0.6489943861961365, -0.3016470968723297, -0.8933737277984619, 0.018188467249274254, 1.1690927743911743, 0.5860543847084045, 0.5539555549621582, 0.2834498882293701, 0.9970256686210632, -0.9717835187911987, -1.3027088642120361, 1.1971800327301025, 1.1678597927093506, -0.8029330372810364, 1.8617820739746094, -0.43511122465133667, -0.08691809326410294, -0.09407420456409454, -0.3554612100124359, -2.9076247215270996, 1.351184606552124, 0.0996985211968422, 2.017228126525879, -1.3815455436706543, -1.2984763383865356, -0.7355647683143616, 0.44281578063964844, -1.1712990999221802, -0.8915325403213501, -1.582457184791565, 1.2589694261550903, 0.22326253354549408, -2.0453929901123047, 1.1100419759750366, -1.3980189561843872, 0.8044938445091248, 0.18044282495975494, 0.5610438585281372, -0.08607050776481628, 0.6280252933502197, -1.081897497177124, 0.5672657489776611, -0.8591939210891724, -0.8948452472686768, -0.05472322553396225, -1.749219536781311, 0.8392342925071716, 0.46242740750312805, -0.1853267401456833, 0.4014819264411926, -0.021690072491765022, 0.08327068388462067, -0.21451832354068756], [-0.05412834510207176, -2.4673573970794678, 0.5682385563850403, -0.16644538938999176, 0.25047117471694946, 1.2531133890151978, 0.65437251329422, -1.7001219987869263, -0.8409892320632935, 0.4999430775642395, -0.45019495487213135, 1.0647085905075073, -0.06606623530387878, 0.7445269227027893, 0.4407636821269989, -1.4664779901504517, -1.1450670957565308, 1.2531991004943848, 2.317448616027832, 1.3803499937057495, -1.1017179489135742, 3.3866617679595947, 0.22175714373588562, 0.982786238193512, -0.7938669323921204, -0.6693069934844971, 1.5444307327270508, -0.4465011954307556, 0.5839235186576843, -1.027463436126709, -0.5432330369949341, 1.5843538045883179, 1.3788056373596191, 0.48645830154418945, 0.3985166847705841, 0.5211160778999329, -0.145124614238739, 0.5411804914474487, -1.9467679262161255, -0.5634820461273193, -0.4971564710140228, 0.8488296866416931, 0.20060144364833832, -1.7148584127426147, 1.2730878591537476, 0.4899405837059021, 0.6169427037239075, 1.6591606140136719, 2.6511852741241455, -0.4878091514110565, -2.666109561920166, 0.07990452647209167, 1.004832148551941, 1.3068156242370605, -1.191483497619629, -0.27064716815948486, -0.07026772201061249, 1.4700993299484253, -0.35035401582717896, 1.7671058177947998, 1.045813798904419, -0.626123309135437, -0.9297590255737305, 1.221636414527893, -0.6031032800674438, 0.04541999474167824, 0.9114776253700256, -0.25620922446250916, -2.0959107875823975, -0.18037375807762146, -0.16949711740016937, 0.8195289969444275, -0.8933489322662354, 0.13033296167850494, -1.2392014265060425, -1.3969931602478027, -0.900166928768158, 0.26045218110084534, -0.26572611927986145, 1.4489177465438843, 1.6005237102508545, -0.4363512098789215, -1.4064795970916748, 0.3220770061016083, -0.8907853960990906, 1.3580201864242554, -0.6043823957443237, -2.4996588230133057, 0.5016751885414124, -0.7449815273284912, -0.8075138926506042, -0.635076105594635, -0.39301276206970215, 0.519789457321167, -0.4552907645702362, 0.5792872905731201, 0.3127136528491974, 1.0433743000030518, 0.7932568192481995, 1.2850744724273682, 0.5813409090042114, 0.27254146337509155, 0.9141282439231873, 1.0628771781921387, 0.9494888782501221, 0.8394521474838257, 0.19751279056072235, -1.9632680416107178, -1.5291293859481812, -1.0615326166152954, -0.6738331913948059, 0.18074001371860504, 0.1790669858455658, -0.044485267251729965, 1.5766892433166504, -1.307528018951416, -0.6137707233428955, -0.3443671464920044, -0.7708815336227417, 0.2927432060241699, 0.5639531016349792, 1.7580941915512085, 0.40961429476737976, 1.081870436668396, 1.873834490776062, -0.04267342388629913, 0.4281776547431946, -0.09547629207372665, -1.5290476083755493], [-0.816095232963562, 1.7107956409454346, 1.0896575450897217, 0.13458527624607086, 0.3043872117996216, 0.3331925570964813, -0.1843324601650238, -1.0529484748840332, 0.6663655638694763, 0.6464582085609436, -0.836997926235199, 0.5519359111785889, -0.019205721095204353, 0.09011533856391907, 0.6732496619224548, 1.3135672807693481, -0.07596375793218613, -0.6502915024757385, -0.5920378565788269, -0.29030272364616394, 0.12735112011432648, 1.0539612770080566, 0.5424553751945496, 0.032838545739650726, -0.12446938455104828, 1.532011866569519, 1.2546244859695435, -0.5776970982551575, 0.6625683307647705, -0.955867350101471, 0.8427038192749023, -2.1551835536956787, -0.553684651851654, 1.0764765739440918, -0.5741804838180542, 0.040330611169338226, 1.0692254304885864, 0.01650371588766575, 0.8338656425476074, 0.7985245585441589, -1.7792407274246216, 0.8426321148872375, -1.388218879699707, -0.4525682330131531, -0.3434221148490906, -0.6722365021705627, 0.23982252180576324, -0.3045749068260193, -0.16684134304523468, -0.23788173496723175, -0.7145661115646362, -0.17545394599437714, 0.40943384170532227, -0.8156147003173828, -0.5696172118186951, -0.26373302936553955, -1.6703219413757324, 0.9566150307655334, 0.22563476860523224, -0.040041763335466385, -1.7442207336425781, 0.7310177087783813, 0.8005783557891846, 0.46825727820396423, -0.03298141434788704, -0.7440661787986755, -0.927964448928833, 0.8290479779243469, -1.0385888814926147, -1.2135556936264038, -0.9695783853530884, -0.174159437417984, -1.736399531364441, 1.2627288103103638, 1.1849254369735718, 1.8300915956497192, 0.2742890417575836, -0.08892647176980972, 0.6656615138053894, 2.1019034385681152, -0.7117426991462708, 1.2474558353424072, -1.3321399688720703, 0.42439284920692444, 0.2723226249217987, -0.9895179867744446, 0.7376602292060852, -0.37688031792640686, 1.1453198194503784, -0.05423225834965706, 0.30404314398765564, 0.5750851631164551, -0.006903196219354868, 0.8087783455848694, 0.22830834984779358, 0.017290258780121803, -1.0805577039718628, 0.08615145832300186, -1.1516684293746948, 1.0196871757507324, -1.3605165481567383, -0.2854599058628082, 0.8944607973098755, -0.7941858768463135, -0.6519840359687805, -1.6970486640930176, 0.8183531165122986, 0.24464333057403564, -0.48323854804039, -0.03469757363200188, 0.7130065560340881, 0.8057190179824829, -0.8802980184555054, -0.7752498984336853, 0.01210907381027937, -0.2282024621963501, -0.258329838514328, -0.342480331659317, -0.9173052310943604, -0.24264784157276154, 0.6316514015197754, -0.10087878257036209, 0.34361106157302856, 0.04024583846330643, -0.2119513750076294, -2.1116507053375244, 0.10216661542654037, -0.9724658131599426, -0.42380186915397644], [1.1039395332336426, 0.5599616169929504, -0.0017206310294568539, 1.426657795906067, 1.0032161474227905, 0.11977250128984451, 0.17104144394397736, -0.14757658541202545, -0.7198737859725952, 0.08805525302886963, 1.2519036531448364, -0.640590250492096, -0.3404003381729126, 0.08285968750715256, -0.6487175226211548, -0.03286612033843994, 0.9995275139808655, -1.6320281028747559, 0.6385738253593445, 0.4742089509963989, 0.146891787648201, 1.1482434272766113, 1.2194023132324219, 0.5832836031913757, 0.7579065561294556, 0.3306032121181488, 1.186803936958313, 0.5417991280555725, -1.9211831092834473, 1.3435354232788086, 0.9166398644447327, 0.5568447113037109, 0.6264888644218445, 0.5816038250923157, -0.23565147817134857, 2.0310938358306885, 0.8863360285758972, -0.27180758118629456, -0.07136189192533493, -0.7043863534927368, -0.4823087453842163, -0.427742600440979, -0.11061267554759979, 0.08649259060621262, 0.6952962279319763, -0.29080429673194885, -1.0601645708084106, -0.18696574866771698, -0.10902658849954605, -2.6843016147613525, 1.7177791595458984, -0.49766767024993896, 0.11768966168165207, 0.3038567900657654, -0.40247154235839844, 1.2018355131149292, -0.78999263048172, -0.41222307085990906, 0.938748836517334, -0.4374505281448364, -1.0828428268432617, 0.6769949793815613, -0.568040132522583, 0.7665150761604309, -0.6301554441452026, 0.009899023920297623, 0.13500364124774933, -1.121845006942749, -0.46051105856895447, -0.09826690703630447, 0.8180686831474304, 0.4406748116016388, 0.04787361994385719, -0.5657392144203186, 0.286028653383255, 0.5021212100982666, 1.201863169670105, -1.4394879341125488, -1.0996334552764893, -0.389487624168396, -0.1895926296710968, -1.3906893730163574, 0.5700244307518005, -1.017229437828064, 1.3045881986618042, -0.5979753136634827, 0.8858762979507446, -2.0025389194488525, 0.5223963856697083, 0.9979363083839417, 0.26225951313972473, 1.844722867012024, 0.4729405641555786, -0.11785271763801575, -0.6512410640716553, -0.4908190369606018, -0.13198864459991455, -0.8469780087471008, 0.937942385673523, -0.6834215521812439, 0.34976688027381897, -0.5354777574539185, -0.2386346161365509, 0.03738504648208618, 0.49628663063049316, 1.0978351831436157, -0.5006980895996094, -1.5698928833007812, 0.8310444951057434, -1.534877061843872, 0.04822933301329613, 1.620313048362732, -1.2801204919815063, -1.4931451082229614, -0.8649594783782959, -2.427732229232788, -0.9434708952903748, 0.1779729127883911, -0.4769952893257141, 0.5024418234825134, -0.27425816655158997, 0.9884631037712097, -1.679824948310852, -0.2698262929916382, -0.1352481245994568, 1.205613374710083, -0.3595814108848572, 1.7206159830093384, 0.2569323778152466], [-0.2885013222694397, 0.6647588014602661, -0.8410112261772156, -0.7250093817710876, -0.12413393706083298, 0.8416810631752014, 0.45272064208984375, -2.885768175125122, -0.37443748116493225, -0.16820128262043, 1.5265427827835083, 0.40666013956069946, 0.48576751351356506, 0.30721837282180786, -0.4326312839984894, 0.8780053853988647, -1.6659200191497803, 0.8124339580535889, 0.9007622599601746, 1.5251997709274292, 0.038091015070676804, 1.617720603942871, -0.2615105211734772, 1.646259069442749, 0.031788170337677, -0.2457527369260788, -0.3511139750480652, 0.2597126364707947, 0.7288905382156372, -0.3866921663284302, 0.41891300678253174, 0.13880908489227295, 1.4534189701080322, 0.4329077899456024, -0.24690844118595123, 1.1007179021835327, 0.048411861062049866, 1.2771590948104858, 1.5534439086914062, 0.0859917551279068, 0.9973823428153992, 0.8283072710037231, 0.24095354974269867, 1.3716554641723633, 0.35300305485725403, -2.173985719680786, 0.7955456972122192, -0.7991901636123657, -0.12626194953918457, -0.4847562909126282, -0.004299856256693602, 1.4275665283203125, -0.559016764163971, 2.6752171516418457, 0.1893487423658371, 0.649150550365448, 0.5075315237045288, 1.7414313554763794, -0.22673065960407257, 0.8565385341644287, 0.5390394330024719, 2.348918914794922, 1.0974847078323364, 1.8098304271697998, -0.6427667737007141, -0.15489475429058075, 1.1378757953643799, 0.21305282413959503, 0.7638290524482727, 0.2435149848461151, -0.3757114112377167, -0.3355371356010437, -0.9050394892692566, 0.9684725403785706, -1.7461086511611938, -1.2076706886291504, 0.986927330493927, 0.010096835903823376, 1.2248667478561401, -2.1123194694519043, 0.16163350641727448, -1.5493571758270264, 0.8435709476470947, 0.6279240250587463, 1.2268004417419434, 1.3284696340560913, 0.5627357363700867, -0.600879967212677, 1.006941795349121, 0.4842560589313507, 0.42019909620285034, 0.42468786239624023, 1.0224391222000122, 1.127318263053894, -1.5627647638320923, 0.37020766735076904, -0.5537461042404175, 0.13461273908615112, -0.6097626686096191, -0.013352707959711552, -0.6127506494522095, 0.5430022478103638, -0.5146881937980652, 0.26976603269577026, 1.0593743324279785, 1.843667984008789, -0.029047876596450806, 0.5675662755966187, 1.4718259572982788, -0.7726732492446899, -0.6699153184890747, -1.5750422477722168, 1.7390661239624023, 0.4537779688835144, -0.976703405380249, 0.9829500913619995, 0.5878359079360962, 0.14977054297924042, -2.2018699645996094, 0.4159015715122223, 0.7251715660095215, 0.4406741261482239, 0.5964178442955017, -0.4039306640625, 1.2069882154464722, -0.17951607704162598, -1.172527551651001, 1.9643640518188477, 0.1666816622018814], [0.017660103738307953, -0.12676148116588593, -0.08356212824583054, 0.7459379434585571, 0.0002651085960678756, -0.295712947845459, 0.26359036564826965, -0.7641584873199463, -1.0435043573379517, -0.12644653022289276, 1.1197736263275146, -1.34917151927948, -1.325921893119812, 0.5805164575576782, -0.04963601753115654, -0.9895777106285095, 0.8014488220214844, 1.1659311056137085, -0.3970281183719635, -0.7464744448661804, 1.2239371538162231, 1.5070281028747559, -0.7282072901725769, 1.0681836605072021, 1.3415333032608032, 1.1268033981323242, 0.8682869672775269, -2.42507266998291, 1.971318006515503, -0.08582288026809692, 0.2862100899219513, -1.4322837591171265, 1.7571783065795898, 0.4587542414665222, -1.3916258811950684, -1.2640701532363892, -0.24335168302059174, -1.277505874633789, 0.4126465320587158, -0.2687242329120636, -1.8978065252304077, 1.0959806442260742, -0.47490450739860535, -0.5563321113586426, 1.0942256450653076, -0.5800398588180542, 0.8774967789649963, -0.03420695662498474, -0.4046679735183716, 0.2724045217037201, 1.8177096843719482, -0.42230144143104553, -0.5188937187194824, -0.12571817636489868, -2.3002493381500244, -0.523215115070343, 1.1750202178955078, 0.940142810344696, -0.2962561249732971, -1.5699626207351685, 1.0200417041778564, -0.22700099647045135, 1.0364627838134766, -1.0077879428863525, -0.8598008155822754, -0.5569244623184204, 0.7982049584388733, -0.08407622575759888, 0.7089303731918335, 0.9383787512779236, 0.4828869104385376, 1.9944959878921509, -0.43429139256477356, -1.1810951232910156, 0.5636975765228271, 1.2919293642044067, -0.008032726123929024, 1.7978578805923462, 0.8454493284225464, -1.0485336780548096, -0.35749268531799316, 0.4647310674190521, 0.980577826499939, -0.5964018106460571, -0.44231778383255005, -0.4348975121974945, -0.14727531373500824, 0.0646602138876915, -0.8841291666030884, 0.28321099281311035, 0.02745826356112957, -0.5596620440483093, 0.5752711892127991, 0.6033873558044434, -1.1085054874420166, -1.1816710233688354, 0.678880512714386, 0.592191219329834, -0.28986555337905884, 0.9523940682411194, 0.1312241554260254, 1.734290599822998, -0.5481730103492737, 1.3372149467468262, -0.07035170495510101, -1.1075115203857422, 0.3778909742832184, -0.7176176309585571, 0.49329227209091187, -1.1442309617996216, -0.4716647267341614, 1.5392837524414062, 0.545974612236023, 1.2143117189407349, -0.741092324256897, 0.18967179954051971, 0.2220907211303711, 0.47570544481277466, -0.10466200858354568, 2.277097225189209, 1.3207056522369385, -0.0934504047036171, -1.683387041091919, 0.8461707234382629, 2.4182348251342773, -0.32514411211013794, -0.31581902503967285, 0.47635507583618164, 0.7637384533882141], [0.2166932076215744, -2.5738964080810547, -0.6165157556533813, -0.948123037815094, -0.8964441418647766, -0.7351003885269165, 0.8929922580718994, 0.45544421672821045, 0.7955083847045898, 0.851805567741394, -0.8641775846481323, 1.928797721862793, -2.549851655960083, -0.9952656030654907, 0.3407648205757141, 0.15031877160072327, -1.6640348434448242, 1.2923868894577026, 0.1262122243642807, -1.904167890548706, -0.38428375124931335, 0.4986134469509125, 0.402591735124588, 0.6740179061889648, 1.194000244140625, 1.6866238117218018, -0.6002762317657471, 0.601935625076294, 0.3249552845954895, -0.714921236038208, -1.2191786766052246, 1.3846145868301392, -0.1905333697795868, -0.6697138547897339, 0.810969352722168, 0.8176414370536804, -0.6033579111099243, 0.8064755201339722, -0.32524916529655457, -0.6624409556388855, 0.1863335818052292, -0.18029670417308807, 1.4610530138015747, 0.4526507556438446, 0.8514428734779358, 0.0954202264547348, 0.8962705731391907, 0.6572434902191162, 1.542636513710022, 0.5740310549736023, 1.0879508256912231, 0.7704988718032837, 0.8563479781150818, 0.7767040133476257, 0.4295506179332733, 1.8727340698242188, -0.241852805018425, -0.31501129269599915, 0.82338947057724, -0.19980116188526154, 0.9982963800430298, 0.5386070609092712, -0.270559161901474, 0.08637259155511856, -0.3839356303215027, -0.42377299070358276, -0.8578647971153259, 1.393800139427185, 1.1393799781799316, -1.1320692300796509, -0.22762608528137207, 0.9356856346130371, 1.138535976409912, -1.0946710109710693, 0.9323068261146545, 0.03097480908036232, -0.07576406747102737, 0.09716877341270447, 0.5961503982543945, -0.6338706016540527, -2.6011412143707275, 0.16730229556560516, 1.0227922201156616, -0.30927762389183044, 1.4547595977783203, 1.472703218460083, -1.061794638633728, -0.44412046670913696, -1.2921481132507324, -0.4898144006729126, 0.1937698870897293, 0.154444620013237, -0.035132355988025665, 1.48555326461792, -0.5093367099761963, 0.09600323438644409, -0.5496701002120972, 0.304826557636261, 0.426362007856369, -1.6591891050338745, -0.12574326992034912, -0.6857088804244995, 1.318851113319397, 1.0355114936828613, -0.46766695380210876, -1.9105483293533325, -0.20819886028766632, 0.27247342467308044, 1.878764033317566, 0.5637543201446533, 0.7280988693237305, 0.5032931566238403, 0.19175425171852112, -0.7663258910179138, 0.053410932421684265, -1.7633020877838135, -0.037536460906267166, -1.6204469203948975, -1.3032901287078857, -0.5149194002151489, 0.018887635320425034, -0.8172043561935425, 0.01899339258670807, 0.021233076229691505, 1.0363093614578247, 1.418637990951538, -0.8970457315444946, -1.0543360710144043, -0.0741652175784111], [1.4316142797470093, 0.6194774508476257, 0.0646899938583374, 0.09184237569570541, 0.62910395860672, -0.001459730789065361, 0.046133022755384445, -0.43923282623291016, 0.4998244643211365, 0.6993417739868164, -0.27725788950920105, 0.9659627676010132, -0.5916270017623901, -1.5292330980300903, 0.6250689029693604, -0.6380763053894043, 0.15977993607521057, -0.20348690450191498, -1.5879732370376587, -0.6119983196258545, -0.3022507131099701, -0.4439162611961365, 0.11523110419511795, 0.37096261978149414, 0.3317716717720032, 0.4435049593448639, 2.115773916244507, -1.4153157472610474, 0.013858462683856487, 1.5448548793792725, -0.011495329439640045, 1.852423906326294, 0.9251219630241394, -1.8645915985107422, 0.971884548664093, 0.4282490313053131, 1.2633144855499268, -0.1540941596031189, -1.3708430528640747, -0.3033265471458435, 0.8496472835540771, 0.360248863697052, -0.9591386914253235, -0.6371216773986816, -0.44026997685432434, -1.8448777198791504, -0.3363751769065857, 0.30061179399490356, 0.2741324305534363, -0.03279919549822807, -0.9341040849685669, 1.0468463897705078, 1.6580933332443237, 0.5160824656486511, -0.4344577491283417, 0.712908148765564, -0.1540573239326477, -0.44173604249954224, -0.7497485280036926, -1.1391838788986206, -1.2438687086105347, -1.6419745683670044, -1.5677423477172852, 0.2338312566280365, -0.5937424302101135, -0.013522637076675892, -0.03315156325697899, 1.3169820308685303, -1.8131378889083862, -0.42704063653945923, -1.3442354202270508, 0.14477790892124176, -0.18113824725151062, 0.4574897587299347, 1.178091287612915, 0.26511627435684204, 1.3345669507980347, -0.7328084707260132, -1.833483099937439, -1.4514997005462646, -1.548580288887024, 0.3903972804546356, -0.982041597366333, 0.08019540458917618, 0.3935180902481079, 0.2412148416042328, -0.08252684772014618, -1.0098795890808105, -0.04060042276978493, -0.3813971281051636, -1.172053337097168, -1.4631257057189941, 0.6996896266937256, -2.541520118713379, -0.28347912430763245, 0.7914343476295471, -0.60056471824646, -0.45300135016441345, 1.4214935302734375, -0.22944937646389008, 0.1836170107126236, -1.4505048990249634, -0.05267002806067467, -0.005859578028321266, -0.2851005494594574, -0.9595624208450317, -0.3073759377002716, -0.48938775062561035, 0.23714329302310944, -1.2769912481307983, 0.7818389534950256, -2.285504102706909, 0.29745590686798096, -1.0159393548965454, 0.8080628514289856, 0.7267820239067078, 2.632779121398926, 0.0008187192142941058, 0.37230080366134644, -0.6021263599395752, -1.701340913772583, 0.5689869523048401, 0.6911436915397644, -1.7796530723571777, 0.651524007320404, -1.3350365161895752, 2.9535322189331055, 1.2709987163543701, -0.27113959193229675], [2.3435919284820557, 0.6812953352928162, -0.32484033703804016, 0.6550211906433105, 0.3220774531364441, -0.498216450214386, -1.3300915956497192, 0.027850858867168427, 1.7396612167358398, -1.314131259918213, -0.3895414173603058, -1.7999757528305054, 0.7741157412528992, 1.0776290893554688, -0.5926639437675476, 0.23326939344406128, 1.608350157737732, 1.2353492975234985, 0.2022462785243988, 0.9678325057029724, 1.1843489408493042, 1.9171528816223145, 0.17165875434875488, -0.7989676594734192, -0.6518952250480652, -0.8422529697418213, 0.014069624245166779, -0.410001277923584, -0.07853903621435165, -0.1883613020181656, -1.3228821754455566, 0.10363614559173584, 1.2298979759216309, 0.08549965173006058, 0.11816554516553879, -0.8101209402084351, 0.42480984330177307, 0.8305981755256653, -0.5471042990684509, -0.9977206587791443, 0.441928505897522, -0.5946714878082275, 0.6139407157897949, -0.10482964664697647, -0.7858790755271912, -0.7616807818412781, -0.1770731806755066, 1.286051869392395, 0.037608757615089417, 1.3251450061798096, -0.2811131775379181, -0.4552290141582489, 0.12324751168489456, -0.3233553469181061, -0.8127326369285583, 1.4364691972732544, -0.4607786536216736, -1.5489535331726074, -1.2198262214660645, 0.5838906168937683, -0.7707348465919495, -1.3631222248077393, 0.23111121356487274, 0.1719854176044464, 0.5144423842430115, -1.0815016031265259, 0.7501005530357361, -0.3322352468967438, -0.6477884650230408, 0.3861459493637085, -0.8720112442970276, -1.7805145978927612, -1.2375887632369995, -0.5215585231781006, 0.06338593363761902, 1.1532608270645142, 0.49465858936309814, 0.9204358458518982, -1.662980318069458, 0.09318486601114273, 0.23307733237743378, 0.6216942071914673, -1.2286134958267212, -0.8495146632194519, -1.82184898853302, 0.05894092470407486, 1.194252371788025, 1.1003870964050293, -0.1870122104883194, 1.773195505142212, 0.7218529582023621, -0.0840209424495697, -0.013740172609686852, 1.8355621099472046, 0.2618564963340759, 0.9332343935966492, 0.262077271938324, 1.1102906465530396, 0.09843911230564117, -1.1121500730514526, -0.7737747430801392, -0.2903217673301697, 0.08568153530359268, -0.9598042368888855, 0.30855873227119446, -0.6739003658294678, 0.5848198533058167, -0.48510870337486267, -0.4666295647621155, -0.2753288149833679, 1.1823686361312866, 0.55558180809021, 2.2963595390319824, 1.0973317623138428, 2.1824042797088623, 0.7818572521209717, 0.8639910817146301, -0.17076915502548218, 1.1160153150558472, -1.482622504234314, -0.3716733455657959, -0.750903308391571, -0.88802570104599, 0.2804747223854065, 1.2836939096450806, -1.8139431476593018, 0.5734760761260986, 0.9406799077987671, 0.3806000053882599], [-1.0098191499710083, -0.47732144594192505, -1.4188380241394043, 2.0287983417510986, -1.3978896141052246, -0.3293834924697876, 0.299726665019989, 0.1199006661772728, 1.1673024892807007, 0.3235822021961212, -0.727857232093811, 0.635759711265564, 0.27705684304237366, 0.5392943024635315, -0.6557150483131409, 1.103550672531128, -1.3106056451797485, -0.07861846685409546, 0.821199893951416, -0.22000913321971893, -0.44698286056518555, -0.006709638051688671, -0.38232216238975525, 0.7799838781356812, -0.821527898311615, 0.7587224245071411, 1.3151553869247437, 0.636040210723877, -0.6233848333358765, -1.3103116750717163, -0.6800044178962708, 1.44290030002594, -0.36680302023887634, -0.7291048169136047, -0.008088365197181702, -0.4727233052253723, -0.7720364928245544, -0.22023804485797882, 1.021738886833191, 1.433050513267517, -1.0301786661148071, 0.527587354183197, 1.7457484006881714, -0.5022115707397461, 0.21781478822231293, 1.2074640989303589, -1.2349724769592285, 0.028887681663036346, 0.7296512126922607, 1.274733066558838, 0.2884959280490875, 0.17833778262138367, 1.0642691850662231, 0.8452509641647339, -1.1354522705078125, 0.514710009098053, 0.5805478096008301, 0.2289908081293106, 0.6591864228248596, -0.8581199049949646, 2.712451696395874, 0.5156720876693726, -0.001370217651128769, 0.49331897497177124, 1.9483619928359985, -0.12837016582489014, 0.38350483775138855, 0.049021150916814804, 0.17905132472515106, 0.30898475646972656, -0.75092613697052, 0.07328549027442932, -0.2775042951107025, -2.0276923179626465, 1.6450504064559937, -0.24458041787147522, -0.07560339570045471, -0.8111403584480286, -0.7777935862541199, -0.4820433259010315, 0.5927846431732178, 0.41445887088775635, -0.823198676109314, -0.06205850839614868, 0.14006590843200684, -1.057214379310608, 0.9934715628623962, 1.0196598768234253, -0.30455324053764343, -1.701554536819458, -1.1423593759536743, 3.150428295135498, -0.8967899680137634, -1.0769338607788086, -0.31961095333099365, 2.6332011222839355, 0.026624498888850212, 0.0063707553781569, 1.2315871715545654, 1.837668538093567, 0.0656922310590744, -0.5300332903862, -0.27182844281196594, 0.7000930309295654, 0.34086915850639343, -0.06047075614333153, -1.151269793510437, 0.059734195470809937, 2.308027505874634, -1.8076497316360474, 0.07206865400075912, 2.0047812461853027, -0.32094115018844604, -0.7762245535850525, 1.7611860036849976, -0.7832421064376831, -0.13887587189674377, 0.6456026434898376, 1.340039610862732, 1.4539809226989746, -0.027085622772574425, 0.9644222855567932, 0.27930861711502075, -0.07686682045459747, 0.5127880573272705, 1.838570237159729, 0.3198908865451813, 1.0281481742858887, 0.5667136907577515], [-1.117586612701416, -0.09972892701625824, 1.1244444847106934, -0.32719239592552185, -0.5349976420402527, 0.628652036190033, -1.8856757879257202, 0.4545834958553314, 0.6976591944694519, 0.7275388240814209, 0.0771782398223877, 1.1371781826019287, 0.4259307086467743, 0.300976425409317, -1.3619284629821777, -0.0939461961388588, -0.8872450590133667, 2.3381693363189697, 0.5253874063491821, 1.2010462284088135, -0.22674259543418884, 0.4438481032848358, -2.51934814453125, -1.0564903020858765, 0.048540957272052765, 0.6039850115776062, -1.8261018991470337, -0.0680026262998581, -0.19145092368125916, 0.41377875208854675, 0.6573363542556763, 0.8117632865905762, -0.5852633118629456, 1.0453505516052246, -0.041553132236003876, 0.9320446252822876, 0.9983375072479248, 0.5495843887329102, 0.7617411613464355, 0.2959917187690735, 0.39383959770202637, 0.614847719669342, 1.2074437141418457, 0.8077600002288818, -2.7974741458892822, 0.47831669449806213, 0.46873098611831665, -0.7506119012832642, -0.5688950419425964, -0.045746006071567535, -1.0786515474319458, 0.22886714339256287, 0.13130690157413483, 0.5942419171333313, 0.570114254951477, -0.22980481386184692, -0.6335793733596802, -0.13994482159614563, 1.540581226348877, 0.45230209827423096, 0.5356227159500122, 1.011641263961792, 1.5953798294067383, -0.17398172616958618, 0.3241793215274811, -0.38817086815834045, -0.7292378544807434, 0.5150806307792664, 2.30118989944458, -0.6727657318115234, -0.142013818025589, 0.8574985861778259, -0.17609365284442902, -0.09857211261987686, 0.12489072233438492, -0.10089725255966187, -0.44653865694999695, -1.4985382556915283, 0.4820888042449951, -1.0449411869049072, 0.05972930043935776, -0.9203691482543945, -0.16274000704288483, -0.10803087800741196, -0.04358698055148125, -0.20171219110488892, 0.9282639622688293, -0.6458074450492859, 0.20646420121192932, -0.7491084933280945, -1.0933101177215576, -0.9693618416786194, 1.0908445119857788, 0.1355823129415512, 2.2629826068878174, 0.09642812609672546, 0.9943593144416809, -1.4415714740753174, 1.522889256477356, -0.03405948355793953, -0.6631225347518921, 0.07708556950092316, -0.8971555233001709, -0.22371557354927063, -0.2727430760860443, 0.6875066161155701, -0.5604525804519653, -0.5475027561187744, -0.4941365420818329, 1.0031325817108154, 1.4348326921463013, -0.5552107691764832, -1.5754421949386597, -0.06732256710529327, 2.1539206504821777, 0.3363496661186218, 1.6833423376083374, 0.9850143790245056, -1.4389606714248657, 0.01507622841745615, -1.0239125490188599, -1.109395980834961, 0.47623300552368164, 0.624276340007782, 1.3100919723510742, -0.3252319097518921, -1.8271925449371338, 0.8548552393913269, -1.905232310295105], [-1.85906982421875, 0.028793323785066605, 1.4054125547409058, 2.439757823944092, -0.44011202454566956, 0.7679985165596008, -1.4226772785186768, -2.6776630878448486, -0.263234943151474, 0.981243371963501, -0.845401406288147, -0.3845711648464203, 0.4663889408111572, 0.2367151528596878, 0.8329944610595703, -2.44384765625, 0.9133069515228271, -0.8876228332519531, -0.5788016319274902, 1.2506734132766724, -0.7407369613647461, 0.4168046712875366, -0.44255080819129944, 0.5549665689468384, 0.4463057219982147, -1.4716808795928955, 0.5177756547927856, 0.7337731719017029, -0.2665312886238098, 1.7937490940093994, 1.1230663061141968, 1.0610512495040894, 0.5232272744178772, -0.24389147758483887, 1.390350580215454, 0.7287653684616089, -0.33244264125823975, -0.3615550100803375, 1.3656660318374634, 1.9205756187438965, -0.018022537231445312, 0.782158613204956, 0.156919926404953, 0.8306781053543091, -1.280559778213501, 1.1778143644332886, -0.3953045606613159, -0.07813297212123871, 0.20600652694702148, 0.2857081890106201, 1.077551245689392, 0.5789247751235962, -0.5712556838989258, 1.6334083080291748, -0.9547020196914673, -0.04883711412549019, 1.4444602727890015, 0.4892932176589966, 0.1564946323633194, -0.00653365021571517, 0.8266197443008423, -0.5289179682731628, -0.12651415169239044, -0.3775208592414856, 0.5142434239387512, 0.7967632412910461, -0.32726508378982544, -0.7910529971122742, 1.1186965703964233, 0.6883633732795715, -1.5092710256576538, -0.31358200311660767, 0.8697831630706787, -0.10932542383670807, -0.6840065717697144, 2.1902592182159424, -1.545241117477417, -0.7443841695785522, -0.8973836898803711, -1.1276124715805054, 0.11429920792579651, 0.29613247513771057, -2.2663605213165283, -0.1371987760066986, -0.8682814240455627, 1.2804951667785645, 0.2748582363128662, -0.1479998230934143, -1.0655821561813354, 0.12055741250514984, 1.814355731010437, 0.7514727115631104, -0.9538478255271912, 0.5752508640289307, -1.0646331310272217, -0.29690101742744446, 0.7935397028923035, 0.15480442345142365, -0.12487305700778961, -1.0782012939453125, 0.26409026980400085, 0.06690200418233871, 0.07334459573030472, 0.1808524876832962, 1.4784388542175293, -1.1626131534576416, 0.17902658879756927, -0.9814482927322388, 0.3062613606452942, 1.1131516695022583, -1.3897773027420044, -0.42789459228515625, -0.49487531185150146, -1.5975769758224487, -0.1667906939983368, -0.429698646068573, 0.3668849766254425, 0.11501695960760117, 0.3525972366333008, 1.5342144966125488, -1.0127369165420532, 0.4755310118198395, 0.719219446182251, -0.24463781714439392, 0.05269716680049896, 0.5562245845794678, 0.5627001523971558, -0.5191733241081238, -1.7576662302017212], [-0.04881104454398155, -0.4576530456542969, 0.2658613920211792, -2.169442653656006, -0.25391316413879395, 1.1850529909133911, -1.1789577007293701, -0.7360130548477173, -0.2370212823152542, 0.6844599843025208, 0.16190679371356964, 0.6389201283454895, -0.3495885729789734, -1.3313847780227661, 0.8131932020187378, 0.7155925035476685, -0.35464659333229065, -1.6660826206207275, 0.5915394425392151, 0.042532797902822495, 0.01548011228442192, 0.402121901512146, -2.333513021469116, -0.46283087134361267, 0.21038909256458282, 1.587837815284729, 1.2931369543075562, 0.7754610180854797, -0.6746845245361328, -0.44425374269485474, -0.3411754071712494, 0.6326270699501038, 2.095057487487793, -0.3573465049266815, -0.8011265993118286, -0.06958215683698654, -0.900132954120636, -0.3248401880264282, 0.5972438454627991, -1.573796033859253, 0.36281901597976685, 2.238131523132324, -1.0527230501174927, -0.7572195529937744, -0.8463616371154785, 0.3272094130516052, -3.007251024246216, 1.1121364831924438, 0.217429980635643, 1.4067590236663818, 0.9469850063323975, -0.4305334687232971, -1.180446743965149, 1.2062835693359375, -1.1375508308410645, 0.012632003054022789, 0.44696494936943054, -0.7991341352462769, -0.7911036610603333, -0.2990685701370239, -1.1378892660140991, -0.11834928393363953, 0.9842812418937683, 0.09895046055316925, -0.976043164730072, 0.5332115292549133, 0.4510039687156677, 0.8872824907302856, -0.39170143008232117, -2.259862184524536, 2.2660906314849854, -0.4872160851955414, -0.8223388195037842, 0.2430635541677475, 0.5812488794326782, -0.5318171977996826, 0.6385173201560974, -0.48613834381103516, 0.405186265707016, 0.6690037846565247, 0.8884657025337219, 0.1480361521244049, -0.19685953855514526, -0.08985576778650284, -0.5586436986923218, -0.3836185932159424, -2.017075538635254, 1.1035720109939575, 1.8097326755523682, 0.26543399691581726, -0.6262725591659546, -0.7789071798324585, -1.7591183185577393, -0.8592321276664734, 1.034660816192627, -1.0591788291931152, -0.2458251714706421, -0.9629155397415161, 2.210862636566162, 0.6329268217086792, 0.22799310088157654, -0.806568443775177, -0.929422914981842, 0.47008249163627625, 0.20795400440692902, -0.504743218421936, -0.2662089467048645, 0.4783151149749756, -2.666668176651001, 0.15949636697769165, 1.1489392518997192, -0.19198746979236603, 1.6092149019241333, -0.018581125885248184, -0.5250956416130066, 0.8686173558235168, 1.8058499097824097, -0.5535743832588196, -0.15549053251743317, -2.92409610748291, -0.5863950252532959, 0.6900601387023926, 0.5662465691566467, -0.2219649702310562, -1.2969012260437012, -0.21986524760723114, 0.7752448916435242, -1.261275053024292, -1.5608617067337036], [-0.043801311403512955, -0.46754106879234314, -0.9015238881111145, -1.1346068382263184, 0.02892119809985161, -1.305725336074829, 1.5331295728683472, -0.20459270477294922, -0.14507895708084106, 0.4242616295814514, 1.5456265211105347, -0.014730535447597504, -0.39121341705322266, 0.18967342376708984, 2.0327672958374023, 2.0737574100494385, 1.0209312438964844, -2.650514841079712, -1.5939162969589233, -0.7754925489425659, 0.22522993385791779, -1.5338101387023926, 0.19364336133003235, -0.9354379177093506, 0.5333402752876282, 0.7538408637046814, 0.5350902080535889, -1.2605479955673218, -1.8871315717697144, 0.8766380548477173, -1.3540449142456055, -0.5505821108818054, -0.7255934476852417, -0.1633555144071579, 0.22523261606693268, 1.0336679220199585, 0.8098540306091309, -0.5999254584312439, -0.2883880138397217, -0.9459580779075623, -0.894597589969635, 0.08602545410394669, 0.07924528419971466, -0.04539871588349342, 0.37729811668395996, 0.34555235505104065, -0.06956595182418823, -1.0704158544540405, 0.28463852405548096, 0.3333560824394226, -0.39872413873672485, -0.4923075735569, -1.0071632862091064, -1.0107309818267822, 0.880564272403717, -0.043200526386499405, 0.14320558309555054, 1.023735523223877, 1.8433352708816528, 1.7064741849899292, -0.7148528695106506, -0.7424882054328918, -0.49560341238975525, 0.18447504937648773, 0.49167880415916443, -0.31095731258392334, 1.0119887590408325, 2.5482394695281982, 0.04310030862689018, -0.6853867769241333, 1.941910743713379, -0.7437325119972229, -0.8897212743759155, -0.887870728969574, 0.4660857915878296, 1.1526212692260742, 0.22998742759227753, -0.6785630583763123, 0.7324633598327637, -0.48888424038887024, 0.6229485273361206, 1.490906834602356, 1.1718554496765137, 1.0324203968048096, -0.30978062748908997, 1.3246848583221436, -1.1635252237319946, 1.030861496925354, -0.4705756604671478, -0.2164953649044037, -1.0046149492263794, 0.07049381732940674, -0.9188161492347717, -0.1634548008441925, 0.44519418478012085, 0.589803159236908, 0.0968007817864418, 0.317536324262619, -1.3812557458877563, 1.4225659370422363, 0.48548418283462524, 0.5690402388572693, 0.12502321600914001, 0.5182554721832275, -1.6048096418380737, -0.09560839086771011, 0.7157149910926819, 0.20661330223083496, 0.9049668312072754, 1.1542503833770752, -0.990847110748291, -0.10201137512922287, 0.15563979744911194, -0.6421757340431213, 1.15655517578125, 0.0409669391810894, -1.2325210571289062, 1.6717907190322876, -1.23789644241333, -0.5985800623893738, 0.17802567780017853, -0.7845204472541809, 0.35731223225593567, -1.3165534734725952, -1.6699954271316528, -0.35125917196273804, 1.1954560279846191, 0.5126600861549377, 0.05095185711979866], [-1.0020612478256226, -0.40296798944473267, 1.2286465167999268, 1.1275522708892822, -0.7307607531547546, -0.41010555624961853, 0.24445444345474243, -0.6103262901306152, -1.6943098306655884, 0.026819707825779915, -0.11161697655916214, 1.2189339399337769, 0.4513299763202667, 0.09869036078453064, -0.8828054666519165, -0.19960153102874756, 1.762542963027954, -0.28120461106300354, 0.6259541511535645, -0.6839460730552673, 1.109299898147583, 1.0858633518218994, 0.28953292965888977, -0.7245484590530396, 0.2782084047794342, -0.8209529519081116, -0.01861705631017685, 0.37499672174453735, 0.3038957118988037, -0.012144056148827076, 1.1236674785614014, 0.5851062536239624, 0.09553897380828857, -1.2521706819534302, 1.0018155574798584, -0.0918583795428276, -0.2806670367717743, 0.6785598993301392, -0.03611849993467331, -0.32855603098869324, 0.7241008281707764, 1.3185656070709229, 1.6630953550338745, 0.016259506344795227, -0.49996426701545715, -0.43520763516426086, -0.7949748039245605, 0.1704244315624237, -0.03341762349009514, 1.249393343925476, 1.1382619142532349, 0.3668380379676819, 3.1427881717681885, 0.22454018890857697, -1.0349878072738647, 1.0290746688842773, -2.1954715251922607, -0.958865225315094, -0.3649505376815796, 0.06635045260190964, 0.677663266658783, -1.0449918508529663, 1.6354554891586304, 0.4328160881996155, -0.20738476514816284, -0.8319310545921326, 1.174441933631897, 1.2880332469940186, -0.765291154384613, -0.47841379046440125, -1.1439391374588013, 2.2147181034088135, 0.3927932679653168, 1.203029990196228, -0.5904074311256409, -0.7382224202156067, -1.9524732828140259, 0.3405784070491791, 0.28338220715522766, 0.8283975124359131, 0.9547048211097717, -0.572259247303009, -0.2625693380832672, 1.1073675155639648, 0.1730033904314041, 0.9154276251792908, -0.6435121893882751, 1.543705701828003, 1.5813887119293213, -0.7579840421676636, 0.5566999912261963, -0.4859426021575928, 1.0418596267700195, 0.33998504281044006, -1.1012896299362183, 0.20200088620185852, 0.46801963448524475, -0.7334380745887756, -2.115095376968384, 0.6077286601066589, -0.47741249203681946, -0.46865081787109375, -1.3346436023712158, 0.717073917388916, 0.9667417407035828, -0.4288843274116516, -0.4674510955810547, 1.194399356842041, 0.6181254982948303, 0.7162742614746094, -0.20781727135181427, -1.4914324283599854, 1.2717128992080688, 1.2369896173477173, 0.10709071904420853, -1.1940556764602661, 0.9076671004295349, -0.7103132009506226, -0.4234558045864105, 0.5158514380455017, -0.7936984300613403, 1.9836677312850952, -0.41447463631629944, 0.5638152360916138, 1.1934893131256104, 1.2080146074295044, 0.5838143229484558, -0.934708833694458, -1.1185343265533447], [-1.5834342241287231, -0.8296605348587036, -0.5487555265426636, -1.0665925741195679, -1.2473546266555786, -1.3393324613571167, -0.7492817640304565, -1.2426021099090576, -0.6410296559333801, -0.7369909286499023, 0.7601919174194336, 0.9192952513694763, 1.7851101160049438, -0.8223922252655029, -0.8832515478134155, 0.21270598471164703, -0.16934700310230255, -0.18745510280132294, -0.8748539090156555, 0.28733399510383606, 0.3216601610183716, 1.0003923177719116, -1.7698042392730713, -0.008348086848855019, 0.9069849252700806, 0.7496466040611267, -0.06555888801813126, -0.08516662567853928, 0.2276812046766281, 0.24677665531635284, -1.5126919746398926, -0.24245299398899078, -2.0255484580993652, 1.0522172451019287, 0.9974281787872314, 0.04322700947523117, -1.7710295915603638, 0.6096170544624329, 1.3464961051940918, -0.9894676208496094, -0.7118769884109497, 0.30098268389701843, -1.2631045579910278, -1.4815990924835205, 2.2331748008728027, 0.049407754093408585, -0.7628410458564758, -0.3307342529296875, -1.2147846221923828, 0.4871666729450226, 1.6104276180267334, 0.5113945007324219, 0.46781936287879944, -1.168439507484436, 0.3134896159172058, 2.18226957321167, -0.29901209473609924, -0.2654450535774231, 1.6500037908554077, -1.0416386127471924, -0.06246362626552582, 0.46016427874565125, 0.43673384189605713, 0.208839550614357, -1.0108518600463867, 0.2450694739818573, -1.6235284805297852, -0.25845229625701904, -0.36008062958717346, 0.9437586069107056, 0.01467225793749094, -0.29315346479415894, 1.1608359813690186, 0.4590793251991272, -0.4137221574783325, -0.9228286743164062, -0.3134091794490814, 0.18474628031253815, -1.8848628997802734, -1.1830612421035767, -0.11585585027933121, -0.5231656432151794, -0.15459123253822327, 0.36877620220184326, -0.636746346950531, 1.1306099891662598, 0.06245145574212074, -0.9588193893432617, -0.4863128662109375, -1.7195603847503662, 0.01445950847119093, -0.9747710227966309, 0.7747430205345154, 0.022316856309771538, 2.1501717567443848, 0.37422701716423035, -0.8120785355567932, 0.7634389996528625, -0.7041381001472473, -1.1281903982162476, 1.1050163507461548, -1.3277337551116943, 0.09346780180931091, 0.5120805501937866, 1.5810025930404663, -0.3990461230278015, 0.4582647383213043, -1.0029773712158203, -1.4672667980194092, 0.08780962973833084, 0.49293550848960876, -0.4043998122215271, 1.0535304546356201, 1.6824499368667603, 0.5974021553993225, -0.9058082699775696, 0.46191030740737915, -0.33222031593322754, -1.0738660097122192, -0.429190069437027, -0.12495146691799164, -0.23030643165111542, 0.31119081377983093, -0.4126977026462555, 0.6757832765579224, -0.0913274958729744, 0.45333218574523926, 0.6661815643310547, -2.374523162841797], [0.158376544713974, -0.24075788259506226, 1.032282829284668, -0.169438436627388, 0.49302762746810913, 0.14744453132152557, -0.04669783264398575, 1.6462982892990112, 1.0595828294754028, -0.18850481510162354, 1.0658529996871948, 1.0010669231414795, 0.8452135920524597, 0.5073105096817017, 2.290449380874634, -0.6607750058174133, 1.6594197750091553, -1.0482553243637085, -0.05239947885274887, 1.0255725383758545, -1.272093415260315, -0.38043537735939026, 1.5021061897277832, 0.20133858919143677, -0.8725250959396362, 0.07841476052999496, 0.950552225112915, -1.059536099433899, 0.7483066320419312, -0.007743634283542633, -1.0083565711975098, -0.5180249214172363, -0.00695689395070076, -0.9834706783294678, 0.47142520546913147, -0.7354253530502319, -0.455157995223999, 0.7655918598175049, -0.5753737092018127, -0.7453792095184326, 0.9935491681098938, -1.2444016933441162, 0.15897613763809204, -0.5503923296928406, -0.30867281556129456, -1.4178751707077026, 0.9118714332580566, 1.5411732196807861, 1.9831255674362183, 0.7616466283798218, -0.20835454761981964, 0.28601351380348206, 0.24701032042503357, -0.20114561915397644, -1.2184330224990845, 0.010093224234879017, 1.2271573543548584, 1.7853206396102905, -0.006203628610819578, -0.4243869185447693, -0.24168133735656738, -1.1579349040985107, -1.5958631038665771, -0.7251668572425842, 0.6216611862182617, -1.1712485551834106, 0.6386139988899231, -0.48111701011657715, -1.8351244926452637, 0.928525447845459, -1.136191725730896, 0.15220025181770325, -0.45468103885650635, 1.945296049118042, 0.9690975546836853, 0.7973379492759705, -0.13651195168495178, -1.22053861618042, -0.8787139654159546, -2.0545449256896973, 0.8427749276161194, -1.825163722038269, 0.4882325828075409, 0.5140112042427063, 0.3625844717025757, -1.5863739252090454, -0.7364649176597595, -0.9557895064353943, 0.5631726980209351, 0.9651728868484497, -0.6500304937362671, -1.2952746152877808, -1.0612635612487793, 1.7398265600204468, -0.05702965334057808, -1.0477993488311768, 0.12481896579265594, 0.525594174861908, -0.740671694278717, -1.6225131750106812, 1.4274451732635498, -0.660984218120575, 1.0365983247756958, 1.2631162405014038, -0.5968770980834961, -2.560075283050537, -0.21845678985118866, -0.10806915909051895, -0.15109655261039734, 1.2766389846801758, 0.7607144713401794, 0.7896045446395874, 0.7410906553268433, -0.11557891219854355, 0.8672707080841064, -0.31036990880966187, 0.16605108976364136, 0.7135014533996582, -1.4397541284561157, -0.44364476203918457, 0.9887921810150146, 0.23081307113170624, -0.6055083274841309, -0.27781644463539124, 0.6629608869552612, 0.7711677551269531, -0.7802480459213257, 0.376950204372406, -0.2122475653886795], [-1.3401515483856201, -2.0322072505950928, 0.2006322741508484, 0.791673481464386, -0.28269878029823303, -0.6380666494369507, -1.311544418334961, 1.475488305091858, 0.03605562075972557, -0.3072200417518616, 0.8652850985527039, -1.1361939907073975, -0.7306493520736694, -0.9635961055755615, 0.9424474239349365, -0.9131749272346497, 0.011952468194067478, -0.006081925239413977, -1.7404329776763916, -1.253474235534668, 2.6078310012817383, -1.294988989830017, -0.3365309238433838, 0.6790002584457397, 1.0089092254638672, 1.5594232082366943, -0.24821442365646362, -0.8292111158370972, 1.192544937133789, -0.7258518934249878, -0.8648432493209839, 0.6293637156486511, -2.3739969730377197, 0.770979642868042, 1.282523274421692, -0.2498125582933426, -0.480631947517395, -1.2178398370742798, -0.17181068658828735, 0.16515226662158966, -1.687396764755249, -0.16788697242736816, -2.51657772064209, 0.2011014223098755, 0.9572707414627075, 0.008090629242360592, 0.3862389624118805, -3.213550567626953, -1.0227402448654175, -0.908735990524292, -1.6325913667678833, -1.281556248664856, -0.019443925470113754, 0.27860507369041443, 0.07944285124540329, -0.2578752934932709, 2.438687324523926, -0.14002831280231476, 0.5284879207611084, 2.7046186923980713, -0.9482976198196411, -0.8492084741592407, 0.33775612711906433, -1.8833227157592773, 0.1677519530057907, -0.8039330244064331, -0.7165084481239319, 0.8779616951942444, 0.9098650813102722, -0.4350990951061249, 0.8739460706710815, -1.871783971786499, 0.8059280514717102, 1.3316082954406738, 0.808056652545929, -1.5644118785858154, 0.07093250751495361, 0.10711240768432617, 1.7491710186004639, 0.8149653077125549, 0.2592627704143524, 1.1097288131713867, -1.2254481315612793, 0.4530058801174164, 0.09306798130273819, -0.6156525015830994, 0.27235347032546997, -0.4593603312969208, 0.39742955565452576, 0.518356442451477, 0.7851143479347229, -0.9354257583618164, -0.04587647691369057, -0.7441521883010864, -0.4692898690700531, 0.35086074471473694, 0.167332723736763, 0.9021961688995361, -0.2040957361459732, -1.3073463439941406, -0.03070804476737976, -2.2187368869781494, -0.8828225135803223, 0.8773424625396729, -0.299068808555603, -1.092507004737854, 0.18822655081748962, -0.3298192620277405, -0.5635672211647034, -1.205505609512329, 0.42244821786880493, -0.7511823177337646, 0.3578236699104309, 0.32626932859420776, 0.47139281034469604, -1.860169529914856, 2.188183546066284, -0.7785951495170593, -1.0820200443267822, 0.6862583756446838, -0.7261580228805542, -1.733904242515564, 2.5003459453582764, 0.464883953332901, -1.849263310432434, -1.279527187347412, 0.7885446548461914, -0.1732928603887558, -0.3902150094509125], [-0.8799083828926086, -0.24236062169075012, -0.6445592045783997, -0.2667566239833832, -1.6919511556625366, 0.03975936025381088, -0.4621526300907135, -0.479588121175766, -1.0588046312332153, 0.5387079119682312, 0.528494119644165, -0.1548602283000946, 0.4691324532032013, 0.10667930543422699, 1.193481206893921, 1.7846052646636963, 0.12056722491979599, 0.7311701774597168, 0.4139471650123596, 1.162850022315979, 0.5756534337997437, -0.3577570915222168, 0.9891000986099243, 0.6415373086929321, -0.30127018690109253, 0.31565162539482117, 0.4998781383037567, -0.2350926399230957, 0.4310351610183716, 0.5622428059577942, -0.6789567470550537, 0.2540033161640167, -1.0500097274780273, -0.7370399832725525, -0.6605082750320435, -0.5518812537193298, 0.2703900635242462, -1.8184106349945068, -0.1753213256597519, 0.7830066680908203, 0.17512665688991547, -0.49154043197631836, 0.5379361510276794, -0.35700738430023193, 0.5453472137451172, 1.3961024284362793, 2.292752981185913, 0.35504665970802307, -1.3813695907592773, -0.09914600104093552, -0.9697083830833435, 1.445504903793335, -1.1636700630187988, -1.4393478631973267, -1.4617005586624146, 0.38942751288414, -0.027054037898778915, 0.08746837824583054, 1.0413094758987427, -0.6641266942024231, 0.3157844543457031, 0.026169275864958763, 0.8725642561912537, -0.659793496131897, 0.07466869056224823, -0.6942070722579956, -0.45010530948638916, -0.5121119022369385, 1.7891793251037598, -1.5295231342315674, -0.10605204105377197, -1.009517788887024, -1.2708414793014526, -1.030745267868042, -1.0814509391784668, -1.2758431434631348, -1.1742768287658691, 0.06525275856256485, 1.6940115690231323, -0.7324278354644775, 0.4586355984210968, -1.3718538284301758, 0.6931513547897339, 1.3399529457092285, 1.755346655845642, 0.2835729122161865, -1.1753756999969482, -0.20839929580688477, -0.3504847288131714, 1.4850956201553345, -1.222516655921936, -0.5036215782165527, 0.3687483072280884, -0.3412141501903534, -0.8995821475982666, -9.030442015500739e-05, 0.43204012513160706, 0.115585096180439, 0.6175601482391357, -0.9877084493637085, -0.7180984616279602, 0.5993024706840515, -1.0570025444030762, -1.0535002946853638, 0.5039151310920715, -0.29592761397361755, 0.42847272753715515, -1.2381900548934937, 2.198955774307251, 1.8771326541900635, -1.6987391710281372, 0.41239383816719055, 0.6104590892791748, -0.5503323674201965, -1.231644630432129, -0.49284324049949646, -0.345419704914093, 0.6070258021354675, 0.844677209854126, -0.3272833228111267, -0.0761033222079277, 0.3684093654155731, 1.111011028289795, 0.5355252623558044, 0.6803283095359802, -0.5869233012199402, -0.05322942137718201, -0.07498679310083389, 0.6092492341995239], [0.38363510370254517, 0.9052408933639526, -2.404171943664551, 0.49052509665489197, -0.3859879672527313, -0.3198113739490509, -0.9291613101959229, 0.5797649025917053, 0.09149137884378433, -1.207738995552063, 0.3909248411655426, -0.38534611463546753, -0.5342435240745544, 1.222573161125183, -0.7882888317108154, 0.09728141874074936, 0.48446789383888245, -0.5281733274459839, 2.069335699081421, -0.6663898825645447, 1.089152216911316, 0.0433909147977829, -0.8177098035812378, 0.5678091049194336, -0.9304317235946655, -0.8881655335426331, -0.30774933099746704, -0.054907821118831635, -0.22440004348754883, 0.08012500405311584, -0.6414609551429749, -2.081970691680908, 0.7991806268692017, -0.021733172237873077, 0.6279785633087158, -0.7997183799743652, -1.5165493488311768, 1.554128646850586, 0.3478797972202301, -0.23835286498069763, 1.6696451902389526, -0.7290629148483276, 1.984344244003296, -2.688216209411621, -0.010269084945321083, -2.0124738216400146, 0.8016220331192017, 0.12239660322666168, 1.6796226501464844, 0.9109876751899719, -1.467560887336731, 0.22368690371513367, -0.18830564618110657, 1.1081891059875488, 1.9856359958648682, 0.6363874673843384, 0.5213130116462708, -0.06772515922784805, 1.0694494247436523, -0.61505526304245, -0.0902196392416954, 1.9401910305023193, 0.048749424517154694, 0.5945835709571838, -0.9051790237426758, 0.5511688590049744, -0.6674211621284485, 0.8393519520759583, 0.40407687425613403, 1.4281915426254272, 0.7623472809791565, 2.4402003288269043, -0.537412703037262, 0.9223048090934753, -1.5584455728530884, 0.27533817291259766, 0.4381522536277771, 0.33692118525505066, 0.560478687286377, 0.0018568409141153097, 1.222127914428711, -0.39970216155052185, -1.3116668462753296, -0.14701661467552185, -0.24956081807613373, -1.9981545209884644, 0.6709108352661133, -0.3237825334072113, -0.19702084362506866, 0.943439245223999, 1.4457411766052246, 0.7538689970970154, -0.280742347240448, 0.039582476019859314, 0.5541884303092957, 1.322471022605896, 0.016617855057120323, -0.6908833980560303, -0.6556414365768433, 1.0335233211517334, -0.13732361793518066, 0.8225845098495483, -0.12227609008550644, -0.40676143765449524, -0.11724036186933517, -0.01933184824883938, 0.3130857050418854, 0.09369022399187088, 0.6950855851173401, 1.8631311655044556, 0.05012977495789528, 0.16690167784690857, -0.7063941955566406, -1.9294668436050415, -0.3574925661087036, 1.998038649559021, 1.3566862344741821, -1.6009107828140259, -0.9588263630867004, 0.18344208598136902, 0.5812085866928101, 0.7909636497497559, -1.0419450998306274, -2.8429834842681885, 1.5793555974960327, 1.4352329969406128, 0.592046856880188, 1.1642566919326782, 1.1546165943145752], [-0.9127897024154663, -0.05445488169789314, -1.018413782119751, 0.150151327252388, -2.1210274696350098, 0.03214513510465622, -0.4260707497596741, -1.2565343379974365, 0.3438365161418915, 1.9414337873458862, 0.49521398544311523, -0.3279029428958893, -1.0426489114761353, 0.3644336462020874, -1.3367516994476318, -0.3108839690685272, 0.6612100601196289, 0.5855329036712646, 0.5033780932426453, 1.096200704574585, -0.14877623319625854, 0.12420204281806946, 0.9844357967376709, -0.3993690609931946, -1.7281297445297241, 1.351016879081726, -0.24295364320278168, -0.5020949244499207, 0.13215498626232147, 0.9899824857711792, 1.1720635890960693, 0.4646712839603424, -0.901461660861969, -0.906974196434021, 0.406080424785614, -0.7194095849990845, 1.4704196453094482, -0.3535594344139099, 0.280731201171875, -1.455052137374878, -0.2203882783651352, -0.9194272756576538, -0.41542115807533264, 0.33940231800079346, 0.17726996541023254, -0.021087361499667168, 0.5367311239242554, -2.0419509410858154, 0.08135519176721573, 0.5359079837799072, -0.23750561475753784, 0.41136446595191956, -1.6287397146224976, -2.457530975341797, 0.5120792388916016, 0.5027204751968384, 0.24833939969539642, 2.6484427452087402, -0.7938986420631409, 0.4767914414405823, -1.178351640701294, -0.21283923089504242, -0.6273792386054993, -2.380791187286377, 0.25955361127853394, -1.6797757148742676, 0.40972766280174255, 0.6408481597900391, -0.4322260022163391, 0.5360559821128845, -0.29909181594848633, -1.5308921337127686, -0.7249781489372253, 0.8753411769866943, 0.3512851893901825, 1.391945719718933, 0.8727411031723022, -0.13364103436470032, 1.292184591293335, 1.7605386972427368, 1.3435524702072144, 0.7141118049621582, 0.5524975657463074, 0.16450659930706024, 1.681378960609436, 0.4020805358886719, -1.3875339031219482, -0.2530808746814728, 0.43705493211746216, 2.037085771560669, -1.0078084468841553, -0.24448254704475403, -2.217839479446411, 0.5127131938934326, -1.9585570096969604, 0.21646170318126678, 0.17251171171665192, 1.0100189447402954, 0.7017024755477905, 0.6148860454559326, -0.9830117225646973, -1.2131954431533813, -0.633390486240387, -0.23273605108261108, -0.42690664529800415, -1.9230709075927734, 0.7457758784294128, 0.07404005527496338, 2.0470497608184814, -0.2659900486469269, 0.23769164085388184, 0.506536602973938, 0.7525827884674072, -1.5483018159866333, 2.1788699626922607, 0.5773090720176697, -1.0724982023239136, 1.6957931518554688, -0.26263609528541565, -1.1375954151153564, -1.228053331375122, -0.041102632880210876, -0.08907674998044968, -0.22829383611679077, 0.26874926686286926, 0.4992902874946594, -2.485466241836548, -0.44156745076179504, -0.7773541212081909], [-0.5156528949737549, -0.19944803416728973, -1.4891767501831055, -0.2877211272716522, -0.6609635353088379, -0.49605032801628113, -2.010052442550659, -0.6216624975204468, 1.4125949144363403, -0.08601824939250946, -0.5148714780807495, 0.04343441128730774, -0.17884306609630585, -2.535583019256592, -0.9451318979263306, 0.25158214569091797, -1.629959225654602, -0.303678959608078, -0.14353515207767487, -0.9393189549446106, -0.2278963029384613, 1.4018250703811646, 0.2500593066215515, 0.7541923522949219, 1.156269907951355, -0.9807156920433044, 1.263756513595581, 1.2330288887023926, -2.244093418121338, 1.7035571336746216, -1.237365484237671, -1.2657452821731567, -0.971997082233429, 0.26594239473342896, 1.08980131149292, 1.0505460500717163, 0.8945843577384949, 1.4458882808685303, -0.6429706811904907, 0.23588360846042633, -0.07790851593017578, 0.07217481732368469, -1.4797712564468384, 0.4420199990272522, -0.9629873633384705, 0.5611927509307861, -0.5715023279190063, -0.4160269498825073, 0.18879400193691254, 0.7299233078956604, -0.29447048902511597, 1.3957992792129517, 0.04768476262688637, 0.6834679841995239, -1.4691766500473022, 2.295194149017334, 0.6854575872421265, 0.9862843751907349, 0.7220720052719116, -0.0702018141746521, -1.1748201847076416, 0.033830899745225906, 0.3009246289730072, 0.4083884358406067, -1.4041215181350708, -0.702734649181366, 1.1088608503341675, 0.45713675022125244, -0.13797946274280548, 0.33523860573768616, -0.2976663112640381, -1.0099786520004272, -0.2330372929573059, 0.20468029379844666, -0.37993496656417847, 0.9091085195541382, 1.779098391532898, 0.3718366026878357, 0.447861909866333, 0.4299325942993164, 0.28240713477134705, -0.5105270147323608, -1.0109727382659912, -1.3102425336837769, -1.3251405954360962, 0.6100513339042664, -1.0210028886795044, -0.4437454640865326, -0.42042145133018494, 0.017752520740032196, 0.9319517016410828, 0.3140903115272522, -0.584312379360199, 1.1625829935073853, -0.8353748917579651, -0.1626463383436203, -0.1340533196926117, -0.18016299605369568, 1.507644534111023, 0.43251118063926697, 1.29819655418396, 1.302872896194458, -0.16099470853805542, -1.0843470096588135, -1.6202025413513184, -0.09969808906316757, -0.7435550093650818, -1.6489850282669067, -0.22762362658977509, -1.3030996322631836, 0.6033653020858765, 0.3251044750213623, 0.17936156690120697, 1.5076497793197632, -0.13140524923801422, -1.135461688041687, 1.9301854372024536, -0.25244298577308655, -1.517168641090393, 0.18978577852249146, 0.8659589886665344, 1.4362915754318237, 0.08709029853343964, -1.4169340133666992, 0.8535165190696716, 0.954388439655304, 0.3788682818412781, 0.10836423188447952, -1.0897417068481445], [-0.07184315472841263, -0.16725614666938782, 0.6301252245903015, -0.6609347462654114, -0.06394113600254059, 0.26987746357917786, 1.6019071340560913, -0.05237828195095062, 1.1065480709075928, 0.28605425357818604, -0.8377535343170166, -2.178997278213501, 0.49660471081733704, -1.3251343965530396, -0.3541543781757355, -0.9064772725105286, -1.7160309553146362, -0.30531832575798035, -0.7317490577697754, -0.2982562482357025, 0.7842314839363098, -0.3612177073955536, 0.24325180053710938, -1.4592522382736206, 1.8770473003387451, 0.7801110148429871, 0.301558256149292, -0.16812846064567566, -0.8505380749702454, 0.5561196208000183, 2.3457510471343994, -2.3863441944122314, -1.0060794353485107, -0.4274657070636749, 1.9268531799316406, -0.9911202192306519, -0.6424815058708191, -0.3337288200855255, 0.05745859071612358, -0.3126280605792999, 1.846282958984375, 0.7339490056037903, -0.19320787489414215, 2.214052438735962, -0.24057886004447937, -1.2116299867630005, 0.445401132106781, 1.3993993997573853, 1.065712571144104, 0.2663947343826294, -1.6617697477340698, 1.4886500835418701, 0.23664727807044983, 0.3843161463737488, 1.2377290725708008, -0.36253678798675537, 1.1301144361495972, 0.572425127029419, -0.18281914293766022, -0.098717100918293, 1.3204935789108276, -0.07331330329179764, 0.6341825127601624, 0.35988742113113403, 0.8145661950111389, -2.8630759716033936, 0.17520573735237122, 1.7650341987609863, -0.23764169216156006, -0.6169844269752502, -0.30880290269851685, 0.5686507225036621, 0.1502814143896103, 0.874584972858429, 0.5028434991836548, -0.40297731757164, 1.2008697986602783, -1.5754591226577759, 1.9964144229888916, 0.48423007130622864, 0.03350580111145973, 1.8325132131576538, -0.7589634656906128, 0.20954540371894836, 0.45727694034576416, 1.7752324342727661, 0.6475337147712708, -0.829607367515564, 0.36900806427001953, -1.6786658763885498, -0.9058412909507751, -1.233810544013977, -1.9984935522079468, 0.33024951815605164, 0.838533878326416, 1.0829957723617554, -0.20974726974964142, -0.915103018283844, 0.16788962483406067, 0.0012534346897155046, 0.2690526247024536, -0.15104350447654724, -1.0143709182739258, -1.2103462219238281, -1.2349380254745483, 0.43418729305267334, 0.39558351039886475, 0.2951609194278717, -0.2048601508140564, -0.06147441267967224, -0.2889624536037445, -0.25648653507232666, 0.06446386873722076, -0.5699228048324585, 2.605898380279541, -0.32152020931243896, 0.3867724537849426, 0.24217481911182404, -0.3071227967739105, 0.4256422817707062, 0.5912978053092957, 0.3174957036972046, -1.2440181970596313, -0.3572218716144562, -0.02804689109325409, 0.28698816895484924, -0.2658419907093048, -2.4899024963378906, -1.6736761331558228], [-0.04264987260103226, -1.4108855724334717, -0.3526816666126251, 0.553888738155365, 0.8227263689041138, 0.9845796823501587, -0.2596723735332489, -0.7467305064201355, 1.9233900308609009, -0.8552206754684448, 0.8718207478523254, -1.7137577533721924, 1.7574207782745361, 1.3718165159225464, 1.5814926624298096, 1.6941533088684082, -0.10432811826467514, -1.960625410079956, -1.8104829788208008, 0.31280502676963806, 1.5112309455871582, 0.659203052520752, 0.13844002783298492, -0.7655004858970642, 1.1811312437057495, -1.002761960029602, 0.6737181544303894, -0.2029590904712677, 0.14466990530490875, 0.4016447067260742, -0.1679084599018097, 0.2669421434402466, -0.21275217831134796, 1.0990082025527954, -0.33283817768096924, 0.28573542833328247, -0.14771893620491028, 1.7521226406097412, 0.8987387418746948, -1.458369493484497, 1.44825279712677, 1.06032395362854, -0.11882834136486053, 1.0577600002288818, -0.1360558122396469, 0.9261859655380249, 0.7233598828315735, 0.11850880086421967, 0.192052960395813, 1.3398226499557495, 1.0994092226028442, 2.1449828147888184, 1.2836692333221436, -0.7892807722091675, 0.5666490793228149, 0.7350049614906311, 0.7317106127738953, -0.9139939546585083, -0.47591638565063477, -0.8478288650512695, -0.8042188882827759, 0.9995439052581787, 0.5414881706237793, -0.05511981621384621, -0.6576169729232788, 0.676677405834198, 0.9702264666557312, -1.025136113166809, 2.6021406650543213, 0.1862959861755371, -0.5327033400535583, -0.5844618082046509, -2.7675013542175293, -0.708428144454956, -1.1795510053634644, 0.4598968029022217, 0.6688860654830933, 0.7676330208778381, -0.29014652967453003, 0.6270219087600708, 0.1692485213279724, -0.5977250337600708, 0.21212983131408691, -0.9070476293563843, -1.1692824363708496, -0.23566295206546783, 0.7455440759658813, -0.7803648114204407, -1.001551628112793, 2.337920665740967, 2.1425881385803223, 1.552093744277954, -0.21593652665615082, -0.05105122551321983, -0.8171644806861877, 1.630791425704956, -0.19483628869056702, -0.22053606808185577, 0.5189688801765442, -0.8196365833282471, 0.2465897500514984, -0.993478536605835, 0.7304636836051941, -1.4601033926010132, 0.5918581485748291, -0.665696918964386, -0.5695953965187073, -0.19144804775714874, 1.4361437559127808, -0.03560580313205719, -0.2638886570930481, 0.15254445374011993, -0.5660180449485779, -0.3583839237689972, 0.31411710381507874, 0.7487540245056152, 0.02085711807012558, 0.44583985209465027, 0.1425734907388687, -0.554230809211731, -1.4283703565597534, -1.1813173294067383, 0.10011578351259232, -0.5207656025886536, 1.1827178001403809, 0.45045340061187744, -0.42183804512023926, 0.552242636680603, 0.5755021572113037], [-0.5871167778968811, 0.057489339262247086, -1.4032119512557983, 1.218599557876587, -0.11922638863325119, -0.22960959374904633, -0.999836266040802, -2.1121108531951904, -0.7035742402076721, 0.05837845429778099, 0.7008045315742493, 1.154846429824829, -2.040221929550171, -2.2544806003570557, -0.09857350587844849, -0.8223060369491577, -0.3769042193889618, 0.8750351667404175, 0.9656354188919067, 0.88887619972229, 0.0032135394867509604, 1.0153108835220337, -0.8274868130683899, 0.604571521282196, 1.5314834117889404, 0.18148648738861084, -0.3874860107898712, 0.6740677356719971, -1.246629238128662, 1.6165426969528198, 0.5692178606987, 0.3439280092716217, -0.18897537887096405, 0.43974918127059937, 0.9789709448814392, -0.05379743129014969, -1.6705951690673828, -1.3181310892105103, 0.33345288038253784, 0.814077615737915, -0.8569720983505249, 1.0301977396011353, 1.22287118434906, -1.5186033248901367, -0.1913687288761139, -0.11675289273262024, -2.2355289459228516, 0.07503314316272736, 0.1724380999803543, 0.939568281173706, 0.477512001991272, 0.5267682075500488, 0.8079738020896912, -2.04339599609375, 0.9720466136932373, -1.1364989280700684, -0.9444399476051331, -1.1993250846862793, 0.6380266547203064, -0.6365771889686584, -0.32166892290115356, -0.8418059349060059, 1.9296365976333618, -0.9074587225914001, 0.07999030500650406, 2.799105167388916, 0.42136210203170776, -1.0870596170425415, -0.3263714015483856, -0.770850658416748, 0.43258824944496155, 1.7191253900527954, -1.620309829711914, 0.854080319404602, 2.178467035293579, -1.5168941020965576, 0.8775245547294617, -0.42323118448257446, 0.910909116268158, 0.8483930230140686, 0.19398225843906403, 0.004306548275053501, -0.04553510993719101, 0.14211907982826233, 0.04693016782402992, -1.667410969734192, -0.16819985210895538, 0.7178049087524414, 0.6260147094726562, -0.49173519015312195, -0.41471347212791443, 1.5382041931152344, -0.3543398678302765, 2.070533037185669, 0.49333521723747253, 0.31824707984924316, 0.2627421021461487, 0.6677603721618652, 1.042377233505249, -1.2085072994232178, -0.012661054730415344, -0.8466612100601196, -0.434855580329895, 1.0163055658340454, 0.7729507684707642, -2.068568229675293, 0.493934690952301, -0.6925063133239746, 0.31510263681411743, -0.022695045918226242, 0.08733426034450531, 0.3094949424266815, 0.8059130311012268, 0.5746995210647583, 1.6855467557907104, -0.13144336640834808, 0.04574904218316078, 1.0388550758361816, -0.28045719861984253, -0.14983490109443665, 2.1571309566497803, -0.4561058282852173, 0.8127864003181458, -1.009181261062622, 0.3154732286930084, 2.1090612411499023, 0.8889704346656799, 0.36002272367477417, -0.6962212324142456], [-0.9481746554374695, 0.3184438645839691, 0.7214871644973755, 0.5911306142807007, -0.6394530534744263, 1.0476021766662598, -0.24063260853290558, -0.5002546310424805, -0.7218105792999268, 0.43784499168395996, 1.2393226623535156, 0.30642765760421753, -0.5546433925628662, 0.07618112862110138, 0.12921829521656036, 0.5789610743522644, -1.3924602270126343, -2.3156332969665527, -1.2402194738388062, -0.19839031994342804, -2.3246655464172363, -1.157098412513733, -0.7331231832504272, -0.03968102112412453, -0.5972960591316223, 1.0493083000183105, 0.7653409838676453, 0.2811215817928314, -0.41670557856559753, -0.5365416407585144, 0.222954660654068, 1.5848183631896973, 0.4358164370059967, 0.12577256560325623, -1.4498721361160278, -1.1338714361190796, -1.3495938777923584, -0.16212992370128632, 0.3583146035671234, 0.5224037766456604, -0.7569959163665771, 1.1312875747680664, 1.2308825254440308, -0.5486829876899719, -0.7682576179504395, 0.06437698006629944, 0.5684521198272705, 0.11176196485757828, 0.12547267973423004, 0.0865572839975357, -0.4242619276046753, -1.6113570928573608, 0.7738460898399353, -0.19883465766906738, -0.3312208652496338, -0.973630964756012, -0.19639232754707336, 0.8071387410163879, 0.2940594553947449, -0.7289735078811646, -1.4914720058441162, -0.1438116878271103, -1.066975712776184, -0.6280803680419922, -1.79826021194458, 0.8133134245872498, -0.626348614692688, -0.5454869270324707, -0.2939901351928711, -0.4329181909561157, 0.028643015772104263, 2.137120485305786, -1.7441154718399048, 0.7305938005447388, -0.3546994626522064, -0.04895740747451782, 0.11919613182544708, -0.32644620537757874, -1.6992098093032837, -1.3916503190994263, 2.6591553688049316, 0.8309900760650635, -0.8598986864089966, -0.7258409261703491, 2.1187803745269775, -0.974769651889801, 1.0036752223968506, 0.4148353636264801, -1.424490213394165, -0.05626260116696358, -0.48767393827438354, -0.8184070587158203, 1.2992198467254639, 0.4049951732158661, 0.49731725454330444, 0.9599883556365967, 0.08145240694284439, -0.6475105285644531, -2.327296733856201, 0.07309675216674805, 2.5894033908843994, -0.492388516664505, 0.46348804235458374, -2.1703433990478516, 1.5961993932724, 1.2542850971221924, -1.5495221614837646, -1.0420804023742676, 1.2615506649017334, 0.374409556388855, -0.2695794999599457, -0.3572251796722412, -0.791727602481842, -1.3091164827346802, -1.1563773155212402, -0.15257319808006287, -0.8102104663848877, 1.1913683414459229, 1.2917296886444092, -1.1482834815979004, 0.17416056990623474, -1.6749241352081299, -0.6714369654655457, 0.14884090423583984, 1.1439257860183716, -0.939108669757843, 1.6475858688354492, 1.9923858642578125, 0.08630245923995972], [0.6063876748085022, 0.46104639768600464, 0.3572646975517273, -0.8913715481758118, 0.21395611763000488, 0.9266232848167419, -0.3707410395145416, -1.9734575748443604, 0.5809245109558105, 1.5621590614318848, -0.861700713634491, -0.009819639846682549, -0.376801460981369, -0.04720115661621094, 0.8830889463424683, 1.2532044649124146, 0.5365763306617737, -1.3500707149505615, -0.26515042781829834, 0.6426725387573242, 0.12177427858114243, -1.2669353485107422, 0.016962863504886627, -1.8480947017669678, 0.21050992608070374, -0.31489038467407227, -0.0899074524641037, -0.03946872800588608, 1.080317497253418, 1.1256163120269775, 1.2187482118606567, 0.0807684138417244, 0.7555655241012573, -0.5442488193511963, 0.9087170958518982, -0.37858840823173523, 0.4503799378871918, -0.4048088490962982, -0.48295167088508606, 0.37135130167007446, 0.03374684602022171, -1.8878908157348633, -0.5468524098396301, 0.7370051741600037, -2.112959384918213, -0.7462303638458252, 0.3925710916519165, 0.07756666094064713, -0.14601650834083557, -1.2998688220977783, 1.0448776483535767, -0.852465033531189, 1.032501220703125, -0.7424524426460266, -1.1677565574645996, -0.6960904598236084, 0.3685811758041382, -0.2959992289543152, -0.30060696601867676, 2.6858229637145996, 1.7973581552505493, -0.9096572399139404, -1.3721097707748413, -0.30597159266471863, 0.04866199195384979, 1.1673356294631958, 0.2075347751379013, -0.8932083249092102, 0.9264863729476929, 0.3730064630508423, 0.30403298139572144, -1.326019287109375, 0.6213865876197815, -0.5364243984222412, 0.019378840923309326, 1.3263558149337769, -0.1530616730451584, 0.02762738987803459, -1.8769776821136475, -0.879067599773407, -0.346698522567749, -0.44538411498069763, 1.4963924884796143, -1.0337507724761963, -0.9471110105514526, -0.6893507838249207, -1.9631599187850952, -0.8070658445358276, -0.22463485598564148, 0.8440525531768799, 0.500015914440155, -1.1160085201263428, -0.5302340984344482, 0.5399157404899597, 1.164985179901123, 1.140934944152832, -1.3957539796829224, -1.4421448707580566, 0.5406230092048645, 0.43350112438201904, -0.5467777848243713, -0.39868149161338806, -0.20981796085834503, 0.7515777349472046, 1.0369449853897095, -0.25236865878105164, 0.16688081622123718, -0.09676258265972137, 1.0969314575195312, -1.073061466217041, 1.5302613973617554, -0.7501896619796753, 0.7648734450340271, 0.24085746705532074, 1.1890220642089844, -1.39834725856781, 0.557906985282898, -0.21521306037902832, -0.8165067434310913, -0.2247253656387329, 0.3887113034725189, 1.0819015502929688, 1.2597326040267944, -1.5013960599899292, 0.40589749813079834, -1.179059624671936, -0.05916013568639755, -1.415163278579712, 0.8106515407562256], [-0.045838452875614166, 0.2586456537246704, 0.09595280140638351, 1.4709919691085815, -1.1274889707565308, -2.403843402862549, 0.8394965529441833, 0.322890967130661, 0.9608784317970276, 1.1007335186004639, -0.5518903732299805, 0.47495540976524353, 0.186277836561203, 0.4489564001560211, -1.6085331439971924, 0.7593216300010681, 1.7804279327392578, 0.2766552269458771, -0.9553729891777039, -0.6602708697319031, -0.3248745799064636, -1.4104809761047363, -1.0874981880187988, 0.6189265847206116, -0.6119097471237183, 0.6660025715827942, -0.45641499757766724, -1.9800841808319092, 0.8821905851364136, -1.8201531171798706, 0.8041072487831116, -0.2491147220134735, -0.3124808669090271, -0.19644492864608765, -0.20408345758914948, -0.8191093802452087, -0.7885057926177979, -0.9652062058448792, -2.5323612689971924, 1.5657368898391724, 0.7753041386604309, 1.5971367359161377, -2.970036506652832, 1.534065842628479, 0.4734691381454468, -0.1279401332139969, 1.074691891670227, -0.673132061958313, 0.5335692763328552, 0.5293142199516296, 0.7639660835266113, -0.2917085886001587, -0.866439163684845, 0.48802125453948975, 0.5606130957603455, 1.2039917707443237, -0.4281795620918274, -0.04832739755511284, 1.0229554176330566, -0.6587800979614258, 1.2102530002593994, -1.7846499681472778, 1.2273935079574585, 2.445587158203125, -1.5485084056854248, -0.11297100782394409, -0.8965178728103638, -0.08767379820346832, 0.24111102521419525, 0.6726567149162292, -0.222652405500412, -0.041430018842220306, -0.5230061411857605, -0.08271009474992752, -0.15579435229301453, -0.7579954266548157, 1.547350525856018, 0.3585290014743805, 0.27562013268470764, -0.6822359561920166, -0.21077130734920502, 0.19491924345493317, 0.2780080735683441, -0.7172120213508606, 0.22068890929222107, 0.498419851064682, 1.659627914428711, -0.07696844637393951, -1.6830215454101562, 1.4613314867019653, 2.441120147705078, 1.4264733791351318, 0.5110436081886292, 1.2836558818817139, -1.2921656370162964, -0.9915112257003784, 0.7274425625801086, -0.6005551218986511, 0.045747384428977966, 0.4133733808994293, 2.0557618141174316, 0.20602738857269287, -0.06584908068180084, 1.5843366384506226, -0.30361995100975037, -0.20398248732089996, 1.7396652698516846, -0.6590224504470825, -0.014601531438529491, -1.0307937860488892, -1.5528619289398193, -1.2325178384780884, -0.031104344874620438, -1.1493972539901733, 0.9901825785636902, 0.19308702647686005, -1.2526764869689941, -1.5942020416259766, 0.4540020525455475, -1.6928138732910156, -2.147090196609497, 0.29245874285697937, -1.187572717666626, 0.8611130714416504, 0.7189900875091553, 2.207594633102417, -0.9716166257858276, 0.847646176815033, -0.9751793146133423], [-0.17557194828987122, -0.9372673034667969, 1.2466646432876587, 1.106929063796997, -1.1012215614318848, -0.523618757724762, 1.0788484811782837, -0.3648305833339691, 0.9245895743370056, -0.9383367300033569, 1.3416945934295654, -1.0965871810913086, -0.410129189491272, -0.847359836101532, -1.5639262199401855, -1.08810555934906, -0.5876206159591675, 1.2825334072113037, -0.5287221074104309, -1.2587544918060303, 0.18935875594615936, 0.8247941732406616, -0.7455766797065735, -0.33481335639953613, -0.5235515236854553, -1.7152656316757202, -0.7422366142272949, 0.9574651122093201, -0.752066433429718, -0.3448251485824585, -1.4833614826202393, 1.713178277015686, -1.0289907455444336, -0.11677490174770355, 1.2523835897445679, 0.007935306057333946, -0.36949190497398376, -0.38667723536491394, -1.140307903289795, 1.20860755443573, -1.0158971548080444, 1.7710597515106201, -0.2587999403476715, 1.3929919004440308, -0.22734059393405914, 0.881284773349762, -0.7039451599121094, 0.6829644441604614, -0.6629123091697693, 0.8698420524597168, -0.7492188811302185, 0.3953615427017212, 0.31807366013526917, -1.3060269355773926, -0.8349627256393433, -1.4475691318511963, 0.8685052394866943, -2.639777183532715, 1.5006095170974731, -0.23613333702087402, -1.2608593702316284, -0.9788602590560913, -1.289531946182251, 0.7507520318031311, -0.5092592835426331, -0.19977033138275146, 0.49461641907691956, 0.27778035402297974, 0.33408159017562866, 1.1986901760101318, -0.5890894532203674, 0.0967089980840683, -0.8514139652252197, 1.091214656829834, -0.9800068140029907, 0.9714615941047668, 0.5081961154937744, 0.681718647480011, -0.7687660455703735, -0.1489957869052887, 1.0989115238189697, -1.2444056272506714, -1.7187987565994263, -0.9629250764846802, -1.1151518821716309, -0.5271246433258057, 1.6294612884521484, -0.7780357599258423, -0.16182664036750793, -1.3246575593948364, -1.4656109809875488, -2.861851692199707, 1.1406186819076538, -0.25506511330604553, -0.6153877377510071, 1.1451847553253174, 1.4161953926086426, 1.3424910306930542, 0.19855228066444397, -2.182680606842041, -0.2986420691013336, 1.313882827758789, -0.2995239794254303, -0.09868105500936508, -0.4646602272987366, -1.413865327835083, -2.084989547729492, 1.4494355916976929, -1.1027593612670898, -1.6805363893508911, 2.2802765369415283, 0.9783469438552856, -0.18395404517650604, -1.7676231861114502, -0.5849090218544006, -1.0249841213226318, -0.18401682376861572, -0.19199863076210022, 0.32711562514305115, 1.3253097534179688, 0.3320651948451996, 0.4129543602466583, 0.6728482842445374, 0.474519282579422, 1.8784310817718506, 0.7479258179664612, -1.928780198097229, 0.9191927313804626, 0.22084607183933258], [-1.4148814678192139, 0.9065293669700623, -0.9812305569648743, 0.6044219136238098, 0.810831606388092, -0.30549731850624084, -1.5811041593551636, 0.366632342338562, -0.37379977107048035, 1.2575480937957764, -0.3824247419834137, 0.889224112033844, 0.9345415830612183, 0.4084584712982178, 0.9606503248214722, -0.831006646156311, -0.04896239936351776, -1.1267485618591309, -2.162227153778076, -0.8841335773468018, 0.4131351411342621, -1.427402377128601, 0.7370972037315369, 0.4381810426712036, -1.3758080005645752, -1.3557310104370117, -0.5562418699264526, 0.6599318981170654, 0.6720807552337646, 0.21981218457221985, 0.35523566603660583, -0.8664358258247375, 0.33866646885871887, -0.03736092522740364, -0.8575403690338135, -1.256859302520752, 0.34672391414642334, -0.30012476444244385, 0.9801498651504517, -0.4888414442539215, 0.3966333568096161, -0.21664537489414215, 0.8269948959350586, 0.7603668570518494, 1.4271883964538574, 0.4382809102535248, -0.9644758105278015, 0.9746086597442627, -0.458492249250412, 1.486498475074768, -0.7303125262260437, -0.8991227746009827, 0.2654280364513397, 0.18678149580955505, 0.256513774394989, -1.4509421586990356, 0.03609379380941391, 0.5336744785308838, 0.979939877986908, 0.1495031863451004, -0.9576451182365417, -0.9074217677116394, 1.0864717960357666, -0.2059132158756256, -0.4532857835292816, -0.9069761037826538, -0.6829352378845215, 0.8533226847648621, -1.5768283605575562, 1.4811067581176758, 0.05460776016116142, -1.0667407512664795, 1.1120506525039673, 2.2360310554504395, 0.6831375360488892, 1.1377596855163574, -1.3191910982131958, 2.010053873062134, -0.1763308346271515, 0.07960296422243118, 0.010201397351920605, -1.0933661460876465, 1.0992316007614136, 0.16992168128490448, 0.836519718170166, -0.24448150396347046, 0.9239946603775024, 1.5089092254638672, -1.0746382474899292, 2.839315891265869, 0.4452790319919586, 0.44754162430763245, 0.07949531078338623, -1.09522545337677, 1.2094427347183228, 0.1611848771572113, -0.09649409353733063, 0.33199432492256165, -0.2766323983669281, 0.48373639583587646, -0.4133516252040863, 0.731025218963623, -1.989192008972168, -0.1492963284254074, 2.059614896774292, 2.029641628265381, -1.1312958002090454, -1.1213570833206177, -0.5031812191009521, 1.0351120233535767, 0.38420337438583374, 0.8654929995536804, -1.9228914976119995, -1.7485402822494507, 0.15684591233730316, 0.6661333441734314, -0.6207754015922546, -0.18314674496650696, 0.03804435953497887, -0.7472542524337769, -0.08827449381351471, 1.2192555665969849, -0.6625120639801025, -0.6723365783691406, -0.47680068016052246, 0.42304256558418274, -0.4858269989490509, -0.17474660277366638, 0.3503740727901459], [-1.0475034713745117, -1.1192494630813599, -0.0970953032374382, -0.9248026609420776, -0.784695029258728, 0.08598171174526215, 0.6381869912147522, -1.219847321510315, -0.5400497913360596, -0.4713081419467926, -0.41301774978637695, -1.5902328491210938, 1.7536860704421997, 0.970341682434082, -0.6662585735321045, -0.06648869812488556, -1.7490178346633911, -1.434110403060913, 1.1271427869796753, 2.475795030593872, 1.312318205833435, 0.0469498448073864, -0.2077658623456955, 1.265799880027771, -0.5373420119285583, -0.25321394205093384, 0.35385897755622864, -0.5049522519111633, 0.42185264825820923, 1.4847490787506104, -0.9191937446594238, -1.1907120943069458, -0.2721014618873596, 0.2455849051475525, -0.6842936277389526, 1.8983957767486572, -0.17867493629455566, -1.1732808351516724, 0.15126392245292664, -1.0958216190338135, -0.9699243903160095, 0.47324639558792114, -1.2112482786178589, -0.888465166091919, 0.9524861574172974, -1.064403772354126, 1.4473179578781128, 0.3507588505744934, -0.053902219980955124, -0.531653642654419, 0.170330211520195, 0.23605991899967194, -1.23627507686615, -0.9278026819229126, 0.6817140579223633, -0.647688090801239, -0.1860678642988205, -0.05311082303524017, 1.2982944250106812, -0.2908650040626526, -0.6254779696464539, -0.8316473364830017, -2.6883184909820557, -0.3102554976940155, -2.071425676345825, -0.46054670214653015, -0.9681110978126526, -0.5593824982643127, -0.8983527421951294, 0.02084166370332241, 1.7832818031311035, -0.20550879836082458, 0.5260964632034302, 0.5880973935127258, 1.1845847368240356, -0.9258582592010498, -0.4888647198677063, 2.107667922973633, -1.154504656791687, 0.08978547155857086, 0.7746800184249878, 0.4407104253768921, -0.1109846755862236, -1.0069609880447388, -1.6938139200210571, -0.5195426344871521, 0.004646909423172474, -0.38798651099205017, -0.37457144260406494, -0.6596413850784302, 0.8470784425735474, 0.8218130469322205, -0.5565311312675476, -1.0026814937591553, -2.227870464324951, 0.717652440071106, 1.5697627067565918, 0.504413366317749, -1.1079168319702148, 0.4431358575820923, -1.1316404342651367, -0.013140695169568062, 0.232387512922287, 0.9168713688850403, -0.5725909471511841, -0.6157478094100952, 1.3205267190933228, -0.5465806126594543, 1.0186352729797363, -1.1421267986297607, -0.11328692734241486, 0.5384767651557922, -1.1921091079711914, 1.1803722381591797, -1.0939832925796509, 0.6465178728103638, 0.13980835676193237, -0.7602087259292603, -0.6333982348442078, 1.4534021615982056, -0.8574599623680115, 0.7109484672546387, 0.2213486284017563, -1.3294317722320557, 1.064800500869751, 1.3449615240097046, -1.3084238767623901, -0.1000838354229927, -0.07854291796684265], [0.669912576675415, -0.4583635628223419, 0.17199882864952087, 0.09705038368701935, -1.671926736831665, -0.16306442022323608, 1.6943830251693726, -0.0265667662024498, 1.2245278358459473, 1.4924273490905762, -1.7220171689987183, 0.6120063066482544, 0.06240539625287056, -1.8568241596221924, -0.36613979935646057, 0.898638904094696, 0.9161518216133118, 0.06392813473939896, -0.5077964663505554, 0.1906510889530182, -0.7528129816055298, -1.3903898000717163, 0.6652312278747559, 0.238639235496521, -1.2209348678588867, -0.1804715096950531, 0.30095237493515015, -0.439836323261261, 1.387744665145874, -0.34117311239242554, 1.5299698114395142, -0.4312945306301117, 0.9748854637145996, -0.7665651440620422, -0.14034192264080048, -0.09833759814500809, -0.8640596270561218, -0.45741844177246094, -1.5038644075393677, 0.32962000370025635, 1.8126800060272217, -0.7372931838035583, -0.48664960265159607, -0.9939747452735901, -0.5506875514984131, 0.31921327114105225, -0.29364877939224243, 1.0459277629852295, 0.9566239714622498, 0.23479114472866058, 1.508884310722351, -0.5407352447509766, -0.2068868726491928, 1.8410645723342896, -0.3397614061832428, -0.04459593817591667, -0.6226400136947632, 0.9707645773887634, -0.13045741617679596, -1.3595839738845825, -0.4713430404663086, 0.9648707509040833, -0.6176207065582275, 1.9367834329605103, 0.825997531414032, -0.9729433655738831, -0.8831409215927124, 0.8546030521392822, -0.3183459937572479, -0.03951483964920044, 0.25754866003990173, -0.5301799178123474, 0.08075734972953796, -1.1428248882293701, -0.2785736620426178, 0.8256975412368774, 0.6702612042427063, -0.04509305953979492, -0.1295747607946396, -1.334154486656189, -0.09386009722948074, 0.1905720829963684, -0.3651849925518036, 1.2888967990875244, -0.28887811303138733, -0.41753920912742615, 0.3464702367782593, 1.9120901823043823, 0.6630392074584961, -0.6623823642730713, 0.6908833384513855, 0.13313856720924377, -0.27962562441825867, 2.2515347003936768, -1.2889915704727173, -0.23484677076339722, -0.2795853912830353, -1.795868158340454, -0.26579317450523376, -1.4406852722167969, 0.678837239742279, 0.31602227687835693, -0.5319139957427979, 0.6236903667449951, -0.5274699926376343, 1.0280152559280396, -0.8256489634513855, 0.35329335927963257, 2.2899692058563232, 0.6699283123016357, -1.579278826713562, 0.5132797360420227, 3.7213308811187744, 0.49368545413017273, -0.35253167152404785, 1.1569594144821167, 0.31711849570274353, 0.7959179282188416, -0.3464714288711548, 0.03393279388546944, 0.3430700898170471, 1.3663108348846436, 0.8880917429924011, -0.29459479451179504, 0.5413731932640076, 0.38304829597473145, 0.6528419256210327, 2.8022682666778564, -0.1815221607685089], [1.5300445556640625, -1.1736962795257568, -0.4372354745864868, 0.9251371026039124, -0.2560080289840698, 1.592627763748169, -0.8478038907051086, -0.8045403957366943, -1.300010085105896, -1.1460890769958496, 0.9445950388908386, -1.2885874509811401, -0.595215916633606, -1.3793387413024902, 0.8383473753929138, 0.036154840141534805, -0.1613306850194931, -1.4497965574264526, 0.16918441653251648, 0.7767155766487122, -1.3843719959259033, 1.6212886571884155, -1.4041938781738281, -0.06256747990846634, 1.1396982669830322, 0.45149311423301697, 1.2395429611206055, 0.4424359202384949, -0.5628243088722229, 0.8213515877723694, 0.7149513959884644, -0.3481440246105194, 0.38715577125549316, -0.41077324748039246, 0.22487780451774597, 0.05445382371544838, -1.3451611995697021, -0.012332950718700886, -0.17443203926086426, -0.4715292155742645, 1.0296027660369873, -0.998044490814209, -0.709962010383606, -1.1234126091003418, -0.285585880279541, 0.4166848957538605, 0.996676504611969, -0.04527251422405243, -1.7387162446975708, 0.07250964641571045, 0.5151689052581787, -0.3842516839504242, 0.10260843485593796, -1.5900428295135498, -0.14558787643909454, 0.43304958939552307, 0.322682648897171, 0.735551118850708, -0.5484119653701782, 0.001594087458215654, -0.47210073471069336, 0.38792040944099426, -0.6187113523483276, -0.0810760036110878, 0.7483005523681641, -0.14012163877487183, 1.6191633939743042, 0.5370888113975525, 0.7038357257843018, -1.374449610710144, 0.30133768916130066, -0.6575523018836975, -0.4974345266819, 0.06580377370119095, 1.0029627084732056, -0.32252299785614014, 0.5474295020103455, 1.263755440711975, 0.5208517909049988, 1.1983067989349365, 0.1475025713443756, -0.4657813310623169, -1.6956076622009277, 0.44312864542007446, -0.8390461802482605, -0.06246258318424225, -1.1037551164627075, 1.0998971462249756, -0.11975059658288956, 1.2503159046173096, -1.1038190126419067, -0.27072805166244507, -0.5822930335998535, -1.5075277090072632, 1.5152249336242676, -0.09680645167827606, -0.09751737117767334, 2.4112985134124756, 0.8159107565879822, 0.24908314645290375, 0.8122389912605286, -1.343923807144165, -0.90805983543396, -1.476336121559143, -1.3363933563232422, 0.3494938611984253, 1.3119288682937622, -0.23566783964633942, 1.149551272392273, -0.19937646389007568, -0.35887011885643005, 0.18247433006763458, 0.9090389013290405, -0.9257763028144836, -1.4095063209533691, 1.9738677740097046, 0.22700969874858856, -2.115974187850952, 1.30197012424469, -0.8750721216201782, -1.1029301881790161, 0.7166870832443237, 1.340868592262268, -0.16631443798542023, 0.7000712752342224, -0.45944637060165405, 1.9193757772445679, 1.7988635301589966, -0.8756802678108215], [1.4114497900009155, -0.5436955094337463, 1.175033450126648, 2.842384099960327, -0.03912864252924919, -0.5897865295410156, 0.2238072156906128, -0.28271231055259705, 1.5587629079818726, 0.9310596585273743, 0.7459288239479065, -1.3355481624603271, -0.9678940176963806, -0.338891863822937, -0.4434514045715332, -0.9976882934570312, -0.0199118684977293, -0.3205816447734833, 0.8951016664505005, 0.12295942008495331, 1.0762476921081543, -0.7381855249404907, -0.03494476154446602, 0.4505029320716858, 2.0418829917907715, -0.15160106122493744, -0.6339171528816223, 1.0428431034088135, 0.44651558995246887, -1.0884687900543213, 0.5822513103485107, -0.31434839963912964, 0.9949865937232971, -1.0279425382614136, -0.9843912124633789, 1.292405366897583, 1.377397894859314, 0.69370436668396, 0.5691338181495667, 1.1746238470077515, -0.4064388871192932, 0.6491640210151672, 2.133441925048828, -0.5573566555976868, -0.5293202996253967, -1.2089027166366577, 1.0382745265960693, 0.7339493036270142, -0.7862469553947449, -1.929651141166687, -0.4188581109046936, -0.873322069644928, 0.3590281903743744, -0.46085405349731445, 0.4275812804698944, 1.4602653980255127, -0.25210630893707275, 1.5446372032165527, 0.2627146244049072, 0.8670122623443604, 0.6271964311599731, 1.376291036605835, -0.13837316632270813, -0.9881077408790588, 1.015779972076416, 0.5196311473846436, 0.07792799174785614, 1.7046905755996704, -0.7532227635383606, -0.6663874387741089, -0.07489574700593948, 0.7387869954109192, -0.03829075023531914, -1.378775715827942, 1.0850163698196411, 0.7661912441253662, -0.6435254812240601, 0.6145368218421936, -0.49855631589889526, -0.3344276547431946, -1.4370527267456055, 1.735590934753418, -0.11493822187185287, -0.5339744687080383, 0.8758354187011719, 1.5019434690475464, -2.205047130584717, 2.1971840858459473, -0.3407994508743286, -0.3541765511035919, -0.5737257599830627, 0.23580458760261536, 0.31068041920661926, -1.8742823600769043, 1.1495366096496582, -0.06140550225973129, -0.8754802346229553, 1.9019604921340942, 0.7902020812034607, 1.3799511194229126, -0.046202946454286575, -1.2147412300109863, -1.2978559732437134, -0.7712319493293762, 0.39822813868522644, 1.4096583127975464, 2.1429126262664795, -0.4729088842868805, 0.5043198466300964, -0.94479900598526, 0.07489486783742905, 0.6008983254432678, 0.25490301847457886, 0.5542442798614502, -0.19921144843101501, -1.2793387174606323, 0.4262950122356415, -0.5871462225914001, -0.3580474257469177, 1.7084672451019287, 0.27076929807662964, -0.7953771948814392, -0.08932603150606155, 0.330060750246048, 0.7530138492584229, 0.2911081910133362, 0.3589416742324829, 0.6983155608177185, -2.1381995677948], [-0.7168322801589966, -0.9772967100143433, 1.5524108409881592, 1.0015206336975098, 0.5218891501426697, -2.4746501445770264, -0.4943893551826477, -1.4383857250213623, 0.47096970677375793, -1.226141095161438, 0.6458808779716492, -0.9818269610404968, 0.4378426671028137, 2.293445587158203, 0.39579087495803833, 0.29051798582077026, 0.803205132484436, 0.8367363214492798, -1.6901664733886719, 0.02528093196451664, 1.8390882015228271, 0.6525395512580872, -0.9128044843673706, 0.03631788492202759, -0.5046406984329224, -0.0051545649766922, 0.32681310176849365, 1.0866652727127075, -1.8723136186599731, -0.5540217161178589, 0.3594682514667511, -0.01521097868680954, -1.9910752773284912, -1.335414171218872, 0.06654173135757446, 0.6189604997634888, -0.07740026712417603, 0.06806673109531403, 0.168370321393013, -0.24387246370315552, -0.5907863974571228, 0.8936972618103027, -0.05702733248472214, -0.2341635823249817, -1.9359701871871948, 2.707869291305542, -0.9531662464141846, 0.22708068788051605, 0.41874706745147705, -0.6901251673698425, 0.2328062355518341, 0.5188756585121155, 0.529406726360321, -0.8425847291946411, -0.3163916766643524, 1.2420711517333984, -1.2614341974258423, 0.893467903137207, -0.5261223316192627, 2.200470209121704, 1.2618881464004517, 0.9564656019210815, 0.28495359420776367, -1.8547197580337524, 1.7010700702667236, -0.6365110874176025, -0.32589808106422424, 0.47838321328163147, -0.3653576672077179, -0.9157164692878723, -0.395860880613327, -1.0887665748596191, 1.6891756057739258, -1.4213844537734985, -0.40921467542648315, 0.4457223117351532, -1.4705784320831299, -1.4618695974349976, -0.9883117079734802, 1.297454833984375, 1.1445457935333252, -1.6118717193603516, 1.0646148920059204, 0.678642749786377, -0.31483861804008484, -0.4919086992740631, 1.04843270778656, -0.4861812889575958, -0.12893293797969818, -0.453840970993042, -0.3855147957801819, 0.17799489200115204, -2.1970865726470947, 0.341253399848938, -1.1341384649276733, 2.2221298217773438, 1.6493476629257202, 1.1081624031066895, -3.136293649673462, -1.1818737983703613, -0.5471700429916382, -1.1959818601608276, 1.2888108491897583, -2.3354604244232178, 0.03930690139532089, -0.8365144729614258, 2.328998327255249, 0.8447557687759399, -1.2414934635162354, 1.0237151384353638, 1.3851938247680664, -1.0223244428634644, -1.0851008892059326, -0.9312190413475037, -0.5118579268455505, -1.5995206832885742, 0.37689894437789917, 0.4977792203426361, -0.7491056323051453, 1.1076563596725464, 1.937923550605774, 0.8876242637634277, -0.6268026828765869, 1.5299880504608154, -0.36307063698768616, 0.35126012563705444, -0.019180072471499443, -1.1941474676132202, 1.3616560697555542], [-1.0342488288879395, -0.3718908131122589, 0.9822304844856262, 0.60201096534729, -1.253477931022644, 0.6732317209243774, 0.42614662647247314, 0.13815093040466309, 0.8524606823921204, -1.0671955347061157, -1.065459966659546, -0.41907474398612976, -0.8409081101417542, 1.1608860492706299, -0.9019145965576172, -0.49301835894584656, 0.32868728041648865, 0.2106957882642746, -0.07750268280506134, 1.4027376174926758, -1.6823838949203491, -2.0185861587524414, -2.1313319206237793, 1.5232479572296143, 0.1748759150505066, 2.0102267265319824, -0.04124539718031883, -0.15938930213451385, 0.5440582036972046, 0.45993953943252563, -0.09547301381826401, 0.5916306376457214, -0.4151444435119629, 0.552111029624939, 0.060798920691013336, 0.9353134632110596, 0.2574370801448822, -1.2411783933639526, -1.1569242477416992, 1.0484288930892944, 1.2539725303649902, -1.228519082069397, 0.1994117647409439, -0.9821832776069641, -2.1308319568634033, 0.28983885049819946, 0.7036222219467163, -1.1904276609420776, 0.3757009506225586, 1.0972239971160889, -0.21639806032180786, -0.17965111136436462, 0.8046442866325378, 2.364396810531616, -1.211002230644226, -0.03501756116747856, 1.9606860876083374, -0.0800996646285057, -0.18070974946022034, -0.07975243777036667, -0.442164808511734, 0.836206316947937, -2.148099899291992, 0.015511658973991871, 1.3016717433929443, 0.17898377776145935, -0.432981014251709, 0.3561778962612152, 0.3214712142944336, 1.281259298324585, 0.548241913318634, 1.8316370248794556, -0.37596502900123596, -0.4410470128059387, 0.502459704875946, -0.12984782457351685, 0.14194579422473907, 0.4595010280609131, -0.20606505870819092, -0.2055656462907791, 0.6354876756668091, -0.5835562944412231, 0.8003203868865967, 0.2672284245491028, -1.0686628818511963, 0.9931312203407288, 0.06279002875089645, -0.1025146022439003, -0.9856541156768799, 1.1013572216033936, 1.7053537368774414, 0.12889361381530762, -0.4722561240196228, 1.6395562887191772, -0.44905635714530945, 1.0319061279296875, 0.5053449869155884, 1.5650140047073364, 2.0973129272460938, -0.6819759607315063, -0.1413765847682953, -0.7730895280838013, 0.0148603031411767, -1.492043137550354, 1.6428589820861816, 0.329587459564209, -1.4397413730621338, 0.8409712910652161, -1.792243480682373, -2.1061112880706787, 0.052121181041002274, -1.409475564956665, -1.096891164779663, 1.131049633026123, -2.024811267852783, -1.11566162109375, 0.8366734385490417, -1.3030579090118408, 1.0260847806930542, 0.7384447455406189, 0.713713526725769, -1.1942473649978638, 2.359987735748291, 1.3483426570892334, 0.7107017636299133, 1.564632534980774, 0.5839180946350098, -1.3270225524902344, 0.27537018060684204], [-0.7563385367393494, 0.40954846143722534, 1.9566593170166016, 2.0560379028320312, -1.196955680847168, 0.8501129150390625, 1.7508513927459717, -0.2368127852678299, 0.2279290407896042, 0.024383239448070526, 0.31173959374427795, 0.24098694324493408, -0.26807740330696106, -1.2504127025604248, 0.7852392196655273, -1.0398621559143066, -0.07844143360853195, 0.9468129873275757, -1.0900272130966187, 1.2226001024246216, 1.3699935674667358, 0.43005481362342834, 1.7120542526245117, 0.675422191619873, 1.9469609260559082, 1.3533909320831299, -0.1321079134941101, -0.5453929901123047, -0.4478214681148529, 0.38637715578079224, -0.7839350700378418, 0.7618823051452637, 0.8313008546829224, -0.8940091133117676, 0.499881774187088, 1.0173311233520508, -1.5538820028305054, -0.7943578362464905, -0.37965768575668335, -0.7494333982467651, -1.0889945030212402, 0.9674924612045288, 0.8620506525039673, -0.45357540249824524, -0.9149990677833557, 0.07195921242237091, -1.2290685176849365, -0.8376381397247314, 0.5416117906570435, 0.15695226192474365, -0.013715364038944244, -0.7062250971794128, -0.7611951231956482, 1.6756062507629395, 0.1932263970375061, 1.2254594564437866, 0.6095369458198547, -0.229999378323555, -0.06345925480127335, -0.362920343875885, -0.4337118864059448, 0.584649920463562, 0.4587445855140686, -0.5513451099395752, -1.8337122201919556, 0.3943425714969635, -1.3344448804855347, -0.3749968111515045, 0.3669549822807312, 1.0808428525924683, -0.9134382605552673, 0.276871919631958, -0.8146906495094299, -0.15574637055397034, 0.4373912811279297, 1.1813913583755493, 0.39560550451278687, 2.5703723430633545, -1.868800163269043, 0.637100338935852, 0.056189827620983124, 0.8259139060974121, -0.09634646773338318, -0.4473923146724701, -0.13690175116062164, -2.1119298934936523, -0.23427286744117737, -0.26720190048217773, -0.4280274212360382, -0.20307934284210205, -0.3412989675998688, -1.3280274868011475, 2.095590114593506, 0.9025005102157593, -0.608638346195221, 0.11933627724647522, -0.2397908717393875, 0.1058020144701004, -0.07171911001205444, -1.6261500120162964, -2.494784355163574, -0.7959310412406921, 0.6720086932182312, 0.6697928309440613, 0.0755426362156868, 0.10834087431430817, -0.620436429977417, 0.3321758508682251, -1.9555805921554565, -0.04773480072617531, -0.3417161703109741, -0.559425413608551, -0.1093050017952919, 0.08566229790449142, -0.03365306556224823, 0.5069371461868286, -0.24871240556240082, 0.5036870241165161, 1.1966302394866943, -0.6085253357887268, 0.3983490467071533, 0.07118356227874756, -0.1948467344045639, -0.18051575124263763, 0.5185595154762268, -2.383361339569092, -1.9223201274871826, -1.0634541511535645, 1.5440908670425415], [-0.8729752898216248, -1.3091862201690674, -1.5895473957061768, 0.327483594417572, 1.1837879419326782, 2.174499750137329, -1.1920621395111084, -1.2942434549331665, -1.1888788938522339, 1.0697822570800781, 2.3919692039489746, -1.1007646322250366, -0.8118270039558411, -0.6893431544303894, -1.1237359046936035, -0.11372724920511246, -1.4011842012405396, -0.088679239153862, 1.3276070356369019, -0.24043160676956177, -0.03826947137713432, 0.5235996842384338, -1.7244375944137573, -0.08551113307476044, 0.11536258459091187, 0.9182603359222412, -0.37539607286453247, 1.2187106609344482, -0.49990391731262207, 0.150502547621727, 0.7482925653457642, -1.3983899354934692, -0.870140016078949, -0.5497591495513916, -1.3873910903930664, 0.6460016965866089, -0.06393982470035553, -0.4075007736682892, -0.9896400570869446, -0.23345373570919037, 0.9043545126914978, 0.056454211473464966, 0.45252278447151184, -0.3022950291633606, -0.6928052306175232, -2.9895031452178955, -1.2693263292312622, -0.44710102677345276, -0.8993580937385559, -1.074122428894043, -0.2006741166114807, -0.08281063288450241, -1.6124752759933472, 1.56273353099823, -1.2377110719680786, 0.11955246329307556, -0.31643059849739075, 0.610194206237793, -0.9714497327804565, 3.216053009033203, 0.32799315452575684, -0.7089375257492065, 0.3379858732223511, 0.005677997600287199, 0.16124007105827332, 0.2416795790195465, -0.8572510480880737, 0.2779387831687927, -0.8994965553283691, -0.6529301404953003, -0.5096496939659119, 1.4071279764175415, 1.7908215522766113, -0.6252047419548035, -0.496830016374588, -0.4207213222980499, 0.6214382648468018, 1.4446845054626465, 0.06781354546546936, 1.495448112487793, 0.13132008910179138, -1.448325514793396, -0.6690850257873535, 3.1803553104400635, 0.24190117418766022, 0.8900582790374756, 0.8169363737106323, 0.6699347496032715, -0.041422583162784576, -1.3468043804168701, -1.300835132598877, -1.9851891994476318, 0.07376638799905777, 2.162186861038208, 1.6357122659683228, -1.703917145729065, -0.8653216361999512, -1.0224177837371826, 0.8928074836730957, 0.8704544305801392, -1.0375237464904785, 1.0769892930984497, 1.0099937915802002, -1.7705105543136597, -0.108698271214962, 0.9991841912269592, 2.241182327270508, -0.48315349221229553, 1.5888903141021729, 0.457854300737381, 0.23096562922000885, 0.486347496509552, 0.08979085832834244, 0.31436994671821594, -0.004769085440784693, -0.6571217179298401, 0.2070332169532776, 0.22398491203784943, 0.3619039058685303, -0.24057123064994812, -1.1773427724838257, -1.424879550933838, -1.1277703046798706, 0.4506050944328308, 0.5595042705535889, -1.1413216590881348, -0.7990357875823975, -1.319838285446167, -0.2000548243522644], [-1.1611050367355347, 0.965912938117981, -0.39126935601234436, 1.4428799152374268, -0.8313763737678528, 0.06127895414829254, -0.7446978092193604, 1.2710635662078857, 0.13918355107307434, -1.1678497791290283, 1.0855337381362915, -1.834644079208374, 1.2739108800888062, 0.2080204337835312, -3.354314088821411, -0.4650106430053711, -0.015785062685608864, -0.10291864722967148, 1.4081357717514038, -0.4030999541282654, -0.638500988483429, 0.8825216889381409, -0.5312523245811462, 0.4915328323841095, 0.12903551757335663, 0.8970299363136292, 1.5256707668304443, -0.2277030348777771, -0.9597050547599792, -0.26815372705459595, 0.7787026762962341, -0.4716170132160187, -1.8078445196151733, 1.3464267253875732, 0.4884141981601715, 1.939687967300415, -0.3692673444747925, -0.9859562516212463, 0.12478826940059662, 0.25824224948883057, -0.6324078440666199, -0.23960256576538086, 0.1375524252653122, -1.4077457189559937, 1.4087518453598022, 0.5713257193565369, 0.8091391324996948, -1.7291220426559448, -0.13966332376003265, -0.7394760251045227, -0.6132954955101013, 0.6649404764175415, 0.9177941679954529, -0.26835131645202637, 2.1743521690368652, 1.7374275922775269, -1.8181354999542236, -0.12099713087081909, 0.06383492052555084, -0.00018678820924833417, -0.4503973424434662, -0.3503764867782593, -0.5750619173049927, 1.8288543224334717, 0.6046115756034851, -0.2745477855205536, 0.5115252733230591, -0.16417574882507324, -0.34500351548194885, -0.1727464348077774, -0.37296050786972046, 0.5006458163261414, 1.734163522720337, -0.15732213854789734, -1.5034922361373901, 2.5925018787384033, -1.432483196258545, 1.0550036430358887, -1.1298521757125854, -1.1068758964538574, -0.9645174741744995, -0.10339652001857758, -0.18207530677318573, 0.08193176239728928, 1.129132628440857, 0.7711115479469299, 1.0143777132034302, -0.35685399174690247, 0.12588246166706085, 1.5557246208190918, -0.3778640627861023, -0.2931682765483856, 0.41952401399612427, 0.6234222650527954, -0.06388339400291443, -0.88646399974823, 0.4190956652164459, -0.9321651458740234, 0.014612091705203056, 0.72576504945755, 0.9824259877204895, 0.6427008509635925, -1.2345187664031982, 0.16111811995506287, 0.9940046072006226, 1.8564523458480835, -0.05707428976893425, -0.2978716492652893, -0.6593403220176697, 0.7068080306053162, -0.22567987442016602, -0.3073149621486664, -0.24838809669017792, -1.6784613132476807, -0.4391285181045532, -1.1293238401412964, 2.4499075412750244, -0.48666059970855713, -0.2718191146850586, 0.014218559488654137, -1.276672601699829, -0.9655272364616394, 1.9034473896026611, -0.13143546879291534, -0.3451479375362396, -0.9408103823661804, -1.984233021736145, -0.13327379524707794, -0.1854850947856903], [-1.4179866313934326, -0.2761777341365814, -1.2758963108062744, 0.910488486289978, 0.4535053074359894, 0.7938809394836426, 1.2483469247817993, -0.23035766184329987, -1.6337461471557617, -1.884002447128296, -0.902421236038208, -0.08528857678174973, 0.4081563949584961, 0.7784478664398193, -1.2215713262557983, -0.757063627243042, 1.9274828433990479, 0.17491896450519562, 0.9940961003303528, -1.4444001913070679, 0.7403804063796997, -2.253009557723999, 0.9370729327201843, 0.7066624164581299, 0.801184892654419, -0.11193150281906128, 0.14586549997329712, 0.8194165229797363, -0.0008275836589746177, 0.36531969904899597, -0.595785915851593, -0.20673590898513794, -0.8997995853424072, 1.8258928060531616, -0.117363840341568, -0.1681097149848938, 1.0870122909545898, -0.38476207852363586, 0.6435506939888, -0.6763940453529358, 2.158092975616455, -0.9039261341094971, 0.6359143853187561, -0.344754159450531, 0.04686561971902847, -0.14431744813919067, 0.3080844283103943, 0.23880437016487122, -0.9000705480575562, -1.3779672384262085, 0.5015515685081482, -2.2205770015716553, 1.6150656938552856, 1.6638844013214111, 0.04211297631263733, -0.5443394184112549, -2.5939674377441406, 0.2150096297264099, -0.47238025069236755, -1.4116652011871338, -0.4766494035720825, 0.6813102960586548, 1.1342922449111938, 2.103469133377075, 0.029353711754083633, 0.9962942600250244, 0.11181994527578354, -0.33047670125961304, -0.35059595108032227, -0.42872869968414307, -0.8029125332832336, -0.1272662729024887, -0.26407021284103394, -0.09142319113016129, -0.4007422924041748, -1.5209954977035522, 0.4374375641345978, -0.24649429321289062, -0.6211531758308411, 0.9376574158668518, -0.0927850529551506, 1.4735832214355469, -0.2612552344799042, -0.30109909176826477, -0.3904232084751129, 0.5341231226921082, -0.5647026300430298, -1.9327216148376465, -1.0050866603851318, 1.7276756763458252, 0.20756031572818756, 0.688891589641571, 0.23824970424175262, 0.5743239521980286, -0.7806599736213684, -1.2740700244903564, -0.5817009210586548, -0.20587490499019623, -0.276412695646286, -0.046486228704452515, -1.6003491878509521, 0.7688924074172974, 1.0350018739700317, 0.5029416084289551, 0.30750447511672974, 1.0343716144561768, 2.403235912322998, 1.1444623470306396, -0.37129679322242737, 1.286080241203308, -0.4969690144062042, -1.0855956077575684, -0.7897874116897583, -0.5517682433128357, -0.45243972539901733, 0.916740357875824, 0.3404298722743988, -0.3864128291606903, -0.7182775139808655, -0.08840255439281464, -0.6017278432846069, 0.09244886040687561, 0.2379351705312729, 1.456540584564209, -0.07902944087982178, -0.02203754149377346, 0.06096749007701874, -0.6769789457321167, -0.3568643629550934], [-1.904403805732727, -0.4278612434864044, -1.06937575340271, -1.3206651210784912, 0.6560834646224976, -0.440932959318161, 1.054694414138794, 0.19710606336593628, 0.5606284141540527, -0.1920728087425232, -0.48414498567581177, 0.8510426878929138, 0.4114525616168976, 0.9819701313972473, -0.35630130767822266, 0.5726978182792664, 1.7162929773330688, -0.13058796525001526, -0.048152629286050797, -0.6006262302398682, 1.7558038234710693, -2.415680170059204, -1.2280235290527344, 0.12213160842657089, 0.24986539781093597, 1.0308208465576172, -0.11648744344711304, -1.114140272140503, -0.347410649061203, -0.43925195932388306, -1.4626387357711792, -0.1260092705488205, -0.028858214616775513, -0.34632769227027893, -0.30823129415512085, -0.04729005694389343, 1.0328668355941772, 1.765006422996521, 1.7062878608703613, 0.842511773109436, 0.22555775940418243, 1.1141579151153564, 0.195186048746109, -1.2445560693740845, -1.7341320514678955, 0.9457864761352539, 0.6747630834579468, 0.26518335938453674, 1.0093847513198853, 0.5220403075218201, -0.9048203825950623, -0.8784235119819641, 0.29710718989372253, 0.19300775229930878, 0.15586495399475098, 0.5171023011207581, -1.4306235313415527, 0.0127076655626297, 0.051403213292360306, -1.483771800994873, -0.5888959169387817, 0.5739904642105103, -0.30861368775367737, 0.42297035455703735, -0.03092535026371479, -1.1445711851119995, 0.6834455728530884, 0.4591866433620453, -0.43461060523986816, 0.06040523201227188, -0.16124799847602844, -0.1741487979888916, -0.16342109441757202, -0.6006535291671753, 0.9329923987388611, -0.5179829001426697, -0.6696823835372925, 0.8978955745697021, -0.17649808526039124, 1.5834230184555054, 1.3341714143753052, -0.7684048414230347, -0.2573803663253784, 0.46877145767211914, -3.051778554916382, 0.4486255347728729, -0.4044037461280823, -0.5622366070747375, -0.9653605222702026, -1.3530330657958984, 0.14141085743904114, 0.5265084505081177, 0.5194129347801208, -1.6324890851974487, 0.9613897800445557, -0.17908518016338348, 0.49998876452445984, -1.1173210144042969, -1.0252625942230225, 0.48202431201934814, 0.6078559756278992, -1.5473421812057495, 0.8422885537147522, -0.038718242198228836, -1.0715999603271484, -0.9059999585151672, 0.9925397634506226, -0.9150921702384949, -1.3006045818328857, 0.1622539460659027, 0.5025880336761475, -1.550850510597229, 0.20309385657310486, -1.3038694858551025, 1.5197157859802246, 2.508715867996216, -0.5425989031791687, 0.5831913352012634, -0.10292121767997742, 1.6741160154342651, 0.4730941951274872, -0.7672475576400757, -0.8587303161621094, 1.6772925853729248, -0.2512224316596985, -1.144720196723938, -0.5542138814926147, 0.10672841966152191, -0.24037185311317444], [-0.6400079131126404, 2.0893490314483643, 0.08395237475633621, -0.627025842666626, 1.8823727369308472, -0.528130292892456, -0.6639792919158936, -1.712560772895813, 2.015117645263672, 0.07899686694145203, 0.8114339709281921, -0.096489317715168, -1.7124788761138916, -0.8002084493637085, -0.15564978122711182, -1.4275033473968506, -0.21094828844070435, 0.22800856828689575, -0.7987726926803589, 0.03499249741435051, -1.5812468528747559, -1.04884934425354, -0.4674518406391144, -0.9143822193145752, 1.5907179117202759, -0.2464253306388855, -1.0868691205978394, 1.090823769569397, 0.34666314721107483, 1.7827152013778687, 0.2421284019947052, -0.48411548137664795, 0.45405223965644836, 0.052425310015678406, 0.5747465491294861, 1.5177737474441528, -0.46598154306411743, 1.2102049589157104, -1.3454781770706177, 1.6188552379608154, 0.11828713864088058, 0.1672622263431549, 0.1942148506641388, -2.2026455402374268, -0.6088127493858337, 0.2981385886669159, 1.7220150232315063, -0.7422268986701965, -0.8683210611343384, -0.06544607132673264, 1.3033089637756348, -0.8808900713920593, 1.258426308631897, 1.328658938407898, -0.6465056538581848, -1.024749517440796, 0.9275210499763489, 0.9464121460914612, 0.9675427079200745, 1.6506474018096924, -0.5849620699882507, -0.29501447081565857, -0.07258239388465881, 1.7982341051101685, 0.7840142846107483, 0.0191151425242424, 0.10405895113945007, -1.4194817543029785, -0.867236852645874, 0.5361372828483582, -0.25234779715538025, 0.18783779442310333, -0.14058278501033783, -1.3362516164779663, 0.6229661703109741, -0.26120567321777344, -1.0603282451629639, 1.002891182899475, 0.5592415928840637, -0.11469276249408722, 0.11095242947340012, 0.9703230261802673, 0.7499328255653381, 0.22527366876602173, -0.29918643832206726, -0.026520630344748497, 0.6540413498878479, 0.945067286491394, 0.3666338622570038, 0.04599869251251221, -0.4369606673717499, -1.764374852180481, 1.5691163539886475, 1.0655874013900757, -0.915837824344635, -0.11179229617118835, 0.9946019649505615, -0.8347750902175903, -0.67451012134552, -1.5244982242584229, -0.8311710953712463, 0.591679573059082, -1.575676441192627, -1.4671300649642944, -0.3611041009426117, 0.43377265334129333, 0.6301693916320801, -0.11451338231563568, 0.09498697519302368, -0.2063138782978058, 0.3791654407978058, -0.10496417433023453, 1.0567646026611328, 1.460752248764038, -0.269926518201828, -2.3098058700561523, -0.28111422061920166, -0.5574438571929932, 0.12790359556674957, 0.20074568688869476, 1.134658932685852, 0.5571631193161011, 1.3033363819122314, 0.47424572706222534, -1.6353261470794678, -0.709071695804596, 0.31286925077438354, 0.6362606883049011, -0.7103291749954224], [-0.8248708844184875, -0.5259466171264648, 0.0948038250207901, -0.5958280563354492, -0.6661695241928101, -0.16185735166072845, 0.12623943388462067, -0.005338401533663273, 0.6866806745529175, 1.5825244188308716, 1.546409249305725, 0.0674087405204773, -1.0861581563949585, 0.8296146392822266, 0.4528653025627136, -0.5248699188232422, 0.5110618472099304, 1.9260996580123901, -1.5665380954742432, 0.28020766377449036, -0.45073187351226807, 1.06902277469635, -1.2987980842590332, -1.3509560823440552, 0.3407643139362335, 0.886983335018158, 0.7381982207298279, 1.4443734884262085, 0.14416785538196564, 1.317362904548645, -1.1610051393508911, -0.03312478959560394, 0.832861602306366, 0.3334674835205078, -1.16159987449646, 0.19740884006023407, 0.975710391998291, -0.03420056402683258, -1.3258856534957886, -0.043431397527456284, 1.2909671068191528, 0.19470900297164917, 1.0209904909133911, -1.1385738849639893, 0.6298301219940186, 0.8144221305847168, -0.1266687959432602, 0.023778334259986877, 0.12364166229963303, -1.1136888265609741, -0.5314040184020996, 0.5892052054405212, -1.53257155418396, -0.09056703746318817, -0.0030088494531810284, -0.5665155053138733, -1.806736707687378, 0.5041976571083069, 1.0391281843185425, 0.311160147190094, 0.14468955993652344, 0.7349330186843872, -1.1718968152999878, 0.018980199471116066, 0.09902346879243851, 1.636961579322815, -0.5712687373161316, 1.2703893184661865, -0.2532446086406708, -0.5889322757720947, 1.5465401411056519, 0.17800304293632507, -0.011435406282544136, 0.06471168249845505, -0.37791284918785095, -1.0115920305252075, -0.30610042810440063, 0.04407870024442673, 0.28977492451667786, -2.356361150741577, 0.18436793982982635, -1.0933016538619995, -0.22845184803009033, 1.6503665447235107, 1.2649335861206055, -0.16719456017017365, 1.6157405376434326, 1.5486352443695068, -0.6051971912384033, 0.607765257358551, -1.0675421953201294, -0.43132805824279785, -0.336160808801651, -0.7456096410751343, 0.02376890368759632, 0.6859789490699768, 0.19074073433876038, -1.0242904424667358, 1.0849096775054932, 1.5508980751037598, 0.26366525888442993, -1.9140894412994385, 0.4878758490085602, 1.4554375410079956, 0.7714942097663879, 0.04097672179341316, 0.6196229457855225, 0.13300566375255585, 0.9825051426887512, 1.67897367477417, 0.5328410863876343, -3.0110864639282227, 0.0583769790828228, 0.06863092631101608, 0.13070997595787048, -0.4797825515270233, 0.07002705335617065, 0.29750096797943115, -0.1785905659198761, 0.7069467902183533, 0.9249487519264221, 0.38041120767593384, -2.2192482948303223, -1.2993730306625366, -0.7673718929290771, 0.4650808870792389, -0.8483813405036926, 1.61565101146698, -0.2345241904258728], [0.934212863445282, -0.8157374858856201, 1.011999249458313, 1.1084851026535034, -0.4756331741809845, -1.4723819494247437, 1.6564810276031494, -1.6315155029296875, 0.8376167416572571, 2.0673739910125732, 0.7385834455490112, 0.46701666712760925, -0.22535477578639984, 1.5228537321090698, 0.3798426389694214, 1.5612788200378418, -1.1933212280273438, -0.5618359446525574, 0.16439974308013916, 0.23136737942695618, 0.3672557473182678, -0.538249135017395, -0.5842919945716858, -0.050245556980371475, 0.37492823600769043, 0.7981643676757812, -1.284738302230835, 1.2445199489593506, 0.297325998544693, 0.15368250012397766, -0.10698822140693665, -0.3304409086704254, 2.37091064453125, -0.29431143403053284, -1.211742639541626, -0.3049144446849823, 0.5062435269355774, 0.7764503359794617, 0.9058820009231567, 0.7953604459762573, 0.6389037370681763, -2.675257444381714, -0.9974988102912903, 0.1677987426519394, 1.0410799980163574, -0.5397271513938904, 1.1475030183792114, -0.16329853236675262, -1.5339595079421997, 0.4734882116317749, 1.7265771627426147, 0.330879271030426, 0.7166886925697327, 0.8506053686141968, -0.8056526780128479, -0.6718535423278809, 0.0282774455845356, 0.9510021805763245, 0.29856979846954346, -1.1560633182525635, 0.7451412677764893, 1.3588619232177734, 1.0574302673339844, 0.33279240131378174, 2.1219534873962402, 1.3905055522918701, 1.2270653247833252, 0.12731914222240448, 0.7335111498832703, 0.7760828137397766, -0.0779973492026329, 0.09357579052448273, 0.0054290443658828735, -0.5916066765785217, 0.22217856347560883, 1.3244848251342773, 0.5589876770973206, 0.3761298656463623, 0.7532354593276978, 0.23370897769927979, 0.982395350933075, -0.5853580832481384, -1.9184069633483887, -0.6567768454551697, 1.4005093574523926, -0.8568806648254395, -0.8269617557525635, -0.1341276466846466, -1.6788004636764526, 0.2679922878742218, -0.6284293532371521, 0.1275721788406372, 0.06093379110097885, 0.17235451936721802, -0.306196004152298, 0.3891459107398987, 0.8265212178230286, -0.8691572546958923, 0.5742619633674622, -0.9503127932548523, 0.3830503225326538, -0.8781999945640564, -2.551943778991699, 0.5466089248657227, 0.27519986033439636, 1.0283493995666504, 0.28337129950523376, 2.105012893676758, 0.781247079372406, 0.24481594562530518, -1.6307462453842163, -0.3480686843395233, 1.7136850357055664, 0.32837989926338196, 0.4238201081752777, -0.37267741560935974, -1.0824605226516724, 0.36456045508384705, 0.15953318774700165, -0.06570310145616531, 0.3712533712387085, -0.250811368227005, -0.7141793370246887, -1.5137465000152588, -1.2187488079071045, 0.7387787103652954, 0.7410613894462585, -0.7173055410385132, 0.30600807070732117], [0.5283246636390686, 1.7884242534637451, 0.23541727662086487, 0.7721948027610779, 0.3689703643321991, 0.7669086456298828, -0.4982103407382965, 0.6584122776985168, 0.6270111203193665, 0.661847710609436, 1.4559990167617798, -0.2506876289844513, 0.461542546749115, 0.042684365063905716, -0.728932797908783, 0.08971603214740753, 0.862913966178894, -1.0489767789840698, 0.08016865700483322, -0.48320290446281433, -0.35334810614585876, -1.4129754304885864, 1.4104911088943481, -0.9283702969551086, 0.40338394045829773, -0.5196236968040466, 0.13726192712783813, 0.009549682959914207, 0.592146098613739, -0.32307174801826477, -0.1286158412694931, -0.09044165909290314, 0.4530053436756134, -0.5210641622543335, 0.5161787271499634, 0.4338698983192444, 0.8538711071014404, -0.3293539583683014, -0.11941490322351456, -0.06744331866502762, -0.32914111018180847, -0.06781534850597382, 0.09852490574121475, 0.7818663120269775, 1.0630288124084473, -0.7769522666931152, -0.17098744213581085, -0.1047929897904396, -0.6552709341049194, 1.4520410299301147, 0.11440218985080719, 0.17343831062316895, -0.47806516289711, -1.3688135147094727, 0.757953941822052, 2.807745933532715, -0.22931940853595734, -1.6000655889511108, -0.3676537573337555, -2.2442047595977783, -1.90091872215271, 0.5855984091758728, 0.4348178803920746, 1.4287413358688354, 1.0110846757888794, -0.6560925841331482, -1.7742358446121216, 0.21021538972854614, 0.38067305088043213, -0.06984592229127884, 0.7127591371536255, -0.3370254337787628, 0.044881872832775116, -0.870762288570404, -1.7270811796188354, 0.45685675740242004, -0.5873837471008301, 0.17223425209522247, -0.5895158648490906, -0.6102359294891357, 1.2310969829559326, 0.34467941522598267, -0.15430191159248352, 0.4586288630962372, 0.4268176853656769, 0.1342998892068863, 0.8524025678634644, 0.7194995284080505, -0.006935573183000088, 1.5645626783370972, 0.7311208844184875, -0.445211797952652, 0.026324966922402382, 0.4769079089164734, 0.6979578733444214, 1.9120930433273315, -0.1182786375284195, 2.5687313079833984, -1.8874882459640503, -0.6009383797645569, 0.44914504885673523, -0.3624289929866791, -1.4147065877914429, 0.16446149349212646, -0.5382668972015381, 0.6360859274864197, 0.3646489083766937, 0.26082882285118103, -0.7855896353721619, 1.9993715286254883, 0.5363211035728455, -1.2362323999404907, -1.7491382360458374, 0.6583021879196167, 0.10455194860696793, -0.7721617817878723, -0.663522481918335, 1.1434063911437988, -0.029674040153622627, 0.4637252390384674, 0.43152734637260437, -0.8648228049278259, -0.4742324948310852, 0.8526915311813354, 0.5639138221740723, 0.8024143576622009, -1.523916244506836, 0.10530249029397964, 0.8523618578910828], [0.7075849175453186, 0.7373979687690735, 0.1276254951953888, -0.8622079491615295, 0.11005843430757523, 2.013054132461548, 1.073664903640747, 1.082655429840088, 0.44134122133255005, 0.771609365940094, -0.5935595631599426, -0.018862824887037277, -0.6613523960113525, 0.7061172127723694, 1.3224327564239502, -1.0433595180511475, -0.7919999361038208, -0.30051150918006897, 0.874287486076355, 0.44538962841033936, 0.42689448595046997, -0.23303785920143127, -0.382051020860672, 0.1825438141822815, 0.7751691937446594, 1.0843554735183716, -1.5849895477294922, 1.3847309350967407, -0.1301102638244629, -0.8673227429389954, 0.748802661895752, 0.02337273769080639, 1.502434253692627, -0.00915258564054966, -0.7449748516082764, 0.9624804854393005, -1.1970431804656982, 0.8947102427482605, 0.922236442565918, -0.03385985270142555, -0.2838275730609894, -1.8612592220306396, 0.8349912762641907, 0.8115416169166565, -0.7200904488563538, 1.1259502172470093, -1.4022998809814453, -0.5929803848266602, 0.6270543932914734, -1.2955527305603027, -0.5170650482177734, 0.01718449592590332, -0.19383662939071655, 0.5912730097770691, 1.4518808126449585, 0.22053848206996918, 0.28424879908561707, -1.2303990125656128, 1.5422662496566772, 0.5956892371177673, -0.9337054491043091, 1.1759390830993652, 0.585228443145752, -0.42233285307884216, -0.4147377610206604, -1.027521014213562, 0.693300187587738, -0.06490831077098846, 0.07825076580047607, 0.28666263818740845, 1.0606521368026733, -0.17260925471782684, 2.508422374725342, -2.0472707748413086, -0.11966779083013535, -0.3943234384059906, 0.831727921962738, 0.8571423888206482, 0.2925257682800293, 0.5968503355979919, -0.6488116979598999, -0.7439525127410889, -0.09358870983123779, 0.13695719838142395, -0.8723102807998657, -0.706023097038269, 1.6693211793899536, -0.040209561586380005, 1.010252833366394, 0.8906434774398804, -0.6549729108810425, 0.2789711356163025, 0.6690658926963806, -0.36317625641822815, -0.018395572900772095, 0.3271420896053314, 0.8859264850616455, 1.288152813911438, -0.44380810856819153, -1.0070942640304565, -1.0469179153442383, 1.3049039840698242, -0.5790727138519287, 2.6728298664093018, -1.7512542009353638, 1.7638707160949707, -0.5737819075584412, -0.1303291767835617, -0.05372368171811104, -0.21864134073257446, 1.6363787651062012, -2.155924081802368, 1.3701348304748535, 0.1440165787935257, 2.007641553878784, -2.186061382293701, 0.6374084949493408, 0.7650119662284851, 1.793062686920166, 1.7367109060287476, -1.403967261314392, -0.35517942905426025, 0.13098351657390594, 0.10267748683691025, -1.8823022842407227, 0.5492836236953735, -0.006781427189707756, -0.7444565892219543, -0.7742375135421753], [-0.3721671402454376, 1.1464133262634277, -0.7352924942970276, 0.40260574221611023, 0.18608082830905914, 1.3375400304794312, -0.7667673230171204, -1.0355137586593628, 1.0605686902999878, -1.1434550285339355, -1.7189953327178955, 0.3353722095489502, -0.008817355148494244, 0.017195196822285652, 0.12021781504154205, -1.0128318071365356, 0.44745227694511414, -1.3950870037078857, -0.4906688928604126, -0.06291183829307556, 0.14639727771282196, -0.7316242456436157, 0.1755685806274414, 0.47172653675079346, 0.7040730714797974, 0.6517540216445923, -0.3870409429073334, 1.7195072174072266, 0.6560052037239075, -1.3387718200683594, -0.4463801383972168, -0.1721559762954712, 1.2899171113967896, 0.9181355834007263, -0.9511749148368835, 0.9057679772377014, -0.22043098509311676, -0.6468906998634338, 0.7803080081939697, -1.0722676515579224, -0.5153859257698059, -0.09861845523118973, -1.651251196861267, 0.2688145339488983, -0.5684418678283691, 0.7551924586296082, -0.9661920070648193, -1.4061253070831299, 0.7135034799575806, 0.7983829379081726, 0.5554462671279907, 1.0912991762161255, 1.3248291015625, -0.5622408390045166, -0.733539879322052, 0.12389053404331207, -1.9157960414886475, -0.5520463585853577, -0.5862107276916504, 0.7986510396003723, 0.22709393501281738, 0.7494948506355286, 0.06794783473014832, -0.6062803268432617, 1.218868613243103, -1.2826926708221436, 0.5473855137825012, -0.5964871644973755, -1.5383068323135376, -0.1017095148563385, -1.3543689250946045, -1.5767406225204468, -0.7315250635147095, 1.3225444555282593, 0.4868847727775574, 1.06557035446167, 3.3075692653656006, 1.3415981531143188, 0.2990752160549164, -0.06224484369158745, -0.1735817939043045, -1.0053369998931885, 0.14762580394744873, -1.0129703283309937, -2.178828239440918, -0.16931824386119843, -0.17154298722743988, -0.2086620181798935, -1.001482605934143, -0.39877381920814514, 0.29669198393821716, 0.5128225684165955, 0.19474324584007263, 2.6393723487854004, -0.7539355754852295, 0.43148475885391235, -0.4112086296081543, -0.6711925864219666, -0.9403656721115112, 0.09978748112916946, -0.8721714615821838, 0.6110634803771973, -0.9264029264450073, 0.6989961862564087, -0.622406542301178, 0.6452656388282776, -0.29230156540870667, -1.5394374132156372, -0.1636658012866974, -0.8739514350891113, -0.8455724120140076, -0.9238330125808716, 1.604384183883667, 1.6284401416778564, 0.8472203016281128, 0.34957021474838257, -0.8405128717422485, 0.056965891271829605, -2.0520808696746826, 0.11247792094945908, -0.3186296820640564, 1.3637415170669556, 0.8334166407585144, -0.04395719990134239, 0.047087352722883224, -1.7431052923202515, -0.9921733736991882, -0.07847968488931656, 1.6757378578186035], [0.22703364491462708, 0.2389344871044159, 0.38947853446006775, 0.38421115279197693, -0.13081718981266022, 1.1163171529769897, -0.36334389448165894, -0.45278748869895935, -0.9547953605651855, -0.03451678529381752, -0.7201990485191345, 0.13665415346622467, 1.3380076885223389, -0.9957998394966125, -0.22466173768043518, -0.2949828505516052, 0.7091073393821716, -0.5051916837692261, 1.5343011617660522, -1.7321585416793823, 0.5649241209030151, 2.6522772312164307, 0.25900596380233765, 0.5157250761985779, 0.4427417516708374, 0.9652459025382996, 0.44295477867126465, 1.3193435668945312, 0.27530771493911743, 1.4335405826568604, -0.7410581707954407, 2.5311503410339355, 1.0786120891571045, -0.3937377631664276, 0.33916938304901123, 0.004713749513030052, 0.9187490344047546, 0.8402595520019531, -0.7783527970314026, -2.3620364665985107, -1.852979063987732, -0.8584108948707581, -0.43613317608833313, 1.7670142650604248, 1.8438855409622192, 0.24633297324180603, -1.0591174364089966, -1.2941861152648926, 0.8381043672561646, 0.1494140475988388, -1.6358656883239746, 0.02061600424349308, -0.9530701041221619, 0.5751040577888489, 0.8814325928688049, 0.5795010328292847, 0.49412792921066284, -1.340337872505188, 0.662881076335907, 1.2406221628189087, -0.944537341594696, -1.717164158821106, 0.9434071779251099, -0.7183548808097839, -1.2244006395339966, 0.19707906246185303, -0.1968478560447693, 1.3285777568817139, -0.04126515984535217, 1.0241196155548096, 0.9938182830810547, -0.33335012197494507, -1.2435760498046875, 0.22835154831409454, 0.7539722919464111, -0.4662530720233917, 1.6507989168167114, 0.9719168543815613, 1.5524816513061523, 0.2017839401960373, -0.48350387811660767, 1.2537574768066406, 1.858659029006958, -0.3799494206905365, -0.6691117882728577, -1.1834237575531006, -2.0732128620147705, 0.7874614000320435, -1.9449467658996582, -1.3957606554031372, 0.7941760420799255, -0.9382700324058533, -1.9030498266220093, 0.5258371233940125, 1.7098898887634277, 1.061797022819519, -1.8122215270996094, 0.3199474811553955, -1.1579188108444214, 1.0159369707107544, -0.6902874708175659, 0.7291852235794067, -1.5418360233306885, 0.49987098574638367, -0.7815613150596619, -0.24763478338718414, 1.2982134819030762, -1.4162683486938477, 1.366795539855957, 1.037675142288208, -1.134435772895813, -0.5362993478775024, 0.9000809192657471, -2.664989948272705, 1.288314700126648, -1.5555561780929565, -2.464221954345703, 1.8010600805282593, -1.4318443536758423, 0.6297702789306641, -2.405071496963501, 1.0826866626739502, -0.08514609187841415, 0.766631543636322, 0.20520247519016266, -0.4071873724460602, 1.1646437644958496, -0.13212184607982635, -0.04821588471531868], [-0.35708707571029663, 0.016539951786398888, 0.3422016203403473, 0.1568654626607895, -0.8175243139266968, 0.5310763716697693, 0.13119059801101685, 0.90591961145401, 1.6399403810501099, -0.3054090738296509, 0.9589858651161194, -1.776058316230774, -0.5056528449058533, 1.802270531654358, -0.42535293102264404, -0.0456601157784462, 2.2017264366149902, 1.5897201299667358, 1.2138422727584839, -0.06448586285114288, 0.6951988935470581, -0.045031361281871796, -1.3681806325912476, -0.24485428631305695, -0.059169553220272064, -0.1113055869936943, 1.1309082508087158, -0.6142242550849915, 0.18461857736110687, -0.4353125989437103, 0.7398894429206848, 0.5175358653068542, -0.796444296836853, 0.7224999666213989, 2.1524593830108643, 2.144348382949829, -0.16064949333667755, -0.5601991415023804, 0.3649236261844635, -0.36368826031684875, -2.516021728515625, -1.609961748123169, 0.6160901784896851, -0.7422316670417786, -0.8118056654930115, 1.666709065437317, 0.010192970745265484, -0.07561438530683517, -2.358814239501953, 1.2823518514633179, 0.6842043995857239, 0.2670937776565552, 0.5791623592376709, 0.5002854466438293, -0.6625282168388367, -0.06368957459926605, 0.1262144297361374, -0.5085724592208862, 1.3798354864120483, -0.2452971190214157, -0.6217402815818787, 0.18032607436180115, -0.7907567024230957, 0.6988141536712646, 0.8910166621208191, -0.629828929901123, -0.07814284414052963, -0.26607784628868103, -0.330912709236145, -0.06929842382669449, -0.08894969522953033, -1.192348599433899, 0.4542198181152344, -0.27326902747154236, 2.0826516151428223, -0.3548036515712738, -0.1059543639421463, 0.6712794303894043, -0.056474506855010986, 0.2689865827560425, -0.19515754282474518, -0.19362618029117584, 0.21198411285877228, 0.36274221539497375, -0.2036074995994568, -1.850041389465332, -0.1417647749185562, -0.020891599357128143, 1.8830033540725708, -0.18045972287654877, -0.06928040832281113, -1.0204622745513916, -0.32013845443725586, 0.12660767138004303, -0.8575901985168457, 0.5862570405006409, 0.6151089668273926, -0.871913492679596, 2.5999090671539307, 0.22022899985313416, -0.8518237471580505, -0.04260401055216789, 0.6928080916404724, -1.179795742034912, 0.21222881972789764, -0.41132089495658875, -2.0123701095581055, -0.005691691301763058, 0.15981778502464294, -1.0720134973526, 0.9278523325920105, 0.7118266224861145, 0.3013755977153778, 1.9465998411178589, -0.24999822676181793, -1.5645841360092163, 0.21510468423366547, -1.0357537269592285, -0.07550829648971558, -0.7262641191482544, 1.0168482065200806, -2.289515972137451, -0.2811260521411896, -1.1540627479553223, -0.5697619318962097, -0.5400371551513672, 0.40682798624038696, -1.1573500633239746, -0.11611364781856537], [0.29928359389305115, 0.11875320225954056, -1.4757659435272217, 1.5821386575698853, -1.4534467458724976, -0.35580524802207947, 0.05010978505015373, 0.780188798904419, 0.601612389087677, 0.4856927990913391, -0.030651861801743507, 0.12253437936306, -0.7104606628417969, 0.8747175335884094, 0.4643557369709015, -0.7316797375679016, 0.38281625509262085, -0.3261650800704956, -1.2850967645645142, 0.8860970735549927, 1.2337350845336914, 1.400391697883606, -0.6460609436035156, 0.6808741092681885, -1.1127049922943115, 0.3445412218570709, 1.1138880252838135, -1.8839752674102783, -1.8548943996429443, 0.1313665211200714, 0.2733367681503296, 0.13303278386592865, 0.2812885046005249, 0.7369672060012817, 0.767228901386261, -1.4530739784240723, 0.26895076036453247, -1.6745574474334717, 1.845633625984192, -0.8921116590499878, 1.4700754880905151, -0.32934123277664185, 0.15621601045131683, -0.4474277198314667, 1.9245244264602661, 1.0060975551605225, 0.32898569107055664, 0.3364914059638977, -0.7711547017097473, -1.3066346645355225, 0.985497772693634, -0.31239935755729675, 1.1755940914154053, -0.8792027831077576, 2.6234071254730225, -0.045080866664648056, 0.04693181440234184, -0.8850955367088318, 2.0538179874420166, 1.5716731548309326, 0.46377426385879517, 0.8687911629676819, -0.18228305876255035, -0.5348970293998718, -1.3720968961715698, 0.5322169065475464, -0.4926597476005554, 0.04818040877580643, -1.8998689651489258, -0.28380462527275085, -0.07079538702964783, 1.681242823600769, -0.33334484696388245, -0.4587963819503784, 1.519079327583313, 0.7383459806442261, -0.28141042590141296, -1.0574419498443604, 0.13987962901592255, -0.521538496017456, -0.7286278605461121, 0.5763784646987915, -0.10209820419549942, 0.8997034430503845, -0.21621529757976532, 1.9123820066452026, -0.7416590452194214, 0.5897477269172668, 0.02319989539682865, -0.5791327357292175, 2.2585675716400146, -0.0026568921748548746, -0.9131296873092651, -1.8297119140625, -1.072634220123291, -0.41291651129722595, 0.6547547578811646, 0.06381683051586151, 0.8457616567611694, -0.8456197381019592, -0.19259193539619446, -2.4569649696350098, -1.8046153783798218, 0.6805618405342102, -2.0404372215270996, -1.1192344427108765, 0.5783841013908386, -0.437188982963562, -0.09691707044839859, -0.6588370203971863, -2.0598838329315186, 0.3572084605693817, -0.08993107080459595, -0.011472390964627266, 1.7476489543914795, -1.1207770109176636, -0.4167774021625519, -0.18428069353103638, -1.5989656448364258, -0.17571072280406952, 0.022966641932725906, -0.2689081132411957, -0.3323321044445038, 0.5691226124763489, 0.4386668801307678, 0.11619402468204498, -0.342365562915802, 0.17792825400829315, -2.362287759780884], [-1.2746813297271729, -1.6460957527160645, -1.0917332172393799, -0.5382635593414307, 1.5214533805847168, -0.2869930565357208, -0.27687177062034607, -0.15787753462791443, 1.7647006511688232, 0.30623987317085266, -0.5356221199035645, -0.9412612318992615, 0.7028490900993347, -0.8891518712043762, 0.01605108007788658, 1.672154188156128, -1.2018970251083374, 1.5745726823806763, -0.822910726070404, -0.5944958925247192, 0.9554967880249023, 1.5896196365356445, -0.7758593559265137, -0.691774308681488, 2.042229175567627, 0.010833758860826492, -0.43117690086364746, -0.3238139748573303, 1.3288545608520508, 0.3271344006061554, 1.6781748533248901, 0.5200595259666443, 1.345134973526001, 0.5703346729278564, 0.2769227623939514, 1.6748454570770264, -0.4261219799518585, 0.03666143864393234, -0.17741659283638, 0.5367821455001831, -0.4613723158836365, -0.1394932121038437, 2.003812313079834, -0.4797363877296448, -0.4360942840576172, -0.8250950574874878, -2.6537160873413086, 0.7098367810249329, 0.14199714362621307, -0.24492290616035461, 1.2821199893951416, -0.13294526934623718, 0.09240399301052094, -2.301149606704712, -0.5237334966659546, 0.7886875867843628, -0.41793540120124817, -0.3200579881668091, -0.06502941250801086, 0.9083406925201416, 0.24543927609920502, -1.1469546556472778, 0.09377340227365494, -0.5786727070808411, -0.7021196484565735, 1.042658805847168, 1.0192170143127441, -0.6159098148345947, 0.12800845503807068, 0.17905296385288239, 0.36667582392692566, -0.550162672996521, -2.0265953540802, 0.37407222390174866, 1.2373124361038208, -0.27338993549346924, 0.46596574783325195, -0.7122884392738342, 3.6018130779266357, 0.6836508512496948, 0.381278395652771, 1.3457399606704712, 1.0764775276184082, 1.9386597871780396, 0.3890840709209442, -1.1406346559524536, -0.27705082297325134, -0.3964802324771881, 0.2411605417728424, 0.7048242092132568, 0.8948420882225037, 0.3447563648223877, 0.8767862319946289, -0.23068687319755554, -2.0412802696228027, 0.7123114466667175, 0.08987446129322052, -0.3969312608242035, 0.22509582340717316, -0.8891376852989197, 0.5502756237983704, 0.8050842881202698, 2.2578563690185547, 0.6463022828102112, -0.9612834453582764, -1.4739375114440918, 1.4303979873657227, -1.8106504678726196, -0.496747761964798, -2.7730712890625, 3.087423324584961, 0.9663899540901184, 0.19675137102603912, 0.02688237838447094, 0.06584230810403824, -1.1523160934448242, 0.7526717185974121, 0.4533163607120514, 0.15470130741596222, -1.66011381149292, 1.1225141286849976, -1.9022045135498047, 0.09480540454387665, 0.9902551174163818, -0.6324178576469421, -0.7340560555458069, 0.39425843954086304, 0.41726720333099365, 0.6283791661262512], [-1.2369247674942017, -0.45949941873550415, -0.3162524998188019, 0.05199158936738968, 1.023726224899292, -0.21386268734931946, 1.101115345954895, 0.8541821837425232, 0.2500775158405304, 1.6986714601516724, 1.547859787940979, -1.6659862995147705, 0.7095586657524109, -0.9575614333152771, 0.22457599639892578, -1.527465581893921, -0.7263758778572083, 0.7478673458099365, -1.153031587600708, -0.9952065348625183, -0.536033034324646, -1.040585994720459, -0.42597031593322754, 0.1035253182053566, -1.8837707042694092, 0.3355262875556946, -0.8773465156555176, -1.1381521224975586, -1.4988776445388794, -0.1582743376493454, 0.15519210696220398, 0.05708397179841995, -0.4016014039516449, 1.1904995441436768, 1.3338022232055664, -0.32927772402763367, 1.2613937854766846, 0.03233093023300171, 1.1872962713241577, 0.6490614414215088, 0.4206331670284271, 0.07053027302026749, 0.40065518021583557, 0.8402898907661438, -0.5514979958534241, 0.5201772451400757, -0.2818900942802429, -0.5752555131912231, -0.003899934235960245, 0.6054608225822449, -0.8168254494667053, -2.119978904724121, 0.8147241473197937, 0.2873903810977936, 0.17979511618614197, -0.6943734288215637, 2.8682432174682617, -1.1188325881958008, -1.830263376235962, -1.3400323390960693, 0.33844542503356934, 1.3040437698364258, -0.5577141046524048, -0.4256362020969391, -1.8870097398757935, 0.6794019341468811, -1.5532983541488647, 0.6697486639022827, 0.20661474764347076, -0.6965367794036865, 0.7931768894195557, 0.8514776825904846, -0.5120260119438171, 0.867591142654419, 0.2583809494972229, 1.6205623149871826, 0.9843435287475586, 0.6247891783714294, -0.23537610471248627, 0.9167318344116211, -0.36442455649375916, -0.10650576651096344, -1.20619535446167, 0.14705967903137207, 0.008583311922848225, -0.12160060554742813, 0.21172931790351868, 0.5569007396697998, -0.7059710025787354, -0.36180543899536133, -2.722309112548828, 0.8178901076316833, -0.9384626746177673, 0.4895540475845337, 1.0581568479537964, -0.8654895424842834, 0.10259073227643967, 0.09740396589040756, 0.21853159368038177, -0.7668948769569397, 1.0769904851913452, -0.3750595450401306, -0.27435004711151123, 0.22819890081882477, 0.05365594103932381, 1.2257983684539795, -1.3279680013656616, -2.278721570968628, 0.1744609922170639, -0.3986496925354004, 1.6587671041488647, -0.7305372357368469, 0.3975859582424164, -0.10101069509983063, 0.06838015466928482, -0.34885546565055847, -0.7164365649223328, 0.9466131329536438, 0.5228127837181091, 1.1841588020324707, -0.760205864906311, 0.3845149576663971, 0.7142679691314697, -1.4059700965881348, -0.5754936933517456, -0.5544036030769348, -0.6583464741706848, 1.738182544708252, -0.46980372071266174], [1.5469061136245728, -0.5612393617630005, 0.40549832582473755, -1.270404577255249, -1.8832191228866577, 1.1277785301208496, 0.5328275561332703, 1.2590748071670532, 0.5715809464454651, -0.19819305837154388, 0.8157219886779785, -1.1272248029708862, 0.03994515910744667, -0.04112264886498451, -0.5757363438606262, -0.5805628299713135, -0.8858008980751038, 2.443796396255493, 1.3376805782318115, -0.14765256643295288, -1.2296785116195679, 2.4890332221984863, -0.3236021101474762, 0.40450945496559143, 0.8503987193107605, -0.8156275749206543, 0.033550698310136795, 1.9005488157272339, -1.3303056955337524, -1.2560629844665527, 0.3893299698829651, -1.5275933742523193, 0.43030086159706116, -0.4720727205276489, 1.607437014579773, -0.06723374873399734, 0.3943300247192383, 0.2937944531440735, 1.6671416759490967, 0.7652803659439087, -0.7542892098426819, -0.3929389715194702, -0.5132325887680054, 0.18685495853424072, -0.10226288437843323, -0.1927061378955841, 0.1962631791830063, -0.3231208324432373, 0.5144456028938293, -0.45825037360191345, -0.771329402923584, 0.15900841355323792, -0.6339653730392456, -2.729156255722046, 0.4603024423122406, 1.1290167570114136, 1.3642432689666748, -1.1951932907104492, 0.8361546397209167, -1.3385496139526367, 0.563971757888794, -0.6902664303779602, -0.35391128063201904, -0.5474386811256409, 1.3371410369873047, -0.007706902921199799, -0.6742686629295349, -0.43020403385162354, 0.10523729026317596, 0.6126885414123535, -1.8249695301055908, 1.36650812625885, 0.6613609194755554, 1.0941188335418701, 2.008898973464966, 0.5488808751106262, -0.4081976115703583, 1.5533522367477417, 0.8121817111968994, 2.18929123878479, 0.5155897736549377, 1.4590681791305542, -0.33683711290359497, 0.2703607976436615, -1.4510971307754517, 0.34592437744140625, 0.06030764430761337, 0.83956378698349, 1.8283766508102417, 1.364970326423645, 1.5363540649414062, 1.7956191301345825, -0.8231088519096375, 0.6088289022445679, 0.4677913188934326, 0.3712778389453888, -0.5187708735466003, 0.3065086901187897, 2.096086263656616, -0.3480626046657562, -1.6603997945785522, 0.7628237009048462, 1.5003639459609985, -0.012904969044029713, 0.33036869764328003, -0.6414098143577576, 0.6058437824249268, -0.04501785710453987, 0.6009914875030518, -0.9975132346153259, -0.2934911549091339, -0.10477235913276672, -2.880683183670044, 1.6074334383010864, 1.869652509689331, -1.2788286209106445, 1.4130985736846924, -1.2391282320022583, 1.337583065032959, -0.6322493553161621, -2.1700470447540283, 1.7924716472625732, -0.5136798620223999, 1.226822018623352, -0.22293896973133087, -0.30602166056632996, 0.8055983185768127, 1.3398184776306152, 0.0026675239205360413], [-0.09151992201805115, -0.951817512512207, -0.3319060206413269, 0.7768070697784424, -0.9179048538208008, -0.404289186000824, 0.18320722877979279, -1.151852011680603, -0.6043957471847534, 0.023296229541301727, -0.6357858777046204, -1.6273561716079712, 0.6667278409004211, 0.22274062037467957, -1.2647373676300049, -0.35098838806152344, -0.636115312576294, 0.9135993123054504, 0.47632962465286255, -0.5104188919067383, -1.8104649782180786, -0.3880905210971832, -1.4879859685897827, -0.08095642179250717, -0.6791321039199829, 0.41690948605537415, -1.1534209251403809, -0.14890266954898834, 0.060074321925640106, -0.42790302634239197, -0.047643933445215225, 0.8408727645874023, 0.09626630693674088, 0.7442682981491089, -0.6261993050575256, -0.538061261177063, -0.00285645155236125, 1.0715998411178589, 0.23590154945850372, 0.6362299919128418, 1.6347442865371704, 1.8284181356430054, -1.1110072135925293, -0.4076247811317444, -0.798801064491272, -0.08897382766008377, -0.5973889827728271, -0.877359926700592, 0.25510847568511963, 0.6923224925994873, 0.8892531394958496, -0.08187262713909149, 0.04544176161289215, 0.4502221941947937, -0.6890427470207214, -0.7786471247673035, -1.060333013534546, 1.9062069654464722, 0.1594177782535553, 0.40075451135635376, 0.24901534616947174, 1.1023340225219727, 0.7845636010169983, -0.6823804378509521, -1.7663943767547607, 0.534454345703125, -1.3517009019851685, 0.48695623874664307, -1.4432638883590698, -0.47576868534088135, -1.6172289848327637, -0.834050178527832, -0.6188110709190369, -1.1946710348129272, -0.07177713513374329, -0.5146248936653137, -0.4854179322719574, 0.8736166954040527, -1.933190941810608, -1.5151008367538452, -0.2532959282398224, -1.7004525661468506, -1.717302918434143, 1.0252041816711426, 2.1019411087036133, 1.0118584632873535, -1.427773356437683, 1.9826722145080566, 0.6920921206474304, -0.060214024037122726, 0.3009317219257355, -0.005882727447897196, 0.26881399750709534, 0.8275837898254395, 0.8069482445716858, 0.30489635467529297, 0.9603952169418335, -0.2541501820087433, -0.3932029902935028, -0.9347960352897644, -0.5899795889854431, 1.081769347190857, -0.20011495053768158, -0.005040397867560387, -0.3822742998600006, 0.5200469493865967, -0.8518480658531189, 1.815351963043213, 1.6239326000213623, -0.6081069707870483, 0.6549748778343201, -1.0600907802581787, -0.7289641499519348, -0.5936408042907715, -0.6932408809661865, -1.0639333724975586, 0.32684025168418884, 1.939321756362915, 0.266598641872406, -0.35666096210479736, 2.1111769676208496, 1.9358985424041748, -0.32652992010116577, -1.3436996936798096, 0.7028263807296753, -0.10246861726045609, -0.5506712794303894, 0.9656694531440735, -0.7636650204658508], [-1.3690460920333862, 0.3653603792190552, -0.03422220051288605, 0.18204453587532043, 0.14315582811832428, -1.0438488721847534, 0.9196535348892212, 0.9970581531524658, 0.06759319454431534, -1.3215738534927368, -0.5871518850326538, -1.0247081518173218, 0.33861178159713745, 0.4565636217594147, 0.8904653787612915, 0.04134612902998924, 0.01310473121702671, -0.2602616548538208, 1.058660864830017, 1.7376736402511597, 0.42377984523773193, 1.4960942268371582, 1.4032682180404663, 0.7351754903793335, 1.576859712600708, -0.07453484833240509, 0.9636712074279785, -0.29275888204574585, -1.3714489936828613, 1.3591508865356445, 2.1631338596343994, 1.3045190572738647, -1.3306349515914917, 0.35978081822395325, 1.25080406665802, -1.5066434144973755, 0.7738751769065857, -0.9148526191711426, -0.05165708065032959, 0.017805302515625954, 0.27252957224845886, 0.6062162518501282, -0.03493892401456833, 0.22929076850414276, 0.5317764282226562, -0.3930692672729492, -0.4380088448524475, -0.24541382491588593, 0.4876369833946228, 1.5312888622283936, 0.7090479135513306, -0.7404693961143494, -1.8776308298110962, -0.3862605094909668, -0.42481517791748047, 0.34212496876716614, 0.7356129884719849, 1.3695017099380493, -0.8382555246353149, 2.092472791671753, 0.037213392555713654, -0.16857826709747314, -0.4122135639190674, -1.582131266593933, 0.25985726714134216, 0.43701839447021484, -0.26300910115242004, 0.4228682816028595, 1.5319184064865112, 0.6179186701774597, 0.4248755872249603, 0.2851337492465973, -1.3058921098709106, -0.11368633806705475, -1.252798318862915, -0.46557343006134033, -1.4963496923446655, -0.006370851304382086, 0.6725323796272278, 0.1302453577518463, -0.4400513768196106, 0.5492385625839233, -0.9747969508171082, 0.03478877991437912, -0.2113499492406845, 0.6620804667472839, 0.15930810570716858, -1.8003039360046387, 0.9928362965583801, -0.8025533556938171, -1.195918321609497, -1.3941890001296997, -0.6855621337890625, -0.6923984289169312, -0.10460307449102402, -1.6116684675216675, 0.24350214004516602, -0.7828916907310486, -1.3246866464614868, 1.075151801109314, -0.11854200810194016, -0.3115122318267822, -1.0219289064407349, -0.7015188336372375, -1.1299837827682495, -0.3830834627151489, 0.6904390454292297, -0.6761146187782288, -0.6527197957038879, -0.1994108408689499, 1.2633819580078125, 0.09961657226085663, 0.059337664395570755, -1.5426981449127197, 0.49293819069862366, 0.3170059621334076, 1.2045084238052368, 0.42383643984794617, -0.412575364112854, 0.21225197613239288, -2.1490914821624756, 0.38708260655403137, 2.239328145980835, -0.3264622390270233, 0.5243415832519531, -0.9464281797409058, -0.9380591511726379, 0.783232569694519, 0.9797541499137878], [-0.16970603168010712, -0.09714125841856003, -0.043359141796827316, -0.5630757212638855, -0.809154212474823, -0.6996565461158752, 1.5988125801086426, -0.5011590719223022, 1.095096230506897, 0.14269514381885529, -1.67052161693573, 0.10721833258867264, -1.1627277135849, -0.9105452299118042, -0.08128520101308823, -0.07729817181825638, -0.1938285529613495, -1.539940357208252, 0.26356929540634155, -0.6022258996963501, -0.8792082071304321, 0.14812374114990234, 0.9243895411491394, -1.0972175598144531, -1.9922858476638794, 0.1816650778055191, -0.14649459719657898, -0.43469101190567017, 0.7102759480476379, -1.5423296689987183, 0.6033690571784973, 0.5963112711906433, 0.9289859533309937, -0.25575268268585205, -1.7625131607055664, 0.7045693397521973, -0.504833996295929, 1.3806859254837036, -0.2798791825771332, 0.5827780961990356, -0.8688937425613403, 0.8916111588478088, -0.3203747570514679, -1.2064028978347778, 0.06373835355043411, -0.3411124646663666, -0.20220303535461426, -1.3540785312652588, -0.07059439271688461, 0.43755054473876953, -0.052408911287784576, -0.37932854890823364, -0.37100672721862793, 1.6594748497009277, -1.1078636646270752, 0.20735572278499603, 0.00646565156057477, -0.29058903455734253, 1.7008795738220215, -0.795903205871582, 0.575553834438324, -0.983002781867981, -0.13306523859500885, -0.45608022809028625, 0.9591337442398071, -0.5747022032737732, -1.7458529472351074, 1.5113263130187988, -0.6445251107215881, 0.25210291147232056, 0.1274091601371765, -1.0679787397384644, 0.25754064321517944, -1.4899293184280396, 1.7654805183410645, 0.47836947441101074, 1.4828935861587524, 0.767184317111969, -0.6890395283699036, 0.3860512971878052, 0.42210936546325684, -1.362345814704895, -1.5998162031173706, 0.20004184544086456, -1.72360098361969, 0.8215717673301697, -1.1858954429626465, -0.048833634704351425, 1.2871894836425781, -0.4596497416496277, 0.7439257502555847, -0.2748042643070221, -1.427573323249817, -0.157541424036026, -0.6081231236457825, 1.3936810493469238, -1.0802419185638428, 1.5883783102035522, 1.114009141921997, -0.3024471402168274, -0.7365745306015015, 1.1499394178390503, -0.10123814642429352, -0.16624529659748077, -0.5628204345703125, 0.11708921939134598, 0.22967709600925446, -0.44606199860572815, -2.900994062423706, -1.3271472454071045, -2.629556655883789, -0.13308016955852509, -2.0002758502960205, 0.806876540184021, 0.6998921632766724, -0.05801104009151459, 1.2474489212036133, -0.20508019626140594, 0.8554781675338745, -0.28655022382736206, 0.9776076078414917, -0.2467651665210724, -0.04224636033177376, -1.8390092849731445, 1.7644761800765991, 0.5046696066856384, -0.1660432070493698, -1.1572049856185913, 0.6637293696403503], [-0.5474374890327454, -1.6189842224121094, 0.4990306794643402, -1.2756520509719849, 1.363134503364563, -0.14984895288944244, -0.7956185340881348, -0.0343170240521431, 1.4372613430023193, 0.6247823238372803, -0.7031654715538025, -0.08132375031709671, 0.27127525210380554, 1.4833505153656006, 0.49322420358657837, 1.4635666608810425, -0.030301112681627274, 0.08788753300905228, -0.06830359250307083, 0.12270012497901917, -0.03736802190542221, 0.8728908896446228, 0.30331480503082275, -0.753562867641449, 0.47872796654701233, -0.9535057544708252, 0.9778530597686768, 0.7434705495834351, -0.8304255604743958, -0.06584146618843079, 1.743313193321228, -0.3701344430446625, -2.2607169151306152, 0.2641931176185608, -0.4528316259384155, -0.03925555571913719, -0.6043127775192261, 0.46864059567451477, 1.2429580688476562, -0.33718642592430115, -0.6362929940223694, 1.0393155813217163, 1.2084121704101562, -0.47132331132888794, -1.7986856698989868, 0.5638858079910278, -0.5485072731971741, 0.5393328666687012, -0.40196144580841064, -0.4505358040332794, 0.544050931930542, -0.3529548943042755, -0.3238694667816162, 0.7868896722793579, 0.5549973845481873, 0.28682535886764526, -1.0647528171539307, -0.5322871208190918, 1.1038631200790405, 0.32415542006492615, -0.2540343403816223, -1.6423323154449463, 0.04358980804681778, -1.0335755348205566, -0.4947446286678314, -1.3107142448425293, -0.27307653427124023, 0.15415659546852112, 1.525943636894226, 0.08117388188838959, 0.08930641412734985, 0.06964258849620819, 0.4416371285915375, 0.41375845670700073, -0.32295939326286316, -1.7956916093826294, -0.05326401814818382, -0.35834309458732605, -1.788476586341858, 2.167433977127075, -1.386979341506958, 1.1476202011108398, -0.2620798349380493, -0.8606071472167969, -0.7360045909881592, 0.7078110575675964, -0.611115574836731, 1.3354264497756958, -0.10480273514986038, 0.6282311081886292, 0.38377252221107483, 0.008508496917784214, 1.9663550853729248, -0.3764605224132538, -0.9287779927253723, -0.367581307888031, -1.061119556427002, 0.08807821571826935, 1.0412427186965942, -0.8786731362342834, -0.6647562980651855, -2.370716094970703, 0.1893366426229477, 0.5830580592155457, 0.8038902878761292, 0.3670744299888611, -1.370867371559143, 0.32424646615982056, -0.11209575086832047, 0.15210694074630737, 0.5349655151367188, -0.508014440536499, -0.6287582516670227, 0.9759216904640198, -0.0955759584903717, -1.3852137327194214, 0.31412357091903687, -1.6498947143554688, 1.3883610963821411, 0.6571546196937561, -1.0110712051391602, -1.9265841245651245, -0.007821185514330864, 0.4862060546875, -0.9147237539291382, -0.3225236237049103, -0.5327802300453186, 0.4975864589214325, 1.0282453298568726], [1.557908535003662, 0.783987820148468, 0.18547506630420685, -0.8570678234100342, -1.6594465970993042, 1.113078236579895, -0.8192127346992493, -0.7141698002815247, -1.1938245296478271, -0.8517863154411316, -1.8648717403411865, -0.8853429555892944, -0.19087667763233185, 0.9908697605133057, -1.2648252248764038, -0.7824019193649292, -0.5760173201560974, -0.3875696063041687, -0.4235566258430481, -0.31255367398262024, -1.6152458190917969, 1.0730526447296143, -0.0260500255972147, -0.1604684591293335, -1.0823928117752075, -0.12076364457607269, 0.37675637006759644, -0.48928195238113403, -1.0585565567016602, 0.37332990765571594, 0.08561621606349945, -2.645871877670288, 0.06761325895786285, -0.4935935139656067, -2.1641459465026855, 1.4689100980758667, 0.1135122999548912, 0.07423345744609833, -0.2970794439315796, -1.0274111032485962, -0.7766358256340027, 0.4928825795650482, 0.30882716178894043, 0.04830975458025932, -1.6405388116836548, -1.1724804639816284, -0.276105135679245, 0.2632986903190613, 0.04432826489210129, 0.6014180183410645, 0.7266245484352112, -1.3361351490020752, 1.397440791130066, 1.2810006141662598, 1.8039560317993164, -1.2821872234344482, 0.4422261714935303, 1.0558931827545166, -0.04148443043231964, -1.2155457735061646, -0.2667519450187683, -0.7882822751998901, -1.051740288734436, 0.22813887894153595, -0.6317481994628906, 1.1332505941390991, -1.0074487924575806, 0.39022696018218994, -1.004680871963501, -0.9572336077690125, -0.6559751629829407, 0.13104629516601562, 1.035063624382019, -1.009261965751648, 0.5344569087028503, -1.459691047668457, 0.7197704315185547, -0.38991159200668335, -2.806032180786133, 0.3912116289138794, 0.26056888699531555, 0.3939124643802643, -1.9305896759033203, -0.2530660331249237, 0.07236209511756897, -0.4787293076515198, -1.2083138227462769, 0.5959503650665283, -0.7445878982543945, -0.9949352741241455, -0.21407818794250488, -0.165130153298378, 0.865620493888855, -0.34451088309288025, 0.37441542744636536, -1.0267449617385864, 1.831005334854126, -0.4564889371395111, 1.5667725801467896, 0.472685843706131, -1.335962176322937, -0.5637922286987305, -0.32287028431892395, 0.37738803029060364, 0.7641656398773193, 1.236629605293274, 0.5403633713722229, -1.127810001373291, 0.9705525636672974, 1.4558099508285522, -0.06309892982244492, 0.07126779854297638, 0.7447640299797058, 0.9338353872299194, 1.1537259817123413, 0.1143755316734314, -0.1011214479804039, -0.8621442914009094, 0.8079379796981812, -1.1739577054977417, 0.2548825442790985, -1.559625267982483, 0.5032014846801758, -0.857191801071167, 0.6914222240447998, -0.33837443590164185, -1.2229760885238647, -0.8152194023132324, 1.9756301641464233], [1.3992805480957031, 0.6229449510574341, 1.632491946220398, 1.0423213243484497, -0.5823388695716858, 1.6089812517166138, 0.5186023116111755, 0.8582445383071899, 0.649750292301178, 0.4392789304256439, -0.41271543502807617, 0.07979580014944077, 0.029524078592658043, -0.46316054463386536, 1.12039315700531, -1.7610797882080078, 0.6502636075019836, 0.17124047875404358, 0.4228556454181671, -0.12790291011333466, -0.24374301731586456, -0.05046846717596054, -0.2197517454624176, -0.28057000041007996, -0.9467651844024658, -0.749165415763855, 0.4554913341999054, 0.40120401978492737, 0.36273717880249023, 0.6562930941581726, 0.05257213115692139, 0.009517250582575798, -0.08562281727790833, -0.9841321110725403, -0.24390855431556702, -1.5950697660446167, -0.31991514563560486, -0.23423704504966736, -0.039503760635852814, 0.5219286680221558, 0.16367687284946442, -0.8474375605583191, 0.3330216109752655, -0.9234063625335693, -0.016552802175283432, 0.3482285439968109, -0.43404439091682434, 0.281545490026474, -1.1894731521606445, 0.06630872189998627, 1.1002296209335327, 0.9459354877471924, 0.9325759410858154, 0.7043918371200562, 0.8108760714530945, 0.11694319546222687, 0.06422263383865356, -1.161306619644165, 0.2279527187347412, 0.4460417330265045, -1.1616679430007935, -0.5142508745193481, 1.7923264503479004, 0.6437850594520569, 0.48613736033439636, -1.6885939836502075, 0.0685446709394455, -1.4550236463546753, 1.0178552865982056, -1.0129315853118896, -0.9934418201446533, 0.7189929485321045, -0.44737717509269714, -0.02835148386657238, -2.282270908355713, -2.7244699001312256, -0.18348771333694458, 0.786554753780365, -0.7308153510093689, -1.7590268850326538, 2.107137441635132, 2.411567211151123, 2.4050190448760986, -0.6772233247756958, -1.2221428155899048, -1.149737000465393, 0.2922227680683136, -1.6277296543121338, 0.6145793795585632, 0.07375498861074448, 0.1840948760509491, -0.395938515663147, 1.2051448822021484, 0.6407964825630188, 0.20528040826320648, 0.040073566138744354, -1.1365916728973389, -0.8867552876472473, 1.1873486042022705, 1.9757122993469238, -0.4785861074924469, -0.9676566123962402, -1.4397891759872437, -1.1147606372833252, -0.3964230418205261, -0.4848118722438812, 0.0801636278629303, 1.312284231185913, -0.37938788533210754, 1.0396772623062134, -1.5222145318984985, 0.5264840722084045, -0.8199790716171265, -0.824341893196106, 0.4776679277420044, -0.06840654462575912, 0.1112980842590332, 0.8271358013153076, -1.2434548139572144, 0.6561494469642639, 1.1079187393188477, -0.5967115163803101, 0.4164397418498993, -0.5871310830116272, 1.0204644203186035, -0.1476851999759674, -0.5824625492095947, 1.9589043855667114, -1.110575795173645], [-0.8765438199043274, 2.2128841876983643, 0.1642163246870041, -0.8037984371185303, -0.09110046178102493, 0.09384789317846298, -1.5246927738189697, 0.29951533675193787, -0.8440871834754944, 0.9145299196243286, 0.15105275809764862, 0.5929792523384094, -0.3560195565223694, -1.9897339344024658, -0.8384031653404236, 0.1882491558790207, 1.4308298826217651, 1.6077765226364136, -0.08436743170022964, -1.711483120918274, -0.8835382461547852, 0.802088737487793, 1.8981235027313232, -0.610679566860199, 1.270424485206604, -0.5257503390312195, -0.17187733948230743, 1.100769281387329, -0.5108548998832703, -0.10755892097949982, -0.03307216987013817, -1.524398922920227, 0.06910941749811172, -1.5812561511993408, -0.08751112967729568, 0.19997435808181763, -0.7514593601226807, -1.087997555732727, -0.502951443195343, -0.7240197658538818, -3.261512279510498, 2.5471901893615723, -1.4136357307434082, -0.950434148311615, -0.27675676345825195, 1.7726919651031494, -0.9667665362358093, 0.8902077674865723, 0.04522720351815224, -0.10419289022684097, 0.16950856149196625, 1.7471249103546143, -0.37405309081077576, -0.5828953981399536, 1.0615594387054443, -2.370305299758911, -1.339481234550476, -1.744372010231018, -1.6344650983810425, -1.5035982131958008, 1.0318286418914795, -0.09383820742368698, -0.5259334444999695, -1.496543049812317, 0.3634357452392578, -0.3513498604297638, -1.4251850843429565, 1.0784791707992554, 2.226966619491577, 0.3053514361381531, -0.07379181683063507, 0.862517237663269, -0.8064918518066406, 1.0335837602615356, -0.5401026606559753, 0.35676848888397217, 0.36696872115135193, 1.3375877141952515, 0.763939380645752, -1.167245864868164, -0.8451130390167236, -1.1112720966339111, -0.6373023986816406, 0.31203803420066833, 0.06640087813138962, -1.8099164962768555, 1.1803123950958252, -0.46539559960365295, 0.3880675435066223, -0.7106641530990601, 0.10748379677534103, 0.7090800404548645, -1.3958656787872314, -0.11039223521947861, 1.6218576431274414, 0.7571068406105042, 0.6322669386863708, 2.035017967224121, -0.46735453605651855, -1.934605360031128, 1.6412969827651978, 1.3255499601364136, 1.0180963277816772, 0.743608832359314, 0.5699353218078613, -0.057472046464681625, 1.071890115737915, -1.0656250715255737, 0.34154659509658813, -0.01997121050953865, 1.451022982597351, 0.30687186121940613, 0.14861717820167542, 1.2277899980545044, 0.3858248293399811, -1.6987107992172241, -0.2800350785255432, -0.17147324979305267, 0.3912808299064636, 0.4868711829185486, -0.38710659742355347, 0.8427606821060181, -0.40442028641700745, 0.14602743089199066, -1.6011897325515747, 1.0897995233535767, 0.3527570366859436, -1.0575721263885498, 0.36343297362327576], [1.9849480390548706, -0.2808251678943634, -0.016257451847195625, 2.206122875213623, -1.099212884902954, -1.6932240724563599, -0.20497025549411774, -0.21643152832984924, -0.7111752033233643, -1.3955862522125244, -0.06500346958637238, 1.2447726726531982, 0.3842363953590393, 0.32126080989837646, -2.1781327724456787, 0.668662428855896, 0.7727250456809998, -0.23963330686092377, -1.117933988571167, -0.5710429549217224, 0.08926869928836823, -0.5124028921127319, 1.8917735815048218, -0.5103740096092224, -0.7288366556167603, -0.3594394624233246, 1.8985949754714966, -0.12771211564540863, -0.3649313449859619, 0.8796950578689575, 0.6738083958625793, -1.640496850013733, -1.7726019620895386, -1.1423792839050293, -0.16480372846126556, -1.115922451019287, 1.2017548084259033, -0.4589148163795471, -1.6388715505599976, 0.924278974533081, 1.2930017709732056, 0.4309689402580261, 0.7942893505096436, -0.23637911677360535, -0.11666113138198853, -0.09550231695175171, -0.21960099041461945, 1.9082753658294678, -0.18475383520126343, -1.016883373260498, 0.5199931263923645, 1.2476274967193604, 0.8991180062294006, 0.29204052686691284, -0.2076963633298874, -1.0640511512756348, -0.453755646944046, -1.3963500261306763, -2.2503604888916016, -1.1938416957855225, 0.11312537640333176, 1.0400906801223755, -0.8145962357521057, -0.26440685987472534, -0.8189078569412231, 0.25995585322380066, 1.7589658498764038, -1.430155634880066, -0.09929509460926056, -2.7588720321655273, -0.2204163670539856, -1.1994764804840088, 0.5491172075271606, 1.3409050703048706, 0.7075427770614624, -0.563225269317627, -1.0676023960113525, -0.6909908056259155, -0.7676054835319519, 0.07873327285051346, 0.9409297108650208, -0.8879203200340271, 1.242087721824646, 0.018605830147862434, -0.13390959799289703, 1.740293025970459, -0.7234109044075012, -0.35831567645072937, -0.12372437119483948, -0.8497238159179688, 0.7370400428771973, 1.3134856224060059, 0.25273510813713074, -1.7183196544647217, -0.762569010257721, 1.0651216506958008, -0.7869488000869751, 0.49413222074508667, 0.21670754253864288, 0.6613420844078064, 0.8236550688743591, -0.11445353925228119, -0.6394048929214478, -0.17924365401268005, -0.28477680683135986, -0.15151487290859222, 1.00810706615448, -0.07567143440246582, -0.21360966563224792, 1.4652268886566162, -0.3110286593437195, 0.2183840274810791, -0.28408482670783997, -0.0041009970009326935, -0.46335476636886597, -1.1978448629379272, 1.0250362157821655, -0.8791532516479492, 2.116239309310913, -1.3271528482437134, 0.3336215913295746, -0.2860696315765381, 0.0456826388835907, -0.33717814087867737, -0.6078808307647705, -0.16692370176315308, -0.14862030744552612, -0.6198538541793823, -0.5422083139419556], [0.4364057779312134, -0.8979191780090332, 0.8110480308532715, -0.4293690025806427, 2.0774974822998047, -1.5238189697265625, 0.0907006710767746, -0.525728166103363, 0.17934592068195343, -1.3896145820617676, -0.36749640107154846, -2.097538471221924, -1.1626858711242676, -0.13064251840114594, -0.08261427283287048, 0.23920470476150513, -0.5651640892028809, -1.7426270246505737, -1.3589377403259277, -0.01151223760098219, -0.7887471318244934, 0.4879225194454193, -0.8103658556938171, -1.6701196432113647, 1.2201353311538696, -0.255096971988678, -1.8553751707077026, 0.29149413108825684, -0.6845736503601074, 0.804050087928772, 1.4630872011184692, 0.2789592146873474, 1.663506031036377, -0.7325499057769775, 0.4607129395008087, -0.3262890577316284, -1.379510760307312, 0.2556193172931671, -0.20920418202877045, -0.4811299741268158, -0.423489511013031, -0.04120989143848419, 1.0206066370010376, 1.9505130052566528, 0.3228578269481659, -0.3075638711452484, -0.6658031940460205, -0.7734617590904236, 0.5561947226524353, -0.7461490631103516, 0.14808933436870575, 0.23829899728298187, 0.5347703099250793, -0.9757578372955322, 1.258992075920105, -1.650536060333252, 0.005505816079676151, 1.5130996704101562, 0.5692216753959656, -1.0770878791809082, -1.7403254508972168, -1.07699453830719, -1.9614851474761963, -0.19950038194656372, 0.737300455570221, -2.0408036708831787, -0.1700885146856308, 0.15308651328086853, 0.563297688961029, -0.21244053542613983, -0.7242992520332336, 0.8528891205787659, 0.3754500150680542, 0.8848593235015869, 0.1654892861843109, 0.42700812220573425, 0.467185914516449, 0.4340423047542572, -0.0011077274102717638, -0.09960807859897614, 0.7864234447479248, 0.8574482202529907, 0.7648794054985046, -0.6006341576576233, -1.11058509349823, 0.40358659625053406, -0.019056245684623718, 1.1559702157974243, -1.269303798675537, -1.0337995290756226, 0.6663627028465271, -0.7075039744377136, 0.24483083188533783, -0.33506056666374207, 0.5529821515083313, 1.949267864227295, 0.8559798002243042, 0.5850433707237244, 0.2661708891391754, 1.367812156677246, -1.6867363452911377, 1.8206101655960083, 0.1746579110622406, -0.7213224172592163, -0.6840410828590393, 1.2848180532455444, -0.6102270483970642, 0.4190947115421295, 0.05665072053670883, -0.3183067739009857, -0.5565807223320007, 0.9347445368766785, 0.4636242687702179, 1.1538437604904175, 0.7890346050262451, 0.6941588521003723, 1.0512847900390625, 0.1135127916932106, 0.21180270612239838, 0.44130897521972656, -0.36769649386405945, -0.736815869808197, 0.3685845136642456, 0.6965931057929993, -0.2399856448173523, -0.9835041761398315, -1.1493446826934814, -0.06956994533538818, 1.5608350038528442], [0.0330079011619091, -0.21381384134292603, 1.0872883796691895, 1.0602192878723145, -1.1038355827331543, -0.3068024218082428, -1.8658254146575928, 1.0368084907531738, 0.024173220619559288, -0.0920562893152237, -0.7866122126579285, 0.527959406375885, -1.099702000617981, 1.1464344263076782, -0.2836341857910156, 0.24680717289447784, 0.4175334870815277, 0.034653689712285995, 1.2728267908096313, -1.5977661609649658, -0.40130603313446045, -0.0017692382680252194, -0.7431894540786743, 2.0791752338409424, 0.8616840243339539, 0.275173544883728, -0.12984208762645721, -0.4743358790874481, -0.2088214010000229, 0.7867017388343811, 0.2951531410217285, 1.0774086713790894, 0.19680051505565643, -0.9679864645004272, -0.9743492603302002, -0.19587863981723785, -0.6486045122146606, -1.1370422840118408, -0.5373296737670898, 0.3616694509983063, 0.018709754571318626, 0.8421655893325806, 0.1546771228313446, -1.6289409399032593, -1.1837445497512817, -1.4974044561386108, 1.6014622449874878, 1.8960634469985962, 0.11743468791246414, 1.1843445301055908, 0.6962272524833679, -0.30017152428627014, -0.2567291259765625, -1.5953974723815918, -1.2639087438583374, 0.41754239797592163, 1.5353349447250366, 1.242725133895874, 1.7481290102005005, 0.6532617807388306, 0.6056429743766785, -0.6868848204612732, 2.939152956008911, -1.329022765159607, 1.851111650466919, -0.5535979866981506, 0.10999801754951477, 0.2665935754776001, 1.340088963508606, 1.8620469570159912, -0.3812233507633209, 1.7583487033843994, 0.11867930740118027, 1.1813991069793701, -0.2486140877008438, -0.5597035884857178, 0.20291969180107117, -0.650143563747406, -0.18531666696071625, -0.2691713273525238, 1.1289260387420654, 1.5316110849380493, 2.43424916267395, -0.3562929630279541, 0.7688316702842712, -0.12438850849866867, -0.7006739974021912, 1.4509934186935425, 1.719767689704895, -0.6055566668510437, -1.94223153591156, 0.539338231086731, -0.2429773360490799, -1.5503864288330078, 1.345226764678955, 1.061421275138855, 0.6328732967376709, -0.3072608411312103, 1.2411696910858154, 0.37022435665130615, -0.08614794164896011, 0.20033130049705505, -1.079880952835083, -2.043379068374634, -0.09091854840517044, -1.2223433256149292, 0.30764949321746826, -1.1834100484848022, -0.617135226726532, 0.9202564358711243, 1.0989655256271362, -0.7191756963729858, 0.004148468375205994, 1.1372287273406982, -0.6072561740875244, -1.638248324394226, -1.3482820987701416, -0.2883426249027252, 0.210269957780838, -2.07954478263855, -1.2578425407409668, -0.2333277016878128, -0.38750484585762024, -1.1237969398498535, 0.5469672083854675, 0.05545276030898094, -0.5610928535461426, -0.4669853448867798, -0.881939172744751], [-1.3742847442626953, -1.5645673274993896, -0.08745279908180237, -1.3572359085083008, 0.555765688419342, -0.8214238286018372, -0.47755733132362366, 1.3067799806594849, -1.0587388277053833, 2.5548105239868164, 0.9547390341758728, 0.42826157808303833, 2.173956871032715, 1.3596067428588867, -0.15974660217761993, 0.420687735080719, 1.6583025455474854, 1.1356874704360962, 1.447757601737976, -0.10036160796880722, -0.01573166809976101, -0.4116213619709015, 0.33004507422447205, -0.6211411356925964, 0.7352766394615173, -1.8774973154067993, 0.4655182957649231, -1.4294828176498413, -0.521926760673523, 1.644675850868225, -0.2958020269870758, -0.6243617534637451, 0.5918583273887634, 0.44862890243530273, -1.9655369520187378, 1.3456319570541382, -1.0785877704620361, 1.5197211503982544, 0.6447204351425171, 1.5077606439590454, -1.7410101890563965, -0.7180660963058472, -0.4348137378692627, 0.5288383960723877, -1.2391949892044067, 0.556474506855011, -1.6334421634674072, 0.9577645063400269, 0.16799363493919373, -0.8453192710876465, -0.23861272633075714, -0.48795458674430847, -0.2596289813518524, 0.36469098925590515, 1.3899726867675781, -0.7503630518913269, -0.024459583684802055, -0.6499069333076477, 1.7168012857437134, -0.2587895691394806, 0.9869454503059387, 0.7209150195121765, -0.25545424222946167, 1.4194645881652832, 0.19115252792835236, 0.3474802076816559, 1.2621740102767944, -0.43938666582107544, 0.9605665802955627, -0.6788421273231506, -0.768337070941925, -0.5075910091400146, -0.17972899973392487, -2.3431739807128906, 0.11510356515645981, -1.7874315977096558, -0.8429183959960938, 0.44330355525016785, -0.9591201543807983, 0.20299944281578064, -0.9009183645248413, -0.2922556698322296, 2.977627992630005, 0.8016330599784851, -0.2266068011522293, 0.24190588295459747, -0.31450915336608887, 0.9425033926963806, -0.8787060379981995, 0.24747957289218903, 0.6402745842933655, -1.9134702682495117, -0.9898307919502258, -0.28441399335861206, 0.10660423338413239, -0.22335846722126007, 0.3257666528224945, -2.6527929306030273, -1.4522053003311157, -0.23825986683368683, -0.6270157098770142, -1.5537853240966797, 0.9924024939537048, -0.9323925971984863, -0.325960636138916, 0.23516559600830078, 0.7933382987976074, -0.8178461194038391, 0.16219480335712433, -0.25059181451797485, 0.8131282925605774, -0.5078625082969666, -0.05675845965743065, 0.06702222675085068, -0.9674797058105469, -0.8005834221839905, -0.02024981379508972, -0.6851420998573303, 0.21723350882530212, 0.5074716806411743, 0.2517034411430359, 1.5454965829849243, 0.03654487058520317, -0.6700214147567749, 0.6261617541313171, 0.30450770258903503, 0.3290388286113739, 1.9340813159942627, -0.739733874797821], [0.0413210391998291, 1.9469964504241943, 0.809980034828186, -0.006044507026672363, 0.360106498003006, -0.034119874238967896, 0.8642788529396057, 0.6451984643936157, 0.5173304677009583, 1.938220500946045, -0.7893436551094055, -0.9063649773597717, 1.1700880527496338, -0.9322591423988342, 2.255592107772827, 0.6314946413040161, -0.6831161379814148, -1.275213360786438, 0.4416591227054596, -1.9950828552246094, 0.2019120305776596, 0.02574821375310421, -1.4892630577087402, 1.8460220098495483, -1.2438874244689941, -1.1842951774597168, 0.7815181016921997, -2.2541654109954834, -0.5851117372512817, 1.484958529472351, -1.2517845630645752, 0.7583329081535339, -0.03950824588537216, -1.0467301607131958, 0.5306835770606995, -0.4006056785583496, 0.9852650761604309, 0.6890143752098083, 0.46397221088409424, -0.16882570087909698, -0.20063316822052002, -0.2039247453212738, 0.5947088003158569, 1.1217139959335327, -0.456257700920105, 1.3380036354064941, 0.09965696185827255, -0.8052400350570679, 1.5052229166030884, -0.07553786784410477, -0.7050437331199646, -0.4906785786151886, -0.7075461745262146, -0.2161853015422821, -0.1699400544166565, -0.41509902477264404, 1.2302526235580444, 0.19222953915596008, -0.3403007388114929, -1.064367413520813, 0.025795046240091324, 1.4527781009674072, 1.3936470746994019, 1.2682114839553833, -0.9436364769935608, 0.5307629704475403, 1.4183379411697388, -0.21114163100719452, -1.783306360244751, -0.06147053465247154, 1.1322705745697021, -0.5379230976104736, 1.2393656969070435, -0.23873133957386017, -1.8018041849136353, 2.727783203125, -0.5358002781867981, 1.1421253681182861, -0.10539495199918747, -0.5946022272109985, 0.19139699637889862, -1.5390472412109375, 1.157442569732666, 0.674968421459198, 0.5986977815628052, 0.35412925481796265, 1.935770034790039, 0.04930730164051056, -1.4940530061721802, 0.3131868243217468, 0.8277607560157776, 1.2871068716049194, 1.1527905464172363, 0.4539036750793457, 0.008294223807752132, -2.793815851211548, -0.846442461013794, -2.023794412612915, -1.044400691986084, 0.2608088254928589, 0.6551499962806702, -0.9544633626937866, -0.5631637573242188, 0.21306487917900085, 1.17099928855896, 1.6207572221755981, -1.954399585723877, -0.5905295014381409, 1.216687798500061, -0.9033592939376831, 0.8780823349952698, 0.927710235118866, -0.18066361546516418, -1.0597445964813232, -0.09474003314971924, -0.18507088720798492, -0.6171603202819824, 1.625217080116272, 0.11764799058437347, 0.7037678956985474, -1.0670504570007324, 0.04383807256817818, 0.3634259104728699, -0.006726414896547794, 0.6231494545936584, -0.5213704109191895, -1.7613554000854492, -0.5660709142684937, -0.7448472380638123], [-0.8960738778114319, 1.0907999277114868, -0.2923007309436798, -1.0276464223861694, 0.21552374958992004, 0.12837441265583038, 0.238462895154953, -0.2337213158607483, -0.11373517662286758, 0.5574203729629517, 0.9432721734046936, -0.2795902490615845, -2.019303798675537, 1.3248118162155151, 3.062880754470825, -1.2603113651275635, 0.09464553743600845, -1.0039007663726807, 1.7708359956741333, 0.9369183778762817, -0.026696665212512016, -0.4821833670139313, -0.8064259886741638, -0.3870317339897156, -0.2824755907058716, 0.8072075247764587, 0.05733592435717583, 0.4845869839191437, -1.2511926889419556, 0.35718753933906555, -2.228294610977173, -0.8505852818489075, 0.8204008340835571, 0.9021851420402527, -0.4469219148159027, 0.7509787678718567, -0.09018498659133911, 0.6507179737091064, 0.8586466312408447, -0.2957329750061035, 0.5428977608680725, -0.4234734773635864, 0.623555064201355, 0.6333680152893066, 0.37004369497299194, 0.16033025085926056, -0.866098165512085, 0.3040251135826111, -1.15716552734375, -0.6009218096733093, 0.42345717549324036, 0.07196418941020966, -1.8868380784988403, -0.09980712831020355, -0.3365875482559204, -1.730667233467102, 0.010114714503288269, -0.3552597165107727, -0.8699651956558228, 0.593888521194458, -0.4325545132160187, -0.4692472517490387, -0.052214402705430984, -1.2037432193756104, -0.17325730621814728, 1.2766883373260498, 0.10452860593795776, 0.8440985083580017, 0.6141772866249084, 0.00033616446307860315, 0.7660136222839355, 1.4662654399871826, 0.3844600021839142, -1.3704577684402466, 0.22254765033721924, 0.19416038691997528, -0.41833075881004333, -0.17836765944957733, -0.27172186970710754, -1.1188156604766846, 1.3453819751739502, 1.613586664199829, -1.2709543704986572, -0.8202222585678101, -0.15582868456840515, 0.44532328844070435, -0.10850562155246735, 0.5138915777206421, 1.561073660850525, 1.7275136709213257, -1.7201288938522339, -0.26860707998275757, 0.8952683806419373, -1.273002028465271, 1.7277101278305054, -0.41772544384002686, 0.3958572447299957, -0.34138861298561096, -0.9988234043121338, -0.7910776734352112, 0.9552733898162842, 1.8358310461044312, 0.5181751847267151, -0.97036212682724, -0.8740800023078918, 0.4810560345649719, 1.1142488718032837, 1.1924647092819214, -1.790151596069336, 1.0669163465499878, -0.378694623708725, 0.5691022872924805, 0.7821623086929321, 0.3925752341747284, -0.4764406681060791, 0.3915139138698578, -1.441880702972412, 1.0058716535568237, -0.5391089916229248, 0.5354101061820984, -0.8728896975517273, -2.3825247287750244, -1.1667561531066895, -0.49878498911857605, -0.6441053748130798, 1.1361404657363892, -0.4131428301334381, -1.3914356231689453, -1.9092682600021362], [-0.2027123123407364, -0.14419852197170258, -1.0368015766143799, -0.763051450252533, 0.6902305483818054, 0.05691343918442726, 2.122828483581543, -2.4922611713409424, 1.677647352218628, 0.07271666824817657, -0.47053390741348267, -0.06400445848703384, 1.829045295715332, 1.1185234785079956, 1.477252721786499, 0.06478060781955719, 1.4181792736053467, -0.519995927810669, 0.881632924079895, -1.65729820728302, 0.20260243117809296, 1.2583094835281372, 0.2987370491027832, 0.2969737648963928, -0.9306899309158325, 1.161842942237854, -1.3865830898284912, 0.895237147808075, 0.6053182482719421, -2.3282458782196045, 1.1485298871994019, -0.0786653384566307, 0.5205554962158203, -1.4902909994125366, 0.300523042678833, 0.03092404641211033, 2.3137335777282715, -0.9954626560211182, -0.1568744033575058, 0.45277947187423706, 1.0091896057128906, -1.0109559297561646, -0.020297763869166374, -0.22575634717941284, 0.44228675961494446, -1.6188584566116333, 0.1780768483877182, -1.388601541519165, -0.8201141953468323, 0.9110094904899597, -1.3768889904022217, 1.885668396949768, 0.009632698260247707, 1.9295440912246704, -0.4558519423007965, 1.6533565521240234, 0.6650370955467224, -0.8795620799064636, -0.7666576504707336, -0.0828472152352333, 0.23568035662174225, 0.42990827560424805, 1.5703657865524292, 1.4416630268096924, -0.4832974374294281, 0.4988875091075897, -1.465491533279419, 1.539942979812622, 1.9390678405761719, 0.7599905133247375, 1.0592870712280273, -0.7464220523834229, -1.091894268989563, -0.5105042457580566, -1.1722687482833862, 2.668407678604126, -1.8010116815567017, -1.5783149003982544, -0.47847557067871094, -0.1760246455669403, 0.7999860048294067, 1.4143569469451904, 0.046334005892276764, 2.610877513885498, -0.5204406380653381, 1.498595952987671, 0.4379778206348419, -0.19586533308029175, -0.21443024277687073, -1.084100365638733, -0.01990439184010029, -0.5469327569007874, -0.6757532358169556, -0.4854457378387451, 1.1420572996139526, 1.7068567276000977, -0.10558434575796127, -0.20247246325016022, 0.5239700078964233, -1.7712008953094482, -2.3904740810394287, -0.10085234045982361, -1.3031225204467773, 0.21247248351573944, 1.192883849143982, -0.05969920754432678, 0.40763694047927856, 0.08888992667198181, -1.6677087545394897, 0.6582381129264832, 1.0536195039749146, -0.7280367612838745, -0.1404198855161667, 0.025418806821107864, -1.9261125326156616, 0.5767630338668823, -0.9491085410118103, 0.22449460625648499, 0.049378033727407455, 0.6913612484931946, 0.6974411606788635, 1.9906291961669922, -1.626286268234253, 0.33094295859336853, 0.5513128638267517, 1.6019995212554932, -0.627056360244751, -0.15766625106334686, -0.45909178256988525], [0.1984293907880783, 0.1549004763364792, 1.9956843852996826, -0.12552788853645325, 1.215218424797058, 1.6472690105438232, 0.2041511982679367, 1.129028558731079, -3.4485905170440674, 0.5763192176818848, 1.4066672325134277, -0.8299065232276917, -2.485884428024292, -1.1519663333892822, 0.9253281950950623, 0.14462827146053314, 3.702279806137085, -0.5175001621246338, -0.6090814471244812, 0.8752223253250122, 0.5958160758018494, 0.6233388185501099, 0.6776280403137207, 0.3342529237270355, -0.07881312817335129, -0.4903281331062317, 0.8622307181358337, -1.1977570056915283, -0.1156623512506485, 0.8296772241592407, 0.5281811356544495, 0.6745638847351074, -1.921967625617981, 1.1534701585769653, -0.46592527627944946, -0.5067375898361206, -1.7503691911697388, -0.37729981541633606, -0.3579745292663574, -0.8739520311355591, 0.4546149671077728, 1.742077112197876, 1.3495067358016968, 0.71532142162323, -0.7612062692642212, -1.3828760385513306, 0.2555581033229828, -0.5702671408653259, 0.23647595942020416, 0.0435054711997509, -2.736464738845825, -2.2996504306793213, 0.08159495145082474, -0.7359405159950256, 0.46384042501449585, 2.017526388168335, 0.24856901168823242, 1.144574761390686, 0.4907006621360779, -0.4856562912464142, -0.5159794092178345, -0.14727361500263214, 0.6194418668746948, -0.21242213249206543, 0.7954441905021667, 0.7062864303588867, -0.20342865586280823, -0.5250508189201355, -0.5590019822120667, 1.1383898258209229, -0.6261308789253235, -0.5185772776603699, 0.7898188233375549, -0.0621301531791687, -0.010752329602837563, -0.41422054171562195, -1.3244833946228027, 1.209921956062317, -0.388405978679657, -0.3977890908718109, 0.05402795970439911, 0.34484511613845825, 0.7721619606018066, -0.42865586280822754, -1.1979719400405884, -1.9908196926116943, 0.35678964853286743, -1.0668706893920898, -0.05123334750533104, 0.02048284374177456, -0.9603942632675171, 0.6643067002296448, -0.9968107342720032, 0.2958323657512665, -1.0387599468231201, -1.865762710571289, 1.3256933689117432, 0.3993625044822693, 0.5430372953414917, 0.6105243563652039, 0.33051273226737976, 0.9436781406402588, 0.6292473077774048, -0.4760315716266632, -0.16441667079925537, 0.36390218138694763, 1.7570388317108154, -0.5028406381607056, -1.1594288349151611, 1.283042550086975, 0.7048110961914062, -1.590829610824585, 1.1516742706298828, 0.015475085936486721, 1.221867322921753, -1.3582184314727783, 0.1183931976556778, -0.15844087302684784, -0.11608168482780457, -0.8028427958488464, 1.2681701183319092, 0.07887446880340576, 0.9297181367874146, -0.5320186614990234, 1.0369101762771606, 0.30554187297821045, -0.041787534952163696, 0.8633369207382202, 0.3968450427055359], [0.06561753898859024, -0.6037641763687134, 1.8103301525115967, -0.8895895481109619, -0.30794984102249146, 1.0860850811004639, 1.6131235361099243, -0.30538636445999146, 1.5425019264221191, 0.6643378734588623, 0.5653388500213623, 0.27698537707328796, 0.32892119884490967, -0.16923001408576965, 2.4475865364074707, 0.6110290288925171, 0.7378683686256409, 0.5298345685005188, 0.08088698983192444, 0.06871602684259415, 0.05617709830403328, 1.4210394620895386, 1.2534358501434326, 1.6189507246017456, -0.5197475552558899, -0.47387370467185974, -1.1512782573699951, -0.8440445065498352, 0.5825757384300232, -0.8971740007400513, 0.7764158248901367, 0.4224807918071747, -0.10660409182310104, 0.11783790588378906, 1.282021403312683, 0.763007402420044, -0.45830726623535156, -0.283861368894577, 1.7443208694458008, 0.05291499197483063, -3.128330707550049, 0.5479954481124878, 0.06104692071676254, -0.468508243560791, -0.29984334111213684, 0.038534015417099, -0.16688759624958038, 0.9653242230415344, 1.5945197343826294, -0.016439490020275116, 1.8035187721252441, 0.7729138135910034, -0.01090335939079523, -0.9763965606689453, 0.3061522841453552, 0.8024461269378662, -1.3533567190170288, -0.2927781045436859, -0.3718106150627136, 0.2572551667690277, 0.8678738474845886, 0.1668708473443985, 0.600112795829773, -0.8473764061927795, 1.1701254844665527, 1.1468532085418701, 0.7336176633834839, 0.5054314136505127, 0.9686489701271057, 0.5932558178901672, -0.015836142003536224, -1.9600369930267334, 0.36952587962150574, 1.1441134214401245, 0.7054994106292725, 1.1487574577331543, -0.03867533430457115, 0.10403662919998169, 0.8569316267967224, -0.604304313659668, -1.0577210187911987, -0.23832444846630096, -0.1409279704093933, -1.3566944599151611, 0.7474720478057861, -1.6979299783706665, -0.24372127652168274, 2.0330495834350586, 0.35881972312927246, 0.1502886563539505, -0.8688780665397644, 0.345418781042099, 0.1533524990081787, -0.38578200340270996, -0.5867359042167664, -0.07214013487100601, 0.3466101884841919, 0.529634416103363, 1.7564318180084229, -2.0206446647644043, -0.3889148235321045, -0.4658837914466858, -0.7631080150604248, 0.7682743668556213, -1.1664263010025024, 0.20065277814865112, 1.4755117893218994, 0.1165875643491745, -0.2924192249774933, -0.49283814430236816, 1.0267682075500488, 0.8617081642150879, 0.18118463456630707, 0.8820959329605103, 0.43067654967308044, 0.9261965155601501, -1.1527280807495117, -0.09049683064222336, -0.19222193956375122, 0.044007740914821625, -0.8887830376625061, 0.6053948402404785, 1.229783296585083, 0.10749413073062897, -0.04496180638670921, 0.9509339928627014, 1.7153598070144653, 0.1391661912202835, 2.166668176651001], [1.1192071437835693, 1.8562322854995728, 0.3301798403263092, 1.2174768447875977, -1.2146580219268799, 0.2761388421058655, -0.3273065686225891, -0.8296968340873718, 1.1934082508087158, -1.585555911064148, 0.345146507024765, 1.4629918336868286, -0.742188572883606, -0.3630412220954895, -1.3642677068710327, -1.0692245960235596, 0.9915178418159485, 0.31016695499420166, 0.4327447712421417, 1.4845842123031616, -0.5015279650688171, 1.2440177202224731, -0.9202597141265869, -0.591188371181488, 0.3854663670063019, 1.8587160110473633, -0.3897927701473236, -2.219660997390747, 0.5993307828903198, -1.2725247144699097, -1.6089426279067993, 0.7603133320808411, -1.0777884721755981, -0.9071203470230103, -1.4224722385406494, -1.8471049070358276, 0.9507737755775452, 1.5702139139175415, -0.2712908387184143, -1.1521245241165161, -0.2824336886405945, 0.4141872823238373, 0.7084311842918396, 0.5606557130813599, -2.0971035957336426, 0.7234392762184143, 0.7737531065940857, -0.04454828426241875, 1.8003885746002197, -0.18151269853115082, -1.043986201286316, -1.4898773431777954, 1.9871184825897217, 0.3394295871257782, 1.2699017524719238, 0.6697801947593689, -1.1796835660934448, 0.67560213804245, 0.8959388732910156, -0.5106457471847534, -1.2551671266555786, 0.29079243540763855, -0.7885336875915527, 2.172729253768921, 0.11136791855096817, 0.4308505952358246, 0.3989778757095337, 1.0341747999191284, -0.27254194021224976, -1.4161826372146606, -0.6604974269866943, -0.96872478723526, 0.32988065481185913, -1.7791578769683838, 1.8199750185012817, -1.2018879652023315, -0.8284968137741089, 0.0007513808086514473, 1.4777779579162598, -0.5505337119102478, 0.8988184928894043, 0.45376428961753845, -1.188786268234253, 0.9119458198547363, 0.34066271781921387, -1.5029109716415405, -1.7792062759399414, 0.44178181886672974, 1.4669100046157837, 0.787278413772583, 1.175854206085205, 0.703511655330658, 1.267329216003418, -0.13923022150993347, 0.5736967921257019, 0.3382679522037506, 1.3343664407730103, -1.5974268913269043, 2.380007266998291, 1.9480040073394775, 1.0737329721450806, 0.7924009561538696, -1.1713002920150757, -1.4136213064193726, -0.026480412110686302, -0.19439099729061127, -1.1136471033096313, -0.6804431080818176, -0.3165810704231262, 0.5846952795982361, -0.2858889400959015, -1.299121618270874, -0.6298992037773132, -1.0845614671707153, 0.7796585559844971, -0.019692707806825638, -1.7545831203460693, 0.667304515838623, -0.6449801325798035, -1.1560478210449219, 1.3671733140945435, 0.8991531729698181, -1.0809097290039062, 1.2025816440582275, 0.217873215675354, 1.0275696516036987, 0.01796005293726921, 0.13045187294483185, -0.6183714866638184], [-0.7026145458221436, -0.05003008618950844, 0.9411957263946533, 0.20330949127674103, -0.38342517614364624, 0.1395784169435501, -1.0285276174545288, 1.5381779670715332, 1.220151662826538, 0.5241658687591553, -1.9796727895736694, -0.4484589099884033, 1.0821354389190674, 0.38494768738746643, 0.17108313739299774, 0.6993392109870911, 1.1405880451202393, 0.1521415412425995, -0.011908741667866707, 1.437977910041809, -1.5510555505752563, -0.6794459819793701, 1.483164668083191, 0.1825917363166809, 0.7931992411613464, -0.7207571268081665, 0.2876838743686676, 0.6639134287834167, 0.22171421349048615, -0.8680016994476318, 1.336599349975586, 0.013329622335731983, -1.6743078231811523, -0.2890287935733795, -1.055922508239746, -0.2206496000289917, -0.6352924108505249, 0.375460684299469, 0.59095698595047, 0.1427881419658661, -0.6499626636505127, -0.46677783131599426, -0.5434327721595764, 1.7076799869537354, -0.5398305058479309, 1.8339804410934448, 1.2877873182296753, -0.3260937035083771, 0.12938936054706573, 1.140445590019226, 0.17985838651657104, -1.6727406978607178, 0.05884762480854988, 0.07656712830066681, -0.538431704044342, 0.5030277371406555, -0.9759247899055481, -2.031475067138672, -0.4286240041255951, -0.6011895537376404, -0.09406495094299316, 0.9353514909744263, 0.6739935874938965, 1.2665332555770874, 0.6351193785667419, -0.23598437011241913, 0.8111984729766846, -0.4028806984424591, 1.0862983465194702, -0.016115667298436165, -0.9604695439338684, -0.1649993509054184, 0.42395076155662537, -0.7874040007591248, -0.236282616853714, 0.562579870223999, -0.2256648987531662, -0.8466908931732178, -0.6568888425827026, -0.4486062228679657, 1.0130385160446167, 1.5137288570404053, 2.3274927139282227, 0.619608461856842, -0.1950593888759613, -0.5112766623497009, 0.6446004509925842, 1.2893407344818115, 0.23884287476539612, -0.912224292755127, -0.7551100254058838, -0.17249494791030884, 0.29331594705581665, -0.41522541642189026, -0.19643835723400116, 0.682364821434021, 0.2203059047460556, -0.9228357672691345, 1.612584114074707, -0.8348334431648254, -0.4178584814071655, -0.0334407314658165, -0.989055335521698, -0.20911164581775665, 0.4839954972267151, -1.4985885620117188, -0.8332302570343018, 0.0247016791254282, 0.27295437455177307, -0.4919906258583069, -1.4876714944839478, 1.5070589780807495, 1.212448000907898, -1.744749903678894, 0.0012322691036388278, -0.6699599623680115, 0.9126340746879578, 1.31204354763031, -0.07491162419319153, 0.33406171202659607, 0.06232381984591484, 0.09875033795833588, 0.6703261733055115, 0.7544894218444824, -0.9183627963066101, -0.45145416259765625, 1.6848098039627075, -0.4017769396305084, 0.5724976658821106], [-1.548973560333252, -1.0732463598251343, -0.44684314727783203, 0.11427275836467743, -0.13432978093624115, 0.026553679257631302, 0.44152936339378357, 0.3064197897911072, -2.6855618953704834, -1.1740952730178833, -0.39158883690834045, 0.8280351758003235, 2.0965001583099365, 0.7895823121070862, 0.4184858798980713, -1.1305146217346191, -1.7438968420028687, -1.2348142862319946, -1.3413628339767456, 0.46755754947662354, 0.17608997225761414, 0.6176097393035889, 0.1300307810306549, -0.42901375889778137, 0.4086173176765442, 1.1046615839004517, 1.2270565032958984, -0.04703541845083237, 0.14496351778507233, 0.25937414169311523, 1.1027448177337646, -0.629206120967865, 0.1066342294216156, 1.1699798107147217, 0.0029400561470538378, -1.04279625415802, -1.2614049911499023, -0.9272518754005432, -0.2642306685447693, -0.6058658957481384, 2.20633602142334, 0.7166600823402405, -0.9939978718757629, -2.0195839405059814, -0.8546050190925598, -0.9088307619094849, 0.9119923710823059, -0.6095136404037476, 0.08149104565382004, -0.6185617446899414, -1.5125703811645508, -0.5731469392776489, -1.1464990377426147, 0.5833463072776794, -1.2077279090881348, -0.2154826819896698, -0.004842674359679222, -0.2955382764339447, -0.3333019018173218, -0.7286094427108765, 0.9911257028579712, 0.15687058866024017, 0.2947304844856262, -1.1920750141143799, 1.1848584413528442, 1.2453609704971313, 0.08805173635482788, 0.22314466536045074, 0.060698118060827255, -0.4378095865249634, 0.555370569229126, -1.1287384033203125, -1.120441198348999, -1.7705296277999878, 0.17141146957874298, 0.3180100619792938, 1.0190998315811157, -1.4384379386901855, 1.1678556203842163, -1.0620839595794678, 0.16886954009532928, -1.0967599153518677, -0.8092912435531616, 0.3261857032775879, 0.1647905558347702, 0.8476320505142212, 0.1778344362974167, 0.2382289469242096, -0.49523913860321045, 0.30672529339790344, 0.04453924298286438, -0.16752511262893677, 0.31492879986763, 1.8510305881500244, -0.1815924197435379, -0.6982918977737427, 1.3068376779556274, -1.4219776391983032, -0.68758225440979, 0.6595155000686646, 0.1437535136938095, -0.8681192994117737, 0.8588051199913025, -0.7968368530273438, 0.9983273148536682, 2.0784413814544678, 0.6695806980133057, 0.12107393890619278, -0.7964418530464172, 0.21852564811706543, 1.776921272277832, 0.5713105201721191, -0.8694865703582764, 0.7062145471572876, 1.6359059810638428, 1.6438122987747192, 0.9129467606544495, 2.445505380630493, -0.1889180988073349, -0.23957304656505585, -0.3169091045856476, 0.31251242756843567, -0.49121659994125366, -0.4707762897014618, 1.0283845663070679, 0.623284101486206, 1.6348278522491455, 0.1551162302494049, 0.4366708993911743], [1.2440017461776733, 0.3205122947692871, -0.3328076899051666, -0.11569776386022568, -0.24408882856369019, -0.9544366598129272, -0.4066988527774811, -0.289829283952713, 0.532072126865387, -0.510908305644989, -1.0304673910140991, -1.2016104459762573, -1.0266971588134766, -0.7605013847351074, 1.0172187089920044, 1.1261850595474243, 2.1482901573181152, 0.7148706912994385, 1.7934941053390503, 0.2937479019165039, -0.3084503710269928, -0.04051623493432999, 1.9044328927993774, -0.5949967503547668, -1.70784330368042, -0.2091534435749054, -1.1179125308990479, 0.3595784902572632, -1.1525180339813232, -0.34745028614997864, -1.4859205484390259, -0.6615324020385742, -0.19017647206783295, -0.07459793239831924, 0.4239034354686737, -0.7735432982444763, -0.7878517508506775, 1.583644151687622, -0.8289079666137695, -1.1501765251159668, 0.9446475505828857, 0.5769569873809814, 1.7647321224212646, -0.6241741180419922, -2.5461387634277344, -0.939860463142395, -0.8396310210227966, -1.3807193040847778, -0.46335941553115845, 1.3808493614196777, -0.46445396542549133, 1.3305962085723877, -0.15626320242881775, 1.279983401298523, 0.12723876535892487, 1.6213079690933228, -0.26771464943885803, 0.8231337666511536, -0.19827905297279358, 1.6211997270584106, -0.2602604031562805, 0.18444672226905823, 1.8778883218765259, 0.5003128051757812, -1.2543398141860962, 0.006840437650680542, -0.34524086117744446, -0.43051213026046753, -0.16545099020004272, 0.9094389081001282, -1.3761274814605713, 0.28808125853538513, 0.05297734588384628, -0.804056704044342, -0.5967610478401184, -1.0142849683761597, 1.7314146757125854, 0.05626543238759041, 0.04631806164979935, -0.23201821744441986, -1.0764299631118774, 3.040484666824341, 0.7490741014480591, 1.0913622379302979, -0.2938339412212372, -0.424637109041214, -0.28658637404441833, -0.7084147930145264, -0.41852623224258423, -0.5255183577537537, -1.3265063762664795, -0.9079311490058899, -0.1536581963300705, 0.14144682884216309, -0.09361759573221207, -0.29669684171676636, 0.11066091060638428, -0.9944614768028259, 0.5412878394126892, 0.26726487278938293, 0.861064612865448, -1.2557662725448608, -0.11248593032360077, 1.269194483757019, 1.4887187480926514, -0.02717837691307068, 0.6350910067558289, -0.5239733457565308, 0.42931681871414185, -0.45715200901031494, -0.6511006355285645, 1.131372332572937, 0.6895400881767273, -0.9906436800956726, 0.7182843089103699, 1.2763768434524536, 0.32354727387428284, 0.5587649345397949, 0.16781318187713623, -1.4138561487197876, -0.10760538280010223, -0.5739492774009705, -1.292323112487793, 0.7314040660858154, 0.35204756259918213, 0.13947901129722595, 0.822981595993042, -0.4371473491191864, -0.28500398993492126], [1.734971284866333, 0.29989394545555115, 1.6625030040740967, 0.8886454105377197, 0.46361851692199707, -0.4957427382469177, 2.946107864379883, -0.8138421773910522, -1.6652454137802124, 0.4325088858604431, 0.8141151070594788, -1.1457606554031372, -0.5465163588523865, 0.3267122805118561, 1.4995203018188477, -0.8245185017585754, -1.0289933681488037, -0.03961769491434097, 1.1794756650924683, 1.0406310558319092, -1.6192326545715332, 0.2222517728805542, 1.221914291381836, -0.8401597142219543, 1.5395785570144653, -0.9854253530502319, 1.20464026927948, -0.6028361320495605, 0.8974968194961548, -0.15443681180477142, 0.34288474917411804, -0.5381121039390564, -0.2071298509836197, -0.6224806308746338, 1.2767906188964844, -0.19845427572727203, -0.5441837310791016, -0.5949870944023132, -0.43759316205978394, 0.22808556258678436, 0.2726636230945587, -1.2821037769317627, 0.1443198174238205, -0.21528878808021545, -0.36198970675468445, -0.9304578304290771, -1.4957339763641357, -1.7110729217529297, 2.15346360206604, 0.14327672123908997, 0.28430095314979553, -0.585536003112793, 0.11858689039945602, -1.3520829677581787, 0.057545293122529984, -0.0034596016630530357, 0.6277771592140198, 0.7885921597480774, -0.08343061059713364, 1.1628319025039673, -2.0213162899017334, 0.5790103673934937, 1.2995272874832153, -0.9968805909156799, 0.11020965129137039, 1.4108977317810059, 0.3770062327384949, 1.634700059890747, 1.3654464483261108, 0.5036125779151917, 0.8299593925476074, 1.5057111978530884, -0.6807042360305786, 1.8832650184631348, -0.5016606450080872, 1.5038470029830933, 0.4721975326538086, -0.8735922574996948, 0.1741408258676529, -0.4872303605079651, -0.6310272216796875, 1.712570309638977, 0.08648339658975601, 0.32180309295654297, 0.9050952792167664, -0.8669654726982117, -0.41937941312789917, 2.2972726821899414, -0.136490598320961, -0.016078351065516472, -1.6338130235671997, -0.9627976417541504, -1.189601182937622, 2.2547292709350586, -0.7880198955535889, 1.0756804943084717, 1.3171205520629883, -0.3085235059261322, 0.31656986474990845, 1.7101737260818481, 2.077305793762207, 0.23666812479496002, -1.0342615842819214, -0.8244054317474365, 0.9273933172225952, -0.40465056896209717, 0.961482048034668, -0.058433521538972855, -1.2615489959716797, -0.7149812579154968, -0.5502463579177856, -0.26830053329467773, -2.381570816040039, 2.6359238624572754, -0.2228628545999527, -2.3099594116210938, 0.9508782029151917, -1.3373421430587769, 0.7029511332511902, -0.038781341165304184, 0.14982551336288452, 0.3591310381889343, -0.3040432035923004, 1.3429348468780518, 2.0845589637756348, -0.9699397683143616, -0.4969286620616913, -0.2896606922149658, 1.3717446327209473], [-0.8933359384536743, -0.21239779889583588, 0.4743931293487549, 0.2812384068965912, -0.09693345427513123, -2.172832489013672, -1.4512380361557007, -0.3105376958847046, 2.027198314666748, -1.1601499319076538, 0.9401010870933533, -0.12754271924495697, -1.098965048789978, -0.24701543152332306, -0.6456617116928101, 0.09887784719467163, -0.8165191411972046, 0.19012568891048431, -0.12217232584953308, -0.563396155834198, -0.4493483603000641, 0.9927031993865967, -0.5628445744514465, -0.43842166662216187, 0.0551689974963665, 0.7774920463562012, -0.4842834174633026, 1.5099482536315918, 0.05229125916957855, -0.4407542943954468, 0.7752242088317871, 0.10250070691108704, -0.14516128599643707, -0.9448996186256409, 0.6014003753662109, 0.6803950071334839, -2.0990116596221924, 0.08858568966388702, 0.031533241271972656, -0.21663887798786163, -1.2940374612808228, 0.4062398672103882, 1.450290322303772, -0.7116026878356934, 1.0346548557281494, -1.328618049621582, -0.3648196756839752, 0.13830207288265228, -0.16675293445587158, 0.5366442203521729, -0.2974245250225067, 0.6295362114906311, 0.44773030281066895, 0.6132177710533142, -0.5139868855476379, -0.2292291671037674, 0.793845534324646, -1.9746603965759277, -0.3515079617500305, -0.8709548711776733, 0.15982270240783691, 1.369256615638733, 0.7953149676322937, 1.042798638343811, 1.4274225234985352, -0.5167484283447266, -2.5920357704162598, 1.2440072298049927, 2.1397452354431152, -0.9495350122451782, 0.3635314106941223, -0.6356930732727051, -0.5492900609970093, 0.8698389530181885, -0.560356616973877, -0.23495341837406158, 1.1271121501922607, -1.270788550376892, 0.012843739241361618, -0.6712529063224792, -2.9434385299682617, -0.14705225825309753, -0.047913484275341034, -1.04006826877594, 1.041723608970642, 0.5347170233726501, -0.6367199420928955, 0.2155788391828537, -0.6475827097892761, -0.984020471572876, 1.0279446840286255, 0.8184646368026733, 0.06342211365699768, -0.3025762438774109, 0.6763471364974976, 0.02123081125319004, -1.930733323097229, -1.511674404144287, -0.9211199283599854, 0.03158905729651451, -1.1916075944900513, 0.3748653829097748, -0.38741588592529297, -0.19453485310077667, -2.0816969871520996, 0.7967197299003601, -0.2926522195339203, 0.4012126326560974, -0.34729713201522827, -0.34225019812583923, 1.6061797142028809, 0.8401910066604614, -1.1630725860595703, -0.06313055753707886, -1.0113329887390137, 1.1518949270248413, -1.4423726797103882, 0.791427731513977, -0.2087191939353943, 1.0703821182250977, -1.8798414468765259, 1.3714573383331299, -0.2662374675273895, -1.1295857429504395, 0.15022628009319305, -1.6639827489852905, 0.587533175945282, 1.602730393409729, -0.37463608384132385], [0.2562769651412964, 1.359370470046997, -0.5778999924659729, 1.5884671211242676, 0.2615102231502533, -0.12788479030132294, -0.1068471297621727, -0.036564216017723083, -1.6261296272277832, -1.3099161386489868, 0.7261268496513367, 1.4551256895065308, 0.4321776330471039, -1.592727541923523, 0.20260554552078247, 1.0561131238937378, 0.04652523621916771, 1.278145432472229, -0.1210247203707695, -0.0983104482293129, 0.19267353415489197, 0.5395271182060242, 1.5136796236038208, -0.5564026832580566, -0.24880218505859375, -0.260904997587204, -0.6824219822883606, -0.30159682035446167, 0.6441056132316589, 0.5158941149711609, -0.2626536190509796, 0.8452677726745605, 0.32890990376472473, -0.08185049146413803, 1.2078256607055664, 0.06507288664579391, -1.1186038255691528, 0.9445209503173828, -0.11827659606933594, 0.9738967418670654, -0.5339018702507019, 0.8477298021316528, 0.10317526757717133, 0.755763053894043, 1.7811002731323242, 1.0716439485549927, -0.3799620568752289, -0.5688489675521851, -0.5065915584564209, -0.8443989753723145, 1.022512674331665, -0.5016040205955505, 0.5777149796485901, -1.2739695310592651, -0.6737029552459717, 1.4057103395462036, -0.7477656602859497, -1.857221245765686, 0.04716695100069046, 0.4302232563495636, 2.742957592010498, -0.30074578523635864, -0.838778018951416, -0.5017732977867126, 0.041310302913188934, -0.07004804909229279, -0.17489513754844666, 1.30479097366333, -0.7151395678520203, -0.20346078276634216, -0.15662413835525513, 0.01733812689781189, 1.249825119972229, -0.19557446241378784, -1.5283806324005127, -0.44104668498039246, -2.123842239379883, -0.453053742647171, 0.5059882998466492, 0.038489531725645065, 0.12615786492824554, 0.5184496641159058, -1.1063199043273926, -0.777178168296814, -0.5035948157310486, 1.1450648307800293, -0.17172174155712128, -0.0667731761932373, -0.9647731184959412, -1.091141700744629, -1.2554327249526978, -0.8557461500167847, 0.42395883798599243, -1.0552167892456055, 0.08742570877075195, -0.5155154466629028, 0.6805093884468079, -0.2637662887573242, 0.7697400450706482, 0.18062131106853485, -0.025520849972963333, -0.19187618792057037, 0.33592531085014343, -0.5624998807907104, 0.7633350491523743, -1.350150227546692, -0.9766570329666138, -0.9452760219573975, -0.27520644664764404, 0.20373119413852692, -0.5910748243331909, 2.0709068775177, -0.416036456823349, 1.0225967168807983, 1.834287405014038, 0.49819985032081604, -0.9522077441215515, -1.3534772396087646, 0.40061891078948975, 0.11773227155208588, -0.7785696983337402, 1.6488550901412964, -0.34382498264312744, -0.42545002698898315, 1.1769627332687378, -1.448514461517334, -1.0791608095169067, -0.1267186999320984, 1.5442010164260864], [-0.7027126550674438, -0.9339861869812012, -0.173793762922287, -1.2168545722961426, 0.853556752204895, -0.7305299639701843, -0.6150515079498291, -0.19649973511695862, -1.2546558380126953, -0.46801891922950745, -1.3882733583450317, -0.3604806959629059, 1.4851360321044922, -0.34658414125442505, 0.7433556914329529, -0.7028014063835144, -0.7566343545913696, -0.5316720008850098, -1.6016825437545776, 0.15817882120609283, 0.9565196633338928, 0.24179032444953918, -0.5025138854980469, 0.11200062185525894, 0.2231992781162262, 2.42465877532959, -1.1240358352661133, 0.13231565058231354, -0.7752285003662109, -1.1980295181274414, -0.7627974152565002, 1.0058315992355347, 0.8269144296646118, -0.014152134768664837, -2.136502981185913, -1.089968204498291, 1.498457670211792, -0.0884438082575798, 0.3504332900047302, 1.4525372982025146, -0.7139008641242981, -0.7298848032951355, -0.21082496643066406, -0.030712923035025597, 0.9976353645324707, -0.5236765146255493, -1.6069831848144531, 0.1258973628282547, -1.4622631072998047, -0.2934427559375763, 0.905857264995575, -0.7651058435440063, -0.9642835855484009, -0.1513383686542511, 1.2509543895721436, 1.5700868368148804, 0.35565268993377686, 1.41888427734375, 0.9823694229125977, 0.8912085294723511, -0.10558761656284332, 1.3011029958724976, 0.021750284358859062, -1.8475494384765625, -0.8446236252784729, -1.0574995279312134, 0.11943002790212631, -1.9594532251358032, 0.13109806180000305, -0.9016858339309692, -1.1583731174468994, 0.5299820303916931, -0.11013390123844147, 1.664333462715149, 2.3017096519470215, -1.5722110271453857, -0.2512444257736206, 0.2507452666759491, 0.10027279704809189, -0.11444739997386932, -0.47395020723342896, 0.9554381370544434, -2.2733054161071777, -0.5604628324508667, -1.2515175342559814, -1.6180187463760376, 1.6294445991516113, -0.47882094979286194, -0.23592199385166168, 0.6938459277153015, -0.3816887140274048, 0.5549683570861816, -1.350551724433899, -0.20387904345989227, -0.2852749526500702, -0.3051337003707886, -0.07114001363515854, 0.8128271698951721, 0.4271274507045746, 0.33484360575675964, -1.6010019779205322, -0.4825238585472107, -1.2082687616348267, 0.17586089670658112, 0.4538123607635498, 0.6171798706054688, 1.5899451971054077, 0.5566216111183167, 0.9627969264984131, -0.1973721981048584, -0.6226615905761719, -0.44041258096694946, -0.1987266093492508, -1.0763105154037476, 0.8436375260353088, -0.5636139512062073, 1.0813171863555908, 0.43162742257118225, -1.1740552186965942, 0.8001235127449036, 1.1914597749710083, -1.3279579877853394, -0.4584878087043762, 0.2387438863515854, 0.09301576018333435, 0.9806187152862549, 1.8911848068237305, 0.40496888756752014, 1.2845746278762817], [-0.6123723387718201, -1.1983375549316406, 0.9601648449897766, 0.1165163591504097, -0.051282916218042374, 0.5685257315635681, 0.040864765644073486, -0.49888718128204346, 0.43239444494247437, 1.109459638595581, -1.2455356121063232, -0.08886153250932693, -1.4024229049682617, 0.7649339437484741, -0.5472310781478882, 0.09962069988250732, 0.8906682729721069, 0.6442180871963501, -1.5338563919067383, -0.6097662448883057, -1.150955319404602, -1.4665168523788452, -0.18724939227104187, 0.40868741273880005, -0.9219741821289062, 0.36316198110580444, -1.9021331071853638, -0.03793922811746597, 0.8192306756973267, 0.15482796728610992, 0.34513166546821594, -0.9920960068702698, 0.47258490324020386, 1.1983901262283325, 0.6113513708114624, -1.1400765180587769, -0.9039173722267151, 1.0892724990844727, 1.4606044292449951, 0.17097777128219604, 0.3701280653476715, -0.6041819453239441, 0.771669864654541, -1.2590540647506714, 0.8303437232971191, -0.013513170182704926, -0.2321631759405136, 1.1838908195495605, 0.5284616351127625, 0.4502653479576111, -0.050860967487096786, 0.38553547859191895, -1.8287631273269653, -1.3737763166427612, -1.1842209100723267, 2.180986166000366, 1.2709071636199951, -0.21479283273220062, -0.29801321029663086, -0.46252942085266113, -0.22319896519184113, 0.7665062546730042, -0.05168820172548294, -0.34244266152381897, 0.7663993835449219, 0.3463398218154907, 1.3277487754821777, -0.0035377303138375282, 0.006384738255292177, -1.0928372144699097, 1.2520140409469604, -1.8692282438278198, 0.31690800189971924, 0.33147579431533813, 0.9728346467018127, 0.4844748079776764, 1.3005170822143555, 0.5260598063468933, -0.7938979268074036, -0.5098648071289062, 1.2510205507278442, -0.252966046333313, 0.6783762574195862, 0.3964702785015106, 0.47112274169921875, -1.2120600938796997, -0.7314813733100891, 1.4961825609207153, 1.0468999147415161, -0.008509786799550056, -0.9111505150794983, 0.4503418505191803, 2.06284761428833, -1.3652485609054565, -0.9472084641456604, -0.5839132070541382, 1.7462592124938965, -1.362831473350525, 0.5488200187683105, 0.7093876600265503, -0.7894090414047241, 0.4268558919429779, -1.582865595817566, 2.096062421798706, 0.7118596434593201, -0.7163171768188477, -0.3917921483516693, -0.9061277508735657, -1.363594651222229, -1.0122103691101074, -1.3939391374588013, 0.8785709738731384, 1.044816493988037, -0.6513534784317017, -0.14934054017066956, 1.4960541725158691, -1.0524773597717285, 0.6052201986312866, 0.7502397298812866, -0.6611064672470093, 0.9563215374946594, 0.4616886079311371, -0.07356036454439163, 1.7306251525878906, 2.1660993099212646, 0.9723348617553711, -1.4717296361923218, 0.8140591382980347, 1.6252645254135132], [0.289929062128067, -1.3042420148849487, -0.3609103560447693, 0.6492697596549988, -0.13091294467449188, -0.5412583947181702, 0.45393040776252747, 1.3137540817260742, -1.7652547359466553, -0.6727639436721802, 0.20271286368370056, 0.45953071117401123, -0.7652801871299744, -0.3806479871273041, -0.9928502440452576, -0.052699293941259384, -0.8150147199630737, 1.5097529888153076, -0.8789024353027344, 0.2850880026817322, 0.38143470883369446, 2.0650315284729004, 1.1344709396362305, 1.4856230020523071, -0.46793821454048157, 1.0589253902435303, -3.5858185291290283, -2.3421854972839355, -2.0403177738189697, -0.24173198640346527, 0.5253840684890747, -0.23962460458278656, -0.13328562676906586, 0.26451727747917175, 0.12672825157642365, -0.21783797442913055, 1.2617172002792358, 0.31697991490364075, 0.05485747754573822, 0.37536248564720154, 1.2565232515335083, -1.156654715538025, -0.8094742298126221, -0.5958530902862549, 0.13290084898471832, -1.3298022747039795, 0.7962260842323303, 0.16863968968391418, -0.08672245591878891, 1.2282614707946777, 0.4161357879638672, 1.3495279550552368, 0.891133189201355, 2.2603421211242676, -2.436951160430908, -0.7324126362800598, 0.46669575572013855, -0.09328257292509079, 1.2719290256500244, 0.3293027877807617, -2.1159510612487793, -1.9364814758300781, -0.333234965801239, -0.07866063714027405, 0.3971536457538605, -1.1712220907211304, 1.0431534051895142, 0.03972155973315239, 0.45714104175567627, 0.8019856214523315, -1.0337611436843872, -1.0511730909347534, 0.2488832175731659, -1.1219578981399536, -0.5065301656723022, 0.36283013224601746, -2.246103525161743, -0.8042875528335571, 2.140050172805786, 0.2950115501880646, 1.4917398691177368, 1.6901099681854248, 0.5286687016487122, 0.9267748594284058, -1.0726332664489746, -0.35625526309013367, 2.2622382640838623, -0.2216072529554367, 0.8833611011505127, 1.05376136302948, 0.8031972050666809, -0.14237704873085022, 1.2325537204742432, -1.4168208837509155, -0.35881906747817993, -0.691949188709259, 0.8855205178260803, 0.18944405019283295, -0.6167480945587158, 0.7175669074058533, -0.3863673508167267, -0.3349570333957672, 0.4135037958621979, 0.5305927991867065, 1.021850347518921, -1.6466630697250366, -0.29067322611808777, 0.5501315593719482, -0.8815715312957764, -0.8054243922233582, -0.7271986603736877, -0.4546862244606018, 1.938478946685791, 1.0433521270751953, 0.43501365184783936, 0.49258357286453247, 0.16906335949897766, -1.6340056657791138, 1.3917382955551147, -0.09536075592041016, -0.46307042241096497, -0.6852652430534363, -0.4435884952545166, -0.5462475419044495, -0.995961606502533, 0.22048187255859375, -0.2957071363925934, 0.8679707646369934, -0.6519927978515625], [-0.07162471860647202, 0.8107953071594238, 0.8421950936317444, 0.03765256702899933, -1.1825181245803833, 1.8798314332962036, -0.5431864261627197, 0.3359951674938202, -0.1103045642375946, -0.11999893933534622, 0.09865862876176834, -0.3840913474559784, -1.6472679376602173, -0.8150375485420227, 0.8133371472358704, 1.1896482706069946, -0.13793396949768066, 0.493514746427536, 0.03143109753727913, 0.029959242790937424, 1.413763165473938, -1.7433100938796997, 0.1377299427986145, 0.6371168494224548, 0.03088035061955452, 0.9263778924942017, -1.8676958084106445, -1.2825171947479248, -0.08889186382293701, 0.8678882718086243, -2.1895737648010254, 0.6254887580871582, -0.2430325299501419, 0.4299088418483734, -0.6028293371200562, 1.036104679107666, 0.10158461332321167, -1.1207606792449951, 0.0853087529540062, 0.5009954571723938, -1.0035810470581055, -0.6721478700637817, -1.4510729312896729, 1.2069287300109863, 0.15961425006389618, -1.480538249015808, -0.08616502583026886, 0.3504698872566223, -1.5147751569747925, -1.2507011890411377, 0.37999165058135986, 0.004592874087393284, -0.5063938498497009, -0.8307124376296997, -1.473996639251709, 0.5311228036880493, 0.9114424586296082, -0.4641328454017639, -2.135815143585205, 2.492295742034912, -3.961111545562744, 0.19451041519641876, -0.34882310032844543, -0.3298984169960022, -0.4205724895000458, 0.154195636510849, -0.3560884892940521, 0.6075003743171692, 0.634753942489624, -0.1334507167339325, 1.1795976161956787, 0.34613871574401855, 1.5695629119873047, -1.8014322519302368, 0.4624950885772705, -0.8326119780540466, -0.5294711589813232, 0.8134447336196899, 0.2603745758533478, 1.6319458484649658, -0.42736855149269104, 0.48697444796562195, -1.0068495273590088, 1.2263516187667847, -0.6015766859054565, -0.8218265175819397, -1.6998120546340942, -0.1835760623216629, -0.04825939983129501, 1.3213268518447876, -0.21115387976169586, -0.6150866150856018, -0.8464274406433105, -0.23960228264331818, -0.023690413683652878, 0.48246753215789795, 0.201971173286438, 0.7416303753852844, 0.2513163983821869, 0.03390892967581749, -0.5020216703414917, 0.26999539136886597, 0.6081798672676086, 0.7375159859657288, -0.5807680487632751, 2.6533894538879395, -1.0181955099105835, 0.4853609800338745, -1.8300702571868896, 1.0882294178009033, -1.7483859062194824, -1.3286391496658325, -0.13172022998332977, 0.8246228098869324, 1.552443027496338, 0.09432483464479446, 0.021167665719985962, 0.9804202318191528, 0.07092823088169098, 0.0919000431895256, 0.23827940225601196, 0.7320228219032288, -1.522379994392395, -0.13114506006240845, -0.15119405090808868, 0.20179393887519836, -0.30643323063850403, 0.1260118931531906, -1.4589226245880127], [-1.820056438446045, 1.451988935470581, -1.5799843072891235, 0.005838441662490368, 0.42996057868003845, -1.3738479614257812, 1.0742616653442383, -0.2766052782535553, -2.3651087284088135, -0.7140007615089417, -1.2809557914733887, 0.5436130166053772, 0.76007479429245, -2.8296144008636475, 1.1598411798477173, -0.7108290791511536, -0.9194270372390747, -0.33721020817756653, 0.592656672000885, -0.43095722794532776, 1.1735458374023438, -0.014153847470879555, 0.4526604115962982, 1.109460711479187, -1.0862483978271484, 0.7826011180877686, -1.232361078262329, -0.4201139807701111, 0.7716915011405945, -1.6486061811447144, 0.49164626002311707, -0.19374781847000122, 0.34870558977127075, -0.9293453097343445, -1.0027265548706055, 0.7266258001327515, 1.8207933902740479, 0.20358580350875854, 0.9638876914978027, -1.5640811920166016, 0.7045746445655823, 1.700191617012024, 0.517275869846344, -0.7272242903709412, -0.027893241494894028, 0.3742629587650299, -1.4357281923294067, 1.2843866348266602, -0.045880213379859924, 2.1029481887817383, 0.9256840348243713, 1.2853736877441406, -1.201993465423584, 0.2608420252799988, 1.463637113571167, 0.8526737689971924, 0.5679672360420227, 0.03217514976859093, -0.39568376541137695, -1.05545175075531, -0.2537992596626282, -0.5266913175582886, -0.48911580443382263, 1.7030812501907349, 1.1009037494659424, 1.2002440690994263, -1.2987126111984253, -0.1804412454366684, 1.4561766386032104, 0.23366783559322357, 0.3773684501647949, -0.29718253016471863, -0.26902246475219727, 0.5419201254844666, -0.9188236594200134, 0.15908455848693848, -0.07151918858289719, -1.3374946117401123, -0.3832845389842987, -0.5629345774650574, 0.9911062717437744, 0.2811256945133209, 0.09914005547761917, -0.22038263082504272, -0.2504047453403473, -1.3495104312896729, -1.0196805000305176, 0.24585723876953125, -0.3760630488395691, 1.114659309387207, 1.8759793043136597, 1.1138650178909302, -0.9601899981498718, -1.0821373462677002, -0.07069631665945053, -0.2771860361099243, 0.8807042837142944, 0.3329584300518036, 2.2235469818115234, 0.4409606456756592, 0.3043482303619385, -1.4143874645233154, -0.6892826557159424, 0.07875334471464157, 0.7945488691329956, 0.04762306436896324, 0.5520878434181213, 2.111412763595581, 0.5349388718605042, -1.6875863075256348, -0.2169644981622696, -0.7024837732315063, -0.06348875164985657, -0.425520122051239, 1.2008718252182007, -0.3337942659854889, -1.5825337171554565, -0.7400413155555725, -0.23726768791675568, 0.12618732452392578, -0.3172473907470703, -0.015285278670489788, -1.3243821859359741, -1.4625548124313354, -0.2714262902736664, 0.35858967900276184, 0.6158376932144165, -0.9236797094345093, -0.009763000532984734], [0.8270548582077026, 0.5557321310043335, 0.4565849304199219, -0.48734432458877563, -1.8402544260025024, 0.31218481063842773, -0.19409722089767456, -0.21115025877952576, -1.4510345458984375, -0.6052754521369934, -1.8625781536102295, 0.3289291262626648, 0.21131931245326996, 0.7822604775428772, -0.7900161147117615, 2.23024845123291, -0.5919120907783508, -0.8850372433662415, 1.8495150804519653, 0.22802291810512543, -0.7351223230361938, 0.3281886875629425, 1.1911903619766235, -1.6143943071365356, -0.8626686930656433, 1.0116266012191772, -0.832633912563324, -0.3971785604953766, 1.156416416168213, -0.5657100081443787, 0.5029907822608948, -0.2581404745578766, 1.0239650011062622, -1.2396156787872314, 0.6238643527030945, -0.12584610283374786, 1.550018072128296, 0.2238481640815735, -0.27169737219810486, -0.6003907918930054, 0.40426814556121826, 0.4875446856021881, 0.1497117429971695, -1.2230334281921387, 0.16597460210323334, -0.1654708832502365, -0.4371472895145416, 2.2372114658355713, 0.44266176223754883, -0.07327822595834732, -0.10753967612981796, -0.23878644406795502, -1.331934928894043, 0.28519123792648315, 0.624062180519104, -0.1881837695837021, 0.7066840529441833, -0.847295343875885, 0.15780627727508545, -0.6233358383178711, 0.6474300026893616, -0.34111666679382324, -2.1235268115997314, -0.33432748913764954, -0.5027765035629272, 1.4470462799072266, 0.6212284564971924, -0.51316237449646, 1.8187967538833618, 0.2581369876861572, -1.3887429237365723, 1.4478988647460938, -0.8657440543174744, -1.1601992845535278, -1.6899449825286865, -0.47876009345054626, 0.08965785056352615, 1.0270521640777588, 0.5125513672828674, 1.8297526836395264, -1.625068187713623, 0.8012414574623108, -0.14474701881408691, -0.8413723111152649, 0.3573354482650757, -0.8954271674156189, -0.4883939325809479, 0.4384264349937439, 2.652019500732422, -0.16758595407009125, -0.018673252314329147, -0.42736533284187317, -0.9463067650794983, -1.1043871641159058, -0.017977790907025337, -0.5853532552719116, 1.634229063987732, -0.6484308242797852, 0.8654305934906006, 0.2821449339389801, 1.9837692975997925, 0.6978298425674438, -1.0405629873275757, -0.2392992228269577, 0.776406466960907, -0.13516117632389069, 1.11176598072052, -0.8091342449188232, -2.3963441848754883, 0.11895134299993515, 0.9094260931015015, 1.511918306350708, -0.8477847576141357, 0.6465582251548767, -0.5009887218475342, -0.3071715235710144, -2.1096153259277344, 0.7309252619743347, -0.5794985890388489, 0.5510363578796387, 1.7379794120788574, 0.11480589210987091, 0.6478698253631592, -0.762398898601532, -0.24510295689105988, 2.2634496688842773, -1.081298828125, -0.5432504415512085, -0.5857834219932556], [-0.2132999449968338, -0.9474842548370361, 1.0176362991333008, -0.5050522089004517, -0.03038768097758293, -1.0640010833740234, 1.2809791564941406, 1.7034752368927002, -0.3315078914165497, 0.017051689326763153, 0.9495929479598999, 0.45405489206314087, -0.5261214971542358, 0.1251269429922104, -2.560443878173828, 1.3781237602233887, -1.0077648162841797, -0.39735832810401917, 0.46741464734077454, -0.15505802631378174, -0.06034017726778984, 0.9391270279884338, 0.6548206806182861, 0.2824784517288208, -0.1727742850780487, 0.5168068408966064, -1.0315171480178833, -0.6094335317611694, 0.33419302105903625, -2.1880316734313965, 1.7466251850128174, -1.0496314764022827, -1.4892802238464355, 1.1976020336151123, -0.6870885491371155, -0.1298026591539383, 0.0318230502307415, 0.7258656024932861, -0.3134336471557617, 0.7432153820991516, 0.07435856014490128, 0.6482053399085999, -0.04501902312040329, 0.7697126865386963, 1.3597294092178345, -0.8651959300041199, -0.19445106387138367, -0.07928227633237839, -1.0692919492721558, 1.479018211364746, -0.21860384941101074, -1.5274919271469116, -1.4391032457351685, 0.5112728476524353, -2.4186489582061768, -0.8642178177833557, -0.7465123534202576, 0.6368275880813599, -2.816869020462036, 0.7830021977424622, 1.6847831010818481, -1.4564388990402222, 0.335477739572525, -1.246978759765625, -0.7907770276069641, -0.4527938961982727, -1.7924195528030396, -2.00423002243042, -2.339646339416504, -1.498496413230896, -0.27513745427131653, 0.9052483439445496, -0.07070262730121613, -1.549803614616394, -0.40404483675956726, 2.3599374294281006, 0.9631003737449646, -1.099544644355774, -0.43390393257141113, 1.5872606039047241, -1.742393970489502, 0.5686777830123901, -1.1038960218429565, -0.897396445274353, 2.0286142826080322, -0.3946661055088043, -0.16556720435619354, -1.1040220260620117, -0.023926904425024986, 0.9244269132614136, -1.0082511901855469, 0.346321702003479, 0.2321028858423233, 1.0018006563186646, -1.5811471939086914, 1.5377788543701172, -0.15065541863441467, -0.9687649607658386, -0.944463849067688, 0.6326454877853394, 0.8423548340797424, 2.0083372592926025, 1.4496649503707886, 0.5671643018722534, -0.6927632093429565, 0.9290880560874939, 1.1199220418930054, -0.3007355034351349, -0.5846890211105347, 0.2281518429517746, 0.2348249852657318, -0.04685339704155922, 0.31349658966064453, 0.08116800338029861, 0.8644710183143616, -0.22421486675739288, 1.3676680326461792, -0.4342208206653595, 0.44037362933158875, 0.8231933116912842, -0.10633603483438492, 0.11247138679027557, 1.3403680324554443, -0.5186192989349365, 0.26698577404022217, 0.7852712273597717, 0.177556112408638, 0.274050235748291, -0.2982501983642578], [-0.4115935266017914, -1.0925568342208862, 0.11032255738973618, 1.0793559551239014, 0.1085282564163208, 0.27502670884132385, 1.6003737449645996, -0.09466178715229034, -0.45063894987106323, 0.8625024557113647, 0.8682996034622192, 0.8879303932189941, -1.6283817291259766, -1.6597354412078857, 2.92234206199646, 1.1528804302215576, 1.2601349353790283, 0.06405685842037201, 0.5977238416671753, 1.722139596939087, 0.19859208166599274, 1.1023430824279785, 0.07299747318029404, 1.1969079971313477, 0.5254574418067932, -1.7089271545410156, 0.39340919256210327, 2.1309616565704346, 1.1700072288513184, 0.6580101847648621, -1.1708561182022095, -1.0457602739334106, 1.0767228603363037, 0.27766120433807373, -0.2966139614582062, 0.22256781160831451, -0.48994264006614685, 0.5072025060653687, 0.8242541551589966, -0.4769758880138397, 0.33462750911712646, -0.8444846272468567, 0.8969066143035889, 0.5846412181854248, 0.8483942151069641, -0.6407338976860046, -0.1976863443851471, 0.13069117069244385, -1.7566617727279663, -0.08178862184286118, 0.7926480174064636, -0.6117161512374878, 0.8121699094772339, 0.7017967104911804, 1.719572901725769, -0.4485491216182709, 0.591313362121582, -0.915818989276886, -0.7677541375160217, 0.7498175501823425, -0.567599892616272, -0.6210520267486572, -0.0038436518516391516, 0.1798926740884781, 0.28595682978630066, -1.5448418855667114, -1.4773468971252441, 1.1684551239013672, -0.20109350979328156, 0.6716249585151672, -0.24247127771377563, 1.1221120357513428, -0.9054640531539917, -0.371987521648407, -0.8588465452194214, 1.976257562637329, 0.6560826301574707, 0.4692980945110321, -0.11043096333742142, 0.9768533110618591, -1.9368828535079956, -0.0143484091386199, -0.4571181535720825, 1.0791875123977661, -1.8402259349822998, -0.10640057176351547, -0.8633902668952942, -0.154069185256958, -0.09746850281953812, 1.9855592250823975, 1.0662742853164673, -0.1308622509241104, 0.8976256251335144, 0.19385065138339996, 0.6537643671035767, -1.7238131761550903, -0.44879764318466187, -0.506482720375061, 0.30641236901283264, -1.1272039413452148, -0.25972768664360046, 1.7054686546325684, -1.0703176259994507, 1.5138486623764038, 0.7761281728744507, -0.8748490810394287, -1.1709561347961426, 0.5515462160110474, -1.932630181312561, 0.3683064579963684, 0.09864402562379837, -0.2323550134897232, 0.6951448321342468, -1.0092215538024902, -0.4945146143436432, 0.12811410427093506, 0.48250260949134827, 0.8583746552467346, -0.6521013379096985, -0.9565448760986328, -1.5431660413742065, 0.935088574886322, -0.030886929482221603, 0.8633956909179688, -0.5944041013717651, -1.0342605113983154, 1.092949390411377, -0.5028483271598816, 0.45139363408088684], [0.7416075468063354, 0.21542514860630035, 0.054868634790182114, -1.066477656364441, -0.42967909574508667, 0.1467004418373108, -0.3521692752838135, -0.5510785579681396, -1.7384084463119507, -1.2434971332550049, -0.628273606300354, -1.5819505453109741, 0.6141551733016968, -0.13422906398773193, 1.611777663230896, 0.2860451340675354, 1.4149631261825562, -1.2644070386886597, 0.3548475503921509, 0.6984944939613342, -1.0077903270721436, 1.3030221462249756, 0.9093005657196045, 0.4326752722263336, 0.3511609435081482, 1.032109022140503, 1.3257733583450317, 1.8650144338607788, -2.459714889526367, 1.6148816347122192, -0.5805695652961731, 1.842734694480896, 1.3408482074737549, 0.3616834580898285, 0.30567288398742676, -1.8663214445114136, -0.3011738955974579, 0.41755035519599915, 1.9592945575714111, 0.38621631264686584, 0.45421546697616577, 1.9394491910934448, 0.8163593411445618, 0.8692420721054077, 0.4315275549888611, 0.036529541015625, -1.6677961349487305, -0.8967307806015015, -0.8659471869468689, -0.4344506561756134, 0.2533327341079712, 0.33200931549072266, 0.3751128017902374, 0.5626569390296936, 0.08714217692613602, -1.0327092409133911, -0.4932558536529541, -1.0177451372146606, 0.2343062311410904, -0.4663533568382263, -0.3848782181739807, -2.0659801959991455, 0.18341468274593353, 0.6355894207954407, 0.9722299575805664, 0.36170732975006104, 0.6676826477050781, 0.7558972239494324, -0.9066910743713379, 0.0024365868885070086, 0.3254684805870056, -0.40774157643318176, 0.9108255505561829, -1.0830628871917725, -0.02887287363409996, -0.44035887718200684, -1.0908384323120117, 0.5155288577079773, 0.0020002739038318396, -1.2600115537643433, -1.2290964126586914, -0.1212218776345253, -0.8867595195770264, -0.6987779140472412, 1.2095868587493896, -0.2469693273305893, 0.2727104425430298, -0.7976580858230591, 1.3917864561080933, 0.06005604937672615, -0.025051677599549294, -1.3567556142807007, 0.17452217638492584, 0.569023609161377, -0.734959602355957, -0.7138978838920593, -1.8622918128967285, -1.1234064102172852, -0.7543603181838989, -1.202483892440796, -0.8014392852783203, 1.8452390432357788, -0.7875198721885681, -0.35056760907173157, -0.38135817646980286, 0.8370589017868042, -1.088414192199707, 1.176113486289978, 0.8913757801055908, 0.47355684638023376, 0.5368097424507141, 1.5876652002334595, -0.43059998750686646, -0.64103102684021, -1.7597243785858154, 0.5104038119316101, -0.6994921565055847, -1.5563558340072632, 2.3447961807250977, -0.19986608624458313, 1.4638350009918213, -0.6289238929748535, 2.0552191734313965, -0.04443495720624924, -0.41227200627326965, 1.0548079013824463, -1.0864378213882446, 0.7831328511238098, -1.9467052221298218], [0.3276155889034271, 0.007008267100900412, 0.21637825667858124, 1.368510365486145, 0.9650594592094421, -1.095835566520691, -0.9206291437149048, 0.4634098410606384, -0.38347768783569336, -1.3500937223434448, 0.07890598475933075, 1.2969074249267578, 1.0485762357711792, -0.8408376574516296, -2.028378963470459, -0.6891648769378662, -0.02347867749631405, -1.196433186531067, -0.9523595571517944, 1.0885047912597656, 0.39667555689811707, -0.34210479259490967, 0.4739084839820862, -1.4396921396255493, 0.967357337474823, -0.5744447708129883, 0.29308196902275085, -0.35066378116607666, -0.20315119624137878, -0.13310697674751282, 1.1746188402175903, -0.33444899320602417, 0.37454497814178467, -0.21118438243865967, 1.7388310432434082, -0.02312198095023632, -0.9126940369606018, -1.9485902786254883, 0.9144796133041382, -0.7027764916419983, -1.4704948663711548, -0.43468785285949707, -0.06632454693317413, -0.5233272910118103, -0.9392779469490051, -1.1578514575958252, -0.0685882419347763, -0.8496149182319641, 0.2931019067764282, -0.28614944219589233, 1.9853790998458862, 0.06564395874738693, -0.1898682713508606, -0.5904040932655334, -1.3098342418670654, 0.21454742550849915, 0.25330135226249695, -0.6968620419502258, -0.344500333070755, 0.036612775176763535, -1.164060354232788, -0.24772484600543976, -1.468820333480835, 2.389648675918579, -1.4890882968902588, -1.2587867975234985, 0.1615685522556305, -0.19453692436218262, -0.046922724694013596, 0.07470047473907471, 0.6899099349975586, 1.8385484218597412, 0.6646183133125305, -1.811151385307312, 0.900528609752655, 1.7455484867095947, 0.6804128885269165, 0.6556442379951477, -0.7133691310882568, -0.12249420583248138, 0.9384950399398804, 0.0006867394549772143, -0.3172977566719055, -0.19739434123039246, 0.06751788407564163, 0.21226611733436584, 0.5452331304550171, -0.4676465392112732, 2.3078880310058594, 1.1538206338882446, -0.1570311039686203, 1.0850815773010254, -0.12880048155784607, 0.6516051888465881, 2.3125667572021484, 0.18641473352909088, 0.3523915410041809, 0.7182124853134155, -0.2561403512954712, -0.3588108420372009, -0.46402156352996826, 0.3735477030277252, -0.1572285294532776, -1.9960864782333374, -0.5353001952171326, 0.6334110498428345, 1.4700517654418945, 0.9875768423080444, -1.0720163583755493, 0.1038934737443924, -0.6181567311286926, 1.7159812450408936, -1.48358952999115, 0.189208522439003, 0.22552667558193207, -0.37701165676116943, 1.7760653495788574, 1.903894066810608, -0.6478148698806763, -0.10289155691862106, -0.593431830406189, 1.23960542678833, 1.399673581123352, 0.08259575814008713, -0.5350810289382935, 1.2141876220703125, 1.4884980916976929, 0.3123232424259186, -1.423989176750183], [-0.48067769408226013, -1.0532550811767578, -0.8075035810470581, -1.1254165172576904, -0.46604079008102417, 0.3932912051677704, 0.588539183139801, 1.0126817226409912, 0.01128442119807005, -0.37958741188049316, 0.15272679924964905, 0.5277031064033508, -0.3565126359462738, 1.28838312625885, 0.14693933725357056, 0.8074567317962646, 2.2105860710144043, -1.661946415901184, 0.48779863119125366, -1.5822325944900513, 2.3117799758911133, 0.15354323387145996, 1.0998711585998535, 1.1061270236968994, 0.6179348230361938, -0.09858474135398865, 1.3237704038619995, -0.20934300124645233, -0.27982622385025024, 0.5452538728713989, 0.22612681984901428, -0.48496943712234497, 0.8405210375785828, 0.34223368763923645, 0.13090510666370392, -1.085931420326233, -1.1458237171173096, -0.04131615161895752, 0.21573255956172943, -1.4154553413391113, -1.9858919382095337, 0.9978246688842773, 1.485480546951294, 0.2861902713775635, -0.04793309047818184, 1.5253710746765137, -0.6478026509284973, 3.0816025733947754, -0.46140429377555847, -0.1760571449995041, 0.9192864894866943, -1.350889801979065, -0.3107767105102539, 0.7919964790344238, -1.4629374742507935, -1.3869168758392334, 1.2667272090911865, 0.2551271319389343, 0.6665832996368408, -1.533619999885559, -0.43871989846229553, 0.6035782694816589, -0.41020193696022034, -1.7414565086364746, 0.01845870539546013, 0.16937518119812012, 0.09901010245084763, -1.653445839881897, -0.026360131800174713, -1.9595913887023926, 0.4729820191860199, 0.9026168584823608, -0.8374884724617004, -1.4516994953155518, 0.15985603630542755, 0.7511901259422302, -0.3296547830104828, -1.5899957418441772, -0.8347502946853638, -0.6638575196266174, -0.08705367892980576, -0.398674875497818, 0.23931404948234558, -0.8392588496208191, 2.2433276176452637, -0.256332665681839, -0.8835298418998718, -0.3169972598552704, 0.14810976386070251, -0.0401308573782444, -0.21282358467578888, 0.5969356894493103, -0.46983614563941956, -0.5188688635826111, -1.1988894939422607, 1.060110330581665, -1.1672323942184448, 1.4385462999343872, -0.7734915614128113, 1.0034464597702026, -0.059136901050806046, -0.4968121647834778, 0.12799814343452454, -0.7877625226974487, -0.7768123745918274, 0.7332906723022461, -0.7403167486190796, 1.6577025651931763, 1.4616694450378418, -0.4205262064933777, -0.5158780217170715, 0.23787321150302887, 0.5954883694648743, -0.45499294996261597, 0.6689966917037964, 0.48801854252815247, 1.291611671447754, 2.28328537940979, -0.598190426826477, 1.6573201417922974, 0.7614875435829163, 0.3831733763217926, 1.6393526792526245, -0.8808367252349854, -0.42789116501808167, 1.4931646585464478, 0.2113628387451172, 0.5830725431442261, -0.20415323972702026], [1.1425753831863403, 0.6047044396400452, 0.11489690095186234, 0.3396848440170288, 1.538625955581665, 0.7918612360954285, -1.4188843965530396, 1.3692249059677124, -1.4527771472930908, 0.3137754201889038, 0.8788070678710938, -0.39521148800849915, 1.154193639755249, 3.375997304916382, 1.422279715538025, 0.428434282541275, -0.30620548129081726, -1.437583327293396, -0.8850985169410706, -1.6047238111495972, -0.6262474060058594, -0.9268887639045715, 0.34805911779403687, 0.6119268536567688, -2.707430362701416, -0.8873947858810425, -1.7139902114868164, 1.3507981300354004, -0.3003849983215332, 0.7664973735809326, 1.5765184164047241, 0.8881438970565796, 0.3559003472328186, 0.2937355637550354, 0.14724284410476685, 1.2834296226501465, 0.8190186619758606, 0.10137369483709335, 0.16397956013679504, -0.8561703562736511, -0.6756880879402161, 1.0344252586364746, 0.6836600303649902, 1.0399439334869385, 1.3201066255569458, 0.27087709307670593, -1.1547788381576538, -0.15786300599575043, -1.3673533201217651, 0.22778916358947754, -1.0067602396011353, -0.27445101737976074, -0.4560813307762146, -0.22541677951812744, -0.7365750670433044, -0.6308788657188416, 0.5867424011230469, 0.9558510780334473, 1.5407835245132446, -0.024549204856157303, -0.7459635734558105, 2.422435998916626, 2.5726513862609863, 3.0568480491638184, -0.2865045666694641, 0.47562268376350403, -0.49060043692588806, 0.5532559156417847, -0.9029361009597778, -0.43500128388404846, -1.0118184089660645, -1.4111099243164062, -0.047891899943351746, 0.11291663348674774, -1.1538490056991577, -0.44647136330604553, 1.916483759880066, 1.4844589233398438, 2.275883436203003, 2.0880651473999023, 0.542682945728302, -1.2314348220825195, -0.91438227891922, -1.4870572090148926, -1.6321711540222168, 0.8910501599311829, 2.3514132499694824, 1.2306934595108032, -0.8152043223381042, 0.5951669216156006, -1.1925190687179565, 0.2234870195388794, -0.3797641694545746, -0.4323285222053528, -0.13593922555446625, -0.3963192403316498, -0.5608872175216675, 0.9582775831222534, -1.8542320728302002, -0.052229274064302444, -1.9109079837799072, -1.4634037017822266, -0.9347617030143738, 0.17367032170295715, 0.5201879143714905, -0.843009352684021, -0.0729009211063385, 0.0955253466963768, 0.10534030199050903, -0.10969429463148117, 1.1119378805160522, -1.7180876731872559, 0.6945669054985046, -0.7969305515289307, -1.1593409776687622, -0.17090673744678497, 1.690116286277771, -0.5346814393997192, -1.1663031578063965, 0.8413293957710266, 1.1390782594680786, 0.24724078178405762, 0.6915003657341003, 0.6811006665229797, -0.7670196890830994, -0.28832632303237915, 1.4899712800979614, -1.2618660926818848, -2.0358612537384033], [-0.4206452965736389, 1.802631139755249, 0.9479624032974243, 0.3310008943080902, -0.1206786036491394, 1.1930798292160034, -0.7450115084648132, -2.4153897762298584, 0.6981467008590698, 0.9365230798721313, 0.7526982426643372, 0.21775074303150177, 1.213026762008667, -0.5266280174255371, 0.04432424157857895, -1.079785704612732, -0.36669301986694336, -0.3428763747215271, -0.9715970158576965, 0.9116112589836121, -0.6423735618591309, 0.4853788912296295, -0.12392140179872513, 0.43103694915771484, -0.8195306062698364, 0.36714112758636475, 1.0621627569198608, -0.48119866847991943, 1.2692986726760864, 0.9759379029273987, -0.48167818784713745, -0.03643433749675751, -2.438699722290039, 0.3501337468624115, -0.373765766620636, 0.2006116658449173, -2.189593553543091, 0.34078866243362427, -2.524040460586548, -1.1527715921401978, 0.1807582825422287, -0.21597115695476532, 1.262258529663086, 0.7533929347991943, 0.3455161452293396, 0.1929136961698532, 0.9416097402572632, 0.30843719840049744, -0.5946357250213623, -1.1772193908691406, -0.4107956886291504, 0.6691263318061829, -0.011475118808448315, 0.5307152271270752, -0.2642707824707031, 1.6112192869186401, 0.63140869140625, 1.0717229843139648, -0.11924130469560623, -0.17147569358348846, 0.5035873651504517, -0.7653954029083252, 3.7663257122039795, 0.4281180202960968, 0.02692534029483795, 0.36344870924949646, 1.9775046110153198, -1.1203268766403198, 1.5360292196273804, 0.596667468547821, -1.2291641235351562, -0.7135398983955383, 0.8341819047927856, -0.25682130455970764, -0.3326261043548584, -0.5112942457199097, -0.45186254382133484, -0.7975021004676819, 0.4936327636241913, -0.6179258227348328, -0.578264594078064, 0.8516628742218018, 0.1585978865623474, 0.838593602180481, 0.5782472491264343, -0.9947943687438965, 1.7373114824295044, -0.3137359619140625, -0.7927491068840027, -0.16312016546726227, 0.6039202809333801, -1.7672686576843262, 0.12874944508075714, 0.20594695210456848, 1.1763246059417725, 1.158375859260559, 0.4236789643764496, -0.4653163254261017, 0.09911960363388062, 0.6356667280197144, -0.2543296813964844, -0.863262951374054, -0.6227304339408875, 1.6803243160247803, 0.06548846513032913, 0.039819542318582535, -0.26080963015556335, 0.15907824039459229, 0.994074821472168, -0.8744610548019409, -1.1322975158691406, -0.23955653607845306, 1.7560256719589233, 0.7871624231338501, 0.5547127723693848, 1.3117793798446655, -0.39556577801704407, -0.2313937097787857, -1.117281198501587, -1.088833212852478, 1.0055843591690063, -0.5753814578056335, -0.18844851851463318, -0.26074767112731934, 0.23465628921985626, 1.796157717704773, -0.39917662739753723, -0.585192084312439, -1.861159324645996], [0.7814244031906128, 0.7678472399711609, 0.14184357225894928, 0.9756956100463867, 0.23512005805969238, 0.20636919140815735, -0.20546448230743408, -0.24578188359737396, -0.8101284503936768, 0.9290327429771423, 1.4759764671325684, 0.1134045273065567, 1.0995639562606812, -0.8609073758125305, -0.5535628795623779, 0.7183734178543091, -0.8383926749229431, -0.3012649416923523, -0.37994417548179626, 1.1814992427825928, 0.30680331587791443, 1.2080549001693726, -0.614278256893158, 0.33292967081069946, 0.07525746524333954, -0.6362085342407227, 0.12885329127311707, 1.0037527084350586, -0.6003483533859253, -1.868157982826233, -1.0938987731933594, 0.10968047380447388, -0.7403608560562134, 0.2068917453289032, 0.4150746166706085, -1.746264934539795, -0.9223626255989075, 0.9138305187225342, 0.6140134930610657, -1.1506413221359253, 0.7918442487716675, 0.27668410539627075, -0.0731429010629654, 0.20017580687999725, -1.5426307916641235, -0.5991924405097961, -0.576840341091156, 0.9015606641769409, -0.386343777179718, 0.03492578864097595, 1.337349772453308, -0.19199135899543762, 0.30036234855651855, 0.6952793598175049, -1.7234488725662231, -0.08672408759593964, 1.218042016029358, -0.3420014977455139, 0.26291725039482117, -0.25094884634017944, 0.2592563033103943, -0.348412424325943, 0.41898980736732483, -0.027109162881970406, 2.1717822551727295, 0.6013630628585815, -0.7673311233520508, -0.5941289663314819, -0.19558651745319366, -0.6104673147201538, -1.2887226343154907, 1.8792389631271362, -0.21458236873149872, -0.4585099220275879, -0.5016638040542603, -0.39159339666366577, 0.2715282440185547, -2.2853827476501465, -0.028534604236483574, 0.3651778995990753, -3.067695140838623, 1.4669091701507568, -0.09605958312749863, -1.1555612087249756, -0.5282440185546875, 0.22747135162353516, 0.9754117727279663, -0.9522823095321655, -0.7401129007339478, -0.01076261606067419, 0.6716548800468445, -1.1859607696533203, 1.5748475790023804, -0.04936770349740982, -0.11317546665668488, -3.1262567043304443, 1.9183224439620972, -0.14902670681476593, -0.12751218676567078, 0.22825458645820618, -0.12069100141525269, -0.35315069556236267, 0.6011175513267517, 0.013296136632561684, -1.2449711561203003, -0.1813696026802063, -0.256895512342453, -0.4027637243270874, 0.5758273601531982, -1.0050345659255981, -1.4159311056137085, -0.266580194234848, -0.29748088121414185, -0.5835676789283752, 0.20981062948703766, -0.4085976779460907, -0.3630278408527374, -2.2988665103912354, -0.5672858357429504, -1.2133878469467163, -0.6363691687583923, 0.8038942813873291, -0.4960094690322876, -0.3836560547351837, -1.5364141464233398, 1.0954995155334473, 0.2681471109390259, -0.7148017287254333, 2.4643452167510986], [0.6227419972419739, 1.0423105955123901, 0.6012762784957886, -0.6417070031166077, -0.18518584966659546, 0.2762784957885742, 0.06897544860839844, 0.7617093324661255, 0.19541870057582855, -0.5757001042366028, 0.7377732396125793, -2.0176970958709717, 1.4217288494110107, -0.949135959148407, -0.0868508517742157, -1.0742439031600952, 1.174674391746521, 0.7576829791069031, -0.6779998540878296, 1.0641638040542603, 1.4881141185760498, 0.3503856062889099, 0.3692035973072052, 1.1630291938781738, -0.3185688853263855, 0.4482533931732178, 1.3274412155151367, 1.516776442527771, -0.08514351397752762, 1.3754994869232178, -0.17560428380966187, -0.15898694097995758, 1.6934903860092163, -0.18517380952835083, -0.009067080914974213, 2.047980785369873, 0.1450313925743103, -1.654828667640686, 1.3928231000900269, -0.6187796592712402, 0.33951449394226074, 0.2465965747833252, 0.7993974685668945, -0.4112657606601715, 0.08332933485507965, -0.25234147906303406, -1.880309820175171, 0.12609697878360748, -2.319892168045044, 0.6643179655075073, -0.11403987556695938, -0.7019249796867371, -0.38449302315711975, -1.2750437259674072, 0.7986176609992981, 0.02764025330543518, 1.0748356580734253, -2.137985944747925, -1.9265364408493042, -0.6322870850563049, 1.5920443534851074, -0.24772928655147552, -0.9726323485374451, 0.6099262833595276, 1.271561861038208, 0.720611572265625, 0.9078177213668823, -0.6678040623664856, -1.0127274990081787, -1.414512038230896, -1.8550605773925781, -0.9593830704689026, -0.37979593873023987, 0.98366379737854, -2.103579044342041, -0.879080593585968, -0.09252143651247025, 0.04147607088088989, -0.13732659816741943, 1.9768035411834717, 1.3247572183609009, 0.3304212689399719, -2.1363062858581543, 0.715108335018158, -0.8430495262145996, 1.5079208612442017, 0.25590455532073975, 0.5622395873069763, -1.3913516998291016, -1.2173720598220825, 0.1553838551044464, 0.12158702313899994, 1.07502019405365, 1.2053182125091553, -0.02095184661448002, 0.6580571532249451, 0.7842678427696228, -0.7079258561134338, -0.6004338264465332, -0.4629702866077423, 0.8114398121833801, -0.3337249755859375, 0.6213542222976685, -0.4169604182243347, -1.5984716415405273, -0.5729576349258423, -1.2314492464065552, -0.7087069749832153, 1.843336582183838, 1.1572867631912231, -0.822368860244751, 0.6797711849212646, 2.9305243492126465, 0.6367794871330261, -0.08116508275270462, -0.5683258175849915, -1.521111011505127, 0.8872240781784058, -1.7820305824279785, -0.2675679326057434, 1.1583622694015503, -1.4427108764648438, -0.7278348803520203, 0.5772866010665894, 0.023861393332481384, 1.7478212118148804, 0.44513189792633057, -0.21887199580669403, 0.15011870861053467], [-1.301672339439392, 0.14932064712047577, -0.9404690861701965, 0.24445289373397827, -1.649659276008606, -0.2898550033569336, -0.8671791553497314, 0.3879176378250122, -0.47182658314704895, 1.0379658937454224, 1.9427210092544556, 1.2418049573898315, -0.05617663636803627, -0.140549898147583, -0.671893298625946, 0.3476027548313141, -2.017683744430542, 0.7531418800354004, -0.7972906827926636, 0.3662647604942322, -0.8045709133148193, -0.04446876049041748, -0.624505341053009, -0.7570108771324158, -0.8398178815841675, 1.165709137916565, -1.3231011629104614, -1.5951907634735107, -0.04565209895372391, -0.7840906381607056, -1.3176896572113037, -0.6518064141273499, 2.388700008392334, -1.2295829057693481, -0.8898847103118896, -0.16435810923576355, 0.2562105357646942, 1.1915510892868042, 0.7919047474861145, -0.6321582794189453, -0.3490956127643585, 0.06701988726854324, -0.41789451241493225, 0.9337162971496582, -0.30958443880081177, -1.2026880979537964, 1.229273796081543, 0.3594498336315155, 0.3725423812866211, -0.6538885235786438, 0.7844676971435547, 0.19890040159225464, 0.3289487957954407, -0.7690378427505493, 0.4343818128108978, -0.4399888515472412, 0.9072892069816589, -0.26337477564811707, -0.6069450974464417, 0.550134539604187, 0.23241344094276428, -0.2659035325050354, -0.9491930603981018, -1.4785921573638916, -0.23314277827739716, 0.4221183657646179, 1.188788652420044, 0.8776556849479675, 1.4193459749221802, 2.231351613998413, 0.3315415680408478, -0.3019682466983795, -0.44704002141952515, -1.590297818183899, 1.033893346786499, -0.33170267939567566, -0.3808194398880005, -0.340656578540802, 0.1670701503753662, -0.9221293330192566, -0.5215407013893127, -1.280374526977539, -1.387912392616272, 0.11911296099424362, 0.18081972002983093, 0.19167156517505646, 0.8782790899276733, -1.644512414932251, 0.19748179614543915, 0.21482522785663605, 1.8631500005722046, 1.1553500890731812, -0.16255921125411987, 0.07401929795742035, -1.8366881608963013, -1.0477550029754639, -0.6047849059104919, -1.3691296577453613, 0.06761067360639572, 0.02528565563261509, -2.504807710647583, -0.26834017038345337, -0.4462790787220001, -0.1868833601474762, -0.8773103356361389, -0.9374343752861023, -1.141434669494629, 1.7303982973098755, 1.367841362953186, -0.6260654926300049, -0.4724244177341461, 0.37477976083755493, 1.4683315753936768, 0.6432086229324341, -0.2552781403064728, -0.797870934009552, -0.5834890007972717, 0.14630600810050964, -1.0888264179229736, -0.28107956051826477, 0.2040061354637146, 0.11009863764047623, 0.05920456349849701, -0.25100448727607727, -1.7749000787734985, -1.7941945791244507, 0.06952358037233353, 0.5237084031105042, 2.447702646255493], [0.7130782604217529, -0.6845256090164185, -1.7366867065429688, 1.8934413194656372, -1.4329959154129028, -0.1782374233007431, -0.05364562198519707, 0.36349523067474365, 0.8594450950622559, 1.4395829439163208, -0.22606444358825684, 0.5456914901733398, -0.4864087700843811, 0.08965001255273819, -0.6385812163352966, -1.6524404287338257, -0.12122388184070587, 0.028606686741113663, -1.4130579233169556, -0.7264292240142822, -1.6601321697235107, -0.2825944423675537, 2.2089648246765137, -0.3919156789779663, -0.1408502161502838, -1.571408987045288, -0.4026804566383362, 1.8502601385116577, 1.1592174768447876, 0.11377077549695969, 0.114385224878788, 0.8511975407600403, 1.0253238677978516, -0.2899135649204254, -0.8311982154846191, 0.1611371636390686, -0.30319446325302124, 2.169309139251709, -0.41729140281677246, 1.6007368564605713, -1.4976754188537598, -0.7583619952201843, -0.30824753642082214, -1.3483471870422363, 0.7111027836799622, 1.4078774452209473, 0.2585970461368561, 0.2838539481163025, -1.0041426420211792, 1.0786908864974976, -0.3773728013038635, 0.5304610133171082, -0.6979429721832275, 0.43775415420532227, -0.28945687413215637, -0.3154696226119995, 0.15270179510116577, -0.6565576791763306, 0.09424177557229996, -1.9623833894729614, -1.7544127702713013, -0.1059262678027153, 1.5848886966705322, 0.006246983539313078, -1.0373101234436035, 0.7210187315940857, -0.5753028988838196, -0.04033257067203522, 1.2970099449157715, -0.5827255845069885, 0.14321239292621613, -0.6137610673904419, -0.6965033411979675, 1.191113829612732, -1.5955970287322998, 1.0360149145126343, -0.4509787857532501, -1.0896998643875122, 1.9608476161956787, 1.0176013708114624, 0.6692176461219788, -0.7529458999633789, -0.2309037744998932, -0.09192756563425064, 0.14251020550727844, -0.8691270351409912, -0.7662375569343567, -0.7123101949691772, 0.3851267099380493, -0.42979925870895386, 1.228897213935852, 1.3760545253753662, -0.7112000584602356, -1.0294160842895508, 0.5523470640182495, 0.41144874691963196, -0.18221187591552734, -0.5341399312019348, 1.0675339698791504, -1.117549180984497, -1.5256179571151733, -0.016606194898486137, 0.04796477034687996, -0.18315650522708893, -0.828482985496521, -0.2100544273853302, -0.8333982229232788, 1.0437448024749756, 1.6314696073532104, 0.2713549733161926, -0.04458891227841377, 0.21979451179504395, -0.2696090340614319, -0.9218744039535522, -0.451286256313324, -0.7745572328567505, -0.38521096110343933, -0.01586250402033329, 0.21649494767189026, -1.3320488929748535, -0.5910149812698364, 0.07903135567903519, 0.539356529712677, -0.053844355046749115, 0.11130347102880478, 1.3276798725128174, -1.077800989151001, -1.2840052843093872, 0.014703445136547089], [1.369757056236267, 0.1065736934542656, -0.4098327159881592, 1.0452733039855957, 0.36280906200408936, -0.15285971760749817, 0.09482864290475845, -0.18058814108371735, -0.06432802230119705, -1.0257115364074707, 1.1482596397399902, -0.06392782926559448, 1.3222758769989014, 0.6684540510177612, -0.9180508255958557, 0.26027965545654297, -0.5055239200592041, -0.6531910300254822, 1.0810209512710571, -1.3331687450408936, -1.7691287994384766, -1.1864383220672607, 0.5330285429954529, -1.7277601957321167, 0.27348053455352783, 0.11520232260227203, 0.7639697790145874, -0.8728135228157043, 0.1983719915151596, 0.055392004549503326, -2.0655324459075928, -0.710277795791626, 1.4715969562530518, 0.6703161597251892, -0.2648114562034607, 0.213515043258667, 0.02225044183433056, -1.1515954732894897, -0.9080381393432617, -0.785519003868103, -1.0904486179351807, 0.10057094693183899, -0.7975181341171265, 0.6303305625915527, 0.9920854568481445, 2.4448740482330322, -0.9460247755050659, 0.5657926201820374, -1.189774751663208, -0.7141753435134888, 0.3066956400871277, -0.2058963179588318, -1.7942558526992798, -0.4924532473087311, -1.2317410707473755, -1.2710427045822144, -0.6070171594619751, -0.300471693277359, -1.2713496685028076, 0.024228202179074287, -0.9304046034812927, 0.6870874166488647, 1.1816372871398926, 0.7875921726226807, 0.3148185610771179, 0.8702394366264343, 1.264836072921753, -0.5768157243728638, -0.010984434746205807, -0.03092130459845066, 0.554768979549408, 0.11997076869010925, -0.7502819895744324, -0.598004937171936, -0.9527444243431091, 0.43441465497016907, -0.5408282279968262, -0.2713758051395416, 1.1170223951339722, 0.9667342901229858, 0.8622428178787231, -0.21564587950706482, 0.4331895112991333, 0.48689988255500793, -1.5332632064819336, -0.009574889205396175, -0.38646411895751953, -1.2115345001220703, 1.1106823682785034, -1.0595097541809082, -0.36009377241134644, 0.3573050796985626, -0.15744635462760925, -0.29989340901374817, -1.779335856437683, -1.7744884490966797, 0.7048795223236084, 0.10171712934970856, -1.1234418153762817, -0.5742084383964539, 1.0414631366729736, 0.5413150191307068, -0.4837616980075836, -0.8842030763626099, -1.109038233757019, -0.4416901171207428, -0.8217303156852722, 2.805473566055298, -0.686281681060791, 0.6079956293106079, -0.37423500418663025, -0.5524334907531738, -0.6725284457206726, 0.31517812609672546, -0.07534731924533844, -1.446121335029602, 0.11700980365276337, 0.44431963562965393, 0.9873089790344238, 1.4351783990859985, -0.34160059690475464, -0.8108741641044617, 1.1359566450119019, -0.7922976016998291, 2.484714984893799, -0.8258477449417114, -1.2952992916107178, 1.665402889251709, 0.515207827091217], [-0.8001217246055603, 0.6013746857643127, 0.18865138292312622, 0.32105758786201477, 2.1964573860168457, 0.5604404211044312, -0.6506814956665039, 0.8520073294639587, -1.4518461227416992, 1.925469160079956, 0.5173826217651367, -1.7333447933197021, -0.3162311613559723, 1.1722265481948853, -0.6105757355690002, -0.3620086908340454, 2.135497808456421, 2.0230181217193604, 0.5948358774185181, -0.7250896692276001, -0.6541916131973267, -1.3258874416351318, -0.5019116401672363, -0.6935853362083435, -0.3779143691062927, -0.5975324511528015, -0.09975156933069229, -1.269959568977356, 0.6751485466957092, -1.8414431810379028, -0.44795897603034973, 0.4044613838195801, -0.6041414141654968, -0.41747650504112244, 0.43365269899368286, -0.8184435367584229, 0.06219657510519028, -1.3446378707885742, -0.6918803453445435, -0.24016064405441284, -0.18230892717838287, -0.6312815546989441, 1.263027548789978, 1.0670357942581177, -1.5192813873291016, -1.0479086637496948, -0.3361559212207794, 0.9326308965682983, -1.6716047525405884, -0.2854805588722229, 0.4522917866706848, 0.8234028816223145, -0.18596087396144867, -1.4921797513961792, -0.4107402563095093, 1.2020597457885742, 0.32063648104667664, 1.1691619157791138, 0.2510567605495453, 0.6924169659614563, 0.38056549429893494, 0.9448094964027405, 1.5133310556411743, -0.6770889759063721, -0.006762884557247162, 1.279944896697998, 0.42485669255256653, -0.4399203956127167, -0.07125275582075119, -1.3599588871002197, 0.9131177663803101, 0.45492491126060486, 0.4253048002719879, 0.518679141998291, -1.528573751449585, 0.8149151802062988, -0.968654453754425, -0.8736196756362915, -1.8111391067504883, 0.967215895652771, -0.030508574098348618, -0.031018689274787903, 0.36288607120513916, 1.6489300727844238, 0.4064018726348877, 1.2871021032333374, -1.4390521049499512, 0.5820181369781494, -0.26473674178123474, -1.1663144826889038, 0.7369511723518372, 0.047316428273916245, -0.7584483027458191, -0.6180660724639893, 0.11150842159986496, -0.683921754360199, 1.8346731662750244, 0.10911509394645691, -0.4477105438709259, 1.3307193517684937, -0.7885975241661072, -0.6921687722206116, -0.08676595240831375, 0.052275918424129486, 0.6894273161888123, -0.3952181041240692, -0.14717863500118256, -1.0889207124710083, -0.28592658042907715, -0.3605787754058838, 2.2571613788604736, 0.8265854716300964, -0.07928776741027832, 0.876438558101654, -1.827606201171875, -0.5860466361045837, 0.310043603181839, -0.6651360392570496, -0.6575779318809509, 1.306235909461975, -0.9920311570167542, -0.41711559891700745, 0.40889400243759155, -0.7693403959274292, -1.2784355878829956, -0.3693465292453766, -0.11507600545883179, -1.8163949251174927, 0.2367798537015915], [0.29824697971343994, 0.28572651743888855, -0.11228562146425247, -1.2488600015640259, -1.5952484607696533, -1.0127248764038086, -0.19396263360977173, 0.8780150413513184, 1.1999218463897705, -0.4797365963459015, -1.2351808547973633, -0.2903529107570648, 0.25352537631988525, -0.421822190284729, 0.6822831630706787, 1.2191779613494873, 0.2536219358444214, 1.5255160331726074, 0.19007568061351776, -0.2306702435016632, -1.2061234712600708, -0.8865719437599182, 1.0530747175216675, 0.8670313954353333, 0.17566601932048798, 1.5822677612304688, 0.14998158812522888, 1.394684910774231, 0.2375199943780899, -0.2779324948787689, 0.1730981022119522, -0.7803915143013, -1.4170585870742798, 0.20439155399799347, -0.027569055557250977, 0.40415969491004944, 0.16342929005622864, 1.8369635343551636, -0.42371413111686707, -0.25075459480285645, -0.006487173959612846, -0.610873281955719, -1.0196911096572876, 1.051666259765625, -1.2450015544891357, 0.6628016233444214, -0.11514912545681, -1.6858820915222168, 0.6420746445655823, 0.7766566276550293, -2.0575923919677734, 0.10733307898044586, 0.5641323328018188, 0.033398665487766266, -0.22906933724880219, 0.5552466511726379, 0.884428083896637, -0.28471654653549194, -2.746962070465088, 1.0689442157745361, -0.46467891335487366, -0.3834906816482544, -0.028381623327732086, -0.5601160526275635, 0.06800337135791779, -0.07285590469837189, 0.45851993560791016, -0.48731136322021484, -0.041497159749269485, -2.5213115215301514, 2.0556089878082275, -0.2506752908229828, -3.3931593894958496, 0.530561625957489, -0.1973959058523178, 0.7713024616241455, 0.12382481247186661, -0.5723333358764648, -0.14248502254486084, 0.3690323531627655, 0.5203424096107483, -1.399998664855957, -0.10044309496879578, -0.5487202405929565, 0.07148725539445877, 0.07710045576095581, 1.1245445013046265, -0.9706827402114868, -1.1488099098205566, 0.3717488646507263, 0.8164516091346741, -1.5639653205871582, -1.615053653717041, 0.25291410088539124, -0.8995064496994019, 1.9833176136016846, 0.46733883023262024, -0.7397693395614624, -0.7566078305244446, -0.659919798374176, 0.0828782320022583, -0.7325738668441772, 1.9311243295669556, 2.30875825881958, -1.1224483251571655, 0.90355384349823, -0.5351952910423279, -0.9011455178260803, 0.18401814997196198, -0.3138420581817627, -1.3025201559066772, 1.1728112697601318, 0.9605238437652588, -1.468147873878479, 0.2795141935348511, -1.7253221273422241, 1.000542163848877, 0.2035161256790161, -0.7305962443351746, 1.403298020362854, -0.16662803292274475, 0.5311597585678101, 0.0801183432340622, 0.2666414976119995, -0.2307630181312561, 0.09808287769556046, -2.028566360473633, 0.4389064908027649, -1.6691275835037231], [0.7623195052146912, 0.22723273932933807, -0.3994498550891876, -0.5393279790878296, -0.5025716423988342, 1.2499265670776367, 1.6031562089920044, -0.4306279420852661, -0.3911513090133667, -2.004948616027832, -0.22656840085983276, -0.4438745975494385, 1.2866326570510864, -0.4123166501522064, 0.48663339018821716, -0.37227076292037964, -0.761233925819397, 2.333754301071167, 1.9012669324874878, -0.8513789176940918, 0.7101256251335144, -1.3004058599472046, -0.3963949978351593, -0.8079048991203308, 1.172102689743042, -0.825411319732666, 0.368063747882843, -0.7911449670791626, 0.7461075782775879, -0.567413330078125, 1.4482765197753906, -2.4769365787506104, 0.5791009068489075, 2.3744118213653564, 1.3093109130859375, -0.8751815557479858, 1.484025001525879, 0.9519133567810059, 0.8998486399650574, 0.49447959661483765, -0.6806662678718567, -1.9276506900787354, -0.8040299415588379, -0.05302111804485321, 0.4143953025341034, 0.7364214062690735, 1.4435031414031982, -0.5220410823822021, -0.38197794556617737, -0.8529009819030762, 0.40891435742378235, -0.01817878894507885, -1.0291321277618408, 1.567905306816101, 0.03692612051963806, -1.0806964635849, 0.03163371980190277, -1.2136718034744263, -1.2108808755874634, 0.2082621455192566, -0.3502425253391266, 0.21712875366210938, 0.2961331307888031, 0.4837735593318939, 0.8209105730056763, 0.45984840393066406, -0.9644718170166016, -1.9321424961090088, 1.1803417205810547, 0.7257165908813477, 0.5365777611732483, -0.06064877659082413, -0.3831460475921631, -0.019661588594317436, -1.1551696062088013, -0.07412474602460861, -1.8026233911514282, 0.4053612947463989, -1.9215072393417358, 0.3047102689743042, -3.3038642406463623, -1.1961694955825806, -0.7874947786331177, -0.8774352669715881, -0.25111374258995056, -0.7454002499580383, -1.2034454345703125, 1.1706465482711792, 0.6276370286941528, -1.696540117263794, 1.2243047952651978, 0.22275890409946442, -0.423930287361145, 0.17734362185001373, 1.0927551984786987, -1.2670868635177612, -1.2133604288101196, -2.4463720321655273, -0.7400827407836914, 0.8298287391662598, -0.6422697901725769, 0.7297728657722473, 1.4310128688812256, 0.702060341835022, -0.7787867188453674, 1.5728209018707275, -0.7218993306159973, 0.8099860548973083, -0.4738180637359619, -1.019644021987915, -0.5876402854919434, -0.08921597898006439, -0.6947497725486755, 1.9862682819366455, -0.3523491322994232, -1.3464637994766235, 0.26754745841026306, 1.955772876739502, -0.2692287862300873, -0.7698643207550049, -0.9666921496391296, 1.7885030508041382, -0.10761840641498566, -0.597215473651886, 2.632316827774048, 0.9540337324142456, -0.2943178415298462, -0.0482967309653759, 0.7624789476394653], [-0.3296830356121063, 0.6279439330101013, 0.4692729711532593, -0.36441025137901306, -0.5400164127349854, 0.3983646333217621, 0.3341868817806244, 1.2606945037841797, -1.910003423690796, 0.6175252199172974, -0.2857978045940399, 1.9062100648880005, 1.5306490659713745, 0.24515536427497864, 0.15323412418365479, -0.17179249227046967, 0.29259181022644043, 0.6355875134468079, 1.1634637117385864, 0.05961794778704643, 1.4852631092071533, 0.25331178307533264, -1.0016698837280273, -1.7906336784362793, 0.5836141109466553, -1.377455472946167, -1.2302910089492798, -0.06222864240407944, 1.5128285884857178, 2.401092052459717, 1.070705771446228, 0.36261647939682007, 1.247588038444519, 0.4215761125087738, -2.090933322906494, -0.2506234645843506, 0.8505029678344727, -0.5639119148254395, 0.9028513431549072, -0.1682446002960205, -0.029744993895292282, 1.0941365957260132, -0.3127526342868805, -0.4749494791030884, -0.8263428211212158, -0.5682666301727295, -1.9186638593673706, 1.9087486267089844, 0.08001219481229782, 0.8520345687866211, 1.4659321308135986, 1.7566123008728027, 0.7217516303062439, -1.7545139789581299, 0.45511990785598755, -0.43068501353263855, -0.9580283164978027, 1.4771422147750854, -0.7917810678482056, -1.5118824243545532, -1.446150541305542, 0.5124967098236084, 0.14525938034057617, -2.4555721282958984, 0.8951348066329956, 2.08620023727417, 2.4651222229003906, 1.7480545043945312, 0.2436932921409607, 0.4216519296169281, 0.7368721961975098, 0.2238183617591858, -0.23870457708835602, 0.36016014218330383, 0.8375144600868225, -1.6609694957733154, 1.724987268447876, -0.1346128284931183, -0.4061258137226105, -1.799370288848877, 1.315161943435669, 0.096038319170475, -0.17439723014831543, 0.2906253933906555, -0.9514226317405701, -0.12311755865812302, -0.009539252147078514, -0.5257370471954346, -0.652301549911499, -0.9838526248931885, 1.307523250579834, 0.17061050236225128, -0.10190941393375397, -0.34167999029159546, 0.2366115152835846, 0.6432022452354431, -0.12988898158073425, -0.16428197920322418, -0.11019138246774673, 0.6441858410835266, -0.735267162322998, -1.412353277206421, -1.136223316192627, -0.027570223435759544, 0.15867231786251068, -0.8057207465171814, 0.3814260959625244, 0.6820454597473145, -1.7299059629440308, 1.1570097208023071, -0.21264076232910156, -1.1213960647583008, -1.2690377235412598, -0.18800438940525055, 0.6564448475837708, 1.8346260786056519, 0.4674323499202728, -0.8362311720848083, 0.24839067459106445, -0.23623226583003998, -0.040005870163440704, -0.3144407272338867, 0.3573070168495178, 1.1568156480789185, 0.6657966375350952, -1.1349060535430908, -0.03403853252530098, -0.41496288776397705, 1.5557061433792114], [0.014247852377593517, -0.9944905638694763, -0.007672493811696768, -0.4023524224758148, -1.5132282972335815, 0.8535393476486206, -1.0869824886322021, -0.0882187932729721, -1.1902799606323242, -0.9934572577476501, 0.2507410943508148, 0.43166661262512207, -0.29529958963394165, 1.629235863685608, -2.652914047241211, 1.4231122732162476, -0.08063390851020813, 1.1135940551757812, 0.45263195037841797, 0.9230268001556396, 0.9035080671310425, -0.189913809299469, 0.10580702871084213, 0.6931226849555969, -2.3768398761749268, -0.18915261328220367, 0.6694732308387756, -0.10081528127193451, 2.350571632385254, 0.04579756781458855, -0.09671410173177719, -0.9647320508956909, -0.19744987785816193, -1.3351709842681885, -0.23761826753616333, 0.887357771396637, 0.13476327061653137, 0.2599640190601349, 0.4319877624511719, -2.2595176696777344, 0.1935936063528061, -0.3212384581565857, -0.589093804359436, 1.2870101928710938, 0.3229479491710663, -0.47924214601516724, -1.3759734630584717, 0.743093729019165, 0.20064766705036163, 1.1502526998519897, 1.264824390411377, 2.2938649654388428, 0.6717896461486816, -1.2467304468154907, 0.2931309640407562, -0.009494496509432793, 0.5047909617424011, -1.1701611280441284, 0.664338231086731, 0.06240842863917351, 1.095961570739746, -0.8336777687072754, -0.10476760566234589, -0.4189753532409668, 1.5288726091384888, 1.2007731199264526, 0.2179195135831833, -1.3019402027130127, -1.6262961626052856, 0.4325033724308014, 0.6930717825889587, -0.4845649302005768, 0.857096791267395, -1.8718565702438354, -0.10484002530574799, -0.5930807590484619, 2.7689173221588135, 1.003225564956665, 0.583046555519104, 1.3534313440322876, -0.15667584538459778, 0.1877715289592743, -0.47904253005981445, 1.1490976810455322, -0.8286935091018677, 0.9198687672615051, 2.1744349002838135, -2.97294020652771, 0.7031789422035217, -1.4824082851409912, 0.5714190006256104, -1.1810522079467773, 1.3578178882598877, 0.5469920635223389, -1.3266255855560303, 1.611043095588684, 0.5467950701713562, -0.5593314170837402, -1.2881124019622803, -0.2872088551521301, -0.06041599437594414, 0.18342408537864685, -1.2832578420639038, 2.5684103965759277, 0.23378299176692963, -1.9127074480056763, -0.44235557317733765, 0.7553678154945374, 0.15657779574394226, -1.344749927520752, -1.0306456089019775, 1.467465877532959, -0.3331747055053711, 0.7276095747947693, -1.0644582509994507, -0.4253727197647095, 0.5023592710494995, -1.3769744634628296, -0.09158708155155182, 0.03292108699679375, -1.342803716659546, -0.04647207260131836, 0.15780247747898102, 0.36785441637039185, 0.3381020724773407, -1.3518210649490356, 0.4394339323043823, -1.2104421854019165, 0.6934071183204651], [-1.6645969152450562, -0.048644937574863434, 0.6204683780670166, -0.9143546223640442, -0.693679928779602, -0.24301154911518097, 1.625734806060791, -1.872384786605835, -0.15485508739948273, -0.7866097688674927, -0.15011976659297943, 1.8934638500213623, -0.39660194516181946, -0.6888563632965088, 0.33829426765441895, 0.7150722742080688, 1.2658778429031372, -0.24908335506916046, -0.5088875889778137, 0.0747937262058258, -0.7246989607810974, -0.7942342758178711, 2.4076454639434814, 0.13902097940444946, 1.1020822525024414, 1.1316423416137695, 1.6134395599365234, -0.29673123359680176, -0.5443715453147888, 0.15442584455013275, 0.02533516101539135, -2.0866518020629883, 1.8888739347457886, 0.6972991824150085, -0.040458034723997116, -1.785013198852539, -0.23121511936187744, 0.3201615512371063, -0.7472419142723083, -0.206486776471138, -1.1350549459457397, 1.443881869316101, 1.033314824104309, 1.5271575450897217, 1.8460955619812012, 0.7219544649124146, 0.6947157382965088, -2.371638059616089, -1.2276486158370972, 0.5162997245788574, 0.5798017978668213, -1.0834919214248657, -2.8734488487243652, -0.11304714530706406, -1.122814416885376, -0.9450224041938782, -0.6627299189567566, -0.04338230565190315, 1.142602562904358, 1.5760514736175537, 0.7243173718452454, -0.40290939807891846, 0.8840721249580383, -0.7659140229225159, -0.8041131496429443, -1.7231943607330322, -0.3765079379081726, -1.993951439857483, 0.0793805941939354, 0.5904208421707153, 0.15918943285942078, -2.1931886672973633, -0.7838369607925415, -0.12970319390296936, 3.2464592456817627, -0.5102687478065491, 0.11620596796274185, -0.5545396208763123, -1.4596047401428223, -2.2394678592681885, 0.13381455838680267, -1.8087818622589111, 0.3106617033481598, -0.7274786233901978, -0.38130319118499756, -0.06060541793704033, -1.4010132551193237, -1.0863711833953857, 0.01598549261689186, -0.05260308086872101, -0.5763285756111145, 0.9386994242668152, 1.3099987506866455, 0.6467798948287964, 0.005601020529866219, 1.02651846408844, 1.9875985383987427, 0.1865086555480957, -0.8125295639038086, -1.411792278289795, -0.027937810868024826, -0.43095701932907104, 0.05608855187892914, -0.16132394969463348, -0.7007196545600891, 0.5716342329978943, 0.8398621678352356, 0.060536373406648636, 0.48142242431640625, 0.3398541808128357, -0.022828374058008194, -0.888599157333374, -0.2975665032863617, -0.2308400422334671, -1.1612998247146606, -3.3001863956451416, 0.6733620762825012, 0.5743618011474609, 0.13344566524028778, -1.4575562477111816, 0.2498759925365448, -0.5811436176300049, -0.15854330360889435, 0.4319377839565277, 0.6147887110710144, -0.48815304040908813, 0.32114648818969727, -0.5167602896690369, 0.05630099028348923], [0.13585585355758667, 0.2395005077123642, -1.2144545316696167, 2.4632139205932617, -0.554438591003418, -0.04913200065493584, 1.450583815574646, -0.6953415274620056, -0.16684889793395996, -0.2797054350376129, -2.133502960205078, 0.3853284418582916, -0.30069929361343384, 0.5009955167770386, 0.9884171485900879, -0.31926068663597107, -0.2958059310913086, -0.6147943139076233, 1.5512768030166626, 0.7624454498291016, 0.6804732084274292, 0.9542672038078308, -0.8180298805236816, -0.6562279462814331, 1.7744592428207397, 0.040600720793008804, 0.7512370347976685, -0.998860239982605, -0.04556742310523987, 0.6844290494918823, 0.731395959854126, 0.34625735878944397, -0.4509597718715668, 1.2167432308197021, -0.6912480592727661, -0.4249633550643921, 0.007422678638249636, -1.6969623565673828, -0.3533141613006592, -1.3234163522720337, 0.44763433933258057, -1.3394018411636353, -0.029615530744194984, -0.4871726632118225, 0.7230433821678162, 0.11259789764881134, -0.5200151205062866, -0.22143562138080597, 0.03714631870388985, 1.132685661315918, 0.45221519470214844, 0.3433733284473419, 0.726973295211792, -3.716157516464591e-05, -0.5488241910934448, -1.0962365865707397, -0.47999662160873413, -0.2532561421394348, 0.5669938921928406, 1.5080686807632446, -2.3982057571411133, 1.7861902713775635, -1.2971904277801514, 0.7276809215545654, -0.8370646238327026, 0.33706900477409363, -1.017870545387268, 0.6437268853187561, -0.4413388669490814, -0.4525962471961975, -0.47720348834991455, -0.8480396866798401, 1.1324576139450073, 1.0019360780715942, -0.1289134919643402, -0.056913360953330994, -2.061552047729492, -1.1139419078826904, -2.1038177013397217, 0.036279965192079544, 1.692792296409607, -2.5684192180633545, -0.019475948065519333, -0.09632983058691025, 0.6833130121231079, 1.10178804397583, -1.118788719177246, 0.8438253402709961, -1.1561415195465088, 1.1023530960083008, -1.1659220457077026, 0.8003472089767456, 0.6087542772293091, 0.4371946156024933, 0.6468528509140015, 0.406377911567688, 0.6753526329994202, 0.485273540019989, -0.1840168684720993, 0.014433520846068859, -0.6405252814292908, 1.991683006286621, -1.0628677606582642, 0.7287209630012512, -0.4098178446292877, 0.5798444747924805, -0.5568227171897888, -1.0630416870117188, -0.010610422119498253, 0.9386958479881287, -1.0622825622558594, 0.23563824594020844, 1.0638670921325684, 0.2003450095653534, -0.8819852471351624, -0.4200410842895508, -1.3569937944412231, -1.5433571338653564, 0.002172218170017004, 1.3003606796264648, -1.1707549095153809, -1.053789496421814, 1.4760102033615112, 0.5132893919944763, -0.8769528865814209, -0.12320809811353683, -0.05983235687017441, -1.105657935142517, -2.1182003021240234], [0.19305036962032318, 1.3555166721343994, 0.2943292260169983, 0.3229537010192871, 0.6276746392250061, -0.45234885811805725, -0.2498873919248581, 0.6992440819740295, 0.7990017533302307, 0.449940025806427, -2.1018424034118652, 1.488457202911377, -0.23633278906345367, 1.4577573537826538, 0.796345591545105, -0.24276533722877502, 0.760174572467804, 0.29060181975364685, -0.8305038809776306, -0.03932693973183632, 0.4768507778644562, 1.9488734006881714, -0.052658241242170334, -0.9154788255691528, 0.19980065524578094, -1.0574557781219482, -0.4647720456123352, 0.08052777498960495, 0.12451934069395065, -2.4197990894317627, -1.4504880905151367, -1.7593858242034912, -0.662811815738678, -0.8671013116836548, 0.6504665613174438, -1.2495120763778687, 0.005632214248180389, -0.1794266253709793, -0.03739312291145325, 0.5549812912940979, 0.8650504350662231, 0.7854498624801636, -1.455074429512024, 0.5897484421730042, -0.04592817276716232, -0.5727880001068115, -0.9123765826225281, -0.12389880418777466, 0.3646031320095062, -1.1795289516448975, -0.32879558205604553, 0.3280307948589325, 1.4047048091888428, -1.784971833229065, -0.6465684175491333, -0.651715874671936, -0.9504516124725342, 0.05460049584507942, -1.2999540567398071, -1.1673394441604614, -1.9776753187179565, 1.7183935642242432, 0.4081404209136963, 0.6490635275840759, 0.3589962124824524, -0.27753448486328125, -0.6440848112106323, -1.606136441230774, -1.4037346839904785, 0.5280399322509766, 1.4464229345321655, -1.3572983741760254, -0.32497677206993103, -0.6141614317893982, 0.31618577241897583, 0.2989945113658905, 0.7437135577201843, -0.1446024775505066, -0.05047450587153435, -0.6586088538169861, 0.9395825862884521, 0.8734310269355774, -0.7556608319282532, 1.0013335943222046, 0.40496259927749634, -0.5757967233657837, -0.34385353326797485, -0.10496417433023453, 0.626822292804718, 1.0450060367584229, -0.29624494910240173, 1.2762258052825928, 1.4978317022323608, -0.0248321034014225, 1.405370831489563, -1.1196262836456299, 1.6513981819152832, -2.0595109462738037, -0.542952299118042, -0.8215901255607605, 2.329418182373047, 0.23482388257980347, -0.19986365735530853, 0.32707518339157104, -2.113837242126465, -1.695392370223999, 1.7424184083938599, 0.37392088770866394, -0.5413538813591003, 1.3714088201522827, -0.7697712779045105, 1.3982218503952026, -0.6025589108467102, 0.11479318886995316, 1.258926510810852, -2.4642248153686523, 1.1812739372253418, -0.4602048993110657, -0.9309571385383606, -0.8936147093772888, 0.36952415108680725, -0.7750868797302246, 0.3112468123435974, 0.6101108193397522, 1.7833242416381836, -1.9133615493774414, 1.297163963317871, -0.8163713812828064, 0.9714208841323853], [-0.6832907795906067, -0.7077123522758484, -0.5163106322288513, 0.2590884566307068, -0.8099103569984436, -0.6395093202590942, 1.6436090469360352, -0.640714704990387, 0.07312265038490295, 0.6875812411308289, 0.5257379412651062, -0.8313493132591248, 1.4140814542770386, -0.850832998752594, -1.7168331146240234, -1.1472477912902832, -0.9917911291122437, -0.42781931161880493, -0.5999017953872681, -0.07870406657457352, 1.4268814325332642, 1.9276039600372314, -1.2120062112808228, 0.8200744390487671, -1.3153889179229736, -0.6030541658401489, -0.12865237891674042, -0.8328067064285278, -2.1749284267425537, -0.42605239152908325, -1.494489073753357, -1.5992116928100586, -1.3367832899093628, -0.9851500988006592, -2.75117826461792, 0.7371849417686462, 0.16161660850048065, 0.049690183252096176, 1.810077428817749, -0.37342020869255066, -0.20475682616233826, -0.313393235206604, -0.08464028686285019, -1.0689576864242554, 1.6824194192886353, -0.6842656135559082, -0.6875647306442261, -1.23581862449646, -0.22079281508922577, 0.21485522389411926, -0.10387042909860611, 1.2561204433441162, -0.9425832629203796, -2.171266555786133, 2.239394426345825, -1.452405333518982, -0.8312296867370605, 1.2724353075027466, 0.656586766242981, 0.6198428869247437, 0.4944445788860321, -1.470597743988037, 0.6221620440483093, -0.8057098984718323, -0.5845984816551208, 0.5876258611679077, -0.72710782289505, -0.3043496608734131, -0.7833675146102905, 0.3852120339870453, -1.3021166324615479, -1.4600037336349487, 1.105539321899414, 1.6745003461837769, -1.9796535968780518, -0.16444560885429382, -1.2316374778747559, 0.02895016223192215, 0.6560916900634766, 0.07499171048402786, -1.0417487621307373, -1.2541040182113647, 1.1083308458328247, 0.38063231110572815, -0.10845812410116196, 2.0912652015686035, -1.8539012670516968, -1.8309533596038818, 1.9092199802398682, -0.16041862964630127, 0.18985885381698608, 0.39748436212539673, 0.9784544110298157, -1.5684655904769897, 1.149763584136963, 0.20420804619789124, -1.3802199363708496, -1.7330403327941895, -0.9800503253936768, -0.42081913352012634, 0.28846505284309387, 0.014752838760614395, 1.1555206775665283, -0.7507067918777466, -0.13360579311847687, -1.0250005722045898, -1.3737156391143799, -0.7331132292747498, 0.18217414617538452, 2.074974775314331, 1.1638081073760986, 0.8956668376922607, 1.5991572141647339, 0.12606120109558105, 1.9952261447906494, 0.11546463519334793, -0.04324662312865257, -0.6625521183013916, 0.5359389781951904, -0.0497092604637146, -1.5801777839660645, -2.015916585922241, -0.7044923901557922, 1.2169129848480225, 0.5407759547233582, 1.0721694231033325, 2.1559910774230957, 0.4321939945220947, -0.47328537702560425], [-0.8527644872665405, -0.5774236917495728, 0.28064972162246704, 1.3374159336090088, -0.42432641983032227, 1.3621509075164795, 1.5515857934951782, -0.38894122838974, -0.16632695496082306, -2.4742398262023926, -1.8256210088729858, -1.0810556411743164, -0.2486511766910553, -1.3327176570892334, 0.0830727070569992, 0.27260562777519226, -0.9310891032218933, -1.2436978816986084, -0.2210882157087326, -0.4375137984752655, 0.3605131208896637, 1.724225401878357, 0.6047987341880798, -1.022943377494812, 0.5051106214523315, 0.8322192430496216, 0.4239419996738434, 2.417387008666992, -0.8679813742637634, 0.04569945111870766, -0.8061557412147522, 0.9566349387168884, -1.020951271057129, -0.616340696811676, -2.722309112548828, 0.05324659124016762, -0.5538166761398315, -0.026846064254641533, -1.6995429992675781, 3.1344900131225586, 0.15237532556056976, -0.9725955724716187, -0.29348650574684143, -0.011208745650947094, 0.17838728427886963, -0.7543625235557556, -0.04700355976819992, -0.4110361933708191, -0.18460911512374878, -0.8820403218269348, 1.265572190284729, -0.030100896954536438, -0.4718853235244751, -0.06398297101259232, 0.0322968065738678, 0.6637340784072876, -1.0811831951141357, -0.6040393710136414, -0.024823537096381187, 0.28317031264305115, 0.3686302602291107, 1.2713510990142822, 0.29971885681152344, -0.6599379181861877, 1.2254441976547241, -0.1235484704375267, 0.14606119692325592, 1.177811861038208, 0.7439570426940918, -0.13548417389392853, -0.06501799821853638, -0.7118091583251953, 0.6685733795166016, 0.6731852293014526, 1.7867450714111328, -0.17228734493255615, 0.49192991852760315, -0.5104209780693054, -0.38163867592811584, -1.098159670829773, -1.1116613149642944, -0.9701706767082214, 1.1919505596160889, 1.3261692523956299, -0.6181379556655884, -0.9061732888221741, -0.30410775542259216, -1.1217635869979858, 1.5294065475463867, 0.37378042936325073, -0.9751002788543701, -0.73331218957901, 0.6303531527519226, -1.1705902814865112, -1.307262897491455, -3.750872850418091, -0.12029733508825302, 0.7520315051078796, 0.9991171360015869, -1.3468230962753296, -0.16443432867527008, 1.1249990463256836, 0.871147096157074, 0.37484073638916016, -0.5110661387443542, -0.480291485786438, 1.0147101879119873, 0.961739182472229, 0.5588777661323547, -0.6007282137870789, 0.12260492891073227, 0.6029160618782043, 1.4651243686676025, -0.9683550000190735, 1.4853169918060303, -0.382149875164032, 0.11329306662082672, 0.41447556018829346, -0.6437571048736572, -0.9698081612586975, 0.00024632856366224587, -0.13568857312202454, -0.558765709400177, 0.678049623966217, -0.3651653826236725, 1.2527530193328857, 1.5849690437316895, 2.543461561203003, -0.7381606698036194], [0.6752883195877075, 1.8405048847198486, 0.4399299919605255, 1.7471038103103638, 0.880011796951294, 1.6382317543029785, 0.1040172204375267, -0.28460952639579773, 0.7260022163391113, 0.8034005761146545, 0.04072222858667374, -0.1278149038553238, -0.4132118225097656, -0.16206295788288116, 0.27025604248046875, -0.5125504732131958, 0.7060723304748535, -1.8984239101409912, -0.19527392089366913, 0.018793625757098198, 0.592772901058197, -0.3919067680835724, 0.6785231232643127, 3.295543670654297, -0.5056259036064148, 0.10849709808826447, -0.7853420972824097, 1.022804617881775, 0.1916593611240387, 1.5281109809875488, -0.822065532207489, 0.6084861159324646, -1.8520817756652832, -0.7414342164993286, -1.3315761089324951, -2.2089202404022217, 1.6106913089752197, 1.8172783851623535, 0.9736072421073914, -1.6784268617630005, -0.6585862636566162, -0.8510048985481262, -0.006911218632012606, 1.1256598234176636, -0.9183158874511719, -0.745680570602417, -0.18377012014389038, 0.16900016367435455, -0.1801217943429947, -0.5457432866096497, -0.3840380311012268, -0.5569427609443665, 0.8831233978271484, -0.9489946961402893, 0.3834706246852875, 0.47416332364082336, 0.188446506857872, 1.8038973808288574, -1.152112364768982, -0.6122671365737915, 0.6721136569976807, -0.8991261720657349, 2.107118844985962, -1.0539458990097046, 0.017567122355103493, -0.36598315834999084, 0.9005307555198669, 1.1418544054031372, -0.33736974000930786, -0.3736422657966614, 1.2546813488006592, -0.41870811581611633, 0.14341777563095093, -0.14918109774589539, -0.9465243220329285, -0.799250066280365, -0.5396878123283386, 0.5156240463256836, 0.6563944816589355, -1.0049790143966675, -1.2741585969924927, -1.0769377946853638, 0.22290296852588654, -1.4351038932800293, 0.027656104415655136, -0.33360883593559265, -1.1877155303955078, -1.707222819328308, -0.5447089076042175, -0.5434957146644592, 0.5394837260246277, 0.6345134377479553, -1.3743199110031128, 1.524226427078247, -0.6310167908668518, 0.9399195909500122, 0.5175626873970032, 1.2072142362594604, 1.0015498399734497, -0.8315204977989197, 1.2038466930389404, -0.7286056876182556, 0.20730546116828918, 1.0704073905944824, -1.3851537704467773, -0.13166941702365875, 2.0935120582580566, -0.6240753531455994, 0.019528469070792198, -0.9512993097305298, -0.2511407434940338, -1.410171627998352, 2.0795061588287354, 0.6691396832466125, 0.11949054151773453, -0.638213038444519, -0.47622933983802795, -0.383245050907135, 1.071362018585205, -1.6489742994308472, -0.7361279726028442, -1.2962192296981812, 0.09076324850320816, 1.2428808212280273, -0.622787356376648, 1.2646199464797974, 0.5888786315917969, 0.05303027853369713, -0.2746502459049225], [2.448848009109497, -0.35049521923065186, 0.9193934202194214, 0.4086627662181854, -1.6780846118927002, 1.1707262992858887, 1.9498580694198608, 0.6459589004516602, 1.61185622215271, -0.31177398562431335, -0.6425755023956299, 0.7112753987312317, -0.8943997025489807, 0.47019150853157043, 1.3775311708450317, -0.23045727610588074, 0.13328972458839417, 1.189092993736267, 0.20225776731967926, -0.807030975818634, 0.9643433094024658, 0.2091629058122635, 2.115915298461914, 0.8701276779174805, 0.9378833174705505, -0.5760799050331116, 0.4263981878757477, 0.7942851185798645, 0.26804858446121216, 0.3193875551223755, -1.0707049369812012, 0.3974941670894623, -0.20195063948631287, -0.8251887559890747, -0.3802761733531952, 0.05933641269803047, 2.064783811569214, -0.01484254002571106, 1.3908555507659912, -0.3398193418979645, 0.7350066304206848, -0.6914392113685608, -0.6142752766609192, 0.8159162402153015, 0.7605680823326111, -0.717898964881897, -0.3179575204849243, -0.18266157805919647, -1.913476586341858, 0.6982065439224243, -1.4113227128982544, 0.5309760570526123, -0.14892259240150452, -1.6260019540786743, -1.6559584140777588, 0.785722553730011, -1.1549464464187622, 0.13998976349830627, 0.1482720524072647, 0.02129187434911728, 1.4183545112609863, 0.812723696231842, 0.27261456847190857, -1.2242562770843506, 0.7847900986671448, 0.6396638751029968, -0.307168573141098, -1.607182502746582, -0.956259548664093, -1.4082680940628052, -1.5941872596740723, -1.8230700492858887, 0.09682323038578033, 0.5871837139129639, -0.8930835127830505, 0.3401104509830475, 0.8501421213150024, -1.118241786956787, -1.4697821140289307, 0.6653485298156738, -0.7688371539115906, 1.993956446647644, -0.4953049421310425, 0.3319498896598816, -1.8715556859970093, -0.08604034781455994, 2.5552785396575928, -0.3024861216545105, 0.010568978264927864, 0.29532119631767273, -0.09814941138029099, -0.3258104920387268, 1.3432515859603882, 0.2970026135444641, 1.733106255531311, -0.05431386083364487, 0.3671831488609314, -1.1670397520065308, -1.906273603439331, 0.21192625164985657, -1.049997329711914, 1.3705984354019165, -0.29657456278800964, 2.145106077194214, 1.259602665901184, 1.5792992115020752, -0.3044421672821045, -1.1343978643417358, -0.743531346321106, -0.7411964535713196, -0.7084910869598389, 0.16496588289737701, 0.6612476110458374, 0.7585718035697937, 0.9309810996055603, 0.49023184180259705, -1.0664279460906982, 0.48892852663993835, -1.3072659969329834, 0.9780195355415344, -0.4535203278064728, -0.8346956968307495, 0.9658022522926331, 0.4131326377391815, -0.15358446538448334, -0.008176104165613651, 1.515390157699585, 0.7350251078605652, 0.02127683162689209]], "dtype": "float32"}], "e": {"__nd__": [[5.965022563934326, 14.753453254699707, -9.33409595489502, 21.011951446533203, 25.269134521484375, 10.177948951721191, -29.386863708496094, -14.665610313415527, 3.8899576663970947, -35.116119384765625, -35.79648971557617, 8.495931625366211, -37.78215789794922, 4.385868549346924, -40.99420166015625, 8.241477966308594, -4.946835517883301, -33.96315002441406, -8.287643432617188, 2.1357553005218506, -1.3095650672912598, 14.370140075683594, 23.272510528564453, 14.48029613494873, 41.57703399658203, 12.906758308410645, 6.414305686950684, -8.760579109191895, -32.000675201416016, -2.8315887451171875, 15.69940185546875, 1.7637100219726562, 3.629697561264038, 28.37873649597168, 11.670863151550293, -7.9210076332092285, -24.019697189331055, -1.1455243825912476, 19.408313751220703, 4.840285301208496, -6.972848415374756, -1.5548086166381836, -14.026387214660645, 2.5870859622955322, -8.925469398498535, 19.848526000976562, 8.486614227294922, -4.2617998123168945, 3.321692705154419, 24.55682373046875, 9.733288764953613, -7.207742214202881, -8.812588691711426, 1.9885625839233398, 12.021039962768555, 6.977652549743652, -5.601741313934326, 42.9501838684082, 2.604517936706543, -16.79043197631836, -50.3969612121582, -8.07930850982666, -3.4760897159576416, 12.028915405273438, -4.550256729125977, -0.6519330739974976, 21.285308837890625, -12.262240409851074, -6.038595199584961, 3.6111927032470703, 2.132075786590576, 20.030893325805664, -8.313158988952637, -10.188835144042969, -5.334841251373291, 10.506095886230469, -2.415708303451538, -31.41799545288086, 24.99007797241211, -12.148590087890625, 11.81997013092041, 11.137014389038086, -5.656845569610596, 24.01821517944336, -17.325645446777344, 2.552335739135742, -31.262468338012695, 2.9167838096618652, -11.692609786987305, 25.465763092041016, 6.62447452545166, 6.149730682373047, 23.410890579223633, -2.4577903747558594, -5.806427955627441, -38.5649528503418, 0.3995670974254608, 5.738502025604248, 12.677231788635254, -34.67507553100586, -11.166150093078613, 35.2641716003418, -19.758726119995117, -20.651508331298828, -9.541726112365723, 13.911442756652832, 13.495336532592773, -22.981922149658203, -33.45602798461914, -0.7973881363868713, -9.687654495239258, 20.02389144897461, 1.0225980281829834, -23.939716339111328, -16.253427505493164, -17.072355270385742, 26.301328659057617, -23.41106605529785, 27.068443298339844, -0.7747790813446045, -1.5860707759857178, 29.0775146484375, -26.038986206054688, 2.370863199234009, 4.302059650421143, -34.62615203857422, -20.76375961303711, -3.1939895153045654, 10.52133846282959], [1.9324994087219238, 17.287508010864258, -35.239593505859375, -1.061834692955017, -10.922236442565918, -11.249289512634277, 27.737812042236328, 2.920180082321167, -20.420438766479492, -11.659281730651855, 28.69301414489746, -15.735438346862793, -12.296957015991211, -4.10875129699707, 16.557674407958984, -46.78235626220703, 5.371962547302246, 9.198578834533691, 0.9881969094276428, -4.643924236297607, 1.9448179006576538, 4.43297815322876, -2.9003562927246094, 10.161019325256348, 6.0387749671936035, -26.370697021484375, -36.01748275756836, -11.05278491973877, -9.03592300415039, -21.091833114624023, -7.044819355010986, 20.061771392822266, 3.74601149559021, -1.4999717473983765, 29.81585693359375, 7.3090667724609375, -5.436601161956787, 6.043216705322266, 5.384593963623047, 4.159642696380615, -6.986303329467773, 4.1521992683410645, -16.918336868286133, 25.79338836669922, 29.084510803222656, 6.024879455566406, -19.767444610595703, 24.007843017578125, 10.116683006286621, 7.763800621032715, 36.97332000732422, -1.6708812713623047, -36.6474494934082, -38.00408172607422, 6.754063606262207, 0.11095234006643295, 8.568532943725586, 13.944560050964355, -13.97290325164795, -1.8770391941070557, -1.486818790435791, 8.606001853942871, -7.506353855133057, -39.848228454589844, 11.117768287658691, 12.34399127960205, -6.52030086517334, -10.028573989868164, -18.388216018676758, -25.7900333404541, 16.729955673217773, -11.873044967651367, 13.970072746276855, 5.087723255157471, -1.0245802402496338, 48.40066146850586, -9.77636432647705, -15.289573669433594, -13.954386711120605, -1.931738018989563, -33.05942916870117, 10.385499954223633, -13.393672943115234, 8.38259220123291, 5.551228046417236, 14.529123306274414, 9.980864524841309, -28.461875915527344, -2.902981996536255, -5.601836204528809, -26.74080467224121, -2.5706169605255127, -0.7545226812362671, 6.591200828552246, 9.131159782409668, -11.842745780944824, -31.251333236694336, 0.26977238059043884, 2.6428043842315674, 23.196300506591797, 0.9206519722938538, 20.32912826538086, 15.83763599395752, -4.734497547149658, -13.956966400146484, -8.414079666137695, -5.750197410583496, -16.354278564453125, -10.02875804901123, -15.323663711547852, 7.596815586090088, -6.500640392303467, -13.055967330932617, 9.446189880371094, 18.073095321655273, -6.946335792541504, 23.66672134399414, -13.249140739440918, 6.3937811851501465, -0.23551200330257416, -32.42170715332031, -0.869184136390686, 25.13499641418457, 29.4289493560791, -10.764914512634277, -17.5665283203125, 25.066003799438477, 2.9851367473602295, 7.457615375518799], [-1.3602571487426758, -29.71185874938965, 10.106712341308594, 1.2579439878463745, 11.911744117736816, 1.4107415676116943, 29.795974731445312, -22.325193405151367, 26.921689987182617, -4.602153301239014, 17.82464027404785, -0.38748791813850403, 41.73161697387695, -6.482529163360596, 9.812421798706055, 3.8546369075775146, -7.169492244720459, -6.733279228210449, -36.05950927734375, -7.303217887878418, 37.681636810302734, 12.094964981079102, -1.7510415315628052, -34.642032623291016, 8.357069969177246, 25.86251449584961, 14.92612075805664, -26.16606330871582, -34.63148880004883, 0.48006564378738403, -3.222236156463623, -24.5511417388916, 2.4609780311584473, -28.742216110229492, 5.818698883056641, -26.900930404663086, -5.862970352172852, -4.347097873687744, 14.501435279846191, -25.776723861694336, 16.465049743652344, -29.469079971313477, -8.468931198120117, -9.029929161071777, -25.006986618041992, -1.6829551458358765, -17.700794219970703, -30.98738670349121, 12.477738380432129, -39.34680938720703, 10.395835876464844, 7.688312530517578, 12.783949851989746, 10.057111740112305, -13.843214988708496, -17.257455825805664, 11.893331527709961, -5.691262245178223, 17.39912223815918, 13.935931205749512, -10.110689163208008, 30.306913375854492, -2.4911906719207764, -17.940088272094727, -21.87405014038086, 1.0074790716171265, 17.320146560668945, -11.655290603637695, -5.699337482452393, -18.06207847595215, -26.992389678955078, -42.85380935668945, -3.6588547229766846, 7.870575904846191, -7.105319023132324, -6.4204325675964355, -44.0771369934082, 19.979955673217773, 7.03706693649292, 5.862055778503418, 24.15599250793457, -10.773798942565918, -20.96101951599121, 0.061433762311935425, -16.768831253051758, 5.318121433258057, -12.504411697387695, -17.639142990112305, 27.12818145751953, -2.4951469898223877, -4.566553592681885, 9.641305923461914, -1.1832009553909302, -18.645309448242188, 2.4857494831085205, 13.211984634399414, 15.255319595336914, -15.709681510925293, -48.632774353027344, -7.362728118896484, -6.086137771606445, -28.975887298583984, -5.491453647613525, 5.5811944007873535, -8.774247169494629, 11.925762176513672, -2.092574119567871, -2.656972646713257, -3.5463929176330566, 11.709086418151855, -9.087740898132324, -14.222464561462402, 13.132516860961914, 21.85618782043457, -4.882569313049316, -2.2895195484161377, 11.752445220947266, 19.219886779785156, 18.356361389160156, -5.974075794219971, 36.84453201293945, -33.531471252441406, 33.944820404052734, -19.568613052368164, -31.13275146484375, -20.552831649780273, -4.1730241775512695, 12.85123062133789, -10.081857681274414], [-2.9796457290649414, 25.19710922241211, 18.036420822143555, 6.4503889083862305, -13.427688598632812, -11.177361488342285, -9.166097640991211, 2.0016579627990723, -24.293731689453125, -18.310306549072266, -5.494751930236816, -10.174704551696777, 14.894908905029297, 34.44710922241211, 2.334167957305908, 23.974092483520508, 1.6144473552703857, -14.602693557739258, 9.5626859664917, 7.087547779083252, 14.381487846374512, 10.486620903015137, -7.658386707305908, 9.87502384185791, -19.68492317199707, 14.2539644241333, -4.3568243980407715, 42.04635238647461, -20.056926727294922, -10.234360694885254, -3.825079917907715, 8.831172943115234, -42.58661651611328, 12.505403518676758, 15.754108428955078, -0.8555253744125366, 3.327939987182617, -14.363353729248047, -10.316596984863281, 5.65723991394043, 15.418898582458496, 13.482397079467773, 12.441744804382324, 11.112744331359863, -26.740203857421875, -8.287332534790039, 16.877931594848633, 13.345309257507324, 15.969929695129395, -24.786102294921875, -0.40285608172416687, -15.53037166595459, -2.0900824069976807, -9.078533172607422, -0.6861270070075989, 16.225751876831055, -26.559236526489258, 24.664913177490234, 59.18328094482422, -17.862648010253906, 17.735328674316406, -14.60057544708252, 20.583341598510742, 0.11520658433437347, -1.2718595266342163, -2.744743824005127, -16.882244110107422, -28.10146713256836, -16.2089900970459, -19.94166374206543, -4.999673843383789, 20.739089965820312, 3.9669222831726074, 13.93307876586914, -1.9994609355926514, -39.177642822265625, 4.036069869995117, -23.322284698486328, -8.93977165222168, -45.732486724853516, 7.444687366485596, 20.849878311157227, 6.112667083740234, 28.440027236938477, -7.380735397338867, -12.630226135253906, -24.90289306640625, -40.356834411621094, 2.127647876739502, -1.8842062950134277, -0.5750729441642761, -44.41358184814453, 12.612871170043945, 28.130081176757812, -11.707509994506836, 37.79875564575195, -6.040497303009033, -1.8595070838928223, -10.451563835144043, 20.011505126953125, -8.179620742797852, -7.859896659851074, -12.141327857971191, -5.557493686676025, -6.945404529571533, -17.034976959228516, -19.845212936401367, 27.30712890625, -2.0043909549713135, 7.186913013458252, 20.82966423034668, -17.282014846801758, -2.3404786586761475, 20.564584732055664, 15.651612281799316, -10.774625778198242, -12.025191307067871, -14.412398338317871, -47.782386779785156, -40.27552795410156, 7.895882606506348, -6.349061489105225, 0.44708144664764404, 25.845529556274414, -1.6741214990615845, 12.235618591308594, -1.372849941253662, -16.035907745361328, -29.924396514892578], [-6.003142356872559, -22.25861358642578, 26.757596969604492, 46.97274398803711, 26.779582977294922, 28.608121871948242, 22.317432403564453, -31.398225784301758, -25.68443489074707, 2.055229663848877, -39.40241622924805, 2.2007408142089844, -23.907485961914062, 6.926701068878174, 43.09098815917969, -20.48115348815918, 14.556623458862305, -38.55200958251953, -11.866497039794922, -4.179678916931152, -28.470380783081055, -23.14788246154785, 32.732383728027344, -15.060577392578125, 7.009006977081299, 6.288068771362305, -1.3831573724746704, 20.850444793701172, 15.665410041809082, -13.818618774414062, -10.327606201171875, 17.74388313293457, 26.612468719482422, -13.95296859741211, -14.449113845825195, -36.67003631591797, 7.475584506988525, -21.82729721069336, -4.769711017608643, -16.773170471191406, 12.170886993408203, -6.250421047210693, -14.151071548461914, 7.422837734222412, 4.546515464782715, -23.89422607421875, -54.108367919921875, 10.856474876403809, -8.190566062927246, -26.976430892944336, 30.26317596435547, -6.970365047454834, -14.783751487731934, -25.30292320251465, 11.477398872375488, -14.453630447387695, -3.2441964149475098, 13.424997329711914, -17.94917106628418, -11.868985176086426, -45.640811920166016, -43.8709716796875, -0.4824964702129364, -1.208061695098877, 12.671001434326172, 7.772095203399658, 12.011168479919434, 27.95905876159668, 11.067078590393066, -3.9302496910095215, -27.53465461730957, -14.524652481079102, 4.990228176116943, -1.014019250869751, -7.767427921295166, 19.074596405029297, -5.793243408203125, 4.914365291595459, -4.7879719734191895, -7.446558952331543, -6.051990032196045, -6.408895015716553, 6.480895519256592, -7.9111104011535645, -15.9893159866333, 9.665635108947754, -11.289567947387695, 16.2674617767334, 34.90693664550781, 8.456087112426758, 9.290838241577148, 21.768218994140625, 7.080633163452148, 16.76409149169922, -7.706118583679199, -13.548295974731445, 12.967127799987793, 6.049363136291504, 2.4516360759735107, 38.735469818115234, 27.84490394592285, -12.93413257598877, -38.67210006713867, 12.537620544433594, -10.702333450317383, 4.10923957824707, 19.08458709716797, 6.415045738220215, -9.64635944366455, -4.055855751037598, -5.265160083770752, -6.459402084350586, -0.19610555469989777, -9.721000671386719, 1.3141261339187622, -9.255807876586914, -5.516516208648682, 18.660451889038086, -29.29239273071289, -2.9218125343322754, 38.206905364990234, 12.67608642578125, -7.761556625366211, -13.804936408996582, 8.66370964050293, -2.0536129474639893, 0.7542616724967957, -9.074262619018555, 23.98522186279297]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]], "dtype": "float32"}, {"__nd__": [[1.1614781618118286, -1.168588638305664, 0.6685959100723267, 0.11078453063964844, 0.4656485319137573, -0.027509579434990883, 1.8085906505584717, -0.5511820912361145, -0.25441375374794006, -0.11204741150140762, 0.03570594638586044, 0.08488547056913376, -1.8130311965942383, 0.9036855101585388, -0.17652393877506256, 1.6382287740707397, -0.05251174047589302], [0.5520599484443665, -0.09989111125469208, 0.5731106400489807, -0.7807968854904175, -0.614578366279602, 1.231959581375122, 0.4062361717224121, 0.8830996751785278, 1.2442636489868164, 1.054718017578125, -1.9387959241867065, -0.15408584475517273, 1.054183840751648, -0.5511282086372375, -0.3748876452445984, 0.3601100742816925, -0.3831285834312439], [-0.5112664103507996, -1.7379724979400635, 1.6845417022705078, -0.3898513615131378, 2.945523977279663, -1.045864224433899, 0.24680884182453156, 0.2891887426376343, 0.0024194081779569387, -0.4076298475265503, 0.25311264395713806, -1.4353830814361572, -0.1887165904045105, -0.02244318276643753, -0.5802040696144104, -0.6742333769798279, -0.9921287894248962], [-0.9625059962272644, 0.0908195748925209, -1.255054235458374, 0.3071123957633972, 0.34202098846435547, 1.1500474214553833, 1.3558664321899414, 0.9277841448783875, -1.231643795967102, 0.7370036840438843, -0.21373668313026428, -0.731985330581665, 1.6993502378463745, -1.1459869146347046, 0.8108246922492981, 0.1941567063331604, 0.5165068507194519], [-1.2926125526428223, 0.7165169715881348, -0.39561039209365845, -1.7015711069107056, -3.104464054107666, 0.8925644159317017, -1.1426937580108643, 1.1177655458450317, -0.2948853075504303, 0.06843023002147675, -0.09427052736282349, 0.9387674331665039, 0.14421169459819794, 0.24477317929267883, 0.9423349499702454, 1.886193037033081, -0.9501217603683472], [0.4471232295036316, 0.28228408098220825, 1.006272792816162, -0.7578164935112, -0.0006004875176586211, -0.3487266004085541, -0.19105631113052368, 0.10195944458246231, 1.6157519817352295, 0.8078869581222534, 0.6466975808143616, -1.1457979679107666, -0.2978084981441498, -0.9643558263778687, -0.5900033116340637, 0.152094304561615, -0.23629583418369293], [-0.6758818626403809, -0.5784849524497986, 1.2839032411575317, -0.5652052164077759, 0.5422675609588623, 0.708106279373169, -0.23343494534492493, -0.1354549676179886, 1.241170048713684, 0.9100873470306396, 0.49880439043045044, -0.5774974822998047, -0.14252905547618866, -1.4764289855957031, 2.121335744857788, -0.5925326347351074, -0.565711498260498], [0.5516615509986877, 1.7111222743988037, -0.5081571936607361, 0.5088491439819336, 0.7712281346321106, -1.0978264808654785, 2.057523727416992, 0.3908860981464386, 0.1311505287885666, -1.5836396217346191, 0.6241387128829956, 0.4309728741645813, 1.2102411985397339, 0.6870387196540833, 0.514578640460968, 0.44791069626808167, 0.40427589416503906], [-1.045062780380249, 1.792913556098938, -0.47540995478630066, 0.5945501327514648, -0.40573742985725403, 0.1949131339788437, -0.3481943607330322, 0.21003979444503784, -0.17867587506771088, 1.0194307565689087, 1.5370266437530518, 0.07765749096870422, 0.20314063131809235, 0.1174284890294075, -0.6897146105766296, -1.4896425008773804, -2.147407293319702], [-1.2495530843734741, 0.3023548424243927, 0.4781974256038666, 1.088560700416565, -1.3613539934158325, -2.0966885089874268, -1.2065047025680542, -0.08333034813404083, -0.014956247061491013, 0.15236087143421173, -1.3927204608917236, -0.38955336809158325, -0.022291211411356926, 0.1915515959262848, 0.424498051404953, -0.8605777025222778, -1.044689416885376], [-0.12040333449840546, -0.9094305038452148, -0.5806179046630859, 2.328784704208374, -2.974750280380249, -1.665902853012085, 0.4692191779613495, 0.8051718473434448, 1.2660146951675415, -0.3883439898490906, 1.078377366065979, -2.231116533279419, -0.08875216543674469, 3.1635794639587402, 1.0731980800628662, 0.7166609764099121, 0.22103697061538696], [-1.6492207050323486, 2.7500712871551514, 0.5492988228797913, -1.07375168800354, 1.7462900876998901, 1.2472423315048218, 0.06763888895511627, 1.222408413887024, -1.648842692375183, -2.801071882247925, 0.5587285757064819, -1.4212102890014648, -1.3350303173065186, -0.13705167174339294, -0.24082690477371216, 1.2808295488357544, -0.12525931000709534], [0.008543689735233784, 1.720851182937622, -0.46572741866111755, -0.8947381973266602, 1.3153886795043945, -1.5474085807800293, -0.42366310954093933, 0.9718901515007019, -0.1267564594745636, -0.49217671155929565, -0.5310481190681458, 2.6661205291748047, 1.4900469779968262, -0.3650358021259308, -1.3222236633300781, -0.7180961966514587, -2.1330111026763916], [1.3870735168457031, -2.2288105487823486, 2.0747575759887695, 0.8614515066146851, -0.5436177253723145, -0.043105799704790115, -0.4648756980895996, 1.167811632156372, -0.8897534608840942, 0.8783462643623352, 1.6211403608322144, -0.31431615352630615, 0.05932832136750221, -0.0728447288274765, 0.2767157256603241, -1.0095692873001099, -1.1986756324768066], [-1.1453977823257446, 0.04190535098314285, -1.4821254014968872, 0.4624991714954376, 1.0719213485717773, 0.7757446765899658, 0.24889817833900452, 1.58240807056427, -0.41379427909851074, -1.3547660112380981, -0.8833305835723877, 0.6458855867385864, 0.9844874143600464, 1.6396154165267944, 1.3499088287353516, -1.1745271682739258, -1.596384048461914], [2.0528743267059326, 0.018510473892092705, 0.4658055901527405, -0.15372231602668762, -0.36366376280784607, 0.8811660408973694, -0.42070791125297546, 0.33700352907180786, -1.4697229862213135, -0.027923252433538437, -0.7986010313034058, 1.077273964881897, -0.23502130806446075, -0.13982653617858887, 0.21149630844593048, -1.0380724668502808, -1.3208284378051758], [-0.5403220653533936, -0.8029400706291199, -0.5875487327575684, -0.34520190954208374, -0.867133378982544, 0.3818015158176422, -1.0520360469818115, 0.23051364719867706, 0.14352454245090485, 1.6775727272033691, -1.797445297241211, 1.0546139478683472, 0.14047464728355408, 0.7039214968681335, 0.10870252549648285, -1.0100475549697876, 0.7070821523666382], [0.5794859528541565, -0.5889976620674133, 0.1939316689968109, 0.027508728206157684, -0.2332405149936676, -0.184341698884964, 0.04664172604680061, 1.695277214050293, 0.8916943073272705, 0.9946292042732239, -1.53965163230896, 0.9328185319900513, 0.7180827856063843, -0.1461150199174881, -1.6900389194488525, 2.2939865589141846, 0.7973570823669434], [0.6177749037742615, 0.7409769892692566, -0.1523245871067047, 0.3309367597103119, 0.2680709958076477, 0.9201411604881287, 0.7585053443908691, -0.49956178665161133, 0.608304500579834, 1.4132484197616577, -2.286412477493286, -0.1136622279882431, -0.37246882915496826, 1.4577490091323853, -1.756752848625183, 0.7270932197570801, -0.14700932800769806], [-1.3594592809677124, -1.0579378604888916, 0.727493405342102, 0.027313759550452232, 1.0416594743728638, -0.4208802878856659, 1.2803754806518555, -0.26623350381851196, 0.1262645423412323, -0.8376392722129822, -1.1017851829528809, 0.6413938403129578, -1.4150235652923584, -0.9595386981964111, -2.0570995807647705, 0.19827057421207428, 0.25299492478370667], [-0.7469654679298401, -0.17164187133312225, 0.3041100800037384, 0.7292163968086243, -0.27314460277557373, -1.031455636024475, -0.28448691964149475, 0.28115513920783997, 0.6065614819526672, 0.13287241756916046, -0.0032826659735292196, 0.34526392817497253, -1.7996639013290405, 0.15173043310642242, -0.17951907217502594, 2.259446144104004, 0.5689256191253662], [-1.2325892448425293, 0.432657390832901, -0.6180405616760254, -0.5268965363502502, 1.692015528678894, -0.11808576434850693, 0.5646330118179321, -1.015217661857605, 0.6616835594177246, -0.5124821066856384, 0.3979562222957611, 2.161109447479248, 1.287667989730835, 0.22841450572013855, -1.774553656578064, -1.2556008100509644, 0.5185611844062805], [-0.21863724291324615, -1.2255513668060303, -1.5820279121398926, -2.2255666255950928, -1.0334051847457886, 0.2674465775489807, -0.6106505393981934, -0.6118473410606384, -0.5775336623191833, -0.39282578229904175, -1.3154363632202148, -0.14675767719745636, -0.4295620024204254, -0.2612987458705902, -1.2050670385360718, 0.4768948554992676, -0.016362264752388], [0.9346954822540283, 0.9781157374382019, -1.3347461223602295, -1.8384525775909424, -0.7696306109428406, -0.4208522439002991, -0.6246948838233948, 0.8627586960792542, 0.08337126672267914, -1.5452240705490112, -0.7594577074050903, -0.1131066232919693, 0.9554653763771057, -1.1729474067687988, -0.5292380452156067, -0.869990348815918, 0.8173093795776367], [-1.5516958236694336, -1.4033641815185547, -1.9004135131835938, -1.0863865613937378, 0.17922276258468628, -0.9199537634849548, 2.214191436767578, 1.450953722000122, 0.6854308843612671, -0.3694404661655426, 0.32549265027046204, -1.5475231409072876, 0.2206258773803711, -0.9310951232910156, 0.27876806259155273, -0.7424272894859314, -1.2736598253250122], [-0.2615744173526764, 0.9065732359886169, -0.11043272912502289, -0.7379282116889954, 0.8857733607292175, 1.964460849761963, -0.09488055855035782, -1.6217468976974487, -1.586707592010498, 0.5987356901168823, -0.3174974322319031, 1.0155836343765259, 1.0298454761505127, -0.7651979923248291, 0.07096041738986969, 0.271915078163147, -0.035630736500024796], [1.4102288484573364, 1.2110265493392944, -0.9408137202262878, -0.8028277158737183, -0.35489368438720703, -0.4348882734775543, 0.7815552353858948, 0.4811670780181885, 1.163120985031128, -0.4612080454826355, 0.7977160215377808, -0.25975486636161804, -1.2555111646652222, -0.03513447940349579, -0.8383429646492004, -0.525806188583374, -0.5402752757072449], [-0.0361095629632473, -0.2903013229370117, 0.6623964905738831, 0.7997479438781738, -1.2322877645492554, 0.11163884401321411, -0.2540799081325531, -0.1050189957022667, -1.23673677444458, 0.848475992679596, -1.2943733930587769, 0.40525585412979126, -0.15846553444862366, -0.3422660529613495, -0.4562983810901642, 0.601824164390564, -1.0211083889007568], [0.3094329237937927, -0.22882507741451263, 0.21094255149364471, -1.1497572660446167, -0.05109144002199173, -1.6848443746566772, -0.5173606872558594, 1.5746766328811646, 1.7103074789047241, 0.259977787733078, 1.614962100982666, -0.2722133696079254, -1.5183452367782593, -0.9752135872840881, 0.9173703193664551, 3.515455961227417, 0.7376238107681274], [0.49364808201789856, -0.772544264793396, 0.6143262386322021, 0.4332350492477417, -1.7002235651016235, 0.47939732670783997, -0.5673176646232605, -0.5384065508842468, -0.24646523594856262, -0.1048518642783165, -0.3330121636390686, 0.420552134513855, -1.4047054052352905, 1.8434032201766968, 0.6020156145095825, 0.30819886922836304, 0.4378381073474884], [-0.6352910995483398, 1.138601303100586, 0.20686626434326172, 0.946063220500946, 0.5006831288337708, 1.226668357849121, 0.49340423941612244, -0.4577915668487549, -0.27042311429977417, -0.7962584495544434, -0.4546493887901306, 0.7287143468856812, -0.6521199345588684, 0.9579348564147949, -1.4792474508285522, 0.3278404176235199, 1.033156156539917], [0.8408260941505432, 1.1389636993408203, 0.7954379916191101, 0.5595601201057434, -0.1998487114906311, -0.15203730762004852, -0.931091845035553, -1.3255691528320312, -0.44701501727104187, -0.2973499596118927, -0.24614176154136658, -0.5580992698669434, 0.40407323837280273, 0.6977592706680298, 0.07623913139104843, -0.08083760738372803, -0.09772683680057526], [1.203027606010437, -0.4909696578979492, -1.4037870168685913, -0.5042696595191956, 0.7381492257118225, 0.023538032546639442, 1.3849595785140991, 0.48389294743537903, 0.8274803161621094, -0.9975430369377136, 0.1533399224281311, -0.6661942005157471, -0.5165414214134216, 1.867452621459961, 0.18687790632247925, 0.23502546548843384, -1.0577248334884644], [0.18769970536231995, 1.067959189414978, -1.3507542610168457, 1.3666739463806152, -1.6077682971954346, 2.793058156967163, -2.8941268920898438, -0.21862898766994476, -0.7221524119377136, -0.2007138729095459, -0.4705173373222351, 0.5550577640533447, -0.5741846561431885, -1.0090417861938477, 0.5857229828834534, -0.31036391854286194, -0.5540643334388733], [0.047170255333185196, 1.8556506633758545, -0.44987308979034424, 0.26477885246276855, 0.711428701877594, 0.2619304955005646, -1.2448586225509644, 0.14002981781959534, -1.7405240535736084, -1.4477996826171875, -0.21136435866355896, -0.05803564935922623, 0.36648935079574585, -2.5891692638397217, -0.6409109830856323, -1.4942020177841187, -1.5988367795944214], [0.6939666867256165, 0.21389763057231903, 0.5514057874679565, -1.2889313697814941, -0.16024518013000488, 0.5008809566497803, 0.2766536474227905, 0.8699880242347717, -0.02759844996035099, -0.23642216622829437, 2.072726249694824, 2.2328922748565674, 0.5775668621063232, -0.5768400430679321, -0.06570982933044434, 0.9932706356048584, 0.8112373352050781], [-0.914104163646698, 0.2797974646091461, 0.05268091708421707, -0.09410616755485535, -0.9936376810073853, -0.12986716628074646, 0.3902595639228821, 0.6416707038879395, 0.6414533853530884, 1.2036250829696655, -1.7762466669082642, 0.04877133294939995, 3.182833433151245, 1.2089037895202637, -0.9960243105888367, 1.7463726997375488, -0.6694440245628357], [0.023553647100925446, 0.013290663249790668, 0.056088197976350784, 1.3666919469833374, -0.04914776235818863, -0.8678648471832275, -0.9176263213157654, -0.5632599592208862, 1.162980318069458, 1.0255892276763916, 1.805193305015564, -1.3747667074203491, -1.1056997776031494, 0.13598579168319702, 0.6376349925994873, -1.9513981342315674, 0.8226718306541443], [-2.7150371074676514, -0.30396971106529236, 0.4102361798286438, -0.9627500772476196, 0.8949965238571167, -0.25788795948028564, 0.4883008599281311, -2.4548587799072266, 2.382915496826172, -0.40975335240364075, -0.8163479566574097, 0.7351853847503662, -0.08246473222970963, 1.0393025875091553, -1.2114506959915161, 0.823749303817749, -0.5709015727043152], [-0.3779210150241852, -0.9839464426040649, 0.8875642418861389, 0.9106884598731995, -0.6731351017951965, -0.11267514526844025, -0.13672739267349243, -0.15128690004348755, -0.4355616569519043, -0.6545403003692627, -0.936414361000061, 0.04980747029185295, 0.003747417591512203, -0.07919634878635406, -1.6152857542037964, -0.8524101376533508, 0.8053380846977234], [0.8266510367393494, -0.347174733877182, 0.9613128900527954, 0.47722408175468445, -0.781343400478363, -1.4586164951324463, 1.6471987962722778, 0.5104811787605286, -0.19088925421237946, 0.9094066619873047, -0.27016517519950867, 0.928984522819519, 1.3637807369232178, 0.7752188444137573, -1.2985575199127197, 0.1970537155866623, -1.2907845973968506], [-0.9429466128349304, 0.6181135773658752, 0.13385747373104095, -0.4884302616119385, 0.810080885887146, -0.2652156949043274, -1.434682011604309, 0.3785608112812042, 0.46983614563941956, -1.7400598526000977, -0.9638985395431519, 0.992096483707428, 0.7690914273262024, -0.1521931290626526, 0.9456242322921753, -0.22439004480838776, -1.058606505393982], [-0.550275981426239, 0.08035571873188019, -0.5947614312171936, -0.8247841596603394, -0.36024296283721924, -0.06715735793113708, 2.047778367996216, -0.5923676490783691, -0.2915455400943756, 0.17402410507202148, 0.2173229306936264, 0.14145304262638092, 1.2919989824295044, -0.23118354380130768, -1.3819658756256104, -0.5435879230499268, -0.5732836127281189], [-0.0015474514802917838, 0.26343590021133423, 0.07843798398971558, -0.40902644395828247, -0.8865757584571838, -0.10265616327524185, -0.02647005021572113, -0.4509248733520508, 2.6557750701904297, -0.3442482650279999, -0.5924245715141296, 0.5047276616096497, 0.7994343638420105, 0.07009468227624893, 1.6160200834274292, 1.2507870197296143, -0.7989500164985657], [-0.2551763653755188, -0.7652575969696045, -0.08653775602579117, -0.7540591359138489, 1.0431060791015625, 0.7916714549064636, 0.24842707812786102, -0.828942060470581, 0.7449548840522766, 0.8637508153915405, -0.7482186555862427, 0.5851341485977173, -2.6169192790985107, 1.240096926689148, -2.1643271446228027, -2.284346580505371, -0.6727381348609924], [-1.2267102003097534, 1.0007063150405884, 0.7164575457572937, -0.44632968306541443, 0.029231781139969826, -0.04592696949839592, 1.4276593923568726, -1.330704689025879, -1.4807195663452148, 1.188172459602356, 2.23276424407959, 0.11413946747779846, 1.4916491508483887, -0.0532902367413044, -0.008555573411285877, -0.5885509848594666, -0.23677709698677063], [0.45527100563049316, 0.9817196130752563, 0.40005946159362793, 0.5288305282592773, -0.46837133169174194, -0.021967146545648575, -0.3091752529144287, -0.3765672445297241, 0.6855042576789856, 0.7501175403594971, 0.7853220105171204, -2.512352466583252, 0.4617656171321869, 0.4794686436653137, 1.8493008613586426, -0.7249227166175842, 0.4868258237838745], [0.7995337843894958, -1.2762441635131836, 1.5185694694519043, -0.3811017572879791, 0.44407129287719727, -0.13568052649497986, 0.11827884614467621, 0.9298524260520935, -0.45130205154418945, -3.16549015045166, 0.5403849482536316, -0.04149356111884117, 1.3236613273620605, -1.4993215799331665, -0.3331741988658905, 0.311042845249176, 1.6129798889160156]], "dtype": "float32"}], "e": {"__nd__": [[1.1614781618118286, -1.168588638305664, 0.6685959100723267, 0.11078453063964844, 0.4656485319137573, -0.027509579434990883, 1.8085906505584717, -0.5511820912361145, -0.25441375374794006, -0.11204741150140762, 0.03570594638586044, 0.08488547056913376, -1.8130311965942383, 0.9036855101585388, -0.17652393877506256, 1.6382287740707397, -0.05251174047589302], [0.5520599484443665, -0.09989111125469208, 0.5731106400489807, -0.7807968854904175, -0.614578366279602, 1.231959581375122, 0.4062361717224121, 0.8830996751785278, 1.2442636489868164, 1.054718017578125, -1.9387959241867065, -0.15408584475517273, 1.054183840751648, -0.5511282086372375, -0.3748876452445984, 0.3601100742816925, -0.3831285834312439], [-0.5112664103507996, -1.7379724979400635, 1.6845417022705078, -0.3898513615131378, 2.945523977279663, -1.045864224433899, 0.24680884182453156, 0.2891887426376343, 0.0024194081779569387, -0.4076298475265503, 0.25311264395713806, -1.4353830814361572, -0.1887165904045105, -0.02244318276643753, -0.5802040696144104, -0.6742333769798279, -0.9921287894248962], [-0.9625059962272644, 0.0908195748925209, -1.255054235458374, 0.3071123957633972, 0.34202098846435547, 1.1500474214553833, 1.3558664321899414, 0.9277841448783875, -1.231643795967102, 0.7370036840438843, -0.21373668313026428, -0.731985330581665, 1.6993502378463745, -1.1459869146347046, 0.8108246922492981, 0.1941567063331604, 0.5165068507194519], [-1.2926125526428223, 0.7165169715881348, -0.39561039209365845, -1.7015711069107056, -3.104464054107666, 0.8925644159317017, -1.1426937580108643, 1.1177655458450317, -0.2948853075504303, 0.06843023002147675, -0.09427052736282349, 0.9387674331665039, 0.14421169459819794, 0.24477317929267883, 0.9423349499702454, 1.886193037033081, -0.9501217603683472], [0.4471232295036316, 0.28228408098220825, 1.006272792816162, -0.7578164935112, -0.0006004875176586211, -0.3487266004085541, -0.19105631113052368, 0.10195944458246231, 1.6157519817352295, 0.8078869581222534, 0.6466975808143616, -1.1457979679107666, -0.2978084981441498, -0.9643558263778687, -0.5900033116340637, 0.152094304561615, -0.23629583418369293], [-0.6758818626403809, -0.5784849524497986, 1.2839032411575317, -0.5652052164077759, 0.5422675609588623, 0.708106279373169, -0.23343494534492493, -0.1354549676179886, 1.241170048713684, 0.9100873470306396, 0.49880439043045044, -0.5774974822998047, -0.14252905547618866, -1.4764289855957031, 2.121335744857788, -0.5925326347351074, -0.565711498260498], [0.5516615509986877, 1.7111222743988037, -0.5081571936607361, 0.5088491439819336, 0.7712281346321106, -1.0978264808654785, 2.057523727416992, 0.3908860981464386, 0.1311505287885666, -1.5836396217346191, 0.6241387128829956, 0.4309728741645813, 1.2102411985397339, 0.6870387196540833, 0.514578640460968, 0.44791069626808167, 0.40427589416503906], [-1.045062780380249, 1.792913556098938, -0.47540995478630066, 0.5945501327514648, -0.40573742985725403, 0.1949131339788437, -0.3481943607330322, 0.21003979444503784, -0.17867587506771088, 1.0194307565689087, 1.5370266437530518, 0.07765749096870422, 0.20314063131809235, 0.1174284890294075, -0.6897146105766296, -1.4896425008773804, -2.147407293319702], [-1.2495530843734741, 0.3023548424243927, 0.4781974256038666, 1.088560700416565, -1.3613539934158325, -2.0966885089874268, -1.2065047025680542, -0.08333034813404083, -0.014956247061491013, 0.15236087143421173, -1.3927204608917236, -0.38955336809158325, -0.022291211411356926, 0.1915515959262848, 0.424498051404953, -0.8605777025222778, -1.044689416885376], [-0.12040333449840546, -0.9094305038452148, -0.5806179046630859, 2.328784704208374, -2.974750280380249, -1.665902853012085, 0.4692191779613495, 0.8051718473434448, 1.2660146951675415, -0.3883439898490906, 1.078377366065979, -2.231116533279419, -0.08875216543674469, 3.1635794639587402, 1.0731980800628662, 0.7166609764099121, 0.22103697061538696], [-1.6492207050323486, 2.7500712871551514, 0.5492988228797913, -1.07375168800354, 1.7462900876998901, 1.2472423315048218, 0.06763888895511627, 1.222408413887024, -1.648842692375183, -2.801071882247925, 0.5587285757064819, -1.4212102890014648, -1.3350303173065186, -0.13705167174339294, -0.24082690477371216, 1.2808295488357544, -0.12525931000709534], [0.008543689735233784, 1.720851182937622, -0.46572741866111755, -0.8947381973266602, 1.3153886795043945, -1.5474085807800293, -0.42366310954093933, 0.9718901515007019, -0.1267564594745636, -0.49217671155929565, -0.5310481190681458, 2.6661205291748047, 1.4900469779968262, -0.3650358021259308, -1.3222236633300781, -0.7180961966514587, -2.1330111026763916], [1.3870735168457031, -2.2288105487823486, 2.0747575759887695, 0.8614515066146851, -0.5436177253723145, -0.043105799704790115, -0.4648756980895996, 1.167811632156372, -0.8897534608840942, 0.8783462643623352, 1.6211403608322144, -0.31431615352630615, 0.05932832136750221, -0.0728447288274765, 0.2767157256603241, -1.0095692873001099, -1.1986756324768066], [-1.1453977823257446, 0.04190535098314285, -1.4821254014968872, 0.4624991714954376, 1.0719213485717773, 0.7757446765899658, 0.24889817833900452, 1.58240807056427, -0.41379427909851074, -1.3547660112380981, -0.8833305835723877, 0.6458855867385864, 0.9844874143600464, 1.6396154165267944, 1.3499088287353516, -1.1745271682739258, -1.596384048461914], [2.0528743267059326, 0.018510473892092705, 0.4658055901527405, -0.15372231602668762, -0.36366376280784607, 0.8811660408973694, -0.42070791125297546, 0.33700352907180786, -1.4697229862213135, -0.027923252433538437, -0.7986010313034058, 1.077273964881897, -0.23502130806446075, -0.13982653617858887, 0.21149630844593048, -1.0380724668502808, -1.3208284378051758], [-0.5403220653533936, -0.8029400706291199, -0.5875487327575684, -0.34520190954208374, -0.867133378982544, 0.3818015158176422, -1.0520360469818115, 0.23051364719867706, 0.14352454245090485, 1.6775727272033691, -1.797445297241211, 1.0546139478683472, 0.14047464728355408, 0.7039214968681335, 0.10870252549648285, -1.0100475549697876, 0.7070821523666382], [0.5794859528541565, -0.5889976620674133, 0.1939316689968109, 0.027508728206157684, -0.2332405149936676, -0.184341698884964, 0.04664172604680061, 1.695277214050293, 0.8916943073272705, 0.9946292042732239, -1.53965163230896, 0.9328185319900513, 0.7180827856063843, -0.1461150199174881, -1.6900389194488525, 2.2939865589141846, 0.7973570823669434], [0.6177749037742615, 0.7409769892692566, -0.1523245871067047, 0.3309367597103119, 0.2680709958076477, 0.9201411604881287, 0.7585053443908691, -0.49956178665161133, 0.608304500579834, 1.4132484197616577, -2.286412477493286, -0.1136622279882431, -0.37246882915496826, 1.4577490091323853, -1.756752848625183, 0.7270932197570801, -0.14700932800769806], [-1.3594592809677124, -1.0579378604888916, 0.727493405342102, 0.027313759550452232, 1.0416594743728638, -0.4208802878856659, 1.2803754806518555, -0.26623350381851196, 0.1262645423412323, -0.8376392722129822, -1.1017851829528809, 0.6413938403129578, -1.4150235652923584, -0.9595386981964111, -2.0570995807647705, 0.19827057421207428, 0.25299492478370667], [-0.7469654679298401, -0.17164187133312225, 0.3041100800037384, 0.7292163968086243, -0.27314460277557373, -1.031455636024475, -0.28448691964149475, 0.28115513920783997, 0.6065614819526672, 0.13287241756916046, -0.0032826659735292196, 0.34526392817497253, -1.7996639013290405, 0.15173043310642242, -0.17951907217502594, 2.259446144104004, 0.5689256191253662], [-1.2325892448425293, 0.432657390832901, -0.6180405616760254, -0.5268965363502502, 1.692015528678894, -0.11808576434850693, 0.5646330118179321, -1.015217661857605, 0.6616835594177246, -0.5124821066856384, 0.3979562222957611, 2.161109447479248, 1.287667989730835, 0.22841450572013855, -1.774553656578064, -1.2556008100509644, 0.5185611844062805], [-0.21863724291324615, -1.2255513668060303, -1.5820279121398926, -2.2255666255950928, -1.0334051847457886, 0.2674465775489807, -0.6106505393981934, -0.6118473410606384, -0.5775336623191833, -0.39282578229904175, -1.3154363632202148, -0.14675767719745636, -0.4295620024204254, -0.2612987458705902, -1.2050670385360718, 0.4768948554992676, -0.016362264752388], [0.9346954822540283, 0.9781157374382019, -1.3347461223602295, -1.8384525775909424, -0.7696306109428406, -0.4208522439002991, -0.6246948838233948, 0.8627586960792542, 0.08337126672267914, -1.5452240705490112, -0.7594577074050903, -0.1131066232919693, 0.9554653763771057, -1.1729474067687988, -0.5292380452156067, -0.869990348815918, 0.8173093795776367], [-1.5516958236694336, -1.4033641815185547, -1.9004135131835938, -1.0863865613937378, 0.17922276258468628, -0.9199537634849548, 2.214191436767578, 1.450953722000122, 0.6854308843612671, -0.3694404661655426, 0.32549265027046204, -1.5475231409072876, 0.2206258773803711, -0.9310951232910156, 0.27876806259155273, -0.7424272894859314, -1.2736598253250122], [-0.2615744173526764, 0.9065732359886169, -0.11043272912502289, -0.7379282116889954, 0.8857733607292175, 1.964460849761963, -0.09488055855035782, -1.6217468976974487, -1.586707592010498, 0.5987356901168823, -0.3174974322319031, 1.0155836343765259, 1.0298454761505127, -0.7651979923248291, 0.07096041738986969, 0.271915078163147, -0.035630736500024796], [1.4102288484573364, 1.2110265493392944, -0.9408137202262878, -0.8028277158737183, -0.35489368438720703, -0.4348882734775543, 0.7815552353858948, 0.4811670780181885, 1.163120985031128, -0.4612080454826355, 0.7977160215377808, -0.25975486636161804, -1.2555111646652222, -0.03513447940349579, -0.8383429646492004, -0.525806188583374, -0.5402752757072449], [-0.0361095629632473, -0.2903013229370117, 0.6623964905738831, 0.7997479438781738, -1.2322877645492554, 0.11163884401321411, -0.2540799081325531, -0.1050189957022667, -1.23673677444458, 0.848475992679596, -1.2943733930587769, 0.40525585412979126, -0.15846553444862366, -0.3422660529613495, -0.4562983810901642, 0.601824164390564, -1.0211083889007568], [0.3094329237937927, -0.22882507741451263, 0.21094255149364471, -1.1497572660446167, -0.05109144002199173, -1.6848443746566772, -0.5173606872558594, 1.5746766328811646, 1.7103074789047241, 0.259977787733078, 1.614962100982666, -0.2722133696079254, -1.5183452367782593, -0.9752135872840881, 0.9173703193664551, 3.515455961227417, 0.7376238107681274], [0.49364808201789856, -0.772544264793396, 0.6143262386322021, 0.4332350492477417, -1.7002235651016235, 0.47939732670783997, -0.5673176646232605, -0.5384065508842468, -0.24646523594856262, -0.1048518642783165, -0.3330121636390686, 0.420552134513855, -1.4047054052352905, 1.8434032201766968, 0.6020156145095825, 0.30819886922836304, 0.4378381073474884], [-0.6352910995483398, 1.138601303100586, 0.20686626434326172, 0.946063220500946, 0.5006831288337708, 1.226668357849121, 0.49340423941612244, -0.4577915668487549, -0.27042311429977417, -0.7962584495544434, -0.4546493887901306, 0.7287143468856812, -0.6521199345588684, 0.9579348564147949, -1.4792474508285522, 0.3278404176235199, 1.033156156539917], [0.8408260941505432, 1.1389636993408203, 0.7954379916191101, 0.5595601201057434, -0.1998487114906311, -0.15203730762004852, -0.931091845035553, -1.3255691528320312, -0.44701501727104187, -0.2973499596118927, -0.24614176154136658, -0.5580992698669434, 0.40407323837280273, 0.6977592706680298, 0.07623913139104843, -0.08083760738372803, -0.09772683680057526], [1.203027606010437, -0.4909696578979492, -1.4037870168685913, -0.5042696595191956, 0.7381492257118225, 0.023538032546639442, 1.3849595785140991, 0.48389294743537903, 0.8274803161621094, -0.9975430369377136, 0.1533399224281311, -0.6661942005157471, -0.5165414214134216, 1.867452621459961, 0.18687790632247925, 0.23502546548843384, -1.0577248334884644], [0.18769970536231995, 1.067959189414978, -1.3507542610168457, 1.3666739463806152, -1.6077682971954346, 2.793058156967163, -2.8941268920898438, -0.21862898766994476, -0.7221524119377136, -0.2007138729095459, -0.4705173373222351, 0.5550577640533447, -0.5741846561431885, -1.0090417861938477, 0.5857229828834534, -0.31036391854286194, -0.5540643334388733], [0.047170255333185196, 1.8556506633758545, -0.44987308979034424, 0.26477885246276855, 0.711428701877594, 0.2619304955005646, -1.2448586225509644, 0.14002981781959534, -1.7405240535736084, -1.4477996826171875, -0.21136435866355896, -0.05803564935922623, 0.36648935079574585, -2.5891692638397217, -0.6409109830856323, -1.4942020177841187, -1.5988367795944214], [0.6939666867256165, 0.21389763057231903, 0.5514057874679565, -1.2889313697814941, -0.16024518013000488, 0.5008809566497803, 0.2766536474227905, 0.8699880242347717, -0.02759844996035099, -0.23642216622829437, 2.072726249694824, 2.2328922748565674, 0.5775668621063232, -0.5768400430679321, -0.06570982933044434, 0.9932706356048584, 0.8112373352050781], [-0.914104163646698, 0.2797974646091461, 0.05268091708421707, -0.09410616755485535, -0.9936376810073853, -0.12986716628074646, 0.3902595639228821, 0.6416707038879395, 0.6414533853530884, 1.2036250829696655, -1.7762466669082642, 0.04877133294939995, 3.182833433151245, 1.2089037895202637, -0.9960243105888367, 1.7463726997375488, -0.6694440245628357], [0.023553647100925446, 0.013290663249790668, 0.056088197976350784, 1.3666919469833374, -0.04914776235818863, -0.8678648471832275, -0.9176263213157654, -0.5632599592208862, 1.162980318069458, 1.0255892276763916, 1.805193305015564, -1.3747667074203491, -1.1056997776031494, 0.13598579168319702, 0.6376349925994873, -1.9513981342315674, 0.8226718306541443], [-2.7150371074676514, -0.30396971106529236, 0.4102361798286438, -0.9627500772476196, 0.8949965238571167, -0.25788795948028564, 0.4883008599281311, -2.4548587799072266, 2.382915496826172, -0.40975335240364075, -0.8163479566574097, 0.7351853847503662, -0.08246473222970963, 1.0393025875091553, -1.2114506959915161, 0.823749303817749, -0.5709015727043152], [-0.3779210150241852, -0.9839464426040649, 0.8875642418861389, 0.9106884598731995, -0.6731351017951965, -0.11267514526844025, -0.13672739267349243, -0.15128690004348755, -0.4355616569519043, -0.6545403003692627, -0.936414361000061, 0.04980747029185295, 0.003747417591512203, -0.07919634878635406, -1.6152857542037964, -0.8524101376533508, 0.8053380846977234], [0.8266510367393494, -0.347174733877182, 0.9613128900527954, 0.47722408175468445, -0.781343400478363, -1.4586164951324463, 1.6471987962722778, 0.5104811787605286, -0.19088925421237946, 0.9094066619873047, -0.27016517519950867, 0.928984522819519, 1.3637807369232178, 0.7752188444137573, -1.2985575199127197, 0.1970537155866623, -1.2907845973968506], [-0.9429466128349304, 0.6181135773658752, 0.13385747373104095, -0.4884302616119385, 0.810080885887146, -0.2652156949043274, -1.434682011604309, 0.3785608112812042, 0.46983614563941956, -1.7400598526000977, -0.9638985395431519, 0.992096483707428, 0.7690914273262024, -0.1521931290626526, 0.9456242322921753, -0.22439004480838776, -1.058606505393982], [-0.550275981426239, 0.08035571873188019, -0.5947614312171936, -0.8247841596603394, -0.36024296283721924, -0.06715735793113708, 2.047778367996216, -0.5923676490783691, -0.2915455400943756, 0.17402410507202148, 0.2173229306936264, 0.14145304262638092, 1.2919989824295044, -0.23118354380130768, -1.3819658756256104, -0.5435879230499268, -0.5732836127281189], [-0.0015474514802917838, 0.26343590021133423, 0.07843798398971558, -0.40902644395828247, -0.8865757584571838, -0.10265616327524185, -0.02647005021572113, -0.4509248733520508, 2.6557750701904297, -0.3442482650279999, -0.5924245715141296, 0.5047276616096497, 0.7994343638420105, 0.07009468227624893, 1.6160200834274292, 1.2507870197296143, -0.7989500164985657], [-0.2551763653755188, -0.7652575969696045, -0.08653775602579117, -0.7540591359138489, 1.0431060791015625, 0.7916714549064636, 0.24842707812786102, -0.828942060470581, 0.7449548840522766, 0.8637508153915405, -0.7482186555862427, 0.5851341485977173, -2.6169192790985107, 1.240096926689148, -2.1643271446228027, -2.284346580505371, -0.6727381348609924], [-1.2267102003097534, 1.0007063150405884, 0.7164575457572937, -0.44632968306541443, 0.029231781139969826, -0.04592696949839592, 1.4276593923568726, -1.330704689025879, -1.4807195663452148, 1.188172459602356, 2.23276424407959, 0.11413946747779846, 1.4916491508483887, -0.0532902367413044, -0.008555573411285877, -0.5885509848594666, -0.23677709698677063], [0.45527100563049316, 0.9817196130752563, 0.40005946159362793, 0.5288305282592773, -0.46837133169174194, -0.021967146545648575, -0.3091752529144287, -0.3765672445297241, 0.6855042576789856, 0.7501175403594971, 0.7853220105171204, -2.512352466583252, 0.4617656171321869, 0.4794686436653137, 1.8493008613586426, -0.7249227166175842, 0.4868258237838745], [0.7995337843894958, -1.2762441635131836, 1.5185694694519043, -0.3811017572879791, 0.44407129287719727, -0.13568052649497986, 0.11827884614467621, 0.9298524260520935, -0.45130205154418945, -3.16549015045166, 0.5403849482536316, -0.04149356111884117, 1.3236613273620605, -1.4993215799331665, -0.3331741988658905, 0.311042845249176, 1.6129798889160156]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(matmul, json.loads(_TESTS), 'triton', *[0.0001, 0.0001])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")